# AKANE E11-3：箱の形のベクトルを GPU で学習する（部分と全体・病気・地理）

E11-3 の比べのうち、箱の学習だけが手元の計算機では重すぎたので、ここで流します（中身と数は E11-2 と同じ）。
正解はこのノートに入っていません。手で書いた語の一覧もありません。材料は E10-1_results の鎖と着きです。

## 手順
1. 「ランタイム」→「ランタイムのタイプを変更」→ **L4 GPU**（T4 でも可）。
2. 「すべてのセルを実行」。ドライブの許可を聞かれたら許可。
3. 終わると `AKANE_アーキテクチャ/E11-3_results/box_P.json`・`box_D.json`・`box_G.json` ができます。20〜40分の見込み。

In [ ]:
import os, json, gzip, time, sys, types
import numpy as np
from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/AKANE_アーキテクチャ"
R101 = f"{BASE}/E10-1_results"; OUT = f"{BASE}/E11-3_results"; os.makedirs(OUT, exist_ok=True)
for n in ["E101_steps", "E101_res_P", "E101_res_D", "E101_res_G"]:
    assert os.path.exists(f"{R101}/{n}.json.gz"), n
os.environ["BOX_DEVICE"] = "cuda"

In [ ]:
# 箱の中身（analysis/e11/e112_box.py と同じ）
"""E11-2 の箱の形のベクトル（計画 plans/E11-2_deep_chain_and_box.md「手で決めた所」の数のまま）。
E8 の出した組だけで学ぶ：段（意味→頭の名詞の意味、重み＝段の NLI の含意）と、着き（意味→分類名、重み＝着きの NLI の含意）。
WordNet も正解も使わない。箱は Li ほか 2019（ICLR、softbox）の滑らかな体積、P(A の中に B) = vol(A∩B)/vol(A)。"""
import numpy as np
import torch

DIM, EPOCHS, LR, NEG, BETA = 32, 200, 0.01, 5, 1.0
import os
DEVICE = os.environ.get("BOX_DEVICE", "cpu")   # E11-3 の Colab では "cuda"（数は同じ）


def log_vol(lo, hi):
    return torch.nn.functional.softplus(hi - lo, beta=1 / BETA).clamp_min(1e-10).log().sum(-1)


class Box(torch.nn.Module):
    def __init__(self, n, seed):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.z = torch.nn.Parameter((torch.rand(n, DIM, generator=g) * 0.1).to(DEVICE))
        self.d = torch.nn.Parameter((torch.rand(n, DIM, generator=g) * 0.1 + 1.0).to(DEVICE))

    def logp(self, a, b):
        """log P(a は b の一種) = log vol(a∩b) − log vol(a)"""
        za, Za = self.z[a], self.z[a] + torch.nn.functional.softplus(self.d[a])
        zb, Zb = self.z[b], self.z[b] + torch.nn.functional.softplus(self.d[b])
        return (log_vol(torch.maximum(za, zb), torch.minimum(Za, Zb)) - log_vol(za, Za)).clamp(max=0.0)


def train(n, pairs, w, seed, log=None):
    """pairs: (子, 親) の番号の配列、w: 0〜1 の重み（目標の確からしさ）。負の組は親をでたらめに入れ替えて目標0。"""
    torch.manual_seed(seed)
    m = Box(n, seed); opt = torch.optim.Adam(m.parameters(), lr=LR)
    a = torch.tensor(pairs[:, 0], device=DEVICE); b = torch.tensor(pairs[:, 1], device=DEVICE); t = torch.tensor(w, dtype=torch.float32, device=DEVICE)
    rng = np.random.default_rng(seed)
    for ep in range(EPOCHS):
        nb = torch.tensor(rng.integers(0, n, (len(a), NEG)), device=DEVICE)
        lp = m.logp(a, b); ln = m.logp(a[:, None].expand(-1, NEG), nb)
        p = lp.exp().clamp(1e-6, 1 - 1e-6); q = ln.exp().clamp(1e-6, 1 - 1e-6)
        loss = -(t * p.log() + (1 - t) * (1 - p).log()).mean() - (1 - q).log().mean()
        opt.zero_grad(); loss.backward(); opt.step()
        if log and ep % 50 == 0: log(f"  箱 種{seed} 回{ep} 損 {loss.item():.4f}")
    return m


def score(m, a, b):
    with torch.no_grad():
        return m.logp(torch.tensor(a, device=DEVICE), torch.tensor(b, device=DEVICE)).exp().cpu().numpy()

In [ ]:
BXM = types.SimpleNamespace(train=train, score=score)
ITEMS = json.loads(r'''{"items": [{"id": 0, "dom": "P", "ent": "action", "cls": "architecture"}, {"id": 1, "dom": "P", "ent": "action", "cls": "bacteria"}, {"id": 2, "dom": "P", "ent": "action", "cls": "beef"}, {"id": 3, "dom": "P", "ent": "action", "cls": "bridge"}, {"id": 4, "dom": "P", "ent": "action", "cls": "bud"}, {"id": 5, "dom": "P", "ent": "action", "cls": "cat"}, {"id": 6, "dom": "P", "ent": "action", "cls": "chapel"}, {"id": 7, "dom": "P", "ent": "action", "cls": "chick"}, {"id": 8, "dom": "P", "ent": "action", "cls": "clock"}, {"id": 9, "dom": "P", "ent": "action", "cls": "deer"}, {"id": 10, "dom": "P", "ent": "action", "cls": "dog"}, {"id": 11, "dom": "P", "ent": "action", "cls": "elephant"}, {"id": 12, "dom": "P", "ent": "action", "cls": "embassy"}, {"id": 13, "dom": "P", "ent": "action", "cls": "garage"}, {"id": 14, "dom": "P", "ent": "action", "cls": "highway"}, {"id": 15, "dom": "P", "ent": "action", "cls": "kidney"}, {"id": 16, "dom": "P", "ent": "action", "cls": "lamb"}, {"id": 17, "dom": "P", "ent": "action", "cls": "male"}, {"id": 18, "dom": "P", "ent": "action", "cls": "marijuana"}, {"id": 19, "dom": "P", "ent": "action", "cls": "puppy"}, {"id": 20, "dom": "P", "ent": "action", "cls": "receiver"}, {"id": 21, "dom": "P", "ent": "action", "cls": "reed"}, {"id": 22, "dom": "P", "ent": "action", "cls": "rifle"}, {"id": 23, "dom": "P", "ent": "action", "cls": "road"}, {"id": 24, "dom": "P", "ent": "action", "cls": "salmon"}, {"id": 25, "dom": "P", "ent": "action", "cls": "shark"}, {"id": 26, "dom": "P", "ent": "action", "cls": "sheep"}, {"id": 27, "dom": "P", "ent": "action", "cls": "shook"}, {"id": 28, "dom": "P", "ent": "action", "cls": "tank"}, {"id": 29, "dom": "P", "ent": "action", "cls": "wheat"}, {"id": 30, "dom": "P", "ent": "aerial", "cls": "architecture"}, {"id": 31, "dom": "P", "ent": "aerial", "cls": "bacteria"}, {"id": 32, "dom": "P", "ent": "aerial", "cls": "beef"}, {"id": 33, "dom": "P", "ent": "aerial", "cls": "bridge"}, {"id": 34, "dom": "P", "ent": "aerial", "cls": "bud"}, {"id": 35, "dom": "P", "ent": "aerial", "cls": "cat"}, {"id": 36, "dom": "P", "ent": "aerial", "cls": "chapel"}, {"id": 37, "dom": "P", "ent": "aerial", "cls": "chick"}, {"id": 38, "dom": "P", "ent": "aerial", "cls": "clock"}, {"id": 39, "dom": "P", "ent": "aerial", "cls": "deer"}, {"id": 40, "dom": "P", "ent": "aerial", "cls": "dog"}, {"id": 41, "dom": "P", "ent": "aerial", "cls": "elephant"}, {"id": 42, "dom": "P", "ent": "aerial", "cls": "embassy"}, {"id": 43, "dom": "P", "ent": "aerial", "cls": "garage"}, {"id": 44, "dom": "P", "ent": "aerial", "cls": "highway"}, {"id": 45, "dom": "P", "ent": "aerial", "cls": "kidney"}, {"id": 46, "dom": "P", "ent": "aerial", "cls": "lamb"}, {"id": 47, "dom": "P", "ent": "aerial", "cls": "male"}, {"id": 48, "dom": "P", "ent": "aerial", "cls": "marijuana"}, {"id": 49, "dom": "P", "ent": "aerial", "cls": "puppy"}, {"id": 50, "dom": "P", "ent": "aerial", "cls": "receiver"}, {"id": 51, "dom": "P", "ent": "aerial", "cls": "reed"}, {"id": 52, "dom": "P", "ent": "aerial", "cls": "rifle"}, {"id": 53, "dom": "P", "ent": "aerial", "cls": "road"}, {"id": 54, "dom": "P", "ent": "aerial", "cls": "salmon"}, {"id": 55, "dom": "P", "ent": "aerial", "cls": "shark"}, {"id": 56, "dom": "P", "ent": "aerial", "cls": "sheep"}, {"id": 57, "dom": "P", "ent": "aerial", "cls": "shook"}, {"id": 58, "dom": "P", "ent": "aerial", "cls": "tank"}, {"id": 59, "dom": "P", "ent": "aerial", "cls": "wheat"}, {"id": 60, "dom": "P", "ent": "annex", "cls": "architecture"}, {"id": 61, "dom": "P", "ent": "annex", "cls": "bacteria"}, {"id": 62, "dom": "P", "ent": "annex", "cls": "beef"}, {"id": 63, "dom": "P", "ent": "annex", "cls": "bridge"}, {"id": 64, "dom": "P", "ent": "annex", "cls": "bud"}, {"id": 65, "dom": "P", "ent": "annex", "cls": "cat"}, {"id": 66, "dom": "P", "ent": "annex", "cls": "chapel"}, {"id": 67, "dom": "P", "ent": "annex", "cls": "chick"}, {"id": 68, "dom": "P", "ent": "annex", "cls": "clock"}, {"id": 69, "dom": "P", "ent": "annex", "cls": "deer"}, {"id": 70, "dom": "P", "ent": "annex", "cls": "dog"}, {"id": 71, "dom": "P", "ent": "annex", "cls": "elephant"}, {"id": 72, "dom": "P", "ent": "annex", "cls": "embassy"}, {"id": 73, "dom": "P", "ent": "annex", "cls": "garage"}, {"id": 74, "dom": "P", "ent": "annex", "cls": "highway"}, {"id": 75, "dom": "P", "ent": "annex", "cls": "kidney"}, {"id": 76, "dom": "P", "ent": "annex", "cls": "lamb"}, {"id": 77, "dom": "P", "ent": "annex", "cls": "male"}, {"id": 78, "dom": "P", "ent": "annex", "cls": "marijuana"}, {"id": 79, "dom": "P", "ent": "annex", "cls": "puppy"}, {"id": 80, "dom": "P", "ent": "annex", "cls": "receiver"}, {"id": 81, "dom": "P", "ent": "annex", "cls": "reed"}, {"id": 82, "dom": "P", "ent": "annex", "cls": "rifle"}, {"id": 83, "dom": "P", "ent": "annex", "cls": "road"}, {"id": 84, "dom": "P", "ent": "annex", "cls": "salmon"}, {"id": 85, "dom": "P", "ent": "annex", "cls": "shark"}, {"id": 86, "dom": "P", "ent": "annex", "cls": "sheep"}, {"id": 87, "dom": "P", "ent": "annex", "cls": "shook"}, {"id": 88, "dom": "P", "ent": "annex", "cls": "tank"}, {"id": 89, "dom": "P", "ent": "annex", "cls": "wheat"}, {"id": 90, "dom": "P", "ent": "antenna", "cls": "architecture"}, {"id": 91, "dom": "P", "ent": "antenna", "cls": "bacteria"}, {"id": 92, "dom": "P", "ent": "antenna", "cls": "beef"}, {"id": 93, "dom": "P", "ent": "antenna", "cls": "bridge"}, {"id": 94, "dom": "P", "ent": "antenna", "cls": "bud"}, {"id": 95, "dom": "P", "ent": "antenna", "cls": "cat"}, {"id": 96, "dom": "P", "ent": "antenna", "cls": "chapel"}, {"id": 97, "dom": "P", "ent": "antenna", "cls": "chick"}, {"id": 98, "dom": "P", "ent": "antenna", "cls": "clock"}, {"id": 99, "dom": "P", "ent": "antenna", "cls": "deer"}, {"id": 100, "dom": "P", "ent": "antenna", "cls": "dog"}, {"id": 101, "dom": "P", "ent": "antenna", "cls": "elephant"}, {"id": 102, "dom": "P", "ent": "antenna", "cls": "embassy"}, {"id": 103, "dom": "P", "ent": "antenna", "cls": "garage"}, {"id": 104, "dom": "P", "ent": "antenna", "cls": "highway"}, {"id": 105, "dom": "P", "ent": "antenna", "cls": "kidney"}, {"id": 106, "dom": "P", "ent": "antenna", "cls": "lamb"}, {"id": 107, "dom": "P", "ent": "antenna", "cls": "male"}, {"id": 108, "dom": "P", "ent": "antenna", "cls": "marijuana"}, {"id": 109, "dom": "P", "ent": "antenna", "cls": "puppy"}, {"id": 110, "dom": "P", "ent": "antenna", "cls": "receiver"}, {"id": 111, "dom": "P", "ent": "antenna", "cls": "reed"}, {"id": 112, "dom": "P", "ent": "antenna", "cls": "rifle"}, {"id": 113, "dom": "P", "ent": "antenna", "cls": "road"}, {"id": 114, "dom": "P", "ent": "antenna", "cls": "salmon"}, {"id": 115, "dom": "P", "ent": "antenna", "cls": "shark"}, {"id": 116, "dom": "P", "ent": "antenna", "cls": "sheep"}, {"id": 117, "dom": "P", "ent": "antenna", "cls": "shook"}, {"id": 118, "dom": "P", "ent": "antenna", "cls": "tank"}, {"id": 119, "dom": "P", "ent": "antenna", "cls": "wheat"}, {"id": 120, "dom": "P", "ent": "anterior", "cls": "architecture"}, {"id": 121, "dom": "P", "ent": "anterior", "cls": "bacteria"}, {"id": 122, "dom": "P", "ent": "anterior", "cls": "beef"}, {"id": 123, "dom": "P", "ent": "anterior", "cls": "bridge"}, {"id": 124, "dom": "P", "ent": "anterior", "cls": "bud"}, {"id": 125, "dom": "P", "ent": "anterior", "cls": "cat"}, {"id": 126, "dom": "P", "ent": "anterior", "cls": "chapel"}, {"id": 127, "dom": "P", "ent": "anterior", "cls": "chick"}, {"id": 128, "dom": "P", "ent": "anterior", "cls": "clock"}, {"id": 129, "dom": "P", "ent": "anterior", "cls": "deer"}, {"id": 130, "dom": "P", "ent": "anterior", "cls": "dog"}, {"id": 131, "dom": "P", "ent": "anterior", "cls": "elephant"}, {"id": 132, "dom": "P", "ent": "anterior", "cls": "embassy"}, {"id": 133, "dom": "P", "ent": "anterior", "cls": "garage"}, {"id": 134, "dom": "P", "ent": "anterior", "cls": "highway"}, {"id": 135, "dom": "P", "ent": "anterior", "cls": "kidney"}, {"id": 136, "dom": "P", "ent": "anterior", "cls": "lamb"}, {"id": 137, "dom": "P", "ent": "anterior", "cls": "male"}, {"id": 138, "dom": "P", "ent": "anterior", "cls": "marijuana"}, {"id": 139, "dom": "P", "ent": "anterior", "cls": "puppy"}, {"id": 140, "dom": "P", "ent": "anterior", "cls": "receiver"}, {"id": 141, "dom": "P", "ent": "anterior", "cls": "reed"}, {"id": 142, "dom": "P", "ent": "anterior", "cls": "rifle"}, {"id": 143, "dom": "P", "ent": "anterior", "cls": "road"}, {"id": 144, "dom": "P", "ent": "anterior", "cls": "salmon"}, {"id": 145, "dom": "P", "ent": "anterior", "cls": "shark"}, {"id": 146, "dom": "P", "ent": "anterior", "cls": "sheep"}, {"id": 147, "dom": "P", "ent": "anterior", "cls": "shook"}, {"id": 148, "dom": "P", "ent": "anterior", "cls": "tank"}, {"id": 149, "dom": "P", "ent": "anterior", "cls": "wheat"}, {"id": 150, "dom": "P", "ent": "antique", "cls": "architecture"}, {"id": 151, "dom": "P", "ent": "antique", "cls": "bacteria"}, {"id": 152, "dom": "P", "ent": "antique", "cls": "beef"}, {"id": 153, "dom": "P", "ent": "antique", "cls": "bridge"}, {"id": 154, "dom": "P", "ent": "antique", "cls": "bud"}, {"id": 155, "dom": "P", "ent": "antique", "cls": "cat"}, {"id": 156, "dom": "P", "ent": "antique", "cls": "chapel"}, {"id": 157, "dom": "P", "ent": "antique", "cls": "chick"}, {"id": 158, "dom": "P", "ent": "antique", "cls": "clock"}, {"id": 159, "dom": "P", "ent": "antique", "cls": "deer"}, {"id": 160, "dom": "P", "ent": "antique", "cls": "dog"}, {"id": 161, "dom": "P", "ent": "antique", "cls": "elephant"}, {"id": 162, "dom": "P", "ent": "antique", "cls": "embassy"}, {"id": 163, "dom": "P", "ent": "antique", "cls": "garage"}, {"id": 164, "dom": "P", "ent": "antique", "cls": "highway"}, {"id": 165, "dom": "P", "ent": "antique", "cls": "kidney"}, {"id": 166, "dom": "P", "ent": "antique", "cls": "lamb"}, {"id": 167, "dom": "P", "ent": "antique", "cls": "male"}, {"id": 168, "dom": "P", "ent": "antique", "cls": "marijuana"}, {"id": 169, "dom": "P", "ent": "antique", "cls": "puppy"}, {"id": 170, "dom": "P", "ent": "antique", "cls": "receiver"}, {"id": 171, "dom": "P", "ent": "antique", "cls": "reed"}, {"id": 172, "dom": "P", "ent": "antique", "cls": "rifle"}, {"id": 173, "dom": "P", "ent": "antique", "cls": "road"}, {"id": 174, "dom": "P", "ent": "antique", "cls": "salmon"}, {"id": 175, "dom": "P", "ent": "antique", "cls": "shark"}, {"id": 176, "dom": "P", "ent": "antique", "cls": "sheep"}, {"id": 177, "dom": "P", "ent": "antique", "cls": "shook"}, {"id": 178, "dom": "P", "ent": "antique", "cls": "tank"}, {"id": 179, "dom": "P", "ent": "antique", "cls": "wheat"}, {"id": 180, "dom": "P", "ent": "apartment", "cls": "architecture"}, {"id": 181, "dom": "P", "ent": "apartment", "cls": "bacteria"}, {"id": 182, "dom": "P", "ent": "apartment", "cls": "beef"}, {"id": 183, "dom": "P", "ent": "apartment", "cls": "bridge"}, {"id": 184, "dom": "P", "ent": "apartment", "cls": "bud"}, {"id": 185, "dom": "P", "ent": "apartment", "cls": "cat"}, {"id": 186, "dom": "P", "ent": "apartment", "cls": "chapel"}, {"id": 187, "dom": "P", "ent": "apartment", "cls": "chick"}, {"id": 188, "dom": "P", "ent": "apartment", "cls": "clock"}, {"id": 189, "dom": "P", "ent": "apartment", "cls": "deer"}, {"id": 190, "dom": "P", "ent": "apartment", "cls": "dog"}, {"id": 191, "dom": "P", "ent": "apartment", "cls": "elephant"}, {"id": 192, "dom": "P", "ent": "apartment", "cls": "embassy"}, {"id": 193, "dom": "P", "ent": "apartment", "cls": "garage"}, {"id": 194, "dom": "P", "ent": "apartment", "cls": "highway"}, {"id": 195, "dom": "P", "ent": "apartment", "cls": "kidney"}, {"id": 196, "dom": "P", "ent": "apartment", "cls": "lamb"}, {"id": 197, "dom": "P", "ent": "apartment", "cls": "male"}, {"id": 198, "dom": "P", "ent": "apartment", "cls": "marijuana"}, {"id": 199, "dom": "P", "ent": "apartment", "cls": "puppy"}, {"id": 200, "dom": "P", "ent": "apartment", "cls": "receiver"}, {"id": 201, "dom": "P", "ent": "apartment", "cls": "reed"}, {"id": 202, "dom": "P", "ent": "apartment", "cls": "rifle"}, {"id": 203, "dom": "P", "ent": "apartment", "cls": "road"}, {"id": 204, "dom": "P", "ent": "apartment", "cls": "salmon"}, {"id": 205, "dom": "P", "ent": "apartment", "cls": "shark"}, {"id": 206, "dom": "P", "ent": "apartment", "cls": "sheep"}, {"id": 207, "dom": "P", "ent": "apartment", "cls": "shook"}, {"id": 208, "dom": "P", "ent": "apartment", "cls": "tank"}, {"id": 209, "dom": "P", "ent": "apartment", "cls": "wheat"}, {"id": 210, "dom": "P", "ent": "aperture", "cls": "architecture"}, {"id": 211, "dom": "P", "ent": "aperture", "cls": "bacteria"}, {"id": 212, "dom": "P", "ent": "aperture", "cls": "beef"}, {"id": 213, "dom": "P", "ent": "aperture", "cls": "bridge"}, {"id": 214, "dom": "P", "ent": "aperture", "cls": "bud"}, {"id": 215, "dom": "P", "ent": "aperture", "cls": "cat"}, {"id": 216, "dom": "P", "ent": "aperture", "cls": "chapel"}, {"id": 217, "dom": "P", "ent": "aperture", "cls": "chick"}, {"id": 218, "dom": "P", "ent": "aperture", "cls": "clock"}, {"id": 219, "dom": "P", "ent": "aperture", "cls": "deer"}, {"id": 220, "dom": "P", "ent": "aperture", "cls": "dog"}, {"id": 221, "dom": "P", "ent": "aperture", "cls": "elephant"}, {"id": 222, "dom": "P", "ent": "aperture", "cls": "embassy"}, {"id": 223, "dom": "P", "ent": "aperture", "cls": "garage"}, {"id": 224, "dom": "P", "ent": "aperture", "cls": "highway"}, {"id": 225, "dom": "P", "ent": "aperture", "cls": "kidney"}, {"id": 226, "dom": "P", "ent": "aperture", "cls": "lamb"}, {"id": 227, "dom": "P", "ent": "aperture", "cls": "male"}, {"id": 228, "dom": "P", "ent": "aperture", "cls": "marijuana"}, {"id": 229, "dom": "P", "ent": "aperture", "cls": "puppy"}, {"id": 230, "dom": "P", "ent": "aperture", "cls": "receiver"}, {"id": 231, "dom": "P", "ent": "aperture", "cls": "reed"}, {"id": 232, "dom": "P", "ent": "aperture", "cls": "rifle"}, {"id": 233, "dom": "P", "ent": "aperture", "cls": "road"}, {"id": 234, "dom": "P", "ent": "aperture", "cls": "salmon"}, {"id": 235, "dom": "P", "ent": "aperture", "cls": "shark"}, {"id": 236, "dom": "P", "ent": "aperture", "cls": "sheep"}, {"id": 237, "dom": "P", "ent": "aperture", "cls": "shook"}, {"id": 238, "dom": "P", "ent": "aperture", "cls": "tank"}, {"id": 239, "dom": "P", "ent": "aperture", "cls": "wheat"}, {"id": 240, "dom": "P", "ent": "approach", "cls": "architecture"}, {"id": 241, "dom": "P", "ent": "approach", "cls": "bacteria"}, {"id": 242, "dom": "P", "ent": "approach", "cls": "beef"}, {"id": 243, "dom": "P", "ent": "approach", "cls": "bridge"}, {"id": 244, "dom": "P", "ent": "approach", "cls": "bud"}, {"id": 245, "dom": "P", "ent": "approach", "cls": "cat"}, {"id": 246, "dom": "P", "ent": "approach", "cls": "chapel"}, {"id": 247, "dom": "P", "ent": "approach", "cls": "chick"}, {"id": 248, "dom": "P", "ent": "approach", "cls": "clock"}, {"id": 249, "dom": "P", "ent": "approach", "cls": "deer"}, {"id": 250, "dom": "P", "ent": "approach", "cls": "dog"}, {"id": 251, "dom": "P", "ent": "approach", "cls": "elephant"}, {"id": 252, "dom": "P", "ent": "approach", "cls": "embassy"}, {"id": 253, "dom": "P", "ent": "approach", "cls": "garage"}, {"id": 254, "dom": "P", "ent": "approach", "cls": "highway"}, {"id": 255, "dom": "P", "ent": "approach", "cls": "kidney"}, {"id": 256, "dom": "P", "ent": "approach", "cls": "lamb"}, {"id": 257, "dom": "P", "ent": "approach", "cls": "male"}, {"id": 258, "dom": "P", "ent": "approach", "cls": "marijuana"}, {"id": 259, "dom": "P", "ent": "approach", "cls": "puppy"}, {"id": 260, "dom": "P", "ent": "approach", "cls": "receiver"}, {"id": 261, "dom": "P", "ent": "approach", "cls": "reed"}, {"id": 262, "dom": "P", "ent": "approach", "cls": "rifle"}, {"id": 263, "dom": "P", "ent": "approach", "cls": "road"}, {"id": 264, "dom": "P", "ent": "approach", "cls": "salmon"}, {"id": 265, "dom": "P", "ent": "approach", "cls": "shark"}, {"id": 266, "dom": "P", "ent": "approach", "cls": "sheep"}, {"id": 267, "dom": "P", "ent": "approach", "cls": "shook"}, {"id": 268, "dom": "P", "ent": "approach", "cls": "tank"}, {"id": 269, "dom": "P", "ent": "approach", "cls": "wheat"}, {"id": 270, "dom": "P", "ent": "arch", "cls": "architecture"}, {"id": 271, "dom": "P", "ent": "arch", "cls": "bacteria"}, {"id": 272, "dom": "P", "ent": "arch", "cls": "beef"}, {"id": 273, "dom": "P", "ent": "arch", "cls": "bridge"}, {"id": 274, "dom": "P", "ent": "arch", "cls": "bud"}, {"id": 275, "dom": "P", "ent": "arch", "cls": "cat"}, {"id": 276, "dom": "P", "ent": "arch", "cls": "chapel"}, {"id": 277, "dom": "P", "ent": "arch", "cls": "chick"}, {"id": 278, "dom": "P", "ent": "arch", "cls": "clock"}, {"id": 279, "dom": "P", "ent": "arch", "cls": "deer"}, {"id": 280, "dom": "P", "ent": "arch", "cls": "dog"}, {"id": 281, "dom": "P", "ent": "arch", "cls": "elephant"}, {"id": 282, "dom": "P", "ent": "arch", "cls": "embassy"}, {"id": 283, "dom": "P", "ent": "arch", "cls": "garage"}, {"id": 284, "dom": "P", "ent": "arch", "cls": "highway"}, {"id": 285, "dom": "P", "ent": "arch", "cls": "kidney"}, {"id": 286, "dom": "P", "ent": "arch", "cls": "lamb"}, {"id": 287, "dom": "P", "ent": "arch", "cls": "male"}, {"id": 288, "dom": "P", "ent": "arch", "cls": "marijuana"}, {"id": 289, "dom": "P", "ent": "arch", "cls": "puppy"}, {"id": 290, "dom": "P", "ent": "arch", "cls": "receiver"}, {"id": 291, "dom": "P", "ent": "arch", "cls": "reed"}, {"id": 292, "dom": "P", "ent": "arch", "cls": "rifle"}, {"id": 293, "dom": "P", "ent": "arch", "cls": "road"}, {"id": 294, "dom": "P", "ent": "arch", "cls": "salmon"}, {"id": 295, "dom": "P", "ent": "arch", "cls": "shark"}, {"id": 296, "dom": "P", "ent": "arch", "cls": "sheep"}, {"id": 297, "dom": "P", "ent": "arch", "cls": "shook"}, {"id": 298, "dom": "P", "ent": "arch", "cls": "tank"}, {"id": 299, "dom": "P", "ent": "arch", "cls": "wheat"}, {"id": 300, "dom": "P", "ent": "ash", "cls": "architecture"}, {"id": 301, "dom": "P", "ent": "ash", "cls": "bacteria"}, {"id": 302, "dom": "P", "ent": "ash", "cls": "beef"}, {"id": 303, "dom": "P", "ent": "ash", "cls": "bridge"}, {"id": 304, "dom": "P", "ent": "ash", "cls": "bud"}, {"id": 305, "dom": "P", "ent": "ash", "cls": "cat"}, {"id": 306, "dom": "P", "ent": "ash", "cls": "chapel"}, {"id": 307, "dom": "P", "ent": "ash", "cls": "chick"}, {"id": 308, "dom": "P", "ent": "ash", "cls": "clock"}, {"id": 309, "dom": "P", "ent": "ash", "cls": "deer"}, {"id": 310, "dom": "P", "ent": "ash", "cls": "dog"}, {"id": 311, "dom": "P", "ent": "ash", "cls": "elephant"}, {"id": 312, "dom": "P", "ent": "ash", "cls": "embassy"}, {"id": 313, "dom": "P", "ent": "ash", "cls": "garage"}, {"id": 314, "dom": "P", "ent": "ash", "cls": "highway"}, {"id": 315, "dom": "P", "ent": "ash", "cls": "kidney"}, {"id": 316, "dom": "P", "ent": "ash", "cls": "lamb"}, {"id": 317, "dom": "P", "ent": "ash", "cls": "male"}, {"id": 318, "dom": "P", "ent": "ash", "cls": "marijuana"}, {"id": 319, "dom": "P", "ent": "ash", "cls": "puppy"}, {"id": 320, "dom": "P", "ent": "ash", "cls": "receiver"}, {"id": 321, "dom": "P", "ent": "ash", "cls": "reed"}, {"id": 322, "dom": "P", "ent": "ash", "cls": "rifle"}, {"id": 323, "dom": "P", "ent": "ash", "cls": "road"}, {"id": 324, "dom": "P", "ent": "ash", "cls": "salmon"}, {"id": 325, "dom": "P", "ent": "ash", "cls": "shark"}, {"id": 326, "dom": "P", "ent": "ash", "cls": "sheep"}, {"id": 327, "dom": "P", "ent": "ash", "cls": "shook"}, {"id": 328, "dom": "P", "ent": "ash", "cls": "tank"}, {"id": 329, "dom": "P", "ent": "ash", "cls": "wheat"}, {"id": 330, "dom": "P", "ent": "ash-key", "cls": "architecture"}, {"id": 331, "dom": "P", "ent": "ash-key", "cls": "bacteria"}, {"id": 332, "dom": "P", "ent": "ash-key", "cls": "beef"}, {"id": 333, "dom": "P", "ent": "ash-key", "cls": "bridge"}, {"id": 334, "dom": "P", "ent": "ash-key", "cls": "bud"}, {"id": 335, "dom": "P", "ent": "ash-key", "cls": "cat"}, {"id": 336, "dom": "P", "ent": "ash-key", "cls": "chapel"}, {"id": 337, "dom": "P", "ent": "ash-key", "cls": "chick"}, {"id": 338, "dom": "P", "ent": "ash-key", "cls": "clock"}, {"id": 339, "dom": "P", "ent": "ash-key", "cls": "deer"}, {"id": 340, "dom": "P", "ent": "ash-key", "cls": "dog"}, {"id": 341, "dom": "P", "ent": "ash-key", "cls": "elephant"}, {"id": 342, "dom": "P", "ent": "ash-key", "cls": "embassy"}, {"id": 343, "dom": "P", "ent": "ash-key", "cls": "garage"}, {"id": 344, "dom": "P", "ent": "ash-key", "cls": "highway"}, {"id": 345, "dom": "P", "ent": "ash-key", "cls": "kidney"}, {"id": 346, "dom": "P", "ent": "ash-key", "cls": "lamb"}, {"id": 347, "dom": "P", "ent": "ash-key", "cls": "male"}, {"id": 348, "dom": "P", "ent": "ash-key", "cls": "marijuana"}, {"id": 349, "dom": "P", "ent": "ash-key", "cls": "puppy"}, {"id": 350, "dom": "P", "ent": "ash-key", "cls": "receiver"}, {"id": 351, "dom": "P", "ent": "ash-key", "cls": "reed"}, {"id": 352, "dom": "P", "ent": "ash-key", "cls": "rifle"}, {"id": 353, "dom": "P", "ent": "ash-key", "cls": "road"}, {"id": 354, "dom": "P", "ent": "ash-key", "cls": "salmon"}, {"id": 355, "dom": "P", "ent": "ash-key", "cls": "shark"}, {"id": 356, "dom": "P", "ent": "ash-key", "cls": "sheep"}, {"id": 357, "dom": "P", "ent": "ash-key", "cls": "shook"}, {"id": 358, "dom": "P", "ent": "ash-key", "cls": "tank"}, {"id": 359, "dom": "P", "ent": "ash-key", "cls": "wheat"}, {"id": 360, "dom": "P", "ent": "axle", "cls": "architecture"}, {"id": 361, "dom": "P", "ent": "axle", "cls": "bacteria"}, {"id": 362, "dom": "P", "ent": "axle", "cls": "beef"}, {"id": 363, "dom": "P", "ent": "axle", "cls": "bridge"}, {"id": 364, "dom": "P", "ent": "axle", "cls": "bud"}, {"id": 365, "dom": "P", "ent": "axle", "cls": "cat"}, {"id": 366, "dom": "P", "ent": "axle", "cls": "chapel"}, {"id": 367, "dom": "P", "ent": "axle", "cls": "chick"}, {"id": 368, "dom": "P", "ent": "axle", "cls": "clock"}, {"id": 369, "dom": "P", "ent": "axle", "cls": "deer"}, {"id": 370, "dom": "P", "ent": "axle", "cls": "dog"}, {"id": 371, "dom": "P", "ent": "axle", "cls": "elephant"}, {"id": 372, "dom": "P", "ent": "axle", "cls": "embassy"}, {"id": 373, "dom": "P", "ent": "axle", "cls": "garage"}, {"id": 374, "dom": "P", "ent": "axle", "cls": "highway"}, {"id": 375, "dom": "P", "ent": "axle", "cls": "kidney"}, {"id": 376, "dom": "P", "ent": "axle", "cls": "lamb"}, {"id": 377, "dom": "P", "ent": "axle", "cls": "male"}, {"id": 378, "dom": "P", "ent": "axle", "cls": "marijuana"}, {"id": 379, "dom": "P", "ent": "axle", "cls": "puppy"}, {"id": 380, "dom": "P", "ent": "axle", "cls": "receiver"}, {"id": 381, "dom": "P", "ent": "axle", "cls": "reed"}, {"id": 382, "dom": "P", "ent": "axle", "cls": "rifle"}, {"id": 383, "dom": "P", "ent": "axle", "cls": "road"}, {"id": 384, "dom": "P", "ent": "axle", "cls": "salmon"}, {"id": 385, "dom": "P", "ent": "axle", "cls": "shark"}, {"id": 386, "dom": "P", "ent": "axle", "cls": "sheep"}, {"id": 387, "dom": "P", "ent": "axle", "cls": "shook"}, {"id": 388, "dom": "P", "ent": "axle", "cls": "tank"}, {"id": 389, "dom": "P", "ent": "axle", "cls": "wheat"}, {"id": 390, "dom": "P", "ent": "balance", "cls": "architecture"}, {"id": 391, "dom": "P", "ent": "balance", "cls": "bacteria"}, {"id": 392, "dom": "P", "ent": "balance", "cls": "beef"}, {"id": 393, "dom": "P", "ent": "balance", "cls": "bridge"}, {"id": 394, "dom": "P", "ent": "balance", "cls": "bud"}, {"id": 395, "dom": "P", "ent": "balance", "cls": "cat"}, {"id": 396, "dom": "P", "ent": "balance", "cls": "chapel"}, {"id": 397, "dom": "P", "ent": "balance", "cls": "chick"}, {"id": 398, "dom": "P", "ent": "balance", "cls": "clock"}, {"id": 399, "dom": "P", "ent": "balance", "cls": "deer"}, {"id": 400, "dom": "P", "ent": "balance", "cls": "dog"}, {"id": 401, "dom": "P", "ent": "balance", "cls": "elephant"}, {"id": 402, "dom": "P", "ent": "balance", "cls": "embassy"}, {"id": 403, "dom": "P", "ent": "balance", "cls": "garage"}, {"id": 404, "dom": "P", "ent": "balance", "cls": "highway"}, {"id": 405, "dom": "P", "ent": "balance", "cls": "kidney"}, {"id": 406, "dom": "P", "ent": "balance", "cls": "lamb"}, {"id": 407, "dom": "P", "ent": "balance", "cls": "male"}, {"id": 408, "dom": "P", "ent": "balance", "cls": "marijuana"}, {"id": 409, "dom": "P", "ent": "balance", "cls": "puppy"}, {"id": 410, "dom": "P", "ent": "balance", "cls": "receiver"}, {"id": 411, "dom": "P", "ent": "balance", "cls": "reed"}, {"id": 412, "dom": "P", "ent": "balance", "cls": "rifle"}, {"id": 413, "dom": "P", "ent": "balance", "cls": "road"}, {"id": 414, "dom": "P", "ent": "balance", "cls": "salmon"}, {"id": 415, "dom": "P", "ent": "balance", "cls": "shark"}, {"id": 416, "dom": "P", "ent": "balance", "cls": "sheep"}, {"id": 417, "dom": "P", "ent": "balance", "cls": "shook"}, {"id": 418, "dom": "P", "ent": "balance", "cls": "tank"}, {"id": 419, "dom": "P", "ent": "balance", "cls": "wheat"}, {"id": 420, "dom": "P", "ent": "bang", "cls": "architecture"}, {"id": 421, "dom": "P", "ent": "bang", "cls": "bacteria"}, {"id": 422, "dom": "P", "ent": "bang", "cls": "beef"}, {"id": 423, "dom": "P", "ent": "bang", "cls": "bridge"}, {"id": 424, "dom": "P", "ent": "bang", "cls": "bud"}, {"id": 425, "dom": "P", "ent": "bang", "cls": "cat"}, {"id": 426, "dom": "P", "ent": "bang", "cls": "chapel"}, {"id": 427, "dom": "P", "ent": "bang", "cls": "chick"}, {"id": 428, "dom": "P", "ent": "bang", "cls": "clock"}, {"id": 429, "dom": "P", "ent": "bang", "cls": "deer"}, {"id": 430, "dom": "P", "ent": "bang", "cls": "dog"}, {"id": 431, "dom": "P", "ent": "bang", "cls": "elephant"}, {"id": 432, "dom": "P", "ent": "bang", "cls": "embassy"}, {"id": 433, "dom": "P", "ent": "bang", "cls": "garage"}, {"id": 434, "dom": "P", "ent": "bang", "cls": "highway"}, {"id": 435, "dom": "P", "ent": "bang", "cls": "kidney"}, {"id": 436, "dom": "P", "ent": "bang", "cls": "lamb"}, {"id": 437, "dom": "P", "ent": "bang", "cls": "male"}, {"id": 438, "dom": "P", "ent": "bang", "cls": "marijuana"}, {"id": 439, "dom": "P", "ent": "bang", "cls": "puppy"}, {"id": 440, "dom": "P", "ent": "bang", "cls": "receiver"}, {"id": 441, "dom": "P", "ent": "bang", "cls": "reed"}, {"id": 442, "dom": "P", "ent": "bang", "cls": "rifle"}, {"id": 443, "dom": "P", "ent": "bang", "cls": "road"}, {"id": 444, "dom": "P", "ent": "bang", "cls": "salmon"}, {"id": 445, "dom": "P", "ent": "bang", "cls": "shark"}, {"id": 446, "dom": "P", "ent": "bang", "cls": "sheep"}, {"id": 447, "dom": "P", "ent": "bang", "cls": "shook"}, {"id": 448, "dom": "P", "ent": "bang", "cls": "tank"}, {"id": 449, "dom": "P", "ent": "bang", "cls": "wheat"}, {"id": 450, "dom": "P", "ent": "barb", "cls": "architecture"}, {"id": 451, "dom": "P", "ent": "barb", "cls": "bacteria"}, {"id": 452, "dom": "P", "ent": "barb", "cls": "beef"}, {"id": 453, "dom": "P", "ent": "barb", "cls": "bridge"}, {"id": 454, "dom": "P", "ent": "barb", "cls": "bud"}, {"id": 455, "dom": "P", "ent": "barb", "cls": "cat"}, {"id": 456, "dom": "P", "ent": "barb", "cls": "chapel"}, {"id": 457, "dom": "P", "ent": "barb", "cls": "chick"}, {"id": 458, "dom": "P", "ent": "barb", "cls": "clock"}, {"id": 459, "dom": "P", "ent": "barb", "cls": "deer"}, {"id": 460, "dom": "P", "ent": "barb", "cls": "dog"}, {"id": 461, "dom": "P", "ent": "barb", "cls": "elephant"}, {"id": 462, "dom": "P", "ent": "barb", "cls": "embassy"}, {"id": 463, "dom": "P", "ent": "barb", "cls": "garage"}, {"id": 464, "dom": "P", "ent": "barb", "cls": "highway"}, {"id": 465, "dom": "P", "ent": "barb", "cls": "kidney"}, {"id": 466, "dom": "P", "ent": "barb", "cls": "lamb"}, {"id": 467, "dom": "P", "ent": "barb", "cls": "male"}, {"id": 468, "dom": "P", "ent": "barb", "cls": "marijuana"}, {"id": 469, "dom": "P", "ent": "barb", "cls": "puppy"}, {"id": 470, "dom": "P", "ent": "barb", "cls": "receiver"}, {"id": 471, "dom": "P", "ent": "barb", "cls": "reed"}, {"id": 472, "dom": "P", "ent": "barb", "cls": "rifle"}, {"id": 473, "dom": "P", "ent": "barb", "cls": "road"}, {"id": 474, "dom": "P", "ent": "barb", "cls": "salmon"}, {"id": 475, "dom": "P", "ent": "barb", "cls": "shark"}, {"id": 476, "dom": "P", "ent": "barb", "cls": "sheep"}, {"id": 477, "dom": "P", "ent": "barb", "cls": "shook"}, {"id": 478, "dom": "P", "ent": "barb", "cls": "tank"}, {"id": 479, "dom": "P", "ent": "barb", "cls": "wheat"}, {"id": 480, "dom": "P", "ent": "barrel", "cls": "architecture"}, {"id": 481, "dom": "P", "ent": "barrel", "cls": "bacteria"}, {"id": 482, "dom": "P", "ent": "barrel", "cls": "beef"}, {"id": 483, "dom": "P", "ent": "barrel", "cls": "bridge"}, {"id": 484, "dom": "P", "ent": "barrel", "cls": "bud"}, {"id": 485, "dom": "P", "ent": "barrel", "cls": "cat"}, {"id": 486, "dom": "P", "ent": "barrel", "cls": "chapel"}, {"id": 487, "dom": "P", "ent": "barrel", "cls": "chick"}, {"id": 488, "dom": "P", "ent": "barrel", "cls": "clock"}, {"id": 489, "dom": "P", "ent": "barrel", "cls": "deer"}, {"id": 490, "dom": "P", "ent": "barrel", "cls": "dog"}, {"id": 491, "dom": "P", "ent": "barrel", "cls": "elephant"}, {"id": 492, "dom": "P", "ent": "barrel", "cls": "embassy"}, {"id": 493, "dom": "P", "ent": "barrel", "cls": "garage"}, {"id": 494, "dom": "P", "ent": "barrel", "cls": "highway"}, {"id": 495, "dom": "P", "ent": "barrel", "cls": "kidney"}, {"id": 496, "dom": "P", "ent": "barrel", "cls": "lamb"}, {"id": 497, "dom": "P", "ent": "barrel", "cls": "male"}, {"id": 498, "dom": "P", "ent": "barrel", "cls": "marijuana"}, {"id": 499, "dom": "P", "ent": "barrel", "cls": "puppy"}, {"id": 500, "dom": "P", "ent": "barrel", "cls": "receiver"}, {"id": 501, "dom": "P", "ent": "barrel", "cls": "reed"}, {"id": 502, "dom": "P", "ent": "barrel", "cls": "rifle"}, {"id": 503, "dom": "P", "ent": "barrel", "cls": "road"}, {"id": 504, "dom": "P", "ent": "barrel", "cls": "salmon"}, {"id": 505, "dom": "P", "ent": "barrel", "cls": "shark"}, {"id": 506, "dom": "P", "ent": "barrel", "cls": "sheep"}, {"id": 507, "dom": "P", "ent": "barrel", "cls": "shook"}, {"id": 508, "dom": "P", "ent": "barrel", "cls": "tank"}, {"id": 509, "dom": "P", "ent": "barrel", "cls": "wheat"}, {"id": 510, "dom": "P", "ent": "base", "cls": "architecture"}, {"id": 511, "dom": "P", "ent": "base", "cls": "bacteria"}, {"id": 512, "dom": "P", "ent": "base", "cls": "beef"}, {"id": 513, "dom": "P", "ent": "base", "cls": "bridge"}, {"id": 514, "dom": "P", "ent": "base", "cls": "bud"}, {"id": 515, "dom": "P", "ent": "base", "cls": "cat"}, {"id": 516, "dom": "P", "ent": "base", "cls": "chapel"}, {"id": 517, "dom": "P", "ent": "base", "cls": "chick"}, {"id": 518, "dom": "P", "ent": "base", "cls": "clock"}, {"id": 519, "dom": "P", "ent": "base", "cls": "deer"}, {"id": 520, "dom": "P", "ent": "base", "cls": "dog"}, {"id": 521, "dom": "P", "ent": "base", "cls": "elephant"}, {"id": 522, "dom": "P", "ent": "base", "cls": "embassy"}, {"id": 523, "dom": "P", "ent": "base", "cls": "garage"}, {"id": 524, "dom": "P", "ent": "base", "cls": "highway"}, {"id": 525, "dom": "P", "ent": "base", "cls": "kidney"}, {"id": 526, "dom": "P", "ent": "base", "cls": "lamb"}, {"id": 527, "dom": "P", "ent": "base", "cls": "male"}, {"id": 528, "dom": "P", "ent": "base", "cls": "marijuana"}, {"id": 529, "dom": "P", "ent": "base", "cls": "puppy"}, {"id": 530, "dom": "P", "ent": "base", "cls": "receiver"}, {"id": 531, "dom": "P", "ent": "base", "cls": "reed"}, {"id": 532, "dom": "P", "ent": "base", "cls": "rifle"}, {"id": 533, "dom": "P", "ent": "base", "cls": "road"}, {"id": 534, "dom": "P", "ent": "base", "cls": "salmon"}, {"id": 535, "dom": "P", "ent": "base", "cls": "shark"}, {"id": 536, "dom": "P", "ent": "base", "cls": "sheep"}, {"id": 537, "dom": "P", "ent": "base", "cls": "shook"}, {"id": 538, "dom": "P", "ent": "base", "cls": "tank"}, {"id": 539, "dom": "P", "ent": "base", "cls": "wheat"}, {"id": 540, "dom": "P", "ent": "batting", "cls": "architecture"}, {"id": 541, "dom": "P", "ent": "batting", "cls": "bacteria"}, {"id": 542, "dom": "P", "ent": "batting", "cls": "beef"}, {"id": 543, "dom": "P", "ent": "batting", "cls": "bridge"}, {"id": 544, "dom": "P", "ent": "batting", "cls": "bud"}, {"id": 545, "dom": "P", "ent": "batting", "cls": "cat"}, {"id": 546, "dom": "P", "ent": "batting", "cls": "chapel"}, {"id": 547, "dom": "P", "ent": "batting", "cls": "chick"}, {"id": 548, "dom": "P", "ent": "batting", "cls": "clock"}, {"id": 549, "dom": "P", "ent": "batting", "cls": "deer"}, {"id": 550, "dom": "P", "ent": "batting", "cls": "dog"}, {"id": 551, "dom": "P", "ent": "batting", "cls": "elephant"}, {"id": 552, "dom": "P", "ent": "batting", "cls": "embassy"}, {"id": 553, "dom": "P", "ent": "batting", "cls": "garage"}, {"id": 554, "dom": "P", "ent": "batting", "cls": "highway"}, {"id": 555, "dom": "P", "ent": "batting", "cls": "kidney"}, {"id": 556, "dom": "P", "ent": "batting", "cls": "lamb"}, {"id": 557, "dom": "P", "ent": "batting", "cls": "male"}, {"id": 558, "dom": "P", "ent": "batting", "cls": "marijuana"}, {"id": 559, "dom": "P", "ent": "batting", "cls": "puppy"}, {"id": 560, "dom": "P", "ent": "batting", "cls": "receiver"}, {"id": 561, "dom": "P", "ent": "batting", "cls": "reed"}, {"id": 562, "dom": "P", "ent": "batting", "cls": "rifle"}, {"id": 563, "dom": "P", "ent": "batting", "cls": "road"}, {"id": 564, "dom": "P", "ent": "batting", "cls": "salmon"}, {"id": 565, "dom": "P", "ent": "batting", "cls": "shark"}, {"id": 566, "dom": "P", "ent": "batting", "cls": "sheep"}, {"id": 567, "dom": "P", "ent": "batting", "cls": "shook"}, {"id": 568, "dom": "P", "ent": "batting", "cls": "tank"}, {"id": 569, "dom": "P", "ent": "batting", "cls": "wheat"}, {"id": 570, "dom": "P", "ent": "beak", "cls": "architecture"}, {"id": 571, "dom": "P", "ent": "beak", "cls": "bacteria"}, {"id": 572, "dom": "P", "ent": "beak", "cls": "beef"}, {"id": 573, "dom": "P", "ent": "beak", "cls": "bridge"}, {"id": 574, "dom": "P", "ent": "beak", "cls": "bud"}, {"id": 575, "dom": "P", "ent": "beak", "cls": "cat"}, {"id": 576, "dom": "P", "ent": "beak", "cls": "chapel"}, {"id": 577, "dom": "P", "ent": "beak", "cls": "chick"}, {"id": 578, "dom": "P", "ent": "beak", "cls": "clock"}, {"id": 579, "dom": "P", "ent": "beak", "cls": "deer"}, {"id": 580, "dom": "P", "ent": "beak", "cls": "dog"}, {"id": 581, "dom": "P", "ent": "beak", "cls": "elephant"}, {"id": 582, "dom": "P", "ent": "beak", "cls": "embassy"}, {"id": 583, "dom": "P", "ent": "beak", "cls": "garage"}, {"id": 584, "dom": "P", "ent": "beak", "cls": "highway"}, {"id": 585, "dom": "P", "ent": "beak", "cls": "kidney"}, {"id": 586, "dom": "P", "ent": "beak", "cls": "lamb"}, {"id": 587, "dom": "P", "ent": "beak", "cls": "male"}, {"id": 588, "dom": "P", "ent": "beak", "cls": "marijuana"}, {"id": 589, "dom": "P", "ent": "beak", "cls": "puppy"}, {"id": 590, "dom": "P", "ent": "beak", "cls": "receiver"}, {"id": 591, "dom": "P", "ent": "beak", "cls": "reed"}, {"id": 592, "dom": "P", "ent": "beak", "cls": "rifle"}, {"id": 593, "dom": "P", "ent": "beak", "cls": "road"}, {"id": 594, "dom": "P", "ent": "beak", "cls": "salmon"}, {"id": 595, "dom": "P", "ent": "beak", "cls": "shark"}, {"id": 596, "dom": "P", "ent": "beak", "cls": "sheep"}, {"id": 597, "dom": "P", "ent": "beak", "cls": "shook"}, {"id": 598, "dom": "P", "ent": "beak", "cls": "tank"}, {"id": 599, "dom": "P", "ent": "beak", "cls": "wheat"}, {"id": 600, "dom": "P", "ent": "beard", "cls": "architecture"}, {"id": 601, "dom": "P", "ent": "beard", "cls": "bacteria"}, {"id": 602, "dom": "P", "ent": "beard", "cls": "beef"}, {"id": 603, "dom": "P", "ent": "beard", "cls": "bridge"}, {"id": 604, "dom": "P", "ent": "beard", "cls": "bud"}, {"id": 605, "dom": "P", "ent": "beard", "cls": "cat"}, {"id": 606, "dom": "P", "ent": "beard", "cls": "chapel"}, {"id": 607, "dom": "P", "ent": "beard", "cls": "chick"}, {"id": 608, "dom": "P", "ent": "beard", "cls": "clock"}, {"id": 609, "dom": "P", "ent": "beard", "cls": "deer"}, {"id": 610, "dom": "P", "ent": "beard", "cls": "dog"}, {"id": 611, "dom": "P", "ent": "beard", "cls": "elephant"}, {"id": 612, "dom": "P", "ent": "beard", "cls": "embassy"}, {"id": 613, "dom": "P", "ent": "beard", "cls": "garage"}, {"id": 614, "dom": "P", "ent": "beard", "cls": "highway"}, {"id": 615, "dom": "P", "ent": "beard", "cls": "kidney"}, {"id": 616, "dom": "P", "ent": "beard", "cls": "lamb"}, {"id": 617, "dom": "P", "ent": "beard", "cls": "male"}, {"id": 618, "dom": "P", "ent": "beard", "cls": "marijuana"}, {"id": 619, "dom": "P", "ent": "beard", "cls": "puppy"}, {"id": 620, "dom": "P", "ent": "beard", "cls": "receiver"}, {"id": 621, "dom": "P", "ent": "beard", "cls": "reed"}, {"id": 622, "dom": "P", "ent": "beard", "cls": "rifle"}, {"id": 623, "dom": "P", "ent": "beard", "cls": "road"}, {"id": 624, "dom": "P", "ent": "beard", "cls": "salmon"}, {"id": 625, "dom": "P", "ent": "beard", "cls": "shark"}, {"id": 626, "dom": "P", "ent": "beard", "cls": "sheep"}, {"id": 627, "dom": "P", "ent": "beard", "cls": "shook"}, {"id": 628, "dom": "P", "ent": "beard", "cls": "tank"}, {"id": 629, "dom": "P", "ent": "beard", "cls": "wheat"}, {"id": 630, "dom": "P", "ent": "beetle", "cls": "architecture"}, {"id": 631, "dom": "P", "ent": "beetle", "cls": "bacteria"}, {"id": 632, "dom": "P", "ent": "beetle", "cls": "beef"}, {"id": 633, "dom": "P", "ent": "beetle", "cls": "bridge"}, {"id": 634, "dom": "P", "ent": "beetle", "cls": "bud"}, {"id": 635, "dom": "P", "ent": "beetle", "cls": "cat"}, {"id": 636, "dom": "P", "ent": "beetle", "cls": "chapel"}, {"id": 637, "dom": "P", "ent": "beetle", "cls": "chick"}, {"id": 638, "dom": "P", "ent": "beetle", "cls": "clock"}, {"id": 639, "dom": "P", "ent": "beetle", "cls": "deer"}, {"id": 640, "dom": "P", "ent": "beetle", "cls": "dog"}, {"id": 641, "dom": "P", "ent": "beetle", "cls": "elephant"}, {"id": 642, "dom": "P", "ent": "beetle", "cls": "embassy"}, {"id": 643, "dom": "P", "ent": "beetle", "cls": "garage"}, {"id": 644, "dom": "P", "ent": "beetle", "cls": "highway"}, {"id": 645, "dom": "P", "ent": "beetle", "cls": "kidney"}, {"id": 646, "dom": "P", "ent": "beetle", "cls": "lamb"}, {"id": 647, "dom": "P", "ent": "beetle", "cls": "male"}, {"id": 648, "dom": "P", "ent": "beetle", "cls": "marijuana"}, {"id": 649, "dom": "P", "ent": "beetle", "cls": "puppy"}, {"id": 650, "dom": "P", "ent": "beetle", "cls": "receiver"}, {"id": 651, "dom": "P", "ent": "beetle", "cls": "reed"}, {"id": 652, "dom": "P", "ent": "beetle", "cls": "rifle"}, {"id": 653, "dom": "P", "ent": "beetle", "cls": "road"}, {"id": 654, "dom": "P", "ent": "beetle", "cls": "salmon"}, {"id": 655, "dom": "P", "ent": "beetle", "cls": "shark"}, {"id": 656, "dom": "P", "ent": "beetle", "cls": "sheep"}, {"id": 657, "dom": "P", "ent": "beetle", "cls": "shook"}, {"id": 658, "dom": "P", "ent": "beetle", "cls": "tank"}, {"id": 659, "dom": "P", "ent": "beetle", "cls": "wheat"}, {"id": 660, "dom": "P", "ent": "bell", "cls": "architecture"}, {"id": 661, "dom": "P", "ent": "bell", "cls": "bacteria"}, {"id": 662, "dom": "P", "ent": "bell", "cls": "beef"}, {"id": 663, "dom": "P", "ent": "bell", "cls": "bridge"}, {"id": 664, "dom": "P", "ent": "bell", "cls": "bud"}, {"id": 665, "dom": "P", "ent": "bell", "cls": "cat"}, {"id": 666, "dom": "P", "ent": "bell", "cls": "chapel"}, {"id": 667, "dom": "P", "ent": "bell", "cls": "chick"}, {"id": 668, "dom": "P", "ent": "bell", "cls": "clock"}, {"id": 669, "dom": "P", "ent": "bell", "cls": "deer"}, {"id": 670, "dom": "P", "ent": "bell", "cls": "dog"}, {"id": 671, "dom": "P", "ent": "bell", "cls": "elephant"}, {"id": 672, "dom": "P", "ent": "bell", "cls": "embassy"}, {"id": 673, "dom": "P", "ent": "bell", "cls": "garage"}, {"id": 674, "dom": "P", "ent": "bell", "cls": "highway"}, {"id": 675, "dom": "P", "ent": "bell", "cls": "kidney"}, {"id": 676, "dom": "P", "ent": "bell", "cls": "lamb"}, {"id": 677, "dom": "P", "ent": "bell", "cls": "male"}, {"id": 678, "dom": "P", "ent": "bell", "cls": "marijuana"}, {"id": 679, "dom": "P", "ent": "bell", "cls": "puppy"}, {"id": 680, "dom": "P", "ent": "bell", "cls": "receiver"}, {"id": 681, "dom": "P", "ent": "bell", "cls": "reed"}, {"id": 682, "dom": "P", "ent": "bell", "cls": "rifle"}, {"id": 683, "dom": "P", "ent": "bell", "cls": "road"}, {"id": 684, "dom": "P", "ent": "bell", "cls": "salmon"}, {"id": 685, "dom": "P", "ent": "bell", "cls": "shark"}, {"id": 686, "dom": "P", "ent": "bell", "cls": "sheep"}, {"id": 687, "dom": "P", "ent": "bell", "cls": "shook"}, {"id": 688, "dom": "P", "ent": "bell", "cls": "tank"}, {"id": 689, "dom": "P", "ent": "bell", "cls": "wheat"}, {"id": 690, "dom": "P", "ent": "belly", "cls": "architecture"}, {"id": 691, "dom": "P", "ent": "belly", "cls": "bacteria"}, {"id": 692, "dom": "P", "ent": "belly", "cls": "beef"}, {"id": 693, "dom": "P", "ent": "belly", "cls": "bridge"}, {"id": 694, "dom": "P", "ent": "belly", "cls": "bud"}, {"id": 695, "dom": "P", "ent": "belly", "cls": "cat"}, {"id": 696, "dom": "P", "ent": "belly", "cls": "chapel"}, {"id": 697, "dom": "P", "ent": "belly", "cls": "chick"}, {"id": 698, "dom": "P", "ent": "belly", "cls": "clock"}, {"id": 699, "dom": "P", "ent": "belly", "cls": "deer"}, {"id": 700, "dom": "P", "ent": "belly", "cls": "dog"}, {"id": 701, "dom": "P", "ent": "belly", "cls": "elephant"}, {"id": 702, "dom": "P", "ent": "belly", "cls": "embassy"}, {"id": 703, "dom": "P", "ent": "belly", "cls": "garage"}, {"id": 704, "dom": "P", "ent": "belly", "cls": "highway"}, {"id": 705, "dom": "P", "ent": "belly", "cls": "kidney"}, {"id": 706, "dom": "P", "ent": "belly", "cls": "lamb"}, {"id": 707, "dom": "P", "ent": "belly", "cls": "male"}, {"id": 708, "dom": "P", "ent": "belly", "cls": "marijuana"}, {"id": 709, "dom": "P", "ent": "belly", "cls": "puppy"}, {"id": 710, "dom": "P", "ent": "belly", "cls": "receiver"}, {"id": 711, "dom": "P", "ent": "belly", "cls": "reed"}, {"id": 712, "dom": "P", "ent": "belly", "cls": "rifle"}, {"id": 713, "dom": "P", "ent": "belly", "cls": "road"}, {"id": 714, "dom": "P", "ent": "belly", "cls": "salmon"}, {"id": 715, "dom": "P", "ent": "belly", "cls": "shark"}, {"id": 716, "dom": "P", "ent": "belly", "cls": "sheep"}, {"id": 717, "dom": "P", "ent": "belly", "cls": "shook"}, {"id": 718, "dom": "P", "ent": "belly", "cls": "tank"}, {"id": 719, "dom": "P", "ent": "belly", "cls": "wheat"}, {"id": 720, "dom": "P", "ent": "bend", "cls": "architecture"}, {"id": 721, "dom": "P", "ent": "bend", "cls": "bacteria"}, {"id": 722, "dom": "P", "ent": "bend", "cls": "beef"}, {"id": 723, "dom": "P", "ent": "bend", "cls": "bridge"}, {"id": 724, "dom": "P", "ent": "bend", "cls": "bud"}, {"id": 725, "dom": "P", "ent": "bend", "cls": "cat"}, {"id": 726, "dom": "P", "ent": "bend", "cls": "chapel"}, {"id": 727, "dom": "P", "ent": "bend", "cls": "chick"}, {"id": 728, "dom": "P", "ent": "bend", "cls": "clock"}, {"id": 729, "dom": "P", "ent": "bend", "cls": "deer"}, {"id": 730, "dom": "P", "ent": "bend", "cls": "dog"}, {"id": 731, "dom": "P", "ent": "bend", "cls": "elephant"}, {"id": 732, "dom": "P", "ent": "bend", "cls": "embassy"}, {"id": 733, "dom": "P", "ent": "bend", "cls": "garage"}, {"id": 734, "dom": "P", "ent": "bend", "cls": "highway"}, {"id": 735, "dom": "P", "ent": "bend", "cls": "kidney"}, {"id": 736, "dom": "P", "ent": "bend", "cls": "lamb"}, {"id": 737, "dom": "P", "ent": "bend", "cls": "male"}, {"id": 738, "dom": "P", "ent": "bend", "cls": "marijuana"}, {"id": 739, "dom": "P", "ent": "bend", "cls": "puppy"}, {"id": 740, "dom": "P", "ent": "bend", "cls": "receiver"}, {"id": 741, "dom": "P", "ent": "bend", "cls": "reed"}, {"id": 742, "dom": "P", "ent": "bend", "cls": "rifle"}, {"id": 743, "dom": "P", "ent": "bend", "cls": "road"}, {"id": 744, "dom": "P", "ent": "bend", "cls": "salmon"}, {"id": 745, "dom": "P", "ent": "bend", "cls": "shark"}, {"id": 746, "dom": "P", "ent": "bend", "cls": "sheep"}, {"id": 747, "dom": "P", "ent": "bend", "cls": "shook"}, {"id": 748, "dom": "P", "ent": "bend", "cls": "tank"}, {"id": 749, "dom": "P", "ent": "bend", "cls": "wheat"}, {"id": 750, "dom": "P", "ent": "bill", "cls": "architecture"}, {"id": 751, "dom": "P", "ent": "bill", "cls": "bacteria"}, {"id": 752, "dom": "P", "ent": "bill", "cls": "beef"}, {"id": 753, "dom": "P", "ent": "bill", "cls": "bridge"}, {"id": 754, "dom": "P", "ent": "bill", "cls": "bud"}, {"id": 755, "dom": "P", "ent": "bill", "cls": "cat"}, {"id": 756, "dom": "P", "ent": "bill", "cls": "chapel"}, {"id": 757, "dom": "P", "ent": "bill", "cls": "chick"}, {"id": 758, "dom": "P", "ent": "bill", "cls": "clock"}, {"id": 759, "dom": "P", "ent": "bill", "cls": "deer"}, {"id": 760, "dom": "P", "ent": "bill", "cls": "dog"}, {"id": 761, "dom": "P", "ent": "bill", "cls": "elephant"}, {"id": 762, "dom": "P", "ent": "bill", "cls": "embassy"}, {"id": 763, "dom": "P", "ent": "bill", "cls": "garage"}, {"id": 764, "dom": "P", "ent": "bill", "cls": "highway"}, {"id": 765, "dom": "P", "ent": "bill", "cls": "kidney"}, {"id": 766, "dom": "P", "ent": "bill", "cls": "lamb"}, {"id": 767, "dom": "P", "ent": "bill", "cls": "male"}, {"id": 768, "dom": "P", "ent": "bill", "cls": "marijuana"}, {"id": 769, "dom": "P", "ent": "bill", "cls": "puppy"}, {"id": 770, "dom": "P", "ent": "bill", "cls": "receiver"}, {"id": 771, "dom": "P", "ent": "bill", "cls": "reed"}, {"id": 772, "dom": "P", "ent": "bill", "cls": "rifle"}, {"id": 773, "dom": "P", "ent": "bill", "cls": "road"}, {"id": 774, "dom": "P", "ent": "bill", "cls": "salmon"}, {"id": 775, "dom": "P", "ent": "bill", "cls": "shark"}, {"id": 776, "dom": "P", "ent": "bill", "cls": "sheep"}, {"id": 777, "dom": "P", "ent": "bill", "cls": "shook"}, {"id": 778, "dom": "P", "ent": "bill", "cls": "tank"}, {"id": 779, "dom": "P", "ent": "bill", "cls": "wheat"}, {"id": 780, "dom": "P", "ent": "blood", "cls": "architecture"}, {"id": 781, "dom": "P", "ent": "blood", "cls": "bacteria"}, {"id": 782, "dom": "P", "ent": "blood", "cls": "beef"}, {"id": 783, "dom": "P", "ent": "blood", "cls": "bridge"}, {"id": 784, "dom": "P", "ent": "blood", "cls": "bud"}, {"id": 785, "dom": "P", "ent": "blood", "cls": "cat"}, {"id": 786, "dom": "P", "ent": "blood", "cls": "chapel"}, {"id": 787, "dom": "P", "ent": "blood", "cls": "chick"}, {"id": 788, "dom": "P", "ent": "blood", "cls": "clock"}, {"id": 789, "dom": "P", "ent": "blood", "cls": "deer"}, {"id": 790, "dom": "P", "ent": "blood", "cls": "dog"}, {"id": 791, "dom": "P", "ent": "blood", "cls": "elephant"}, {"id": 792, "dom": "P", "ent": "blood", "cls": "embassy"}, {"id": 793, "dom": "P", "ent": "blood", "cls": "garage"}, {"id": 794, "dom": "P", "ent": "blood", "cls": "highway"}, {"id": 795, "dom": "P", "ent": "blood", "cls": "kidney"}, {"id": 796, "dom": "P", "ent": "blood", "cls": "lamb"}, {"id": 797, "dom": "P", "ent": "blood", "cls": "male"}, {"id": 798, "dom": "P", "ent": "blood", "cls": "marijuana"}, {"id": 799, "dom": "P", "ent": "blood", "cls": "puppy"}, {"id": 800, "dom": "P", "ent": "blood", "cls": "receiver"}, {"id": 801, "dom": "P", "ent": "blood", "cls": "reed"}, {"id": 802, "dom": "P", "ent": "blood", "cls": "rifle"}, {"id": 803, "dom": "P", "ent": "blood", "cls": "road"}, {"id": 804, "dom": "P", "ent": "blood", "cls": "salmon"}, {"id": 805, "dom": "P", "ent": "blood", "cls": "shark"}, {"id": 806, "dom": "P", "ent": "blood", "cls": "sheep"}, {"id": 807, "dom": "P", "ent": "blood", "cls": "shook"}, {"id": 808, "dom": "P", "ent": "blood", "cls": "tank"}, {"id": 809, "dom": "P", "ent": "blood", "cls": "wheat"}, {"id": 810, "dom": "P", "ent": "blue-belly", "cls": "architecture"}, {"id": 811, "dom": "P", "ent": "blue-belly", "cls": "bacteria"}, {"id": 812, "dom": "P", "ent": "blue-belly", "cls": "beef"}, {"id": 813, "dom": "P", "ent": "blue-belly", "cls": "bridge"}, {"id": 814, "dom": "P", "ent": "blue-belly", "cls": "bud"}, {"id": 815, "dom": "P", "ent": "blue-belly", "cls": "cat"}, {"id": 816, "dom": "P", "ent": "blue-belly", "cls": "chapel"}, {"id": 817, "dom": "P", "ent": "blue-belly", "cls": "chick"}, {"id": 818, "dom": "P", "ent": "blue-belly", "cls": "clock"}, {"id": 819, "dom": "P", "ent": "blue-belly", "cls": "deer"}, {"id": 820, "dom": "P", "ent": "blue-belly", "cls": "dog"}, {"id": 821, "dom": "P", "ent": "blue-belly", "cls": "elephant"}, {"id": 822, "dom": "P", "ent": "blue-belly", "cls": "embassy"}, {"id": 823, "dom": "P", "ent": "blue-belly", "cls": "garage"}, {"id": 824, "dom": "P", "ent": "blue-belly", "cls": "highway"}, {"id": 825, "dom": "P", "ent": "blue-belly", "cls": "kidney"}, {"id": 826, "dom": "P", "ent": "blue-belly", "cls": "lamb"}, {"id": 827, "dom": "P", "ent": "blue-belly", "cls": "male"}, {"id": 828, "dom": "P", "ent": "blue-belly", "cls": "marijuana"}, {"id": 829, "dom": "P", "ent": "blue-belly", "cls": "puppy"}, {"id": 830, "dom": "P", "ent": "blue-belly", "cls": "receiver"}, {"id": 831, "dom": "P", "ent": "blue-belly", "cls": "reed"}, {"id": 832, "dom": "P", "ent": "blue-belly", "cls": "rifle"}, {"id": 833, "dom": "P", "ent": "blue-belly", "cls": "road"}, {"id": 834, "dom": "P", "ent": "blue-belly", "cls": "salmon"}, {"id": 835, "dom": "P", "ent": "blue-belly", "cls": "shark"}, {"id": 836, "dom": "P", "ent": "blue-belly", "cls": "sheep"}, {"id": 837, "dom": "P", "ent": "blue-belly", "cls": "shook"}, {"id": 838, "dom": "P", "ent": "blue-belly", "cls": "tank"}, {"id": 839, "dom": "P", "ent": "blue-belly", "cls": "wheat"}, {"id": 840, "dom": "P", "ent": "boiler", "cls": "architecture"}, {"id": 841, "dom": "P", "ent": "boiler", "cls": "bacteria"}, {"id": 842, "dom": "P", "ent": "boiler", "cls": "beef"}, {"id": 843, "dom": "P", "ent": "boiler", "cls": "bridge"}, {"id": 844, "dom": "P", "ent": "boiler", "cls": "bud"}, {"id": 845, "dom": "P", "ent": "boiler", "cls": "cat"}, {"id": 846, "dom": "P", "ent": "boiler", "cls": "chapel"}, {"id": 847, "dom": "P", "ent": "boiler", "cls": "chick"}, {"id": 848, "dom": "P", "ent": "boiler", "cls": "clock"}, {"id": 849, "dom": "P", "ent": "boiler", "cls": "deer"}, {"id": 850, "dom": "P", "ent": "boiler", "cls": "dog"}, {"id": 851, "dom": "P", "ent": "boiler", "cls": "elephant"}, {"id": 852, "dom": "P", "ent": "boiler", "cls": "embassy"}, {"id": 853, "dom": "P", "ent": "boiler", "cls": "garage"}, {"id": 854, "dom": "P", "ent": "boiler", "cls": "highway"}, {"id": 855, "dom": "P", "ent": "boiler", "cls": "kidney"}, {"id": 856, "dom": "P", "ent": "boiler", "cls": "lamb"}, {"id": 857, "dom": "P", "ent": "boiler", "cls": "male"}, {"id": 858, "dom": "P", "ent": "boiler", "cls": "marijuana"}, {"id": 859, "dom": "P", "ent": "boiler", "cls": "puppy"}, {"id": 860, "dom": "P", "ent": "boiler", "cls": "receiver"}, {"id": 861, "dom": "P", "ent": "boiler", "cls": "reed"}, {"id": 862, "dom": "P", "ent": "boiler", "cls": "rifle"}, {"id": 863, "dom": "P", "ent": "boiler", "cls": "road"}, {"id": 864, "dom": "P", "ent": "boiler", "cls": "salmon"}, {"id": 865, "dom": "P", "ent": "boiler", "cls": "shark"}, {"id": 866, "dom": "P", "ent": "boiler", "cls": "sheep"}, {"id": 867, "dom": "P", "ent": "boiler", "cls": "shook"}, {"id": 868, "dom": "P", "ent": "boiler", "cls": "tank"}, {"id": 869, "dom": "P", "ent": "boiler", "cls": "wheat"}, {"id": 870, "dom": "P", "ent": "bolt", "cls": "architecture"}, {"id": 871, "dom": "P", "ent": "bolt", "cls": "bacteria"}, {"id": 872, "dom": "P", "ent": "bolt", "cls": "beef"}, {"id": 873, "dom": "P", "ent": "bolt", "cls": "bridge"}, {"id": 874, "dom": "P", "ent": "bolt", "cls": "bud"}, {"id": 875, "dom": "P", "ent": "bolt", "cls": "cat"}, {"id": 876, "dom": "P", "ent": "bolt", "cls": "chapel"}, {"id": 877, "dom": "P", "ent": "bolt", "cls": "chick"}, {"id": 878, "dom": "P", "ent": "bolt", "cls": "clock"}, {"id": 879, "dom": "P", "ent": "bolt", "cls": "deer"}, {"id": 880, "dom": "P", "ent": "bolt", "cls": "dog"}, {"id": 881, "dom": "P", "ent": "bolt", "cls": "elephant"}, {"id": 882, "dom": "P", "ent": "bolt", "cls": "embassy"}, {"id": 883, "dom": "P", "ent": "bolt", "cls": "garage"}, {"id": 884, "dom": "P", "ent": "bolt", "cls": "highway"}, {"id": 885, "dom": "P", "ent": "bolt", "cls": "kidney"}, {"id": 886, "dom": "P", "ent": "bolt", "cls": "lamb"}, {"id": 887, "dom": "P", "ent": "bolt", "cls": "male"}, {"id": 888, "dom": "P", "ent": "bolt", "cls": "marijuana"}, {"id": 889, "dom": "P", "ent": "bolt", "cls": "puppy"}, {"id": 890, "dom": "P", "ent": "bolt", "cls": "receiver"}, {"id": 891, "dom": "P", "ent": "bolt", "cls": "reed"}, {"id": 892, "dom": "P", "ent": "bolt", "cls": "rifle"}, {"id": 893, "dom": "P", "ent": "bolt", "cls": "road"}, {"id": 894, "dom": "P", "ent": "bolt", "cls": "salmon"}, {"id": 895, "dom": "P", "ent": "bolt", "cls": "shark"}, {"id": 896, "dom": "P", "ent": "bolt", "cls": "sheep"}, {"id": 897, "dom": "P", "ent": "bolt", "cls": "shook"}, {"id": 898, "dom": "P", "ent": "bolt", "cls": "tank"}, {"id": 899, "dom": "P", "ent": "bolt", "cls": "wheat"}, {"id": 900, "dom": "P", "ent": "bottle", "cls": "architecture"}, {"id": 901, "dom": "P", "ent": "bottle", "cls": "bacteria"}, {"id": 902, "dom": "P", "ent": "bottle", "cls": "beef"}, {"id": 903, "dom": "P", "ent": "bottle", "cls": "bridge"}, {"id": 904, "dom": "P", "ent": "bottle", "cls": "bud"}, {"id": 905, "dom": "P", "ent": "bottle", "cls": "cat"}, {"id": 906, "dom": "P", "ent": "bottle", "cls": "chapel"}, {"id": 907, "dom": "P", "ent": "bottle", "cls": "chick"}, {"id": 908, "dom": "P", "ent": "bottle", "cls": "clock"}, {"id": 909, "dom": "P", "ent": "bottle", "cls": "deer"}, {"id": 910, "dom": "P", "ent": "bottle", "cls": "dog"}, {"id": 911, "dom": "P", "ent": "bottle", "cls": "elephant"}, {"id": 912, "dom": "P", "ent": "bottle", "cls": "embassy"}, {"id": 913, "dom": "P", "ent": "bottle", "cls": "garage"}, {"id": 914, "dom": "P", "ent": "bottle", "cls": "highway"}, {"id": 915, "dom": "P", "ent": "bottle", "cls": "kidney"}, {"id": 916, "dom": "P", "ent": "bottle", "cls": "lamb"}, {"id": 917, "dom": "P", "ent": "bottle", "cls": "male"}, {"id": 918, "dom": "P", "ent": "bottle", "cls": "marijuana"}, {"id": 919, "dom": "P", "ent": "bottle", "cls": "puppy"}, {"id": 920, "dom": "P", "ent": "bottle", "cls": "receiver"}, {"id": 921, "dom": "P", "ent": "bottle", "cls": "reed"}, {"id": 922, "dom": "P", "ent": "bottle", "cls": "rifle"}, {"id": 923, "dom": "P", "ent": "bottle", "cls": "road"}, {"id": 924, "dom": "P", "ent": "bottle", "cls": "salmon"}, {"id": 925, "dom": "P", "ent": "bottle", "cls": "shark"}, {"id": 926, "dom": "P", "ent": "bottle", "cls": "sheep"}, {"id": 927, "dom": "P", "ent": "bottle", "cls": "shook"}, {"id": 928, "dom": "P", "ent": "bottle", "cls": "tank"}, {"id": 929, "dom": "P", "ent": "bottle", "cls": "wheat"}, {"id": 930, "dom": "P", "ent": "bottle-tree", "cls": "architecture"}, {"id": 931, "dom": "P", "ent": "bottle-tree", "cls": "bacteria"}, {"id": 932, "dom": "P", "ent": "bottle-tree", "cls": "beef"}, {"id": 933, "dom": "P", "ent": "bottle-tree", "cls": "bridge"}, {"id": 934, "dom": "P", "ent": "bottle-tree", "cls": "bud"}, {"id": 935, "dom": "P", "ent": "bottle-tree", "cls": "cat"}, {"id": 936, "dom": "P", "ent": "bottle-tree", "cls": "chapel"}, {"id": 937, "dom": "P", "ent": "bottle-tree", "cls": "chick"}, {"id": 938, "dom": "P", "ent": "bottle-tree", "cls": "clock"}, {"id": 939, "dom": "P", "ent": "bottle-tree", "cls": "deer"}, {"id": 940, "dom": "P", "ent": "bottle-tree", "cls": "dog"}, {"id": 941, "dom": "P", "ent": "bottle-tree", "cls": "elephant"}, {"id": 942, "dom": "P", "ent": "bottle-tree", "cls": "embassy"}, {"id": 943, "dom": "P", "ent": "bottle-tree", "cls": "garage"}, {"id": 944, "dom": "P", "ent": "bottle-tree", "cls": "highway"}, {"id": 945, "dom": "P", "ent": "bottle-tree", "cls": "kidney"}, {"id": 946, "dom": "P", "ent": "bottle-tree", "cls": "lamb"}, {"id": 947, "dom": "P", "ent": "bottle-tree", "cls": "male"}, {"id": 948, "dom": "P", "ent": "bottle-tree", "cls": "marijuana"}, {"id": 949, "dom": "P", "ent": "bottle-tree", "cls": "puppy"}, {"id": 950, "dom": "P", "ent": "bottle-tree", "cls": "receiver"}, {"id": 951, "dom": "P", "ent": "bottle-tree", "cls": "reed"}, {"id": 952, "dom": "P", "ent": "bottle-tree", "cls": "rifle"}, {"id": 953, "dom": "P", "ent": "bottle-tree", "cls": "road"}, {"id": 954, "dom": "P", "ent": "bottle-tree", "cls": "salmon"}, {"id": 955, "dom": "P", "ent": "bottle-tree", "cls": "shark"}, {"id": 956, "dom": "P", "ent": "bottle-tree", "cls": "sheep"}, {"id": 957, "dom": "P", "ent": "bottle-tree", "cls": "shook"}, {"id": 958, "dom": "P", "ent": "bottle-tree", "cls": "tank"}, {"id": 959, "dom": "P", "ent": "bottle-tree", "cls": "wheat"}, {"id": 960, "dom": "P", "ent": "boulevard", "cls": "architecture"}, {"id": 961, "dom": "P", "ent": "boulevard", "cls": "bacteria"}, {"id": 962, "dom": "P", "ent": "boulevard", "cls": "beef"}, {"id": 963, "dom": "P", "ent": "boulevard", "cls": "bridge"}, {"id": 964, "dom": "P", "ent": "boulevard", "cls": "bud"}, {"id": 965, "dom": "P", "ent": "boulevard", "cls": "cat"}, {"id": 966, "dom": "P", "ent": "boulevard", "cls": "chapel"}, {"id": 967, "dom": "P", "ent": "boulevard", "cls": "chick"}, {"id": 968, "dom": "P", "ent": "boulevard", "cls": "clock"}, {"id": 969, "dom": "P", "ent": "boulevard", "cls": "deer"}, {"id": 970, "dom": "P", "ent": "boulevard", "cls": "dog"}, {"id": 971, "dom": "P", "ent": "boulevard", "cls": "elephant"}, {"id": 972, "dom": "P", "ent": "boulevard", "cls": "embassy"}, {"id": 973, "dom": "P", "ent": "boulevard", "cls": "garage"}, {"id": 974, "dom": "P", "ent": "boulevard", "cls": "highway"}, {"id": 975, "dom": "P", "ent": "boulevard", "cls": "kidney"}, {"id": 976, "dom": "P", "ent": "boulevard", "cls": "lamb"}, {"id": 977, "dom": "P", "ent": "boulevard", "cls": "male"}, {"id": 978, "dom": "P", "ent": "boulevard", "cls": "marijuana"}, {"id": 979, "dom": "P", "ent": "boulevard", "cls": "puppy"}, {"id": 980, "dom": "P", "ent": "boulevard", "cls": "receiver"}, {"id": 981, "dom": "P", "ent": "boulevard", "cls": "reed"}, {"id": 982, "dom": "P", "ent": "boulevard", "cls": "rifle"}, {"id": 983, "dom": "P", "ent": "boulevard", "cls": "road"}, {"id": 984, "dom": "P", "ent": "boulevard", "cls": "salmon"}, {"id": 985, "dom": "P", "ent": "boulevard", "cls": "shark"}, {"id": 986, "dom": "P", "ent": "boulevard", "cls": "sheep"}, {"id": 987, "dom": "P", "ent": "boulevard", "cls": "shook"}, {"id": 988, "dom": "P", "ent": "boulevard", "cls": "tank"}, {"id": 989, "dom": "P", "ent": "boulevard", "cls": "wheat"}, {"id": 990, "dom": "P", "ent": "brain", "cls": "architecture"}, {"id": 991, "dom": "P", "ent": "brain", "cls": "bacteria"}, {"id": 992, "dom": "P", "ent": "brain", "cls": "beef"}, {"id": 993, "dom": "P", "ent": "brain", "cls": "bridge"}, {"id": 994, "dom": "P", "ent": "brain", "cls": "bud"}, {"id": 995, "dom": "P", "ent": "brain", "cls": "cat"}, {"id": 996, "dom": "P", "ent": "brain", "cls": "chapel"}, {"id": 997, "dom": "P", "ent": "brain", "cls": "chick"}, {"id": 998, "dom": "P", "ent": "brain", "cls": "clock"}, {"id": 999, "dom": "P", "ent": "brain", "cls": "deer"}, {"id": 1000, "dom": "P", "ent": "brain", "cls": "dog"}, {"id": 1001, "dom": "P", "ent": "brain", "cls": "elephant"}, {"id": 1002, "dom": "P", "ent": "brain", "cls": "embassy"}, {"id": 1003, "dom": "P", "ent": "brain", "cls": "garage"}, {"id": 1004, "dom": "P", "ent": "brain", "cls": "highway"}, {"id": 1005, "dom": "P", "ent": "brain", "cls": "kidney"}, {"id": 1006, "dom": "P", "ent": "brain", "cls": "lamb"}, {"id": 1007, "dom": "P", "ent": "brain", "cls": "male"}, {"id": 1008, "dom": "P", "ent": "brain", "cls": "marijuana"}, {"id": 1009, "dom": "P", "ent": "brain", "cls": "puppy"}, {"id": 1010, "dom": "P", "ent": "brain", "cls": "receiver"}, {"id": 1011, "dom": "P", "ent": "brain", "cls": "reed"}, {"id": 1012, "dom": "P", "ent": "brain", "cls": "rifle"}, {"id": 1013, "dom": "P", "ent": "brain", "cls": "road"}, {"id": 1014, "dom": "P", "ent": "brain", "cls": "salmon"}, {"id": 1015, "dom": "P", "ent": "brain", "cls": "shark"}, {"id": 1016, "dom": "P", "ent": "brain", "cls": "sheep"}, {"id": 1017, "dom": "P", "ent": "brain", "cls": "shook"}, {"id": 1018, "dom": "P", "ent": "brain", "cls": "tank"}, {"id": 1019, "dom": "P", "ent": "brain", "cls": "wheat"}, {"id": 1020, "dom": "P", "ent": "brain-stem", "cls": "architecture"}, {"id": 1021, "dom": "P", "ent": "brain-stem", "cls": "bacteria"}, {"id": 1022, "dom": "P", "ent": "brain-stem", "cls": "beef"}, {"id": 1023, "dom": "P", "ent": "brain-stem", "cls": "bridge"}, {"id": 1024, "dom": "P", "ent": "brain-stem", "cls": "bud"}, {"id": 1025, "dom": "P", "ent": "brain-stem", "cls": "cat"}, {"id": 1026, "dom": "P", "ent": "brain-stem", "cls": "chapel"}, {"id": 1027, "dom": "P", "ent": "brain-stem", "cls": "chick"}, {"id": 1028, "dom": "P", "ent": "brain-stem", "cls": "clock"}, {"id": 1029, "dom": "P", "ent": "brain-stem", "cls": "deer"}, {"id": 1030, "dom": "P", "ent": "brain-stem", "cls": "dog"}, {"id": 1031, "dom": "P", "ent": "brain-stem", "cls": "elephant"}, {"id": 1032, "dom": "P", "ent": "brain-stem", "cls": "embassy"}, {"id": 1033, "dom": "P", "ent": "brain-stem", "cls": "garage"}, {"id": 1034, "dom": "P", "ent": "brain-stem", "cls": "highway"}, {"id": 1035, "dom": "P", "ent": "brain-stem", "cls": "kidney"}, {"id": 1036, "dom": "P", "ent": "brain-stem", "cls": "lamb"}, {"id": 1037, "dom": "P", "ent": "brain-stem", "cls": "male"}, {"id": 1038, "dom": "P", "ent": "brain-stem", "cls": "marijuana"}, {"id": 1039, "dom": "P", "ent": "brain-stem", "cls": "puppy"}, {"id": 1040, "dom": "P", "ent": "brain-stem", "cls": "receiver"}, {"id": 1041, "dom": "P", "ent": "brain-stem", "cls": "reed"}, {"id": 1042, "dom": "P", "ent": "brain-stem", "cls": "rifle"}, {"id": 1043, "dom": "P", "ent": "brain-stem", "cls": "road"}, {"id": 1044, "dom": "P", "ent": "brain-stem", "cls": "salmon"}, {"id": 1045, "dom": "P", "ent": "brain-stem", "cls": "shark"}, {"id": 1046, "dom": "P", "ent": "brain-stem", "cls": "sheep"}, {"id": 1047, "dom": "P", "ent": "brain-stem", "cls": "shook"}, {"id": 1048, "dom": "P", "ent": "brain-stem", "cls": "tank"}, {"id": 1049, "dom": "P", "ent": "brain-stem", "cls": "wheat"}, {"id": 1050, "dom": "P", "ent": "brake", "cls": "architecture"}, {"id": 1051, "dom": "P", "ent": "brake", "cls": "bacteria"}, {"id": 1052, "dom": "P", "ent": "brake", "cls": "beef"}, {"id": 1053, "dom": "P", "ent": "brake", "cls": "bridge"}, {"id": 1054, "dom": "P", "ent": "brake", "cls": "bud"}, {"id": 1055, "dom": "P", "ent": "brake", "cls": "cat"}, {"id": 1056, "dom": "P", "ent": "brake", "cls": "chapel"}, {"id": 1057, "dom": "P", "ent": "brake", "cls": "chick"}, {"id": 1058, "dom": "P", "ent": "brake", "cls": "clock"}, {"id": 1059, "dom": "P", "ent": "brake", "cls": "deer"}, {"id": 1060, "dom": "P", "ent": "brake", "cls": "dog"}, {"id": 1061, "dom": "P", "ent": "brake", "cls": "elephant"}, {"id": 1062, "dom": "P", "ent": "brake", "cls": "embassy"}, {"id": 1063, "dom": "P", "ent": "brake", "cls": "garage"}, {"id": 1064, "dom": "P", "ent": "brake", "cls": "highway"}, {"id": 1065, "dom": "P", "ent": "brake", "cls": "kidney"}, {"id": 1066, "dom": "P", "ent": "brake", "cls": "lamb"}, {"id": 1067, "dom": "P", "ent": "brake", "cls": "male"}, {"id": 1068, "dom": "P", "ent": "brake", "cls": "marijuana"}, {"id": 1069, "dom": "P", "ent": "brake", "cls": "puppy"}, {"id": 1070, "dom": "P", "ent": "brake", "cls": "receiver"}, {"id": 1071, "dom": "P", "ent": "brake", "cls": "reed"}, {"id": 1072, "dom": "P", "ent": "brake", "cls": "rifle"}, {"id": 1073, "dom": "P", "ent": "brake", "cls": "road"}, {"id": 1074, "dom": "P", "ent": "brake", "cls": "salmon"}, {"id": 1075, "dom": "P", "ent": "brake", "cls": "shark"}, {"id": 1076, "dom": "P", "ent": "brake", "cls": "sheep"}, {"id": 1077, "dom": "P", "ent": "brake", "cls": "shook"}, {"id": 1078, "dom": "P", "ent": "brake", "cls": "tank"}, {"id": 1079, "dom": "P", "ent": "brake", "cls": "wheat"}, {"id": 1080, "dom": "P", "ent": "breast", "cls": "architecture"}, {"id": 1081, "dom": "P", "ent": "breast", "cls": "bacteria"}, {"id": 1082, "dom": "P", "ent": "breast", "cls": "beef"}, {"id": 1083, "dom": "P", "ent": "breast", "cls": "bridge"}, {"id": 1084, "dom": "P", "ent": "breast", "cls": "bud"}, {"id": 1085, "dom": "P", "ent": "breast", "cls": "cat"}, {"id": 1086, "dom": "P", "ent": "breast", "cls": "chapel"}, {"id": 1087, "dom": "P", "ent": "breast", "cls": "chick"}, {"id": 1088, "dom": "P", "ent": "breast", "cls": "clock"}, {"id": 1089, "dom": "P", "ent": "breast", "cls": "deer"}, {"id": 1090, "dom": "P", "ent": "breast", "cls": "dog"}, {"id": 1091, "dom": "P", "ent": "breast", "cls": "elephant"}, {"id": 1092, "dom": "P", "ent": "breast", "cls": "embassy"}, {"id": 1093, "dom": "P", "ent": "breast", "cls": "garage"}, {"id": 1094, "dom": "P", "ent": "breast", "cls": "highway"}, {"id": 1095, "dom": "P", "ent": "breast", "cls": "kidney"}, {"id": 1096, "dom": "P", "ent": "breast", "cls": "lamb"}, {"id": 1097, "dom": "P", "ent": "breast", "cls": "male"}, {"id": 1098, "dom": "P", "ent": "breast", "cls": "marijuana"}, {"id": 1099, "dom": "P", "ent": "breast", "cls": "puppy"}, {"id": 1100, "dom": "P", "ent": "breast", "cls": "receiver"}, {"id": 1101, "dom": "P", "ent": "breast", "cls": "reed"}, {"id": 1102, "dom": "P", "ent": "breast", "cls": "rifle"}, {"id": 1103, "dom": "P", "ent": "breast", "cls": "road"}, {"id": 1104, "dom": "P", "ent": "breast", "cls": "salmon"}, {"id": 1105, "dom": "P", "ent": "breast", "cls": "shark"}, {"id": 1106, "dom": "P", "ent": "breast", "cls": "sheep"}, {"id": 1107, "dom": "P", "ent": "breast", "cls": "shook"}, {"id": 1108, "dom": "P", "ent": "breast", "cls": "tank"}, {"id": 1109, "dom": "P", "ent": "breast", "cls": "wheat"}, {"id": 1110, "dom": "P", "ent": "brim", "cls": "architecture"}, {"id": 1111, "dom": "P", "ent": "brim", "cls": "bacteria"}, {"id": 1112, "dom": "P", "ent": "brim", "cls": "beef"}, {"id": 1113, "dom": "P", "ent": "brim", "cls": "bridge"}, {"id": 1114, "dom": "P", "ent": "brim", "cls": "bud"}, {"id": 1115, "dom": "P", "ent": "brim", "cls": "cat"}, {"id": 1116, "dom": "P", "ent": "brim", "cls": "chapel"}, {"id": 1117, "dom": "P", "ent": "brim", "cls": "chick"}, {"id": 1118, "dom": "P", "ent": "brim", "cls": "clock"}, {"id": 1119, "dom": "P", "ent": "brim", "cls": "deer"}, {"id": 1120, "dom": "P", "ent": "brim", "cls": "dog"}, {"id": 1121, "dom": "P", "ent": "brim", "cls": "elephant"}, {"id": 1122, "dom": "P", "ent": "brim", "cls": "embassy"}, {"id": 1123, "dom": "P", "ent": "brim", "cls": "garage"}, {"id": 1124, "dom": "P", "ent": "brim", "cls": "highway"}, {"id": 1125, "dom": "P", "ent": "brim", "cls": "kidney"}, {"id": 1126, "dom": "P", "ent": "brim", "cls": "lamb"}, {"id": 1127, "dom": "P", "ent": "brim", "cls": "male"}, {"id": 1128, "dom": "P", "ent": "brim", "cls": "marijuana"}, {"id": 1129, "dom": "P", "ent": "brim", "cls": "puppy"}, {"id": 1130, "dom": "P", "ent": "brim", "cls": "receiver"}, {"id": 1131, "dom": "P", "ent": "brim", "cls": "reed"}, {"id": 1132, "dom": "P", "ent": "brim", "cls": "rifle"}, {"id": 1133, "dom": "P", "ent": "brim", "cls": "road"}, {"id": 1134, "dom": "P", "ent": "brim", "cls": "salmon"}, {"id": 1135, "dom": "P", "ent": "brim", "cls": "shark"}, {"id": 1136, "dom": "P", "ent": "brim", "cls": "sheep"}, {"id": 1137, "dom": "P", "ent": "brim", "cls": "shook"}, {"id": 1138, "dom": "P", "ent": "brim", "cls": "tank"}, {"id": 1139, "dom": "P", "ent": "brim", "cls": "wheat"}, {"id": 1140, "dom": "P", "ent": "brow", "cls": "architecture"}, {"id": 1141, "dom": "P", "ent": "brow", "cls": "bacteria"}, {"id": 1142, "dom": "P", "ent": "brow", "cls": "beef"}, {"id": 1143, "dom": "P", "ent": "brow", "cls": "bridge"}, {"id": 1144, "dom": "P", "ent": "brow", "cls": "bud"}, {"id": 1145, "dom": "P", "ent": "brow", "cls": "cat"}, {"id": 1146, "dom": "P", "ent": "brow", "cls": "chapel"}, {"id": 1147, "dom": "P", "ent": "brow", "cls": "chick"}, {"id": 1148, "dom": "P", "ent": "brow", "cls": "clock"}, {"id": 1149, "dom": "P", "ent": "brow", "cls": "deer"}, {"id": 1150, "dom": "P", "ent": "brow", "cls": "dog"}, {"id": 1151, "dom": "P", "ent": "brow", "cls": "elephant"}, {"id": 1152, "dom": "P", "ent": "brow", "cls": "embassy"}, {"id": 1153, "dom": "P", "ent": "brow", "cls": "garage"}, {"id": 1154, "dom": "P", "ent": "brow", "cls": "highway"}, {"id": 1155, "dom": "P", "ent": "brow", "cls": "kidney"}, {"id": 1156, "dom": "P", "ent": "brow", "cls": "lamb"}, {"id": 1157, "dom": "P", "ent": "brow", "cls": "male"}, {"id": 1158, "dom": "P", "ent": "brow", "cls": "marijuana"}, {"id": 1159, "dom": "P", "ent": "brow", "cls": "puppy"}, {"id": 1160, "dom": "P", "ent": "brow", "cls": "receiver"}, {"id": 1161, "dom": "P", "ent": "brow", "cls": "reed"}, {"id": 1162, "dom": "P", "ent": "brow", "cls": "rifle"}, {"id": 1163, "dom": "P", "ent": "brow", "cls": "road"}, {"id": 1164, "dom": "P", "ent": "brow", "cls": "salmon"}, {"id": 1165, "dom": "P", "ent": "brow", "cls": "shark"}, {"id": 1166, "dom": "P", "ent": "brow", "cls": "sheep"}, {"id": 1167, "dom": "P", "ent": "brow", "cls": "shook"}, {"id": 1168, "dom": "P", "ent": "brow", "cls": "tank"}, {"id": 1169, "dom": "P", "ent": "brow", "cls": "wheat"}, {"id": 1170, "dom": "P", "ent": "bulb", "cls": "architecture"}, {"id": 1171, "dom": "P", "ent": "bulb", "cls": "bacteria"}, {"id": 1172, "dom": "P", "ent": "bulb", "cls": "beef"}, {"id": 1173, "dom": "P", "ent": "bulb", "cls": "bridge"}, {"id": 1174, "dom": "P", "ent": "bulb", "cls": "bud"}, {"id": 1175, "dom": "P", "ent": "bulb", "cls": "cat"}, {"id": 1176, "dom": "P", "ent": "bulb", "cls": "chapel"}, {"id": 1177, "dom": "P", "ent": "bulb", "cls": "chick"}, {"id": 1178, "dom": "P", "ent": "bulb", "cls": "clock"}, {"id": 1179, "dom": "P", "ent": "bulb", "cls": "deer"}, {"id": 1180, "dom": "P", "ent": "bulb", "cls": "dog"}, {"id": 1181, "dom": "P", "ent": "bulb", "cls": "elephant"}, {"id": 1182, "dom": "P", "ent": "bulb", "cls": "embassy"}, {"id": 1183, "dom": "P", "ent": "bulb", "cls": "garage"}, {"id": 1184, "dom": "P", "ent": "bulb", "cls": "highway"}, {"id": 1185, "dom": "P", "ent": "bulb", "cls": "kidney"}, {"id": 1186, "dom": "P", "ent": "bulb", "cls": "lamb"}, {"id": 1187, "dom": "P", "ent": "bulb", "cls": "male"}, {"id": 1188, "dom": "P", "ent": "bulb", "cls": "marijuana"}, {"id": 1189, "dom": "P", "ent": "bulb", "cls": "puppy"}, {"id": 1190, "dom": "P", "ent": "bulb", "cls": "receiver"}, {"id": 1191, "dom": "P", "ent": "bulb", "cls": "reed"}, {"id": 1192, "dom": "P", "ent": "bulb", "cls": "rifle"}, {"id": 1193, "dom": "P", "ent": "bulb", "cls": "road"}, {"id": 1194, "dom": "P", "ent": "bulb", "cls": "salmon"}, {"id": 1195, "dom": "P", "ent": "bulb", "cls": "shark"}, {"id": 1196, "dom": "P", "ent": "bulb", "cls": "sheep"}, {"id": 1197, "dom": "P", "ent": "bulb", "cls": "shook"}, {"id": 1198, "dom": "P", "ent": "bulb", "cls": "tank"}, {"id": 1199, "dom": "P", "ent": "bulb", "cls": "wheat"}, {"id": 1200, "dom": "P", "ent": "bullet", "cls": "architecture"}, {"id": 1201, "dom": "P", "ent": "bullet", "cls": "bacteria"}, {"id": 1202, "dom": "P", "ent": "bullet", "cls": "beef"}, {"id": 1203, "dom": "P", "ent": "bullet", "cls": "bridge"}, {"id": 1204, "dom": "P", "ent": "bullet", "cls": "bud"}, {"id": 1205, "dom": "P", "ent": "bullet", "cls": "cat"}, {"id": 1206, "dom": "P", "ent": "bullet", "cls": "chapel"}, {"id": 1207, "dom": "P", "ent": "bullet", "cls": "chick"}, {"id": 1208, "dom": "P", "ent": "bullet", "cls": "clock"}, {"id": 1209, "dom": "P", "ent": "bullet", "cls": "deer"}, {"id": 1210, "dom": "P", "ent": "bullet", "cls": "dog"}, {"id": 1211, "dom": "P", "ent": "bullet", "cls": "elephant"}, {"id": 1212, "dom": "P", "ent": "bullet", "cls": "embassy"}, {"id": 1213, "dom": "P", "ent": "bullet", "cls": "garage"}, {"id": 1214, "dom": "P", "ent": "bullet", "cls": "highway"}, {"id": 1215, "dom": "P", "ent": "bullet", "cls": "kidney"}, {"id": 1216, "dom": "P", "ent": "bullet", "cls": "lamb"}, {"id": 1217, "dom": "P", "ent": "bullet", "cls": "male"}, {"id": 1218, "dom": "P", "ent": "bullet", "cls": "marijuana"}, {"id": 1219, "dom": "P", "ent": "bullet", "cls": "puppy"}, {"id": 1220, "dom": "P", "ent": "bullet", "cls": "receiver"}, {"id": 1221, "dom": "P", "ent": "bullet", "cls": "reed"}, {"id": 1222, "dom": "P", "ent": "bullet", "cls": "rifle"}, {"id": 1223, "dom": "P", "ent": "bullet", "cls": "road"}, {"id": 1224, "dom": "P", "ent": "bullet", "cls": "salmon"}, {"id": 1225, "dom": "P", "ent": "bullet", "cls": "shark"}, {"id": 1226, "dom": "P", "ent": "bullet", "cls": "sheep"}, {"id": 1227, "dom": "P", "ent": "bullet", "cls": "shook"}, {"id": 1228, "dom": "P", "ent": "bullet", "cls": "tank"}, {"id": 1229, "dom": "P", "ent": "bullet", "cls": "wheat"}, {"id": 1230, "dom": "P", "ent": "burger", "cls": "architecture"}, {"id": 1231, "dom": "P", "ent": "burger", "cls": "bacteria"}, {"id": 1232, "dom": "P", "ent": "burger", "cls": "beef"}, {"id": 1233, "dom": "P", "ent": "burger", "cls": "bridge"}, {"id": 1234, "dom": "P", "ent": "burger", "cls": "bud"}, {"id": 1235, "dom": "P", "ent": "burger", "cls": "cat"}, {"id": 1236, "dom": "P", "ent": "burger", "cls": "chapel"}, {"id": 1237, "dom": "P", "ent": "burger", "cls": "chick"}, {"id": 1238, "dom": "P", "ent": "burger", "cls": "clock"}, {"id": 1239, "dom": "P", "ent": "burger", "cls": "deer"}, {"id": 1240, "dom": "P", "ent": "burger", "cls": "dog"}, {"id": 1241, "dom": "P", "ent": "burger", "cls": "elephant"}, {"id": 1242, "dom": "P", "ent": "burger", "cls": "embassy"}, {"id": 1243, "dom": "P", "ent": "burger", "cls": "garage"}, {"id": 1244, "dom": "P", "ent": "burger", "cls": "highway"}, {"id": 1245, "dom": "P", "ent": "burger", "cls": "kidney"}, {"id": 1246, "dom": "P", "ent": "burger", "cls": "lamb"}, {"id": 1247, "dom": "P", "ent": "burger", "cls": "male"}, {"id": 1248, "dom": "P", "ent": "burger", "cls": "marijuana"}, {"id": 1249, "dom": "P", "ent": "burger", "cls": "puppy"}, {"id": 1250, "dom": "P", "ent": "burger", "cls": "receiver"}, {"id": 1251, "dom": "P", "ent": "burger", "cls": "reed"}, {"id": 1252, "dom": "P", "ent": "burger", "cls": "rifle"}, {"id": 1253, "dom": "P", "ent": "burger", "cls": "road"}, {"id": 1254, "dom": "P", "ent": "burger", "cls": "salmon"}, {"id": 1255, "dom": "P", "ent": "burger", "cls": "shark"}, {"id": 1256, "dom": "P", "ent": "burger", "cls": "sheep"}, {"id": 1257, "dom": "P", "ent": "burger", "cls": "shook"}, {"id": 1258, "dom": "P", "ent": "burger", "cls": "tank"}, {"id": 1259, "dom": "P", "ent": "burger", "cls": "wheat"}, {"id": 1260, "dom": "P", "ent": "bypass", "cls": "architecture"}, {"id": 1261, "dom": "P", "ent": "bypass", "cls": "bacteria"}, {"id": 1262, "dom": "P", "ent": "bypass", "cls": "beef"}, {"id": 1263, "dom": "P", "ent": "bypass", "cls": "bridge"}, {"id": 1264, "dom": "P", "ent": "bypass", "cls": "bud"}, {"id": 1265, "dom": "P", "ent": "bypass", "cls": "cat"}, {"id": 1266, "dom": "P", "ent": "bypass", "cls": "chapel"}, {"id": 1267, "dom": "P", "ent": "bypass", "cls": "chick"}, {"id": 1268, "dom": "P", "ent": "bypass", "cls": "clock"}, {"id": 1269, "dom": "P", "ent": "bypass", "cls": "deer"}, {"id": 1270, "dom": "P", "ent": "bypass", "cls": "dog"}, {"id": 1271, "dom": "P", "ent": "bypass", "cls": "elephant"}, {"id": 1272, "dom": "P", "ent": "bypass", "cls": "embassy"}, {"id": 1273, "dom": "P", "ent": "bypass", "cls": "garage"}, {"id": 1274, "dom": "P", "ent": "bypass", "cls": "highway"}, {"id": 1275, "dom": "P", "ent": "bypass", "cls": "kidney"}, {"id": 1276, "dom": "P", "ent": "bypass", "cls": "lamb"}, {"id": 1277, "dom": "P", "ent": "bypass", "cls": "male"}, {"id": 1278, "dom": "P", "ent": "bypass", "cls": "marijuana"}, {"id": 1279, "dom": "P", "ent": "bypass", "cls": "puppy"}, {"id": 1280, "dom": "P", "ent": "bypass", "cls": "receiver"}, {"id": 1281, "dom": "P", "ent": "bypass", "cls": "reed"}, {"id": 1282, "dom": "P", "ent": "bypass", "cls": "rifle"}, {"id": 1283, "dom": "P", "ent": "bypass", "cls": "road"}, {"id": 1284, "dom": "P", "ent": "bypass", "cls": "salmon"}, {"id": 1285, "dom": "P", "ent": "bypass", "cls": "shark"}, {"id": 1286, "dom": "P", "ent": "bypass", "cls": "sheep"}, {"id": 1287, "dom": "P", "ent": "bypass", "cls": "shook"}, {"id": 1288, "dom": "P", "ent": "bypass", "cls": "tank"}, {"id": 1289, "dom": "P", "ent": "bypass", "cls": "wheat"}, {"id": 1290, "dom": "P", "ent": "canal", "cls": "architecture"}, {"id": 1291, "dom": "P", "ent": "canal", "cls": "bacteria"}, {"id": 1292, "dom": "P", "ent": "canal", "cls": "beef"}, {"id": 1293, "dom": "P", "ent": "canal", "cls": "bridge"}, {"id": 1294, "dom": "P", "ent": "canal", "cls": "bud"}, {"id": 1295, "dom": "P", "ent": "canal", "cls": "cat"}, {"id": 1296, "dom": "P", "ent": "canal", "cls": "chapel"}, {"id": 1297, "dom": "P", "ent": "canal", "cls": "chick"}, {"id": 1298, "dom": "P", "ent": "canal", "cls": "clock"}, {"id": 1299, "dom": "P", "ent": "canal", "cls": "deer"}, {"id": 1300, "dom": "P", "ent": "canal", "cls": "dog"}, {"id": 1301, "dom": "P", "ent": "canal", "cls": "elephant"}, {"id": 1302, "dom": "P", "ent": "canal", "cls": "embassy"}, {"id": 1303, "dom": "P", "ent": "canal", "cls": "garage"}, {"id": 1304, "dom": "P", "ent": "canal", "cls": "highway"}, {"id": 1305, "dom": "P", "ent": "canal", "cls": "kidney"}, {"id": 1306, "dom": "P", "ent": "canal", "cls": "lamb"}, {"id": 1307, "dom": "P", "ent": "canal", "cls": "male"}, {"id": 1308, "dom": "P", "ent": "canal", "cls": "marijuana"}, {"id": 1309, "dom": "P", "ent": "canal", "cls": "puppy"}, {"id": 1310, "dom": "P", "ent": "canal", "cls": "receiver"}, {"id": 1311, "dom": "P", "ent": "canal", "cls": "reed"}, {"id": 1312, "dom": "P", "ent": "canal", "cls": "rifle"}, {"id": 1313, "dom": "P", "ent": "canal", "cls": "road"}, {"id": 1314, "dom": "P", "ent": "canal", "cls": "salmon"}, {"id": 1315, "dom": "P", "ent": "canal", "cls": "shark"}, {"id": 1316, "dom": "P", "ent": "canal", "cls": "sheep"}, {"id": 1317, "dom": "P", "ent": "canal", "cls": "shook"}, {"id": 1318, "dom": "P", "ent": "canal", "cls": "tank"}, {"id": 1319, "dom": "P", "ent": "canal", "cls": "wheat"}, {"id": 1320, "dom": "P", "ent": "cannabis", "cls": "architecture"}, {"id": 1321, "dom": "P", "ent": "cannabis", "cls": "bacteria"}, {"id": 1322, "dom": "P", "ent": "cannabis", "cls": "beef"}, {"id": 1323, "dom": "P", "ent": "cannabis", "cls": "bridge"}, {"id": 1324, "dom": "P", "ent": "cannabis", "cls": "bud"}, {"id": 1325, "dom": "P", "ent": "cannabis", "cls": "cat"}, {"id": 1326, "dom": "P", "ent": "cannabis", "cls": "chapel"}, {"id": 1327, "dom": "P", "ent": "cannabis", "cls": "chick"}, {"id": 1328, "dom": "P", "ent": "cannabis", "cls": "clock"}, {"id": 1329, "dom": "P", "ent": "cannabis", "cls": "deer"}, {"id": 1330, "dom": "P", "ent": "cannabis", "cls": "dog"}, {"id": 1331, "dom": "P", "ent": "cannabis", "cls": "elephant"}, {"id": 1332, "dom": "P", "ent": "cannabis", "cls": "embassy"}, {"id": 1333, "dom": "P", "ent": "cannabis", "cls": "garage"}, {"id": 1334, "dom": "P", "ent": "cannabis", "cls": "highway"}, {"id": 1335, "dom": "P", "ent": "cannabis", "cls": "kidney"}, {"id": 1336, "dom": "P", "ent": "cannabis", "cls": "lamb"}, {"id": 1337, "dom": "P", "ent": "cannabis", "cls": "male"}, {"id": 1338, "dom": "P", "ent": "cannabis", "cls": "marijuana"}, {"id": 1339, "dom": "P", "ent": "cannabis", "cls": "puppy"}, {"id": 1340, "dom": "P", "ent": "cannabis", "cls": "receiver"}, {"id": 1341, "dom": "P", "ent": "cannabis", "cls": "reed"}, {"id": 1342, "dom": "P", "ent": "cannabis", "cls": "rifle"}, {"id": 1343, "dom": "P", "ent": "cannabis", "cls": "road"}, {"id": 1344, "dom": "P", "ent": "cannabis", "cls": "salmon"}, {"id": 1345, "dom": "P", "ent": "cannabis", "cls": "shark"}, {"id": 1346, "dom": "P", "ent": "cannabis", "cls": "sheep"}, {"id": 1347, "dom": "P", "ent": "cannabis", "cls": "shook"}, {"id": 1348, "dom": "P", "ent": "cannabis", "cls": "tank"}, {"id": 1349, "dom": "P", "ent": "cannabis", "cls": "wheat"}, {"id": 1350, "dom": "P", "ent": "cannon", "cls": "architecture"}, {"id": 1351, "dom": "P", "ent": "cannon", "cls": "bacteria"}, {"id": 1352, "dom": "P", "ent": "cannon", "cls": "beef"}, {"id": 1353, "dom": "P", "ent": "cannon", "cls": "bridge"}, {"id": 1354, "dom": "P", "ent": "cannon", "cls": "bud"}, {"id": 1355, "dom": "P", "ent": "cannon", "cls": "cat"}, {"id": 1356, "dom": "P", "ent": "cannon", "cls": "chapel"}, {"id": 1357, "dom": "P", "ent": "cannon", "cls": "chick"}, {"id": 1358, "dom": "P", "ent": "cannon", "cls": "clock"}, {"id": 1359, "dom": "P", "ent": "cannon", "cls": "deer"}, {"id": 1360, "dom": "P", "ent": "cannon", "cls": "dog"}, {"id": 1361, "dom": "P", "ent": "cannon", "cls": "elephant"}, {"id": 1362, "dom": "P", "ent": "cannon", "cls": "embassy"}, {"id": 1363, "dom": "P", "ent": "cannon", "cls": "garage"}, {"id": 1364, "dom": "P", "ent": "cannon", "cls": "highway"}, {"id": 1365, "dom": "P", "ent": "cannon", "cls": "kidney"}, {"id": 1366, "dom": "P", "ent": "cannon", "cls": "lamb"}, {"id": 1367, "dom": "P", "ent": "cannon", "cls": "male"}, {"id": 1368, "dom": "P", "ent": "cannon", "cls": "marijuana"}, {"id": 1369, "dom": "P", "ent": "cannon", "cls": "puppy"}, {"id": 1370, "dom": "P", "ent": "cannon", "cls": "receiver"}, {"id": 1371, "dom": "P", "ent": "cannon", "cls": "reed"}, {"id": 1372, "dom": "P", "ent": "cannon", "cls": "rifle"}, {"id": 1373, "dom": "P", "ent": "cannon", "cls": "road"}, {"id": 1374, "dom": "P", "ent": "cannon", "cls": "salmon"}, {"id": 1375, "dom": "P", "ent": "cannon", "cls": "shark"}, {"id": 1376, "dom": "P", "ent": "cannon", "cls": "sheep"}, {"id": 1377, "dom": "P", "ent": "cannon", "cls": "shook"}, {"id": 1378, "dom": "P", "ent": "cannon", "cls": "tank"}, {"id": 1379, "dom": "P", "ent": "cannon", "cls": "wheat"}, {"id": 1380, "dom": "P", "ent": "cap", "cls": "architecture"}, {"id": 1381, "dom": "P", "ent": "cap", "cls": "bacteria"}, {"id": 1382, "dom": "P", "ent": "cap", "cls": "beef"}, {"id": 1383, "dom": "P", "ent": "cap", "cls": "bridge"}, {"id": 1384, "dom": "P", "ent": "cap", "cls": "bud"}, {"id": 1385, "dom": "P", "ent": "cap", "cls": "cat"}, {"id": 1386, "dom": "P", "ent": "cap", "cls": "chapel"}, {"id": 1387, "dom": "P", "ent": "cap", "cls": "chick"}, {"id": 1388, "dom": "P", "ent": "cap", "cls": "clock"}, {"id": 1389, "dom": "P", "ent": "cap", "cls": "deer"}, {"id": 1390, "dom": "P", "ent": "cap", "cls": "dog"}, {"id": 1391, "dom": "P", "ent": "cap", "cls": "elephant"}, {"id": 1392, "dom": "P", "ent": "cap", "cls": "embassy"}, {"id": 1393, "dom": "P", "ent": "cap", "cls": "garage"}, {"id": 1394, "dom": "P", "ent": "cap", "cls": "highway"}, {"id": 1395, "dom": "P", "ent": "cap", "cls": "kidney"}, {"id": 1396, "dom": "P", "ent": "cap", "cls": "lamb"}, {"id": 1397, "dom": "P", "ent": "cap", "cls": "male"}, {"id": 1398, "dom": "P", "ent": "cap", "cls": "marijuana"}, {"id": 1399, "dom": "P", "ent": "cap", "cls": "puppy"}, {"id": 1400, "dom": "P", "ent": "cap", "cls": "receiver"}, {"id": 1401, "dom": "P", "ent": "cap", "cls": "reed"}, {"id": 1402, "dom": "P", "ent": "cap", "cls": "rifle"}, {"id": 1403, "dom": "P", "ent": "cap", "cls": "road"}, {"id": 1404, "dom": "P", "ent": "cap", "cls": "salmon"}, {"id": 1405, "dom": "P", "ent": "cap", "cls": "shark"}, {"id": 1406, "dom": "P", "ent": "cap", "cls": "sheep"}, {"id": 1407, "dom": "P", "ent": "cap", "cls": "shook"}, {"id": 1408, "dom": "P", "ent": "cap", "cls": "tank"}, {"id": 1409, "dom": "P", "ent": "cap", "cls": "wheat"}, {"id": 1410, "dom": "P", "ent": "capacitor", "cls": "architecture"}, {"id": 1411, "dom": "P", "ent": "capacitor", "cls": "bacteria"}, {"id": 1412, "dom": "P", "ent": "capacitor", "cls": "beef"}, {"id": 1413, "dom": "P", "ent": "capacitor", "cls": "bridge"}, {"id": 1414, "dom": "P", "ent": "capacitor", "cls": "bud"}, {"id": 1415, "dom": "P", "ent": "capacitor", "cls": "cat"}, {"id": 1416, "dom": "P", "ent": "capacitor", "cls": "chapel"}, {"id": 1417, "dom": "P", "ent": "capacitor", "cls": "chick"}, {"id": 1418, "dom": "P", "ent": "capacitor", "cls": "clock"}, {"id": 1419, "dom": "P", "ent": "capacitor", "cls": "deer"}, {"id": 1420, "dom": "P", "ent": "capacitor", "cls": "dog"}, {"id": 1421, "dom": "P", "ent": "capacitor", "cls": "elephant"}, {"id": 1422, "dom": "P", "ent": "capacitor", "cls": "embassy"}, {"id": 1423, "dom": "P", "ent": "capacitor", "cls": "garage"}, {"id": 1424, "dom": "P", "ent": "capacitor", "cls": "highway"}, {"id": 1425, "dom": "P", "ent": "capacitor", "cls": "kidney"}, {"id": 1426, "dom": "P", "ent": "capacitor", "cls": "lamb"}, {"id": 1427, "dom": "P", "ent": "capacitor", "cls": "male"}, {"id": 1428, "dom": "P", "ent": "capacitor", "cls": "marijuana"}, {"id": 1429, "dom": "P", "ent": "capacitor", "cls": "puppy"}, {"id": 1430, "dom": "P", "ent": "capacitor", "cls": "receiver"}, {"id": 1431, "dom": "P", "ent": "capacitor", "cls": "reed"}, {"id": 1432, "dom": "P", "ent": "capacitor", "cls": "rifle"}, {"id": 1433, "dom": "P", "ent": "capacitor", "cls": "road"}, {"id": 1434, "dom": "P", "ent": "capacitor", "cls": "salmon"}, {"id": 1435, "dom": "P", "ent": "capacitor", "cls": "shark"}, {"id": 1436, "dom": "P", "ent": "capacitor", "cls": "sheep"}, {"id": 1437, "dom": "P", "ent": "capacitor", "cls": "shook"}, {"id": 1438, "dom": "P", "ent": "capacitor", "cls": "tank"}, {"id": 1439, "dom": "P", "ent": "capacitor", "cls": "wheat"}, {"id": 1440, "dom": "P", "ent": "capital", "cls": "architecture"}, {"id": 1441, "dom": "P", "ent": "capital", "cls": "bacteria"}, {"id": 1442, "dom": "P", "ent": "capital", "cls": "beef"}, {"id": 1443, "dom": "P", "ent": "capital", "cls": "bridge"}, {"id": 1444, "dom": "P", "ent": "capital", "cls": "bud"}, {"id": 1445, "dom": "P", "ent": "capital", "cls": "cat"}, {"id": 1446, "dom": "P", "ent": "capital", "cls": "chapel"}, {"id": 1447, "dom": "P", "ent": "capital", "cls": "chick"}, {"id": 1448, "dom": "P", "ent": "capital", "cls": "clock"}, {"id": 1449, "dom": "P", "ent": "capital", "cls": "deer"}, {"id": 1450, "dom": "P", "ent": "capital", "cls": "dog"}, {"id": 1451, "dom": "P", "ent": "capital", "cls": "elephant"}, {"id": 1452, "dom": "P", "ent": "capital", "cls": "embassy"}, {"id": 1453, "dom": "P", "ent": "capital", "cls": "garage"}, {"id": 1454, "dom": "P", "ent": "capital", "cls": "highway"}, {"id": 1455, "dom": "P", "ent": "capital", "cls": "kidney"}, {"id": 1456, "dom": "P", "ent": "capital", "cls": "lamb"}, {"id": 1457, "dom": "P", "ent": "capital", "cls": "male"}, {"id": 1458, "dom": "P", "ent": "capital", "cls": "marijuana"}, {"id": 1459, "dom": "P", "ent": "capital", "cls": "puppy"}, {"id": 1460, "dom": "P", "ent": "capital", "cls": "receiver"}, {"id": 1461, "dom": "P", "ent": "capital", "cls": "reed"}, {"id": 1462, "dom": "P", "ent": "capital", "cls": "rifle"}, {"id": 1463, "dom": "P", "ent": "capital", "cls": "road"}, {"id": 1464, "dom": "P", "ent": "capital", "cls": "salmon"}, {"id": 1465, "dom": "P", "ent": "capital", "cls": "shark"}, {"id": 1466, "dom": "P", "ent": "capital", "cls": "sheep"}, {"id": 1467, "dom": "P", "ent": "capital", "cls": "shook"}, {"id": 1468, "dom": "P", "ent": "capital", "cls": "tank"}, {"id": 1469, "dom": "P", "ent": "capital", "cls": "wheat"}, {"id": 1470, "dom": "P", "ent": "cardinal", "cls": "architecture"}, {"id": 1471, "dom": "P", "ent": "cardinal", "cls": "bacteria"}, {"id": 1472, "dom": "P", "ent": "cardinal", "cls": "beef"}, {"id": 1473, "dom": "P", "ent": "cardinal", "cls": "bridge"}, {"id": 1474, "dom": "P", "ent": "cardinal", "cls": "bud"}, {"id": 1475, "dom": "P", "ent": "cardinal", "cls": "cat"}, {"id": 1476, "dom": "P", "ent": "cardinal", "cls": "chapel"}, {"id": 1477, "dom": "P", "ent": "cardinal", "cls": "chick"}, {"id": 1478, "dom": "P", "ent": "cardinal", "cls": "clock"}, {"id": 1479, "dom": "P", "ent": "cardinal", "cls": "deer"}, {"id": 1480, "dom": "P", "ent": "cardinal", "cls": "dog"}, {"id": 1481, "dom": "P", "ent": "cardinal", "cls": "elephant"}, {"id": 1482, "dom": "P", "ent": "cardinal", "cls": "embassy"}, {"id": 1483, "dom": "P", "ent": "cardinal", "cls": "garage"}, {"id": 1484, "dom": "P", "ent": "cardinal", "cls": "highway"}, {"id": 1485, "dom": "P", "ent": "cardinal", "cls": "kidney"}, {"id": 1486, "dom": "P", "ent": "cardinal", "cls": "lamb"}, {"id": 1487, "dom": "P", "ent": "cardinal", "cls": "male"}, {"id": 1488, "dom": "P", "ent": "cardinal", "cls": "marijuana"}, {"id": 1489, "dom": "P", "ent": "cardinal", "cls": "puppy"}, {"id": 1490, "dom": "P", "ent": "cardinal", "cls": "receiver"}, {"id": 1491, "dom": "P", "ent": "cardinal", "cls": "reed"}, {"id": 1492, "dom": "P", "ent": "cardinal", "cls": "rifle"}, {"id": 1493, "dom": "P", "ent": "cardinal", "cls": "road"}, {"id": 1494, "dom": "P", "ent": "cardinal", "cls": "salmon"}, {"id": 1495, "dom": "P", "ent": "cardinal", "cls": "shark"}, {"id": 1496, "dom": "P", "ent": "cardinal", "cls": "sheep"}, {"id": 1497, "dom": "P", "ent": "cardinal", "cls": "shook"}, {"id": 1498, "dom": "P", "ent": "cardinal", "cls": "tank"}, {"id": 1499, "dom": "P", "ent": "cardinal", "cls": "wheat"}, {"id": 1500, "dom": "P", "ent": "carpet", "cls": "architecture"}, {"id": 1501, "dom": "P", "ent": "carpet", "cls": "bacteria"}, {"id": 1502, "dom": "P", "ent": "carpet", "cls": "beef"}, {"id": 1503, "dom": "P", "ent": "carpet", "cls": "bridge"}, {"id": 1504, "dom": "P", "ent": "carpet", "cls": "bud"}, {"id": 1505, "dom": "P", "ent": "carpet", "cls": "cat"}, {"id": 1506, "dom": "P", "ent": "carpet", "cls": "chapel"}, {"id": 1507, "dom": "P", "ent": "carpet", "cls": "chick"}, {"id": 1508, "dom": "P", "ent": "carpet", "cls": "clock"}, {"id": 1509, "dom": "P", "ent": "carpet", "cls": "deer"}, {"id": 1510, "dom": "P", "ent": "carpet", "cls": "dog"}, {"id": 1511, "dom": "P", "ent": "carpet", "cls": "elephant"}, {"id": 1512, "dom": "P", "ent": "carpet", "cls": "embassy"}, {"id": 1513, "dom": "P", "ent": "carpet", "cls": "garage"}, {"id": 1514, "dom": "P", "ent": "carpet", "cls": "highway"}, {"id": 1515, "dom": "P", "ent": "carpet", "cls": "kidney"}, {"id": 1516, "dom": "P", "ent": "carpet", "cls": "lamb"}, {"id": 1517, "dom": "P", "ent": "carpet", "cls": "male"}, {"id": 1518, "dom": "P", "ent": "carpet", "cls": "marijuana"}, {"id": 1519, "dom": "P", "ent": "carpet", "cls": "puppy"}, {"id": 1520, "dom": "P", "ent": "carpet", "cls": "receiver"}, {"id": 1521, "dom": "P", "ent": "carpet", "cls": "reed"}, {"id": 1522, "dom": "P", "ent": "carpet", "cls": "rifle"}, {"id": 1523, "dom": "P", "ent": "carpet", "cls": "road"}, {"id": 1524, "dom": "P", "ent": "carpet", "cls": "salmon"}, {"id": 1525, "dom": "P", "ent": "carpet", "cls": "shark"}, {"id": 1526, "dom": "P", "ent": "carpet", "cls": "sheep"}, {"id": 1527, "dom": "P", "ent": "carpet", "cls": "shook"}, {"id": 1528, "dom": "P", "ent": "carpet", "cls": "tank"}, {"id": 1529, "dom": "P", "ent": "carpet", "cls": "wheat"}, {"id": 1530, "dom": "P", "ent": "carving", "cls": "architecture"}, {"id": 1531, "dom": "P", "ent": "carving", "cls": "bacteria"}, {"id": 1532, "dom": "P", "ent": "carving", "cls": "beef"}, {"id": 1533, "dom": "P", "ent": "carving", "cls": "bridge"}, {"id": 1534, "dom": "P", "ent": "carving", "cls": "bud"}, {"id": 1535, "dom": "P", "ent": "carving", "cls": "cat"}, {"id": 1536, "dom": "P", "ent": "carving", "cls": "chapel"}, {"id": 1537, "dom": "P", "ent": "carving", "cls": "chick"}, {"id": 1538, "dom": "P", "ent": "carving", "cls": "clock"}, {"id": 1539, "dom": "P", "ent": "carving", "cls": "deer"}, {"id": 1540, "dom": "P", "ent": "carving", "cls": "dog"}, {"id": 1541, "dom": "P", "ent": "carving", "cls": "elephant"}, {"id": 1542, "dom": "P", "ent": "carving", "cls": "embassy"}, {"id": 1543, "dom": "P", "ent": "carving", "cls": "garage"}, {"id": 1544, "dom": "P", "ent": "carving", "cls": "highway"}, {"id": 1545, "dom": "P", "ent": "carving", "cls": "kidney"}, {"id": 1546, "dom": "P", "ent": "carving", "cls": "lamb"}, {"id": 1547, "dom": "P", "ent": "carving", "cls": "male"}, {"id": 1548, "dom": "P", "ent": "carving", "cls": "marijuana"}, {"id": 1549, "dom": "P", "ent": "carving", "cls": "puppy"}, {"id": 1550, "dom": "P", "ent": "carving", "cls": "receiver"}, {"id": 1551, "dom": "P", "ent": "carving", "cls": "reed"}, {"id": 1552, "dom": "P", "ent": "carving", "cls": "rifle"}, {"id": 1553, "dom": "P", "ent": "carving", "cls": "road"}, {"id": 1554, "dom": "P", "ent": "carving", "cls": "salmon"}, {"id": 1555, "dom": "P", "ent": "carving", "cls": "shark"}, {"id": 1556, "dom": "P", "ent": "carving", "cls": "sheep"}, {"id": 1557, "dom": "P", "ent": "carving", "cls": "shook"}, {"id": 1558, "dom": "P", "ent": "carving", "cls": "tank"}, {"id": 1559, "dom": "P", "ent": "carving", "cls": "wheat"}, {"id": 1560, "dom": "P", "ent": "case", "cls": "architecture"}, {"id": 1561, "dom": "P", "ent": "case", "cls": "bacteria"}, {"id": 1562, "dom": "P", "ent": "case", "cls": "beef"}, {"id": 1563, "dom": "P", "ent": "case", "cls": "bridge"}, {"id": 1564, "dom": "P", "ent": "case", "cls": "bud"}, {"id": 1565, "dom": "P", "ent": "case", "cls": "cat"}, {"id": 1566, "dom": "P", "ent": "case", "cls": "chapel"}, {"id": 1567, "dom": "P", "ent": "case", "cls": "chick"}, {"id": 1568, "dom": "P", "ent": "case", "cls": "clock"}, {"id": 1569, "dom": "P", "ent": "case", "cls": "deer"}, {"id": 1570, "dom": "P", "ent": "case", "cls": "dog"}, {"id": 1571, "dom": "P", "ent": "case", "cls": "elephant"}, {"id": 1572, "dom": "P", "ent": "case", "cls": "embassy"}, {"id": 1573, "dom": "P", "ent": "case", "cls": "garage"}, {"id": 1574, "dom": "P", "ent": "case", "cls": "highway"}, {"id": 1575, "dom": "P", "ent": "case", "cls": "kidney"}, {"id": 1576, "dom": "P", "ent": "case", "cls": "lamb"}, {"id": 1577, "dom": "P", "ent": "case", "cls": "male"}, {"id": 1578, "dom": "P", "ent": "case", "cls": "marijuana"}, {"id": 1579, "dom": "P", "ent": "case", "cls": "puppy"}, {"id": 1580, "dom": "P", "ent": "case", "cls": "receiver"}, {"id": 1581, "dom": "P", "ent": "case", "cls": "reed"}, {"id": 1582, "dom": "P", "ent": "case", "cls": "rifle"}, {"id": 1583, "dom": "P", "ent": "case", "cls": "road"}, {"id": 1584, "dom": "P", "ent": "case", "cls": "salmon"}, {"id": 1585, "dom": "P", "ent": "case", "cls": "shark"}, {"id": 1586, "dom": "P", "ent": "case", "cls": "sheep"}, {"id": 1587, "dom": "P", "ent": "case", "cls": "shook"}, {"id": 1588, "dom": "P", "ent": "case", "cls": "tank"}, {"id": 1589, "dom": "P", "ent": "case", "cls": "wheat"}, {"id": 1590, "dom": "P", "ent": "casing", "cls": "architecture"}, {"id": 1591, "dom": "P", "ent": "casing", "cls": "bacteria"}, {"id": 1592, "dom": "P", "ent": "casing", "cls": "beef"}, {"id": 1593, "dom": "P", "ent": "casing", "cls": "bridge"}, {"id": 1594, "dom": "P", "ent": "casing", "cls": "bud"}, {"id": 1595, "dom": "P", "ent": "casing", "cls": "cat"}, {"id": 1596, "dom": "P", "ent": "casing", "cls": "chapel"}, {"id": 1597, "dom": "P", "ent": "casing", "cls": "chick"}, {"id": 1598, "dom": "P", "ent": "casing", "cls": "clock"}, {"id": 1599, "dom": "P", "ent": "casing", "cls": "deer"}, {"id": 1600, "dom": "P", "ent": "casing", "cls": "dog"}, {"id": 1601, "dom": "P", "ent": "casing", "cls": "elephant"}, {"id": 1602, "dom": "P", "ent": "casing", "cls": "embassy"}, {"id": 1603, "dom": "P", "ent": "casing", "cls": "garage"}, {"id": 1604, "dom": "P", "ent": "casing", "cls": "highway"}, {"id": 1605, "dom": "P", "ent": "casing", "cls": "kidney"}, {"id": 1606, "dom": "P", "ent": "casing", "cls": "lamb"}, {"id": 1607, "dom": "P", "ent": "casing", "cls": "male"}, {"id": 1608, "dom": "P", "ent": "casing", "cls": "marijuana"}, {"id": 1609, "dom": "P", "ent": "casing", "cls": "puppy"}, {"id": 1610, "dom": "P", "ent": "casing", "cls": "receiver"}, {"id": 1611, "dom": "P", "ent": "casing", "cls": "reed"}, {"id": 1612, "dom": "P", "ent": "casing", "cls": "rifle"}, {"id": 1613, "dom": "P", "ent": "casing", "cls": "road"}, {"id": 1614, "dom": "P", "ent": "casing", "cls": "salmon"}, {"id": 1615, "dom": "P", "ent": "casing", "cls": "shark"}, {"id": 1616, "dom": "P", "ent": "casing", "cls": "sheep"}, {"id": 1617, "dom": "P", "ent": "casing", "cls": "shook"}, {"id": 1618, "dom": "P", "ent": "casing", "cls": "tank"}, {"id": 1619, "dom": "P", "ent": "casing", "cls": "wheat"}, {"id": 1620, "dom": "P", "ent": "cathode", "cls": "architecture"}, {"id": 1621, "dom": "P", "ent": "cathode", "cls": "bacteria"}, {"id": 1622, "dom": "P", "ent": "cathode", "cls": "beef"}, {"id": 1623, "dom": "P", "ent": "cathode", "cls": "bridge"}, {"id": 1624, "dom": "P", "ent": "cathode", "cls": "bud"}, {"id": 1625, "dom": "P", "ent": "cathode", "cls": "cat"}, {"id": 1626, "dom": "P", "ent": "cathode", "cls": "chapel"}, {"id": 1627, "dom": "P", "ent": "cathode", "cls": "chick"}, {"id": 1628, "dom": "P", "ent": "cathode", "cls": "clock"}, {"id": 1629, "dom": "P", "ent": "cathode", "cls": "deer"}, {"id": 1630, "dom": "P", "ent": "cathode", "cls": "dog"}, {"id": 1631, "dom": "P", "ent": "cathode", "cls": "elephant"}, {"id": 1632, "dom": "P", "ent": "cathode", "cls": "embassy"}, {"id": 1633, "dom": "P", "ent": "cathode", "cls": "garage"}, {"id": 1634, "dom": "P", "ent": "cathode", "cls": "highway"}, {"id": 1635, "dom": "P", "ent": "cathode", "cls": "kidney"}, {"id": 1636, "dom": "P", "ent": "cathode", "cls": "lamb"}, {"id": 1637, "dom": "P", "ent": "cathode", "cls": "male"}, {"id": 1638, "dom": "P", "ent": "cathode", "cls": "marijuana"}, {"id": 1639, "dom": "P", "ent": "cathode", "cls": "puppy"}, {"id": 1640, "dom": "P", "ent": "cathode", "cls": "receiver"}, {"id": 1641, "dom": "P", "ent": "cathode", "cls": "reed"}, {"id": 1642, "dom": "P", "ent": "cathode", "cls": "rifle"}, {"id": 1643, "dom": "P", "ent": "cathode", "cls": "road"}, {"id": 1644, "dom": "P", "ent": "cathode", "cls": "salmon"}, {"id": 1645, "dom": "P", "ent": "cathode", "cls": "shark"}, {"id": 1646, "dom": "P", "ent": "cathode", "cls": "sheep"}, {"id": 1647, "dom": "P", "ent": "cathode", "cls": "shook"}, {"id": 1648, "dom": "P", "ent": "cathode", "cls": "tank"}, {"id": 1649, "dom": "P", "ent": "cathode", "cls": "wheat"}, {"id": 1650, "dom": "P", "ent": "ceiling", "cls": "architecture"}, {"id": 1651, "dom": "P", "ent": "ceiling", "cls": "bacteria"}, {"id": 1652, "dom": "P", "ent": "ceiling", "cls": "beef"}, {"id": 1653, "dom": "P", "ent": "ceiling", "cls": "bridge"}, {"id": 1654, "dom": "P", "ent": "ceiling", "cls": "bud"}, {"id": 1655, "dom": "P", "ent": "ceiling", "cls": "cat"}, {"id": 1656, "dom": "P", "ent": "ceiling", "cls": "chapel"}, {"id": 1657, "dom": "P", "ent": "ceiling", "cls": "chick"}, {"id": 1658, "dom": "P", "ent": "ceiling", "cls": "clock"}, {"id": 1659, "dom": "P", "ent": "ceiling", "cls": "deer"}, {"id": 1660, "dom": "P", "ent": "ceiling", "cls": "dog"}, {"id": 1661, "dom": "P", "ent": "ceiling", "cls": "elephant"}, {"id": 1662, "dom": "P", "ent": "ceiling", "cls": "embassy"}, {"id": 1663, "dom": "P", "ent": "ceiling", "cls": "garage"}, {"id": 1664, "dom": "P", "ent": "ceiling", "cls": "highway"}, {"id": 1665, "dom": "P", "ent": "ceiling", "cls": "kidney"}, {"id": 1666, "dom": "P", "ent": "ceiling", "cls": "lamb"}, {"id": 1667, "dom": "P", "ent": "ceiling", "cls": "male"}, {"id": 1668, "dom": "P", "ent": "ceiling", "cls": "marijuana"}, {"id": 1669, "dom": "P", "ent": "ceiling", "cls": "puppy"}, {"id": 1670, "dom": "P", "ent": "ceiling", "cls": "receiver"}, {"id": 1671, "dom": "P", "ent": "ceiling", "cls": "reed"}, {"id": 1672, "dom": "P", "ent": "ceiling", "cls": "rifle"}, {"id": 1673, "dom": "P", "ent": "ceiling", "cls": "road"}, {"id": 1674, "dom": "P", "ent": "ceiling", "cls": "salmon"}, {"id": 1675, "dom": "P", "ent": "ceiling", "cls": "shark"}, {"id": 1676, "dom": "P", "ent": "ceiling", "cls": "sheep"}, {"id": 1677, "dom": "P", "ent": "ceiling", "cls": "shook"}, {"id": 1678, "dom": "P", "ent": "ceiling", "cls": "tank"}, {"id": 1679, "dom": "P", "ent": "ceiling", "cls": "wheat"}, {"id": 1680, "dom": "P", "ent": "chart", "cls": "architecture"}, {"id": 1681, "dom": "P", "ent": "chart", "cls": "bacteria"}, {"id": 1682, "dom": "P", "ent": "chart", "cls": "beef"}, {"id": 1683, "dom": "P", "ent": "chart", "cls": "bridge"}, {"id": 1684, "dom": "P", "ent": "chart", "cls": "bud"}, {"id": 1685, "dom": "P", "ent": "chart", "cls": "cat"}, {"id": 1686, "dom": "P", "ent": "chart", "cls": "chapel"}, {"id": 1687, "dom": "P", "ent": "chart", "cls": "chick"}, {"id": 1688, "dom": "P", "ent": "chart", "cls": "clock"}, {"id": 1689, "dom": "P", "ent": "chart", "cls": "deer"}, {"id": 1690, "dom": "P", "ent": "chart", "cls": "dog"}, {"id": 1691, "dom": "P", "ent": "chart", "cls": "elephant"}, {"id": 1692, "dom": "P", "ent": "chart", "cls": "embassy"}, {"id": 1693, "dom": "P", "ent": "chart", "cls": "garage"}, {"id": 1694, "dom": "P", "ent": "chart", "cls": "highway"}, {"id": 1695, "dom": "P", "ent": "chart", "cls": "kidney"}, {"id": 1696, "dom": "P", "ent": "chart", "cls": "lamb"}, {"id": 1697, "dom": "P", "ent": "chart", "cls": "male"}, {"id": 1698, "dom": "P", "ent": "chart", "cls": "marijuana"}, {"id": 1699, "dom": "P", "ent": "chart", "cls": "puppy"}, {"id": 1700, "dom": "P", "ent": "chart", "cls": "receiver"}, {"id": 1701, "dom": "P", "ent": "chart", "cls": "reed"}, {"id": 1702, "dom": "P", "ent": "chart", "cls": "rifle"}, {"id": 1703, "dom": "P", "ent": "chart", "cls": "road"}, {"id": 1704, "dom": "P", "ent": "chart", "cls": "salmon"}, {"id": 1705, "dom": "P", "ent": "chart", "cls": "shark"}, {"id": 1706, "dom": "P", "ent": "chart", "cls": "sheep"}, {"id": 1707, "dom": "P", "ent": "chart", "cls": "shook"}, {"id": 1708, "dom": "P", "ent": "chart", "cls": "tank"}, {"id": 1709, "dom": "P", "ent": "chart", "cls": "wheat"}, {"id": 1710, "dom": "P", "ent": "cheat", "cls": "architecture"}, {"id": 1711, "dom": "P", "ent": "cheat", "cls": "bacteria"}, {"id": 1712, "dom": "P", "ent": "cheat", "cls": "beef"}, {"id": 1713, "dom": "P", "ent": "cheat", "cls": "bridge"}, {"id": 1714, "dom": "P", "ent": "cheat", "cls": "bud"}, {"id": 1715, "dom": "P", "ent": "cheat", "cls": "cat"}, {"id": 1716, "dom": "P", "ent": "cheat", "cls": "chapel"}, {"id": 1717, "dom": "P", "ent": "cheat", "cls": "chick"}, {"id": 1718, "dom": "P", "ent": "cheat", "cls": "clock"}, {"id": 1719, "dom": "P", "ent": "cheat", "cls": "deer"}, {"id": 1720, "dom": "P", "ent": "cheat", "cls": "dog"}, {"id": 1721, "dom": "P", "ent": "cheat", "cls": "elephant"}, {"id": 1722, "dom": "P", "ent": "cheat", "cls": "embassy"}, {"id": 1723, "dom": "P", "ent": "cheat", "cls": "garage"}, {"id": 1724, "dom": "P", "ent": "cheat", "cls": "highway"}, {"id": 1725, "dom": "P", "ent": "cheat", "cls": "kidney"}, {"id": 1726, "dom": "P", "ent": "cheat", "cls": "lamb"}, {"id": 1727, "dom": "P", "ent": "cheat", "cls": "male"}, {"id": 1728, "dom": "P", "ent": "cheat", "cls": "marijuana"}, {"id": 1729, "dom": "P", "ent": "cheat", "cls": "puppy"}, {"id": 1730, "dom": "P", "ent": "cheat", "cls": "receiver"}, {"id": 1731, "dom": "P", "ent": "cheat", "cls": "reed"}, {"id": 1732, "dom": "P", "ent": "cheat", "cls": "rifle"}, {"id": 1733, "dom": "P", "ent": "cheat", "cls": "road"}, {"id": 1734, "dom": "P", "ent": "cheat", "cls": "salmon"}, {"id": 1735, "dom": "P", "ent": "cheat", "cls": "shark"}, {"id": 1736, "dom": "P", "ent": "cheat", "cls": "sheep"}, {"id": 1737, "dom": "P", "ent": "cheat", "cls": "shook"}, {"id": 1738, "dom": "P", "ent": "cheat", "cls": "tank"}, {"id": 1739, "dom": "P", "ent": "cheat", "cls": "wheat"}, {"id": 1740, "dom": "P", "ent": "cheek", "cls": "architecture"}, {"id": 1741, "dom": "P", "ent": "cheek", "cls": "bacteria"}, {"id": 1742, "dom": "P", "ent": "cheek", "cls": "beef"}, {"id": 1743, "dom": "P", "ent": "cheek", "cls": "bridge"}, {"id": 1744, "dom": "P", "ent": "cheek", "cls": "bud"}, {"id": 1745, "dom": "P", "ent": "cheek", "cls": "cat"}, {"id": 1746, "dom": "P", "ent": "cheek", "cls": "chapel"}, {"id": 1747, "dom": "P", "ent": "cheek", "cls": "chick"}, {"id": 1748, "dom": "P", "ent": "cheek", "cls": "clock"}, {"id": 1749, "dom": "P", "ent": "cheek", "cls": "deer"}, {"id": 1750, "dom": "P", "ent": "cheek", "cls": "dog"}, {"id": 1751, "dom": "P", "ent": "cheek", "cls": "elephant"}, {"id": 1752, "dom": "P", "ent": "cheek", "cls": "embassy"}, {"id": 1753, "dom": "P", "ent": "cheek", "cls": "garage"}, {"id": 1754, "dom": "P", "ent": "cheek", "cls": "highway"}, {"id": 1755, "dom": "P", "ent": "cheek", "cls": "kidney"}, {"id": 1756, "dom": "P", "ent": "cheek", "cls": "lamb"}, {"id": 1757, "dom": "P", "ent": "cheek", "cls": "male"}, {"id": 1758, "dom": "P", "ent": "cheek", "cls": "marijuana"}, {"id": 1759, "dom": "P", "ent": "cheek", "cls": "puppy"}, {"id": 1760, "dom": "P", "ent": "cheek", "cls": "receiver"}, {"id": 1761, "dom": "P", "ent": "cheek", "cls": "reed"}, {"id": 1762, "dom": "P", "ent": "cheek", "cls": "rifle"}, {"id": 1763, "dom": "P", "ent": "cheek", "cls": "road"}, {"id": 1764, "dom": "P", "ent": "cheek", "cls": "salmon"}, {"id": 1765, "dom": "P", "ent": "cheek", "cls": "shark"}, {"id": 1766, "dom": "P", "ent": "cheek", "cls": "sheep"}, {"id": 1767, "dom": "P", "ent": "cheek", "cls": "shook"}, {"id": 1768, "dom": "P", "ent": "cheek", "cls": "tank"}, {"id": 1769, "dom": "P", "ent": "cheek", "cls": "wheat"}, {"id": 1770, "dom": "P", "ent": "chest", "cls": "architecture"}, {"id": 1771, "dom": "P", "ent": "chest", "cls": "bacteria"}, {"id": 1772, "dom": "P", "ent": "chest", "cls": "beef"}, {"id": 1773, "dom": "P", "ent": "chest", "cls": "bridge"}, {"id": 1774, "dom": "P", "ent": "chest", "cls": "bud"}, {"id": 1775, "dom": "P", "ent": "chest", "cls": "cat"}, {"id": 1776, "dom": "P", "ent": "chest", "cls": "chapel"}, {"id": 1777, "dom": "P", "ent": "chest", "cls": "chick"}, {"id": 1778, "dom": "P", "ent": "chest", "cls": "clock"}, {"id": 1779, "dom": "P", "ent": "chest", "cls": "deer"}, {"id": 1780, "dom": "P", "ent": "chest", "cls": "dog"}, {"id": 1781, "dom": "P", "ent": "chest", "cls": "elephant"}, {"id": 1782, "dom": "P", "ent": "chest", "cls": "embassy"}, {"id": 1783, "dom": "P", "ent": "chest", "cls": "garage"}, {"id": 1784, "dom": "P", "ent": "chest", "cls": "highway"}, {"id": 1785, "dom": "P", "ent": "chest", "cls": "kidney"}, {"id": 1786, "dom": "P", "ent": "chest", "cls": "lamb"}, {"id": 1787, "dom": "P", "ent": "chest", "cls": "male"}, {"id": 1788, "dom": "P", "ent": "chest", "cls": "marijuana"}, {"id": 1789, "dom": "P", "ent": "chest", "cls": "puppy"}, {"id": 1790, "dom": "P", "ent": "chest", "cls": "receiver"}, {"id": 1791, "dom": "P", "ent": "chest", "cls": "reed"}, {"id": 1792, "dom": "P", "ent": "chest", "cls": "rifle"}, {"id": 1793, "dom": "P", "ent": "chest", "cls": "road"}, {"id": 1794, "dom": "P", "ent": "chest", "cls": "salmon"}, {"id": 1795, "dom": "P", "ent": "chest", "cls": "shark"}, {"id": 1796, "dom": "P", "ent": "chest", "cls": "sheep"}, {"id": 1797, "dom": "P", "ent": "chest", "cls": "shook"}, {"id": 1798, "dom": "P", "ent": "chest", "cls": "tank"}, {"id": 1799, "dom": "P", "ent": "chest", "cls": "wheat"}, {"id": 1800, "dom": "P", "ent": "chin", "cls": "architecture"}, {"id": 1801, "dom": "P", "ent": "chin", "cls": "bacteria"}, {"id": 1802, "dom": "P", "ent": "chin", "cls": "beef"}, {"id": 1803, "dom": "P", "ent": "chin", "cls": "bridge"}, {"id": 1804, "dom": "P", "ent": "chin", "cls": "bud"}, {"id": 1805, "dom": "P", "ent": "chin", "cls": "cat"}, {"id": 1806, "dom": "P", "ent": "chin", "cls": "chapel"}, {"id": 1807, "dom": "P", "ent": "chin", "cls": "chick"}, {"id": 1808, "dom": "P", "ent": "chin", "cls": "clock"}, {"id": 1809, "dom": "P", "ent": "chin", "cls": "deer"}, {"id": 1810, "dom": "P", "ent": "chin", "cls": "dog"}, {"id": 1811, "dom": "P", "ent": "chin", "cls": "elephant"}, {"id": 1812, "dom": "P", "ent": "chin", "cls": "embassy"}, {"id": 1813, "dom": "P", "ent": "chin", "cls": "garage"}, {"id": 1814, "dom": "P", "ent": "chin", "cls": "highway"}, {"id": 1815, "dom": "P", "ent": "chin", "cls": "kidney"}, {"id": 1816, "dom": "P", "ent": "chin", "cls": "lamb"}, {"id": 1817, "dom": "P", "ent": "chin", "cls": "male"}, {"id": 1818, "dom": "P", "ent": "chin", "cls": "marijuana"}, {"id": 1819, "dom": "P", "ent": "chin", "cls": "puppy"}, {"id": 1820, "dom": "P", "ent": "chin", "cls": "receiver"}, {"id": 1821, "dom": "P", "ent": "chin", "cls": "reed"}, {"id": 1822, "dom": "P", "ent": "chin", "cls": "rifle"}, {"id": 1823, "dom": "P", "ent": "chin", "cls": "road"}, {"id": 1824, "dom": "P", "ent": "chin", "cls": "salmon"}, {"id": 1825, "dom": "P", "ent": "chin", "cls": "shark"}, {"id": 1826, "dom": "P", "ent": "chin", "cls": "sheep"}, {"id": 1827, "dom": "P", "ent": "chin", "cls": "shook"}, {"id": 1828, "dom": "P", "ent": "chin", "cls": "tank"}, {"id": 1829, "dom": "P", "ent": "chin", "cls": "wheat"}, {"id": 1830, "dom": "P", "ent": "choke", "cls": "architecture"}, {"id": 1831, "dom": "P", "ent": "choke", "cls": "bacteria"}, {"id": 1832, "dom": "P", "ent": "choke", "cls": "beef"}, {"id": 1833, "dom": "P", "ent": "choke", "cls": "bridge"}, {"id": 1834, "dom": "P", "ent": "choke", "cls": "bud"}, {"id": 1835, "dom": "P", "ent": "choke", "cls": "cat"}, {"id": 1836, "dom": "P", "ent": "choke", "cls": "chapel"}, {"id": 1837, "dom": "P", "ent": "choke", "cls": "chick"}, {"id": 1838, "dom": "P", "ent": "choke", "cls": "clock"}, {"id": 1839, "dom": "P", "ent": "choke", "cls": "deer"}, {"id": 1840, "dom": "P", "ent": "choke", "cls": "dog"}, {"id": 1841, "dom": "P", "ent": "choke", "cls": "elephant"}, {"id": 1842, "dom": "P", "ent": "choke", "cls": "embassy"}, {"id": 1843, "dom": "P", "ent": "choke", "cls": "garage"}, {"id": 1844, "dom": "P", "ent": "choke", "cls": "highway"}, {"id": 1845, "dom": "P", "ent": "choke", "cls": "kidney"}, {"id": 1846, "dom": "P", "ent": "choke", "cls": "lamb"}, {"id": 1847, "dom": "P", "ent": "choke", "cls": "male"}, {"id": 1848, "dom": "P", "ent": "choke", "cls": "marijuana"}, {"id": 1849, "dom": "P", "ent": "choke", "cls": "puppy"}, {"id": 1850, "dom": "P", "ent": "choke", "cls": "receiver"}, {"id": 1851, "dom": "P", "ent": "choke", "cls": "reed"}, {"id": 1852, "dom": "P", "ent": "choke", "cls": "rifle"}, {"id": 1853, "dom": "P", "ent": "choke", "cls": "road"}, {"id": 1854, "dom": "P", "ent": "choke", "cls": "salmon"}, {"id": 1855, "dom": "P", "ent": "choke", "cls": "shark"}, {"id": 1856, "dom": "P", "ent": "choke", "cls": "sheep"}, {"id": 1857, "dom": "P", "ent": "choke", "cls": "shook"}, {"id": 1858, "dom": "P", "ent": "choke", "cls": "tank"}, {"id": 1859, "dom": "P", "ent": "choke", "cls": "wheat"}, {"id": 1860, "dom": "P", "ent": "chromosome", "cls": "architecture"}, {"id": 1861, "dom": "P", "ent": "chromosome", "cls": "bacteria"}, {"id": 1862, "dom": "P", "ent": "chromosome", "cls": "beef"}, {"id": 1863, "dom": "P", "ent": "chromosome", "cls": "bridge"}, {"id": 1864, "dom": "P", "ent": "chromosome", "cls": "bud"}, {"id": 1865, "dom": "P", "ent": "chromosome", "cls": "cat"}, {"id": 1866, "dom": "P", "ent": "chromosome", "cls": "chapel"}, {"id": 1867, "dom": "P", "ent": "chromosome", "cls": "chick"}, {"id": 1868, "dom": "P", "ent": "chromosome", "cls": "clock"}, {"id": 1869, "dom": "P", "ent": "chromosome", "cls": "deer"}, {"id": 1870, "dom": "P", "ent": "chromosome", "cls": "dog"}, {"id": 1871, "dom": "P", "ent": "chromosome", "cls": "elephant"}, {"id": 1872, "dom": "P", "ent": "chromosome", "cls": "embassy"}, {"id": 1873, "dom": "P", "ent": "chromosome", "cls": "garage"}, {"id": 1874, "dom": "P", "ent": "chromosome", "cls": "highway"}, {"id": 1875, "dom": "P", "ent": "chromosome", "cls": "kidney"}, {"id": 1876, "dom": "P", "ent": "chromosome", "cls": "lamb"}, {"id": 1877, "dom": "P", "ent": "chromosome", "cls": "male"}, {"id": 1878, "dom": "P", "ent": "chromosome", "cls": "marijuana"}, {"id": 1879, "dom": "P", "ent": "chromosome", "cls": "puppy"}, {"id": 1880, "dom": "P", "ent": "chromosome", "cls": "receiver"}, {"id": 1881, "dom": "P", "ent": "chromosome", "cls": "reed"}, {"id": 1882, "dom": "P", "ent": "chromosome", "cls": "rifle"}, {"id": 1883, "dom": "P", "ent": "chromosome", "cls": "road"}, {"id": 1884, "dom": "P", "ent": "chromosome", "cls": "salmon"}, {"id": 1885, "dom": "P", "ent": "chromosome", "cls": "shark"}, {"id": 1886, "dom": "P", "ent": "chromosome", "cls": "sheep"}, {"id": 1887, "dom": "P", "ent": "chromosome", "cls": "shook"}, {"id": 1888, "dom": "P", "ent": "chromosome", "cls": "tank"}, {"id": 1889, "dom": "P", "ent": "chromosome", "cls": "wheat"}, {"id": 1890, "dom": "P", "ent": "circle", "cls": "architecture"}, {"id": 1891, "dom": "P", "ent": "circle", "cls": "bacteria"}, {"id": 1892, "dom": "P", "ent": "circle", "cls": "beef"}, {"id": 1893, "dom": "P", "ent": "circle", "cls": "bridge"}, {"id": 1894, "dom": "P", "ent": "circle", "cls": "bud"}, {"id": 1895, "dom": "P", "ent": "circle", "cls": "cat"}, {"id": 1896, "dom": "P", "ent": "circle", "cls": "chapel"}, {"id": 1897, "dom": "P", "ent": "circle", "cls": "chick"}, {"id": 1898, "dom": "P", "ent": "circle", "cls": "clock"}, {"id": 1899, "dom": "P", "ent": "circle", "cls": "deer"}, {"id": 1900, "dom": "P", "ent": "circle", "cls": "dog"}, {"id": 1901, "dom": "P", "ent": "circle", "cls": "elephant"}, {"id": 1902, "dom": "P", "ent": "circle", "cls": "embassy"}, {"id": 1903, "dom": "P", "ent": "circle", "cls": "garage"}, {"id": 1904, "dom": "P", "ent": "circle", "cls": "highway"}, {"id": 1905, "dom": "P", "ent": "circle", "cls": "kidney"}, {"id": 1906, "dom": "P", "ent": "circle", "cls": "lamb"}, {"id": 1907, "dom": "P", "ent": "circle", "cls": "male"}, {"id": 1908, "dom": "P", "ent": "circle", "cls": "marijuana"}, {"id": 1909, "dom": "P", "ent": "circle", "cls": "puppy"}, {"id": 1910, "dom": "P", "ent": "circle", "cls": "receiver"}, {"id": 1911, "dom": "P", "ent": "circle", "cls": "reed"}, {"id": 1912, "dom": "P", "ent": "circle", "cls": "rifle"}, {"id": 1913, "dom": "P", "ent": "circle", "cls": "road"}, {"id": 1914, "dom": "P", "ent": "circle", "cls": "salmon"}, {"id": 1915, "dom": "P", "ent": "circle", "cls": "shark"}, {"id": 1916, "dom": "P", "ent": "circle", "cls": "sheep"}, {"id": 1917, "dom": "P", "ent": "circle", "cls": "shook"}, {"id": 1918, "dom": "P", "ent": "circle", "cls": "tank"}, {"id": 1919, "dom": "P", "ent": "circle", "cls": "wheat"}, {"id": 1920, "dom": "P", "ent": "circuit", "cls": "architecture"}, {"id": 1921, "dom": "P", "ent": "circuit", "cls": "bacteria"}, {"id": 1922, "dom": "P", "ent": "circuit", "cls": "beef"}, {"id": 1923, "dom": "P", "ent": "circuit", "cls": "bridge"}, {"id": 1924, "dom": "P", "ent": "circuit", "cls": "bud"}, {"id": 1925, "dom": "P", "ent": "circuit", "cls": "cat"}, {"id": 1926, "dom": "P", "ent": "circuit", "cls": "chapel"}, {"id": 1927, "dom": "P", "ent": "circuit", "cls": "chick"}, {"id": 1928, "dom": "P", "ent": "circuit", "cls": "clock"}, {"id": 1929, "dom": "P", "ent": "circuit", "cls": "deer"}, {"id": 1930, "dom": "P", "ent": "circuit", "cls": "dog"}, {"id": 1931, "dom": "P", "ent": "circuit", "cls": "elephant"}, {"id": 1932, "dom": "P", "ent": "circuit", "cls": "embassy"}, {"id": 1933, "dom": "P", "ent": "circuit", "cls": "garage"}, {"id": 1934, "dom": "P", "ent": "circuit", "cls": "highway"}, {"id": 1935, "dom": "P", "ent": "circuit", "cls": "kidney"}, {"id": 1936, "dom": "P", "ent": "circuit", "cls": "lamb"}, {"id": 1937, "dom": "P", "ent": "circuit", "cls": "male"}, {"id": 1938, "dom": "P", "ent": "circuit", "cls": "marijuana"}, {"id": 1939, "dom": "P", "ent": "circuit", "cls": "puppy"}, {"id": 1940, "dom": "P", "ent": "circuit", "cls": "receiver"}, {"id": 1941, "dom": "P", "ent": "circuit", "cls": "reed"}, {"id": 1942, "dom": "P", "ent": "circuit", "cls": "rifle"}, {"id": 1943, "dom": "P", "ent": "circuit", "cls": "road"}, {"id": 1944, "dom": "P", "ent": "circuit", "cls": "salmon"}, {"id": 1945, "dom": "P", "ent": "circuit", "cls": "shark"}, {"id": 1946, "dom": "P", "ent": "circuit", "cls": "sheep"}, {"id": 1947, "dom": "P", "ent": "circuit", "cls": "shook"}, {"id": 1948, "dom": "P", "ent": "circuit", "cls": "tank"}, {"id": 1949, "dom": "P", "ent": "circuit", "cls": "wheat"}, {"id": 1950, "dom": "P", "ent": "cleaner", "cls": "architecture"}, {"id": 1951, "dom": "P", "ent": "cleaner", "cls": "bacteria"}, {"id": 1952, "dom": "P", "ent": "cleaner", "cls": "beef"}, {"id": 1953, "dom": "P", "ent": "cleaner", "cls": "bridge"}, {"id": 1954, "dom": "P", "ent": "cleaner", "cls": "bud"}, {"id": 1955, "dom": "P", "ent": "cleaner", "cls": "cat"}, {"id": 1956, "dom": "P", "ent": "cleaner", "cls": "chapel"}, {"id": 1957, "dom": "P", "ent": "cleaner", "cls": "chick"}, {"id": 1958, "dom": "P", "ent": "cleaner", "cls": "clock"}, {"id": 1959, "dom": "P", "ent": "cleaner", "cls": "deer"}, {"id": 1960, "dom": "P", "ent": "cleaner", "cls": "dog"}, {"id": 1961, "dom": "P", "ent": "cleaner", "cls": "elephant"}, {"id": 1962, "dom": "P", "ent": "cleaner", "cls": "embassy"}, {"id": 1963, "dom": "P", "ent": "cleaner", "cls": "garage"}, {"id": 1964, "dom": "P", "ent": "cleaner", "cls": "highway"}, {"id": 1965, "dom": "P", "ent": "cleaner", "cls": "kidney"}, {"id": 1966, "dom": "P", "ent": "cleaner", "cls": "lamb"}, {"id": 1967, "dom": "P", "ent": "cleaner", "cls": "male"}, {"id": 1968, "dom": "P", "ent": "cleaner", "cls": "marijuana"}, {"id": 1969, "dom": "P", "ent": "cleaner", "cls": "puppy"}, {"id": 1970, "dom": "P", "ent": "cleaner", "cls": "receiver"}, {"id": 1971, "dom": "P", "ent": "cleaner", "cls": "reed"}, {"id": 1972, "dom": "P", "ent": "cleaner", "cls": "rifle"}, {"id": 1973, "dom": "P", "ent": "cleaner", "cls": "road"}, {"id": 1974, "dom": "P", "ent": "cleaner", "cls": "salmon"}, {"id": 1975, "dom": "P", "ent": "cleaner", "cls": "shark"}, {"id": 1976, "dom": "P", "ent": "cleaner", "cls": "sheep"}, {"id": 1977, "dom": "P", "ent": "cleaner", "cls": "shook"}, {"id": 1978, "dom": "P", "ent": "cleaner", "cls": "tank"}, {"id": 1979, "dom": "P", "ent": "cleaner", "cls": "wheat"}, {"id": 1980, "dom": "P", "ent": "clip", "cls": "architecture"}, {"id": 1981, "dom": "P", "ent": "clip", "cls": "bacteria"}, {"id": 1982, "dom": "P", "ent": "clip", "cls": "beef"}, {"id": 1983, "dom": "P", "ent": "clip", "cls": "bridge"}, {"id": 1984, "dom": "P", "ent": "clip", "cls": "bud"}, {"id": 1985, "dom": "P", "ent": "clip", "cls": "cat"}, {"id": 1986, "dom": "P", "ent": "clip", "cls": "chapel"}, {"id": 1987, "dom": "P", "ent": "clip", "cls": "chick"}, {"id": 1988, "dom": "P", "ent": "clip", "cls": "clock"}, {"id": 1989, "dom": "P", "ent": "clip", "cls": "deer"}, {"id": 1990, "dom": "P", "ent": "clip", "cls": "dog"}, {"id": 1991, "dom": "P", "ent": "clip", "cls": "elephant"}, {"id": 1992, "dom": "P", "ent": "clip", "cls": "embassy"}, {"id": 1993, "dom": "P", "ent": "clip", "cls": "garage"}, {"id": 1994, "dom": "P", "ent": "clip", "cls": "highway"}, {"id": 1995, "dom": "P", "ent": "clip", "cls": "kidney"}, {"id": 1996, "dom": "P", "ent": "clip", "cls": "lamb"}, {"id": 1997, "dom": "P", "ent": "clip", "cls": "male"}, {"id": 1998, "dom": "P", "ent": "clip", "cls": "marijuana"}, {"id": 1999, "dom": "P", "ent": "clip", "cls": "puppy"}, {"id": 2000, "dom": "P", "ent": "clip", "cls": "receiver"}, {"id": 2001, "dom": "P", "ent": "clip", "cls": "reed"}, {"id": 2002, "dom": "P", "ent": "clip", "cls": "rifle"}, {"id": 2003, "dom": "P", "ent": "clip", "cls": "road"}, {"id": 2004, "dom": "P", "ent": "clip", "cls": "salmon"}, {"id": 2005, "dom": "P", "ent": "clip", "cls": "shark"}, {"id": 2006, "dom": "P", "ent": "clip", "cls": "sheep"}, {"id": 2007, "dom": "P", "ent": "clip", "cls": "shook"}, {"id": 2008, "dom": "P", "ent": "clip", "cls": "tank"}, {"id": 2009, "dom": "P", "ent": "clip", "cls": "wheat"}, {"id": 2010, "dom": "P", "ent": "clone", "cls": "architecture"}, {"id": 2011, "dom": "P", "ent": "clone", "cls": "bacteria"}, {"id": 2012, "dom": "P", "ent": "clone", "cls": "beef"}, {"id": 2013, "dom": "P", "ent": "clone", "cls": "bridge"}, {"id": 2014, "dom": "P", "ent": "clone", "cls": "bud"}, {"id": 2015, "dom": "P", "ent": "clone", "cls": "cat"}, {"id": 2016, "dom": "P", "ent": "clone", "cls": "chapel"}, {"id": 2017, "dom": "P", "ent": "clone", "cls": "chick"}, {"id": 2018, "dom": "P", "ent": "clone", "cls": "clock"}, {"id": 2019, "dom": "P", "ent": "clone", "cls": "deer"}, {"id": 2020, "dom": "P", "ent": "clone", "cls": "dog"}, {"id": 2021, "dom": "P", "ent": "clone", "cls": "elephant"}, {"id": 2022, "dom": "P", "ent": "clone", "cls": "embassy"}, {"id": 2023, "dom": "P", "ent": "clone", "cls": "garage"}, {"id": 2024, "dom": "P", "ent": "clone", "cls": "highway"}, {"id": 2025, "dom": "P", "ent": "clone", "cls": "kidney"}, {"id": 2026, "dom": "P", "ent": "clone", "cls": "lamb"}, {"id": 2027, "dom": "P", "ent": "clone", "cls": "male"}, {"id": 2028, "dom": "P", "ent": "clone", "cls": "marijuana"}, {"id": 2029, "dom": "P", "ent": "clone", "cls": "puppy"}, {"id": 2030, "dom": "P", "ent": "clone", "cls": "receiver"}, {"id": 2031, "dom": "P", "ent": "clone", "cls": "reed"}, {"id": 2032, "dom": "P", "ent": "clone", "cls": "rifle"}, {"id": 2033, "dom": "P", "ent": "clone", "cls": "road"}, {"id": 2034, "dom": "P", "ent": "clone", "cls": "salmon"}, {"id": 2035, "dom": "P", "ent": "clone", "cls": "shark"}, {"id": 2036, "dom": "P", "ent": "clone", "cls": "sheep"}, {"id": 2037, "dom": "P", "ent": "clone", "cls": "shook"}, {"id": 2038, "dom": "P", "ent": "clone", "cls": "tank"}, {"id": 2039, "dom": "P", "ent": "clone", "cls": "wheat"}, {"id": 2040, "dom": "P", "ent": "club", "cls": "architecture"}, {"id": 2041, "dom": "P", "ent": "club", "cls": "bacteria"}, {"id": 2042, "dom": "P", "ent": "club", "cls": "beef"}, {"id": 2043, "dom": "P", "ent": "club", "cls": "bridge"}, {"id": 2044, "dom": "P", "ent": "club", "cls": "bud"}, {"id": 2045, "dom": "P", "ent": "club", "cls": "cat"}, {"id": 2046, "dom": "P", "ent": "club", "cls": "chapel"}, {"id": 2047, "dom": "P", "ent": "club", "cls": "chick"}, {"id": 2048, "dom": "P", "ent": "club", "cls": "clock"}, {"id": 2049, "dom": "P", "ent": "club", "cls": "deer"}, {"id": 2050, "dom": "P", "ent": "club", "cls": "dog"}, {"id": 2051, "dom": "P", "ent": "club", "cls": "elephant"}, {"id": 2052, "dom": "P", "ent": "club", "cls": "embassy"}, {"id": 2053, "dom": "P", "ent": "club", "cls": "garage"}, {"id": 2054, "dom": "P", "ent": "club", "cls": "highway"}, {"id": 2055, "dom": "P", "ent": "club", "cls": "kidney"}, {"id": 2056, "dom": "P", "ent": "club", "cls": "lamb"}, {"id": 2057, "dom": "P", "ent": "club", "cls": "male"}, {"id": 2058, "dom": "P", "ent": "club", "cls": "marijuana"}, {"id": 2059, "dom": "P", "ent": "club", "cls": "puppy"}, {"id": 2060, "dom": "P", "ent": "club", "cls": "receiver"}, {"id": 2061, "dom": "P", "ent": "club", "cls": "reed"}, {"id": 2062, "dom": "P", "ent": "club", "cls": "rifle"}, {"id": 2063, "dom": "P", "ent": "club", "cls": "road"}, {"id": 2064, "dom": "P", "ent": "club", "cls": "salmon"}, {"id": 2065, "dom": "P", "ent": "club", "cls": "shark"}, {"id": 2066, "dom": "P", "ent": "club", "cls": "sheep"}, {"id": 2067, "dom": "P", "ent": "club", "cls": "shook"}, {"id": 2068, "dom": "P", "ent": "club", "cls": "tank"}, {"id": 2069, "dom": "P", "ent": "club", "cls": "wheat"}, {"id": 2070, "dom": "P", "ent": "coat", "cls": "architecture"}, {"id": 2071, "dom": "P", "ent": "coat", "cls": "bacteria"}, {"id": 2072, "dom": "P", "ent": "coat", "cls": "beef"}, {"id": 2073, "dom": "P", "ent": "coat", "cls": "bridge"}, {"id": 2074, "dom": "P", "ent": "coat", "cls": "bud"}, {"id": 2075, "dom": "P", "ent": "coat", "cls": "cat"}, {"id": 2076, "dom": "P", "ent": "coat", "cls": "chapel"}, {"id": 2077, "dom": "P", "ent": "coat", "cls": "chick"}, {"id": 2078, "dom": "P", "ent": "coat", "cls": "clock"}, {"id": 2079, "dom": "P", "ent": "coat", "cls": "deer"}, {"id": 2080, "dom": "P", "ent": "coat", "cls": "dog"}, {"id": 2081, "dom": "P", "ent": "coat", "cls": "elephant"}, {"id": 2082, "dom": "P", "ent": "coat", "cls": "embassy"}, {"id": 2083, "dom": "P", "ent": "coat", "cls": "garage"}, {"id": 2084, "dom": "P", "ent": "coat", "cls": "highway"}, {"id": 2085, "dom": "P", "ent": "coat", "cls": "kidney"}, {"id": 2086, "dom": "P", "ent": "coat", "cls": "lamb"}, {"id": 2087, "dom": "P", "ent": "coat", "cls": "male"}, {"id": 2088, "dom": "P", "ent": "coat", "cls": "marijuana"}, {"id": 2089, "dom": "P", "ent": "coat", "cls": "puppy"}, {"id": 2090, "dom": "P", "ent": "coat", "cls": "receiver"}, {"id": 2091, "dom": "P", "ent": "coat", "cls": "reed"}, {"id": 2092, "dom": "P", "ent": "coat", "cls": "rifle"}, {"id": 2093, "dom": "P", "ent": "coat", "cls": "road"}, {"id": 2094, "dom": "P", "ent": "coat", "cls": "salmon"}, {"id": 2095, "dom": "P", "ent": "coat", "cls": "shark"}, {"id": 2096, "dom": "P", "ent": "coat", "cls": "sheep"}, {"id": 2097, "dom": "P", "ent": "coat", "cls": "shook"}, {"id": 2098, "dom": "P", "ent": "coat", "cls": "tank"}, {"id": 2099, "dom": "P", "ent": "coat", "cls": "wheat"}, {"id": 2100, "dom": "P", "ent": "cock", "cls": "architecture"}, {"id": 2101, "dom": "P", "ent": "cock", "cls": "bacteria"}, {"id": 2102, "dom": "P", "ent": "cock", "cls": "beef"}, {"id": 2103, "dom": "P", "ent": "cock", "cls": "bridge"}, {"id": 2104, "dom": "P", "ent": "cock", "cls": "bud"}, {"id": 2105, "dom": "P", "ent": "cock", "cls": "cat"}, {"id": 2106, "dom": "P", "ent": "cock", "cls": "chapel"}, {"id": 2107, "dom": "P", "ent": "cock", "cls": "chick"}, {"id": 2108, "dom": "P", "ent": "cock", "cls": "clock"}, {"id": 2109, "dom": "P", "ent": "cock", "cls": "deer"}, {"id": 2110, "dom": "P", "ent": "cock", "cls": "dog"}, {"id": 2111, "dom": "P", "ent": "cock", "cls": "elephant"}, {"id": 2112, "dom": "P", "ent": "cock", "cls": "embassy"}, {"id": 2113, "dom": "P", "ent": "cock", "cls": "garage"}, {"id": 2114, "dom": "P", "ent": "cock", "cls": "highway"}, {"id": 2115, "dom": "P", "ent": "cock", "cls": "kidney"}, {"id": 2116, "dom": "P", "ent": "cock", "cls": "lamb"}, {"id": 2117, "dom": "P", "ent": "cock", "cls": "male"}, {"id": 2118, "dom": "P", "ent": "cock", "cls": "marijuana"}, {"id": 2119, "dom": "P", "ent": "cock", "cls": "puppy"}, {"id": 2120, "dom": "P", "ent": "cock", "cls": "receiver"}, {"id": 2121, "dom": "P", "ent": "cock", "cls": "reed"}, {"id": 2122, "dom": "P", "ent": "cock", "cls": "rifle"}, {"id": 2123, "dom": "P", "ent": "cock", "cls": "road"}, {"id": 2124, "dom": "P", "ent": "cock", "cls": "salmon"}, {"id": 2125, "dom": "P", "ent": "cock", "cls": "shark"}, {"id": 2126, "dom": "P", "ent": "cock", "cls": "sheep"}, {"id": 2127, "dom": "P", "ent": "cock", "cls": "shook"}, {"id": 2128, "dom": "P", "ent": "cock", "cls": "tank"}, {"id": 2129, "dom": "P", "ent": "cock", "cls": "wheat"}, {"id": 2130, "dom": "P", "ent": "comb", "cls": "architecture"}, {"id": 2131, "dom": "P", "ent": "comb", "cls": "bacteria"}, {"id": 2132, "dom": "P", "ent": "comb", "cls": "beef"}, {"id": 2133, "dom": "P", "ent": "comb", "cls": "bridge"}, {"id": 2134, "dom": "P", "ent": "comb", "cls": "bud"}, {"id": 2135, "dom": "P", "ent": "comb", "cls": "cat"}, {"id": 2136, "dom": "P", "ent": "comb", "cls": "chapel"}, {"id": 2137, "dom": "P", "ent": "comb", "cls": "chick"}, {"id": 2138, "dom": "P", "ent": "comb", "cls": "clock"}, {"id": 2139, "dom": "P", "ent": "comb", "cls": "deer"}, {"id": 2140, "dom": "P", "ent": "comb", "cls": "dog"}, {"id": 2141, "dom": "P", "ent": "comb", "cls": "elephant"}, {"id": 2142, "dom": "P", "ent": "comb", "cls": "embassy"}, {"id": 2143, "dom": "P", "ent": "comb", "cls": "garage"}, {"id": 2144, "dom": "P", "ent": "comb", "cls": "highway"}, {"id": 2145, "dom": "P", "ent": "comb", "cls": "kidney"}, {"id": 2146, "dom": "P", "ent": "comb", "cls": "lamb"}, {"id": 2147, "dom": "P", "ent": "comb", "cls": "male"}, {"id": 2148, "dom": "P", "ent": "comb", "cls": "marijuana"}, {"id": 2149, "dom": "P", "ent": "comb", "cls": "puppy"}, {"id": 2150, "dom": "P", "ent": "comb", "cls": "receiver"}, {"id": 2151, "dom": "P", "ent": "comb", "cls": "reed"}, {"id": 2152, "dom": "P", "ent": "comb", "cls": "rifle"}, {"id": 2153, "dom": "P", "ent": "comb", "cls": "road"}, {"id": 2154, "dom": "P", "ent": "comb", "cls": "salmon"}, {"id": 2155, "dom": "P", "ent": "comb", "cls": "shark"}, {"id": 2156, "dom": "P", "ent": "comb", "cls": "sheep"}, {"id": 2157, "dom": "P", "ent": "comb", "cls": "shook"}, {"id": 2158, "dom": "P", "ent": "comb", "cls": "tank"}, {"id": 2159, "dom": "P", "ent": "comb", "cls": "wheat"}, {"id": 2160, "dom": "P", "ent": "cone", "cls": "architecture"}, {"id": 2161, "dom": "P", "ent": "cone", "cls": "bacteria"}, {"id": 2162, "dom": "P", "ent": "cone", "cls": "beef"}, {"id": 2163, "dom": "P", "ent": "cone", "cls": "bridge"}, {"id": 2164, "dom": "P", "ent": "cone", "cls": "bud"}, {"id": 2165, "dom": "P", "ent": "cone", "cls": "cat"}, {"id": 2166, "dom": "P", "ent": "cone", "cls": "chapel"}, {"id": 2167, "dom": "P", "ent": "cone", "cls": "chick"}, {"id": 2168, "dom": "P", "ent": "cone", "cls": "clock"}, {"id": 2169, "dom": "P", "ent": "cone", "cls": "deer"}, {"id": 2170, "dom": "P", "ent": "cone", "cls": "dog"}, {"id": 2171, "dom": "P", "ent": "cone", "cls": "elephant"}, {"id": 2172, "dom": "P", "ent": "cone", "cls": "embassy"}, {"id": 2173, "dom": "P", "ent": "cone", "cls": "garage"}, {"id": 2174, "dom": "P", "ent": "cone", "cls": "highway"}, {"id": 2175, "dom": "P", "ent": "cone", "cls": "kidney"}, {"id": 2176, "dom": "P", "ent": "cone", "cls": "lamb"}, {"id": 2177, "dom": "P", "ent": "cone", "cls": "male"}, {"id": 2178, "dom": "P", "ent": "cone", "cls": "marijuana"}, {"id": 2179, "dom": "P", "ent": "cone", "cls": "puppy"}, {"id": 2180, "dom": "P", "ent": "cone", "cls": "receiver"}, {"id": 2181, "dom": "P", "ent": "cone", "cls": "reed"}, {"id": 2182, "dom": "P", "ent": "cone", "cls": "rifle"}, {"id": 2183, "dom": "P", "ent": "cone", "cls": "road"}, {"id": 2184, "dom": "P", "ent": "cone", "cls": "salmon"}, {"id": 2185, "dom": "P", "ent": "cone", "cls": "shark"}, {"id": 2186, "dom": "P", "ent": "cone", "cls": "sheep"}, {"id": 2187, "dom": "P", "ent": "cone", "cls": "shook"}, {"id": 2188, "dom": "P", "ent": "cone", "cls": "tank"}, {"id": 2189, "dom": "P", "ent": "cone", "cls": "wheat"}, {"id": 2190, "dom": "P", "ent": "contact", "cls": "architecture"}, {"id": 2191, "dom": "P", "ent": "contact", "cls": "bacteria"}, {"id": 2192, "dom": "P", "ent": "contact", "cls": "beef"}, {"id": 2193, "dom": "P", "ent": "contact", "cls": "bridge"}, {"id": 2194, "dom": "P", "ent": "contact", "cls": "bud"}, {"id": 2195, "dom": "P", "ent": "contact", "cls": "cat"}, {"id": 2196, "dom": "P", "ent": "contact", "cls": "chapel"}, {"id": 2197, "dom": "P", "ent": "contact", "cls": "chick"}, {"id": 2198, "dom": "P", "ent": "contact", "cls": "clock"}, {"id": 2199, "dom": "P", "ent": "contact", "cls": "deer"}, {"id": 2200, "dom": "P", "ent": "contact", "cls": "dog"}, {"id": 2201, "dom": "P", "ent": "contact", "cls": "elephant"}, {"id": 2202, "dom": "P", "ent": "contact", "cls": "embassy"}, {"id": 2203, "dom": "P", "ent": "contact", "cls": "garage"}, {"id": 2204, "dom": "P", "ent": "contact", "cls": "highway"}, {"id": 2205, "dom": "P", "ent": "contact", "cls": "kidney"}, {"id": 2206, "dom": "P", "ent": "contact", "cls": "lamb"}, {"id": 2207, "dom": "P", "ent": "contact", "cls": "male"}, {"id": 2208, "dom": "P", "ent": "contact", "cls": "marijuana"}, {"id": 2209, "dom": "P", "ent": "contact", "cls": "puppy"}, {"id": 2210, "dom": "P", "ent": "contact", "cls": "receiver"}, {"id": 2211, "dom": "P", "ent": "contact", "cls": "reed"}, {"id": 2212, "dom": "P", "ent": "contact", "cls": "rifle"}, {"id": 2213, "dom": "P", "ent": "contact", "cls": "road"}, {"id": 2214, "dom": "P", "ent": "contact", "cls": "salmon"}, {"id": 2215, "dom": "P", "ent": "contact", "cls": "shark"}, {"id": 2216, "dom": "P", "ent": "contact", "cls": "sheep"}, {"id": 2217, "dom": "P", "ent": "contact", "cls": "shook"}, {"id": 2218, "dom": "P", "ent": "contact", "cls": "tank"}, {"id": 2219, "dom": "P", "ent": "contact", "cls": "wheat"}, {"id": 2220, "dom": "P", "ent": "cope", "cls": "architecture"}, {"id": 2221, "dom": "P", "ent": "cope", "cls": "bacteria"}, {"id": 2222, "dom": "P", "ent": "cope", "cls": "beef"}, {"id": 2223, "dom": "P", "ent": "cope", "cls": "bridge"}, {"id": 2224, "dom": "P", "ent": "cope", "cls": "bud"}, {"id": 2225, "dom": "P", "ent": "cope", "cls": "cat"}, {"id": 2226, "dom": "P", "ent": "cope", "cls": "chapel"}, {"id": 2227, "dom": "P", "ent": "cope", "cls": "chick"}, {"id": 2228, "dom": "P", "ent": "cope", "cls": "clock"}, {"id": 2229, "dom": "P", "ent": "cope", "cls": "deer"}, {"id": 2230, "dom": "P", "ent": "cope", "cls": "dog"}, {"id": 2231, "dom": "P", "ent": "cope", "cls": "elephant"}, {"id": 2232, "dom": "P", "ent": "cope", "cls": "embassy"}, {"id": 2233, "dom": "P", "ent": "cope", "cls": "garage"}, {"id": 2234, "dom": "P", "ent": "cope", "cls": "highway"}, {"id": 2235, "dom": "P", "ent": "cope", "cls": "kidney"}, {"id": 2236, "dom": "P", "ent": "cope", "cls": "lamb"}, {"id": 2237, "dom": "P", "ent": "cope", "cls": "male"}, {"id": 2238, "dom": "P", "ent": "cope", "cls": "marijuana"}, {"id": 2239, "dom": "P", "ent": "cope", "cls": "puppy"}, {"id": 2240, "dom": "P", "ent": "cope", "cls": "receiver"}, {"id": 2241, "dom": "P", "ent": "cope", "cls": "reed"}, {"id": 2242, "dom": "P", "ent": "cope", "cls": "rifle"}, {"id": 2243, "dom": "P", "ent": "cope", "cls": "road"}, {"id": 2244, "dom": "P", "ent": "cope", "cls": "salmon"}, {"id": 2245, "dom": "P", "ent": "cope", "cls": "shark"}, {"id": 2246, "dom": "P", "ent": "cope", "cls": "sheep"}, {"id": 2247, "dom": "P", "ent": "cope", "cls": "shook"}, {"id": 2248, "dom": "P", "ent": "cope", "cls": "tank"}, {"id": 2249, "dom": "P", "ent": "cope", "cls": "wheat"}, {"id": 2250, "dom": "P", "ent": "coping", "cls": "architecture"}, {"id": 2251, "dom": "P", "ent": "coping", "cls": "bacteria"}, {"id": 2252, "dom": "P", "ent": "coping", "cls": "beef"}, {"id": 2253, "dom": "P", "ent": "coping", "cls": "bridge"}, {"id": 2254, "dom": "P", "ent": "coping", "cls": "bud"}, {"id": 2255, "dom": "P", "ent": "coping", "cls": "cat"}, {"id": 2256, "dom": "P", "ent": "coping", "cls": "chapel"}, {"id": 2257, "dom": "P", "ent": "coping", "cls": "chick"}, {"id": 2258, "dom": "P", "ent": "coping", "cls": "clock"}, {"id": 2259, "dom": "P", "ent": "coping", "cls": "deer"}, {"id": 2260, "dom": "P", "ent": "coping", "cls": "dog"}, {"id": 2261, "dom": "P", "ent": "coping", "cls": "elephant"}, {"id": 2262, "dom": "P", "ent": "coping", "cls": "embassy"}, {"id": 2263, "dom": "P", "ent": "coping", "cls": "garage"}, {"id": 2264, "dom": "P", "ent": "coping", "cls": "highway"}, {"id": 2265, "dom": "P", "ent": "coping", "cls": "kidney"}, {"id": 2266, "dom": "P", "ent": "coping", "cls": "lamb"}, {"id": 2267, "dom": "P", "ent": "coping", "cls": "male"}, {"id": 2268, "dom": "P", "ent": "coping", "cls": "marijuana"}, {"id": 2269, "dom": "P", "ent": "coping", "cls": "puppy"}, {"id": 2270, "dom": "P", "ent": "coping", "cls": "receiver"}, {"id": 2271, "dom": "P", "ent": "coping", "cls": "reed"}, {"id": 2272, "dom": "P", "ent": "coping", "cls": "rifle"}, {"id": 2273, "dom": "P", "ent": "coping", "cls": "road"}, {"id": 2274, "dom": "P", "ent": "coping", "cls": "salmon"}, {"id": 2275, "dom": "P", "ent": "coping", "cls": "shark"}, {"id": 2276, "dom": "P", "ent": "coping", "cls": "sheep"}, {"id": 2277, "dom": "P", "ent": "coping", "cls": "shook"}, {"id": 2278, "dom": "P", "ent": "coping", "cls": "tank"}, {"id": 2279, "dom": "P", "ent": "coping", "cls": "wheat"}, {"id": 2280, "dom": "P", "ent": "corner", "cls": "architecture"}, {"id": 2281, "dom": "P", "ent": "corner", "cls": "bacteria"}, {"id": 2282, "dom": "P", "ent": "corner", "cls": "beef"}, {"id": 2283, "dom": "P", "ent": "corner", "cls": "bridge"}, {"id": 2284, "dom": "P", "ent": "corner", "cls": "bud"}, {"id": 2285, "dom": "P", "ent": "corner", "cls": "cat"}, {"id": 2286, "dom": "P", "ent": "corner", "cls": "chapel"}, {"id": 2287, "dom": "P", "ent": "corner", "cls": "chick"}, {"id": 2288, "dom": "P", "ent": "corner", "cls": "clock"}, {"id": 2289, "dom": "P", "ent": "corner", "cls": "deer"}, {"id": 2290, "dom": "P", "ent": "corner", "cls": "dog"}, {"id": 2291, "dom": "P", "ent": "corner", "cls": "elephant"}, {"id": 2292, "dom": "P", "ent": "corner", "cls": "embassy"}, {"id": 2293, "dom": "P", "ent": "corner", "cls": "garage"}, {"id": 2294, "dom": "P", "ent": "corner", "cls": "highway"}, {"id": 2295, "dom": "P", "ent": "corner", "cls": "kidney"}, {"id": 2296, "dom": "P", "ent": "corner", "cls": "lamb"}, {"id": 2297, "dom": "P", "ent": "corner", "cls": "male"}, {"id": 2298, "dom": "P", "ent": "corner", "cls": "marijuana"}, {"id": 2299, "dom": "P", "ent": "corner", "cls": "puppy"}, {"id": 2300, "dom": "P", "ent": "corner", "cls": "receiver"}, {"id": 2301, "dom": "P", "ent": "corner", "cls": "reed"}, {"id": 2302, "dom": "P", "ent": "corner", "cls": "rifle"}, {"id": 2303, "dom": "P", "ent": "corner", "cls": "road"}, {"id": 2304, "dom": "P", "ent": "corner", "cls": "salmon"}, {"id": 2305, "dom": "P", "ent": "corner", "cls": "shark"}, {"id": 2306, "dom": "P", "ent": "corner", "cls": "sheep"}, {"id": 2307, "dom": "P", "ent": "corner", "cls": "shook"}, {"id": 2308, "dom": "P", "ent": "corner", "cls": "tank"}, {"id": 2309, "dom": "P", "ent": "corner", "cls": "wheat"}, {"id": 2310, "dom": "P", "ent": "cornerstone", "cls": "architecture"}, {"id": 2311, "dom": "P", "ent": "cornerstone", "cls": "bacteria"}, {"id": 2312, "dom": "P", "ent": "cornerstone", "cls": "beef"}, {"id": 2313, "dom": "P", "ent": "cornerstone", "cls": "bridge"}, {"id": 2314, "dom": "P", "ent": "cornerstone", "cls": "bud"}, {"id": 2315, "dom": "P", "ent": "cornerstone", "cls": "cat"}, {"id": 2316, "dom": "P", "ent": "cornerstone", "cls": "chapel"}, {"id": 2317, "dom": "P", "ent": "cornerstone", "cls": "chick"}, {"id": 2318, "dom": "P", "ent": "cornerstone", "cls": "clock"}, {"id": 2319, "dom": "P", "ent": "cornerstone", "cls": "deer"}, {"id": 2320, "dom": "P", "ent": "cornerstone", "cls": "dog"}, {"id": 2321, "dom": "P", "ent": "cornerstone", "cls": "elephant"}, {"id": 2322, "dom": "P", "ent": "cornerstone", "cls": "embassy"}, {"id": 2323, "dom": "P", "ent": "cornerstone", "cls": "garage"}, {"id": 2324, "dom": "P", "ent": "cornerstone", "cls": "highway"}, {"id": 2325, "dom": "P", "ent": "cornerstone", "cls": "kidney"}, {"id": 2326, "dom": "P", "ent": "cornerstone", "cls": "lamb"}, {"id": 2327, "dom": "P", "ent": "cornerstone", "cls": "male"}, {"id": 2328, "dom": "P", "ent": "cornerstone", "cls": "marijuana"}, {"id": 2329, "dom": "P", "ent": "cornerstone", "cls": "puppy"}, {"id": 2330, "dom": "P", "ent": "cornerstone", "cls": "receiver"}, {"id": 2331, "dom": "P", "ent": "cornerstone", "cls": "reed"}, {"id": 2332, "dom": "P", "ent": "cornerstone", "cls": "rifle"}, {"id": 2333, "dom": "P", "ent": "cornerstone", "cls": "road"}, {"id": 2334, "dom": "P", "ent": "cornerstone", "cls": "salmon"}, {"id": 2335, "dom": "P", "ent": "cornerstone", "cls": "shark"}, {"id": 2336, "dom": "P", "ent": "cornerstone", "cls": "sheep"}, {"id": 2337, "dom": "P", "ent": "cornerstone", "cls": "shook"}, {"id": 2338, "dom": "P", "ent": "cornerstone", "cls": "tank"}, {"id": 2339, "dom": "P", "ent": "cornerstone", "cls": "wheat"}, {"id": 2340, "dom": "P", "ent": "corolla", "cls": "architecture"}, {"id": 2341, "dom": "P", "ent": "corolla", "cls": "bacteria"}, {"id": 2342, "dom": "P", "ent": "corolla", "cls": "beef"}, {"id": 2343, "dom": "P", "ent": "corolla", "cls": "bridge"}, {"id": 2344, "dom": "P", "ent": "corolla", "cls": "bud"}, {"id": 2345, "dom": "P", "ent": "corolla", "cls": "cat"}, {"id": 2346, "dom": "P", "ent": "corolla", "cls": "chapel"}, {"id": 2347, "dom": "P", "ent": "corolla", "cls": "chick"}, {"id": 2348, "dom": "P", "ent": "corolla", "cls": "clock"}, {"id": 2349, "dom": "P", "ent": "corolla", "cls": "deer"}, {"id": 2350, "dom": "P", "ent": "corolla", "cls": "dog"}, {"id": 2351, "dom": "P", "ent": "corolla", "cls": "elephant"}, {"id": 2352, "dom": "P", "ent": "corolla", "cls": "embassy"}, {"id": 2353, "dom": "P", "ent": "corolla", "cls": "garage"}, {"id": 2354, "dom": "P", "ent": "corolla", "cls": "highway"}, {"id": 2355, "dom": "P", "ent": "corolla", "cls": "kidney"}, {"id": 2356, "dom": "P", "ent": "corolla", "cls": "lamb"}, {"id": 2357, "dom": "P", "ent": "corolla", "cls": "male"}, {"id": 2358, "dom": "P", "ent": "corolla", "cls": "marijuana"}, {"id": 2359, "dom": "P", "ent": "corolla", "cls": "puppy"}, {"id": 2360, "dom": "P", "ent": "corolla", "cls": "receiver"}, {"id": 2361, "dom": "P", "ent": "corolla", "cls": "reed"}, {"id": 2362, "dom": "P", "ent": "corolla", "cls": "rifle"}, {"id": 2363, "dom": "P", "ent": "corolla", "cls": "road"}, {"id": 2364, "dom": "P", "ent": "corolla", "cls": "salmon"}, {"id": 2365, "dom": "P", "ent": "corolla", "cls": "shark"}, {"id": 2366, "dom": "P", "ent": "corolla", "cls": "sheep"}, {"id": 2367, "dom": "P", "ent": "corolla", "cls": "shook"}, {"id": 2368, "dom": "P", "ent": "corolla", "cls": "tank"}, {"id": 2369, "dom": "P", "ent": "corolla", "cls": "wheat"}, {"id": 2370, "dom": "P", "ent": "corona", "cls": "architecture"}, {"id": 2371, "dom": "P", "ent": "corona", "cls": "bacteria"}, {"id": 2372, "dom": "P", "ent": "corona", "cls": "beef"}, {"id": 2373, "dom": "P", "ent": "corona", "cls": "bridge"}, {"id": 2374, "dom": "P", "ent": "corona", "cls": "bud"}, {"id": 2375, "dom": "P", "ent": "corona", "cls": "cat"}, {"id": 2376, "dom": "P", "ent": "corona", "cls": "chapel"}, {"id": 2377, "dom": "P", "ent": "corona", "cls": "chick"}, {"id": 2378, "dom": "P", "ent": "corona", "cls": "clock"}, {"id": 2379, "dom": "P", "ent": "corona", "cls": "deer"}, {"id": 2380, "dom": "P", "ent": "corona", "cls": "dog"}, {"id": 2381, "dom": "P", "ent": "corona", "cls": "elephant"}, {"id": 2382, "dom": "P", "ent": "corona", "cls": "embassy"}, {"id": 2383, "dom": "P", "ent": "corona", "cls": "garage"}, {"id": 2384, "dom": "P", "ent": "corona", "cls": "highway"}, {"id": 2385, "dom": "P", "ent": "corona", "cls": "kidney"}, {"id": 2386, "dom": "P", "ent": "corona", "cls": "lamb"}, {"id": 2387, "dom": "P", "ent": "corona", "cls": "male"}, {"id": 2388, "dom": "P", "ent": "corona", "cls": "marijuana"}, {"id": 2389, "dom": "P", "ent": "corona", "cls": "puppy"}, {"id": 2390, "dom": "P", "ent": "corona", "cls": "receiver"}, {"id": 2391, "dom": "P", "ent": "corona", "cls": "reed"}, {"id": 2392, "dom": "P", "ent": "corona", "cls": "rifle"}, {"id": 2393, "dom": "P", "ent": "corona", "cls": "road"}, {"id": 2394, "dom": "P", "ent": "corona", "cls": "salmon"}, {"id": 2395, "dom": "P", "ent": "corona", "cls": "shark"}, {"id": 2396, "dom": "P", "ent": "corona", "cls": "sheep"}, {"id": 2397, "dom": "P", "ent": "corona", "cls": "shook"}, {"id": 2398, "dom": "P", "ent": "corona", "cls": "tank"}, {"id": 2399, "dom": "P", "ent": "corona", "cls": "wheat"}, {"id": 2400, "dom": "P", "ent": "corpus", "cls": "architecture"}, {"id": 2401, "dom": "P", "ent": "corpus", "cls": "bacteria"}, {"id": 2402, "dom": "P", "ent": "corpus", "cls": "beef"}, {"id": 2403, "dom": "P", "ent": "corpus", "cls": "bridge"}, {"id": 2404, "dom": "P", "ent": "corpus", "cls": "bud"}, {"id": 2405, "dom": "P", "ent": "corpus", "cls": "cat"}, {"id": 2406, "dom": "P", "ent": "corpus", "cls": "chapel"}, {"id": 2407, "dom": "P", "ent": "corpus", "cls": "chick"}, {"id": 2408, "dom": "P", "ent": "corpus", "cls": "clock"}, {"id": 2409, "dom": "P", "ent": "corpus", "cls": "deer"}, {"id": 2410, "dom": "P", "ent": "corpus", "cls": "dog"}, {"id": 2411, "dom": "P", "ent": "corpus", "cls": "elephant"}, {"id": 2412, "dom": "P", "ent": "corpus", "cls": "embassy"}, {"id": 2413, "dom": "P", "ent": "corpus", "cls": "garage"}, {"id": 2414, "dom": "P", "ent": "corpus", "cls": "highway"}, {"id": 2415, "dom": "P", "ent": "corpus", "cls": "kidney"}, {"id": 2416, "dom": "P", "ent": "corpus", "cls": "lamb"}, {"id": 2417, "dom": "P", "ent": "corpus", "cls": "male"}, {"id": 2418, "dom": "P", "ent": "corpus", "cls": "marijuana"}, {"id": 2419, "dom": "P", "ent": "corpus", "cls": "puppy"}, {"id": 2420, "dom": "P", "ent": "corpus", "cls": "receiver"}, {"id": 2421, "dom": "P", "ent": "corpus", "cls": "reed"}, {"id": 2422, "dom": "P", "ent": "corpus", "cls": "rifle"}, {"id": 2423, "dom": "P", "ent": "corpus", "cls": "road"}, {"id": 2424, "dom": "P", "ent": "corpus", "cls": "salmon"}, {"id": 2425, "dom": "P", "ent": "corpus", "cls": "shark"}, {"id": 2426, "dom": "P", "ent": "corpus", "cls": "sheep"}, {"id": 2427, "dom": "P", "ent": "corpus", "cls": "shook"}, {"id": 2428, "dom": "P", "ent": "corpus", "cls": "tank"}, {"id": 2429, "dom": "P", "ent": "corpus", "cls": "wheat"}, {"id": 2430, "dom": "P", "ent": "cortex", "cls": "architecture"}, {"id": 2431, "dom": "P", "ent": "cortex", "cls": "bacteria"}, {"id": 2432, "dom": "P", "ent": "cortex", "cls": "beef"}, {"id": 2433, "dom": "P", "ent": "cortex", "cls": "bridge"}, {"id": 2434, "dom": "P", "ent": "cortex", "cls": "bud"}, {"id": 2435, "dom": "P", "ent": "cortex", "cls": "cat"}, {"id": 2436, "dom": "P", "ent": "cortex", "cls": "chapel"}, {"id": 2437, "dom": "P", "ent": "cortex", "cls": "chick"}, {"id": 2438, "dom": "P", "ent": "cortex", "cls": "clock"}, {"id": 2439, "dom": "P", "ent": "cortex", "cls": "deer"}, {"id": 2440, "dom": "P", "ent": "cortex", "cls": "dog"}, {"id": 2441, "dom": "P", "ent": "cortex", "cls": "elephant"}, {"id": 2442, "dom": "P", "ent": "cortex", "cls": "embassy"}, {"id": 2443, "dom": "P", "ent": "cortex", "cls": "garage"}, {"id": 2444, "dom": "P", "ent": "cortex", "cls": "highway"}, {"id": 2445, "dom": "P", "ent": "cortex", "cls": "kidney"}, {"id": 2446, "dom": "P", "ent": "cortex", "cls": "lamb"}, {"id": 2447, "dom": "P", "ent": "cortex", "cls": "male"}, {"id": 2448, "dom": "P", "ent": "cortex", "cls": "marijuana"}, {"id": 2449, "dom": "P", "ent": "cortex", "cls": "puppy"}, {"id": 2450, "dom": "P", "ent": "cortex", "cls": "receiver"}, {"id": 2451, "dom": "P", "ent": "cortex", "cls": "reed"}, {"id": 2452, "dom": "P", "ent": "cortex", "cls": "rifle"}, {"id": 2453, "dom": "P", "ent": "cortex", "cls": "road"}, {"id": 2454, "dom": "P", "ent": "cortex", "cls": "salmon"}, {"id": 2455, "dom": "P", "ent": "cortex", "cls": "shark"}, {"id": 2456, "dom": "P", "ent": "cortex", "cls": "sheep"}, {"id": 2457, "dom": "P", "ent": "cortex", "cls": "shook"}, {"id": 2458, "dom": "P", "ent": "cortex", "cls": "tank"}, {"id": 2459, "dom": "P", "ent": "cortex", "cls": "wheat"}, {"id": 2460, "dom": "P", "ent": "costa", "cls": "architecture"}, {"id": 2461, "dom": "P", "ent": "costa", "cls": "bacteria"}, {"id": 2462, "dom": "P", "ent": "costa", "cls": "beef"}, {"id": 2463, "dom": "P", "ent": "costa", "cls": "bridge"}, {"id": 2464, "dom": "P", "ent": "costa", "cls": "bud"}, {"id": 2465, "dom": "P", "ent": "costa", "cls": "cat"}, {"id": 2466, "dom": "P", "ent": "costa", "cls": "chapel"}, {"id": 2467, "dom": "P", "ent": "costa", "cls": "chick"}, {"id": 2468, "dom": "P", "ent": "costa", "cls": "clock"}, {"id": 2469, "dom": "P", "ent": "costa", "cls": "deer"}, {"id": 2470, "dom": "P", "ent": "costa", "cls": "dog"}, {"id": 2471, "dom": "P", "ent": "costa", "cls": "elephant"}, {"id": 2472, "dom": "P", "ent": "costa", "cls": "embassy"}, {"id": 2473, "dom": "P", "ent": "costa", "cls": "garage"}, {"id": 2474, "dom": "P", "ent": "costa", "cls": "highway"}, {"id": 2475, "dom": "P", "ent": "costa", "cls": "kidney"}, {"id": 2476, "dom": "P", "ent": "costa", "cls": "lamb"}, {"id": 2477, "dom": "P", "ent": "costa", "cls": "male"}, {"id": 2478, "dom": "P", "ent": "costa", "cls": "marijuana"}, {"id": 2479, "dom": "P", "ent": "costa", "cls": "puppy"}, {"id": 2480, "dom": "P", "ent": "costa", "cls": "receiver"}, {"id": 2481, "dom": "P", "ent": "costa", "cls": "reed"}, {"id": 2482, "dom": "P", "ent": "costa", "cls": "rifle"}, {"id": 2483, "dom": "P", "ent": "costa", "cls": "road"}, {"id": 2484, "dom": "P", "ent": "costa", "cls": "salmon"}, {"id": 2485, "dom": "P", "ent": "costa", "cls": "shark"}, {"id": 2486, "dom": "P", "ent": "costa", "cls": "sheep"}, {"id": 2487, "dom": "P", "ent": "costa", "cls": "shook"}, {"id": 2488, "dom": "P", "ent": "costa", "cls": "tank"}, {"id": 2489, "dom": "P", "ent": "costa", "cls": "wheat"}, {"id": 2490, "dom": "P", "ent": "course", "cls": "architecture"}, {"id": 2491, "dom": "P", "ent": "course", "cls": "bacteria"}, {"id": 2492, "dom": "P", "ent": "course", "cls": "beef"}, {"id": 2493, "dom": "P", "ent": "course", "cls": "bridge"}, {"id": 2494, "dom": "P", "ent": "course", "cls": "bud"}, {"id": 2495, "dom": "P", "ent": "course", "cls": "cat"}, {"id": 2496, "dom": "P", "ent": "course", "cls": "chapel"}, {"id": 2497, "dom": "P", "ent": "course", "cls": "chick"}, {"id": 2498, "dom": "P", "ent": "course", "cls": "clock"}, {"id": 2499, "dom": "P", "ent": "course", "cls": "deer"}, {"id": 2500, "dom": "P", "ent": "course", "cls": "dog"}, {"id": 2501, "dom": "P", "ent": "course", "cls": "elephant"}, {"id": 2502, "dom": "P", "ent": "course", "cls": "embassy"}, {"id": 2503, "dom": "P", "ent": "course", "cls": "garage"}, {"id": 2504, "dom": "P", "ent": "course", "cls": "highway"}, {"id": 2505, "dom": "P", "ent": "course", "cls": "kidney"}, {"id": 2506, "dom": "P", "ent": "course", "cls": "lamb"}, {"id": 2507, "dom": "P", "ent": "course", "cls": "male"}, {"id": 2508, "dom": "P", "ent": "course", "cls": "marijuana"}, {"id": 2509, "dom": "P", "ent": "course", "cls": "puppy"}, {"id": 2510, "dom": "P", "ent": "course", "cls": "receiver"}, {"id": 2511, "dom": "P", "ent": "course", "cls": "reed"}, {"id": 2512, "dom": "P", "ent": "course", "cls": "rifle"}, {"id": 2513, "dom": "P", "ent": "course", "cls": "road"}, {"id": 2514, "dom": "P", "ent": "course", "cls": "salmon"}, {"id": 2515, "dom": "P", "ent": "course", "cls": "shark"}, {"id": 2516, "dom": "P", "ent": "course", "cls": "sheep"}, {"id": 2517, "dom": "P", "ent": "course", "cls": "shook"}, {"id": 2518, "dom": "P", "ent": "course", "cls": "tank"}, {"id": 2519, "dom": "P", "ent": "course", "cls": "wheat"}, {"id": 2520, "dom": "P", "ent": "court", "cls": "architecture"}, {"id": 2521, "dom": "P", "ent": "court", "cls": "bacteria"}, {"id": 2522, "dom": "P", "ent": "court", "cls": "beef"}, {"id": 2523, "dom": "P", "ent": "court", "cls": "bridge"}, {"id": 2524, "dom": "P", "ent": "court", "cls": "bud"}, {"id": 2525, "dom": "P", "ent": "court", "cls": "cat"}, {"id": 2526, "dom": "P", "ent": "court", "cls": "chapel"}, {"id": 2527, "dom": "P", "ent": "court", "cls": "chick"}, {"id": 2528, "dom": "P", "ent": "court", "cls": "clock"}, {"id": 2529, "dom": "P", "ent": "court", "cls": "deer"}, {"id": 2530, "dom": "P", "ent": "court", "cls": "dog"}, {"id": 2531, "dom": "P", "ent": "court", "cls": "elephant"}, {"id": 2532, "dom": "P", "ent": "court", "cls": "embassy"}, {"id": 2533, "dom": "P", "ent": "court", "cls": "garage"}, {"id": 2534, "dom": "P", "ent": "court", "cls": "highway"}, {"id": 2535, "dom": "P", "ent": "court", "cls": "kidney"}, {"id": 2536, "dom": "P", "ent": "court", "cls": "lamb"}, {"id": 2537, "dom": "P", "ent": "court", "cls": "male"}, {"id": 2538, "dom": "P", "ent": "court", "cls": "marijuana"}, {"id": 2539, "dom": "P", "ent": "court", "cls": "puppy"}, {"id": 2540, "dom": "P", "ent": "court", "cls": "receiver"}, {"id": 2541, "dom": "P", "ent": "court", "cls": "reed"}, {"id": 2542, "dom": "P", "ent": "court", "cls": "rifle"}, {"id": 2543, "dom": "P", "ent": "court", "cls": "road"}, {"id": 2544, "dom": "P", "ent": "court", "cls": "salmon"}, {"id": 2545, "dom": "P", "ent": "court", "cls": "shark"}, {"id": 2546, "dom": "P", "ent": "court", "cls": "sheep"}, {"id": 2547, "dom": "P", "ent": "court", "cls": "shook"}, {"id": 2548, "dom": "P", "ent": "court", "cls": "tank"}, {"id": 2549, "dom": "P", "ent": "court", "cls": "wheat"}, {"id": 2550, "dom": "P", "ent": "courtyard", "cls": "architecture"}, {"id": 2551, "dom": "P", "ent": "courtyard", "cls": "bacteria"}, {"id": 2552, "dom": "P", "ent": "courtyard", "cls": "beef"}, {"id": 2553, "dom": "P", "ent": "courtyard", "cls": "bridge"}, {"id": 2554, "dom": "P", "ent": "courtyard", "cls": "bud"}, {"id": 2555, "dom": "P", "ent": "courtyard", "cls": "cat"}, {"id": 2556, "dom": "P", "ent": "courtyard", "cls": "chapel"}, {"id": 2557, "dom": "P", "ent": "courtyard", "cls": "chick"}, {"id": 2558, "dom": "P", "ent": "courtyard", "cls": "clock"}, {"id": 2559, "dom": "P", "ent": "courtyard", "cls": "deer"}, {"id": 2560, "dom": "P", "ent": "courtyard", "cls": "dog"}, {"id": 2561, "dom": "P", "ent": "courtyard", "cls": "elephant"}, {"id": 2562, "dom": "P", "ent": "courtyard", "cls": "embassy"}, {"id": 2563, "dom": "P", "ent": "courtyard", "cls": "garage"}, {"id": 2564, "dom": "P", "ent": "courtyard", "cls": "highway"}, {"id": 2565, "dom": "P", "ent": "courtyard", "cls": "kidney"}, {"id": 2566, "dom": "P", "ent": "courtyard", "cls": "lamb"}, {"id": 2567, "dom": "P", "ent": "courtyard", "cls": "male"}, {"id": 2568, "dom": "P", "ent": "courtyard", "cls": "marijuana"}, {"id": 2569, "dom": "P", "ent": "courtyard", "cls": "puppy"}, {"id": 2570, "dom": "P", "ent": "courtyard", "cls": "receiver"}, {"id": 2571, "dom": "P", "ent": "courtyard", "cls": "reed"}, {"id": 2572, "dom": "P", "ent": "courtyard", "cls": "rifle"}, {"id": 2573, "dom": "P", "ent": "courtyard", "cls": "road"}, {"id": 2574, "dom": "P", "ent": "courtyard", "cls": "salmon"}, {"id": 2575, "dom": "P", "ent": "courtyard", "cls": "shark"}, {"id": 2576, "dom": "P", "ent": "courtyard", "cls": "sheep"}, {"id": 2577, "dom": "P", "ent": "courtyard", "cls": "shook"}, {"id": 2578, "dom": "P", "ent": "courtyard", "cls": "tank"}, {"id": 2579, "dom": "P", "ent": "courtyard", "cls": "wheat"}, {"id": 2580, "dom": "P", "ent": "cradle", "cls": "architecture"}, {"id": 2581, "dom": "P", "ent": "cradle", "cls": "bacteria"}, {"id": 2582, "dom": "P", "ent": "cradle", "cls": "beef"}, {"id": 2583, "dom": "P", "ent": "cradle", "cls": "bridge"}, {"id": 2584, "dom": "P", "ent": "cradle", "cls": "bud"}, {"id": 2585, "dom": "P", "ent": "cradle", "cls": "cat"}, {"id": 2586, "dom": "P", "ent": "cradle", "cls": "chapel"}, {"id": 2587, "dom": "P", "ent": "cradle", "cls": "chick"}, {"id": 2588, "dom": "P", "ent": "cradle", "cls": "clock"}, {"id": 2589, "dom": "P", "ent": "cradle", "cls": "deer"}, {"id": 2590, "dom": "P", "ent": "cradle", "cls": "dog"}, {"id": 2591, "dom": "P", "ent": "cradle", "cls": "elephant"}, {"id": 2592, "dom": "P", "ent": "cradle", "cls": "embassy"}, {"id": 2593, "dom": "P", "ent": "cradle", "cls": "garage"}, {"id": 2594, "dom": "P", "ent": "cradle", "cls": "highway"}, {"id": 2595, "dom": "P", "ent": "cradle", "cls": "kidney"}, {"id": 2596, "dom": "P", "ent": "cradle", "cls": "lamb"}, {"id": 2597, "dom": "P", "ent": "cradle", "cls": "male"}, {"id": 2598, "dom": "P", "ent": "cradle", "cls": "marijuana"}, {"id": 2599, "dom": "P", "ent": "cradle", "cls": "puppy"}, {"id": 2600, "dom": "P", "ent": "cradle", "cls": "receiver"}, {"id": 2601, "dom": "P", "ent": "cradle", "cls": "reed"}, {"id": 2602, "dom": "P", "ent": "cradle", "cls": "rifle"}, {"id": 2603, "dom": "P", "ent": "cradle", "cls": "road"}, {"id": 2604, "dom": "P", "ent": "cradle", "cls": "salmon"}, {"id": 2605, "dom": "P", "ent": "cradle", "cls": "shark"}, {"id": 2606, "dom": "P", "ent": "cradle", "cls": "sheep"}, {"id": 2607, "dom": "P", "ent": "cradle", "cls": "shook"}, {"id": 2608, "dom": "P", "ent": "cradle", "cls": "tank"}, {"id": 2609, "dom": "P", "ent": "cradle", "cls": "wheat"}, {"id": 2610, "dom": "P", "ent": "crest", "cls": "architecture"}, {"id": 2611, "dom": "P", "ent": "crest", "cls": "bacteria"}, {"id": 2612, "dom": "P", "ent": "crest", "cls": "beef"}, {"id": 2613, "dom": "P", "ent": "crest", "cls": "bridge"}, {"id": 2614, "dom": "P", "ent": "crest", "cls": "bud"}, {"id": 2615, "dom": "P", "ent": "crest", "cls": "cat"}, {"id": 2616, "dom": "P", "ent": "crest", "cls": "chapel"}, {"id": 2617, "dom": "P", "ent": "crest", "cls": "chick"}, {"id": 2618, "dom": "P", "ent": "crest", "cls": "clock"}, {"id": 2619, "dom": "P", "ent": "crest", "cls": "deer"}, {"id": 2620, "dom": "P", "ent": "crest", "cls": "dog"}, {"id": 2621, "dom": "P", "ent": "crest", "cls": "elephant"}, {"id": 2622, "dom": "P", "ent": "crest", "cls": "embassy"}, {"id": 2623, "dom": "P", "ent": "crest", "cls": "garage"}, {"id": 2624, "dom": "P", "ent": "crest", "cls": "highway"}, {"id": 2625, "dom": "P", "ent": "crest", "cls": "kidney"}, {"id": 2626, "dom": "P", "ent": "crest", "cls": "lamb"}, {"id": 2627, "dom": "P", "ent": "crest", "cls": "male"}, {"id": 2628, "dom": "P", "ent": "crest", "cls": "marijuana"}, {"id": 2629, "dom": "P", "ent": "crest", "cls": "puppy"}, {"id": 2630, "dom": "P", "ent": "crest", "cls": "receiver"}, {"id": 2631, "dom": "P", "ent": "crest", "cls": "reed"}, {"id": 2632, "dom": "P", "ent": "crest", "cls": "rifle"}, {"id": 2633, "dom": "P", "ent": "crest", "cls": "road"}, {"id": 2634, "dom": "P", "ent": "crest", "cls": "salmon"}, {"id": 2635, "dom": "P", "ent": "crest", "cls": "shark"}, {"id": 2636, "dom": "P", "ent": "crest", "cls": "sheep"}, {"id": 2637, "dom": "P", "ent": "crest", "cls": "shook"}, {"id": 2638, "dom": "P", "ent": "crest", "cls": "tank"}, {"id": 2639, "dom": "P", "ent": "crest", "cls": "wheat"}, {"id": 2640, "dom": "P", "ent": "crossing", "cls": "architecture"}, {"id": 2641, "dom": "P", "ent": "crossing", "cls": "bacteria"}, {"id": 2642, "dom": "P", "ent": "crossing", "cls": "beef"}, {"id": 2643, "dom": "P", "ent": "crossing", "cls": "bridge"}, {"id": 2644, "dom": "P", "ent": "crossing", "cls": "bud"}, {"id": 2645, "dom": "P", "ent": "crossing", "cls": "cat"}, {"id": 2646, "dom": "P", "ent": "crossing", "cls": "chapel"}, {"id": 2647, "dom": "P", "ent": "crossing", "cls": "chick"}, {"id": 2648, "dom": "P", "ent": "crossing", "cls": "clock"}, {"id": 2649, "dom": "P", "ent": "crossing", "cls": "deer"}, {"id": 2650, "dom": "P", "ent": "crossing", "cls": "dog"}, {"id": 2651, "dom": "P", "ent": "crossing", "cls": "elephant"}, {"id": 2652, "dom": "P", "ent": "crossing", "cls": "embassy"}, {"id": 2653, "dom": "P", "ent": "crossing", "cls": "garage"}, {"id": 2654, "dom": "P", "ent": "crossing", "cls": "highway"}, {"id": 2655, "dom": "P", "ent": "crossing", "cls": "kidney"}, {"id": 2656, "dom": "P", "ent": "crossing", "cls": "lamb"}, {"id": 2657, "dom": "P", "ent": "crossing", "cls": "male"}, {"id": 2658, "dom": "P", "ent": "crossing", "cls": "marijuana"}, {"id": 2659, "dom": "P", "ent": "crossing", "cls": "puppy"}, {"id": 2660, "dom": "P", "ent": "crossing", "cls": "receiver"}, {"id": 2661, "dom": "P", "ent": "crossing", "cls": "reed"}, {"id": 2662, "dom": "P", "ent": "crossing", "cls": "rifle"}, {"id": 2663, "dom": "P", "ent": "crossing", "cls": "road"}, {"id": 2664, "dom": "P", "ent": "crossing", "cls": "salmon"}, {"id": 2665, "dom": "P", "ent": "crossing", "cls": "shark"}, {"id": 2666, "dom": "P", "ent": "crossing", "cls": "sheep"}, {"id": 2667, "dom": "P", "ent": "crossing", "cls": "shook"}, {"id": 2668, "dom": "P", "ent": "crossing", "cls": "tank"}, {"id": 2669, "dom": "P", "ent": "crossing", "cls": "wheat"}, {"id": 2670, "dom": "P", "ent": "crown", "cls": "architecture"}, {"id": 2671, "dom": "P", "ent": "crown", "cls": "bacteria"}, {"id": 2672, "dom": "P", "ent": "crown", "cls": "beef"}, {"id": 2673, "dom": "P", "ent": "crown", "cls": "bridge"}, {"id": 2674, "dom": "P", "ent": "crown", "cls": "bud"}, {"id": 2675, "dom": "P", "ent": "crown", "cls": "cat"}, {"id": 2676, "dom": "P", "ent": "crown", "cls": "chapel"}, {"id": 2677, "dom": "P", "ent": "crown", "cls": "chick"}, {"id": 2678, "dom": "P", "ent": "crown", "cls": "clock"}, {"id": 2679, "dom": "P", "ent": "crown", "cls": "deer"}, {"id": 2680, "dom": "P", "ent": "crown", "cls": "dog"}, {"id": 2681, "dom": "P", "ent": "crown", "cls": "elephant"}, {"id": 2682, "dom": "P", "ent": "crown", "cls": "embassy"}, {"id": 2683, "dom": "P", "ent": "crown", "cls": "garage"}, {"id": 2684, "dom": "P", "ent": "crown", "cls": "highway"}, {"id": 2685, "dom": "P", "ent": "crown", "cls": "kidney"}, {"id": 2686, "dom": "P", "ent": "crown", "cls": "lamb"}, {"id": 2687, "dom": "P", "ent": "crown", "cls": "male"}, {"id": 2688, "dom": "P", "ent": "crown", "cls": "marijuana"}, {"id": 2689, "dom": "P", "ent": "crown", "cls": "puppy"}, {"id": 2690, "dom": "P", "ent": "crown", "cls": "receiver"}, {"id": 2691, "dom": "P", "ent": "crown", "cls": "reed"}, {"id": 2692, "dom": "P", "ent": "crown", "cls": "rifle"}, {"id": 2693, "dom": "P", "ent": "crown", "cls": "road"}, {"id": 2694, "dom": "P", "ent": "crown", "cls": "salmon"}, {"id": 2695, "dom": "P", "ent": "crown", "cls": "shark"}, {"id": 2696, "dom": "P", "ent": "crown", "cls": "sheep"}, {"id": 2697, "dom": "P", "ent": "crown", "cls": "shook"}, {"id": 2698, "dom": "P", "ent": "crown", "cls": "tank"}, {"id": 2699, "dom": "P", "ent": "crown", "cls": "wheat"}, {"id": 2700, "dom": "P", "ent": "curve", "cls": "architecture"}, {"id": 2701, "dom": "P", "ent": "curve", "cls": "bacteria"}, {"id": 2702, "dom": "P", "ent": "curve", "cls": "beef"}, {"id": 2703, "dom": "P", "ent": "curve", "cls": "bridge"}, {"id": 2704, "dom": "P", "ent": "curve", "cls": "bud"}, {"id": 2705, "dom": "P", "ent": "curve", "cls": "cat"}, {"id": 2706, "dom": "P", "ent": "curve", "cls": "chapel"}, {"id": 2707, "dom": "P", "ent": "curve", "cls": "chick"}, {"id": 2708, "dom": "P", "ent": "curve", "cls": "clock"}, {"id": 2709, "dom": "P", "ent": "curve", "cls": "deer"}, {"id": 2710, "dom": "P", "ent": "curve", "cls": "dog"}, {"id": 2711, "dom": "P", "ent": "curve", "cls": "elephant"}, {"id": 2712, "dom": "P", "ent": "curve", "cls": "embassy"}, {"id": 2713, "dom": "P", "ent": "curve", "cls": "garage"}, {"id": 2714, "dom": "P", "ent": "curve", "cls": "highway"}, {"id": 2715, "dom": "P", "ent": "curve", "cls": "kidney"}, {"id": 2716, "dom": "P", "ent": "curve", "cls": "lamb"}, {"id": 2717, "dom": "P", "ent": "curve", "cls": "male"}, {"id": 2718, "dom": "P", "ent": "curve", "cls": "marijuana"}, {"id": 2719, "dom": "P", "ent": "curve", "cls": "puppy"}, {"id": 2720, "dom": "P", "ent": "curve", "cls": "receiver"}, {"id": 2721, "dom": "P", "ent": "curve", "cls": "reed"}, {"id": 2722, "dom": "P", "ent": "curve", "cls": "rifle"}, {"id": 2723, "dom": "P", "ent": "curve", "cls": "road"}, {"id": 2724, "dom": "P", "ent": "curve", "cls": "salmon"}, {"id": 2725, "dom": "P", "ent": "curve", "cls": "shark"}, {"id": 2726, "dom": "P", "ent": "curve", "cls": "sheep"}, {"id": 2727, "dom": "P", "ent": "curve", "cls": "shook"}, {"id": 2728, "dom": "P", "ent": "curve", "cls": "tank"}, {"id": 2729, "dom": "P", "ent": "curve", "cls": "wheat"}, {"id": 2730, "dom": "P", "ent": "dashboard", "cls": "architecture"}, {"id": 2731, "dom": "P", "ent": "dashboard", "cls": "bacteria"}, {"id": 2732, "dom": "P", "ent": "dashboard", "cls": "beef"}, {"id": 2733, "dom": "P", "ent": "dashboard", "cls": "bridge"}, {"id": 2734, "dom": "P", "ent": "dashboard", "cls": "bud"}, {"id": 2735, "dom": "P", "ent": "dashboard", "cls": "cat"}, {"id": 2736, "dom": "P", "ent": "dashboard", "cls": "chapel"}, {"id": 2737, "dom": "P", "ent": "dashboard", "cls": "chick"}, {"id": 2738, "dom": "P", "ent": "dashboard", "cls": "clock"}, {"id": 2739, "dom": "P", "ent": "dashboard", "cls": "deer"}, {"id": 2740, "dom": "P", "ent": "dashboard", "cls": "dog"}, {"id": 2741, "dom": "P", "ent": "dashboard", "cls": "elephant"}, {"id": 2742, "dom": "P", "ent": "dashboard", "cls": "embassy"}, {"id": 2743, "dom": "P", "ent": "dashboard", "cls": "garage"}, {"id": 2744, "dom": "P", "ent": "dashboard", "cls": "highway"}, {"id": 2745, "dom": "P", "ent": "dashboard", "cls": "kidney"}, {"id": 2746, "dom": "P", "ent": "dashboard", "cls": "lamb"}, {"id": 2747, "dom": "P", "ent": "dashboard", "cls": "male"}, {"id": 2748, "dom": "P", "ent": "dashboard", "cls": "marijuana"}, {"id": 2749, "dom": "P", "ent": "dashboard", "cls": "puppy"}, {"id": 2750, "dom": "P", "ent": "dashboard", "cls": "receiver"}, {"id": 2751, "dom": "P", "ent": "dashboard", "cls": "reed"}, {"id": 2752, "dom": "P", "ent": "dashboard", "cls": "rifle"}, {"id": 2753, "dom": "P", "ent": "dashboard", "cls": "road"}, {"id": 2754, "dom": "P", "ent": "dashboard", "cls": "salmon"}, {"id": 2755, "dom": "P", "ent": "dashboard", "cls": "shark"}, {"id": 2756, "dom": "P", "ent": "dashboard", "cls": "sheep"}, {"id": 2757, "dom": "P", "ent": "dashboard", "cls": "shook"}, {"id": 2758, "dom": "P", "ent": "dashboard", "cls": "tank"}, {"id": 2759, "dom": "P", "ent": "dashboard", "cls": "wheat"}, {"id": 2760, "dom": "P", "ent": "dial", "cls": "architecture"}, {"id": 2761, "dom": "P", "ent": "dial", "cls": "bacteria"}, {"id": 2762, "dom": "P", "ent": "dial", "cls": "beef"}, {"id": 2763, "dom": "P", "ent": "dial", "cls": "bridge"}, {"id": 2764, "dom": "P", "ent": "dial", "cls": "bud"}, {"id": 2765, "dom": "P", "ent": "dial", "cls": "cat"}, {"id": 2766, "dom": "P", "ent": "dial", "cls": "chapel"}, {"id": 2767, "dom": "P", "ent": "dial", "cls": "chick"}, {"id": 2768, "dom": "P", "ent": "dial", "cls": "clock"}, {"id": 2769, "dom": "P", "ent": "dial", "cls": "deer"}, {"id": 2770, "dom": "P", "ent": "dial", "cls": "dog"}, {"id": 2771, "dom": "P", "ent": "dial", "cls": "elephant"}, {"id": 2772, "dom": "P", "ent": "dial", "cls": "embassy"}, {"id": 2773, "dom": "P", "ent": "dial", "cls": "garage"}, {"id": 2774, "dom": "P", "ent": "dial", "cls": "highway"}, {"id": 2775, "dom": "P", "ent": "dial", "cls": "kidney"}, {"id": 2776, "dom": "P", "ent": "dial", "cls": "lamb"}, {"id": 2777, "dom": "P", "ent": "dial", "cls": "male"}, {"id": 2778, "dom": "P", "ent": "dial", "cls": "marijuana"}, {"id": 2779, "dom": "P", "ent": "dial", "cls": "puppy"}, {"id": 2780, "dom": "P", "ent": "dial", "cls": "receiver"}, {"id": 2781, "dom": "P", "ent": "dial", "cls": "reed"}, {"id": 2782, "dom": "P", "ent": "dial", "cls": "rifle"}, {"id": 2783, "dom": "P", "ent": "dial", "cls": "road"}, {"id": 2784, "dom": "P", "ent": "dial", "cls": "salmon"}, {"id": 2785, "dom": "P", "ent": "dial", "cls": "shark"}, {"id": 2786, "dom": "P", "ent": "dial", "cls": "sheep"}, {"id": 2787, "dom": "P", "ent": "dial", "cls": "shook"}, {"id": 2788, "dom": "P", "ent": "dial", "cls": "tank"}, {"id": 2789, "dom": "P", "ent": "dial", "cls": "wheat"}, {"id": 2790, "dom": "P", "ent": "dice", "cls": "architecture"}, {"id": 2791, "dom": "P", "ent": "dice", "cls": "bacteria"}, {"id": 2792, "dom": "P", "ent": "dice", "cls": "beef"}, {"id": 2793, "dom": "P", "ent": "dice", "cls": "bridge"}, {"id": 2794, "dom": "P", "ent": "dice", "cls": "bud"}, {"id": 2795, "dom": "P", "ent": "dice", "cls": "cat"}, {"id": 2796, "dom": "P", "ent": "dice", "cls": "chapel"}, {"id": 2797, "dom": "P", "ent": "dice", "cls": "chick"}, {"id": 2798, "dom": "P", "ent": "dice", "cls": "clock"}, {"id": 2799, "dom": "P", "ent": "dice", "cls": "deer"}, {"id": 2800, "dom": "P", "ent": "dice", "cls": "dog"}, {"id": 2801, "dom": "P", "ent": "dice", "cls": "elephant"}, {"id": 2802, "dom": "P", "ent": "dice", "cls": "embassy"}, {"id": 2803, "dom": "P", "ent": "dice", "cls": "garage"}, {"id": 2804, "dom": "P", "ent": "dice", "cls": "highway"}, {"id": 2805, "dom": "P", "ent": "dice", "cls": "kidney"}, {"id": 2806, "dom": "P", "ent": "dice", "cls": "lamb"}, {"id": 2807, "dom": "P", "ent": "dice", "cls": "male"}, {"id": 2808, "dom": "P", "ent": "dice", "cls": "marijuana"}, {"id": 2809, "dom": "P", "ent": "dice", "cls": "puppy"}, {"id": 2810, "dom": "P", "ent": "dice", "cls": "receiver"}, {"id": 2811, "dom": "P", "ent": "dice", "cls": "reed"}, {"id": 2812, "dom": "P", "ent": "dice", "cls": "rifle"}, {"id": 2813, "dom": "P", "ent": "dice", "cls": "road"}, {"id": 2814, "dom": "P", "ent": "dice", "cls": "salmon"}, {"id": 2815, "dom": "P", "ent": "dice", "cls": "shark"}, {"id": 2816, "dom": "P", "ent": "dice", "cls": "sheep"}, {"id": 2817, "dom": "P", "ent": "dice", "cls": "shook"}, {"id": 2818, "dom": "P", "ent": "dice", "cls": "tank"}, {"id": 2819, "dom": "P", "ent": "dice", "cls": "wheat"}, {"id": 2820, "dom": "P", "ent": "differential", "cls": "architecture"}, {"id": 2821, "dom": "P", "ent": "differential", "cls": "bacteria"}, {"id": 2822, "dom": "P", "ent": "differential", "cls": "beef"}, {"id": 2823, "dom": "P", "ent": "differential", "cls": "bridge"}, {"id": 2824, "dom": "P", "ent": "differential", "cls": "bud"}, {"id": 2825, "dom": "P", "ent": "differential", "cls": "cat"}, {"id": 2826, "dom": "P", "ent": "differential", "cls": "chapel"}, {"id": 2827, "dom": "P", "ent": "differential", "cls": "chick"}, {"id": 2828, "dom": "P", "ent": "differential", "cls": "clock"}, {"id": 2829, "dom": "P", "ent": "differential", "cls": "deer"}, {"id": 2830, "dom": "P", "ent": "differential", "cls": "dog"}, {"id": 2831, "dom": "P", "ent": "differential", "cls": "elephant"}, {"id": 2832, "dom": "P", "ent": "differential", "cls": "embassy"}, {"id": 2833, "dom": "P", "ent": "differential", "cls": "garage"}, {"id": 2834, "dom": "P", "ent": "differential", "cls": "highway"}, {"id": 2835, "dom": "P", "ent": "differential", "cls": "kidney"}, {"id": 2836, "dom": "P", "ent": "differential", "cls": "lamb"}, {"id": 2837, "dom": "P", "ent": "differential", "cls": "male"}, {"id": 2838, "dom": "P", "ent": "differential", "cls": "marijuana"}, {"id": 2839, "dom": "P", "ent": "differential", "cls": "puppy"}, {"id": 2840, "dom": "P", "ent": "differential", "cls": "receiver"}, {"id": 2841, "dom": "P", "ent": "differential", "cls": "reed"}, {"id": 2842, "dom": "P", "ent": "differential", "cls": "rifle"}, {"id": 2843, "dom": "P", "ent": "differential", "cls": "road"}, {"id": 2844, "dom": "P", "ent": "differential", "cls": "salmon"}, {"id": 2845, "dom": "P", "ent": "differential", "cls": "shark"}, {"id": 2846, "dom": "P", "ent": "differential", "cls": "sheep"}, {"id": 2847, "dom": "P", "ent": "differential", "cls": "shook"}, {"id": 2848, "dom": "P", "ent": "differential", "cls": "tank"}, {"id": 2849, "dom": "P", "ent": "differential", "cls": "wheat"}, {"id": 2850, "dom": "P", "ent": "digit", "cls": "architecture"}, {"id": 2851, "dom": "P", "ent": "digit", "cls": "bacteria"}, {"id": 2852, "dom": "P", "ent": "digit", "cls": "beef"}, {"id": 2853, "dom": "P", "ent": "digit", "cls": "bridge"}, {"id": 2854, "dom": "P", "ent": "digit", "cls": "bud"}, {"id": 2855, "dom": "P", "ent": "digit", "cls": "cat"}, {"id": 2856, "dom": "P", "ent": "digit", "cls": "chapel"}, {"id": 2857, "dom": "P", "ent": "digit", "cls": "chick"}, {"id": 2858, "dom": "P", "ent": "digit", "cls": "clock"}, {"id": 2859, "dom": "P", "ent": "digit", "cls": "deer"}, {"id": 2860, "dom": "P", "ent": "digit", "cls": "dog"}, {"id": 2861, "dom": "P", "ent": "digit", "cls": "elephant"}, {"id": 2862, "dom": "P", "ent": "digit", "cls": "embassy"}, {"id": 2863, "dom": "P", "ent": "digit", "cls": "garage"}, {"id": 2864, "dom": "P", "ent": "digit", "cls": "highway"}, {"id": 2865, "dom": "P", "ent": "digit", "cls": "kidney"}, {"id": 2866, "dom": "P", "ent": "digit", "cls": "lamb"}, {"id": 2867, "dom": "P", "ent": "digit", "cls": "male"}, {"id": 2868, "dom": "P", "ent": "digit", "cls": "marijuana"}, {"id": 2869, "dom": "P", "ent": "digit", "cls": "puppy"}, {"id": 2870, "dom": "P", "ent": "digit", "cls": "receiver"}, {"id": 2871, "dom": "P", "ent": "digit", "cls": "reed"}, {"id": 2872, "dom": "P", "ent": "digit", "cls": "rifle"}, {"id": 2873, "dom": "P", "ent": "digit", "cls": "road"}, {"id": 2874, "dom": "P", "ent": "digit", "cls": "salmon"}, {"id": 2875, "dom": "P", "ent": "digit", "cls": "shark"}, {"id": 2876, "dom": "P", "ent": "digit", "cls": "sheep"}, {"id": 2877, "dom": "P", "ent": "digit", "cls": "shook"}, {"id": 2878, "dom": "P", "ent": "digit", "cls": "tank"}, {"id": 2879, "dom": "P", "ent": "digit", "cls": "wheat"}, {"id": 2880, "dom": "P", "ent": "dock", "cls": "architecture"}, {"id": 2881, "dom": "P", "ent": "dock", "cls": "bacteria"}, {"id": 2882, "dom": "P", "ent": "dock", "cls": "beef"}, {"id": 2883, "dom": "P", "ent": "dock", "cls": "bridge"}, {"id": 2884, "dom": "P", "ent": "dock", "cls": "bud"}, {"id": 2885, "dom": "P", "ent": "dock", "cls": "cat"}, {"id": 2886, "dom": "P", "ent": "dock", "cls": "chapel"}, {"id": 2887, "dom": "P", "ent": "dock", "cls": "chick"}, {"id": 2888, "dom": "P", "ent": "dock", "cls": "clock"}, {"id": 2889, "dom": "P", "ent": "dock", "cls": "deer"}, {"id": 2890, "dom": "P", "ent": "dock", "cls": "dog"}, {"id": 2891, "dom": "P", "ent": "dock", "cls": "elephant"}, {"id": 2892, "dom": "P", "ent": "dock", "cls": "embassy"}, {"id": 2893, "dom": "P", "ent": "dock", "cls": "garage"}, {"id": 2894, "dom": "P", "ent": "dock", "cls": "highway"}, {"id": 2895, "dom": "P", "ent": "dock", "cls": "kidney"}, {"id": 2896, "dom": "P", "ent": "dock", "cls": "lamb"}, {"id": 2897, "dom": "P", "ent": "dock", "cls": "male"}, {"id": 2898, "dom": "P", "ent": "dock", "cls": "marijuana"}, {"id": 2899, "dom": "P", "ent": "dock", "cls": "puppy"}, {"id": 2900, "dom": "P", "ent": "dock", "cls": "receiver"}, {"id": 2901, "dom": "P", "ent": "dock", "cls": "reed"}, {"id": 2902, "dom": "P", "ent": "dock", "cls": "rifle"}, {"id": 2903, "dom": "P", "ent": "dock", "cls": "road"}, {"id": 2904, "dom": "P", "ent": "dock", "cls": "salmon"}, {"id": 2905, "dom": "P", "ent": "dock", "cls": "shark"}, {"id": 2906, "dom": "P", "ent": "dock", "cls": "sheep"}, {"id": 2907, "dom": "P", "ent": "dock", "cls": "shook"}, {"id": 2908, "dom": "P", "ent": "dock", "cls": "tank"}, {"id": 2909, "dom": "P", "ent": "dock", "cls": "wheat"}, {"id": 2910, "dom": "P", "ent": "door", "cls": "architecture"}, {"id": 2911, "dom": "P", "ent": "door", "cls": "bacteria"}, {"id": 2912, "dom": "P", "ent": "door", "cls": "beef"}, {"id": 2913, "dom": "P", "ent": "door", "cls": "bridge"}, {"id": 2914, "dom": "P", "ent": "door", "cls": "bud"}, {"id": 2915, "dom": "P", "ent": "door", "cls": "cat"}, {"id": 2916, "dom": "P", "ent": "door", "cls": "chapel"}, {"id": 2917, "dom": "P", "ent": "door", "cls": "chick"}, {"id": 2918, "dom": "P", "ent": "door", "cls": "clock"}, {"id": 2919, "dom": "P", "ent": "door", "cls": "deer"}, {"id": 2920, "dom": "P", "ent": "door", "cls": "dog"}, {"id": 2921, "dom": "P", "ent": "door", "cls": "elephant"}, {"id": 2922, "dom": "P", "ent": "door", "cls": "embassy"}, {"id": 2923, "dom": "P", "ent": "door", "cls": "garage"}, {"id": 2924, "dom": "P", "ent": "door", "cls": "highway"}, {"id": 2925, "dom": "P", "ent": "door", "cls": "kidney"}, {"id": 2926, "dom": "P", "ent": "door", "cls": "lamb"}, {"id": 2927, "dom": "P", "ent": "door", "cls": "male"}, {"id": 2928, "dom": "P", "ent": "door", "cls": "marijuana"}, {"id": 2929, "dom": "P", "ent": "door", "cls": "puppy"}, {"id": 2930, "dom": "P", "ent": "door", "cls": "receiver"}, {"id": 2931, "dom": "P", "ent": "door", "cls": "reed"}, {"id": 2932, "dom": "P", "ent": "door", "cls": "rifle"}, {"id": 2933, "dom": "P", "ent": "door", "cls": "road"}, {"id": 2934, "dom": "P", "ent": "door", "cls": "salmon"}, {"id": 2935, "dom": "P", "ent": "door", "cls": "shark"}, {"id": 2936, "dom": "P", "ent": "door", "cls": "sheep"}, {"id": 2937, "dom": "P", "ent": "door", "cls": "shook"}, {"id": 2938, "dom": "P", "ent": "door", "cls": "tank"}, {"id": 2939, "dom": "P", "ent": "door", "cls": "wheat"}, {"id": 2940, "dom": "P", "ent": "doorstep", "cls": "architecture"}, {"id": 2941, "dom": "P", "ent": "doorstep", "cls": "bacteria"}, {"id": 2942, "dom": "P", "ent": "doorstep", "cls": "beef"}, {"id": 2943, "dom": "P", "ent": "doorstep", "cls": "bridge"}, {"id": 2944, "dom": "P", "ent": "doorstep", "cls": "bud"}, {"id": 2945, "dom": "P", "ent": "doorstep", "cls": "cat"}, {"id": 2946, "dom": "P", "ent": "doorstep", "cls": "chapel"}, {"id": 2947, "dom": "P", "ent": "doorstep", "cls": "chick"}, {"id": 2948, "dom": "P", "ent": "doorstep", "cls": "clock"}, {"id": 2949, "dom": "P", "ent": "doorstep", "cls": "deer"}, {"id": 2950, "dom": "P", "ent": "doorstep", "cls": "dog"}, {"id": 2951, "dom": "P", "ent": "doorstep", "cls": "elephant"}, {"id": 2952, "dom": "P", "ent": "doorstep", "cls": "embassy"}, {"id": 2953, "dom": "P", "ent": "doorstep", "cls": "garage"}, {"id": 2954, "dom": "P", "ent": "doorstep", "cls": "highway"}, {"id": 2955, "dom": "P", "ent": "doorstep", "cls": "kidney"}, {"id": 2956, "dom": "P", "ent": "doorstep", "cls": "lamb"}, {"id": 2957, "dom": "P", "ent": "doorstep", "cls": "male"}, {"id": 2958, "dom": "P", "ent": "doorstep", "cls": "marijuana"}, {"id": 2959, "dom": "P", "ent": "doorstep", "cls": "puppy"}, {"id": 2960, "dom": "P", "ent": "doorstep", "cls": "receiver"}, {"id": 2961, "dom": "P", "ent": "doorstep", "cls": "reed"}, {"id": 2962, "dom": "P", "ent": "doorstep", "cls": "rifle"}, {"id": 2963, "dom": "P", "ent": "doorstep", "cls": "road"}, {"id": 2964, "dom": "P", "ent": "doorstep", "cls": "salmon"}, {"id": 2965, "dom": "P", "ent": "doorstep", "cls": "shark"}, {"id": 2966, "dom": "P", "ent": "doorstep", "cls": "sheep"}, {"id": 2967, "dom": "P", "ent": "doorstep", "cls": "shook"}, {"id": 2968, "dom": "P", "ent": "doorstep", "cls": "tank"}, {"id": 2969, "dom": "P", "ent": "doorstep", "cls": "wheat"}, {"id": 2970, "dom": "P", "ent": "doorway", "cls": "architecture"}, {"id": 2971, "dom": "P", "ent": "doorway", "cls": "bacteria"}, {"id": 2972, "dom": "P", "ent": "doorway", "cls": "beef"}, {"id": 2973, "dom": "P", "ent": "doorway", "cls": "bridge"}, {"id": 2974, "dom": "P", "ent": "doorway", "cls": "bud"}, {"id": 2975, "dom": "P", "ent": "doorway", "cls": "cat"}, {"id": 2976, "dom": "P", "ent": "doorway", "cls": "chapel"}, {"id": 2977, "dom": "P", "ent": "doorway", "cls": "chick"}, {"id": 2978, "dom": "P", "ent": "doorway", "cls": "clock"}, {"id": 2979, "dom": "P", "ent": "doorway", "cls": "deer"}, {"id": 2980, "dom": "P", "ent": "doorway", "cls": "dog"}, {"id": 2981, "dom": "P", "ent": "doorway", "cls": "elephant"}, {"id": 2982, "dom": "P", "ent": "doorway", "cls": "embassy"}, {"id": 2983, "dom": "P", "ent": "doorway", "cls": "garage"}, {"id": 2984, "dom": "P", "ent": "doorway", "cls": "highway"}, {"id": 2985, "dom": "P", "ent": "doorway", "cls": "kidney"}, {"id": 2986, "dom": "P", "ent": "doorway", "cls": "lamb"}, {"id": 2987, "dom": "P", "ent": "doorway", "cls": "male"}, {"id": 2988, "dom": "P", "ent": "doorway", "cls": "marijuana"}, {"id": 2989, "dom": "P", "ent": "doorway", "cls": "puppy"}, {"id": 2990, "dom": "P", "ent": "doorway", "cls": "receiver"}, {"id": 2991, "dom": "P", "ent": "doorway", "cls": "reed"}, {"id": 2992, "dom": "P", "ent": "doorway", "cls": "rifle"}, {"id": 2993, "dom": "P", "ent": "doorway", "cls": "road"}, {"id": 2994, "dom": "P", "ent": "doorway", "cls": "salmon"}, {"id": 2995, "dom": "P", "ent": "doorway", "cls": "shark"}, {"id": 2996, "dom": "P", "ent": "doorway", "cls": "sheep"}, {"id": 2997, "dom": "P", "ent": "doorway", "cls": "shook"}, {"id": 2998, "dom": "P", "ent": "doorway", "cls": "tank"}, {"id": 2999, "dom": "P", "ent": "doorway", "cls": "wheat"}, {"id": 3000, "dom": "P", "ent": "driver", "cls": "architecture"}, {"id": 3001, "dom": "P", "ent": "driver", "cls": "bacteria"}, {"id": 3002, "dom": "P", "ent": "driver", "cls": "beef"}, {"id": 3003, "dom": "P", "ent": "driver", "cls": "bridge"}, {"id": 3004, "dom": "P", "ent": "driver", "cls": "bud"}, {"id": 3005, "dom": "P", "ent": "driver", "cls": "cat"}, {"id": 3006, "dom": "P", "ent": "driver", "cls": "chapel"}, {"id": 3007, "dom": "P", "ent": "driver", "cls": "chick"}, {"id": 3008, "dom": "P", "ent": "driver", "cls": "clock"}, {"id": 3009, "dom": "P", "ent": "driver", "cls": "deer"}, {"id": 3010, "dom": "P", "ent": "driver", "cls": "dog"}, {"id": 3011, "dom": "P", "ent": "driver", "cls": "elephant"}, {"id": 3012, "dom": "P", "ent": "driver", "cls": "embassy"}, {"id": 3013, "dom": "P", "ent": "driver", "cls": "garage"}, {"id": 3014, "dom": "P", "ent": "driver", "cls": "highway"}, {"id": 3015, "dom": "P", "ent": "driver", "cls": "kidney"}, {"id": 3016, "dom": "P", "ent": "driver", "cls": "lamb"}, {"id": 3017, "dom": "P", "ent": "driver", "cls": "male"}, {"id": 3018, "dom": "P", "ent": "driver", "cls": "marijuana"}, {"id": 3019, "dom": "P", "ent": "driver", "cls": "puppy"}, {"id": 3020, "dom": "P", "ent": "driver", "cls": "receiver"}, {"id": 3021, "dom": "P", "ent": "driver", "cls": "reed"}, {"id": 3022, "dom": "P", "ent": "driver", "cls": "rifle"}, {"id": 3023, "dom": "P", "ent": "driver", "cls": "road"}, {"id": 3024, "dom": "P", "ent": "driver", "cls": "salmon"}, {"id": 3025, "dom": "P", "ent": "driver", "cls": "shark"}, {"id": 3026, "dom": "P", "ent": "driver", "cls": "sheep"}, {"id": 3027, "dom": "P", "ent": "driver", "cls": "shook"}, {"id": 3028, "dom": "P", "ent": "driver", "cls": "tank"}, {"id": 3029, "dom": "P", "ent": "driver", "cls": "wheat"}, {"id": 3030, "dom": "P", "ent": "duct", "cls": "architecture"}, {"id": 3031, "dom": "P", "ent": "duct", "cls": "bacteria"}, {"id": 3032, "dom": "P", "ent": "duct", "cls": "beef"}, {"id": 3033, "dom": "P", "ent": "duct", "cls": "bridge"}, {"id": 3034, "dom": "P", "ent": "duct", "cls": "bud"}, {"id": 3035, "dom": "P", "ent": "duct", "cls": "cat"}, {"id": 3036, "dom": "P", "ent": "duct", "cls": "chapel"}, {"id": 3037, "dom": "P", "ent": "duct", "cls": "chick"}, {"id": 3038, "dom": "P", "ent": "duct", "cls": "clock"}, {"id": 3039, "dom": "P", "ent": "duct", "cls": "deer"}, {"id": 3040, "dom": "P", "ent": "duct", "cls": "dog"}, {"id": 3041, "dom": "P", "ent": "duct", "cls": "elephant"}, {"id": 3042, "dom": "P", "ent": "duct", "cls": "embassy"}, {"id": 3043, "dom": "P", "ent": "duct", "cls": "garage"}, {"id": 3044, "dom": "P", "ent": "duct", "cls": "highway"}, {"id": 3045, "dom": "P", "ent": "duct", "cls": "kidney"}, {"id": 3046, "dom": "P", "ent": "duct", "cls": "lamb"}, {"id": 3047, "dom": "P", "ent": "duct", "cls": "male"}, {"id": 3048, "dom": "P", "ent": "duct", "cls": "marijuana"}, {"id": 3049, "dom": "P", "ent": "duct", "cls": "puppy"}, {"id": 3050, "dom": "P", "ent": "duct", "cls": "receiver"}, {"id": 3051, "dom": "P", "ent": "duct", "cls": "reed"}, {"id": 3052, "dom": "P", "ent": "duct", "cls": "rifle"}, {"id": 3053, "dom": "P", "ent": "duct", "cls": "road"}, {"id": 3054, "dom": "P", "ent": "duct", "cls": "salmon"}, {"id": 3055, "dom": "P", "ent": "duct", "cls": "shark"}, {"id": 3056, "dom": "P", "ent": "duct", "cls": "sheep"}, {"id": 3057, "dom": "P", "ent": "duct", "cls": "shook"}, {"id": 3058, "dom": "P", "ent": "duct", "cls": "tank"}, {"id": 3059, "dom": "P", "ent": "duct", "cls": "wheat"}, {"id": 3060, "dom": "P", "ent": "ear", "cls": "architecture"}, {"id": 3061, "dom": "P", "ent": "ear", "cls": "bacteria"}, {"id": 3062, "dom": "P", "ent": "ear", "cls": "beef"}, {"id": 3063, "dom": "P", "ent": "ear", "cls": "bridge"}, {"id": 3064, "dom": "P", "ent": "ear", "cls": "bud"}, {"id": 3065, "dom": "P", "ent": "ear", "cls": "cat"}, {"id": 3066, "dom": "P", "ent": "ear", "cls": "chapel"}, {"id": 3067, "dom": "P", "ent": "ear", "cls": "chick"}, {"id": 3068, "dom": "P", "ent": "ear", "cls": "clock"}, {"id": 3069, "dom": "P", "ent": "ear", "cls": "deer"}, {"id": 3070, "dom": "P", "ent": "ear", "cls": "dog"}, {"id": 3071, "dom": "P", "ent": "ear", "cls": "elephant"}, {"id": 3072, "dom": "P", "ent": "ear", "cls": "embassy"}, {"id": 3073, "dom": "P", "ent": "ear", "cls": "garage"}, {"id": 3074, "dom": "P", "ent": "ear", "cls": "highway"}, {"id": 3075, "dom": "P", "ent": "ear", "cls": "kidney"}, {"id": 3076, "dom": "P", "ent": "ear", "cls": "lamb"}, {"id": 3077, "dom": "P", "ent": "ear", "cls": "male"}, {"id": 3078, "dom": "P", "ent": "ear", "cls": "marijuana"}, {"id": 3079, "dom": "P", "ent": "ear", "cls": "puppy"}, {"id": 3080, "dom": "P", "ent": "ear", "cls": "receiver"}, {"id": 3081, "dom": "P", "ent": "ear", "cls": "reed"}, {"id": 3082, "dom": "P", "ent": "ear", "cls": "rifle"}, {"id": 3083, "dom": "P", "ent": "ear", "cls": "road"}, {"id": 3084, "dom": "P", "ent": "ear", "cls": "salmon"}, {"id": 3085, "dom": "P", "ent": "ear", "cls": "shark"}, {"id": 3086, "dom": "P", "ent": "ear", "cls": "sheep"}, {"id": 3087, "dom": "P", "ent": "ear", "cls": "shook"}, {"id": 3088, "dom": "P", "ent": "ear", "cls": "tank"}, {"id": 3089, "dom": "P", "ent": "ear", "cls": "wheat"}, {"id": 3090, "dom": "P", "ent": "edible", "cls": "architecture"}, {"id": 3091, "dom": "P", "ent": "edible", "cls": "bacteria"}, {"id": 3092, "dom": "P", "ent": "edible", "cls": "beef"}, {"id": 3093, "dom": "P", "ent": "edible", "cls": "bridge"}, {"id": 3094, "dom": "P", "ent": "edible", "cls": "bud"}, {"id": 3095, "dom": "P", "ent": "edible", "cls": "cat"}, {"id": 3096, "dom": "P", "ent": "edible", "cls": "chapel"}, {"id": 3097, "dom": "P", "ent": "edible", "cls": "chick"}, {"id": 3098, "dom": "P", "ent": "edible", "cls": "clock"}, {"id": 3099, "dom": "P", "ent": "edible", "cls": "deer"}, {"id": 3100, "dom": "P", "ent": "edible", "cls": "dog"}, {"id": 3101, "dom": "P", "ent": "edible", "cls": "elephant"}, {"id": 3102, "dom": "P", "ent": "edible", "cls": "embassy"}, {"id": 3103, "dom": "P", "ent": "edible", "cls": "garage"}, {"id": 3104, "dom": "P", "ent": "edible", "cls": "highway"}, {"id": 3105, "dom": "P", "ent": "edible", "cls": "kidney"}, {"id": 3106, "dom": "P", "ent": "edible", "cls": "lamb"}, {"id": 3107, "dom": "P", "ent": "edible", "cls": "male"}, {"id": 3108, "dom": "P", "ent": "edible", "cls": "marijuana"}, {"id": 3109, "dom": "P", "ent": "edible", "cls": "puppy"}, {"id": 3110, "dom": "P", "ent": "edible", "cls": "receiver"}, {"id": 3111, "dom": "P", "ent": "edible", "cls": "reed"}, {"id": 3112, "dom": "P", "ent": "edible", "cls": "rifle"}, {"id": 3113, "dom": "P", "ent": "edible", "cls": "road"}, {"id": 3114, "dom": "P", "ent": "edible", "cls": "salmon"}, {"id": 3115, "dom": "P", "ent": "edible", "cls": "shark"}, {"id": 3116, "dom": "P", "ent": "edible", "cls": "sheep"}, {"id": 3117, "dom": "P", "ent": "edible", "cls": "shook"}, {"id": 3118, "dom": "P", "ent": "edible", "cls": "tank"}, {"id": 3119, "dom": "P", "ent": "edible", "cls": "wheat"}, {"id": 3120, "dom": "P", "ent": "electrode", "cls": "architecture"}, {"id": 3121, "dom": "P", "ent": "electrode", "cls": "bacteria"}, {"id": 3122, "dom": "P", "ent": "electrode", "cls": "beef"}, {"id": 3123, "dom": "P", "ent": "electrode", "cls": "bridge"}, {"id": 3124, "dom": "P", "ent": "electrode", "cls": "bud"}, {"id": 3125, "dom": "P", "ent": "electrode", "cls": "cat"}, {"id": 3126, "dom": "P", "ent": "electrode", "cls": "chapel"}, {"id": 3127, "dom": "P", "ent": "electrode", "cls": "chick"}, {"id": 3128, "dom": "P", "ent": "electrode", "cls": "clock"}, {"id": 3129, "dom": "P", "ent": "electrode", "cls": "deer"}, {"id": 3130, "dom": "P", "ent": "electrode", "cls": "dog"}, {"id": 3131, "dom": "P", "ent": "electrode", "cls": "elephant"}, {"id": 3132, "dom": "P", "ent": "electrode", "cls": "embassy"}, {"id": 3133, "dom": "P", "ent": "electrode", "cls": "garage"}, {"id": 3134, "dom": "P", "ent": "electrode", "cls": "highway"}, {"id": 3135, "dom": "P", "ent": "electrode", "cls": "kidney"}, {"id": 3136, "dom": "P", "ent": "electrode", "cls": "lamb"}, {"id": 3137, "dom": "P", "ent": "electrode", "cls": "male"}, {"id": 3138, "dom": "P", "ent": "electrode", "cls": "marijuana"}, {"id": 3139, "dom": "P", "ent": "electrode", "cls": "puppy"}, {"id": 3140, "dom": "P", "ent": "electrode", "cls": "receiver"}, {"id": 3141, "dom": "P", "ent": "electrode", "cls": "reed"}, {"id": 3142, "dom": "P", "ent": "electrode", "cls": "rifle"}, {"id": 3143, "dom": "P", "ent": "electrode", "cls": "road"}, {"id": 3144, "dom": "P", "ent": "electrode", "cls": "salmon"}, {"id": 3145, "dom": "P", "ent": "electrode", "cls": "shark"}, {"id": 3146, "dom": "P", "ent": "electrode", "cls": "sheep"}, {"id": 3147, "dom": "P", "ent": "electrode", "cls": "shook"}, {"id": 3148, "dom": "P", "ent": "electrode", "cls": "tank"}, {"id": 3149, "dom": "P", "ent": "electrode", "cls": "wheat"}, {"id": 3150, "dom": "P", "ent": "elevator", "cls": "architecture"}, {"id": 3151, "dom": "P", "ent": "elevator", "cls": "bacteria"}, {"id": 3152, "dom": "P", "ent": "elevator", "cls": "beef"}, {"id": 3153, "dom": "P", "ent": "elevator", "cls": "bridge"}, {"id": 3154, "dom": "P", "ent": "elevator", "cls": "bud"}, {"id": 3155, "dom": "P", "ent": "elevator", "cls": "cat"}, {"id": 3156, "dom": "P", "ent": "elevator", "cls": "chapel"}, {"id": 3157, "dom": "P", "ent": "elevator", "cls": "chick"}, {"id": 3158, "dom": "P", "ent": "elevator", "cls": "clock"}, {"id": 3159, "dom": "P", "ent": "elevator", "cls": "deer"}, {"id": 3160, "dom": "P", "ent": "elevator", "cls": "dog"}, {"id": 3161, "dom": "P", "ent": "elevator", "cls": "elephant"}, {"id": 3162, "dom": "P", "ent": "elevator", "cls": "embassy"}, {"id": 3163, "dom": "P", "ent": "elevator", "cls": "garage"}, {"id": 3164, "dom": "P", "ent": "elevator", "cls": "highway"}, {"id": 3165, "dom": "P", "ent": "elevator", "cls": "kidney"}, {"id": 3166, "dom": "P", "ent": "elevator", "cls": "lamb"}, {"id": 3167, "dom": "P", "ent": "elevator", "cls": "male"}, {"id": 3168, "dom": "P", "ent": "elevator", "cls": "marijuana"}, {"id": 3169, "dom": "P", "ent": "elevator", "cls": "puppy"}, {"id": 3170, "dom": "P", "ent": "elevator", "cls": "receiver"}, {"id": 3171, "dom": "P", "ent": "elevator", "cls": "reed"}, {"id": 3172, "dom": "P", "ent": "elevator", "cls": "rifle"}, {"id": 3173, "dom": "P", "ent": "elevator", "cls": "road"}, {"id": 3174, "dom": "P", "ent": "elevator", "cls": "salmon"}, {"id": 3175, "dom": "P", "ent": "elevator", "cls": "shark"}, {"id": 3176, "dom": "P", "ent": "elevator", "cls": "sheep"}, {"id": 3177, "dom": "P", "ent": "elevator", "cls": "shook"}, {"id": 3178, "dom": "P", "ent": "elevator", "cls": "tank"}, {"id": 3179, "dom": "P", "ent": "elevator", "cls": "wheat"}, {"id": 3180, "dom": "P", "ent": "extension", "cls": "architecture"}, {"id": 3181, "dom": "P", "ent": "extension", "cls": "bacteria"}, {"id": 3182, "dom": "P", "ent": "extension", "cls": "beef"}, {"id": 3183, "dom": "P", "ent": "extension", "cls": "bridge"}, {"id": 3184, "dom": "P", "ent": "extension", "cls": "bud"}, {"id": 3185, "dom": "P", "ent": "extension", "cls": "cat"}, {"id": 3186, "dom": "P", "ent": "extension", "cls": "chapel"}, {"id": 3187, "dom": "P", "ent": "extension", "cls": "chick"}, {"id": 3188, "dom": "P", "ent": "extension", "cls": "clock"}, {"id": 3189, "dom": "P", "ent": "extension", "cls": "deer"}, {"id": 3190, "dom": "P", "ent": "extension", "cls": "dog"}, {"id": 3191, "dom": "P", "ent": "extension", "cls": "elephant"}, {"id": 3192, "dom": "P", "ent": "extension", "cls": "embassy"}, {"id": 3193, "dom": "P", "ent": "extension", "cls": "garage"}, {"id": 3194, "dom": "P", "ent": "extension", "cls": "highway"}, {"id": 3195, "dom": "P", "ent": "extension", "cls": "kidney"}, {"id": 3196, "dom": "P", "ent": "extension", "cls": "lamb"}, {"id": 3197, "dom": "P", "ent": "extension", "cls": "male"}, {"id": 3198, "dom": "P", "ent": "extension", "cls": "marijuana"}, {"id": 3199, "dom": "P", "ent": "extension", "cls": "puppy"}, {"id": 3200, "dom": "P", "ent": "extension", "cls": "receiver"}, {"id": 3201, "dom": "P", "ent": "extension", "cls": "reed"}, {"id": 3202, "dom": "P", "ent": "extension", "cls": "rifle"}, {"id": 3203, "dom": "P", "ent": "extension", "cls": "road"}, {"id": 3204, "dom": "P", "ent": "extension", "cls": "salmon"}, {"id": 3205, "dom": "P", "ent": "extension", "cls": "shark"}, {"id": 3206, "dom": "P", "ent": "extension", "cls": "sheep"}, {"id": 3207, "dom": "P", "ent": "extension", "cls": "shook"}, {"id": 3208, "dom": "P", "ent": "extension", "cls": "tank"}, {"id": 3209, "dom": "P", "ent": "extension", "cls": "wheat"}, {"id": 3210, "dom": "P", "ent": "eyeball", "cls": "architecture"}, {"id": 3211, "dom": "P", "ent": "eyeball", "cls": "bacteria"}, {"id": 3212, "dom": "P", "ent": "eyeball", "cls": "beef"}, {"id": 3213, "dom": "P", "ent": "eyeball", "cls": "bridge"}, {"id": 3214, "dom": "P", "ent": "eyeball", "cls": "bud"}, {"id": 3215, "dom": "P", "ent": "eyeball", "cls": "cat"}, {"id": 3216, "dom": "P", "ent": "eyeball", "cls": "chapel"}, {"id": 3217, "dom": "P", "ent": "eyeball", "cls": "chick"}, {"id": 3218, "dom": "P", "ent": "eyeball", "cls": "clock"}, {"id": 3219, "dom": "P", "ent": "eyeball", "cls": "deer"}, {"id": 3220, "dom": "P", "ent": "eyeball", "cls": "dog"}, {"id": 3221, "dom": "P", "ent": "eyeball", "cls": "elephant"}, {"id": 3222, "dom": "P", "ent": "eyeball", "cls": "embassy"}, {"id": 3223, "dom": "P", "ent": "eyeball", "cls": "garage"}, {"id": 3224, "dom": "P", "ent": "eyeball", "cls": "highway"}, {"id": 3225, "dom": "P", "ent": "eyeball", "cls": "kidney"}, {"id": 3226, "dom": "P", "ent": "eyeball", "cls": "lamb"}, {"id": 3227, "dom": "P", "ent": "eyeball", "cls": "male"}, {"id": 3228, "dom": "P", "ent": "eyeball", "cls": "marijuana"}, {"id": 3229, "dom": "P", "ent": "eyeball", "cls": "puppy"}, {"id": 3230, "dom": "P", "ent": "eyeball", "cls": "receiver"}, {"id": 3231, "dom": "P", "ent": "eyeball", "cls": "reed"}, {"id": 3232, "dom": "P", "ent": "eyeball", "cls": "rifle"}, {"id": 3233, "dom": "P", "ent": "eyeball", "cls": "road"}, {"id": 3234, "dom": "P", "ent": "eyeball", "cls": "salmon"}, {"id": 3235, "dom": "P", "ent": "eyeball", "cls": "shark"}, {"id": 3236, "dom": "P", "ent": "eyeball", "cls": "sheep"}, {"id": 3237, "dom": "P", "ent": "eyeball", "cls": "shook"}, {"id": 3238, "dom": "P", "ent": "eyeball", "cls": "tank"}, {"id": 3239, "dom": "P", "ent": "eyeball", "cls": "wheat"}, {"id": 3240, "dom": "P", "ent": "eyebrow", "cls": "architecture"}, {"id": 3241, "dom": "P", "ent": "eyebrow", "cls": "bacteria"}, {"id": 3242, "dom": "P", "ent": "eyebrow", "cls": "beef"}, {"id": 3243, "dom": "P", "ent": "eyebrow", "cls": "bridge"}, {"id": 3244, "dom": "P", "ent": "eyebrow", "cls": "bud"}, {"id": 3245, "dom": "P", "ent": "eyebrow", "cls": "cat"}, {"id": 3246, "dom": "P", "ent": "eyebrow", "cls": "chapel"}, {"id": 3247, "dom": "P", "ent": "eyebrow", "cls": "chick"}, {"id": 3248, "dom": "P", "ent": "eyebrow", "cls": "clock"}, {"id": 3249, "dom": "P", "ent": "eyebrow", "cls": "deer"}, {"id": 3250, "dom": "P", "ent": "eyebrow", "cls": "dog"}, {"id": 3251, "dom": "P", "ent": "eyebrow", "cls": "elephant"}, {"id": 3252, "dom": "P", "ent": "eyebrow", "cls": "embassy"}, {"id": 3253, "dom": "P", "ent": "eyebrow", "cls": "garage"}, {"id": 3254, "dom": "P", "ent": "eyebrow", "cls": "highway"}, {"id": 3255, "dom": "P", "ent": "eyebrow", "cls": "kidney"}, {"id": 3256, "dom": "P", "ent": "eyebrow", "cls": "lamb"}, {"id": 3257, "dom": "P", "ent": "eyebrow", "cls": "male"}, {"id": 3258, "dom": "P", "ent": "eyebrow", "cls": "marijuana"}, {"id": 3259, "dom": "P", "ent": "eyebrow", "cls": "puppy"}, {"id": 3260, "dom": "P", "ent": "eyebrow", "cls": "receiver"}, {"id": 3261, "dom": "P", "ent": "eyebrow", "cls": "reed"}, {"id": 3262, "dom": "P", "ent": "eyebrow", "cls": "rifle"}, {"id": 3263, "dom": "P", "ent": "eyebrow", "cls": "road"}, {"id": 3264, "dom": "P", "ent": "eyebrow", "cls": "salmon"}, {"id": 3265, "dom": "P", "ent": "eyebrow", "cls": "shark"}, {"id": 3266, "dom": "P", "ent": "eyebrow", "cls": "sheep"}, {"id": 3267, "dom": "P", "ent": "eyebrow", "cls": "shook"}, {"id": 3268, "dom": "P", "ent": "eyebrow", "cls": "tank"}, {"id": 3269, "dom": "P", "ent": "eyebrow", "cls": "wheat"}, {"id": 3270, "dom": "P", "ent": "eyelid", "cls": "architecture"}, {"id": 3271, "dom": "P", "ent": "eyelid", "cls": "bacteria"}, {"id": 3272, "dom": "P", "ent": "eyelid", "cls": "beef"}, {"id": 3273, "dom": "P", "ent": "eyelid", "cls": "bridge"}, {"id": 3274, "dom": "P", "ent": "eyelid", "cls": "bud"}, {"id": 3275, "dom": "P", "ent": "eyelid", "cls": "cat"}, {"id": 3276, "dom": "P", "ent": "eyelid", "cls": "chapel"}, {"id": 3277, "dom": "P", "ent": "eyelid", "cls": "chick"}, {"id": 3278, "dom": "P", "ent": "eyelid", "cls": "clock"}, {"id": 3279, "dom": "P", "ent": "eyelid", "cls": "deer"}, {"id": 3280, "dom": "P", "ent": "eyelid", "cls": "dog"}, {"id": 3281, "dom": "P", "ent": "eyelid", "cls": "elephant"}, {"id": 3282, "dom": "P", "ent": "eyelid", "cls": "embassy"}, {"id": 3283, "dom": "P", "ent": "eyelid", "cls": "garage"}, {"id": 3284, "dom": "P", "ent": "eyelid", "cls": "highway"}, {"id": 3285, "dom": "P", "ent": "eyelid", "cls": "kidney"}, {"id": 3286, "dom": "P", "ent": "eyelid", "cls": "lamb"}, {"id": 3287, "dom": "P", "ent": "eyelid", "cls": "male"}, {"id": 3288, "dom": "P", "ent": "eyelid", "cls": "marijuana"}, {"id": 3289, "dom": "P", "ent": "eyelid", "cls": "puppy"}, {"id": 3290, "dom": "P", "ent": "eyelid", "cls": "receiver"}, {"id": 3291, "dom": "P", "ent": "eyelid", "cls": "reed"}, {"id": 3292, "dom": "P", "ent": "eyelid", "cls": "rifle"}, {"id": 3293, "dom": "P", "ent": "eyelid", "cls": "road"}, {"id": 3294, "dom": "P", "ent": "eyelid", "cls": "salmon"}, {"id": 3295, "dom": "P", "ent": "eyelid", "cls": "shark"}, {"id": 3296, "dom": "P", "ent": "eyelid", "cls": "sheep"}, {"id": 3297, "dom": "P", "ent": "eyelid", "cls": "shook"}, {"id": 3298, "dom": "P", "ent": "eyelid", "cls": "tank"}, {"id": 3299, "dom": "P", "ent": "eyelid", "cls": "wheat"}, {"id": 3300, "dom": "P", "ent": "facial", "cls": "architecture"}, {"id": 3301, "dom": "P", "ent": "facial", "cls": "bacteria"}, {"id": 3302, "dom": "P", "ent": "facial", "cls": "beef"}, {"id": 3303, "dom": "P", "ent": "facial", "cls": "bridge"}, {"id": 3304, "dom": "P", "ent": "facial", "cls": "bud"}, {"id": 3305, "dom": "P", "ent": "facial", "cls": "cat"}, {"id": 3306, "dom": "P", "ent": "facial", "cls": "chapel"}, {"id": 3307, "dom": "P", "ent": "facial", "cls": "chick"}, {"id": 3308, "dom": "P", "ent": "facial", "cls": "clock"}, {"id": 3309, "dom": "P", "ent": "facial", "cls": "deer"}, {"id": 3310, "dom": "P", "ent": "facial", "cls": "dog"}, {"id": 3311, "dom": "P", "ent": "facial", "cls": "elephant"}, {"id": 3312, "dom": "P", "ent": "facial", "cls": "embassy"}, {"id": 3313, "dom": "P", "ent": "facial", "cls": "garage"}, {"id": 3314, "dom": "P", "ent": "facial", "cls": "highway"}, {"id": 3315, "dom": "P", "ent": "facial", "cls": "kidney"}, {"id": 3316, "dom": "P", "ent": "facial", "cls": "lamb"}, {"id": 3317, "dom": "P", "ent": "facial", "cls": "male"}, {"id": 3318, "dom": "P", "ent": "facial", "cls": "marijuana"}, {"id": 3319, "dom": "P", "ent": "facial", "cls": "puppy"}, {"id": 3320, "dom": "P", "ent": "facial", "cls": "receiver"}, {"id": 3321, "dom": "P", "ent": "facial", "cls": "reed"}, {"id": 3322, "dom": "P", "ent": "facial", "cls": "rifle"}, {"id": 3323, "dom": "P", "ent": "facial", "cls": "road"}, {"id": 3324, "dom": "P", "ent": "facial", "cls": "salmon"}, {"id": 3325, "dom": "P", "ent": "facial", "cls": "shark"}, {"id": 3326, "dom": "P", "ent": "facial", "cls": "sheep"}, {"id": 3327, "dom": "P", "ent": "facial", "cls": "shook"}, {"id": 3328, "dom": "P", "ent": "facial", "cls": "tank"}, {"id": 3329, "dom": "P", "ent": "facial", "cls": "wheat"}, {"id": 3330, "dom": "P", "ent": "factor", "cls": "architecture"}, {"id": 3331, "dom": "P", "ent": "factor", "cls": "bacteria"}, {"id": 3332, "dom": "P", "ent": "factor", "cls": "beef"}, {"id": 3333, "dom": "P", "ent": "factor", "cls": "bridge"}, {"id": 3334, "dom": "P", "ent": "factor", "cls": "bud"}, {"id": 3335, "dom": "P", "ent": "factor", "cls": "cat"}, {"id": 3336, "dom": "P", "ent": "factor", "cls": "chapel"}, {"id": 3337, "dom": "P", "ent": "factor", "cls": "chick"}, {"id": 3338, "dom": "P", "ent": "factor", "cls": "clock"}, {"id": 3339, "dom": "P", "ent": "factor", "cls": "deer"}, {"id": 3340, "dom": "P", "ent": "factor", "cls": "dog"}, {"id": 3341, "dom": "P", "ent": "factor", "cls": "elephant"}, {"id": 3342, "dom": "P", "ent": "factor", "cls": "embassy"}, {"id": 3343, "dom": "P", "ent": "factor", "cls": "garage"}, {"id": 3344, "dom": "P", "ent": "factor", "cls": "highway"}, {"id": 3345, "dom": "P", "ent": "factor", "cls": "kidney"}, {"id": 3346, "dom": "P", "ent": "factor", "cls": "lamb"}, {"id": 3347, "dom": "P", "ent": "factor", "cls": "male"}, {"id": 3348, "dom": "P", "ent": "factor", "cls": "marijuana"}, {"id": 3349, "dom": "P", "ent": "factor", "cls": "puppy"}, {"id": 3350, "dom": "P", "ent": "factor", "cls": "receiver"}, {"id": 3351, "dom": "P", "ent": "factor", "cls": "reed"}, {"id": 3352, "dom": "P", "ent": "factor", "cls": "rifle"}, {"id": 3353, "dom": "P", "ent": "factor", "cls": "road"}, {"id": 3354, "dom": "P", "ent": "factor", "cls": "salmon"}, {"id": 3355, "dom": "P", "ent": "factor", "cls": "shark"}, {"id": 3356, "dom": "P", "ent": "factor", "cls": "sheep"}, {"id": 3357, "dom": "P", "ent": "factor", "cls": "shook"}, {"id": 3358, "dom": "P", "ent": "factor", "cls": "tank"}, {"id": 3359, "dom": "P", "ent": "factor", "cls": "wheat"}, {"id": 3360, "dom": "P", "ent": "feather", "cls": "architecture"}, {"id": 3361, "dom": "P", "ent": "feather", "cls": "bacteria"}, {"id": 3362, "dom": "P", "ent": "feather", "cls": "beef"}, {"id": 3363, "dom": "P", "ent": "feather", "cls": "bridge"}, {"id": 3364, "dom": "P", "ent": "feather", "cls": "bud"}, {"id": 3365, "dom": "P", "ent": "feather", "cls": "cat"}, {"id": 3366, "dom": "P", "ent": "feather", "cls": "chapel"}, {"id": 3367, "dom": "P", "ent": "feather", "cls": "chick"}, {"id": 3368, "dom": "P", "ent": "feather", "cls": "clock"}, {"id": 3369, "dom": "P", "ent": "feather", "cls": "deer"}, {"id": 3370, "dom": "P", "ent": "feather", "cls": "dog"}, {"id": 3371, "dom": "P", "ent": "feather", "cls": "elephant"}, {"id": 3372, "dom": "P", "ent": "feather", "cls": "embassy"}, {"id": 3373, "dom": "P", "ent": "feather", "cls": "garage"}, {"id": 3374, "dom": "P", "ent": "feather", "cls": "highway"}, {"id": 3375, "dom": "P", "ent": "feather", "cls": "kidney"}, {"id": 3376, "dom": "P", "ent": "feather", "cls": "lamb"}, {"id": 3377, "dom": "P", "ent": "feather", "cls": "male"}, {"id": 3378, "dom": "P", "ent": "feather", "cls": "marijuana"}, {"id": 3379, "dom": "P", "ent": "feather", "cls": "puppy"}, {"id": 3380, "dom": "P", "ent": "feather", "cls": "receiver"}, {"id": 3381, "dom": "P", "ent": "feather", "cls": "reed"}, {"id": 3382, "dom": "P", "ent": "feather", "cls": "rifle"}, {"id": 3383, "dom": "P", "ent": "feather", "cls": "road"}, {"id": 3384, "dom": "P", "ent": "feather", "cls": "salmon"}, {"id": 3385, "dom": "P", "ent": "feather", "cls": "shark"}, {"id": 3386, "dom": "P", "ent": "feather", "cls": "sheep"}, {"id": 3387, "dom": "P", "ent": "feather", "cls": "shook"}, {"id": 3388, "dom": "P", "ent": "feather", "cls": "tank"}, {"id": 3389, "dom": "P", "ent": "feather", "cls": "wheat"}, {"id": 3390, "dom": "P", "ent": "feature", "cls": "architecture"}, {"id": 3391, "dom": "P", "ent": "feature", "cls": "bacteria"}, {"id": 3392, "dom": "P", "ent": "feature", "cls": "beef"}, {"id": 3393, "dom": "P", "ent": "feature", "cls": "bridge"}, {"id": 3394, "dom": "P", "ent": "feature", "cls": "bud"}, {"id": 3395, "dom": "P", "ent": "feature", "cls": "cat"}, {"id": 3396, "dom": "P", "ent": "feature", "cls": "chapel"}, {"id": 3397, "dom": "P", "ent": "feature", "cls": "chick"}, {"id": 3398, "dom": "P", "ent": "feature", "cls": "clock"}, {"id": 3399, "dom": "P", "ent": "feature", "cls": "deer"}, {"id": 3400, "dom": "P", "ent": "feature", "cls": "dog"}, {"id": 3401, "dom": "P", "ent": "feature", "cls": "elephant"}, {"id": 3402, "dom": "P", "ent": "feature", "cls": "embassy"}, {"id": 3403, "dom": "P", "ent": "feature", "cls": "garage"}, {"id": 3404, "dom": "P", "ent": "feature", "cls": "highway"}, {"id": 3405, "dom": "P", "ent": "feature", "cls": "kidney"}, {"id": 3406, "dom": "P", "ent": "feature", "cls": "lamb"}, {"id": 3407, "dom": "P", "ent": "feature", "cls": "male"}, {"id": 3408, "dom": "P", "ent": "feature", "cls": "marijuana"}, {"id": 3409, "dom": "P", "ent": "feature", "cls": "puppy"}, {"id": 3410, "dom": "P", "ent": "feature", "cls": "receiver"}, {"id": 3411, "dom": "P", "ent": "feature", "cls": "reed"}, {"id": 3412, "dom": "P", "ent": "feature", "cls": "rifle"}, {"id": 3413, "dom": "P", "ent": "feature", "cls": "road"}, {"id": 3414, "dom": "P", "ent": "feature", "cls": "salmon"}, {"id": 3415, "dom": "P", "ent": "feature", "cls": "shark"}, {"id": 3416, "dom": "P", "ent": "feature", "cls": "sheep"}, {"id": 3417, "dom": "P", "ent": "feature", "cls": "shook"}, {"id": 3418, "dom": "P", "ent": "feature", "cls": "tank"}, {"id": 3419, "dom": "P", "ent": "feature", "cls": "wheat"}, {"id": 3420, "dom": "P", "ent": "fence", "cls": "architecture"}, {"id": 3421, "dom": "P", "ent": "fence", "cls": "bacteria"}, {"id": 3422, "dom": "P", "ent": "fence", "cls": "beef"}, {"id": 3423, "dom": "P", "ent": "fence", "cls": "bridge"}, {"id": 3424, "dom": "P", "ent": "fence", "cls": "bud"}, {"id": 3425, "dom": "P", "ent": "fence", "cls": "cat"}, {"id": 3426, "dom": "P", "ent": "fence", "cls": "chapel"}, {"id": 3427, "dom": "P", "ent": "fence", "cls": "chick"}, {"id": 3428, "dom": "P", "ent": "fence", "cls": "clock"}, {"id": 3429, "dom": "P", "ent": "fence", "cls": "deer"}, {"id": 3430, "dom": "P", "ent": "fence", "cls": "dog"}, {"id": 3431, "dom": "P", "ent": "fence", "cls": "elephant"}, {"id": 3432, "dom": "P", "ent": "fence", "cls": "embassy"}, {"id": 3433, "dom": "P", "ent": "fence", "cls": "garage"}, {"id": 3434, "dom": "P", "ent": "fence", "cls": "highway"}, {"id": 3435, "dom": "P", "ent": "fence", "cls": "kidney"}, {"id": 3436, "dom": "P", "ent": "fence", "cls": "lamb"}, {"id": 3437, "dom": "P", "ent": "fence", "cls": "male"}, {"id": 3438, "dom": "P", "ent": "fence", "cls": "marijuana"}, {"id": 3439, "dom": "P", "ent": "fence", "cls": "puppy"}, {"id": 3440, "dom": "P", "ent": "fence", "cls": "receiver"}, {"id": 3441, "dom": "P", "ent": "fence", "cls": "reed"}, {"id": 3442, "dom": "P", "ent": "fence", "cls": "rifle"}, {"id": 3443, "dom": "P", "ent": "fence", "cls": "road"}, {"id": 3444, "dom": "P", "ent": "fence", "cls": "salmon"}, {"id": 3445, "dom": "P", "ent": "fence", "cls": "shark"}, {"id": 3446, "dom": "P", "ent": "fence", "cls": "sheep"}, {"id": 3447, "dom": "P", "ent": "fence", "cls": "shook"}, {"id": 3448, "dom": "P", "ent": "fence", "cls": "tank"}, {"id": 3449, "dom": "P", "ent": "fence", "cls": "wheat"}, {"id": 3450, "dom": "P", "ent": "fin", "cls": "architecture"}, {"id": 3451, "dom": "P", "ent": "fin", "cls": "bacteria"}, {"id": 3452, "dom": "P", "ent": "fin", "cls": "beef"}, {"id": 3453, "dom": "P", "ent": "fin", "cls": "bridge"}, {"id": 3454, "dom": "P", "ent": "fin", "cls": "bud"}, {"id": 3455, "dom": "P", "ent": "fin", "cls": "cat"}, {"id": 3456, "dom": "P", "ent": "fin", "cls": "chapel"}, {"id": 3457, "dom": "P", "ent": "fin", "cls": "chick"}, {"id": 3458, "dom": "P", "ent": "fin", "cls": "clock"}, {"id": 3459, "dom": "P", "ent": "fin", "cls": "deer"}, {"id": 3460, "dom": "P", "ent": "fin", "cls": "dog"}, {"id": 3461, "dom": "P", "ent": "fin", "cls": "elephant"}, {"id": 3462, "dom": "P", "ent": "fin", "cls": "embassy"}, {"id": 3463, "dom": "P", "ent": "fin", "cls": "garage"}, {"id": 3464, "dom": "P", "ent": "fin", "cls": "highway"}, {"id": 3465, "dom": "P", "ent": "fin", "cls": "kidney"}, {"id": 3466, "dom": "P", "ent": "fin", "cls": "lamb"}, {"id": 3467, "dom": "P", "ent": "fin", "cls": "male"}, {"id": 3468, "dom": "P", "ent": "fin", "cls": "marijuana"}, {"id": 3469, "dom": "P", "ent": "fin", "cls": "puppy"}, {"id": 3470, "dom": "P", "ent": "fin", "cls": "receiver"}, {"id": 3471, "dom": "P", "ent": "fin", "cls": "reed"}, {"id": 3472, "dom": "P", "ent": "fin", "cls": "rifle"}, {"id": 3473, "dom": "P", "ent": "fin", "cls": "road"}, {"id": 3474, "dom": "P", "ent": "fin", "cls": "salmon"}, {"id": 3475, "dom": "P", "ent": "fin", "cls": "shark"}, {"id": 3476, "dom": "P", "ent": "fin", "cls": "sheep"}, {"id": 3477, "dom": "P", "ent": "fin", "cls": "shook"}, {"id": 3478, "dom": "P", "ent": "fin", "cls": "tank"}, {"id": 3479, "dom": "P", "ent": "fin", "cls": "wheat"}, {"id": 3480, "dom": "P", "ent": "finder", "cls": "architecture"}, {"id": 3481, "dom": "P", "ent": "finder", "cls": "bacteria"}, {"id": 3482, "dom": "P", "ent": "finder", "cls": "beef"}, {"id": 3483, "dom": "P", "ent": "finder", "cls": "bridge"}, {"id": 3484, "dom": "P", "ent": "finder", "cls": "bud"}, {"id": 3485, "dom": "P", "ent": "finder", "cls": "cat"}, {"id": 3486, "dom": "P", "ent": "finder", "cls": "chapel"}, {"id": 3487, "dom": "P", "ent": "finder", "cls": "chick"}, {"id": 3488, "dom": "P", "ent": "finder", "cls": "clock"}, {"id": 3489, "dom": "P", "ent": "finder", "cls": "deer"}, {"id": 3490, "dom": "P", "ent": "finder", "cls": "dog"}, {"id": 3491, "dom": "P", "ent": "finder", "cls": "elephant"}, {"id": 3492, "dom": "P", "ent": "finder", "cls": "embassy"}, {"id": 3493, "dom": "P", "ent": "finder", "cls": "garage"}, {"id": 3494, "dom": "P", "ent": "finder", "cls": "highway"}, {"id": 3495, "dom": "P", "ent": "finder", "cls": "kidney"}, {"id": 3496, "dom": "P", "ent": "finder", "cls": "lamb"}, {"id": 3497, "dom": "P", "ent": "finder", "cls": "male"}, {"id": 3498, "dom": "P", "ent": "finder", "cls": "marijuana"}, {"id": 3499, "dom": "P", "ent": "finder", "cls": "puppy"}, {"id": 3500, "dom": "P", "ent": "finder", "cls": "receiver"}, {"id": 3501, "dom": "P", "ent": "finder", "cls": "reed"}, {"id": 3502, "dom": "P", "ent": "finder", "cls": "rifle"}, {"id": 3503, "dom": "P", "ent": "finder", "cls": "road"}, {"id": 3504, "dom": "P", "ent": "finder", "cls": "salmon"}, {"id": 3505, "dom": "P", "ent": "finder", "cls": "shark"}, {"id": 3506, "dom": "P", "ent": "finder", "cls": "sheep"}, {"id": 3507, "dom": "P", "ent": "finder", "cls": "shook"}, {"id": 3508, "dom": "P", "ent": "finder", "cls": "tank"}, {"id": 3509, "dom": "P", "ent": "finder", "cls": "wheat"}, {"id": 3510, "dom": "P", "ent": "flag", "cls": "architecture"}, {"id": 3511, "dom": "P", "ent": "flag", "cls": "bacteria"}, {"id": 3512, "dom": "P", "ent": "flag", "cls": "beef"}, {"id": 3513, "dom": "P", "ent": "flag", "cls": "bridge"}, {"id": 3514, "dom": "P", "ent": "flag", "cls": "bud"}, {"id": 3515, "dom": "P", "ent": "flag", "cls": "cat"}, {"id": 3516, "dom": "P", "ent": "flag", "cls": "chapel"}, {"id": 3517, "dom": "P", "ent": "flag", "cls": "chick"}, {"id": 3518, "dom": "P", "ent": "flag", "cls": "clock"}, {"id": 3519, "dom": "P", "ent": "flag", "cls": "deer"}, {"id": 3520, "dom": "P", "ent": "flag", "cls": "dog"}, {"id": 3521, "dom": "P", "ent": "flag", "cls": "elephant"}, {"id": 3522, "dom": "P", "ent": "flag", "cls": "embassy"}, {"id": 3523, "dom": "P", "ent": "flag", "cls": "garage"}, {"id": 3524, "dom": "P", "ent": "flag", "cls": "highway"}, {"id": 3525, "dom": "P", "ent": "flag", "cls": "kidney"}, {"id": 3526, "dom": "P", "ent": "flag", "cls": "lamb"}, {"id": 3527, "dom": "P", "ent": "flag", "cls": "male"}, {"id": 3528, "dom": "P", "ent": "flag", "cls": "marijuana"}, {"id": 3529, "dom": "P", "ent": "flag", "cls": "puppy"}, {"id": 3530, "dom": "P", "ent": "flag", "cls": "receiver"}, {"id": 3531, "dom": "P", "ent": "flag", "cls": "reed"}, {"id": 3532, "dom": "P", "ent": "flag", "cls": "rifle"}, {"id": 3533, "dom": "P", "ent": "flag", "cls": "road"}, {"id": 3534, "dom": "P", "ent": "flag", "cls": "salmon"}, {"id": 3535, "dom": "P", "ent": "flag", "cls": "shark"}, {"id": 3536, "dom": "P", "ent": "flag", "cls": "sheep"}, {"id": 3537, "dom": "P", "ent": "flag", "cls": "shook"}, {"id": 3538, "dom": "P", "ent": "flag", "cls": "tank"}, {"id": 3539, "dom": "P", "ent": "flag", "cls": "wheat"}, {"id": 3540, "dom": "P", "ent": "floor", "cls": "architecture"}, {"id": 3541, "dom": "P", "ent": "floor", "cls": "bacteria"}, {"id": 3542, "dom": "P", "ent": "floor", "cls": "beef"}, {"id": 3543, "dom": "P", "ent": "floor", "cls": "bridge"}, {"id": 3544, "dom": "P", "ent": "floor", "cls": "bud"}, {"id": 3545, "dom": "P", "ent": "floor", "cls": "cat"}, {"id": 3546, "dom": "P", "ent": "floor", "cls": "chapel"}, {"id": 3547, "dom": "P", "ent": "floor", "cls": "chick"}, {"id": 3548, "dom": "P", "ent": "floor", "cls": "clock"}, {"id": 3549, "dom": "P", "ent": "floor", "cls": "deer"}, {"id": 3550, "dom": "P", "ent": "floor", "cls": "dog"}, {"id": 3551, "dom": "P", "ent": "floor", "cls": "elephant"}, {"id": 3552, "dom": "P", "ent": "floor", "cls": "embassy"}, {"id": 3553, "dom": "P", "ent": "floor", "cls": "garage"}, {"id": 3554, "dom": "P", "ent": "floor", "cls": "highway"}, {"id": 3555, "dom": "P", "ent": "floor", "cls": "kidney"}, {"id": 3556, "dom": "P", "ent": "floor", "cls": "lamb"}, {"id": 3557, "dom": "P", "ent": "floor", "cls": "male"}, {"id": 3558, "dom": "P", "ent": "floor", "cls": "marijuana"}, {"id": 3559, "dom": "P", "ent": "floor", "cls": "puppy"}, {"id": 3560, "dom": "P", "ent": "floor", "cls": "receiver"}, {"id": 3561, "dom": "P", "ent": "floor", "cls": "reed"}, {"id": 3562, "dom": "P", "ent": "floor", "cls": "rifle"}, {"id": 3563, "dom": "P", "ent": "floor", "cls": "road"}, {"id": 3564, "dom": "P", "ent": "floor", "cls": "salmon"}, {"id": 3565, "dom": "P", "ent": "floor", "cls": "shark"}, {"id": 3566, "dom": "P", "ent": "floor", "cls": "sheep"}, {"id": 3567, "dom": "P", "ent": "floor", "cls": "shook"}, {"id": 3568, "dom": "P", "ent": "floor", "cls": "tank"}, {"id": 3569, "dom": "P", "ent": "floor", "cls": "wheat"}, {"id": 3570, "dom": "P", "ent": "flooring", "cls": "architecture"}, {"id": 3571, "dom": "P", "ent": "flooring", "cls": "bacteria"}, {"id": 3572, "dom": "P", "ent": "flooring", "cls": "beef"}, {"id": 3573, "dom": "P", "ent": "flooring", "cls": "bridge"}, {"id": 3574, "dom": "P", "ent": "flooring", "cls": "bud"}, {"id": 3575, "dom": "P", "ent": "flooring", "cls": "cat"}, {"id": 3576, "dom": "P", "ent": "flooring", "cls": "chapel"}, {"id": 3577, "dom": "P", "ent": "flooring", "cls": "chick"}, {"id": 3578, "dom": "P", "ent": "flooring", "cls": "clock"}, {"id": 3579, "dom": "P", "ent": "flooring", "cls": "deer"}, {"id": 3580, "dom": "P", "ent": "flooring", "cls": "dog"}, {"id": 3581, "dom": "P", "ent": "flooring", "cls": "elephant"}, {"id": 3582, "dom": "P", "ent": "flooring", "cls": "embassy"}, {"id": 3583, "dom": "P", "ent": "flooring", "cls": "garage"}, {"id": 3584, "dom": "P", "ent": "flooring", "cls": "highway"}, {"id": 3585, "dom": "P", "ent": "flooring", "cls": "kidney"}, {"id": 3586, "dom": "P", "ent": "flooring", "cls": "lamb"}, {"id": 3587, "dom": "P", "ent": "flooring", "cls": "male"}, {"id": 3588, "dom": "P", "ent": "flooring", "cls": "marijuana"}, {"id": 3589, "dom": "P", "ent": "flooring", "cls": "puppy"}, {"id": 3590, "dom": "P", "ent": "flooring", "cls": "receiver"}, {"id": 3591, "dom": "P", "ent": "flooring", "cls": "reed"}, {"id": 3592, "dom": "P", "ent": "flooring", "cls": "rifle"}, {"id": 3593, "dom": "P", "ent": "flooring", "cls": "road"}, {"id": 3594, "dom": "P", "ent": "flooring", "cls": "salmon"}, {"id": 3595, "dom": "P", "ent": "flooring", "cls": "shark"}, {"id": 3596, "dom": "P", "ent": "flooring", "cls": "sheep"}, {"id": 3597, "dom": "P", "ent": "flooring", "cls": "shook"}, {"id": 3598, "dom": "P", "ent": "flooring", "cls": "tank"}, {"id": 3599, "dom": "P", "ent": "flooring", "cls": "wheat"}, {"id": 3600, "dom": "P", "ent": "foil", "cls": "architecture"}, {"id": 3601, "dom": "P", "ent": "foil", "cls": "bacteria"}, {"id": 3602, "dom": "P", "ent": "foil", "cls": "beef"}, {"id": 3603, "dom": "P", "ent": "foil", "cls": "bridge"}, {"id": 3604, "dom": "P", "ent": "foil", "cls": "bud"}, {"id": 3605, "dom": "P", "ent": "foil", "cls": "cat"}, {"id": 3606, "dom": "P", "ent": "foil", "cls": "chapel"}, {"id": 3607, "dom": "P", "ent": "foil", "cls": "chick"}, {"id": 3608, "dom": "P", "ent": "foil", "cls": "clock"}, {"id": 3609, "dom": "P", "ent": "foil", "cls": "deer"}, {"id": 3610, "dom": "P", "ent": "foil", "cls": "dog"}, {"id": 3611, "dom": "P", "ent": "foil", "cls": "elephant"}, {"id": 3612, "dom": "P", "ent": "foil", "cls": "embassy"}, {"id": 3613, "dom": "P", "ent": "foil", "cls": "garage"}, {"id": 3614, "dom": "P", "ent": "foil", "cls": "highway"}, {"id": 3615, "dom": "P", "ent": "foil", "cls": "kidney"}, {"id": 3616, "dom": "P", "ent": "foil", "cls": "lamb"}, {"id": 3617, "dom": "P", "ent": "foil", "cls": "male"}, {"id": 3618, "dom": "P", "ent": "foil", "cls": "marijuana"}, {"id": 3619, "dom": "P", "ent": "foil", "cls": "puppy"}, {"id": 3620, "dom": "P", "ent": "foil", "cls": "receiver"}, {"id": 3621, "dom": "P", "ent": "foil", "cls": "reed"}, {"id": 3622, "dom": "P", "ent": "foil", "cls": "rifle"}, {"id": 3623, "dom": "P", "ent": "foil", "cls": "road"}, {"id": 3624, "dom": "P", "ent": "foil", "cls": "salmon"}, {"id": 3625, "dom": "P", "ent": "foil", "cls": "shark"}, {"id": 3626, "dom": "P", "ent": "foil", "cls": "sheep"}, {"id": 3627, "dom": "P", "ent": "foil", "cls": "shook"}, {"id": 3628, "dom": "P", "ent": "foil", "cls": "tank"}, {"id": 3629, "dom": "P", "ent": "foil", "cls": "wheat"}, {"id": 3630, "dom": "P", "ent": "food", "cls": "architecture"}, {"id": 3631, "dom": "P", "ent": "food", "cls": "bacteria"}, {"id": 3632, "dom": "P", "ent": "food", "cls": "beef"}, {"id": 3633, "dom": "P", "ent": "food", "cls": "bridge"}, {"id": 3634, "dom": "P", "ent": "food", "cls": "bud"}, {"id": 3635, "dom": "P", "ent": "food", "cls": "cat"}, {"id": 3636, "dom": "P", "ent": "food", "cls": "chapel"}, {"id": 3637, "dom": "P", "ent": "food", "cls": "chick"}, {"id": 3638, "dom": "P", "ent": "food", "cls": "clock"}, {"id": 3639, "dom": "P", "ent": "food", "cls": "deer"}, {"id": 3640, "dom": "P", "ent": "food", "cls": "dog"}, {"id": 3641, "dom": "P", "ent": "food", "cls": "elephant"}, {"id": 3642, "dom": "P", "ent": "food", "cls": "embassy"}, {"id": 3643, "dom": "P", "ent": "food", "cls": "garage"}, {"id": 3644, "dom": "P", "ent": "food", "cls": "highway"}, {"id": 3645, "dom": "P", "ent": "food", "cls": "kidney"}, {"id": 3646, "dom": "P", "ent": "food", "cls": "lamb"}, {"id": 3647, "dom": "P", "ent": "food", "cls": "male"}, {"id": 3648, "dom": "P", "ent": "food", "cls": "marijuana"}, {"id": 3649, "dom": "P", "ent": "food", "cls": "puppy"}, {"id": 3650, "dom": "P", "ent": "food", "cls": "receiver"}, {"id": 3651, "dom": "P", "ent": "food", "cls": "reed"}, {"id": 3652, "dom": "P", "ent": "food", "cls": "rifle"}, {"id": 3653, "dom": "P", "ent": "food", "cls": "road"}, {"id": 3654, "dom": "P", "ent": "food", "cls": "salmon"}, {"id": 3655, "dom": "P", "ent": "food", "cls": "shark"}, {"id": 3656, "dom": "P", "ent": "food", "cls": "sheep"}, {"id": 3657, "dom": "P", "ent": "food", "cls": "shook"}, {"id": 3658, "dom": "P", "ent": "food", "cls": "tank"}, {"id": 3659, "dom": "P", "ent": "food", "cls": "wheat"}, {"id": 3660, "dom": "P", "ent": "forehead", "cls": "architecture"}, {"id": 3661, "dom": "P", "ent": "forehead", "cls": "bacteria"}, {"id": 3662, "dom": "P", "ent": "forehead", "cls": "beef"}, {"id": 3663, "dom": "P", "ent": "forehead", "cls": "bridge"}, {"id": 3664, "dom": "P", "ent": "forehead", "cls": "bud"}, {"id": 3665, "dom": "P", "ent": "forehead", "cls": "cat"}, {"id": 3666, "dom": "P", "ent": "forehead", "cls": "chapel"}, {"id": 3667, "dom": "P", "ent": "forehead", "cls": "chick"}, {"id": 3668, "dom": "P", "ent": "forehead", "cls": "clock"}, {"id": 3669, "dom": "P", "ent": "forehead", "cls": "deer"}, {"id": 3670, "dom": "P", "ent": "forehead", "cls": "dog"}, {"id": 3671, "dom": "P", "ent": "forehead", "cls": "elephant"}, {"id": 3672, "dom": "P", "ent": "forehead", "cls": "embassy"}, {"id": 3673, "dom": "P", "ent": "forehead", "cls": "garage"}, {"id": 3674, "dom": "P", "ent": "forehead", "cls": "highway"}, {"id": 3675, "dom": "P", "ent": "forehead", "cls": "kidney"}, {"id": 3676, "dom": "P", "ent": "forehead", "cls": "lamb"}, {"id": 3677, "dom": "P", "ent": "forehead", "cls": "male"}, {"id": 3678, "dom": "P", "ent": "forehead", "cls": "marijuana"}, {"id": 3679, "dom": "P", "ent": "forehead", "cls": "puppy"}, {"id": 3680, "dom": "P", "ent": "forehead", "cls": "receiver"}, {"id": 3681, "dom": "P", "ent": "forehead", "cls": "reed"}, {"id": 3682, "dom": "P", "ent": "forehead", "cls": "rifle"}, {"id": 3683, "dom": "P", "ent": "forehead", "cls": "road"}, {"id": 3684, "dom": "P", "ent": "forehead", "cls": "salmon"}, {"id": 3685, "dom": "P", "ent": "forehead", "cls": "shark"}, {"id": 3686, "dom": "P", "ent": "forehead", "cls": "sheep"}, {"id": 3687, "dom": "P", "ent": "forehead", "cls": "shook"}, {"id": 3688, "dom": "P", "ent": "forehead", "cls": "tank"}, {"id": 3689, "dom": "P", "ent": "forehead", "cls": "wheat"}, {"id": 3690, "dom": "P", "ent": "foundation", "cls": "architecture"}, {"id": 3691, "dom": "P", "ent": "foundation", "cls": "bacteria"}, {"id": 3692, "dom": "P", "ent": "foundation", "cls": "beef"}, {"id": 3693, "dom": "P", "ent": "foundation", "cls": "bridge"}, {"id": 3694, "dom": "P", "ent": "foundation", "cls": "bud"}, {"id": 3695, "dom": "P", "ent": "foundation", "cls": "cat"}, {"id": 3696, "dom": "P", "ent": "foundation", "cls": "chapel"}, {"id": 3697, "dom": "P", "ent": "foundation", "cls": "chick"}, {"id": 3698, "dom": "P", "ent": "foundation", "cls": "clock"}, {"id": 3699, "dom": "P", "ent": "foundation", "cls": "deer"}, {"id": 3700, "dom": "P", "ent": "foundation", "cls": "dog"}, {"id": 3701, "dom": "P", "ent": "foundation", "cls": "elephant"}, {"id": 3702, "dom": "P", "ent": "foundation", "cls": "embassy"}, {"id": 3703, "dom": "P", "ent": "foundation", "cls": "garage"}, {"id": 3704, "dom": "P", "ent": "foundation", "cls": "highway"}, {"id": 3705, "dom": "P", "ent": "foundation", "cls": "kidney"}, {"id": 3706, "dom": "P", "ent": "foundation", "cls": "lamb"}, {"id": 3707, "dom": "P", "ent": "foundation", "cls": "male"}, {"id": 3708, "dom": "P", "ent": "foundation", "cls": "marijuana"}, {"id": 3709, "dom": "P", "ent": "foundation", "cls": "puppy"}, {"id": 3710, "dom": "P", "ent": "foundation", "cls": "receiver"}, {"id": 3711, "dom": "P", "ent": "foundation", "cls": "reed"}, {"id": 3712, "dom": "P", "ent": "foundation", "cls": "rifle"}, {"id": 3713, "dom": "P", "ent": "foundation", "cls": "road"}, {"id": 3714, "dom": "P", "ent": "foundation", "cls": "salmon"}, {"id": 3715, "dom": "P", "ent": "foundation", "cls": "shark"}, {"id": 3716, "dom": "P", "ent": "foundation", "cls": "sheep"}, {"id": 3717, "dom": "P", "ent": "foundation", "cls": "shook"}, {"id": 3718, "dom": "P", "ent": "foundation", "cls": "tank"}, {"id": 3719, "dom": "P", "ent": "foundation", "cls": "wheat"}, {"id": 3720, "dom": "P", "ent": "foyer", "cls": "architecture"}, {"id": 3721, "dom": "P", "ent": "foyer", "cls": "bacteria"}, {"id": 3722, "dom": "P", "ent": "foyer", "cls": "beef"}, {"id": 3723, "dom": "P", "ent": "foyer", "cls": "bridge"}, {"id": 3724, "dom": "P", "ent": "foyer", "cls": "bud"}, {"id": 3725, "dom": "P", "ent": "foyer", "cls": "cat"}, {"id": 3726, "dom": "P", "ent": "foyer", "cls": "chapel"}, {"id": 3727, "dom": "P", "ent": "foyer", "cls": "chick"}, {"id": 3728, "dom": "P", "ent": "foyer", "cls": "clock"}, {"id": 3729, "dom": "P", "ent": "foyer", "cls": "deer"}, {"id": 3730, "dom": "P", "ent": "foyer", "cls": "dog"}, {"id": 3731, "dom": "P", "ent": "foyer", "cls": "elephant"}, {"id": 3732, "dom": "P", "ent": "foyer", "cls": "embassy"}, {"id": 3733, "dom": "P", "ent": "foyer", "cls": "garage"}, {"id": 3734, "dom": "P", "ent": "foyer", "cls": "highway"}, {"id": 3735, "dom": "P", "ent": "foyer", "cls": "kidney"}, {"id": 3736, "dom": "P", "ent": "foyer", "cls": "lamb"}, {"id": 3737, "dom": "P", "ent": "foyer", "cls": "male"}, {"id": 3738, "dom": "P", "ent": "foyer", "cls": "marijuana"}, {"id": 3739, "dom": "P", "ent": "foyer", "cls": "puppy"}, {"id": 3740, "dom": "P", "ent": "foyer", "cls": "receiver"}, {"id": 3741, "dom": "P", "ent": "foyer", "cls": "reed"}, {"id": 3742, "dom": "P", "ent": "foyer", "cls": "rifle"}, {"id": 3743, "dom": "P", "ent": "foyer", "cls": "road"}, {"id": 3744, "dom": "P", "ent": "foyer", "cls": "salmon"}, {"id": 3745, "dom": "P", "ent": "foyer", "cls": "shark"}, {"id": 3746, "dom": "P", "ent": "foyer", "cls": "sheep"}, {"id": 3747, "dom": "P", "ent": "foyer", "cls": "shook"}, {"id": 3748, "dom": "P", "ent": "foyer", "cls": "tank"}, {"id": 3749, "dom": "P", "ent": "foyer", "cls": "wheat"}, {"id": 3750, "dom": "P", "ent": "frame", "cls": "architecture"}, {"id": 3751, "dom": "P", "ent": "frame", "cls": "bacteria"}, {"id": 3752, "dom": "P", "ent": "frame", "cls": "beef"}, {"id": 3753, "dom": "P", "ent": "frame", "cls": "bridge"}, {"id": 3754, "dom": "P", "ent": "frame", "cls": "bud"}, {"id": 3755, "dom": "P", "ent": "frame", "cls": "cat"}, {"id": 3756, "dom": "P", "ent": "frame", "cls": "chapel"}, {"id": 3757, "dom": "P", "ent": "frame", "cls": "chick"}, {"id": 3758, "dom": "P", "ent": "frame", "cls": "clock"}, {"id": 3759, "dom": "P", "ent": "frame", "cls": "deer"}, {"id": 3760, "dom": "P", "ent": "frame", "cls": "dog"}, {"id": 3761, "dom": "P", "ent": "frame", "cls": "elephant"}, {"id": 3762, "dom": "P", "ent": "frame", "cls": "embassy"}, {"id": 3763, "dom": "P", "ent": "frame", "cls": "garage"}, {"id": 3764, "dom": "P", "ent": "frame", "cls": "highway"}, {"id": 3765, "dom": "P", "ent": "frame", "cls": "kidney"}, {"id": 3766, "dom": "P", "ent": "frame", "cls": "lamb"}, {"id": 3767, "dom": "P", "ent": "frame", "cls": "male"}, {"id": 3768, "dom": "P", "ent": "frame", "cls": "marijuana"}, {"id": 3769, "dom": "P", "ent": "frame", "cls": "puppy"}, {"id": 3770, "dom": "P", "ent": "frame", "cls": "receiver"}, {"id": 3771, "dom": "P", "ent": "frame", "cls": "reed"}, {"id": 3772, "dom": "P", "ent": "frame", "cls": "rifle"}, {"id": 3773, "dom": "P", "ent": "frame", "cls": "road"}, {"id": 3774, "dom": "P", "ent": "frame", "cls": "salmon"}, {"id": 3775, "dom": "P", "ent": "frame", "cls": "shark"}, {"id": 3776, "dom": "P", "ent": "frame", "cls": "sheep"}, {"id": 3777, "dom": "P", "ent": "frame", "cls": "shook"}, {"id": 3778, "dom": "P", "ent": "frame", "cls": "tank"}, {"id": 3779, "dom": "P", "ent": "frame", "cls": "wheat"}, {"id": 3780, "dom": "P", "ent": "funeral-residence", "cls": "architecture"}, {"id": 3781, "dom": "P", "ent": "funeral-residence", "cls": "bacteria"}, {"id": 3782, "dom": "P", "ent": "funeral-residence", "cls": "beef"}, {"id": 3783, "dom": "P", "ent": "funeral-residence", "cls": "bridge"}, {"id": 3784, "dom": "P", "ent": "funeral-residence", "cls": "bud"}, {"id": 3785, "dom": "P", "ent": "funeral-residence", "cls": "cat"}, {"id": 3786, "dom": "P", "ent": "funeral-residence", "cls": "chapel"}, {"id": 3787, "dom": "P", "ent": "funeral-residence", "cls": "chick"}, {"id": 3788, "dom": "P", "ent": "funeral-residence", "cls": "clock"}, {"id": 3789, "dom": "P", "ent": "funeral-residence", "cls": "deer"}, {"id": 3790, "dom": "P", "ent": "funeral-residence", "cls": "dog"}, {"id": 3791, "dom": "P", "ent": "funeral-residence", "cls": "elephant"}, {"id": 3792, "dom": "P", "ent": "funeral-residence", "cls": "embassy"}, {"id": 3793, "dom": "P", "ent": "funeral-residence", "cls": "garage"}, {"id": 3794, "dom": "P", "ent": "funeral-residence", "cls": "highway"}, {"id": 3795, "dom": "P", "ent": "funeral-residence", "cls": "kidney"}, {"id": 3796, "dom": "P", "ent": "funeral-residence", "cls": "lamb"}, {"id": 3797, "dom": "P", "ent": "funeral-residence", "cls": "male"}, {"id": 3798, "dom": "P", "ent": "funeral-residence", "cls": "marijuana"}, {"id": 3799, "dom": "P", "ent": "funeral-residence", "cls": "puppy"}, {"id": 3800, "dom": "P", "ent": "funeral-residence", "cls": "receiver"}, {"id": 3801, "dom": "P", "ent": "funeral-residence", "cls": "reed"}, {"id": 3802, "dom": "P", "ent": "funeral-residence", "cls": "rifle"}, {"id": 3803, "dom": "P", "ent": "funeral-residence", "cls": "road"}, {"id": 3804, "dom": "P", "ent": "funeral-residence", "cls": "salmon"}, {"id": 3805, "dom": "P", "ent": "funeral-residence", "cls": "shark"}, {"id": 3806, "dom": "P", "ent": "funeral-residence", "cls": "sheep"}, {"id": 3807, "dom": "P", "ent": "funeral-residence", "cls": "shook"}, {"id": 3808, "dom": "P", "ent": "funeral-residence", "cls": "tank"}, {"id": 3809, "dom": "P", "ent": "funeral-residence", "cls": "wheat"}, {"id": 3810, "dom": "P", "ent": "fungus", "cls": "architecture"}, {"id": 3811, "dom": "P", "ent": "fungus", "cls": "bacteria"}, {"id": 3812, "dom": "P", "ent": "fungus", "cls": "beef"}, {"id": 3813, "dom": "P", "ent": "fungus", "cls": "bridge"}, {"id": 3814, "dom": "P", "ent": "fungus", "cls": "bud"}, {"id": 3815, "dom": "P", "ent": "fungus", "cls": "cat"}, {"id": 3816, "dom": "P", "ent": "fungus", "cls": "chapel"}, {"id": 3817, "dom": "P", "ent": "fungus", "cls": "chick"}, {"id": 3818, "dom": "P", "ent": "fungus", "cls": "clock"}, {"id": 3819, "dom": "P", "ent": "fungus", "cls": "deer"}, {"id": 3820, "dom": "P", "ent": "fungus", "cls": "dog"}, {"id": 3821, "dom": "P", "ent": "fungus", "cls": "elephant"}, {"id": 3822, "dom": "P", "ent": "fungus", "cls": "embassy"}, {"id": 3823, "dom": "P", "ent": "fungus", "cls": "garage"}, {"id": 3824, "dom": "P", "ent": "fungus", "cls": "highway"}, {"id": 3825, "dom": "P", "ent": "fungus", "cls": "kidney"}, {"id": 3826, "dom": "P", "ent": "fungus", "cls": "lamb"}, {"id": 3827, "dom": "P", "ent": "fungus", "cls": "male"}, {"id": 3828, "dom": "P", "ent": "fungus", "cls": "marijuana"}, {"id": 3829, "dom": "P", "ent": "fungus", "cls": "puppy"}, {"id": 3830, "dom": "P", "ent": "fungus", "cls": "receiver"}, {"id": 3831, "dom": "P", "ent": "fungus", "cls": "reed"}, {"id": 3832, "dom": "P", "ent": "fungus", "cls": "rifle"}, {"id": 3833, "dom": "P", "ent": "fungus", "cls": "road"}, {"id": 3834, "dom": "P", "ent": "fungus", "cls": "salmon"}, {"id": 3835, "dom": "P", "ent": "fungus", "cls": "shark"}, {"id": 3836, "dom": "P", "ent": "fungus", "cls": "sheep"}, {"id": 3837, "dom": "P", "ent": "fungus", "cls": "shook"}, {"id": 3838, "dom": "P", "ent": "fungus", "cls": "tank"}, {"id": 3839, "dom": "P", "ent": "fungus", "cls": "wheat"}, {"id": 3840, "dom": "P", "ent": "gene", "cls": "architecture"}, {"id": 3841, "dom": "P", "ent": "gene", "cls": "bacteria"}, {"id": 3842, "dom": "P", "ent": "gene", "cls": "beef"}, {"id": 3843, "dom": "P", "ent": "gene", "cls": "bridge"}, {"id": 3844, "dom": "P", "ent": "gene", "cls": "bud"}, {"id": 3845, "dom": "P", "ent": "gene", "cls": "cat"}, {"id": 3846, "dom": "P", "ent": "gene", "cls": "chapel"}, {"id": 3847, "dom": "P", "ent": "gene", "cls": "chick"}, {"id": 3848, "dom": "P", "ent": "gene", "cls": "clock"}, {"id": 3849, "dom": "P", "ent": "gene", "cls": "deer"}, {"id": 3850, "dom": "P", "ent": "gene", "cls": "dog"}, {"id": 3851, "dom": "P", "ent": "gene", "cls": "elephant"}, {"id": 3852, "dom": "P", "ent": "gene", "cls": "embassy"}, {"id": 3853, "dom": "P", "ent": "gene", "cls": "garage"}, {"id": 3854, "dom": "P", "ent": "gene", "cls": "highway"}, {"id": 3855, "dom": "P", "ent": "gene", "cls": "kidney"}, {"id": 3856, "dom": "P", "ent": "gene", "cls": "lamb"}, {"id": 3857, "dom": "P", "ent": "gene", "cls": "male"}, {"id": 3858, "dom": "P", "ent": "gene", "cls": "marijuana"}, {"id": 3859, "dom": "P", "ent": "gene", "cls": "puppy"}, {"id": 3860, "dom": "P", "ent": "gene", "cls": "receiver"}, {"id": 3861, "dom": "P", "ent": "gene", "cls": "reed"}, {"id": 3862, "dom": "P", "ent": "gene", "cls": "rifle"}, {"id": 3863, "dom": "P", "ent": "gene", "cls": "road"}, {"id": 3864, "dom": "P", "ent": "gene", "cls": "salmon"}, {"id": 3865, "dom": "P", "ent": "gene", "cls": "shark"}, {"id": 3866, "dom": "P", "ent": "gene", "cls": "sheep"}, {"id": 3867, "dom": "P", "ent": "gene", "cls": "shook"}, {"id": 3868, "dom": "P", "ent": "gene", "cls": "tank"}, {"id": 3869, "dom": "P", "ent": "gene", "cls": "wheat"}, {"id": 3870, "dom": "P", "ent": "god", "cls": "architecture"}, {"id": 3871, "dom": "P", "ent": "god", "cls": "bacteria"}, {"id": 3872, "dom": "P", "ent": "god", "cls": "beef"}, {"id": 3873, "dom": "P", "ent": "god", "cls": "bridge"}, {"id": 3874, "dom": "P", "ent": "god", "cls": "bud"}, {"id": 3875, "dom": "P", "ent": "god", "cls": "cat"}, {"id": 3876, "dom": "P", "ent": "god", "cls": "chapel"}, {"id": 3877, "dom": "P", "ent": "god", "cls": "chick"}, {"id": 3878, "dom": "P", "ent": "god", "cls": "clock"}, {"id": 3879, "dom": "P", "ent": "god", "cls": "deer"}, {"id": 3880, "dom": "P", "ent": "god", "cls": "dog"}, {"id": 3881, "dom": "P", "ent": "god", "cls": "elephant"}, {"id": 3882, "dom": "P", "ent": "god", "cls": "embassy"}, {"id": 3883, "dom": "P", "ent": "god", "cls": "garage"}, {"id": 3884, "dom": "P", "ent": "god", "cls": "highway"}, {"id": 3885, "dom": "P", "ent": "god", "cls": "kidney"}, {"id": 3886, "dom": "P", "ent": "god", "cls": "lamb"}, {"id": 3887, "dom": "P", "ent": "god", "cls": "male"}, {"id": 3888, "dom": "P", "ent": "god", "cls": "marijuana"}, {"id": 3889, "dom": "P", "ent": "god", "cls": "puppy"}, {"id": 3890, "dom": "P", "ent": "god", "cls": "receiver"}, {"id": 3891, "dom": "P", "ent": "god", "cls": "reed"}, {"id": 3892, "dom": "P", "ent": "god", "cls": "rifle"}, {"id": 3893, "dom": "P", "ent": "god", "cls": "road"}, {"id": 3894, "dom": "P", "ent": "god", "cls": "salmon"}, {"id": 3895, "dom": "P", "ent": "god", "cls": "shark"}, {"id": 3896, "dom": "P", "ent": "god", "cls": "sheep"}, {"id": 3897, "dom": "P", "ent": "god", "cls": "shook"}, {"id": 3898, "dom": "P", "ent": "god", "cls": "tank"}, {"id": 3899, "dom": "P", "ent": "god", "cls": "wheat"}, {"id": 3900, "dom": "P", "ent": "goose", "cls": "architecture"}, {"id": 3901, "dom": "P", "ent": "goose", "cls": "bacteria"}, {"id": 3902, "dom": "P", "ent": "goose", "cls": "beef"}, {"id": 3903, "dom": "P", "ent": "goose", "cls": "bridge"}, {"id": 3904, "dom": "P", "ent": "goose", "cls": "bud"}, {"id": 3905, "dom": "P", "ent": "goose", "cls": "cat"}, {"id": 3906, "dom": "P", "ent": "goose", "cls": "chapel"}, {"id": 3907, "dom": "P", "ent": "goose", "cls": "chick"}, {"id": 3908, "dom": "P", "ent": "goose", "cls": "clock"}, {"id": 3909, "dom": "P", "ent": "goose", "cls": "deer"}, {"id": 3910, "dom": "P", "ent": "goose", "cls": "dog"}, {"id": 3911, "dom": "P", "ent": "goose", "cls": "elephant"}, {"id": 3912, "dom": "P", "ent": "goose", "cls": "embassy"}, {"id": 3913, "dom": "P", "ent": "goose", "cls": "garage"}, {"id": 3914, "dom": "P", "ent": "goose", "cls": "highway"}, {"id": 3915, "dom": "P", "ent": "goose", "cls": "kidney"}, {"id": 3916, "dom": "P", "ent": "goose", "cls": "lamb"}, {"id": 3917, "dom": "P", "ent": "goose", "cls": "male"}, {"id": 3918, "dom": "P", "ent": "goose", "cls": "marijuana"}, {"id": 3919, "dom": "P", "ent": "goose", "cls": "puppy"}, {"id": 3920, "dom": "P", "ent": "goose", "cls": "receiver"}, {"id": 3921, "dom": "P", "ent": "goose", "cls": "reed"}, {"id": 3922, "dom": "P", "ent": "goose", "cls": "rifle"}, {"id": 3923, "dom": "P", "ent": "goose", "cls": "road"}, {"id": 3924, "dom": "P", "ent": "goose", "cls": "salmon"}, {"id": 3925, "dom": "P", "ent": "goose", "cls": "shark"}, {"id": 3926, "dom": "P", "ent": "goose", "cls": "sheep"}, {"id": 3927, "dom": "P", "ent": "goose", "cls": "shook"}, {"id": 3928, "dom": "P", "ent": "goose", "cls": "tank"}, {"id": 3929, "dom": "P", "ent": "goose", "cls": "wheat"}, {"id": 3930, "dom": "P", "ent": "graphic", "cls": "architecture"}, {"id": 3931, "dom": "P", "ent": "graphic", "cls": "bacteria"}, {"id": 3932, "dom": "P", "ent": "graphic", "cls": "beef"}, {"id": 3933, "dom": "P", "ent": "graphic", "cls": "bridge"}, {"id": 3934, "dom": "P", "ent": "graphic", "cls": "bud"}, {"id": 3935, "dom": "P", "ent": "graphic", "cls": "cat"}, {"id": 3936, "dom": "P", "ent": "graphic", "cls": "chapel"}, {"id": 3937, "dom": "P", "ent": "graphic", "cls": "chick"}, {"id": 3938, "dom": "P", "ent": "graphic", "cls": "clock"}, {"id": 3939, "dom": "P", "ent": "graphic", "cls": "deer"}, {"id": 3940, "dom": "P", "ent": "graphic", "cls": "dog"}, {"id": 3941, "dom": "P", "ent": "graphic", "cls": "elephant"}, {"id": 3942, "dom": "P", "ent": "graphic", "cls": "embassy"}, {"id": 3943, "dom": "P", "ent": "graphic", "cls": "garage"}, {"id": 3944, "dom": "P", "ent": "graphic", "cls": "highway"}, {"id": 3945, "dom": "P", "ent": "graphic", "cls": "kidney"}, {"id": 3946, "dom": "P", "ent": "graphic", "cls": "lamb"}, {"id": 3947, "dom": "P", "ent": "graphic", "cls": "male"}, {"id": 3948, "dom": "P", "ent": "graphic", "cls": "marijuana"}, {"id": 3949, "dom": "P", "ent": "graphic", "cls": "puppy"}, {"id": 3950, "dom": "P", "ent": "graphic", "cls": "receiver"}, {"id": 3951, "dom": "P", "ent": "graphic", "cls": "reed"}, {"id": 3952, "dom": "P", "ent": "graphic", "cls": "rifle"}, {"id": 3953, "dom": "P", "ent": "graphic", "cls": "road"}, {"id": 3954, "dom": "P", "ent": "graphic", "cls": "salmon"}, {"id": 3955, "dom": "P", "ent": "graphic", "cls": "shark"}, {"id": 3956, "dom": "P", "ent": "graphic", "cls": "sheep"}, {"id": 3957, "dom": "P", "ent": "graphic", "cls": "shook"}, {"id": 3958, "dom": "P", "ent": "graphic", "cls": "tank"}, {"id": 3959, "dom": "P", "ent": "graphic", "cls": "wheat"}, {"id": 3960, "dom": "P", "ent": "grid", "cls": "architecture"}, {"id": 3961, "dom": "P", "ent": "grid", "cls": "bacteria"}, {"id": 3962, "dom": "P", "ent": "grid", "cls": "beef"}, {"id": 3963, "dom": "P", "ent": "grid", "cls": "bridge"}, {"id": 3964, "dom": "P", "ent": "grid", "cls": "bud"}, {"id": 3965, "dom": "P", "ent": "grid", "cls": "cat"}, {"id": 3966, "dom": "P", "ent": "grid", "cls": "chapel"}, {"id": 3967, "dom": "P", "ent": "grid", "cls": "chick"}, {"id": 3968, "dom": "P", "ent": "grid", "cls": "clock"}, {"id": 3969, "dom": "P", "ent": "grid", "cls": "deer"}, {"id": 3970, "dom": "P", "ent": "grid", "cls": "dog"}, {"id": 3971, "dom": "P", "ent": "grid", "cls": "elephant"}, {"id": 3972, "dom": "P", "ent": "grid", "cls": "embassy"}, {"id": 3973, "dom": "P", "ent": "grid", "cls": "garage"}, {"id": 3974, "dom": "P", "ent": "grid", "cls": "highway"}, {"id": 3975, "dom": "P", "ent": "grid", "cls": "kidney"}, {"id": 3976, "dom": "P", "ent": "grid", "cls": "lamb"}, {"id": 3977, "dom": "P", "ent": "grid", "cls": "male"}, {"id": 3978, "dom": "P", "ent": "grid", "cls": "marijuana"}, {"id": 3979, "dom": "P", "ent": "grid", "cls": "puppy"}, {"id": 3980, "dom": "P", "ent": "grid", "cls": "receiver"}, {"id": 3981, "dom": "P", "ent": "grid", "cls": "reed"}, {"id": 3982, "dom": "P", "ent": "grid", "cls": "rifle"}, {"id": 3983, "dom": "P", "ent": "grid", "cls": "road"}, {"id": 3984, "dom": "P", "ent": "grid", "cls": "salmon"}, {"id": 3985, "dom": "P", "ent": "grid", "cls": "shark"}, {"id": 3986, "dom": "P", "ent": "grid", "cls": "sheep"}, {"id": 3987, "dom": "P", "ent": "grid", "cls": "shook"}, {"id": 3988, "dom": "P", "ent": "grid", "cls": "tank"}, {"id": 3989, "dom": "P", "ent": "grid", "cls": "wheat"}, {"id": 3990, "dom": "P", "ent": "groundwork", "cls": "architecture"}, {"id": 3991, "dom": "P", "ent": "groundwork", "cls": "bacteria"}, {"id": 3992, "dom": "P", "ent": "groundwork", "cls": "beef"}, {"id": 3993, "dom": "P", "ent": "groundwork", "cls": "bridge"}, {"id": 3994, "dom": "P", "ent": "groundwork", "cls": "bud"}, {"id": 3995, "dom": "P", "ent": "groundwork", "cls": "cat"}, {"id": 3996, "dom": "P", "ent": "groundwork", "cls": "chapel"}, {"id": 3997, "dom": "P", "ent": "groundwork", "cls": "chick"}, {"id": 3998, "dom": "P", "ent": "groundwork", "cls": "clock"}, {"id": 3999, "dom": "P", "ent": "groundwork", "cls": "deer"}, {"id": 4000, "dom": "P", "ent": "groundwork", "cls": "dog"}, {"id": 4001, "dom": "P", "ent": "groundwork", "cls": "elephant"}, {"id": 4002, "dom": "P", "ent": "groundwork", "cls": "embassy"}, {"id": 4003, "dom": "P", "ent": "groundwork", "cls": "garage"}, {"id": 4004, "dom": "P", "ent": "groundwork", "cls": "highway"}, {"id": 4005, "dom": "P", "ent": "groundwork", "cls": "kidney"}, {"id": 4006, "dom": "P", "ent": "groundwork", "cls": "lamb"}, {"id": 4007, "dom": "P", "ent": "groundwork", "cls": "male"}, {"id": 4008, "dom": "P", "ent": "groundwork", "cls": "marijuana"}, {"id": 4009, "dom": "P", "ent": "groundwork", "cls": "puppy"}, {"id": 4010, "dom": "P", "ent": "groundwork", "cls": "receiver"}, {"id": 4011, "dom": "P", "ent": "groundwork", "cls": "reed"}, {"id": 4012, "dom": "P", "ent": "groundwork", "cls": "rifle"}, {"id": 4013, "dom": "P", "ent": "groundwork", "cls": "road"}, {"id": 4014, "dom": "P", "ent": "groundwork", "cls": "salmon"}, {"id": 4015, "dom": "P", "ent": "groundwork", "cls": "shark"}, {"id": 4016, "dom": "P", "ent": "groundwork", "cls": "sheep"}, {"id": 4017, "dom": "P", "ent": "groundwork", "cls": "shook"}, {"id": 4018, "dom": "P", "ent": "groundwork", "cls": "tank"}, {"id": 4019, "dom": "P", "ent": "groundwork", "cls": "wheat"}, {"id": 4020, "dom": "P", "ent": "gum", "cls": "architecture"}, {"id": 4021, "dom": "P", "ent": "gum", "cls": "bacteria"}, {"id": 4022, "dom": "P", "ent": "gum", "cls": "beef"}, {"id": 4023, "dom": "P", "ent": "gum", "cls": "bridge"}, {"id": 4024, "dom": "P", "ent": "gum", "cls": "bud"}, {"id": 4025, "dom": "P", "ent": "gum", "cls": "cat"}, {"id": 4026, "dom": "P", "ent": "gum", "cls": "chapel"}, {"id": 4027, "dom": "P", "ent": "gum", "cls": "chick"}, {"id": 4028, "dom": "P", "ent": "gum", "cls": "clock"}, {"id": 4029, "dom": "P", "ent": "gum", "cls": "deer"}, {"id": 4030, "dom": "P", "ent": "gum", "cls": "dog"}, {"id": 4031, "dom": "P", "ent": "gum", "cls": "elephant"}, {"id": 4032, "dom": "P", "ent": "gum", "cls": "embassy"}, {"id": 4033, "dom": "P", "ent": "gum", "cls": "garage"}, {"id": 4034, "dom": "P", "ent": "gum", "cls": "highway"}, {"id": 4035, "dom": "P", "ent": "gum", "cls": "kidney"}, {"id": 4036, "dom": "P", "ent": "gum", "cls": "lamb"}, {"id": 4037, "dom": "P", "ent": "gum", "cls": "male"}, {"id": 4038, "dom": "P", "ent": "gum", "cls": "marijuana"}, {"id": 4039, "dom": "P", "ent": "gum", "cls": "puppy"}, {"id": 4040, "dom": "P", "ent": "gum", "cls": "receiver"}, {"id": 4041, "dom": "P", "ent": "gum", "cls": "reed"}, {"id": 4042, "dom": "P", "ent": "gum", "cls": "rifle"}, {"id": 4043, "dom": "P", "ent": "gum", "cls": "road"}, {"id": 4044, "dom": "P", "ent": "gum", "cls": "salmon"}, {"id": 4045, "dom": "P", "ent": "gum", "cls": "shark"}, {"id": 4046, "dom": "P", "ent": "gum", "cls": "sheep"}, {"id": 4047, "dom": "P", "ent": "gum", "cls": "shook"}, {"id": 4048, "dom": "P", "ent": "gum", "cls": "tank"}, {"id": 4049, "dom": "P", "ent": "gum", "cls": "wheat"}, {"id": 4050, "dom": "P", "ent": "gun-sight", "cls": "architecture"}, {"id": 4051, "dom": "P", "ent": "gun-sight", "cls": "bacteria"}, {"id": 4052, "dom": "P", "ent": "gun-sight", "cls": "beef"}, {"id": 4053, "dom": "P", "ent": "gun-sight", "cls": "bridge"}, {"id": 4054, "dom": "P", "ent": "gun-sight", "cls": "bud"}, {"id": 4055, "dom": "P", "ent": "gun-sight", "cls": "cat"}, {"id": 4056, "dom": "P", "ent": "gun-sight", "cls": "chapel"}, {"id": 4057, "dom": "P", "ent": "gun-sight", "cls": "chick"}, {"id": 4058, "dom": "P", "ent": "gun-sight", "cls": "clock"}, {"id": 4059, "dom": "P", "ent": "gun-sight", "cls": "deer"}, {"id": 4060, "dom": "P", "ent": "gun-sight", "cls": "dog"}, {"id": 4061, "dom": "P", "ent": "gun-sight", "cls": "elephant"}, {"id": 4062, "dom": "P", "ent": "gun-sight", "cls": "embassy"}, {"id": 4063, "dom": "P", "ent": "gun-sight", "cls": "garage"}, {"id": 4064, "dom": "P", "ent": "gun-sight", "cls": "highway"}, {"id": 4065, "dom": "P", "ent": "gun-sight", "cls": "kidney"}, {"id": 4066, "dom": "P", "ent": "gun-sight", "cls": "lamb"}, {"id": 4067, "dom": "P", "ent": "gun-sight", "cls": "male"}, {"id": 4068, "dom": "P", "ent": "gun-sight", "cls": "marijuana"}, {"id": 4069, "dom": "P", "ent": "gun-sight", "cls": "puppy"}, {"id": 4070, "dom": "P", "ent": "gun-sight", "cls": "receiver"}, {"id": 4071, "dom": "P", "ent": "gun-sight", "cls": "reed"}, {"id": 4072, "dom": "P", "ent": "gun-sight", "cls": "rifle"}, {"id": 4073, "dom": "P", "ent": "gun-sight", "cls": "road"}, {"id": 4074, "dom": "P", "ent": "gun-sight", "cls": "salmon"}, {"id": 4075, "dom": "P", "ent": "gun-sight", "cls": "shark"}, {"id": 4076, "dom": "P", "ent": "gun-sight", "cls": "sheep"}, {"id": 4077, "dom": "P", "ent": "gun-sight", "cls": "shook"}, {"id": 4078, "dom": "P", "ent": "gun-sight", "cls": "tank"}, {"id": 4079, "dom": "P", "ent": "gun-sight", "cls": "wheat"}, {"id": 4080, "dom": "P", "ent": "hair", "cls": "architecture"}, {"id": 4081, "dom": "P", "ent": "hair", "cls": "bacteria"}, {"id": 4082, "dom": "P", "ent": "hair", "cls": "beef"}, {"id": 4083, "dom": "P", "ent": "hair", "cls": "bridge"}, {"id": 4084, "dom": "P", "ent": "hair", "cls": "bud"}, {"id": 4085, "dom": "P", "ent": "hair", "cls": "cat"}, {"id": 4086, "dom": "P", "ent": "hair", "cls": "chapel"}, {"id": 4087, "dom": "P", "ent": "hair", "cls": "chick"}, {"id": 4088, "dom": "P", "ent": "hair", "cls": "clock"}, {"id": 4089, "dom": "P", "ent": "hair", "cls": "deer"}, {"id": 4090, "dom": "P", "ent": "hair", "cls": "dog"}, {"id": 4091, "dom": "P", "ent": "hair", "cls": "elephant"}, {"id": 4092, "dom": "P", "ent": "hair", "cls": "embassy"}, {"id": 4093, "dom": "P", "ent": "hair", "cls": "garage"}, {"id": 4094, "dom": "P", "ent": "hair", "cls": "highway"}, {"id": 4095, "dom": "P", "ent": "hair", "cls": "kidney"}, {"id": 4096, "dom": "P", "ent": "hair", "cls": "lamb"}, {"id": 4097, "dom": "P", "ent": "hair", "cls": "male"}, {"id": 4098, "dom": "P", "ent": "hair", "cls": "marijuana"}, {"id": 4099, "dom": "P", "ent": "hair", "cls": "puppy"}, {"id": 4100, "dom": "P", "ent": "hair", "cls": "receiver"}, {"id": 4101, "dom": "P", "ent": "hair", "cls": "reed"}, {"id": 4102, "dom": "P", "ent": "hair", "cls": "rifle"}, {"id": 4103, "dom": "P", "ent": "hair", "cls": "road"}, {"id": 4104, "dom": "P", "ent": "hair", "cls": "salmon"}, {"id": 4105, "dom": "P", "ent": "hair", "cls": "shark"}, {"id": 4106, "dom": "P", "ent": "hair", "cls": "sheep"}, {"id": 4107, "dom": "P", "ent": "hair", "cls": "shook"}, {"id": 4108, "dom": "P", "ent": "hair", "cls": "tank"}, {"id": 4109, "dom": "P", "ent": "hair", "cls": "wheat"}, {"id": 4110, "dom": "P", "ent": "half-moon", "cls": "architecture"}, {"id": 4111, "dom": "P", "ent": "half-moon", "cls": "bacteria"}, {"id": 4112, "dom": "P", "ent": "half-moon", "cls": "beef"}, {"id": 4113, "dom": "P", "ent": "half-moon", "cls": "bridge"}, {"id": 4114, "dom": "P", "ent": "half-moon", "cls": "bud"}, {"id": 4115, "dom": "P", "ent": "half-moon", "cls": "cat"}, {"id": 4116, "dom": "P", "ent": "half-moon", "cls": "chapel"}, {"id": 4117, "dom": "P", "ent": "half-moon", "cls": "chick"}, {"id": 4118, "dom": "P", "ent": "half-moon", "cls": "clock"}, {"id": 4119, "dom": "P", "ent": "half-moon", "cls": "deer"}, {"id": 4120, "dom": "P", "ent": "half-moon", "cls": "dog"}, {"id": 4121, "dom": "P", "ent": "half-moon", "cls": "elephant"}, {"id": 4122, "dom": "P", "ent": "half-moon", "cls": "embassy"}, {"id": 4123, "dom": "P", "ent": "half-moon", "cls": "garage"}, {"id": 4124, "dom": "P", "ent": "half-moon", "cls": "highway"}, {"id": 4125, "dom": "P", "ent": "half-moon", "cls": "kidney"}, {"id": 4126, "dom": "P", "ent": "half-moon", "cls": "lamb"}, {"id": 4127, "dom": "P", "ent": "half-moon", "cls": "male"}, {"id": 4128, "dom": "P", "ent": "half-moon", "cls": "marijuana"}, {"id": 4129, "dom": "P", "ent": "half-moon", "cls": "puppy"}, {"id": 4130, "dom": "P", "ent": "half-moon", "cls": "receiver"}, {"id": 4131, "dom": "P", "ent": "half-moon", "cls": "reed"}, {"id": 4132, "dom": "P", "ent": "half-moon", "cls": "rifle"}, {"id": 4133, "dom": "P", "ent": "half-moon", "cls": "road"}, {"id": 4134, "dom": "P", "ent": "half-moon", "cls": "salmon"}, {"id": 4135, "dom": "P", "ent": "half-moon", "cls": "shark"}, {"id": 4136, "dom": "P", "ent": "half-moon", "cls": "sheep"}, {"id": 4137, "dom": "P", "ent": "half-moon", "cls": "shook"}, {"id": 4138, "dom": "P", "ent": "half-moon", "cls": "tank"}, {"id": 4139, "dom": "P", "ent": "half-moon", "cls": "wheat"}, {"id": 4140, "dom": "P", "ent": "hall", "cls": "architecture"}, {"id": 4141, "dom": "P", "ent": "hall", "cls": "bacteria"}, {"id": 4142, "dom": "P", "ent": "hall", "cls": "beef"}, {"id": 4143, "dom": "P", "ent": "hall", "cls": "bridge"}, {"id": 4144, "dom": "P", "ent": "hall", "cls": "bud"}, {"id": 4145, "dom": "P", "ent": "hall", "cls": "cat"}, {"id": 4146, "dom": "P", "ent": "hall", "cls": "chapel"}, {"id": 4147, "dom": "P", "ent": "hall", "cls": "chick"}, {"id": 4148, "dom": "P", "ent": "hall", "cls": "clock"}, {"id": 4149, "dom": "P", "ent": "hall", "cls": "deer"}, {"id": 4150, "dom": "P", "ent": "hall", "cls": "dog"}, {"id": 4151, "dom": "P", "ent": "hall", "cls": "elephant"}, {"id": 4152, "dom": "P", "ent": "hall", "cls": "embassy"}, {"id": 4153, "dom": "P", "ent": "hall", "cls": "garage"}, {"id": 4154, "dom": "P", "ent": "hall", "cls": "highway"}, {"id": 4155, "dom": "P", "ent": "hall", "cls": "kidney"}, {"id": 4156, "dom": "P", "ent": "hall", "cls": "lamb"}, {"id": 4157, "dom": "P", "ent": "hall", "cls": "male"}, {"id": 4158, "dom": "P", "ent": "hall", "cls": "marijuana"}, {"id": 4159, "dom": "P", "ent": "hall", "cls": "puppy"}, {"id": 4160, "dom": "P", "ent": "hall", "cls": "receiver"}, {"id": 4161, "dom": "P", "ent": "hall", "cls": "reed"}, {"id": 4162, "dom": "P", "ent": "hall", "cls": "rifle"}, {"id": 4163, "dom": "P", "ent": "hall", "cls": "road"}, {"id": 4164, "dom": "P", "ent": "hall", "cls": "salmon"}, {"id": 4165, "dom": "P", "ent": "hall", "cls": "shark"}, {"id": 4166, "dom": "P", "ent": "hall", "cls": "sheep"}, {"id": 4167, "dom": "P", "ent": "hall", "cls": "shook"}, {"id": 4168, "dom": "P", "ent": "hall", "cls": "tank"}, {"id": 4169, "dom": "P", "ent": "hall", "cls": "wheat"}, {"id": 4170, "dom": "P", "ent": "hammer", "cls": "architecture"}, {"id": 4171, "dom": "P", "ent": "hammer", "cls": "bacteria"}, {"id": 4172, "dom": "P", "ent": "hammer", "cls": "beef"}, {"id": 4173, "dom": "P", "ent": "hammer", "cls": "bridge"}, {"id": 4174, "dom": "P", "ent": "hammer", "cls": "bud"}, {"id": 4175, "dom": "P", "ent": "hammer", "cls": "cat"}, {"id": 4176, "dom": "P", "ent": "hammer", "cls": "chapel"}, {"id": 4177, "dom": "P", "ent": "hammer", "cls": "chick"}, {"id": 4178, "dom": "P", "ent": "hammer", "cls": "clock"}, {"id": 4179, "dom": "P", "ent": "hammer", "cls": "deer"}, {"id": 4180, "dom": "P", "ent": "hammer", "cls": "dog"}, {"id": 4181, "dom": "P", "ent": "hammer", "cls": "elephant"}, {"id": 4182, "dom": "P", "ent": "hammer", "cls": "embassy"}, {"id": 4183, "dom": "P", "ent": "hammer", "cls": "garage"}, {"id": 4184, "dom": "P", "ent": "hammer", "cls": "highway"}, {"id": 4185, "dom": "P", "ent": "hammer", "cls": "kidney"}, {"id": 4186, "dom": "P", "ent": "hammer", "cls": "lamb"}, {"id": 4187, "dom": "P", "ent": "hammer", "cls": "male"}, {"id": 4188, "dom": "P", "ent": "hammer", "cls": "marijuana"}, {"id": 4189, "dom": "P", "ent": "hammer", "cls": "puppy"}, {"id": 4190, "dom": "P", "ent": "hammer", "cls": "receiver"}, {"id": 4191, "dom": "P", "ent": "hammer", "cls": "reed"}, {"id": 4192, "dom": "P", "ent": "hammer", "cls": "rifle"}, {"id": 4193, "dom": "P", "ent": "hammer", "cls": "road"}, {"id": 4194, "dom": "P", "ent": "hammer", "cls": "salmon"}, {"id": 4195, "dom": "P", "ent": "hammer", "cls": "shark"}, {"id": 4196, "dom": "P", "ent": "hammer", "cls": "sheep"}, {"id": 4197, "dom": "P", "ent": "hammer", "cls": "shook"}, {"id": 4198, "dom": "P", "ent": "hammer", "cls": "tank"}, {"id": 4199, "dom": "P", "ent": "hammer", "cls": "wheat"}, {"id": 4200, "dom": "P", "ent": "header", "cls": "architecture"}, {"id": 4201, "dom": "P", "ent": "header", "cls": "bacteria"}, {"id": 4202, "dom": "P", "ent": "header", "cls": "beef"}, {"id": 4203, "dom": "P", "ent": "header", "cls": "bridge"}, {"id": 4204, "dom": "P", "ent": "header", "cls": "bud"}, {"id": 4205, "dom": "P", "ent": "header", "cls": "cat"}, {"id": 4206, "dom": "P", "ent": "header", "cls": "chapel"}, {"id": 4207, "dom": "P", "ent": "header", "cls": "chick"}, {"id": 4208, "dom": "P", "ent": "header", "cls": "clock"}, {"id": 4209, "dom": "P", "ent": "header", "cls": "deer"}, {"id": 4210, "dom": "P", "ent": "header", "cls": "dog"}, {"id": 4211, "dom": "P", "ent": "header", "cls": "elephant"}, {"id": 4212, "dom": "P", "ent": "header", "cls": "embassy"}, {"id": 4213, "dom": "P", "ent": "header", "cls": "garage"}, {"id": 4214, "dom": "P", "ent": "header", "cls": "highway"}, {"id": 4215, "dom": "P", "ent": "header", "cls": "kidney"}, {"id": 4216, "dom": "P", "ent": "header", "cls": "lamb"}, {"id": 4217, "dom": "P", "ent": "header", "cls": "male"}, {"id": 4218, "dom": "P", "ent": "header", "cls": "marijuana"}, {"id": 4219, "dom": "P", "ent": "header", "cls": "puppy"}, {"id": 4220, "dom": "P", "ent": "header", "cls": "receiver"}, {"id": 4221, "dom": "P", "ent": "header", "cls": "reed"}, {"id": 4222, "dom": "P", "ent": "header", "cls": "rifle"}, {"id": 4223, "dom": "P", "ent": "header", "cls": "road"}, {"id": 4224, "dom": "P", "ent": "header", "cls": "salmon"}, {"id": 4225, "dom": "P", "ent": "header", "cls": "shark"}, {"id": 4226, "dom": "P", "ent": "header", "cls": "sheep"}, {"id": 4227, "dom": "P", "ent": "header", "cls": "shook"}, {"id": 4228, "dom": "P", "ent": "header", "cls": "tank"}, {"id": 4229, "dom": "P", "ent": "header", "cls": "wheat"}, {"id": 4230, "dom": "P", "ent": "heading", "cls": "architecture"}, {"id": 4231, "dom": "P", "ent": "heading", "cls": "bacteria"}, {"id": 4232, "dom": "P", "ent": "heading", "cls": "beef"}, {"id": 4233, "dom": "P", "ent": "heading", "cls": "bridge"}, {"id": 4234, "dom": "P", "ent": "heading", "cls": "bud"}, {"id": 4235, "dom": "P", "ent": "heading", "cls": "cat"}, {"id": 4236, "dom": "P", "ent": "heading", "cls": "chapel"}, {"id": 4237, "dom": "P", "ent": "heading", "cls": "chick"}, {"id": 4238, "dom": "P", "ent": "heading", "cls": "clock"}, {"id": 4239, "dom": "P", "ent": "heading", "cls": "deer"}, {"id": 4240, "dom": "P", "ent": "heading", "cls": "dog"}, {"id": 4241, "dom": "P", "ent": "heading", "cls": "elephant"}, {"id": 4242, "dom": "P", "ent": "heading", "cls": "embassy"}, {"id": 4243, "dom": "P", "ent": "heading", "cls": "garage"}, {"id": 4244, "dom": "P", "ent": "heading", "cls": "highway"}, {"id": 4245, "dom": "P", "ent": "heading", "cls": "kidney"}, {"id": 4246, "dom": "P", "ent": "heading", "cls": "lamb"}, {"id": 4247, "dom": "P", "ent": "heading", "cls": "male"}, {"id": 4248, "dom": "P", "ent": "heading", "cls": "marijuana"}, {"id": 4249, "dom": "P", "ent": "heading", "cls": "puppy"}, {"id": 4250, "dom": "P", "ent": "heading", "cls": "receiver"}, {"id": 4251, "dom": "P", "ent": "heading", "cls": "reed"}, {"id": 4252, "dom": "P", "ent": "heading", "cls": "rifle"}, {"id": 4253, "dom": "P", "ent": "heading", "cls": "road"}, {"id": 4254, "dom": "P", "ent": "heading", "cls": "salmon"}, {"id": 4255, "dom": "P", "ent": "heading", "cls": "shark"}, {"id": 4256, "dom": "P", "ent": "heading", "cls": "sheep"}, {"id": 4257, "dom": "P", "ent": "heading", "cls": "shook"}, {"id": 4258, "dom": "P", "ent": "heading", "cls": "tank"}, {"id": 4259, "dom": "P", "ent": "heading", "cls": "wheat"}, {"id": 4260, "dom": "P", "ent": "heat", "cls": "architecture"}, {"id": 4261, "dom": "P", "ent": "heat", "cls": "bacteria"}, {"id": 4262, "dom": "P", "ent": "heat", "cls": "beef"}, {"id": 4263, "dom": "P", "ent": "heat", "cls": "bridge"}, {"id": 4264, "dom": "P", "ent": "heat", "cls": "bud"}, {"id": 4265, "dom": "P", "ent": "heat", "cls": "cat"}, {"id": 4266, "dom": "P", "ent": "heat", "cls": "chapel"}, {"id": 4267, "dom": "P", "ent": "heat", "cls": "chick"}, {"id": 4268, "dom": "P", "ent": "heat", "cls": "clock"}, {"id": 4269, "dom": "P", "ent": "heat", "cls": "deer"}, {"id": 4270, "dom": "P", "ent": "heat", "cls": "dog"}, {"id": 4271, "dom": "P", "ent": "heat", "cls": "elephant"}, {"id": 4272, "dom": "P", "ent": "heat", "cls": "embassy"}, {"id": 4273, "dom": "P", "ent": "heat", "cls": "garage"}, {"id": 4274, "dom": "P", "ent": "heat", "cls": "highway"}, {"id": 4275, "dom": "P", "ent": "heat", "cls": "kidney"}, {"id": 4276, "dom": "P", "ent": "heat", "cls": "lamb"}, {"id": 4277, "dom": "P", "ent": "heat", "cls": "male"}, {"id": 4278, "dom": "P", "ent": "heat", "cls": "marijuana"}, {"id": 4279, "dom": "P", "ent": "heat", "cls": "puppy"}, {"id": 4280, "dom": "P", "ent": "heat", "cls": "receiver"}, {"id": 4281, "dom": "P", "ent": "heat", "cls": "reed"}, {"id": 4282, "dom": "P", "ent": "heat", "cls": "rifle"}, {"id": 4283, "dom": "P", "ent": "heat", "cls": "road"}, {"id": 4284, "dom": "P", "ent": "heat", "cls": "salmon"}, {"id": 4285, "dom": "P", "ent": "heat", "cls": "shark"}, {"id": 4286, "dom": "P", "ent": "heat", "cls": "sheep"}, {"id": 4287, "dom": "P", "ent": "heat", "cls": "shook"}, {"id": 4288, "dom": "P", "ent": "heat", "cls": "tank"}, {"id": 4289, "dom": "P", "ent": "heat", "cls": "wheat"}, {"id": 4290, "dom": "P", "ent": "heating", "cls": "architecture"}, {"id": 4291, "dom": "P", "ent": "heating", "cls": "bacteria"}, {"id": 4292, "dom": "P", "ent": "heating", "cls": "beef"}, {"id": 4293, "dom": "P", "ent": "heating", "cls": "bridge"}, {"id": 4294, "dom": "P", "ent": "heating", "cls": "bud"}, {"id": 4295, "dom": "P", "ent": "heating", "cls": "cat"}, {"id": 4296, "dom": "P", "ent": "heating", "cls": "chapel"}, {"id": 4297, "dom": "P", "ent": "heating", "cls": "chick"}, {"id": 4298, "dom": "P", "ent": "heating", "cls": "clock"}, {"id": 4299, "dom": "P", "ent": "heating", "cls": "deer"}, {"id": 4300, "dom": "P", "ent": "heating", "cls": "dog"}, {"id": 4301, "dom": "P", "ent": "heating", "cls": "elephant"}, {"id": 4302, "dom": "P", "ent": "heating", "cls": "embassy"}, {"id": 4303, "dom": "P", "ent": "heating", "cls": "garage"}, {"id": 4304, "dom": "P", "ent": "heating", "cls": "highway"}, {"id": 4305, "dom": "P", "ent": "heating", "cls": "kidney"}, {"id": 4306, "dom": "P", "ent": "heating", "cls": "lamb"}, {"id": 4307, "dom": "P", "ent": "heating", "cls": "male"}, {"id": 4308, "dom": "P", "ent": "heating", "cls": "marijuana"}, {"id": 4309, "dom": "P", "ent": "heating", "cls": "puppy"}, {"id": 4310, "dom": "P", "ent": "heating", "cls": "receiver"}, {"id": 4311, "dom": "P", "ent": "heating", "cls": "reed"}, {"id": 4312, "dom": "P", "ent": "heating", "cls": "rifle"}, {"id": 4313, "dom": "P", "ent": "heating", "cls": "road"}, {"id": 4314, "dom": "P", "ent": "heating", "cls": "salmon"}, {"id": 4315, "dom": "P", "ent": "heating", "cls": "shark"}, {"id": 4316, "dom": "P", "ent": "heating", "cls": "sheep"}, {"id": 4317, "dom": "P", "ent": "heating", "cls": "shook"}, {"id": 4318, "dom": "P", "ent": "heating", "cls": "tank"}, {"id": 4319, "dom": "P", "ent": "heating", "cls": "wheat"}, {"id": 4320, "dom": "P", "ent": "hemisphere", "cls": "architecture"}, {"id": 4321, "dom": "P", "ent": "hemisphere", "cls": "bacteria"}, {"id": 4322, "dom": "P", "ent": "hemisphere", "cls": "beef"}, {"id": 4323, "dom": "P", "ent": "hemisphere", "cls": "bridge"}, {"id": 4324, "dom": "P", "ent": "hemisphere", "cls": "bud"}, {"id": 4325, "dom": "P", "ent": "hemisphere", "cls": "cat"}, {"id": 4326, "dom": "P", "ent": "hemisphere", "cls": "chapel"}, {"id": 4327, "dom": "P", "ent": "hemisphere", "cls": "chick"}, {"id": 4328, "dom": "P", "ent": "hemisphere", "cls": "clock"}, {"id": 4329, "dom": "P", "ent": "hemisphere", "cls": "deer"}, {"id": 4330, "dom": "P", "ent": "hemisphere", "cls": "dog"}, {"id": 4331, "dom": "P", "ent": "hemisphere", "cls": "elephant"}, {"id": 4332, "dom": "P", "ent": "hemisphere", "cls": "embassy"}, {"id": 4333, "dom": "P", "ent": "hemisphere", "cls": "garage"}, {"id": 4334, "dom": "P", "ent": "hemisphere", "cls": "highway"}, {"id": 4335, "dom": "P", "ent": "hemisphere", "cls": "kidney"}, {"id": 4336, "dom": "P", "ent": "hemisphere", "cls": "lamb"}, {"id": 4337, "dom": "P", "ent": "hemisphere", "cls": "male"}, {"id": 4338, "dom": "P", "ent": "hemisphere", "cls": "marijuana"}, {"id": 4339, "dom": "P", "ent": "hemisphere", "cls": "puppy"}, {"id": 4340, "dom": "P", "ent": "hemisphere", "cls": "receiver"}, {"id": 4341, "dom": "P", "ent": "hemisphere", "cls": "reed"}, {"id": 4342, "dom": "P", "ent": "hemisphere", "cls": "rifle"}, {"id": 4343, "dom": "P", "ent": "hemisphere", "cls": "road"}, {"id": 4344, "dom": "P", "ent": "hemisphere", "cls": "salmon"}, {"id": 4345, "dom": "P", "ent": "hemisphere", "cls": "shark"}, {"id": 4346, "dom": "P", "ent": "hemisphere", "cls": "sheep"}, {"id": 4347, "dom": "P", "ent": "hemisphere", "cls": "shook"}, {"id": 4348, "dom": "P", "ent": "hemisphere", "cls": "tank"}, {"id": 4349, "dom": "P", "ent": "hemisphere", "cls": "wheat"}, {"id": 4350, "dom": "P", "ent": "hen", "cls": "architecture"}, {"id": 4351, "dom": "P", "ent": "hen", "cls": "bacteria"}, {"id": 4352, "dom": "P", "ent": "hen", "cls": "beef"}, {"id": 4353, "dom": "P", "ent": "hen", "cls": "bridge"}, {"id": 4354, "dom": "P", "ent": "hen", "cls": "bud"}, {"id": 4355, "dom": "P", "ent": "hen", "cls": "cat"}, {"id": 4356, "dom": "P", "ent": "hen", "cls": "chapel"}, {"id": 4357, "dom": "P", "ent": "hen", "cls": "chick"}, {"id": 4358, "dom": "P", "ent": "hen", "cls": "clock"}, {"id": 4359, "dom": "P", "ent": "hen", "cls": "deer"}, {"id": 4360, "dom": "P", "ent": "hen", "cls": "dog"}, {"id": 4361, "dom": "P", "ent": "hen", "cls": "elephant"}, {"id": 4362, "dom": "P", "ent": "hen", "cls": "embassy"}, {"id": 4363, "dom": "P", "ent": "hen", "cls": "garage"}, {"id": 4364, "dom": "P", "ent": "hen", "cls": "highway"}, {"id": 4365, "dom": "P", "ent": "hen", "cls": "kidney"}, {"id": 4366, "dom": "P", "ent": "hen", "cls": "lamb"}, {"id": 4367, "dom": "P", "ent": "hen", "cls": "male"}, {"id": 4368, "dom": "P", "ent": "hen", "cls": "marijuana"}, {"id": 4369, "dom": "P", "ent": "hen", "cls": "puppy"}, {"id": 4370, "dom": "P", "ent": "hen", "cls": "receiver"}, {"id": 4371, "dom": "P", "ent": "hen", "cls": "reed"}, {"id": 4372, "dom": "P", "ent": "hen", "cls": "rifle"}, {"id": 4373, "dom": "P", "ent": "hen", "cls": "road"}, {"id": 4374, "dom": "P", "ent": "hen", "cls": "salmon"}, {"id": 4375, "dom": "P", "ent": "hen", "cls": "shark"}, {"id": 4376, "dom": "P", "ent": "hen", "cls": "sheep"}, {"id": 4377, "dom": "P", "ent": "hen", "cls": "shook"}, {"id": 4378, "dom": "P", "ent": "hen", "cls": "tank"}, {"id": 4379, "dom": "P", "ent": "hen", "cls": "wheat"}, {"id": 4380, "dom": "P", "ent": "hind", "cls": "architecture"}, {"id": 4381, "dom": "P", "ent": "hind", "cls": "bacteria"}, {"id": 4382, "dom": "P", "ent": "hind", "cls": "beef"}, {"id": 4383, "dom": "P", "ent": "hind", "cls": "bridge"}, {"id": 4384, "dom": "P", "ent": "hind", "cls": "bud"}, {"id": 4385, "dom": "P", "ent": "hind", "cls": "cat"}, {"id": 4386, "dom": "P", "ent": "hind", "cls": "chapel"}, {"id": 4387, "dom": "P", "ent": "hind", "cls": "chick"}, {"id": 4388, "dom": "P", "ent": "hind", "cls": "clock"}, {"id": 4389, "dom": "P", "ent": "hind", "cls": "deer"}, {"id": 4390, "dom": "P", "ent": "hind", "cls": "dog"}, {"id": 4391, "dom": "P", "ent": "hind", "cls": "elephant"}, {"id": 4392, "dom": "P", "ent": "hind", "cls": "embassy"}, {"id": 4393, "dom": "P", "ent": "hind", "cls": "garage"}, {"id": 4394, "dom": "P", "ent": "hind", "cls": "highway"}, {"id": 4395, "dom": "P", "ent": "hind", "cls": "kidney"}, {"id": 4396, "dom": "P", "ent": "hind", "cls": "lamb"}, {"id": 4397, "dom": "P", "ent": "hind", "cls": "male"}, {"id": 4398, "dom": "P", "ent": "hind", "cls": "marijuana"}, {"id": 4399, "dom": "P", "ent": "hind", "cls": "puppy"}, {"id": 4400, "dom": "P", "ent": "hind", "cls": "receiver"}, {"id": 4401, "dom": "P", "ent": "hind", "cls": "reed"}, {"id": 4402, "dom": "P", "ent": "hind", "cls": "rifle"}, {"id": 4403, "dom": "P", "ent": "hind", "cls": "road"}, {"id": 4404, "dom": "P", "ent": "hind", "cls": "salmon"}, {"id": 4405, "dom": "P", "ent": "hind", "cls": "shark"}, {"id": 4406, "dom": "P", "ent": "hind", "cls": "sheep"}, {"id": 4407, "dom": "P", "ent": "hind", "cls": "shook"}, {"id": 4408, "dom": "P", "ent": "hind", "cls": "tank"}, {"id": 4409, "dom": "P", "ent": "hind", "cls": "wheat"}, {"id": 4410, "dom": "P", "ent": "hinge", "cls": "architecture"}, {"id": 4411, "dom": "P", "ent": "hinge", "cls": "bacteria"}, {"id": 4412, "dom": "P", "ent": "hinge", "cls": "beef"}, {"id": 4413, "dom": "P", "ent": "hinge", "cls": "bridge"}, {"id": 4414, "dom": "P", "ent": "hinge", "cls": "bud"}, {"id": 4415, "dom": "P", "ent": "hinge", "cls": "cat"}, {"id": 4416, "dom": "P", "ent": "hinge", "cls": "chapel"}, {"id": 4417, "dom": "P", "ent": "hinge", "cls": "chick"}, {"id": 4418, "dom": "P", "ent": "hinge", "cls": "clock"}, {"id": 4419, "dom": "P", "ent": "hinge", "cls": "deer"}, {"id": 4420, "dom": "P", "ent": "hinge", "cls": "dog"}, {"id": 4421, "dom": "P", "ent": "hinge", "cls": "elephant"}, {"id": 4422, "dom": "P", "ent": "hinge", "cls": "embassy"}, {"id": 4423, "dom": "P", "ent": "hinge", "cls": "garage"}, {"id": 4424, "dom": "P", "ent": "hinge", "cls": "highway"}, {"id": 4425, "dom": "P", "ent": "hinge", "cls": "kidney"}, {"id": 4426, "dom": "P", "ent": "hinge", "cls": "lamb"}, {"id": 4427, "dom": "P", "ent": "hinge", "cls": "male"}, {"id": 4428, "dom": "P", "ent": "hinge", "cls": "marijuana"}, {"id": 4429, "dom": "P", "ent": "hinge", "cls": "puppy"}, {"id": 4430, "dom": "P", "ent": "hinge", "cls": "receiver"}, {"id": 4431, "dom": "P", "ent": "hinge", "cls": "reed"}, {"id": 4432, "dom": "P", "ent": "hinge", "cls": "rifle"}, {"id": 4433, "dom": "P", "ent": "hinge", "cls": "road"}, {"id": 4434, "dom": "P", "ent": "hinge", "cls": "salmon"}, {"id": 4435, "dom": "P", "ent": "hinge", "cls": "shark"}, {"id": 4436, "dom": "P", "ent": "hinge", "cls": "sheep"}, {"id": 4437, "dom": "P", "ent": "hinge", "cls": "shook"}, {"id": 4438, "dom": "P", "ent": "hinge", "cls": "tank"}, {"id": 4439, "dom": "P", "ent": "hinge", "cls": "wheat"}, {"id": 4440, "dom": "P", "ent": "hood", "cls": "architecture"}, {"id": 4441, "dom": "P", "ent": "hood", "cls": "bacteria"}, {"id": 4442, "dom": "P", "ent": "hood", "cls": "beef"}, {"id": 4443, "dom": "P", "ent": "hood", "cls": "bridge"}, {"id": 4444, "dom": "P", "ent": "hood", "cls": "bud"}, {"id": 4445, "dom": "P", "ent": "hood", "cls": "cat"}, {"id": 4446, "dom": "P", "ent": "hood", "cls": "chapel"}, {"id": 4447, "dom": "P", "ent": "hood", "cls": "chick"}, {"id": 4448, "dom": "P", "ent": "hood", "cls": "clock"}, {"id": 4449, "dom": "P", "ent": "hood", "cls": "deer"}, {"id": 4450, "dom": "P", "ent": "hood", "cls": "dog"}, {"id": 4451, "dom": "P", "ent": "hood", "cls": "elephant"}, {"id": 4452, "dom": "P", "ent": "hood", "cls": "embassy"}, {"id": 4453, "dom": "P", "ent": "hood", "cls": "garage"}, {"id": 4454, "dom": "P", "ent": "hood", "cls": "highway"}, {"id": 4455, "dom": "P", "ent": "hood", "cls": "kidney"}, {"id": 4456, "dom": "P", "ent": "hood", "cls": "lamb"}, {"id": 4457, "dom": "P", "ent": "hood", "cls": "male"}, {"id": 4458, "dom": "P", "ent": "hood", "cls": "marijuana"}, {"id": 4459, "dom": "P", "ent": "hood", "cls": "puppy"}, {"id": 4460, "dom": "P", "ent": "hood", "cls": "receiver"}, {"id": 4461, "dom": "P", "ent": "hood", "cls": "reed"}, {"id": 4462, "dom": "P", "ent": "hood", "cls": "rifle"}, {"id": 4463, "dom": "P", "ent": "hood", "cls": "road"}, {"id": 4464, "dom": "P", "ent": "hood", "cls": "salmon"}, {"id": 4465, "dom": "P", "ent": "hood", "cls": "shark"}, {"id": 4466, "dom": "P", "ent": "hood", "cls": "sheep"}, {"id": 4467, "dom": "P", "ent": "hood", "cls": "shook"}, {"id": 4468, "dom": "P", "ent": "hood", "cls": "tank"}, {"id": 4469, "dom": "P", "ent": "hood", "cls": "wheat"}, {"id": 4470, "dom": "P", "ent": "hoof", "cls": "architecture"}, {"id": 4471, "dom": "P", "ent": "hoof", "cls": "bacteria"}, {"id": 4472, "dom": "P", "ent": "hoof", "cls": "beef"}, {"id": 4473, "dom": "P", "ent": "hoof", "cls": "bridge"}, {"id": 4474, "dom": "P", "ent": "hoof", "cls": "bud"}, {"id": 4475, "dom": "P", "ent": "hoof", "cls": "cat"}, {"id": 4476, "dom": "P", "ent": "hoof", "cls": "chapel"}, {"id": 4477, "dom": "P", "ent": "hoof", "cls": "chick"}, {"id": 4478, "dom": "P", "ent": "hoof", "cls": "clock"}, {"id": 4479, "dom": "P", "ent": "hoof", "cls": "deer"}, {"id": 4480, "dom": "P", "ent": "hoof", "cls": "dog"}, {"id": 4481, "dom": "P", "ent": "hoof", "cls": "elephant"}, {"id": 4482, "dom": "P", "ent": "hoof", "cls": "embassy"}, {"id": 4483, "dom": "P", "ent": "hoof", "cls": "garage"}, {"id": 4484, "dom": "P", "ent": "hoof", "cls": "highway"}, {"id": 4485, "dom": "P", "ent": "hoof", "cls": "kidney"}, {"id": 4486, "dom": "P", "ent": "hoof", "cls": "lamb"}, {"id": 4487, "dom": "P", "ent": "hoof", "cls": "male"}, {"id": 4488, "dom": "P", "ent": "hoof", "cls": "marijuana"}, {"id": 4489, "dom": "P", "ent": "hoof", "cls": "puppy"}, {"id": 4490, "dom": "P", "ent": "hoof", "cls": "receiver"}, {"id": 4491, "dom": "P", "ent": "hoof", "cls": "reed"}, {"id": 4492, "dom": "P", "ent": "hoof", "cls": "rifle"}, {"id": 4493, "dom": "P", "ent": "hoof", "cls": "road"}, {"id": 4494, "dom": "P", "ent": "hoof", "cls": "salmon"}, {"id": 4495, "dom": "P", "ent": "hoof", "cls": "shark"}, {"id": 4496, "dom": "P", "ent": "hoof", "cls": "sheep"}, {"id": 4497, "dom": "P", "ent": "hoof", "cls": "shook"}, {"id": 4498, "dom": "P", "ent": "hoof", "cls": "tank"}, {"id": 4499, "dom": "P", "ent": "hoof", "cls": "wheat"}, {"id": 4500, "dom": "P", "ent": "hoop", "cls": "architecture"}, {"id": 4501, "dom": "P", "ent": "hoop", "cls": "bacteria"}, {"id": 4502, "dom": "P", "ent": "hoop", "cls": "beef"}, {"id": 4503, "dom": "P", "ent": "hoop", "cls": "bridge"}, {"id": 4504, "dom": "P", "ent": "hoop", "cls": "bud"}, {"id": 4505, "dom": "P", "ent": "hoop", "cls": "cat"}, {"id": 4506, "dom": "P", "ent": "hoop", "cls": "chapel"}, {"id": 4507, "dom": "P", "ent": "hoop", "cls": "chick"}, {"id": 4508, "dom": "P", "ent": "hoop", "cls": "clock"}, {"id": 4509, "dom": "P", "ent": "hoop", "cls": "deer"}, {"id": 4510, "dom": "P", "ent": "hoop", "cls": "dog"}, {"id": 4511, "dom": "P", "ent": "hoop", "cls": "elephant"}, {"id": 4512, "dom": "P", "ent": "hoop", "cls": "embassy"}, {"id": 4513, "dom": "P", "ent": "hoop", "cls": "garage"}, {"id": 4514, "dom": "P", "ent": "hoop", "cls": "highway"}, {"id": 4515, "dom": "P", "ent": "hoop", "cls": "kidney"}, {"id": 4516, "dom": "P", "ent": "hoop", "cls": "lamb"}, {"id": 4517, "dom": "P", "ent": "hoop", "cls": "male"}, {"id": 4518, "dom": "P", "ent": "hoop", "cls": "marijuana"}, {"id": 4519, "dom": "P", "ent": "hoop", "cls": "puppy"}, {"id": 4520, "dom": "P", "ent": "hoop", "cls": "receiver"}, {"id": 4521, "dom": "P", "ent": "hoop", "cls": "reed"}, {"id": 4522, "dom": "P", "ent": "hoop", "cls": "rifle"}, {"id": 4523, "dom": "P", "ent": "hoop", "cls": "road"}, {"id": 4524, "dom": "P", "ent": "hoop", "cls": "salmon"}, {"id": 4525, "dom": "P", "ent": "hoop", "cls": "shark"}, {"id": 4526, "dom": "P", "ent": "hoop", "cls": "sheep"}, {"id": 4527, "dom": "P", "ent": "hoop", "cls": "shook"}, {"id": 4528, "dom": "P", "ent": "hoop", "cls": "tank"}, {"id": 4529, "dom": "P", "ent": "hoop", "cls": "wheat"}, {"id": 4530, "dom": "P", "ent": "horse-trail", "cls": "architecture"}, {"id": 4531, "dom": "P", "ent": "horse-trail", "cls": "bacteria"}, {"id": 4532, "dom": "P", "ent": "horse-trail", "cls": "beef"}, {"id": 4533, "dom": "P", "ent": "horse-trail", "cls": "bridge"}, {"id": 4534, "dom": "P", "ent": "horse-trail", "cls": "bud"}, {"id": 4535, "dom": "P", "ent": "horse-trail", "cls": "cat"}, {"id": 4536, "dom": "P", "ent": "horse-trail", "cls": "chapel"}, {"id": 4537, "dom": "P", "ent": "horse-trail", "cls": "chick"}, {"id": 4538, "dom": "P", "ent": "horse-trail", "cls": "clock"}, {"id": 4539, "dom": "P", "ent": "horse-trail", "cls": "deer"}, {"id": 4540, "dom": "P", "ent": "horse-trail", "cls": "dog"}, {"id": 4541, "dom": "P", "ent": "horse-trail", "cls": "elephant"}, {"id": 4542, "dom": "P", "ent": "horse-trail", "cls": "embassy"}, {"id": 4543, "dom": "P", "ent": "horse-trail", "cls": "garage"}, {"id": 4544, "dom": "P", "ent": "horse-trail", "cls": "highway"}, {"id": 4545, "dom": "P", "ent": "horse-trail", "cls": "kidney"}, {"id": 4546, "dom": "P", "ent": "horse-trail", "cls": "lamb"}, {"id": 4547, "dom": "P", "ent": "horse-trail", "cls": "male"}, {"id": 4548, "dom": "P", "ent": "horse-trail", "cls": "marijuana"}, {"id": 4549, "dom": "P", "ent": "horse-trail", "cls": "puppy"}, {"id": 4550, "dom": "P", "ent": "horse-trail", "cls": "receiver"}, {"id": 4551, "dom": "P", "ent": "horse-trail", "cls": "reed"}, {"id": 4552, "dom": "P", "ent": "horse-trail", "cls": "rifle"}, {"id": 4553, "dom": "P", "ent": "horse-trail", "cls": "road"}, {"id": 4554, "dom": "P", "ent": "horse-trail", "cls": "salmon"}, {"id": 4555, "dom": "P", "ent": "horse-trail", "cls": "shark"}, {"id": 4556, "dom": "P", "ent": "horse-trail", "cls": "sheep"}, {"id": 4557, "dom": "P", "ent": "horse-trail", "cls": "shook"}, {"id": 4558, "dom": "P", "ent": "horse-trail", "cls": "tank"}, {"id": 4559, "dom": "P", "ent": "horse-trail", "cls": "wheat"}, {"id": 4560, "dom": "P", "ent": "interchange", "cls": "architecture"}, {"id": 4561, "dom": "P", "ent": "interchange", "cls": "bacteria"}, {"id": 4562, "dom": "P", "ent": "interchange", "cls": "beef"}, {"id": 4563, "dom": "P", "ent": "interchange", "cls": "bridge"}, {"id": 4564, "dom": "P", "ent": "interchange", "cls": "bud"}, {"id": 4565, "dom": "P", "ent": "interchange", "cls": "cat"}, {"id": 4566, "dom": "P", "ent": "interchange", "cls": "chapel"}, {"id": 4567, "dom": "P", "ent": "interchange", "cls": "chick"}, {"id": 4568, "dom": "P", "ent": "interchange", "cls": "clock"}, {"id": 4569, "dom": "P", "ent": "interchange", "cls": "deer"}, {"id": 4570, "dom": "P", "ent": "interchange", "cls": "dog"}, {"id": 4571, "dom": "P", "ent": "interchange", "cls": "elephant"}, {"id": 4572, "dom": "P", "ent": "interchange", "cls": "embassy"}, {"id": 4573, "dom": "P", "ent": "interchange", "cls": "garage"}, {"id": 4574, "dom": "P", "ent": "interchange", "cls": "highway"}, {"id": 4575, "dom": "P", "ent": "interchange", "cls": "kidney"}, {"id": 4576, "dom": "P", "ent": "interchange", "cls": "lamb"}, {"id": 4577, "dom": "P", "ent": "interchange", "cls": "male"}, {"id": 4578, "dom": "P", "ent": "interchange", "cls": "marijuana"}, {"id": 4579, "dom": "P", "ent": "interchange", "cls": "puppy"}, {"id": 4580, "dom": "P", "ent": "interchange", "cls": "receiver"}, {"id": 4581, "dom": "P", "ent": "interchange", "cls": "reed"}, {"id": 4582, "dom": "P", "ent": "interchange", "cls": "rifle"}, {"id": 4583, "dom": "P", "ent": "interchange", "cls": "road"}, {"id": 4584, "dom": "P", "ent": "interchange", "cls": "salmon"}, {"id": 4585, "dom": "P", "ent": "interchange", "cls": "shark"}, {"id": 4586, "dom": "P", "ent": "interchange", "cls": "sheep"}, {"id": 4587, "dom": "P", "ent": "interchange", "cls": "shook"}, {"id": 4588, "dom": "P", "ent": "interchange", "cls": "tank"}, {"id": 4589, "dom": "P", "ent": "interchange", "cls": "wheat"}, {"id": 4590, "dom": "P", "ent": "intersection", "cls": "architecture"}, {"id": 4591, "dom": "P", "ent": "intersection", "cls": "bacteria"}, {"id": 4592, "dom": "P", "ent": "intersection", "cls": "beef"}, {"id": 4593, "dom": "P", "ent": "intersection", "cls": "bridge"}, {"id": 4594, "dom": "P", "ent": "intersection", "cls": "bud"}, {"id": 4595, "dom": "P", "ent": "intersection", "cls": "cat"}, {"id": 4596, "dom": "P", "ent": "intersection", "cls": "chapel"}, {"id": 4597, "dom": "P", "ent": "intersection", "cls": "chick"}, {"id": 4598, "dom": "P", "ent": "intersection", "cls": "clock"}, {"id": 4599, "dom": "P", "ent": "intersection", "cls": "deer"}, {"id": 4600, "dom": "P", "ent": "intersection", "cls": "dog"}, {"id": 4601, "dom": "P", "ent": "intersection", "cls": "elephant"}, {"id": 4602, "dom": "P", "ent": "intersection", "cls": "embassy"}, {"id": 4603, "dom": "P", "ent": "intersection", "cls": "garage"}, {"id": 4604, "dom": "P", "ent": "intersection", "cls": "highway"}, {"id": 4605, "dom": "P", "ent": "intersection", "cls": "kidney"}, {"id": 4606, "dom": "P", "ent": "intersection", "cls": "lamb"}, {"id": 4607, "dom": "P", "ent": "intersection", "cls": "male"}, {"id": 4608, "dom": "P", "ent": "intersection", "cls": "marijuana"}, {"id": 4609, "dom": "P", "ent": "intersection", "cls": "puppy"}, {"id": 4610, "dom": "P", "ent": "intersection", "cls": "receiver"}, {"id": 4611, "dom": "P", "ent": "intersection", "cls": "reed"}, {"id": 4612, "dom": "P", "ent": "intersection", "cls": "rifle"}, {"id": 4613, "dom": "P", "ent": "intersection", "cls": "road"}, {"id": 4614, "dom": "P", "ent": "intersection", "cls": "salmon"}, {"id": 4615, "dom": "P", "ent": "intersection", "cls": "shark"}, {"id": 4616, "dom": "P", "ent": "intersection", "cls": "sheep"}, {"id": 4617, "dom": "P", "ent": "intersection", "cls": "shook"}, {"id": 4618, "dom": "P", "ent": "intersection", "cls": "tank"}, {"id": 4619, "dom": "P", "ent": "intersection", "cls": "wheat"}, {"id": 4620, "dom": "P", "ent": "iris", "cls": "architecture"}, {"id": 4621, "dom": "P", "ent": "iris", "cls": "bacteria"}, {"id": 4622, "dom": "P", "ent": "iris", "cls": "beef"}, {"id": 4623, "dom": "P", "ent": "iris", "cls": "bridge"}, {"id": 4624, "dom": "P", "ent": "iris", "cls": "bud"}, {"id": 4625, "dom": "P", "ent": "iris", "cls": "cat"}, {"id": 4626, "dom": "P", "ent": "iris", "cls": "chapel"}, {"id": 4627, "dom": "P", "ent": "iris", "cls": "chick"}, {"id": 4628, "dom": "P", "ent": "iris", "cls": "clock"}, {"id": 4629, "dom": "P", "ent": "iris", "cls": "deer"}, {"id": 4630, "dom": "P", "ent": "iris", "cls": "dog"}, {"id": 4631, "dom": "P", "ent": "iris", "cls": "elephant"}, {"id": 4632, "dom": "P", "ent": "iris", "cls": "embassy"}, {"id": 4633, "dom": "P", "ent": "iris", "cls": "garage"}, {"id": 4634, "dom": "P", "ent": "iris", "cls": "highway"}, {"id": 4635, "dom": "P", "ent": "iris", "cls": "kidney"}, {"id": 4636, "dom": "P", "ent": "iris", "cls": "lamb"}, {"id": 4637, "dom": "P", "ent": "iris", "cls": "male"}, {"id": 4638, "dom": "P", "ent": "iris", "cls": "marijuana"}, {"id": 4639, "dom": "P", "ent": "iris", "cls": "puppy"}, {"id": 4640, "dom": "P", "ent": "iris", "cls": "receiver"}, {"id": 4641, "dom": "P", "ent": "iris", "cls": "reed"}, {"id": 4642, "dom": "P", "ent": "iris", "cls": "rifle"}, {"id": 4643, "dom": "P", "ent": "iris", "cls": "road"}, {"id": 4644, "dom": "P", "ent": "iris", "cls": "salmon"}, {"id": 4645, "dom": "P", "ent": "iris", "cls": "shark"}, {"id": 4646, "dom": "P", "ent": "iris", "cls": "sheep"}, {"id": 4647, "dom": "P", "ent": "iris", "cls": "shook"}, {"id": 4648, "dom": "P", "ent": "iris", "cls": "tank"}, {"id": 4649, "dom": "P", "ent": "iris", "cls": "wheat"}, {"id": 4650, "dom": "P", "ent": "ivy", "cls": "architecture"}, {"id": 4651, "dom": "P", "ent": "ivy", "cls": "bacteria"}, {"id": 4652, "dom": "P", "ent": "ivy", "cls": "beef"}, {"id": 4653, "dom": "P", "ent": "ivy", "cls": "bridge"}, {"id": 4654, "dom": "P", "ent": "ivy", "cls": "bud"}, {"id": 4655, "dom": "P", "ent": "ivy", "cls": "cat"}, {"id": 4656, "dom": "P", "ent": "ivy", "cls": "chapel"}, {"id": 4657, "dom": "P", "ent": "ivy", "cls": "chick"}, {"id": 4658, "dom": "P", "ent": "ivy", "cls": "clock"}, {"id": 4659, "dom": "P", "ent": "ivy", "cls": "deer"}, {"id": 4660, "dom": "P", "ent": "ivy", "cls": "dog"}, {"id": 4661, "dom": "P", "ent": "ivy", "cls": "elephant"}, {"id": 4662, "dom": "P", "ent": "ivy", "cls": "embassy"}, {"id": 4663, "dom": "P", "ent": "ivy", "cls": "garage"}, {"id": 4664, "dom": "P", "ent": "ivy", "cls": "highway"}, {"id": 4665, "dom": "P", "ent": "ivy", "cls": "kidney"}, {"id": 4666, "dom": "P", "ent": "ivy", "cls": "lamb"}, {"id": 4667, "dom": "P", "ent": "ivy", "cls": "male"}, {"id": 4668, "dom": "P", "ent": "ivy", "cls": "marijuana"}, {"id": 4669, "dom": "P", "ent": "ivy", "cls": "puppy"}, {"id": 4670, "dom": "P", "ent": "ivy", "cls": "receiver"}, {"id": 4671, "dom": "P", "ent": "ivy", "cls": "reed"}, {"id": 4672, "dom": "P", "ent": "ivy", "cls": "rifle"}, {"id": 4673, "dom": "P", "ent": "ivy", "cls": "road"}, {"id": 4674, "dom": "P", "ent": "ivy", "cls": "salmon"}, {"id": 4675, "dom": "P", "ent": "ivy", "cls": "shark"}, {"id": 4676, "dom": "P", "ent": "ivy", "cls": "sheep"}, {"id": 4677, "dom": "P", "ent": "ivy", "cls": "shook"}, {"id": 4678, "dom": "P", "ent": "ivy", "cls": "tank"}, {"id": 4679, "dom": "P", "ent": "ivy", "cls": "wheat"}, {"id": 4680, "dom": "P", "ent": "jack", "cls": "architecture"}, {"id": 4681, "dom": "P", "ent": "jack", "cls": "bacteria"}, {"id": 4682, "dom": "P", "ent": "jack", "cls": "beef"}, {"id": 4683, "dom": "P", "ent": "jack", "cls": "bridge"}, {"id": 4684, "dom": "P", "ent": "jack", "cls": "bud"}, {"id": 4685, "dom": "P", "ent": "jack", "cls": "cat"}, {"id": 4686, "dom": "P", "ent": "jack", "cls": "chapel"}, {"id": 4687, "dom": "P", "ent": "jack", "cls": "chick"}, {"id": 4688, "dom": "P", "ent": "jack", "cls": "clock"}, {"id": 4689, "dom": "P", "ent": "jack", "cls": "deer"}, {"id": 4690, "dom": "P", "ent": "jack", "cls": "dog"}, {"id": 4691, "dom": "P", "ent": "jack", "cls": "elephant"}, {"id": 4692, "dom": "P", "ent": "jack", "cls": "embassy"}, {"id": 4693, "dom": "P", "ent": "jack", "cls": "garage"}, {"id": 4694, "dom": "P", "ent": "jack", "cls": "highway"}, {"id": 4695, "dom": "P", "ent": "jack", "cls": "kidney"}, {"id": 4696, "dom": "P", "ent": "jack", "cls": "lamb"}, {"id": 4697, "dom": "P", "ent": "jack", "cls": "male"}, {"id": 4698, "dom": "P", "ent": "jack", "cls": "marijuana"}, {"id": 4699, "dom": "P", "ent": "jack", "cls": "puppy"}, {"id": 4700, "dom": "P", "ent": "jack", "cls": "receiver"}, {"id": 4701, "dom": "P", "ent": "jack", "cls": "reed"}, {"id": 4702, "dom": "P", "ent": "jack", "cls": "rifle"}, {"id": 4703, "dom": "P", "ent": "jack", "cls": "road"}, {"id": 4704, "dom": "P", "ent": "jack", "cls": "salmon"}, {"id": 4705, "dom": "P", "ent": "jack", "cls": "shark"}, {"id": 4706, "dom": "P", "ent": "jack", "cls": "sheep"}, {"id": 4707, "dom": "P", "ent": "jack", "cls": "shook"}, {"id": 4708, "dom": "P", "ent": "jack", "cls": "tank"}, {"id": 4709, "dom": "P", "ent": "jack", "cls": "wheat"}, {"id": 4710, "dom": "P", "ent": "jacket", "cls": "architecture"}, {"id": 4711, "dom": "P", "ent": "jacket", "cls": "bacteria"}, {"id": 4712, "dom": "P", "ent": "jacket", "cls": "beef"}, {"id": 4713, "dom": "P", "ent": "jacket", "cls": "bridge"}, {"id": 4714, "dom": "P", "ent": "jacket", "cls": "bud"}, {"id": 4715, "dom": "P", "ent": "jacket", "cls": "cat"}, {"id": 4716, "dom": "P", "ent": "jacket", "cls": "chapel"}, {"id": 4717, "dom": "P", "ent": "jacket", "cls": "chick"}, {"id": 4718, "dom": "P", "ent": "jacket", "cls": "clock"}, {"id": 4719, "dom": "P", "ent": "jacket", "cls": "deer"}, {"id": 4720, "dom": "P", "ent": "jacket", "cls": "dog"}, {"id": 4721, "dom": "P", "ent": "jacket", "cls": "elephant"}, {"id": 4722, "dom": "P", "ent": "jacket", "cls": "embassy"}, {"id": 4723, "dom": "P", "ent": "jacket", "cls": "garage"}, {"id": 4724, "dom": "P", "ent": "jacket", "cls": "highway"}, {"id": 4725, "dom": "P", "ent": "jacket", "cls": "kidney"}, {"id": 4726, "dom": "P", "ent": "jacket", "cls": "lamb"}, {"id": 4727, "dom": "P", "ent": "jacket", "cls": "male"}, {"id": 4728, "dom": "P", "ent": "jacket", "cls": "marijuana"}, {"id": 4729, "dom": "P", "ent": "jacket", "cls": "puppy"}, {"id": 4730, "dom": "P", "ent": "jacket", "cls": "receiver"}, {"id": 4731, "dom": "P", "ent": "jacket", "cls": "reed"}, {"id": 4732, "dom": "P", "ent": "jacket", "cls": "rifle"}, {"id": 4733, "dom": "P", "ent": "jacket", "cls": "road"}, {"id": 4734, "dom": "P", "ent": "jacket", "cls": "salmon"}, {"id": 4735, "dom": "P", "ent": "jacket", "cls": "shark"}, {"id": 4736, "dom": "P", "ent": "jacket", "cls": "sheep"}, {"id": 4737, "dom": "P", "ent": "jacket", "cls": "shook"}, {"id": 4738, "dom": "P", "ent": "jacket", "cls": "tank"}, {"id": 4739, "dom": "P", "ent": "jacket", "cls": "wheat"}, {"id": 4740, "dom": "P", "ent": "jaw", "cls": "architecture"}, {"id": 4741, "dom": "P", "ent": "jaw", "cls": "bacteria"}, {"id": 4742, "dom": "P", "ent": "jaw", "cls": "beef"}, {"id": 4743, "dom": "P", "ent": "jaw", "cls": "bridge"}, {"id": 4744, "dom": "P", "ent": "jaw", "cls": "bud"}, {"id": 4745, "dom": "P", "ent": "jaw", "cls": "cat"}, {"id": 4746, "dom": "P", "ent": "jaw", "cls": "chapel"}, {"id": 4747, "dom": "P", "ent": "jaw", "cls": "chick"}, {"id": 4748, "dom": "P", "ent": "jaw", "cls": "clock"}, {"id": 4749, "dom": "P", "ent": "jaw", "cls": "deer"}, {"id": 4750, "dom": "P", "ent": "jaw", "cls": "dog"}, {"id": 4751, "dom": "P", "ent": "jaw", "cls": "elephant"}, {"id": 4752, "dom": "P", "ent": "jaw", "cls": "embassy"}, {"id": 4753, "dom": "P", "ent": "jaw", "cls": "garage"}, {"id": 4754, "dom": "P", "ent": "jaw", "cls": "highway"}, {"id": 4755, "dom": "P", "ent": "jaw", "cls": "kidney"}, {"id": 4756, "dom": "P", "ent": "jaw", "cls": "lamb"}, {"id": 4757, "dom": "P", "ent": "jaw", "cls": "male"}, {"id": 4758, "dom": "P", "ent": "jaw", "cls": "marijuana"}, {"id": 4759, "dom": "P", "ent": "jaw", "cls": "puppy"}, {"id": 4760, "dom": "P", "ent": "jaw", "cls": "receiver"}, {"id": 4761, "dom": "P", "ent": "jaw", "cls": "reed"}, {"id": 4762, "dom": "P", "ent": "jaw", "cls": "rifle"}, {"id": 4763, "dom": "P", "ent": "jaw", "cls": "road"}, {"id": 4764, "dom": "P", "ent": "jaw", "cls": "salmon"}, {"id": 4765, "dom": "P", "ent": "jaw", "cls": "shark"}, {"id": 4766, "dom": "P", "ent": "jaw", "cls": "sheep"}, {"id": 4767, "dom": "P", "ent": "jaw", "cls": "shook"}, {"id": 4768, "dom": "P", "ent": "jaw", "cls": "tank"}, {"id": 4769, "dom": "P", "ent": "jaw", "cls": "wheat"}, {"id": 4770, "dom": "P", "ent": "jelly", "cls": "architecture"}, {"id": 4771, "dom": "P", "ent": "jelly", "cls": "bacteria"}, {"id": 4772, "dom": "P", "ent": "jelly", "cls": "beef"}, {"id": 4773, "dom": "P", "ent": "jelly", "cls": "bridge"}, {"id": 4774, "dom": "P", "ent": "jelly", "cls": "bud"}, {"id": 4775, "dom": "P", "ent": "jelly", "cls": "cat"}, {"id": 4776, "dom": "P", "ent": "jelly", "cls": "chapel"}, {"id": 4777, "dom": "P", "ent": "jelly", "cls": "chick"}, {"id": 4778, "dom": "P", "ent": "jelly", "cls": "clock"}, {"id": 4779, "dom": "P", "ent": "jelly", "cls": "deer"}, {"id": 4780, "dom": "P", "ent": "jelly", "cls": "dog"}, {"id": 4781, "dom": "P", "ent": "jelly", "cls": "elephant"}, {"id": 4782, "dom": "P", "ent": "jelly", "cls": "embassy"}, {"id": 4783, "dom": "P", "ent": "jelly", "cls": "garage"}, {"id": 4784, "dom": "P", "ent": "jelly", "cls": "highway"}, {"id": 4785, "dom": "P", "ent": "jelly", "cls": "kidney"}, {"id": 4786, "dom": "P", "ent": "jelly", "cls": "lamb"}, {"id": 4787, "dom": "P", "ent": "jelly", "cls": "male"}, {"id": 4788, "dom": "P", "ent": "jelly", "cls": "marijuana"}, {"id": 4789, "dom": "P", "ent": "jelly", "cls": "puppy"}, {"id": 4790, "dom": "P", "ent": "jelly", "cls": "receiver"}, {"id": 4791, "dom": "P", "ent": "jelly", "cls": "reed"}, {"id": 4792, "dom": "P", "ent": "jelly", "cls": "rifle"}, {"id": 4793, "dom": "P", "ent": "jelly", "cls": "road"}, {"id": 4794, "dom": "P", "ent": "jelly", "cls": "salmon"}, {"id": 4795, "dom": "P", "ent": "jelly", "cls": "shark"}, {"id": 4796, "dom": "P", "ent": "jelly", "cls": "sheep"}, {"id": 4797, "dom": "P", "ent": "jelly", "cls": "shook"}, {"id": 4798, "dom": "P", "ent": "jelly", "cls": "tank"}, {"id": 4799, "dom": "P", "ent": "jelly", "cls": "wheat"}, {"id": 4800, "dom": "P", "ent": "jersey", "cls": "architecture"}, {"id": 4801, "dom": "P", "ent": "jersey", "cls": "bacteria"}, {"id": 4802, "dom": "P", "ent": "jersey", "cls": "beef"}, {"id": 4803, "dom": "P", "ent": "jersey", "cls": "bridge"}, {"id": 4804, "dom": "P", "ent": "jersey", "cls": "bud"}, {"id": 4805, "dom": "P", "ent": "jersey", "cls": "cat"}, {"id": 4806, "dom": "P", "ent": "jersey", "cls": "chapel"}, {"id": 4807, "dom": "P", "ent": "jersey", "cls": "chick"}, {"id": 4808, "dom": "P", "ent": "jersey", "cls": "clock"}, {"id": 4809, "dom": "P", "ent": "jersey", "cls": "deer"}, {"id": 4810, "dom": "P", "ent": "jersey", "cls": "dog"}, {"id": 4811, "dom": "P", "ent": "jersey", "cls": "elephant"}, {"id": 4812, "dom": "P", "ent": "jersey", "cls": "embassy"}, {"id": 4813, "dom": "P", "ent": "jersey", "cls": "garage"}, {"id": 4814, "dom": "P", "ent": "jersey", "cls": "highway"}, {"id": 4815, "dom": "P", "ent": "jersey", "cls": "kidney"}, {"id": 4816, "dom": "P", "ent": "jersey", "cls": "lamb"}, {"id": 4817, "dom": "P", "ent": "jersey", "cls": "male"}, {"id": 4818, "dom": "P", "ent": "jersey", "cls": "marijuana"}, {"id": 4819, "dom": "P", "ent": "jersey", "cls": "puppy"}, {"id": 4820, "dom": "P", "ent": "jersey", "cls": "receiver"}, {"id": 4821, "dom": "P", "ent": "jersey", "cls": "reed"}, {"id": 4822, "dom": "P", "ent": "jersey", "cls": "rifle"}, {"id": 4823, "dom": "P", "ent": "jersey", "cls": "road"}, {"id": 4824, "dom": "P", "ent": "jersey", "cls": "salmon"}, {"id": 4825, "dom": "P", "ent": "jersey", "cls": "shark"}, {"id": 4826, "dom": "P", "ent": "jersey", "cls": "sheep"}, {"id": 4827, "dom": "P", "ent": "jersey", "cls": "shook"}, {"id": 4828, "dom": "P", "ent": "jersey", "cls": "tank"}, {"id": 4829, "dom": "P", "ent": "jersey", "cls": "wheat"}, {"id": 4830, "dom": "P", "ent": "jewel", "cls": "architecture"}, {"id": 4831, "dom": "P", "ent": "jewel", "cls": "bacteria"}, {"id": 4832, "dom": "P", "ent": "jewel", "cls": "beef"}, {"id": 4833, "dom": "P", "ent": "jewel", "cls": "bridge"}, {"id": 4834, "dom": "P", "ent": "jewel", "cls": "bud"}, {"id": 4835, "dom": "P", "ent": "jewel", "cls": "cat"}, {"id": 4836, "dom": "P", "ent": "jewel", "cls": "chapel"}, {"id": 4837, "dom": "P", "ent": "jewel", "cls": "chick"}, {"id": 4838, "dom": "P", "ent": "jewel", "cls": "clock"}, {"id": 4839, "dom": "P", "ent": "jewel", "cls": "deer"}, {"id": 4840, "dom": "P", "ent": "jewel", "cls": "dog"}, {"id": 4841, "dom": "P", "ent": "jewel", "cls": "elephant"}, {"id": 4842, "dom": "P", "ent": "jewel", "cls": "embassy"}, {"id": 4843, "dom": "P", "ent": "jewel", "cls": "garage"}, {"id": 4844, "dom": "P", "ent": "jewel", "cls": "highway"}, {"id": 4845, "dom": "P", "ent": "jewel", "cls": "kidney"}, {"id": 4846, "dom": "P", "ent": "jewel", "cls": "lamb"}, {"id": 4847, "dom": "P", "ent": "jewel", "cls": "male"}, {"id": 4848, "dom": "P", "ent": "jewel", "cls": "marijuana"}, {"id": 4849, "dom": "P", "ent": "jewel", "cls": "puppy"}, {"id": 4850, "dom": "P", "ent": "jewel", "cls": "receiver"}, {"id": 4851, "dom": "P", "ent": "jewel", "cls": "reed"}, {"id": 4852, "dom": "P", "ent": "jewel", "cls": "rifle"}, {"id": 4853, "dom": "P", "ent": "jewel", "cls": "road"}, {"id": 4854, "dom": "P", "ent": "jewel", "cls": "salmon"}, {"id": 4855, "dom": "P", "ent": "jewel", "cls": "shark"}, {"id": 4856, "dom": "P", "ent": "jewel", "cls": "sheep"}, {"id": 4857, "dom": "P", "ent": "jewel", "cls": "shook"}, {"id": 4858, "dom": "P", "ent": "jewel", "cls": "tank"}, {"id": 4859, "dom": "P", "ent": "jewel", "cls": "wheat"}, {"id": 4860, "dom": "P", "ent": "jimmy", "cls": "architecture"}, {"id": 4861, "dom": "P", "ent": "jimmy", "cls": "bacteria"}, {"id": 4862, "dom": "P", "ent": "jimmy", "cls": "beef"}, {"id": 4863, "dom": "P", "ent": "jimmy", "cls": "bridge"}, {"id": 4864, "dom": "P", "ent": "jimmy", "cls": "bud"}, {"id": 4865, "dom": "P", "ent": "jimmy", "cls": "cat"}, {"id": 4866, "dom": "P", "ent": "jimmy", "cls": "chapel"}, {"id": 4867, "dom": "P", "ent": "jimmy", "cls": "chick"}, {"id": 4868, "dom": "P", "ent": "jimmy", "cls": "clock"}, {"id": 4869, "dom": "P", "ent": "jimmy", "cls": "deer"}, {"id": 4870, "dom": "P", "ent": "jimmy", "cls": "dog"}, {"id": 4871, "dom": "P", "ent": "jimmy", "cls": "elephant"}, {"id": 4872, "dom": "P", "ent": "jimmy", "cls": "embassy"}, {"id": 4873, "dom": "P", "ent": "jimmy", "cls": "garage"}, {"id": 4874, "dom": "P", "ent": "jimmy", "cls": "highway"}, {"id": 4875, "dom": "P", "ent": "jimmy", "cls": "kidney"}, {"id": 4876, "dom": "P", "ent": "jimmy", "cls": "lamb"}, {"id": 4877, "dom": "P", "ent": "jimmy", "cls": "male"}, {"id": 4878, "dom": "P", "ent": "jimmy", "cls": "marijuana"}, {"id": 4879, "dom": "P", "ent": "jimmy", "cls": "puppy"}, {"id": 4880, "dom": "P", "ent": "jimmy", "cls": "receiver"}, {"id": 4881, "dom": "P", "ent": "jimmy", "cls": "reed"}, {"id": 4882, "dom": "P", "ent": "jimmy", "cls": "rifle"}, {"id": 4883, "dom": "P", "ent": "jimmy", "cls": "road"}, {"id": 4884, "dom": "P", "ent": "jimmy", "cls": "salmon"}, {"id": 4885, "dom": "P", "ent": "jimmy", "cls": "shark"}, {"id": 4886, "dom": "P", "ent": "jimmy", "cls": "sheep"}, {"id": 4887, "dom": "P", "ent": "jimmy", "cls": "shook"}, {"id": 4888, "dom": "P", "ent": "jimmy", "cls": "tank"}, {"id": 4889, "dom": "P", "ent": "jimmy", "cls": "wheat"}, {"id": 4890, "dom": "P", "ent": "john", "cls": "architecture"}, {"id": 4891, "dom": "P", "ent": "john", "cls": "bacteria"}, {"id": 4892, "dom": "P", "ent": "john", "cls": "beef"}, {"id": 4893, "dom": "P", "ent": "john", "cls": "bridge"}, {"id": 4894, "dom": "P", "ent": "john", "cls": "bud"}, {"id": 4895, "dom": "P", "ent": "john", "cls": "cat"}, {"id": 4896, "dom": "P", "ent": "john", "cls": "chapel"}, {"id": 4897, "dom": "P", "ent": "john", "cls": "chick"}, {"id": 4898, "dom": "P", "ent": "john", "cls": "clock"}, {"id": 4899, "dom": "P", "ent": "john", "cls": "deer"}, {"id": 4900, "dom": "P", "ent": "john", "cls": "dog"}, {"id": 4901, "dom": "P", "ent": "john", "cls": "elephant"}, {"id": 4902, "dom": "P", "ent": "john", "cls": "embassy"}, {"id": 4903, "dom": "P", "ent": "john", "cls": "garage"}, {"id": 4904, "dom": "P", "ent": "john", "cls": "highway"}, {"id": 4905, "dom": "P", "ent": "john", "cls": "kidney"}, {"id": 4906, "dom": "P", "ent": "john", "cls": "lamb"}, {"id": 4907, "dom": "P", "ent": "john", "cls": "male"}, {"id": 4908, "dom": "P", "ent": "john", "cls": "marijuana"}, {"id": 4909, "dom": "P", "ent": "john", "cls": "puppy"}, {"id": 4910, "dom": "P", "ent": "john", "cls": "receiver"}, {"id": 4911, "dom": "P", "ent": "john", "cls": "reed"}, {"id": 4912, "dom": "P", "ent": "john", "cls": "rifle"}, {"id": 4913, "dom": "P", "ent": "john", "cls": "road"}, {"id": 4914, "dom": "P", "ent": "john", "cls": "salmon"}, {"id": 4915, "dom": "P", "ent": "john", "cls": "shark"}, {"id": 4916, "dom": "P", "ent": "john", "cls": "sheep"}, {"id": 4917, "dom": "P", "ent": "john", "cls": "shook"}, {"id": 4918, "dom": "P", "ent": "john", "cls": "tank"}, {"id": 4919, "dom": "P", "ent": "john", "cls": "wheat"}, {"id": 4920, "dom": "P", "ent": "key", "cls": "architecture"}, {"id": 4921, "dom": "P", "ent": "key", "cls": "bacteria"}, {"id": 4922, "dom": "P", "ent": "key", "cls": "beef"}, {"id": 4923, "dom": "P", "ent": "key", "cls": "bridge"}, {"id": 4924, "dom": "P", "ent": "key", "cls": "bud"}, {"id": 4925, "dom": "P", "ent": "key", "cls": "cat"}, {"id": 4926, "dom": "P", "ent": "key", "cls": "chapel"}, {"id": 4927, "dom": "P", "ent": "key", "cls": "chick"}, {"id": 4928, "dom": "P", "ent": "key", "cls": "clock"}, {"id": 4929, "dom": "P", "ent": "key", "cls": "deer"}, {"id": 4930, "dom": "P", "ent": "key", "cls": "dog"}, {"id": 4931, "dom": "P", "ent": "key", "cls": "elephant"}, {"id": 4932, "dom": "P", "ent": "key", "cls": "embassy"}, {"id": 4933, "dom": "P", "ent": "key", "cls": "garage"}, {"id": 4934, "dom": "P", "ent": "key", "cls": "highway"}, {"id": 4935, "dom": "P", "ent": "key", "cls": "kidney"}, {"id": 4936, "dom": "P", "ent": "key", "cls": "lamb"}, {"id": 4937, "dom": "P", "ent": "key", "cls": "male"}, {"id": 4938, "dom": "P", "ent": "key", "cls": "marijuana"}, {"id": 4939, "dom": "P", "ent": "key", "cls": "puppy"}, {"id": 4940, "dom": "P", "ent": "key", "cls": "receiver"}, {"id": 4941, "dom": "P", "ent": "key", "cls": "reed"}, {"id": 4942, "dom": "P", "ent": "key", "cls": "rifle"}, {"id": 4943, "dom": "P", "ent": "key", "cls": "road"}, {"id": 4944, "dom": "P", "ent": "key", "cls": "salmon"}, {"id": 4945, "dom": "P", "ent": "key", "cls": "shark"}, {"id": 4946, "dom": "P", "ent": "key", "cls": "sheep"}, {"id": 4947, "dom": "P", "ent": "key", "cls": "shook"}, {"id": 4948, "dom": "P", "ent": "key", "cls": "tank"}, {"id": 4949, "dom": "P", "ent": "key", "cls": "wheat"}, {"id": 4950, "dom": "P", "ent": "keystone", "cls": "architecture"}, {"id": 4951, "dom": "P", "ent": "keystone", "cls": "bacteria"}, {"id": 4952, "dom": "P", "ent": "keystone", "cls": "beef"}, {"id": 4953, "dom": "P", "ent": "keystone", "cls": "bridge"}, {"id": 4954, "dom": "P", "ent": "keystone", "cls": "bud"}, {"id": 4955, "dom": "P", "ent": "keystone", "cls": "cat"}, {"id": 4956, "dom": "P", "ent": "keystone", "cls": "chapel"}, {"id": 4957, "dom": "P", "ent": "keystone", "cls": "chick"}, {"id": 4958, "dom": "P", "ent": "keystone", "cls": "clock"}, {"id": 4959, "dom": "P", "ent": "keystone", "cls": "deer"}, {"id": 4960, "dom": "P", "ent": "keystone", "cls": "dog"}, {"id": 4961, "dom": "P", "ent": "keystone", "cls": "elephant"}, {"id": 4962, "dom": "P", "ent": "keystone", "cls": "embassy"}, {"id": 4963, "dom": "P", "ent": "keystone", "cls": "garage"}, {"id": 4964, "dom": "P", "ent": "keystone", "cls": "highway"}, {"id": 4965, "dom": "P", "ent": "keystone", "cls": "kidney"}, {"id": 4966, "dom": "P", "ent": "keystone", "cls": "lamb"}, {"id": 4967, "dom": "P", "ent": "keystone", "cls": "male"}, {"id": 4968, "dom": "P", "ent": "keystone", "cls": "marijuana"}, {"id": 4969, "dom": "P", "ent": "keystone", "cls": "puppy"}, {"id": 4970, "dom": "P", "ent": "keystone", "cls": "receiver"}, {"id": 4971, "dom": "P", "ent": "keystone", "cls": "reed"}, {"id": 4972, "dom": "P", "ent": "keystone", "cls": "rifle"}, {"id": 4973, "dom": "P", "ent": "keystone", "cls": "road"}, {"id": 4974, "dom": "P", "ent": "keystone", "cls": "salmon"}, {"id": 4975, "dom": "P", "ent": "keystone", "cls": "shark"}, {"id": 4976, "dom": "P", "ent": "keystone", "cls": "sheep"}, {"id": 4977, "dom": "P", "ent": "keystone", "cls": "shook"}, {"id": 4978, "dom": "P", "ent": "keystone", "cls": "tank"}, {"id": 4979, "dom": "P", "ent": "keystone", "cls": "wheat"}, {"id": 4980, "dom": "P", "ent": "lag", "cls": "architecture"}, {"id": 4981, "dom": "P", "ent": "lag", "cls": "bacteria"}, {"id": 4982, "dom": "P", "ent": "lag", "cls": "beef"}, {"id": 4983, "dom": "P", "ent": "lag", "cls": "bridge"}, {"id": 4984, "dom": "P", "ent": "lag", "cls": "bud"}, {"id": 4985, "dom": "P", "ent": "lag", "cls": "cat"}, {"id": 4986, "dom": "P", "ent": "lag", "cls": "chapel"}, {"id": 4987, "dom": "P", "ent": "lag", "cls": "chick"}, {"id": 4988, "dom": "P", "ent": "lag", "cls": "clock"}, {"id": 4989, "dom": "P", "ent": "lag", "cls": "deer"}, {"id": 4990, "dom": "P", "ent": "lag", "cls": "dog"}, {"id": 4991, "dom": "P", "ent": "lag", "cls": "elephant"}, {"id": 4992, "dom": "P", "ent": "lag", "cls": "embassy"}, {"id": 4993, "dom": "P", "ent": "lag", "cls": "garage"}, {"id": 4994, "dom": "P", "ent": "lag", "cls": "highway"}, {"id": 4995, "dom": "P", "ent": "lag", "cls": "kidney"}, {"id": 4996, "dom": "P", "ent": "lag", "cls": "lamb"}, {"id": 4997, "dom": "P", "ent": "lag", "cls": "male"}, {"id": 4998, "dom": "P", "ent": "lag", "cls": "marijuana"}, {"id": 4999, "dom": "P", "ent": "lag", "cls": "puppy"}, {"id": 5000, "dom": "P", "ent": "lag", "cls": "receiver"}, {"id": 5001, "dom": "P", "ent": "lag", "cls": "reed"}, {"id": 5002, "dom": "P", "ent": "lag", "cls": "rifle"}, {"id": 5003, "dom": "P", "ent": "lag", "cls": "road"}, {"id": 5004, "dom": "P", "ent": "lag", "cls": "salmon"}, {"id": 5005, "dom": "P", "ent": "lag", "cls": "shark"}, {"id": 5006, "dom": "P", "ent": "lag", "cls": "sheep"}, {"id": 5007, "dom": "P", "ent": "lag", "cls": "shook"}, {"id": 5008, "dom": "P", "ent": "lag", "cls": "tank"}, {"id": 5009, "dom": "P", "ent": "lag", "cls": "wheat"}, {"id": 5010, "dom": "P", "ent": "landing", "cls": "architecture"}, {"id": 5011, "dom": "P", "ent": "landing", "cls": "bacteria"}, {"id": 5012, "dom": "P", "ent": "landing", "cls": "beef"}, {"id": 5013, "dom": "P", "ent": "landing", "cls": "bridge"}, {"id": 5014, "dom": "P", "ent": "landing", "cls": "bud"}, {"id": 5015, "dom": "P", "ent": "landing", "cls": "cat"}, {"id": 5016, "dom": "P", "ent": "landing", "cls": "chapel"}, {"id": 5017, "dom": "P", "ent": "landing", "cls": "chick"}, {"id": 5018, "dom": "P", "ent": "landing", "cls": "clock"}, {"id": 5019, "dom": "P", "ent": "landing", "cls": "deer"}, {"id": 5020, "dom": "P", "ent": "landing", "cls": "dog"}, {"id": 5021, "dom": "P", "ent": "landing", "cls": "elephant"}, {"id": 5022, "dom": "P", "ent": "landing", "cls": "embassy"}, {"id": 5023, "dom": "P", "ent": "landing", "cls": "garage"}, {"id": 5024, "dom": "P", "ent": "landing", "cls": "highway"}, {"id": 5025, "dom": "P", "ent": "landing", "cls": "kidney"}, {"id": 5026, "dom": "P", "ent": "landing", "cls": "lamb"}, {"id": 5027, "dom": "P", "ent": "landing", "cls": "male"}, {"id": 5028, "dom": "P", "ent": "landing", "cls": "marijuana"}, {"id": 5029, "dom": "P", "ent": "landing", "cls": "puppy"}, {"id": 5030, "dom": "P", "ent": "landing", "cls": "receiver"}, {"id": 5031, "dom": "P", "ent": "landing", "cls": "reed"}, {"id": 5032, "dom": "P", "ent": "landing", "cls": "rifle"}, {"id": 5033, "dom": "P", "ent": "landing", "cls": "road"}, {"id": 5034, "dom": "P", "ent": "landing", "cls": "salmon"}, {"id": 5035, "dom": "P", "ent": "landing", "cls": "shark"}, {"id": 5036, "dom": "P", "ent": "landing", "cls": "sheep"}, {"id": 5037, "dom": "P", "ent": "landing", "cls": "shook"}, {"id": 5038, "dom": "P", "ent": "landing", "cls": "tank"}, {"id": 5039, "dom": "P", "ent": "landing", "cls": "wheat"}, {"id": 5040, "dom": "P", "ent": "lash", "cls": "architecture"}, {"id": 5041, "dom": "P", "ent": "lash", "cls": "bacteria"}, {"id": 5042, "dom": "P", "ent": "lash", "cls": "beef"}, {"id": 5043, "dom": "P", "ent": "lash", "cls": "bridge"}, {"id": 5044, "dom": "P", "ent": "lash", "cls": "bud"}, {"id": 5045, "dom": "P", "ent": "lash", "cls": "cat"}, {"id": 5046, "dom": "P", "ent": "lash", "cls": "chapel"}, {"id": 5047, "dom": "P", "ent": "lash", "cls": "chick"}, {"id": 5048, "dom": "P", "ent": "lash", "cls": "clock"}, {"id": 5049, "dom": "P", "ent": "lash", "cls": "deer"}, {"id": 5050, "dom": "P", "ent": "lash", "cls": "dog"}, {"id": 5051, "dom": "P", "ent": "lash", "cls": "elephant"}, {"id": 5052, "dom": "P", "ent": "lash", "cls": "embassy"}, {"id": 5053, "dom": "P", "ent": "lash", "cls": "garage"}, {"id": 5054, "dom": "P", "ent": "lash", "cls": "highway"}, {"id": 5055, "dom": "P", "ent": "lash", "cls": "kidney"}, {"id": 5056, "dom": "P", "ent": "lash", "cls": "lamb"}, {"id": 5057, "dom": "P", "ent": "lash", "cls": "male"}, {"id": 5058, "dom": "P", "ent": "lash", "cls": "marijuana"}, {"id": 5059, "dom": "P", "ent": "lash", "cls": "puppy"}, {"id": 5060, "dom": "P", "ent": "lash", "cls": "receiver"}, {"id": 5061, "dom": "P", "ent": "lash", "cls": "reed"}, {"id": 5062, "dom": "P", "ent": "lash", "cls": "rifle"}, {"id": 5063, "dom": "P", "ent": "lash", "cls": "road"}, {"id": 5064, "dom": "P", "ent": "lash", "cls": "salmon"}, {"id": 5065, "dom": "P", "ent": "lash", "cls": "shark"}, {"id": 5066, "dom": "P", "ent": "lash", "cls": "sheep"}, {"id": 5067, "dom": "P", "ent": "lash", "cls": "shook"}, {"id": 5068, "dom": "P", "ent": "lash", "cls": "tank"}, {"id": 5069, "dom": "P", "ent": "lash", "cls": "wheat"}, {"id": 5070, "dom": "P", "ent": "left", "cls": "architecture"}, {"id": 5071, "dom": "P", "ent": "left", "cls": "bacteria"}, {"id": 5072, "dom": "P", "ent": "left", "cls": "beef"}, {"id": 5073, "dom": "P", "ent": "left", "cls": "bridge"}, {"id": 5074, "dom": "P", "ent": "left", "cls": "bud"}, {"id": 5075, "dom": "P", "ent": "left", "cls": "cat"}, {"id": 5076, "dom": "P", "ent": "left", "cls": "chapel"}, {"id": 5077, "dom": "P", "ent": "left", "cls": "chick"}, {"id": 5078, "dom": "P", "ent": "left", "cls": "clock"}, {"id": 5079, "dom": "P", "ent": "left", "cls": "deer"}, {"id": 5080, "dom": "P", "ent": "left", "cls": "dog"}, {"id": 5081, "dom": "P", "ent": "left", "cls": "elephant"}, {"id": 5082, "dom": "P", "ent": "left", "cls": "embassy"}, {"id": 5083, "dom": "P", "ent": "left", "cls": "garage"}, {"id": 5084, "dom": "P", "ent": "left", "cls": "highway"}, {"id": 5085, "dom": "P", "ent": "left", "cls": "kidney"}, {"id": 5086, "dom": "P", "ent": "left", "cls": "lamb"}, {"id": 5087, "dom": "P", "ent": "left", "cls": "male"}, {"id": 5088, "dom": "P", "ent": "left", "cls": "marijuana"}, {"id": 5089, "dom": "P", "ent": "left", "cls": "puppy"}, {"id": 5090, "dom": "P", "ent": "left", "cls": "receiver"}, {"id": 5091, "dom": "P", "ent": "left", "cls": "reed"}, {"id": 5092, "dom": "P", "ent": "left", "cls": "rifle"}, {"id": 5093, "dom": "P", "ent": "left", "cls": "road"}, {"id": 5094, "dom": "P", "ent": "left", "cls": "salmon"}, {"id": 5095, "dom": "P", "ent": "left", "cls": "shark"}, {"id": 5096, "dom": "P", "ent": "left", "cls": "sheep"}, {"id": 5097, "dom": "P", "ent": "left", "cls": "shook"}, {"id": 5098, "dom": "P", "ent": "left", "cls": "tank"}, {"id": 5099, "dom": "P", "ent": "left", "cls": "wheat"}, {"id": 5100, "dom": "P", "ent": "lens", "cls": "architecture"}, {"id": 5101, "dom": "P", "ent": "lens", "cls": "bacteria"}, {"id": 5102, "dom": "P", "ent": "lens", "cls": "beef"}, {"id": 5103, "dom": "P", "ent": "lens", "cls": "bridge"}, {"id": 5104, "dom": "P", "ent": "lens", "cls": "bud"}, {"id": 5105, "dom": "P", "ent": "lens", "cls": "cat"}, {"id": 5106, "dom": "P", "ent": "lens", "cls": "chapel"}, {"id": 5107, "dom": "P", "ent": "lens", "cls": "chick"}, {"id": 5108, "dom": "P", "ent": "lens", "cls": "clock"}, {"id": 5109, "dom": "P", "ent": "lens", "cls": "deer"}, {"id": 5110, "dom": "P", "ent": "lens", "cls": "dog"}, {"id": 5111, "dom": "P", "ent": "lens", "cls": "elephant"}, {"id": 5112, "dom": "P", "ent": "lens", "cls": "embassy"}, {"id": 5113, "dom": "P", "ent": "lens", "cls": "garage"}, {"id": 5114, "dom": "P", "ent": "lens", "cls": "highway"}, {"id": 5115, "dom": "P", "ent": "lens", "cls": "kidney"}, {"id": 5116, "dom": "P", "ent": "lens", "cls": "lamb"}, {"id": 5117, "dom": "P", "ent": "lens", "cls": "male"}, {"id": 5118, "dom": "P", "ent": "lens", "cls": "marijuana"}, {"id": 5119, "dom": "P", "ent": "lens", "cls": "puppy"}, {"id": 5120, "dom": "P", "ent": "lens", "cls": "receiver"}, {"id": 5121, "dom": "P", "ent": "lens", "cls": "reed"}, {"id": 5122, "dom": "P", "ent": "lens", "cls": "rifle"}, {"id": 5123, "dom": "P", "ent": "lens", "cls": "road"}, {"id": 5124, "dom": "P", "ent": "lens", "cls": "salmon"}, {"id": 5125, "dom": "P", "ent": "lens", "cls": "shark"}, {"id": 5126, "dom": "P", "ent": "lens", "cls": "sheep"}, {"id": 5127, "dom": "P", "ent": "lens", "cls": "shook"}, {"id": 5128, "dom": "P", "ent": "lens", "cls": "tank"}, {"id": 5129, "dom": "P", "ent": "lens", "cls": "wheat"}, {"id": 5130, "dom": "P", "ent": "level", "cls": "architecture"}, {"id": 5131, "dom": "P", "ent": "level", "cls": "bacteria"}, {"id": 5132, "dom": "P", "ent": "level", "cls": "beef"}, {"id": 5133, "dom": "P", "ent": "level", "cls": "bridge"}, {"id": 5134, "dom": "P", "ent": "level", "cls": "bud"}, {"id": 5135, "dom": "P", "ent": "level", "cls": "cat"}, {"id": 5136, "dom": "P", "ent": "level", "cls": "chapel"}, {"id": 5137, "dom": "P", "ent": "level", "cls": "chick"}, {"id": 5138, "dom": "P", "ent": "level", "cls": "clock"}, {"id": 5139, "dom": "P", "ent": "level", "cls": "deer"}, {"id": 5140, "dom": "P", "ent": "level", "cls": "dog"}, {"id": 5141, "dom": "P", "ent": "level", "cls": "elephant"}, {"id": 5142, "dom": "P", "ent": "level", "cls": "embassy"}, {"id": 5143, "dom": "P", "ent": "level", "cls": "garage"}, {"id": 5144, "dom": "P", "ent": "level", "cls": "highway"}, {"id": 5145, "dom": "P", "ent": "level", "cls": "kidney"}, {"id": 5146, "dom": "P", "ent": "level", "cls": "lamb"}, {"id": 5147, "dom": "P", "ent": "level", "cls": "male"}, {"id": 5148, "dom": "P", "ent": "level", "cls": "marijuana"}, {"id": 5149, "dom": "P", "ent": "level", "cls": "puppy"}, {"id": 5150, "dom": "P", "ent": "level", "cls": "receiver"}, {"id": 5151, "dom": "P", "ent": "level", "cls": "reed"}, {"id": 5152, "dom": "P", "ent": "level", "cls": "rifle"}, {"id": 5153, "dom": "P", "ent": "level", "cls": "road"}, {"id": 5154, "dom": "P", "ent": "level", "cls": "salmon"}, {"id": 5155, "dom": "P", "ent": "level", "cls": "shark"}, {"id": 5156, "dom": "P", "ent": "level", "cls": "sheep"}, {"id": 5157, "dom": "P", "ent": "level", "cls": "shook"}, {"id": 5158, "dom": "P", "ent": "level", "cls": "tank"}, {"id": 5159, "dom": "P", "ent": "level", "cls": "wheat"}, {"id": 5160, "dom": "P", "ent": "lid", "cls": "architecture"}, {"id": 5161, "dom": "P", "ent": "lid", "cls": "bacteria"}, {"id": 5162, "dom": "P", "ent": "lid", "cls": "beef"}, {"id": 5163, "dom": "P", "ent": "lid", "cls": "bridge"}, {"id": 5164, "dom": "P", "ent": "lid", "cls": "bud"}, {"id": 5165, "dom": "P", "ent": "lid", "cls": "cat"}, {"id": 5166, "dom": "P", "ent": "lid", "cls": "chapel"}, {"id": 5167, "dom": "P", "ent": "lid", "cls": "chick"}, {"id": 5168, "dom": "P", "ent": "lid", "cls": "clock"}, {"id": 5169, "dom": "P", "ent": "lid", "cls": "deer"}, {"id": 5170, "dom": "P", "ent": "lid", "cls": "dog"}, {"id": 5171, "dom": "P", "ent": "lid", "cls": "elephant"}, {"id": 5172, "dom": "P", "ent": "lid", "cls": "embassy"}, {"id": 5173, "dom": "P", "ent": "lid", "cls": "garage"}, {"id": 5174, "dom": "P", "ent": "lid", "cls": "highway"}, {"id": 5175, "dom": "P", "ent": "lid", "cls": "kidney"}, {"id": 5176, "dom": "P", "ent": "lid", "cls": "lamb"}, {"id": 5177, "dom": "P", "ent": "lid", "cls": "male"}, {"id": 5178, "dom": "P", "ent": "lid", "cls": "marijuana"}, {"id": 5179, "dom": "P", "ent": "lid", "cls": "puppy"}, {"id": 5180, "dom": "P", "ent": "lid", "cls": "receiver"}, {"id": 5181, "dom": "P", "ent": "lid", "cls": "reed"}, {"id": 5182, "dom": "P", "ent": "lid", "cls": "rifle"}, {"id": 5183, "dom": "P", "ent": "lid", "cls": "road"}, {"id": 5184, "dom": "P", "ent": "lid", "cls": "salmon"}, {"id": 5185, "dom": "P", "ent": "lid", "cls": "shark"}, {"id": 5186, "dom": "P", "ent": "lid", "cls": "sheep"}, {"id": 5187, "dom": "P", "ent": "lid", "cls": "shook"}, {"id": 5188, "dom": "P", "ent": "lid", "cls": "tank"}, {"id": 5189, "dom": "P", "ent": "lid", "cls": "wheat"}, {"id": 5190, "dom": "P", "ent": "lift", "cls": "architecture"}, {"id": 5191, "dom": "P", "ent": "lift", "cls": "bacteria"}, {"id": 5192, "dom": "P", "ent": "lift", "cls": "beef"}, {"id": 5193, "dom": "P", "ent": "lift", "cls": "bridge"}, {"id": 5194, "dom": "P", "ent": "lift", "cls": "bud"}, {"id": 5195, "dom": "P", "ent": "lift", "cls": "cat"}, {"id": 5196, "dom": "P", "ent": "lift", "cls": "chapel"}, {"id": 5197, "dom": "P", "ent": "lift", "cls": "chick"}, {"id": 5198, "dom": "P", "ent": "lift", "cls": "clock"}, {"id": 5199, "dom": "P", "ent": "lift", "cls": "deer"}, {"id": 5200, "dom": "P", "ent": "lift", "cls": "dog"}, {"id": 5201, "dom": "P", "ent": "lift", "cls": "elephant"}, {"id": 5202, "dom": "P", "ent": "lift", "cls": "embassy"}, {"id": 5203, "dom": "P", "ent": "lift", "cls": "garage"}, {"id": 5204, "dom": "P", "ent": "lift", "cls": "highway"}, {"id": 5205, "dom": "P", "ent": "lift", "cls": "kidney"}, {"id": 5206, "dom": "P", "ent": "lift", "cls": "lamb"}, {"id": 5207, "dom": "P", "ent": "lift", "cls": "male"}, {"id": 5208, "dom": "P", "ent": "lift", "cls": "marijuana"}, {"id": 5209, "dom": "P", "ent": "lift", "cls": "puppy"}, {"id": 5210, "dom": "P", "ent": "lift", "cls": "receiver"}, {"id": 5211, "dom": "P", "ent": "lift", "cls": "reed"}, {"id": 5212, "dom": "P", "ent": "lift", "cls": "rifle"}, {"id": 5213, "dom": "P", "ent": "lift", "cls": "road"}, {"id": 5214, "dom": "P", "ent": "lift", "cls": "salmon"}, {"id": 5215, "dom": "P", "ent": "lift", "cls": "shark"}, {"id": 5216, "dom": "P", "ent": "lift", "cls": "sheep"}, {"id": 5217, "dom": "P", "ent": "lift", "cls": "shook"}, {"id": 5218, "dom": "P", "ent": "lift", "cls": "tank"}, {"id": 5219, "dom": "P", "ent": "lift", "cls": "wheat"}, {"id": 5220, "dom": "P", "ent": "lighthouse", "cls": "architecture"}, {"id": 5221, "dom": "P", "ent": "lighthouse", "cls": "bacteria"}, {"id": 5222, "dom": "P", "ent": "lighthouse", "cls": "beef"}, {"id": 5223, "dom": "P", "ent": "lighthouse", "cls": "bridge"}, {"id": 5224, "dom": "P", "ent": "lighthouse", "cls": "bud"}, {"id": 5225, "dom": "P", "ent": "lighthouse", "cls": "cat"}, {"id": 5226, "dom": "P", "ent": "lighthouse", "cls": "chapel"}, {"id": 5227, "dom": "P", "ent": "lighthouse", "cls": "chick"}, {"id": 5228, "dom": "P", "ent": "lighthouse", "cls": "clock"}, {"id": 5229, "dom": "P", "ent": "lighthouse", "cls": "deer"}, {"id": 5230, "dom": "P", "ent": "lighthouse", "cls": "dog"}, {"id": 5231, "dom": "P", "ent": "lighthouse", "cls": "elephant"}, {"id": 5232, "dom": "P", "ent": "lighthouse", "cls": "embassy"}, {"id": 5233, "dom": "P", "ent": "lighthouse", "cls": "garage"}, {"id": 5234, "dom": "P", "ent": "lighthouse", "cls": "highway"}, {"id": 5235, "dom": "P", "ent": "lighthouse", "cls": "kidney"}, {"id": 5236, "dom": "P", "ent": "lighthouse", "cls": "lamb"}, {"id": 5237, "dom": "P", "ent": "lighthouse", "cls": "male"}, {"id": 5238, "dom": "P", "ent": "lighthouse", "cls": "marijuana"}, {"id": 5239, "dom": "P", "ent": "lighthouse", "cls": "puppy"}, {"id": 5240, "dom": "P", "ent": "lighthouse", "cls": "receiver"}, {"id": 5241, "dom": "P", "ent": "lighthouse", "cls": "reed"}, {"id": 5242, "dom": "P", "ent": "lighthouse", "cls": "rifle"}, {"id": 5243, "dom": "P", "ent": "lighthouse", "cls": "road"}, {"id": 5244, "dom": "P", "ent": "lighthouse", "cls": "salmon"}, {"id": 5245, "dom": "P", "ent": "lighthouse", "cls": "shark"}, {"id": 5246, "dom": "P", "ent": "lighthouse", "cls": "sheep"}, {"id": 5247, "dom": "P", "ent": "lighthouse", "cls": "shook"}, {"id": 5248, "dom": "P", "ent": "lighthouse", "cls": "tank"}, {"id": 5249, "dom": "P", "ent": "lighthouse", "cls": "wheat"}, {"id": 5250, "dom": "P", "ent": "lip", "cls": "architecture"}, {"id": 5251, "dom": "P", "ent": "lip", "cls": "bacteria"}, {"id": 5252, "dom": "P", "ent": "lip", "cls": "beef"}, {"id": 5253, "dom": "P", "ent": "lip", "cls": "bridge"}, {"id": 5254, "dom": "P", "ent": "lip", "cls": "bud"}, {"id": 5255, "dom": "P", "ent": "lip", "cls": "cat"}, {"id": 5256, "dom": "P", "ent": "lip", "cls": "chapel"}, {"id": 5257, "dom": "P", "ent": "lip", "cls": "chick"}, {"id": 5258, "dom": "P", "ent": "lip", "cls": "clock"}, {"id": 5259, "dom": "P", "ent": "lip", "cls": "deer"}, {"id": 5260, "dom": "P", "ent": "lip", "cls": "dog"}, {"id": 5261, "dom": "P", "ent": "lip", "cls": "elephant"}, {"id": 5262, "dom": "P", "ent": "lip", "cls": "embassy"}, {"id": 5263, "dom": "P", "ent": "lip", "cls": "garage"}, {"id": 5264, "dom": "P", "ent": "lip", "cls": "highway"}, {"id": 5265, "dom": "P", "ent": "lip", "cls": "kidney"}, {"id": 5266, "dom": "P", "ent": "lip", "cls": "lamb"}, {"id": 5267, "dom": "P", "ent": "lip", "cls": "male"}, {"id": 5268, "dom": "P", "ent": "lip", "cls": "marijuana"}, {"id": 5269, "dom": "P", "ent": "lip", "cls": "puppy"}, {"id": 5270, "dom": "P", "ent": "lip", "cls": "receiver"}, {"id": 5271, "dom": "P", "ent": "lip", "cls": "reed"}, {"id": 5272, "dom": "P", "ent": "lip", "cls": "rifle"}, {"id": 5273, "dom": "P", "ent": "lip", "cls": "road"}, {"id": 5274, "dom": "P", "ent": "lip", "cls": "salmon"}, {"id": 5275, "dom": "P", "ent": "lip", "cls": "shark"}, {"id": 5276, "dom": "P", "ent": "lip", "cls": "sheep"}, {"id": 5277, "dom": "P", "ent": "lip", "cls": "shook"}, {"id": 5278, "dom": "P", "ent": "lip", "cls": "tank"}, {"id": 5279, "dom": "P", "ent": "lip", "cls": "wheat"}, {"id": 5280, "dom": "P", "ent": "lobby", "cls": "architecture"}, {"id": 5281, "dom": "P", "ent": "lobby", "cls": "bacteria"}, {"id": 5282, "dom": "P", "ent": "lobby", "cls": "beef"}, {"id": 5283, "dom": "P", "ent": "lobby", "cls": "bridge"}, {"id": 5284, "dom": "P", "ent": "lobby", "cls": "bud"}, {"id": 5285, "dom": "P", "ent": "lobby", "cls": "cat"}, {"id": 5286, "dom": "P", "ent": "lobby", "cls": "chapel"}, {"id": 5287, "dom": "P", "ent": "lobby", "cls": "chick"}, {"id": 5288, "dom": "P", "ent": "lobby", "cls": "clock"}, {"id": 5289, "dom": "P", "ent": "lobby", "cls": "deer"}, {"id": 5290, "dom": "P", "ent": "lobby", "cls": "dog"}, {"id": 5291, "dom": "P", "ent": "lobby", "cls": "elephant"}, {"id": 5292, "dom": "P", "ent": "lobby", "cls": "embassy"}, {"id": 5293, "dom": "P", "ent": "lobby", "cls": "garage"}, {"id": 5294, "dom": "P", "ent": "lobby", "cls": "highway"}, {"id": 5295, "dom": "P", "ent": "lobby", "cls": "kidney"}, {"id": 5296, "dom": "P", "ent": "lobby", "cls": "lamb"}, {"id": 5297, "dom": "P", "ent": "lobby", "cls": "male"}, {"id": 5298, "dom": "P", "ent": "lobby", "cls": "marijuana"}, {"id": 5299, "dom": "P", "ent": "lobby", "cls": "puppy"}, {"id": 5300, "dom": "P", "ent": "lobby", "cls": "receiver"}, {"id": 5301, "dom": "P", "ent": "lobby", "cls": "reed"}, {"id": 5302, "dom": "P", "ent": "lobby", "cls": "rifle"}, {"id": 5303, "dom": "P", "ent": "lobby", "cls": "road"}, {"id": 5304, "dom": "P", "ent": "lobby", "cls": "salmon"}, {"id": 5305, "dom": "P", "ent": "lobby", "cls": "shark"}, {"id": 5306, "dom": "P", "ent": "lobby", "cls": "sheep"}, {"id": 5307, "dom": "P", "ent": "lobby", "cls": "shook"}, {"id": 5308, "dom": "P", "ent": "lobby", "cls": "tank"}, {"id": 5309, "dom": "P", "ent": "lobby", "cls": "wheat"}, {"id": 5310, "dom": "P", "ent": "lobe", "cls": "architecture"}, {"id": 5311, "dom": "P", "ent": "lobe", "cls": "bacteria"}, {"id": 5312, "dom": "P", "ent": "lobe", "cls": "beef"}, {"id": 5313, "dom": "P", "ent": "lobe", "cls": "bridge"}, {"id": 5314, "dom": "P", "ent": "lobe", "cls": "bud"}, {"id": 5315, "dom": "P", "ent": "lobe", "cls": "cat"}, {"id": 5316, "dom": "P", "ent": "lobe", "cls": "chapel"}, {"id": 5317, "dom": "P", "ent": "lobe", "cls": "chick"}, {"id": 5318, "dom": "P", "ent": "lobe", "cls": "clock"}, {"id": 5319, "dom": "P", "ent": "lobe", "cls": "deer"}, {"id": 5320, "dom": "P", "ent": "lobe", "cls": "dog"}, {"id": 5321, "dom": "P", "ent": "lobe", "cls": "elephant"}, {"id": 5322, "dom": "P", "ent": "lobe", "cls": "embassy"}, {"id": 5323, "dom": "P", "ent": "lobe", "cls": "garage"}, {"id": 5324, "dom": "P", "ent": "lobe", "cls": "highway"}, {"id": 5325, "dom": "P", "ent": "lobe", "cls": "kidney"}, {"id": 5326, "dom": "P", "ent": "lobe", "cls": "lamb"}, {"id": 5327, "dom": "P", "ent": "lobe", "cls": "male"}, {"id": 5328, "dom": "P", "ent": "lobe", "cls": "marijuana"}, {"id": 5329, "dom": "P", "ent": "lobe", "cls": "puppy"}, {"id": 5330, "dom": "P", "ent": "lobe", "cls": "receiver"}, {"id": 5331, "dom": "P", "ent": "lobe", "cls": "reed"}, {"id": 5332, "dom": "P", "ent": "lobe", "cls": "rifle"}, {"id": 5333, "dom": "P", "ent": "lobe", "cls": "road"}, {"id": 5334, "dom": "P", "ent": "lobe", "cls": "salmon"}, {"id": 5335, "dom": "P", "ent": "lobe", "cls": "shark"}, {"id": 5336, "dom": "P", "ent": "lobe", "cls": "sheep"}, {"id": 5337, "dom": "P", "ent": "lobe", "cls": "shook"}, {"id": 5338, "dom": "P", "ent": "lobe", "cls": "tank"}, {"id": 5339, "dom": "P", "ent": "lobe", "cls": "wheat"}, {"id": 5340, "dom": "P", "ent": "lock", "cls": "architecture"}, {"id": 5341, "dom": "P", "ent": "lock", "cls": "bacteria"}, {"id": 5342, "dom": "P", "ent": "lock", "cls": "beef"}, {"id": 5343, "dom": "P", "ent": "lock", "cls": "bridge"}, {"id": 5344, "dom": "P", "ent": "lock", "cls": "bud"}, {"id": 5345, "dom": "P", "ent": "lock", "cls": "cat"}, {"id": 5346, "dom": "P", "ent": "lock", "cls": "chapel"}, {"id": 5347, "dom": "P", "ent": "lock", "cls": "chick"}, {"id": 5348, "dom": "P", "ent": "lock", "cls": "clock"}, {"id": 5349, "dom": "P", "ent": "lock", "cls": "deer"}, {"id": 5350, "dom": "P", "ent": "lock", "cls": "dog"}, {"id": 5351, "dom": "P", "ent": "lock", "cls": "elephant"}, {"id": 5352, "dom": "P", "ent": "lock", "cls": "embassy"}, {"id": 5353, "dom": "P", "ent": "lock", "cls": "garage"}, {"id": 5354, "dom": "P", "ent": "lock", "cls": "highway"}, {"id": 5355, "dom": "P", "ent": "lock", "cls": "kidney"}, {"id": 5356, "dom": "P", "ent": "lock", "cls": "lamb"}, {"id": 5357, "dom": "P", "ent": "lock", "cls": "male"}, {"id": 5358, "dom": "P", "ent": "lock", "cls": "marijuana"}, {"id": 5359, "dom": "P", "ent": "lock", "cls": "puppy"}, {"id": 5360, "dom": "P", "ent": "lock", "cls": "receiver"}, {"id": 5361, "dom": "P", "ent": "lock", "cls": "reed"}, {"id": 5362, "dom": "P", "ent": "lock", "cls": "rifle"}, {"id": 5363, "dom": "P", "ent": "lock", "cls": "road"}, {"id": 5364, "dom": "P", "ent": "lock", "cls": "salmon"}, {"id": 5365, "dom": "P", "ent": "lock", "cls": "shark"}, {"id": 5366, "dom": "P", "ent": "lock", "cls": "sheep"}, {"id": 5367, "dom": "P", "ent": "lock", "cls": "shook"}, {"id": 5368, "dom": "P", "ent": "lock", "cls": "tank"}, {"id": 5369, "dom": "P", "ent": "lock", "cls": "wheat"}, {"id": 5370, "dom": "P", "ent": "lounge", "cls": "architecture"}, {"id": 5371, "dom": "P", "ent": "lounge", "cls": "bacteria"}, {"id": 5372, "dom": "P", "ent": "lounge", "cls": "beef"}, {"id": 5373, "dom": "P", "ent": "lounge", "cls": "bridge"}, {"id": 5374, "dom": "P", "ent": "lounge", "cls": "bud"}, {"id": 5375, "dom": "P", "ent": "lounge", "cls": "cat"}, {"id": 5376, "dom": "P", "ent": "lounge", "cls": "chapel"}, {"id": 5377, "dom": "P", "ent": "lounge", "cls": "chick"}, {"id": 5378, "dom": "P", "ent": "lounge", "cls": "clock"}, {"id": 5379, "dom": "P", "ent": "lounge", "cls": "deer"}, {"id": 5380, "dom": "P", "ent": "lounge", "cls": "dog"}, {"id": 5381, "dom": "P", "ent": "lounge", "cls": "elephant"}, {"id": 5382, "dom": "P", "ent": "lounge", "cls": "embassy"}, {"id": 5383, "dom": "P", "ent": "lounge", "cls": "garage"}, {"id": 5384, "dom": "P", "ent": "lounge", "cls": "highway"}, {"id": 5385, "dom": "P", "ent": "lounge", "cls": "kidney"}, {"id": 5386, "dom": "P", "ent": "lounge", "cls": "lamb"}, {"id": 5387, "dom": "P", "ent": "lounge", "cls": "male"}, {"id": 5388, "dom": "P", "ent": "lounge", "cls": "marijuana"}, {"id": 5389, "dom": "P", "ent": "lounge", "cls": "puppy"}, {"id": 5390, "dom": "P", "ent": "lounge", "cls": "receiver"}, {"id": 5391, "dom": "P", "ent": "lounge", "cls": "reed"}, {"id": 5392, "dom": "P", "ent": "lounge", "cls": "rifle"}, {"id": 5393, "dom": "P", "ent": "lounge", "cls": "road"}, {"id": 5394, "dom": "P", "ent": "lounge", "cls": "salmon"}, {"id": 5395, "dom": "P", "ent": "lounge", "cls": "shark"}, {"id": 5396, "dom": "P", "ent": "lounge", "cls": "sheep"}, {"id": 5397, "dom": "P", "ent": "lounge", "cls": "shook"}, {"id": 5398, "dom": "P", "ent": "lounge", "cls": "tank"}, {"id": 5399, "dom": "P", "ent": "lounge", "cls": "wheat"}, {"id": 5400, "dom": "P", "ent": "magazine", "cls": "architecture"}, {"id": 5401, "dom": "P", "ent": "magazine", "cls": "bacteria"}, {"id": 5402, "dom": "P", "ent": "magazine", "cls": "beef"}, {"id": 5403, "dom": "P", "ent": "magazine", "cls": "bridge"}, {"id": 5404, "dom": "P", "ent": "magazine", "cls": "bud"}, {"id": 5405, "dom": "P", "ent": "magazine", "cls": "cat"}, {"id": 5406, "dom": "P", "ent": "magazine", "cls": "chapel"}, {"id": 5407, "dom": "P", "ent": "magazine", "cls": "chick"}, {"id": 5408, "dom": "P", "ent": "magazine", "cls": "clock"}, {"id": 5409, "dom": "P", "ent": "magazine", "cls": "deer"}, {"id": 5410, "dom": "P", "ent": "magazine", "cls": "dog"}, {"id": 5411, "dom": "P", "ent": "magazine", "cls": "elephant"}, {"id": 5412, "dom": "P", "ent": "magazine", "cls": "embassy"}, {"id": 5413, "dom": "P", "ent": "magazine", "cls": "garage"}, {"id": 5414, "dom": "P", "ent": "magazine", "cls": "highway"}, {"id": 5415, "dom": "P", "ent": "magazine", "cls": "kidney"}, {"id": 5416, "dom": "P", "ent": "magazine", "cls": "lamb"}, {"id": 5417, "dom": "P", "ent": "magazine", "cls": "male"}, {"id": 5418, "dom": "P", "ent": "magazine", "cls": "marijuana"}, {"id": 5419, "dom": "P", "ent": "magazine", "cls": "puppy"}, {"id": 5420, "dom": "P", "ent": "magazine", "cls": "receiver"}, {"id": 5421, "dom": "P", "ent": "magazine", "cls": "reed"}, {"id": 5422, "dom": "P", "ent": "magazine", "cls": "rifle"}, {"id": 5423, "dom": "P", "ent": "magazine", "cls": "road"}, {"id": 5424, "dom": "P", "ent": "magazine", "cls": "salmon"}, {"id": 5425, "dom": "P", "ent": "magazine", "cls": "shark"}, {"id": 5426, "dom": "P", "ent": "magazine", "cls": "sheep"}, {"id": 5427, "dom": "P", "ent": "magazine", "cls": "shook"}, {"id": 5428, "dom": "P", "ent": "magazine", "cls": "tank"}, {"id": 5429, "dom": "P", "ent": "magazine", "cls": "wheat"}, {"id": 5430, "dom": "P", "ent": "mast", "cls": "architecture"}, {"id": 5431, "dom": "P", "ent": "mast", "cls": "bacteria"}, {"id": 5432, "dom": "P", "ent": "mast", "cls": "beef"}, {"id": 5433, "dom": "P", "ent": "mast", "cls": "bridge"}, {"id": 5434, "dom": "P", "ent": "mast", "cls": "bud"}, {"id": 5435, "dom": "P", "ent": "mast", "cls": "cat"}, {"id": 5436, "dom": "P", "ent": "mast", "cls": "chapel"}, {"id": 5437, "dom": "P", "ent": "mast", "cls": "chick"}, {"id": 5438, "dom": "P", "ent": "mast", "cls": "clock"}, {"id": 5439, "dom": "P", "ent": "mast", "cls": "deer"}, {"id": 5440, "dom": "P", "ent": "mast", "cls": "dog"}, {"id": 5441, "dom": "P", "ent": "mast", "cls": "elephant"}, {"id": 5442, "dom": "P", "ent": "mast", "cls": "embassy"}, {"id": 5443, "dom": "P", "ent": "mast", "cls": "garage"}, {"id": 5444, "dom": "P", "ent": "mast", "cls": "highway"}, {"id": 5445, "dom": "P", "ent": "mast", "cls": "kidney"}, {"id": 5446, "dom": "P", "ent": "mast", "cls": "lamb"}, {"id": 5447, "dom": "P", "ent": "mast", "cls": "male"}, {"id": 5448, "dom": "P", "ent": "mast", "cls": "marijuana"}, {"id": 5449, "dom": "P", "ent": "mast", "cls": "puppy"}, {"id": 5450, "dom": "P", "ent": "mast", "cls": "receiver"}, {"id": 5451, "dom": "P", "ent": "mast", "cls": "reed"}, {"id": 5452, "dom": "P", "ent": "mast", "cls": "rifle"}, {"id": 5453, "dom": "P", "ent": "mast", "cls": "road"}, {"id": 5454, "dom": "P", "ent": "mast", "cls": "salmon"}, {"id": 5455, "dom": "P", "ent": "mast", "cls": "shark"}, {"id": 5456, "dom": "P", "ent": "mast", "cls": "sheep"}, {"id": 5457, "dom": "P", "ent": "mast", "cls": "shook"}, {"id": 5458, "dom": "P", "ent": "mast", "cls": "tank"}, {"id": 5459, "dom": "P", "ent": "mast", "cls": "wheat"}, {"id": 5460, "dom": "P", "ent": "matrix", "cls": "architecture"}, {"id": 5461, "dom": "P", "ent": "matrix", "cls": "bacteria"}, {"id": 5462, "dom": "P", "ent": "matrix", "cls": "beef"}, {"id": 5463, "dom": "P", "ent": "matrix", "cls": "bridge"}, {"id": 5464, "dom": "P", "ent": "matrix", "cls": "bud"}, {"id": 5465, "dom": "P", "ent": "matrix", "cls": "cat"}, {"id": 5466, "dom": "P", "ent": "matrix", "cls": "chapel"}, {"id": 5467, "dom": "P", "ent": "matrix", "cls": "chick"}, {"id": 5468, "dom": "P", "ent": "matrix", "cls": "clock"}, {"id": 5469, "dom": "P", "ent": "matrix", "cls": "deer"}, {"id": 5470, "dom": "P", "ent": "matrix", "cls": "dog"}, {"id": 5471, "dom": "P", "ent": "matrix", "cls": "elephant"}, {"id": 5472, "dom": "P", "ent": "matrix", "cls": "embassy"}, {"id": 5473, "dom": "P", "ent": "matrix", "cls": "garage"}, {"id": 5474, "dom": "P", "ent": "matrix", "cls": "highway"}, {"id": 5475, "dom": "P", "ent": "matrix", "cls": "kidney"}, {"id": 5476, "dom": "P", "ent": "matrix", "cls": "lamb"}, {"id": 5477, "dom": "P", "ent": "matrix", "cls": "male"}, {"id": 5478, "dom": "P", "ent": "matrix", "cls": "marijuana"}, {"id": 5479, "dom": "P", "ent": "matrix", "cls": "puppy"}, {"id": 5480, "dom": "P", "ent": "matrix", "cls": "receiver"}, {"id": 5481, "dom": "P", "ent": "matrix", "cls": "reed"}, {"id": 5482, "dom": "P", "ent": "matrix", "cls": "rifle"}, {"id": 5483, "dom": "P", "ent": "matrix", "cls": "road"}, {"id": 5484, "dom": "P", "ent": "matrix", "cls": "salmon"}, {"id": 5485, "dom": "P", "ent": "matrix", "cls": "shark"}, {"id": 5486, "dom": "P", "ent": "matrix", "cls": "sheep"}, {"id": 5487, "dom": "P", "ent": "matrix", "cls": "shook"}, {"id": 5488, "dom": "P", "ent": "matrix", "cls": "tank"}, {"id": 5489, "dom": "P", "ent": "matrix", "cls": "wheat"}, {"id": 5490, "dom": "P", "ent": "mock-up", "cls": "architecture"}, {"id": 5491, "dom": "P", "ent": "mock-up", "cls": "bacteria"}, {"id": 5492, "dom": "P", "ent": "mock-up", "cls": "beef"}, {"id": 5493, "dom": "P", "ent": "mock-up", "cls": "bridge"}, {"id": 5494, "dom": "P", "ent": "mock-up", "cls": "bud"}, {"id": 5495, "dom": "P", "ent": "mock-up", "cls": "cat"}, {"id": 5496, "dom": "P", "ent": "mock-up", "cls": "chapel"}, {"id": 5497, "dom": "P", "ent": "mock-up", "cls": "chick"}, {"id": 5498, "dom": "P", "ent": "mock-up", "cls": "clock"}, {"id": 5499, "dom": "P", "ent": "mock-up", "cls": "deer"}, {"id": 5500, "dom": "P", "ent": "mock-up", "cls": "dog"}, {"id": 5501, "dom": "P", "ent": "mock-up", "cls": "elephant"}, {"id": 5502, "dom": "P", "ent": "mock-up", "cls": "embassy"}, {"id": 5503, "dom": "P", "ent": "mock-up", "cls": "garage"}, {"id": 5504, "dom": "P", "ent": "mock-up", "cls": "highway"}, {"id": 5505, "dom": "P", "ent": "mock-up", "cls": "kidney"}, {"id": 5506, "dom": "P", "ent": "mock-up", "cls": "lamb"}, {"id": 5507, "dom": "P", "ent": "mock-up", "cls": "male"}, {"id": 5508, "dom": "P", "ent": "mock-up", "cls": "marijuana"}, {"id": 5509, "dom": "P", "ent": "mock-up", "cls": "puppy"}, {"id": 5510, "dom": "P", "ent": "mock-up", "cls": "receiver"}, {"id": 5511, "dom": "P", "ent": "mock-up", "cls": "reed"}, {"id": 5512, "dom": "P", "ent": "mock-up", "cls": "rifle"}, {"id": 5513, "dom": "P", "ent": "mock-up", "cls": "road"}, {"id": 5514, "dom": "P", "ent": "mock-up", "cls": "salmon"}, {"id": 5515, "dom": "P", "ent": "mock-up", "cls": "shark"}, {"id": 5516, "dom": "P", "ent": "mock-up", "cls": "sheep"}, {"id": 5517, "dom": "P", "ent": "mock-up", "cls": "shook"}, {"id": 5518, "dom": "P", "ent": "mock-up", "cls": "tank"}, {"id": 5519, "dom": "P", "ent": "mock-up", "cls": "wheat"}, {"id": 5520, "dom": "P", "ent": "monastery", "cls": "architecture"}, {"id": 5521, "dom": "P", "ent": "monastery", "cls": "bacteria"}, {"id": 5522, "dom": "P", "ent": "monastery", "cls": "beef"}, {"id": 5523, "dom": "P", "ent": "monastery", "cls": "bridge"}, {"id": 5524, "dom": "P", "ent": "monastery", "cls": "bud"}, {"id": 5525, "dom": "P", "ent": "monastery", "cls": "cat"}, {"id": 5526, "dom": "P", "ent": "monastery", "cls": "chapel"}, {"id": 5527, "dom": "P", "ent": "monastery", "cls": "chick"}, {"id": 5528, "dom": "P", "ent": "monastery", "cls": "clock"}, {"id": 5529, "dom": "P", "ent": "monastery", "cls": "deer"}, {"id": 5530, "dom": "P", "ent": "monastery", "cls": "dog"}, {"id": 5531, "dom": "P", "ent": "monastery", "cls": "elephant"}, {"id": 5532, "dom": "P", "ent": "monastery", "cls": "embassy"}, {"id": 5533, "dom": "P", "ent": "monastery", "cls": "garage"}, {"id": 5534, "dom": "P", "ent": "monastery", "cls": "highway"}, {"id": 5535, "dom": "P", "ent": "monastery", "cls": "kidney"}, {"id": 5536, "dom": "P", "ent": "monastery", "cls": "lamb"}, {"id": 5537, "dom": "P", "ent": "monastery", "cls": "male"}, {"id": 5538, "dom": "P", "ent": "monastery", "cls": "marijuana"}, {"id": 5539, "dom": "P", "ent": "monastery", "cls": "puppy"}, {"id": 5540, "dom": "P", "ent": "monastery", "cls": "receiver"}, {"id": 5541, "dom": "P", "ent": "monastery", "cls": "reed"}, {"id": 5542, "dom": "P", "ent": "monastery", "cls": "rifle"}, {"id": 5543, "dom": "P", "ent": "monastery", "cls": "road"}, {"id": 5544, "dom": "P", "ent": "monastery", "cls": "salmon"}, {"id": 5545, "dom": "P", "ent": "monastery", "cls": "shark"}, {"id": 5546, "dom": "P", "ent": "monastery", "cls": "sheep"}, {"id": 5547, "dom": "P", "ent": "monastery", "cls": "shook"}, {"id": 5548, "dom": "P", "ent": "monastery", "cls": "tank"}, {"id": 5549, "dom": "P", "ent": "monastery", "cls": "wheat"}, {"id": 5550, "dom": "P", "ent": "moss", "cls": "architecture"}, {"id": 5551, "dom": "P", "ent": "moss", "cls": "bacteria"}, {"id": 5552, "dom": "P", "ent": "moss", "cls": "beef"}, {"id": 5553, "dom": "P", "ent": "moss", "cls": "bridge"}, {"id": 5554, "dom": "P", "ent": "moss", "cls": "bud"}, {"id": 5555, "dom": "P", "ent": "moss", "cls": "cat"}, {"id": 5556, "dom": "P", "ent": "moss", "cls": "chapel"}, {"id": 5557, "dom": "P", "ent": "moss", "cls": "chick"}, {"id": 5558, "dom": "P", "ent": "moss", "cls": "clock"}, {"id": 5559, "dom": "P", "ent": "moss", "cls": "deer"}, {"id": 5560, "dom": "P", "ent": "moss", "cls": "dog"}, {"id": 5561, "dom": "P", "ent": "moss", "cls": "elephant"}, {"id": 5562, "dom": "P", "ent": "moss", "cls": "embassy"}, {"id": 5563, "dom": "P", "ent": "moss", "cls": "garage"}, {"id": 5564, "dom": "P", "ent": "moss", "cls": "highway"}, {"id": 5565, "dom": "P", "ent": "moss", "cls": "kidney"}, {"id": 5566, "dom": "P", "ent": "moss", "cls": "lamb"}, {"id": 5567, "dom": "P", "ent": "moss", "cls": "male"}, {"id": 5568, "dom": "P", "ent": "moss", "cls": "marijuana"}, {"id": 5569, "dom": "P", "ent": "moss", "cls": "puppy"}, {"id": 5570, "dom": "P", "ent": "moss", "cls": "receiver"}, {"id": 5571, "dom": "P", "ent": "moss", "cls": "reed"}, {"id": 5572, "dom": "P", "ent": "moss", "cls": "rifle"}, {"id": 5573, "dom": "P", "ent": "moss", "cls": "road"}, {"id": 5574, "dom": "P", "ent": "moss", "cls": "salmon"}, {"id": 5575, "dom": "P", "ent": "moss", "cls": "shark"}, {"id": 5576, "dom": "P", "ent": "moss", "cls": "sheep"}, {"id": 5577, "dom": "P", "ent": "moss", "cls": "shook"}, {"id": 5578, "dom": "P", "ent": "moss", "cls": "tank"}, {"id": 5579, "dom": "P", "ent": "moss", "cls": "wheat"}, {"id": 5580, "dom": "P", "ent": "moustache", "cls": "architecture"}, {"id": 5581, "dom": "P", "ent": "moustache", "cls": "bacteria"}, {"id": 5582, "dom": "P", "ent": "moustache", "cls": "beef"}, {"id": 5583, "dom": "P", "ent": "moustache", "cls": "bridge"}, {"id": 5584, "dom": "P", "ent": "moustache", "cls": "bud"}, {"id": 5585, "dom": "P", "ent": "moustache", "cls": "cat"}, {"id": 5586, "dom": "P", "ent": "moustache", "cls": "chapel"}, {"id": 5587, "dom": "P", "ent": "moustache", "cls": "chick"}, {"id": 5588, "dom": "P", "ent": "moustache", "cls": "clock"}, {"id": 5589, "dom": "P", "ent": "moustache", "cls": "deer"}, {"id": 5590, "dom": "P", "ent": "moustache", "cls": "dog"}, {"id": 5591, "dom": "P", "ent": "moustache", "cls": "elephant"}, {"id": 5592, "dom": "P", "ent": "moustache", "cls": "embassy"}, {"id": 5593, "dom": "P", "ent": "moustache", "cls": "garage"}, {"id": 5594, "dom": "P", "ent": "moustache", "cls": "highway"}, {"id": 5595, "dom": "P", "ent": "moustache", "cls": "kidney"}, {"id": 5596, "dom": "P", "ent": "moustache", "cls": "lamb"}, {"id": 5597, "dom": "P", "ent": "moustache", "cls": "male"}, {"id": 5598, "dom": "P", "ent": "moustache", "cls": "marijuana"}, {"id": 5599, "dom": "P", "ent": "moustache", "cls": "puppy"}, {"id": 5600, "dom": "P", "ent": "moustache", "cls": "receiver"}, {"id": 5601, "dom": "P", "ent": "moustache", "cls": "reed"}, {"id": 5602, "dom": "P", "ent": "moustache", "cls": "rifle"}, {"id": 5603, "dom": "P", "ent": "moustache", "cls": "road"}, {"id": 5604, "dom": "P", "ent": "moustache", "cls": "salmon"}, {"id": 5605, "dom": "P", "ent": "moustache", "cls": "shark"}, {"id": 5606, "dom": "P", "ent": "moustache", "cls": "sheep"}, {"id": 5607, "dom": "P", "ent": "moustache", "cls": "shook"}, {"id": 5608, "dom": "P", "ent": "moustache", "cls": "tank"}, {"id": 5609, "dom": "P", "ent": "moustache", "cls": "wheat"}, {"id": 5610, "dom": "P", "ent": "movement", "cls": "architecture"}, {"id": 5611, "dom": "P", "ent": "movement", "cls": "bacteria"}, {"id": 5612, "dom": "P", "ent": "movement", "cls": "beef"}, {"id": 5613, "dom": "P", "ent": "movement", "cls": "bridge"}, {"id": 5614, "dom": "P", "ent": "movement", "cls": "bud"}, {"id": 5615, "dom": "P", "ent": "movement", "cls": "cat"}, {"id": 5616, "dom": "P", "ent": "movement", "cls": "chapel"}, {"id": 5617, "dom": "P", "ent": "movement", "cls": "chick"}, {"id": 5618, "dom": "P", "ent": "movement", "cls": "clock"}, {"id": 5619, "dom": "P", "ent": "movement", "cls": "deer"}, {"id": 5620, "dom": "P", "ent": "movement", "cls": "dog"}, {"id": 5621, "dom": "P", "ent": "movement", "cls": "elephant"}, {"id": 5622, "dom": "P", "ent": "movement", "cls": "embassy"}, {"id": 5623, "dom": "P", "ent": "movement", "cls": "garage"}, {"id": 5624, "dom": "P", "ent": "movement", "cls": "highway"}, {"id": 5625, "dom": "P", "ent": "movement", "cls": "kidney"}, {"id": 5626, "dom": "P", "ent": "movement", "cls": "lamb"}, {"id": 5627, "dom": "P", "ent": "movement", "cls": "male"}, {"id": 5628, "dom": "P", "ent": "movement", "cls": "marijuana"}, {"id": 5629, "dom": "P", "ent": "movement", "cls": "puppy"}, {"id": 5630, "dom": "P", "ent": "movement", "cls": "receiver"}, {"id": 5631, "dom": "P", "ent": "movement", "cls": "reed"}, {"id": 5632, "dom": "P", "ent": "movement", "cls": "rifle"}, {"id": 5633, "dom": "P", "ent": "movement", "cls": "road"}, {"id": 5634, "dom": "P", "ent": "movement", "cls": "salmon"}, {"id": 5635, "dom": "P", "ent": "movement", "cls": "shark"}, {"id": 5636, "dom": "P", "ent": "movement", "cls": "sheep"}, {"id": 5637, "dom": "P", "ent": "movement", "cls": "shook"}, {"id": 5638, "dom": "P", "ent": "movement", "cls": "tank"}, {"id": 5639, "dom": "P", "ent": "movement", "cls": "wheat"}, {"id": 5640, "dom": "P", "ent": "mustache", "cls": "architecture"}, {"id": 5641, "dom": "P", "ent": "mustache", "cls": "bacteria"}, {"id": 5642, "dom": "P", "ent": "mustache", "cls": "beef"}, {"id": 5643, "dom": "P", "ent": "mustache", "cls": "bridge"}, {"id": 5644, "dom": "P", "ent": "mustache", "cls": "bud"}, {"id": 5645, "dom": "P", "ent": "mustache", "cls": "cat"}, {"id": 5646, "dom": "P", "ent": "mustache", "cls": "chapel"}, {"id": 5647, "dom": "P", "ent": "mustache", "cls": "chick"}, {"id": 5648, "dom": "P", "ent": "mustache", "cls": "clock"}, {"id": 5649, "dom": "P", "ent": "mustache", "cls": "deer"}, {"id": 5650, "dom": "P", "ent": "mustache", "cls": "dog"}, {"id": 5651, "dom": "P", "ent": "mustache", "cls": "elephant"}, {"id": 5652, "dom": "P", "ent": "mustache", "cls": "embassy"}, {"id": 5653, "dom": "P", "ent": "mustache", "cls": "garage"}, {"id": 5654, "dom": "P", "ent": "mustache", "cls": "highway"}, {"id": 5655, "dom": "P", "ent": "mustache", "cls": "kidney"}, {"id": 5656, "dom": "P", "ent": "mustache", "cls": "lamb"}, {"id": 5657, "dom": "P", "ent": "mustache", "cls": "male"}, {"id": 5658, "dom": "P", "ent": "mustache", "cls": "marijuana"}, {"id": 5659, "dom": "P", "ent": "mustache", "cls": "puppy"}, {"id": 5660, "dom": "P", "ent": "mustache", "cls": "receiver"}, {"id": 5661, "dom": "P", "ent": "mustache", "cls": "reed"}, {"id": 5662, "dom": "P", "ent": "mustache", "cls": "rifle"}, {"id": 5663, "dom": "P", "ent": "mustache", "cls": "road"}, {"id": 5664, "dom": "P", "ent": "mustache", "cls": "salmon"}, {"id": 5665, "dom": "P", "ent": "mustache", "cls": "shark"}, {"id": 5666, "dom": "P", "ent": "mustache", "cls": "sheep"}, {"id": 5667, "dom": "P", "ent": "mustache", "cls": "shook"}, {"id": 5668, "dom": "P", "ent": "mustache", "cls": "tank"}, {"id": 5669, "dom": "P", "ent": "mustache", "cls": "wheat"}, {"id": 5670, "dom": "P", "ent": "muzzle", "cls": "architecture"}, {"id": 5671, "dom": "P", "ent": "muzzle", "cls": "bacteria"}, {"id": 5672, "dom": "P", "ent": "muzzle", "cls": "beef"}, {"id": 5673, "dom": "P", "ent": "muzzle", "cls": "bridge"}, {"id": 5674, "dom": "P", "ent": "muzzle", "cls": "bud"}, {"id": 5675, "dom": "P", "ent": "muzzle", "cls": "cat"}, {"id": 5676, "dom": "P", "ent": "muzzle", "cls": "chapel"}, {"id": 5677, "dom": "P", "ent": "muzzle", "cls": "chick"}, {"id": 5678, "dom": "P", "ent": "muzzle", "cls": "clock"}, {"id": 5679, "dom": "P", "ent": "muzzle", "cls": "deer"}, {"id": 5680, "dom": "P", "ent": "muzzle", "cls": "dog"}, {"id": 5681, "dom": "P", "ent": "muzzle", "cls": "elephant"}, {"id": 5682, "dom": "P", "ent": "muzzle", "cls": "embassy"}, {"id": 5683, "dom": "P", "ent": "muzzle", "cls": "garage"}, {"id": 5684, "dom": "P", "ent": "muzzle", "cls": "highway"}, {"id": 5685, "dom": "P", "ent": "muzzle", "cls": "kidney"}, {"id": 5686, "dom": "P", "ent": "muzzle", "cls": "lamb"}, {"id": 5687, "dom": "P", "ent": "muzzle", "cls": "male"}, {"id": 5688, "dom": "P", "ent": "muzzle", "cls": "marijuana"}, {"id": 5689, "dom": "P", "ent": "muzzle", "cls": "puppy"}, {"id": 5690, "dom": "P", "ent": "muzzle", "cls": "receiver"}, {"id": 5691, "dom": "P", "ent": "muzzle", "cls": "reed"}, {"id": 5692, "dom": "P", "ent": "muzzle", "cls": "rifle"}, {"id": 5693, "dom": "P", "ent": "muzzle", "cls": "road"}, {"id": 5694, "dom": "P", "ent": "muzzle", "cls": "salmon"}, {"id": 5695, "dom": "P", "ent": "muzzle", "cls": "shark"}, {"id": 5696, "dom": "P", "ent": "muzzle", "cls": "sheep"}, {"id": 5697, "dom": "P", "ent": "muzzle", "cls": "shook"}, {"id": 5698, "dom": "P", "ent": "muzzle", "cls": "tank"}, {"id": 5699, "dom": "P", "ent": "muzzle", "cls": "wheat"}, {"id": 5700, "dom": "P", "ent": "nail", "cls": "architecture"}, {"id": 5701, "dom": "P", "ent": "nail", "cls": "bacteria"}, {"id": 5702, "dom": "P", "ent": "nail", "cls": "beef"}, {"id": 5703, "dom": "P", "ent": "nail", "cls": "bridge"}, {"id": 5704, "dom": "P", "ent": "nail", "cls": "bud"}, {"id": 5705, "dom": "P", "ent": "nail", "cls": "cat"}, {"id": 5706, "dom": "P", "ent": "nail", "cls": "chapel"}, {"id": 5707, "dom": "P", "ent": "nail", "cls": "chick"}, {"id": 5708, "dom": "P", "ent": "nail", "cls": "clock"}, {"id": 5709, "dom": "P", "ent": "nail", "cls": "deer"}, {"id": 5710, "dom": "P", "ent": "nail", "cls": "dog"}, {"id": 5711, "dom": "P", "ent": "nail", "cls": "elephant"}, {"id": 5712, "dom": "P", "ent": "nail", "cls": "embassy"}, {"id": 5713, "dom": "P", "ent": "nail", "cls": "garage"}, {"id": 5714, "dom": "P", "ent": "nail", "cls": "highway"}, {"id": 5715, "dom": "P", "ent": "nail", "cls": "kidney"}, {"id": 5716, "dom": "P", "ent": "nail", "cls": "lamb"}, {"id": 5717, "dom": "P", "ent": "nail", "cls": "male"}, {"id": 5718, "dom": "P", "ent": "nail", "cls": "marijuana"}, {"id": 5719, "dom": "P", "ent": "nail", "cls": "puppy"}, {"id": 5720, "dom": "P", "ent": "nail", "cls": "receiver"}, {"id": 5721, "dom": "P", "ent": "nail", "cls": "reed"}, {"id": 5722, "dom": "P", "ent": "nail", "cls": "rifle"}, {"id": 5723, "dom": "P", "ent": "nail", "cls": "road"}, {"id": 5724, "dom": "P", "ent": "nail", "cls": "salmon"}, {"id": 5725, "dom": "P", "ent": "nail", "cls": "shark"}, {"id": 5726, "dom": "P", "ent": "nail", "cls": "sheep"}, {"id": 5727, "dom": "P", "ent": "nail", "cls": "shook"}, {"id": 5728, "dom": "P", "ent": "nail", "cls": "tank"}, {"id": 5729, "dom": "P", "ent": "nail", "cls": "wheat"}, {"id": 5730, "dom": "P", "ent": "nasal", "cls": "architecture"}, {"id": 5731, "dom": "P", "ent": "nasal", "cls": "bacteria"}, {"id": 5732, "dom": "P", "ent": "nasal", "cls": "beef"}, {"id": 5733, "dom": "P", "ent": "nasal", "cls": "bridge"}, {"id": 5734, "dom": "P", "ent": "nasal", "cls": "bud"}, {"id": 5735, "dom": "P", "ent": "nasal", "cls": "cat"}, {"id": 5736, "dom": "P", "ent": "nasal", "cls": "chapel"}, {"id": 5737, "dom": "P", "ent": "nasal", "cls": "chick"}, {"id": 5738, "dom": "P", "ent": "nasal", "cls": "clock"}, {"id": 5739, "dom": "P", "ent": "nasal", "cls": "deer"}, {"id": 5740, "dom": "P", "ent": "nasal", "cls": "dog"}, {"id": 5741, "dom": "P", "ent": "nasal", "cls": "elephant"}, {"id": 5742, "dom": "P", "ent": "nasal", "cls": "embassy"}, {"id": 5743, "dom": "P", "ent": "nasal", "cls": "garage"}, {"id": 5744, "dom": "P", "ent": "nasal", "cls": "highway"}, {"id": 5745, "dom": "P", "ent": "nasal", "cls": "kidney"}, {"id": 5746, "dom": "P", "ent": "nasal", "cls": "lamb"}, {"id": 5747, "dom": "P", "ent": "nasal", "cls": "male"}, {"id": 5748, "dom": "P", "ent": "nasal", "cls": "marijuana"}, {"id": 5749, "dom": "P", "ent": "nasal", "cls": "puppy"}, {"id": 5750, "dom": "P", "ent": "nasal", "cls": "receiver"}, {"id": 5751, "dom": "P", "ent": "nasal", "cls": "reed"}, {"id": 5752, "dom": "P", "ent": "nasal", "cls": "rifle"}, {"id": 5753, "dom": "P", "ent": "nasal", "cls": "road"}, {"id": 5754, "dom": "P", "ent": "nasal", "cls": "salmon"}, {"id": 5755, "dom": "P", "ent": "nasal", "cls": "shark"}, {"id": 5756, "dom": "P", "ent": "nasal", "cls": "sheep"}, {"id": 5757, "dom": "P", "ent": "nasal", "cls": "shook"}, {"id": 5758, "dom": "P", "ent": "nasal", "cls": "tank"}, {"id": 5759, "dom": "P", "ent": "nasal", "cls": "wheat"}, {"id": 5760, "dom": "P", "ent": "nib", "cls": "architecture"}, {"id": 5761, "dom": "P", "ent": "nib", "cls": "bacteria"}, {"id": 5762, "dom": "P", "ent": "nib", "cls": "beef"}, {"id": 5763, "dom": "P", "ent": "nib", "cls": "bridge"}, {"id": 5764, "dom": "P", "ent": "nib", "cls": "bud"}, {"id": 5765, "dom": "P", "ent": "nib", "cls": "cat"}, {"id": 5766, "dom": "P", "ent": "nib", "cls": "chapel"}, {"id": 5767, "dom": "P", "ent": "nib", "cls": "chick"}, {"id": 5768, "dom": "P", "ent": "nib", "cls": "clock"}, {"id": 5769, "dom": "P", "ent": "nib", "cls": "deer"}, {"id": 5770, "dom": "P", "ent": "nib", "cls": "dog"}, {"id": 5771, "dom": "P", "ent": "nib", "cls": "elephant"}, {"id": 5772, "dom": "P", "ent": "nib", "cls": "embassy"}, {"id": 5773, "dom": "P", "ent": "nib", "cls": "garage"}, {"id": 5774, "dom": "P", "ent": "nib", "cls": "highway"}, {"id": 5775, "dom": "P", "ent": "nib", "cls": "kidney"}, {"id": 5776, "dom": "P", "ent": "nib", "cls": "lamb"}, {"id": 5777, "dom": "P", "ent": "nib", "cls": "male"}, {"id": 5778, "dom": "P", "ent": "nib", "cls": "marijuana"}, {"id": 5779, "dom": "P", "ent": "nib", "cls": "puppy"}, {"id": 5780, "dom": "P", "ent": "nib", "cls": "receiver"}, {"id": 5781, "dom": "P", "ent": "nib", "cls": "reed"}, {"id": 5782, "dom": "P", "ent": "nib", "cls": "rifle"}, {"id": 5783, "dom": "P", "ent": "nib", "cls": "road"}, {"id": 5784, "dom": "P", "ent": "nib", "cls": "salmon"}, {"id": 5785, "dom": "P", "ent": "nib", "cls": "shark"}, {"id": 5786, "dom": "P", "ent": "nib", "cls": "sheep"}, {"id": 5787, "dom": "P", "ent": "nib", "cls": "shook"}, {"id": 5788, "dom": "P", "ent": "nib", "cls": "tank"}, {"id": 5789, "dom": "P", "ent": "nib", "cls": "wheat"}, {"id": 5790, "dom": "P", "ent": "nook", "cls": "architecture"}, {"id": 5791, "dom": "P", "ent": "nook", "cls": "bacteria"}, {"id": 5792, "dom": "P", "ent": "nook", "cls": "beef"}, {"id": 5793, "dom": "P", "ent": "nook", "cls": "bridge"}, {"id": 5794, "dom": "P", "ent": "nook", "cls": "bud"}, {"id": 5795, "dom": "P", "ent": "nook", "cls": "cat"}, {"id": 5796, "dom": "P", "ent": "nook", "cls": "chapel"}, {"id": 5797, "dom": "P", "ent": "nook", "cls": "chick"}, {"id": 5798, "dom": "P", "ent": "nook", "cls": "clock"}, {"id": 5799, "dom": "P", "ent": "nook", "cls": "deer"}, {"id": 5800, "dom": "P", "ent": "nook", "cls": "dog"}, {"id": 5801, "dom": "P", "ent": "nook", "cls": "elephant"}, {"id": 5802, "dom": "P", "ent": "nook", "cls": "embassy"}, {"id": 5803, "dom": "P", "ent": "nook", "cls": "garage"}, {"id": 5804, "dom": "P", "ent": "nook", "cls": "highway"}, {"id": 5805, "dom": "P", "ent": "nook", "cls": "kidney"}, {"id": 5806, "dom": "P", "ent": "nook", "cls": "lamb"}, {"id": 5807, "dom": "P", "ent": "nook", "cls": "male"}, {"id": 5808, "dom": "P", "ent": "nook", "cls": "marijuana"}, {"id": 5809, "dom": "P", "ent": "nook", "cls": "puppy"}, {"id": 5810, "dom": "P", "ent": "nook", "cls": "receiver"}, {"id": 5811, "dom": "P", "ent": "nook", "cls": "reed"}, {"id": 5812, "dom": "P", "ent": "nook", "cls": "rifle"}, {"id": 5813, "dom": "P", "ent": "nook", "cls": "road"}, {"id": 5814, "dom": "P", "ent": "nook", "cls": "salmon"}, {"id": 5815, "dom": "P", "ent": "nook", "cls": "shark"}, {"id": 5816, "dom": "P", "ent": "nook", "cls": "sheep"}, {"id": 5817, "dom": "P", "ent": "nook", "cls": "shook"}, {"id": 5818, "dom": "P", "ent": "nook", "cls": "tank"}, {"id": 5819, "dom": "P", "ent": "nook", "cls": "wheat"}, {"id": 5820, "dom": "P", "ent": "nose", "cls": "architecture"}, {"id": 5821, "dom": "P", "ent": "nose", "cls": "bacteria"}, {"id": 5822, "dom": "P", "ent": "nose", "cls": "beef"}, {"id": 5823, "dom": "P", "ent": "nose", "cls": "bridge"}, {"id": 5824, "dom": "P", "ent": "nose", "cls": "bud"}, {"id": 5825, "dom": "P", "ent": "nose", "cls": "cat"}, {"id": 5826, "dom": "P", "ent": "nose", "cls": "chapel"}, {"id": 5827, "dom": "P", "ent": "nose", "cls": "chick"}, {"id": 5828, "dom": "P", "ent": "nose", "cls": "clock"}, {"id": 5829, "dom": "P", "ent": "nose", "cls": "deer"}, {"id": 5830, "dom": "P", "ent": "nose", "cls": "dog"}, {"id": 5831, "dom": "P", "ent": "nose", "cls": "elephant"}, {"id": 5832, "dom": "P", "ent": "nose", "cls": "embassy"}, {"id": 5833, "dom": "P", "ent": "nose", "cls": "garage"}, {"id": 5834, "dom": "P", "ent": "nose", "cls": "highway"}, {"id": 5835, "dom": "P", "ent": "nose", "cls": "kidney"}, {"id": 5836, "dom": "P", "ent": "nose", "cls": "lamb"}, {"id": 5837, "dom": "P", "ent": "nose", "cls": "male"}, {"id": 5838, "dom": "P", "ent": "nose", "cls": "marijuana"}, {"id": 5839, "dom": "P", "ent": "nose", "cls": "puppy"}, {"id": 5840, "dom": "P", "ent": "nose", "cls": "receiver"}, {"id": 5841, "dom": "P", "ent": "nose", "cls": "reed"}, {"id": 5842, "dom": "P", "ent": "nose", "cls": "rifle"}, {"id": 5843, "dom": "P", "ent": "nose", "cls": "road"}, {"id": 5844, "dom": "P", "ent": "nose", "cls": "salmon"}, {"id": 5845, "dom": "P", "ent": "nose", "cls": "shark"}, {"id": 5846, "dom": "P", "ent": "nose", "cls": "sheep"}, {"id": 5847, "dom": "P", "ent": "nose", "cls": "shook"}, {"id": 5848, "dom": "P", "ent": "nose", "cls": "tank"}, {"id": 5849, "dom": "P", "ent": "nose", "cls": "wheat"}, {"id": 5850, "dom": "P", "ent": "nucleus", "cls": "architecture"}, {"id": 5851, "dom": "P", "ent": "nucleus", "cls": "bacteria"}, {"id": 5852, "dom": "P", "ent": "nucleus", "cls": "beef"}, {"id": 5853, "dom": "P", "ent": "nucleus", "cls": "bridge"}, {"id": 5854, "dom": "P", "ent": "nucleus", "cls": "bud"}, {"id": 5855, "dom": "P", "ent": "nucleus", "cls": "cat"}, {"id": 5856, "dom": "P", "ent": "nucleus", "cls": "chapel"}, {"id": 5857, "dom": "P", "ent": "nucleus", "cls": "chick"}, {"id": 5858, "dom": "P", "ent": "nucleus", "cls": "clock"}, {"id": 5859, "dom": "P", "ent": "nucleus", "cls": "deer"}, {"id": 5860, "dom": "P", "ent": "nucleus", "cls": "dog"}, {"id": 5861, "dom": "P", "ent": "nucleus", "cls": "elephant"}, {"id": 5862, "dom": "P", "ent": "nucleus", "cls": "embassy"}, {"id": 5863, "dom": "P", "ent": "nucleus", "cls": "garage"}, {"id": 5864, "dom": "P", "ent": "nucleus", "cls": "highway"}, {"id": 5865, "dom": "P", "ent": "nucleus", "cls": "kidney"}, {"id": 5866, "dom": "P", "ent": "nucleus", "cls": "lamb"}, {"id": 5867, "dom": "P", "ent": "nucleus", "cls": "male"}, {"id": 5868, "dom": "P", "ent": "nucleus", "cls": "marijuana"}, {"id": 5869, "dom": "P", "ent": "nucleus", "cls": "puppy"}, {"id": 5870, "dom": "P", "ent": "nucleus", "cls": "receiver"}, {"id": 5871, "dom": "P", "ent": "nucleus", "cls": "reed"}, {"id": 5872, "dom": "P", "ent": "nucleus", "cls": "rifle"}, {"id": 5873, "dom": "P", "ent": "nucleus", "cls": "road"}, {"id": 5874, "dom": "P", "ent": "nucleus", "cls": "salmon"}, {"id": 5875, "dom": "P", "ent": "nucleus", "cls": "shark"}, {"id": 5876, "dom": "P", "ent": "nucleus", "cls": "sheep"}, {"id": 5877, "dom": "P", "ent": "nucleus", "cls": "shook"}, {"id": 5878, "dom": "P", "ent": "nucleus", "cls": "tank"}, {"id": 5879, "dom": "P", "ent": "nucleus", "cls": "wheat"}, {"id": 5880, "dom": "P", "ent": "oculus", "cls": "architecture"}, {"id": 5881, "dom": "P", "ent": "oculus", "cls": "bacteria"}, {"id": 5882, "dom": "P", "ent": "oculus", "cls": "beef"}, {"id": 5883, "dom": "P", "ent": "oculus", "cls": "bridge"}, {"id": 5884, "dom": "P", "ent": "oculus", "cls": "bud"}, {"id": 5885, "dom": "P", "ent": "oculus", "cls": "cat"}, {"id": 5886, "dom": "P", "ent": "oculus", "cls": "chapel"}, {"id": 5887, "dom": "P", "ent": "oculus", "cls": "chick"}, {"id": 5888, "dom": "P", "ent": "oculus", "cls": "clock"}, {"id": 5889, "dom": "P", "ent": "oculus", "cls": "deer"}, {"id": 5890, "dom": "P", "ent": "oculus", "cls": "dog"}, {"id": 5891, "dom": "P", "ent": "oculus", "cls": "elephant"}, {"id": 5892, "dom": "P", "ent": "oculus", "cls": "embassy"}, {"id": 5893, "dom": "P", "ent": "oculus", "cls": "garage"}, {"id": 5894, "dom": "P", "ent": "oculus", "cls": "highway"}, {"id": 5895, "dom": "P", "ent": "oculus", "cls": "kidney"}, {"id": 5896, "dom": "P", "ent": "oculus", "cls": "lamb"}, {"id": 5897, "dom": "P", "ent": "oculus", "cls": "male"}, {"id": 5898, "dom": "P", "ent": "oculus", "cls": "marijuana"}, {"id": 5899, "dom": "P", "ent": "oculus", "cls": "puppy"}, {"id": 5900, "dom": "P", "ent": "oculus", "cls": "receiver"}, {"id": 5901, "dom": "P", "ent": "oculus", "cls": "reed"}, {"id": 5902, "dom": "P", "ent": "oculus", "cls": "rifle"}, {"id": 5903, "dom": "P", "ent": "oculus", "cls": "road"}, {"id": 5904, "dom": "P", "ent": "oculus", "cls": "salmon"}, {"id": 5905, "dom": "P", "ent": "oculus", "cls": "shark"}, {"id": 5906, "dom": "P", "ent": "oculus", "cls": "sheep"}, {"id": 5907, "dom": "P", "ent": "oculus", "cls": "shook"}, {"id": 5908, "dom": "P", "ent": "oculus", "cls": "tank"}, {"id": 5909, "dom": "P", "ent": "oculus", "cls": "wheat"}, {"id": 5910, "dom": "P", "ent": "optic", "cls": "architecture"}, {"id": 5911, "dom": "P", "ent": "optic", "cls": "bacteria"}, {"id": 5912, "dom": "P", "ent": "optic", "cls": "beef"}, {"id": 5913, "dom": "P", "ent": "optic", "cls": "bridge"}, {"id": 5914, "dom": "P", "ent": "optic", "cls": "bud"}, {"id": 5915, "dom": "P", "ent": "optic", "cls": "cat"}, {"id": 5916, "dom": "P", "ent": "optic", "cls": "chapel"}, {"id": 5917, "dom": "P", "ent": "optic", "cls": "chick"}, {"id": 5918, "dom": "P", "ent": "optic", "cls": "clock"}, {"id": 5919, "dom": "P", "ent": "optic", "cls": "deer"}, {"id": 5920, "dom": "P", "ent": "optic", "cls": "dog"}, {"id": 5921, "dom": "P", "ent": "optic", "cls": "elephant"}, {"id": 5922, "dom": "P", "ent": "optic", "cls": "embassy"}, {"id": 5923, "dom": "P", "ent": "optic", "cls": "garage"}, {"id": 5924, "dom": "P", "ent": "optic", "cls": "highway"}, {"id": 5925, "dom": "P", "ent": "optic", "cls": "kidney"}, {"id": 5926, "dom": "P", "ent": "optic", "cls": "lamb"}, {"id": 5927, "dom": "P", "ent": "optic", "cls": "male"}, {"id": 5928, "dom": "P", "ent": "optic", "cls": "marijuana"}, {"id": 5929, "dom": "P", "ent": "optic", "cls": "puppy"}, {"id": 5930, "dom": "P", "ent": "optic", "cls": "receiver"}, {"id": 5931, "dom": "P", "ent": "optic", "cls": "reed"}, {"id": 5932, "dom": "P", "ent": "optic", "cls": "rifle"}, {"id": 5933, "dom": "P", "ent": "optic", "cls": "road"}, {"id": 5934, "dom": "P", "ent": "optic", "cls": "salmon"}, {"id": 5935, "dom": "P", "ent": "optic", "cls": "shark"}, {"id": 5936, "dom": "P", "ent": "optic", "cls": "sheep"}, {"id": 5937, "dom": "P", "ent": "optic", "cls": "shook"}, {"id": 5938, "dom": "P", "ent": "optic", "cls": "tank"}, {"id": 5939, "dom": "P", "ent": "optic", "cls": "wheat"}, {"id": 5940, "dom": "P", "ent": "orb", "cls": "architecture"}, {"id": 5941, "dom": "P", "ent": "orb", "cls": "bacteria"}, {"id": 5942, "dom": "P", "ent": "orb", "cls": "beef"}, {"id": 5943, "dom": "P", "ent": "orb", "cls": "bridge"}, {"id": 5944, "dom": "P", "ent": "orb", "cls": "bud"}, {"id": 5945, "dom": "P", "ent": "orb", "cls": "cat"}, {"id": 5946, "dom": "P", "ent": "orb", "cls": "chapel"}, {"id": 5947, "dom": "P", "ent": "orb", "cls": "chick"}, {"id": 5948, "dom": "P", "ent": "orb", "cls": "clock"}, {"id": 5949, "dom": "P", "ent": "orb", "cls": "deer"}, {"id": 5950, "dom": "P", "ent": "orb", "cls": "dog"}, {"id": 5951, "dom": "P", "ent": "orb", "cls": "elephant"}, {"id": 5952, "dom": "P", "ent": "orb", "cls": "embassy"}, {"id": 5953, "dom": "P", "ent": "orb", "cls": "garage"}, {"id": 5954, "dom": "P", "ent": "orb", "cls": "highway"}, {"id": 5955, "dom": "P", "ent": "orb", "cls": "kidney"}, {"id": 5956, "dom": "P", "ent": "orb", "cls": "lamb"}, {"id": 5957, "dom": "P", "ent": "orb", "cls": "male"}, {"id": 5958, "dom": "P", "ent": "orb", "cls": "marijuana"}, {"id": 5959, "dom": "P", "ent": "orb", "cls": "puppy"}, {"id": 5960, "dom": "P", "ent": "orb", "cls": "receiver"}, {"id": 5961, "dom": "P", "ent": "orb", "cls": "reed"}, {"id": 5962, "dom": "P", "ent": "orb", "cls": "rifle"}, {"id": 5963, "dom": "P", "ent": "orb", "cls": "road"}, {"id": 5964, "dom": "P", "ent": "orb", "cls": "salmon"}, {"id": 5965, "dom": "P", "ent": "orb", "cls": "shark"}, {"id": 5966, "dom": "P", "ent": "orb", "cls": "sheep"}, {"id": 5967, "dom": "P", "ent": "orb", "cls": "shook"}, {"id": 5968, "dom": "P", "ent": "orb", "cls": "tank"}, {"id": 5969, "dom": "P", "ent": "orb", "cls": "wheat"}, {"id": 5970, "dom": "P", "ent": "orbit", "cls": "architecture"}, {"id": 5971, "dom": "P", "ent": "orbit", "cls": "bacteria"}, {"id": 5972, "dom": "P", "ent": "orbit", "cls": "beef"}, {"id": 5973, "dom": "P", "ent": "orbit", "cls": "bridge"}, {"id": 5974, "dom": "P", "ent": "orbit", "cls": "bud"}, {"id": 5975, "dom": "P", "ent": "orbit", "cls": "cat"}, {"id": 5976, "dom": "P", "ent": "orbit", "cls": "chapel"}, {"id": 5977, "dom": "P", "ent": "orbit", "cls": "chick"}, {"id": 5978, "dom": "P", "ent": "orbit", "cls": "clock"}, {"id": 5979, "dom": "P", "ent": "orbit", "cls": "deer"}, {"id": 5980, "dom": "P", "ent": "orbit", "cls": "dog"}, {"id": 5981, "dom": "P", "ent": "orbit", "cls": "elephant"}, {"id": 5982, "dom": "P", "ent": "orbit", "cls": "embassy"}, {"id": 5983, "dom": "P", "ent": "orbit", "cls": "garage"}, {"id": 5984, "dom": "P", "ent": "orbit", "cls": "highway"}, {"id": 5985, "dom": "P", "ent": "orbit", "cls": "kidney"}, {"id": 5986, "dom": "P", "ent": "orbit", "cls": "lamb"}, {"id": 5987, "dom": "P", "ent": "orbit", "cls": "male"}, {"id": 5988, "dom": "P", "ent": "orbit", "cls": "marijuana"}, {"id": 5989, "dom": "P", "ent": "orbit", "cls": "puppy"}, {"id": 5990, "dom": "P", "ent": "orbit", "cls": "receiver"}, {"id": 5991, "dom": "P", "ent": "orbit", "cls": "reed"}, {"id": 5992, "dom": "P", "ent": "orbit", "cls": "rifle"}, {"id": 5993, "dom": "P", "ent": "orbit", "cls": "road"}, {"id": 5994, "dom": "P", "ent": "orbit", "cls": "salmon"}, {"id": 5995, "dom": "P", "ent": "orbit", "cls": "shark"}, {"id": 5996, "dom": "P", "ent": "orbit", "cls": "sheep"}, {"id": 5997, "dom": "P", "ent": "orbit", "cls": "shook"}, {"id": 5998, "dom": "P", "ent": "orbit", "cls": "tank"}, {"id": 5999, "dom": "P", "ent": "orbit", "cls": "wheat"}, {"id": 6000, "dom": "P", "ent": "pad", "cls": "architecture"}, {"id": 6001, "dom": "P", "ent": "pad", "cls": "bacteria"}, {"id": 6002, "dom": "P", "ent": "pad", "cls": "beef"}, {"id": 6003, "dom": "P", "ent": "pad", "cls": "bridge"}, {"id": 6004, "dom": "P", "ent": "pad", "cls": "bud"}, {"id": 6005, "dom": "P", "ent": "pad", "cls": "cat"}, {"id": 6006, "dom": "P", "ent": "pad", "cls": "chapel"}, {"id": 6007, "dom": "P", "ent": "pad", "cls": "chick"}, {"id": 6008, "dom": "P", "ent": "pad", "cls": "clock"}, {"id": 6009, "dom": "P", "ent": "pad", "cls": "deer"}, {"id": 6010, "dom": "P", "ent": "pad", "cls": "dog"}, {"id": 6011, "dom": "P", "ent": "pad", "cls": "elephant"}, {"id": 6012, "dom": "P", "ent": "pad", "cls": "embassy"}, {"id": 6013, "dom": "P", "ent": "pad", "cls": "garage"}, {"id": 6014, "dom": "P", "ent": "pad", "cls": "highway"}, {"id": 6015, "dom": "P", "ent": "pad", "cls": "kidney"}, {"id": 6016, "dom": "P", "ent": "pad", "cls": "lamb"}, {"id": 6017, "dom": "P", "ent": "pad", "cls": "male"}, {"id": 6018, "dom": "P", "ent": "pad", "cls": "marijuana"}, {"id": 6019, "dom": "P", "ent": "pad", "cls": "puppy"}, {"id": 6020, "dom": "P", "ent": "pad", "cls": "receiver"}, {"id": 6021, "dom": "P", "ent": "pad", "cls": "reed"}, {"id": 6022, "dom": "P", "ent": "pad", "cls": "rifle"}, {"id": 6023, "dom": "P", "ent": "pad", "cls": "road"}, {"id": 6024, "dom": "P", "ent": "pad", "cls": "salmon"}, {"id": 6025, "dom": "P", "ent": "pad", "cls": "shark"}, {"id": 6026, "dom": "P", "ent": "pad", "cls": "sheep"}, {"id": 6027, "dom": "P", "ent": "pad", "cls": "shook"}, {"id": 6028, "dom": "P", "ent": "pad", "cls": "tank"}, {"id": 6029, "dom": "P", "ent": "pad", "cls": "wheat"}, {"id": 6030, "dom": "P", "ent": "palate", "cls": "architecture"}, {"id": 6031, "dom": "P", "ent": "palate", "cls": "bacteria"}, {"id": 6032, "dom": "P", "ent": "palate", "cls": "beef"}, {"id": 6033, "dom": "P", "ent": "palate", "cls": "bridge"}, {"id": 6034, "dom": "P", "ent": "palate", "cls": "bud"}, {"id": 6035, "dom": "P", "ent": "palate", "cls": "cat"}, {"id": 6036, "dom": "P", "ent": "palate", "cls": "chapel"}, {"id": 6037, "dom": "P", "ent": "palate", "cls": "chick"}, {"id": 6038, "dom": "P", "ent": "palate", "cls": "clock"}, {"id": 6039, "dom": "P", "ent": "palate", "cls": "deer"}, {"id": 6040, "dom": "P", "ent": "palate", "cls": "dog"}, {"id": 6041, "dom": "P", "ent": "palate", "cls": "elephant"}, {"id": 6042, "dom": "P", "ent": "palate", "cls": "embassy"}, {"id": 6043, "dom": "P", "ent": "palate", "cls": "garage"}, {"id": 6044, "dom": "P", "ent": "palate", "cls": "highway"}, {"id": 6045, "dom": "P", "ent": "palate", "cls": "kidney"}, {"id": 6046, "dom": "P", "ent": "palate", "cls": "lamb"}, {"id": 6047, "dom": "P", "ent": "palate", "cls": "male"}, {"id": 6048, "dom": "P", "ent": "palate", "cls": "marijuana"}, {"id": 6049, "dom": "P", "ent": "palate", "cls": "puppy"}, {"id": 6050, "dom": "P", "ent": "palate", "cls": "receiver"}, {"id": 6051, "dom": "P", "ent": "palate", "cls": "reed"}, {"id": 6052, "dom": "P", "ent": "palate", "cls": "rifle"}, {"id": 6053, "dom": "P", "ent": "palate", "cls": "road"}, {"id": 6054, "dom": "P", "ent": "palate", "cls": "salmon"}, {"id": 6055, "dom": "P", "ent": "palate", "cls": "shark"}, {"id": 6056, "dom": "P", "ent": "palate", "cls": "sheep"}, {"id": 6057, "dom": "P", "ent": "palate", "cls": "shook"}, {"id": 6058, "dom": "P", "ent": "palate", "cls": "tank"}, {"id": 6059, "dom": "P", "ent": "palate", "cls": "wheat"}, {"id": 6060, "dom": "P", "ent": "pale", "cls": "architecture"}, {"id": 6061, "dom": "P", "ent": "pale", "cls": "bacteria"}, {"id": 6062, "dom": "P", "ent": "pale", "cls": "beef"}, {"id": 6063, "dom": "P", "ent": "pale", "cls": "bridge"}, {"id": 6064, "dom": "P", "ent": "pale", "cls": "bud"}, {"id": 6065, "dom": "P", "ent": "pale", "cls": "cat"}, {"id": 6066, "dom": "P", "ent": "pale", "cls": "chapel"}, {"id": 6067, "dom": "P", "ent": "pale", "cls": "chick"}, {"id": 6068, "dom": "P", "ent": "pale", "cls": "clock"}, {"id": 6069, "dom": "P", "ent": "pale", "cls": "deer"}, {"id": 6070, "dom": "P", "ent": "pale", "cls": "dog"}, {"id": 6071, "dom": "P", "ent": "pale", "cls": "elephant"}, {"id": 6072, "dom": "P", "ent": "pale", "cls": "embassy"}, {"id": 6073, "dom": "P", "ent": "pale", "cls": "garage"}, {"id": 6074, "dom": "P", "ent": "pale", "cls": "highway"}, {"id": 6075, "dom": "P", "ent": "pale", "cls": "kidney"}, {"id": 6076, "dom": "P", "ent": "pale", "cls": "lamb"}, {"id": 6077, "dom": "P", "ent": "pale", "cls": "male"}, {"id": 6078, "dom": "P", "ent": "pale", "cls": "marijuana"}, {"id": 6079, "dom": "P", "ent": "pale", "cls": "puppy"}, {"id": 6080, "dom": "P", "ent": "pale", "cls": "receiver"}, {"id": 6081, "dom": "P", "ent": "pale", "cls": "reed"}, {"id": 6082, "dom": "P", "ent": "pale", "cls": "rifle"}, {"id": 6083, "dom": "P", "ent": "pale", "cls": "road"}, {"id": 6084, "dom": "P", "ent": "pale", "cls": "salmon"}, {"id": 6085, "dom": "P", "ent": "pale", "cls": "shark"}, {"id": 6086, "dom": "P", "ent": "pale", "cls": "sheep"}, {"id": 6087, "dom": "P", "ent": "pale", "cls": "shook"}, {"id": 6088, "dom": "P", "ent": "pale", "cls": "tank"}, {"id": 6089, "dom": "P", "ent": "pale", "cls": "wheat"}, {"id": 6090, "dom": "P", "ent": "pane", "cls": "architecture"}, {"id": 6091, "dom": "P", "ent": "pane", "cls": "bacteria"}, {"id": 6092, "dom": "P", "ent": "pane", "cls": "beef"}, {"id": 6093, "dom": "P", "ent": "pane", "cls": "bridge"}, {"id": 6094, "dom": "P", "ent": "pane", "cls": "bud"}, {"id": 6095, "dom": "P", "ent": "pane", "cls": "cat"}, {"id": 6096, "dom": "P", "ent": "pane", "cls": "chapel"}, {"id": 6097, "dom": "P", "ent": "pane", "cls": "chick"}, {"id": 6098, "dom": "P", "ent": "pane", "cls": "clock"}, {"id": 6099, "dom": "P", "ent": "pane", "cls": "deer"}, {"id": 6100, "dom": "P", "ent": "pane", "cls": "dog"}, {"id": 6101, "dom": "P", "ent": "pane", "cls": "elephant"}, {"id": 6102, "dom": "P", "ent": "pane", "cls": "embassy"}, {"id": 6103, "dom": "P", "ent": "pane", "cls": "garage"}, {"id": 6104, "dom": "P", "ent": "pane", "cls": "highway"}, {"id": 6105, "dom": "P", "ent": "pane", "cls": "kidney"}, {"id": 6106, "dom": "P", "ent": "pane", "cls": "lamb"}, {"id": 6107, "dom": "P", "ent": "pane", "cls": "male"}, {"id": 6108, "dom": "P", "ent": "pane", "cls": "marijuana"}, {"id": 6109, "dom": "P", "ent": "pane", "cls": "puppy"}, {"id": 6110, "dom": "P", "ent": "pane", "cls": "receiver"}, {"id": 6111, "dom": "P", "ent": "pane", "cls": "reed"}, {"id": 6112, "dom": "P", "ent": "pane", "cls": "rifle"}, {"id": 6113, "dom": "P", "ent": "pane", "cls": "road"}, {"id": 6114, "dom": "P", "ent": "pane", "cls": "salmon"}, {"id": 6115, "dom": "P", "ent": "pane", "cls": "shark"}, {"id": 6116, "dom": "P", "ent": "pane", "cls": "sheep"}, {"id": 6117, "dom": "P", "ent": "pane", "cls": "shook"}, {"id": 6118, "dom": "P", "ent": "pane", "cls": "tank"}, {"id": 6119, "dom": "P", "ent": "pane", "cls": "wheat"}, {"id": 6120, "dom": "P", "ent": "panther", "cls": "architecture"}, {"id": 6121, "dom": "P", "ent": "panther", "cls": "bacteria"}, {"id": 6122, "dom": "P", "ent": "panther", "cls": "beef"}, {"id": 6123, "dom": "P", "ent": "panther", "cls": "bridge"}, {"id": 6124, "dom": "P", "ent": "panther", "cls": "bud"}, {"id": 6125, "dom": "P", "ent": "panther", "cls": "cat"}, {"id": 6126, "dom": "P", "ent": "panther", "cls": "chapel"}, {"id": 6127, "dom": "P", "ent": "panther", "cls": "chick"}, {"id": 6128, "dom": "P", "ent": "panther", "cls": "clock"}, {"id": 6129, "dom": "P", "ent": "panther", "cls": "deer"}, {"id": 6130, "dom": "P", "ent": "panther", "cls": "dog"}, {"id": 6131, "dom": "P", "ent": "panther", "cls": "elephant"}, {"id": 6132, "dom": "P", "ent": "panther", "cls": "embassy"}, {"id": 6133, "dom": "P", "ent": "panther", "cls": "garage"}, {"id": 6134, "dom": "P", "ent": "panther", "cls": "highway"}, {"id": 6135, "dom": "P", "ent": "panther", "cls": "kidney"}, {"id": 6136, "dom": "P", "ent": "panther", "cls": "lamb"}, {"id": 6137, "dom": "P", "ent": "panther", "cls": "male"}, {"id": 6138, "dom": "P", "ent": "panther", "cls": "marijuana"}, {"id": 6139, "dom": "P", "ent": "panther", "cls": "puppy"}, {"id": 6140, "dom": "P", "ent": "panther", "cls": "receiver"}, {"id": 6141, "dom": "P", "ent": "panther", "cls": "reed"}, {"id": 6142, "dom": "P", "ent": "panther", "cls": "rifle"}, {"id": 6143, "dom": "P", "ent": "panther", "cls": "road"}, {"id": 6144, "dom": "P", "ent": "panther", "cls": "salmon"}, {"id": 6145, "dom": "P", "ent": "panther", "cls": "shark"}, {"id": 6146, "dom": "P", "ent": "panther", "cls": "sheep"}, {"id": 6147, "dom": "P", "ent": "panther", "cls": "shook"}, {"id": 6148, "dom": "P", "ent": "panther", "cls": "tank"}, {"id": 6149, "dom": "P", "ent": "panther", "cls": "wheat"}, {"id": 6150, "dom": "P", "ent": "part", "cls": "architecture"}, {"id": 6151, "dom": "P", "ent": "part", "cls": "bacteria"}, {"id": 6152, "dom": "P", "ent": "part", "cls": "beef"}, {"id": 6153, "dom": "P", "ent": "part", "cls": "bridge"}, {"id": 6154, "dom": "P", "ent": "part", "cls": "bud"}, {"id": 6155, "dom": "P", "ent": "part", "cls": "cat"}, {"id": 6156, "dom": "P", "ent": "part", "cls": "chapel"}, {"id": 6157, "dom": "P", "ent": "part", "cls": "chick"}, {"id": 6158, "dom": "P", "ent": "part", "cls": "clock"}, {"id": 6159, "dom": "P", "ent": "part", "cls": "deer"}, {"id": 6160, "dom": "P", "ent": "part", "cls": "dog"}, {"id": 6161, "dom": "P", "ent": "part", "cls": "elephant"}, {"id": 6162, "dom": "P", "ent": "part", "cls": "embassy"}, {"id": 6163, "dom": "P", "ent": "part", "cls": "garage"}, {"id": 6164, "dom": "P", "ent": "part", "cls": "highway"}, {"id": 6165, "dom": "P", "ent": "part", "cls": "kidney"}, {"id": 6166, "dom": "P", "ent": "part", "cls": "lamb"}, {"id": 6167, "dom": "P", "ent": "part", "cls": "male"}, {"id": 6168, "dom": "P", "ent": "part", "cls": "marijuana"}, {"id": 6169, "dom": "P", "ent": "part", "cls": "puppy"}, {"id": 6170, "dom": "P", "ent": "part", "cls": "receiver"}, {"id": 6171, "dom": "P", "ent": "part", "cls": "reed"}, {"id": 6172, "dom": "P", "ent": "part", "cls": "rifle"}, {"id": 6173, "dom": "P", "ent": "part", "cls": "road"}, {"id": 6174, "dom": "P", "ent": "part", "cls": "salmon"}, {"id": 6175, "dom": "P", "ent": "part", "cls": "shark"}, {"id": 6176, "dom": "P", "ent": "part", "cls": "sheep"}, {"id": 6177, "dom": "P", "ent": "part", "cls": "shook"}, {"id": 6178, "dom": "P", "ent": "part", "cls": "tank"}, {"id": 6179, "dom": "P", "ent": "part", "cls": "wheat"}, {"id": 6180, "dom": "P", "ent": "paw", "cls": "architecture"}, {"id": 6181, "dom": "P", "ent": "paw", "cls": "bacteria"}, {"id": 6182, "dom": "P", "ent": "paw", "cls": "beef"}, {"id": 6183, "dom": "P", "ent": "paw", "cls": "bridge"}, {"id": 6184, "dom": "P", "ent": "paw", "cls": "bud"}, {"id": 6185, "dom": "P", "ent": "paw", "cls": "cat"}, {"id": 6186, "dom": "P", "ent": "paw", "cls": "chapel"}, {"id": 6187, "dom": "P", "ent": "paw", "cls": "chick"}, {"id": 6188, "dom": "P", "ent": "paw", "cls": "clock"}, {"id": 6189, "dom": "P", "ent": "paw", "cls": "deer"}, {"id": 6190, "dom": "P", "ent": "paw", "cls": "dog"}, {"id": 6191, "dom": "P", "ent": "paw", "cls": "elephant"}, {"id": 6192, "dom": "P", "ent": "paw", "cls": "embassy"}, {"id": 6193, "dom": "P", "ent": "paw", "cls": "garage"}, {"id": 6194, "dom": "P", "ent": "paw", "cls": "highway"}, {"id": 6195, "dom": "P", "ent": "paw", "cls": "kidney"}, {"id": 6196, "dom": "P", "ent": "paw", "cls": "lamb"}, {"id": 6197, "dom": "P", "ent": "paw", "cls": "male"}, {"id": 6198, "dom": "P", "ent": "paw", "cls": "marijuana"}, {"id": 6199, "dom": "P", "ent": "paw", "cls": "puppy"}, {"id": 6200, "dom": "P", "ent": "paw", "cls": "receiver"}, {"id": 6201, "dom": "P", "ent": "paw", "cls": "reed"}, {"id": 6202, "dom": "P", "ent": "paw", "cls": "rifle"}, {"id": 6203, "dom": "P", "ent": "paw", "cls": "road"}, {"id": 6204, "dom": "P", "ent": "paw", "cls": "salmon"}, {"id": 6205, "dom": "P", "ent": "paw", "cls": "shark"}, {"id": 6206, "dom": "P", "ent": "paw", "cls": "sheep"}, {"id": 6207, "dom": "P", "ent": "paw", "cls": "shook"}, {"id": 6208, "dom": "P", "ent": "paw", "cls": "tank"}, {"id": 6209, "dom": "P", "ent": "paw", "cls": "wheat"}, {"id": 6210, "dom": "P", "ent": "peak", "cls": "architecture"}, {"id": 6211, "dom": "P", "ent": "peak", "cls": "bacteria"}, {"id": 6212, "dom": "P", "ent": "peak", "cls": "beef"}, {"id": 6213, "dom": "P", "ent": "peak", "cls": "bridge"}, {"id": 6214, "dom": "P", "ent": "peak", "cls": "bud"}, {"id": 6215, "dom": "P", "ent": "peak", "cls": "cat"}, {"id": 6216, "dom": "P", "ent": "peak", "cls": "chapel"}, {"id": 6217, "dom": "P", "ent": "peak", "cls": "chick"}, {"id": 6218, "dom": "P", "ent": "peak", "cls": "clock"}, {"id": 6219, "dom": "P", "ent": "peak", "cls": "deer"}, {"id": 6220, "dom": "P", "ent": "peak", "cls": "dog"}, {"id": 6221, "dom": "P", "ent": "peak", "cls": "elephant"}, {"id": 6222, "dom": "P", "ent": "peak", "cls": "embassy"}, {"id": 6223, "dom": "P", "ent": "peak", "cls": "garage"}, {"id": 6224, "dom": "P", "ent": "peak", "cls": "highway"}, {"id": 6225, "dom": "P", "ent": "peak", "cls": "kidney"}, {"id": 6226, "dom": "P", "ent": "peak", "cls": "lamb"}, {"id": 6227, "dom": "P", "ent": "peak", "cls": "male"}, {"id": 6228, "dom": "P", "ent": "peak", "cls": "marijuana"}, {"id": 6229, "dom": "P", "ent": "peak", "cls": "puppy"}, {"id": 6230, "dom": "P", "ent": "peak", "cls": "receiver"}, {"id": 6231, "dom": "P", "ent": "peak", "cls": "reed"}, {"id": 6232, "dom": "P", "ent": "peak", "cls": "rifle"}, {"id": 6233, "dom": "P", "ent": "peak", "cls": "road"}, {"id": 6234, "dom": "P", "ent": "peak", "cls": "salmon"}, {"id": 6235, "dom": "P", "ent": "peak", "cls": "shark"}, {"id": 6236, "dom": "P", "ent": "peak", "cls": "sheep"}, {"id": 6237, "dom": "P", "ent": "peak", "cls": "shook"}, {"id": 6238, "dom": "P", "ent": "peak", "cls": "tank"}, {"id": 6239, "dom": "P", "ent": "peak", "cls": "wheat"}, {"id": 6240, "dom": "P", "ent": "pearl-fish", "cls": "architecture"}, {"id": 6241, "dom": "P", "ent": "pearl-fish", "cls": "bacteria"}, {"id": 6242, "dom": "P", "ent": "pearl-fish", "cls": "beef"}, {"id": 6243, "dom": "P", "ent": "pearl-fish", "cls": "bridge"}, {"id": 6244, "dom": "P", "ent": "pearl-fish", "cls": "bud"}, {"id": 6245, "dom": "P", "ent": "pearl-fish", "cls": "cat"}, {"id": 6246, "dom": "P", "ent": "pearl-fish", "cls": "chapel"}, {"id": 6247, "dom": "P", "ent": "pearl-fish", "cls": "chick"}, {"id": 6248, "dom": "P", "ent": "pearl-fish", "cls": "clock"}, {"id": 6249, "dom": "P", "ent": "pearl-fish", "cls": "deer"}, {"id": 6250, "dom": "P", "ent": "pearl-fish", "cls": "dog"}, {"id": 6251, "dom": "P", "ent": "pearl-fish", "cls": "elephant"}, {"id": 6252, "dom": "P", "ent": "pearl-fish", "cls": "embassy"}, {"id": 6253, "dom": "P", "ent": "pearl-fish", "cls": "garage"}, {"id": 6254, "dom": "P", "ent": "pearl-fish", "cls": "highway"}, {"id": 6255, "dom": "P", "ent": "pearl-fish", "cls": "kidney"}, {"id": 6256, "dom": "P", "ent": "pearl-fish", "cls": "lamb"}, {"id": 6257, "dom": "P", "ent": "pearl-fish", "cls": "male"}, {"id": 6258, "dom": "P", "ent": "pearl-fish", "cls": "marijuana"}, {"id": 6259, "dom": "P", "ent": "pearl-fish", "cls": "puppy"}, {"id": 6260, "dom": "P", "ent": "pearl-fish", "cls": "receiver"}, {"id": 6261, "dom": "P", "ent": "pearl-fish", "cls": "reed"}, {"id": 6262, "dom": "P", "ent": "pearl-fish", "cls": "rifle"}, {"id": 6263, "dom": "P", "ent": "pearl-fish", "cls": "road"}, {"id": 6264, "dom": "P", "ent": "pearl-fish", "cls": "salmon"}, {"id": 6265, "dom": "P", "ent": "pearl-fish", "cls": "shark"}, {"id": 6266, "dom": "P", "ent": "pearl-fish", "cls": "sheep"}, {"id": 6267, "dom": "P", "ent": "pearl-fish", "cls": "shook"}, {"id": 6268, "dom": "P", "ent": "pearl-fish", "cls": "tank"}, {"id": 6269, "dom": "P", "ent": "pearl-fish", "cls": "wheat"}, {"id": 6270, "dom": "P", "ent": "pelvis", "cls": "architecture"}, {"id": 6271, "dom": "P", "ent": "pelvis", "cls": "bacteria"}, {"id": 6272, "dom": "P", "ent": "pelvis", "cls": "beef"}, {"id": 6273, "dom": "P", "ent": "pelvis", "cls": "bridge"}, {"id": 6274, "dom": "P", "ent": "pelvis", "cls": "bud"}, {"id": 6275, "dom": "P", "ent": "pelvis", "cls": "cat"}, {"id": 6276, "dom": "P", "ent": "pelvis", "cls": "chapel"}, {"id": 6277, "dom": "P", "ent": "pelvis", "cls": "chick"}, {"id": 6278, "dom": "P", "ent": "pelvis", "cls": "clock"}, {"id": 6279, "dom": "P", "ent": "pelvis", "cls": "deer"}, {"id": 6280, "dom": "P", "ent": "pelvis", "cls": "dog"}, {"id": 6281, "dom": "P", "ent": "pelvis", "cls": "elephant"}, {"id": 6282, "dom": "P", "ent": "pelvis", "cls": "embassy"}, {"id": 6283, "dom": "P", "ent": "pelvis", "cls": "garage"}, {"id": 6284, "dom": "P", "ent": "pelvis", "cls": "highway"}, {"id": 6285, "dom": "P", "ent": "pelvis", "cls": "kidney"}, {"id": 6286, "dom": "P", "ent": "pelvis", "cls": "lamb"}, {"id": 6287, "dom": "P", "ent": "pelvis", "cls": "male"}, {"id": 6288, "dom": "P", "ent": "pelvis", "cls": "marijuana"}, {"id": 6289, "dom": "P", "ent": "pelvis", "cls": "puppy"}, {"id": 6290, "dom": "P", "ent": "pelvis", "cls": "receiver"}, {"id": 6291, "dom": "P", "ent": "pelvis", "cls": "reed"}, {"id": 6292, "dom": "P", "ent": "pelvis", "cls": "rifle"}, {"id": 6293, "dom": "P", "ent": "pelvis", "cls": "road"}, {"id": 6294, "dom": "P", "ent": "pelvis", "cls": "salmon"}, {"id": 6295, "dom": "P", "ent": "pelvis", "cls": "shark"}, {"id": 6296, "dom": "P", "ent": "pelvis", "cls": "sheep"}, {"id": 6297, "dom": "P", "ent": "pelvis", "cls": "shook"}, {"id": 6298, "dom": "P", "ent": "pelvis", "cls": "tank"}, {"id": 6299, "dom": "P", "ent": "pelvis", "cls": "wheat"}, {"id": 6300, "dom": "P", "ent": "petal", "cls": "architecture"}, {"id": 6301, "dom": "P", "ent": "petal", "cls": "bacteria"}, {"id": 6302, "dom": "P", "ent": "petal", "cls": "beef"}, {"id": 6303, "dom": "P", "ent": "petal", "cls": "bridge"}, {"id": 6304, "dom": "P", "ent": "petal", "cls": "bud"}, {"id": 6305, "dom": "P", "ent": "petal", "cls": "cat"}, {"id": 6306, "dom": "P", "ent": "petal", "cls": "chapel"}, {"id": 6307, "dom": "P", "ent": "petal", "cls": "chick"}, {"id": 6308, "dom": "P", "ent": "petal", "cls": "clock"}, {"id": 6309, "dom": "P", "ent": "petal", "cls": "deer"}, {"id": 6310, "dom": "P", "ent": "petal", "cls": "dog"}, {"id": 6311, "dom": "P", "ent": "petal", "cls": "elephant"}, {"id": 6312, "dom": "P", "ent": "petal", "cls": "embassy"}, {"id": 6313, "dom": "P", "ent": "petal", "cls": "garage"}, {"id": 6314, "dom": "P", "ent": "petal", "cls": "highway"}, {"id": 6315, "dom": "P", "ent": "petal", "cls": "kidney"}, {"id": 6316, "dom": "P", "ent": "petal", "cls": "lamb"}, {"id": 6317, "dom": "P", "ent": "petal", "cls": "male"}, {"id": 6318, "dom": "P", "ent": "petal", "cls": "marijuana"}, {"id": 6319, "dom": "P", "ent": "petal", "cls": "puppy"}, {"id": 6320, "dom": "P", "ent": "petal", "cls": "receiver"}, {"id": 6321, "dom": "P", "ent": "petal", "cls": "reed"}, {"id": 6322, "dom": "P", "ent": "petal", "cls": "rifle"}, {"id": 6323, "dom": "P", "ent": "petal", "cls": "road"}, {"id": 6324, "dom": "P", "ent": "petal", "cls": "salmon"}, {"id": 6325, "dom": "P", "ent": "petal", "cls": "shark"}, {"id": 6326, "dom": "P", "ent": "petal", "cls": "sheep"}, {"id": 6327, "dom": "P", "ent": "petal", "cls": "shook"}, {"id": 6328, "dom": "P", "ent": "petal", "cls": "tank"}, {"id": 6329, "dom": "P", "ent": "petal", "cls": "wheat"}, {"id": 6330, "dom": "P", "ent": "photograph", "cls": "architecture"}, {"id": 6331, "dom": "P", "ent": "photograph", "cls": "bacteria"}, {"id": 6332, "dom": "P", "ent": "photograph", "cls": "beef"}, {"id": 6333, "dom": "P", "ent": "photograph", "cls": "bridge"}, {"id": 6334, "dom": "P", "ent": "photograph", "cls": "bud"}, {"id": 6335, "dom": "P", "ent": "photograph", "cls": "cat"}, {"id": 6336, "dom": "P", "ent": "photograph", "cls": "chapel"}, {"id": 6337, "dom": "P", "ent": "photograph", "cls": "chick"}, {"id": 6338, "dom": "P", "ent": "photograph", "cls": "clock"}, {"id": 6339, "dom": "P", "ent": "photograph", "cls": "deer"}, {"id": 6340, "dom": "P", "ent": "photograph", "cls": "dog"}, {"id": 6341, "dom": "P", "ent": "photograph", "cls": "elephant"}, {"id": 6342, "dom": "P", "ent": "photograph", "cls": "embassy"}, {"id": 6343, "dom": "P", "ent": "photograph", "cls": "garage"}, {"id": 6344, "dom": "P", "ent": "photograph", "cls": "highway"}, {"id": 6345, "dom": "P", "ent": "photograph", "cls": "kidney"}, {"id": 6346, "dom": "P", "ent": "photograph", "cls": "lamb"}, {"id": 6347, "dom": "P", "ent": "photograph", "cls": "male"}, {"id": 6348, "dom": "P", "ent": "photograph", "cls": "marijuana"}, {"id": 6349, "dom": "P", "ent": "photograph", "cls": "puppy"}, {"id": 6350, "dom": "P", "ent": "photograph", "cls": "receiver"}, {"id": 6351, "dom": "P", "ent": "photograph", "cls": "reed"}, {"id": 6352, "dom": "P", "ent": "photograph", "cls": "rifle"}, {"id": 6353, "dom": "P", "ent": "photograph", "cls": "road"}, {"id": 6354, "dom": "P", "ent": "photograph", "cls": "salmon"}, {"id": 6355, "dom": "P", "ent": "photograph", "cls": "shark"}, {"id": 6356, "dom": "P", "ent": "photograph", "cls": "sheep"}, {"id": 6357, "dom": "P", "ent": "photograph", "cls": "shook"}, {"id": 6358, "dom": "P", "ent": "photograph", "cls": "tank"}, {"id": 6359, "dom": "P", "ent": "photograph", "cls": "wheat"}, {"id": 6360, "dom": "P", "ent": "pier", "cls": "architecture"}, {"id": 6361, "dom": "P", "ent": "pier", "cls": "bacteria"}, {"id": 6362, "dom": "P", "ent": "pier", "cls": "beef"}, {"id": 6363, "dom": "P", "ent": "pier", "cls": "bridge"}, {"id": 6364, "dom": "P", "ent": "pier", "cls": "bud"}, {"id": 6365, "dom": "P", "ent": "pier", "cls": "cat"}, {"id": 6366, "dom": "P", "ent": "pier", "cls": "chapel"}, {"id": 6367, "dom": "P", "ent": "pier", "cls": "chick"}, {"id": 6368, "dom": "P", "ent": "pier", "cls": "clock"}, {"id": 6369, "dom": "P", "ent": "pier", "cls": "deer"}, {"id": 6370, "dom": "P", "ent": "pier", "cls": "dog"}, {"id": 6371, "dom": "P", "ent": "pier", "cls": "elephant"}, {"id": 6372, "dom": "P", "ent": "pier", "cls": "embassy"}, {"id": 6373, "dom": "P", "ent": "pier", "cls": "garage"}, {"id": 6374, "dom": "P", "ent": "pier", "cls": "highway"}, {"id": 6375, "dom": "P", "ent": "pier", "cls": "kidney"}, {"id": 6376, "dom": "P", "ent": "pier", "cls": "lamb"}, {"id": 6377, "dom": "P", "ent": "pier", "cls": "male"}, {"id": 6378, "dom": "P", "ent": "pier", "cls": "marijuana"}, {"id": 6379, "dom": "P", "ent": "pier", "cls": "puppy"}, {"id": 6380, "dom": "P", "ent": "pier", "cls": "receiver"}, {"id": 6381, "dom": "P", "ent": "pier", "cls": "reed"}, {"id": 6382, "dom": "P", "ent": "pier", "cls": "rifle"}, {"id": 6383, "dom": "P", "ent": "pier", "cls": "road"}, {"id": 6384, "dom": "P", "ent": "pier", "cls": "salmon"}, {"id": 6385, "dom": "P", "ent": "pier", "cls": "shark"}, {"id": 6386, "dom": "P", "ent": "pier", "cls": "sheep"}, {"id": 6387, "dom": "P", "ent": "pier", "cls": "shook"}, {"id": 6388, "dom": "P", "ent": "pier", "cls": "tank"}, {"id": 6389, "dom": "P", "ent": "pier", "cls": "wheat"}, {"id": 6390, "dom": "P", "ent": "pineapple", "cls": "architecture"}, {"id": 6391, "dom": "P", "ent": "pineapple", "cls": "bacteria"}, {"id": 6392, "dom": "P", "ent": "pineapple", "cls": "beef"}, {"id": 6393, "dom": "P", "ent": "pineapple", "cls": "bridge"}, {"id": 6394, "dom": "P", "ent": "pineapple", "cls": "bud"}, {"id": 6395, "dom": "P", "ent": "pineapple", "cls": "cat"}, {"id": 6396, "dom": "P", "ent": "pineapple", "cls": "chapel"}, {"id": 6397, "dom": "P", "ent": "pineapple", "cls": "chick"}, {"id": 6398, "dom": "P", "ent": "pineapple", "cls": "clock"}, {"id": 6399, "dom": "P", "ent": "pineapple", "cls": "deer"}, {"id": 6400, "dom": "P", "ent": "pineapple", "cls": "dog"}, {"id": 6401, "dom": "P", "ent": "pineapple", "cls": "elephant"}, {"id": 6402, "dom": "P", "ent": "pineapple", "cls": "embassy"}, {"id": 6403, "dom": "P", "ent": "pineapple", "cls": "garage"}, {"id": 6404, "dom": "P", "ent": "pineapple", "cls": "highway"}, {"id": 6405, "dom": "P", "ent": "pineapple", "cls": "kidney"}, {"id": 6406, "dom": "P", "ent": "pineapple", "cls": "lamb"}, {"id": 6407, "dom": "P", "ent": "pineapple", "cls": "male"}, {"id": 6408, "dom": "P", "ent": "pineapple", "cls": "marijuana"}, {"id": 6409, "dom": "P", "ent": "pineapple", "cls": "puppy"}, {"id": 6410, "dom": "P", "ent": "pineapple", "cls": "receiver"}, {"id": 6411, "dom": "P", "ent": "pineapple", "cls": "reed"}, {"id": 6412, "dom": "P", "ent": "pineapple", "cls": "rifle"}, {"id": 6413, "dom": "P", "ent": "pineapple", "cls": "road"}, {"id": 6414, "dom": "P", "ent": "pineapple", "cls": "salmon"}, {"id": 6415, "dom": "P", "ent": "pineapple", "cls": "shark"}, {"id": 6416, "dom": "P", "ent": "pineapple", "cls": "sheep"}, {"id": 6417, "dom": "P", "ent": "pineapple", "cls": "shook"}, {"id": 6418, "dom": "P", "ent": "pineapple", "cls": "tank"}, {"id": 6419, "dom": "P", "ent": "pineapple", "cls": "wheat"}, {"id": 6420, "dom": "P", "ent": "placenta", "cls": "architecture"}, {"id": 6421, "dom": "P", "ent": "placenta", "cls": "bacteria"}, {"id": 6422, "dom": "P", "ent": "placenta", "cls": "beef"}, {"id": 6423, "dom": "P", "ent": "placenta", "cls": "bridge"}, {"id": 6424, "dom": "P", "ent": "placenta", "cls": "bud"}, {"id": 6425, "dom": "P", "ent": "placenta", "cls": "cat"}, {"id": 6426, "dom": "P", "ent": "placenta", "cls": "chapel"}, {"id": 6427, "dom": "P", "ent": "placenta", "cls": "chick"}, {"id": 6428, "dom": "P", "ent": "placenta", "cls": "clock"}, {"id": 6429, "dom": "P", "ent": "placenta", "cls": "deer"}, {"id": 6430, "dom": "P", "ent": "placenta", "cls": "dog"}, {"id": 6431, "dom": "P", "ent": "placenta", "cls": "elephant"}, {"id": 6432, "dom": "P", "ent": "placenta", "cls": "embassy"}, {"id": 6433, "dom": "P", "ent": "placenta", "cls": "garage"}, {"id": 6434, "dom": "P", "ent": "placenta", "cls": "highway"}, {"id": 6435, "dom": "P", "ent": "placenta", "cls": "kidney"}, {"id": 6436, "dom": "P", "ent": "placenta", "cls": "lamb"}, {"id": 6437, "dom": "P", "ent": "placenta", "cls": "male"}, {"id": 6438, "dom": "P", "ent": "placenta", "cls": "marijuana"}, {"id": 6439, "dom": "P", "ent": "placenta", "cls": "puppy"}, {"id": 6440, "dom": "P", "ent": "placenta", "cls": "receiver"}, {"id": 6441, "dom": "P", "ent": "placenta", "cls": "reed"}, {"id": 6442, "dom": "P", "ent": "placenta", "cls": "rifle"}, {"id": 6443, "dom": "P", "ent": "placenta", "cls": "road"}, {"id": 6444, "dom": "P", "ent": "placenta", "cls": "salmon"}, {"id": 6445, "dom": "P", "ent": "placenta", "cls": "shark"}, {"id": 6446, "dom": "P", "ent": "placenta", "cls": "sheep"}, {"id": 6447, "dom": "P", "ent": "placenta", "cls": "shook"}, {"id": 6448, "dom": "P", "ent": "placenta", "cls": "tank"}, {"id": 6449, "dom": "P", "ent": "placenta", "cls": "wheat"}, {"id": 6450, "dom": "P", "ent": "plain", "cls": "architecture"}, {"id": 6451, "dom": "P", "ent": "plain", "cls": "bacteria"}, {"id": 6452, "dom": "P", "ent": "plain", "cls": "beef"}, {"id": 6453, "dom": "P", "ent": "plain", "cls": "bridge"}, {"id": 6454, "dom": "P", "ent": "plain", "cls": "bud"}, {"id": 6455, "dom": "P", "ent": "plain", "cls": "cat"}, {"id": 6456, "dom": "P", "ent": "plain", "cls": "chapel"}, {"id": 6457, "dom": "P", "ent": "plain", "cls": "chick"}, {"id": 6458, "dom": "P", "ent": "plain", "cls": "clock"}, {"id": 6459, "dom": "P", "ent": "plain", "cls": "deer"}, {"id": 6460, "dom": "P", "ent": "plain", "cls": "dog"}, {"id": 6461, "dom": "P", "ent": "plain", "cls": "elephant"}, {"id": 6462, "dom": "P", "ent": "plain", "cls": "embassy"}, {"id": 6463, "dom": "P", "ent": "plain", "cls": "garage"}, {"id": 6464, "dom": "P", "ent": "plain", "cls": "highway"}, {"id": 6465, "dom": "P", "ent": "plain", "cls": "kidney"}, {"id": 6466, "dom": "P", "ent": "plain", "cls": "lamb"}, {"id": 6467, "dom": "P", "ent": "plain", "cls": "male"}, {"id": 6468, "dom": "P", "ent": "plain", "cls": "marijuana"}, {"id": 6469, "dom": "P", "ent": "plain", "cls": "puppy"}, {"id": 6470, "dom": "P", "ent": "plain", "cls": "receiver"}, {"id": 6471, "dom": "P", "ent": "plain", "cls": "reed"}, {"id": 6472, "dom": "P", "ent": "plain", "cls": "rifle"}, {"id": 6473, "dom": "P", "ent": "plain", "cls": "road"}, {"id": 6474, "dom": "P", "ent": "plain", "cls": "salmon"}, {"id": 6475, "dom": "P", "ent": "plain", "cls": "shark"}, {"id": 6476, "dom": "P", "ent": "plain", "cls": "sheep"}, {"id": 6477, "dom": "P", "ent": "plain", "cls": "shook"}, {"id": 6478, "dom": "P", "ent": "plain", "cls": "tank"}, {"id": 6479, "dom": "P", "ent": "plain", "cls": "wheat"}, {"id": 6480, "dom": "P", "ent": "plate", "cls": "architecture"}, {"id": 6481, "dom": "P", "ent": "plate", "cls": "bacteria"}, {"id": 6482, "dom": "P", "ent": "plate", "cls": "beef"}, {"id": 6483, "dom": "P", "ent": "plate", "cls": "bridge"}, {"id": 6484, "dom": "P", "ent": "plate", "cls": "bud"}, {"id": 6485, "dom": "P", "ent": "plate", "cls": "cat"}, {"id": 6486, "dom": "P", "ent": "plate", "cls": "chapel"}, {"id": 6487, "dom": "P", "ent": "plate", "cls": "chick"}, {"id": 6488, "dom": "P", "ent": "plate", "cls": "clock"}, {"id": 6489, "dom": "P", "ent": "plate", "cls": "deer"}, {"id": 6490, "dom": "P", "ent": "plate", "cls": "dog"}, {"id": 6491, "dom": "P", "ent": "plate", "cls": "elephant"}, {"id": 6492, "dom": "P", "ent": "plate", "cls": "embassy"}, {"id": 6493, "dom": "P", "ent": "plate", "cls": "garage"}, {"id": 6494, "dom": "P", "ent": "plate", "cls": "highway"}, {"id": 6495, "dom": "P", "ent": "plate", "cls": "kidney"}, {"id": 6496, "dom": "P", "ent": "plate", "cls": "lamb"}, {"id": 6497, "dom": "P", "ent": "plate", "cls": "male"}, {"id": 6498, "dom": "P", "ent": "plate", "cls": "marijuana"}, {"id": 6499, "dom": "P", "ent": "plate", "cls": "puppy"}, {"id": 6500, "dom": "P", "ent": "plate", "cls": "receiver"}, {"id": 6501, "dom": "P", "ent": "plate", "cls": "reed"}, {"id": 6502, "dom": "P", "ent": "plate", "cls": "rifle"}, {"id": 6503, "dom": "P", "ent": "plate", "cls": "road"}, {"id": 6504, "dom": "P", "ent": "plate", "cls": "salmon"}, {"id": 6505, "dom": "P", "ent": "plate", "cls": "shark"}, {"id": 6506, "dom": "P", "ent": "plate", "cls": "sheep"}, {"id": 6507, "dom": "P", "ent": "plate", "cls": "shook"}, {"id": 6508, "dom": "P", "ent": "plate", "cls": "tank"}, {"id": 6509, "dom": "P", "ent": "plate", "cls": "wheat"}, {"id": 6510, "dom": "P", "ent": "plume", "cls": "architecture"}, {"id": 6511, "dom": "P", "ent": "plume", "cls": "bacteria"}, {"id": 6512, "dom": "P", "ent": "plume", "cls": "beef"}, {"id": 6513, "dom": "P", "ent": "plume", "cls": "bridge"}, {"id": 6514, "dom": "P", "ent": "plume", "cls": "bud"}, {"id": 6515, "dom": "P", "ent": "plume", "cls": "cat"}, {"id": 6516, "dom": "P", "ent": "plume", "cls": "chapel"}, {"id": 6517, "dom": "P", "ent": "plume", "cls": "chick"}, {"id": 6518, "dom": "P", "ent": "plume", "cls": "clock"}, {"id": 6519, "dom": "P", "ent": "plume", "cls": "deer"}, {"id": 6520, "dom": "P", "ent": "plume", "cls": "dog"}, {"id": 6521, "dom": "P", "ent": "plume", "cls": "elephant"}, {"id": 6522, "dom": "P", "ent": "plume", "cls": "embassy"}, {"id": 6523, "dom": "P", "ent": "plume", "cls": "garage"}, {"id": 6524, "dom": "P", "ent": "plume", "cls": "highway"}, {"id": 6525, "dom": "P", "ent": "plume", "cls": "kidney"}, {"id": 6526, "dom": "P", "ent": "plume", "cls": "lamb"}, {"id": 6527, "dom": "P", "ent": "plume", "cls": "male"}, {"id": 6528, "dom": "P", "ent": "plume", "cls": "marijuana"}, {"id": 6529, "dom": "P", "ent": "plume", "cls": "puppy"}, {"id": 6530, "dom": "P", "ent": "plume", "cls": "receiver"}, {"id": 6531, "dom": "P", "ent": "plume", "cls": "reed"}, {"id": 6532, "dom": "P", "ent": "plume", "cls": "rifle"}, {"id": 6533, "dom": "P", "ent": "plume", "cls": "road"}, {"id": 6534, "dom": "P", "ent": "plume", "cls": "salmon"}, {"id": 6535, "dom": "P", "ent": "plume", "cls": "shark"}, {"id": 6536, "dom": "P", "ent": "plume", "cls": "sheep"}, {"id": 6537, "dom": "P", "ent": "plume", "cls": "shook"}, {"id": 6538, "dom": "P", "ent": "plume", "cls": "tank"}, {"id": 6539, "dom": "P", "ent": "plume", "cls": "wheat"}, {"id": 6540, "dom": "P", "ent": "poison-berry", "cls": "architecture"}, {"id": 6541, "dom": "P", "ent": "poison-berry", "cls": "bacteria"}, {"id": 6542, "dom": "P", "ent": "poison-berry", "cls": "beef"}, {"id": 6543, "dom": "P", "ent": "poison-berry", "cls": "bridge"}, {"id": 6544, "dom": "P", "ent": "poison-berry", "cls": "bud"}, {"id": 6545, "dom": "P", "ent": "poison-berry", "cls": "cat"}, {"id": 6546, "dom": "P", "ent": "poison-berry", "cls": "chapel"}, {"id": 6547, "dom": "P", "ent": "poison-berry", "cls": "chick"}, {"id": 6548, "dom": "P", "ent": "poison-berry", "cls": "clock"}, {"id": 6549, "dom": "P", "ent": "poison-berry", "cls": "deer"}, {"id": 6550, "dom": "P", "ent": "poison-berry", "cls": "dog"}, {"id": 6551, "dom": "P", "ent": "poison-berry", "cls": "elephant"}, {"id": 6552, "dom": "P", "ent": "poison-berry", "cls": "embassy"}, {"id": 6553, "dom": "P", "ent": "poison-berry", "cls": "garage"}, {"id": 6554, "dom": "P", "ent": "poison-berry", "cls": "highway"}, {"id": 6555, "dom": "P", "ent": "poison-berry", "cls": "kidney"}, {"id": 6556, "dom": "P", "ent": "poison-berry", "cls": "lamb"}, {"id": 6557, "dom": "P", "ent": "poison-berry", "cls": "male"}, {"id": 6558, "dom": "P", "ent": "poison-berry", "cls": "marijuana"}, {"id": 6559, "dom": "P", "ent": "poison-berry", "cls": "puppy"}, {"id": 6560, "dom": "P", "ent": "poison-berry", "cls": "receiver"}, {"id": 6561, "dom": "P", "ent": "poison-berry", "cls": "reed"}, {"id": 6562, "dom": "P", "ent": "poison-berry", "cls": "rifle"}, {"id": 6563, "dom": "P", "ent": "poison-berry", "cls": "road"}, {"id": 6564, "dom": "P", "ent": "poison-berry", "cls": "salmon"}, {"id": 6565, "dom": "P", "ent": "poison-berry", "cls": "shark"}, {"id": 6566, "dom": "P", "ent": "poison-berry", "cls": "sheep"}, {"id": 6567, "dom": "P", "ent": "poison-berry", "cls": "shook"}, {"id": 6568, "dom": "P", "ent": "poison-berry", "cls": "tank"}, {"id": 6569, "dom": "P", "ent": "poison-berry", "cls": "wheat"}, {"id": 6570, "dom": "P", "ent": "portion", "cls": "architecture"}, {"id": 6571, "dom": "P", "ent": "portion", "cls": "bacteria"}, {"id": 6572, "dom": "P", "ent": "portion", "cls": "beef"}, {"id": 6573, "dom": "P", "ent": "portion", "cls": "bridge"}, {"id": 6574, "dom": "P", "ent": "portion", "cls": "bud"}, {"id": 6575, "dom": "P", "ent": "portion", "cls": "cat"}, {"id": 6576, "dom": "P", "ent": "portion", "cls": "chapel"}, {"id": 6577, "dom": "P", "ent": "portion", "cls": "chick"}, {"id": 6578, "dom": "P", "ent": "portion", "cls": "clock"}, {"id": 6579, "dom": "P", "ent": "portion", "cls": "deer"}, {"id": 6580, "dom": "P", "ent": "portion", "cls": "dog"}, {"id": 6581, "dom": "P", "ent": "portion", "cls": "elephant"}, {"id": 6582, "dom": "P", "ent": "portion", "cls": "embassy"}, {"id": 6583, "dom": "P", "ent": "portion", "cls": "garage"}, {"id": 6584, "dom": "P", "ent": "portion", "cls": "highway"}, {"id": 6585, "dom": "P", "ent": "portion", "cls": "kidney"}, {"id": 6586, "dom": "P", "ent": "portion", "cls": "lamb"}, {"id": 6587, "dom": "P", "ent": "portion", "cls": "male"}, {"id": 6588, "dom": "P", "ent": "portion", "cls": "marijuana"}, {"id": 6589, "dom": "P", "ent": "portion", "cls": "puppy"}, {"id": 6590, "dom": "P", "ent": "portion", "cls": "receiver"}, {"id": 6591, "dom": "P", "ent": "portion", "cls": "reed"}, {"id": 6592, "dom": "P", "ent": "portion", "cls": "rifle"}, {"id": 6593, "dom": "P", "ent": "portion", "cls": "road"}, {"id": 6594, "dom": "P", "ent": "portion", "cls": "salmon"}, {"id": 6595, "dom": "P", "ent": "portion", "cls": "shark"}, {"id": 6596, "dom": "P", "ent": "portion", "cls": "sheep"}, {"id": 6597, "dom": "P", "ent": "portion", "cls": "shook"}, {"id": 6598, "dom": "P", "ent": "portion", "cls": "tank"}, {"id": 6599, "dom": "P", "ent": "portion", "cls": "wheat"}, {"id": 6600, "dom": "P", "ent": "portrait", "cls": "architecture"}, {"id": 6601, "dom": "P", "ent": "portrait", "cls": "bacteria"}, {"id": 6602, "dom": "P", "ent": "portrait", "cls": "beef"}, {"id": 6603, "dom": "P", "ent": "portrait", "cls": "bridge"}, {"id": 6604, "dom": "P", "ent": "portrait", "cls": "bud"}, {"id": 6605, "dom": "P", "ent": "portrait", "cls": "cat"}, {"id": 6606, "dom": "P", "ent": "portrait", "cls": "chapel"}, {"id": 6607, "dom": "P", "ent": "portrait", "cls": "chick"}, {"id": 6608, "dom": "P", "ent": "portrait", "cls": "clock"}, {"id": 6609, "dom": "P", "ent": "portrait", "cls": "deer"}, {"id": 6610, "dom": "P", "ent": "portrait", "cls": "dog"}, {"id": 6611, "dom": "P", "ent": "portrait", "cls": "elephant"}, {"id": 6612, "dom": "P", "ent": "portrait", "cls": "embassy"}, {"id": 6613, "dom": "P", "ent": "portrait", "cls": "garage"}, {"id": 6614, "dom": "P", "ent": "portrait", "cls": "highway"}, {"id": 6615, "dom": "P", "ent": "portrait", "cls": "kidney"}, {"id": 6616, "dom": "P", "ent": "portrait", "cls": "lamb"}, {"id": 6617, "dom": "P", "ent": "portrait", "cls": "male"}, {"id": 6618, "dom": "P", "ent": "portrait", "cls": "marijuana"}, {"id": 6619, "dom": "P", "ent": "portrait", "cls": "puppy"}, {"id": 6620, "dom": "P", "ent": "portrait", "cls": "receiver"}, {"id": 6621, "dom": "P", "ent": "portrait", "cls": "reed"}, {"id": 6622, "dom": "P", "ent": "portrait", "cls": "rifle"}, {"id": 6623, "dom": "P", "ent": "portrait", "cls": "road"}, {"id": 6624, "dom": "P", "ent": "portrait", "cls": "salmon"}, {"id": 6625, "dom": "P", "ent": "portrait", "cls": "shark"}, {"id": 6626, "dom": "P", "ent": "portrait", "cls": "sheep"}, {"id": 6627, "dom": "P", "ent": "portrait", "cls": "shook"}, {"id": 6628, "dom": "P", "ent": "portrait", "cls": "tank"}, {"id": 6629, "dom": "P", "ent": "portrait", "cls": "wheat"}, {"id": 6630, "dom": "P", "ent": "product", "cls": "architecture"}, {"id": 6631, "dom": "P", "ent": "product", "cls": "bacteria"}, {"id": 6632, "dom": "P", "ent": "product", "cls": "beef"}, {"id": 6633, "dom": "P", "ent": "product", "cls": "bridge"}, {"id": 6634, "dom": "P", "ent": "product", "cls": "bud"}, {"id": 6635, "dom": "P", "ent": "product", "cls": "cat"}, {"id": 6636, "dom": "P", "ent": "product", "cls": "chapel"}, {"id": 6637, "dom": "P", "ent": "product", "cls": "chick"}, {"id": 6638, "dom": "P", "ent": "product", "cls": "clock"}, {"id": 6639, "dom": "P", "ent": "product", "cls": "deer"}, {"id": 6640, "dom": "P", "ent": "product", "cls": "dog"}, {"id": 6641, "dom": "P", "ent": "product", "cls": "elephant"}, {"id": 6642, "dom": "P", "ent": "product", "cls": "embassy"}, {"id": 6643, "dom": "P", "ent": "product", "cls": "garage"}, {"id": 6644, "dom": "P", "ent": "product", "cls": "highway"}, {"id": 6645, "dom": "P", "ent": "product", "cls": "kidney"}, {"id": 6646, "dom": "P", "ent": "product", "cls": "lamb"}, {"id": 6647, "dom": "P", "ent": "product", "cls": "male"}, {"id": 6648, "dom": "P", "ent": "product", "cls": "marijuana"}, {"id": 6649, "dom": "P", "ent": "product", "cls": "puppy"}, {"id": 6650, "dom": "P", "ent": "product", "cls": "receiver"}, {"id": 6651, "dom": "P", "ent": "product", "cls": "reed"}, {"id": 6652, "dom": "P", "ent": "product", "cls": "rifle"}, {"id": 6653, "dom": "P", "ent": "product", "cls": "road"}, {"id": 6654, "dom": "P", "ent": "product", "cls": "salmon"}, {"id": 6655, "dom": "P", "ent": "product", "cls": "shark"}, {"id": 6656, "dom": "P", "ent": "product", "cls": "sheep"}, {"id": 6657, "dom": "P", "ent": "product", "cls": "shook"}, {"id": 6658, "dom": "P", "ent": "product", "cls": "tank"}, {"id": 6659, "dom": "P", "ent": "product", "cls": "wheat"}, {"id": 6660, "dom": "P", "ent": "punch", "cls": "architecture"}, {"id": 6661, "dom": "P", "ent": "punch", "cls": "bacteria"}, {"id": 6662, "dom": "P", "ent": "punch", "cls": "beef"}, {"id": 6663, "dom": "P", "ent": "punch", "cls": "bridge"}, {"id": 6664, "dom": "P", "ent": "punch", "cls": "bud"}, {"id": 6665, "dom": "P", "ent": "punch", "cls": "cat"}, {"id": 6666, "dom": "P", "ent": "punch", "cls": "chapel"}, {"id": 6667, "dom": "P", "ent": "punch", "cls": "chick"}, {"id": 6668, "dom": "P", "ent": "punch", "cls": "clock"}, {"id": 6669, "dom": "P", "ent": "punch", "cls": "deer"}, {"id": 6670, "dom": "P", "ent": "punch", "cls": "dog"}, {"id": 6671, "dom": "P", "ent": "punch", "cls": "elephant"}, {"id": 6672, "dom": "P", "ent": "punch", "cls": "embassy"}, {"id": 6673, "dom": "P", "ent": "punch", "cls": "garage"}, {"id": 6674, "dom": "P", "ent": "punch", "cls": "highway"}, {"id": 6675, "dom": "P", "ent": "punch", "cls": "kidney"}, {"id": 6676, "dom": "P", "ent": "punch", "cls": "lamb"}, {"id": 6677, "dom": "P", "ent": "punch", "cls": "male"}, {"id": 6678, "dom": "P", "ent": "punch", "cls": "marijuana"}, {"id": 6679, "dom": "P", "ent": "punch", "cls": "puppy"}, {"id": 6680, "dom": "P", "ent": "punch", "cls": "receiver"}, {"id": 6681, "dom": "P", "ent": "punch", "cls": "reed"}, {"id": 6682, "dom": "P", "ent": "punch", "cls": "rifle"}, {"id": 6683, "dom": "P", "ent": "punch", "cls": "road"}, {"id": 6684, "dom": "P", "ent": "punch", "cls": "salmon"}, {"id": 6685, "dom": "P", "ent": "punch", "cls": "shark"}, {"id": 6686, "dom": "P", "ent": "punch", "cls": "sheep"}, {"id": 6687, "dom": "P", "ent": "punch", "cls": "shook"}, {"id": 6688, "dom": "P", "ent": "punch", "cls": "tank"}, {"id": 6689, "dom": "P", "ent": "punch", "cls": "wheat"}, {"id": 6690, "dom": "P", "ent": "pupil", "cls": "architecture"}, {"id": 6691, "dom": "P", "ent": "pupil", "cls": "bacteria"}, {"id": 6692, "dom": "P", "ent": "pupil", "cls": "beef"}, {"id": 6693, "dom": "P", "ent": "pupil", "cls": "bridge"}, {"id": 6694, "dom": "P", "ent": "pupil", "cls": "bud"}, {"id": 6695, "dom": "P", "ent": "pupil", "cls": "cat"}, {"id": 6696, "dom": "P", "ent": "pupil", "cls": "chapel"}, {"id": 6697, "dom": "P", "ent": "pupil", "cls": "chick"}, {"id": 6698, "dom": "P", "ent": "pupil", "cls": "clock"}, {"id": 6699, "dom": "P", "ent": "pupil", "cls": "deer"}, {"id": 6700, "dom": "P", "ent": "pupil", "cls": "dog"}, {"id": 6701, "dom": "P", "ent": "pupil", "cls": "elephant"}, {"id": 6702, "dom": "P", "ent": "pupil", "cls": "embassy"}, {"id": 6703, "dom": "P", "ent": "pupil", "cls": "garage"}, {"id": 6704, "dom": "P", "ent": "pupil", "cls": "highway"}, {"id": 6705, "dom": "P", "ent": "pupil", "cls": "kidney"}, {"id": 6706, "dom": "P", "ent": "pupil", "cls": "lamb"}, {"id": 6707, "dom": "P", "ent": "pupil", "cls": "male"}, {"id": 6708, "dom": "P", "ent": "pupil", "cls": "marijuana"}, {"id": 6709, "dom": "P", "ent": "pupil", "cls": "puppy"}, {"id": 6710, "dom": "P", "ent": "pupil", "cls": "receiver"}, {"id": 6711, "dom": "P", "ent": "pupil", "cls": "reed"}, {"id": 6712, "dom": "P", "ent": "pupil", "cls": "rifle"}, {"id": 6713, "dom": "P", "ent": "pupil", "cls": "road"}, {"id": 6714, "dom": "P", "ent": "pupil", "cls": "salmon"}, {"id": 6715, "dom": "P", "ent": "pupil", "cls": "shark"}, {"id": 6716, "dom": "P", "ent": "pupil", "cls": "sheep"}, {"id": 6717, "dom": "P", "ent": "pupil", "cls": "shook"}, {"id": 6718, "dom": "P", "ent": "pupil", "cls": "tank"}, {"id": 6719, "dom": "P", "ent": "pupil", "cls": "wheat"}, {"id": 6720, "dom": "P", "ent": "quill", "cls": "architecture"}, {"id": 6721, "dom": "P", "ent": "quill", "cls": "bacteria"}, {"id": 6722, "dom": "P", "ent": "quill", "cls": "beef"}, {"id": 6723, "dom": "P", "ent": "quill", "cls": "bridge"}, {"id": 6724, "dom": "P", "ent": "quill", "cls": "bud"}, {"id": 6725, "dom": "P", "ent": "quill", "cls": "cat"}, {"id": 6726, "dom": "P", "ent": "quill", "cls": "chapel"}, {"id": 6727, "dom": "P", "ent": "quill", "cls": "chick"}, {"id": 6728, "dom": "P", "ent": "quill", "cls": "clock"}, {"id": 6729, "dom": "P", "ent": "quill", "cls": "deer"}, {"id": 6730, "dom": "P", "ent": "quill", "cls": "dog"}, {"id": 6731, "dom": "P", "ent": "quill", "cls": "elephant"}, {"id": 6732, "dom": "P", "ent": "quill", "cls": "embassy"}, {"id": 6733, "dom": "P", "ent": "quill", "cls": "garage"}, {"id": 6734, "dom": "P", "ent": "quill", "cls": "highway"}, {"id": 6735, "dom": "P", "ent": "quill", "cls": "kidney"}, {"id": 6736, "dom": "P", "ent": "quill", "cls": "lamb"}, {"id": 6737, "dom": "P", "ent": "quill", "cls": "male"}, {"id": 6738, "dom": "P", "ent": "quill", "cls": "marijuana"}, {"id": 6739, "dom": "P", "ent": "quill", "cls": "puppy"}, {"id": 6740, "dom": "P", "ent": "quill", "cls": "receiver"}, {"id": 6741, "dom": "P", "ent": "quill", "cls": "reed"}, {"id": 6742, "dom": "P", "ent": "quill", "cls": "rifle"}, {"id": 6743, "dom": "P", "ent": "quill", "cls": "road"}, {"id": 6744, "dom": "P", "ent": "quill", "cls": "salmon"}, {"id": 6745, "dom": "P", "ent": "quill", "cls": "shark"}, {"id": 6746, "dom": "P", "ent": "quill", "cls": "sheep"}, {"id": 6747, "dom": "P", "ent": "quill", "cls": "shook"}, {"id": 6748, "dom": "P", "ent": "quill", "cls": "tank"}, {"id": 6749, "dom": "P", "ent": "quill", "cls": "wheat"}, {"id": 6750, "dom": "P", "ent": "radiator", "cls": "architecture"}, {"id": 6751, "dom": "P", "ent": "radiator", "cls": "bacteria"}, {"id": 6752, "dom": "P", "ent": "radiator", "cls": "beef"}, {"id": 6753, "dom": "P", "ent": "radiator", "cls": "bridge"}, {"id": 6754, "dom": "P", "ent": "radiator", "cls": "bud"}, {"id": 6755, "dom": "P", "ent": "radiator", "cls": "cat"}, {"id": 6756, "dom": "P", "ent": "radiator", "cls": "chapel"}, {"id": 6757, "dom": "P", "ent": "radiator", "cls": "chick"}, {"id": 6758, "dom": "P", "ent": "radiator", "cls": "clock"}, {"id": 6759, "dom": "P", "ent": "radiator", "cls": "deer"}, {"id": 6760, "dom": "P", "ent": "radiator", "cls": "dog"}, {"id": 6761, "dom": "P", "ent": "radiator", "cls": "elephant"}, {"id": 6762, "dom": "P", "ent": "radiator", "cls": "embassy"}, {"id": 6763, "dom": "P", "ent": "radiator", "cls": "garage"}, {"id": 6764, "dom": "P", "ent": "radiator", "cls": "highway"}, {"id": 6765, "dom": "P", "ent": "radiator", "cls": "kidney"}, {"id": 6766, "dom": "P", "ent": "radiator", "cls": "lamb"}, {"id": 6767, "dom": "P", "ent": "radiator", "cls": "male"}, {"id": 6768, "dom": "P", "ent": "radiator", "cls": "marijuana"}, {"id": 6769, "dom": "P", "ent": "radiator", "cls": "puppy"}, {"id": 6770, "dom": "P", "ent": "radiator", "cls": "receiver"}, {"id": 6771, "dom": "P", "ent": "radiator", "cls": "reed"}, {"id": 6772, "dom": "P", "ent": "radiator", "cls": "rifle"}, {"id": 6773, "dom": "P", "ent": "radiator", "cls": "road"}, {"id": 6774, "dom": "P", "ent": "radiator", "cls": "salmon"}, {"id": 6775, "dom": "P", "ent": "radiator", "cls": "shark"}, {"id": 6776, "dom": "P", "ent": "radiator", "cls": "sheep"}, {"id": 6777, "dom": "P", "ent": "radiator", "cls": "shook"}, {"id": 6778, "dom": "P", "ent": "radiator", "cls": "tank"}, {"id": 6779, "dom": "P", "ent": "radiator", "cls": "wheat"}, {"id": 6780, "dom": "P", "ent": "ray", "cls": "architecture"}, {"id": 6781, "dom": "P", "ent": "ray", "cls": "bacteria"}, {"id": 6782, "dom": "P", "ent": "ray", "cls": "beef"}, {"id": 6783, "dom": "P", "ent": "ray", "cls": "bridge"}, {"id": 6784, "dom": "P", "ent": "ray", "cls": "bud"}, {"id": 6785, "dom": "P", "ent": "ray", "cls": "cat"}, {"id": 6786, "dom": "P", "ent": "ray", "cls": "chapel"}, {"id": 6787, "dom": "P", "ent": "ray", "cls": "chick"}, {"id": 6788, "dom": "P", "ent": "ray", "cls": "clock"}, {"id": 6789, "dom": "P", "ent": "ray", "cls": "deer"}, {"id": 6790, "dom": "P", "ent": "ray", "cls": "dog"}, {"id": 6791, "dom": "P", "ent": "ray", "cls": "elephant"}, {"id": 6792, "dom": "P", "ent": "ray", "cls": "embassy"}, {"id": 6793, "dom": "P", "ent": "ray", "cls": "garage"}, {"id": 6794, "dom": "P", "ent": "ray", "cls": "highway"}, {"id": 6795, "dom": "P", "ent": "ray", "cls": "kidney"}, {"id": 6796, "dom": "P", "ent": "ray", "cls": "lamb"}, {"id": 6797, "dom": "P", "ent": "ray", "cls": "male"}, {"id": 6798, "dom": "P", "ent": "ray", "cls": "marijuana"}, {"id": 6799, "dom": "P", "ent": "ray", "cls": "puppy"}, {"id": 6800, "dom": "P", "ent": "ray", "cls": "receiver"}, {"id": 6801, "dom": "P", "ent": "ray", "cls": "reed"}, {"id": 6802, "dom": "P", "ent": "ray", "cls": "rifle"}, {"id": 6803, "dom": "P", "ent": "ray", "cls": "road"}, {"id": 6804, "dom": "P", "ent": "ray", "cls": "salmon"}, {"id": 6805, "dom": "P", "ent": "ray", "cls": "shark"}, {"id": 6806, "dom": "P", "ent": "ray", "cls": "sheep"}, {"id": 6807, "dom": "P", "ent": "ray", "cls": "shook"}, {"id": 6808, "dom": "P", "ent": "ray", "cls": "tank"}, {"id": 6809, "dom": "P", "ent": "ray", "cls": "wheat"}, {"id": 6810, "dom": "P", "ent": "relay", "cls": "architecture"}, {"id": 6811, "dom": "P", "ent": "relay", "cls": "bacteria"}, {"id": 6812, "dom": "P", "ent": "relay", "cls": "beef"}, {"id": 6813, "dom": "P", "ent": "relay", "cls": "bridge"}, {"id": 6814, "dom": "P", "ent": "relay", "cls": "bud"}, {"id": 6815, "dom": "P", "ent": "relay", "cls": "cat"}, {"id": 6816, "dom": "P", "ent": "relay", "cls": "chapel"}, {"id": 6817, "dom": "P", "ent": "relay", "cls": "chick"}, {"id": 6818, "dom": "P", "ent": "relay", "cls": "clock"}, {"id": 6819, "dom": "P", "ent": "relay", "cls": "deer"}, {"id": 6820, "dom": "P", "ent": "relay", "cls": "dog"}, {"id": 6821, "dom": "P", "ent": "relay", "cls": "elephant"}, {"id": 6822, "dom": "P", "ent": "relay", "cls": "embassy"}, {"id": 6823, "dom": "P", "ent": "relay", "cls": "garage"}, {"id": 6824, "dom": "P", "ent": "relay", "cls": "highway"}, {"id": 6825, "dom": "P", "ent": "relay", "cls": "kidney"}, {"id": 6826, "dom": "P", "ent": "relay", "cls": "lamb"}, {"id": 6827, "dom": "P", "ent": "relay", "cls": "male"}, {"id": 6828, "dom": "P", "ent": "relay", "cls": "marijuana"}, {"id": 6829, "dom": "P", "ent": "relay", "cls": "puppy"}, {"id": 6830, "dom": "P", "ent": "relay", "cls": "receiver"}, {"id": 6831, "dom": "P", "ent": "relay", "cls": "reed"}, {"id": 6832, "dom": "P", "ent": "relay", "cls": "rifle"}, {"id": 6833, "dom": "P", "ent": "relay", "cls": "road"}, {"id": 6834, "dom": "P", "ent": "relay", "cls": "salmon"}, {"id": 6835, "dom": "P", "ent": "relay", "cls": "shark"}, {"id": 6836, "dom": "P", "ent": "relay", "cls": "sheep"}, {"id": 6837, "dom": "P", "ent": "relay", "cls": "shook"}, {"id": 6838, "dom": "P", "ent": "relay", "cls": "tank"}, {"id": 6839, "dom": "P", "ent": "relay", "cls": "wheat"}, {"id": 6840, "dom": "P", "ent": "resistance", "cls": "architecture"}, {"id": 6841, "dom": "P", "ent": "resistance", "cls": "bacteria"}, {"id": 6842, "dom": "P", "ent": "resistance", "cls": "beef"}, {"id": 6843, "dom": "P", "ent": "resistance", "cls": "bridge"}, {"id": 6844, "dom": "P", "ent": "resistance", "cls": "bud"}, {"id": 6845, "dom": "P", "ent": "resistance", "cls": "cat"}, {"id": 6846, "dom": "P", "ent": "resistance", "cls": "chapel"}, {"id": 6847, "dom": "P", "ent": "resistance", "cls": "chick"}, {"id": 6848, "dom": "P", "ent": "resistance", "cls": "clock"}, {"id": 6849, "dom": "P", "ent": "resistance", "cls": "deer"}, {"id": 6850, "dom": "P", "ent": "resistance", "cls": "dog"}, {"id": 6851, "dom": "P", "ent": "resistance", "cls": "elephant"}, {"id": 6852, "dom": "P", "ent": "resistance", "cls": "embassy"}, {"id": 6853, "dom": "P", "ent": "resistance", "cls": "garage"}, {"id": 6854, "dom": "P", "ent": "resistance", "cls": "highway"}, {"id": 6855, "dom": "P", "ent": "resistance", "cls": "kidney"}, {"id": 6856, "dom": "P", "ent": "resistance", "cls": "lamb"}, {"id": 6857, "dom": "P", "ent": "resistance", "cls": "male"}, {"id": 6858, "dom": "P", "ent": "resistance", "cls": "marijuana"}, {"id": 6859, "dom": "P", "ent": "resistance", "cls": "puppy"}, {"id": 6860, "dom": "P", "ent": "resistance", "cls": "receiver"}, {"id": 6861, "dom": "P", "ent": "resistance", "cls": "reed"}, {"id": 6862, "dom": "P", "ent": "resistance", "cls": "rifle"}, {"id": 6863, "dom": "P", "ent": "resistance", "cls": "road"}, {"id": 6864, "dom": "P", "ent": "resistance", "cls": "salmon"}, {"id": 6865, "dom": "P", "ent": "resistance", "cls": "shark"}, {"id": 6866, "dom": "P", "ent": "resistance", "cls": "sheep"}, {"id": 6867, "dom": "P", "ent": "resistance", "cls": "shook"}, {"id": 6868, "dom": "P", "ent": "resistance", "cls": "tank"}, {"id": 6869, "dom": "P", "ent": "resistance", "cls": "wheat"}, {"id": 6870, "dom": "P", "ent": "resistor", "cls": "architecture"}, {"id": 6871, "dom": "P", "ent": "resistor", "cls": "bacteria"}, {"id": 6872, "dom": "P", "ent": "resistor", "cls": "beef"}, {"id": 6873, "dom": "P", "ent": "resistor", "cls": "bridge"}, {"id": 6874, "dom": "P", "ent": "resistor", "cls": "bud"}, {"id": 6875, "dom": "P", "ent": "resistor", "cls": "cat"}, {"id": 6876, "dom": "P", "ent": "resistor", "cls": "chapel"}, {"id": 6877, "dom": "P", "ent": "resistor", "cls": "chick"}, {"id": 6878, "dom": "P", "ent": "resistor", "cls": "clock"}, {"id": 6879, "dom": "P", "ent": "resistor", "cls": "deer"}, {"id": 6880, "dom": "P", "ent": "resistor", "cls": "dog"}, {"id": 6881, "dom": "P", "ent": "resistor", "cls": "elephant"}, {"id": 6882, "dom": "P", "ent": "resistor", "cls": "embassy"}, {"id": 6883, "dom": "P", "ent": "resistor", "cls": "garage"}, {"id": 6884, "dom": "P", "ent": "resistor", "cls": "highway"}, {"id": 6885, "dom": "P", "ent": "resistor", "cls": "kidney"}, {"id": 6886, "dom": "P", "ent": "resistor", "cls": "lamb"}, {"id": 6887, "dom": "P", "ent": "resistor", "cls": "male"}, {"id": 6888, "dom": "P", "ent": "resistor", "cls": "marijuana"}, {"id": 6889, "dom": "P", "ent": "resistor", "cls": "puppy"}, {"id": 6890, "dom": "P", "ent": "resistor", "cls": "receiver"}, {"id": 6891, "dom": "P", "ent": "resistor", "cls": "reed"}, {"id": 6892, "dom": "P", "ent": "resistor", "cls": "rifle"}, {"id": 6893, "dom": "P", "ent": "resistor", "cls": "road"}, {"id": 6894, "dom": "P", "ent": "resistor", "cls": "salmon"}, {"id": 6895, "dom": "P", "ent": "resistor", "cls": "shark"}, {"id": 6896, "dom": "P", "ent": "resistor", "cls": "sheep"}, {"id": 6897, "dom": "P", "ent": "resistor", "cls": "shook"}, {"id": 6898, "dom": "P", "ent": "resistor", "cls": "tank"}, {"id": 6899, "dom": "P", "ent": "resistor", "cls": "wheat"}, {"id": 6900, "dom": "P", "ent": "retina", "cls": "architecture"}, {"id": 6901, "dom": "P", "ent": "retina", "cls": "bacteria"}, {"id": 6902, "dom": "P", "ent": "retina", "cls": "beef"}, {"id": 6903, "dom": "P", "ent": "retina", "cls": "bridge"}, {"id": 6904, "dom": "P", "ent": "retina", "cls": "bud"}, {"id": 6905, "dom": "P", "ent": "retina", "cls": "cat"}, {"id": 6906, "dom": "P", "ent": "retina", "cls": "chapel"}, {"id": 6907, "dom": "P", "ent": "retina", "cls": "chick"}, {"id": 6908, "dom": "P", "ent": "retina", "cls": "clock"}, {"id": 6909, "dom": "P", "ent": "retina", "cls": "deer"}, {"id": 6910, "dom": "P", "ent": "retina", "cls": "dog"}, {"id": 6911, "dom": "P", "ent": "retina", "cls": "elephant"}, {"id": 6912, "dom": "P", "ent": "retina", "cls": "embassy"}, {"id": 6913, "dom": "P", "ent": "retina", "cls": "garage"}, {"id": 6914, "dom": "P", "ent": "retina", "cls": "highway"}, {"id": 6915, "dom": "P", "ent": "retina", "cls": "kidney"}, {"id": 6916, "dom": "P", "ent": "retina", "cls": "lamb"}, {"id": 6917, "dom": "P", "ent": "retina", "cls": "male"}, {"id": 6918, "dom": "P", "ent": "retina", "cls": "marijuana"}, {"id": 6919, "dom": "P", "ent": "retina", "cls": "puppy"}, {"id": 6920, "dom": "P", "ent": "retina", "cls": "receiver"}, {"id": 6921, "dom": "P", "ent": "retina", "cls": "reed"}, {"id": 6922, "dom": "P", "ent": "retina", "cls": "rifle"}, {"id": 6923, "dom": "P", "ent": "retina", "cls": "road"}, {"id": 6924, "dom": "P", "ent": "retina", "cls": "salmon"}, {"id": 6925, "dom": "P", "ent": "retina", "cls": "shark"}, {"id": 6926, "dom": "P", "ent": "retina", "cls": "sheep"}, {"id": 6927, "dom": "P", "ent": "retina", "cls": "shook"}, {"id": 6928, "dom": "P", "ent": "retina", "cls": "tank"}, {"id": 6929, "dom": "P", "ent": "retina", "cls": "wheat"}, {"id": 6930, "dom": "P", "ent": "rib", "cls": "architecture"}, {"id": 6931, "dom": "P", "ent": "rib", "cls": "bacteria"}, {"id": 6932, "dom": "P", "ent": "rib", "cls": "beef"}, {"id": 6933, "dom": "P", "ent": "rib", "cls": "bridge"}, {"id": 6934, "dom": "P", "ent": "rib", "cls": "bud"}, {"id": 6935, "dom": "P", "ent": "rib", "cls": "cat"}, {"id": 6936, "dom": "P", "ent": "rib", "cls": "chapel"}, {"id": 6937, "dom": "P", "ent": "rib", "cls": "chick"}, {"id": 6938, "dom": "P", "ent": "rib", "cls": "clock"}, {"id": 6939, "dom": "P", "ent": "rib", "cls": "deer"}, {"id": 6940, "dom": "P", "ent": "rib", "cls": "dog"}, {"id": 6941, "dom": "P", "ent": "rib", "cls": "elephant"}, {"id": 6942, "dom": "P", "ent": "rib", "cls": "embassy"}, {"id": 6943, "dom": "P", "ent": "rib", "cls": "garage"}, {"id": 6944, "dom": "P", "ent": "rib", "cls": "highway"}, {"id": 6945, "dom": "P", "ent": "rib", "cls": "kidney"}, {"id": 6946, "dom": "P", "ent": "rib", "cls": "lamb"}, {"id": 6947, "dom": "P", "ent": "rib", "cls": "male"}, {"id": 6948, "dom": "P", "ent": "rib", "cls": "marijuana"}, {"id": 6949, "dom": "P", "ent": "rib", "cls": "puppy"}, {"id": 6950, "dom": "P", "ent": "rib", "cls": "receiver"}, {"id": 6951, "dom": "P", "ent": "rib", "cls": "reed"}, {"id": 6952, "dom": "P", "ent": "rib", "cls": "rifle"}, {"id": 6953, "dom": "P", "ent": "rib", "cls": "road"}, {"id": 6954, "dom": "P", "ent": "rib", "cls": "salmon"}, {"id": 6955, "dom": "P", "ent": "rib", "cls": "shark"}, {"id": 6956, "dom": "P", "ent": "rib", "cls": "sheep"}, {"id": 6957, "dom": "P", "ent": "rib", "cls": "shook"}, {"id": 6958, "dom": "P", "ent": "rib", "cls": "tank"}, {"id": 6959, "dom": "P", "ent": "rib", "cls": "wheat"}, {"id": 6960, "dom": "P", "ent": "rim", "cls": "architecture"}, {"id": 6961, "dom": "P", "ent": "rim", "cls": "bacteria"}, {"id": 6962, "dom": "P", "ent": "rim", "cls": "beef"}, {"id": 6963, "dom": "P", "ent": "rim", "cls": "bridge"}, {"id": 6964, "dom": "P", "ent": "rim", "cls": "bud"}, {"id": 6965, "dom": "P", "ent": "rim", "cls": "cat"}, {"id": 6966, "dom": "P", "ent": "rim", "cls": "chapel"}, {"id": 6967, "dom": "P", "ent": "rim", "cls": "chick"}, {"id": 6968, "dom": "P", "ent": "rim", "cls": "clock"}, {"id": 6969, "dom": "P", "ent": "rim", "cls": "deer"}, {"id": 6970, "dom": "P", "ent": "rim", "cls": "dog"}, {"id": 6971, "dom": "P", "ent": "rim", "cls": "elephant"}, {"id": 6972, "dom": "P", "ent": "rim", "cls": "embassy"}, {"id": 6973, "dom": "P", "ent": "rim", "cls": "garage"}, {"id": 6974, "dom": "P", "ent": "rim", "cls": "highway"}, {"id": 6975, "dom": "P", "ent": "rim", "cls": "kidney"}, {"id": 6976, "dom": "P", "ent": "rim", "cls": "lamb"}, {"id": 6977, "dom": "P", "ent": "rim", "cls": "male"}, {"id": 6978, "dom": "P", "ent": "rim", "cls": "marijuana"}, {"id": 6979, "dom": "P", "ent": "rim", "cls": "puppy"}, {"id": 6980, "dom": "P", "ent": "rim", "cls": "receiver"}, {"id": 6981, "dom": "P", "ent": "rim", "cls": "reed"}, {"id": 6982, "dom": "P", "ent": "rim", "cls": "rifle"}, {"id": 6983, "dom": "P", "ent": "rim", "cls": "road"}, {"id": 6984, "dom": "P", "ent": "rim", "cls": "salmon"}, {"id": 6985, "dom": "P", "ent": "rim", "cls": "shark"}, {"id": 6986, "dom": "P", "ent": "rim", "cls": "sheep"}, {"id": 6987, "dom": "P", "ent": "rim", "cls": "shook"}, {"id": 6988, "dom": "P", "ent": "rim", "cls": "tank"}, {"id": 6989, "dom": "P", "ent": "rim", "cls": "wheat"}, {"id": 6990, "dom": "P", "ent": "ring", "cls": "architecture"}, {"id": 6991, "dom": "P", "ent": "ring", "cls": "bacteria"}, {"id": 6992, "dom": "P", "ent": "ring", "cls": "beef"}, {"id": 6993, "dom": "P", "ent": "ring", "cls": "bridge"}, {"id": 6994, "dom": "P", "ent": "ring", "cls": "bud"}, {"id": 6995, "dom": "P", "ent": "ring", "cls": "cat"}, {"id": 6996, "dom": "P", "ent": "ring", "cls": "chapel"}, {"id": 6997, "dom": "P", "ent": "ring", "cls": "chick"}, {"id": 6998, "dom": "P", "ent": "ring", "cls": "clock"}, {"id": 6999, "dom": "P", "ent": "ring", "cls": "deer"}, {"id": 7000, "dom": "P", "ent": "ring", "cls": "dog"}, {"id": 7001, "dom": "P", "ent": "ring", "cls": "elephant"}, {"id": 7002, "dom": "P", "ent": "ring", "cls": "embassy"}, {"id": 7003, "dom": "P", "ent": "ring", "cls": "garage"}, {"id": 7004, "dom": "P", "ent": "ring", "cls": "highway"}, {"id": 7005, "dom": "P", "ent": "ring", "cls": "kidney"}, {"id": 7006, "dom": "P", "ent": "ring", "cls": "lamb"}, {"id": 7007, "dom": "P", "ent": "ring", "cls": "male"}, {"id": 7008, "dom": "P", "ent": "ring", "cls": "marijuana"}, {"id": 7009, "dom": "P", "ent": "ring", "cls": "puppy"}, {"id": 7010, "dom": "P", "ent": "ring", "cls": "receiver"}, {"id": 7011, "dom": "P", "ent": "ring", "cls": "reed"}, {"id": 7012, "dom": "P", "ent": "ring", "cls": "rifle"}, {"id": 7013, "dom": "P", "ent": "ring", "cls": "road"}, {"id": 7014, "dom": "P", "ent": "ring", "cls": "salmon"}, {"id": 7015, "dom": "P", "ent": "ring", "cls": "shark"}, {"id": 7016, "dom": "P", "ent": "ring", "cls": "sheep"}, {"id": 7017, "dom": "P", "ent": "ring", "cls": "shook"}, {"id": 7018, "dom": "P", "ent": "ring", "cls": "tank"}, {"id": 7019, "dom": "P", "ent": "ring", "cls": "wheat"}, {"id": 7020, "dom": "P", "ent": "roadside", "cls": "architecture"}, {"id": 7021, "dom": "P", "ent": "roadside", "cls": "bacteria"}, {"id": 7022, "dom": "P", "ent": "roadside", "cls": "beef"}, {"id": 7023, "dom": "P", "ent": "roadside", "cls": "bridge"}, {"id": 7024, "dom": "P", "ent": "roadside", "cls": "bud"}, {"id": 7025, "dom": "P", "ent": "roadside", "cls": "cat"}, {"id": 7026, "dom": "P", "ent": "roadside", "cls": "chapel"}, {"id": 7027, "dom": "P", "ent": "roadside", "cls": "chick"}, {"id": 7028, "dom": "P", "ent": "roadside", "cls": "clock"}, {"id": 7029, "dom": "P", "ent": "roadside", "cls": "deer"}, {"id": 7030, "dom": "P", "ent": "roadside", "cls": "dog"}, {"id": 7031, "dom": "P", "ent": "roadside", "cls": "elephant"}, {"id": 7032, "dom": "P", "ent": "roadside", "cls": "embassy"}, {"id": 7033, "dom": "P", "ent": "roadside", "cls": "garage"}, {"id": 7034, "dom": "P", "ent": "roadside", "cls": "highway"}, {"id": 7035, "dom": "P", "ent": "roadside", "cls": "kidney"}, {"id": 7036, "dom": "P", "ent": "roadside", "cls": "lamb"}, {"id": 7037, "dom": "P", "ent": "roadside", "cls": "male"}, {"id": 7038, "dom": "P", "ent": "roadside", "cls": "marijuana"}, {"id": 7039, "dom": "P", "ent": "roadside", "cls": "puppy"}, {"id": 7040, "dom": "P", "ent": "roadside", "cls": "receiver"}, {"id": 7041, "dom": "P", "ent": "roadside", "cls": "reed"}, {"id": 7042, "dom": "P", "ent": "roadside", "cls": "rifle"}, {"id": 7043, "dom": "P", "ent": "roadside", "cls": "road"}, {"id": 7044, "dom": "P", "ent": "roadside", "cls": "salmon"}, {"id": 7045, "dom": "P", "ent": "roadside", "cls": "shark"}, {"id": 7046, "dom": "P", "ent": "roadside", "cls": "sheep"}, {"id": 7047, "dom": "P", "ent": "roadside", "cls": "shook"}, {"id": 7048, "dom": "P", "ent": "roadside", "cls": "tank"}, {"id": 7049, "dom": "P", "ent": "roadside", "cls": "wheat"}, {"id": 7050, "dom": "P", "ent": "rod", "cls": "architecture"}, {"id": 7051, "dom": "P", "ent": "rod", "cls": "bacteria"}, {"id": 7052, "dom": "P", "ent": "rod", "cls": "beef"}, {"id": 7053, "dom": "P", "ent": "rod", "cls": "bridge"}, {"id": 7054, "dom": "P", "ent": "rod", "cls": "bud"}, {"id": 7055, "dom": "P", "ent": "rod", "cls": "cat"}, {"id": 7056, "dom": "P", "ent": "rod", "cls": "chapel"}, {"id": 7057, "dom": "P", "ent": "rod", "cls": "chick"}, {"id": 7058, "dom": "P", "ent": "rod", "cls": "clock"}, {"id": 7059, "dom": "P", "ent": "rod", "cls": "deer"}, {"id": 7060, "dom": "P", "ent": "rod", "cls": "dog"}, {"id": 7061, "dom": "P", "ent": "rod", "cls": "elephant"}, {"id": 7062, "dom": "P", "ent": "rod", "cls": "embassy"}, {"id": 7063, "dom": "P", "ent": "rod", "cls": "garage"}, {"id": 7064, "dom": "P", "ent": "rod", "cls": "highway"}, {"id": 7065, "dom": "P", "ent": "rod", "cls": "kidney"}, {"id": 7066, "dom": "P", "ent": "rod", "cls": "lamb"}, {"id": 7067, "dom": "P", "ent": "rod", "cls": "male"}, {"id": 7068, "dom": "P", "ent": "rod", "cls": "marijuana"}, {"id": 7069, "dom": "P", "ent": "rod", "cls": "puppy"}, {"id": 7070, "dom": "P", "ent": "rod", "cls": "receiver"}, {"id": 7071, "dom": "P", "ent": "rod", "cls": "reed"}, {"id": 7072, "dom": "P", "ent": "rod", "cls": "rifle"}, {"id": 7073, "dom": "P", "ent": "rod", "cls": "road"}, {"id": 7074, "dom": "P", "ent": "rod", "cls": "salmon"}, {"id": 7075, "dom": "P", "ent": "rod", "cls": "shark"}, {"id": 7076, "dom": "P", "ent": "rod", "cls": "sheep"}, {"id": 7077, "dom": "P", "ent": "rod", "cls": "shook"}, {"id": 7078, "dom": "P", "ent": "rod", "cls": "tank"}, {"id": 7079, "dom": "P", "ent": "rod", "cls": "wheat"}, {"id": 7080, "dom": "P", "ent": "roe", "cls": "architecture"}, {"id": 7081, "dom": "P", "ent": "roe", "cls": "bacteria"}, {"id": 7082, "dom": "P", "ent": "roe", "cls": "beef"}, {"id": 7083, "dom": "P", "ent": "roe", "cls": "bridge"}, {"id": 7084, "dom": "P", "ent": "roe", "cls": "bud"}, {"id": 7085, "dom": "P", "ent": "roe", "cls": "cat"}, {"id": 7086, "dom": "P", "ent": "roe", "cls": "chapel"}, {"id": 7087, "dom": "P", "ent": "roe", "cls": "chick"}, {"id": 7088, "dom": "P", "ent": "roe", "cls": "clock"}, {"id": 7089, "dom": "P", "ent": "roe", "cls": "deer"}, {"id": 7090, "dom": "P", "ent": "roe", "cls": "dog"}, {"id": 7091, "dom": "P", "ent": "roe", "cls": "elephant"}, {"id": 7092, "dom": "P", "ent": "roe", "cls": "embassy"}, {"id": 7093, "dom": "P", "ent": "roe", "cls": "garage"}, {"id": 7094, "dom": "P", "ent": "roe", "cls": "highway"}, {"id": 7095, "dom": "P", "ent": "roe", "cls": "kidney"}, {"id": 7096, "dom": "P", "ent": "roe", "cls": "lamb"}, {"id": 7097, "dom": "P", "ent": "roe", "cls": "male"}, {"id": 7098, "dom": "P", "ent": "roe", "cls": "marijuana"}, {"id": 7099, "dom": "P", "ent": "roe", "cls": "puppy"}, {"id": 7100, "dom": "P", "ent": "roe", "cls": "receiver"}, {"id": 7101, "dom": "P", "ent": "roe", "cls": "reed"}, {"id": 7102, "dom": "P", "ent": "roe", "cls": "rifle"}, {"id": 7103, "dom": "P", "ent": "roe", "cls": "road"}, {"id": 7104, "dom": "P", "ent": "roe", "cls": "salmon"}, {"id": 7105, "dom": "P", "ent": "roe", "cls": "shark"}, {"id": 7106, "dom": "P", "ent": "roe", "cls": "sheep"}, {"id": 7107, "dom": "P", "ent": "roe", "cls": "shook"}, {"id": 7108, "dom": "P", "ent": "roe", "cls": "tank"}, {"id": 7109, "dom": "P", "ent": "roe", "cls": "wheat"}, {"id": 7110, "dom": "P", "ent": "roof", "cls": "architecture"}, {"id": 7111, "dom": "P", "ent": "roof", "cls": "bacteria"}, {"id": 7112, "dom": "P", "ent": "roof", "cls": "beef"}, {"id": 7113, "dom": "P", "ent": "roof", "cls": "bridge"}, {"id": 7114, "dom": "P", "ent": "roof", "cls": "bud"}, {"id": 7115, "dom": "P", "ent": "roof", "cls": "cat"}, {"id": 7116, "dom": "P", "ent": "roof", "cls": "chapel"}, {"id": 7117, "dom": "P", "ent": "roof", "cls": "chick"}, {"id": 7118, "dom": "P", "ent": "roof", "cls": "clock"}, {"id": 7119, "dom": "P", "ent": "roof", "cls": "deer"}, {"id": 7120, "dom": "P", "ent": "roof", "cls": "dog"}, {"id": 7121, "dom": "P", "ent": "roof", "cls": "elephant"}, {"id": 7122, "dom": "P", "ent": "roof", "cls": "embassy"}, {"id": 7123, "dom": "P", "ent": "roof", "cls": "garage"}, {"id": 7124, "dom": "P", "ent": "roof", "cls": "highway"}, {"id": 7125, "dom": "P", "ent": "roof", "cls": "kidney"}, {"id": 7126, "dom": "P", "ent": "roof", "cls": "lamb"}, {"id": 7127, "dom": "P", "ent": "roof", "cls": "male"}, {"id": 7128, "dom": "P", "ent": "roof", "cls": "marijuana"}, {"id": 7129, "dom": "P", "ent": "roof", "cls": "puppy"}, {"id": 7130, "dom": "P", "ent": "roof", "cls": "receiver"}, {"id": 7131, "dom": "P", "ent": "roof", "cls": "reed"}, {"id": 7132, "dom": "P", "ent": "roof", "cls": "rifle"}, {"id": 7133, "dom": "P", "ent": "roof", "cls": "road"}, {"id": 7134, "dom": "P", "ent": "roof", "cls": "salmon"}, {"id": 7135, "dom": "P", "ent": "roof", "cls": "shark"}, {"id": 7136, "dom": "P", "ent": "roof", "cls": "sheep"}, {"id": 7137, "dom": "P", "ent": "roof", "cls": "shook"}, {"id": 7138, "dom": "P", "ent": "roof", "cls": "tank"}, {"id": 7139, "dom": "P", "ent": "roof", "cls": "wheat"}, {"id": 7140, "dom": "P", "ent": "room", "cls": "architecture"}, {"id": 7141, "dom": "P", "ent": "room", "cls": "bacteria"}, {"id": 7142, "dom": "P", "ent": "room", "cls": "beef"}, {"id": 7143, "dom": "P", "ent": "room", "cls": "bridge"}, {"id": 7144, "dom": "P", "ent": "room", "cls": "bud"}, {"id": 7145, "dom": "P", "ent": "room", "cls": "cat"}, {"id": 7146, "dom": "P", "ent": "room", "cls": "chapel"}, {"id": 7147, "dom": "P", "ent": "room", "cls": "chick"}, {"id": 7148, "dom": "P", "ent": "room", "cls": "clock"}, {"id": 7149, "dom": "P", "ent": "room", "cls": "deer"}, {"id": 7150, "dom": "P", "ent": "room", "cls": "dog"}, {"id": 7151, "dom": "P", "ent": "room", "cls": "elephant"}, {"id": 7152, "dom": "P", "ent": "room", "cls": "embassy"}, {"id": 7153, "dom": "P", "ent": "room", "cls": "garage"}, {"id": 7154, "dom": "P", "ent": "room", "cls": "highway"}, {"id": 7155, "dom": "P", "ent": "room", "cls": "kidney"}, {"id": 7156, "dom": "P", "ent": "room", "cls": "lamb"}, {"id": 7157, "dom": "P", "ent": "room", "cls": "male"}, {"id": 7158, "dom": "P", "ent": "room", "cls": "marijuana"}, {"id": 7159, "dom": "P", "ent": "room", "cls": "puppy"}, {"id": 7160, "dom": "P", "ent": "room", "cls": "receiver"}, {"id": 7161, "dom": "P", "ent": "room", "cls": "reed"}, {"id": 7162, "dom": "P", "ent": "room", "cls": "rifle"}, {"id": 7163, "dom": "P", "ent": "room", "cls": "road"}, {"id": 7164, "dom": "P", "ent": "room", "cls": "salmon"}, {"id": 7165, "dom": "P", "ent": "room", "cls": "shark"}, {"id": 7166, "dom": "P", "ent": "room", "cls": "sheep"}, {"id": 7167, "dom": "P", "ent": "room", "cls": "shook"}, {"id": 7168, "dom": "P", "ent": "room", "cls": "tank"}, {"id": 7169, "dom": "P", "ent": "room", "cls": "wheat"}, {"id": 7170, "dom": "P", "ent": "rotary", "cls": "architecture"}, {"id": 7171, "dom": "P", "ent": "rotary", "cls": "bacteria"}, {"id": 7172, "dom": "P", "ent": "rotary", "cls": "beef"}, {"id": 7173, "dom": "P", "ent": "rotary", "cls": "bridge"}, {"id": 7174, "dom": "P", "ent": "rotary", "cls": "bud"}, {"id": 7175, "dom": "P", "ent": "rotary", "cls": "cat"}, {"id": 7176, "dom": "P", "ent": "rotary", "cls": "chapel"}, {"id": 7177, "dom": "P", "ent": "rotary", "cls": "chick"}, {"id": 7178, "dom": "P", "ent": "rotary", "cls": "clock"}, {"id": 7179, "dom": "P", "ent": "rotary", "cls": "deer"}, {"id": 7180, "dom": "P", "ent": "rotary", "cls": "dog"}, {"id": 7181, "dom": "P", "ent": "rotary", "cls": "elephant"}, {"id": 7182, "dom": "P", "ent": "rotary", "cls": "embassy"}, {"id": 7183, "dom": "P", "ent": "rotary", "cls": "garage"}, {"id": 7184, "dom": "P", "ent": "rotary", "cls": "highway"}, {"id": 7185, "dom": "P", "ent": "rotary", "cls": "kidney"}, {"id": 7186, "dom": "P", "ent": "rotary", "cls": "lamb"}, {"id": 7187, "dom": "P", "ent": "rotary", "cls": "male"}, {"id": 7188, "dom": "P", "ent": "rotary", "cls": "marijuana"}, {"id": 7189, "dom": "P", "ent": "rotary", "cls": "puppy"}, {"id": 7190, "dom": "P", "ent": "rotary", "cls": "receiver"}, {"id": 7191, "dom": "P", "ent": "rotary", "cls": "reed"}, {"id": 7192, "dom": "P", "ent": "rotary", "cls": "rifle"}, {"id": 7193, "dom": "P", "ent": "rotary", "cls": "road"}, {"id": 7194, "dom": "P", "ent": "rotary", "cls": "salmon"}, {"id": 7195, "dom": "P", "ent": "rotary", "cls": "shark"}, {"id": 7196, "dom": "P", "ent": "rotary", "cls": "sheep"}, {"id": 7197, "dom": "P", "ent": "rotary", "cls": "shook"}, {"id": 7198, "dom": "P", "ent": "rotary", "cls": "tank"}, {"id": 7199, "dom": "P", "ent": "rotary", "cls": "wheat"}, {"id": 7200, "dom": "P", "ent": "roundabout", "cls": "architecture"}, {"id": 7201, "dom": "P", "ent": "roundabout", "cls": "bacteria"}, {"id": 7202, "dom": "P", "ent": "roundabout", "cls": "beef"}, {"id": 7203, "dom": "P", "ent": "roundabout", "cls": "bridge"}, {"id": 7204, "dom": "P", "ent": "roundabout", "cls": "bud"}, {"id": 7205, "dom": "P", "ent": "roundabout", "cls": "cat"}, {"id": 7206, "dom": "P", "ent": "roundabout", "cls": "chapel"}, {"id": 7207, "dom": "P", "ent": "roundabout", "cls": "chick"}, {"id": 7208, "dom": "P", "ent": "roundabout", "cls": "clock"}, {"id": 7209, "dom": "P", "ent": "roundabout", "cls": "deer"}, {"id": 7210, "dom": "P", "ent": "roundabout", "cls": "dog"}, {"id": 7211, "dom": "P", "ent": "roundabout", "cls": "elephant"}, {"id": 7212, "dom": "P", "ent": "roundabout", "cls": "embassy"}, {"id": 7213, "dom": "P", "ent": "roundabout", "cls": "garage"}, {"id": 7214, "dom": "P", "ent": "roundabout", "cls": "highway"}, {"id": 7215, "dom": "P", "ent": "roundabout", "cls": "kidney"}, {"id": 7216, "dom": "P", "ent": "roundabout", "cls": "lamb"}, {"id": 7217, "dom": "P", "ent": "roundabout", "cls": "male"}, {"id": 7218, "dom": "P", "ent": "roundabout", "cls": "marijuana"}, {"id": 7219, "dom": "P", "ent": "roundabout", "cls": "puppy"}, {"id": 7220, "dom": "P", "ent": "roundabout", "cls": "receiver"}, {"id": 7221, "dom": "P", "ent": "roundabout", "cls": "reed"}, {"id": 7222, "dom": "P", "ent": "roundabout", "cls": "rifle"}, {"id": 7223, "dom": "P", "ent": "roundabout", "cls": "road"}, {"id": 7224, "dom": "P", "ent": "roundabout", "cls": "salmon"}, {"id": 7225, "dom": "P", "ent": "roundabout", "cls": "shark"}, {"id": 7226, "dom": "P", "ent": "roundabout", "cls": "sheep"}, {"id": 7227, "dom": "P", "ent": "roundabout", "cls": "shook"}, {"id": 7228, "dom": "P", "ent": "roundabout", "cls": "tank"}, {"id": 7229, "dom": "P", "ent": "roundabout", "cls": "wheat"}, {"id": 7230, "dom": "P", "ent": "row", "cls": "architecture"}, {"id": 7231, "dom": "P", "ent": "row", "cls": "bacteria"}, {"id": 7232, "dom": "P", "ent": "row", "cls": "beef"}, {"id": 7233, "dom": "P", "ent": "row", "cls": "bridge"}, {"id": 7234, "dom": "P", "ent": "row", "cls": "bud"}, {"id": 7235, "dom": "P", "ent": "row", "cls": "cat"}, {"id": 7236, "dom": "P", "ent": "row", "cls": "chapel"}, {"id": 7237, "dom": "P", "ent": "row", "cls": "chick"}, {"id": 7238, "dom": "P", "ent": "row", "cls": "clock"}, {"id": 7239, "dom": "P", "ent": "row", "cls": "deer"}, {"id": 7240, "dom": "P", "ent": "row", "cls": "dog"}, {"id": 7241, "dom": "P", "ent": "row", "cls": "elephant"}, {"id": 7242, "dom": "P", "ent": "row", "cls": "embassy"}, {"id": 7243, "dom": "P", "ent": "row", "cls": "garage"}, {"id": 7244, "dom": "P", "ent": "row", "cls": "highway"}, {"id": 7245, "dom": "P", "ent": "row", "cls": "kidney"}, {"id": 7246, "dom": "P", "ent": "row", "cls": "lamb"}, {"id": 7247, "dom": "P", "ent": "row", "cls": "male"}, {"id": 7248, "dom": "P", "ent": "row", "cls": "marijuana"}, {"id": 7249, "dom": "P", "ent": "row", "cls": "puppy"}, {"id": 7250, "dom": "P", "ent": "row", "cls": "receiver"}, {"id": 7251, "dom": "P", "ent": "row", "cls": "reed"}, {"id": 7252, "dom": "P", "ent": "row", "cls": "rifle"}, {"id": 7253, "dom": "P", "ent": "row", "cls": "road"}, {"id": 7254, "dom": "P", "ent": "row", "cls": "salmon"}, {"id": 7255, "dom": "P", "ent": "row", "cls": "shark"}, {"id": 7256, "dom": "P", "ent": "row", "cls": "sheep"}, {"id": 7257, "dom": "P", "ent": "row", "cls": "shook"}, {"id": 7258, "dom": "P", "ent": "row", "cls": "tank"}, {"id": 7259, "dom": "P", "ent": "row", "cls": "wheat"}, {"id": 7260, "dom": "P", "ent": "ruin", "cls": "architecture"}, {"id": 7261, "dom": "P", "ent": "ruin", "cls": "bacteria"}, {"id": 7262, "dom": "P", "ent": "ruin", "cls": "beef"}, {"id": 7263, "dom": "P", "ent": "ruin", "cls": "bridge"}, {"id": 7264, "dom": "P", "ent": "ruin", "cls": "bud"}, {"id": 7265, "dom": "P", "ent": "ruin", "cls": "cat"}, {"id": 7266, "dom": "P", "ent": "ruin", "cls": "chapel"}, {"id": 7267, "dom": "P", "ent": "ruin", "cls": "chick"}, {"id": 7268, "dom": "P", "ent": "ruin", "cls": "clock"}, {"id": 7269, "dom": "P", "ent": "ruin", "cls": "deer"}, {"id": 7270, "dom": "P", "ent": "ruin", "cls": "dog"}, {"id": 7271, "dom": "P", "ent": "ruin", "cls": "elephant"}, {"id": 7272, "dom": "P", "ent": "ruin", "cls": "embassy"}, {"id": 7273, "dom": "P", "ent": "ruin", "cls": "garage"}, {"id": 7274, "dom": "P", "ent": "ruin", "cls": "highway"}, {"id": 7275, "dom": "P", "ent": "ruin", "cls": "kidney"}, {"id": 7276, "dom": "P", "ent": "ruin", "cls": "lamb"}, {"id": 7277, "dom": "P", "ent": "ruin", "cls": "male"}, {"id": 7278, "dom": "P", "ent": "ruin", "cls": "marijuana"}, {"id": 7279, "dom": "P", "ent": "ruin", "cls": "puppy"}, {"id": 7280, "dom": "P", "ent": "ruin", "cls": "receiver"}, {"id": 7281, "dom": "P", "ent": "ruin", "cls": "reed"}, {"id": 7282, "dom": "P", "ent": "ruin", "cls": "rifle"}, {"id": 7283, "dom": "P", "ent": "ruin", "cls": "road"}, {"id": 7284, "dom": "P", "ent": "ruin", "cls": "salmon"}, {"id": 7285, "dom": "P", "ent": "ruin", "cls": "shark"}, {"id": 7286, "dom": "P", "ent": "ruin", "cls": "sheep"}, {"id": 7287, "dom": "P", "ent": "ruin", "cls": "shook"}, {"id": 7288, "dom": "P", "ent": "ruin", "cls": "tank"}, {"id": 7289, "dom": "P", "ent": "ruin", "cls": "wheat"}, {"id": 7290, "dom": "P", "ent": "rump", "cls": "architecture"}, {"id": 7291, "dom": "P", "ent": "rump", "cls": "bacteria"}, {"id": 7292, "dom": "P", "ent": "rump", "cls": "beef"}, {"id": 7293, "dom": "P", "ent": "rump", "cls": "bridge"}, {"id": 7294, "dom": "P", "ent": "rump", "cls": "bud"}, {"id": 7295, "dom": "P", "ent": "rump", "cls": "cat"}, {"id": 7296, "dom": "P", "ent": "rump", "cls": "chapel"}, {"id": 7297, "dom": "P", "ent": "rump", "cls": "chick"}, {"id": 7298, "dom": "P", "ent": "rump", "cls": "clock"}, {"id": 7299, "dom": "P", "ent": "rump", "cls": "deer"}, {"id": 7300, "dom": "P", "ent": "rump", "cls": "dog"}, {"id": 7301, "dom": "P", "ent": "rump", "cls": "elephant"}, {"id": 7302, "dom": "P", "ent": "rump", "cls": "embassy"}, {"id": 7303, "dom": "P", "ent": "rump", "cls": "garage"}, {"id": 7304, "dom": "P", "ent": "rump", "cls": "highway"}, {"id": 7305, "dom": "P", "ent": "rump", "cls": "kidney"}, {"id": 7306, "dom": "P", "ent": "rump", "cls": "lamb"}, {"id": 7307, "dom": "P", "ent": "rump", "cls": "male"}, {"id": 7308, "dom": "P", "ent": "rump", "cls": "marijuana"}, {"id": 7309, "dom": "P", "ent": "rump", "cls": "puppy"}, {"id": 7310, "dom": "P", "ent": "rump", "cls": "receiver"}, {"id": 7311, "dom": "P", "ent": "rump", "cls": "reed"}, {"id": 7312, "dom": "P", "ent": "rump", "cls": "rifle"}, {"id": 7313, "dom": "P", "ent": "rump", "cls": "road"}, {"id": 7314, "dom": "P", "ent": "rump", "cls": "salmon"}, {"id": 7315, "dom": "P", "ent": "rump", "cls": "shark"}, {"id": 7316, "dom": "P", "ent": "rump", "cls": "sheep"}, {"id": 7317, "dom": "P", "ent": "rump", "cls": "shook"}, {"id": 7318, "dom": "P", "ent": "rump", "cls": "tank"}, {"id": 7319, "dom": "P", "ent": "rump", "cls": "wheat"}, {"id": 7320, "dom": "P", "ent": "saddle", "cls": "architecture"}, {"id": 7321, "dom": "P", "ent": "saddle", "cls": "bacteria"}, {"id": 7322, "dom": "P", "ent": "saddle", "cls": "beef"}, {"id": 7323, "dom": "P", "ent": "saddle", "cls": "bridge"}, {"id": 7324, "dom": "P", "ent": "saddle", "cls": "bud"}, {"id": 7325, "dom": "P", "ent": "saddle", "cls": "cat"}, {"id": 7326, "dom": "P", "ent": "saddle", "cls": "chapel"}, {"id": 7327, "dom": "P", "ent": "saddle", "cls": "chick"}, {"id": 7328, "dom": "P", "ent": "saddle", "cls": "clock"}, {"id": 7329, "dom": "P", "ent": "saddle", "cls": "deer"}, {"id": 7330, "dom": "P", "ent": "saddle", "cls": "dog"}, {"id": 7331, "dom": "P", "ent": "saddle", "cls": "elephant"}, {"id": 7332, "dom": "P", "ent": "saddle", "cls": "embassy"}, {"id": 7333, "dom": "P", "ent": "saddle", "cls": "garage"}, {"id": 7334, "dom": "P", "ent": "saddle", "cls": "highway"}, {"id": 7335, "dom": "P", "ent": "saddle", "cls": "kidney"}, {"id": 7336, "dom": "P", "ent": "saddle", "cls": "lamb"}, {"id": 7337, "dom": "P", "ent": "saddle", "cls": "male"}, {"id": 7338, "dom": "P", "ent": "saddle", "cls": "marijuana"}, {"id": 7339, "dom": "P", "ent": "saddle", "cls": "puppy"}, {"id": 7340, "dom": "P", "ent": "saddle", "cls": "receiver"}, {"id": 7341, "dom": "P", "ent": "saddle", "cls": "reed"}, {"id": 7342, "dom": "P", "ent": "saddle", "cls": "rifle"}, {"id": 7343, "dom": "P", "ent": "saddle", "cls": "road"}, {"id": 7344, "dom": "P", "ent": "saddle", "cls": "salmon"}, {"id": 7345, "dom": "P", "ent": "saddle", "cls": "shark"}, {"id": 7346, "dom": "P", "ent": "saddle", "cls": "sheep"}, {"id": 7347, "dom": "P", "ent": "saddle", "cls": "shook"}, {"id": 7348, "dom": "P", "ent": "saddle", "cls": "tank"}, {"id": 7349, "dom": "P", "ent": "saddle", "cls": "wheat"}, {"id": 7350, "dom": "P", "ent": "saliva", "cls": "architecture"}, {"id": 7351, "dom": "P", "ent": "saliva", "cls": "bacteria"}, {"id": 7352, "dom": "P", "ent": "saliva", "cls": "beef"}, {"id": 7353, "dom": "P", "ent": "saliva", "cls": "bridge"}, {"id": 7354, "dom": "P", "ent": "saliva", "cls": "bud"}, {"id": 7355, "dom": "P", "ent": "saliva", "cls": "cat"}, {"id": 7356, "dom": "P", "ent": "saliva", "cls": "chapel"}, {"id": 7357, "dom": "P", "ent": "saliva", "cls": "chick"}, {"id": 7358, "dom": "P", "ent": "saliva", "cls": "clock"}, {"id": 7359, "dom": "P", "ent": "saliva", "cls": "deer"}, {"id": 7360, "dom": "P", "ent": "saliva", "cls": "dog"}, {"id": 7361, "dom": "P", "ent": "saliva", "cls": "elephant"}, {"id": 7362, "dom": "P", "ent": "saliva", "cls": "embassy"}, {"id": 7363, "dom": "P", "ent": "saliva", "cls": "garage"}, {"id": 7364, "dom": "P", "ent": "saliva", "cls": "highway"}, {"id": 7365, "dom": "P", "ent": "saliva", "cls": "kidney"}, {"id": 7366, "dom": "P", "ent": "saliva", "cls": "lamb"}, {"id": 7367, "dom": "P", "ent": "saliva", "cls": "male"}, {"id": 7368, "dom": "P", "ent": "saliva", "cls": "marijuana"}, {"id": 7369, "dom": "P", "ent": "saliva", "cls": "puppy"}, {"id": 7370, "dom": "P", "ent": "saliva", "cls": "receiver"}, {"id": 7371, "dom": "P", "ent": "saliva", "cls": "reed"}, {"id": 7372, "dom": "P", "ent": "saliva", "cls": "rifle"}, {"id": 7373, "dom": "P", "ent": "saliva", "cls": "road"}, {"id": 7374, "dom": "P", "ent": "saliva", "cls": "salmon"}, {"id": 7375, "dom": "P", "ent": "saliva", "cls": "shark"}, {"id": 7376, "dom": "P", "ent": "saliva", "cls": "sheep"}, {"id": 7377, "dom": "P", "ent": "saliva", "cls": "shook"}, {"id": 7378, "dom": "P", "ent": "saliva", "cls": "tank"}, {"id": 7379, "dom": "P", "ent": "saliva", "cls": "wheat"}, {"id": 7380, "dom": "P", "ent": "sash", "cls": "architecture"}, {"id": 7381, "dom": "P", "ent": "sash", "cls": "bacteria"}, {"id": 7382, "dom": "P", "ent": "sash", "cls": "beef"}, {"id": 7383, "dom": "P", "ent": "sash", "cls": "bridge"}, {"id": 7384, "dom": "P", "ent": "sash", "cls": "bud"}, {"id": 7385, "dom": "P", "ent": "sash", "cls": "cat"}, {"id": 7386, "dom": "P", "ent": "sash", "cls": "chapel"}, {"id": 7387, "dom": "P", "ent": "sash", "cls": "chick"}, {"id": 7388, "dom": "P", "ent": "sash", "cls": "clock"}, {"id": 7389, "dom": "P", "ent": "sash", "cls": "deer"}, {"id": 7390, "dom": "P", "ent": "sash", "cls": "dog"}, {"id": 7391, "dom": "P", "ent": "sash", "cls": "elephant"}, {"id": 7392, "dom": "P", "ent": "sash", "cls": "embassy"}, {"id": 7393, "dom": "P", "ent": "sash", "cls": "garage"}, {"id": 7394, "dom": "P", "ent": "sash", "cls": "highway"}, {"id": 7395, "dom": "P", "ent": "sash", "cls": "kidney"}, {"id": 7396, "dom": "P", "ent": "sash", "cls": "lamb"}, {"id": 7397, "dom": "P", "ent": "sash", "cls": "male"}, {"id": 7398, "dom": "P", "ent": "sash", "cls": "marijuana"}, {"id": 7399, "dom": "P", "ent": "sash", "cls": "puppy"}, {"id": 7400, "dom": "P", "ent": "sash", "cls": "receiver"}, {"id": 7401, "dom": "P", "ent": "sash", "cls": "reed"}, {"id": 7402, "dom": "P", "ent": "sash", "cls": "rifle"}, {"id": 7403, "dom": "P", "ent": "sash", "cls": "road"}, {"id": 7404, "dom": "P", "ent": "sash", "cls": "salmon"}, {"id": 7405, "dom": "P", "ent": "sash", "cls": "shark"}, {"id": 7406, "dom": "P", "ent": "sash", "cls": "sheep"}, {"id": 7407, "dom": "P", "ent": "sash", "cls": "shook"}, {"id": 7408, "dom": "P", "ent": "sash", "cls": "tank"}, {"id": 7409, "dom": "P", "ent": "sash", "cls": "wheat"}, {"id": 7410, "dom": "P", "ent": "scum", "cls": "architecture"}, {"id": 7411, "dom": "P", "ent": "scum", "cls": "bacteria"}, {"id": 7412, "dom": "P", "ent": "scum", "cls": "beef"}, {"id": 7413, "dom": "P", "ent": "scum", "cls": "bridge"}, {"id": 7414, "dom": "P", "ent": "scum", "cls": "bud"}, {"id": 7415, "dom": "P", "ent": "scum", "cls": "cat"}, {"id": 7416, "dom": "P", "ent": "scum", "cls": "chapel"}, {"id": 7417, "dom": "P", "ent": "scum", "cls": "chick"}, {"id": 7418, "dom": "P", "ent": "scum", "cls": "clock"}, {"id": 7419, "dom": "P", "ent": "scum", "cls": "deer"}, {"id": 7420, "dom": "P", "ent": "scum", "cls": "dog"}, {"id": 7421, "dom": "P", "ent": "scum", "cls": "elephant"}, {"id": 7422, "dom": "P", "ent": "scum", "cls": "embassy"}, {"id": 7423, "dom": "P", "ent": "scum", "cls": "garage"}, {"id": 7424, "dom": "P", "ent": "scum", "cls": "highway"}, {"id": 7425, "dom": "P", "ent": "scum", "cls": "kidney"}, {"id": 7426, "dom": "P", "ent": "scum", "cls": "lamb"}, {"id": 7427, "dom": "P", "ent": "scum", "cls": "male"}, {"id": 7428, "dom": "P", "ent": "scum", "cls": "marijuana"}, {"id": 7429, "dom": "P", "ent": "scum", "cls": "puppy"}, {"id": 7430, "dom": "P", "ent": "scum", "cls": "receiver"}, {"id": 7431, "dom": "P", "ent": "scum", "cls": "reed"}, {"id": 7432, "dom": "P", "ent": "scum", "cls": "rifle"}, {"id": 7433, "dom": "P", "ent": "scum", "cls": "road"}, {"id": 7434, "dom": "P", "ent": "scum", "cls": "salmon"}, {"id": 7435, "dom": "P", "ent": "scum", "cls": "shark"}, {"id": 7436, "dom": "P", "ent": "scum", "cls": "sheep"}, {"id": 7437, "dom": "P", "ent": "scum", "cls": "shook"}, {"id": 7438, "dom": "P", "ent": "scum", "cls": "tank"}, {"id": 7439, "dom": "P", "ent": "scum", "cls": "wheat"}, {"id": 7440, "dom": "P", "ent": "section", "cls": "architecture"}, {"id": 7441, "dom": "P", "ent": "section", "cls": "bacteria"}, {"id": 7442, "dom": "P", "ent": "section", "cls": "beef"}, {"id": 7443, "dom": "P", "ent": "section", "cls": "bridge"}, {"id": 7444, "dom": "P", "ent": "section", "cls": "bud"}, {"id": 7445, "dom": "P", "ent": "section", "cls": "cat"}, {"id": 7446, "dom": "P", "ent": "section", "cls": "chapel"}, {"id": 7447, "dom": "P", "ent": "section", "cls": "chick"}, {"id": 7448, "dom": "P", "ent": "section", "cls": "clock"}, {"id": 7449, "dom": "P", "ent": "section", "cls": "deer"}, {"id": 7450, "dom": "P", "ent": "section", "cls": "dog"}, {"id": 7451, "dom": "P", "ent": "section", "cls": "elephant"}, {"id": 7452, "dom": "P", "ent": "section", "cls": "embassy"}, {"id": 7453, "dom": "P", "ent": "section", "cls": "garage"}, {"id": 7454, "dom": "P", "ent": "section", "cls": "highway"}, {"id": 7455, "dom": "P", "ent": "section", "cls": "kidney"}, {"id": 7456, "dom": "P", "ent": "section", "cls": "lamb"}, {"id": 7457, "dom": "P", "ent": "section", "cls": "male"}, {"id": 7458, "dom": "P", "ent": "section", "cls": "marijuana"}, {"id": 7459, "dom": "P", "ent": "section", "cls": "puppy"}, {"id": 7460, "dom": "P", "ent": "section", "cls": "receiver"}, {"id": 7461, "dom": "P", "ent": "section", "cls": "reed"}, {"id": 7462, "dom": "P", "ent": "section", "cls": "rifle"}, {"id": 7463, "dom": "P", "ent": "section", "cls": "road"}, {"id": 7464, "dom": "P", "ent": "section", "cls": "salmon"}, {"id": 7465, "dom": "P", "ent": "section", "cls": "shark"}, {"id": 7466, "dom": "P", "ent": "section", "cls": "sheep"}, {"id": 7467, "dom": "P", "ent": "section", "cls": "shook"}, {"id": 7468, "dom": "P", "ent": "section", "cls": "tank"}, {"id": 7469, "dom": "P", "ent": "section", "cls": "wheat"}, {"id": 7470, "dom": "P", "ent": "segment", "cls": "architecture"}, {"id": 7471, "dom": "P", "ent": "segment", "cls": "bacteria"}, {"id": 7472, "dom": "P", "ent": "segment", "cls": "beef"}, {"id": 7473, "dom": "P", "ent": "segment", "cls": "bridge"}, {"id": 7474, "dom": "P", "ent": "segment", "cls": "bud"}, {"id": 7475, "dom": "P", "ent": "segment", "cls": "cat"}, {"id": 7476, "dom": "P", "ent": "segment", "cls": "chapel"}, {"id": 7477, "dom": "P", "ent": "segment", "cls": "chick"}, {"id": 7478, "dom": "P", "ent": "segment", "cls": "clock"}, {"id": 7479, "dom": "P", "ent": "segment", "cls": "deer"}, {"id": 7480, "dom": "P", "ent": "segment", "cls": "dog"}, {"id": 7481, "dom": "P", "ent": "segment", "cls": "elephant"}, {"id": 7482, "dom": "P", "ent": "segment", "cls": "embassy"}, {"id": 7483, "dom": "P", "ent": "segment", "cls": "garage"}, {"id": 7484, "dom": "P", "ent": "segment", "cls": "highway"}, {"id": 7485, "dom": "P", "ent": "segment", "cls": "kidney"}, {"id": 7486, "dom": "P", "ent": "segment", "cls": "lamb"}, {"id": 7487, "dom": "P", "ent": "segment", "cls": "male"}, {"id": 7488, "dom": "P", "ent": "segment", "cls": "marijuana"}, {"id": 7489, "dom": "P", "ent": "segment", "cls": "puppy"}, {"id": 7490, "dom": "P", "ent": "segment", "cls": "receiver"}, {"id": 7491, "dom": "P", "ent": "segment", "cls": "reed"}, {"id": 7492, "dom": "P", "ent": "segment", "cls": "rifle"}, {"id": 7493, "dom": "P", "ent": "segment", "cls": "road"}, {"id": 7494, "dom": "P", "ent": "segment", "cls": "salmon"}, {"id": 7495, "dom": "P", "ent": "segment", "cls": "shark"}, {"id": 7496, "dom": "P", "ent": "segment", "cls": "sheep"}, {"id": 7497, "dom": "P", "ent": "segment", "cls": "shook"}, {"id": 7498, "dom": "P", "ent": "segment", "cls": "tank"}, {"id": 7499, "dom": "P", "ent": "segment", "cls": "wheat"}, {"id": 7500, "dom": "P", "ent": "serum", "cls": "architecture"}, {"id": 7501, "dom": "P", "ent": "serum", "cls": "bacteria"}, {"id": 7502, "dom": "P", "ent": "serum", "cls": "beef"}, {"id": 7503, "dom": "P", "ent": "serum", "cls": "bridge"}, {"id": 7504, "dom": "P", "ent": "serum", "cls": "bud"}, {"id": 7505, "dom": "P", "ent": "serum", "cls": "cat"}, {"id": 7506, "dom": "P", "ent": "serum", "cls": "chapel"}, {"id": 7507, "dom": "P", "ent": "serum", "cls": "chick"}, {"id": 7508, "dom": "P", "ent": "serum", "cls": "clock"}, {"id": 7509, "dom": "P", "ent": "serum", "cls": "deer"}, {"id": 7510, "dom": "P", "ent": "serum", "cls": "dog"}, {"id": 7511, "dom": "P", "ent": "serum", "cls": "elephant"}, {"id": 7512, "dom": "P", "ent": "serum", "cls": "embassy"}, {"id": 7513, "dom": "P", "ent": "serum", "cls": "garage"}, {"id": 7514, "dom": "P", "ent": "serum", "cls": "highway"}, {"id": 7515, "dom": "P", "ent": "serum", "cls": "kidney"}, {"id": 7516, "dom": "P", "ent": "serum", "cls": "lamb"}, {"id": 7517, "dom": "P", "ent": "serum", "cls": "male"}, {"id": 7518, "dom": "P", "ent": "serum", "cls": "marijuana"}, {"id": 7519, "dom": "P", "ent": "serum", "cls": "puppy"}, {"id": 7520, "dom": "P", "ent": "serum", "cls": "receiver"}, {"id": 7521, "dom": "P", "ent": "serum", "cls": "reed"}, {"id": 7522, "dom": "P", "ent": "serum", "cls": "rifle"}, {"id": 7523, "dom": "P", "ent": "serum", "cls": "road"}, {"id": 7524, "dom": "P", "ent": "serum", "cls": "salmon"}, {"id": 7525, "dom": "P", "ent": "serum", "cls": "shark"}, {"id": 7526, "dom": "P", "ent": "serum", "cls": "sheep"}, {"id": 7527, "dom": "P", "ent": "serum", "cls": "shook"}, {"id": 7528, "dom": "P", "ent": "serum", "cls": "tank"}, {"id": 7529, "dom": "P", "ent": "serum", "cls": "wheat"}, {"id": 7530, "dom": "P", "ent": "shaft", "cls": "architecture"}, {"id": 7531, "dom": "P", "ent": "shaft", "cls": "bacteria"}, {"id": 7532, "dom": "P", "ent": "shaft", "cls": "beef"}, {"id": 7533, "dom": "P", "ent": "shaft", "cls": "bridge"}, {"id": 7534, "dom": "P", "ent": "shaft", "cls": "bud"}, {"id": 7535, "dom": "P", "ent": "shaft", "cls": "cat"}, {"id": 7536, "dom": "P", "ent": "shaft", "cls": "chapel"}, {"id": 7537, "dom": "P", "ent": "shaft", "cls": "chick"}, {"id": 7538, "dom": "P", "ent": "shaft", "cls": "clock"}, {"id": 7539, "dom": "P", "ent": "shaft", "cls": "deer"}, {"id": 7540, "dom": "P", "ent": "shaft", "cls": "dog"}, {"id": 7541, "dom": "P", "ent": "shaft", "cls": "elephant"}, {"id": 7542, "dom": "P", "ent": "shaft", "cls": "embassy"}, {"id": 7543, "dom": "P", "ent": "shaft", "cls": "garage"}, {"id": 7544, "dom": "P", "ent": "shaft", "cls": "highway"}, {"id": 7545, "dom": "P", "ent": "shaft", "cls": "kidney"}, {"id": 7546, "dom": "P", "ent": "shaft", "cls": "lamb"}, {"id": 7547, "dom": "P", "ent": "shaft", "cls": "male"}, {"id": 7548, "dom": "P", "ent": "shaft", "cls": "marijuana"}, {"id": 7549, "dom": "P", "ent": "shaft", "cls": "puppy"}, {"id": 7550, "dom": "P", "ent": "shaft", "cls": "receiver"}, {"id": 7551, "dom": "P", "ent": "shaft", "cls": "reed"}, {"id": 7552, "dom": "P", "ent": "shaft", "cls": "rifle"}, {"id": 7553, "dom": "P", "ent": "shaft", "cls": "road"}, {"id": 7554, "dom": "P", "ent": "shaft", "cls": "salmon"}, {"id": 7555, "dom": "P", "ent": "shaft", "cls": "shark"}, {"id": 7556, "dom": "P", "ent": "shaft", "cls": "sheep"}, {"id": 7557, "dom": "P", "ent": "shaft", "cls": "shook"}, {"id": 7558, "dom": "P", "ent": "shaft", "cls": "tank"}, {"id": 7559, "dom": "P", "ent": "shaft", "cls": "wheat"}, {"id": 7560, "dom": "P", "ent": "shank", "cls": "architecture"}, {"id": 7561, "dom": "P", "ent": "shank", "cls": "bacteria"}, {"id": 7562, "dom": "P", "ent": "shank", "cls": "beef"}, {"id": 7563, "dom": "P", "ent": "shank", "cls": "bridge"}, {"id": 7564, "dom": "P", "ent": "shank", "cls": "bud"}, {"id": 7565, "dom": "P", "ent": "shank", "cls": "cat"}, {"id": 7566, "dom": "P", "ent": "shank", "cls": "chapel"}, {"id": 7567, "dom": "P", "ent": "shank", "cls": "chick"}, {"id": 7568, "dom": "P", "ent": "shank", "cls": "clock"}, {"id": 7569, "dom": "P", "ent": "shank", "cls": "deer"}, {"id": 7570, "dom": "P", "ent": "shank", "cls": "dog"}, {"id": 7571, "dom": "P", "ent": "shank", "cls": "elephant"}, {"id": 7572, "dom": "P", "ent": "shank", "cls": "embassy"}, {"id": 7573, "dom": "P", "ent": "shank", "cls": "garage"}, {"id": 7574, "dom": "P", "ent": "shank", "cls": "highway"}, {"id": 7575, "dom": "P", "ent": "shank", "cls": "kidney"}, {"id": 7576, "dom": "P", "ent": "shank", "cls": "lamb"}, {"id": 7577, "dom": "P", "ent": "shank", "cls": "male"}, {"id": 7578, "dom": "P", "ent": "shank", "cls": "marijuana"}, {"id": 7579, "dom": "P", "ent": "shank", "cls": "puppy"}, {"id": 7580, "dom": "P", "ent": "shank", "cls": "receiver"}, {"id": 7581, "dom": "P", "ent": "shank", "cls": "reed"}, {"id": 7582, "dom": "P", "ent": "shank", "cls": "rifle"}, {"id": 7583, "dom": "P", "ent": "shank", "cls": "road"}, {"id": 7584, "dom": "P", "ent": "shank", "cls": "salmon"}, {"id": 7585, "dom": "P", "ent": "shank", "cls": "shark"}, {"id": 7586, "dom": "P", "ent": "shank", "cls": "sheep"}, {"id": 7587, "dom": "P", "ent": "shank", "cls": "shook"}, {"id": 7588, "dom": "P", "ent": "shank", "cls": "tank"}, {"id": 7589, "dom": "P", "ent": "shank", "cls": "wheat"}, {"id": 7590, "dom": "P", "ent": "sheep-tick", "cls": "architecture"}, {"id": 7591, "dom": "P", "ent": "sheep-tick", "cls": "bacteria"}, {"id": 7592, "dom": "P", "ent": "sheep-tick", "cls": "beef"}, {"id": 7593, "dom": "P", "ent": "sheep-tick", "cls": "bridge"}, {"id": 7594, "dom": "P", "ent": "sheep-tick", "cls": "bud"}, {"id": 7595, "dom": "P", "ent": "sheep-tick", "cls": "cat"}, {"id": 7596, "dom": "P", "ent": "sheep-tick", "cls": "chapel"}, {"id": 7597, "dom": "P", "ent": "sheep-tick", "cls": "chick"}, {"id": 7598, "dom": "P", "ent": "sheep-tick", "cls": "clock"}, {"id": 7599, "dom": "P", "ent": "sheep-tick", "cls": "deer"}, {"id": 7600, "dom": "P", "ent": "sheep-tick", "cls": "dog"}, {"id": 7601, "dom": "P", "ent": "sheep-tick", "cls": "elephant"}, {"id": 7602, "dom": "P", "ent": "sheep-tick", "cls": "embassy"}, {"id": 7603, "dom": "P", "ent": "sheep-tick", "cls": "garage"}, {"id": 7604, "dom": "P", "ent": "sheep-tick", "cls": "highway"}, {"id": 7605, "dom": "P", "ent": "sheep-tick", "cls": "kidney"}, {"id": 7606, "dom": "P", "ent": "sheep-tick", "cls": "lamb"}, {"id": 7607, "dom": "P", "ent": "sheep-tick", "cls": "male"}, {"id": 7608, "dom": "P", "ent": "sheep-tick", "cls": "marijuana"}, {"id": 7609, "dom": "P", "ent": "sheep-tick", "cls": "puppy"}, {"id": 7610, "dom": "P", "ent": "sheep-tick", "cls": "receiver"}, {"id": 7611, "dom": "P", "ent": "sheep-tick", "cls": "reed"}, {"id": 7612, "dom": "P", "ent": "sheep-tick", "cls": "rifle"}, {"id": 7613, "dom": "P", "ent": "sheep-tick", "cls": "road"}, {"id": 7614, "dom": "P", "ent": "sheep-tick", "cls": "salmon"}, {"id": 7615, "dom": "P", "ent": "sheep-tick", "cls": "shark"}, {"id": 7616, "dom": "P", "ent": "sheep-tick", "cls": "sheep"}, {"id": 7617, "dom": "P", "ent": "sheep-tick", "cls": "shook"}, {"id": 7618, "dom": "P", "ent": "sheep-tick", "cls": "tank"}, {"id": 7619, "dom": "P", "ent": "sheep-tick", "cls": "wheat"}, {"id": 7620, "dom": "P", "ent": "shift", "cls": "architecture"}, {"id": 7621, "dom": "P", "ent": "shift", "cls": "bacteria"}, {"id": 7622, "dom": "P", "ent": "shift", "cls": "beef"}, {"id": 7623, "dom": "P", "ent": "shift", "cls": "bridge"}, {"id": 7624, "dom": "P", "ent": "shift", "cls": "bud"}, {"id": 7625, "dom": "P", "ent": "shift", "cls": "cat"}, {"id": 7626, "dom": "P", "ent": "shift", "cls": "chapel"}, {"id": 7627, "dom": "P", "ent": "shift", "cls": "chick"}, {"id": 7628, "dom": "P", "ent": "shift", "cls": "clock"}, {"id": 7629, "dom": "P", "ent": "shift", "cls": "deer"}, {"id": 7630, "dom": "P", "ent": "shift", "cls": "dog"}, {"id": 7631, "dom": "P", "ent": "shift", "cls": "elephant"}, {"id": 7632, "dom": "P", "ent": "shift", "cls": "embassy"}, {"id": 7633, "dom": "P", "ent": "shift", "cls": "garage"}, {"id": 7634, "dom": "P", "ent": "shift", "cls": "highway"}, {"id": 7635, "dom": "P", "ent": "shift", "cls": "kidney"}, {"id": 7636, "dom": "P", "ent": "shift", "cls": "lamb"}, {"id": 7637, "dom": "P", "ent": "shift", "cls": "male"}, {"id": 7638, "dom": "P", "ent": "shift", "cls": "marijuana"}, {"id": 7639, "dom": "P", "ent": "shift", "cls": "puppy"}, {"id": 7640, "dom": "P", "ent": "shift", "cls": "receiver"}, {"id": 7641, "dom": "P", "ent": "shift", "cls": "reed"}, {"id": 7642, "dom": "P", "ent": "shift", "cls": "rifle"}, {"id": 7643, "dom": "P", "ent": "shift", "cls": "road"}, {"id": 7644, "dom": "P", "ent": "shift", "cls": "salmon"}, {"id": 7645, "dom": "P", "ent": "shift", "cls": "shark"}, {"id": 7646, "dom": "P", "ent": "shift", "cls": "sheep"}, {"id": 7647, "dom": "P", "ent": "shift", "cls": "shook"}, {"id": 7648, "dom": "P", "ent": "shift", "cls": "tank"}, {"id": 7649, "dom": "P", "ent": "shift", "cls": "wheat"}, {"id": 7650, "dom": "P", "ent": "shoot", "cls": "architecture"}, {"id": 7651, "dom": "P", "ent": "shoot", "cls": "bacteria"}, {"id": 7652, "dom": "P", "ent": "shoot", "cls": "beef"}, {"id": 7653, "dom": "P", "ent": "shoot", "cls": "bridge"}, {"id": 7654, "dom": "P", "ent": "shoot", "cls": "bud"}, {"id": 7655, "dom": "P", "ent": "shoot", "cls": "cat"}, {"id": 7656, "dom": "P", "ent": "shoot", "cls": "chapel"}, {"id": 7657, "dom": "P", "ent": "shoot", "cls": "chick"}, {"id": 7658, "dom": "P", "ent": "shoot", "cls": "clock"}, {"id": 7659, "dom": "P", "ent": "shoot", "cls": "deer"}, {"id": 7660, "dom": "P", "ent": "shoot", "cls": "dog"}, {"id": 7661, "dom": "P", "ent": "shoot", "cls": "elephant"}, {"id": 7662, "dom": "P", "ent": "shoot", "cls": "embassy"}, {"id": 7663, "dom": "P", "ent": "shoot", "cls": "garage"}, {"id": 7664, "dom": "P", "ent": "shoot", "cls": "highway"}, {"id": 7665, "dom": "P", "ent": "shoot", "cls": "kidney"}, {"id": 7666, "dom": "P", "ent": "shoot", "cls": "lamb"}, {"id": 7667, "dom": "P", "ent": "shoot", "cls": "male"}, {"id": 7668, "dom": "P", "ent": "shoot", "cls": "marijuana"}, {"id": 7669, "dom": "P", "ent": "shoot", "cls": "puppy"}, {"id": 7670, "dom": "P", "ent": "shoot", "cls": "receiver"}, {"id": 7671, "dom": "P", "ent": "shoot", "cls": "reed"}, {"id": 7672, "dom": "P", "ent": "shoot", "cls": "rifle"}, {"id": 7673, "dom": "P", "ent": "shoot", "cls": "road"}, {"id": 7674, "dom": "P", "ent": "shoot", "cls": "salmon"}, {"id": 7675, "dom": "P", "ent": "shoot", "cls": "shark"}, {"id": 7676, "dom": "P", "ent": "shoot", "cls": "sheep"}, {"id": 7677, "dom": "P", "ent": "shoot", "cls": "shook"}, {"id": 7678, "dom": "P", "ent": "shoot", "cls": "tank"}, {"id": 7679, "dom": "P", "ent": "shoot", "cls": "wheat"}, {"id": 7680, "dom": "P", "ent": "short", "cls": "architecture"}, {"id": 7681, "dom": "P", "ent": "short", "cls": "bacteria"}, {"id": 7682, "dom": "P", "ent": "short", "cls": "beef"}, {"id": 7683, "dom": "P", "ent": "short", "cls": "bridge"}, {"id": 7684, "dom": "P", "ent": "short", "cls": "bud"}, {"id": 7685, "dom": "P", "ent": "short", "cls": "cat"}, {"id": 7686, "dom": "P", "ent": "short", "cls": "chapel"}, {"id": 7687, "dom": "P", "ent": "short", "cls": "chick"}, {"id": 7688, "dom": "P", "ent": "short", "cls": "clock"}, {"id": 7689, "dom": "P", "ent": "short", "cls": "deer"}, {"id": 7690, "dom": "P", "ent": "short", "cls": "dog"}, {"id": 7691, "dom": "P", "ent": "short", "cls": "elephant"}, {"id": 7692, "dom": "P", "ent": "short", "cls": "embassy"}, {"id": 7693, "dom": "P", "ent": "short", "cls": "garage"}, {"id": 7694, "dom": "P", "ent": "short", "cls": "highway"}, {"id": 7695, "dom": "P", "ent": "short", "cls": "kidney"}, {"id": 7696, "dom": "P", "ent": "short", "cls": "lamb"}, {"id": 7697, "dom": "P", "ent": "short", "cls": "male"}, {"id": 7698, "dom": "P", "ent": "short", "cls": "marijuana"}, {"id": 7699, "dom": "P", "ent": "short", "cls": "puppy"}, {"id": 7700, "dom": "P", "ent": "short", "cls": "receiver"}, {"id": 7701, "dom": "P", "ent": "short", "cls": "reed"}, {"id": 7702, "dom": "P", "ent": "short", "cls": "rifle"}, {"id": 7703, "dom": "P", "ent": "short", "cls": "road"}, {"id": 7704, "dom": "P", "ent": "short", "cls": "salmon"}, {"id": 7705, "dom": "P", "ent": "short", "cls": "shark"}, {"id": 7706, "dom": "P", "ent": "short", "cls": "sheep"}, {"id": 7707, "dom": "P", "ent": "short", "cls": "shook"}, {"id": 7708, "dom": "P", "ent": "short", "cls": "tank"}, {"id": 7709, "dom": "P", "ent": "short", "cls": "wheat"}, {"id": 7710, "dom": "P", "ent": "shoulder", "cls": "architecture"}, {"id": 7711, "dom": "P", "ent": "shoulder", "cls": "bacteria"}, {"id": 7712, "dom": "P", "ent": "shoulder", "cls": "beef"}, {"id": 7713, "dom": "P", "ent": "shoulder", "cls": "bridge"}, {"id": 7714, "dom": "P", "ent": "shoulder", "cls": "bud"}, {"id": 7715, "dom": "P", "ent": "shoulder", "cls": "cat"}, {"id": 7716, "dom": "P", "ent": "shoulder", "cls": "chapel"}, {"id": 7717, "dom": "P", "ent": "shoulder", "cls": "chick"}, {"id": 7718, "dom": "P", "ent": "shoulder", "cls": "clock"}, {"id": 7719, "dom": "P", "ent": "shoulder", "cls": "deer"}, {"id": 7720, "dom": "P", "ent": "shoulder", "cls": "dog"}, {"id": 7721, "dom": "P", "ent": "shoulder", "cls": "elephant"}, {"id": 7722, "dom": "P", "ent": "shoulder", "cls": "embassy"}, {"id": 7723, "dom": "P", "ent": "shoulder", "cls": "garage"}, {"id": 7724, "dom": "P", "ent": "shoulder", "cls": "highway"}, {"id": 7725, "dom": "P", "ent": "shoulder", "cls": "kidney"}, {"id": 7726, "dom": "P", "ent": "shoulder", "cls": "lamb"}, {"id": 7727, "dom": "P", "ent": "shoulder", "cls": "male"}, {"id": 7728, "dom": "P", "ent": "shoulder", "cls": "marijuana"}, {"id": 7729, "dom": "P", "ent": "shoulder", "cls": "puppy"}, {"id": 7730, "dom": "P", "ent": "shoulder", "cls": "receiver"}, {"id": 7731, "dom": "P", "ent": "shoulder", "cls": "reed"}, {"id": 7732, "dom": "P", "ent": "shoulder", "cls": "rifle"}, {"id": 7733, "dom": "P", "ent": "shoulder", "cls": "road"}, {"id": 7734, "dom": "P", "ent": "shoulder", "cls": "salmon"}, {"id": 7735, "dom": "P", "ent": "shoulder", "cls": "shark"}, {"id": 7736, "dom": "P", "ent": "shoulder", "cls": "sheep"}, {"id": 7737, "dom": "P", "ent": "shoulder", "cls": "shook"}, {"id": 7738, "dom": "P", "ent": "shoulder", "cls": "tank"}, {"id": 7739, "dom": "P", "ent": "shoulder", "cls": "wheat"}, {"id": 7740, "dom": "P", "ent": "sights", "cls": "architecture"}, {"id": 7741, "dom": "P", "ent": "sights", "cls": "bacteria"}, {"id": 7742, "dom": "P", "ent": "sights", "cls": "beef"}, {"id": 7743, "dom": "P", "ent": "sights", "cls": "bridge"}, {"id": 7744, "dom": "P", "ent": "sights", "cls": "bud"}, {"id": 7745, "dom": "P", "ent": "sights", "cls": "cat"}, {"id": 7746, "dom": "P", "ent": "sights", "cls": "chapel"}, {"id": 7747, "dom": "P", "ent": "sights", "cls": "chick"}, {"id": 7748, "dom": "P", "ent": "sights", "cls": "clock"}, {"id": 7749, "dom": "P", "ent": "sights", "cls": "deer"}, {"id": 7750, "dom": "P", "ent": "sights", "cls": "dog"}, {"id": 7751, "dom": "P", "ent": "sights", "cls": "elephant"}, {"id": 7752, "dom": "P", "ent": "sights", "cls": "embassy"}, {"id": 7753, "dom": "P", "ent": "sights", "cls": "garage"}, {"id": 7754, "dom": "P", "ent": "sights", "cls": "highway"}, {"id": 7755, "dom": "P", "ent": "sights", "cls": "kidney"}, {"id": 7756, "dom": "P", "ent": "sights", "cls": "lamb"}, {"id": 7757, "dom": "P", "ent": "sights", "cls": "male"}, {"id": 7758, "dom": "P", "ent": "sights", "cls": "marijuana"}, {"id": 7759, "dom": "P", "ent": "sights", "cls": "puppy"}, {"id": 7760, "dom": "P", "ent": "sights", "cls": "receiver"}, {"id": 7761, "dom": "P", "ent": "sights", "cls": "reed"}, {"id": 7762, "dom": "P", "ent": "sights", "cls": "rifle"}, {"id": 7763, "dom": "P", "ent": "sights", "cls": "road"}, {"id": 7764, "dom": "P", "ent": "sights", "cls": "salmon"}, {"id": 7765, "dom": "P", "ent": "sights", "cls": "shark"}, {"id": 7766, "dom": "P", "ent": "sights", "cls": "sheep"}, {"id": 7767, "dom": "P", "ent": "sights", "cls": "shook"}, {"id": 7768, "dom": "P", "ent": "sights", "cls": "tank"}, {"id": 7769, "dom": "P", "ent": "sights", "cls": "wheat"}, {"id": 7770, "dom": "P", "ent": "skeleton", "cls": "architecture"}, {"id": 7771, "dom": "P", "ent": "skeleton", "cls": "bacteria"}, {"id": 7772, "dom": "P", "ent": "skeleton", "cls": "beef"}, {"id": 7773, "dom": "P", "ent": "skeleton", "cls": "bridge"}, {"id": 7774, "dom": "P", "ent": "skeleton", "cls": "bud"}, {"id": 7775, "dom": "P", "ent": "skeleton", "cls": "cat"}, {"id": 7776, "dom": "P", "ent": "skeleton", "cls": "chapel"}, {"id": 7777, "dom": "P", "ent": "skeleton", "cls": "chick"}, {"id": 7778, "dom": "P", "ent": "skeleton", "cls": "clock"}, {"id": 7779, "dom": "P", "ent": "skeleton", "cls": "deer"}, {"id": 7780, "dom": "P", "ent": "skeleton", "cls": "dog"}, {"id": 7781, "dom": "P", "ent": "skeleton", "cls": "elephant"}, {"id": 7782, "dom": "P", "ent": "skeleton", "cls": "embassy"}, {"id": 7783, "dom": "P", "ent": "skeleton", "cls": "garage"}, {"id": 7784, "dom": "P", "ent": "skeleton", "cls": "highway"}, {"id": 7785, "dom": "P", "ent": "skeleton", "cls": "kidney"}, {"id": 7786, "dom": "P", "ent": "skeleton", "cls": "lamb"}, {"id": 7787, "dom": "P", "ent": "skeleton", "cls": "male"}, {"id": 7788, "dom": "P", "ent": "skeleton", "cls": "marijuana"}, {"id": 7789, "dom": "P", "ent": "skeleton", "cls": "puppy"}, {"id": 7790, "dom": "P", "ent": "skeleton", "cls": "receiver"}, {"id": 7791, "dom": "P", "ent": "skeleton", "cls": "reed"}, {"id": 7792, "dom": "P", "ent": "skeleton", "cls": "rifle"}, {"id": 7793, "dom": "P", "ent": "skeleton", "cls": "road"}, {"id": 7794, "dom": "P", "ent": "skeleton", "cls": "salmon"}, {"id": 7795, "dom": "P", "ent": "skeleton", "cls": "shark"}, {"id": 7796, "dom": "P", "ent": "skeleton", "cls": "sheep"}, {"id": 7797, "dom": "P", "ent": "skeleton", "cls": "shook"}, {"id": 7798, "dom": "P", "ent": "skeleton", "cls": "tank"}, {"id": 7799, "dom": "P", "ent": "skeleton", "cls": "wheat"}, {"id": 7800, "dom": "P", "ent": "skull", "cls": "architecture"}, {"id": 7801, "dom": "P", "ent": "skull", "cls": "bacteria"}, {"id": 7802, "dom": "P", "ent": "skull", "cls": "beef"}, {"id": 7803, "dom": "P", "ent": "skull", "cls": "bridge"}, {"id": 7804, "dom": "P", "ent": "skull", "cls": "bud"}, {"id": 7805, "dom": "P", "ent": "skull", "cls": "cat"}, {"id": 7806, "dom": "P", "ent": "skull", "cls": "chapel"}, {"id": 7807, "dom": "P", "ent": "skull", "cls": "chick"}, {"id": 7808, "dom": "P", "ent": "skull", "cls": "clock"}, {"id": 7809, "dom": "P", "ent": "skull", "cls": "deer"}, {"id": 7810, "dom": "P", "ent": "skull", "cls": "dog"}, {"id": 7811, "dom": "P", "ent": "skull", "cls": "elephant"}, {"id": 7812, "dom": "P", "ent": "skull", "cls": "embassy"}, {"id": 7813, "dom": "P", "ent": "skull", "cls": "garage"}, {"id": 7814, "dom": "P", "ent": "skull", "cls": "highway"}, {"id": 7815, "dom": "P", "ent": "skull", "cls": "kidney"}, {"id": 7816, "dom": "P", "ent": "skull", "cls": "lamb"}, {"id": 7817, "dom": "P", "ent": "skull", "cls": "male"}, {"id": 7818, "dom": "P", "ent": "skull", "cls": "marijuana"}, {"id": 7819, "dom": "P", "ent": "skull", "cls": "puppy"}, {"id": 7820, "dom": "P", "ent": "skull", "cls": "receiver"}, {"id": 7821, "dom": "P", "ent": "skull", "cls": "reed"}, {"id": 7822, "dom": "P", "ent": "skull", "cls": "rifle"}, {"id": 7823, "dom": "P", "ent": "skull", "cls": "road"}, {"id": 7824, "dom": "P", "ent": "skull", "cls": "salmon"}, {"id": 7825, "dom": "P", "ent": "skull", "cls": "shark"}, {"id": 7826, "dom": "P", "ent": "skull", "cls": "sheep"}, {"id": 7827, "dom": "P", "ent": "skull", "cls": "shook"}, {"id": 7828, "dom": "P", "ent": "skull", "cls": "tank"}, {"id": 7829, "dom": "P", "ent": "skull", "cls": "wheat"}, {"id": 7830, "dom": "P", "ent": "slide", "cls": "architecture"}, {"id": 7831, "dom": "P", "ent": "slide", "cls": "bacteria"}, {"id": 7832, "dom": "P", "ent": "slide", "cls": "beef"}, {"id": 7833, "dom": "P", "ent": "slide", "cls": "bridge"}, {"id": 7834, "dom": "P", "ent": "slide", "cls": "bud"}, {"id": 7835, "dom": "P", "ent": "slide", "cls": "cat"}, {"id": 7836, "dom": "P", "ent": "slide", "cls": "chapel"}, {"id": 7837, "dom": "P", "ent": "slide", "cls": "chick"}, {"id": 7838, "dom": "P", "ent": "slide", "cls": "clock"}, {"id": 7839, "dom": "P", "ent": "slide", "cls": "deer"}, {"id": 7840, "dom": "P", "ent": "slide", "cls": "dog"}, {"id": 7841, "dom": "P", "ent": "slide", "cls": "elephant"}, {"id": 7842, "dom": "P", "ent": "slide", "cls": "embassy"}, {"id": 7843, "dom": "P", "ent": "slide", "cls": "garage"}, {"id": 7844, "dom": "P", "ent": "slide", "cls": "highway"}, {"id": 7845, "dom": "P", "ent": "slide", "cls": "kidney"}, {"id": 7846, "dom": "P", "ent": "slide", "cls": "lamb"}, {"id": 7847, "dom": "P", "ent": "slide", "cls": "male"}, {"id": 7848, "dom": "P", "ent": "slide", "cls": "marijuana"}, {"id": 7849, "dom": "P", "ent": "slide", "cls": "puppy"}, {"id": 7850, "dom": "P", "ent": "slide", "cls": "receiver"}, {"id": 7851, "dom": "P", "ent": "slide", "cls": "reed"}, {"id": 7852, "dom": "P", "ent": "slide", "cls": "rifle"}, {"id": 7853, "dom": "P", "ent": "slide", "cls": "road"}, {"id": 7854, "dom": "P", "ent": "slide", "cls": "salmon"}, {"id": 7855, "dom": "P", "ent": "slide", "cls": "shark"}, {"id": 7856, "dom": "P", "ent": "slide", "cls": "sheep"}, {"id": 7857, "dom": "P", "ent": "slide", "cls": "shook"}, {"id": 7858, "dom": "P", "ent": "slide", "cls": "tank"}, {"id": 7859, "dom": "P", "ent": "slide", "cls": "wheat"}, {"id": 7860, "dom": "P", "ent": "smoke", "cls": "architecture"}, {"id": 7861, "dom": "P", "ent": "smoke", "cls": "bacteria"}, {"id": 7862, "dom": "P", "ent": "smoke", "cls": "beef"}, {"id": 7863, "dom": "P", "ent": "smoke", "cls": "bridge"}, {"id": 7864, "dom": "P", "ent": "smoke", "cls": "bud"}, {"id": 7865, "dom": "P", "ent": "smoke", "cls": "cat"}, {"id": 7866, "dom": "P", "ent": "smoke", "cls": "chapel"}, {"id": 7867, "dom": "P", "ent": "smoke", "cls": "chick"}, {"id": 7868, "dom": "P", "ent": "smoke", "cls": "clock"}, {"id": 7869, "dom": "P", "ent": "smoke", "cls": "deer"}, {"id": 7870, "dom": "P", "ent": "smoke", "cls": "dog"}, {"id": 7871, "dom": "P", "ent": "smoke", "cls": "elephant"}, {"id": 7872, "dom": "P", "ent": "smoke", "cls": "embassy"}, {"id": 7873, "dom": "P", "ent": "smoke", "cls": "garage"}, {"id": 7874, "dom": "P", "ent": "smoke", "cls": "highway"}, {"id": 7875, "dom": "P", "ent": "smoke", "cls": "kidney"}, {"id": 7876, "dom": "P", "ent": "smoke", "cls": "lamb"}, {"id": 7877, "dom": "P", "ent": "smoke", "cls": "male"}, {"id": 7878, "dom": "P", "ent": "smoke", "cls": "marijuana"}, {"id": 7879, "dom": "P", "ent": "smoke", "cls": "puppy"}, {"id": 7880, "dom": "P", "ent": "smoke", "cls": "receiver"}, {"id": 7881, "dom": "P", "ent": "smoke", "cls": "reed"}, {"id": 7882, "dom": "P", "ent": "smoke", "cls": "rifle"}, {"id": 7883, "dom": "P", "ent": "smoke", "cls": "road"}, {"id": 7884, "dom": "P", "ent": "smoke", "cls": "salmon"}, {"id": 7885, "dom": "P", "ent": "smoke", "cls": "shark"}, {"id": 7886, "dom": "P", "ent": "smoke", "cls": "sheep"}, {"id": 7887, "dom": "P", "ent": "smoke", "cls": "shook"}, {"id": 7888, "dom": "P", "ent": "smoke", "cls": "tank"}, {"id": 7889, "dom": "P", "ent": "smoke", "cls": "wheat"}, {"id": 7890, "dom": "P", "ent": "snout", "cls": "architecture"}, {"id": 7891, "dom": "P", "ent": "snout", "cls": "bacteria"}, {"id": 7892, "dom": "P", "ent": "snout", "cls": "beef"}, {"id": 7893, "dom": "P", "ent": "snout", "cls": "bridge"}, {"id": 7894, "dom": "P", "ent": "snout", "cls": "bud"}, {"id": 7895, "dom": "P", "ent": "snout", "cls": "cat"}, {"id": 7896, "dom": "P", "ent": "snout", "cls": "chapel"}, {"id": 7897, "dom": "P", "ent": "snout", "cls": "chick"}, {"id": 7898, "dom": "P", "ent": "snout", "cls": "clock"}, {"id": 7899, "dom": "P", "ent": "snout", "cls": "deer"}, {"id": 7900, "dom": "P", "ent": "snout", "cls": "dog"}, {"id": 7901, "dom": "P", "ent": "snout", "cls": "elephant"}, {"id": 7902, "dom": "P", "ent": "snout", "cls": "embassy"}, {"id": 7903, "dom": "P", "ent": "snout", "cls": "garage"}, {"id": 7904, "dom": "P", "ent": "snout", "cls": "highway"}, {"id": 7905, "dom": "P", "ent": "snout", "cls": "kidney"}, {"id": 7906, "dom": "P", "ent": "snout", "cls": "lamb"}, {"id": 7907, "dom": "P", "ent": "snout", "cls": "male"}, {"id": 7908, "dom": "P", "ent": "snout", "cls": "marijuana"}, {"id": 7909, "dom": "P", "ent": "snout", "cls": "puppy"}, {"id": 7910, "dom": "P", "ent": "snout", "cls": "receiver"}, {"id": 7911, "dom": "P", "ent": "snout", "cls": "reed"}, {"id": 7912, "dom": "P", "ent": "snout", "cls": "rifle"}, {"id": 7913, "dom": "P", "ent": "snout", "cls": "road"}, {"id": 7914, "dom": "P", "ent": "snout", "cls": "salmon"}, {"id": 7915, "dom": "P", "ent": "snout", "cls": "shark"}, {"id": 7916, "dom": "P", "ent": "snout", "cls": "sheep"}, {"id": 7917, "dom": "P", "ent": "snout", "cls": "shook"}, {"id": 7918, "dom": "P", "ent": "snout", "cls": "tank"}, {"id": 7919, "dom": "P", "ent": "snout", "cls": "wheat"}, {"id": 7920, "dom": "P", "ent": "soldier-fish", "cls": "architecture"}, {"id": 7921, "dom": "P", "ent": "soldier-fish", "cls": "bacteria"}, {"id": 7922, "dom": "P", "ent": "soldier-fish", "cls": "beef"}, {"id": 7923, "dom": "P", "ent": "soldier-fish", "cls": "bridge"}, {"id": 7924, "dom": "P", "ent": "soldier-fish", "cls": "bud"}, {"id": 7925, "dom": "P", "ent": "soldier-fish", "cls": "cat"}, {"id": 7926, "dom": "P", "ent": "soldier-fish", "cls": "chapel"}, {"id": 7927, "dom": "P", "ent": "soldier-fish", "cls": "chick"}, {"id": 7928, "dom": "P", "ent": "soldier-fish", "cls": "clock"}, {"id": 7929, "dom": "P", "ent": "soldier-fish", "cls": "deer"}, {"id": 7930, "dom": "P", "ent": "soldier-fish", "cls": "dog"}, {"id": 7931, "dom": "P", "ent": "soldier-fish", "cls": "elephant"}, {"id": 7932, "dom": "P", "ent": "soldier-fish", "cls": "embassy"}, {"id": 7933, "dom": "P", "ent": "soldier-fish", "cls": "garage"}, {"id": 7934, "dom": "P", "ent": "soldier-fish", "cls": "highway"}, {"id": 7935, "dom": "P", "ent": "soldier-fish", "cls": "kidney"}, {"id": 7936, "dom": "P", "ent": "soldier-fish", "cls": "lamb"}, {"id": 7937, "dom": "P", "ent": "soldier-fish", "cls": "male"}, {"id": 7938, "dom": "P", "ent": "soldier-fish", "cls": "marijuana"}, {"id": 7939, "dom": "P", "ent": "soldier-fish", "cls": "puppy"}, {"id": 7940, "dom": "P", "ent": "soldier-fish", "cls": "receiver"}, {"id": 7941, "dom": "P", "ent": "soldier-fish", "cls": "reed"}, {"id": 7942, "dom": "P", "ent": "soldier-fish", "cls": "rifle"}, {"id": 7943, "dom": "P", "ent": "soldier-fish", "cls": "road"}, {"id": 7944, "dom": "P", "ent": "soldier-fish", "cls": "salmon"}, {"id": 7945, "dom": "P", "ent": "soldier-fish", "cls": "shark"}, {"id": 7946, "dom": "P", "ent": "soldier-fish", "cls": "sheep"}, {"id": 7947, "dom": "P", "ent": "soldier-fish", "cls": "shook"}, {"id": 7948, "dom": "P", "ent": "soldier-fish", "cls": "tank"}, {"id": 7949, "dom": "P", "ent": "soldier-fish", "cls": "wheat"}, {"id": 7950, "dom": "P", "ent": "spit", "cls": "architecture"}, {"id": 7951, "dom": "P", "ent": "spit", "cls": "bacteria"}, {"id": 7952, "dom": "P", "ent": "spit", "cls": "beef"}, {"id": 7953, "dom": "P", "ent": "spit", "cls": "bridge"}, {"id": 7954, "dom": "P", "ent": "spit", "cls": "bud"}, {"id": 7955, "dom": "P", "ent": "spit", "cls": "cat"}, {"id": 7956, "dom": "P", "ent": "spit", "cls": "chapel"}, {"id": 7957, "dom": "P", "ent": "spit", "cls": "chick"}, {"id": 7958, "dom": "P", "ent": "spit", "cls": "clock"}, {"id": 7959, "dom": "P", "ent": "spit", "cls": "deer"}, {"id": 7960, "dom": "P", "ent": "spit", "cls": "dog"}, {"id": 7961, "dom": "P", "ent": "spit", "cls": "elephant"}, {"id": 7962, "dom": "P", "ent": "spit", "cls": "embassy"}, {"id": 7963, "dom": "P", "ent": "spit", "cls": "garage"}, {"id": 7964, "dom": "P", "ent": "spit", "cls": "highway"}, {"id": 7965, "dom": "P", "ent": "spit", "cls": "kidney"}, {"id": 7966, "dom": "P", "ent": "spit", "cls": "lamb"}, {"id": 7967, "dom": "P", "ent": "spit", "cls": "male"}, {"id": 7968, "dom": "P", "ent": "spit", "cls": "marijuana"}, {"id": 7969, "dom": "P", "ent": "spit", "cls": "puppy"}, {"id": 7970, "dom": "P", "ent": "spit", "cls": "receiver"}, {"id": 7971, "dom": "P", "ent": "spit", "cls": "reed"}, {"id": 7972, "dom": "P", "ent": "spit", "cls": "rifle"}, {"id": 7973, "dom": "P", "ent": "spit", "cls": "road"}, {"id": 7974, "dom": "P", "ent": "spit", "cls": "salmon"}, {"id": 7975, "dom": "P", "ent": "spit", "cls": "shark"}, {"id": 7976, "dom": "P", "ent": "spit", "cls": "sheep"}, {"id": 7977, "dom": "P", "ent": "spit", "cls": "shook"}, {"id": 7978, "dom": "P", "ent": "spit", "cls": "tank"}, {"id": 7979, "dom": "P", "ent": "spit", "cls": "wheat"}, {"id": 7980, "dom": "P", "ent": "springer", "cls": "architecture"}, {"id": 7981, "dom": "P", "ent": "springer", "cls": "bacteria"}, {"id": 7982, "dom": "P", "ent": "springer", "cls": "beef"}, {"id": 7983, "dom": "P", "ent": "springer", "cls": "bridge"}, {"id": 7984, "dom": "P", "ent": "springer", "cls": "bud"}, {"id": 7985, "dom": "P", "ent": "springer", "cls": "cat"}, {"id": 7986, "dom": "P", "ent": "springer", "cls": "chapel"}, {"id": 7987, "dom": "P", "ent": "springer", "cls": "chick"}, {"id": 7988, "dom": "P", "ent": "springer", "cls": "clock"}, {"id": 7989, "dom": "P", "ent": "springer", "cls": "deer"}, {"id": 7990, "dom": "P", "ent": "springer", "cls": "dog"}, {"id": 7991, "dom": "P", "ent": "springer", "cls": "elephant"}, {"id": 7992, "dom": "P", "ent": "springer", "cls": "embassy"}, {"id": 7993, "dom": "P", "ent": "springer", "cls": "garage"}, {"id": 7994, "dom": "P", "ent": "springer", "cls": "highway"}, {"id": 7995, "dom": "P", "ent": "springer", "cls": "kidney"}, {"id": 7996, "dom": "P", "ent": "springer", "cls": "lamb"}, {"id": 7997, "dom": "P", "ent": "springer", "cls": "male"}, {"id": 7998, "dom": "P", "ent": "springer", "cls": "marijuana"}, {"id": 7999, "dom": "P", "ent": "springer", "cls": "puppy"}, {"id": 8000, "dom": "P", "ent": "springer", "cls": "receiver"}, {"id": 8001, "dom": "P", "ent": "springer", "cls": "reed"}, {"id": 8002, "dom": "P", "ent": "springer", "cls": "rifle"}, {"id": 8003, "dom": "P", "ent": "springer", "cls": "road"}, {"id": 8004, "dom": "P", "ent": "springer", "cls": "salmon"}, {"id": 8005, "dom": "P", "ent": "springer", "cls": "shark"}, {"id": 8006, "dom": "P", "ent": "springer", "cls": "sheep"}, {"id": 8007, "dom": "P", "ent": "springer", "cls": "shook"}, {"id": 8008, "dom": "P", "ent": "springer", "cls": "tank"}, {"id": 8009, "dom": "P", "ent": "springer", "cls": "wheat"}, {"id": 8010, "dom": "P", "ent": "stair", "cls": "architecture"}, {"id": 8011, "dom": "P", "ent": "stair", "cls": "bacteria"}, {"id": 8012, "dom": "P", "ent": "stair", "cls": "beef"}, {"id": 8013, "dom": "P", "ent": "stair", "cls": "bridge"}, {"id": 8014, "dom": "P", "ent": "stair", "cls": "bud"}, {"id": 8015, "dom": "P", "ent": "stair", "cls": "cat"}, {"id": 8016, "dom": "P", "ent": "stair", "cls": "chapel"}, {"id": 8017, "dom": "P", "ent": "stair", "cls": "chick"}, {"id": 8018, "dom": "P", "ent": "stair", "cls": "clock"}, {"id": 8019, "dom": "P", "ent": "stair", "cls": "deer"}, {"id": 8020, "dom": "P", "ent": "stair", "cls": "dog"}, {"id": 8021, "dom": "P", "ent": "stair", "cls": "elephant"}, {"id": 8022, "dom": "P", "ent": "stair", "cls": "embassy"}, {"id": 8023, "dom": "P", "ent": "stair", "cls": "garage"}, {"id": 8024, "dom": "P", "ent": "stair", "cls": "highway"}, {"id": 8025, "dom": "P", "ent": "stair", "cls": "kidney"}, {"id": 8026, "dom": "P", "ent": "stair", "cls": "lamb"}, {"id": 8027, "dom": "P", "ent": "stair", "cls": "male"}, {"id": 8028, "dom": "P", "ent": "stair", "cls": "marijuana"}, {"id": 8029, "dom": "P", "ent": "stair", "cls": "puppy"}, {"id": 8030, "dom": "P", "ent": "stair", "cls": "receiver"}, {"id": 8031, "dom": "P", "ent": "stair", "cls": "reed"}, {"id": 8032, "dom": "P", "ent": "stair", "cls": "rifle"}, {"id": 8033, "dom": "P", "ent": "stair", "cls": "road"}, {"id": 8034, "dom": "P", "ent": "stair", "cls": "salmon"}, {"id": 8035, "dom": "P", "ent": "stair", "cls": "shark"}, {"id": 8036, "dom": "P", "ent": "stair", "cls": "sheep"}, {"id": 8037, "dom": "P", "ent": "stair", "cls": "shook"}, {"id": 8038, "dom": "P", "ent": "stair", "cls": "tank"}, {"id": 8039, "dom": "P", "ent": "stair", "cls": "wheat"}, {"id": 8040, "dom": "P", "ent": "staircase", "cls": "architecture"}, {"id": 8041, "dom": "P", "ent": "staircase", "cls": "bacteria"}, {"id": 8042, "dom": "P", "ent": "staircase", "cls": "beef"}, {"id": 8043, "dom": "P", "ent": "staircase", "cls": "bridge"}, {"id": 8044, "dom": "P", "ent": "staircase", "cls": "bud"}, {"id": 8045, "dom": "P", "ent": "staircase", "cls": "cat"}, {"id": 8046, "dom": "P", "ent": "staircase", "cls": "chapel"}, {"id": 8047, "dom": "P", "ent": "staircase", "cls": "chick"}, {"id": 8048, "dom": "P", "ent": "staircase", "cls": "clock"}, {"id": 8049, "dom": "P", "ent": "staircase", "cls": "deer"}, {"id": 8050, "dom": "P", "ent": "staircase", "cls": "dog"}, {"id": 8051, "dom": "P", "ent": "staircase", "cls": "elephant"}, {"id": 8052, "dom": "P", "ent": "staircase", "cls": "embassy"}, {"id": 8053, "dom": "P", "ent": "staircase", "cls": "garage"}, {"id": 8054, "dom": "P", "ent": "staircase", "cls": "highway"}, {"id": 8055, "dom": "P", "ent": "staircase", "cls": "kidney"}, {"id": 8056, "dom": "P", "ent": "staircase", "cls": "lamb"}, {"id": 8057, "dom": "P", "ent": "staircase", "cls": "male"}, {"id": 8058, "dom": "P", "ent": "staircase", "cls": "marijuana"}, {"id": 8059, "dom": "P", "ent": "staircase", "cls": "puppy"}, {"id": 8060, "dom": "P", "ent": "staircase", "cls": "receiver"}, {"id": 8061, "dom": "P", "ent": "staircase", "cls": "reed"}, {"id": 8062, "dom": "P", "ent": "staircase", "cls": "rifle"}, {"id": 8063, "dom": "P", "ent": "staircase", "cls": "road"}, {"id": 8064, "dom": "P", "ent": "staircase", "cls": "salmon"}, {"id": 8065, "dom": "P", "ent": "staircase", "cls": "shark"}, {"id": 8066, "dom": "P", "ent": "staircase", "cls": "sheep"}, {"id": 8067, "dom": "P", "ent": "staircase", "cls": "shook"}, {"id": 8068, "dom": "P", "ent": "staircase", "cls": "tank"}, {"id": 8069, "dom": "P", "ent": "staircase", "cls": "wheat"}, {"id": 8070, "dom": "P", "ent": "stairway", "cls": "architecture"}, {"id": 8071, "dom": "P", "ent": "stairway", "cls": "bacteria"}, {"id": 8072, "dom": "P", "ent": "stairway", "cls": "beef"}, {"id": 8073, "dom": "P", "ent": "stairway", "cls": "bridge"}, {"id": 8074, "dom": "P", "ent": "stairway", "cls": "bud"}, {"id": 8075, "dom": "P", "ent": "stairway", "cls": "cat"}, {"id": 8076, "dom": "P", "ent": "stairway", "cls": "chapel"}, {"id": 8077, "dom": "P", "ent": "stairway", "cls": "chick"}, {"id": 8078, "dom": "P", "ent": "stairway", "cls": "clock"}, {"id": 8079, "dom": "P", "ent": "stairway", "cls": "deer"}, {"id": 8080, "dom": "P", "ent": "stairway", "cls": "dog"}, {"id": 8081, "dom": "P", "ent": "stairway", "cls": "elephant"}, {"id": 8082, "dom": "P", "ent": "stairway", "cls": "embassy"}, {"id": 8083, "dom": "P", "ent": "stairway", "cls": "garage"}, {"id": 8084, "dom": "P", "ent": "stairway", "cls": "highway"}, {"id": 8085, "dom": "P", "ent": "stairway", "cls": "kidney"}, {"id": 8086, "dom": "P", "ent": "stairway", "cls": "lamb"}, {"id": 8087, "dom": "P", "ent": "stairway", "cls": "male"}, {"id": 8088, "dom": "P", "ent": "stairway", "cls": "marijuana"}, {"id": 8089, "dom": "P", "ent": "stairway", "cls": "puppy"}, {"id": 8090, "dom": "P", "ent": "stairway", "cls": "receiver"}, {"id": 8091, "dom": "P", "ent": "stairway", "cls": "reed"}, {"id": 8092, "dom": "P", "ent": "stairway", "cls": "rifle"}, {"id": 8093, "dom": "P", "ent": "stairway", "cls": "road"}, {"id": 8094, "dom": "P", "ent": "stairway", "cls": "salmon"}, {"id": 8095, "dom": "P", "ent": "stairway", "cls": "shark"}, {"id": 8096, "dom": "P", "ent": "stairway", "cls": "sheep"}, {"id": 8097, "dom": "P", "ent": "stairway", "cls": "shook"}, {"id": 8098, "dom": "P", "ent": "stairway", "cls": "tank"}, {"id": 8099, "dom": "P", "ent": "stairway", "cls": "wheat"}, {"id": 8100, "dom": "P", "ent": "stave", "cls": "architecture"}, {"id": 8101, "dom": "P", "ent": "stave", "cls": "bacteria"}, {"id": 8102, "dom": "P", "ent": "stave", "cls": "beef"}, {"id": 8103, "dom": "P", "ent": "stave", "cls": "bridge"}, {"id": 8104, "dom": "P", "ent": "stave", "cls": "bud"}, {"id": 8105, "dom": "P", "ent": "stave", "cls": "cat"}, {"id": 8106, "dom": "P", "ent": "stave", "cls": "chapel"}, {"id": 8107, "dom": "P", "ent": "stave", "cls": "chick"}, {"id": 8108, "dom": "P", "ent": "stave", "cls": "clock"}, {"id": 8109, "dom": "P", "ent": "stave", "cls": "deer"}, {"id": 8110, "dom": "P", "ent": "stave", "cls": "dog"}, {"id": 8111, "dom": "P", "ent": "stave", "cls": "elephant"}, {"id": 8112, "dom": "P", "ent": "stave", "cls": "embassy"}, {"id": 8113, "dom": "P", "ent": "stave", "cls": "garage"}, {"id": 8114, "dom": "P", "ent": "stave", "cls": "highway"}, {"id": 8115, "dom": "P", "ent": "stave", "cls": "kidney"}, {"id": 8116, "dom": "P", "ent": "stave", "cls": "lamb"}, {"id": 8117, "dom": "P", "ent": "stave", "cls": "male"}, {"id": 8118, "dom": "P", "ent": "stave", "cls": "marijuana"}, {"id": 8119, "dom": "P", "ent": "stave", "cls": "puppy"}, {"id": 8120, "dom": "P", "ent": "stave", "cls": "receiver"}, {"id": 8121, "dom": "P", "ent": "stave", "cls": "reed"}, {"id": 8122, "dom": "P", "ent": "stave", "cls": "rifle"}, {"id": 8123, "dom": "P", "ent": "stave", "cls": "road"}, {"id": 8124, "dom": "P", "ent": "stave", "cls": "salmon"}, {"id": 8125, "dom": "P", "ent": "stave", "cls": "shark"}, {"id": 8126, "dom": "P", "ent": "stave", "cls": "sheep"}, {"id": 8127, "dom": "P", "ent": "stave", "cls": "shook"}, {"id": 8128, "dom": "P", "ent": "stave", "cls": "tank"}, {"id": 8129, "dom": "P", "ent": "stave", "cls": "wheat"}, {"id": 8130, "dom": "P", "ent": "step", "cls": "architecture"}, {"id": 8131, "dom": "P", "ent": "step", "cls": "bacteria"}, {"id": 8132, "dom": "P", "ent": "step", "cls": "beef"}, {"id": 8133, "dom": "P", "ent": "step", "cls": "bridge"}, {"id": 8134, "dom": "P", "ent": "step", "cls": "bud"}, {"id": 8135, "dom": "P", "ent": "step", "cls": "cat"}, {"id": 8136, "dom": "P", "ent": "step", "cls": "chapel"}, {"id": 8137, "dom": "P", "ent": "step", "cls": "chick"}, {"id": 8138, "dom": "P", "ent": "step", "cls": "clock"}, {"id": 8139, "dom": "P", "ent": "step", "cls": "deer"}, {"id": 8140, "dom": "P", "ent": "step", "cls": "dog"}, {"id": 8141, "dom": "P", "ent": "step", "cls": "elephant"}, {"id": 8142, "dom": "P", "ent": "step", "cls": "embassy"}, {"id": 8143, "dom": "P", "ent": "step", "cls": "garage"}, {"id": 8144, "dom": "P", "ent": "step", "cls": "highway"}, {"id": 8145, "dom": "P", "ent": "step", "cls": "kidney"}, {"id": 8146, "dom": "P", "ent": "step", "cls": "lamb"}, {"id": 8147, "dom": "P", "ent": "step", "cls": "male"}, {"id": 8148, "dom": "P", "ent": "step", "cls": "marijuana"}, {"id": 8149, "dom": "P", "ent": "step", "cls": "puppy"}, {"id": 8150, "dom": "P", "ent": "step", "cls": "receiver"}, {"id": 8151, "dom": "P", "ent": "step", "cls": "reed"}, {"id": 8152, "dom": "P", "ent": "step", "cls": "rifle"}, {"id": 8153, "dom": "P", "ent": "step", "cls": "road"}, {"id": 8154, "dom": "P", "ent": "step", "cls": "salmon"}, {"id": 8155, "dom": "P", "ent": "step", "cls": "shark"}, {"id": 8156, "dom": "P", "ent": "step", "cls": "sheep"}, {"id": 8157, "dom": "P", "ent": "step", "cls": "shook"}, {"id": 8158, "dom": "P", "ent": "step", "cls": "tank"}, {"id": 8159, "dom": "P", "ent": "step", "cls": "wheat"}, {"id": 8160, "dom": "P", "ent": "stigma", "cls": "architecture"}, {"id": 8161, "dom": "P", "ent": "stigma", "cls": "bacteria"}, {"id": 8162, "dom": "P", "ent": "stigma", "cls": "beef"}, {"id": 8163, "dom": "P", "ent": "stigma", "cls": "bridge"}, {"id": 8164, "dom": "P", "ent": "stigma", "cls": "bud"}, {"id": 8165, "dom": "P", "ent": "stigma", "cls": "cat"}, {"id": 8166, "dom": "P", "ent": "stigma", "cls": "chapel"}, {"id": 8167, "dom": "P", "ent": "stigma", "cls": "chick"}, {"id": 8168, "dom": "P", "ent": "stigma", "cls": "clock"}, {"id": 8169, "dom": "P", "ent": "stigma", "cls": "deer"}, {"id": 8170, "dom": "P", "ent": "stigma", "cls": "dog"}, {"id": 8171, "dom": "P", "ent": "stigma", "cls": "elephant"}, {"id": 8172, "dom": "P", "ent": "stigma", "cls": "embassy"}, {"id": 8173, "dom": "P", "ent": "stigma", "cls": "garage"}, {"id": 8174, "dom": "P", "ent": "stigma", "cls": "highway"}, {"id": 8175, "dom": "P", "ent": "stigma", "cls": "kidney"}, {"id": 8176, "dom": "P", "ent": "stigma", "cls": "lamb"}, {"id": 8177, "dom": "P", "ent": "stigma", "cls": "male"}, {"id": 8178, "dom": "P", "ent": "stigma", "cls": "marijuana"}, {"id": 8179, "dom": "P", "ent": "stigma", "cls": "puppy"}, {"id": 8180, "dom": "P", "ent": "stigma", "cls": "receiver"}, {"id": 8181, "dom": "P", "ent": "stigma", "cls": "reed"}, {"id": 8182, "dom": "P", "ent": "stigma", "cls": "rifle"}, {"id": 8183, "dom": "P", "ent": "stigma", "cls": "road"}, {"id": 8184, "dom": "P", "ent": "stigma", "cls": "salmon"}, {"id": 8185, "dom": "P", "ent": "stigma", "cls": "shark"}, {"id": 8186, "dom": "P", "ent": "stigma", "cls": "sheep"}, {"id": 8187, "dom": "P", "ent": "stigma", "cls": "shook"}, {"id": 8188, "dom": "P", "ent": "stigma", "cls": "tank"}, {"id": 8189, "dom": "P", "ent": "stigma", "cls": "wheat"}, {"id": 8190, "dom": "P", "ent": "stock", "cls": "architecture"}, {"id": 8191, "dom": "P", "ent": "stock", "cls": "bacteria"}, {"id": 8192, "dom": "P", "ent": "stock", "cls": "beef"}, {"id": 8193, "dom": "P", "ent": "stock", "cls": "bridge"}, {"id": 8194, "dom": "P", "ent": "stock", "cls": "bud"}, {"id": 8195, "dom": "P", "ent": "stock", "cls": "cat"}, {"id": 8196, "dom": "P", "ent": "stock", "cls": "chapel"}, {"id": 8197, "dom": "P", "ent": "stock", "cls": "chick"}, {"id": 8198, "dom": "P", "ent": "stock", "cls": "clock"}, {"id": 8199, "dom": "P", "ent": "stock", "cls": "deer"}, {"id": 8200, "dom": "P", "ent": "stock", "cls": "dog"}, {"id": 8201, "dom": "P", "ent": "stock", "cls": "elephant"}, {"id": 8202, "dom": "P", "ent": "stock", "cls": "embassy"}, {"id": 8203, "dom": "P", "ent": "stock", "cls": "garage"}, {"id": 8204, "dom": "P", "ent": "stock", "cls": "highway"}, {"id": 8205, "dom": "P", "ent": "stock", "cls": "kidney"}, {"id": 8206, "dom": "P", "ent": "stock", "cls": "lamb"}, {"id": 8207, "dom": "P", "ent": "stock", "cls": "male"}, {"id": 8208, "dom": "P", "ent": "stock", "cls": "marijuana"}, {"id": 8209, "dom": "P", "ent": "stock", "cls": "puppy"}, {"id": 8210, "dom": "P", "ent": "stock", "cls": "receiver"}, {"id": 8211, "dom": "P", "ent": "stock", "cls": "reed"}, {"id": 8212, "dom": "P", "ent": "stock", "cls": "rifle"}, {"id": 8213, "dom": "P", "ent": "stock", "cls": "road"}, {"id": 8214, "dom": "P", "ent": "stock", "cls": "salmon"}, {"id": 8215, "dom": "P", "ent": "stock", "cls": "shark"}, {"id": 8216, "dom": "P", "ent": "stock", "cls": "sheep"}, {"id": 8217, "dom": "P", "ent": "stock", "cls": "shook"}, {"id": 8218, "dom": "P", "ent": "stock", "cls": "tank"}, {"id": 8219, "dom": "P", "ent": "stock", "cls": "wheat"}, {"id": 8220, "dom": "P", "ent": "storey", "cls": "architecture"}, {"id": 8221, "dom": "P", "ent": "storey", "cls": "bacteria"}, {"id": 8222, "dom": "P", "ent": "storey", "cls": "beef"}, {"id": 8223, "dom": "P", "ent": "storey", "cls": "bridge"}, {"id": 8224, "dom": "P", "ent": "storey", "cls": "bud"}, {"id": 8225, "dom": "P", "ent": "storey", "cls": "cat"}, {"id": 8226, "dom": "P", "ent": "storey", "cls": "chapel"}, {"id": 8227, "dom": "P", "ent": "storey", "cls": "chick"}, {"id": 8228, "dom": "P", "ent": "storey", "cls": "clock"}, {"id": 8229, "dom": "P", "ent": "storey", "cls": "deer"}, {"id": 8230, "dom": "P", "ent": "storey", "cls": "dog"}, {"id": 8231, "dom": "P", "ent": "storey", "cls": "elephant"}, {"id": 8232, "dom": "P", "ent": "storey", "cls": "embassy"}, {"id": 8233, "dom": "P", "ent": "storey", "cls": "garage"}, {"id": 8234, "dom": "P", "ent": "storey", "cls": "highway"}, {"id": 8235, "dom": "P", "ent": "storey", "cls": "kidney"}, {"id": 8236, "dom": "P", "ent": "storey", "cls": "lamb"}, {"id": 8237, "dom": "P", "ent": "storey", "cls": "male"}, {"id": 8238, "dom": "P", "ent": "storey", "cls": "marijuana"}, {"id": 8239, "dom": "P", "ent": "storey", "cls": "puppy"}, {"id": 8240, "dom": "P", "ent": "storey", "cls": "receiver"}, {"id": 8241, "dom": "P", "ent": "storey", "cls": "reed"}, {"id": 8242, "dom": "P", "ent": "storey", "cls": "rifle"}, {"id": 8243, "dom": "P", "ent": "storey", "cls": "road"}, {"id": 8244, "dom": "P", "ent": "storey", "cls": "salmon"}, {"id": 8245, "dom": "P", "ent": "storey", "cls": "shark"}, {"id": 8246, "dom": "P", "ent": "storey", "cls": "sheep"}, {"id": 8247, "dom": "P", "ent": "storey", "cls": "shook"}, {"id": 8248, "dom": "P", "ent": "storey", "cls": "tank"}, {"id": 8249, "dom": "P", "ent": "storey", "cls": "wheat"}, {"id": 8250, "dom": "P", "ent": "story", "cls": "architecture"}, {"id": 8251, "dom": "P", "ent": "story", "cls": "bacteria"}, {"id": 8252, "dom": "P", "ent": "story", "cls": "beef"}, {"id": 8253, "dom": "P", "ent": "story", "cls": "bridge"}, {"id": 8254, "dom": "P", "ent": "story", "cls": "bud"}, {"id": 8255, "dom": "P", "ent": "story", "cls": "cat"}, {"id": 8256, "dom": "P", "ent": "story", "cls": "chapel"}, {"id": 8257, "dom": "P", "ent": "story", "cls": "chick"}, {"id": 8258, "dom": "P", "ent": "story", "cls": "clock"}, {"id": 8259, "dom": "P", "ent": "story", "cls": "deer"}, {"id": 8260, "dom": "P", "ent": "story", "cls": "dog"}, {"id": 8261, "dom": "P", "ent": "story", "cls": "elephant"}, {"id": 8262, "dom": "P", "ent": "story", "cls": "embassy"}, {"id": 8263, "dom": "P", "ent": "story", "cls": "garage"}, {"id": 8264, "dom": "P", "ent": "story", "cls": "highway"}, {"id": 8265, "dom": "P", "ent": "story", "cls": "kidney"}, {"id": 8266, "dom": "P", "ent": "story", "cls": "lamb"}, {"id": 8267, "dom": "P", "ent": "story", "cls": "male"}, {"id": 8268, "dom": "P", "ent": "story", "cls": "marijuana"}, {"id": 8269, "dom": "P", "ent": "story", "cls": "puppy"}, {"id": 8270, "dom": "P", "ent": "story", "cls": "receiver"}, {"id": 8271, "dom": "P", "ent": "story", "cls": "reed"}, {"id": 8272, "dom": "P", "ent": "story", "cls": "rifle"}, {"id": 8273, "dom": "P", "ent": "story", "cls": "road"}, {"id": 8274, "dom": "P", "ent": "story", "cls": "salmon"}, {"id": 8275, "dom": "P", "ent": "story", "cls": "shark"}, {"id": 8276, "dom": "P", "ent": "story", "cls": "sheep"}, {"id": 8277, "dom": "P", "ent": "story", "cls": "shook"}, {"id": 8278, "dom": "P", "ent": "story", "cls": "tank"}, {"id": 8279, "dom": "P", "ent": "story", "cls": "wheat"}, {"id": 8280, "dom": "P", "ent": "stretcher", "cls": "architecture"}, {"id": 8281, "dom": "P", "ent": "stretcher", "cls": "bacteria"}, {"id": 8282, "dom": "P", "ent": "stretcher", "cls": "beef"}, {"id": 8283, "dom": "P", "ent": "stretcher", "cls": "bridge"}, {"id": 8284, "dom": "P", "ent": "stretcher", "cls": "bud"}, {"id": 8285, "dom": "P", "ent": "stretcher", "cls": "cat"}, {"id": 8286, "dom": "P", "ent": "stretcher", "cls": "chapel"}, {"id": 8287, "dom": "P", "ent": "stretcher", "cls": "chick"}, {"id": 8288, "dom": "P", "ent": "stretcher", "cls": "clock"}, {"id": 8289, "dom": "P", "ent": "stretcher", "cls": "deer"}, {"id": 8290, "dom": "P", "ent": "stretcher", "cls": "dog"}, {"id": 8291, "dom": "P", "ent": "stretcher", "cls": "elephant"}, {"id": 8292, "dom": "P", "ent": "stretcher", "cls": "embassy"}, {"id": 8293, "dom": "P", "ent": "stretcher", "cls": "garage"}, {"id": 8294, "dom": "P", "ent": "stretcher", "cls": "highway"}, {"id": 8295, "dom": "P", "ent": "stretcher", "cls": "kidney"}, {"id": 8296, "dom": "P", "ent": "stretcher", "cls": "lamb"}, {"id": 8297, "dom": "P", "ent": "stretcher", "cls": "male"}, {"id": 8298, "dom": "P", "ent": "stretcher", "cls": "marijuana"}, {"id": 8299, "dom": "P", "ent": "stretcher", "cls": "puppy"}, {"id": 8300, "dom": "P", "ent": "stretcher", "cls": "receiver"}, {"id": 8301, "dom": "P", "ent": "stretcher", "cls": "reed"}, {"id": 8302, "dom": "P", "ent": "stretcher", "cls": "rifle"}, {"id": 8303, "dom": "P", "ent": "stretcher", "cls": "road"}, {"id": 8304, "dom": "P", "ent": "stretcher", "cls": "salmon"}, {"id": 8305, "dom": "P", "ent": "stretcher", "cls": "shark"}, {"id": 8306, "dom": "P", "ent": "stretcher", "cls": "sheep"}, {"id": 8307, "dom": "P", "ent": "stretcher", "cls": "shook"}, {"id": 8308, "dom": "P", "ent": "stretcher", "cls": "tank"}, {"id": 8309, "dom": "P", "ent": "stretcher", "cls": "wheat"}, {"id": 8310, "dom": "P", "ent": "stud", "cls": "architecture"}, {"id": 8311, "dom": "P", "ent": "stud", "cls": "bacteria"}, {"id": 8312, "dom": "P", "ent": "stud", "cls": "beef"}, {"id": 8313, "dom": "P", "ent": "stud", "cls": "bridge"}, {"id": 8314, "dom": "P", "ent": "stud", "cls": "bud"}, {"id": 8315, "dom": "P", "ent": "stud", "cls": "cat"}, {"id": 8316, "dom": "P", "ent": "stud", "cls": "chapel"}, {"id": 8317, "dom": "P", "ent": "stud", "cls": "chick"}, {"id": 8318, "dom": "P", "ent": "stud", "cls": "clock"}, {"id": 8319, "dom": "P", "ent": "stud", "cls": "deer"}, {"id": 8320, "dom": "P", "ent": "stud", "cls": "dog"}, {"id": 8321, "dom": "P", "ent": "stud", "cls": "elephant"}, {"id": 8322, "dom": "P", "ent": "stud", "cls": "embassy"}, {"id": 8323, "dom": "P", "ent": "stud", "cls": "garage"}, {"id": 8324, "dom": "P", "ent": "stud", "cls": "highway"}, {"id": 8325, "dom": "P", "ent": "stud", "cls": "kidney"}, {"id": 8326, "dom": "P", "ent": "stud", "cls": "lamb"}, {"id": 8327, "dom": "P", "ent": "stud", "cls": "male"}, {"id": 8328, "dom": "P", "ent": "stud", "cls": "marijuana"}, {"id": 8329, "dom": "P", "ent": "stud", "cls": "puppy"}, {"id": 8330, "dom": "P", "ent": "stud", "cls": "receiver"}, {"id": 8331, "dom": "P", "ent": "stud", "cls": "reed"}, {"id": 8332, "dom": "P", "ent": "stud", "cls": "rifle"}, {"id": 8333, "dom": "P", "ent": "stud", "cls": "road"}, {"id": 8334, "dom": "P", "ent": "stud", "cls": "salmon"}, {"id": 8335, "dom": "P", "ent": "stud", "cls": "shark"}, {"id": 8336, "dom": "P", "ent": "stud", "cls": "sheep"}, {"id": 8337, "dom": "P", "ent": "stud", "cls": "shook"}, {"id": 8338, "dom": "P", "ent": "stud", "cls": "tank"}, {"id": 8339, "dom": "P", "ent": "stud", "cls": "wheat"}, {"id": 8340, "dom": "P", "ent": "style", "cls": "architecture"}, {"id": 8341, "dom": "P", "ent": "style", "cls": "bacteria"}, {"id": 8342, "dom": "P", "ent": "style", "cls": "beef"}, {"id": 8343, "dom": "P", "ent": "style", "cls": "bridge"}, {"id": 8344, "dom": "P", "ent": "style", "cls": "bud"}, {"id": 8345, "dom": "P", "ent": "style", "cls": "cat"}, {"id": 8346, "dom": "P", "ent": "style", "cls": "chapel"}, {"id": 8347, "dom": "P", "ent": "style", "cls": "chick"}, {"id": 8348, "dom": "P", "ent": "style", "cls": "clock"}, {"id": 8349, "dom": "P", "ent": "style", "cls": "deer"}, {"id": 8350, "dom": "P", "ent": "style", "cls": "dog"}, {"id": 8351, "dom": "P", "ent": "style", "cls": "elephant"}, {"id": 8352, "dom": "P", "ent": "style", "cls": "embassy"}, {"id": 8353, "dom": "P", "ent": "style", "cls": "garage"}, {"id": 8354, "dom": "P", "ent": "style", "cls": "highway"}, {"id": 8355, "dom": "P", "ent": "style", "cls": "kidney"}, {"id": 8356, "dom": "P", "ent": "style", "cls": "lamb"}, {"id": 8357, "dom": "P", "ent": "style", "cls": "male"}, {"id": 8358, "dom": "P", "ent": "style", "cls": "marijuana"}, {"id": 8359, "dom": "P", "ent": "style", "cls": "puppy"}, {"id": 8360, "dom": "P", "ent": "style", "cls": "receiver"}, {"id": 8361, "dom": "P", "ent": "style", "cls": "reed"}, {"id": 8362, "dom": "P", "ent": "style", "cls": "rifle"}, {"id": 8363, "dom": "P", "ent": "style", "cls": "road"}, {"id": 8364, "dom": "P", "ent": "style", "cls": "salmon"}, {"id": 8365, "dom": "P", "ent": "style", "cls": "shark"}, {"id": 8366, "dom": "P", "ent": "style", "cls": "sheep"}, {"id": 8367, "dom": "P", "ent": "style", "cls": "shook"}, {"id": 8368, "dom": "P", "ent": "style", "cls": "tank"}, {"id": 8369, "dom": "P", "ent": "style", "cls": "wheat"}, {"id": 8370, "dom": "P", "ent": "sucker", "cls": "architecture"}, {"id": 8371, "dom": "P", "ent": "sucker", "cls": "bacteria"}, {"id": 8372, "dom": "P", "ent": "sucker", "cls": "beef"}, {"id": 8373, "dom": "P", "ent": "sucker", "cls": "bridge"}, {"id": 8374, "dom": "P", "ent": "sucker", "cls": "bud"}, {"id": 8375, "dom": "P", "ent": "sucker", "cls": "cat"}, {"id": 8376, "dom": "P", "ent": "sucker", "cls": "chapel"}, {"id": 8377, "dom": "P", "ent": "sucker", "cls": "chick"}, {"id": 8378, "dom": "P", "ent": "sucker", "cls": "clock"}, {"id": 8379, "dom": "P", "ent": "sucker", "cls": "deer"}, {"id": 8380, "dom": "P", "ent": "sucker", "cls": "dog"}, {"id": 8381, "dom": "P", "ent": "sucker", "cls": "elephant"}, {"id": 8382, "dom": "P", "ent": "sucker", "cls": "embassy"}, {"id": 8383, "dom": "P", "ent": "sucker", "cls": "garage"}, {"id": 8384, "dom": "P", "ent": "sucker", "cls": "highway"}, {"id": 8385, "dom": "P", "ent": "sucker", "cls": "kidney"}, {"id": 8386, "dom": "P", "ent": "sucker", "cls": "lamb"}, {"id": 8387, "dom": "P", "ent": "sucker", "cls": "male"}, {"id": 8388, "dom": "P", "ent": "sucker", "cls": "marijuana"}, {"id": 8389, "dom": "P", "ent": "sucker", "cls": "puppy"}, {"id": 8390, "dom": "P", "ent": "sucker", "cls": "receiver"}, {"id": 8391, "dom": "P", "ent": "sucker", "cls": "reed"}, {"id": 8392, "dom": "P", "ent": "sucker", "cls": "rifle"}, {"id": 8393, "dom": "P", "ent": "sucker", "cls": "road"}, {"id": 8394, "dom": "P", "ent": "sucker", "cls": "salmon"}, {"id": 8395, "dom": "P", "ent": "sucker", "cls": "shark"}, {"id": 8396, "dom": "P", "ent": "sucker", "cls": "sheep"}, {"id": 8397, "dom": "P", "ent": "sucker", "cls": "shook"}, {"id": 8398, "dom": "P", "ent": "sucker", "cls": "tank"}, {"id": 8399, "dom": "P", "ent": "sucker", "cls": "wheat"}, {"id": 8400, "dom": "P", "ent": "sushi", "cls": "architecture"}, {"id": 8401, "dom": "P", "ent": "sushi", "cls": "bacteria"}, {"id": 8402, "dom": "P", "ent": "sushi", "cls": "beef"}, {"id": 8403, "dom": "P", "ent": "sushi", "cls": "bridge"}, {"id": 8404, "dom": "P", "ent": "sushi", "cls": "bud"}, {"id": 8405, "dom": "P", "ent": "sushi", "cls": "cat"}, {"id": 8406, "dom": "P", "ent": "sushi", "cls": "chapel"}, {"id": 8407, "dom": "P", "ent": "sushi", "cls": "chick"}, {"id": 8408, "dom": "P", "ent": "sushi", "cls": "clock"}, {"id": 8409, "dom": "P", "ent": "sushi", "cls": "deer"}, {"id": 8410, "dom": "P", "ent": "sushi", "cls": "dog"}, {"id": 8411, "dom": "P", "ent": "sushi", "cls": "elephant"}, {"id": 8412, "dom": "P", "ent": "sushi", "cls": "embassy"}, {"id": 8413, "dom": "P", "ent": "sushi", "cls": "garage"}, {"id": 8414, "dom": "P", "ent": "sushi", "cls": "highway"}, {"id": 8415, "dom": "P", "ent": "sushi", "cls": "kidney"}, {"id": 8416, "dom": "P", "ent": "sushi", "cls": "lamb"}, {"id": 8417, "dom": "P", "ent": "sushi", "cls": "male"}, {"id": 8418, "dom": "P", "ent": "sushi", "cls": "marijuana"}, {"id": 8419, "dom": "P", "ent": "sushi", "cls": "puppy"}, {"id": 8420, "dom": "P", "ent": "sushi", "cls": "receiver"}, {"id": 8421, "dom": "P", "ent": "sushi", "cls": "reed"}, {"id": 8422, "dom": "P", "ent": "sushi", "cls": "rifle"}, {"id": 8423, "dom": "P", "ent": "sushi", "cls": "road"}, {"id": 8424, "dom": "P", "ent": "sushi", "cls": "salmon"}, {"id": 8425, "dom": "P", "ent": "sushi", "cls": "shark"}, {"id": 8426, "dom": "P", "ent": "sushi", "cls": "sheep"}, {"id": 8427, "dom": "P", "ent": "sushi", "cls": "shook"}, {"id": 8428, "dom": "P", "ent": "sushi", "cls": "tank"}, {"id": 8429, "dom": "P", "ent": "sushi", "cls": "wheat"}, {"id": 8430, "dom": "P", "ent": "tail", "cls": "architecture"}, {"id": 8431, "dom": "P", "ent": "tail", "cls": "bacteria"}, {"id": 8432, "dom": "P", "ent": "tail", "cls": "beef"}, {"id": 8433, "dom": "P", "ent": "tail", "cls": "bridge"}, {"id": 8434, "dom": "P", "ent": "tail", "cls": "bud"}, {"id": 8435, "dom": "P", "ent": "tail", "cls": "cat"}, {"id": 8436, "dom": "P", "ent": "tail", "cls": "chapel"}, {"id": 8437, "dom": "P", "ent": "tail", "cls": "chick"}, {"id": 8438, "dom": "P", "ent": "tail", "cls": "clock"}, {"id": 8439, "dom": "P", "ent": "tail", "cls": "deer"}, {"id": 8440, "dom": "P", "ent": "tail", "cls": "dog"}, {"id": 8441, "dom": "P", "ent": "tail", "cls": "elephant"}, {"id": 8442, "dom": "P", "ent": "tail", "cls": "embassy"}, {"id": 8443, "dom": "P", "ent": "tail", "cls": "garage"}, {"id": 8444, "dom": "P", "ent": "tail", "cls": "highway"}, {"id": 8445, "dom": "P", "ent": "tail", "cls": "kidney"}, {"id": 8446, "dom": "P", "ent": "tail", "cls": "lamb"}, {"id": 8447, "dom": "P", "ent": "tail", "cls": "male"}, {"id": 8448, "dom": "P", "ent": "tail", "cls": "marijuana"}, {"id": 8449, "dom": "P", "ent": "tail", "cls": "puppy"}, {"id": 8450, "dom": "P", "ent": "tail", "cls": "receiver"}, {"id": 8451, "dom": "P", "ent": "tail", "cls": "reed"}, {"id": 8452, "dom": "P", "ent": "tail", "cls": "rifle"}, {"id": 8453, "dom": "P", "ent": "tail", "cls": "road"}, {"id": 8454, "dom": "P", "ent": "tail", "cls": "salmon"}, {"id": 8455, "dom": "P", "ent": "tail", "cls": "shark"}, {"id": 8456, "dom": "P", "ent": "tail", "cls": "sheep"}, {"id": 8457, "dom": "P", "ent": "tail", "cls": "shook"}, {"id": 8458, "dom": "P", "ent": "tail", "cls": "tank"}, {"id": 8459, "dom": "P", "ent": "tail", "cls": "wheat"}, {"id": 8460, "dom": "P", "ent": "tandem", "cls": "architecture"}, {"id": 8461, "dom": "P", "ent": "tandem", "cls": "bacteria"}, {"id": 8462, "dom": "P", "ent": "tandem", "cls": "beef"}, {"id": 8463, "dom": "P", "ent": "tandem", "cls": "bridge"}, {"id": 8464, "dom": "P", "ent": "tandem", "cls": "bud"}, {"id": 8465, "dom": "P", "ent": "tandem", "cls": "cat"}, {"id": 8466, "dom": "P", "ent": "tandem", "cls": "chapel"}, {"id": 8467, "dom": "P", "ent": "tandem", "cls": "chick"}, {"id": 8468, "dom": "P", "ent": "tandem", "cls": "clock"}, {"id": 8469, "dom": "P", "ent": "tandem", "cls": "deer"}, {"id": 8470, "dom": "P", "ent": "tandem", "cls": "dog"}, {"id": 8471, "dom": "P", "ent": "tandem", "cls": "elephant"}, {"id": 8472, "dom": "P", "ent": "tandem", "cls": "embassy"}, {"id": 8473, "dom": "P", "ent": "tandem", "cls": "garage"}, {"id": 8474, "dom": "P", "ent": "tandem", "cls": "highway"}, {"id": 8475, "dom": "P", "ent": "tandem", "cls": "kidney"}, {"id": 8476, "dom": "P", "ent": "tandem", "cls": "lamb"}, {"id": 8477, "dom": "P", "ent": "tandem", "cls": "male"}, {"id": 8478, "dom": "P", "ent": "tandem", "cls": "marijuana"}, {"id": 8479, "dom": "P", "ent": "tandem", "cls": "puppy"}, {"id": 8480, "dom": "P", "ent": "tandem", "cls": "receiver"}, {"id": 8481, "dom": "P", "ent": "tandem", "cls": "reed"}, {"id": 8482, "dom": "P", "ent": "tandem", "cls": "rifle"}, {"id": 8483, "dom": "P", "ent": "tandem", "cls": "road"}, {"id": 8484, "dom": "P", "ent": "tandem", "cls": "salmon"}, {"id": 8485, "dom": "P", "ent": "tandem", "cls": "shark"}, {"id": 8486, "dom": "P", "ent": "tandem", "cls": "sheep"}, {"id": 8487, "dom": "P", "ent": "tandem", "cls": "shook"}, {"id": 8488, "dom": "P", "ent": "tandem", "cls": "tank"}, {"id": 8489, "dom": "P", "ent": "tandem", "cls": "wheat"}, {"id": 8490, "dom": "P", "ent": "tap", "cls": "architecture"}, {"id": 8491, "dom": "P", "ent": "tap", "cls": "bacteria"}, {"id": 8492, "dom": "P", "ent": "tap", "cls": "beef"}, {"id": 8493, "dom": "P", "ent": "tap", "cls": "bridge"}, {"id": 8494, "dom": "P", "ent": "tap", "cls": "bud"}, {"id": 8495, "dom": "P", "ent": "tap", "cls": "cat"}, {"id": 8496, "dom": "P", "ent": "tap", "cls": "chapel"}, {"id": 8497, "dom": "P", "ent": "tap", "cls": "chick"}, {"id": 8498, "dom": "P", "ent": "tap", "cls": "clock"}, {"id": 8499, "dom": "P", "ent": "tap", "cls": "deer"}, {"id": 8500, "dom": "P", "ent": "tap", "cls": "dog"}, {"id": 8501, "dom": "P", "ent": "tap", "cls": "elephant"}, {"id": 8502, "dom": "P", "ent": "tap", "cls": "embassy"}, {"id": 8503, "dom": "P", "ent": "tap", "cls": "garage"}, {"id": 8504, "dom": "P", "ent": "tap", "cls": "highway"}, {"id": 8505, "dom": "P", "ent": "tap", "cls": "kidney"}, {"id": 8506, "dom": "P", "ent": "tap", "cls": "lamb"}, {"id": 8507, "dom": "P", "ent": "tap", "cls": "male"}, {"id": 8508, "dom": "P", "ent": "tap", "cls": "marijuana"}, {"id": 8509, "dom": "P", "ent": "tap", "cls": "puppy"}, {"id": 8510, "dom": "P", "ent": "tap", "cls": "receiver"}, {"id": 8511, "dom": "P", "ent": "tap", "cls": "reed"}, {"id": 8512, "dom": "P", "ent": "tap", "cls": "rifle"}, {"id": 8513, "dom": "P", "ent": "tap", "cls": "road"}, {"id": 8514, "dom": "P", "ent": "tap", "cls": "salmon"}, {"id": 8515, "dom": "P", "ent": "tap", "cls": "shark"}, {"id": 8516, "dom": "P", "ent": "tap", "cls": "sheep"}, {"id": 8517, "dom": "P", "ent": "tap", "cls": "shook"}, {"id": 8518, "dom": "P", "ent": "tap", "cls": "tank"}, {"id": 8519, "dom": "P", "ent": "tap", "cls": "wheat"}, {"id": 8520, "dom": "P", "ent": "target", "cls": "architecture"}, {"id": 8521, "dom": "P", "ent": "target", "cls": "bacteria"}, {"id": 8522, "dom": "P", "ent": "target", "cls": "beef"}, {"id": 8523, "dom": "P", "ent": "target", "cls": "bridge"}, {"id": 8524, "dom": "P", "ent": "target", "cls": "bud"}, {"id": 8525, "dom": "P", "ent": "target", "cls": "cat"}, {"id": 8526, "dom": "P", "ent": "target", "cls": "chapel"}, {"id": 8527, "dom": "P", "ent": "target", "cls": "chick"}, {"id": 8528, "dom": "P", "ent": "target", "cls": "clock"}, {"id": 8529, "dom": "P", "ent": "target", "cls": "deer"}, {"id": 8530, "dom": "P", "ent": "target", "cls": "dog"}, {"id": 8531, "dom": "P", "ent": "target", "cls": "elephant"}, {"id": 8532, "dom": "P", "ent": "target", "cls": "embassy"}, {"id": 8533, "dom": "P", "ent": "target", "cls": "garage"}, {"id": 8534, "dom": "P", "ent": "target", "cls": "highway"}, {"id": 8535, "dom": "P", "ent": "target", "cls": "kidney"}, {"id": 8536, "dom": "P", "ent": "target", "cls": "lamb"}, {"id": 8537, "dom": "P", "ent": "target", "cls": "male"}, {"id": 8538, "dom": "P", "ent": "target", "cls": "marijuana"}, {"id": 8539, "dom": "P", "ent": "target", "cls": "puppy"}, {"id": 8540, "dom": "P", "ent": "target", "cls": "receiver"}, {"id": 8541, "dom": "P", "ent": "target", "cls": "reed"}, {"id": 8542, "dom": "P", "ent": "target", "cls": "rifle"}, {"id": 8543, "dom": "P", "ent": "target", "cls": "road"}, {"id": 8544, "dom": "P", "ent": "target", "cls": "salmon"}, {"id": 8545, "dom": "P", "ent": "target", "cls": "shark"}, {"id": 8546, "dom": "P", "ent": "target", "cls": "sheep"}, {"id": 8547, "dom": "P", "ent": "target", "cls": "shook"}, {"id": 8548, "dom": "P", "ent": "target", "cls": "tank"}, {"id": 8549, "dom": "P", "ent": "target", "cls": "wheat"}, {"id": 8550, "dom": "P", "ent": "teeth", "cls": "architecture"}, {"id": 8551, "dom": "P", "ent": "teeth", "cls": "bacteria"}, {"id": 8552, "dom": "P", "ent": "teeth", "cls": "beef"}, {"id": 8553, "dom": "P", "ent": "teeth", "cls": "bridge"}, {"id": 8554, "dom": "P", "ent": "teeth", "cls": "bud"}, {"id": 8555, "dom": "P", "ent": "teeth", "cls": "cat"}, {"id": 8556, "dom": "P", "ent": "teeth", "cls": "chapel"}, {"id": 8557, "dom": "P", "ent": "teeth", "cls": "chick"}, {"id": 8558, "dom": "P", "ent": "teeth", "cls": "clock"}, {"id": 8559, "dom": "P", "ent": "teeth", "cls": "deer"}, {"id": 8560, "dom": "P", "ent": "teeth", "cls": "dog"}, {"id": 8561, "dom": "P", "ent": "teeth", "cls": "elephant"}, {"id": 8562, "dom": "P", "ent": "teeth", "cls": "embassy"}, {"id": 8563, "dom": "P", "ent": "teeth", "cls": "garage"}, {"id": 8564, "dom": "P", "ent": "teeth", "cls": "highway"}, {"id": 8565, "dom": "P", "ent": "teeth", "cls": "kidney"}, {"id": 8566, "dom": "P", "ent": "teeth", "cls": "lamb"}, {"id": 8567, "dom": "P", "ent": "teeth", "cls": "male"}, {"id": 8568, "dom": "P", "ent": "teeth", "cls": "marijuana"}, {"id": 8569, "dom": "P", "ent": "teeth", "cls": "puppy"}, {"id": 8570, "dom": "P", "ent": "teeth", "cls": "receiver"}, {"id": 8571, "dom": "P", "ent": "teeth", "cls": "reed"}, {"id": 8572, "dom": "P", "ent": "teeth", "cls": "rifle"}, {"id": 8573, "dom": "P", "ent": "teeth", "cls": "road"}, {"id": 8574, "dom": "P", "ent": "teeth", "cls": "salmon"}, {"id": 8575, "dom": "P", "ent": "teeth", "cls": "shark"}, {"id": 8576, "dom": "P", "ent": "teeth", "cls": "sheep"}, {"id": 8577, "dom": "P", "ent": "teeth", "cls": "shook"}, {"id": 8578, "dom": "P", "ent": "teeth", "cls": "tank"}, {"id": 8579, "dom": "P", "ent": "teeth", "cls": "wheat"}, {"id": 8580, "dom": "P", "ent": "temple", "cls": "architecture"}, {"id": 8581, "dom": "P", "ent": "temple", "cls": "bacteria"}, {"id": 8582, "dom": "P", "ent": "temple", "cls": "beef"}, {"id": 8583, "dom": "P", "ent": "temple", "cls": "bridge"}, {"id": 8584, "dom": "P", "ent": "temple", "cls": "bud"}, {"id": 8585, "dom": "P", "ent": "temple", "cls": "cat"}, {"id": 8586, "dom": "P", "ent": "temple", "cls": "chapel"}, {"id": 8587, "dom": "P", "ent": "temple", "cls": "chick"}, {"id": 8588, "dom": "P", "ent": "temple", "cls": "clock"}, {"id": 8589, "dom": "P", "ent": "temple", "cls": "deer"}, {"id": 8590, "dom": "P", "ent": "temple", "cls": "dog"}, {"id": 8591, "dom": "P", "ent": "temple", "cls": "elephant"}, {"id": 8592, "dom": "P", "ent": "temple", "cls": "embassy"}, {"id": 8593, "dom": "P", "ent": "temple", "cls": "garage"}, {"id": 8594, "dom": "P", "ent": "temple", "cls": "highway"}, {"id": 8595, "dom": "P", "ent": "temple", "cls": "kidney"}, {"id": 8596, "dom": "P", "ent": "temple", "cls": "lamb"}, {"id": 8597, "dom": "P", "ent": "temple", "cls": "male"}, {"id": 8598, "dom": "P", "ent": "temple", "cls": "marijuana"}, {"id": 8599, "dom": "P", "ent": "temple", "cls": "puppy"}, {"id": 8600, "dom": "P", "ent": "temple", "cls": "receiver"}, {"id": 8601, "dom": "P", "ent": "temple", "cls": "reed"}, {"id": 8602, "dom": "P", "ent": "temple", "cls": "rifle"}, {"id": 8603, "dom": "P", "ent": "temple", "cls": "road"}, {"id": 8604, "dom": "P", "ent": "temple", "cls": "salmon"}, {"id": 8605, "dom": "P", "ent": "temple", "cls": "shark"}, {"id": 8606, "dom": "P", "ent": "temple", "cls": "sheep"}, {"id": 8607, "dom": "P", "ent": "temple", "cls": "shook"}, {"id": 8608, "dom": "P", "ent": "temple", "cls": "tank"}, {"id": 8609, "dom": "P", "ent": "temple", "cls": "wheat"}, {"id": 8610, "dom": "P", "ent": "thick-knee", "cls": "architecture"}, {"id": 8611, "dom": "P", "ent": "thick-knee", "cls": "bacteria"}, {"id": 8612, "dom": "P", "ent": "thick-knee", "cls": "beef"}, {"id": 8613, "dom": "P", "ent": "thick-knee", "cls": "bridge"}, {"id": 8614, "dom": "P", "ent": "thick-knee", "cls": "bud"}, {"id": 8615, "dom": "P", "ent": "thick-knee", "cls": "cat"}, {"id": 8616, "dom": "P", "ent": "thick-knee", "cls": "chapel"}, {"id": 8617, "dom": "P", "ent": "thick-knee", "cls": "chick"}, {"id": 8618, "dom": "P", "ent": "thick-knee", "cls": "clock"}, {"id": 8619, "dom": "P", "ent": "thick-knee", "cls": "deer"}, {"id": 8620, "dom": "P", "ent": "thick-knee", "cls": "dog"}, {"id": 8621, "dom": "P", "ent": "thick-knee", "cls": "elephant"}, {"id": 8622, "dom": "P", "ent": "thick-knee", "cls": "embassy"}, {"id": 8623, "dom": "P", "ent": "thick-knee", "cls": "garage"}, {"id": 8624, "dom": "P", "ent": "thick-knee", "cls": "highway"}, {"id": 8625, "dom": "P", "ent": "thick-knee", "cls": "kidney"}, {"id": 8626, "dom": "P", "ent": "thick-knee", "cls": "lamb"}, {"id": 8627, "dom": "P", "ent": "thick-knee", "cls": "male"}, {"id": 8628, "dom": "P", "ent": "thick-knee", "cls": "marijuana"}, {"id": 8629, "dom": "P", "ent": "thick-knee", "cls": "puppy"}, {"id": 8630, "dom": "P", "ent": "thick-knee", "cls": "receiver"}, {"id": 8631, "dom": "P", "ent": "thick-knee", "cls": "reed"}, {"id": 8632, "dom": "P", "ent": "thick-knee", "cls": "rifle"}, {"id": 8633, "dom": "P", "ent": "thick-knee", "cls": "road"}, {"id": 8634, "dom": "P", "ent": "thick-knee", "cls": "salmon"}, {"id": 8635, "dom": "P", "ent": "thick-knee", "cls": "shark"}, {"id": 8636, "dom": "P", "ent": "thick-knee", "cls": "sheep"}, {"id": 8637, "dom": "P", "ent": "thick-knee", "cls": "shook"}, {"id": 8638, "dom": "P", "ent": "thick-knee", "cls": "tank"}, {"id": 8639, "dom": "P", "ent": "thick-knee", "cls": "wheat"}, {"id": 8640, "dom": "P", "ent": "threshold", "cls": "architecture"}, {"id": 8641, "dom": "P", "ent": "threshold", "cls": "bacteria"}, {"id": 8642, "dom": "P", "ent": "threshold", "cls": "beef"}, {"id": 8643, "dom": "P", "ent": "threshold", "cls": "bridge"}, {"id": 8644, "dom": "P", "ent": "threshold", "cls": "bud"}, {"id": 8645, "dom": "P", "ent": "threshold", "cls": "cat"}, {"id": 8646, "dom": "P", "ent": "threshold", "cls": "chapel"}, {"id": 8647, "dom": "P", "ent": "threshold", "cls": "chick"}, {"id": 8648, "dom": "P", "ent": "threshold", "cls": "clock"}, {"id": 8649, "dom": "P", "ent": "threshold", "cls": "deer"}, {"id": 8650, "dom": "P", "ent": "threshold", "cls": "dog"}, {"id": 8651, "dom": "P", "ent": "threshold", "cls": "elephant"}, {"id": 8652, "dom": "P", "ent": "threshold", "cls": "embassy"}, {"id": 8653, "dom": "P", "ent": "threshold", "cls": "garage"}, {"id": 8654, "dom": "P", "ent": "threshold", "cls": "highway"}, {"id": 8655, "dom": "P", "ent": "threshold", "cls": "kidney"}, {"id": 8656, "dom": "P", "ent": "threshold", "cls": "lamb"}, {"id": 8657, "dom": "P", "ent": "threshold", "cls": "male"}, {"id": 8658, "dom": "P", "ent": "threshold", "cls": "marijuana"}, {"id": 8659, "dom": "P", "ent": "threshold", "cls": "puppy"}, {"id": 8660, "dom": "P", "ent": "threshold", "cls": "receiver"}, {"id": 8661, "dom": "P", "ent": "threshold", "cls": "reed"}, {"id": 8662, "dom": "P", "ent": "threshold", "cls": "rifle"}, {"id": 8663, "dom": "P", "ent": "threshold", "cls": "road"}, {"id": 8664, "dom": "P", "ent": "threshold", "cls": "salmon"}, {"id": 8665, "dom": "P", "ent": "threshold", "cls": "shark"}, {"id": 8666, "dom": "P", "ent": "threshold", "cls": "sheep"}, {"id": 8667, "dom": "P", "ent": "threshold", "cls": "shook"}, {"id": 8668, "dom": "P", "ent": "threshold", "cls": "tank"}, {"id": 8669, "dom": "P", "ent": "threshold", "cls": "wheat"}, {"id": 8670, "dom": "P", "ent": "time-ball", "cls": "architecture"}, {"id": 8671, "dom": "P", "ent": "time-ball", "cls": "bacteria"}, {"id": 8672, "dom": "P", "ent": "time-ball", "cls": "beef"}, {"id": 8673, "dom": "P", "ent": "time-ball", "cls": "bridge"}, {"id": 8674, "dom": "P", "ent": "time-ball", "cls": "bud"}, {"id": 8675, "dom": "P", "ent": "time-ball", "cls": "cat"}, {"id": 8676, "dom": "P", "ent": "time-ball", "cls": "chapel"}, {"id": 8677, "dom": "P", "ent": "time-ball", "cls": "chick"}, {"id": 8678, "dom": "P", "ent": "time-ball", "cls": "clock"}, {"id": 8679, "dom": "P", "ent": "time-ball", "cls": "deer"}, {"id": 8680, "dom": "P", "ent": "time-ball", "cls": "dog"}, {"id": 8681, "dom": "P", "ent": "time-ball", "cls": "elephant"}, {"id": 8682, "dom": "P", "ent": "time-ball", "cls": "embassy"}, {"id": 8683, "dom": "P", "ent": "time-ball", "cls": "garage"}, {"id": 8684, "dom": "P", "ent": "time-ball", "cls": "highway"}, {"id": 8685, "dom": "P", "ent": "time-ball", "cls": "kidney"}, {"id": 8686, "dom": "P", "ent": "time-ball", "cls": "lamb"}, {"id": 8687, "dom": "P", "ent": "time-ball", "cls": "male"}, {"id": 8688, "dom": "P", "ent": "time-ball", "cls": "marijuana"}, {"id": 8689, "dom": "P", "ent": "time-ball", "cls": "puppy"}, {"id": 8690, "dom": "P", "ent": "time-ball", "cls": "receiver"}, {"id": 8691, "dom": "P", "ent": "time-ball", "cls": "reed"}, {"id": 8692, "dom": "P", "ent": "time-ball", "cls": "rifle"}, {"id": 8693, "dom": "P", "ent": "time-ball", "cls": "road"}, {"id": 8694, "dom": "P", "ent": "time-ball", "cls": "salmon"}, {"id": 8695, "dom": "P", "ent": "time-ball", "cls": "shark"}, {"id": 8696, "dom": "P", "ent": "time-ball", "cls": "sheep"}, {"id": 8697, "dom": "P", "ent": "time-ball", "cls": "shook"}, {"id": 8698, "dom": "P", "ent": "time-ball", "cls": "tank"}, {"id": 8699, "dom": "P", "ent": "time-ball", "cls": "wheat"}, {"id": 8700, "dom": "P", "ent": "toe", "cls": "architecture"}, {"id": 8701, "dom": "P", "ent": "toe", "cls": "bacteria"}, {"id": 8702, "dom": "P", "ent": "toe", "cls": "beef"}, {"id": 8703, "dom": "P", "ent": "toe", "cls": "bridge"}, {"id": 8704, "dom": "P", "ent": "toe", "cls": "bud"}, {"id": 8705, "dom": "P", "ent": "toe", "cls": "cat"}, {"id": 8706, "dom": "P", "ent": "toe", "cls": "chapel"}, {"id": 8707, "dom": "P", "ent": "toe", "cls": "chick"}, {"id": 8708, "dom": "P", "ent": "toe", "cls": "clock"}, {"id": 8709, "dom": "P", "ent": "toe", "cls": "deer"}, {"id": 8710, "dom": "P", "ent": "toe", "cls": "dog"}, {"id": 8711, "dom": "P", "ent": "toe", "cls": "elephant"}, {"id": 8712, "dom": "P", "ent": "toe", "cls": "embassy"}, {"id": 8713, "dom": "P", "ent": "toe", "cls": "garage"}, {"id": 8714, "dom": "P", "ent": "toe", "cls": "highway"}, {"id": 8715, "dom": "P", "ent": "toe", "cls": "kidney"}, {"id": 8716, "dom": "P", "ent": "toe", "cls": "lamb"}, {"id": 8717, "dom": "P", "ent": "toe", "cls": "male"}, {"id": 8718, "dom": "P", "ent": "toe", "cls": "marijuana"}, {"id": 8719, "dom": "P", "ent": "toe", "cls": "puppy"}, {"id": 8720, "dom": "P", "ent": "toe", "cls": "receiver"}, {"id": 8721, "dom": "P", "ent": "toe", "cls": "reed"}, {"id": 8722, "dom": "P", "ent": "toe", "cls": "rifle"}, {"id": 8723, "dom": "P", "ent": "toe", "cls": "road"}, {"id": 8724, "dom": "P", "ent": "toe", "cls": "salmon"}, {"id": 8725, "dom": "P", "ent": "toe", "cls": "shark"}, {"id": 8726, "dom": "P", "ent": "toe", "cls": "sheep"}, {"id": 8727, "dom": "P", "ent": "toe", "cls": "shook"}, {"id": 8728, "dom": "P", "ent": "toe", "cls": "tank"}, {"id": 8729, "dom": "P", "ent": "toe", "cls": "wheat"}, {"id": 8730, "dom": "P", "ent": "tom-tom", "cls": "architecture"}, {"id": 8731, "dom": "P", "ent": "tom-tom", "cls": "bacteria"}, {"id": 8732, "dom": "P", "ent": "tom-tom", "cls": "beef"}, {"id": 8733, "dom": "P", "ent": "tom-tom", "cls": "bridge"}, {"id": 8734, "dom": "P", "ent": "tom-tom", "cls": "bud"}, {"id": 8735, "dom": "P", "ent": "tom-tom", "cls": "cat"}, {"id": 8736, "dom": "P", "ent": "tom-tom", "cls": "chapel"}, {"id": 8737, "dom": "P", "ent": "tom-tom", "cls": "chick"}, {"id": 8738, "dom": "P", "ent": "tom-tom", "cls": "clock"}, {"id": 8739, "dom": "P", "ent": "tom-tom", "cls": "deer"}, {"id": 8740, "dom": "P", "ent": "tom-tom", "cls": "dog"}, {"id": 8741, "dom": "P", "ent": "tom-tom", "cls": "elephant"}, {"id": 8742, "dom": "P", "ent": "tom-tom", "cls": "embassy"}, {"id": 8743, "dom": "P", "ent": "tom-tom", "cls": "garage"}, {"id": 8744, "dom": "P", "ent": "tom-tom", "cls": "highway"}, {"id": 8745, "dom": "P", "ent": "tom-tom", "cls": "kidney"}, {"id": 8746, "dom": "P", "ent": "tom-tom", "cls": "lamb"}, {"id": 8747, "dom": "P", "ent": "tom-tom", "cls": "male"}, {"id": 8748, "dom": "P", "ent": "tom-tom", "cls": "marijuana"}, {"id": 8749, "dom": "P", "ent": "tom-tom", "cls": "puppy"}, {"id": 8750, "dom": "P", "ent": "tom-tom", "cls": "receiver"}, {"id": 8751, "dom": "P", "ent": "tom-tom", "cls": "reed"}, {"id": 8752, "dom": "P", "ent": "tom-tom", "cls": "rifle"}, {"id": 8753, "dom": "P", "ent": "tom-tom", "cls": "road"}, {"id": 8754, "dom": "P", "ent": "tom-tom", "cls": "salmon"}, {"id": 8755, "dom": "P", "ent": "tom-tom", "cls": "shark"}, {"id": 8756, "dom": "P", "ent": "tom-tom", "cls": "sheep"}, {"id": 8757, "dom": "P", "ent": "tom-tom", "cls": "shook"}, {"id": 8758, "dom": "P", "ent": "tom-tom", "cls": "tank"}, {"id": 8759, "dom": "P", "ent": "tom-tom", "cls": "wheat"}, {"id": 8760, "dom": "P", "ent": "tongue", "cls": "architecture"}, {"id": 8761, "dom": "P", "ent": "tongue", "cls": "bacteria"}, {"id": 8762, "dom": "P", "ent": "tongue", "cls": "beef"}, {"id": 8763, "dom": "P", "ent": "tongue", "cls": "bridge"}, {"id": 8764, "dom": "P", "ent": "tongue", "cls": "bud"}, {"id": 8765, "dom": "P", "ent": "tongue", "cls": "cat"}, {"id": 8766, "dom": "P", "ent": "tongue", "cls": "chapel"}, {"id": 8767, "dom": "P", "ent": "tongue", "cls": "chick"}, {"id": 8768, "dom": "P", "ent": "tongue", "cls": "clock"}, {"id": 8769, "dom": "P", "ent": "tongue", "cls": "deer"}, {"id": 8770, "dom": "P", "ent": "tongue", "cls": "dog"}, {"id": 8771, "dom": "P", "ent": "tongue", "cls": "elephant"}, {"id": 8772, "dom": "P", "ent": "tongue", "cls": "embassy"}, {"id": 8773, "dom": "P", "ent": "tongue", "cls": "garage"}, {"id": 8774, "dom": "P", "ent": "tongue", "cls": "highway"}, {"id": 8775, "dom": "P", "ent": "tongue", "cls": "kidney"}, {"id": 8776, "dom": "P", "ent": "tongue", "cls": "lamb"}, {"id": 8777, "dom": "P", "ent": "tongue", "cls": "male"}, {"id": 8778, "dom": "P", "ent": "tongue", "cls": "marijuana"}, {"id": 8779, "dom": "P", "ent": "tongue", "cls": "puppy"}, {"id": 8780, "dom": "P", "ent": "tongue", "cls": "receiver"}, {"id": 8781, "dom": "P", "ent": "tongue", "cls": "reed"}, {"id": 8782, "dom": "P", "ent": "tongue", "cls": "rifle"}, {"id": 8783, "dom": "P", "ent": "tongue", "cls": "road"}, {"id": 8784, "dom": "P", "ent": "tongue", "cls": "salmon"}, {"id": 8785, "dom": "P", "ent": "tongue", "cls": "shark"}, {"id": 8786, "dom": "P", "ent": "tongue", "cls": "sheep"}, {"id": 8787, "dom": "P", "ent": "tongue", "cls": "shook"}, {"id": 8788, "dom": "P", "ent": "tongue", "cls": "tank"}, {"id": 8789, "dom": "P", "ent": "tongue", "cls": "wheat"}, {"id": 8790, "dom": "P", "ent": "tongue-flower", "cls": "architecture"}, {"id": 8791, "dom": "P", "ent": "tongue-flower", "cls": "bacteria"}, {"id": 8792, "dom": "P", "ent": "tongue-flower", "cls": "beef"}, {"id": 8793, "dom": "P", "ent": "tongue-flower", "cls": "bridge"}, {"id": 8794, "dom": "P", "ent": "tongue-flower", "cls": "bud"}, {"id": 8795, "dom": "P", "ent": "tongue-flower", "cls": "cat"}, {"id": 8796, "dom": "P", "ent": "tongue-flower", "cls": "chapel"}, {"id": 8797, "dom": "P", "ent": "tongue-flower", "cls": "chick"}, {"id": 8798, "dom": "P", "ent": "tongue-flower", "cls": "clock"}, {"id": 8799, "dom": "P", "ent": "tongue-flower", "cls": "deer"}, {"id": 8800, "dom": "P", "ent": "tongue-flower", "cls": "dog"}, {"id": 8801, "dom": "P", "ent": "tongue-flower", "cls": "elephant"}, {"id": 8802, "dom": "P", "ent": "tongue-flower", "cls": "embassy"}, {"id": 8803, "dom": "P", "ent": "tongue-flower", "cls": "garage"}, {"id": 8804, "dom": "P", "ent": "tongue-flower", "cls": "highway"}, {"id": 8805, "dom": "P", "ent": "tongue-flower", "cls": "kidney"}, {"id": 8806, "dom": "P", "ent": "tongue-flower", "cls": "lamb"}, {"id": 8807, "dom": "P", "ent": "tongue-flower", "cls": "male"}, {"id": 8808, "dom": "P", "ent": "tongue-flower", "cls": "marijuana"}, {"id": 8809, "dom": "P", "ent": "tongue-flower", "cls": "puppy"}, {"id": 8810, "dom": "P", "ent": "tongue-flower", "cls": "receiver"}, {"id": 8811, "dom": "P", "ent": "tongue-flower", "cls": "reed"}, {"id": 8812, "dom": "P", "ent": "tongue-flower", "cls": "rifle"}, {"id": 8813, "dom": "P", "ent": "tongue-flower", "cls": "road"}, {"id": 8814, "dom": "P", "ent": "tongue-flower", "cls": "salmon"}, {"id": 8815, "dom": "P", "ent": "tongue-flower", "cls": "shark"}, {"id": 8816, "dom": "P", "ent": "tongue-flower", "cls": "sheep"}, {"id": 8817, "dom": "P", "ent": "tongue-flower", "cls": "shook"}, {"id": 8818, "dom": "P", "ent": "tongue-flower", "cls": "tank"}, {"id": 8819, "dom": "P", "ent": "tongue-flower", "cls": "wheat"}, {"id": 8820, "dom": "P", "ent": "tornado", "cls": "architecture"}, {"id": 8821, "dom": "P", "ent": "tornado", "cls": "bacteria"}, {"id": 8822, "dom": "P", "ent": "tornado", "cls": "beef"}, {"id": 8823, "dom": "P", "ent": "tornado", "cls": "bridge"}, {"id": 8824, "dom": "P", "ent": "tornado", "cls": "bud"}, {"id": 8825, "dom": "P", "ent": "tornado", "cls": "cat"}, {"id": 8826, "dom": "P", "ent": "tornado", "cls": "chapel"}, {"id": 8827, "dom": "P", "ent": "tornado", "cls": "chick"}, {"id": 8828, "dom": "P", "ent": "tornado", "cls": "clock"}, {"id": 8829, "dom": "P", "ent": "tornado", "cls": "deer"}, {"id": 8830, "dom": "P", "ent": "tornado", "cls": "dog"}, {"id": 8831, "dom": "P", "ent": "tornado", "cls": "elephant"}, {"id": 8832, "dom": "P", "ent": "tornado", "cls": "embassy"}, {"id": 8833, "dom": "P", "ent": "tornado", "cls": "garage"}, {"id": 8834, "dom": "P", "ent": "tornado", "cls": "highway"}, {"id": 8835, "dom": "P", "ent": "tornado", "cls": "kidney"}, {"id": 8836, "dom": "P", "ent": "tornado", "cls": "lamb"}, {"id": 8837, "dom": "P", "ent": "tornado", "cls": "male"}, {"id": 8838, "dom": "P", "ent": "tornado", "cls": "marijuana"}, {"id": 8839, "dom": "P", "ent": "tornado", "cls": "puppy"}, {"id": 8840, "dom": "P", "ent": "tornado", "cls": "receiver"}, {"id": 8841, "dom": "P", "ent": "tornado", "cls": "reed"}, {"id": 8842, "dom": "P", "ent": "tornado", "cls": "rifle"}, {"id": 8843, "dom": "P", "ent": "tornado", "cls": "road"}, {"id": 8844, "dom": "P", "ent": "tornado", "cls": "salmon"}, {"id": 8845, "dom": "P", "ent": "tornado", "cls": "shark"}, {"id": 8846, "dom": "P", "ent": "tornado", "cls": "sheep"}, {"id": 8847, "dom": "P", "ent": "tornado", "cls": "shook"}, {"id": 8848, "dom": "P", "ent": "tornado", "cls": "tank"}, {"id": 8849, "dom": "P", "ent": "tornado", "cls": "wheat"}, {"id": 8850, "dom": "P", "ent": "total", "cls": "architecture"}, {"id": 8851, "dom": "P", "ent": "total", "cls": "bacteria"}, {"id": 8852, "dom": "P", "ent": "total", "cls": "beef"}, {"id": 8853, "dom": "P", "ent": "total", "cls": "bridge"}, {"id": 8854, "dom": "P", "ent": "total", "cls": "bud"}, {"id": 8855, "dom": "P", "ent": "total", "cls": "cat"}, {"id": 8856, "dom": "P", "ent": "total", "cls": "chapel"}, {"id": 8857, "dom": "P", "ent": "total", "cls": "chick"}, {"id": 8858, "dom": "P", "ent": "total", "cls": "clock"}, {"id": 8859, "dom": "P", "ent": "total", "cls": "deer"}, {"id": 8860, "dom": "P", "ent": "total", "cls": "dog"}, {"id": 8861, "dom": "P", "ent": "total", "cls": "elephant"}, {"id": 8862, "dom": "P", "ent": "total", "cls": "embassy"}, {"id": 8863, "dom": "P", "ent": "total", "cls": "garage"}, {"id": 8864, "dom": "P", "ent": "total", "cls": "highway"}, {"id": 8865, "dom": "P", "ent": "total", "cls": "kidney"}, {"id": 8866, "dom": "P", "ent": "total", "cls": "lamb"}, {"id": 8867, "dom": "P", "ent": "total", "cls": "male"}, {"id": 8868, "dom": "P", "ent": "total", "cls": "marijuana"}, {"id": 8869, "dom": "P", "ent": "total", "cls": "puppy"}, {"id": 8870, "dom": "P", "ent": "total", "cls": "receiver"}, {"id": 8871, "dom": "P", "ent": "total", "cls": "reed"}, {"id": 8872, "dom": "P", "ent": "total", "cls": "rifle"}, {"id": 8873, "dom": "P", "ent": "total", "cls": "road"}, {"id": 8874, "dom": "P", "ent": "total", "cls": "salmon"}, {"id": 8875, "dom": "P", "ent": "total", "cls": "shark"}, {"id": 8876, "dom": "P", "ent": "total", "cls": "sheep"}, {"id": 8877, "dom": "P", "ent": "total", "cls": "shook"}, {"id": 8878, "dom": "P", "ent": "total", "cls": "tank"}, {"id": 8879, "dom": "P", "ent": "total", "cls": "wheat"}, {"id": 8880, "dom": "P", "ent": "track", "cls": "architecture"}, {"id": 8881, "dom": "P", "ent": "track", "cls": "bacteria"}, {"id": 8882, "dom": "P", "ent": "track", "cls": "beef"}, {"id": 8883, "dom": "P", "ent": "track", "cls": "bridge"}, {"id": 8884, "dom": "P", "ent": "track", "cls": "bud"}, {"id": 8885, "dom": "P", "ent": "track", "cls": "cat"}, {"id": 8886, "dom": "P", "ent": "track", "cls": "chapel"}, {"id": 8887, "dom": "P", "ent": "track", "cls": "chick"}, {"id": 8888, "dom": "P", "ent": "track", "cls": "clock"}, {"id": 8889, "dom": "P", "ent": "track", "cls": "deer"}, {"id": 8890, "dom": "P", "ent": "track", "cls": "dog"}, {"id": 8891, "dom": "P", "ent": "track", "cls": "elephant"}, {"id": 8892, "dom": "P", "ent": "track", "cls": "embassy"}, {"id": 8893, "dom": "P", "ent": "track", "cls": "garage"}, {"id": 8894, "dom": "P", "ent": "track", "cls": "highway"}, {"id": 8895, "dom": "P", "ent": "track", "cls": "kidney"}, {"id": 8896, "dom": "P", "ent": "track", "cls": "lamb"}, {"id": 8897, "dom": "P", "ent": "track", "cls": "male"}, {"id": 8898, "dom": "P", "ent": "track", "cls": "marijuana"}, {"id": 8899, "dom": "P", "ent": "track", "cls": "puppy"}, {"id": 8900, "dom": "P", "ent": "track", "cls": "receiver"}, {"id": 8901, "dom": "P", "ent": "track", "cls": "reed"}, {"id": 8902, "dom": "P", "ent": "track", "cls": "rifle"}, {"id": 8903, "dom": "P", "ent": "track", "cls": "road"}, {"id": 8904, "dom": "P", "ent": "track", "cls": "salmon"}, {"id": 8905, "dom": "P", "ent": "track", "cls": "shark"}, {"id": 8906, "dom": "P", "ent": "track", "cls": "sheep"}, {"id": 8907, "dom": "P", "ent": "track", "cls": "shook"}, {"id": 8908, "dom": "P", "ent": "track", "cls": "tank"}, {"id": 8909, "dom": "P", "ent": "track", "cls": "wheat"}, {"id": 8910, "dom": "P", "ent": "tread", "cls": "architecture"}, {"id": 8911, "dom": "P", "ent": "tread", "cls": "bacteria"}, {"id": 8912, "dom": "P", "ent": "tread", "cls": "beef"}, {"id": 8913, "dom": "P", "ent": "tread", "cls": "bridge"}, {"id": 8914, "dom": "P", "ent": "tread", "cls": "bud"}, {"id": 8915, "dom": "P", "ent": "tread", "cls": "cat"}, {"id": 8916, "dom": "P", "ent": "tread", "cls": "chapel"}, {"id": 8917, "dom": "P", "ent": "tread", "cls": "chick"}, {"id": 8918, "dom": "P", "ent": "tread", "cls": "clock"}, {"id": 8919, "dom": "P", "ent": "tread", "cls": "deer"}, {"id": 8920, "dom": "P", "ent": "tread", "cls": "dog"}, {"id": 8921, "dom": "P", "ent": "tread", "cls": "elephant"}, {"id": 8922, "dom": "P", "ent": "tread", "cls": "embassy"}, {"id": 8923, "dom": "P", "ent": "tread", "cls": "garage"}, {"id": 8924, "dom": "P", "ent": "tread", "cls": "highway"}, {"id": 8925, "dom": "P", "ent": "tread", "cls": "kidney"}, {"id": 8926, "dom": "P", "ent": "tread", "cls": "lamb"}, {"id": 8927, "dom": "P", "ent": "tread", "cls": "male"}, {"id": 8928, "dom": "P", "ent": "tread", "cls": "marijuana"}, {"id": 8929, "dom": "P", "ent": "tread", "cls": "puppy"}, {"id": 8930, "dom": "P", "ent": "tread", "cls": "receiver"}, {"id": 8931, "dom": "P", "ent": "tread", "cls": "reed"}, {"id": 8932, "dom": "P", "ent": "tread", "cls": "rifle"}, {"id": 8933, "dom": "P", "ent": "tread", "cls": "road"}, {"id": 8934, "dom": "P", "ent": "tread", "cls": "salmon"}, {"id": 8935, "dom": "P", "ent": "tread", "cls": "shark"}, {"id": 8936, "dom": "P", "ent": "tread", "cls": "sheep"}, {"id": 8937, "dom": "P", "ent": "tread", "cls": "shook"}, {"id": 8938, "dom": "P", "ent": "tread", "cls": "tank"}, {"id": 8939, "dom": "P", "ent": "tread", "cls": "wheat"}, {"id": 8940, "dom": "P", "ent": "trigger", "cls": "architecture"}, {"id": 8941, "dom": "P", "ent": "trigger", "cls": "bacteria"}, {"id": 8942, "dom": "P", "ent": "trigger", "cls": "beef"}, {"id": 8943, "dom": "P", "ent": "trigger", "cls": "bridge"}, {"id": 8944, "dom": "P", "ent": "trigger", "cls": "bud"}, {"id": 8945, "dom": "P", "ent": "trigger", "cls": "cat"}, {"id": 8946, "dom": "P", "ent": "trigger", "cls": "chapel"}, {"id": 8947, "dom": "P", "ent": "trigger", "cls": "chick"}, {"id": 8948, "dom": "P", "ent": "trigger", "cls": "clock"}, {"id": 8949, "dom": "P", "ent": "trigger", "cls": "deer"}, {"id": 8950, "dom": "P", "ent": "trigger", "cls": "dog"}, {"id": 8951, "dom": "P", "ent": "trigger", "cls": "elephant"}, {"id": 8952, "dom": "P", "ent": "trigger", "cls": "embassy"}, {"id": 8953, "dom": "P", "ent": "trigger", "cls": "garage"}, {"id": 8954, "dom": "P", "ent": "trigger", "cls": "highway"}, {"id": 8955, "dom": "P", "ent": "trigger", "cls": "kidney"}, {"id": 8956, "dom": "P", "ent": "trigger", "cls": "lamb"}, {"id": 8957, "dom": "P", "ent": "trigger", "cls": "male"}, {"id": 8958, "dom": "P", "ent": "trigger", "cls": "marijuana"}, {"id": 8959, "dom": "P", "ent": "trigger", "cls": "puppy"}, {"id": 8960, "dom": "P", "ent": "trigger", "cls": "receiver"}, {"id": 8961, "dom": "P", "ent": "trigger", "cls": "reed"}, {"id": 8962, "dom": "P", "ent": "trigger", "cls": "rifle"}, {"id": 8963, "dom": "P", "ent": "trigger", "cls": "road"}, {"id": 8964, "dom": "P", "ent": "trigger", "cls": "salmon"}, {"id": 8965, "dom": "P", "ent": "trigger", "cls": "shark"}, {"id": 8966, "dom": "P", "ent": "trigger", "cls": "sheep"}, {"id": 8967, "dom": "P", "ent": "trigger", "cls": "shook"}, {"id": 8968, "dom": "P", "ent": "trigger", "cls": "tank"}, {"id": 8969, "dom": "P", "ent": "trigger", "cls": "wheat"}, {"id": 8970, "dom": "P", "ent": "trunk", "cls": "architecture"}, {"id": 8971, "dom": "P", "ent": "trunk", "cls": "bacteria"}, {"id": 8972, "dom": "P", "ent": "trunk", "cls": "beef"}, {"id": 8973, "dom": "P", "ent": "trunk", "cls": "bridge"}, {"id": 8974, "dom": "P", "ent": "trunk", "cls": "bud"}, {"id": 8975, "dom": "P", "ent": "trunk", "cls": "cat"}, {"id": 8976, "dom": "P", "ent": "trunk", "cls": "chapel"}, {"id": 8977, "dom": "P", "ent": "trunk", "cls": "chick"}, {"id": 8978, "dom": "P", "ent": "trunk", "cls": "clock"}, {"id": 8979, "dom": "P", "ent": "trunk", "cls": "deer"}, {"id": 8980, "dom": "P", "ent": "trunk", "cls": "dog"}, {"id": 8981, "dom": "P", "ent": "trunk", "cls": "elephant"}, {"id": 8982, "dom": "P", "ent": "trunk", "cls": "embassy"}, {"id": 8983, "dom": "P", "ent": "trunk", "cls": "garage"}, {"id": 8984, "dom": "P", "ent": "trunk", "cls": "highway"}, {"id": 8985, "dom": "P", "ent": "trunk", "cls": "kidney"}, {"id": 8986, "dom": "P", "ent": "trunk", "cls": "lamb"}, {"id": 8987, "dom": "P", "ent": "trunk", "cls": "male"}, {"id": 8988, "dom": "P", "ent": "trunk", "cls": "marijuana"}, {"id": 8989, "dom": "P", "ent": "trunk", "cls": "puppy"}, {"id": 8990, "dom": "P", "ent": "trunk", "cls": "receiver"}, {"id": 8991, "dom": "P", "ent": "trunk", "cls": "reed"}, {"id": 8992, "dom": "P", "ent": "trunk", "cls": "rifle"}, {"id": 8993, "dom": "P", "ent": "trunk", "cls": "road"}, {"id": 8994, "dom": "P", "ent": "trunk", "cls": "salmon"}, {"id": 8995, "dom": "P", "ent": "trunk", "cls": "shark"}, {"id": 8996, "dom": "P", "ent": "trunk", "cls": "sheep"}, {"id": 8997, "dom": "P", "ent": "trunk", "cls": "shook"}, {"id": 8998, "dom": "P", "ent": "trunk", "cls": "tank"}, {"id": 8999, "dom": "P", "ent": "trunk", "cls": "wheat"}, {"id": 9000, "dom": "P", "ent": "tube", "cls": "architecture"}, {"id": 9001, "dom": "P", "ent": "tube", "cls": "bacteria"}, {"id": 9002, "dom": "P", "ent": "tube", "cls": "beef"}, {"id": 9003, "dom": "P", "ent": "tube", "cls": "bridge"}, {"id": 9004, "dom": "P", "ent": "tube", "cls": "bud"}, {"id": 9005, "dom": "P", "ent": "tube", "cls": "cat"}, {"id": 9006, "dom": "P", "ent": "tube", "cls": "chapel"}, {"id": 9007, "dom": "P", "ent": "tube", "cls": "chick"}, {"id": 9008, "dom": "P", "ent": "tube", "cls": "clock"}, {"id": 9009, "dom": "P", "ent": "tube", "cls": "deer"}, {"id": 9010, "dom": "P", "ent": "tube", "cls": "dog"}, {"id": 9011, "dom": "P", "ent": "tube", "cls": "elephant"}, {"id": 9012, "dom": "P", "ent": "tube", "cls": "embassy"}, {"id": 9013, "dom": "P", "ent": "tube", "cls": "garage"}, {"id": 9014, "dom": "P", "ent": "tube", "cls": "highway"}, {"id": 9015, "dom": "P", "ent": "tube", "cls": "kidney"}, {"id": 9016, "dom": "P", "ent": "tube", "cls": "lamb"}, {"id": 9017, "dom": "P", "ent": "tube", "cls": "male"}, {"id": 9018, "dom": "P", "ent": "tube", "cls": "marijuana"}, {"id": 9019, "dom": "P", "ent": "tube", "cls": "puppy"}, {"id": 9020, "dom": "P", "ent": "tube", "cls": "receiver"}, {"id": 9021, "dom": "P", "ent": "tube", "cls": "reed"}, {"id": 9022, "dom": "P", "ent": "tube", "cls": "rifle"}, {"id": 9023, "dom": "P", "ent": "tube", "cls": "road"}, {"id": 9024, "dom": "P", "ent": "tube", "cls": "salmon"}, {"id": 9025, "dom": "P", "ent": "tube", "cls": "shark"}, {"id": 9026, "dom": "P", "ent": "tube", "cls": "sheep"}, {"id": 9027, "dom": "P", "ent": "tube", "cls": "shook"}, {"id": 9028, "dom": "P", "ent": "tube", "cls": "tank"}, {"id": 9029, "dom": "P", "ent": "tube", "cls": "wheat"}, {"id": 9030, "dom": "P", "ent": "turnaround", "cls": "architecture"}, {"id": 9031, "dom": "P", "ent": "turnaround", "cls": "bacteria"}, {"id": 9032, "dom": "P", "ent": "turnaround", "cls": "beef"}, {"id": 9033, "dom": "P", "ent": "turnaround", "cls": "bridge"}, {"id": 9034, "dom": "P", "ent": "turnaround", "cls": "bud"}, {"id": 9035, "dom": "P", "ent": "turnaround", "cls": "cat"}, {"id": 9036, "dom": "P", "ent": "turnaround", "cls": "chapel"}, {"id": 9037, "dom": "P", "ent": "turnaround", "cls": "chick"}, {"id": 9038, "dom": "P", "ent": "turnaround", "cls": "clock"}, {"id": 9039, "dom": "P", "ent": "turnaround", "cls": "deer"}, {"id": 9040, "dom": "P", "ent": "turnaround", "cls": "dog"}, {"id": 9041, "dom": "P", "ent": "turnaround", "cls": "elephant"}, {"id": 9042, "dom": "P", "ent": "turnaround", "cls": "embassy"}, {"id": 9043, "dom": "P", "ent": "turnaround", "cls": "garage"}, {"id": 9044, "dom": "P", "ent": "turnaround", "cls": "highway"}, {"id": 9045, "dom": "P", "ent": "turnaround", "cls": "kidney"}, {"id": 9046, "dom": "P", "ent": "turnaround", "cls": "lamb"}, {"id": 9047, "dom": "P", "ent": "turnaround", "cls": "male"}, {"id": 9048, "dom": "P", "ent": "turnaround", "cls": "marijuana"}, {"id": 9049, "dom": "P", "ent": "turnaround", "cls": "puppy"}, {"id": 9050, "dom": "P", "ent": "turnaround", "cls": "receiver"}, {"id": 9051, "dom": "P", "ent": "turnaround", "cls": "reed"}, {"id": 9052, "dom": "P", "ent": "turnaround", "cls": "rifle"}, {"id": 9053, "dom": "P", "ent": "turnaround", "cls": "road"}, {"id": 9054, "dom": "P", "ent": "turnaround", "cls": "salmon"}, {"id": 9055, "dom": "P", "ent": "turnaround", "cls": "shark"}, {"id": 9056, "dom": "P", "ent": "turnaround", "cls": "sheep"}, {"id": 9057, "dom": "P", "ent": "turnaround", "cls": "shook"}, {"id": 9058, "dom": "P", "ent": "turnaround", "cls": "tank"}, {"id": 9059, "dom": "P", "ent": "turnaround", "cls": "wheat"}, {"id": 9060, "dom": "P", "ent": "turnout", "cls": "architecture"}, {"id": 9061, "dom": "P", "ent": "turnout", "cls": "bacteria"}, {"id": 9062, "dom": "P", "ent": "turnout", "cls": "beef"}, {"id": 9063, "dom": "P", "ent": "turnout", "cls": "bridge"}, {"id": 9064, "dom": "P", "ent": "turnout", "cls": "bud"}, {"id": 9065, "dom": "P", "ent": "turnout", "cls": "cat"}, {"id": 9066, "dom": "P", "ent": "turnout", "cls": "chapel"}, {"id": 9067, "dom": "P", "ent": "turnout", "cls": "chick"}, {"id": 9068, "dom": "P", "ent": "turnout", "cls": "clock"}, {"id": 9069, "dom": "P", "ent": "turnout", "cls": "deer"}, {"id": 9070, "dom": "P", "ent": "turnout", "cls": "dog"}, {"id": 9071, "dom": "P", "ent": "turnout", "cls": "elephant"}, {"id": 9072, "dom": "P", "ent": "turnout", "cls": "embassy"}, {"id": 9073, "dom": "P", "ent": "turnout", "cls": "garage"}, {"id": 9074, "dom": "P", "ent": "turnout", "cls": "highway"}, {"id": 9075, "dom": "P", "ent": "turnout", "cls": "kidney"}, {"id": 9076, "dom": "P", "ent": "turnout", "cls": "lamb"}, {"id": 9077, "dom": "P", "ent": "turnout", "cls": "male"}, {"id": 9078, "dom": "P", "ent": "turnout", "cls": "marijuana"}, {"id": 9079, "dom": "P", "ent": "turnout", "cls": "puppy"}, {"id": 9080, "dom": "P", "ent": "turnout", "cls": "receiver"}, {"id": 9081, "dom": "P", "ent": "turnout", "cls": "reed"}, {"id": 9082, "dom": "P", "ent": "turnout", "cls": "rifle"}, {"id": 9083, "dom": "P", "ent": "turnout", "cls": "road"}, {"id": 9084, "dom": "P", "ent": "turnout", "cls": "salmon"}, {"id": 9085, "dom": "P", "ent": "turnout", "cls": "shark"}, {"id": 9086, "dom": "P", "ent": "turnout", "cls": "sheep"}, {"id": 9087, "dom": "P", "ent": "turnout", "cls": "shook"}, {"id": 9088, "dom": "P", "ent": "turnout", "cls": "tank"}, {"id": 9089, "dom": "P", "ent": "turnout", "cls": "wheat"}, {"id": 9090, "dom": "P", "ent": "turret", "cls": "architecture"}, {"id": 9091, "dom": "P", "ent": "turret", "cls": "bacteria"}, {"id": 9092, "dom": "P", "ent": "turret", "cls": "beef"}, {"id": 9093, "dom": "P", "ent": "turret", "cls": "bridge"}, {"id": 9094, "dom": "P", "ent": "turret", "cls": "bud"}, {"id": 9095, "dom": "P", "ent": "turret", "cls": "cat"}, {"id": 9096, "dom": "P", "ent": "turret", "cls": "chapel"}, {"id": 9097, "dom": "P", "ent": "turret", "cls": "chick"}, {"id": 9098, "dom": "P", "ent": "turret", "cls": "clock"}, {"id": 9099, "dom": "P", "ent": "turret", "cls": "deer"}, {"id": 9100, "dom": "P", "ent": "turret", "cls": "dog"}, {"id": 9101, "dom": "P", "ent": "turret", "cls": "elephant"}, {"id": 9102, "dom": "P", "ent": "turret", "cls": "embassy"}, {"id": 9103, "dom": "P", "ent": "turret", "cls": "garage"}, {"id": 9104, "dom": "P", "ent": "turret", "cls": "highway"}, {"id": 9105, "dom": "P", "ent": "turret", "cls": "kidney"}, {"id": 9106, "dom": "P", "ent": "turret", "cls": "lamb"}, {"id": 9107, "dom": "P", "ent": "turret", "cls": "male"}, {"id": 9108, "dom": "P", "ent": "turret", "cls": "marijuana"}, {"id": 9109, "dom": "P", "ent": "turret", "cls": "puppy"}, {"id": 9110, "dom": "P", "ent": "turret", "cls": "receiver"}, {"id": 9111, "dom": "P", "ent": "turret", "cls": "reed"}, {"id": 9112, "dom": "P", "ent": "turret", "cls": "rifle"}, {"id": 9113, "dom": "P", "ent": "turret", "cls": "road"}, {"id": 9114, "dom": "P", "ent": "turret", "cls": "salmon"}, {"id": 9115, "dom": "P", "ent": "turret", "cls": "shark"}, {"id": 9116, "dom": "P", "ent": "turret", "cls": "sheep"}, {"id": 9117, "dom": "P", "ent": "turret", "cls": "shook"}, {"id": 9118, "dom": "P", "ent": "turret", "cls": "tank"}, {"id": 9119, "dom": "P", "ent": "turret", "cls": "wheat"}, {"id": 9120, "dom": "P", "ent": "tyre", "cls": "architecture"}, {"id": 9121, "dom": "P", "ent": "tyre", "cls": "bacteria"}, {"id": 9122, "dom": "P", "ent": "tyre", "cls": "beef"}, {"id": 9123, "dom": "P", "ent": "tyre", "cls": "bridge"}, {"id": 9124, "dom": "P", "ent": "tyre", "cls": "bud"}, {"id": 9125, "dom": "P", "ent": "tyre", "cls": "cat"}, {"id": 9126, "dom": "P", "ent": "tyre", "cls": "chapel"}, {"id": 9127, "dom": "P", "ent": "tyre", "cls": "chick"}, {"id": 9128, "dom": "P", "ent": "tyre", "cls": "clock"}, {"id": 9129, "dom": "P", "ent": "tyre", "cls": "deer"}, {"id": 9130, "dom": "P", "ent": "tyre", "cls": "dog"}, {"id": 9131, "dom": "P", "ent": "tyre", "cls": "elephant"}, {"id": 9132, "dom": "P", "ent": "tyre", "cls": "embassy"}, {"id": 9133, "dom": "P", "ent": "tyre", "cls": "garage"}, {"id": 9134, "dom": "P", "ent": "tyre", "cls": "highway"}, {"id": 9135, "dom": "P", "ent": "tyre", "cls": "kidney"}, {"id": 9136, "dom": "P", "ent": "tyre", "cls": "lamb"}, {"id": 9137, "dom": "P", "ent": "tyre", "cls": "male"}, {"id": 9138, "dom": "P", "ent": "tyre", "cls": "marijuana"}, {"id": 9139, "dom": "P", "ent": "tyre", "cls": "puppy"}, {"id": 9140, "dom": "P", "ent": "tyre", "cls": "receiver"}, {"id": 9141, "dom": "P", "ent": "tyre", "cls": "reed"}, {"id": 9142, "dom": "P", "ent": "tyre", "cls": "rifle"}, {"id": 9143, "dom": "P", "ent": "tyre", "cls": "road"}, {"id": 9144, "dom": "P", "ent": "tyre", "cls": "salmon"}, {"id": 9145, "dom": "P", "ent": "tyre", "cls": "shark"}, {"id": 9146, "dom": "P", "ent": "tyre", "cls": "sheep"}, {"id": 9147, "dom": "P", "ent": "tyre", "cls": "shook"}, {"id": 9148, "dom": "P", "ent": "tyre", "cls": "tank"}, {"id": 9149, "dom": "P", "ent": "tyre", "cls": "wheat"}, {"id": 9150, "dom": "P", "ent": "upstairs", "cls": "architecture"}, {"id": 9151, "dom": "P", "ent": "upstairs", "cls": "bacteria"}, {"id": 9152, "dom": "P", "ent": "upstairs", "cls": "beef"}, {"id": 9153, "dom": "P", "ent": "upstairs", "cls": "bridge"}, {"id": 9154, "dom": "P", "ent": "upstairs", "cls": "bud"}, {"id": 9155, "dom": "P", "ent": "upstairs", "cls": "cat"}, {"id": 9156, "dom": "P", "ent": "upstairs", "cls": "chapel"}, {"id": 9157, "dom": "P", "ent": "upstairs", "cls": "chick"}, {"id": 9158, "dom": "P", "ent": "upstairs", "cls": "clock"}, {"id": 9159, "dom": "P", "ent": "upstairs", "cls": "deer"}, {"id": 9160, "dom": "P", "ent": "upstairs", "cls": "dog"}, {"id": 9161, "dom": "P", "ent": "upstairs", "cls": "elephant"}, {"id": 9162, "dom": "P", "ent": "upstairs", "cls": "embassy"}, {"id": 9163, "dom": "P", "ent": "upstairs", "cls": "garage"}, {"id": 9164, "dom": "P", "ent": "upstairs", "cls": "highway"}, {"id": 9165, "dom": "P", "ent": "upstairs", "cls": "kidney"}, {"id": 9166, "dom": "P", "ent": "upstairs", "cls": "lamb"}, {"id": 9167, "dom": "P", "ent": "upstairs", "cls": "male"}, {"id": 9168, "dom": "P", "ent": "upstairs", "cls": "marijuana"}, {"id": 9169, "dom": "P", "ent": "upstairs", "cls": "puppy"}, {"id": 9170, "dom": "P", "ent": "upstairs", "cls": "receiver"}, {"id": 9171, "dom": "P", "ent": "upstairs", "cls": "reed"}, {"id": 9172, "dom": "P", "ent": "upstairs", "cls": "rifle"}, {"id": 9173, "dom": "P", "ent": "upstairs", "cls": "road"}, {"id": 9174, "dom": "P", "ent": "upstairs", "cls": "salmon"}, {"id": 9175, "dom": "P", "ent": "upstairs", "cls": "shark"}, {"id": 9176, "dom": "P", "ent": "upstairs", "cls": "sheep"}, {"id": 9177, "dom": "P", "ent": "upstairs", "cls": "shook"}, {"id": 9178, "dom": "P", "ent": "upstairs", "cls": "tank"}, {"id": 9179, "dom": "P", "ent": "upstairs", "cls": "wheat"}, {"id": 9180, "dom": "P", "ent": "vaccine", "cls": "architecture"}, {"id": 9181, "dom": "P", "ent": "vaccine", "cls": "bacteria"}, {"id": 9182, "dom": "P", "ent": "vaccine", "cls": "beef"}, {"id": 9183, "dom": "P", "ent": "vaccine", "cls": "bridge"}, {"id": 9184, "dom": "P", "ent": "vaccine", "cls": "bud"}, {"id": 9185, "dom": "P", "ent": "vaccine", "cls": "cat"}, {"id": 9186, "dom": "P", "ent": "vaccine", "cls": "chapel"}, {"id": 9187, "dom": "P", "ent": "vaccine", "cls": "chick"}, {"id": 9188, "dom": "P", "ent": "vaccine", "cls": "clock"}, {"id": 9189, "dom": "P", "ent": "vaccine", "cls": "deer"}, {"id": 9190, "dom": "P", "ent": "vaccine", "cls": "dog"}, {"id": 9191, "dom": "P", "ent": "vaccine", "cls": "elephant"}, {"id": 9192, "dom": "P", "ent": "vaccine", "cls": "embassy"}, {"id": 9193, "dom": "P", "ent": "vaccine", "cls": "garage"}, {"id": 9194, "dom": "P", "ent": "vaccine", "cls": "highway"}, {"id": 9195, "dom": "P", "ent": "vaccine", "cls": "kidney"}, {"id": 9196, "dom": "P", "ent": "vaccine", "cls": "lamb"}, {"id": 9197, "dom": "P", "ent": "vaccine", "cls": "male"}, {"id": 9198, "dom": "P", "ent": "vaccine", "cls": "marijuana"}, {"id": 9199, "dom": "P", "ent": "vaccine", "cls": "puppy"}, {"id": 9200, "dom": "P", "ent": "vaccine", "cls": "receiver"}, {"id": 9201, "dom": "P", "ent": "vaccine", "cls": "reed"}, {"id": 9202, "dom": "P", "ent": "vaccine", "cls": "rifle"}, {"id": 9203, "dom": "P", "ent": "vaccine", "cls": "road"}, {"id": 9204, "dom": "P", "ent": "vaccine", "cls": "salmon"}, {"id": 9205, "dom": "P", "ent": "vaccine", "cls": "shark"}, {"id": 9206, "dom": "P", "ent": "vaccine", "cls": "sheep"}, {"id": 9207, "dom": "P", "ent": "vaccine", "cls": "shook"}, {"id": 9208, "dom": "P", "ent": "vaccine", "cls": "tank"}, {"id": 9209, "dom": "P", "ent": "vaccine", "cls": "wheat"}, {"id": 9210, "dom": "P", "ent": "vane", "cls": "architecture"}, {"id": 9211, "dom": "P", "ent": "vane", "cls": "bacteria"}, {"id": 9212, "dom": "P", "ent": "vane", "cls": "beef"}, {"id": 9213, "dom": "P", "ent": "vane", "cls": "bridge"}, {"id": 9214, "dom": "P", "ent": "vane", "cls": "bud"}, {"id": 9215, "dom": "P", "ent": "vane", "cls": "cat"}, {"id": 9216, "dom": "P", "ent": "vane", "cls": "chapel"}, {"id": 9217, "dom": "P", "ent": "vane", "cls": "chick"}, {"id": 9218, "dom": "P", "ent": "vane", "cls": "clock"}, {"id": 9219, "dom": "P", "ent": "vane", "cls": "deer"}, {"id": 9220, "dom": "P", "ent": "vane", "cls": "dog"}, {"id": 9221, "dom": "P", "ent": "vane", "cls": "elephant"}, {"id": 9222, "dom": "P", "ent": "vane", "cls": "embassy"}, {"id": 9223, "dom": "P", "ent": "vane", "cls": "garage"}, {"id": 9224, "dom": "P", "ent": "vane", "cls": "highway"}, {"id": 9225, "dom": "P", "ent": "vane", "cls": "kidney"}, {"id": 9226, "dom": "P", "ent": "vane", "cls": "lamb"}, {"id": 9227, "dom": "P", "ent": "vane", "cls": "male"}, {"id": 9228, "dom": "P", "ent": "vane", "cls": "marijuana"}, {"id": 9229, "dom": "P", "ent": "vane", "cls": "puppy"}, {"id": 9230, "dom": "P", "ent": "vane", "cls": "receiver"}, {"id": 9231, "dom": "P", "ent": "vane", "cls": "reed"}, {"id": 9232, "dom": "P", "ent": "vane", "cls": "rifle"}, {"id": 9233, "dom": "P", "ent": "vane", "cls": "road"}, {"id": 9234, "dom": "P", "ent": "vane", "cls": "salmon"}, {"id": 9235, "dom": "P", "ent": "vane", "cls": "shark"}, {"id": 9236, "dom": "P", "ent": "vane", "cls": "sheep"}, {"id": 9237, "dom": "P", "ent": "vane", "cls": "shook"}, {"id": 9238, "dom": "P", "ent": "vane", "cls": "tank"}, {"id": 9239, "dom": "P", "ent": "vane", "cls": "wheat"}, {"id": 9240, "dom": "P", "ent": "variation", "cls": "architecture"}, {"id": 9241, "dom": "P", "ent": "variation", "cls": "bacteria"}, {"id": 9242, "dom": "P", "ent": "variation", "cls": "beef"}, {"id": 9243, "dom": "P", "ent": "variation", "cls": "bridge"}, {"id": 9244, "dom": "P", "ent": "variation", "cls": "bud"}, {"id": 9245, "dom": "P", "ent": "variation", "cls": "cat"}, {"id": 9246, "dom": "P", "ent": "variation", "cls": "chapel"}, {"id": 9247, "dom": "P", "ent": "variation", "cls": "chick"}, {"id": 9248, "dom": "P", "ent": "variation", "cls": "clock"}, {"id": 9249, "dom": "P", "ent": "variation", "cls": "deer"}, {"id": 9250, "dom": "P", "ent": "variation", "cls": "dog"}, {"id": 9251, "dom": "P", "ent": "variation", "cls": "elephant"}, {"id": 9252, "dom": "P", "ent": "variation", "cls": "embassy"}, {"id": 9253, "dom": "P", "ent": "variation", "cls": "garage"}, {"id": 9254, "dom": "P", "ent": "variation", "cls": "highway"}, {"id": 9255, "dom": "P", "ent": "variation", "cls": "kidney"}, {"id": 9256, "dom": "P", "ent": "variation", "cls": "lamb"}, {"id": 9257, "dom": "P", "ent": "variation", "cls": "male"}, {"id": 9258, "dom": "P", "ent": "variation", "cls": "marijuana"}, {"id": 9259, "dom": "P", "ent": "variation", "cls": "puppy"}, {"id": 9260, "dom": "P", "ent": "variation", "cls": "receiver"}, {"id": 9261, "dom": "P", "ent": "variation", "cls": "reed"}, {"id": 9262, "dom": "P", "ent": "variation", "cls": "rifle"}, {"id": 9263, "dom": "P", "ent": "variation", "cls": "road"}, {"id": 9264, "dom": "P", "ent": "variation", "cls": "salmon"}, {"id": 9265, "dom": "P", "ent": "variation", "cls": "shark"}, {"id": 9266, "dom": "P", "ent": "variation", "cls": "sheep"}, {"id": 9267, "dom": "P", "ent": "variation", "cls": "shook"}, {"id": 9268, "dom": "P", "ent": "variation", "cls": "tank"}, {"id": 9269, "dom": "P", "ent": "variation", "cls": "wheat"}, {"id": 9270, "dom": "P", "ent": "vas", "cls": "architecture"}, {"id": 9271, "dom": "P", "ent": "vas", "cls": "bacteria"}, {"id": 9272, "dom": "P", "ent": "vas", "cls": "beef"}, {"id": 9273, "dom": "P", "ent": "vas", "cls": "bridge"}, {"id": 9274, "dom": "P", "ent": "vas", "cls": "bud"}, {"id": 9275, "dom": "P", "ent": "vas", "cls": "cat"}, {"id": 9276, "dom": "P", "ent": "vas", "cls": "chapel"}, {"id": 9277, "dom": "P", "ent": "vas", "cls": "chick"}, {"id": 9278, "dom": "P", "ent": "vas", "cls": "clock"}, {"id": 9279, "dom": "P", "ent": "vas", "cls": "deer"}, {"id": 9280, "dom": "P", "ent": "vas", "cls": "dog"}, {"id": 9281, "dom": "P", "ent": "vas", "cls": "elephant"}, {"id": 9282, "dom": "P", "ent": "vas", "cls": "embassy"}, {"id": 9283, "dom": "P", "ent": "vas", "cls": "garage"}, {"id": 9284, "dom": "P", "ent": "vas", "cls": "highway"}, {"id": 9285, "dom": "P", "ent": "vas", "cls": "kidney"}, {"id": 9286, "dom": "P", "ent": "vas", "cls": "lamb"}, {"id": 9287, "dom": "P", "ent": "vas", "cls": "male"}, {"id": 9288, "dom": "P", "ent": "vas", "cls": "marijuana"}, {"id": 9289, "dom": "P", "ent": "vas", "cls": "puppy"}, {"id": 9290, "dom": "P", "ent": "vas", "cls": "receiver"}, {"id": 9291, "dom": "P", "ent": "vas", "cls": "reed"}, {"id": 9292, "dom": "P", "ent": "vas", "cls": "rifle"}, {"id": 9293, "dom": "P", "ent": "vas", "cls": "road"}, {"id": 9294, "dom": "P", "ent": "vas", "cls": "salmon"}, {"id": 9295, "dom": "P", "ent": "vas", "cls": "shark"}, {"id": 9296, "dom": "P", "ent": "vas", "cls": "sheep"}, {"id": 9297, "dom": "P", "ent": "vas", "cls": "shook"}, {"id": 9298, "dom": "P", "ent": "vas", "cls": "tank"}, {"id": 9299, "dom": "P", "ent": "vas", "cls": "wheat"}, {"id": 9300, "dom": "P", "ent": "vent-hole", "cls": "architecture"}, {"id": 9301, "dom": "P", "ent": "vent-hole", "cls": "bacteria"}, {"id": 9302, "dom": "P", "ent": "vent-hole", "cls": "beef"}, {"id": 9303, "dom": "P", "ent": "vent-hole", "cls": "bridge"}, {"id": 9304, "dom": "P", "ent": "vent-hole", "cls": "bud"}, {"id": 9305, "dom": "P", "ent": "vent-hole", "cls": "cat"}, {"id": 9306, "dom": "P", "ent": "vent-hole", "cls": "chapel"}, {"id": 9307, "dom": "P", "ent": "vent-hole", "cls": "chick"}, {"id": 9308, "dom": "P", "ent": "vent-hole", "cls": "clock"}, {"id": 9309, "dom": "P", "ent": "vent-hole", "cls": "deer"}, {"id": 9310, "dom": "P", "ent": "vent-hole", "cls": "dog"}, {"id": 9311, "dom": "P", "ent": "vent-hole", "cls": "elephant"}, {"id": 9312, "dom": "P", "ent": "vent-hole", "cls": "embassy"}, {"id": 9313, "dom": "P", "ent": "vent-hole", "cls": "garage"}, {"id": 9314, "dom": "P", "ent": "vent-hole", "cls": "highway"}, {"id": 9315, "dom": "P", "ent": "vent-hole", "cls": "kidney"}, {"id": 9316, "dom": "P", "ent": "vent-hole", "cls": "lamb"}, {"id": 9317, "dom": "P", "ent": "vent-hole", "cls": "male"}, {"id": 9318, "dom": "P", "ent": "vent-hole", "cls": "marijuana"}, {"id": 9319, "dom": "P", "ent": "vent-hole", "cls": "puppy"}, {"id": 9320, "dom": "P", "ent": "vent-hole", "cls": "receiver"}, {"id": 9321, "dom": "P", "ent": "vent-hole", "cls": "reed"}, {"id": 9322, "dom": "P", "ent": "vent-hole", "cls": "rifle"}, {"id": 9323, "dom": "P", "ent": "vent-hole", "cls": "road"}, {"id": 9324, "dom": "P", "ent": "vent-hole", "cls": "salmon"}, {"id": 9325, "dom": "P", "ent": "vent-hole", "cls": "shark"}, {"id": 9326, "dom": "P", "ent": "vent-hole", "cls": "sheep"}, {"id": 9327, "dom": "P", "ent": "vent-hole", "cls": "shook"}, {"id": 9328, "dom": "P", "ent": "vent-hole", "cls": "tank"}, {"id": 9329, "dom": "P", "ent": "vent-hole", "cls": "wheat"}, {"id": 9330, "dom": "P", "ent": "vertical", "cls": "architecture"}, {"id": 9331, "dom": "P", "ent": "vertical", "cls": "bacteria"}, {"id": 9332, "dom": "P", "ent": "vertical", "cls": "beef"}, {"id": 9333, "dom": "P", "ent": "vertical", "cls": "bridge"}, {"id": 9334, "dom": "P", "ent": "vertical", "cls": "bud"}, {"id": 9335, "dom": "P", "ent": "vertical", "cls": "cat"}, {"id": 9336, "dom": "P", "ent": "vertical", "cls": "chapel"}, {"id": 9337, "dom": "P", "ent": "vertical", "cls": "chick"}, {"id": 9338, "dom": "P", "ent": "vertical", "cls": "clock"}, {"id": 9339, "dom": "P", "ent": "vertical", "cls": "deer"}, {"id": 9340, "dom": "P", "ent": "vertical", "cls": "dog"}, {"id": 9341, "dom": "P", "ent": "vertical", "cls": "elephant"}, {"id": 9342, "dom": "P", "ent": "vertical", "cls": "embassy"}, {"id": 9343, "dom": "P", "ent": "vertical", "cls": "garage"}, {"id": 9344, "dom": "P", "ent": "vertical", "cls": "highway"}, {"id": 9345, "dom": "P", "ent": "vertical", "cls": "kidney"}, {"id": 9346, "dom": "P", "ent": "vertical", "cls": "lamb"}, {"id": 9347, "dom": "P", "ent": "vertical", "cls": "male"}, {"id": 9348, "dom": "P", "ent": "vertical", "cls": "marijuana"}, {"id": 9349, "dom": "P", "ent": "vertical", "cls": "puppy"}, {"id": 9350, "dom": "P", "ent": "vertical", "cls": "receiver"}, {"id": 9351, "dom": "P", "ent": "vertical", "cls": "reed"}, {"id": 9352, "dom": "P", "ent": "vertical", "cls": "rifle"}, {"id": 9353, "dom": "P", "ent": "vertical", "cls": "road"}, {"id": 9354, "dom": "P", "ent": "vertical", "cls": "salmon"}, {"id": 9355, "dom": "P", "ent": "vertical", "cls": "shark"}, {"id": 9356, "dom": "P", "ent": "vertical", "cls": "sheep"}, {"id": 9357, "dom": "P", "ent": "vertical", "cls": "shook"}, {"id": 9358, "dom": "P", "ent": "vertical", "cls": "tank"}, {"id": 9359, "dom": "P", "ent": "vertical", "cls": "wheat"}, {"id": 9360, "dom": "P", "ent": "vessel", "cls": "architecture"}, {"id": 9361, "dom": "P", "ent": "vessel", "cls": "bacteria"}, {"id": 9362, "dom": "P", "ent": "vessel", "cls": "beef"}, {"id": 9363, "dom": "P", "ent": "vessel", "cls": "bridge"}, {"id": 9364, "dom": "P", "ent": "vessel", "cls": "bud"}, {"id": 9365, "dom": "P", "ent": "vessel", "cls": "cat"}, {"id": 9366, "dom": "P", "ent": "vessel", "cls": "chapel"}, {"id": 9367, "dom": "P", "ent": "vessel", "cls": "chick"}, {"id": 9368, "dom": "P", "ent": "vessel", "cls": "clock"}, {"id": 9369, "dom": "P", "ent": "vessel", "cls": "deer"}, {"id": 9370, "dom": "P", "ent": "vessel", "cls": "dog"}, {"id": 9371, "dom": "P", "ent": "vessel", "cls": "elephant"}, {"id": 9372, "dom": "P", "ent": "vessel", "cls": "embassy"}, {"id": 9373, "dom": "P", "ent": "vessel", "cls": "garage"}, {"id": 9374, "dom": "P", "ent": "vessel", "cls": "highway"}, {"id": 9375, "dom": "P", "ent": "vessel", "cls": "kidney"}, {"id": 9376, "dom": "P", "ent": "vessel", "cls": "lamb"}, {"id": 9377, "dom": "P", "ent": "vessel", "cls": "male"}, {"id": 9378, "dom": "P", "ent": "vessel", "cls": "marijuana"}, {"id": 9379, "dom": "P", "ent": "vessel", "cls": "puppy"}, {"id": 9380, "dom": "P", "ent": "vessel", "cls": "receiver"}, {"id": 9381, "dom": "P", "ent": "vessel", "cls": "reed"}, {"id": 9382, "dom": "P", "ent": "vessel", "cls": "rifle"}, {"id": 9383, "dom": "P", "ent": "vessel", "cls": "road"}, {"id": 9384, "dom": "P", "ent": "vessel", "cls": "salmon"}, {"id": 9385, "dom": "P", "ent": "vessel", "cls": "shark"}, {"id": 9386, "dom": "P", "ent": "vessel", "cls": "sheep"}, {"id": 9387, "dom": "P", "ent": "vessel", "cls": "shook"}, {"id": 9388, "dom": "P", "ent": "vessel", "cls": "tank"}, {"id": 9389, "dom": "P", "ent": "vessel", "cls": "wheat"}, {"id": 9390, "dom": "P", "ent": "volleyball", "cls": "architecture"}, {"id": 9391, "dom": "P", "ent": "volleyball", "cls": "bacteria"}, {"id": 9392, "dom": "P", "ent": "volleyball", "cls": "beef"}, {"id": 9393, "dom": "P", "ent": "volleyball", "cls": "bridge"}, {"id": 9394, "dom": "P", "ent": "volleyball", "cls": "bud"}, {"id": 9395, "dom": "P", "ent": "volleyball", "cls": "cat"}, {"id": 9396, "dom": "P", "ent": "volleyball", "cls": "chapel"}, {"id": 9397, "dom": "P", "ent": "volleyball", "cls": "chick"}, {"id": 9398, "dom": "P", "ent": "volleyball", "cls": "clock"}, {"id": 9399, "dom": "P", "ent": "volleyball", "cls": "deer"}, {"id": 9400, "dom": "P", "ent": "volleyball", "cls": "dog"}, {"id": 9401, "dom": "P", "ent": "volleyball", "cls": "elephant"}, {"id": 9402, "dom": "P", "ent": "volleyball", "cls": "embassy"}, {"id": 9403, "dom": "P", "ent": "volleyball", "cls": "garage"}, {"id": 9404, "dom": "P", "ent": "volleyball", "cls": "highway"}, {"id": 9405, "dom": "P", "ent": "volleyball", "cls": "kidney"}, {"id": 9406, "dom": "P", "ent": "volleyball", "cls": "lamb"}, {"id": 9407, "dom": "P", "ent": "volleyball", "cls": "male"}, {"id": 9408, "dom": "P", "ent": "volleyball", "cls": "marijuana"}, {"id": 9409, "dom": "P", "ent": "volleyball", "cls": "puppy"}, {"id": 9410, "dom": "P", "ent": "volleyball", "cls": "receiver"}, {"id": 9411, "dom": "P", "ent": "volleyball", "cls": "reed"}, {"id": 9412, "dom": "P", "ent": "volleyball", "cls": "rifle"}, {"id": 9413, "dom": "P", "ent": "volleyball", "cls": "road"}, {"id": 9414, "dom": "P", "ent": "volleyball", "cls": "salmon"}, {"id": 9415, "dom": "P", "ent": "volleyball", "cls": "shark"}, {"id": 9416, "dom": "P", "ent": "volleyball", "cls": "sheep"}, {"id": 9417, "dom": "P", "ent": "volleyball", "cls": "shook"}, {"id": 9418, "dom": "P", "ent": "volleyball", "cls": "tank"}, {"id": 9419, "dom": "P", "ent": "volleyball", "cls": "wheat"}, {"id": 9420, "dom": "P", "ent": "wall", "cls": "architecture"}, {"id": 9421, "dom": "P", "ent": "wall", "cls": "bacteria"}, {"id": 9422, "dom": "P", "ent": "wall", "cls": "beef"}, {"id": 9423, "dom": "P", "ent": "wall", "cls": "bridge"}, {"id": 9424, "dom": "P", "ent": "wall", "cls": "bud"}, {"id": 9425, "dom": "P", "ent": "wall", "cls": "cat"}, {"id": 9426, "dom": "P", "ent": "wall", "cls": "chapel"}, {"id": 9427, "dom": "P", "ent": "wall", "cls": "chick"}, {"id": 9428, "dom": "P", "ent": "wall", "cls": "clock"}, {"id": 9429, "dom": "P", "ent": "wall", "cls": "deer"}, {"id": 9430, "dom": "P", "ent": "wall", "cls": "dog"}, {"id": 9431, "dom": "P", "ent": "wall", "cls": "elephant"}, {"id": 9432, "dom": "P", "ent": "wall", "cls": "embassy"}, {"id": 9433, "dom": "P", "ent": "wall", "cls": "garage"}, {"id": 9434, "dom": "P", "ent": "wall", "cls": "highway"}, {"id": 9435, "dom": "P", "ent": "wall", "cls": "kidney"}, {"id": 9436, "dom": "P", "ent": "wall", "cls": "lamb"}, {"id": 9437, "dom": "P", "ent": "wall", "cls": "male"}, {"id": 9438, "dom": "P", "ent": "wall", "cls": "marijuana"}, {"id": 9439, "dom": "P", "ent": "wall", "cls": "puppy"}, {"id": 9440, "dom": "P", "ent": "wall", "cls": "receiver"}, {"id": 9441, "dom": "P", "ent": "wall", "cls": "reed"}, {"id": 9442, "dom": "P", "ent": "wall", "cls": "rifle"}, {"id": 9443, "dom": "P", "ent": "wall", "cls": "road"}, {"id": 9444, "dom": "P", "ent": "wall", "cls": "salmon"}, {"id": 9445, "dom": "P", "ent": "wall", "cls": "shark"}, {"id": 9446, "dom": "P", "ent": "wall", "cls": "sheep"}, {"id": 9447, "dom": "P", "ent": "wall", "cls": "shook"}, {"id": 9448, "dom": "P", "ent": "wall", "cls": "tank"}, {"id": 9449, "dom": "P", "ent": "wall", "cls": "wheat"}, {"id": 9450, "dom": "P", "ent": "wave", "cls": "architecture"}, {"id": 9451, "dom": "P", "ent": "wave", "cls": "bacteria"}, {"id": 9452, "dom": "P", "ent": "wave", "cls": "beef"}, {"id": 9453, "dom": "P", "ent": "wave", "cls": "bridge"}, {"id": 9454, "dom": "P", "ent": "wave", "cls": "bud"}, {"id": 9455, "dom": "P", "ent": "wave", "cls": "cat"}, {"id": 9456, "dom": "P", "ent": "wave", "cls": "chapel"}, {"id": 9457, "dom": "P", "ent": "wave", "cls": "chick"}, {"id": 9458, "dom": "P", "ent": "wave", "cls": "clock"}, {"id": 9459, "dom": "P", "ent": "wave", "cls": "deer"}, {"id": 9460, "dom": "P", "ent": "wave", "cls": "dog"}, {"id": 9461, "dom": "P", "ent": "wave", "cls": "elephant"}, {"id": 9462, "dom": "P", "ent": "wave", "cls": "embassy"}, {"id": 9463, "dom": "P", "ent": "wave", "cls": "garage"}, {"id": 9464, "dom": "P", "ent": "wave", "cls": "highway"}, {"id": 9465, "dom": "P", "ent": "wave", "cls": "kidney"}, {"id": 9466, "dom": "P", "ent": "wave", "cls": "lamb"}, {"id": 9467, "dom": "P", "ent": "wave", "cls": "male"}, {"id": 9468, "dom": "P", "ent": "wave", "cls": "marijuana"}, {"id": 9469, "dom": "P", "ent": "wave", "cls": "puppy"}, {"id": 9470, "dom": "P", "ent": "wave", "cls": "receiver"}, {"id": 9471, "dom": "P", "ent": "wave", "cls": "reed"}, {"id": 9472, "dom": "P", "ent": "wave", "cls": "rifle"}, {"id": 9473, "dom": "P", "ent": "wave", "cls": "road"}, {"id": 9474, "dom": "P", "ent": "wave", "cls": "salmon"}, {"id": 9475, "dom": "P", "ent": "wave", "cls": "shark"}, {"id": 9476, "dom": "P", "ent": "wave", "cls": "sheep"}, {"id": 9477, "dom": "P", "ent": "wave", "cls": "shook"}, {"id": 9478, "dom": "P", "ent": "wave", "cls": "tank"}, {"id": 9479, "dom": "P", "ent": "wave", "cls": "wheat"}, {"id": 9480, "dom": "P", "ent": "web", "cls": "architecture"}, {"id": 9481, "dom": "P", "ent": "web", "cls": "bacteria"}, {"id": 9482, "dom": "P", "ent": "web", "cls": "beef"}, {"id": 9483, "dom": "P", "ent": "web", "cls": "bridge"}, {"id": 9484, "dom": "P", "ent": "web", "cls": "bud"}, {"id": 9485, "dom": "P", "ent": "web", "cls": "cat"}, {"id": 9486, "dom": "P", "ent": "web", "cls": "chapel"}, {"id": 9487, "dom": "P", "ent": "web", "cls": "chick"}, {"id": 9488, "dom": "P", "ent": "web", "cls": "clock"}, {"id": 9489, "dom": "P", "ent": "web", "cls": "deer"}, {"id": 9490, "dom": "P", "ent": "web", "cls": "dog"}, {"id": 9491, "dom": "P", "ent": "web", "cls": "elephant"}, {"id": 9492, "dom": "P", "ent": "web", "cls": "embassy"}, {"id": 9493, "dom": "P", "ent": "web", "cls": "garage"}, {"id": 9494, "dom": "P", "ent": "web", "cls": "highway"}, {"id": 9495, "dom": "P", "ent": "web", "cls": "kidney"}, {"id": 9496, "dom": "P", "ent": "web", "cls": "lamb"}, {"id": 9497, "dom": "P", "ent": "web", "cls": "male"}, {"id": 9498, "dom": "P", "ent": "web", "cls": "marijuana"}, {"id": 9499, "dom": "P", "ent": "web", "cls": "puppy"}, {"id": 9500, "dom": "P", "ent": "web", "cls": "receiver"}, {"id": 9501, "dom": "P", "ent": "web", "cls": "reed"}, {"id": 9502, "dom": "P", "ent": "web", "cls": "rifle"}, {"id": 9503, "dom": "P", "ent": "web", "cls": "road"}, {"id": 9504, "dom": "P", "ent": "web", "cls": "salmon"}, {"id": 9505, "dom": "P", "ent": "web", "cls": "shark"}, {"id": 9506, "dom": "P", "ent": "web", "cls": "sheep"}, {"id": 9507, "dom": "P", "ent": "web", "cls": "shook"}, {"id": 9508, "dom": "P", "ent": "web", "cls": "tank"}, {"id": 9509, "dom": "P", "ent": "web", "cls": "wheat"}, {"id": 9510, "dom": "P", "ent": "weed", "cls": "architecture"}, {"id": 9511, "dom": "P", "ent": "weed", "cls": "bacteria"}, {"id": 9512, "dom": "P", "ent": "weed", "cls": "beef"}, {"id": 9513, "dom": "P", "ent": "weed", "cls": "bridge"}, {"id": 9514, "dom": "P", "ent": "weed", "cls": "bud"}, {"id": 9515, "dom": "P", "ent": "weed", "cls": "cat"}, {"id": 9516, "dom": "P", "ent": "weed", "cls": "chapel"}, {"id": 9517, "dom": "P", "ent": "weed", "cls": "chick"}, {"id": 9518, "dom": "P", "ent": "weed", "cls": "clock"}, {"id": 9519, "dom": "P", "ent": "weed", "cls": "deer"}, {"id": 9520, "dom": "P", "ent": "weed", "cls": "dog"}, {"id": 9521, "dom": "P", "ent": "weed", "cls": "elephant"}, {"id": 9522, "dom": "P", "ent": "weed", "cls": "embassy"}, {"id": 9523, "dom": "P", "ent": "weed", "cls": "garage"}, {"id": 9524, "dom": "P", "ent": "weed", "cls": "highway"}, {"id": 9525, "dom": "P", "ent": "weed", "cls": "kidney"}, {"id": 9526, "dom": "P", "ent": "weed", "cls": "lamb"}, {"id": 9527, "dom": "P", "ent": "weed", "cls": "male"}, {"id": 9528, "dom": "P", "ent": "weed", "cls": "marijuana"}, {"id": 9529, "dom": "P", "ent": "weed", "cls": "puppy"}, {"id": 9530, "dom": "P", "ent": "weed", "cls": "receiver"}, {"id": 9531, "dom": "P", "ent": "weed", "cls": "reed"}, {"id": 9532, "dom": "P", "ent": "weed", "cls": "rifle"}, {"id": 9533, "dom": "P", "ent": "weed", "cls": "road"}, {"id": 9534, "dom": "P", "ent": "weed", "cls": "salmon"}, {"id": 9535, "dom": "P", "ent": "weed", "cls": "shark"}, {"id": 9536, "dom": "P", "ent": "weed", "cls": "sheep"}, {"id": 9537, "dom": "P", "ent": "weed", "cls": "shook"}, {"id": 9538, "dom": "P", "ent": "weed", "cls": "tank"}, {"id": 9539, "dom": "P", "ent": "weed", "cls": "wheat"}, {"id": 9540, "dom": "P", "ent": "weeds", "cls": "architecture"}, {"id": 9541, "dom": "P", "ent": "weeds", "cls": "bacteria"}, {"id": 9542, "dom": "P", "ent": "weeds", "cls": "beef"}, {"id": 9543, "dom": "P", "ent": "weeds", "cls": "bridge"}, {"id": 9544, "dom": "P", "ent": "weeds", "cls": "bud"}, {"id": 9545, "dom": "P", "ent": "weeds", "cls": "cat"}, {"id": 9546, "dom": "P", "ent": "weeds", "cls": "chapel"}, {"id": 9547, "dom": "P", "ent": "weeds", "cls": "chick"}, {"id": 9548, "dom": "P", "ent": "weeds", "cls": "clock"}, {"id": 9549, "dom": "P", "ent": "weeds", "cls": "deer"}, {"id": 9550, "dom": "P", "ent": "weeds", "cls": "dog"}, {"id": 9551, "dom": "P", "ent": "weeds", "cls": "elephant"}, {"id": 9552, "dom": "P", "ent": "weeds", "cls": "embassy"}, {"id": 9553, "dom": "P", "ent": "weeds", "cls": "garage"}, {"id": 9554, "dom": "P", "ent": "weeds", "cls": "highway"}, {"id": 9555, "dom": "P", "ent": "weeds", "cls": "kidney"}, {"id": 9556, "dom": "P", "ent": "weeds", "cls": "lamb"}, {"id": 9557, "dom": "P", "ent": "weeds", "cls": "male"}, {"id": 9558, "dom": "P", "ent": "weeds", "cls": "marijuana"}, {"id": 9559, "dom": "P", "ent": "weeds", "cls": "puppy"}, {"id": 9560, "dom": "P", "ent": "weeds", "cls": "receiver"}, {"id": 9561, "dom": "P", "ent": "weeds", "cls": "reed"}, {"id": 9562, "dom": "P", "ent": "weeds", "cls": "rifle"}, {"id": 9563, "dom": "P", "ent": "weeds", "cls": "road"}, {"id": 9564, "dom": "P", "ent": "weeds", "cls": "salmon"}, {"id": 9565, "dom": "P", "ent": "weeds", "cls": "shark"}, {"id": 9566, "dom": "P", "ent": "weeds", "cls": "sheep"}, {"id": 9567, "dom": "P", "ent": "weeds", "cls": "shook"}, {"id": 9568, "dom": "P", "ent": "weeds", "cls": "tank"}, {"id": 9569, "dom": "P", "ent": "weeds", "cls": "wheat"}, {"id": 9570, "dom": "P", "ent": "wheel", "cls": "architecture"}, {"id": 9571, "dom": "P", "ent": "wheel", "cls": "bacteria"}, {"id": 9572, "dom": "P", "ent": "wheel", "cls": "beef"}, {"id": 9573, "dom": "P", "ent": "wheel", "cls": "bridge"}, {"id": 9574, "dom": "P", "ent": "wheel", "cls": "bud"}, {"id": 9575, "dom": "P", "ent": "wheel", "cls": "cat"}, {"id": 9576, "dom": "P", "ent": "wheel", "cls": "chapel"}, {"id": 9577, "dom": "P", "ent": "wheel", "cls": "chick"}, {"id": 9578, "dom": "P", "ent": "wheel", "cls": "clock"}, {"id": 9579, "dom": "P", "ent": "wheel", "cls": "deer"}, {"id": 9580, "dom": "P", "ent": "wheel", "cls": "dog"}, {"id": 9581, "dom": "P", "ent": "wheel", "cls": "elephant"}, {"id": 9582, "dom": "P", "ent": "wheel", "cls": "embassy"}, {"id": 9583, "dom": "P", "ent": "wheel", "cls": "garage"}, {"id": 9584, "dom": "P", "ent": "wheel", "cls": "highway"}, {"id": 9585, "dom": "P", "ent": "wheel", "cls": "kidney"}, {"id": 9586, "dom": "P", "ent": "wheel", "cls": "lamb"}, {"id": 9587, "dom": "P", "ent": "wheel", "cls": "male"}, {"id": 9588, "dom": "P", "ent": "wheel", "cls": "marijuana"}, {"id": 9589, "dom": "P", "ent": "wheel", "cls": "puppy"}, {"id": 9590, "dom": "P", "ent": "wheel", "cls": "receiver"}, {"id": 9591, "dom": "P", "ent": "wheel", "cls": "reed"}, {"id": 9592, "dom": "P", "ent": "wheel", "cls": "rifle"}, {"id": 9593, "dom": "P", "ent": "wheel", "cls": "road"}, {"id": 9594, "dom": "P", "ent": "wheel", "cls": "salmon"}, {"id": 9595, "dom": "P", "ent": "wheel", "cls": "shark"}, {"id": 9596, "dom": "P", "ent": "wheel", "cls": "sheep"}, {"id": 9597, "dom": "P", "ent": "wheel", "cls": "shook"}, {"id": 9598, "dom": "P", "ent": "wheel", "cls": "tank"}, {"id": 9599, "dom": "P", "ent": "wheel", "cls": "wheat"}, {"id": 9600, "dom": "P", "ent": "widening", "cls": "architecture"}, {"id": 9601, "dom": "P", "ent": "widening", "cls": "bacteria"}, {"id": 9602, "dom": "P", "ent": "widening", "cls": "beef"}, {"id": 9603, "dom": "P", "ent": "widening", "cls": "bridge"}, {"id": 9604, "dom": "P", "ent": "widening", "cls": "bud"}, {"id": 9605, "dom": "P", "ent": "widening", "cls": "cat"}, {"id": 9606, "dom": "P", "ent": "widening", "cls": "chapel"}, {"id": 9607, "dom": "P", "ent": "widening", "cls": "chick"}, {"id": 9608, "dom": "P", "ent": "widening", "cls": "clock"}, {"id": 9609, "dom": "P", "ent": "widening", "cls": "deer"}, {"id": 9610, "dom": "P", "ent": "widening", "cls": "dog"}, {"id": 9611, "dom": "P", "ent": "widening", "cls": "elephant"}, {"id": 9612, "dom": "P", "ent": "widening", "cls": "embassy"}, {"id": 9613, "dom": "P", "ent": "widening", "cls": "garage"}, {"id": 9614, "dom": "P", "ent": "widening", "cls": "highway"}, {"id": 9615, "dom": "P", "ent": "widening", "cls": "kidney"}, {"id": 9616, "dom": "P", "ent": "widening", "cls": "lamb"}, {"id": 9617, "dom": "P", "ent": "widening", "cls": "male"}, {"id": 9618, "dom": "P", "ent": "widening", "cls": "marijuana"}, {"id": 9619, "dom": "P", "ent": "widening", "cls": "puppy"}, {"id": 9620, "dom": "P", "ent": "widening", "cls": "receiver"}, {"id": 9621, "dom": "P", "ent": "widening", "cls": "reed"}, {"id": 9622, "dom": "P", "ent": "widening", "cls": "rifle"}, {"id": 9623, "dom": "P", "ent": "widening", "cls": "road"}, {"id": 9624, "dom": "P", "ent": "widening", "cls": "salmon"}, {"id": 9625, "dom": "P", "ent": "widening", "cls": "shark"}, {"id": 9626, "dom": "P", "ent": "widening", "cls": "sheep"}, {"id": 9627, "dom": "P", "ent": "widening", "cls": "shook"}, {"id": 9628, "dom": "P", "ent": "widening", "cls": "tank"}, {"id": 9629, "dom": "P", "ent": "widening", "cls": "wheat"}, {"id": 9630, "dom": "P", "ent": "window", "cls": "architecture"}, {"id": 9631, "dom": "P", "ent": "window", "cls": "bacteria"}, {"id": 9632, "dom": "P", "ent": "window", "cls": "beef"}, {"id": 9633, "dom": "P", "ent": "window", "cls": "bridge"}, {"id": 9634, "dom": "P", "ent": "window", "cls": "bud"}, {"id": 9635, "dom": "P", "ent": "window", "cls": "cat"}, {"id": 9636, "dom": "P", "ent": "window", "cls": "chapel"}, {"id": 9637, "dom": "P", "ent": "window", "cls": "chick"}, {"id": 9638, "dom": "P", "ent": "window", "cls": "clock"}, {"id": 9639, "dom": "P", "ent": "window", "cls": "deer"}, {"id": 9640, "dom": "P", "ent": "window", "cls": "dog"}, {"id": 9641, "dom": "P", "ent": "window", "cls": "elephant"}, {"id": 9642, "dom": "P", "ent": "window", "cls": "embassy"}, {"id": 9643, "dom": "P", "ent": "window", "cls": "garage"}, {"id": 9644, "dom": "P", "ent": "window", "cls": "highway"}, {"id": 9645, "dom": "P", "ent": "window", "cls": "kidney"}, {"id": 9646, "dom": "P", "ent": "window", "cls": "lamb"}, {"id": 9647, "dom": "P", "ent": "window", "cls": "male"}, {"id": 9648, "dom": "P", "ent": "window", "cls": "marijuana"}, {"id": 9649, "dom": "P", "ent": "window", "cls": "puppy"}, {"id": 9650, "dom": "P", "ent": "window", "cls": "receiver"}, {"id": 9651, "dom": "P", "ent": "window", "cls": "reed"}, {"id": 9652, "dom": "P", "ent": "window", "cls": "rifle"}, {"id": 9653, "dom": "P", "ent": "window", "cls": "road"}, {"id": 9654, "dom": "P", "ent": "window", "cls": "salmon"}, {"id": 9655, "dom": "P", "ent": "window", "cls": "shark"}, {"id": 9656, "dom": "P", "ent": "window", "cls": "sheep"}, {"id": 9657, "dom": "P", "ent": "window", "cls": "shook"}, {"id": 9658, "dom": "P", "ent": "window", "cls": "tank"}, {"id": 9659, "dom": "P", "ent": "window", "cls": "wheat"}, {"id": 9660, "dom": "P", "ent": "wing", "cls": "architecture"}, {"id": 9661, "dom": "P", "ent": "wing", "cls": "bacteria"}, {"id": 9662, "dom": "P", "ent": "wing", "cls": "beef"}, {"id": 9663, "dom": "P", "ent": "wing", "cls": "bridge"}, {"id": 9664, "dom": "P", "ent": "wing", "cls": "bud"}, {"id": 9665, "dom": "P", "ent": "wing", "cls": "cat"}, {"id": 9666, "dom": "P", "ent": "wing", "cls": "chapel"}, {"id": 9667, "dom": "P", "ent": "wing", "cls": "chick"}, {"id": 9668, "dom": "P", "ent": "wing", "cls": "clock"}, {"id": 9669, "dom": "P", "ent": "wing", "cls": "deer"}, {"id": 9670, "dom": "P", "ent": "wing", "cls": "dog"}, {"id": 9671, "dom": "P", "ent": "wing", "cls": "elephant"}, {"id": 9672, "dom": "P", "ent": "wing", "cls": "embassy"}, {"id": 9673, "dom": "P", "ent": "wing", "cls": "garage"}, {"id": 9674, "dom": "P", "ent": "wing", "cls": "highway"}, {"id": 9675, "dom": "P", "ent": "wing", "cls": "kidney"}, {"id": 9676, "dom": "P", "ent": "wing", "cls": "lamb"}, {"id": 9677, "dom": "P", "ent": "wing", "cls": "male"}, {"id": 9678, "dom": "P", "ent": "wing", "cls": "marijuana"}, {"id": 9679, "dom": "P", "ent": "wing", "cls": "puppy"}, {"id": 9680, "dom": "P", "ent": "wing", "cls": "receiver"}, {"id": 9681, "dom": "P", "ent": "wing", "cls": "reed"}, {"id": 9682, "dom": "P", "ent": "wing", "cls": "rifle"}, {"id": 9683, "dom": "P", "ent": "wing", "cls": "road"}, {"id": 9684, "dom": "P", "ent": "wing", "cls": "salmon"}, {"id": 9685, "dom": "P", "ent": "wing", "cls": "shark"}, {"id": 9686, "dom": "P", "ent": "wing", "cls": "sheep"}, {"id": 9687, "dom": "P", "ent": "wing", "cls": "shook"}, {"id": 9688, "dom": "P", "ent": "wing", "cls": "tank"}, {"id": 9689, "dom": "P", "ent": "wing", "cls": "wheat"}, {"id": 9690, "dom": "P", "ent": "wire", "cls": "architecture"}, {"id": 9691, "dom": "P", "ent": "wire", "cls": "bacteria"}, {"id": 9692, "dom": "P", "ent": "wire", "cls": "beef"}, {"id": 9693, "dom": "P", "ent": "wire", "cls": "bridge"}, {"id": 9694, "dom": "P", "ent": "wire", "cls": "bud"}, {"id": 9695, "dom": "P", "ent": "wire", "cls": "cat"}, {"id": 9696, "dom": "P", "ent": "wire", "cls": "chapel"}, {"id": 9697, "dom": "P", "ent": "wire", "cls": "chick"}, {"id": 9698, "dom": "P", "ent": "wire", "cls": "clock"}, {"id": 9699, "dom": "P", "ent": "wire", "cls": "deer"}, {"id": 9700, "dom": "P", "ent": "wire", "cls": "dog"}, {"id": 9701, "dom": "P", "ent": "wire", "cls": "elephant"}, {"id": 9702, "dom": "P", "ent": "wire", "cls": "embassy"}, {"id": 9703, "dom": "P", "ent": "wire", "cls": "garage"}, {"id": 9704, "dom": "P", "ent": "wire", "cls": "highway"}, {"id": 9705, "dom": "P", "ent": "wire", "cls": "kidney"}, {"id": 9706, "dom": "P", "ent": "wire", "cls": "lamb"}, {"id": 9707, "dom": "P", "ent": "wire", "cls": "male"}, {"id": 9708, "dom": "P", "ent": "wire", "cls": "marijuana"}, {"id": 9709, "dom": "P", "ent": "wire", "cls": "puppy"}, {"id": 9710, "dom": "P", "ent": "wire", "cls": "receiver"}, {"id": 9711, "dom": "P", "ent": "wire", "cls": "reed"}, {"id": 9712, "dom": "P", "ent": "wire", "cls": "rifle"}, {"id": 9713, "dom": "P", "ent": "wire", "cls": "road"}, {"id": 9714, "dom": "P", "ent": "wire", "cls": "salmon"}, {"id": 9715, "dom": "P", "ent": "wire", "cls": "shark"}, {"id": 9716, "dom": "P", "ent": "wire", "cls": "sheep"}, {"id": 9717, "dom": "P", "ent": "wire", "cls": "shook"}, {"id": 9718, "dom": "P", "ent": "wire", "cls": "tank"}, {"id": 9719, "dom": "P", "ent": "wire", "cls": "wheat"}, {"id": 9720, "dom": "P", "ent": "work-clothing", "cls": "architecture"}, {"id": 9721, "dom": "P", "ent": "work-clothing", "cls": "bacteria"}, {"id": 9722, "dom": "P", "ent": "work-clothing", "cls": "beef"}, {"id": 9723, "dom": "P", "ent": "work-clothing", "cls": "bridge"}, {"id": 9724, "dom": "P", "ent": "work-clothing", "cls": "bud"}, {"id": 9725, "dom": "P", "ent": "work-clothing", "cls": "cat"}, {"id": 9726, "dom": "P", "ent": "work-clothing", "cls": "chapel"}, {"id": 9727, "dom": "P", "ent": "work-clothing", "cls": "chick"}, {"id": 9728, "dom": "P", "ent": "work-clothing", "cls": "clock"}, {"id": 9729, "dom": "P", "ent": "work-clothing", "cls": "deer"}, {"id": 9730, "dom": "P", "ent": "work-clothing", "cls": "dog"}, {"id": 9731, "dom": "P", "ent": "work-clothing", "cls": "elephant"}, {"id": 9732, "dom": "P", "ent": "work-clothing", "cls": "embassy"}, {"id": 9733, "dom": "P", "ent": "work-clothing", "cls": "garage"}, {"id": 9734, "dom": "P", "ent": "work-clothing", "cls": "highway"}, {"id": 9735, "dom": "P", "ent": "work-clothing", "cls": "kidney"}, {"id": 9736, "dom": "P", "ent": "work-clothing", "cls": "lamb"}, {"id": 9737, "dom": "P", "ent": "work-clothing", "cls": "male"}, {"id": 9738, "dom": "P", "ent": "work-clothing", "cls": "marijuana"}, {"id": 9739, "dom": "P", "ent": "work-clothing", "cls": "puppy"}, {"id": 9740, "dom": "P", "ent": "work-clothing", "cls": "receiver"}, {"id": 9741, "dom": "P", "ent": "work-clothing", "cls": "reed"}, {"id": 9742, "dom": "P", "ent": "work-clothing", "cls": "rifle"}, {"id": 9743, "dom": "P", "ent": "work-clothing", "cls": "road"}, {"id": 9744, "dom": "P", "ent": "work-clothing", "cls": "salmon"}, {"id": 9745, "dom": "P", "ent": "work-clothing", "cls": "shark"}, {"id": 9746, "dom": "P", "ent": "work-clothing", "cls": "sheep"}, {"id": 9747, "dom": "P", "ent": "work-clothing", "cls": "shook"}, {"id": 9748, "dom": "P", "ent": "work-clothing", "cls": "tank"}, {"id": 9749, "dom": "P", "ent": "work-clothing", "cls": "wheat"}, {"id": 9750, "dom": "D", "ent": "acute diarrhea", "cls": "cardio"}, {"id": 9751, "dom": "D", "ent": "acute diarrhea", "cls": "resp"}, {"id": 9752, "dom": "D", "ent": "acute diarrhea", "cls": "gi"}, {"id": 9753, "dom": "D", "ent": "acute diarrhea", "cls": "nerv"}, {"id": 9754, "dom": "D", "ent": "acute diarrhea", "cls": "skin"}, {"id": 9755, "dom": "D", "ent": "acute diarrhea", "cls": "musc"}, {"id": 9756, "dom": "D", "ent": "acute diarrhea", "cls": "urin"}, {"id": 9757, "dom": "D", "ent": "acute diarrhea", "cls": "endo"}, {"id": 9758, "dom": "D", "ent": "acute diarrhea", "cls": "immune"}, {"id": 9759, "dom": "D", "ent": "acute diarrhea", "cls": "blood"}, {"id": 9760, "dom": "D", "ent": "acute diarrhea", "cls": "repro"}, {"id": 9761, "dom": "D", "ent": "acute diarrhea", "cls": "breast"}, {"id": 9762, "dom": "D", "ent": "acute diarrhea", "cls": "chest"}, {"id": 9763, "dom": "D", "ent": "acute diarrhea", "cls": "metab"}, {"id": 9764, "dom": "D", "ent": "acute diarrhea", "cls": "infect"}, {"id": 9765, "dom": "D", "ent": "acute diarrhea", "cls": "tumor"}, {"id": 9766, "dom": "D", "ent": "acute diarrhea", "cls": "mental"}, {"id": 9767, "dom": "D", "ent": "acute diarrhea", "cls": "genetic"}, {"id": 9768, "dom": "D", "ent": "acute diarrhea", "cls": "bact"}, {"id": 9769, "dom": "D", "ent": "acute diarrhea", "cls": "viral"}, {"id": 9770, "dom": "D", "ent": "acute diarrhea", "cls": "fungal"}, {"id": 9771, "dom": "D", "ent": "acute diarrhea", "cls": "parasit"}, {"id": 9772, "dom": "D", "ent": "aspiration pneumonia", "cls": "cardio"}, {"id": 9773, "dom": "D", "ent": "aspiration pneumonia", "cls": "resp"}, {"id": 9774, "dom": "D", "ent": "aspiration pneumonia", "cls": "gi"}, {"id": 9775, "dom": "D", "ent": "aspiration pneumonia", "cls": "nerv"}, {"id": 9776, "dom": "D", "ent": "aspiration pneumonia", "cls": "skin"}, {"id": 9777, "dom": "D", "ent": "aspiration pneumonia", "cls": "musc"}, {"id": 9778, "dom": "D", "ent": "aspiration pneumonia", "cls": "urin"}, {"id": 9779, "dom": "D", "ent": "aspiration pneumonia", "cls": "endo"}, {"id": 9780, "dom": "D", "ent": "aspiration pneumonia", "cls": "immune"}, {"id": 9781, "dom": "D", "ent": "aspiration pneumonia", "cls": "blood"}, {"id": 9782, "dom": "D", "ent": "aspiration pneumonia", "cls": "repro"}, {"id": 9783, "dom": "D", "ent": "aspiration pneumonia", "cls": "breast"}, {"id": 9784, "dom": "D", "ent": "aspiration pneumonia", "cls": "chest"}, {"id": 9785, "dom": "D", "ent": "aspiration pneumonia", "cls": "metab"}, {"id": 9786, "dom": "D", "ent": "aspiration pneumonia", "cls": "infect"}, {"id": 9787, "dom": "D", "ent": "aspiration pneumonia", "cls": "tumor"}, {"id": 9788, "dom": "D", "ent": "aspiration pneumonia", "cls": "mental"}, {"id": 9789, "dom": "D", "ent": "aspiration pneumonia", "cls": "genetic"}, {"id": 9790, "dom": "D", "ent": "aspiration pneumonia", "cls": "bact"}, {"id": 9791, "dom": "D", "ent": "aspiration pneumonia", "cls": "viral"}, {"id": 9792, "dom": "D", "ent": "aspiration pneumonia", "cls": "fungal"}, {"id": 9793, "dom": "D", "ent": "aspiration pneumonia", "cls": "parasit"}, {"id": 9794, "dom": "D", "ent": "swine influenza", "cls": "cardio"}, {"id": 9795, "dom": "D", "ent": "swine influenza", "cls": "resp"}, {"id": 9796, "dom": "D", "ent": "swine influenza", "cls": "gi"}, {"id": 9797, "dom": "D", "ent": "swine influenza", "cls": "nerv"}, {"id": 9798, "dom": "D", "ent": "swine influenza", "cls": "skin"}, {"id": 9799, "dom": "D", "ent": "swine influenza", "cls": "musc"}, {"id": 9800, "dom": "D", "ent": "swine influenza", "cls": "urin"}, {"id": 9801, "dom": "D", "ent": "swine influenza", "cls": "endo"}, {"id": 9802, "dom": "D", "ent": "swine influenza", "cls": "immune"}, {"id": 9803, "dom": "D", "ent": "swine influenza", "cls": "blood"}, {"id": 9804, "dom": "D", "ent": "swine influenza", "cls": "repro"}, {"id": 9805, "dom": "D", "ent": "swine influenza", "cls": "breast"}, {"id": 9806, "dom": "D", "ent": "swine influenza", "cls": "chest"}, {"id": 9807, "dom": "D", "ent": "swine influenza", "cls": "metab"}, {"id": 9808, "dom": "D", "ent": "swine influenza", "cls": "infect"}, {"id": 9809, "dom": "D", "ent": "swine influenza", "cls": "tumor"}, {"id": 9810, "dom": "D", "ent": "swine influenza", "cls": "mental"}, {"id": 9811, "dom": "D", "ent": "swine influenza", "cls": "genetic"}, {"id": 9812, "dom": "D", "ent": "swine influenza", "cls": "bact"}, {"id": 9813, "dom": "D", "ent": "swine influenza", "cls": "viral"}, {"id": 9814, "dom": "D", "ent": "swine influenza", "cls": "fungal"}, {"id": 9815, "dom": "D", "ent": "swine influenza", "cls": "parasit"}, {"id": 9816, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "cardio"}, {"id": 9817, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "resp"}, {"id": 9818, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "gi"}, {"id": 9819, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "nerv"}, {"id": 9820, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "skin"}, {"id": 9821, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "musc"}, {"id": 9822, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "urin"}, {"id": 9823, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "endo"}, {"id": 9824, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "immune"}, {"id": 9825, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "blood"}, {"id": 9826, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "repro"}, {"id": 9827, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "breast"}, {"id": 9828, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "chest"}, {"id": 9829, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "metab"}, {"id": 9830, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "infect"}, {"id": 9831, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "tumor"}, {"id": 9832, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "mental"}, {"id": 9833, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "genetic"}, {"id": 9834, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "bact"}, {"id": 9835, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "viral"}, {"id": 9836, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "fungal"}, {"id": 9837, "dom": "D", "ent": "Stevens-Johnson syndrome", "cls": "parasit"}, {"id": 9838, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "cardio"}, {"id": 9839, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "resp"}, {"id": 9840, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "gi"}, {"id": 9841, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "nerv"}, {"id": 9842, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "skin"}, {"id": 9843, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "musc"}, {"id": 9844, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "urin"}, {"id": 9845, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "endo"}, {"id": 9846, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "immune"}, {"id": 9847, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "blood"}, {"id": 9848, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "repro"}, {"id": 9849, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "breast"}, {"id": 9850, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "chest"}, {"id": 9851, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "metab"}, {"id": 9852, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "infect"}, {"id": 9853, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "tumor"}, {"id": 9854, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "mental"}, {"id": 9855, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "genetic"}, {"id": 9856, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "bact"}, {"id": 9857, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "viral"}, {"id": 9858, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "fungal"}, {"id": 9859, "dom": "D", "ent": "adult T-cell leukemia/lymphoma", "cls": "parasit"}, {"id": 9860, "dom": "D", "ent": "abdominal tuberculosis", "cls": "cardio"}, {"id": 9861, "dom": "D", "ent": "abdominal tuberculosis", "cls": "resp"}, {"id": 9862, "dom": "D", "ent": "abdominal tuberculosis", "cls": "gi"}, {"id": 9863, "dom": "D", "ent": "abdominal tuberculosis", "cls": "nerv"}, {"id": 9864, "dom": "D", "ent": "abdominal tuberculosis", "cls": "skin"}, {"id": 9865, "dom": "D", "ent": "abdominal tuberculosis", "cls": "musc"}, {"id": 9866, "dom": "D", "ent": "abdominal tuberculosis", "cls": "urin"}, {"id": 9867, "dom": "D", "ent": "abdominal tuberculosis", "cls": "endo"}, {"id": 9868, "dom": "D", "ent": "abdominal tuberculosis", "cls": "immune"}, {"id": 9869, "dom": "D", "ent": "abdominal tuberculosis", "cls": "blood"}, {"id": 9870, "dom": "D", "ent": "abdominal tuberculosis", "cls": "repro"}, {"id": 9871, "dom": "D", "ent": "abdominal tuberculosis", "cls": "breast"}, {"id": 9872, "dom": "D", "ent": "abdominal tuberculosis", "cls": "chest"}, {"id": 9873, "dom": "D", "ent": "abdominal tuberculosis", "cls": "metab"}, {"id": 9874, "dom": "D", "ent": "abdominal tuberculosis", "cls": "infect"}, {"id": 9875, "dom": "D", "ent": "abdominal tuberculosis", "cls": "tumor"}, {"id": 9876, "dom": "D", "ent": "abdominal tuberculosis", "cls": "mental"}, {"id": 9877, "dom": "D", "ent": "abdominal tuberculosis", "cls": "genetic"}, {"id": 9878, "dom": "D", "ent": "abdominal tuberculosis", "cls": "bact"}, {"id": 9879, "dom": "D", "ent": "abdominal tuberculosis", "cls": "viral"}, {"id": 9880, "dom": "D", "ent": "abdominal tuberculosis", "cls": "fungal"}, {"id": 9881, "dom": "D", "ent": "abdominal tuberculosis", "cls": "parasit"}, {"id": 9882, "dom": "D", "ent": "alcohol-related birth defects", "cls": "cardio"}, {"id": 9883, "dom": "D", "ent": "alcohol-related birth defects", "cls": "resp"}, {"id": 9884, "dom": "D", "ent": "alcohol-related birth defects", "cls": "gi"}, {"id": 9885, "dom": "D", "ent": "alcohol-related birth defects", "cls": "nerv"}, {"id": 9886, "dom": "D", "ent": "alcohol-related birth defects", "cls": "skin"}, {"id": 9887, "dom": "D", "ent": "alcohol-related birth defects", "cls": "musc"}, {"id": 9888, "dom": "D", "ent": "alcohol-related birth defects", "cls": "urin"}, {"id": 9889, "dom": "D", "ent": "alcohol-related birth defects", "cls": "endo"}, {"id": 9890, "dom": "D", "ent": "alcohol-related birth defects", "cls": "immune"}, {"id": 9891, "dom": "D", "ent": "alcohol-related birth defects", "cls": "blood"}, {"id": 9892, "dom": "D", "ent": "alcohol-related birth defects", "cls": "repro"}, {"id": 9893, "dom": "D", "ent": "alcohol-related birth defects", "cls": "breast"}, {"id": 9894, "dom": "D", "ent": "alcohol-related birth defects", "cls": "chest"}, {"id": 9895, "dom": "D", "ent": "alcohol-related birth defects", "cls": "metab"}, {"id": 9896, "dom": "D", "ent": "alcohol-related birth defects", "cls": "infect"}, {"id": 9897, "dom": "D", "ent": "alcohol-related birth defects", "cls": "tumor"}, {"id": 9898, "dom": "D", "ent": "alcohol-related birth defects", "cls": "mental"}, {"id": 9899, "dom": "D", "ent": "alcohol-related birth defects", "cls": "genetic"}, {"id": 9900, "dom": "D", "ent": "alcohol-related birth defects", "cls": "bact"}, {"id": 9901, "dom": "D", "ent": "alcohol-related birth defects", "cls": "viral"}, {"id": 9902, "dom": "D", "ent": "alcohol-related birth defects", "cls": "fungal"}, {"id": 9903, "dom": "D", "ent": "alcohol-related birth defects", "cls": "parasit"}, {"id": 9904, "dom": "D", "ent": "organ system cancer", "cls": "cardio"}, {"id": 9905, "dom": "D", "ent": "organ system cancer", "cls": "resp"}, {"id": 9906, "dom": "D", "ent": "organ system cancer", "cls": "gi"}, {"id": 9907, "dom": "D", "ent": "organ system cancer", "cls": "nerv"}, {"id": 9908, "dom": "D", "ent": "organ system cancer", "cls": "skin"}, {"id": 9909, "dom": "D", "ent": "organ system cancer", "cls": "musc"}, {"id": 9910, "dom": "D", "ent": "organ system cancer", "cls": "urin"}, {"id": 9911, "dom": "D", "ent": "organ system cancer", "cls": "endo"}, {"id": 9912, "dom": "D", "ent": "organ system cancer", "cls": "immune"}, {"id": 9913, "dom": "D", "ent": "organ system cancer", "cls": "blood"}, {"id": 9914, "dom": "D", "ent": "organ system cancer", "cls": "repro"}, {"id": 9915, "dom": "D", "ent": "organ system cancer", "cls": "breast"}, {"id": 9916, "dom": "D", "ent": "organ system cancer", "cls": "chest"}, {"id": 9917, "dom": "D", "ent": "organ system cancer", "cls": "metab"}, {"id": 9918, "dom": "D", "ent": "organ system cancer", "cls": "infect"}, {"id": 9919, "dom": "D", "ent": "organ system cancer", "cls": "tumor"}, {"id": 9920, "dom": "D", "ent": "organ system cancer", "cls": "mental"}, {"id": 9921, "dom": "D", "ent": "organ system cancer", "cls": "genetic"}, {"id": 9922, "dom": "D", "ent": "organ system cancer", "cls": "bact"}, {"id": 9923, "dom": "D", "ent": "organ system cancer", "cls": "viral"}, {"id": 9924, "dom": "D", "ent": "organ system cancer", "cls": "fungal"}, {"id": 9925, "dom": "D", "ent": "organ system cancer", "cls": "parasit"}, {"id": 9926, "dom": "D", "ent": "Dent disease", "cls": "cardio"}, {"id": 9927, "dom": "D", "ent": "Dent disease", "cls": "resp"}, {"id": 9928, "dom": "D", "ent": "Dent disease", "cls": "gi"}, {"id": 9929, "dom": "D", "ent": "Dent disease", "cls": "nerv"}, {"id": 9930, "dom": "D", "ent": "Dent disease", "cls": "skin"}, {"id": 9931, "dom": "D", "ent": "Dent disease", "cls": "musc"}, {"id": 9932, "dom": "D", "ent": "Dent disease", "cls": "urin"}, {"id": 9933, "dom": "D", "ent": "Dent disease", "cls": "endo"}, {"id": 9934, "dom": "D", "ent": "Dent disease", "cls": "immune"}, {"id": 9935, "dom": "D", "ent": "Dent disease", "cls": "blood"}, {"id": 9936, "dom": "D", "ent": "Dent disease", "cls": "repro"}, {"id": 9937, "dom": "D", "ent": "Dent disease", "cls": "breast"}, {"id": 9938, "dom": "D", "ent": "Dent disease", "cls": "chest"}, {"id": 9939, "dom": "D", "ent": "Dent disease", "cls": "metab"}, {"id": 9940, "dom": "D", "ent": "Dent disease", "cls": "infect"}, {"id": 9941, "dom": "D", "ent": "Dent disease", "cls": "tumor"}, {"id": 9942, "dom": "D", "ent": "Dent disease", "cls": "mental"}, {"id": 9943, "dom": "D", "ent": "Dent disease", "cls": "genetic"}, {"id": 9944, "dom": "D", "ent": "Dent disease", "cls": "bact"}, {"id": 9945, "dom": "D", "ent": "Dent disease", "cls": "viral"}, {"id": 9946, "dom": "D", "ent": "Dent disease", "cls": "fungal"}, {"id": 9947, "dom": "D", "ent": "Dent disease", "cls": "parasit"}, {"id": 9948, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "cardio"}, {"id": 9949, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "resp"}, {"id": 9950, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "gi"}, {"id": 9951, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "nerv"}, {"id": 9952, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "skin"}, {"id": 9953, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "musc"}, {"id": 9954, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "urin"}, {"id": 9955, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "endo"}, {"id": 9956, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "immune"}, {"id": 9957, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "blood"}, {"id": 9958, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "repro"}, {"id": 9959, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "breast"}, {"id": 9960, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "chest"}, {"id": 9961, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "metab"}, {"id": 9962, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "infect"}, {"id": 9963, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "tumor"}, {"id": 9964, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "mental"}, {"id": 9965, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "genetic"}, {"id": 9966, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "bact"}, {"id": 9967, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "viral"}, {"id": 9968, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "fungal"}, {"id": 9969, "dom": "D", "ent": "vitamin metabolic disorder", "cls": "parasit"}, {"id": 9970, "dom": "D", "ent": "nicotine dependence", "cls": "cardio"}, {"id": 9971, "dom": "D", "ent": "nicotine dependence", "cls": "resp"}, {"id": 9972, "dom": "D", "ent": "nicotine dependence", "cls": "gi"}, {"id": 9973, "dom": "D", "ent": "nicotine dependence", "cls": "nerv"}, {"id": 9974, "dom": "D", "ent": "nicotine dependence", "cls": "skin"}, {"id": 9975, "dom": "D", "ent": "nicotine dependence", "cls": "musc"}, {"id": 9976, "dom": "D", "ent": "nicotine dependence", "cls": "urin"}, {"id": 9977, "dom": "D", "ent": "nicotine dependence", "cls": "endo"}, {"id": 9978, "dom": "D", "ent": "nicotine dependence", "cls": "immune"}, {"id": 9979, "dom": "D", "ent": "nicotine dependence", "cls": "blood"}, {"id": 9980, "dom": "D", "ent": "nicotine dependence", "cls": "repro"}, {"id": 9981, "dom": "D", "ent": "nicotine dependence", "cls": "breast"}, {"id": 9982, "dom": "D", "ent": "nicotine dependence", "cls": "chest"}, {"id": 9983, "dom": "D", "ent": "nicotine dependence", "cls": "metab"}, {"id": 9984, "dom": "D", "ent": "nicotine dependence", "cls": "infect"}, {"id": 9985, "dom": "D", "ent": "nicotine dependence", "cls": "tumor"}, {"id": 9986, "dom": "D", "ent": "nicotine dependence", "cls": "mental"}, {"id": 9987, "dom": "D", "ent": "nicotine dependence", "cls": "genetic"}, {"id": 9988, "dom": "D", "ent": "nicotine dependence", "cls": "bact"}, {"id": 9989, "dom": "D", "ent": "nicotine dependence", "cls": "viral"}, {"id": 9990, "dom": "D", "ent": "nicotine dependence", "cls": "fungal"}, {"id": 9991, "dom": "D", "ent": "nicotine dependence", "cls": "parasit"}, {"id": 9992, "dom": "D", "ent": "Ogden syndrome", "cls": "cardio"}, {"id": 9993, "dom": "D", "ent": "Ogden syndrome", "cls": "resp"}, {"id": 9994, "dom": "D", "ent": "Ogden syndrome", "cls": "gi"}, {"id": 9995, "dom": "D", "ent": "Ogden syndrome", "cls": "nerv"}, {"id": 9996, "dom": "D", "ent": "Ogden syndrome", "cls": "skin"}, {"id": 9997, "dom": "D", "ent": "Ogden syndrome", "cls": "musc"}, {"id": 9998, "dom": "D", "ent": "Ogden syndrome", "cls": "urin"}, {"id": 9999, "dom": "D", "ent": "Ogden syndrome", "cls": "endo"}, {"id": 10000, "dom": "D", "ent": "Ogden syndrome", "cls": "immune"}, {"id": 10001, "dom": "D", "ent": "Ogden syndrome", "cls": "blood"}, {"id": 10002, "dom": "D", "ent": "Ogden syndrome", "cls": "repro"}, {"id": 10003, "dom": "D", "ent": "Ogden syndrome", "cls": "breast"}, {"id": 10004, "dom": "D", "ent": "Ogden syndrome", "cls": "chest"}, {"id": 10005, "dom": "D", "ent": "Ogden syndrome", "cls": "metab"}, {"id": 10006, "dom": "D", "ent": "Ogden syndrome", "cls": "infect"}, {"id": 10007, "dom": "D", "ent": "Ogden syndrome", "cls": "tumor"}, {"id": 10008, "dom": "D", "ent": "Ogden syndrome", "cls": "mental"}, {"id": 10009, "dom": "D", "ent": "Ogden syndrome", "cls": "genetic"}, {"id": 10010, "dom": "D", "ent": "Ogden syndrome", "cls": "bact"}, {"id": 10011, "dom": "D", "ent": "Ogden syndrome", "cls": "viral"}, {"id": 10012, "dom": "D", "ent": "Ogden syndrome", "cls": "fungal"}, {"id": 10013, "dom": "D", "ent": "Ogden syndrome", "cls": "parasit"}, {"id": 10014, "dom": "D", "ent": "sensory system cancer", "cls": "cardio"}, {"id": 10015, "dom": "D", "ent": "sensory system cancer", "cls": "resp"}, {"id": 10016, "dom": "D", "ent": "sensory system cancer", "cls": "gi"}, {"id": 10017, "dom": "D", "ent": "sensory system cancer", "cls": "nerv"}, {"id": 10018, "dom": "D", "ent": "sensory system cancer", "cls": "skin"}, {"id": 10019, "dom": "D", "ent": "sensory system cancer", "cls": "musc"}, {"id": 10020, "dom": "D", "ent": "sensory system cancer", "cls": "urin"}, {"id": 10021, "dom": "D", "ent": "sensory system cancer", "cls": "endo"}, {"id": 10022, "dom": "D", "ent": "sensory system cancer", "cls": "immune"}, {"id": 10023, "dom": "D", "ent": "sensory system cancer", "cls": "blood"}, {"id": 10024, "dom": "D", "ent": "sensory system cancer", "cls": "repro"}, {"id": 10025, "dom": "D", "ent": "sensory system cancer", "cls": "breast"}, {"id": 10026, "dom": "D", "ent": "sensory system cancer", "cls": "chest"}, {"id": 10027, "dom": "D", "ent": "sensory system cancer", "cls": "metab"}, {"id": 10028, "dom": "D", "ent": "sensory system cancer", "cls": "infect"}, {"id": 10029, "dom": "D", "ent": "sensory system cancer", "cls": "tumor"}, {"id": 10030, "dom": "D", "ent": "sensory system cancer", "cls": "mental"}, {"id": 10031, "dom": "D", "ent": "sensory system cancer", "cls": "genetic"}, {"id": 10032, "dom": "D", "ent": "sensory system cancer", "cls": "bact"}, {"id": 10033, "dom": "D", "ent": "sensory system cancer", "cls": "viral"}, {"id": 10034, "dom": "D", "ent": "sensory system cancer", "cls": "fungal"}, {"id": 10035, "dom": "D", "ent": "sensory system cancer", "cls": "parasit"}, {"id": 10036, "dom": "D", "ent": "heavy chain disease", "cls": "cardio"}, {"id": 10037, "dom": "D", "ent": "heavy chain disease", "cls": "resp"}, {"id": 10038, "dom": "D", "ent": "heavy chain disease", "cls": "gi"}, {"id": 10039, "dom": "D", "ent": "heavy chain disease", "cls": "nerv"}, {"id": 10040, "dom": "D", "ent": "heavy chain disease", "cls": "skin"}, {"id": 10041, "dom": "D", "ent": "heavy chain disease", "cls": "musc"}, {"id": 10042, "dom": "D", "ent": "heavy chain disease", "cls": "urin"}, {"id": 10043, "dom": "D", "ent": "heavy chain disease", "cls": "endo"}, {"id": 10044, "dom": "D", "ent": "heavy chain disease", "cls": "immune"}, {"id": 10045, "dom": "D", "ent": "heavy chain disease", "cls": "blood"}, {"id": 10046, "dom": "D", "ent": "heavy chain disease", "cls": "repro"}, {"id": 10047, "dom": "D", "ent": "heavy chain disease", "cls": "breast"}, {"id": 10048, "dom": "D", "ent": "heavy chain disease", "cls": "chest"}, {"id": 10049, "dom": "D", "ent": "heavy chain disease", "cls": "metab"}, {"id": 10050, "dom": "D", "ent": "heavy chain disease", "cls": "infect"}, {"id": 10051, "dom": "D", "ent": "heavy chain disease", "cls": "tumor"}, {"id": 10052, "dom": "D", "ent": "heavy chain disease", "cls": "mental"}, {"id": 10053, "dom": "D", "ent": "heavy chain disease", "cls": "genetic"}, {"id": 10054, "dom": "D", "ent": "heavy chain disease", "cls": "bact"}, {"id": 10055, "dom": "D", "ent": "heavy chain disease", "cls": "viral"}, {"id": 10056, "dom": "D", "ent": "heavy chain disease", "cls": "fungal"}, {"id": 10057, "dom": "D", "ent": "heavy chain disease", "cls": "parasit"}, {"id": 10058, "dom": "D", "ent": "delta chain disease", "cls": "cardio"}, {"id": 10059, "dom": "D", "ent": "delta chain disease", "cls": "resp"}, {"id": 10060, "dom": "D", "ent": "delta chain disease", "cls": "gi"}, {"id": 10061, "dom": "D", "ent": "delta chain disease", "cls": "nerv"}, {"id": 10062, "dom": "D", "ent": "delta chain disease", "cls": "skin"}, {"id": 10063, "dom": "D", "ent": "delta chain disease", "cls": "musc"}, {"id": 10064, "dom": "D", "ent": "delta chain disease", "cls": "urin"}, {"id": 10065, "dom": "D", "ent": "delta chain disease", "cls": "endo"}, {"id": 10066, "dom": "D", "ent": "delta chain disease", "cls": "immune"}, {"id": 10067, "dom": "D", "ent": "delta chain disease", "cls": "blood"}, {"id": 10068, "dom": "D", "ent": "delta chain disease", "cls": "repro"}, {"id": 10069, "dom": "D", "ent": "delta chain disease", "cls": "breast"}, {"id": 10070, "dom": "D", "ent": "delta chain disease", "cls": "chest"}, {"id": 10071, "dom": "D", "ent": "delta chain disease", "cls": "metab"}, {"id": 10072, "dom": "D", "ent": "delta chain disease", "cls": "infect"}, {"id": 10073, "dom": "D", "ent": "delta chain disease", "cls": "tumor"}, {"id": 10074, "dom": "D", "ent": "delta chain disease", "cls": "mental"}, {"id": 10075, "dom": "D", "ent": "delta chain disease", "cls": "genetic"}, {"id": 10076, "dom": "D", "ent": "delta chain disease", "cls": "bact"}, {"id": 10077, "dom": "D", "ent": "delta chain disease", "cls": "viral"}, {"id": 10078, "dom": "D", "ent": "delta chain disease", "cls": "fungal"}, {"id": 10079, "dom": "D", "ent": "delta chain disease", "cls": "parasit"}, {"id": 10080, "dom": "D", "ent": "acquired metabolic disease", "cls": "cardio"}, {"id": 10081, "dom": "D", "ent": "acquired metabolic disease", "cls": "resp"}, {"id": 10082, "dom": "D", "ent": "acquired metabolic disease", "cls": "gi"}, {"id": 10083, "dom": "D", "ent": "acquired metabolic disease", "cls": "nerv"}, {"id": 10084, "dom": "D", "ent": "acquired metabolic disease", "cls": "skin"}, {"id": 10085, "dom": "D", "ent": "acquired metabolic disease", "cls": "musc"}, {"id": 10086, "dom": "D", "ent": "acquired metabolic disease", "cls": "urin"}, {"id": 10087, "dom": "D", "ent": "acquired metabolic disease", "cls": "endo"}, {"id": 10088, "dom": "D", "ent": "acquired metabolic disease", "cls": "immune"}, {"id": 10089, "dom": "D", "ent": "acquired metabolic disease", "cls": "blood"}, {"id": 10090, "dom": "D", "ent": "acquired metabolic disease", "cls": "repro"}, {"id": 10091, "dom": "D", "ent": "acquired metabolic disease", "cls": "breast"}, {"id": 10092, "dom": "D", "ent": "acquired metabolic disease", "cls": "chest"}, {"id": 10093, "dom": "D", "ent": "acquired metabolic disease", "cls": "metab"}, {"id": 10094, "dom": "D", "ent": "acquired metabolic disease", "cls": "infect"}, {"id": 10095, "dom": "D", "ent": "acquired metabolic disease", "cls": "tumor"}, {"id": 10096, "dom": "D", "ent": "acquired metabolic disease", "cls": "mental"}, {"id": 10097, "dom": "D", "ent": "acquired metabolic disease", "cls": "genetic"}, {"id": 10098, "dom": "D", "ent": "acquired metabolic disease", "cls": "bact"}, {"id": 10099, "dom": "D", "ent": "acquired metabolic disease", "cls": "viral"}, {"id": 10100, "dom": "D", "ent": "acquired metabolic disease", "cls": "fungal"}, {"id": 10101, "dom": "D", "ent": "acquired metabolic disease", "cls": "parasit"}, {"id": 10102, "dom": "D", "ent": "Kennedy's disease", "cls": "cardio"}, {"id": 10103, "dom": "D", "ent": "Kennedy's disease", "cls": "resp"}, {"id": 10104, "dom": "D", "ent": "Kennedy's disease", "cls": "gi"}, {"id": 10105, "dom": "D", "ent": "Kennedy's disease", "cls": "nerv"}, {"id": 10106, "dom": "D", "ent": "Kennedy's disease", "cls": "skin"}, {"id": 10107, "dom": "D", "ent": "Kennedy's disease", "cls": "musc"}, {"id": 10108, "dom": "D", "ent": "Kennedy's disease", "cls": "urin"}, {"id": 10109, "dom": "D", "ent": "Kennedy's disease", "cls": "endo"}, {"id": 10110, "dom": "D", "ent": "Kennedy's disease", "cls": "immune"}, {"id": 10111, "dom": "D", "ent": "Kennedy's disease", "cls": "blood"}, {"id": 10112, "dom": "D", "ent": "Kennedy's disease", "cls": "repro"}, {"id": 10113, "dom": "D", "ent": "Kennedy's disease", "cls": "breast"}, {"id": 10114, "dom": "D", "ent": "Kennedy's disease", "cls": "chest"}, {"id": 10115, "dom": "D", "ent": "Kennedy's disease", "cls": "metab"}, {"id": 10116, "dom": "D", "ent": "Kennedy's disease", "cls": "infect"}, {"id": 10117, "dom": "D", "ent": "Kennedy's disease", "cls": "tumor"}, {"id": 10118, "dom": "D", "ent": "Kennedy's disease", "cls": "mental"}, {"id": 10119, "dom": "D", "ent": "Kennedy's disease", "cls": "genetic"}, {"id": 10120, "dom": "D", "ent": "Kennedy's disease", "cls": "bact"}, {"id": 10121, "dom": "D", "ent": "Kennedy's disease", "cls": "viral"}, {"id": 10122, "dom": "D", "ent": "Kennedy's disease", "cls": "fungal"}, {"id": 10123, "dom": "D", "ent": "Kennedy's disease", "cls": "parasit"}, {"id": 10124, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "cardio"}, {"id": 10125, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "resp"}, {"id": 10126, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "gi"}, {"id": 10127, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "nerv"}, {"id": 10128, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "skin"}, {"id": 10129, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "musc"}, {"id": 10130, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "urin"}, {"id": 10131, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "endo"}, {"id": 10132, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "immune"}, {"id": 10133, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "blood"}, {"id": 10134, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "repro"}, {"id": 10135, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "breast"}, {"id": 10136, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "chest"}, {"id": 10137, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "metab"}, {"id": 10138, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "infect"}, {"id": 10139, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "tumor"}, {"id": 10140, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "mental"}, {"id": 10141, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "genetic"}, {"id": 10142, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "bact"}, {"id": 10143, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "viral"}, {"id": 10144, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "fungal"}, {"id": 10145, "dom": "D", "ent": "juvenile absence epilepsy", "cls": "parasit"}, {"id": 10146, "dom": "D", "ent": "stuttering", "cls": "cardio"}, {"id": 10147, "dom": "D", "ent": "stuttering", "cls": "resp"}, {"id": 10148, "dom": "D", "ent": "stuttering", "cls": "gi"}, {"id": 10149, "dom": "D", "ent": "stuttering", "cls": "nerv"}, {"id": 10150, "dom": "D", "ent": "stuttering", "cls": "skin"}, {"id": 10151, "dom": "D", "ent": "stuttering", "cls": "musc"}, {"id": 10152, "dom": "D", "ent": "stuttering", "cls": "urin"}, {"id": 10153, "dom": "D", "ent": "stuttering", "cls": "endo"}, {"id": 10154, "dom": "D", "ent": "stuttering", "cls": "immune"}, {"id": 10155, "dom": "D", "ent": "stuttering", "cls": "blood"}, {"id": 10156, "dom": "D", "ent": "stuttering", "cls": "repro"}, {"id": 10157, "dom": "D", "ent": "stuttering", "cls": "breast"}, {"id": 10158, "dom": "D", "ent": "stuttering", "cls": "chest"}, {"id": 10159, "dom": "D", "ent": "stuttering", "cls": "metab"}, {"id": 10160, "dom": "D", "ent": "stuttering", "cls": "infect"}, {"id": 10161, "dom": "D", "ent": "stuttering", "cls": "tumor"}, {"id": 10162, "dom": "D", "ent": "stuttering", "cls": "mental"}, {"id": 10163, "dom": "D", "ent": "stuttering", "cls": "genetic"}, {"id": 10164, "dom": "D", "ent": "stuttering", "cls": "bact"}, {"id": 10165, "dom": "D", "ent": "stuttering", "cls": "viral"}, {"id": 10166, "dom": "D", "ent": "stuttering", "cls": "fungal"}, {"id": 10167, "dom": "D", "ent": "stuttering", "cls": "parasit"}, {"id": 10168, "dom": "D", "ent": "peeling skin syndrome", "cls": "cardio"}, {"id": 10169, "dom": "D", "ent": "peeling skin syndrome", "cls": "resp"}, {"id": 10170, "dom": "D", "ent": "peeling skin syndrome", "cls": "gi"}, {"id": 10171, "dom": "D", "ent": "peeling skin syndrome", "cls": "nerv"}, {"id": 10172, "dom": "D", "ent": "peeling skin syndrome", "cls": "skin"}, {"id": 10173, "dom": "D", "ent": "peeling skin syndrome", "cls": "musc"}, {"id": 10174, "dom": "D", "ent": "peeling skin syndrome", "cls": "urin"}, {"id": 10175, "dom": "D", "ent": "peeling skin syndrome", "cls": "endo"}, {"id": 10176, "dom": "D", "ent": "peeling skin syndrome", "cls": "immune"}, {"id": 10177, "dom": "D", "ent": "peeling skin syndrome", "cls": "blood"}, {"id": 10178, "dom": "D", "ent": "peeling skin syndrome", "cls": "repro"}, {"id": 10179, "dom": "D", "ent": "peeling skin syndrome", "cls": "breast"}, {"id": 10180, "dom": "D", "ent": "peeling skin syndrome", "cls": "chest"}, {"id": 10181, "dom": "D", "ent": "peeling skin syndrome", "cls": "metab"}, {"id": 10182, "dom": "D", "ent": "peeling skin syndrome", "cls": "infect"}, {"id": 10183, "dom": "D", "ent": "peeling skin syndrome", "cls": "tumor"}, {"id": 10184, "dom": "D", "ent": "peeling skin syndrome", "cls": "mental"}, {"id": 10185, "dom": "D", "ent": "peeling skin syndrome", "cls": "genetic"}, {"id": 10186, "dom": "D", "ent": "peeling skin syndrome", "cls": "bact"}, {"id": 10187, "dom": "D", "ent": "peeling skin syndrome", "cls": "viral"}, {"id": 10188, "dom": "D", "ent": "peeling skin syndrome", "cls": "fungal"}, {"id": 10189, "dom": "D", "ent": "peeling skin syndrome", "cls": "parasit"}, {"id": 10190, "dom": "D", "ent": "gastrointestinal allergy", "cls": "cardio"}, {"id": 10191, "dom": "D", "ent": "gastrointestinal allergy", "cls": "resp"}, {"id": 10192, "dom": "D", "ent": "gastrointestinal allergy", "cls": "gi"}, {"id": 10193, "dom": "D", "ent": "gastrointestinal allergy", "cls": "nerv"}, {"id": 10194, "dom": "D", "ent": "gastrointestinal allergy", "cls": "skin"}, {"id": 10195, "dom": "D", "ent": "gastrointestinal allergy", "cls": "musc"}, {"id": 10196, "dom": "D", "ent": "gastrointestinal allergy", "cls": "urin"}, {"id": 10197, "dom": "D", "ent": "gastrointestinal allergy", "cls": "endo"}, {"id": 10198, "dom": "D", "ent": "gastrointestinal allergy", "cls": "immune"}, {"id": 10199, "dom": "D", "ent": "gastrointestinal allergy", "cls": "blood"}, {"id": 10200, "dom": "D", "ent": "gastrointestinal allergy", "cls": "repro"}, {"id": 10201, "dom": "D", "ent": "gastrointestinal allergy", "cls": "breast"}, {"id": 10202, "dom": "D", "ent": "gastrointestinal allergy", "cls": "chest"}, {"id": 10203, "dom": "D", "ent": "gastrointestinal allergy", "cls": "metab"}, {"id": 10204, "dom": "D", "ent": "gastrointestinal allergy", "cls": "infect"}, {"id": 10205, "dom": "D", "ent": "gastrointestinal allergy", "cls": "tumor"}, {"id": 10206, "dom": "D", "ent": "gastrointestinal allergy", "cls": "mental"}, {"id": 10207, "dom": "D", "ent": "gastrointestinal allergy", "cls": "genetic"}, {"id": 10208, "dom": "D", "ent": "gastrointestinal allergy", "cls": "bact"}, {"id": 10209, "dom": "D", "ent": "gastrointestinal allergy", "cls": "viral"}, {"id": 10210, "dom": "D", "ent": "gastrointestinal allergy", "cls": "fungal"}, {"id": 10211, "dom": "D", "ent": "gastrointestinal allergy", "cls": "parasit"}, {"id": 10212, "dom": "D", "ent": "apple allergy", "cls": "cardio"}, {"id": 10213, "dom": "D", "ent": "apple allergy", "cls": "resp"}, {"id": 10214, "dom": "D", "ent": "apple allergy", "cls": "gi"}, {"id": 10215, "dom": "D", "ent": "apple allergy", "cls": "nerv"}, {"id": 10216, "dom": "D", "ent": "apple allergy", "cls": "skin"}, {"id": 10217, "dom": "D", "ent": "apple allergy", "cls": "musc"}, {"id": 10218, "dom": "D", "ent": "apple allergy", "cls": "urin"}, {"id": 10219, "dom": "D", "ent": "apple allergy", "cls": "endo"}, {"id": 10220, "dom": "D", "ent": "apple allergy", "cls": "immune"}, {"id": 10221, "dom": "D", "ent": "apple allergy", "cls": "blood"}, {"id": 10222, "dom": "D", "ent": "apple allergy", "cls": "repro"}, {"id": 10223, "dom": "D", "ent": "apple allergy", "cls": "breast"}, {"id": 10224, "dom": "D", "ent": "apple allergy", "cls": "chest"}, {"id": 10225, "dom": "D", "ent": "apple allergy", "cls": "metab"}, {"id": 10226, "dom": "D", "ent": "apple allergy", "cls": "infect"}, {"id": 10227, "dom": "D", "ent": "apple allergy", "cls": "tumor"}, {"id": 10228, "dom": "D", "ent": "apple allergy", "cls": "mental"}, {"id": 10229, "dom": "D", "ent": "apple allergy", "cls": "genetic"}, {"id": 10230, "dom": "D", "ent": "apple allergy", "cls": "bact"}, {"id": 10231, "dom": "D", "ent": "apple allergy", "cls": "viral"}, {"id": 10232, "dom": "D", "ent": "apple allergy", "cls": "fungal"}, {"id": 10233, "dom": "D", "ent": "apple allergy", "cls": "parasit"}, {"id": 10234, "dom": "D", "ent": "plum allergy", "cls": "cardio"}, {"id": 10235, "dom": "D", "ent": "plum allergy", "cls": "resp"}, {"id": 10236, "dom": "D", "ent": "plum allergy", "cls": "gi"}, {"id": 10237, "dom": "D", "ent": "plum allergy", "cls": "nerv"}, {"id": 10238, "dom": "D", "ent": "plum allergy", "cls": "skin"}, {"id": 10239, "dom": "D", "ent": "plum allergy", "cls": "musc"}, {"id": 10240, "dom": "D", "ent": "plum allergy", "cls": "urin"}, {"id": 10241, "dom": "D", "ent": "plum allergy", "cls": "endo"}, {"id": 10242, "dom": "D", "ent": "plum allergy", "cls": "immune"}, {"id": 10243, "dom": "D", "ent": "plum allergy", "cls": "blood"}, {"id": 10244, "dom": "D", "ent": "plum allergy", "cls": "repro"}, {"id": 10245, "dom": "D", "ent": "plum allergy", "cls": "breast"}, {"id": 10246, "dom": "D", "ent": "plum allergy", "cls": "chest"}, {"id": 10247, "dom": "D", "ent": "plum allergy", "cls": "metab"}, {"id": 10248, "dom": "D", "ent": "plum allergy", "cls": "infect"}, {"id": 10249, "dom": "D", "ent": "plum allergy", "cls": "tumor"}, {"id": 10250, "dom": "D", "ent": "plum allergy", "cls": "mental"}, {"id": 10251, "dom": "D", "ent": "plum allergy", "cls": "genetic"}, {"id": 10252, "dom": "D", "ent": "plum allergy", "cls": "bact"}, {"id": 10253, "dom": "D", "ent": "plum allergy", "cls": "viral"}, {"id": 10254, "dom": "D", "ent": "plum allergy", "cls": "fungal"}, {"id": 10255, "dom": "D", "ent": "plum allergy", "cls": "parasit"}, {"id": 10256, "dom": "D", "ent": "carp allergy", "cls": "cardio"}, {"id": 10257, "dom": "D", "ent": "carp allergy", "cls": "resp"}, {"id": 10258, "dom": "D", "ent": "carp allergy", "cls": "gi"}, {"id": 10259, "dom": "D", "ent": "carp allergy", "cls": "nerv"}, {"id": 10260, "dom": "D", "ent": "carp allergy", "cls": "skin"}, {"id": 10261, "dom": "D", "ent": "carp allergy", "cls": "musc"}, {"id": 10262, "dom": "D", "ent": "carp allergy", "cls": "urin"}, {"id": 10263, "dom": "D", "ent": "carp allergy", "cls": "endo"}, {"id": 10264, "dom": "D", "ent": "carp allergy", "cls": "immune"}, {"id": 10265, "dom": "D", "ent": "carp allergy", "cls": "blood"}, {"id": 10266, "dom": "D", "ent": "carp allergy", "cls": "repro"}, {"id": 10267, "dom": "D", "ent": "carp allergy", "cls": "breast"}, {"id": 10268, "dom": "D", "ent": "carp allergy", "cls": "chest"}, {"id": 10269, "dom": "D", "ent": "carp allergy", "cls": "metab"}, {"id": 10270, "dom": "D", "ent": "carp allergy", "cls": "infect"}, {"id": 10271, "dom": "D", "ent": "carp allergy", "cls": "tumor"}, {"id": 10272, "dom": "D", "ent": "carp allergy", "cls": "mental"}, {"id": 10273, "dom": "D", "ent": "carp allergy", "cls": "genetic"}, {"id": 10274, "dom": "D", "ent": "carp allergy", "cls": "bact"}, {"id": 10275, "dom": "D", "ent": "carp allergy", "cls": "viral"}, {"id": 10276, "dom": "D", "ent": "carp allergy", "cls": "fungal"}, {"id": 10277, "dom": "D", "ent": "carp allergy", "cls": "parasit"}, {"id": 10278, "dom": "D", "ent": "white shrimp allergy", "cls": "cardio"}, {"id": 10279, "dom": "D", "ent": "white shrimp allergy", "cls": "resp"}, {"id": 10280, "dom": "D", "ent": "white shrimp allergy", "cls": "gi"}, {"id": 10281, "dom": "D", "ent": "white shrimp allergy", "cls": "nerv"}, {"id": 10282, "dom": "D", "ent": "white shrimp allergy", "cls": "skin"}, {"id": 10283, "dom": "D", "ent": "white shrimp allergy", "cls": "musc"}, {"id": 10284, "dom": "D", "ent": "white shrimp allergy", "cls": "urin"}, {"id": 10285, "dom": "D", "ent": "white shrimp allergy", "cls": "endo"}, {"id": 10286, "dom": "D", "ent": "white shrimp allergy", "cls": "immune"}, {"id": 10287, "dom": "D", "ent": "white shrimp allergy", "cls": "blood"}, {"id": 10288, "dom": "D", "ent": "white shrimp allergy", "cls": "repro"}, {"id": 10289, "dom": "D", "ent": "white shrimp allergy", "cls": "breast"}, {"id": 10290, "dom": "D", "ent": "white shrimp allergy", "cls": "chest"}, {"id": 10291, "dom": "D", "ent": "white shrimp allergy", "cls": "metab"}, {"id": 10292, "dom": "D", "ent": "white shrimp allergy", "cls": "infect"}, {"id": 10293, "dom": "D", "ent": "white shrimp allergy", "cls": "tumor"}, {"id": 10294, "dom": "D", "ent": "white shrimp allergy", "cls": "mental"}, {"id": 10295, "dom": "D", "ent": "white shrimp allergy", "cls": "genetic"}, {"id": 10296, "dom": "D", "ent": "white shrimp allergy", "cls": "bact"}, {"id": 10297, "dom": "D", "ent": "white shrimp allergy", "cls": "viral"}, {"id": 10298, "dom": "D", "ent": "white shrimp allergy", "cls": "fungal"}, {"id": 10299, "dom": "D", "ent": "white shrimp allergy", "cls": "parasit"}, {"id": 10300, "dom": "D", "ent": "snail allergy", "cls": "cardio"}, {"id": 10301, "dom": "D", "ent": "snail allergy", "cls": "resp"}, {"id": 10302, "dom": "D", "ent": "snail allergy", "cls": "gi"}, {"id": 10303, "dom": "D", "ent": "snail allergy", "cls": "nerv"}, {"id": 10304, "dom": "D", "ent": "snail allergy", "cls": "skin"}, {"id": 10305, "dom": "D", "ent": "snail allergy", "cls": "musc"}, {"id": 10306, "dom": "D", "ent": "snail allergy", "cls": "urin"}, {"id": 10307, "dom": "D", "ent": "snail allergy", "cls": "endo"}, {"id": 10308, "dom": "D", "ent": "snail allergy", "cls": "immune"}, {"id": 10309, "dom": "D", "ent": "snail allergy", "cls": "blood"}, {"id": 10310, "dom": "D", "ent": "snail allergy", "cls": "repro"}, {"id": 10311, "dom": "D", "ent": "snail allergy", "cls": "breast"}, {"id": 10312, "dom": "D", "ent": "snail allergy", "cls": "chest"}, {"id": 10313, "dom": "D", "ent": "snail allergy", "cls": "metab"}, {"id": 10314, "dom": "D", "ent": "snail allergy", "cls": "infect"}, {"id": 10315, "dom": "D", "ent": "snail allergy", "cls": "tumor"}, {"id": 10316, "dom": "D", "ent": "snail allergy", "cls": "mental"}, {"id": 10317, "dom": "D", "ent": "snail allergy", "cls": "genetic"}, {"id": 10318, "dom": "D", "ent": "snail allergy", "cls": "bact"}, {"id": 10319, "dom": "D", "ent": "snail allergy", "cls": "viral"}, {"id": 10320, "dom": "D", "ent": "snail allergy", "cls": "fungal"}, {"id": 10321, "dom": "D", "ent": "snail allergy", "cls": "parasit"}, {"id": 10322, "dom": "D", "ent": "cardiac tuberculosis", "cls": "cardio"}, {"id": 10323, "dom": "D", "ent": "cardiac tuberculosis", "cls": "resp"}, {"id": 10324, "dom": "D", "ent": "cardiac tuberculosis", "cls": "gi"}, {"id": 10325, "dom": "D", "ent": "cardiac tuberculosis", "cls": "nerv"}, {"id": 10326, "dom": "D", "ent": "cardiac tuberculosis", "cls": "skin"}, {"id": 10327, "dom": "D", "ent": "cardiac tuberculosis", "cls": "musc"}, {"id": 10328, "dom": "D", "ent": "cardiac tuberculosis", "cls": "urin"}, {"id": 10329, "dom": "D", "ent": "cardiac tuberculosis", "cls": "endo"}, {"id": 10330, "dom": "D", "ent": "cardiac tuberculosis", "cls": "immune"}, {"id": 10331, "dom": "D", "ent": "cardiac tuberculosis", "cls": "blood"}, {"id": 10332, "dom": "D", "ent": "cardiac tuberculosis", "cls": "repro"}, {"id": 10333, "dom": "D", "ent": "cardiac tuberculosis", "cls": "breast"}, {"id": 10334, "dom": "D", "ent": "cardiac tuberculosis", "cls": "chest"}, {"id": 10335, "dom": "D", "ent": "cardiac tuberculosis", "cls": "metab"}, {"id": 10336, "dom": "D", "ent": "cardiac tuberculosis", "cls": "infect"}, {"id": 10337, "dom": "D", "ent": "cardiac tuberculosis", "cls": "tumor"}, {"id": 10338, "dom": "D", "ent": "cardiac tuberculosis", "cls": "mental"}, {"id": 10339, "dom": "D", "ent": "cardiac tuberculosis", "cls": "genetic"}, {"id": 10340, "dom": "D", "ent": "cardiac tuberculosis", "cls": "bact"}, {"id": 10341, "dom": "D", "ent": "cardiac tuberculosis", "cls": "viral"}, {"id": 10342, "dom": "D", "ent": "cardiac tuberculosis", "cls": "fungal"}, {"id": 10343, "dom": "D", "ent": "cardiac tuberculosis", "cls": "parasit"}, {"id": 10344, "dom": "D", "ent": "congenital diarrhea", "cls": "cardio"}, {"id": 10345, "dom": "D", "ent": "congenital diarrhea", "cls": "resp"}, {"id": 10346, "dom": "D", "ent": "congenital diarrhea", "cls": "gi"}, {"id": 10347, "dom": "D", "ent": "congenital diarrhea", "cls": "nerv"}, {"id": 10348, "dom": "D", "ent": "congenital diarrhea", "cls": "skin"}, {"id": 10349, "dom": "D", "ent": "congenital diarrhea", "cls": "musc"}, {"id": 10350, "dom": "D", "ent": "congenital diarrhea", "cls": "urin"}, {"id": 10351, "dom": "D", "ent": "congenital diarrhea", "cls": "endo"}, {"id": 10352, "dom": "D", "ent": "congenital diarrhea", "cls": "immune"}, {"id": 10353, "dom": "D", "ent": "congenital diarrhea", "cls": "blood"}, {"id": 10354, "dom": "D", "ent": "congenital diarrhea", "cls": "repro"}, {"id": 10355, "dom": "D", "ent": "congenital diarrhea", "cls": "breast"}, {"id": 10356, "dom": "D", "ent": "congenital diarrhea", "cls": "chest"}, {"id": 10357, "dom": "D", "ent": "congenital diarrhea", "cls": "metab"}, {"id": 10358, "dom": "D", "ent": "congenital diarrhea", "cls": "infect"}, {"id": 10359, "dom": "D", "ent": "congenital diarrhea", "cls": "tumor"}, {"id": 10360, "dom": "D", "ent": "congenital diarrhea", "cls": "mental"}, {"id": 10361, "dom": "D", "ent": "congenital diarrhea", "cls": "genetic"}, {"id": 10362, "dom": "D", "ent": "congenital diarrhea", "cls": "bact"}, {"id": 10363, "dom": "D", "ent": "congenital diarrhea", "cls": "viral"}, {"id": 10364, "dom": "D", "ent": "congenital diarrhea", "cls": "fungal"}, {"id": 10365, "dom": "D", "ent": "congenital diarrhea", "cls": "parasit"}, {"id": 10366, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "cardio"}, {"id": 10367, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "resp"}, {"id": 10368, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "gi"}, {"id": 10369, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "nerv"}, {"id": 10370, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "skin"}, {"id": 10371, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "musc"}, {"id": 10372, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "urin"}, {"id": 10373, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "endo"}, {"id": 10374, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "immune"}, {"id": 10375, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "blood"}, {"id": 10376, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "repro"}, {"id": 10377, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "breast"}, {"id": 10378, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "chest"}, {"id": 10379, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "metab"}, {"id": 10380, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "infect"}, {"id": 10381, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "tumor"}, {"id": 10382, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "mental"}, {"id": 10383, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "genetic"}, {"id": 10384, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "bact"}, {"id": 10385, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "viral"}, {"id": 10386, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "fungal"}, {"id": 10387, "dom": "D", "ent": "early-onset Parkinson's disease", "cls": "parasit"}, {"id": 10388, "dom": "D", "ent": "vegetable allergy", "cls": "cardio"}, {"id": 10389, "dom": "D", "ent": "vegetable allergy", "cls": "resp"}, {"id": 10390, "dom": "D", "ent": "vegetable allergy", "cls": "gi"}, {"id": 10391, "dom": "D", "ent": "vegetable allergy", "cls": "nerv"}, {"id": 10392, "dom": "D", "ent": "vegetable allergy", "cls": "skin"}, {"id": 10393, "dom": "D", "ent": "vegetable allergy", "cls": "musc"}, {"id": 10394, "dom": "D", "ent": "vegetable allergy", "cls": "urin"}, {"id": 10395, "dom": "D", "ent": "vegetable allergy", "cls": "endo"}, {"id": 10396, "dom": "D", "ent": "vegetable allergy", "cls": "immune"}, {"id": 10397, "dom": "D", "ent": "vegetable allergy", "cls": "blood"}, {"id": 10398, "dom": "D", "ent": "vegetable allergy", "cls": "repro"}, {"id": 10399, "dom": "D", "ent": "vegetable allergy", "cls": "breast"}, {"id": 10400, "dom": "D", "ent": "vegetable allergy", "cls": "chest"}, {"id": 10401, "dom": "D", "ent": "vegetable allergy", "cls": "metab"}, {"id": 10402, "dom": "D", "ent": "vegetable allergy", "cls": "infect"}, {"id": 10403, "dom": "D", "ent": "vegetable allergy", "cls": "tumor"}, {"id": 10404, "dom": "D", "ent": "vegetable allergy", "cls": "mental"}, {"id": 10405, "dom": "D", "ent": "vegetable allergy", "cls": "genetic"}, {"id": 10406, "dom": "D", "ent": "vegetable allergy", "cls": "bact"}, {"id": 10407, "dom": "D", "ent": "vegetable allergy", "cls": "viral"}, {"id": 10408, "dom": "D", "ent": "vegetable allergy", "cls": "fungal"}, {"id": 10409, "dom": "D", "ent": "vegetable allergy", "cls": "parasit"}, {"id": 10410, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "cardio"}, {"id": 10411, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "resp"}, {"id": 10412, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "gi"}, {"id": 10413, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "nerv"}, {"id": 10414, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "skin"}, {"id": 10415, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "musc"}, {"id": 10416, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "urin"}, {"id": 10417, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "endo"}, {"id": 10418, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "immune"}, {"id": 10419, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "blood"}, {"id": 10420, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "repro"}, {"id": 10421, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "breast"}, {"id": 10422, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "chest"}, {"id": 10423, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "metab"}, {"id": 10424, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "infect"}, {"id": 10425, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "tumor"}, {"id": 10426, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "mental"}, {"id": 10427, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "genetic"}, {"id": 10428, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "bact"}, {"id": 10429, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "viral"}, {"id": 10430, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "fungal"}, {"id": 10431, "dom": "D", "ent": "mitochondrial Parkinson's disease", "cls": "parasit"}, {"id": 10432, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "cardio"}, {"id": 10433, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "resp"}, {"id": 10434, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "gi"}, {"id": 10435, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "nerv"}, {"id": 10436, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "skin"}, {"id": 10437, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "musc"}, {"id": 10438, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "urin"}, {"id": 10439, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "endo"}, {"id": 10440, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "immune"}, {"id": 10441, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "blood"}, {"id": 10442, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "repro"}, {"id": 10443, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "breast"}, {"id": 10444, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "chest"}, {"id": 10445, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "metab"}, {"id": 10446, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "infect"}, {"id": 10447, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "tumor"}, {"id": 10448, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "mental"}, {"id": 10449, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "genetic"}, {"id": 10450, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "bact"}, {"id": 10451, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "viral"}, {"id": 10452, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "fungal"}, {"id": 10453, "dom": "D", "ent": "vascular-type bleeding disorder", "cls": "parasit"}, {"id": 10454, "dom": "D", "ent": "bone remodeling disease", "cls": "cardio"}, {"id": 10455, "dom": "D", "ent": "bone remodeling disease", "cls": "resp"}, {"id": 10456, "dom": "D", "ent": "bone remodeling disease", "cls": "gi"}, {"id": 10457, "dom": "D", "ent": "bone remodeling disease", "cls": "nerv"}, {"id": 10458, "dom": "D", "ent": "bone remodeling disease", "cls": "skin"}, {"id": 10459, "dom": "D", "ent": "bone remodeling disease", "cls": "musc"}, {"id": 10460, "dom": "D", "ent": "bone remodeling disease", "cls": "urin"}, {"id": 10461, "dom": "D", "ent": "bone remodeling disease", "cls": "endo"}, {"id": 10462, "dom": "D", "ent": "bone remodeling disease", "cls": "immune"}, {"id": 10463, "dom": "D", "ent": "bone remodeling disease", "cls": "blood"}, {"id": 10464, "dom": "D", "ent": "bone remodeling disease", "cls": "repro"}, {"id": 10465, "dom": "D", "ent": "bone remodeling disease", "cls": "breast"}, {"id": 10466, "dom": "D", "ent": "bone remodeling disease", "cls": "chest"}, {"id": 10467, "dom": "D", "ent": "bone remodeling disease", "cls": "metab"}, {"id": 10468, "dom": "D", "ent": "bone remodeling disease", "cls": "infect"}, {"id": 10469, "dom": "D", "ent": "bone remodeling disease", "cls": "tumor"}, {"id": 10470, "dom": "D", "ent": "bone remodeling disease", "cls": "mental"}, {"id": 10471, "dom": "D", "ent": "bone remodeling disease", "cls": "genetic"}, {"id": 10472, "dom": "D", "ent": "bone remodeling disease", "cls": "bact"}, {"id": 10473, "dom": "D", "ent": "bone remodeling disease", "cls": "viral"}, {"id": 10474, "dom": "D", "ent": "bone remodeling disease", "cls": "fungal"}, {"id": 10475, "dom": "D", "ent": "bone remodeling disease", "cls": "parasit"}, {"id": 10476, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "cardio"}, {"id": 10477, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "resp"}, {"id": 10478, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "gi"}, {"id": 10479, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "nerv"}, {"id": 10480, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "skin"}, {"id": 10481, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "musc"}, {"id": 10482, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "urin"}, {"id": 10483, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "endo"}, {"id": 10484, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "immune"}, {"id": 10485, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "blood"}, {"id": 10486, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "repro"}, {"id": 10487, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "breast"}, {"id": 10488, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "chest"}, {"id": 10489, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "metab"}, {"id": 10490, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "infect"}, {"id": 10491, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "tumor"}, {"id": 10492, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "mental"}, {"id": 10493, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "genetic"}, {"id": 10494, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "bact"}, {"id": 10495, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "viral"}, {"id": 10496, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "fungal"}, {"id": 10497, "dom": "D", "ent": "Coronavirus infectious disease", "cls": "parasit"}, {"id": 10498, "dom": "D", "ent": "kidney cortex disease", "cls": "cardio"}, {"id": 10499, "dom": "D", "ent": "kidney cortex disease", "cls": "resp"}, {"id": 10500, "dom": "D", "ent": "kidney cortex disease", "cls": "gi"}, {"id": 10501, "dom": "D", "ent": "kidney cortex disease", "cls": "nerv"}, {"id": 10502, "dom": "D", "ent": "kidney cortex disease", "cls": "skin"}, {"id": 10503, "dom": "D", "ent": "kidney cortex disease", "cls": "musc"}, {"id": 10504, "dom": "D", "ent": "kidney cortex disease", "cls": "urin"}, {"id": 10505, "dom": "D", "ent": "kidney cortex disease", "cls": "endo"}, {"id": 10506, "dom": "D", "ent": "kidney cortex disease", "cls": "immune"}, {"id": 10507, "dom": "D", "ent": "kidney cortex disease", "cls": "blood"}, {"id": 10508, "dom": "D", "ent": "kidney cortex disease", "cls": "repro"}, {"id": 10509, "dom": "D", "ent": "kidney cortex disease", "cls": "breast"}, {"id": 10510, "dom": "D", "ent": "kidney cortex disease", "cls": "chest"}, {"id": 10511, "dom": "D", "ent": "kidney cortex disease", "cls": "metab"}, {"id": 10512, "dom": "D", "ent": "kidney cortex disease", "cls": "infect"}, {"id": 10513, "dom": "D", "ent": "kidney cortex disease", "cls": "tumor"}, {"id": 10514, "dom": "D", "ent": "kidney cortex disease", "cls": "mental"}, {"id": 10515, "dom": "D", "ent": "kidney cortex disease", "cls": "genetic"}, {"id": 10516, "dom": "D", "ent": "kidney cortex disease", "cls": "bact"}, {"id": 10517, "dom": "D", "ent": "kidney cortex disease", "cls": "viral"}, {"id": 10518, "dom": "D", "ent": "kidney cortex disease", "cls": "fungal"}, {"id": 10519, "dom": "D", "ent": "kidney cortex disease", "cls": "parasit"}, {"id": 10520, "dom": "D", "ent": "chronic asthma", "cls": "cardio"}, {"id": 10521, "dom": "D", "ent": "chronic asthma", "cls": "resp"}, {"id": 10522, "dom": "D", "ent": "chronic asthma", "cls": "gi"}, {"id": 10523, "dom": "D", "ent": "chronic asthma", "cls": "nerv"}, {"id": 10524, "dom": "D", "ent": "chronic asthma", "cls": "skin"}, {"id": 10525, "dom": "D", "ent": "chronic asthma", "cls": "musc"}, {"id": 10526, "dom": "D", "ent": "chronic asthma", "cls": "urin"}, {"id": 10527, "dom": "D", "ent": "chronic asthma", "cls": "endo"}, {"id": 10528, "dom": "D", "ent": "chronic asthma", "cls": "immune"}, {"id": 10529, "dom": "D", "ent": "chronic asthma", "cls": "blood"}, {"id": 10530, "dom": "D", "ent": "chronic asthma", "cls": "repro"}, {"id": 10531, "dom": "D", "ent": "chronic asthma", "cls": "breast"}, {"id": 10532, "dom": "D", "ent": "chronic asthma", "cls": "chest"}, {"id": 10533, "dom": "D", "ent": "chronic asthma", "cls": "metab"}, {"id": 10534, "dom": "D", "ent": "chronic asthma", "cls": "infect"}, {"id": 10535, "dom": "D", "ent": "chronic asthma", "cls": "tumor"}, {"id": 10536, "dom": "D", "ent": "chronic asthma", "cls": "mental"}, {"id": 10537, "dom": "D", "ent": "chronic asthma", "cls": "genetic"}, {"id": 10538, "dom": "D", "ent": "chronic asthma", "cls": "bact"}, {"id": 10539, "dom": "D", "ent": "chronic asthma", "cls": "viral"}, {"id": 10540, "dom": "D", "ent": "chronic asthma", "cls": "fungal"}, {"id": 10541, "dom": "D", "ent": "chronic asthma", "cls": "parasit"}, {"id": 10542, "dom": "D", "ent": "acute asthma", "cls": "cardio"}, {"id": 10543, "dom": "D", "ent": "acute asthma", "cls": "resp"}, {"id": 10544, "dom": "D", "ent": "acute asthma", "cls": "gi"}, {"id": 10545, "dom": "D", "ent": "acute asthma", "cls": "nerv"}, {"id": 10546, "dom": "D", "ent": "acute asthma", "cls": "skin"}, {"id": 10547, "dom": "D", "ent": "acute asthma", "cls": "musc"}, {"id": 10548, "dom": "D", "ent": "acute asthma", "cls": "urin"}, {"id": 10549, "dom": "D", "ent": "acute asthma", "cls": "endo"}, {"id": 10550, "dom": "D", "ent": "acute asthma", "cls": "immune"}, {"id": 10551, "dom": "D", "ent": "acute asthma", "cls": "blood"}, {"id": 10552, "dom": "D", "ent": "acute asthma", "cls": "repro"}, {"id": 10553, "dom": "D", "ent": "acute asthma", "cls": "breast"}, {"id": 10554, "dom": "D", "ent": "acute asthma", "cls": "chest"}, {"id": 10555, "dom": "D", "ent": "acute asthma", "cls": "metab"}, {"id": 10556, "dom": "D", "ent": "acute asthma", "cls": "infect"}, {"id": 10557, "dom": "D", "ent": "acute asthma", "cls": "tumor"}, {"id": 10558, "dom": "D", "ent": "acute asthma", "cls": "mental"}, {"id": 10559, "dom": "D", "ent": "acute asthma", "cls": "genetic"}, {"id": 10560, "dom": "D", "ent": "acute asthma", "cls": "bact"}, {"id": 10561, "dom": "D", "ent": "acute asthma", "cls": "viral"}, {"id": 10562, "dom": "D", "ent": "acute asthma", "cls": "fungal"}, {"id": 10563, "dom": "D", "ent": "acute asthma", "cls": "parasit"}, {"id": 10564, "dom": "D", "ent": "nocturnal asthma", "cls": "cardio"}, {"id": 10565, "dom": "D", "ent": "nocturnal asthma", "cls": "resp"}, {"id": 10566, "dom": "D", "ent": "nocturnal asthma", "cls": "gi"}, {"id": 10567, "dom": "D", "ent": "nocturnal asthma", "cls": "nerv"}, {"id": 10568, "dom": "D", "ent": "nocturnal asthma", "cls": "skin"}, {"id": 10569, "dom": "D", "ent": "nocturnal asthma", "cls": "musc"}, {"id": 10570, "dom": "D", "ent": "nocturnal asthma", "cls": "urin"}, {"id": 10571, "dom": "D", "ent": "nocturnal asthma", "cls": "endo"}, {"id": 10572, "dom": "D", "ent": "nocturnal asthma", "cls": "immune"}, {"id": 10573, "dom": "D", "ent": "nocturnal asthma", "cls": "blood"}, {"id": 10574, "dom": "D", "ent": "nocturnal asthma", "cls": "repro"}, {"id": 10575, "dom": "D", "ent": "nocturnal asthma", "cls": "breast"}, {"id": 10576, "dom": "D", "ent": "nocturnal asthma", "cls": "chest"}, {"id": 10577, "dom": "D", "ent": "nocturnal asthma", "cls": "metab"}, {"id": 10578, "dom": "D", "ent": "nocturnal asthma", "cls": "infect"}, {"id": 10579, "dom": "D", "ent": "nocturnal asthma", "cls": "tumor"}, {"id": 10580, "dom": "D", "ent": "nocturnal asthma", "cls": "mental"}, {"id": 10581, "dom": "D", "ent": "nocturnal asthma", "cls": "genetic"}, {"id": 10582, "dom": "D", "ent": "nocturnal asthma", "cls": "bact"}, {"id": 10583, "dom": "D", "ent": "nocturnal asthma", "cls": "viral"}, {"id": 10584, "dom": "D", "ent": "nocturnal asthma", "cls": "fungal"}, {"id": 10585, "dom": "D", "ent": "nocturnal asthma", "cls": "parasit"}, {"id": 10586, "dom": "D", "ent": "subjective cognitive decline", "cls": "cardio"}, {"id": 10587, "dom": "D", "ent": "subjective cognitive decline", "cls": "resp"}, {"id": 10588, "dom": "D", "ent": "subjective cognitive decline", "cls": "gi"}, {"id": 10589, "dom": "D", "ent": "subjective cognitive decline", "cls": "nerv"}, {"id": 10590, "dom": "D", "ent": "subjective cognitive decline", "cls": "skin"}, {"id": 10591, "dom": "D", "ent": "subjective cognitive decline", "cls": "musc"}, {"id": 10592, "dom": "D", "ent": "subjective cognitive decline", "cls": "urin"}, {"id": 10593, "dom": "D", "ent": "subjective cognitive decline", "cls": "endo"}, {"id": 10594, "dom": "D", "ent": "subjective cognitive decline", "cls": "immune"}, {"id": 10595, "dom": "D", "ent": "subjective cognitive decline", "cls": "blood"}, {"id": 10596, "dom": "D", "ent": "subjective cognitive decline", "cls": "repro"}, {"id": 10597, "dom": "D", "ent": "subjective cognitive decline", "cls": "breast"}, {"id": 10598, "dom": "D", "ent": "subjective cognitive decline", "cls": "chest"}, {"id": 10599, "dom": "D", "ent": "subjective cognitive decline", "cls": "metab"}, {"id": 10600, "dom": "D", "ent": "subjective cognitive decline", "cls": "infect"}, {"id": 10601, "dom": "D", "ent": "subjective cognitive decline", "cls": "tumor"}, {"id": 10602, "dom": "D", "ent": "subjective cognitive decline", "cls": "mental"}, {"id": 10603, "dom": "D", "ent": "subjective cognitive decline", "cls": "genetic"}, {"id": 10604, "dom": "D", "ent": "subjective cognitive decline", "cls": "bact"}, {"id": 10605, "dom": "D", "ent": "subjective cognitive decline", "cls": "viral"}, {"id": 10606, "dom": "D", "ent": "subjective cognitive decline", "cls": "fungal"}, {"id": 10607, "dom": "D", "ent": "subjective cognitive decline", "cls": "parasit"}, {"id": 10608, "dom": "D", "ent": "parsley allergy", "cls": "cardio"}, {"id": 10609, "dom": "D", "ent": "parsley allergy", "cls": "resp"}, {"id": 10610, "dom": "D", "ent": "parsley allergy", "cls": "gi"}, {"id": 10611, "dom": "D", "ent": "parsley allergy", "cls": "nerv"}, {"id": 10612, "dom": "D", "ent": "parsley allergy", "cls": "skin"}, {"id": 10613, "dom": "D", "ent": "parsley allergy", "cls": "musc"}, {"id": 10614, "dom": "D", "ent": "parsley allergy", "cls": "urin"}, {"id": 10615, "dom": "D", "ent": "parsley allergy", "cls": "endo"}, {"id": 10616, "dom": "D", "ent": "parsley allergy", "cls": "immune"}, {"id": 10617, "dom": "D", "ent": "parsley allergy", "cls": "blood"}, {"id": 10618, "dom": "D", "ent": "parsley allergy", "cls": "repro"}, {"id": 10619, "dom": "D", "ent": "parsley allergy", "cls": "breast"}, {"id": 10620, "dom": "D", "ent": "parsley allergy", "cls": "chest"}, {"id": 10621, "dom": "D", "ent": "parsley allergy", "cls": "metab"}, {"id": 10622, "dom": "D", "ent": "parsley allergy", "cls": "infect"}, {"id": 10623, "dom": "D", "ent": "parsley allergy", "cls": "tumor"}, {"id": 10624, "dom": "D", "ent": "parsley allergy", "cls": "mental"}, {"id": 10625, "dom": "D", "ent": "parsley allergy", "cls": "genetic"}, {"id": 10626, "dom": "D", "ent": "parsley allergy", "cls": "bact"}, {"id": 10627, "dom": "D", "ent": "parsley allergy", "cls": "viral"}, {"id": 10628, "dom": "D", "ent": "parsley allergy", "cls": "fungal"}, {"id": 10629, "dom": "D", "ent": "parsley allergy", "cls": "parasit"}, {"id": 10630, "dom": "D", "ent": "traumatic brain injury", "cls": "cardio"}, {"id": 10631, "dom": "D", "ent": "traumatic brain injury", "cls": "resp"}, {"id": 10632, "dom": "D", "ent": "traumatic brain injury", "cls": "gi"}, {"id": 10633, "dom": "D", "ent": "traumatic brain injury", "cls": "nerv"}, {"id": 10634, "dom": "D", "ent": "traumatic brain injury", "cls": "skin"}, {"id": 10635, "dom": "D", "ent": "traumatic brain injury", "cls": "musc"}, {"id": 10636, "dom": "D", "ent": "traumatic brain injury", "cls": "urin"}, {"id": 10637, "dom": "D", "ent": "traumatic brain injury", "cls": "endo"}, {"id": 10638, "dom": "D", "ent": "traumatic brain injury", "cls": "immune"}, {"id": 10639, "dom": "D", "ent": "traumatic brain injury", "cls": "blood"}, {"id": 10640, "dom": "D", "ent": "traumatic brain injury", "cls": "repro"}, {"id": 10641, "dom": "D", "ent": "traumatic brain injury", "cls": "breast"}, {"id": 10642, "dom": "D", "ent": "traumatic brain injury", "cls": "chest"}, {"id": 10643, "dom": "D", "ent": "traumatic brain injury", "cls": "metab"}, {"id": 10644, "dom": "D", "ent": "traumatic brain injury", "cls": "infect"}, {"id": 10645, "dom": "D", "ent": "traumatic brain injury", "cls": "tumor"}, {"id": 10646, "dom": "D", "ent": "traumatic brain injury", "cls": "mental"}, {"id": 10647, "dom": "D", "ent": "traumatic brain injury", "cls": "genetic"}, {"id": 10648, "dom": "D", "ent": "traumatic brain injury", "cls": "bact"}, {"id": 10649, "dom": "D", "ent": "traumatic brain injury", "cls": "viral"}, {"id": 10650, "dom": "D", "ent": "traumatic brain injury", "cls": "fungal"}, {"id": 10651, "dom": "D", "ent": "traumatic brain injury", "cls": "parasit"}, {"id": 10652, "dom": "D", "ent": "Fraser syndrome", "cls": "cardio"}, {"id": 10653, "dom": "D", "ent": "Fraser syndrome", "cls": "resp"}, {"id": 10654, "dom": "D", "ent": "Fraser syndrome", "cls": "gi"}, {"id": 10655, "dom": "D", "ent": "Fraser syndrome", "cls": "nerv"}, {"id": 10656, "dom": "D", "ent": "Fraser syndrome", "cls": "skin"}, {"id": 10657, "dom": "D", "ent": "Fraser syndrome", "cls": "musc"}, {"id": 10658, "dom": "D", "ent": "Fraser syndrome", "cls": "urin"}, {"id": 10659, "dom": "D", "ent": "Fraser syndrome", "cls": "endo"}, {"id": 10660, "dom": "D", "ent": "Fraser syndrome", "cls": "immune"}, {"id": 10661, "dom": "D", "ent": "Fraser syndrome", "cls": "blood"}, {"id": 10662, "dom": "D", "ent": "Fraser syndrome", "cls": "repro"}, {"id": 10663, "dom": "D", "ent": "Fraser syndrome", "cls": "breast"}, {"id": 10664, "dom": "D", "ent": "Fraser syndrome", "cls": "chest"}, {"id": 10665, "dom": "D", "ent": "Fraser syndrome", "cls": "metab"}, {"id": 10666, "dom": "D", "ent": "Fraser syndrome", "cls": "infect"}, {"id": 10667, "dom": "D", "ent": "Fraser syndrome", "cls": "tumor"}, {"id": 10668, "dom": "D", "ent": "Fraser syndrome", "cls": "mental"}, {"id": 10669, "dom": "D", "ent": "Fraser syndrome", "cls": "genetic"}, {"id": 10670, "dom": "D", "ent": "Fraser syndrome", "cls": "bact"}, {"id": 10671, "dom": "D", "ent": "Fraser syndrome", "cls": "viral"}, {"id": 10672, "dom": "D", "ent": "Fraser syndrome", "cls": "fungal"}, {"id": 10673, "dom": "D", "ent": "Fraser syndrome", "cls": "parasit"}, {"id": 10674, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "cardio"}, {"id": 10675, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "resp"}, {"id": 10676, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "gi"}, {"id": 10677, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "nerv"}, {"id": 10678, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "skin"}, {"id": 10679, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "musc"}, {"id": 10680, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "urin"}, {"id": 10681, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "endo"}, {"id": 10682, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "immune"}, {"id": 10683, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "blood"}, {"id": 10684, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "repro"}, {"id": 10685, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "breast"}, {"id": 10686, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "chest"}, {"id": 10687, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "metab"}, {"id": 10688, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "infect"}, {"id": 10689, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "tumor"}, {"id": 10690, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "mental"}, {"id": 10691, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "genetic"}, {"id": 10692, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "bact"}, {"id": 10693, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "viral"}, {"id": 10694, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "fungal"}, {"id": 10695, "dom": "D", "ent": "enhanced S-cone syndrome", "cls": "parasit"}, {"id": 10696, "dom": "D", "ent": "Ballard syndrome", "cls": "cardio"}, {"id": 10697, "dom": "D", "ent": "Ballard syndrome", "cls": "resp"}, {"id": 10698, "dom": "D", "ent": "Ballard syndrome", "cls": "gi"}, {"id": 10699, "dom": "D", "ent": "Ballard syndrome", "cls": "nerv"}, {"id": 10700, "dom": "D", "ent": "Ballard syndrome", "cls": "skin"}, {"id": 10701, "dom": "D", "ent": "Ballard syndrome", "cls": "musc"}, {"id": 10702, "dom": "D", "ent": "Ballard syndrome", "cls": "urin"}, {"id": 10703, "dom": "D", "ent": "Ballard syndrome", "cls": "endo"}, {"id": 10704, "dom": "D", "ent": "Ballard syndrome", "cls": "immune"}, {"id": 10705, "dom": "D", "ent": "Ballard syndrome", "cls": "blood"}, {"id": 10706, "dom": "D", "ent": "Ballard syndrome", "cls": "repro"}, {"id": 10707, "dom": "D", "ent": "Ballard syndrome", "cls": "breast"}, {"id": 10708, "dom": "D", "ent": "Ballard syndrome", "cls": "chest"}, {"id": 10709, "dom": "D", "ent": "Ballard syndrome", "cls": "metab"}, {"id": 10710, "dom": "D", "ent": "Ballard syndrome", "cls": "infect"}, {"id": 10711, "dom": "D", "ent": "Ballard syndrome", "cls": "tumor"}, {"id": 10712, "dom": "D", "ent": "Ballard syndrome", "cls": "mental"}, {"id": 10713, "dom": "D", "ent": "Ballard syndrome", "cls": "genetic"}, {"id": 10714, "dom": "D", "ent": "Ballard syndrome", "cls": "bact"}, {"id": 10715, "dom": "D", "ent": "Ballard syndrome", "cls": "viral"}, {"id": 10716, "dom": "D", "ent": "Ballard syndrome", "cls": "fungal"}, {"id": 10717, "dom": "D", "ent": "Ballard syndrome", "cls": "parasit"}, {"id": 10718, "dom": "D", "ent": "SHORT syndrome", "cls": "cardio"}, {"id": 10719, "dom": "D", "ent": "SHORT syndrome", "cls": "resp"}, {"id": 10720, "dom": "D", "ent": "SHORT syndrome", "cls": "gi"}, {"id": 10721, "dom": "D", "ent": "SHORT syndrome", "cls": "nerv"}, {"id": 10722, "dom": "D", "ent": "SHORT syndrome", "cls": "skin"}, {"id": 10723, "dom": "D", "ent": "SHORT syndrome", "cls": "musc"}, {"id": 10724, "dom": "D", "ent": "SHORT syndrome", "cls": "urin"}, {"id": 10725, "dom": "D", "ent": "SHORT syndrome", "cls": "endo"}, {"id": 10726, "dom": "D", "ent": "SHORT syndrome", "cls": "immune"}, {"id": 10727, "dom": "D", "ent": "SHORT syndrome", "cls": "blood"}, {"id": 10728, "dom": "D", "ent": "SHORT syndrome", "cls": "repro"}, {"id": 10729, "dom": "D", "ent": "SHORT syndrome", "cls": "breast"}, {"id": 10730, "dom": "D", "ent": "SHORT syndrome", "cls": "chest"}, {"id": 10731, "dom": "D", "ent": "SHORT syndrome", "cls": "metab"}, {"id": 10732, "dom": "D", "ent": "SHORT syndrome", "cls": "infect"}, {"id": 10733, "dom": "D", "ent": "SHORT syndrome", "cls": "tumor"}, {"id": 10734, "dom": "D", "ent": "SHORT syndrome", "cls": "mental"}, {"id": 10735, "dom": "D", "ent": "SHORT syndrome", "cls": "genetic"}, {"id": 10736, "dom": "D", "ent": "SHORT syndrome", "cls": "bact"}, {"id": 10737, "dom": "D", "ent": "SHORT syndrome", "cls": "viral"}, {"id": 10738, "dom": "D", "ent": "SHORT syndrome", "cls": "fungal"}, {"id": 10739, "dom": "D", "ent": "SHORT syndrome", "cls": "parasit"}, {"id": 10740, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "cardio"}, {"id": 10741, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "resp"}, {"id": 10742, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "gi"}, {"id": 10743, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "nerv"}, {"id": 10744, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "skin"}, {"id": 10745, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "musc"}, {"id": 10746, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "urin"}, {"id": 10747, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "endo"}, {"id": 10748, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "immune"}, {"id": 10749, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "blood"}, {"id": 10750, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "repro"}, {"id": 10751, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "breast"}, {"id": 10752, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "chest"}, {"id": 10753, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "metab"}, {"id": 10754, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "infect"}, {"id": 10755, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "tumor"}, {"id": 10756, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "mental"}, {"id": 10757, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "genetic"}, {"id": 10758, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "bact"}, {"id": 10759, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "viral"}, {"id": 10760, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "fungal"}, {"id": 10761, "dom": "D", "ent": "Gordon Holmes syndrome", "cls": "parasit"}, {"id": 10762, "dom": "D", "ent": "CHILD syndrome", "cls": "cardio"}, {"id": 10763, "dom": "D", "ent": "CHILD syndrome", "cls": "resp"}, {"id": 10764, "dom": "D", "ent": "CHILD syndrome", "cls": "gi"}, {"id": 10765, "dom": "D", "ent": "CHILD syndrome", "cls": "nerv"}, {"id": 10766, "dom": "D", "ent": "CHILD syndrome", "cls": "skin"}, {"id": 10767, "dom": "D", "ent": "CHILD syndrome", "cls": "musc"}, {"id": 10768, "dom": "D", "ent": "CHILD syndrome", "cls": "urin"}, {"id": 10769, "dom": "D", "ent": "CHILD syndrome", "cls": "endo"}, {"id": 10770, "dom": "D", "ent": "CHILD syndrome", "cls": "immune"}, {"id": 10771, "dom": "D", "ent": "CHILD syndrome", "cls": "blood"}, {"id": 10772, "dom": "D", "ent": "CHILD syndrome", "cls": "repro"}, {"id": 10773, "dom": "D", "ent": "CHILD syndrome", "cls": "breast"}, {"id": 10774, "dom": "D", "ent": "CHILD syndrome", "cls": "chest"}, {"id": 10775, "dom": "D", "ent": "CHILD syndrome", "cls": "metab"}, {"id": 10776, "dom": "D", "ent": "CHILD syndrome", "cls": "infect"}, {"id": 10777, "dom": "D", "ent": "CHILD syndrome", "cls": "tumor"}, {"id": 10778, "dom": "D", "ent": "CHILD syndrome", "cls": "mental"}, {"id": 10779, "dom": "D", "ent": "CHILD syndrome", "cls": "genetic"}, {"id": 10780, "dom": "D", "ent": "CHILD syndrome", "cls": "bact"}, {"id": 10781, "dom": "D", "ent": "CHILD syndrome", "cls": "viral"}, {"id": 10782, "dom": "D", "ent": "CHILD syndrome", "cls": "fungal"}, {"id": 10783, "dom": "D", "ent": "CHILD syndrome", "cls": "parasit"}, {"id": 10784, "dom": "D", "ent": "CK syndrome", "cls": "cardio"}, {"id": 10785, "dom": "D", "ent": "CK syndrome", "cls": "resp"}, {"id": 10786, "dom": "D", "ent": "CK syndrome", "cls": "gi"}, {"id": 10787, "dom": "D", "ent": "CK syndrome", "cls": "nerv"}, {"id": 10788, "dom": "D", "ent": "CK syndrome", "cls": "skin"}, {"id": 10789, "dom": "D", "ent": "CK syndrome", "cls": "musc"}, {"id": 10790, "dom": "D", "ent": "CK syndrome", "cls": "urin"}, {"id": 10791, "dom": "D", "ent": "CK syndrome", "cls": "endo"}, {"id": 10792, "dom": "D", "ent": "CK syndrome", "cls": "immune"}, {"id": 10793, "dom": "D", "ent": "CK syndrome", "cls": "blood"}, {"id": 10794, "dom": "D", "ent": "CK syndrome", "cls": "repro"}, {"id": 10795, "dom": "D", "ent": "CK syndrome", "cls": "breast"}, {"id": 10796, "dom": "D", "ent": "CK syndrome", "cls": "chest"}, {"id": 10797, "dom": "D", "ent": "CK syndrome", "cls": "metab"}, {"id": 10798, "dom": "D", "ent": "CK syndrome", "cls": "infect"}, {"id": 10799, "dom": "D", "ent": "CK syndrome", "cls": "tumor"}, {"id": 10800, "dom": "D", "ent": "CK syndrome", "cls": "mental"}, {"id": 10801, "dom": "D", "ent": "CK syndrome", "cls": "genetic"}, {"id": 10802, "dom": "D", "ent": "CK syndrome", "cls": "bact"}, {"id": 10803, "dom": "D", "ent": "CK syndrome", "cls": "viral"}, {"id": 10804, "dom": "D", "ent": "CK syndrome", "cls": "fungal"}, {"id": 10805, "dom": "D", "ent": "CK syndrome", "cls": "parasit"}, {"id": 10806, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "cardio"}, {"id": 10807, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "resp"}, {"id": 10808, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "gi"}, {"id": 10809, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "nerv"}, {"id": 10810, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "skin"}, {"id": 10811, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "musc"}, {"id": 10812, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "urin"}, {"id": 10813, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "endo"}, {"id": 10814, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "immune"}, {"id": 10815, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "blood"}, {"id": 10816, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "repro"}, {"id": 10817, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "breast"}, {"id": 10818, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "chest"}, {"id": 10819, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "metab"}, {"id": 10820, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "infect"}, {"id": 10821, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "tumor"}, {"id": 10822, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "mental"}, {"id": 10823, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "genetic"}, {"id": 10824, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "bact"}, {"id": 10825, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "viral"}, {"id": 10826, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "fungal"}, {"id": 10827, "dom": "D", "ent": "pelvic inflammatory disease", "cls": "parasit"}, {"id": 10828, "dom": "D", "ent": "small intestine cancer", "cls": "cardio"}, {"id": 10829, "dom": "D", "ent": "small intestine cancer", "cls": "resp"}, {"id": 10830, "dom": "D", "ent": "small intestine cancer", "cls": "gi"}, {"id": 10831, "dom": "D", "ent": "small intestine cancer", "cls": "nerv"}, {"id": 10832, "dom": "D", "ent": "small intestine cancer", "cls": "skin"}, {"id": 10833, "dom": "D", "ent": "small intestine cancer", "cls": "musc"}, {"id": 10834, "dom": "D", "ent": "small intestine cancer", "cls": "urin"}, {"id": 10835, "dom": "D", "ent": "small intestine cancer", "cls": "endo"}, {"id": 10836, "dom": "D", "ent": "small intestine cancer", "cls": "immune"}, {"id": 10837, "dom": "D", "ent": "small intestine cancer", "cls": "blood"}, {"id": 10838, "dom": "D", "ent": "small intestine cancer", "cls": "repro"}, {"id": 10839, "dom": "D", "ent": "small intestine cancer", "cls": "breast"}, {"id": 10840, "dom": "D", "ent": "small intestine cancer", "cls": "chest"}, {"id": 10841, "dom": "D", "ent": "small intestine cancer", "cls": "metab"}, {"id": 10842, "dom": "D", "ent": "small intestine cancer", "cls": "infect"}, {"id": 10843, "dom": "D", "ent": "small intestine cancer", "cls": "tumor"}, {"id": 10844, "dom": "D", "ent": "small intestine cancer", "cls": "mental"}, {"id": 10845, "dom": "D", "ent": "small intestine cancer", "cls": "genetic"}, {"id": 10846, "dom": "D", "ent": "small intestine cancer", "cls": "bact"}, {"id": 10847, "dom": "D", "ent": "small intestine cancer", "cls": "viral"}, {"id": 10848, "dom": "D", "ent": "small intestine cancer", "cls": "fungal"}, {"id": 10849, "dom": "D", "ent": "small intestine cancer", "cls": "parasit"}, {"id": 10850, "dom": "D", "ent": "breast cyst", "cls": "cardio"}, {"id": 10851, "dom": "D", "ent": "breast cyst", "cls": "resp"}, {"id": 10852, "dom": "D", "ent": "breast cyst", "cls": "gi"}, {"id": 10853, "dom": "D", "ent": "breast cyst", "cls": "nerv"}, {"id": 10854, "dom": "D", "ent": "breast cyst", "cls": "skin"}, {"id": 10855, "dom": "D", "ent": "breast cyst", "cls": "musc"}, {"id": 10856, "dom": "D", "ent": "breast cyst", "cls": "urin"}, {"id": 10857, "dom": "D", "ent": "breast cyst", "cls": "endo"}, {"id": 10858, "dom": "D", "ent": "breast cyst", "cls": "immune"}, {"id": 10859, "dom": "D", "ent": "breast cyst", "cls": "blood"}, {"id": 10860, "dom": "D", "ent": "breast cyst", "cls": "repro"}, {"id": 10861, "dom": "D", "ent": "breast cyst", "cls": "breast"}, {"id": 10862, "dom": "D", "ent": "breast cyst", "cls": "chest"}, {"id": 10863, "dom": "D", "ent": "breast cyst", "cls": "metab"}, {"id": 10864, "dom": "D", "ent": "breast cyst", "cls": "infect"}, {"id": 10865, "dom": "D", "ent": "breast cyst", "cls": "tumor"}, {"id": 10866, "dom": "D", "ent": "breast cyst", "cls": "mental"}, {"id": 10867, "dom": "D", "ent": "breast cyst", "cls": "genetic"}, {"id": 10868, "dom": "D", "ent": "breast cyst", "cls": "bact"}, {"id": 10869, "dom": "D", "ent": "breast cyst", "cls": "viral"}, {"id": 10870, "dom": "D", "ent": "breast cyst", "cls": "fungal"}, {"id": 10871, "dom": "D", "ent": "breast cyst", "cls": "parasit"}, {"id": 10872, "dom": "D", "ent": "stomach cancer", "cls": "cardio"}, {"id": 10873, "dom": "D", "ent": "stomach cancer", "cls": "resp"}, {"id": 10874, "dom": "D", "ent": "stomach cancer", "cls": "gi"}, {"id": 10875, "dom": "D", "ent": "stomach cancer", "cls": "nerv"}, {"id": 10876, "dom": "D", "ent": "stomach cancer", "cls": "skin"}, {"id": 10877, "dom": "D", "ent": "stomach cancer", "cls": "musc"}, {"id": 10878, "dom": "D", "ent": "stomach cancer", "cls": "urin"}, {"id": 10879, "dom": "D", "ent": "stomach cancer", "cls": "endo"}, {"id": 10880, "dom": "D", "ent": "stomach cancer", "cls": "immune"}, {"id": 10881, "dom": "D", "ent": "stomach cancer", "cls": "blood"}, {"id": 10882, "dom": "D", "ent": "stomach cancer", "cls": "repro"}, {"id": 10883, "dom": "D", "ent": "stomach cancer", "cls": "breast"}, {"id": 10884, "dom": "D", "ent": "stomach cancer", "cls": "chest"}, {"id": 10885, "dom": "D", "ent": "stomach cancer", "cls": "metab"}, {"id": 10886, "dom": "D", "ent": "stomach cancer", "cls": "infect"}, {"id": 10887, "dom": "D", "ent": "stomach cancer", "cls": "tumor"}, {"id": 10888, "dom": "D", "ent": "stomach cancer", "cls": "mental"}, {"id": 10889, "dom": "D", "ent": "stomach cancer", "cls": "genetic"}, {"id": 10890, "dom": "D", "ent": "stomach cancer", "cls": "bact"}, {"id": 10891, "dom": "D", "ent": "stomach cancer", "cls": "viral"}, {"id": 10892, "dom": "D", "ent": "stomach cancer", "cls": "fungal"}, {"id": 10893, "dom": "D", "ent": "stomach cancer", "cls": "parasit"}, {"id": 10894, "dom": "D", "ent": "intellectual disability", "cls": "cardio"}, {"id": 10895, "dom": "D", "ent": "intellectual disability", "cls": "resp"}, {"id": 10896, "dom": "D", "ent": "intellectual disability", "cls": "gi"}, {"id": 10897, "dom": "D", "ent": "intellectual disability", "cls": "nerv"}, {"id": 10898, "dom": "D", "ent": "intellectual disability", "cls": "skin"}, {"id": 10899, "dom": "D", "ent": "intellectual disability", "cls": "musc"}, {"id": 10900, "dom": "D", "ent": "intellectual disability", "cls": "urin"}, {"id": 10901, "dom": "D", "ent": "intellectual disability", "cls": "endo"}, {"id": 10902, "dom": "D", "ent": "intellectual disability", "cls": "immune"}, {"id": 10903, "dom": "D", "ent": "intellectual disability", "cls": "blood"}, {"id": 10904, "dom": "D", "ent": "intellectual disability", "cls": "repro"}, {"id": 10905, "dom": "D", "ent": "intellectual disability", "cls": "breast"}, {"id": 10906, "dom": "D", "ent": "intellectual disability", "cls": "chest"}, {"id": 10907, "dom": "D", "ent": "intellectual disability", "cls": "metab"}, {"id": 10908, "dom": "D", "ent": "intellectual disability", "cls": "infect"}, {"id": 10909, "dom": "D", "ent": "intellectual disability", "cls": "tumor"}, {"id": 10910, "dom": "D", "ent": "intellectual disability", "cls": "mental"}, {"id": 10911, "dom": "D", "ent": "intellectual disability", "cls": "genetic"}, {"id": 10912, "dom": "D", "ent": "intellectual disability", "cls": "bact"}, {"id": 10913, "dom": "D", "ent": "intellectual disability", "cls": "viral"}, {"id": 10914, "dom": "D", "ent": "intellectual disability", "cls": "fungal"}, {"id": 10915, "dom": "D", "ent": "intellectual disability", "cls": "parasit"}, {"id": 10916, "dom": "D", "ent": "short bowel syndrome", "cls": "cardio"}, {"id": 10917, "dom": "D", "ent": "short bowel syndrome", "cls": "resp"}, {"id": 10918, "dom": "D", "ent": "short bowel syndrome", "cls": "gi"}, {"id": 10919, "dom": "D", "ent": "short bowel syndrome", "cls": "nerv"}, {"id": 10920, "dom": "D", "ent": "short bowel syndrome", "cls": "skin"}, {"id": 10921, "dom": "D", "ent": "short bowel syndrome", "cls": "musc"}, {"id": 10922, "dom": "D", "ent": "short bowel syndrome", "cls": "urin"}, {"id": 10923, "dom": "D", "ent": "short bowel syndrome", "cls": "endo"}, {"id": 10924, "dom": "D", "ent": "short bowel syndrome", "cls": "immune"}, {"id": 10925, "dom": "D", "ent": "short bowel syndrome", "cls": "blood"}, {"id": 10926, "dom": "D", "ent": "short bowel syndrome", "cls": "repro"}, {"id": 10927, "dom": "D", "ent": "short bowel syndrome", "cls": "breast"}, {"id": 10928, "dom": "D", "ent": "short bowel syndrome", "cls": "chest"}, {"id": 10929, "dom": "D", "ent": "short bowel syndrome", "cls": "metab"}, {"id": 10930, "dom": "D", "ent": "short bowel syndrome", "cls": "infect"}, {"id": 10931, "dom": "D", "ent": "short bowel syndrome", "cls": "tumor"}, {"id": 10932, "dom": "D", "ent": "short bowel syndrome", "cls": "mental"}, {"id": 10933, "dom": "D", "ent": "short bowel syndrome", "cls": "genetic"}, {"id": 10934, "dom": "D", "ent": "short bowel syndrome", "cls": "bact"}, {"id": 10935, "dom": "D", "ent": "short bowel syndrome", "cls": "viral"}, {"id": 10936, "dom": "D", "ent": "short bowel syndrome", "cls": "fungal"}, {"id": 10937, "dom": "D", "ent": "short bowel syndrome", "cls": "parasit"}, {"id": 10938, "dom": "D", "ent": "separation anxiety disorder", "cls": "cardio"}, {"id": 10939, "dom": "D", "ent": "separation anxiety disorder", "cls": "resp"}, {"id": 10940, "dom": "D", "ent": "separation anxiety disorder", "cls": "gi"}, {"id": 10941, "dom": "D", "ent": "separation anxiety disorder", "cls": "nerv"}, {"id": 10942, "dom": "D", "ent": "separation anxiety disorder", "cls": "skin"}, {"id": 10943, "dom": "D", "ent": "separation anxiety disorder", "cls": "musc"}, {"id": 10944, "dom": "D", "ent": "separation anxiety disorder", "cls": "urin"}, {"id": 10945, "dom": "D", "ent": "separation anxiety disorder", "cls": "endo"}, {"id": 10946, "dom": "D", "ent": "separation anxiety disorder", "cls": "immune"}, {"id": 10947, "dom": "D", "ent": "separation anxiety disorder", "cls": "blood"}, {"id": 10948, "dom": "D", "ent": "separation anxiety disorder", "cls": "repro"}, {"id": 10949, "dom": "D", "ent": "separation anxiety disorder", "cls": "breast"}, {"id": 10950, "dom": "D", "ent": "separation anxiety disorder", "cls": "chest"}, {"id": 10951, "dom": "D", "ent": "separation anxiety disorder", "cls": "metab"}, {"id": 10952, "dom": "D", "ent": "separation anxiety disorder", "cls": "infect"}, {"id": 10953, "dom": "D", "ent": "separation anxiety disorder", "cls": "tumor"}, {"id": 10954, "dom": "D", "ent": "separation anxiety disorder", "cls": "mental"}, {"id": 10955, "dom": "D", "ent": "separation anxiety disorder", "cls": "genetic"}, {"id": 10956, "dom": "D", "ent": "separation anxiety disorder", "cls": "bact"}, {"id": 10957, "dom": "D", "ent": "separation anxiety disorder", "cls": "viral"}, {"id": 10958, "dom": "D", "ent": "separation anxiety disorder", "cls": "fungal"}, {"id": 10959, "dom": "D", "ent": "separation anxiety disorder", "cls": "parasit"}, {"id": 10960, "dom": "D", "ent": "renal hypertension", "cls": "cardio"}, {"id": 10961, "dom": "D", "ent": "renal hypertension", "cls": "resp"}, {"id": 10962, "dom": "D", "ent": "renal hypertension", "cls": "gi"}, {"id": 10963, "dom": "D", "ent": "renal hypertension", "cls": "nerv"}, {"id": 10964, "dom": "D", "ent": "renal hypertension", "cls": "skin"}, {"id": 10965, "dom": "D", "ent": "renal hypertension", "cls": "musc"}, {"id": 10966, "dom": "D", "ent": "renal hypertension", "cls": "urin"}, {"id": 10967, "dom": "D", "ent": "renal hypertension", "cls": "endo"}, {"id": 10968, "dom": "D", "ent": "renal hypertension", "cls": "immune"}, {"id": 10969, "dom": "D", "ent": "renal hypertension", "cls": "blood"}, {"id": 10970, "dom": "D", "ent": "renal hypertension", "cls": "repro"}, {"id": 10971, "dom": "D", "ent": "renal hypertension", "cls": "breast"}, {"id": 10972, "dom": "D", "ent": "renal hypertension", "cls": "chest"}, {"id": 10973, "dom": "D", "ent": "renal hypertension", "cls": "metab"}, {"id": 10974, "dom": "D", "ent": "renal hypertension", "cls": "infect"}, {"id": 10975, "dom": "D", "ent": "renal hypertension", "cls": "tumor"}, {"id": 10976, "dom": "D", "ent": "renal hypertension", "cls": "mental"}, {"id": 10977, "dom": "D", "ent": "renal hypertension", "cls": "genetic"}, {"id": 10978, "dom": "D", "ent": "renal hypertension", "cls": "bact"}, {"id": 10979, "dom": "D", "ent": "renal hypertension", "cls": "viral"}, {"id": 10980, "dom": "D", "ent": "renal hypertension", "cls": "fungal"}, {"id": 10981, "dom": "D", "ent": "renal hypertension", "cls": "parasit"}, {"id": 10982, "dom": "D", "ent": "kidney failure", "cls": "cardio"}, {"id": 10983, "dom": "D", "ent": "kidney failure", "cls": "resp"}, {"id": 10984, "dom": "D", "ent": "kidney failure", "cls": "gi"}, {"id": 10985, "dom": "D", "ent": "kidney failure", "cls": "nerv"}, {"id": 10986, "dom": "D", "ent": "kidney failure", "cls": "skin"}, {"id": 10987, "dom": "D", "ent": "kidney failure", "cls": "musc"}, {"id": 10988, "dom": "D", "ent": "kidney failure", "cls": "urin"}, {"id": 10989, "dom": "D", "ent": "kidney failure", "cls": "endo"}, {"id": 10990, "dom": "D", "ent": "kidney failure", "cls": "immune"}, {"id": 10991, "dom": "D", "ent": "kidney failure", "cls": "blood"}, {"id": 10992, "dom": "D", "ent": "kidney failure", "cls": "repro"}, {"id": 10993, "dom": "D", "ent": "kidney failure", "cls": "breast"}, {"id": 10994, "dom": "D", "ent": "kidney failure", "cls": "chest"}, {"id": 10995, "dom": "D", "ent": "kidney failure", "cls": "metab"}, {"id": 10996, "dom": "D", "ent": "kidney failure", "cls": "infect"}, {"id": 10997, "dom": "D", "ent": "kidney failure", "cls": "tumor"}, {"id": 10998, "dom": "D", "ent": "kidney failure", "cls": "mental"}, {"id": 10999, "dom": "D", "ent": "kidney failure", "cls": "genetic"}, {"id": 11000, "dom": "D", "ent": "kidney failure", "cls": "bact"}, {"id": 11001, "dom": "D", "ent": "kidney failure", "cls": "viral"}, {"id": 11002, "dom": "D", "ent": "kidney failure", "cls": "fungal"}, {"id": 11003, "dom": "D", "ent": "kidney failure", "cls": "parasit"}, {"id": 11004, "dom": "D", "ent": "premature menopause", "cls": "cardio"}, {"id": 11005, "dom": "D", "ent": "premature menopause", "cls": "resp"}, {"id": 11006, "dom": "D", "ent": "premature menopause", "cls": "gi"}, {"id": 11007, "dom": "D", "ent": "premature menopause", "cls": "nerv"}, {"id": 11008, "dom": "D", "ent": "premature menopause", "cls": "skin"}, {"id": 11009, "dom": "D", "ent": "premature menopause", "cls": "musc"}, {"id": 11010, "dom": "D", "ent": "premature menopause", "cls": "urin"}, {"id": 11011, "dom": "D", "ent": "premature menopause", "cls": "endo"}, {"id": 11012, "dom": "D", "ent": "premature menopause", "cls": "immune"}, {"id": 11013, "dom": "D", "ent": "premature menopause", "cls": "blood"}, {"id": 11014, "dom": "D", "ent": "premature menopause", "cls": "repro"}, {"id": 11015, "dom": "D", "ent": "premature menopause", "cls": "breast"}, {"id": 11016, "dom": "D", "ent": "premature menopause", "cls": "chest"}, {"id": 11017, "dom": "D", "ent": "premature menopause", "cls": "metab"}, {"id": 11018, "dom": "D", "ent": "premature menopause", "cls": "infect"}, {"id": 11019, "dom": "D", "ent": "premature menopause", "cls": "tumor"}, {"id": 11020, "dom": "D", "ent": "premature menopause", "cls": "mental"}, {"id": 11021, "dom": "D", "ent": "premature menopause", "cls": "genetic"}, {"id": 11022, "dom": "D", "ent": "premature menopause", "cls": "bact"}, {"id": 11023, "dom": "D", "ent": "premature menopause", "cls": "viral"}, {"id": 11024, "dom": "D", "ent": "premature menopause", "cls": "fungal"}, {"id": 11025, "dom": "D", "ent": "premature menopause", "cls": "parasit"}, {"id": 11026, "dom": "D", "ent": "malignant hypertension", "cls": "cardio"}, {"id": 11027, "dom": "D", "ent": "malignant hypertension", "cls": "resp"}, {"id": 11028, "dom": "D", "ent": "malignant hypertension", "cls": "gi"}, {"id": 11029, "dom": "D", "ent": "malignant hypertension", "cls": "nerv"}, {"id": 11030, "dom": "D", "ent": "malignant hypertension", "cls": "skin"}, {"id": 11031, "dom": "D", "ent": "malignant hypertension", "cls": "musc"}, {"id": 11032, "dom": "D", "ent": "malignant hypertension", "cls": "urin"}, {"id": 11033, "dom": "D", "ent": "malignant hypertension", "cls": "endo"}, {"id": 11034, "dom": "D", "ent": "malignant hypertension", "cls": "immune"}, {"id": 11035, "dom": "D", "ent": "malignant hypertension", "cls": "blood"}, {"id": 11036, "dom": "D", "ent": "malignant hypertension", "cls": "repro"}, {"id": 11037, "dom": "D", "ent": "malignant hypertension", "cls": "breast"}, {"id": 11038, "dom": "D", "ent": "malignant hypertension", "cls": "chest"}, {"id": 11039, "dom": "D", "ent": "malignant hypertension", "cls": "metab"}, {"id": 11040, "dom": "D", "ent": "malignant hypertension", "cls": "infect"}, {"id": 11041, "dom": "D", "ent": "malignant hypertension", "cls": "tumor"}, {"id": 11042, "dom": "D", "ent": "malignant hypertension", "cls": "mental"}, {"id": 11043, "dom": "D", "ent": "malignant hypertension", "cls": "genetic"}, {"id": 11044, "dom": "D", "ent": "malignant hypertension", "cls": "bact"}, {"id": 11045, "dom": "D", "ent": "malignant hypertension", "cls": "viral"}, {"id": 11046, "dom": "D", "ent": "malignant hypertension", "cls": "fungal"}, {"id": 11047, "dom": "D", "ent": "malignant hypertension", "cls": "parasit"}, {"id": 11048, "dom": "D", "ent": "tooth disease", "cls": "cardio"}, {"id": 11049, "dom": "D", "ent": "tooth disease", "cls": "resp"}, {"id": 11050, "dom": "D", "ent": "tooth disease", "cls": "gi"}, {"id": 11051, "dom": "D", "ent": "tooth disease", "cls": "nerv"}, {"id": 11052, "dom": "D", "ent": "tooth disease", "cls": "skin"}, {"id": 11053, "dom": "D", "ent": "tooth disease", "cls": "musc"}, {"id": 11054, "dom": "D", "ent": "tooth disease", "cls": "urin"}, {"id": 11055, "dom": "D", "ent": "tooth disease", "cls": "endo"}, {"id": 11056, "dom": "D", "ent": "tooth disease", "cls": "immune"}, {"id": 11057, "dom": "D", "ent": "tooth disease", "cls": "blood"}, {"id": 11058, "dom": "D", "ent": "tooth disease", "cls": "repro"}, {"id": 11059, "dom": "D", "ent": "tooth disease", "cls": "breast"}, {"id": 11060, "dom": "D", "ent": "tooth disease", "cls": "chest"}, {"id": 11061, "dom": "D", "ent": "tooth disease", "cls": "metab"}, {"id": 11062, "dom": "D", "ent": "tooth disease", "cls": "infect"}, {"id": 11063, "dom": "D", "ent": "tooth disease", "cls": "tumor"}, {"id": 11064, "dom": "D", "ent": "tooth disease", "cls": "mental"}, {"id": 11065, "dom": "D", "ent": "tooth disease", "cls": "genetic"}, {"id": 11066, "dom": "D", "ent": "tooth disease", "cls": "bact"}, {"id": 11067, "dom": "D", "ent": "tooth disease", "cls": "viral"}, {"id": 11068, "dom": "D", "ent": "tooth disease", "cls": "fungal"}, {"id": 11069, "dom": "D", "ent": "tooth disease", "cls": "parasit"}, {"id": 11070, "dom": "D", "ent": "borderline personality disorder", "cls": "cardio"}, {"id": 11071, "dom": "D", "ent": "borderline personality disorder", "cls": "resp"}, {"id": 11072, "dom": "D", "ent": "borderline personality disorder", "cls": "gi"}, {"id": 11073, "dom": "D", "ent": "borderline personality disorder", "cls": "nerv"}, {"id": 11074, "dom": "D", "ent": "borderline personality disorder", "cls": "skin"}, {"id": 11075, "dom": "D", "ent": "borderline personality disorder", "cls": "musc"}, {"id": 11076, "dom": "D", "ent": "borderline personality disorder", "cls": "urin"}, {"id": 11077, "dom": "D", "ent": "borderline personality disorder", "cls": "endo"}, {"id": 11078, "dom": "D", "ent": "borderline personality disorder", "cls": "immune"}, {"id": 11079, "dom": "D", "ent": "borderline personality disorder", "cls": "blood"}, {"id": 11080, "dom": "D", "ent": "borderline personality disorder", "cls": "repro"}, {"id": 11081, "dom": "D", "ent": "borderline personality disorder", "cls": "breast"}, {"id": 11082, "dom": "D", "ent": "borderline personality disorder", "cls": "chest"}, {"id": 11083, "dom": "D", "ent": "borderline personality disorder", "cls": "metab"}, {"id": 11084, "dom": "D", "ent": "borderline personality disorder", "cls": "infect"}, {"id": 11085, "dom": "D", "ent": "borderline personality disorder", "cls": "tumor"}, {"id": 11086, "dom": "D", "ent": "borderline personality disorder", "cls": "mental"}, {"id": 11087, "dom": "D", "ent": "borderline personality disorder", "cls": "genetic"}, {"id": 11088, "dom": "D", "ent": "borderline personality disorder", "cls": "bact"}, {"id": 11089, "dom": "D", "ent": "borderline personality disorder", "cls": "viral"}, {"id": 11090, "dom": "D", "ent": "borderline personality disorder", "cls": "fungal"}, {"id": 11091, "dom": "D", "ent": "borderline personality disorder", "cls": "parasit"}, {"id": 11092, "dom": "D", "ent": "dependent personality disorder", "cls": "cardio"}, {"id": 11093, "dom": "D", "ent": "dependent personality disorder", "cls": "resp"}, {"id": 11094, "dom": "D", "ent": "dependent personality disorder", "cls": "gi"}, {"id": 11095, "dom": "D", "ent": "dependent personality disorder", "cls": "nerv"}, {"id": 11096, "dom": "D", "ent": "dependent personality disorder", "cls": "skin"}, {"id": 11097, "dom": "D", "ent": "dependent personality disorder", "cls": "musc"}, {"id": 11098, "dom": "D", "ent": "dependent personality disorder", "cls": "urin"}, {"id": 11099, "dom": "D", "ent": "dependent personality disorder", "cls": "endo"}, {"id": 11100, "dom": "D", "ent": "dependent personality disorder", "cls": "immune"}, {"id": 11101, "dom": "D", "ent": "dependent personality disorder", "cls": "blood"}, {"id": 11102, "dom": "D", "ent": "dependent personality disorder", "cls": "repro"}, {"id": 11103, "dom": "D", "ent": "dependent personality disorder", "cls": "breast"}, {"id": 11104, "dom": "D", "ent": "dependent personality disorder", "cls": "chest"}, {"id": 11105, "dom": "D", "ent": "dependent personality disorder", "cls": "metab"}, {"id": 11106, "dom": "D", "ent": "dependent personality disorder", "cls": "infect"}, {"id": 11107, "dom": "D", "ent": "dependent personality disorder", "cls": "tumor"}, {"id": 11108, "dom": "D", "ent": "dependent personality disorder", "cls": "mental"}, {"id": 11109, "dom": "D", "ent": "dependent personality disorder", "cls": "genetic"}, {"id": 11110, "dom": "D", "ent": "dependent personality disorder", "cls": "bact"}, {"id": 11111, "dom": "D", "ent": "dependent personality disorder", "cls": "viral"}, {"id": 11112, "dom": "D", "ent": "dependent personality disorder", "cls": "fungal"}, {"id": 11113, "dom": "D", "ent": "dependent personality disorder", "cls": "parasit"}, {"id": 11114, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "cardio"}, {"id": 11115, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "resp"}, {"id": 11116, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "gi"}, {"id": 11117, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "nerv"}, {"id": 11118, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "skin"}, {"id": 11119, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "musc"}, {"id": 11120, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "urin"}, {"id": 11121, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "endo"}, {"id": 11122, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "immune"}, {"id": 11123, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "blood"}, {"id": 11124, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "repro"}, {"id": 11125, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "breast"}, {"id": 11126, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "chest"}, {"id": 11127, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "metab"}, {"id": 11128, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "infect"}, {"id": 11129, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "tumor"}, {"id": 11130, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "mental"}, {"id": 11131, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "genetic"}, {"id": 11132, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "bact"}, {"id": 11133, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "viral"}, {"id": 11134, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "fungal"}, {"id": 11135, "dom": "D", "ent": "obsessive-compulsive disorder", "cls": "parasit"}, {"id": 11136, "dom": "D", "ent": "multiple personality disorder", "cls": "cardio"}, {"id": 11137, "dom": "D", "ent": "multiple personality disorder", "cls": "resp"}, {"id": 11138, "dom": "D", "ent": "multiple personality disorder", "cls": "gi"}, {"id": 11139, "dom": "D", "ent": "multiple personality disorder", "cls": "nerv"}, {"id": 11140, "dom": "D", "ent": "multiple personality disorder", "cls": "skin"}, {"id": 11141, "dom": "D", "ent": "multiple personality disorder", "cls": "musc"}, {"id": 11142, "dom": "D", "ent": "multiple personality disorder", "cls": "urin"}, {"id": 11143, "dom": "D", "ent": "multiple personality disorder", "cls": "endo"}, {"id": 11144, "dom": "D", "ent": "multiple personality disorder", "cls": "immune"}, {"id": 11145, "dom": "D", "ent": "multiple personality disorder", "cls": "blood"}, {"id": 11146, "dom": "D", "ent": "multiple personality disorder", "cls": "repro"}, {"id": 11147, "dom": "D", "ent": "multiple personality disorder", "cls": "breast"}, {"id": 11148, "dom": "D", "ent": "multiple personality disorder", "cls": "chest"}, {"id": 11149, "dom": "D", "ent": "multiple personality disorder", "cls": "metab"}, {"id": 11150, "dom": "D", "ent": "multiple personality disorder", "cls": "infect"}, {"id": 11151, "dom": "D", "ent": "multiple personality disorder", "cls": "tumor"}, {"id": 11152, "dom": "D", "ent": "multiple personality disorder", "cls": "mental"}, {"id": 11153, "dom": "D", "ent": "multiple personality disorder", "cls": "genetic"}, {"id": 11154, "dom": "D", "ent": "multiple personality disorder", "cls": "bact"}, {"id": 11155, "dom": "D", "ent": "multiple personality disorder", "cls": "viral"}, {"id": 11156, "dom": "D", "ent": "multiple personality disorder", "cls": "fungal"}, {"id": 11157, "dom": "D", "ent": "multiple personality disorder", "cls": "parasit"}, {"id": 11158, "dom": "D", "ent": "impulse control disorder", "cls": "cardio"}, {"id": 11159, "dom": "D", "ent": "impulse control disorder", "cls": "resp"}, {"id": 11160, "dom": "D", "ent": "impulse control disorder", "cls": "gi"}, {"id": 11161, "dom": "D", "ent": "impulse control disorder", "cls": "nerv"}, {"id": 11162, "dom": "D", "ent": "impulse control disorder", "cls": "skin"}, {"id": 11163, "dom": "D", "ent": "impulse control disorder", "cls": "musc"}, {"id": 11164, "dom": "D", "ent": "impulse control disorder", "cls": "urin"}, {"id": 11165, "dom": "D", "ent": "impulse control disorder", "cls": "endo"}, {"id": 11166, "dom": "D", "ent": "impulse control disorder", "cls": "immune"}, {"id": 11167, "dom": "D", "ent": "impulse control disorder", "cls": "blood"}, {"id": 11168, "dom": "D", "ent": "impulse control disorder", "cls": "repro"}, {"id": 11169, "dom": "D", "ent": "impulse control disorder", "cls": "breast"}, {"id": 11170, "dom": "D", "ent": "impulse control disorder", "cls": "chest"}, {"id": 11171, "dom": "D", "ent": "impulse control disorder", "cls": "metab"}, {"id": 11172, "dom": "D", "ent": "impulse control disorder", "cls": "infect"}, {"id": 11173, "dom": "D", "ent": "impulse control disorder", "cls": "tumor"}, {"id": 11174, "dom": "D", "ent": "impulse control disorder", "cls": "mental"}, {"id": 11175, "dom": "D", "ent": "impulse control disorder", "cls": "genetic"}, {"id": 11176, "dom": "D", "ent": "impulse control disorder", "cls": "bact"}, {"id": 11177, "dom": "D", "ent": "impulse control disorder", "cls": "viral"}, {"id": 11178, "dom": "D", "ent": "impulse control disorder", "cls": "fungal"}, {"id": 11179, "dom": "D", "ent": "impulse control disorder", "cls": "parasit"}, {"id": 11180, "dom": "D", "ent": "lens disease", "cls": "cardio"}, {"id": 11181, "dom": "D", "ent": "lens disease", "cls": "resp"}, {"id": 11182, "dom": "D", "ent": "lens disease", "cls": "gi"}, {"id": 11183, "dom": "D", "ent": "lens disease", "cls": "nerv"}, {"id": 11184, "dom": "D", "ent": "lens disease", "cls": "skin"}, {"id": 11185, "dom": "D", "ent": "lens disease", "cls": "musc"}, {"id": 11186, "dom": "D", "ent": "lens disease", "cls": "urin"}, {"id": 11187, "dom": "D", "ent": "lens disease", "cls": "endo"}, {"id": 11188, "dom": "D", "ent": "lens disease", "cls": "immune"}, {"id": 11189, "dom": "D", "ent": "lens disease", "cls": "blood"}, {"id": 11190, "dom": "D", "ent": "lens disease", "cls": "repro"}, {"id": 11191, "dom": "D", "ent": "lens disease", "cls": "breast"}, {"id": 11192, "dom": "D", "ent": "lens disease", "cls": "chest"}, {"id": 11193, "dom": "D", "ent": "lens disease", "cls": "metab"}, {"id": 11194, "dom": "D", "ent": "lens disease", "cls": "infect"}, {"id": 11195, "dom": "D", "ent": "lens disease", "cls": "tumor"}, {"id": 11196, "dom": "D", "ent": "lens disease", "cls": "mental"}, {"id": 11197, "dom": "D", "ent": "lens disease", "cls": "genetic"}, {"id": 11198, "dom": "D", "ent": "lens disease", "cls": "bact"}, {"id": 11199, "dom": "D", "ent": "lens disease", "cls": "viral"}, {"id": 11200, "dom": "D", "ent": "lens disease", "cls": "fungal"}, {"id": 11201, "dom": "D", "ent": "lens disease", "cls": "parasit"}, {"id": 11202, "dom": "D", "ent": "trench fever", "cls": "cardio"}, {"id": 11203, "dom": "D", "ent": "trench fever", "cls": "resp"}, {"id": 11204, "dom": "D", "ent": "trench fever", "cls": "gi"}, {"id": 11205, "dom": "D", "ent": "trench fever", "cls": "nerv"}, {"id": 11206, "dom": "D", "ent": "trench fever", "cls": "skin"}, {"id": 11207, "dom": "D", "ent": "trench fever", "cls": "musc"}, {"id": 11208, "dom": "D", "ent": "trench fever", "cls": "urin"}, {"id": 11209, "dom": "D", "ent": "trench fever", "cls": "endo"}, {"id": 11210, "dom": "D", "ent": "trench fever", "cls": "immune"}, {"id": 11211, "dom": "D", "ent": "trench fever", "cls": "blood"}, {"id": 11212, "dom": "D", "ent": "trench fever", "cls": "repro"}, {"id": 11213, "dom": "D", "ent": "trench fever", "cls": "breast"}, {"id": 11214, "dom": "D", "ent": "trench fever", "cls": "chest"}, {"id": 11215, "dom": "D", "ent": "trench fever", "cls": "metab"}, {"id": 11216, "dom": "D", "ent": "trench fever", "cls": "infect"}, {"id": 11217, "dom": "D", "ent": "trench fever", "cls": "tumor"}, {"id": 11218, "dom": "D", "ent": "trench fever", "cls": "mental"}, {"id": 11219, "dom": "D", "ent": "trench fever", "cls": "genetic"}, {"id": 11220, "dom": "D", "ent": "trench fever", "cls": "bact"}, {"id": 11221, "dom": "D", "ent": "trench fever", "cls": "viral"}, {"id": 11222, "dom": "D", "ent": "trench fever", "cls": "fungal"}, {"id": 11223, "dom": "D", "ent": "trench fever", "cls": "parasit"}, {"id": 11224, "dom": "D", "ent": "neonatal respiratory failure", "cls": "cardio"}, {"id": 11225, "dom": "D", "ent": "neonatal respiratory failure", "cls": "resp"}, {"id": 11226, "dom": "D", "ent": "neonatal respiratory failure", "cls": "gi"}, {"id": 11227, "dom": "D", "ent": "neonatal respiratory failure", "cls": "nerv"}, {"id": 11228, "dom": "D", "ent": "neonatal respiratory failure", "cls": "skin"}, {"id": 11229, "dom": "D", "ent": "neonatal respiratory failure", "cls": "musc"}, {"id": 11230, "dom": "D", "ent": "neonatal respiratory failure", "cls": "urin"}, {"id": 11231, "dom": "D", "ent": "neonatal respiratory failure", "cls": "endo"}, {"id": 11232, "dom": "D", "ent": "neonatal respiratory failure", "cls": "immune"}, {"id": 11233, "dom": "D", "ent": "neonatal respiratory failure", "cls": "blood"}, {"id": 11234, "dom": "D", "ent": "neonatal respiratory failure", "cls": "repro"}, {"id": 11235, "dom": "D", "ent": "neonatal respiratory failure", "cls": "breast"}, {"id": 11236, "dom": "D", "ent": "neonatal respiratory failure", "cls": "chest"}, {"id": 11237, "dom": "D", "ent": "neonatal respiratory failure", "cls": "metab"}, {"id": 11238, "dom": "D", "ent": "neonatal respiratory failure", "cls": "infect"}, {"id": 11239, "dom": "D", "ent": "neonatal respiratory failure", "cls": "tumor"}, {"id": 11240, "dom": "D", "ent": "neonatal respiratory failure", "cls": "mental"}, {"id": 11241, "dom": "D", "ent": "neonatal respiratory failure", "cls": "genetic"}, {"id": 11242, "dom": "D", "ent": "neonatal respiratory failure", "cls": "bact"}, {"id": 11243, "dom": "D", "ent": "neonatal respiratory failure", "cls": "viral"}, {"id": 11244, "dom": "D", "ent": "neonatal respiratory failure", "cls": "fungal"}, {"id": 11245, "dom": "D", "ent": "neonatal respiratory failure", "cls": "parasit"}, {"id": 11246, "dom": "D", "ent": "T cell deficiency", "cls": "cardio"}, {"id": 11247, "dom": "D", "ent": "T cell deficiency", "cls": "resp"}, {"id": 11248, "dom": "D", "ent": "T cell deficiency", "cls": "gi"}, {"id": 11249, "dom": "D", "ent": "T cell deficiency", "cls": "nerv"}, {"id": 11250, "dom": "D", "ent": "T cell deficiency", "cls": "skin"}, {"id": 11251, "dom": "D", "ent": "T cell deficiency", "cls": "musc"}, {"id": 11252, "dom": "D", "ent": "T cell deficiency", "cls": "urin"}, {"id": 11253, "dom": "D", "ent": "T cell deficiency", "cls": "endo"}, {"id": 11254, "dom": "D", "ent": "T cell deficiency", "cls": "immune"}, {"id": 11255, "dom": "D", "ent": "T cell deficiency", "cls": "blood"}, {"id": 11256, "dom": "D", "ent": "T cell deficiency", "cls": "repro"}, {"id": 11257, "dom": "D", "ent": "T cell deficiency", "cls": "breast"}, {"id": 11258, "dom": "D", "ent": "T cell deficiency", "cls": "chest"}, {"id": 11259, "dom": "D", "ent": "T cell deficiency", "cls": "metab"}, {"id": 11260, "dom": "D", "ent": "T cell deficiency", "cls": "infect"}, {"id": 11261, "dom": "D", "ent": "T cell deficiency", "cls": "tumor"}, {"id": 11262, "dom": "D", "ent": "T cell deficiency", "cls": "mental"}, {"id": 11263, "dom": "D", "ent": "T cell deficiency", "cls": "genetic"}, {"id": 11264, "dom": "D", "ent": "T cell deficiency", "cls": "bact"}, {"id": 11265, "dom": "D", "ent": "T cell deficiency", "cls": "viral"}, {"id": 11266, "dom": "D", "ent": "T cell deficiency", "cls": "fungal"}, {"id": 11267, "dom": "D", "ent": "T cell deficiency", "cls": "parasit"}, {"id": 11268, "dom": "D", "ent": "acute orbital inflammation", "cls": "cardio"}, {"id": 11269, "dom": "D", "ent": "acute orbital inflammation", "cls": "resp"}, {"id": 11270, "dom": "D", "ent": "acute orbital inflammation", "cls": "gi"}, {"id": 11271, "dom": "D", "ent": "acute orbital inflammation", "cls": "nerv"}, {"id": 11272, "dom": "D", "ent": "acute orbital inflammation", "cls": "skin"}, {"id": 11273, "dom": "D", "ent": "acute orbital inflammation", "cls": "musc"}, {"id": 11274, "dom": "D", "ent": "acute orbital inflammation", "cls": "urin"}, {"id": 11275, "dom": "D", "ent": "acute orbital inflammation", "cls": "endo"}, {"id": 11276, "dom": "D", "ent": "acute orbital inflammation", "cls": "immune"}, {"id": 11277, "dom": "D", "ent": "acute orbital inflammation", "cls": "blood"}, {"id": 11278, "dom": "D", "ent": "acute orbital inflammation", "cls": "repro"}, {"id": 11279, "dom": "D", "ent": "acute orbital inflammation", "cls": "breast"}, {"id": 11280, "dom": "D", "ent": "acute orbital inflammation", "cls": "chest"}, {"id": 11281, "dom": "D", "ent": "acute orbital inflammation", "cls": "metab"}, {"id": 11282, "dom": "D", "ent": "acute orbital inflammation", "cls": "infect"}, {"id": 11283, "dom": "D", "ent": "acute orbital inflammation", "cls": "tumor"}, {"id": 11284, "dom": "D", "ent": "acute orbital inflammation", "cls": "mental"}, {"id": 11285, "dom": "D", "ent": "acute orbital inflammation", "cls": "genetic"}, {"id": 11286, "dom": "D", "ent": "acute orbital inflammation", "cls": "bact"}, {"id": 11287, "dom": "D", "ent": "acute orbital inflammation", "cls": "viral"}, {"id": 11288, "dom": "D", "ent": "acute orbital inflammation", "cls": "fungal"}, {"id": 11289, "dom": "D", "ent": "acute orbital inflammation", "cls": "parasit"}, {"id": 11290, "dom": "D", "ent": "appendix cancer", "cls": "cardio"}, {"id": 11291, "dom": "D", "ent": "appendix cancer", "cls": "resp"}, {"id": 11292, "dom": "D", "ent": "appendix cancer", "cls": "gi"}, {"id": 11293, "dom": "D", "ent": "appendix cancer", "cls": "nerv"}, {"id": 11294, "dom": "D", "ent": "appendix cancer", "cls": "skin"}, {"id": 11295, "dom": "D", "ent": "appendix cancer", "cls": "musc"}, {"id": 11296, "dom": "D", "ent": "appendix cancer", "cls": "urin"}, {"id": 11297, "dom": "D", "ent": "appendix cancer", "cls": "endo"}, {"id": 11298, "dom": "D", "ent": "appendix cancer", "cls": "immune"}, {"id": 11299, "dom": "D", "ent": "appendix cancer", "cls": "blood"}, {"id": 11300, "dom": "D", "ent": "appendix cancer", "cls": "repro"}, {"id": 11301, "dom": "D", "ent": "appendix cancer", "cls": "breast"}, {"id": 11302, "dom": "D", "ent": "appendix cancer", "cls": "chest"}, {"id": 11303, "dom": "D", "ent": "appendix cancer", "cls": "metab"}, {"id": 11304, "dom": "D", "ent": "appendix cancer", "cls": "infect"}, {"id": 11305, "dom": "D", "ent": "appendix cancer", "cls": "tumor"}, {"id": 11306, "dom": "D", "ent": "appendix cancer", "cls": "mental"}, {"id": 11307, "dom": "D", "ent": "appendix cancer", "cls": "genetic"}, {"id": 11308, "dom": "D", "ent": "appendix cancer", "cls": "bact"}, {"id": 11309, "dom": "D", "ent": "appendix cancer", "cls": "viral"}, {"id": 11310, "dom": "D", "ent": "appendix cancer", "cls": "fungal"}, {"id": 11311, "dom": "D", "ent": "appendix cancer", "cls": "parasit"}, {"id": 11312, "dom": "D", "ent": "appendix lymphoma", "cls": "cardio"}, {"id": 11313, "dom": "D", "ent": "appendix lymphoma", "cls": "resp"}, {"id": 11314, "dom": "D", "ent": "appendix lymphoma", "cls": "gi"}, {"id": 11315, "dom": "D", "ent": "appendix lymphoma", "cls": "nerv"}, {"id": 11316, "dom": "D", "ent": "appendix lymphoma", "cls": "skin"}, {"id": 11317, "dom": "D", "ent": "appendix lymphoma", "cls": "musc"}, {"id": 11318, "dom": "D", "ent": "appendix lymphoma", "cls": "urin"}, {"id": 11319, "dom": "D", "ent": "appendix lymphoma", "cls": "endo"}, {"id": 11320, "dom": "D", "ent": "appendix lymphoma", "cls": "immune"}, {"id": 11321, "dom": "D", "ent": "appendix lymphoma", "cls": "blood"}, {"id": 11322, "dom": "D", "ent": "appendix lymphoma", "cls": "repro"}, {"id": 11323, "dom": "D", "ent": "appendix lymphoma", "cls": "breast"}, {"id": 11324, "dom": "D", "ent": "appendix lymphoma", "cls": "chest"}, {"id": 11325, "dom": "D", "ent": "appendix lymphoma", "cls": "metab"}, {"id": 11326, "dom": "D", "ent": "appendix lymphoma", "cls": "infect"}, {"id": 11327, "dom": "D", "ent": "appendix lymphoma", "cls": "tumor"}, {"id": 11328, "dom": "D", "ent": "appendix lymphoma", "cls": "mental"}, {"id": 11329, "dom": "D", "ent": "appendix lymphoma", "cls": "genetic"}, {"id": 11330, "dom": "D", "ent": "appendix lymphoma", "cls": "bact"}, {"id": 11331, "dom": "D", "ent": "appendix lymphoma", "cls": "viral"}, {"id": 11332, "dom": "D", "ent": "appendix lymphoma", "cls": "fungal"}, {"id": 11333, "dom": "D", "ent": "appendix lymphoma", "cls": "parasit"}, {"id": 11334, "dom": "D", "ent": "rabies", "cls": "cardio"}, {"id": 11335, "dom": "D", "ent": "rabies", "cls": "resp"}, {"id": 11336, "dom": "D", "ent": "rabies", "cls": "gi"}, {"id": 11337, "dom": "D", "ent": "rabies", "cls": "nerv"}, {"id": 11338, "dom": "D", "ent": "rabies", "cls": "skin"}, {"id": 11339, "dom": "D", "ent": "rabies", "cls": "musc"}, {"id": 11340, "dom": "D", "ent": "rabies", "cls": "urin"}, {"id": 11341, "dom": "D", "ent": "rabies", "cls": "endo"}, {"id": 11342, "dom": "D", "ent": "rabies", "cls": "immune"}, {"id": 11343, "dom": "D", "ent": "rabies", "cls": "blood"}, {"id": 11344, "dom": "D", "ent": "rabies", "cls": "repro"}, {"id": 11345, "dom": "D", "ent": "rabies", "cls": "breast"}, {"id": 11346, "dom": "D", "ent": "rabies", "cls": "chest"}, {"id": 11347, "dom": "D", "ent": "rabies", "cls": "metab"}, {"id": 11348, "dom": "D", "ent": "rabies", "cls": "infect"}, {"id": 11349, "dom": "D", "ent": "rabies", "cls": "tumor"}, {"id": 11350, "dom": "D", "ent": "rabies", "cls": "mental"}, {"id": 11351, "dom": "D", "ent": "rabies", "cls": "genetic"}, {"id": 11352, "dom": "D", "ent": "rabies", "cls": "bact"}, {"id": 11353, "dom": "D", "ent": "rabies", "cls": "viral"}, {"id": 11354, "dom": "D", "ent": "rabies", "cls": "fungal"}, {"id": 11355, "dom": "D", "ent": "rabies", "cls": "parasit"}, {"id": 11356, "dom": "D", "ent": "expressive language disorder", "cls": "cardio"}, {"id": 11357, "dom": "D", "ent": "expressive language disorder", "cls": "resp"}, {"id": 11358, "dom": "D", "ent": "expressive language disorder", "cls": "gi"}, {"id": 11359, "dom": "D", "ent": "expressive language disorder", "cls": "nerv"}, {"id": 11360, "dom": "D", "ent": "expressive language disorder", "cls": "skin"}, {"id": 11361, "dom": "D", "ent": "expressive language disorder", "cls": "musc"}, {"id": 11362, "dom": "D", "ent": "expressive language disorder", "cls": "urin"}, {"id": 11363, "dom": "D", "ent": "expressive language disorder", "cls": "endo"}, {"id": 11364, "dom": "D", "ent": "expressive language disorder", "cls": "immune"}, {"id": 11365, "dom": "D", "ent": "expressive language disorder", "cls": "blood"}, {"id": 11366, "dom": "D", "ent": "expressive language disorder", "cls": "repro"}, {"id": 11367, "dom": "D", "ent": "expressive language disorder", "cls": "breast"}, {"id": 11368, "dom": "D", "ent": "expressive language disorder", "cls": "chest"}, {"id": 11369, "dom": "D", "ent": "expressive language disorder", "cls": "metab"}, {"id": 11370, "dom": "D", "ent": "expressive language disorder", "cls": "infect"}, {"id": 11371, "dom": "D", "ent": "expressive language disorder", "cls": "tumor"}, {"id": 11372, "dom": "D", "ent": "expressive language disorder", "cls": "mental"}, {"id": 11373, "dom": "D", "ent": "expressive language disorder", "cls": "genetic"}, {"id": 11374, "dom": "D", "ent": "expressive language disorder", "cls": "bact"}, {"id": 11375, "dom": "D", "ent": "expressive language disorder", "cls": "viral"}, {"id": 11376, "dom": "D", "ent": "expressive language disorder", "cls": "fungal"}, {"id": 11377, "dom": "D", "ent": "expressive language disorder", "cls": "parasit"}, {"id": 11378, "dom": "D", "ent": "acquired night blindness", "cls": "cardio"}, {"id": 11379, "dom": "D", "ent": "acquired night blindness", "cls": "resp"}, {"id": 11380, "dom": "D", "ent": "acquired night blindness", "cls": "gi"}, {"id": 11381, "dom": "D", "ent": "acquired night blindness", "cls": "nerv"}, {"id": 11382, "dom": "D", "ent": "acquired night blindness", "cls": "skin"}, {"id": 11383, "dom": "D", "ent": "acquired night blindness", "cls": "musc"}, {"id": 11384, "dom": "D", "ent": "acquired night blindness", "cls": "urin"}, {"id": 11385, "dom": "D", "ent": "acquired night blindness", "cls": "endo"}, {"id": 11386, "dom": "D", "ent": "acquired night blindness", "cls": "immune"}, {"id": 11387, "dom": "D", "ent": "acquired night blindness", "cls": "blood"}, {"id": 11388, "dom": "D", "ent": "acquired night blindness", "cls": "repro"}, {"id": 11389, "dom": "D", "ent": "acquired night blindness", "cls": "breast"}, {"id": 11390, "dom": "D", "ent": "acquired night blindness", "cls": "chest"}, {"id": 11391, "dom": "D", "ent": "acquired night blindness", "cls": "metab"}, {"id": 11392, "dom": "D", "ent": "acquired night blindness", "cls": "infect"}, {"id": 11393, "dom": "D", "ent": "acquired night blindness", "cls": "tumor"}, {"id": 11394, "dom": "D", "ent": "acquired night blindness", "cls": "mental"}, {"id": 11395, "dom": "D", "ent": "acquired night blindness", "cls": "genetic"}, {"id": 11396, "dom": "D", "ent": "acquired night blindness", "cls": "bact"}, {"id": 11397, "dom": "D", "ent": "acquired night blindness", "cls": "viral"}, {"id": 11398, "dom": "D", "ent": "acquired night blindness", "cls": "fungal"}, {"id": 11399, "dom": "D", "ent": "acquired night blindness", "cls": "parasit"}, {"id": 11400, "dom": "D", "ent": "Chandler syndrome", "cls": "cardio"}, {"id": 11401, "dom": "D", "ent": "Chandler syndrome", "cls": "resp"}, {"id": 11402, "dom": "D", "ent": "Chandler syndrome", "cls": "gi"}, {"id": 11403, "dom": "D", "ent": "Chandler syndrome", "cls": "nerv"}, {"id": 11404, "dom": "D", "ent": "Chandler syndrome", "cls": "skin"}, {"id": 11405, "dom": "D", "ent": "Chandler syndrome", "cls": "musc"}, {"id": 11406, "dom": "D", "ent": "Chandler syndrome", "cls": "urin"}, {"id": 11407, "dom": "D", "ent": "Chandler syndrome", "cls": "endo"}, {"id": 11408, "dom": "D", "ent": "Chandler syndrome", "cls": "immune"}, {"id": 11409, "dom": "D", "ent": "Chandler syndrome", "cls": "blood"}, {"id": 11410, "dom": "D", "ent": "Chandler syndrome", "cls": "repro"}, {"id": 11411, "dom": "D", "ent": "Chandler syndrome", "cls": "breast"}, {"id": 11412, "dom": "D", "ent": "Chandler syndrome", "cls": "chest"}, {"id": 11413, "dom": "D", "ent": "Chandler syndrome", "cls": "metab"}, {"id": 11414, "dom": "D", "ent": "Chandler syndrome", "cls": "infect"}, {"id": 11415, "dom": "D", "ent": "Chandler syndrome", "cls": "tumor"}, {"id": 11416, "dom": "D", "ent": "Chandler syndrome", "cls": "mental"}, {"id": 11417, "dom": "D", "ent": "Chandler syndrome", "cls": "genetic"}, {"id": 11418, "dom": "D", "ent": "Chandler syndrome", "cls": "bact"}, {"id": 11419, "dom": "D", "ent": "Chandler syndrome", "cls": "viral"}, {"id": 11420, "dom": "D", "ent": "Chandler syndrome", "cls": "fungal"}, {"id": 11421, "dom": "D", "ent": "Chandler syndrome", "cls": "parasit"}, {"id": 11422, "dom": "D", "ent": "functional gastric disease", "cls": "cardio"}, {"id": 11423, "dom": "D", "ent": "functional gastric disease", "cls": "resp"}, {"id": 11424, "dom": "D", "ent": "functional gastric disease", "cls": "gi"}, {"id": 11425, "dom": "D", "ent": "functional gastric disease", "cls": "nerv"}, {"id": 11426, "dom": "D", "ent": "functional gastric disease", "cls": "skin"}, {"id": 11427, "dom": "D", "ent": "functional gastric disease", "cls": "musc"}, {"id": 11428, "dom": "D", "ent": "functional gastric disease", "cls": "urin"}, {"id": 11429, "dom": "D", "ent": "functional gastric disease", "cls": "endo"}, {"id": 11430, "dom": "D", "ent": "functional gastric disease", "cls": "immune"}, {"id": 11431, "dom": "D", "ent": "functional gastric disease", "cls": "blood"}, {"id": 11432, "dom": "D", "ent": "functional gastric disease", "cls": "repro"}, {"id": 11433, "dom": "D", "ent": "functional gastric disease", "cls": "breast"}, {"id": 11434, "dom": "D", "ent": "functional gastric disease", "cls": "chest"}, {"id": 11435, "dom": "D", "ent": "functional gastric disease", "cls": "metab"}, {"id": 11436, "dom": "D", "ent": "functional gastric disease", "cls": "infect"}, {"id": 11437, "dom": "D", "ent": "functional gastric disease", "cls": "tumor"}, {"id": 11438, "dom": "D", "ent": "functional gastric disease", "cls": "mental"}, {"id": 11439, "dom": "D", "ent": "functional gastric disease", "cls": "genetic"}, {"id": 11440, "dom": "D", "ent": "functional gastric disease", "cls": "bact"}, {"id": 11441, "dom": "D", "ent": "functional gastric disease", "cls": "viral"}, {"id": 11442, "dom": "D", "ent": "functional gastric disease", "cls": "fungal"}, {"id": 11443, "dom": "D", "ent": "functional gastric disease", "cls": "parasit"}, {"id": 11444, "dom": "D", "ent": "pelvic muscle wasting", "cls": "cardio"}, {"id": 11445, "dom": "D", "ent": "pelvic muscle wasting", "cls": "resp"}, {"id": 11446, "dom": "D", "ent": "pelvic muscle wasting", "cls": "gi"}, {"id": 11447, "dom": "D", "ent": "pelvic muscle wasting", "cls": "nerv"}, {"id": 11448, "dom": "D", "ent": "pelvic muscle wasting", "cls": "skin"}, {"id": 11449, "dom": "D", "ent": "pelvic muscle wasting", "cls": "musc"}, {"id": 11450, "dom": "D", "ent": "pelvic muscle wasting", "cls": "urin"}, {"id": 11451, "dom": "D", "ent": "pelvic muscle wasting", "cls": "endo"}, {"id": 11452, "dom": "D", "ent": "pelvic muscle wasting", "cls": "immune"}, {"id": 11453, "dom": "D", "ent": "pelvic muscle wasting", "cls": "blood"}, {"id": 11454, "dom": "D", "ent": "pelvic muscle wasting", "cls": "repro"}, {"id": 11455, "dom": "D", "ent": "pelvic muscle wasting", "cls": "breast"}, {"id": 11456, "dom": "D", "ent": "pelvic muscle wasting", "cls": "chest"}, {"id": 11457, "dom": "D", "ent": "pelvic muscle wasting", "cls": "metab"}, {"id": 11458, "dom": "D", "ent": "pelvic muscle wasting", "cls": "infect"}, {"id": 11459, "dom": "D", "ent": "pelvic muscle wasting", "cls": "tumor"}, {"id": 11460, "dom": "D", "ent": "pelvic muscle wasting", "cls": "mental"}, {"id": 11461, "dom": "D", "ent": "pelvic muscle wasting", "cls": "genetic"}, {"id": 11462, "dom": "D", "ent": "pelvic muscle wasting", "cls": "bact"}, {"id": 11463, "dom": "D", "ent": "pelvic muscle wasting", "cls": "viral"}, {"id": 11464, "dom": "D", "ent": "pelvic muscle wasting", "cls": "fungal"}, {"id": 11465, "dom": "D", "ent": "pelvic muscle wasting", "cls": "parasit"}, {"id": 11466, "dom": "D", "ent": "allergic disease", "cls": "cardio"}, {"id": 11467, "dom": "D", "ent": "allergic disease", "cls": "resp"}, {"id": 11468, "dom": "D", "ent": "allergic disease", "cls": "gi"}, {"id": 11469, "dom": "D", "ent": "allergic disease", "cls": "nerv"}, {"id": 11470, "dom": "D", "ent": "allergic disease", "cls": "skin"}, {"id": 11471, "dom": "D", "ent": "allergic disease", "cls": "musc"}, {"id": 11472, "dom": "D", "ent": "allergic disease", "cls": "urin"}, {"id": 11473, "dom": "D", "ent": "allergic disease", "cls": "endo"}, {"id": 11474, "dom": "D", "ent": "allergic disease", "cls": "immune"}, {"id": 11475, "dom": "D", "ent": "allergic disease", "cls": "blood"}, {"id": 11476, "dom": "D", "ent": "allergic disease", "cls": "repro"}, {"id": 11477, "dom": "D", "ent": "allergic disease", "cls": "breast"}, {"id": 11478, "dom": "D", "ent": "allergic disease", "cls": "chest"}, {"id": 11479, "dom": "D", "ent": "allergic disease", "cls": "metab"}, {"id": 11480, "dom": "D", "ent": "allergic disease", "cls": "infect"}, {"id": 11481, "dom": "D", "ent": "allergic disease", "cls": "tumor"}, {"id": 11482, "dom": "D", "ent": "allergic disease", "cls": "mental"}, {"id": 11483, "dom": "D", "ent": "allergic disease", "cls": "genetic"}, {"id": 11484, "dom": "D", "ent": "allergic disease", "cls": "bact"}, {"id": 11485, "dom": "D", "ent": "allergic disease", "cls": "viral"}, {"id": 11486, "dom": "D", "ent": "allergic disease", "cls": "fungal"}, {"id": 11487, "dom": "D", "ent": "allergic disease", "cls": "parasit"}, {"id": 11488, "dom": "D", "ent": "cartilage disease", "cls": "cardio"}, {"id": 11489, "dom": "D", "ent": "cartilage disease", "cls": "resp"}, {"id": 11490, "dom": "D", "ent": "cartilage disease", "cls": "gi"}, {"id": 11491, "dom": "D", "ent": "cartilage disease", "cls": "nerv"}, {"id": 11492, "dom": "D", "ent": "cartilage disease", "cls": "skin"}, {"id": 11493, "dom": "D", "ent": "cartilage disease", "cls": "musc"}, {"id": 11494, "dom": "D", "ent": "cartilage disease", "cls": "urin"}, {"id": 11495, "dom": "D", "ent": "cartilage disease", "cls": "endo"}, {"id": 11496, "dom": "D", "ent": "cartilage disease", "cls": "immune"}, {"id": 11497, "dom": "D", "ent": "cartilage disease", "cls": "blood"}, {"id": 11498, "dom": "D", "ent": "cartilage disease", "cls": "repro"}, {"id": 11499, "dom": "D", "ent": "cartilage disease", "cls": "breast"}, {"id": 11500, "dom": "D", "ent": "cartilage disease", "cls": "chest"}, {"id": 11501, "dom": "D", "ent": "cartilage disease", "cls": "metab"}, {"id": 11502, "dom": "D", "ent": "cartilage disease", "cls": "infect"}, {"id": 11503, "dom": "D", "ent": "cartilage disease", "cls": "tumor"}, {"id": 11504, "dom": "D", "ent": "cartilage disease", "cls": "mental"}, {"id": 11505, "dom": "D", "ent": "cartilage disease", "cls": "genetic"}, {"id": 11506, "dom": "D", "ent": "cartilage disease", "cls": "bact"}, {"id": 11507, "dom": "D", "ent": "cartilage disease", "cls": "viral"}, {"id": 11508, "dom": "D", "ent": "cartilage disease", "cls": "fungal"}, {"id": 11509, "dom": "D", "ent": "cartilage disease", "cls": "parasit"}, {"id": 11510, "dom": "D", "ent": "globe disease", "cls": "cardio"}, {"id": 11511, "dom": "D", "ent": "globe disease", "cls": "resp"}, {"id": 11512, "dom": "D", "ent": "globe disease", "cls": "gi"}, {"id": 11513, "dom": "D", "ent": "globe disease", "cls": "nerv"}, {"id": 11514, "dom": "D", "ent": "globe disease", "cls": "skin"}, {"id": 11515, "dom": "D", "ent": "globe disease", "cls": "musc"}, {"id": 11516, "dom": "D", "ent": "globe disease", "cls": "urin"}, {"id": 11517, "dom": "D", "ent": "globe disease", "cls": "endo"}, {"id": 11518, "dom": "D", "ent": "globe disease", "cls": "immune"}, {"id": 11519, "dom": "D", "ent": "globe disease", "cls": "blood"}, {"id": 11520, "dom": "D", "ent": "globe disease", "cls": "repro"}, {"id": 11521, "dom": "D", "ent": "globe disease", "cls": "breast"}, {"id": 11522, "dom": "D", "ent": "globe disease", "cls": "chest"}, {"id": 11523, "dom": "D", "ent": "globe disease", "cls": "metab"}, {"id": 11524, "dom": "D", "ent": "globe disease", "cls": "infect"}, {"id": 11525, "dom": "D", "ent": "globe disease", "cls": "tumor"}, {"id": 11526, "dom": "D", "ent": "globe disease", "cls": "mental"}, {"id": 11527, "dom": "D", "ent": "globe disease", "cls": "genetic"}, {"id": 11528, "dom": "D", "ent": "globe disease", "cls": "bact"}, {"id": 11529, "dom": "D", "ent": "globe disease", "cls": "viral"}, {"id": 11530, "dom": "D", "ent": "globe disease", "cls": "fungal"}, {"id": 11531, "dom": "D", "ent": "globe disease", "cls": "parasit"}, {"id": 11532, "dom": "D", "ent": "hepatitis A", "cls": "cardio"}, {"id": 11533, "dom": "D", "ent": "hepatitis A", "cls": "resp"}, {"id": 11534, "dom": "D", "ent": "hepatitis A", "cls": "gi"}, {"id": 11535, "dom": "D", "ent": "hepatitis A", "cls": "nerv"}, {"id": 11536, "dom": "D", "ent": "hepatitis A", "cls": "skin"}, {"id": 11537, "dom": "D", "ent": "hepatitis A", "cls": "musc"}, {"id": 11538, "dom": "D", "ent": "hepatitis A", "cls": "urin"}, {"id": 11539, "dom": "D", "ent": "hepatitis A", "cls": "endo"}, {"id": 11540, "dom": "D", "ent": "hepatitis A", "cls": "immune"}, {"id": 11541, "dom": "D", "ent": "hepatitis A", "cls": "blood"}, {"id": 11542, "dom": "D", "ent": "hepatitis A", "cls": "repro"}, {"id": 11543, "dom": "D", "ent": "hepatitis A", "cls": "breast"}, {"id": 11544, "dom": "D", "ent": "hepatitis A", "cls": "chest"}, {"id": 11545, "dom": "D", "ent": "hepatitis A", "cls": "metab"}, {"id": 11546, "dom": "D", "ent": "hepatitis A", "cls": "infect"}, {"id": 11547, "dom": "D", "ent": "hepatitis A", "cls": "tumor"}, {"id": 11548, "dom": "D", "ent": "hepatitis A", "cls": "mental"}, {"id": 11549, "dom": "D", "ent": "hepatitis A", "cls": "genetic"}, {"id": 11550, "dom": "D", "ent": "hepatitis A", "cls": "bact"}, {"id": 11551, "dom": "D", "ent": "hepatitis A", "cls": "viral"}, {"id": 11552, "dom": "D", "ent": "hepatitis A", "cls": "fungal"}, {"id": 11553, "dom": "D", "ent": "hepatitis A", "cls": "parasit"}, {"id": 11554, "dom": "D", "ent": "diarrhea", "cls": "cardio"}, {"id": 11555, "dom": "D", "ent": "diarrhea", "cls": "resp"}, {"id": 11556, "dom": "D", "ent": "diarrhea", "cls": "gi"}, {"id": 11557, "dom": "D", "ent": "diarrhea", "cls": "nerv"}, {"id": 11558, "dom": "D", "ent": "diarrhea", "cls": "skin"}, {"id": 11559, "dom": "D", "ent": "diarrhea", "cls": "musc"}, {"id": 11560, "dom": "D", "ent": "diarrhea", "cls": "urin"}, {"id": 11561, "dom": "D", "ent": "diarrhea", "cls": "endo"}, {"id": 11562, "dom": "D", "ent": "diarrhea", "cls": "immune"}, {"id": 11563, "dom": "D", "ent": "diarrhea", "cls": "blood"}, {"id": 11564, "dom": "D", "ent": "diarrhea", "cls": "repro"}, {"id": 11565, "dom": "D", "ent": "diarrhea", "cls": "breast"}, {"id": 11566, "dom": "D", "ent": "diarrhea", "cls": "chest"}, {"id": 11567, "dom": "D", "ent": "diarrhea", "cls": "metab"}, {"id": 11568, "dom": "D", "ent": "diarrhea", "cls": "infect"}, {"id": 11569, "dom": "D", "ent": "diarrhea", "cls": "tumor"}, {"id": 11570, "dom": "D", "ent": "diarrhea", "cls": "mental"}, {"id": 11571, "dom": "D", "ent": "diarrhea", "cls": "genetic"}, {"id": 11572, "dom": "D", "ent": "diarrhea", "cls": "bact"}, {"id": 11573, "dom": "D", "ent": "diarrhea", "cls": "viral"}, {"id": 11574, "dom": "D", "ent": "diarrhea", "cls": "fungal"}, {"id": 11575, "dom": "D", "ent": "diarrhea", "cls": "parasit"}, {"id": 11576, "dom": "D", "ent": "reading disorder", "cls": "cardio"}, {"id": 11577, "dom": "D", "ent": "reading disorder", "cls": "resp"}, {"id": 11578, "dom": "D", "ent": "reading disorder", "cls": "gi"}, {"id": 11579, "dom": "D", "ent": "reading disorder", "cls": "nerv"}, {"id": 11580, "dom": "D", "ent": "reading disorder", "cls": "skin"}, {"id": 11581, "dom": "D", "ent": "reading disorder", "cls": "musc"}, {"id": 11582, "dom": "D", "ent": "reading disorder", "cls": "urin"}, {"id": 11583, "dom": "D", "ent": "reading disorder", "cls": "endo"}, {"id": 11584, "dom": "D", "ent": "reading disorder", "cls": "immune"}, {"id": 11585, "dom": "D", "ent": "reading disorder", "cls": "blood"}, {"id": 11586, "dom": "D", "ent": "reading disorder", "cls": "repro"}, {"id": 11587, "dom": "D", "ent": "reading disorder", "cls": "breast"}, {"id": 11588, "dom": "D", "ent": "reading disorder", "cls": "chest"}, {"id": 11589, "dom": "D", "ent": "reading disorder", "cls": "metab"}, {"id": 11590, "dom": "D", "ent": "reading disorder", "cls": "infect"}, {"id": 11591, "dom": "D", "ent": "reading disorder", "cls": "tumor"}, {"id": 11592, "dom": "D", "ent": "reading disorder", "cls": "mental"}, {"id": 11593, "dom": "D", "ent": "reading disorder", "cls": "genetic"}, {"id": 11594, "dom": "D", "ent": "reading disorder", "cls": "bact"}, {"id": 11595, "dom": "D", "ent": "reading disorder", "cls": "viral"}, {"id": 11596, "dom": "D", "ent": "reading disorder", "cls": "fungal"}, {"id": 11597, "dom": "D", "ent": "reading disorder", "cls": "parasit"}, {"id": 11598, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "cardio"}, {"id": 11599, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "resp"}, {"id": 11600, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "gi"}, {"id": 11601, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "nerv"}, {"id": 11602, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "skin"}, {"id": 11603, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "musc"}, {"id": 11604, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "urin"}, {"id": 11605, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "endo"}, {"id": 11606, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "immune"}, {"id": 11607, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "blood"}, {"id": 11608, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "repro"}, {"id": 11609, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "breast"}, {"id": 11610, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "chest"}, {"id": 11611, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "metab"}, {"id": 11612, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "infect"}, {"id": 11613, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "tumor"}, {"id": 11614, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "mental"}, {"id": 11615, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "genetic"}, {"id": 11616, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "bact"}, {"id": 11617, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "viral"}, {"id": 11618, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "fungal"}, {"id": 11619, "dom": "D", "ent": "Stiff-Person syndrome", "cls": "parasit"}, {"id": 11620, "dom": "D", "ent": "color blindness", "cls": "cardio"}, {"id": 11621, "dom": "D", "ent": "color blindness", "cls": "resp"}, {"id": 11622, "dom": "D", "ent": "color blindness", "cls": "gi"}, {"id": 11623, "dom": "D", "ent": "color blindness", "cls": "nerv"}, {"id": 11624, "dom": "D", "ent": "color blindness", "cls": "skin"}, {"id": 11625, "dom": "D", "ent": "color blindness", "cls": "musc"}, {"id": 11626, "dom": "D", "ent": "color blindness", "cls": "urin"}, {"id": 11627, "dom": "D", "ent": "color blindness", "cls": "endo"}, {"id": 11628, "dom": "D", "ent": "color blindness", "cls": "immune"}, {"id": 11629, "dom": "D", "ent": "color blindness", "cls": "blood"}, {"id": 11630, "dom": "D", "ent": "color blindness", "cls": "repro"}, {"id": 11631, "dom": "D", "ent": "color blindness", "cls": "breast"}, {"id": 11632, "dom": "D", "ent": "color blindness", "cls": "chest"}, {"id": 11633, "dom": "D", "ent": "color blindness", "cls": "metab"}, {"id": 11634, "dom": "D", "ent": "color blindness", "cls": "infect"}, {"id": 11635, "dom": "D", "ent": "color blindness", "cls": "tumor"}, {"id": 11636, "dom": "D", "ent": "color blindness", "cls": "mental"}, {"id": 11637, "dom": "D", "ent": "color blindness", "cls": "genetic"}, {"id": 11638, "dom": "D", "ent": "color blindness", "cls": "bact"}, {"id": 11639, "dom": "D", "ent": "color blindness", "cls": "viral"}, {"id": 11640, "dom": "D", "ent": "color blindness", "cls": "fungal"}, {"id": 11641, "dom": "D", "ent": "color blindness", "cls": "parasit"}, {"id": 11642, "dom": "D", "ent": "secondary Parkinson disease", "cls": "cardio"}, {"id": 11643, "dom": "D", "ent": "secondary Parkinson disease", "cls": "resp"}, {"id": 11644, "dom": "D", "ent": "secondary Parkinson disease", "cls": "gi"}, {"id": 11645, "dom": "D", "ent": "secondary Parkinson disease", "cls": "nerv"}, {"id": 11646, "dom": "D", "ent": "secondary Parkinson disease", "cls": "skin"}, {"id": 11647, "dom": "D", "ent": "secondary Parkinson disease", "cls": "musc"}, {"id": 11648, "dom": "D", "ent": "secondary Parkinson disease", "cls": "urin"}, {"id": 11649, "dom": "D", "ent": "secondary Parkinson disease", "cls": "endo"}, {"id": 11650, "dom": "D", "ent": "secondary Parkinson disease", "cls": "immune"}, {"id": 11651, "dom": "D", "ent": "secondary Parkinson disease", "cls": "blood"}, {"id": 11652, "dom": "D", "ent": "secondary Parkinson disease", "cls": "repro"}, {"id": 11653, "dom": "D", "ent": "secondary Parkinson disease", "cls": "breast"}, {"id": 11654, "dom": "D", "ent": "secondary Parkinson disease", "cls": "chest"}, {"id": 11655, "dom": "D", "ent": "secondary Parkinson disease", "cls": "metab"}, {"id": 11656, "dom": "D", "ent": "secondary Parkinson disease", "cls": "infect"}, {"id": 11657, "dom": "D", "ent": "secondary Parkinson disease", "cls": "tumor"}, {"id": 11658, "dom": "D", "ent": "secondary Parkinson disease", "cls": "mental"}, {"id": 11659, "dom": "D", "ent": "secondary Parkinson disease", "cls": "genetic"}, {"id": 11660, "dom": "D", "ent": "secondary Parkinson disease", "cls": "bact"}, {"id": 11661, "dom": "D", "ent": "secondary Parkinson disease", "cls": "viral"}, {"id": 11662, "dom": "D", "ent": "secondary Parkinson disease", "cls": "fungal"}, {"id": 11663, "dom": "D", "ent": "secondary Parkinson disease", "cls": "parasit"}, {"id": 11664, "dom": "D", "ent": "nutmeg liver", "cls": "cardio"}, {"id": 11665, "dom": "D", "ent": "nutmeg liver", "cls": "resp"}, {"id": 11666, "dom": "D", "ent": "nutmeg liver", "cls": "gi"}, {"id": 11667, "dom": "D", "ent": "nutmeg liver", "cls": "nerv"}, {"id": 11668, "dom": "D", "ent": "nutmeg liver", "cls": "skin"}, {"id": 11669, "dom": "D", "ent": "nutmeg liver", "cls": "musc"}, {"id": 11670, "dom": "D", "ent": "nutmeg liver", "cls": "urin"}, {"id": 11671, "dom": "D", "ent": "nutmeg liver", "cls": "endo"}, {"id": 11672, "dom": "D", "ent": "nutmeg liver", "cls": "immune"}, {"id": 11673, "dom": "D", "ent": "nutmeg liver", "cls": "blood"}, {"id": 11674, "dom": "D", "ent": "nutmeg liver", "cls": "repro"}, {"id": 11675, "dom": "D", "ent": "nutmeg liver", "cls": "breast"}, {"id": 11676, "dom": "D", "ent": "nutmeg liver", "cls": "chest"}, {"id": 11677, "dom": "D", "ent": "nutmeg liver", "cls": "metab"}, {"id": 11678, "dom": "D", "ent": "nutmeg liver", "cls": "infect"}, {"id": 11679, "dom": "D", "ent": "nutmeg liver", "cls": "tumor"}, {"id": 11680, "dom": "D", "ent": "nutmeg liver", "cls": "mental"}, {"id": 11681, "dom": "D", "ent": "nutmeg liver", "cls": "genetic"}, {"id": 11682, "dom": "D", "ent": "nutmeg liver", "cls": "bact"}, {"id": 11683, "dom": "D", "ent": "nutmeg liver", "cls": "viral"}, {"id": 11684, "dom": "D", "ent": "nutmeg liver", "cls": "fungal"}, {"id": 11685, "dom": "D", "ent": "nutmeg liver", "cls": "parasit"}, {"id": 11686, "dom": "D", "ent": "excessive tearing", "cls": "cardio"}, {"id": 11687, "dom": "D", "ent": "excessive tearing", "cls": "resp"}, {"id": 11688, "dom": "D", "ent": "excessive tearing", "cls": "gi"}, {"id": 11689, "dom": "D", "ent": "excessive tearing", "cls": "nerv"}, {"id": 11690, "dom": "D", "ent": "excessive tearing", "cls": "skin"}, {"id": 11691, "dom": "D", "ent": "excessive tearing", "cls": "musc"}, {"id": 11692, "dom": "D", "ent": "excessive tearing", "cls": "urin"}, {"id": 11693, "dom": "D", "ent": "excessive tearing", "cls": "endo"}, {"id": 11694, "dom": "D", "ent": "excessive tearing", "cls": "immune"}, {"id": 11695, "dom": "D", "ent": "excessive tearing", "cls": "blood"}, {"id": 11696, "dom": "D", "ent": "excessive tearing", "cls": "repro"}, {"id": 11697, "dom": "D", "ent": "excessive tearing", "cls": "breast"}, {"id": 11698, "dom": "D", "ent": "excessive tearing", "cls": "chest"}, {"id": 11699, "dom": "D", "ent": "excessive tearing", "cls": "metab"}, {"id": 11700, "dom": "D", "ent": "excessive tearing", "cls": "infect"}, {"id": 11701, "dom": "D", "ent": "excessive tearing", "cls": "tumor"}, {"id": 11702, "dom": "D", "ent": "excessive tearing", "cls": "mental"}, {"id": 11703, "dom": "D", "ent": "excessive tearing", "cls": "genetic"}, {"id": 11704, "dom": "D", "ent": "excessive tearing", "cls": "bact"}, {"id": 11705, "dom": "D", "ent": "excessive tearing", "cls": "viral"}, {"id": 11706, "dom": "D", "ent": "excessive tearing", "cls": "fungal"}, {"id": 11707, "dom": "D", "ent": "excessive tearing", "cls": "parasit"}, {"id": 11708, "dom": "D", "ent": "red-green color blindness", "cls": "cardio"}, {"id": 11709, "dom": "D", "ent": "red-green color blindness", "cls": "resp"}, {"id": 11710, "dom": "D", "ent": "red-green color blindness", "cls": "gi"}, {"id": 11711, "dom": "D", "ent": "red-green color blindness", "cls": "nerv"}, {"id": 11712, "dom": "D", "ent": "red-green color blindness", "cls": "skin"}, {"id": 11713, "dom": "D", "ent": "red-green color blindness", "cls": "musc"}, {"id": 11714, "dom": "D", "ent": "red-green color blindness", "cls": "urin"}, {"id": 11715, "dom": "D", "ent": "red-green color blindness", "cls": "endo"}, {"id": 11716, "dom": "D", "ent": "red-green color blindness", "cls": "immune"}, {"id": 11717, "dom": "D", "ent": "red-green color blindness", "cls": "blood"}, {"id": 11718, "dom": "D", "ent": "red-green color blindness", "cls": "repro"}, {"id": 11719, "dom": "D", "ent": "red-green color blindness", "cls": "breast"}, {"id": 11720, "dom": "D", "ent": "red-green color blindness", "cls": "chest"}, {"id": 11721, "dom": "D", "ent": "red-green color blindness", "cls": "metab"}, {"id": 11722, "dom": "D", "ent": "red-green color blindness", "cls": "infect"}, {"id": 11723, "dom": "D", "ent": "red-green color blindness", "cls": "tumor"}, {"id": 11724, "dom": "D", "ent": "red-green color blindness", "cls": "mental"}, {"id": 11725, "dom": "D", "ent": "red-green color blindness", "cls": "genetic"}, {"id": 11726, "dom": "D", "ent": "red-green color blindness", "cls": "bact"}, {"id": 11727, "dom": "D", "ent": "red-green color blindness", "cls": "viral"}, {"id": 11728, "dom": "D", "ent": "red-green color blindness", "cls": "fungal"}, {"id": 11729, "dom": "D", "ent": "red-green color blindness", "cls": "parasit"}, {"id": 11730, "dom": "D", "ent": "cerebral malaria", "cls": "cardio"}, {"id": 11731, "dom": "D", "ent": "cerebral malaria", "cls": "resp"}, {"id": 11732, "dom": "D", "ent": "cerebral malaria", "cls": "gi"}, {"id": 11733, "dom": "D", "ent": "cerebral malaria", "cls": "nerv"}, {"id": 11734, "dom": "D", "ent": "cerebral malaria", "cls": "skin"}, {"id": 11735, "dom": "D", "ent": "cerebral malaria", "cls": "musc"}, {"id": 11736, "dom": "D", "ent": "cerebral malaria", "cls": "urin"}, {"id": 11737, "dom": "D", "ent": "cerebral malaria", "cls": "endo"}, {"id": 11738, "dom": "D", "ent": "cerebral malaria", "cls": "immune"}, {"id": 11739, "dom": "D", "ent": "cerebral malaria", "cls": "blood"}, {"id": 11740, "dom": "D", "ent": "cerebral malaria", "cls": "repro"}, {"id": 11741, "dom": "D", "ent": "cerebral malaria", "cls": "breast"}, {"id": 11742, "dom": "D", "ent": "cerebral malaria", "cls": "chest"}, {"id": 11743, "dom": "D", "ent": "cerebral malaria", "cls": "metab"}, {"id": 11744, "dom": "D", "ent": "cerebral malaria", "cls": "infect"}, {"id": 11745, "dom": "D", "ent": "cerebral malaria", "cls": "tumor"}, {"id": 11746, "dom": "D", "ent": "cerebral malaria", "cls": "mental"}, {"id": 11747, "dom": "D", "ent": "cerebral malaria", "cls": "genetic"}, {"id": 11748, "dom": "D", "ent": "cerebral malaria", "cls": "bact"}, {"id": 11749, "dom": "D", "ent": "cerebral malaria", "cls": "viral"}, {"id": 11750, "dom": "D", "ent": "cerebral malaria", "cls": "fungal"}, {"id": 11751, "dom": "D", "ent": "cerebral malaria", "cls": "parasit"}, {"id": 11752, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "cardio"}, {"id": 11753, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "resp"}, {"id": 11754, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "gi"}, {"id": 11755, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "nerv"}, {"id": 11756, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "skin"}, {"id": 11757, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "musc"}, {"id": 11758, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "urin"}, {"id": 11759, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "endo"}, {"id": 11760, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "immune"}, {"id": 11761, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "blood"}, {"id": 11762, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "repro"}, {"id": 11763, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "breast"}, {"id": 11764, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "chest"}, {"id": 11765, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "metab"}, {"id": 11766, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "infect"}, {"id": 11767, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "tumor"}, {"id": 11768, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "mental"}, {"id": 11769, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "genetic"}, {"id": 11770, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "bact"}, {"id": 11771, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "viral"}, {"id": 11772, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "fungal"}, {"id": 11773, "dom": "D", "ent": "Masters-Allen syndrome", "cls": "parasit"}, {"id": 11774, "dom": "D", "ent": "frozen shoulder", "cls": "cardio"}, {"id": 11775, "dom": "D", "ent": "frozen shoulder", "cls": "resp"}, {"id": 11776, "dom": "D", "ent": "frozen shoulder", "cls": "gi"}, {"id": 11777, "dom": "D", "ent": "frozen shoulder", "cls": "nerv"}, {"id": 11778, "dom": "D", "ent": "frozen shoulder", "cls": "skin"}, {"id": 11779, "dom": "D", "ent": "frozen shoulder", "cls": "musc"}, {"id": 11780, "dom": "D", "ent": "frozen shoulder", "cls": "urin"}, {"id": 11781, "dom": "D", "ent": "frozen shoulder", "cls": "endo"}, {"id": 11782, "dom": "D", "ent": "frozen shoulder", "cls": "immune"}, {"id": 11783, "dom": "D", "ent": "frozen shoulder", "cls": "blood"}, {"id": 11784, "dom": "D", "ent": "frozen shoulder", "cls": "repro"}, {"id": 11785, "dom": "D", "ent": "frozen shoulder", "cls": "breast"}, {"id": 11786, "dom": "D", "ent": "frozen shoulder", "cls": "chest"}, {"id": 11787, "dom": "D", "ent": "frozen shoulder", "cls": "metab"}, {"id": 11788, "dom": "D", "ent": "frozen shoulder", "cls": "infect"}, {"id": 11789, "dom": "D", "ent": "frozen shoulder", "cls": "tumor"}, {"id": 11790, "dom": "D", "ent": "frozen shoulder", "cls": "mental"}, {"id": 11791, "dom": "D", "ent": "frozen shoulder", "cls": "genetic"}, {"id": 11792, "dom": "D", "ent": "frozen shoulder", "cls": "bact"}, {"id": 11793, "dom": "D", "ent": "frozen shoulder", "cls": "viral"}, {"id": 11794, "dom": "D", "ent": "frozen shoulder", "cls": "fungal"}, {"id": 11795, "dom": "D", "ent": "frozen shoulder", "cls": "parasit"}, {"id": 11796, "dom": "D", "ent": "Machado-Joseph disease", "cls": "cardio"}, {"id": 11797, "dom": "D", "ent": "Machado-Joseph disease", "cls": "resp"}, {"id": 11798, "dom": "D", "ent": "Machado-Joseph disease", "cls": "gi"}, {"id": 11799, "dom": "D", "ent": "Machado-Joseph disease", "cls": "nerv"}, {"id": 11800, "dom": "D", "ent": "Machado-Joseph disease", "cls": "skin"}, {"id": 11801, "dom": "D", "ent": "Machado-Joseph disease", "cls": "musc"}, {"id": 11802, "dom": "D", "ent": "Machado-Joseph disease", "cls": "urin"}, {"id": 11803, "dom": "D", "ent": "Machado-Joseph disease", "cls": "endo"}, {"id": 11804, "dom": "D", "ent": "Machado-Joseph disease", "cls": "immune"}, {"id": 11805, "dom": "D", "ent": "Machado-Joseph disease", "cls": "blood"}, {"id": 11806, "dom": "D", "ent": "Machado-Joseph disease", "cls": "repro"}, {"id": 11807, "dom": "D", "ent": "Machado-Joseph disease", "cls": "breast"}, {"id": 11808, "dom": "D", "ent": "Machado-Joseph disease", "cls": "chest"}, {"id": 11809, "dom": "D", "ent": "Machado-Joseph disease", "cls": "metab"}, {"id": 11810, "dom": "D", "ent": "Machado-Joseph disease", "cls": "infect"}, {"id": 11811, "dom": "D", "ent": "Machado-Joseph disease", "cls": "tumor"}, {"id": 11812, "dom": "D", "ent": "Machado-Joseph disease", "cls": "mental"}, {"id": 11813, "dom": "D", "ent": "Machado-Joseph disease", "cls": "genetic"}, {"id": 11814, "dom": "D", "ent": "Machado-Joseph disease", "cls": "bact"}, {"id": 11815, "dom": "D", "ent": "Machado-Joseph disease", "cls": "viral"}, {"id": 11816, "dom": "D", "ent": "Machado-Joseph disease", "cls": "fungal"}, {"id": 11817, "dom": "D", "ent": "Machado-Joseph disease", "cls": "parasit"}, {"id": 11818, "dom": "D", "ent": "cerebral degeneration", "cls": "cardio"}, {"id": 11819, "dom": "D", "ent": "cerebral degeneration", "cls": "resp"}, {"id": 11820, "dom": "D", "ent": "cerebral degeneration", "cls": "gi"}, {"id": 11821, "dom": "D", "ent": "cerebral degeneration", "cls": "nerv"}, {"id": 11822, "dom": "D", "ent": "cerebral degeneration", "cls": "skin"}, {"id": 11823, "dom": "D", "ent": "cerebral degeneration", "cls": "musc"}, {"id": 11824, "dom": "D", "ent": "cerebral degeneration", "cls": "urin"}, {"id": 11825, "dom": "D", "ent": "cerebral degeneration", "cls": "endo"}, {"id": 11826, "dom": "D", "ent": "cerebral degeneration", "cls": "immune"}, {"id": 11827, "dom": "D", "ent": "cerebral degeneration", "cls": "blood"}, {"id": 11828, "dom": "D", "ent": "cerebral degeneration", "cls": "repro"}, {"id": 11829, "dom": "D", "ent": "cerebral degeneration", "cls": "breast"}, {"id": 11830, "dom": "D", "ent": "cerebral degeneration", "cls": "chest"}, {"id": 11831, "dom": "D", "ent": "cerebral degeneration", "cls": "metab"}, {"id": 11832, "dom": "D", "ent": "cerebral degeneration", "cls": "infect"}, {"id": 11833, "dom": "D", "ent": "cerebral degeneration", "cls": "tumor"}, {"id": 11834, "dom": "D", "ent": "cerebral degeneration", "cls": "mental"}, {"id": 11835, "dom": "D", "ent": "cerebral degeneration", "cls": "genetic"}, {"id": 11836, "dom": "D", "ent": "cerebral degeneration", "cls": "bact"}, {"id": 11837, "dom": "D", "ent": "cerebral degeneration", "cls": "viral"}, {"id": 11838, "dom": "D", "ent": "cerebral degeneration", "cls": "fungal"}, {"id": 11839, "dom": "D", "ent": "cerebral degeneration", "cls": "parasit"}, {"id": 11840, "dom": "D", "ent": "geographic tongue", "cls": "cardio"}, {"id": 11841, "dom": "D", "ent": "geographic tongue", "cls": "resp"}, {"id": 11842, "dom": "D", "ent": "geographic tongue", "cls": "gi"}, {"id": 11843, "dom": "D", "ent": "geographic tongue", "cls": "nerv"}, {"id": 11844, "dom": "D", "ent": "geographic tongue", "cls": "skin"}, {"id": 11845, "dom": "D", "ent": "geographic tongue", "cls": "musc"}, {"id": 11846, "dom": "D", "ent": "geographic tongue", "cls": "urin"}, {"id": 11847, "dom": "D", "ent": "geographic tongue", "cls": "endo"}, {"id": 11848, "dom": "D", "ent": "geographic tongue", "cls": "immune"}, {"id": 11849, "dom": "D", "ent": "geographic tongue", "cls": "blood"}, {"id": 11850, "dom": "D", "ent": "geographic tongue", "cls": "repro"}, {"id": 11851, "dom": "D", "ent": "geographic tongue", "cls": "breast"}, {"id": 11852, "dom": "D", "ent": "geographic tongue", "cls": "chest"}, {"id": 11853, "dom": "D", "ent": "geographic tongue", "cls": "metab"}, {"id": 11854, "dom": "D", "ent": "geographic tongue", "cls": "infect"}, {"id": 11855, "dom": "D", "ent": "geographic tongue", "cls": "tumor"}, {"id": 11856, "dom": "D", "ent": "geographic tongue", "cls": "mental"}, {"id": 11857, "dom": "D", "ent": "geographic tongue", "cls": "genetic"}, {"id": 11858, "dom": "D", "ent": "geographic tongue", "cls": "bact"}, {"id": 11859, "dom": "D", "ent": "geographic tongue", "cls": "viral"}, {"id": 11860, "dom": "D", "ent": "geographic tongue", "cls": "fungal"}, {"id": 11861, "dom": "D", "ent": "geographic tongue", "cls": "parasit"}, {"id": 11862, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "cardio"}, {"id": 11863, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "resp"}, {"id": 11864, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "gi"}, {"id": 11865, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "nerv"}, {"id": 11866, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "skin"}, {"id": 11867, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "musc"}, {"id": 11868, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "urin"}, {"id": 11869, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "endo"}, {"id": 11870, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "immune"}, {"id": 11871, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "blood"}, {"id": 11872, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "repro"}, {"id": 11873, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "breast"}, {"id": 11874, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "chest"}, {"id": 11875, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "metab"}, {"id": 11876, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "infect"}, {"id": 11877, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "tumor"}, {"id": 11878, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "mental"}, {"id": 11879, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "genetic"}, {"id": 11880, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "bact"}, {"id": 11881, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "viral"}, {"id": 11882, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "fungal"}, {"id": 11883, "dom": "D", "ent": "primary pulmonary hypertension", "cls": "parasit"}, {"id": 11884, "dom": "D", "ent": "FG syndrome", "cls": "cardio"}, {"id": 11885, "dom": "D", "ent": "FG syndrome", "cls": "resp"}, {"id": 11886, "dom": "D", "ent": "FG syndrome", "cls": "gi"}, {"id": 11887, "dom": "D", "ent": "FG syndrome", "cls": "nerv"}, {"id": 11888, "dom": "D", "ent": "FG syndrome", "cls": "skin"}, {"id": 11889, "dom": "D", "ent": "FG syndrome", "cls": "musc"}, {"id": 11890, "dom": "D", "ent": "FG syndrome", "cls": "urin"}, {"id": 11891, "dom": "D", "ent": "FG syndrome", "cls": "endo"}, {"id": 11892, "dom": "D", "ent": "FG syndrome", "cls": "immune"}, {"id": 11893, "dom": "D", "ent": "FG syndrome", "cls": "blood"}, {"id": 11894, "dom": "D", "ent": "FG syndrome", "cls": "repro"}, {"id": 11895, "dom": "D", "ent": "FG syndrome", "cls": "breast"}, {"id": 11896, "dom": "D", "ent": "FG syndrome", "cls": "chest"}, {"id": 11897, "dom": "D", "ent": "FG syndrome", "cls": "metab"}, {"id": 11898, "dom": "D", "ent": "FG syndrome", "cls": "infect"}, {"id": 11899, "dom": "D", "ent": "FG syndrome", "cls": "tumor"}, {"id": 11900, "dom": "D", "ent": "FG syndrome", "cls": "mental"}, {"id": 11901, "dom": "D", "ent": "FG syndrome", "cls": "genetic"}, {"id": 11902, "dom": "D", "ent": "FG syndrome", "cls": "bact"}, {"id": 11903, "dom": "D", "ent": "FG syndrome", "cls": "viral"}, {"id": 11904, "dom": "D", "ent": "FG syndrome", "cls": "fungal"}, {"id": 11905, "dom": "D", "ent": "FG syndrome", "cls": "parasit"}, {"id": 11906, "dom": "D", "ent": "Larsen syndrome", "cls": "cardio"}, {"id": 11907, "dom": "D", "ent": "Larsen syndrome", "cls": "resp"}, {"id": 11908, "dom": "D", "ent": "Larsen syndrome", "cls": "gi"}, {"id": 11909, "dom": "D", "ent": "Larsen syndrome", "cls": "nerv"}, {"id": 11910, "dom": "D", "ent": "Larsen syndrome", "cls": "skin"}, {"id": 11911, "dom": "D", "ent": "Larsen syndrome", "cls": "musc"}, {"id": 11912, "dom": "D", "ent": "Larsen syndrome", "cls": "urin"}, {"id": 11913, "dom": "D", "ent": "Larsen syndrome", "cls": "endo"}, {"id": 11914, "dom": "D", "ent": "Larsen syndrome", "cls": "immune"}, {"id": 11915, "dom": "D", "ent": "Larsen syndrome", "cls": "blood"}, {"id": 11916, "dom": "D", "ent": "Larsen syndrome", "cls": "repro"}, {"id": 11917, "dom": "D", "ent": "Larsen syndrome", "cls": "breast"}, {"id": 11918, "dom": "D", "ent": "Larsen syndrome", "cls": "chest"}, {"id": 11919, "dom": "D", "ent": "Larsen syndrome", "cls": "metab"}, {"id": 11920, "dom": "D", "ent": "Larsen syndrome", "cls": "infect"}, {"id": 11921, "dom": "D", "ent": "Larsen syndrome", "cls": "tumor"}, {"id": 11922, "dom": "D", "ent": "Larsen syndrome", "cls": "mental"}, {"id": 11923, "dom": "D", "ent": "Larsen syndrome", "cls": "genetic"}, {"id": 11924, "dom": "D", "ent": "Larsen syndrome", "cls": "bact"}, {"id": 11925, "dom": "D", "ent": "Larsen syndrome", "cls": "viral"}, {"id": 11926, "dom": "D", "ent": "Larsen syndrome", "cls": "fungal"}, {"id": 11927, "dom": "D", "ent": "Larsen syndrome", "cls": "parasit"}, {"id": 11928, "dom": "D", "ent": "colon carcinoma", "cls": "cardio"}, {"id": 11929, "dom": "D", "ent": "colon carcinoma", "cls": "resp"}, {"id": 11930, "dom": "D", "ent": "colon carcinoma", "cls": "gi"}, {"id": 11931, "dom": "D", "ent": "colon carcinoma", "cls": "nerv"}, {"id": 11932, "dom": "D", "ent": "colon carcinoma", "cls": "skin"}, {"id": 11933, "dom": "D", "ent": "colon carcinoma", "cls": "musc"}, {"id": 11934, "dom": "D", "ent": "colon carcinoma", "cls": "urin"}, {"id": 11935, "dom": "D", "ent": "colon carcinoma", "cls": "endo"}, {"id": 11936, "dom": "D", "ent": "colon carcinoma", "cls": "immune"}, {"id": 11937, "dom": "D", "ent": "colon carcinoma", "cls": "blood"}, {"id": 11938, "dom": "D", "ent": "colon carcinoma", "cls": "repro"}, {"id": 11939, "dom": "D", "ent": "colon carcinoma", "cls": "breast"}, {"id": 11940, "dom": "D", "ent": "colon carcinoma", "cls": "chest"}, {"id": 11941, "dom": "D", "ent": "colon carcinoma", "cls": "metab"}, {"id": 11942, "dom": "D", "ent": "colon carcinoma", "cls": "infect"}, {"id": 11943, "dom": "D", "ent": "colon carcinoma", "cls": "tumor"}, {"id": 11944, "dom": "D", "ent": "colon carcinoma", "cls": "mental"}, {"id": 11945, "dom": "D", "ent": "colon carcinoma", "cls": "genetic"}, {"id": 11946, "dom": "D", "ent": "colon carcinoma", "cls": "bact"}, {"id": 11947, "dom": "D", "ent": "colon carcinoma", "cls": "viral"}, {"id": 11948, "dom": "D", "ent": "colon carcinoma", "cls": "fungal"}, {"id": 11949, "dom": "D", "ent": "colon carcinoma", "cls": "parasit"}, {"id": 11950, "dom": "D", "ent": "colon lymphoma", "cls": "cardio"}, {"id": 11951, "dom": "D", "ent": "colon lymphoma", "cls": "resp"}, {"id": 11952, "dom": "D", "ent": "colon lymphoma", "cls": "gi"}, {"id": 11953, "dom": "D", "ent": "colon lymphoma", "cls": "nerv"}, {"id": 11954, "dom": "D", "ent": "colon lymphoma", "cls": "skin"}, {"id": 11955, "dom": "D", "ent": "colon lymphoma", "cls": "musc"}, {"id": 11956, "dom": "D", "ent": "colon lymphoma", "cls": "urin"}, {"id": 11957, "dom": "D", "ent": "colon lymphoma", "cls": "endo"}, {"id": 11958, "dom": "D", "ent": "colon lymphoma", "cls": "immune"}, {"id": 11959, "dom": "D", "ent": "colon lymphoma", "cls": "blood"}, {"id": 11960, "dom": "D", "ent": "colon lymphoma", "cls": "repro"}, {"id": 11961, "dom": "D", "ent": "colon lymphoma", "cls": "breast"}, {"id": 11962, "dom": "D", "ent": "colon lymphoma", "cls": "chest"}, {"id": 11963, "dom": "D", "ent": "colon lymphoma", "cls": "metab"}, {"id": 11964, "dom": "D", "ent": "colon lymphoma", "cls": "infect"}, {"id": 11965, "dom": "D", "ent": "colon lymphoma", "cls": "tumor"}, {"id": 11966, "dom": "D", "ent": "colon lymphoma", "cls": "mental"}, {"id": 11967, "dom": "D", "ent": "colon lymphoma", "cls": "genetic"}, {"id": 11968, "dom": "D", "ent": "colon lymphoma", "cls": "bact"}, {"id": 11969, "dom": "D", "ent": "colon lymphoma", "cls": "viral"}, {"id": 11970, "dom": "D", "ent": "colon lymphoma", "cls": "fungal"}, {"id": 11971, "dom": "D", "ent": "colon lymphoma", "cls": "parasit"}, {"id": 11972, "dom": "D", "ent": "alcohol use disorder", "cls": "cardio"}, {"id": 11973, "dom": "D", "ent": "alcohol use disorder", "cls": "resp"}, {"id": 11974, "dom": "D", "ent": "alcohol use disorder", "cls": "gi"}, {"id": 11975, "dom": "D", "ent": "alcohol use disorder", "cls": "nerv"}, {"id": 11976, "dom": "D", "ent": "alcohol use disorder", "cls": "skin"}, {"id": 11977, "dom": "D", "ent": "alcohol use disorder", "cls": "musc"}, {"id": 11978, "dom": "D", "ent": "alcohol use disorder", "cls": "urin"}, {"id": 11979, "dom": "D", "ent": "alcohol use disorder", "cls": "endo"}, {"id": 11980, "dom": "D", "ent": "alcohol use disorder", "cls": "immune"}, {"id": 11981, "dom": "D", "ent": "alcohol use disorder", "cls": "blood"}, {"id": 11982, "dom": "D", "ent": "alcohol use disorder", "cls": "repro"}, {"id": 11983, "dom": "D", "ent": "alcohol use disorder", "cls": "breast"}, {"id": 11984, "dom": "D", "ent": "alcohol use disorder", "cls": "chest"}, {"id": 11985, "dom": "D", "ent": "alcohol use disorder", "cls": "metab"}, {"id": 11986, "dom": "D", "ent": "alcohol use disorder", "cls": "infect"}, {"id": 11987, "dom": "D", "ent": "alcohol use disorder", "cls": "tumor"}, {"id": 11988, "dom": "D", "ent": "alcohol use disorder", "cls": "mental"}, {"id": 11989, "dom": "D", "ent": "alcohol use disorder", "cls": "genetic"}, {"id": 11990, "dom": "D", "ent": "alcohol use disorder", "cls": "bact"}, {"id": 11991, "dom": "D", "ent": "alcohol use disorder", "cls": "viral"}, {"id": 11992, "dom": "D", "ent": "alcohol use disorder", "cls": "fungal"}, {"id": 11993, "dom": "D", "ent": "alcohol use disorder", "cls": "parasit"}, {"id": 11994, "dom": "D", "ent": "acute chest syndrome", "cls": "cardio"}, {"id": 11995, "dom": "D", "ent": "acute chest syndrome", "cls": "resp"}, {"id": 11996, "dom": "D", "ent": "acute chest syndrome", "cls": "gi"}, {"id": 11997, "dom": "D", "ent": "acute chest syndrome", "cls": "nerv"}, {"id": 11998, "dom": "D", "ent": "acute chest syndrome", "cls": "skin"}, {"id": 11999, "dom": "D", "ent": "acute chest syndrome", "cls": "musc"}, {"id": 12000, "dom": "D", "ent": "acute chest syndrome", "cls": "urin"}, {"id": 12001, "dom": "D", "ent": "acute chest syndrome", "cls": "endo"}, {"id": 12002, "dom": "D", "ent": "acute chest syndrome", "cls": "immune"}, {"id": 12003, "dom": "D", "ent": "acute chest syndrome", "cls": "blood"}, {"id": 12004, "dom": "D", "ent": "acute chest syndrome", "cls": "repro"}, {"id": 12005, "dom": "D", "ent": "acute chest syndrome", "cls": "breast"}, {"id": 12006, "dom": "D", "ent": "acute chest syndrome", "cls": "chest"}, {"id": 12007, "dom": "D", "ent": "acute chest syndrome", "cls": "metab"}, {"id": 12008, "dom": "D", "ent": "acute chest syndrome", "cls": "infect"}, {"id": 12009, "dom": "D", "ent": "acute chest syndrome", "cls": "tumor"}, {"id": 12010, "dom": "D", "ent": "acute chest syndrome", "cls": "mental"}, {"id": 12011, "dom": "D", "ent": "acute chest syndrome", "cls": "genetic"}, {"id": 12012, "dom": "D", "ent": "acute chest syndrome", "cls": "bact"}, {"id": 12013, "dom": "D", "ent": "acute chest syndrome", "cls": "viral"}, {"id": 12014, "dom": "D", "ent": "acute chest syndrome", "cls": "fungal"}, {"id": 12015, "dom": "D", "ent": "acute chest syndrome", "cls": "parasit"}, {"id": 12016, "dom": "D", "ent": "childhood absence epilepsy", "cls": "cardio"}, {"id": 12017, "dom": "D", "ent": "childhood absence epilepsy", "cls": "resp"}, {"id": 12018, "dom": "D", "ent": "childhood absence epilepsy", "cls": "gi"}, {"id": 12019, "dom": "D", "ent": "childhood absence epilepsy", "cls": "nerv"}, {"id": 12020, "dom": "D", "ent": "childhood absence epilepsy", "cls": "skin"}, {"id": 12021, "dom": "D", "ent": "childhood absence epilepsy", "cls": "musc"}, {"id": 12022, "dom": "D", "ent": "childhood absence epilepsy", "cls": "urin"}, {"id": 12023, "dom": "D", "ent": "childhood absence epilepsy", "cls": "endo"}, {"id": 12024, "dom": "D", "ent": "childhood absence epilepsy", "cls": "immune"}, {"id": 12025, "dom": "D", "ent": "childhood absence epilepsy", "cls": "blood"}, {"id": 12026, "dom": "D", "ent": "childhood absence epilepsy", "cls": "repro"}, {"id": 12027, "dom": "D", "ent": "childhood absence epilepsy", "cls": "breast"}, {"id": 12028, "dom": "D", "ent": "childhood absence epilepsy", "cls": "chest"}, {"id": 12029, "dom": "D", "ent": "childhood absence epilepsy", "cls": "metab"}, {"id": 12030, "dom": "D", "ent": "childhood absence epilepsy", "cls": "infect"}, {"id": 12031, "dom": "D", "ent": "childhood absence epilepsy", "cls": "tumor"}, {"id": 12032, "dom": "D", "ent": "childhood absence epilepsy", "cls": "mental"}, {"id": 12033, "dom": "D", "ent": "childhood absence epilepsy", "cls": "genetic"}, {"id": 12034, "dom": "D", "ent": "childhood absence epilepsy", "cls": "bact"}, {"id": 12035, "dom": "D", "ent": "childhood absence epilepsy", "cls": "viral"}, {"id": 12036, "dom": "D", "ent": "childhood absence epilepsy", "cls": "fungal"}, {"id": 12037, "dom": "D", "ent": "childhood absence epilepsy", "cls": "parasit"}, {"id": 12038, "dom": "D", "ent": "epilepsy", "cls": "cardio"}, {"id": 12039, "dom": "D", "ent": "epilepsy", "cls": "resp"}, {"id": 12040, "dom": "D", "ent": "epilepsy", "cls": "gi"}, {"id": 12041, "dom": "D", "ent": "epilepsy", "cls": "nerv"}, {"id": 12042, "dom": "D", "ent": "epilepsy", "cls": "skin"}, {"id": 12043, "dom": "D", "ent": "epilepsy", "cls": "musc"}, {"id": 12044, "dom": "D", "ent": "epilepsy", "cls": "urin"}, {"id": 12045, "dom": "D", "ent": "epilepsy", "cls": "endo"}, {"id": 12046, "dom": "D", "ent": "epilepsy", "cls": "immune"}, {"id": 12047, "dom": "D", "ent": "epilepsy", "cls": "blood"}, {"id": 12048, "dom": "D", "ent": "epilepsy", "cls": "repro"}, {"id": 12049, "dom": "D", "ent": "epilepsy", "cls": "breast"}, {"id": 12050, "dom": "D", "ent": "epilepsy", "cls": "chest"}, {"id": 12051, "dom": "D", "ent": "epilepsy", "cls": "metab"}, {"id": 12052, "dom": "D", "ent": "epilepsy", "cls": "infect"}, {"id": 12053, "dom": "D", "ent": "epilepsy", "cls": "tumor"}, {"id": 12054, "dom": "D", "ent": "epilepsy", "cls": "mental"}, {"id": 12055, "dom": "D", "ent": "epilepsy", "cls": "genetic"}, {"id": 12056, "dom": "D", "ent": "epilepsy", "cls": "bact"}, {"id": 12057, "dom": "D", "ent": "epilepsy", "cls": "viral"}, {"id": 12058, "dom": "D", "ent": "epilepsy", "cls": "fungal"}, {"id": 12059, "dom": "D", "ent": "epilepsy", "cls": "parasit"}, {"id": 12060, "dom": "D", "ent": "cannabis dependence", "cls": "cardio"}, {"id": 12061, "dom": "D", "ent": "cannabis dependence", "cls": "resp"}, {"id": 12062, "dom": "D", "ent": "cannabis dependence", "cls": "gi"}, {"id": 12063, "dom": "D", "ent": "cannabis dependence", "cls": "nerv"}, {"id": 12064, "dom": "D", "ent": "cannabis dependence", "cls": "skin"}, {"id": 12065, "dom": "D", "ent": "cannabis dependence", "cls": "musc"}, {"id": 12066, "dom": "D", "ent": "cannabis dependence", "cls": "urin"}, {"id": 12067, "dom": "D", "ent": "cannabis dependence", "cls": "endo"}, {"id": 12068, "dom": "D", "ent": "cannabis dependence", "cls": "immune"}, {"id": 12069, "dom": "D", "ent": "cannabis dependence", "cls": "blood"}, {"id": 12070, "dom": "D", "ent": "cannabis dependence", "cls": "repro"}, {"id": 12071, "dom": "D", "ent": "cannabis dependence", "cls": "breast"}, {"id": 12072, "dom": "D", "ent": "cannabis dependence", "cls": "chest"}, {"id": 12073, "dom": "D", "ent": "cannabis dependence", "cls": "metab"}, {"id": 12074, "dom": "D", "ent": "cannabis dependence", "cls": "infect"}, {"id": 12075, "dom": "D", "ent": "cannabis dependence", "cls": "tumor"}, {"id": 12076, "dom": "D", "ent": "cannabis dependence", "cls": "mental"}, {"id": 12077, "dom": "D", "ent": "cannabis dependence", "cls": "genetic"}, {"id": 12078, "dom": "D", "ent": "cannabis dependence", "cls": "bact"}, {"id": 12079, "dom": "D", "ent": "cannabis dependence", "cls": "viral"}, {"id": 12080, "dom": "D", "ent": "cannabis dependence", "cls": "fungal"}, {"id": 12081, "dom": "D", "ent": "cannabis dependence", "cls": "parasit"}, {"id": 12082, "dom": "D", "ent": "skull cancer", "cls": "cardio"}, {"id": 12083, "dom": "D", "ent": "skull cancer", "cls": "resp"}, {"id": 12084, "dom": "D", "ent": "skull cancer", "cls": "gi"}, {"id": 12085, "dom": "D", "ent": "skull cancer", "cls": "nerv"}, {"id": 12086, "dom": "D", "ent": "skull cancer", "cls": "skin"}, {"id": 12087, "dom": "D", "ent": "skull cancer", "cls": "musc"}, {"id": 12088, "dom": "D", "ent": "skull cancer", "cls": "urin"}, {"id": 12089, "dom": "D", "ent": "skull cancer", "cls": "endo"}, {"id": 12090, "dom": "D", "ent": "skull cancer", "cls": "immune"}, {"id": 12091, "dom": "D", "ent": "skull cancer", "cls": "blood"}, {"id": 12092, "dom": "D", "ent": "skull cancer", "cls": "repro"}, {"id": 12093, "dom": "D", "ent": "skull cancer", "cls": "breast"}, {"id": 12094, "dom": "D", "ent": "skull cancer", "cls": "chest"}, {"id": 12095, "dom": "D", "ent": "skull cancer", "cls": "metab"}, {"id": 12096, "dom": "D", "ent": "skull cancer", "cls": "infect"}, {"id": 12097, "dom": "D", "ent": "skull cancer", "cls": "tumor"}, {"id": 12098, "dom": "D", "ent": "skull cancer", "cls": "mental"}, {"id": 12099, "dom": "D", "ent": "skull cancer", "cls": "genetic"}, {"id": 12100, "dom": "D", "ent": "skull cancer", "cls": "bact"}, {"id": 12101, "dom": "D", "ent": "skull cancer", "cls": "viral"}, {"id": 12102, "dom": "D", "ent": "skull cancer", "cls": "fungal"}, {"id": 12103, "dom": "D", "ent": "skull cancer", "cls": "parasit"}, {"id": 12104, "dom": "D", "ent": "optic nerve disease", "cls": "cardio"}, {"id": 12105, "dom": "D", "ent": "optic nerve disease", "cls": "resp"}, {"id": 12106, "dom": "D", "ent": "optic nerve disease", "cls": "gi"}, {"id": 12107, "dom": "D", "ent": "optic nerve disease", "cls": "nerv"}, {"id": 12108, "dom": "D", "ent": "optic nerve disease", "cls": "skin"}, {"id": 12109, "dom": "D", "ent": "optic nerve disease", "cls": "musc"}, {"id": 12110, "dom": "D", "ent": "optic nerve disease", "cls": "urin"}, {"id": 12111, "dom": "D", "ent": "optic nerve disease", "cls": "endo"}, {"id": 12112, "dom": "D", "ent": "optic nerve disease", "cls": "immune"}, {"id": 12113, "dom": "D", "ent": "optic nerve disease", "cls": "blood"}, {"id": 12114, "dom": "D", "ent": "optic nerve disease", "cls": "repro"}, {"id": 12115, "dom": "D", "ent": "optic nerve disease", "cls": "breast"}, {"id": 12116, "dom": "D", "ent": "optic nerve disease", "cls": "chest"}, {"id": 12117, "dom": "D", "ent": "optic nerve disease", "cls": "metab"}, {"id": 12118, "dom": "D", "ent": "optic nerve disease", "cls": "infect"}, {"id": 12119, "dom": "D", "ent": "optic nerve disease", "cls": "tumor"}, {"id": 12120, "dom": "D", "ent": "optic nerve disease", "cls": "mental"}, {"id": 12121, "dom": "D", "ent": "optic nerve disease", "cls": "genetic"}, {"id": 12122, "dom": "D", "ent": "optic nerve disease", "cls": "bact"}, {"id": 12123, "dom": "D", "ent": "optic nerve disease", "cls": "viral"}, {"id": 12124, "dom": "D", "ent": "optic nerve disease", "cls": "fungal"}, {"id": 12125, "dom": "D", "ent": "optic nerve disease", "cls": "parasit"}, {"id": 12126, "dom": "D", "ent": "placenta cancer", "cls": "cardio"}, {"id": 12127, "dom": "D", "ent": "placenta cancer", "cls": "resp"}, {"id": 12128, "dom": "D", "ent": "placenta cancer", "cls": "gi"}, {"id": 12129, "dom": "D", "ent": "placenta cancer", "cls": "nerv"}, {"id": 12130, "dom": "D", "ent": "placenta cancer", "cls": "skin"}, {"id": 12131, "dom": "D", "ent": "placenta cancer", "cls": "musc"}, {"id": 12132, "dom": "D", "ent": "placenta cancer", "cls": "urin"}, {"id": 12133, "dom": "D", "ent": "placenta cancer", "cls": "endo"}, {"id": 12134, "dom": "D", "ent": "placenta cancer", "cls": "immune"}, {"id": 12135, "dom": "D", "ent": "placenta cancer", "cls": "blood"}, {"id": 12136, "dom": "D", "ent": "placenta cancer", "cls": "repro"}, {"id": 12137, "dom": "D", "ent": "placenta cancer", "cls": "breast"}, {"id": 12138, "dom": "D", "ent": "placenta cancer", "cls": "chest"}, {"id": 12139, "dom": "D", "ent": "placenta cancer", "cls": "metab"}, {"id": 12140, "dom": "D", "ent": "placenta cancer", "cls": "infect"}, {"id": 12141, "dom": "D", "ent": "placenta cancer", "cls": "tumor"}, {"id": 12142, "dom": "D", "ent": "placenta cancer", "cls": "mental"}, {"id": 12143, "dom": "D", "ent": "placenta cancer", "cls": "genetic"}, {"id": 12144, "dom": "D", "ent": "placenta cancer", "cls": "bact"}, {"id": 12145, "dom": "D", "ent": "placenta cancer", "cls": "viral"}, {"id": 12146, "dom": "D", "ent": "placenta cancer", "cls": "fungal"}, {"id": 12147, "dom": "D", "ent": "placenta cancer", "cls": "parasit"}, {"id": 12148, "dom": "D", "ent": "communication disorder", "cls": "cardio"}, {"id": 12149, "dom": "D", "ent": "communication disorder", "cls": "resp"}, {"id": 12150, "dom": "D", "ent": "communication disorder", "cls": "gi"}, {"id": 12151, "dom": "D", "ent": "communication disorder", "cls": "nerv"}, {"id": 12152, "dom": "D", "ent": "communication disorder", "cls": "skin"}, {"id": 12153, "dom": "D", "ent": "communication disorder", "cls": "musc"}, {"id": 12154, "dom": "D", "ent": "communication disorder", "cls": "urin"}, {"id": 12155, "dom": "D", "ent": "communication disorder", "cls": "endo"}, {"id": 12156, "dom": "D", "ent": "communication disorder", "cls": "immune"}, {"id": 12157, "dom": "D", "ent": "communication disorder", "cls": "blood"}, {"id": 12158, "dom": "D", "ent": "communication disorder", "cls": "repro"}, {"id": 12159, "dom": "D", "ent": "communication disorder", "cls": "breast"}, {"id": 12160, "dom": "D", "ent": "communication disorder", "cls": "chest"}, {"id": 12161, "dom": "D", "ent": "communication disorder", "cls": "metab"}, {"id": 12162, "dom": "D", "ent": "communication disorder", "cls": "infect"}, {"id": 12163, "dom": "D", "ent": "communication disorder", "cls": "tumor"}, {"id": 12164, "dom": "D", "ent": "communication disorder", "cls": "mental"}, {"id": 12165, "dom": "D", "ent": "communication disorder", "cls": "genetic"}, {"id": 12166, "dom": "D", "ent": "communication disorder", "cls": "bact"}, {"id": 12167, "dom": "D", "ent": "communication disorder", "cls": "viral"}, {"id": 12168, "dom": "D", "ent": "communication disorder", "cls": "fungal"}, {"id": 12169, "dom": "D", "ent": "communication disorder", "cls": "parasit"}, {"id": 12170, "dom": "D", "ent": "hepatitis D", "cls": "cardio"}, {"id": 12171, "dom": "D", "ent": "hepatitis D", "cls": "resp"}, {"id": 12172, "dom": "D", "ent": "hepatitis D", "cls": "gi"}, {"id": 12173, "dom": "D", "ent": "hepatitis D", "cls": "nerv"}, {"id": 12174, "dom": "D", "ent": "hepatitis D", "cls": "skin"}, {"id": 12175, "dom": "D", "ent": "hepatitis D", "cls": "musc"}, {"id": 12176, "dom": "D", "ent": "hepatitis D", "cls": "urin"}, {"id": 12177, "dom": "D", "ent": "hepatitis D", "cls": "endo"}, {"id": 12178, "dom": "D", "ent": "hepatitis D", "cls": "immune"}, {"id": 12179, "dom": "D", "ent": "hepatitis D", "cls": "blood"}, {"id": 12180, "dom": "D", "ent": "hepatitis D", "cls": "repro"}, {"id": 12181, "dom": "D", "ent": "hepatitis D", "cls": "breast"}, {"id": 12182, "dom": "D", "ent": "hepatitis D", "cls": "chest"}, {"id": 12183, "dom": "D", "ent": "hepatitis D", "cls": "metab"}, {"id": 12184, "dom": "D", "ent": "hepatitis D", "cls": "infect"}, {"id": 12185, "dom": "D", "ent": "hepatitis D", "cls": "tumor"}, {"id": 12186, "dom": "D", "ent": "hepatitis D", "cls": "mental"}, {"id": 12187, "dom": "D", "ent": "hepatitis D", "cls": "genetic"}, {"id": 12188, "dom": "D", "ent": "hepatitis D", "cls": "bact"}, {"id": 12189, "dom": "D", "ent": "hepatitis D", "cls": "viral"}, {"id": 12190, "dom": "D", "ent": "hepatitis D", "cls": "fungal"}, {"id": 12191, "dom": "D", "ent": "hepatitis D", "cls": "parasit"}, {"id": 12192, "dom": "D", "ent": "autoimmune hepatitis", "cls": "cardio"}, {"id": 12193, "dom": "D", "ent": "autoimmune hepatitis", "cls": "resp"}, {"id": 12194, "dom": "D", "ent": "autoimmune hepatitis", "cls": "gi"}, {"id": 12195, "dom": "D", "ent": "autoimmune hepatitis", "cls": "nerv"}, {"id": 12196, "dom": "D", "ent": "autoimmune hepatitis", "cls": "skin"}, {"id": 12197, "dom": "D", "ent": "autoimmune hepatitis", "cls": "musc"}, {"id": 12198, "dom": "D", "ent": "autoimmune hepatitis", "cls": "urin"}, {"id": 12199, "dom": "D", "ent": "autoimmune hepatitis", "cls": "endo"}, {"id": 12200, "dom": "D", "ent": "autoimmune hepatitis", "cls": "immune"}, {"id": 12201, "dom": "D", "ent": "autoimmune hepatitis", "cls": "blood"}, {"id": 12202, "dom": "D", "ent": "autoimmune hepatitis", "cls": "repro"}, {"id": 12203, "dom": "D", "ent": "autoimmune hepatitis", "cls": "breast"}, {"id": 12204, "dom": "D", "ent": "autoimmune hepatitis", "cls": "chest"}, {"id": 12205, "dom": "D", "ent": "autoimmune hepatitis", "cls": "metab"}, {"id": 12206, "dom": "D", "ent": "autoimmune hepatitis", "cls": "infect"}, {"id": 12207, "dom": "D", "ent": "autoimmune hepatitis", "cls": "tumor"}, {"id": 12208, "dom": "D", "ent": "autoimmune hepatitis", "cls": "mental"}, {"id": 12209, "dom": "D", "ent": "autoimmune hepatitis", "cls": "genetic"}, {"id": 12210, "dom": "D", "ent": "autoimmune hepatitis", "cls": "bact"}, {"id": 12211, "dom": "D", "ent": "autoimmune hepatitis", "cls": "viral"}, {"id": 12212, "dom": "D", "ent": "autoimmune hepatitis", "cls": "fungal"}, {"id": 12213, "dom": "D", "ent": "autoimmune hepatitis", "cls": "parasit"}, {"id": 12214, "dom": "D", "ent": "factor XIII deficiency", "cls": "cardio"}, {"id": 12215, "dom": "D", "ent": "factor XIII deficiency", "cls": "resp"}, {"id": 12216, "dom": "D", "ent": "factor XIII deficiency", "cls": "gi"}, {"id": 12217, "dom": "D", "ent": "factor XIII deficiency", "cls": "nerv"}, {"id": 12218, "dom": "D", "ent": "factor XIII deficiency", "cls": "skin"}, {"id": 12219, "dom": "D", "ent": "factor XIII deficiency", "cls": "musc"}, {"id": 12220, "dom": "D", "ent": "factor XIII deficiency", "cls": "urin"}, {"id": 12221, "dom": "D", "ent": "factor XIII deficiency", "cls": "endo"}, {"id": 12222, "dom": "D", "ent": "factor XIII deficiency", "cls": "immune"}, {"id": 12223, "dom": "D", "ent": "factor XIII deficiency", "cls": "blood"}, {"id": 12224, "dom": "D", "ent": "factor XIII deficiency", "cls": "repro"}, {"id": 12225, "dom": "D", "ent": "factor XIII deficiency", "cls": "breast"}, {"id": 12226, "dom": "D", "ent": "factor XIII deficiency", "cls": "chest"}, {"id": 12227, "dom": "D", "ent": "factor XIII deficiency", "cls": "metab"}, {"id": 12228, "dom": "D", "ent": "factor XIII deficiency", "cls": "infect"}, {"id": 12229, "dom": "D", "ent": "factor XIII deficiency", "cls": "tumor"}, {"id": 12230, "dom": "D", "ent": "factor XIII deficiency", "cls": "mental"}, {"id": 12231, "dom": "D", "ent": "factor XIII deficiency", "cls": "genetic"}, {"id": 12232, "dom": "D", "ent": "factor XIII deficiency", "cls": "bact"}, {"id": 12233, "dom": "D", "ent": "factor XIII deficiency", "cls": "viral"}, {"id": 12234, "dom": "D", "ent": "factor XIII deficiency", "cls": "fungal"}, {"id": 12235, "dom": "D", "ent": "factor XIII deficiency", "cls": "parasit"}, {"id": 12236, "dom": "D", "ent": "factor XII deficiency", "cls": "cardio"}, {"id": 12237, "dom": "D", "ent": "factor XII deficiency", "cls": "resp"}, {"id": 12238, "dom": "D", "ent": "factor XII deficiency", "cls": "gi"}, {"id": 12239, "dom": "D", "ent": "factor XII deficiency", "cls": "nerv"}, {"id": 12240, "dom": "D", "ent": "factor XII deficiency", "cls": "skin"}, {"id": 12241, "dom": "D", "ent": "factor XII deficiency", "cls": "musc"}, {"id": 12242, "dom": "D", "ent": "factor XII deficiency", "cls": "urin"}, {"id": 12243, "dom": "D", "ent": "factor XII deficiency", "cls": "endo"}, {"id": 12244, "dom": "D", "ent": "factor XII deficiency", "cls": "immune"}, {"id": 12245, "dom": "D", "ent": "factor XII deficiency", "cls": "blood"}, {"id": 12246, "dom": "D", "ent": "factor XII deficiency", "cls": "repro"}, {"id": 12247, "dom": "D", "ent": "factor XII deficiency", "cls": "breast"}, {"id": 12248, "dom": "D", "ent": "factor XII deficiency", "cls": "chest"}, {"id": 12249, "dom": "D", "ent": "factor XII deficiency", "cls": "metab"}, {"id": 12250, "dom": "D", "ent": "factor XII deficiency", "cls": "infect"}, {"id": 12251, "dom": "D", "ent": "factor XII deficiency", "cls": "tumor"}, {"id": 12252, "dom": "D", "ent": "factor XII deficiency", "cls": "mental"}, {"id": 12253, "dom": "D", "ent": "factor XII deficiency", "cls": "genetic"}, {"id": 12254, "dom": "D", "ent": "factor XII deficiency", "cls": "bact"}, {"id": 12255, "dom": "D", "ent": "factor XII deficiency", "cls": "viral"}, {"id": 12256, "dom": "D", "ent": "factor XII deficiency", "cls": "fungal"}, {"id": 12257, "dom": "D", "ent": "factor XII deficiency", "cls": "parasit"}, {"id": 12258, "dom": "D", "ent": "hepatitis", "cls": "cardio"}, {"id": 12259, "dom": "D", "ent": "hepatitis", "cls": "resp"}, {"id": 12260, "dom": "D", "ent": "hepatitis", "cls": "gi"}, {"id": 12261, "dom": "D", "ent": "hepatitis", "cls": "nerv"}, {"id": 12262, "dom": "D", "ent": "hepatitis", "cls": "skin"}, {"id": 12263, "dom": "D", "ent": "hepatitis", "cls": "musc"}, {"id": 12264, "dom": "D", "ent": "hepatitis", "cls": "urin"}, {"id": 12265, "dom": "D", "ent": "hepatitis", "cls": "endo"}, {"id": 12266, "dom": "D", "ent": "hepatitis", "cls": "immune"}, {"id": 12267, "dom": "D", "ent": "hepatitis", "cls": "blood"}, {"id": 12268, "dom": "D", "ent": "hepatitis", "cls": "repro"}, {"id": 12269, "dom": "D", "ent": "hepatitis", "cls": "breast"}, {"id": 12270, "dom": "D", "ent": "hepatitis", "cls": "chest"}, {"id": 12271, "dom": "D", "ent": "hepatitis", "cls": "metab"}, {"id": 12272, "dom": "D", "ent": "hepatitis", "cls": "infect"}, {"id": 12273, "dom": "D", "ent": "hepatitis", "cls": "tumor"}, {"id": 12274, "dom": "D", "ent": "hepatitis", "cls": "mental"}, {"id": 12275, "dom": "D", "ent": "hepatitis", "cls": "genetic"}, {"id": 12276, "dom": "D", "ent": "hepatitis", "cls": "bact"}, {"id": 12277, "dom": "D", "ent": "hepatitis", "cls": "viral"}, {"id": 12278, "dom": "D", "ent": "hepatitis", "cls": "fungal"}, {"id": 12279, "dom": "D", "ent": "hepatitis", "cls": "parasit"}, {"id": 12280, "dom": "D", "ent": "cervix disease", "cls": "cardio"}, {"id": 12281, "dom": "D", "ent": "cervix disease", "cls": "resp"}, {"id": 12282, "dom": "D", "ent": "cervix disease", "cls": "gi"}, {"id": 12283, "dom": "D", "ent": "cervix disease", "cls": "nerv"}, {"id": 12284, "dom": "D", "ent": "cervix disease", "cls": "skin"}, {"id": 12285, "dom": "D", "ent": "cervix disease", "cls": "musc"}, {"id": 12286, "dom": "D", "ent": "cervix disease", "cls": "urin"}, {"id": 12287, "dom": "D", "ent": "cervix disease", "cls": "endo"}, {"id": 12288, "dom": "D", "ent": "cervix disease", "cls": "immune"}, {"id": 12289, "dom": "D", "ent": "cervix disease", "cls": "blood"}, {"id": 12290, "dom": "D", "ent": "cervix disease", "cls": "repro"}, {"id": 12291, "dom": "D", "ent": "cervix disease", "cls": "breast"}, {"id": 12292, "dom": "D", "ent": "cervix disease", "cls": "chest"}, {"id": 12293, "dom": "D", "ent": "cervix disease", "cls": "metab"}, {"id": 12294, "dom": "D", "ent": "cervix disease", "cls": "infect"}, {"id": 12295, "dom": "D", "ent": "cervix disease", "cls": "tumor"}, {"id": 12296, "dom": "D", "ent": "cervix disease", "cls": "mental"}, {"id": 12297, "dom": "D", "ent": "cervix disease", "cls": "genetic"}, {"id": 12298, "dom": "D", "ent": "cervix disease", "cls": "bact"}, {"id": 12299, "dom": "D", "ent": "cervix disease", "cls": "viral"}, {"id": 12300, "dom": "D", "ent": "cervix disease", "cls": "fungal"}, {"id": 12301, "dom": "D", "ent": "cervix disease", "cls": "parasit"}, {"id": 12302, "dom": "D", "ent": "lateral sclerosis", "cls": "cardio"}, {"id": 12303, "dom": "D", "ent": "lateral sclerosis", "cls": "resp"}, {"id": 12304, "dom": "D", "ent": "lateral sclerosis", "cls": "gi"}, {"id": 12305, "dom": "D", "ent": "lateral sclerosis", "cls": "nerv"}, {"id": 12306, "dom": "D", "ent": "lateral sclerosis", "cls": "skin"}, {"id": 12307, "dom": "D", "ent": "lateral sclerosis", "cls": "musc"}, {"id": 12308, "dom": "D", "ent": "lateral sclerosis", "cls": "urin"}, {"id": 12309, "dom": "D", "ent": "lateral sclerosis", "cls": "endo"}, {"id": 12310, "dom": "D", "ent": "lateral sclerosis", "cls": "immune"}, {"id": 12311, "dom": "D", "ent": "lateral sclerosis", "cls": "blood"}, {"id": 12312, "dom": "D", "ent": "lateral sclerosis", "cls": "repro"}, {"id": 12313, "dom": "D", "ent": "lateral sclerosis", "cls": "breast"}, {"id": 12314, "dom": "D", "ent": "lateral sclerosis", "cls": "chest"}, {"id": 12315, "dom": "D", "ent": "lateral sclerosis", "cls": "metab"}, {"id": 12316, "dom": "D", "ent": "lateral sclerosis", "cls": "infect"}, {"id": 12317, "dom": "D", "ent": "lateral sclerosis", "cls": "tumor"}, {"id": 12318, "dom": "D", "ent": "lateral sclerosis", "cls": "mental"}, {"id": 12319, "dom": "D", "ent": "lateral sclerosis", "cls": "genetic"}, {"id": 12320, "dom": "D", "ent": "lateral sclerosis", "cls": "bact"}, {"id": 12321, "dom": "D", "ent": "lateral sclerosis", "cls": "viral"}, {"id": 12322, "dom": "D", "ent": "lateral sclerosis", "cls": "fungal"}, {"id": 12323, "dom": "D", "ent": "lateral sclerosis", "cls": "parasit"}, {"id": 12324, "dom": "D", "ent": "West Nile fever", "cls": "cardio"}, {"id": 12325, "dom": "D", "ent": "West Nile fever", "cls": "resp"}, {"id": 12326, "dom": "D", "ent": "West Nile fever", "cls": "gi"}, {"id": 12327, "dom": "D", "ent": "West Nile fever", "cls": "nerv"}, {"id": 12328, "dom": "D", "ent": "West Nile fever", "cls": "skin"}, {"id": 12329, "dom": "D", "ent": "West Nile fever", "cls": "musc"}, {"id": 12330, "dom": "D", "ent": "West Nile fever", "cls": "urin"}, {"id": 12331, "dom": "D", "ent": "West Nile fever", "cls": "endo"}, {"id": 12332, "dom": "D", "ent": "West Nile fever", "cls": "immune"}, {"id": 12333, "dom": "D", "ent": "West Nile fever", "cls": "blood"}, {"id": 12334, "dom": "D", "ent": "West Nile fever", "cls": "repro"}, {"id": 12335, "dom": "D", "ent": "West Nile fever", "cls": "breast"}, {"id": 12336, "dom": "D", "ent": "West Nile fever", "cls": "chest"}, {"id": 12337, "dom": "D", "ent": "West Nile fever", "cls": "metab"}, {"id": 12338, "dom": "D", "ent": "West Nile fever", "cls": "infect"}, {"id": 12339, "dom": "D", "ent": "West Nile fever", "cls": "tumor"}, {"id": 12340, "dom": "D", "ent": "West Nile fever", "cls": "mental"}, {"id": 12341, "dom": "D", "ent": "West Nile fever", "cls": "genetic"}, {"id": 12342, "dom": "D", "ent": "West Nile fever", "cls": "bact"}, {"id": 12343, "dom": "D", "ent": "West Nile fever", "cls": "viral"}, {"id": 12344, "dom": "D", "ent": "West Nile fever", "cls": "fungal"}, {"id": 12345, "dom": "D", "ent": "West Nile fever", "cls": "parasit"}, {"id": 12346, "dom": "D", "ent": "multiple sclerosis", "cls": "cardio"}, {"id": 12347, "dom": "D", "ent": "multiple sclerosis", "cls": "resp"}, {"id": 12348, "dom": "D", "ent": "multiple sclerosis", "cls": "gi"}, {"id": 12349, "dom": "D", "ent": "multiple sclerosis", "cls": "nerv"}, {"id": 12350, "dom": "D", "ent": "multiple sclerosis", "cls": "skin"}, {"id": 12351, "dom": "D", "ent": "multiple sclerosis", "cls": "musc"}, {"id": 12352, "dom": "D", "ent": "multiple sclerosis", "cls": "urin"}, {"id": 12353, "dom": "D", "ent": "multiple sclerosis", "cls": "endo"}, {"id": 12354, "dom": "D", "ent": "multiple sclerosis", "cls": "immune"}, {"id": 12355, "dom": "D", "ent": "multiple sclerosis", "cls": "blood"}, {"id": 12356, "dom": "D", "ent": "multiple sclerosis", "cls": "repro"}, {"id": 12357, "dom": "D", "ent": "multiple sclerosis", "cls": "breast"}, {"id": 12358, "dom": "D", "ent": "multiple sclerosis", "cls": "chest"}, {"id": 12359, "dom": "D", "ent": "multiple sclerosis", "cls": "metab"}, {"id": 12360, "dom": "D", "ent": "multiple sclerosis", "cls": "infect"}, {"id": 12361, "dom": "D", "ent": "multiple sclerosis", "cls": "tumor"}, {"id": 12362, "dom": "D", "ent": "multiple sclerosis", "cls": "mental"}, {"id": 12363, "dom": "D", "ent": "multiple sclerosis", "cls": "genetic"}, {"id": 12364, "dom": "D", "ent": "multiple sclerosis", "cls": "bact"}, {"id": 12365, "dom": "D", "ent": "multiple sclerosis", "cls": "viral"}, {"id": 12366, "dom": "D", "ent": "multiple sclerosis", "cls": "fungal"}, {"id": 12367, "dom": "D", "ent": "multiple sclerosis", "cls": "parasit"}, {"id": 12368, "dom": "D", "ent": "iris disease", "cls": "cardio"}, {"id": 12369, "dom": "D", "ent": "iris disease", "cls": "resp"}, {"id": 12370, "dom": "D", "ent": "iris disease", "cls": "gi"}, {"id": 12371, "dom": "D", "ent": "iris disease", "cls": "nerv"}, {"id": 12372, "dom": "D", "ent": "iris disease", "cls": "skin"}, {"id": 12373, "dom": "D", "ent": "iris disease", "cls": "musc"}, {"id": 12374, "dom": "D", "ent": "iris disease", "cls": "urin"}, {"id": 12375, "dom": "D", "ent": "iris disease", "cls": "endo"}, {"id": 12376, "dom": "D", "ent": "iris disease", "cls": "immune"}, {"id": 12377, "dom": "D", "ent": "iris disease", "cls": "blood"}, {"id": 12378, "dom": "D", "ent": "iris disease", "cls": "repro"}, {"id": 12379, "dom": "D", "ent": "iris disease", "cls": "breast"}, {"id": 12380, "dom": "D", "ent": "iris disease", "cls": "chest"}, {"id": 12381, "dom": "D", "ent": "iris disease", "cls": "metab"}, {"id": 12382, "dom": "D", "ent": "iris disease", "cls": "infect"}, {"id": 12383, "dom": "D", "ent": "iris disease", "cls": "tumor"}, {"id": 12384, "dom": "D", "ent": "iris disease", "cls": "mental"}, {"id": 12385, "dom": "D", "ent": "iris disease", "cls": "genetic"}, {"id": 12386, "dom": "D", "ent": "iris disease", "cls": "bact"}, {"id": 12387, "dom": "D", "ent": "iris disease", "cls": "viral"}, {"id": 12388, "dom": "D", "ent": "iris disease", "cls": "fungal"}, {"id": 12389, "dom": "D", "ent": "iris disease", "cls": "parasit"}, {"id": 12390, "dom": "D", "ent": "tooth erosion", "cls": "cardio"}, {"id": 12391, "dom": "D", "ent": "tooth erosion", "cls": "resp"}, {"id": 12392, "dom": "D", "ent": "tooth erosion", "cls": "gi"}, {"id": 12393, "dom": "D", "ent": "tooth erosion", "cls": "nerv"}, {"id": 12394, "dom": "D", "ent": "tooth erosion", "cls": "skin"}, {"id": 12395, "dom": "D", "ent": "tooth erosion", "cls": "musc"}, {"id": 12396, "dom": "D", "ent": "tooth erosion", "cls": "urin"}, {"id": 12397, "dom": "D", "ent": "tooth erosion", "cls": "endo"}, {"id": 12398, "dom": "D", "ent": "tooth erosion", "cls": "immune"}, {"id": 12399, "dom": "D", "ent": "tooth erosion", "cls": "blood"}, {"id": 12400, "dom": "D", "ent": "tooth erosion", "cls": "repro"}, {"id": 12401, "dom": "D", "ent": "tooth erosion", "cls": "breast"}, {"id": 12402, "dom": "D", "ent": "tooth erosion", "cls": "chest"}, {"id": 12403, "dom": "D", "ent": "tooth erosion", "cls": "metab"}, {"id": 12404, "dom": "D", "ent": "tooth erosion", "cls": "infect"}, {"id": 12405, "dom": "D", "ent": "tooth erosion", "cls": "tumor"}, {"id": 12406, "dom": "D", "ent": "tooth erosion", "cls": "mental"}, {"id": 12407, "dom": "D", "ent": "tooth erosion", "cls": "genetic"}, {"id": 12408, "dom": "D", "ent": "tooth erosion", "cls": "bact"}, {"id": 12409, "dom": "D", "ent": "tooth erosion", "cls": "viral"}, {"id": 12410, "dom": "D", "ent": "tooth erosion", "cls": "fungal"}, {"id": 12411, "dom": "D", "ent": "tooth erosion", "cls": "parasit"}, {"id": 12412, "dom": "D", "ent": "alcoholic psychosis", "cls": "cardio"}, {"id": 12413, "dom": "D", "ent": "alcoholic psychosis", "cls": "resp"}, {"id": 12414, "dom": "D", "ent": "alcoholic psychosis", "cls": "gi"}, {"id": 12415, "dom": "D", "ent": "alcoholic psychosis", "cls": "nerv"}, {"id": 12416, "dom": "D", "ent": "alcoholic psychosis", "cls": "skin"}, {"id": 12417, "dom": "D", "ent": "alcoholic psychosis", "cls": "musc"}, {"id": 12418, "dom": "D", "ent": "alcoholic psychosis", "cls": "urin"}, {"id": 12419, "dom": "D", "ent": "alcoholic psychosis", "cls": "endo"}, {"id": 12420, "dom": "D", "ent": "alcoholic psychosis", "cls": "immune"}, {"id": 12421, "dom": "D", "ent": "alcoholic psychosis", "cls": "blood"}, {"id": 12422, "dom": "D", "ent": "alcoholic psychosis", "cls": "repro"}, {"id": 12423, "dom": "D", "ent": "alcoholic psychosis", "cls": "breast"}, {"id": 12424, "dom": "D", "ent": "alcoholic psychosis", "cls": "chest"}, {"id": 12425, "dom": "D", "ent": "alcoholic psychosis", "cls": "metab"}, {"id": 12426, "dom": "D", "ent": "alcoholic psychosis", "cls": "infect"}, {"id": 12427, "dom": "D", "ent": "alcoholic psychosis", "cls": "tumor"}, {"id": 12428, "dom": "D", "ent": "alcoholic psychosis", "cls": "mental"}, {"id": 12429, "dom": "D", "ent": "alcoholic psychosis", "cls": "genetic"}, {"id": 12430, "dom": "D", "ent": "alcoholic psychosis", "cls": "bact"}, {"id": 12431, "dom": "D", "ent": "alcoholic psychosis", "cls": "viral"}, {"id": 12432, "dom": "D", "ent": "alcoholic psychosis", "cls": "fungal"}, {"id": 12433, "dom": "D", "ent": "alcoholic psychosis", "cls": "parasit"}, {"id": 12434, "dom": "D", "ent": "Gilbert syndrome", "cls": "cardio"}, {"id": 12435, "dom": "D", "ent": "Gilbert syndrome", "cls": "resp"}, {"id": 12436, "dom": "D", "ent": "Gilbert syndrome", "cls": "gi"}, {"id": 12437, "dom": "D", "ent": "Gilbert syndrome", "cls": "nerv"}, {"id": 12438, "dom": "D", "ent": "Gilbert syndrome", "cls": "skin"}, {"id": 12439, "dom": "D", "ent": "Gilbert syndrome", "cls": "musc"}, {"id": 12440, "dom": "D", "ent": "Gilbert syndrome", "cls": "urin"}, {"id": 12441, "dom": "D", "ent": "Gilbert syndrome", "cls": "endo"}, {"id": 12442, "dom": "D", "ent": "Gilbert syndrome", "cls": "immune"}, {"id": 12443, "dom": "D", "ent": "Gilbert syndrome", "cls": "blood"}, {"id": 12444, "dom": "D", "ent": "Gilbert syndrome", "cls": "repro"}, {"id": 12445, "dom": "D", "ent": "Gilbert syndrome", "cls": "breast"}, {"id": 12446, "dom": "D", "ent": "Gilbert syndrome", "cls": "chest"}, {"id": 12447, "dom": "D", "ent": "Gilbert syndrome", "cls": "metab"}, {"id": 12448, "dom": "D", "ent": "Gilbert syndrome", "cls": "infect"}, {"id": 12449, "dom": "D", "ent": "Gilbert syndrome", "cls": "tumor"}, {"id": 12450, "dom": "D", "ent": "Gilbert syndrome", "cls": "mental"}, {"id": 12451, "dom": "D", "ent": "Gilbert syndrome", "cls": "genetic"}, {"id": 12452, "dom": "D", "ent": "Gilbert syndrome", "cls": "bact"}, {"id": 12453, "dom": "D", "ent": "Gilbert syndrome", "cls": "viral"}, {"id": 12454, "dom": "D", "ent": "Gilbert syndrome", "cls": "fungal"}, {"id": 12455, "dom": "D", "ent": "Gilbert syndrome", "cls": "parasit"}, {"id": 12456, "dom": "D", "ent": "auditory system disease", "cls": "cardio"}, {"id": 12457, "dom": "D", "ent": "auditory system disease", "cls": "resp"}, {"id": 12458, "dom": "D", "ent": "auditory system disease", "cls": "gi"}, {"id": 12459, "dom": "D", "ent": "auditory system disease", "cls": "nerv"}, {"id": 12460, "dom": "D", "ent": "auditory system disease", "cls": "skin"}, {"id": 12461, "dom": "D", "ent": "auditory system disease", "cls": "musc"}, {"id": 12462, "dom": "D", "ent": "auditory system disease", "cls": "urin"}, {"id": 12463, "dom": "D", "ent": "auditory system disease", "cls": "endo"}, {"id": 12464, "dom": "D", "ent": "auditory system disease", "cls": "immune"}, {"id": 12465, "dom": "D", "ent": "auditory system disease", "cls": "blood"}, {"id": 12466, "dom": "D", "ent": "auditory system disease", "cls": "repro"}, {"id": 12467, "dom": "D", "ent": "auditory system disease", "cls": "breast"}, {"id": 12468, "dom": "D", "ent": "auditory system disease", "cls": "chest"}, {"id": 12469, "dom": "D", "ent": "auditory system disease", "cls": "metab"}, {"id": 12470, "dom": "D", "ent": "auditory system disease", "cls": "infect"}, {"id": 12471, "dom": "D", "ent": "auditory system disease", "cls": "tumor"}, {"id": 12472, "dom": "D", "ent": "auditory system disease", "cls": "mental"}, {"id": 12473, "dom": "D", "ent": "auditory system disease", "cls": "genetic"}, {"id": 12474, "dom": "D", "ent": "auditory system disease", "cls": "bact"}, {"id": 12475, "dom": "D", "ent": "auditory system disease", "cls": "viral"}, {"id": 12476, "dom": "D", "ent": "auditory system disease", "cls": "fungal"}, {"id": 12477, "dom": "D", "ent": "auditory system disease", "cls": "parasit"}, {"id": 12478, "dom": "D", "ent": "narcissistic personality disorder", "cls": "cardio"}, {"id": 12479, "dom": "D", "ent": "narcissistic personality disorder", "cls": "resp"}, {"id": 12480, "dom": "D", "ent": "narcissistic personality disorder", "cls": "gi"}, {"id": 12481, "dom": "D", "ent": "narcissistic personality disorder", "cls": "nerv"}, {"id": 12482, "dom": "D", "ent": "narcissistic personality disorder", "cls": "skin"}, {"id": 12483, "dom": "D", "ent": "narcissistic personality disorder", "cls": "musc"}, {"id": 12484, "dom": "D", "ent": "narcissistic personality disorder", "cls": "urin"}, {"id": 12485, "dom": "D", "ent": "narcissistic personality disorder", "cls": "endo"}, {"id": 12486, "dom": "D", "ent": "narcissistic personality disorder", "cls": "immune"}, {"id": 12487, "dom": "D", "ent": "narcissistic personality disorder", "cls": "blood"}, {"id": 12488, "dom": "D", "ent": "narcissistic personality disorder", "cls": "repro"}, {"id": 12489, "dom": "D", "ent": "narcissistic personality disorder", "cls": "breast"}, {"id": 12490, "dom": "D", "ent": "narcissistic personality disorder", "cls": "chest"}, {"id": 12491, "dom": "D", "ent": "narcissistic personality disorder", "cls": "metab"}, {"id": 12492, "dom": "D", "ent": "narcissistic personality disorder", "cls": "infect"}, {"id": 12493, "dom": "D", "ent": "narcissistic personality disorder", "cls": "tumor"}, {"id": 12494, "dom": "D", "ent": "narcissistic personality disorder", "cls": "mental"}, {"id": 12495, "dom": "D", "ent": "narcissistic personality disorder", "cls": "genetic"}, {"id": 12496, "dom": "D", "ent": "narcissistic personality disorder", "cls": "bact"}, {"id": 12497, "dom": "D", "ent": "narcissistic personality disorder", "cls": "viral"}, {"id": 12498, "dom": "D", "ent": "narcissistic personality disorder", "cls": "fungal"}, {"id": 12499, "dom": "D", "ent": "narcissistic personality disorder", "cls": "parasit"}, {"id": 12500, "dom": "D", "ent": "long QT syndrome", "cls": "cardio"}, {"id": 12501, "dom": "D", "ent": "long QT syndrome", "cls": "resp"}, {"id": 12502, "dom": "D", "ent": "long QT syndrome", "cls": "gi"}, {"id": 12503, "dom": "D", "ent": "long QT syndrome", "cls": "nerv"}, {"id": 12504, "dom": "D", "ent": "long QT syndrome", "cls": "skin"}, {"id": 12505, "dom": "D", "ent": "long QT syndrome", "cls": "musc"}, {"id": 12506, "dom": "D", "ent": "long QT syndrome", "cls": "urin"}, {"id": 12507, "dom": "D", "ent": "long QT syndrome", "cls": "endo"}, {"id": 12508, "dom": "D", "ent": "long QT syndrome", "cls": "immune"}, {"id": 12509, "dom": "D", "ent": "long QT syndrome", "cls": "blood"}, {"id": 12510, "dom": "D", "ent": "long QT syndrome", "cls": "repro"}, {"id": 12511, "dom": "D", "ent": "long QT syndrome", "cls": "breast"}, {"id": 12512, "dom": "D", "ent": "long QT syndrome", "cls": "chest"}, {"id": 12513, "dom": "D", "ent": "long QT syndrome", "cls": "metab"}, {"id": 12514, "dom": "D", "ent": "long QT syndrome", "cls": "infect"}, {"id": 12515, "dom": "D", "ent": "long QT syndrome", "cls": "tumor"}, {"id": 12516, "dom": "D", "ent": "long QT syndrome", "cls": "mental"}, {"id": 12517, "dom": "D", "ent": "long QT syndrome", "cls": "genetic"}, {"id": 12518, "dom": "D", "ent": "long QT syndrome", "cls": "bact"}, {"id": 12519, "dom": "D", "ent": "long QT syndrome", "cls": "viral"}, {"id": 12520, "dom": "D", "ent": "long QT syndrome", "cls": "fungal"}, {"id": 12521, "dom": "D", "ent": "long QT syndrome", "cls": "parasit"}, {"id": 12522, "dom": "D", "ent": "familial Mediterranean fever", "cls": "cardio"}, {"id": 12523, "dom": "D", "ent": "familial Mediterranean fever", "cls": "resp"}, {"id": 12524, "dom": "D", "ent": "familial Mediterranean fever", "cls": "gi"}, {"id": 12525, "dom": "D", "ent": "familial Mediterranean fever", "cls": "nerv"}, {"id": 12526, "dom": "D", "ent": "familial Mediterranean fever", "cls": "skin"}, {"id": 12527, "dom": "D", "ent": "familial Mediterranean fever", "cls": "musc"}, {"id": 12528, "dom": "D", "ent": "familial Mediterranean fever", "cls": "urin"}, {"id": 12529, "dom": "D", "ent": "familial Mediterranean fever", "cls": "endo"}, {"id": 12530, "dom": "D", "ent": "familial Mediterranean fever", "cls": "immune"}, {"id": 12531, "dom": "D", "ent": "familial Mediterranean fever", "cls": "blood"}, {"id": 12532, "dom": "D", "ent": "familial Mediterranean fever", "cls": "repro"}, {"id": 12533, "dom": "D", "ent": "familial Mediterranean fever", "cls": "breast"}, {"id": 12534, "dom": "D", "ent": "familial Mediterranean fever", "cls": "chest"}, {"id": 12535, "dom": "D", "ent": "familial Mediterranean fever", "cls": "metab"}, {"id": 12536, "dom": "D", "ent": "familial Mediterranean fever", "cls": "infect"}, {"id": 12537, "dom": "D", "ent": "familial Mediterranean fever", "cls": "tumor"}, {"id": 12538, "dom": "D", "ent": "familial Mediterranean fever", "cls": "mental"}, {"id": 12539, "dom": "D", "ent": "familial Mediterranean fever", "cls": "genetic"}, {"id": 12540, "dom": "D", "ent": "familial Mediterranean fever", "cls": "bact"}, {"id": 12541, "dom": "D", "ent": "familial Mediterranean fever", "cls": "viral"}, {"id": 12542, "dom": "D", "ent": "familial Mediterranean fever", "cls": "fungal"}, {"id": 12543, "dom": "D", "ent": "familial Mediterranean fever", "cls": "parasit"}, {"id": 12544, "dom": "D", "ent": "substance-related disorder", "cls": "cardio"}, {"id": 12545, "dom": "D", "ent": "substance-related disorder", "cls": "resp"}, {"id": 12546, "dom": "D", "ent": "substance-related disorder", "cls": "gi"}, {"id": 12547, "dom": "D", "ent": "substance-related disorder", "cls": "nerv"}, {"id": 12548, "dom": "D", "ent": "substance-related disorder", "cls": "skin"}, {"id": 12549, "dom": "D", "ent": "substance-related disorder", "cls": "musc"}, {"id": 12550, "dom": "D", "ent": "substance-related disorder", "cls": "urin"}, {"id": 12551, "dom": "D", "ent": "substance-related disorder", "cls": "endo"}, {"id": 12552, "dom": "D", "ent": "substance-related disorder", "cls": "immune"}, {"id": 12553, "dom": "D", "ent": "substance-related disorder", "cls": "blood"}, {"id": 12554, "dom": "D", "ent": "substance-related disorder", "cls": "repro"}, {"id": 12555, "dom": "D", "ent": "substance-related disorder", "cls": "breast"}, {"id": 12556, "dom": "D", "ent": "substance-related disorder", "cls": "chest"}, {"id": 12557, "dom": "D", "ent": "substance-related disorder", "cls": "metab"}, {"id": 12558, "dom": "D", "ent": "substance-related disorder", "cls": "infect"}, {"id": 12559, "dom": "D", "ent": "substance-related disorder", "cls": "tumor"}, {"id": 12560, "dom": "D", "ent": "substance-related disorder", "cls": "mental"}, {"id": 12561, "dom": "D", "ent": "substance-related disorder", "cls": "genetic"}, {"id": 12562, "dom": "D", "ent": "substance-related disorder", "cls": "bact"}, {"id": 12563, "dom": "D", "ent": "substance-related disorder", "cls": "viral"}, {"id": 12564, "dom": "D", "ent": "substance-related disorder", "cls": "fungal"}, {"id": 12565, "dom": "D", "ent": "substance-related disorder", "cls": "parasit"}, {"id": 12566, "dom": "D", "ent": "spinal cord disease", "cls": "cardio"}, {"id": 12567, "dom": "D", "ent": "spinal cord disease", "cls": "resp"}, {"id": 12568, "dom": "D", "ent": "spinal cord disease", "cls": "gi"}, {"id": 12569, "dom": "D", "ent": "spinal cord disease", "cls": "nerv"}, {"id": 12570, "dom": "D", "ent": "spinal cord disease", "cls": "skin"}, {"id": 12571, "dom": "D", "ent": "spinal cord disease", "cls": "musc"}, {"id": 12572, "dom": "D", "ent": "spinal cord disease", "cls": "urin"}, {"id": 12573, "dom": "D", "ent": "spinal cord disease", "cls": "endo"}, {"id": 12574, "dom": "D", "ent": "spinal cord disease", "cls": "immune"}, {"id": 12575, "dom": "D", "ent": "spinal cord disease", "cls": "blood"}, {"id": 12576, "dom": "D", "ent": "spinal cord disease", "cls": "repro"}, {"id": 12577, "dom": "D", "ent": "spinal cord disease", "cls": "breast"}, {"id": 12578, "dom": "D", "ent": "spinal cord disease", "cls": "chest"}, {"id": 12579, "dom": "D", "ent": "spinal cord disease", "cls": "metab"}, {"id": 12580, "dom": "D", "ent": "spinal cord disease", "cls": "infect"}, {"id": 12581, "dom": "D", "ent": "spinal cord disease", "cls": "tumor"}, {"id": 12582, "dom": "D", "ent": "spinal cord disease", "cls": "mental"}, {"id": 12583, "dom": "D", "ent": "spinal cord disease", "cls": "genetic"}, {"id": 12584, "dom": "D", "ent": "spinal cord disease", "cls": "bact"}, {"id": 12585, "dom": "D", "ent": "spinal cord disease", "cls": "viral"}, {"id": 12586, "dom": "D", "ent": "spinal cord disease", "cls": "fungal"}, {"id": 12587, "dom": "D", "ent": "spinal cord disease", "cls": "parasit"}, {"id": 12588, "dom": "D", "ent": "plague", "cls": "cardio"}, {"id": 12589, "dom": "D", "ent": "plague", "cls": "resp"}, {"id": 12590, "dom": "D", "ent": "plague", "cls": "gi"}, {"id": 12591, "dom": "D", "ent": "plague", "cls": "nerv"}, {"id": 12592, "dom": "D", "ent": "plague", "cls": "skin"}, {"id": 12593, "dom": "D", "ent": "plague", "cls": "musc"}, {"id": 12594, "dom": "D", "ent": "plague", "cls": "urin"}, {"id": 12595, "dom": "D", "ent": "plague", "cls": "endo"}, {"id": 12596, "dom": "D", "ent": "plague", "cls": "immune"}, {"id": 12597, "dom": "D", "ent": "plague", "cls": "blood"}, {"id": 12598, "dom": "D", "ent": "plague", "cls": "repro"}, {"id": 12599, "dom": "D", "ent": "plague", "cls": "breast"}, {"id": 12600, "dom": "D", "ent": "plague", "cls": "chest"}, {"id": 12601, "dom": "D", "ent": "plague", "cls": "metab"}, {"id": 12602, "dom": "D", "ent": "plague", "cls": "infect"}, {"id": 12603, "dom": "D", "ent": "plague", "cls": "tumor"}, {"id": 12604, "dom": "D", "ent": "plague", "cls": "mental"}, {"id": 12605, "dom": "D", "ent": "plague", "cls": "genetic"}, {"id": 12606, "dom": "D", "ent": "plague", "cls": "bact"}, {"id": 12607, "dom": "D", "ent": "plague", "cls": "viral"}, {"id": 12608, "dom": "D", "ent": "plague", "cls": "fungal"}, {"id": 12609, "dom": "D", "ent": "plague", "cls": "parasit"}, {"id": 12610, "dom": "D", "ent": "cerebral arterial disease", "cls": "cardio"}, {"id": 12611, "dom": "D", "ent": "cerebral arterial disease", "cls": "resp"}, {"id": 12612, "dom": "D", "ent": "cerebral arterial disease", "cls": "gi"}, {"id": 12613, "dom": "D", "ent": "cerebral arterial disease", "cls": "nerv"}, {"id": 12614, "dom": "D", "ent": "cerebral arterial disease", "cls": "skin"}, {"id": 12615, "dom": "D", "ent": "cerebral arterial disease", "cls": "musc"}, {"id": 12616, "dom": "D", "ent": "cerebral arterial disease", "cls": "urin"}, {"id": 12617, "dom": "D", "ent": "cerebral arterial disease", "cls": "endo"}, {"id": 12618, "dom": "D", "ent": "cerebral arterial disease", "cls": "immune"}, {"id": 12619, "dom": "D", "ent": "cerebral arterial disease", "cls": "blood"}, {"id": 12620, "dom": "D", "ent": "cerebral arterial disease", "cls": "repro"}, {"id": 12621, "dom": "D", "ent": "cerebral arterial disease", "cls": "breast"}, {"id": 12622, "dom": "D", "ent": "cerebral arterial disease", "cls": "chest"}, {"id": 12623, "dom": "D", "ent": "cerebral arterial disease", "cls": "metab"}, {"id": 12624, "dom": "D", "ent": "cerebral arterial disease", "cls": "infect"}, {"id": 12625, "dom": "D", "ent": "cerebral arterial disease", "cls": "tumor"}, {"id": 12626, "dom": "D", "ent": "cerebral arterial disease", "cls": "mental"}, {"id": 12627, "dom": "D", "ent": "cerebral arterial disease", "cls": "genetic"}, {"id": 12628, "dom": "D", "ent": "cerebral arterial disease", "cls": "bact"}, {"id": 12629, "dom": "D", "ent": "cerebral arterial disease", "cls": "viral"}, {"id": 12630, "dom": "D", "ent": "cerebral arterial disease", "cls": "fungal"}, {"id": 12631, "dom": "D", "ent": "cerebral arterial disease", "cls": "parasit"}, {"id": 12632, "dom": "D", "ent": "anterior compartment syndrome", "cls": "cardio"}, {"id": 12633, "dom": "D", "ent": "anterior compartment syndrome", "cls": "resp"}, {"id": 12634, "dom": "D", "ent": "anterior compartment syndrome", "cls": "gi"}, {"id": 12635, "dom": "D", "ent": "anterior compartment syndrome", "cls": "nerv"}, {"id": 12636, "dom": "D", "ent": "anterior compartment syndrome", "cls": "skin"}, {"id": 12637, "dom": "D", "ent": "anterior compartment syndrome", "cls": "musc"}, {"id": 12638, "dom": "D", "ent": "anterior compartment syndrome", "cls": "urin"}, {"id": 12639, "dom": "D", "ent": "anterior compartment syndrome", "cls": "endo"}, {"id": 12640, "dom": "D", "ent": "anterior compartment syndrome", "cls": "immune"}, {"id": 12641, "dom": "D", "ent": "anterior compartment syndrome", "cls": "blood"}, {"id": 12642, "dom": "D", "ent": "anterior compartment syndrome", "cls": "repro"}, {"id": 12643, "dom": "D", "ent": "anterior compartment syndrome", "cls": "breast"}, {"id": 12644, "dom": "D", "ent": "anterior compartment syndrome", "cls": "chest"}, {"id": 12645, "dom": "D", "ent": "anterior compartment syndrome", "cls": "metab"}, {"id": 12646, "dom": "D", "ent": "anterior compartment syndrome", "cls": "infect"}, {"id": 12647, "dom": "D", "ent": "anterior compartment syndrome", "cls": "tumor"}, {"id": 12648, "dom": "D", "ent": "anterior compartment syndrome", "cls": "mental"}, {"id": 12649, "dom": "D", "ent": "anterior compartment syndrome", "cls": "genetic"}, {"id": 12650, "dom": "D", "ent": "anterior compartment syndrome", "cls": "bact"}, {"id": 12651, "dom": "D", "ent": "anterior compartment syndrome", "cls": "viral"}, {"id": 12652, "dom": "D", "ent": "anterior compartment syndrome", "cls": "fungal"}, {"id": 12653, "dom": "D", "ent": "anterior compartment syndrome", "cls": "parasit"}, {"id": 12654, "dom": "D", "ent": "heart valve disease", "cls": "cardio"}, {"id": 12655, "dom": "D", "ent": "heart valve disease", "cls": "resp"}, {"id": 12656, "dom": "D", "ent": "heart valve disease", "cls": "gi"}, {"id": 12657, "dom": "D", "ent": "heart valve disease", "cls": "nerv"}, {"id": 12658, "dom": "D", "ent": "heart valve disease", "cls": "skin"}, {"id": 12659, "dom": "D", "ent": "heart valve disease", "cls": "musc"}, {"id": 12660, "dom": "D", "ent": "heart valve disease", "cls": "urin"}, {"id": 12661, "dom": "D", "ent": "heart valve disease", "cls": "endo"}, {"id": 12662, "dom": "D", "ent": "heart valve disease", "cls": "immune"}, {"id": 12663, "dom": "D", "ent": "heart valve disease", "cls": "blood"}, {"id": 12664, "dom": "D", "ent": "heart valve disease", "cls": "repro"}, {"id": 12665, "dom": "D", "ent": "heart valve disease", "cls": "breast"}, {"id": 12666, "dom": "D", "ent": "heart valve disease", "cls": "chest"}, {"id": 12667, "dom": "D", "ent": "heart valve disease", "cls": "metab"}, {"id": 12668, "dom": "D", "ent": "heart valve disease", "cls": "infect"}, {"id": 12669, "dom": "D", "ent": "heart valve disease", "cls": "tumor"}, {"id": 12670, "dom": "D", "ent": "heart valve disease", "cls": "mental"}, {"id": 12671, "dom": "D", "ent": "heart valve disease", "cls": "genetic"}, {"id": 12672, "dom": "D", "ent": "heart valve disease", "cls": "bact"}, {"id": 12673, "dom": "D", "ent": "heart valve disease", "cls": "viral"}, {"id": 12674, "dom": "D", "ent": "heart valve disease", "cls": "fungal"}, {"id": 12675, "dom": "D", "ent": "heart valve disease", "cls": "parasit"}, {"id": 12676, "dom": "D", "ent": "bile duct disease", "cls": "cardio"}, {"id": 12677, "dom": "D", "ent": "bile duct disease", "cls": "resp"}, {"id": 12678, "dom": "D", "ent": "bile duct disease", "cls": "gi"}, {"id": 12679, "dom": "D", "ent": "bile duct disease", "cls": "nerv"}, {"id": 12680, "dom": "D", "ent": "bile duct disease", "cls": "skin"}, {"id": 12681, "dom": "D", "ent": "bile duct disease", "cls": "musc"}, {"id": 12682, "dom": "D", "ent": "bile duct disease", "cls": "urin"}, {"id": 12683, "dom": "D", "ent": "bile duct disease", "cls": "endo"}, {"id": 12684, "dom": "D", "ent": "bile duct disease", "cls": "immune"}, {"id": 12685, "dom": "D", "ent": "bile duct disease", "cls": "blood"}, {"id": 12686, "dom": "D", "ent": "bile duct disease", "cls": "repro"}, {"id": 12687, "dom": "D", "ent": "bile duct disease", "cls": "breast"}, {"id": 12688, "dom": "D", "ent": "bile duct disease", "cls": "chest"}, {"id": 12689, "dom": "D", "ent": "bile duct disease", "cls": "metab"}, {"id": 12690, "dom": "D", "ent": "bile duct disease", "cls": "infect"}, {"id": 12691, "dom": "D", "ent": "bile duct disease", "cls": "tumor"}, {"id": 12692, "dom": "D", "ent": "bile duct disease", "cls": "mental"}, {"id": 12693, "dom": "D", "ent": "bile duct disease", "cls": "genetic"}, {"id": 12694, "dom": "D", "ent": "bile duct disease", "cls": "bact"}, {"id": 12695, "dom": "D", "ent": "bile duct disease", "cls": "viral"}, {"id": 12696, "dom": "D", "ent": "bile duct disease", "cls": "fungal"}, {"id": 12697, "dom": "D", "ent": "bile duct disease", "cls": "parasit"}, {"id": 12698, "dom": "D", "ent": "skin cancer", "cls": "cardio"}, {"id": 12699, "dom": "D", "ent": "skin cancer", "cls": "resp"}, {"id": 12700, "dom": "D", "ent": "skin cancer", "cls": "gi"}, {"id": 12701, "dom": "D", "ent": "skin cancer", "cls": "nerv"}, {"id": 12702, "dom": "D", "ent": "skin cancer", "cls": "skin"}, {"id": 12703, "dom": "D", "ent": "skin cancer", "cls": "musc"}, {"id": 12704, "dom": "D", "ent": "skin cancer", "cls": "urin"}, {"id": 12705, "dom": "D", "ent": "skin cancer", "cls": "endo"}, {"id": 12706, "dom": "D", "ent": "skin cancer", "cls": "immune"}, {"id": 12707, "dom": "D", "ent": "skin cancer", "cls": "blood"}, {"id": 12708, "dom": "D", "ent": "skin cancer", "cls": "repro"}, {"id": 12709, "dom": "D", "ent": "skin cancer", "cls": "breast"}, {"id": 12710, "dom": "D", "ent": "skin cancer", "cls": "chest"}, {"id": 12711, "dom": "D", "ent": "skin cancer", "cls": "metab"}, {"id": 12712, "dom": "D", "ent": "skin cancer", "cls": "infect"}, {"id": 12713, "dom": "D", "ent": "skin cancer", "cls": "tumor"}, {"id": 12714, "dom": "D", "ent": "skin cancer", "cls": "mental"}, {"id": 12715, "dom": "D", "ent": "skin cancer", "cls": "genetic"}, {"id": 12716, "dom": "D", "ent": "skin cancer", "cls": "bact"}, {"id": 12717, "dom": "D", "ent": "skin cancer", "cls": "viral"}, {"id": 12718, "dom": "D", "ent": "skin cancer", "cls": "fungal"}, {"id": 12719, "dom": "D", "ent": "skin cancer", "cls": "parasit"}, {"id": 12720, "dom": "D", "ent": "Alexander disease", "cls": "cardio"}, {"id": 12721, "dom": "D", "ent": "Alexander disease", "cls": "resp"}, {"id": 12722, "dom": "D", "ent": "Alexander disease", "cls": "gi"}, {"id": 12723, "dom": "D", "ent": "Alexander disease", "cls": "nerv"}, {"id": 12724, "dom": "D", "ent": "Alexander disease", "cls": "skin"}, {"id": 12725, "dom": "D", "ent": "Alexander disease", "cls": "musc"}, {"id": 12726, "dom": "D", "ent": "Alexander disease", "cls": "urin"}, {"id": 12727, "dom": "D", "ent": "Alexander disease", "cls": "endo"}, {"id": 12728, "dom": "D", "ent": "Alexander disease", "cls": "immune"}, {"id": 12729, "dom": "D", "ent": "Alexander disease", "cls": "blood"}, {"id": 12730, "dom": "D", "ent": "Alexander disease", "cls": "repro"}, {"id": 12731, "dom": "D", "ent": "Alexander disease", "cls": "breast"}, {"id": 12732, "dom": "D", "ent": "Alexander disease", "cls": "chest"}, {"id": 12733, "dom": "D", "ent": "Alexander disease", "cls": "metab"}, {"id": 12734, "dom": "D", "ent": "Alexander disease", "cls": "infect"}, {"id": 12735, "dom": "D", "ent": "Alexander disease", "cls": "tumor"}, {"id": 12736, "dom": "D", "ent": "Alexander disease", "cls": "mental"}, {"id": 12737, "dom": "D", "ent": "Alexander disease", "cls": "genetic"}, {"id": 12738, "dom": "D", "ent": "Alexander disease", "cls": "bact"}, {"id": 12739, "dom": "D", "ent": "Alexander disease", "cls": "viral"}, {"id": 12740, "dom": "D", "ent": "Alexander disease", "cls": "fungal"}, {"id": 12741, "dom": "D", "ent": "Alexander disease", "cls": "parasit"}, {"id": 12742, "dom": "D", "ent": "external ear carcinoma", "cls": "cardio"}, {"id": 12743, "dom": "D", "ent": "external ear carcinoma", "cls": "resp"}, {"id": 12744, "dom": "D", "ent": "external ear carcinoma", "cls": "gi"}, {"id": 12745, "dom": "D", "ent": "external ear carcinoma", "cls": "nerv"}, {"id": 12746, "dom": "D", "ent": "external ear carcinoma", "cls": "skin"}, {"id": 12747, "dom": "D", "ent": "external ear carcinoma", "cls": "musc"}, {"id": 12748, "dom": "D", "ent": "external ear carcinoma", "cls": "urin"}, {"id": 12749, "dom": "D", "ent": "external ear carcinoma", "cls": "endo"}, {"id": 12750, "dom": "D", "ent": "external ear carcinoma", "cls": "immune"}, {"id": 12751, "dom": "D", "ent": "external ear carcinoma", "cls": "blood"}, {"id": 12752, "dom": "D", "ent": "external ear carcinoma", "cls": "repro"}, {"id": 12753, "dom": "D", "ent": "external ear carcinoma", "cls": "breast"}, {"id": 12754, "dom": "D", "ent": "external ear carcinoma", "cls": "chest"}, {"id": 12755, "dom": "D", "ent": "external ear carcinoma", "cls": "metab"}, {"id": 12756, "dom": "D", "ent": "external ear carcinoma", "cls": "infect"}, {"id": 12757, "dom": "D", "ent": "external ear carcinoma", "cls": "tumor"}, {"id": 12758, "dom": "D", "ent": "external ear carcinoma", "cls": "mental"}, {"id": 12759, "dom": "D", "ent": "external ear carcinoma", "cls": "genetic"}, {"id": 12760, "dom": "D", "ent": "external ear carcinoma", "cls": "bact"}, {"id": 12761, "dom": "D", "ent": "external ear carcinoma", "cls": "viral"}, {"id": 12762, "dom": "D", "ent": "external ear carcinoma", "cls": "fungal"}, {"id": 12763, "dom": "D", "ent": "external ear carcinoma", "cls": "parasit"}, {"id": 12764, "dom": "D", "ent": "smooth muscle tumor", "cls": "cardio"}, {"id": 12765, "dom": "D", "ent": "smooth muscle tumor", "cls": "resp"}, {"id": 12766, "dom": "D", "ent": "smooth muscle tumor", "cls": "gi"}, {"id": 12767, "dom": "D", "ent": "smooth muscle tumor", "cls": "nerv"}, {"id": 12768, "dom": "D", "ent": "smooth muscle tumor", "cls": "skin"}, {"id": 12769, "dom": "D", "ent": "smooth muscle tumor", "cls": "musc"}, {"id": 12770, "dom": "D", "ent": "smooth muscle tumor", "cls": "urin"}, {"id": 12771, "dom": "D", "ent": "smooth muscle tumor", "cls": "endo"}, {"id": 12772, "dom": "D", "ent": "smooth muscle tumor", "cls": "immune"}, {"id": 12773, "dom": "D", "ent": "smooth muscle tumor", "cls": "blood"}, {"id": 12774, "dom": "D", "ent": "smooth muscle tumor", "cls": "repro"}, {"id": 12775, "dom": "D", "ent": "smooth muscle tumor", "cls": "breast"}, {"id": 12776, "dom": "D", "ent": "smooth muscle tumor", "cls": "chest"}, {"id": 12777, "dom": "D", "ent": "smooth muscle tumor", "cls": "metab"}, {"id": 12778, "dom": "D", "ent": "smooth muscle tumor", "cls": "infect"}, {"id": 12779, "dom": "D", "ent": "smooth muscle tumor", "cls": "tumor"}, {"id": 12780, "dom": "D", "ent": "smooth muscle tumor", "cls": "mental"}, {"id": 12781, "dom": "D", "ent": "smooth muscle tumor", "cls": "genetic"}, {"id": 12782, "dom": "D", "ent": "smooth muscle tumor", "cls": "bact"}, {"id": 12783, "dom": "D", "ent": "smooth muscle tumor", "cls": "viral"}, {"id": 12784, "dom": "D", "ent": "smooth muscle tumor", "cls": "fungal"}, {"id": 12785, "dom": "D", "ent": "smooth muscle tumor", "cls": "parasit"}, {"id": 12786, "dom": "D", "ent": "retinal cancer", "cls": "cardio"}, {"id": 12787, "dom": "D", "ent": "retinal cancer", "cls": "resp"}, {"id": 12788, "dom": "D", "ent": "retinal cancer", "cls": "gi"}, {"id": 12789, "dom": "D", "ent": "retinal cancer", "cls": "nerv"}, {"id": 12790, "dom": "D", "ent": "retinal cancer", "cls": "skin"}, {"id": 12791, "dom": "D", "ent": "retinal cancer", "cls": "musc"}, {"id": 12792, "dom": "D", "ent": "retinal cancer", "cls": "urin"}, {"id": 12793, "dom": "D", "ent": "retinal cancer", "cls": "endo"}, {"id": 12794, "dom": "D", "ent": "retinal cancer", "cls": "immune"}, {"id": 12795, "dom": "D", "ent": "retinal cancer", "cls": "blood"}, {"id": 12796, "dom": "D", "ent": "retinal cancer", "cls": "repro"}, {"id": 12797, "dom": "D", "ent": "retinal cancer", "cls": "breast"}, {"id": 12798, "dom": "D", "ent": "retinal cancer", "cls": "chest"}, {"id": 12799, "dom": "D", "ent": "retinal cancer", "cls": "metab"}, {"id": 12800, "dom": "D", "ent": "retinal cancer", "cls": "infect"}, {"id": 12801, "dom": "D", "ent": "retinal cancer", "cls": "tumor"}, {"id": 12802, "dom": "D", "ent": "retinal cancer", "cls": "mental"}, {"id": 12803, "dom": "D", "ent": "retinal cancer", "cls": "genetic"}, {"id": 12804, "dom": "D", "ent": "retinal cancer", "cls": "bact"}, {"id": 12805, "dom": "D", "ent": "retinal cancer", "cls": "viral"}, {"id": 12806, "dom": "D", "ent": "retinal cancer", "cls": "fungal"}, {"id": 12807, "dom": "D", "ent": "retinal cancer", "cls": "parasit"}, {"id": 12808, "dom": "D", "ent": "Nelson syndrome", "cls": "cardio"}, {"id": 12809, "dom": "D", "ent": "Nelson syndrome", "cls": "resp"}, {"id": 12810, "dom": "D", "ent": "Nelson syndrome", "cls": "gi"}, {"id": 12811, "dom": "D", "ent": "Nelson syndrome", "cls": "nerv"}, {"id": 12812, "dom": "D", "ent": "Nelson syndrome", "cls": "skin"}, {"id": 12813, "dom": "D", "ent": "Nelson syndrome", "cls": "musc"}, {"id": 12814, "dom": "D", "ent": "Nelson syndrome", "cls": "urin"}, {"id": 12815, "dom": "D", "ent": "Nelson syndrome", "cls": "endo"}, {"id": 12816, "dom": "D", "ent": "Nelson syndrome", "cls": "immune"}, {"id": 12817, "dom": "D", "ent": "Nelson syndrome", "cls": "blood"}, {"id": 12818, "dom": "D", "ent": "Nelson syndrome", "cls": "repro"}, {"id": 12819, "dom": "D", "ent": "Nelson syndrome", "cls": "breast"}, {"id": 12820, "dom": "D", "ent": "Nelson syndrome", "cls": "chest"}, {"id": 12821, "dom": "D", "ent": "Nelson syndrome", "cls": "metab"}, {"id": 12822, "dom": "D", "ent": "Nelson syndrome", "cls": "infect"}, {"id": 12823, "dom": "D", "ent": "Nelson syndrome", "cls": "tumor"}, {"id": 12824, "dom": "D", "ent": "Nelson syndrome", "cls": "mental"}, {"id": 12825, "dom": "D", "ent": "Nelson syndrome", "cls": "genetic"}, {"id": 12826, "dom": "D", "ent": "Nelson syndrome", "cls": "bact"}, {"id": 12827, "dom": "D", "ent": "Nelson syndrome", "cls": "viral"}, {"id": 12828, "dom": "D", "ent": "Nelson syndrome", "cls": "fungal"}, {"id": 12829, "dom": "D", "ent": "Nelson syndrome", "cls": "parasit"}, {"id": 12830, "dom": "D", "ent": "thoracic cancer", "cls": "cardio"}, {"id": 12831, "dom": "D", "ent": "thoracic cancer", "cls": "resp"}, {"id": 12832, "dom": "D", "ent": "thoracic cancer", "cls": "gi"}, {"id": 12833, "dom": "D", "ent": "thoracic cancer", "cls": "nerv"}, {"id": 12834, "dom": "D", "ent": "thoracic cancer", "cls": "skin"}, {"id": 12835, "dom": "D", "ent": "thoracic cancer", "cls": "musc"}, {"id": 12836, "dom": "D", "ent": "thoracic cancer", "cls": "urin"}, {"id": 12837, "dom": "D", "ent": "thoracic cancer", "cls": "endo"}, {"id": 12838, "dom": "D", "ent": "thoracic cancer", "cls": "immune"}, {"id": 12839, "dom": "D", "ent": "thoracic cancer", "cls": "blood"}, {"id": 12840, "dom": "D", "ent": "thoracic cancer", "cls": "repro"}, {"id": 12841, "dom": "D", "ent": "thoracic cancer", "cls": "breast"}, {"id": 12842, "dom": "D", "ent": "thoracic cancer", "cls": "chest"}, {"id": 12843, "dom": "D", "ent": "thoracic cancer", "cls": "metab"}, {"id": 12844, "dom": "D", "ent": "thoracic cancer", "cls": "infect"}, {"id": 12845, "dom": "D", "ent": "thoracic cancer", "cls": "tumor"}, {"id": 12846, "dom": "D", "ent": "thoracic cancer", "cls": "mental"}, {"id": 12847, "dom": "D", "ent": "thoracic cancer", "cls": "genetic"}, {"id": 12848, "dom": "D", "ent": "thoracic cancer", "cls": "bact"}, {"id": 12849, "dom": "D", "ent": "thoracic cancer", "cls": "viral"}, {"id": 12850, "dom": "D", "ent": "thoracic cancer", "cls": "fungal"}, {"id": 12851, "dom": "D", "ent": "thoracic cancer", "cls": "parasit"}, {"id": 12852, "dom": "D", "ent": "ear cancer", "cls": "cardio"}, {"id": 12853, "dom": "D", "ent": "ear cancer", "cls": "resp"}, {"id": 12854, "dom": "D", "ent": "ear cancer", "cls": "gi"}, {"id": 12855, "dom": "D", "ent": "ear cancer", "cls": "nerv"}, {"id": 12856, "dom": "D", "ent": "ear cancer", "cls": "skin"}, {"id": 12857, "dom": "D", "ent": "ear cancer", "cls": "musc"}, {"id": 12858, "dom": "D", "ent": "ear cancer", "cls": "urin"}, {"id": 12859, "dom": "D", "ent": "ear cancer", "cls": "endo"}, {"id": 12860, "dom": "D", "ent": "ear cancer", "cls": "immune"}, {"id": 12861, "dom": "D", "ent": "ear cancer", "cls": "blood"}, {"id": 12862, "dom": "D", "ent": "ear cancer", "cls": "repro"}, {"id": 12863, "dom": "D", "ent": "ear cancer", "cls": "breast"}, {"id": 12864, "dom": "D", "ent": "ear cancer", "cls": "chest"}, {"id": 12865, "dom": "D", "ent": "ear cancer", "cls": "metab"}, {"id": 12866, "dom": "D", "ent": "ear cancer", "cls": "infect"}, {"id": 12867, "dom": "D", "ent": "ear cancer", "cls": "tumor"}, {"id": 12868, "dom": "D", "ent": "ear cancer", "cls": "mental"}, {"id": 12869, "dom": "D", "ent": "ear cancer", "cls": "genetic"}, {"id": 12870, "dom": "D", "ent": "ear cancer", "cls": "bact"}, {"id": 12871, "dom": "D", "ent": "ear cancer", "cls": "viral"}, {"id": 12872, "dom": "D", "ent": "ear cancer", "cls": "fungal"}, {"id": 12873, "dom": "D", "ent": "ear cancer", "cls": "parasit"}, {"id": 12874, "dom": "D", "ent": "inner ear cancer", "cls": "cardio"}, {"id": 12875, "dom": "D", "ent": "inner ear cancer", "cls": "resp"}, {"id": 12876, "dom": "D", "ent": "inner ear cancer", "cls": "gi"}, {"id": 12877, "dom": "D", "ent": "inner ear cancer", "cls": "nerv"}, {"id": 12878, "dom": "D", "ent": "inner ear cancer", "cls": "skin"}, {"id": 12879, "dom": "D", "ent": "inner ear cancer", "cls": "musc"}, {"id": 12880, "dom": "D", "ent": "inner ear cancer", "cls": "urin"}, {"id": 12881, "dom": "D", "ent": "inner ear cancer", "cls": "endo"}, {"id": 12882, "dom": "D", "ent": "inner ear cancer", "cls": "immune"}, {"id": 12883, "dom": "D", "ent": "inner ear cancer", "cls": "blood"}, {"id": 12884, "dom": "D", "ent": "inner ear cancer", "cls": "repro"}, {"id": 12885, "dom": "D", "ent": "inner ear cancer", "cls": "breast"}, {"id": 12886, "dom": "D", "ent": "inner ear cancer", "cls": "chest"}, {"id": 12887, "dom": "D", "ent": "inner ear cancer", "cls": "metab"}, {"id": 12888, "dom": "D", "ent": "inner ear cancer", "cls": "infect"}, {"id": 12889, "dom": "D", "ent": "inner ear cancer", "cls": "tumor"}, {"id": 12890, "dom": "D", "ent": "inner ear cancer", "cls": "mental"}, {"id": 12891, "dom": "D", "ent": "inner ear cancer", "cls": "genetic"}, {"id": 12892, "dom": "D", "ent": "inner ear cancer", "cls": "bact"}, {"id": 12893, "dom": "D", "ent": "inner ear cancer", "cls": "viral"}, {"id": 12894, "dom": "D", "ent": "inner ear cancer", "cls": "fungal"}, {"id": 12895, "dom": "D", "ent": "inner ear cancer", "cls": "parasit"}, {"id": 12896, "dom": "D", "ent": "simple partial epilepsy", "cls": "cardio"}, {"id": 12897, "dom": "D", "ent": "simple partial epilepsy", "cls": "resp"}, {"id": 12898, "dom": "D", "ent": "simple partial epilepsy", "cls": "gi"}, {"id": 12899, "dom": "D", "ent": "simple partial epilepsy", "cls": "nerv"}, {"id": 12900, "dom": "D", "ent": "simple partial epilepsy", "cls": "skin"}, {"id": 12901, "dom": "D", "ent": "simple partial epilepsy", "cls": "musc"}, {"id": 12902, "dom": "D", "ent": "simple partial epilepsy", "cls": "urin"}, {"id": 12903, "dom": "D", "ent": "simple partial epilepsy", "cls": "endo"}, {"id": 12904, "dom": "D", "ent": "simple partial epilepsy", "cls": "immune"}, {"id": 12905, "dom": "D", "ent": "simple partial epilepsy", "cls": "blood"}, {"id": 12906, "dom": "D", "ent": "simple partial epilepsy", "cls": "repro"}, {"id": 12907, "dom": "D", "ent": "simple partial epilepsy", "cls": "breast"}, {"id": 12908, "dom": "D", "ent": "simple partial epilepsy", "cls": "chest"}, {"id": 12909, "dom": "D", "ent": "simple partial epilepsy", "cls": "metab"}, {"id": 12910, "dom": "D", "ent": "simple partial epilepsy", "cls": "infect"}, {"id": 12911, "dom": "D", "ent": "simple partial epilepsy", "cls": "tumor"}, {"id": 12912, "dom": "D", "ent": "simple partial epilepsy", "cls": "mental"}, {"id": 12913, "dom": "D", "ent": "simple partial epilepsy", "cls": "genetic"}, {"id": 12914, "dom": "D", "ent": "simple partial epilepsy", "cls": "bact"}, {"id": 12915, "dom": "D", "ent": "simple partial epilepsy", "cls": "viral"}, {"id": 12916, "dom": "D", "ent": "simple partial epilepsy", "cls": "fungal"}, {"id": 12917, "dom": "D", "ent": "simple partial epilepsy", "cls": "parasit"}, {"id": 12918, "dom": "D", "ent": "infertility", "cls": "cardio"}, {"id": 12919, "dom": "D", "ent": "infertility", "cls": "resp"}, {"id": 12920, "dom": "D", "ent": "infertility", "cls": "gi"}, {"id": 12921, "dom": "D", "ent": "infertility", "cls": "nerv"}, {"id": 12922, "dom": "D", "ent": "infertility", "cls": "skin"}, {"id": 12923, "dom": "D", "ent": "infertility", "cls": "musc"}, {"id": 12924, "dom": "D", "ent": "infertility", "cls": "urin"}, {"id": 12925, "dom": "D", "ent": "infertility", "cls": "endo"}, {"id": 12926, "dom": "D", "ent": "infertility", "cls": "immune"}, {"id": 12927, "dom": "D", "ent": "infertility", "cls": "blood"}, {"id": 12928, "dom": "D", "ent": "infertility", "cls": "repro"}, {"id": 12929, "dom": "D", "ent": "infertility", "cls": "breast"}, {"id": 12930, "dom": "D", "ent": "infertility", "cls": "chest"}, {"id": 12931, "dom": "D", "ent": "infertility", "cls": "metab"}, {"id": 12932, "dom": "D", "ent": "infertility", "cls": "infect"}, {"id": 12933, "dom": "D", "ent": "infertility", "cls": "tumor"}, {"id": 12934, "dom": "D", "ent": "infertility", "cls": "mental"}, {"id": 12935, "dom": "D", "ent": "infertility", "cls": "genetic"}, {"id": 12936, "dom": "D", "ent": "infertility", "cls": "bact"}, {"id": 12937, "dom": "D", "ent": "infertility", "cls": "viral"}, {"id": 12938, "dom": "D", "ent": "infertility", "cls": "fungal"}, {"id": 12939, "dom": "D", "ent": "infertility", "cls": "parasit"}, {"id": 12940, "dom": "D", "ent": "inhibited male orgasm", "cls": "cardio"}, {"id": 12941, "dom": "D", "ent": "inhibited male orgasm", "cls": "resp"}, {"id": 12942, "dom": "D", "ent": "inhibited male orgasm", "cls": "gi"}, {"id": 12943, "dom": "D", "ent": "inhibited male orgasm", "cls": "nerv"}, {"id": 12944, "dom": "D", "ent": "inhibited male orgasm", "cls": "skin"}, {"id": 12945, "dom": "D", "ent": "inhibited male orgasm", "cls": "musc"}, {"id": 12946, "dom": "D", "ent": "inhibited male orgasm", "cls": "urin"}, {"id": 12947, "dom": "D", "ent": "inhibited male orgasm", "cls": "endo"}, {"id": 12948, "dom": "D", "ent": "inhibited male orgasm", "cls": "immune"}, {"id": 12949, "dom": "D", "ent": "inhibited male orgasm", "cls": "blood"}, {"id": 12950, "dom": "D", "ent": "inhibited male orgasm", "cls": "repro"}, {"id": 12951, "dom": "D", "ent": "inhibited male orgasm", "cls": "breast"}, {"id": 12952, "dom": "D", "ent": "inhibited male orgasm", "cls": "chest"}, {"id": 12953, "dom": "D", "ent": "inhibited male orgasm", "cls": "metab"}, {"id": 12954, "dom": "D", "ent": "inhibited male orgasm", "cls": "infect"}, {"id": 12955, "dom": "D", "ent": "inhibited male orgasm", "cls": "tumor"}, {"id": 12956, "dom": "D", "ent": "inhibited male orgasm", "cls": "mental"}, {"id": 12957, "dom": "D", "ent": "inhibited male orgasm", "cls": "genetic"}, {"id": 12958, "dom": "D", "ent": "inhibited male orgasm", "cls": "bact"}, {"id": 12959, "dom": "D", "ent": "inhibited male orgasm", "cls": "viral"}, {"id": 12960, "dom": "D", "ent": "inhibited male orgasm", "cls": "fungal"}, {"id": 12961, "dom": "D", "ent": "inhibited male orgasm", "cls": "parasit"}, {"id": 12962, "dom": "D", "ent": "external ear cancer", "cls": "cardio"}, {"id": 12963, "dom": "D", "ent": "external ear cancer", "cls": "resp"}, {"id": 12964, "dom": "D", "ent": "external ear cancer", "cls": "gi"}, {"id": 12965, "dom": "D", "ent": "external ear cancer", "cls": "nerv"}, {"id": 12966, "dom": "D", "ent": "external ear cancer", "cls": "skin"}, {"id": 12967, "dom": "D", "ent": "external ear cancer", "cls": "musc"}, {"id": 12968, "dom": "D", "ent": "external ear cancer", "cls": "urin"}, {"id": 12969, "dom": "D", "ent": "external ear cancer", "cls": "endo"}, {"id": 12970, "dom": "D", "ent": "external ear cancer", "cls": "immune"}, {"id": 12971, "dom": "D", "ent": "external ear cancer", "cls": "blood"}, {"id": 12972, "dom": "D", "ent": "external ear cancer", "cls": "repro"}, {"id": 12973, "dom": "D", "ent": "external ear cancer", "cls": "breast"}, {"id": 12974, "dom": "D", "ent": "external ear cancer", "cls": "chest"}, {"id": 12975, "dom": "D", "ent": "external ear cancer", "cls": "metab"}, {"id": 12976, "dom": "D", "ent": "external ear cancer", "cls": "infect"}, {"id": 12977, "dom": "D", "ent": "external ear cancer", "cls": "tumor"}, {"id": 12978, "dom": "D", "ent": "external ear cancer", "cls": "mental"}, {"id": 12979, "dom": "D", "ent": "external ear cancer", "cls": "genetic"}, {"id": 12980, "dom": "D", "ent": "external ear cancer", "cls": "bact"}, {"id": 12981, "dom": "D", "ent": "external ear cancer", "cls": "viral"}, {"id": 12982, "dom": "D", "ent": "external ear cancer", "cls": "fungal"}, {"id": 12983, "dom": "D", "ent": "external ear cancer", "cls": "parasit"}, {"id": 12984, "dom": "D", "ent": "pulmonary valve disease", "cls": "cardio"}, {"id": 12985, "dom": "D", "ent": "pulmonary valve disease", "cls": "resp"}, {"id": 12986, "dom": "D", "ent": "pulmonary valve disease", "cls": "gi"}, {"id": 12987, "dom": "D", "ent": "pulmonary valve disease", "cls": "nerv"}, {"id": 12988, "dom": "D", "ent": "pulmonary valve disease", "cls": "skin"}, {"id": 12989, "dom": "D", "ent": "pulmonary valve disease", "cls": "musc"}, {"id": 12990, "dom": "D", "ent": "pulmonary valve disease", "cls": "urin"}, {"id": 12991, "dom": "D", "ent": "pulmonary valve disease", "cls": "endo"}, {"id": 12992, "dom": "D", "ent": "pulmonary valve disease", "cls": "immune"}, {"id": 12993, "dom": "D", "ent": "pulmonary valve disease", "cls": "blood"}, {"id": 12994, "dom": "D", "ent": "pulmonary valve disease", "cls": "repro"}, {"id": 12995, "dom": "D", "ent": "pulmonary valve disease", "cls": "breast"}, {"id": 12996, "dom": "D", "ent": "pulmonary valve disease", "cls": "chest"}, {"id": 12997, "dom": "D", "ent": "pulmonary valve disease", "cls": "metab"}, {"id": 12998, "dom": "D", "ent": "pulmonary valve disease", "cls": "infect"}, {"id": 12999, "dom": "D", "ent": "pulmonary valve disease", "cls": "tumor"}, {"id": 13000, "dom": "D", "ent": "pulmonary valve disease", "cls": "mental"}, {"id": 13001, "dom": "D", "ent": "pulmonary valve disease", "cls": "genetic"}, {"id": 13002, "dom": "D", "ent": "pulmonary valve disease", "cls": "bact"}, {"id": 13003, "dom": "D", "ent": "pulmonary valve disease", "cls": "viral"}, {"id": 13004, "dom": "D", "ent": "pulmonary valve disease", "cls": "fungal"}, {"id": 13005, "dom": "D", "ent": "pulmonary valve disease", "cls": "parasit"}, {"id": 13006, "dom": "D", "ent": "composite lymphoma", "cls": "cardio"}, {"id": 13007, "dom": "D", "ent": "composite lymphoma", "cls": "resp"}, {"id": 13008, "dom": "D", "ent": "composite lymphoma", "cls": "gi"}, {"id": 13009, "dom": "D", "ent": "composite lymphoma", "cls": "nerv"}, {"id": 13010, "dom": "D", "ent": "composite lymphoma", "cls": "skin"}, {"id": 13011, "dom": "D", "ent": "composite lymphoma", "cls": "musc"}, {"id": 13012, "dom": "D", "ent": "composite lymphoma", "cls": "urin"}, {"id": 13013, "dom": "D", "ent": "composite lymphoma", "cls": "endo"}, {"id": 13014, "dom": "D", "ent": "composite lymphoma", "cls": "immune"}, {"id": 13015, "dom": "D", "ent": "composite lymphoma", "cls": "blood"}, {"id": 13016, "dom": "D", "ent": "composite lymphoma", "cls": "repro"}, {"id": 13017, "dom": "D", "ent": "composite lymphoma", "cls": "breast"}, {"id": 13018, "dom": "D", "ent": "composite lymphoma", "cls": "chest"}, {"id": 13019, "dom": "D", "ent": "composite lymphoma", "cls": "metab"}, {"id": 13020, "dom": "D", "ent": "composite lymphoma", "cls": "infect"}, {"id": 13021, "dom": "D", "ent": "composite lymphoma", "cls": "tumor"}, {"id": 13022, "dom": "D", "ent": "composite lymphoma", "cls": "mental"}, {"id": 13023, "dom": "D", "ent": "composite lymphoma", "cls": "genetic"}, {"id": 13024, "dom": "D", "ent": "composite lymphoma", "cls": "bact"}, {"id": 13025, "dom": "D", "ent": "composite lymphoma", "cls": "viral"}, {"id": 13026, "dom": "D", "ent": "composite lymphoma", "cls": "fungal"}, {"id": 13027, "dom": "D", "ent": "composite lymphoma", "cls": "parasit"}, {"id": 13028, "dom": "D", "ent": "adult lymphoma", "cls": "cardio"}, {"id": 13029, "dom": "D", "ent": "adult lymphoma", "cls": "resp"}, {"id": 13030, "dom": "D", "ent": "adult lymphoma", "cls": "gi"}, {"id": 13031, "dom": "D", "ent": "adult lymphoma", "cls": "nerv"}, {"id": 13032, "dom": "D", "ent": "adult lymphoma", "cls": "skin"}, {"id": 13033, "dom": "D", "ent": "adult lymphoma", "cls": "musc"}, {"id": 13034, "dom": "D", "ent": "adult lymphoma", "cls": "urin"}, {"id": 13035, "dom": "D", "ent": "adult lymphoma", "cls": "endo"}, {"id": 13036, "dom": "D", "ent": "adult lymphoma", "cls": "immune"}, {"id": 13037, "dom": "D", "ent": "adult lymphoma", "cls": "blood"}, {"id": 13038, "dom": "D", "ent": "adult lymphoma", "cls": "repro"}, {"id": 13039, "dom": "D", "ent": "adult lymphoma", "cls": "breast"}, {"id": 13040, "dom": "D", "ent": "adult lymphoma", "cls": "chest"}, {"id": 13041, "dom": "D", "ent": "adult lymphoma", "cls": "metab"}, {"id": 13042, "dom": "D", "ent": "adult lymphoma", "cls": "infect"}, {"id": 13043, "dom": "D", "ent": "adult lymphoma", "cls": "tumor"}, {"id": 13044, "dom": "D", "ent": "adult lymphoma", "cls": "mental"}, {"id": 13045, "dom": "D", "ent": "adult lymphoma", "cls": "genetic"}, {"id": 13046, "dom": "D", "ent": "adult lymphoma", "cls": "bact"}, {"id": 13047, "dom": "D", "ent": "adult lymphoma", "cls": "viral"}, {"id": 13048, "dom": "D", "ent": "adult lymphoma", "cls": "fungal"}, {"id": 13049, "dom": "D", "ent": "adult lymphoma", "cls": "parasit"}, {"id": 13050, "dom": "D", "ent": "benign vascular tumor", "cls": "cardio"}, {"id": 13051, "dom": "D", "ent": "benign vascular tumor", "cls": "resp"}, {"id": 13052, "dom": "D", "ent": "benign vascular tumor", "cls": "gi"}, {"id": 13053, "dom": "D", "ent": "benign vascular tumor", "cls": "nerv"}, {"id": 13054, "dom": "D", "ent": "benign vascular tumor", "cls": "skin"}, {"id": 13055, "dom": "D", "ent": "benign vascular tumor", "cls": "musc"}, {"id": 13056, "dom": "D", "ent": "benign vascular tumor", "cls": "urin"}, {"id": 13057, "dom": "D", "ent": "benign vascular tumor", "cls": "endo"}, {"id": 13058, "dom": "D", "ent": "benign vascular tumor", "cls": "immune"}, {"id": 13059, "dom": "D", "ent": "benign vascular tumor", "cls": "blood"}, {"id": 13060, "dom": "D", "ent": "benign vascular tumor", "cls": "repro"}, {"id": 13061, "dom": "D", "ent": "benign vascular tumor", "cls": "breast"}, {"id": 13062, "dom": "D", "ent": "benign vascular tumor", "cls": "chest"}, {"id": 13063, "dom": "D", "ent": "benign vascular tumor", "cls": "metab"}, {"id": 13064, "dom": "D", "ent": "benign vascular tumor", "cls": "infect"}, {"id": 13065, "dom": "D", "ent": "benign vascular tumor", "cls": "tumor"}, {"id": 13066, "dom": "D", "ent": "benign vascular tumor", "cls": "mental"}, {"id": 13067, "dom": "D", "ent": "benign vascular tumor", "cls": "genetic"}, {"id": 13068, "dom": "D", "ent": "benign vascular tumor", "cls": "bact"}, {"id": 13069, "dom": "D", "ent": "benign vascular tumor", "cls": "viral"}, {"id": 13070, "dom": "D", "ent": "benign vascular tumor", "cls": "fungal"}, {"id": 13071, "dom": "D", "ent": "benign vascular tumor", "cls": "parasit"}, {"id": 13072, "dom": "D", "ent": "reactive arthritis", "cls": "cardio"}, {"id": 13073, "dom": "D", "ent": "reactive arthritis", "cls": "resp"}, {"id": 13074, "dom": "D", "ent": "reactive arthritis", "cls": "gi"}, {"id": 13075, "dom": "D", "ent": "reactive arthritis", "cls": "nerv"}, {"id": 13076, "dom": "D", "ent": "reactive arthritis", "cls": "skin"}, {"id": 13077, "dom": "D", "ent": "reactive arthritis", "cls": "musc"}, {"id": 13078, "dom": "D", "ent": "reactive arthritis", "cls": "urin"}, {"id": 13079, "dom": "D", "ent": "reactive arthritis", "cls": "endo"}, {"id": 13080, "dom": "D", "ent": "reactive arthritis", "cls": "immune"}, {"id": 13081, "dom": "D", "ent": "reactive arthritis", "cls": "blood"}, {"id": 13082, "dom": "D", "ent": "reactive arthritis", "cls": "repro"}, {"id": 13083, "dom": "D", "ent": "reactive arthritis", "cls": "breast"}, {"id": 13084, "dom": "D", "ent": "reactive arthritis", "cls": "chest"}, {"id": 13085, "dom": "D", "ent": "reactive arthritis", "cls": "metab"}, {"id": 13086, "dom": "D", "ent": "reactive arthritis", "cls": "infect"}, {"id": 13087, "dom": "D", "ent": "reactive arthritis", "cls": "tumor"}, {"id": 13088, "dom": "D", "ent": "reactive arthritis", "cls": "mental"}, {"id": 13089, "dom": "D", "ent": "reactive arthritis", "cls": "genetic"}, {"id": 13090, "dom": "D", "ent": "reactive arthritis", "cls": "bact"}, {"id": 13091, "dom": "D", "ent": "reactive arthritis", "cls": "viral"}, {"id": 13092, "dom": "D", "ent": "reactive arthritis", "cls": "fungal"}, {"id": 13093, "dom": "D", "ent": "reactive arthritis", "cls": "parasit"}, {"id": 13094, "dom": "D", "ent": "acne", "cls": "cardio"}, {"id": 13095, "dom": "D", "ent": "acne", "cls": "resp"}, {"id": 13096, "dom": "D", "ent": "acne", "cls": "gi"}, {"id": 13097, "dom": "D", "ent": "acne", "cls": "nerv"}, {"id": 13098, "dom": "D", "ent": "acne", "cls": "skin"}, {"id": 13099, "dom": "D", "ent": "acne", "cls": "musc"}, {"id": 13100, "dom": "D", "ent": "acne", "cls": "urin"}, {"id": 13101, "dom": "D", "ent": "acne", "cls": "endo"}, {"id": 13102, "dom": "D", "ent": "acne", "cls": "immune"}, {"id": 13103, "dom": "D", "ent": "acne", "cls": "blood"}, {"id": 13104, "dom": "D", "ent": "acne", "cls": "repro"}, {"id": 13105, "dom": "D", "ent": "acne", "cls": "breast"}, {"id": 13106, "dom": "D", "ent": "acne", "cls": "chest"}, {"id": 13107, "dom": "D", "ent": "acne", "cls": "metab"}, {"id": 13108, "dom": "D", "ent": "acne", "cls": "infect"}, {"id": 13109, "dom": "D", "ent": "acne", "cls": "tumor"}, {"id": 13110, "dom": "D", "ent": "acne", "cls": "mental"}, {"id": 13111, "dom": "D", "ent": "acne", "cls": "genetic"}, {"id": 13112, "dom": "D", "ent": "acne", "cls": "bact"}, {"id": 13113, "dom": "D", "ent": "acne", "cls": "viral"}, {"id": 13114, "dom": "D", "ent": "acne", "cls": "fungal"}, {"id": 13115, "dom": "D", "ent": "acne", "cls": "parasit"}, {"id": 13116, "dom": "D", "ent": "heart lymphoma", "cls": "cardio"}, {"id": 13117, "dom": "D", "ent": "heart lymphoma", "cls": "resp"}, {"id": 13118, "dom": "D", "ent": "heart lymphoma", "cls": "gi"}, {"id": 13119, "dom": "D", "ent": "heart lymphoma", "cls": "nerv"}, {"id": 13120, "dom": "D", "ent": "heart lymphoma", "cls": "skin"}, {"id": 13121, "dom": "D", "ent": "heart lymphoma", "cls": "musc"}, {"id": 13122, "dom": "D", "ent": "heart lymphoma", "cls": "urin"}, {"id": 13123, "dom": "D", "ent": "heart lymphoma", "cls": "endo"}, {"id": 13124, "dom": "D", "ent": "heart lymphoma", "cls": "immune"}, {"id": 13125, "dom": "D", "ent": "heart lymphoma", "cls": "blood"}, {"id": 13126, "dom": "D", "ent": "heart lymphoma", "cls": "repro"}, {"id": 13127, "dom": "D", "ent": "heart lymphoma", "cls": "breast"}, {"id": 13128, "dom": "D", "ent": "heart lymphoma", "cls": "chest"}, {"id": 13129, "dom": "D", "ent": "heart lymphoma", "cls": "metab"}, {"id": 13130, "dom": "D", "ent": "heart lymphoma", "cls": "infect"}, {"id": 13131, "dom": "D", "ent": "heart lymphoma", "cls": "tumor"}, {"id": 13132, "dom": "D", "ent": "heart lymphoma", "cls": "mental"}, {"id": 13133, "dom": "D", "ent": "heart lymphoma", "cls": "genetic"}, {"id": 13134, "dom": "D", "ent": "heart lymphoma", "cls": "bact"}, {"id": 13135, "dom": "D", "ent": "heart lymphoma", "cls": "viral"}, {"id": 13136, "dom": "D", "ent": "heart lymphoma", "cls": "fungal"}, {"id": 13137, "dom": "D", "ent": "heart lymphoma", "cls": "parasit"}, {"id": 13138, "dom": "D", "ent": "chest wall lymphoma", "cls": "cardio"}, {"id": 13139, "dom": "D", "ent": "chest wall lymphoma", "cls": "resp"}, {"id": 13140, "dom": "D", "ent": "chest wall lymphoma", "cls": "gi"}, {"id": 13141, "dom": "D", "ent": "chest wall lymphoma", "cls": "nerv"}, {"id": 13142, "dom": "D", "ent": "chest wall lymphoma", "cls": "skin"}, {"id": 13143, "dom": "D", "ent": "chest wall lymphoma", "cls": "musc"}, {"id": 13144, "dom": "D", "ent": "chest wall lymphoma", "cls": "urin"}, {"id": 13145, "dom": "D", "ent": "chest wall lymphoma", "cls": "endo"}, {"id": 13146, "dom": "D", "ent": "chest wall lymphoma", "cls": "immune"}, {"id": 13147, "dom": "D", "ent": "chest wall lymphoma", "cls": "blood"}, {"id": 13148, "dom": "D", "ent": "chest wall lymphoma", "cls": "repro"}, {"id": 13149, "dom": "D", "ent": "chest wall lymphoma", "cls": "breast"}, {"id": 13150, "dom": "D", "ent": "chest wall lymphoma", "cls": "chest"}, {"id": 13151, "dom": "D", "ent": "chest wall lymphoma", "cls": "metab"}, {"id": 13152, "dom": "D", "ent": "chest wall lymphoma", "cls": "infect"}, {"id": 13153, "dom": "D", "ent": "chest wall lymphoma", "cls": "tumor"}, {"id": 13154, "dom": "D", "ent": "chest wall lymphoma", "cls": "mental"}, {"id": 13155, "dom": "D", "ent": "chest wall lymphoma", "cls": "genetic"}, {"id": 13156, "dom": "D", "ent": "chest wall lymphoma", "cls": "bact"}, {"id": 13157, "dom": "D", "ent": "chest wall lymphoma", "cls": "viral"}, {"id": 13158, "dom": "D", "ent": "chest wall lymphoma", "cls": "fungal"}, {"id": 13159, "dom": "D", "ent": "chest wall lymphoma", "cls": "parasit"}, {"id": 13160, "dom": "D", "ent": "lung lymphoma", "cls": "cardio"}, {"id": 13161, "dom": "D", "ent": "lung lymphoma", "cls": "resp"}, {"id": 13162, "dom": "D", "ent": "lung lymphoma", "cls": "gi"}, {"id": 13163, "dom": "D", "ent": "lung lymphoma", "cls": "nerv"}, {"id": 13164, "dom": "D", "ent": "lung lymphoma", "cls": "skin"}, {"id": 13165, "dom": "D", "ent": "lung lymphoma", "cls": "musc"}, {"id": 13166, "dom": "D", "ent": "lung lymphoma", "cls": "urin"}, {"id": 13167, "dom": "D", "ent": "lung lymphoma", "cls": "endo"}, {"id": 13168, "dom": "D", "ent": "lung lymphoma", "cls": "immune"}, {"id": 13169, "dom": "D", "ent": "lung lymphoma", "cls": "blood"}, {"id": 13170, "dom": "D", "ent": "lung lymphoma", "cls": "repro"}, {"id": 13171, "dom": "D", "ent": "lung lymphoma", "cls": "breast"}, {"id": 13172, "dom": "D", "ent": "lung lymphoma", "cls": "chest"}, {"id": 13173, "dom": "D", "ent": "lung lymphoma", "cls": "metab"}, {"id": 13174, "dom": "D", "ent": "lung lymphoma", "cls": "infect"}, {"id": 13175, "dom": "D", "ent": "lung lymphoma", "cls": "tumor"}, {"id": 13176, "dom": "D", "ent": "lung lymphoma", "cls": "mental"}, {"id": 13177, "dom": "D", "ent": "lung lymphoma", "cls": "genetic"}, {"id": 13178, "dom": "D", "ent": "lung lymphoma", "cls": "bact"}, {"id": 13179, "dom": "D", "ent": "lung lymphoma", "cls": "viral"}, {"id": 13180, "dom": "D", "ent": "lung lymphoma", "cls": "fungal"}, {"id": 13181, "dom": "D", "ent": "lung lymphoma", "cls": "parasit"}, {"id": 13182, "dom": "D", "ent": "eye lymphoma", "cls": "cardio"}, {"id": 13183, "dom": "D", "ent": "eye lymphoma", "cls": "resp"}, {"id": 13184, "dom": "D", "ent": "eye lymphoma", "cls": "gi"}, {"id": 13185, "dom": "D", "ent": "eye lymphoma", "cls": "nerv"}, {"id": 13186, "dom": "D", "ent": "eye lymphoma", "cls": "skin"}, {"id": 13187, "dom": "D", "ent": "eye lymphoma", "cls": "musc"}, {"id": 13188, "dom": "D", "ent": "eye lymphoma", "cls": "urin"}, {"id": 13189, "dom": "D", "ent": "eye lymphoma", "cls": "endo"}, {"id": 13190, "dom": "D", "ent": "eye lymphoma", "cls": "immune"}, {"id": 13191, "dom": "D", "ent": "eye lymphoma", "cls": "blood"}, {"id": 13192, "dom": "D", "ent": "eye lymphoma", "cls": "repro"}, {"id": 13193, "dom": "D", "ent": "eye lymphoma", "cls": "breast"}, {"id": 13194, "dom": "D", "ent": "eye lymphoma", "cls": "chest"}, {"id": 13195, "dom": "D", "ent": "eye lymphoma", "cls": "metab"}, {"id": 13196, "dom": "D", "ent": "eye lymphoma", "cls": "infect"}, {"id": 13197, "dom": "D", "ent": "eye lymphoma", "cls": "tumor"}, {"id": 13198, "dom": "D", "ent": "eye lymphoma", "cls": "mental"}, {"id": 13199, "dom": "D", "ent": "eye lymphoma", "cls": "genetic"}, {"id": 13200, "dom": "D", "ent": "eye lymphoma", "cls": "bact"}, {"id": 13201, "dom": "D", "ent": "eye lymphoma", "cls": "viral"}, {"id": 13202, "dom": "D", "ent": "eye lymphoma", "cls": "fungal"}, {"id": 13203, "dom": "D", "ent": "eye lymphoma", "cls": "parasit"}, {"id": 13204, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "cardio"}, {"id": 13205, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "resp"}, {"id": 13206, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "gi"}, {"id": 13207, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "nerv"}, {"id": 13208, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "skin"}, {"id": 13209, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "musc"}, {"id": 13210, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "urin"}, {"id": 13211, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "endo"}, {"id": 13212, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "immune"}, {"id": 13213, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "blood"}, {"id": 13214, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "repro"}, {"id": 13215, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "breast"}, {"id": 13216, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "chest"}, {"id": 13217, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "metab"}, {"id": 13218, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "infect"}, {"id": 13219, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "tumor"}, {"id": 13220, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "mental"}, {"id": 13221, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "genetic"}, {"id": 13222, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "bact"}, {"id": 13223, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "viral"}, {"id": 13224, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "fungal"}, {"id": 13225, "dom": "D", "ent": "mitochondrial metabolism disease", "cls": "parasit"}, {"id": 13226, "dom": "D", "ent": "B-cell lymphoma", "cls": "cardio"}, {"id": 13227, "dom": "D", "ent": "B-cell lymphoma", "cls": "resp"}, {"id": 13228, "dom": "D", "ent": "B-cell lymphoma", "cls": "gi"}, {"id": 13229, "dom": "D", "ent": "B-cell lymphoma", "cls": "nerv"}, {"id": 13230, "dom": "D", "ent": "B-cell lymphoma", "cls": "skin"}, {"id": 13231, "dom": "D", "ent": "B-cell lymphoma", "cls": "musc"}, {"id": 13232, "dom": "D", "ent": "B-cell lymphoma", "cls": "urin"}, {"id": 13233, "dom": "D", "ent": "B-cell lymphoma", "cls": "endo"}, {"id": 13234, "dom": "D", "ent": "B-cell lymphoma", "cls": "immune"}, {"id": 13235, "dom": "D", "ent": "B-cell lymphoma", "cls": "blood"}, {"id": 13236, "dom": "D", "ent": "B-cell lymphoma", "cls": "repro"}, {"id": 13237, "dom": "D", "ent": "B-cell lymphoma", "cls": "breast"}, {"id": 13238, "dom": "D", "ent": "B-cell lymphoma", "cls": "chest"}, {"id": 13239, "dom": "D", "ent": "B-cell lymphoma", "cls": "metab"}, {"id": 13240, "dom": "D", "ent": "B-cell lymphoma", "cls": "infect"}, {"id": 13241, "dom": "D", "ent": "B-cell lymphoma", "cls": "tumor"}, {"id": 13242, "dom": "D", "ent": "B-cell lymphoma", "cls": "mental"}, {"id": 13243, "dom": "D", "ent": "B-cell lymphoma", "cls": "genetic"}, {"id": 13244, "dom": "D", "ent": "B-cell lymphoma", "cls": "bact"}, {"id": 13245, "dom": "D", "ent": "B-cell lymphoma", "cls": "viral"}, {"id": 13246, "dom": "D", "ent": "B-cell lymphoma", "cls": "fungal"}, {"id": 13247, "dom": "D", "ent": "B-cell lymphoma", "cls": "parasit"}, {"id": 13248, "dom": "D", "ent": "childhood leukemia", "cls": "cardio"}, {"id": 13249, "dom": "D", "ent": "childhood leukemia", "cls": "resp"}, {"id": 13250, "dom": "D", "ent": "childhood leukemia", "cls": "gi"}, {"id": 13251, "dom": "D", "ent": "childhood leukemia", "cls": "nerv"}, {"id": 13252, "dom": "D", "ent": "childhood leukemia", "cls": "skin"}, {"id": 13253, "dom": "D", "ent": "childhood leukemia", "cls": "musc"}, {"id": 13254, "dom": "D", "ent": "childhood leukemia", "cls": "urin"}, {"id": 13255, "dom": "D", "ent": "childhood leukemia", "cls": "endo"}, {"id": 13256, "dom": "D", "ent": "childhood leukemia", "cls": "immune"}, {"id": 13257, "dom": "D", "ent": "childhood leukemia", "cls": "blood"}, {"id": 13258, "dom": "D", "ent": "childhood leukemia", "cls": "repro"}, {"id": 13259, "dom": "D", "ent": "childhood leukemia", "cls": "breast"}, {"id": 13260, "dom": "D", "ent": "childhood leukemia", "cls": "chest"}, {"id": 13261, "dom": "D", "ent": "childhood leukemia", "cls": "metab"}, {"id": 13262, "dom": "D", "ent": "childhood leukemia", "cls": "infect"}, {"id": 13263, "dom": "D", "ent": "childhood leukemia", "cls": "tumor"}, {"id": 13264, "dom": "D", "ent": "childhood leukemia", "cls": "mental"}, {"id": 13265, "dom": "D", "ent": "childhood leukemia", "cls": "genetic"}, {"id": 13266, "dom": "D", "ent": "childhood leukemia", "cls": "bact"}, {"id": 13267, "dom": "D", "ent": "childhood leukemia", "cls": "viral"}, {"id": 13268, "dom": "D", "ent": "childhood leukemia", "cls": "fungal"}, {"id": 13269, "dom": "D", "ent": "childhood leukemia", "cls": "parasit"}, {"id": 13270, "dom": "D", "ent": "placenta disease", "cls": "cardio"}, {"id": 13271, "dom": "D", "ent": "placenta disease", "cls": "resp"}, {"id": 13272, "dom": "D", "ent": "placenta disease", "cls": "gi"}, {"id": 13273, "dom": "D", "ent": "placenta disease", "cls": "nerv"}, {"id": 13274, "dom": "D", "ent": "placenta disease", "cls": "skin"}, {"id": 13275, "dom": "D", "ent": "placenta disease", "cls": "musc"}, {"id": 13276, "dom": "D", "ent": "placenta disease", "cls": "urin"}, {"id": 13277, "dom": "D", "ent": "placenta disease", "cls": "endo"}, {"id": 13278, "dom": "D", "ent": "placenta disease", "cls": "immune"}, {"id": 13279, "dom": "D", "ent": "placenta disease", "cls": "blood"}, {"id": 13280, "dom": "D", "ent": "placenta disease", "cls": "repro"}, {"id": 13281, "dom": "D", "ent": "placenta disease", "cls": "breast"}, {"id": 13282, "dom": "D", "ent": "placenta disease", "cls": "chest"}, {"id": 13283, "dom": "D", "ent": "placenta disease", "cls": "metab"}, {"id": 13284, "dom": "D", "ent": "placenta disease", "cls": "infect"}, {"id": 13285, "dom": "D", "ent": "placenta disease", "cls": "tumor"}, {"id": 13286, "dom": "D", "ent": "placenta disease", "cls": "mental"}, {"id": 13287, "dom": "D", "ent": "placenta disease", "cls": "genetic"}, {"id": 13288, "dom": "D", "ent": "placenta disease", "cls": "bact"}, {"id": 13289, "dom": "D", "ent": "placenta disease", "cls": "viral"}, {"id": 13290, "dom": "D", "ent": "placenta disease", "cls": "fungal"}, {"id": 13291, "dom": "D", "ent": "placenta disease", "cls": "parasit"}, {"id": 13292, "dom": "D", "ent": "cocaine abuse", "cls": "cardio"}, {"id": 13293, "dom": "D", "ent": "cocaine abuse", "cls": "resp"}, {"id": 13294, "dom": "D", "ent": "cocaine abuse", "cls": "gi"}, {"id": 13295, "dom": "D", "ent": "cocaine abuse", "cls": "nerv"}, {"id": 13296, "dom": "D", "ent": "cocaine abuse", "cls": "skin"}, {"id": 13297, "dom": "D", "ent": "cocaine abuse", "cls": "musc"}, {"id": 13298, "dom": "D", "ent": "cocaine abuse", "cls": "urin"}, {"id": 13299, "dom": "D", "ent": "cocaine abuse", "cls": "endo"}, {"id": 13300, "dom": "D", "ent": "cocaine abuse", "cls": "immune"}, {"id": 13301, "dom": "D", "ent": "cocaine abuse", "cls": "blood"}, {"id": 13302, "dom": "D", "ent": "cocaine abuse", "cls": "repro"}, {"id": 13303, "dom": "D", "ent": "cocaine abuse", "cls": "breast"}, {"id": 13304, "dom": "D", "ent": "cocaine abuse", "cls": "chest"}, {"id": 13305, "dom": "D", "ent": "cocaine abuse", "cls": "metab"}, {"id": 13306, "dom": "D", "ent": "cocaine abuse", "cls": "infect"}, {"id": 13307, "dom": "D", "ent": "cocaine abuse", "cls": "tumor"}, {"id": 13308, "dom": "D", "ent": "cocaine abuse", "cls": "mental"}, {"id": 13309, "dom": "D", "ent": "cocaine abuse", "cls": "genetic"}, {"id": 13310, "dom": "D", "ent": "cocaine abuse", "cls": "bact"}, {"id": 13311, "dom": "D", "ent": "cocaine abuse", "cls": "viral"}, {"id": 13312, "dom": "D", "ent": "cocaine abuse", "cls": "fungal"}, {"id": 13313, "dom": "D", "ent": "cocaine abuse", "cls": "parasit"}, {"id": 13314, "dom": "D", "ent": "cork-handlers' disease", "cls": "cardio"}, {"id": 13315, "dom": "D", "ent": "cork-handlers' disease", "cls": "resp"}, {"id": 13316, "dom": "D", "ent": "cork-handlers' disease", "cls": "gi"}, {"id": 13317, "dom": "D", "ent": "cork-handlers' disease", "cls": "nerv"}, {"id": 13318, "dom": "D", "ent": "cork-handlers' disease", "cls": "skin"}, {"id": 13319, "dom": "D", "ent": "cork-handlers' disease", "cls": "musc"}, {"id": 13320, "dom": "D", "ent": "cork-handlers' disease", "cls": "urin"}, {"id": 13321, "dom": "D", "ent": "cork-handlers' disease", "cls": "endo"}, {"id": 13322, "dom": "D", "ent": "cork-handlers' disease", "cls": "immune"}, {"id": 13323, "dom": "D", "ent": "cork-handlers' disease", "cls": "blood"}, {"id": 13324, "dom": "D", "ent": "cork-handlers' disease", "cls": "repro"}, {"id": 13325, "dom": "D", "ent": "cork-handlers' disease", "cls": "breast"}, {"id": 13326, "dom": "D", "ent": "cork-handlers' disease", "cls": "chest"}, {"id": 13327, "dom": "D", "ent": "cork-handlers' disease", "cls": "metab"}, {"id": 13328, "dom": "D", "ent": "cork-handlers' disease", "cls": "infect"}, {"id": 13329, "dom": "D", "ent": "cork-handlers' disease", "cls": "tumor"}, {"id": 13330, "dom": "D", "ent": "cork-handlers' disease", "cls": "mental"}, {"id": 13331, "dom": "D", "ent": "cork-handlers' disease", "cls": "genetic"}, {"id": 13332, "dom": "D", "ent": "cork-handlers' disease", "cls": "bact"}, {"id": 13333, "dom": "D", "ent": "cork-handlers' disease", "cls": "viral"}, {"id": 13334, "dom": "D", "ent": "cork-handlers' disease", "cls": "fungal"}, {"id": 13335, "dom": "D", "ent": "cork-handlers' disease", "cls": "parasit"}, {"id": 13336, "dom": "D", "ent": "gum cancer", "cls": "cardio"}, {"id": 13337, "dom": "D", "ent": "gum cancer", "cls": "resp"}, {"id": 13338, "dom": "D", "ent": "gum cancer", "cls": "gi"}, {"id": 13339, "dom": "D", "ent": "gum cancer", "cls": "nerv"}, {"id": 13340, "dom": "D", "ent": "gum cancer", "cls": "skin"}, {"id": 13341, "dom": "D", "ent": "gum cancer", "cls": "musc"}, {"id": 13342, "dom": "D", "ent": "gum cancer", "cls": "urin"}, {"id": 13343, "dom": "D", "ent": "gum cancer", "cls": "endo"}, {"id": 13344, "dom": "D", "ent": "gum cancer", "cls": "immune"}, {"id": 13345, "dom": "D", "ent": "gum cancer", "cls": "blood"}, {"id": 13346, "dom": "D", "ent": "gum cancer", "cls": "repro"}, {"id": 13347, "dom": "D", "ent": "gum cancer", "cls": "breast"}, {"id": 13348, "dom": "D", "ent": "gum cancer", "cls": "chest"}, {"id": 13349, "dom": "D", "ent": "gum cancer", "cls": "metab"}, {"id": 13350, "dom": "D", "ent": "gum cancer", "cls": "infect"}, {"id": 13351, "dom": "D", "ent": "gum cancer", "cls": "tumor"}, {"id": 13352, "dom": "D", "ent": "gum cancer", "cls": "mental"}, {"id": 13353, "dom": "D", "ent": "gum cancer", "cls": "genetic"}, {"id": 13354, "dom": "D", "ent": "gum cancer", "cls": "bact"}, {"id": 13355, "dom": "D", "ent": "gum cancer", "cls": "viral"}, {"id": 13356, "dom": "D", "ent": "gum cancer", "cls": "fungal"}, {"id": 13357, "dom": "D", "ent": "gum cancer", "cls": "parasit"}, {"id": 13358, "dom": "D", "ent": "upper lip cancer", "cls": "cardio"}, {"id": 13359, "dom": "D", "ent": "upper lip cancer", "cls": "resp"}, {"id": 13360, "dom": "D", "ent": "upper lip cancer", "cls": "gi"}, {"id": 13361, "dom": "D", "ent": "upper lip cancer", "cls": "nerv"}, {"id": 13362, "dom": "D", "ent": "upper lip cancer", "cls": "skin"}, {"id": 13363, "dom": "D", "ent": "upper lip cancer", "cls": "musc"}, {"id": 13364, "dom": "D", "ent": "upper lip cancer", "cls": "urin"}, {"id": 13365, "dom": "D", "ent": "upper lip cancer", "cls": "endo"}, {"id": 13366, "dom": "D", "ent": "upper lip cancer", "cls": "immune"}, {"id": 13367, "dom": "D", "ent": "upper lip cancer", "cls": "blood"}, {"id": 13368, "dom": "D", "ent": "upper lip cancer", "cls": "repro"}, {"id": 13369, "dom": "D", "ent": "upper lip cancer", "cls": "breast"}, {"id": 13370, "dom": "D", "ent": "upper lip cancer", "cls": "chest"}, {"id": 13371, "dom": "D", "ent": "upper lip cancer", "cls": "metab"}, {"id": 13372, "dom": "D", "ent": "upper lip cancer", "cls": "infect"}, {"id": 13373, "dom": "D", "ent": "upper lip cancer", "cls": "tumor"}, {"id": 13374, "dom": "D", "ent": "upper lip cancer", "cls": "mental"}, {"id": 13375, "dom": "D", "ent": "upper lip cancer", "cls": "genetic"}, {"id": 13376, "dom": "D", "ent": "upper lip cancer", "cls": "bact"}, {"id": 13377, "dom": "D", "ent": "upper lip cancer", "cls": "viral"}, {"id": 13378, "dom": "D", "ent": "upper lip cancer", "cls": "fungal"}, {"id": 13379, "dom": "D", "ent": "upper lip cancer", "cls": "parasit"}, {"id": 13380, "dom": "D", "ent": "smallpox", "cls": "cardio"}, {"id": 13381, "dom": "D", "ent": "smallpox", "cls": "resp"}, {"id": 13382, "dom": "D", "ent": "smallpox", "cls": "gi"}, {"id": 13383, "dom": "D", "ent": "smallpox", "cls": "nerv"}, {"id": 13384, "dom": "D", "ent": "smallpox", "cls": "skin"}, {"id": 13385, "dom": "D", "ent": "smallpox", "cls": "musc"}, {"id": 13386, "dom": "D", "ent": "smallpox", "cls": "urin"}, {"id": 13387, "dom": "D", "ent": "smallpox", "cls": "endo"}, {"id": 13388, "dom": "D", "ent": "smallpox", "cls": "immune"}, {"id": 13389, "dom": "D", "ent": "smallpox", "cls": "blood"}, {"id": 13390, "dom": "D", "ent": "smallpox", "cls": "repro"}, {"id": 13391, "dom": "D", "ent": "smallpox", "cls": "breast"}, {"id": 13392, "dom": "D", "ent": "smallpox", "cls": "chest"}, {"id": 13393, "dom": "D", "ent": "smallpox", "cls": "metab"}, {"id": 13394, "dom": "D", "ent": "smallpox", "cls": "infect"}, {"id": 13395, "dom": "D", "ent": "smallpox", "cls": "tumor"}, {"id": 13396, "dom": "D", "ent": "smallpox", "cls": "mental"}, {"id": 13397, "dom": "D", "ent": "smallpox", "cls": "genetic"}, {"id": 13398, "dom": "D", "ent": "smallpox", "cls": "bact"}, {"id": 13399, "dom": "D", "ent": "smallpox", "cls": "viral"}, {"id": 13400, "dom": "D", "ent": "smallpox", "cls": "fungal"}, {"id": 13401, "dom": "D", "ent": "smallpox", "cls": "parasit"}, {"id": 13402, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "cardio"}, {"id": 13403, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "resp"}, {"id": 13404, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "gi"}, {"id": 13405, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "nerv"}, {"id": 13406, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "skin"}, {"id": 13407, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "musc"}, {"id": 13408, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "urin"}, {"id": 13409, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "endo"}, {"id": 13410, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "immune"}, {"id": 13411, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "blood"}, {"id": 13412, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "repro"}, {"id": 13413, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "breast"}, {"id": 13414, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "chest"}, {"id": 13415, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "metab"}, {"id": 13416, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "infect"}, {"id": 13417, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "tumor"}, {"id": 13418, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "mental"}, {"id": 13419, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "genetic"}, {"id": 13420, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "bact"}, {"id": 13421, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "viral"}, {"id": 13422, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "fungal"}, {"id": 13423, "dom": "D", "ent": "intermediate coronary syndrome", "cls": "parasit"}, {"id": 13424, "dom": "D", "ent": "Wilson disease", "cls": "cardio"}, {"id": 13425, "dom": "D", "ent": "Wilson disease", "cls": "resp"}, {"id": 13426, "dom": "D", "ent": "Wilson disease", "cls": "gi"}, {"id": 13427, "dom": "D", "ent": "Wilson disease", "cls": "nerv"}, {"id": 13428, "dom": "D", "ent": "Wilson disease", "cls": "skin"}, {"id": 13429, "dom": "D", "ent": "Wilson disease", "cls": "musc"}, {"id": 13430, "dom": "D", "ent": "Wilson disease", "cls": "urin"}, {"id": 13431, "dom": "D", "ent": "Wilson disease", "cls": "endo"}, {"id": 13432, "dom": "D", "ent": "Wilson disease", "cls": "immune"}, {"id": 13433, "dom": "D", "ent": "Wilson disease", "cls": "blood"}, {"id": 13434, "dom": "D", "ent": "Wilson disease", "cls": "repro"}, {"id": 13435, "dom": "D", "ent": "Wilson disease", "cls": "breast"}, {"id": 13436, "dom": "D", "ent": "Wilson disease", "cls": "chest"}, {"id": 13437, "dom": "D", "ent": "Wilson disease", "cls": "metab"}, {"id": 13438, "dom": "D", "ent": "Wilson disease", "cls": "infect"}, {"id": 13439, "dom": "D", "ent": "Wilson disease", "cls": "tumor"}, {"id": 13440, "dom": "D", "ent": "Wilson disease", "cls": "mental"}, {"id": 13441, "dom": "D", "ent": "Wilson disease", "cls": "genetic"}, {"id": 13442, "dom": "D", "ent": "Wilson disease", "cls": "bact"}, {"id": 13443, "dom": "D", "ent": "Wilson disease", "cls": "viral"}, {"id": 13444, "dom": "D", "ent": "Wilson disease", "cls": "fungal"}, {"id": 13445, "dom": "D", "ent": "Wilson disease", "cls": "parasit"}, {"id": 13446, "dom": "D", "ent": "Evans' syndrome", "cls": "cardio"}, {"id": 13447, "dom": "D", "ent": "Evans' syndrome", "cls": "resp"}, {"id": 13448, "dom": "D", "ent": "Evans' syndrome", "cls": "gi"}, {"id": 13449, "dom": "D", "ent": "Evans' syndrome", "cls": "nerv"}, {"id": 13450, "dom": "D", "ent": "Evans' syndrome", "cls": "skin"}, {"id": 13451, "dom": "D", "ent": "Evans' syndrome", "cls": "musc"}, {"id": 13452, "dom": "D", "ent": "Evans' syndrome", "cls": "urin"}, {"id": 13453, "dom": "D", "ent": "Evans' syndrome", "cls": "endo"}, {"id": 13454, "dom": "D", "ent": "Evans' syndrome", "cls": "immune"}, {"id": 13455, "dom": "D", "ent": "Evans' syndrome", "cls": "blood"}, {"id": 13456, "dom": "D", "ent": "Evans' syndrome", "cls": "repro"}, {"id": 13457, "dom": "D", "ent": "Evans' syndrome", "cls": "breast"}, {"id": 13458, "dom": "D", "ent": "Evans' syndrome", "cls": "chest"}, {"id": 13459, "dom": "D", "ent": "Evans' syndrome", "cls": "metab"}, {"id": 13460, "dom": "D", "ent": "Evans' syndrome", "cls": "infect"}, {"id": 13461, "dom": "D", "ent": "Evans' syndrome", "cls": "tumor"}, {"id": 13462, "dom": "D", "ent": "Evans' syndrome", "cls": "mental"}, {"id": 13463, "dom": "D", "ent": "Evans' syndrome", "cls": "genetic"}, {"id": 13464, "dom": "D", "ent": "Evans' syndrome", "cls": "bact"}, {"id": 13465, "dom": "D", "ent": "Evans' syndrome", "cls": "viral"}, {"id": 13466, "dom": "D", "ent": "Evans' syndrome", "cls": "fungal"}, {"id": 13467, "dom": "D", "ent": "Evans' syndrome", "cls": "parasit"}, {"id": 13468, "dom": "D", "ent": "lower gum cancer", "cls": "cardio"}, {"id": 13469, "dom": "D", "ent": "lower gum cancer", "cls": "resp"}, {"id": 13470, "dom": "D", "ent": "lower gum cancer", "cls": "gi"}, {"id": 13471, "dom": "D", "ent": "lower gum cancer", "cls": "nerv"}, {"id": 13472, "dom": "D", "ent": "lower gum cancer", "cls": "skin"}, {"id": 13473, "dom": "D", "ent": "lower gum cancer", "cls": "musc"}, {"id": 13474, "dom": "D", "ent": "lower gum cancer", "cls": "urin"}, {"id": 13475, "dom": "D", "ent": "lower gum cancer", "cls": "endo"}, {"id": 13476, "dom": "D", "ent": "lower gum cancer", "cls": "immune"}, {"id": 13477, "dom": "D", "ent": "lower gum cancer", "cls": "blood"}, {"id": 13478, "dom": "D", "ent": "lower gum cancer", "cls": "repro"}, {"id": 13479, "dom": "D", "ent": "lower gum cancer", "cls": "breast"}, {"id": 13480, "dom": "D", "ent": "lower gum cancer", "cls": "chest"}, {"id": 13481, "dom": "D", "ent": "lower gum cancer", "cls": "metab"}, {"id": 13482, "dom": "D", "ent": "lower gum cancer", "cls": "infect"}, {"id": 13483, "dom": "D", "ent": "lower gum cancer", "cls": "tumor"}, {"id": 13484, "dom": "D", "ent": "lower gum cancer", "cls": "mental"}, {"id": 13485, "dom": "D", "ent": "lower gum cancer", "cls": "genetic"}, {"id": 13486, "dom": "D", "ent": "lower gum cancer", "cls": "bact"}, {"id": 13487, "dom": "D", "ent": "lower gum cancer", "cls": "viral"}, {"id": 13488, "dom": "D", "ent": "lower gum cancer", "cls": "fungal"}, {"id": 13489, "dom": "D", "ent": "lower gum cancer", "cls": "parasit"}, {"id": 13490, "dom": "D", "ent": "allergic asthma", "cls": "cardio"}, {"id": 13491, "dom": "D", "ent": "allergic asthma", "cls": "resp"}, {"id": 13492, "dom": "D", "ent": "allergic asthma", "cls": "gi"}, {"id": 13493, "dom": "D", "ent": "allergic asthma", "cls": "nerv"}, {"id": 13494, "dom": "D", "ent": "allergic asthma", "cls": "skin"}, {"id": 13495, "dom": "D", "ent": "allergic asthma", "cls": "musc"}, {"id": 13496, "dom": "D", "ent": "allergic asthma", "cls": "urin"}, {"id": 13497, "dom": "D", "ent": "allergic asthma", "cls": "endo"}, {"id": 13498, "dom": "D", "ent": "allergic asthma", "cls": "immune"}, {"id": 13499, "dom": "D", "ent": "allergic asthma", "cls": "blood"}, {"id": 13500, "dom": "D", "ent": "allergic asthma", "cls": "repro"}, {"id": 13501, "dom": "D", "ent": "allergic asthma", "cls": "breast"}, {"id": 13502, "dom": "D", "ent": "allergic asthma", "cls": "chest"}, {"id": 13503, "dom": "D", "ent": "allergic asthma", "cls": "metab"}, {"id": 13504, "dom": "D", "ent": "allergic asthma", "cls": "infect"}, {"id": 13505, "dom": "D", "ent": "allergic asthma", "cls": "tumor"}, {"id": 13506, "dom": "D", "ent": "allergic asthma", "cls": "mental"}, {"id": 13507, "dom": "D", "ent": "allergic asthma", "cls": "genetic"}, {"id": 13508, "dom": "D", "ent": "allergic asthma", "cls": "bact"}, {"id": 13509, "dom": "D", "ent": "allergic asthma", "cls": "viral"}, {"id": 13510, "dom": "D", "ent": "allergic asthma", "cls": "fungal"}, {"id": 13511, "dom": "D", "ent": "allergic asthma", "cls": "parasit"}, {"id": 13512, "dom": "D", "ent": "Sheehan syndrome", "cls": "cardio"}, {"id": 13513, "dom": "D", "ent": "Sheehan syndrome", "cls": "resp"}, {"id": 13514, "dom": "D", "ent": "Sheehan syndrome", "cls": "gi"}, {"id": 13515, "dom": "D", "ent": "Sheehan syndrome", "cls": "nerv"}, {"id": 13516, "dom": "D", "ent": "Sheehan syndrome", "cls": "skin"}, {"id": 13517, "dom": "D", "ent": "Sheehan syndrome", "cls": "musc"}, {"id": 13518, "dom": "D", "ent": "Sheehan syndrome", "cls": "urin"}, {"id": 13519, "dom": "D", "ent": "Sheehan syndrome", "cls": "endo"}, {"id": 13520, "dom": "D", "ent": "Sheehan syndrome", "cls": "immune"}, {"id": 13521, "dom": "D", "ent": "Sheehan syndrome", "cls": "blood"}, {"id": 13522, "dom": "D", "ent": "Sheehan syndrome", "cls": "repro"}, {"id": 13523, "dom": "D", "ent": "Sheehan syndrome", "cls": "breast"}, {"id": 13524, "dom": "D", "ent": "Sheehan syndrome", "cls": "chest"}, {"id": 13525, "dom": "D", "ent": "Sheehan syndrome", "cls": "metab"}, {"id": 13526, "dom": "D", "ent": "Sheehan syndrome", "cls": "infect"}, {"id": 13527, "dom": "D", "ent": "Sheehan syndrome", "cls": "tumor"}, {"id": 13528, "dom": "D", "ent": "Sheehan syndrome", "cls": "mental"}, {"id": 13529, "dom": "D", "ent": "Sheehan syndrome", "cls": "genetic"}, {"id": 13530, "dom": "D", "ent": "Sheehan syndrome", "cls": "bact"}, {"id": 13531, "dom": "D", "ent": "Sheehan syndrome", "cls": "viral"}, {"id": 13532, "dom": "D", "ent": "Sheehan syndrome", "cls": "fungal"}, {"id": 13533, "dom": "D", "ent": "Sheehan syndrome", "cls": "parasit"}, {"id": 13534, "dom": "D", "ent": "blue drum syndrome", "cls": "cardio"}, {"id": 13535, "dom": "D", "ent": "blue drum syndrome", "cls": "resp"}, {"id": 13536, "dom": "D", "ent": "blue drum syndrome", "cls": "gi"}, {"id": 13537, "dom": "D", "ent": "blue drum syndrome", "cls": "nerv"}, {"id": 13538, "dom": "D", "ent": "blue drum syndrome", "cls": "skin"}, {"id": 13539, "dom": "D", "ent": "blue drum syndrome", "cls": "musc"}, {"id": 13540, "dom": "D", "ent": "blue drum syndrome", "cls": "urin"}, {"id": 13541, "dom": "D", "ent": "blue drum syndrome", "cls": "endo"}, {"id": 13542, "dom": "D", "ent": "blue drum syndrome", "cls": "immune"}, {"id": 13543, "dom": "D", "ent": "blue drum syndrome", "cls": "blood"}, {"id": 13544, "dom": "D", "ent": "blue drum syndrome", "cls": "repro"}, {"id": 13545, "dom": "D", "ent": "blue drum syndrome", "cls": "breast"}, {"id": 13546, "dom": "D", "ent": "blue drum syndrome", "cls": "chest"}, {"id": 13547, "dom": "D", "ent": "blue drum syndrome", "cls": "metab"}, {"id": 13548, "dom": "D", "ent": "blue drum syndrome", "cls": "infect"}, {"id": 13549, "dom": "D", "ent": "blue drum syndrome", "cls": "tumor"}, {"id": 13550, "dom": "D", "ent": "blue drum syndrome", "cls": "mental"}, {"id": 13551, "dom": "D", "ent": "blue drum syndrome", "cls": "genetic"}, {"id": 13552, "dom": "D", "ent": "blue drum syndrome", "cls": "bact"}, {"id": 13553, "dom": "D", "ent": "blue drum syndrome", "cls": "viral"}, {"id": 13554, "dom": "D", "ent": "blue drum syndrome", "cls": "fungal"}, {"id": 13555, "dom": "D", "ent": "blue drum syndrome", "cls": "parasit"}, {"id": 13556, "dom": "D", "ent": "peripheral vertigo", "cls": "cardio"}, {"id": 13557, "dom": "D", "ent": "peripheral vertigo", "cls": "resp"}, {"id": 13558, "dom": "D", "ent": "peripheral vertigo", "cls": "gi"}, {"id": 13559, "dom": "D", "ent": "peripheral vertigo", "cls": "nerv"}, {"id": 13560, "dom": "D", "ent": "peripheral vertigo", "cls": "skin"}, {"id": 13561, "dom": "D", "ent": "peripheral vertigo", "cls": "musc"}, {"id": 13562, "dom": "D", "ent": "peripheral vertigo", "cls": "urin"}, {"id": 13563, "dom": "D", "ent": "peripheral vertigo", "cls": "endo"}, {"id": 13564, "dom": "D", "ent": "peripheral vertigo", "cls": "immune"}, {"id": 13565, "dom": "D", "ent": "peripheral vertigo", "cls": "blood"}, {"id": 13566, "dom": "D", "ent": "peripheral vertigo", "cls": "repro"}, {"id": 13567, "dom": "D", "ent": "peripheral vertigo", "cls": "breast"}, {"id": 13568, "dom": "D", "ent": "peripheral vertigo", "cls": "chest"}, {"id": 13569, "dom": "D", "ent": "peripheral vertigo", "cls": "metab"}, {"id": 13570, "dom": "D", "ent": "peripheral vertigo", "cls": "infect"}, {"id": 13571, "dom": "D", "ent": "peripheral vertigo", "cls": "tumor"}, {"id": 13572, "dom": "D", "ent": "peripheral vertigo", "cls": "mental"}, {"id": 13573, "dom": "D", "ent": "peripheral vertigo", "cls": "genetic"}, {"id": 13574, "dom": "D", "ent": "peripheral vertigo", "cls": "bact"}, {"id": 13575, "dom": "D", "ent": "peripheral vertigo", "cls": "viral"}, {"id": 13576, "dom": "D", "ent": "peripheral vertigo", "cls": "fungal"}, {"id": 13577, "dom": "D", "ent": "peripheral vertigo", "cls": "parasit"}, {"id": 13578, "dom": "D", "ent": "congenital heart block", "cls": "cardio"}, {"id": 13579, "dom": "D", "ent": "congenital heart block", "cls": "resp"}, {"id": 13580, "dom": "D", "ent": "congenital heart block", "cls": "gi"}, {"id": 13581, "dom": "D", "ent": "congenital heart block", "cls": "nerv"}, {"id": 13582, "dom": "D", "ent": "congenital heart block", "cls": "skin"}, {"id": 13583, "dom": "D", "ent": "congenital heart block", "cls": "musc"}, {"id": 13584, "dom": "D", "ent": "congenital heart block", "cls": "urin"}, {"id": 13585, "dom": "D", "ent": "congenital heart block", "cls": "endo"}, {"id": 13586, "dom": "D", "ent": "congenital heart block", "cls": "immune"}, {"id": 13587, "dom": "D", "ent": "congenital heart block", "cls": "blood"}, {"id": 13588, "dom": "D", "ent": "congenital heart block", "cls": "repro"}, {"id": 13589, "dom": "D", "ent": "congenital heart block", "cls": "breast"}, {"id": 13590, "dom": "D", "ent": "congenital heart block", "cls": "chest"}, {"id": 13591, "dom": "D", "ent": "congenital heart block", "cls": "metab"}, {"id": 13592, "dom": "D", "ent": "congenital heart block", "cls": "infect"}, {"id": 13593, "dom": "D", "ent": "congenital heart block", "cls": "tumor"}, {"id": 13594, "dom": "D", "ent": "congenital heart block", "cls": "mental"}, {"id": 13595, "dom": "D", "ent": "congenital heart block", "cls": "genetic"}, {"id": 13596, "dom": "D", "ent": "congenital heart block", "cls": "bact"}, {"id": 13597, "dom": "D", "ent": "congenital heart block", "cls": "viral"}, {"id": 13598, "dom": "D", "ent": "congenital heart block", "cls": "fungal"}, {"id": 13599, "dom": "D", "ent": "congenital heart block", "cls": "parasit"}, {"id": 13600, "dom": "D", "ent": "lymph node disease", "cls": "cardio"}, {"id": 13601, "dom": "D", "ent": "lymph node disease", "cls": "resp"}, {"id": 13602, "dom": "D", "ent": "lymph node disease", "cls": "gi"}, {"id": 13603, "dom": "D", "ent": "lymph node disease", "cls": "nerv"}, {"id": 13604, "dom": "D", "ent": "lymph node disease", "cls": "skin"}, {"id": 13605, "dom": "D", "ent": "lymph node disease", "cls": "musc"}, {"id": 13606, "dom": "D", "ent": "lymph node disease", "cls": "urin"}, {"id": 13607, "dom": "D", "ent": "lymph node disease", "cls": "endo"}, {"id": 13608, "dom": "D", "ent": "lymph node disease", "cls": "immune"}, {"id": 13609, "dom": "D", "ent": "lymph node disease", "cls": "blood"}, {"id": 13610, "dom": "D", "ent": "lymph node disease", "cls": "repro"}, {"id": 13611, "dom": "D", "ent": "lymph node disease", "cls": "breast"}, {"id": 13612, "dom": "D", "ent": "lymph node disease", "cls": "chest"}, {"id": 13613, "dom": "D", "ent": "lymph node disease", "cls": "metab"}, {"id": 13614, "dom": "D", "ent": "lymph node disease", "cls": "infect"}, {"id": 13615, "dom": "D", "ent": "lymph node disease", "cls": "tumor"}, {"id": 13616, "dom": "D", "ent": "lymph node disease", "cls": "mental"}, {"id": 13617, "dom": "D", "ent": "lymph node disease", "cls": "genetic"}, {"id": 13618, "dom": "D", "ent": "lymph node disease", "cls": "bact"}, {"id": 13619, "dom": "D", "ent": "lymph node disease", "cls": "viral"}, {"id": 13620, "dom": "D", "ent": "lymph node disease", "cls": "fungal"}, {"id": 13621, "dom": "D", "ent": "lymph node disease", "cls": "parasit"}, {"id": 13622, "dom": "D", "ent": "cocaine dependence", "cls": "cardio"}, {"id": 13623, "dom": "D", "ent": "cocaine dependence", "cls": "resp"}, {"id": 13624, "dom": "D", "ent": "cocaine dependence", "cls": "gi"}, {"id": 13625, "dom": "D", "ent": "cocaine dependence", "cls": "nerv"}, {"id": 13626, "dom": "D", "ent": "cocaine dependence", "cls": "skin"}, {"id": 13627, "dom": "D", "ent": "cocaine dependence", "cls": "musc"}, {"id": 13628, "dom": "D", "ent": "cocaine dependence", "cls": "urin"}, {"id": 13629, "dom": "D", "ent": "cocaine dependence", "cls": "endo"}, {"id": 13630, "dom": "D", "ent": "cocaine dependence", "cls": "immune"}, {"id": 13631, "dom": "D", "ent": "cocaine dependence", "cls": "blood"}, {"id": 13632, "dom": "D", "ent": "cocaine dependence", "cls": "repro"}, {"id": 13633, "dom": "D", "ent": "cocaine dependence", "cls": "breast"}, {"id": 13634, "dom": "D", "ent": "cocaine dependence", "cls": "chest"}, {"id": 13635, "dom": "D", "ent": "cocaine dependence", "cls": "metab"}, {"id": 13636, "dom": "D", "ent": "cocaine dependence", "cls": "infect"}, {"id": 13637, "dom": "D", "ent": "cocaine dependence", "cls": "tumor"}, {"id": 13638, "dom": "D", "ent": "cocaine dependence", "cls": "mental"}, {"id": 13639, "dom": "D", "ent": "cocaine dependence", "cls": "genetic"}, {"id": 13640, "dom": "D", "ent": "cocaine dependence", "cls": "bact"}, {"id": 13641, "dom": "D", "ent": "cocaine dependence", "cls": "viral"}, {"id": 13642, "dom": "D", "ent": "cocaine dependence", "cls": "fungal"}, {"id": 13643, "dom": "D", "ent": "cocaine dependence", "cls": "parasit"}, {"id": 13644, "dom": "G", "ent": "Bani Yas City", "cls": "Australia and New Zealand"}, {"id": 13645, "dom": "G", "ent": "Bani Yas City", "cls": "Caribbean"}, {"id": 13646, "dom": "G", "ent": "Bani Yas City", "cls": "Central America"}, {"id": 13647, "dom": "G", "ent": "Bani Yas City", "cls": "Central Asia"}, {"id": 13648, "dom": "G", "ent": "Bani Yas City", "cls": "Eastern Africa"}, {"id": 13649, "dom": "G", "ent": "Bani Yas City", "cls": "Eastern Asia"}, {"id": 13650, "dom": "G", "ent": "Bani Yas City", "cls": "Eastern Europe"}, {"id": 13651, "dom": "G", "ent": "Bani Yas City", "cls": "Melanesia"}, {"id": 13652, "dom": "G", "ent": "Bani Yas City", "cls": "Micronesia"}, {"id": 13653, "dom": "G", "ent": "Bani Yas City", "cls": "Middle Africa"}, {"id": 13654, "dom": "G", "ent": "Bani Yas City", "cls": "Northern Africa"}, {"id": 13655, "dom": "G", "ent": "Bani Yas City", "cls": "Northern America"}, {"id": 13656, "dom": "G", "ent": "Bani Yas City", "cls": "Northern Europe"}, {"id": 13657, "dom": "G", "ent": "Bani Yas City", "cls": "Polynesia"}, {"id": 13658, "dom": "G", "ent": "Bani Yas City", "cls": "South America"}, {"id": 13659, "dom": "G", "ent": "Bani Yas City", "cls": "South-eastern Asia"}, {"id": 13660, "dom": "G", "ent": "Bani Yas City", "cls": "Southern Africa"}, {"id": 13661, "dom": "G", "ent": "Bani Yas City", "cls": "Southern Asia"}, {"id": 13662, "dom": "G", "ent": "Bani Yas City", "cls": "Southern Europe"}, {"id": 13663, "dom": "G", "ent": "Bani Yas City", "cls": "Western Africa"}, {"id": 13664, "dom": "G", "ent": "Bani Yas City", "cls": "Western Asia"}, {"id": 13665, "dom": "G", "ent": "Bani Yas City", "cls": "Western Europe"}, {"id": 13666, "dom": "G", "ent": "Al Karama", "cls": "Australia and New Zealand"}, {"id": 13667, "dom": "G", "ent": "Al Karama", "cls": "Caribbean"}, {"id": 13668, "dom": "G", "ent": "Al Karama", "cls": "Central America"}, {"id": 13669, "dom": "G", "ent": "Al Karama", "cls": "Central Asia"}, {"id": 13670, "dom": "G", "ent": "Al Karama", "cls": "Eastern Africa"}, {"id": 13671, "dom": "G", "ent": "Al Karama", "cls": "Eastern Asia"}, {"id": 13672, "dom": "G", "ent": "Al Karama", "cls": "Eastern Europe"}, {"id": 13673, "dom": "G", "ent": "Al Karama", "cls": "Melanesia"}, {"id": 13674, "dom": "G", "ent": "Al Karama", "cls": "Micronesia"}, {"id": 13675, "dom": "G", "ent": "Al Karama", "cls": "Middle Africa"}, {"id": 13676, "dom": "G", "ent": "Al Karama", "cls": "Northern Africa"}, {"id": 13677, "dom": "G", "ent": "Al Karama", "cls": "Northern America"}, {"id": 13678, "dom": "G", "ent": "Al Karama", "cls": "Northern Europe"}, {"id": 13679, "dom": "G", "ent": "Al Karama", "cls": "Polynesia"}, {"id": 13680, "dom": "G", "ent": "Al Karama", "cls": "South America"}, {"id": 13681, "dom": "G", "ent": "Al Karama", "cls": "South-eastern Asia"}, {"id": 13682, "dom": "G", "ent": "Al Karama", "cls": "Southern Africa"}, {"id": 13683, "dom": "G", "ent": "Al Karama", "cls": "Southern Asia"}, {"id": 13684, "dom": "G", "ent": "Al Karama", "cls": "Southern Europe"}, {"id": 13685, "dom": "G", "ent": "Al Karama", "cls": "Western Africa"}, {"id": 13686, "dom": "G", "ent": "Al Karama", "cls": "Western Asia"}, {"id": 13687, "dom": "G", "ent": "Al Karama", "cls": "Western Europe"}, {"id": 13688, "dom": "G", "ent": "The Palm Jumeirah", "cls": "Australia and New Zealand"}, {"id": 13689, "dom": "G", "ent": "The Palm Jumeirah", "cls": "Caribbean"}, {"id": 13690, "dom": "G", "ent": "The Palm Jumeirah", "cls": "Central America"}, {"id": 13691, "dom": "G", "ent": "The Palm Jumeirah", "cls": "Central Asia"}, {"id": 13692, "dom": "G", "ent": "The Palm Jumeirah", "cls": "Eastern Africa"}, {"id": 13693, "dom": "G", "ent": "The Palm Jumeirah", "cls": "Eastern Asia"}, {"id": 13694, "dom": "G", "ent": "The Palm Jumeirah", "cls": "Eastern Europe"}, {"id": 13695, "dom": "G", "ent": "The Palm Jumeirah", "cls": "Melanesia"}, {"id": 13696, "dom": "G", "ent": "The Palm Jumeirah", "cls": "Micronesia"}, {"id": 13697, "dom": "G", "ent": "The Palm Jumeirah", "cls": "Middle Africa"}, {"id": 13698, "dom": "G", "ent": "The Palm Jumeirah", "cls": "Northern Africa"}, {"id": 13699, "dom": "G", "ent": "The Palm Jumeirah", "cls": "Northern America"}, {"id": 13700, "dom": "G", "ent": "The Palm Jumeirah", "cls": "Northern Europe"}, {"id": 13701, "dom": "G", "ent": "The Palm Jumeirah", "cls": "Polynesia"}, {"id": 13702, "dom": "G", "ent": "The Palm Jumeirah", "cls": "South America"}, {"id": 13703, "dom": "G", "ent": "The Palm Jumeirah", "cls": "South-eastern Asia"}, {"id": 13704, "dom": "G", "ent": "The Palm Jumeirah", "cls": "Southern Africa"}, {"id": 13705, "dom": "G", "ent": "The Palm Jumeirah", "cls": "Southern Asia"}, {"id": 13706, "dom": "G", "ent": "The Palm Jumeirah", "cls": "Southern Europe"}, {"id": 13707, "dom": "G", "ent": "The Palm Jumeirah", "cls": "Western Africa"}, {"id": 13708, "dom": "G", "ent": "The Palm Jumeirah", "cls": "Western Asia"}, {"id": 13709, "dom": "G", "ent": "The Palm Jumeirah", "cls": "Western Europe"}, {"id": 13710, "dom": "G", "ent": "Al Mizhar First", "cls": "Australia and New Zealand"}, {"id": 13711, "dom": "G", "ent": "Al Mizhar First", "cls": "Caribbean"}, {"id": 13712, "dom": "G", "ent": "Al Mizhar First", "cls": "Central America"}, {"id": 13713, "dom": "G", "ent": "Al Mizhar First", "cls": "Central Asia"}, {"id": 13714, "dom": "G", "ent": "Al Mizhar First", "cls": "Eastern Africa"}, {"id": 13715, "dom": "G", "ent": "Al Mizhar First", "cls": "Eastern Asia"}, {"id": 13716, "dom": "G", "ent": "Al Mizhar First", "cls": "Eastern Europe"}, {"id": 13717, "dom": "G", "ent": "Al Mizhar First", "cls": "Melanesia"}, {"id": 13718, "dom": "G", "ent": "Al Mizhar First", "cls": "Micronesia"}, {"id": 13719, "dom": "G", "ent": "Al Mizhar First", "cls": "Middle Africa"}, {"id": 13720, "dom": "G", "ent": "Al Mizhar First", "cls": "Northern Africa"}, {"id": 13721, "dom": "G", "ent": "Al Mizhar First", "cls": "Northern America"}, {"id": 13722, "dom": "G", "ent": "Al Mizhar First", "cls": "Northern Europe"}, {"id": 13723, "dom": "G", "ent": "Al Mizhar First", "cls": "Polynesia"}, {"id": 13724, "dom": "G", "ent": "Al Mizhar First", "cls": "South America"}, {"id": 13725, "dom": "G", "ent": "Al Mizhar First", "cls": "South-eastern Asia"}, {"id": 13726, "dom": "G", "ent": "Al Mizhar First", "cls": "Southern Africa"}, {"id": 13727, "dom": "G", "ent": "Al Mizhar First", "cls": "Southern Asia"}, {"id": 13728, "dom": "G", "ent": "Al Mizhar First", "cls": "Southern Europe"}, {"id": 13729, "dom": "G", "ent": "Al Mizhar First", "cls": "Western Africa"}, {"id": 13730, "dom": "G", "ent": "Al Mizhar First", "cls": "Western Asia"}, {"id": 13731, "dom": "G", "ent": "Al Mizhar First", "cls": "Western Europe"}, {"id": 13732, "dom": "G", "ent": "Khulm", "cls": "Australia and New Zealand"}, {"id": 13733, "dom": "G", "ent": "Khulm", "cls": "Caribbean"}, {"id": 13734, "dom": "G", "ent": "Khulm", "cls": "Central America"}, {"id": 13735, "dom": "G", "ent": "Khulm", "cls": "Central Asia"}, {"id": 13736, "dom": "G", "ent": "Khulm", "cls": "Eastern Africa"}, {"id": 13737, "dom": "G", "ent": "Khulm", "cls": "Eastern Asia"}, {"id": 13738, "dom": "G", "ent": "Khulm", "cls": "Eastern Europe"}, {"id": 13739, "dom": "G", "ent": "Khulm", "cls": "Melanesia"}, {"id": 13740, "dom": "G", "ent": "Khulm", "cls": "Micronesia"}, {"id": 13741, "dom": "G", "ent": "Khulm", "cls": "Middle Africa"}, {"id": 13742, "dom": "G", "ent": "Khulm", "cls": "Northern Africa"}, {"id": 13743, "dom": "G", "ent": "Khulm", "cls": "Northern America"}, {"id": 13744, "dom": "G", "ent": "Khulm", "cls": "Northern Europe"}, {"id": 13745, "dom": "G", "ent": "Khulm", "cls": "Polynesia"}, {"id": 13746, "dom": "G", "ent": "Khulm", "cls": "South America"}, {"id": 13747, "dom": "G", "ent": "Khulm", "cls": "South-eastern Asia"}, {"id": 13748, "dom": "G", "ent": "Khulm", "cls": "Southern Africa"}, {"id": 13749, "dom": "G", "ent": "Khulm", "cls": "Southern Asia"}, {"id": 13750, "dom": "G", "ent": "Khulm", "cls": "Southern Europe"}, {"id": 13751, "dom": "G", "ent": "Khulm", "cls": "Western Africa"}, {"id": 13752, "dom": "G", "ent": "Khulm", "cls": "Western Asia"}, {"id": 13753, "dom": "G", "ent": "Khulm", "cls": "Western Europe"}, {"id": 13754, "dom": "G", "ent": "Taloqan", "cls": "Australia and New Zealand"}, {"id": 13755, "dom": "G", "ent": "Taloqan", "cls": "Caribbean"}, {"id": 13756, "dom": "G", "ent": "Taloqan", "cls": "Central America"}, {"id": 13757, "dom": "G", "ent": "Taloqan", "cls": "Central Asia"}, {"id": 13758, "dom": "G", "ent": "Taloqan", "cls": "Eastern Africa"}, {"id": 13759, "dom": "G", "ent": "Taloqan", "cls": "Eastern Asia"}, {"id": 13760, "dom": "G", "ent": "Taloqan", "cls": "Eastern Europe"}, {"id": 13761, "dom": "G", "ent": "Taloqan", "cls": "Melanesia"}, {"id": 13762, "dom": "G", "ent": "Taloqan", "cls": "Micronesia"}, {"id": 13763, "dom": "G", "ent": "Taloqan", "cls": "Middle Africa"}, {"id": 13764, "dom": "G", "ent": "Taloqan", "cls": "Northern Africa"}, {"id": 13765, "dom": "G", "ent": "Taloqan", "cls": "Northern America"}, {"id": 13766, "dom": "G", "ent": "Taloqan", "cls": "Northern Europe"}, {"id": 13767, "dom": "G", "ent": "Taloqan", "cls": "Polynesia"}, {"id": 13768, "dom": "G", "ent": "Taloqan", "cls": "South America"}, {"id": 13769, "dom": "G", "ent": "Taloqan", "cls": "South-eastern Asia"}, {"id": 13770, "dom": "G", "ent": "Taloqan", "cls": "Southern Africa"}, {"id": 13771, "dom": "G", "ent": "Taloqan", "cls": "Southern Asia"}, {"id": 13772, "dom": "G", "ent": "Taloqan", "cls": "Southern Europe"}, {"id": 13773, "dom": "G", "ent": "Taloqan", "cls": "Western Africa"}, {"id": 13774, "dom": "G", "ent": "Taloqan", "cls": "Western Asia"}, {"id": 13775, "dom": "G", "ent": "Taloqan", "cls": "Western Europe"}, {"id": 13776, "dom": "G", "ent": "Eslam Qaleh", "cls": "Australia and New Zealand"}, {"id": 13777, "dom": "G", "ent": "Eslam Qaleh", "cls": "Caribbean"}, {"id": 13778, "dom": "G", "ent": "Eslam Qaleh", "cls": "Central America"}, {"id": 13779, "dom": "G", "ent": "Eslam Qaleh", "cls": "Central Asia"}, {"id": 13780, "dom": "G", "ent": "Eslam Qaleh", "cls": "Eastern Africa"}, {"id": 13781, "dom": "G", "ent": "Eslam Qaleh", "cls": "Eastern Asia"}, {"id": 13782, "dom": "G", "ent": "Eslam Qaleh", "cls": "Eastern Europe"}, {"id": 13783, "dom": "G", "ent": "Eslam Qaleh", "cls": "Melanesia"}, {"id": 13784, "dom": "G", "ent": "Eslam Qaleh", "cls": "Micronesia"}, {"id": 13785, "dom": "G", "ent": "Eslam Qaleh", "cls": "Middle Africa"}, {"id": 13786, "dom": "G", "ent": "Eslam Qaleh", "cls": "Northern Africa"}, {"id": 13787, "dom": "G", "ent": "Eslam Qaleh", "cls": "Northern America"}, {"id": 13788, "dom": "G", "ent": "Eslam Qaleh", "cls": "Northern Europe"}, {"id": 13789, "dom": "G", "ent": "Eslam Qaleh", "cls": "Polynesia"}, {"id": 13790, "dom": "G", "ent": "Eslam Qaleh", "cls": "South America"}, {"id": 13791, "dom": "G", "ent": "Eslam Qaleh", "cls": "South-eastern Asia"}, {"id": 13792, "dom": "G", "ent": "Eslam Qaleh", "cls": "Southern Africa"}, {"id": 13793, "dom": "G", "ent": "Eslam Qaleh", "cls": "Southern Asia"}, {"id": 13794, "dom": "G", "ent": "Eslam Qaleh", "cls": "Southern Europe"}, {"id": 13795, "dom": "G", "ent": "Eslam Qaleh", "cls": "Western Africa"}, {"id": 13796, "dom": "G", "ent": "Eslam Qaleh", "cls": "Western Asia"}, {"id": 13797, "dom": "G", "ent": "Eslam Qaleh", "cls": "Western Europe"}, {"id": 13798, "dom": "G", "ent": "Karukh", "cls": "Australia and New Zealand"}, {"id": 13799, "dom": "G", "ent": "Karukh", "cls": "Caribbean"}, {"id": 13800, "dom": "G", "ent": "Karukh", "cls": "Central America"}, {"id": 13801, "dom": "G", "ent": "Karukh", "cls": "Central Asia"}, {"id": 13802, "dom": "G", "ent": "Karukh", "cls": "Eastern Africa"}, {"id": 13803, "dom": "G", "ent": "Karukh", "cls": "Eastern Asia"}, {"id": 13804, "dom": "G", "ent": "Karukh", "cls": "Eastern Europe"}, {"id": 13805, "dom": "G", "ent": "Karukh", "cls": "Melanesia"}, {"id": 13806, "dom": "G", "ent": "Karukh", "cls": "Micronesia"}, {"id": 13807, "dom": "G", "ent": "Karukh", "cls": "Middle Africa"}, {"id": 13808, "dom": "G", "ent": "Karukh", "cls": "Northern Africa"}, {"id": 13809, "dom": "G", "ent": "Karukh", "cls": "Northern America"}, {"id": 13810, "dom": "G", "ent": "Karukh", "cls": "Northern Europe"}, {"id": 13811, "dom": "G", "ent": "Karukh", "cls": "Polynesia"}, {"id": 13812, "dom": "G", "ent": "Karukh", "cls": "South America"}, {"id": 13813, "dom": "G", "ent": "Karukh", "cls": "South-eastern Asia"}, {"id": 13814, "dom": "G", "ent": "Karukh", "cls": "Southern Africa"}, {"id": 13815, "dom": "G", "ent": "Karukh", "cls": "Southern Asia"}, {"id": 13816, "dom": "G", "ent": "Karukh", "cls": "Southern Europe"}, {"id": 13817, "dom": "G", "ent": "Karukh", "cls": "Western Africa"}, {"id": 13818, "dom": "G", "ent": "Karukh", "cls": "Western Asia"}, {"id": 13819, "dom": "G", "ent": "Karukh", "cls": "Western Europe"}, {"id": 13820, "dom": "G", "ent": "Elbasan", "cls": "Australia and New Zealand"}, {"id": 13821, "dom": "G", "ent": "Elbasan", "cls": "Caribbean"}, {"id": 13822, "dom": "G", "ent": "Elbasan", "cls": "Central America"}, {"id": 13823, "dom": "G", "ent": "Elbasan", "cls": "Central Asia"}, {"id": 13824, "dom": "G", "ent": "Elbasan", "cls": "Eastern Africa"}, {"id": 13825, "dom": "G", "ent": "Elbasan", "cls": "Eastern Asia"}, {"id": 13826, "dom": "G", "ent": "Elbasan", "cls": "Eastern Europe"}, {"id": 13827, "dom": "G", "ent": "Elbasan", "cls": "Melanesia"}, {"id": 13828, "dom": "G", "ent": "Elbasan", "cls": "Micronesia"}, {"id": 13829, "dom": "G", "ent": "Elbasan", "cls": "Middle Africa"}, {"id": 13830, "dom": "G", "ent": "Elbasan", "cls": "Northern Africa"}, {"id": 13831, "dom": "G", "ent": "Elbasan", "cls": "Northern America"}, {"id": 13832, "dom": "G", "ent": "Elbasan", "cls": "Northern Europe"}, {"id": 13833, "dom": "G", "ent": "Elbasan", "cls": "Polynesia"}, {"id": 13834, "dom": "G", "ent": "Elbasan", "cls": "South America"}, {"id": 13835, "dom": "G", "ent": "Elbasan", "cls": "South-eastern Asia"}, {"id": 13836, "dom": "G", "ent": "Elbasan", "cls": "Southern Africa"}, {"id": 13837, "dom": "G", "ent": "Elbasan", "cls": "Southern Asia"}, {"id": 13838, "dom": "G", "ent": "Elbasan", "cls": "Southern Europe"}, {"id": 13839, "dom": "G", "ent": "Elbasan", "cls": "Western Africa"}, {"id": 13840, "dom": "G", "ent": "Elbasan", "cls": "Western Asia"}, {"id": 13841, "dom": "G", "ent": "Elbasan", "cls": "Western Europe"}, {"id": 13842, "dom": "G", "ent": "Berat", "cls": "Australia and New Zealand"}, {"id": 13843, "dom": "G", "ent": "Berat", "cls": "Caribbean"}, {"id": 13844, "dom": "G", "ent": "Berat", "cls": "Central America"}, {"id": 13845, "dom": "G", "ent": "Berat", "cls": "Central Asia"}, {"id": 13846, "dom": "G", "ent": "Berat", "cls": "Eastern Africa"}, {"id": 13847, "dom": "G", "ent": "Berat", "cls": "Eastern Asia"}, {"id": 13848, "dom": "G", "ent": "Berat", "cls": "Eastern Europe"}, {"id": 13849, "dom": "G", "ent": "Berat", "cls": "Melanesia"}, {"id": 13850, "dom": "G", "ent": "Berat", "cls": "Micronesia"}, {"id": 13851, "dom": "G", "ent": "Berat", "cls": "Middle Africa"}, {"id": 13852, "dom": "G", "ent": "Berat", "cls": "Northern Africa"}, {"id": 13853, "dom": "G", "ent": "Berat", "cls": "Northern America"}, {"id": 13854, "dom": "G", "ent": "Berat", "cls": "Northern Europe"}, {"id": 13855, "dom": "G", "ent": "Berat", "cls": "Polynesia"}, {"id": 13856, "dom": "G", "ent": "Berat", "cls": "South America"}, {"id": 13857, "dom": "G", "ent": "Berat", "cls": "South-eastern Asia"}, {"id": 13858, "dom": "G", "ent": "Berat", "cls": "Southern Africa"}, {"id": 13859, "dom": "G", "ent": "Berat", "cls": "Southern Asia"}, {"id": 13860, "dom": "G", "ent": "Berat", "cls": "Southern Europe"}, {"id": 13861, "dom": "G", "ent": "Berat", "cls": "Western Africa"}, {"id": 13862, "dom": "G", "ent": "Berat", "cls": "Western Asia"}, {"id": 13863, "dom": "G", "ent": "Berat", "cls": "Western Europe"}, {"id": 13864, "dom": "G", "ent": "Fier", "cls": "Australia and New Zealand"}, {"id": 13865, "dom": "G", "ent": "Fier", "cls": "Caribbean"}, {"id": 13866, "dom": "G", "ent": "Fier", "cls": "Central America"}, {"id": 13867, "dom": "G", "ent": "Fier", "cls": "Central Asia"}, {"id": 13868, "dom": "G", "ent": "Fier", "cls": "Eastern Africa"}, {"id": 13869, "dom": "G", "ent": "Fier", "cls": "Eastern Asia"}, {"id": 13870, "dom": "G", "ent": "Fier", "cls": "Eastern Europe"}, {"id": 13871, "dom": "G", "ent": "Fier", "cls": "Melanesia"}, {"id": 13872, "dom": "G", "ent": "Fier", "cls": "Micronesia"}, {"id": 13873, "dom": "G", "ent": "Fier", "cls": "Middle Africa"}, {"id": 13874, "dom": "G", "ent": "Fier", "cls": "Northern Africa"}, {"id": 13875, "dom": "G", "ent": "Fier", "cls": "Northern America"}, {"id": 13876, "dom": "G", "ent": "Fier", "cls": "Northern Europe"}, {"id": 13877, "dom": "G", "ent": "Fier", "cls": "Polynesia"}, {"id": 13878, "dom": "G", "ent": "Fier", "cls": "South America"}, {"id": 13879, "dom": "G", "ent": "Fier", "cls": "South-eastern Asia"}, {"id": 13880, "dom": "G", "ent": "Fier", "cls": "Southern Africa"}, {"id": 13881, "dom": "G", "ent": "Fier", "cls": "Southern Asia"}, {"id": 13882, "dom": "G", "ent": "Fier", "cls": "Southern Europe"}, {"id": 13883, "dom": "G", "ent": "Fier", "cls": "Western Africa"}, {"id": 13884, "dom": "G", "ent": "Fier", "cls": "Western Asia"}, {"id": 13885, "dom": "G", "ent": "Fier", "cls": "Western Europe"}, {"id": 13886, "dom": "G", "ent": "Peqin", "cls": "Australia and New Zealand"}, {"id": 13887, "dom": "G", "ent": "Peqin", "cls": "Caribbean"}, {"id": 13888, "dom": "G", "ent": "Peqin", "cls": "Central America"}, {"id": 13889, "dom": "G", "ent": "Peqin", "cls": "Central Asia"}, {"id": 13890, "dom": "G", "ent": "Peqin", "cls": "Eastern Africa"}, {"id": 13891, "dom": "G", "ent": "Peqin", "cls": "Eastern Asia"}, {"id": 13892, "dom": "G", "ent": "Peqin", "cls": "Eastern Europe"}, {"id": 13893, "dom": "G", "ent": "Peqin", "cls": "Melanesia"}, {"id": 13894, "dom": "G", "ent": "Peqin", "cls": "Micronesia"}, {"id": 13895, "dom": "G", "ent": "Peqin", "cls": "Middle Africa"}, {"id": 13896, "dom": "G", "ent": "Peqin", "cls": "Northern Africa"}, {"id": 13897, "dom": "G", "ent": "Peqin", "cls": "Northern America"}, {"id": 13898, "dom": "G", "ent": "Peqin", "cls": "Northern Europe"}, {"id": 13899, "dom": "G", "ent": "Peqin", "cls": "Polynesia"}, {"id": 13900, "dom": "G", "ent": "Peqin", "cls": "South America"}, {"id": 13901, "dom": "G", "ent": "Peqin", "cls": "South-eastern Asia"}, {"id": 13902, "dom": "G", "ent": "Peqin", "cls": "Southern Africa"}, {"id": 13903, "dom": "G", "ent": "Peqin", "cls": "Southern Asia"}, {"id": 13904, "dom": "G", "ent": "Peqin", "cls": "Southern Europe"}, {"id": 13905, "dom": "G", "ent": "Peqin", "cls": "Western Africa"}, {"id": 13906, "dom": "G", "ent": "Peqin", "cls": "Western Asia"}, {"id": 13907, "dom": "G", "ent": "Peqin", "cls": "Western Europe"}, {"id": 13908, "dom": "G", "ent": "Luquembo", "cls": "Australia and New Zealand"}, {"id": 13909, "dom": "G", "ent": "Luquembo", "cls": "Caribbean"}, {"id": 13910, "dom": "G", "ent": "Luquembo", "cls": "Central America"}, {"id": 13911, "dom": "G", "ent": "Luquembo", "cls": "Central Asia"}, {"id": 13912, "dom": "G", "ent": "Luquembo", "cls": "Eastern Africa"}, {"id": 13913, "dom": "G", "ent": "Luquembo", "cls": "Eastern Asia"}, {"id": 13914, "dom": "G", "ent": "Luquembo", "cls": "Eastern Europe"}, {"id": 13915, "dom": "G", "ent": "Luquembo", "cls": "Melanesia"}, {"id": 13916, "dom": "G", "ent": "Luquembo", "cls": "Micronesia"}, {"id": 13917, "dom": "G", "ent": "Luquembo", "cls": "Middle Africa"}, {"id": 13918, "dom": "G", "ent": "Luquembo", "cls": "Northern Africa"}, {"id": 13919, "dom": "G", "ent": "Luquembo", "cls": "Northern America"}, {"id": 13920, "dom": "G", "ent": "Luquembo", "cls": "Northern Europe"}, {"id": 13921, "dom": "G", "ent": "Luquembo", "cls": "Polynesia"}, {"id": 13922, "dom": "G", "ent": "Luquembo", "cls": "South America"}, {"id": 13923, "dom": "G", "ent": "Luquembo", "cls": "South-eastern Asia"}, {"id": 13924, "dom": "G", "ent": "Luquembo", "cls": "Southern Africa"}, {"id": 13925, "dom": "G", "ent": "Luquembo", "cls": "Southern Asia"}, {"id": 13926, "dom": "G", "ent": "Luquembo", "cls": "Southern Europe"}, {"id": 13927, "dom": "G", "ent": "Luquembo", "cls": "Western Africa"}, {"id": 13928, "dom": "G", "ent": "Luquembo", "cls": "Western Asia"}, {"id": 13929, "dom": "G", "ent": "Luquembo", "cls": "Western Europe"}, {"id": 13930, "dom": "G", "ent": "N'zeto", "cls": "Australia and New Zealand"}, {"id": 13931, "dom": "G", "ent": "N'zeto", "cls": "Caribbean"}, {"id": 13932, "dom": "G", "ent": "N'zeto", "cls": "Central America"}, {"id": 13933, "dom": "G", "ent": "N'zeto", "cls": "Central Asia"}, {"id": 13934, "dom": "G", "ent": "N'zeto", "cls": "Eastern Africa"}, {"id": 13935, "dom": "G", "ent": "N'zeto", "cls": "Eastern Asia"}, {"id": 13936, "dom": "G", "ent": "N'zeto", "cls": "Eastern Europe"}, {"id": 13937, "dom": "G", "ent": "N'zeto", "cls": "Melanesia"}, {"id": 13938, "dom": "G", "ent": "N'zeto", "cls": "Micronesia"}, {"id": 13939, "dom": "G", "ent": "N'zeto", "cls": "Middle Africa"}, {"id": 13940, "dom": "G", "ent": "N'zeto", "cls": "Northern Africa"}, {"id": 13941, "dom": "G", "ent": "N'zeto", "cls": "Northern America"}, {"id": 13942, "dom": "G", "ent": "N'zeto", "cls": "Northern Europe"}, {"id": 13943, "dom": "G", "ent": "N'zeto", "cls": "Polynesia"}, {"id": 13944, "dom": "G", "ent": "N'zeto", "cls": "South America"}, {"id": 13945, "dom": "G", "ent": "N'zeto", "cls": "South-eastern Asia"}, {"id": 13946, "dom": "G", "ent": "N'zeto", "cls": "Southern Africa"}, {"id": 13947, "dom": "G", "ent": "N'zeto", "cls": "Southern Asia"}, {"id": 13948, "dom": "G", "ent": "N'zeto", "cls": "Southern Europe"}, {"id": 13949, "dom": "G", "ent": "N'zeto", "cls": "Western Africa"}, {"id": 13950, "dom": "G", "ent": "N'zeto", "cls": "Western Asia"}, {"id": 13951, "dom": "G", "ent": "N'zeto", "cls": "Western Europe"}, {"id": 13952, "dom": "G", "ent": "Quipungo", "cls": "Australia and New Zealand"}, {"id": 13953, "dom": "G", "ent": "Quipungo", "cls": "Caribbean"}, {"id": 13954, "dom": "G", "ent": "Quipungo", "cls": "Central America"}, {"id": 13955, "dom": "G", "ent": "Quipungo", "cls": "Central Asia"}, {"id": 13956, "dom": "G", "ent": "Quipungo", "cls": "Eastern Africa"}, {"id": 13957, "dom": "G", "ent": "Quipungo", "cls": "Eastern Asia"}, {"id": 13958, "dom": "G", "ent": "Quipungo", "cls": "Eastern Europe"}, {"id": 13959, "dom": "G", "ent": "Quipungo", "cls": "Melanesia"}, {"id": 13960, "dom": "G", "ent": "Quipungo", "cls": "Micronesia"}, {"id": 13961, "dom": "G", "ent": "Quipungo", "cls": "Middle Africa"}, {"id": 13962, "dom": "G", "ent": "Quipungo", "cls": "Northern Africa"}, {"id": 13963, "dom": "G", "ent": "Quipungo", "cls": "Northern America"}, {"id": 13964, "dom": "G", "ent": "Quipungo", "cls": "Northern Europe"}, {"id": 13965, "dom": "G", "ent": "Quipungo", "cls": "Polynesia"}, {"id": 13966, "dom": "G", "ent": "Quipungo", "cls": "South America"}, {"id": 13967, "dom": "G", "ent": "Quipungo", "cls": "South-eastern Asia"}, {"id": 13968, "dom": "G", "ent": "Quipungo", "cls": "Southern Africa"}, {"id": 13969, "dom": "G", "ent": "Quipungo", "cls": "Southern Asia"}, {"id": 13970, "dom": "G", "ent": "Quipungo", "cls": "Southern Europe"}, {"id": 13971, "dom": "G", "ent": "Quipungo", "cls": "Western Africa"}, {"id": 13972, "dom": "G", "ent": "Quipungo", "cls": "Western Asia"}, {"id": 13973, "dom": "G", "ent": "Quipungo", "cls": "Western Europe"}, {"id": 13974, "dom": "G", "ent": "Kunda dya Baze", "cls": "Australia and New Zealand"}, {"id": 13975, "dom": "G", "ent": "Kunda dya Baze", "cls": "Caribbean"}, {"id": 13976, "dom": "G", "ent": "Kunda dya Baze", "cls": "Central America"}, {"id": 13977, "dom": "G", "ent": "Kunda dya Baze", "cls": "Central Asia"}, {"id": 13978, "dom": "G", "ent": "Kunda dya Baze", "cls": "Eastern Africa"}, {"id": 13979, "dom": "G", "ent": "Kunda dya Baze", "cls": "Eastern Asia"}, {"id": 13980, "dom": "G", "ent": "Kunda dya Baze", "cls": "Eastern Europe"}, {"id": 13981, "dom": "G", "ent": "Kunda dya Baze", "cls": "Melanesia"}, {"id": 13982, "dom": "G", "ent": "Kunda dya Baze", "cls": "Micronesia"}, {"id": 13983, "dom": "G", "ent": "Kunda dya Baze", "cls": "Middle Africa"}, {"id": 13984, "dom": "G", "ent": "Kunda dya Baze", "cls": "Northern Africa"}, {"id": 13985, "dom": "G", "ent": "Kunda dya Baze", "cls": "Northern America"}, {"id": 13986, "dom": "G", "ent": "Kunda dya Baze", "cls": "Northern Europe"}, {"id": 13987, "dom": "G", "ent": "Kunda dya Baze", "cls": "Polynesia"}, {"id": 13988, "dom": "G", "ent": "Kunda dya Baze", "cls": "South America"}, {"id": 13989, "dom": "G", "ent": "Kunda dya Baze", "cls": "South-eastern Asia"}, {"id": 13990, "dom": "G", "ent": "Kunda dya Baze", "cls": "Southern Africa"}, {"id": 13991, "dom": "G", "ent": "Kunda dya Baze", "cls": "Southern Asia"}, {"id": 13992, "dom": "G", "ent": "Kunda dya Baze", "cls": "Southern Europe"}, {"id": 13993, "dom": "G", "ent": "Kunda dya Baze", "cls": "Western Africa"}, {"id": 13994, "dom": "G", "ent": "Kunda dya Baze", "cls": "Western Asia"}, {"id": 13995, "dom": "G", "ent": "Kunda dya Baze", "cls": "Western Europe"}, {"id": 13996, "dom": "G", "ent": "Lomas de Zamora", "cls": "Australia and New Zealand"}, {"id": 13997, "dom": "G", "ent": "Lomas de Zamora", "cls": "Caribbean"}, {"id": 13998, "dom": "G", "ent": "Lomas de Zamora", "cls": "Central America"}, {"id": 13999, "dom": "G", "ent": "Lomas de Zamora", "cls": "Central Asia"}, {"id": 14000, "dom": "G", "ent": "Lomas de Zamora", "cls": "Eastern Africa"}, {"id": 14001, "dom": "G", "ent": "Lomas de Zamora", "cls": "Eastern Asia"}, {"id": 14002, "dom": "G", "ent": "Lomas de Zamora", "cls": "Eastern Europe"}, {"id": 14003, "dom": "G", "ent": "Lomas de Zamora", "cls": "Melanesia"}, {"id": 14004, "dom": "G", "ent": "Lomas de Zamora", "cls": "Micronesia"}, {"id": 14005, "dom": "G", "ent": "Lomas de Zamora", "cls": "Middle Africa"}, {"id": 14006, "dom": "G", "ent": "Lomas de Zamora", "cls": "Northern Africa"}, {"id": 14007, "dom": "G", "ent": "Lomas de Zamora", "cls": "Northern America"}, {"id": 14008, "dom": "G", "ent": "Lomas de Zamora", "cls": "Northern Europe"}, {"id": 14009, "dom": "G", "ent": "Lomas de Zamora", "cls": "Polynesia"}, {"id": 14010, "dom": "G", "ent": "Lomas de Zamora", "cls": "South America"}, {"id": 14011, "dom": "G", "ent": "Lomas de Zamora", "cls": "South-eastern Asia"}, {"id": 14012, "dom": "G", "ent": "Lomas de Zamora", "cls": "Southern Africa"}, {"id": 14013, "dom": "G", "ent": "Lomas de Zamora", "cls": "Southern Asia"}, {"id": 14014, "dom": "G", "ent": "Lomas de Zamora", "cls": "Southern Europe"}, {"id": 14015, "dom": "G", "ent": "Lomas de Zamora", "cls": "Western Africa"}, {"id": 14016, "dom": "G", "ent": "Lomas de Zamora", "cls": "Western Asia"}, {"id": 14017, "dom": "G", "ent": "Lomas de Zamora", "cls": "Western Europe"}, {"id": 14018, "dom": "G", "ent": "Caleta Olivia", "cls": "Australia and New Zealand"}, {"id": 14019, "dom": "G", "ent": "Caleta Olivia", "cls": "Caribbean"}, {"id": 14020, "dom": "G", "ent": "Caleta Olivia", "cls": "Central America"}, {"id": 14021, "dom": "G", "ent": "Caleta Olivia", "cls": "Central Asia"}, {"id": 14022, "dom": "G", "ent": "Caleta Olivia", "cls": "Eastern Africa"}, {"id": 14023, "dom": "G", "ent": "Caleta Olivia", "cls": "Eastern Asia"}, {"id": 14024, "dom": "G", "ent": "Caleta Olivia", "cls": "Eastern Europe"}, {"id": 14025, "dom": "G", "ent": "Caleta Olivia", "cls": "Melanesia"}, {"id": 14026, "dom": "G", "ent": "Caleta Olivia", "cls": "Micronesia"}, {"id": 14027, "dom": "G", "ent": "Caleta Olivia", "cls": "Middle Africa"}, {"id": 14028, "dom": "G", "ent": "Caleta Olivia", "cls": "Northern Africa"}, {"id": 14029, "dom": "G", "ent": "Caleta Olivia", "cls": "Northern America"}, {"id": 14030, "dom": "G", "ent": "Caleta Olivia", "cls": "Northern Europe"}, {"id": 14031, "dom": "G", "ent": "Caleta Olivia", "cls": "Polynesia"}, {"id": 14032, "dom": "G", "ent": "Caleta Olivia", "cls": "South America"}, {"id": 14033, "dom": "G", "ent": "Caleta Olivia", "cls": "South-eastern Asia"}, {"id": 14034, "dom": "G", "ent": "Caleta Olivia", "cls": "Southern Africa"}, {"id": 14035, "dom": "G", "ent": "Caleta Olivia", "cls": "Southern Asia"}, {"id": 14036, "dom": "G", "ent": "Caleta Olivia", "cls": "Southern Europe"}, {"id": 14037, "dom": "G", "ent": "Caleta Olivia", "cls": "Western Africa"}, {"id": 14038, "dom": "G", "ent": "Caleta Olivia", "cls": "Western Asia"}, {"id": 14039, "dom": "G", "ent": "Caleta Olivia", "cls": "Western Europe"}, {"id": 14040, "dom": "G", "ent": "Lobos", "cls": "Australia and New Zealand"}, {"id": 14041, "dom": "G", "ent": "Lobos", "cls": "Caribbean"}, {"id": 14042, "dom": "G", "ent": "Lobos", "cls": "Central America"}, {"id": 14043, "dom": "G", "ent": "Lobos", "cls": "Central Asia"}, {"id": 14044, "dom": "G", "ent": "Lobos", "cls": "Eastern Africa"}, {"id": 14045, "dom": "G", "ent": "Lobos", "cls": "Eastern Asia"}, {"id": 14046, "dom": "G", "ent": "Lobos", "cls": "Eastern Europe"}, {"id": 14047, "dom": "G", "ent": "Lobos", "cls": "Melanesia"}, {"id": 14048, "dom": "G", "ent": "Lobos", "cls": "Micronesia"}, {"id": 14049, "dom": "G", "ent": "Lobos", "cls": "Middle Africa"}, {"id": 14050, "dom": "G", "ent": "Lobos", "cls": "Northern Africa"}, {"id": 14051, "dom": "G", "ent": "Lobos", "cls": "Northern America"}, {"id": 14052, "dom": "G", "ent": "Lobos", "cls": "Northern Europe"}, {"id": 14053, "dom": "G", "ent": "Lobos", "cls": "Polynesia"}, {"id": 14054, "dom": "G", "ent": "Lobos", "cls": "South America"}, {"id": 14055, "dom": "G", "ent": "Lobos", "cls": "South-eastern Asia"}, {"id": 14056, "dom": "G", "ent": "Lobos", "cls": "Southern Africa"}, {"id": 14057, "dom": "G", "ent": "Lobos", "cls": "Southern Asia"}, {"id": 14058, "dom": "G", "ent": "Lobos", "cls": "Southern Europe"}, {"id": 14059, "dom": "G", "ent": "Lobos", "cls": "Western Africa"}, {"id": 14060, "dom": "G", "ent": "Lobos", "cls": "Western Asia"}, {"id": 14061, "dom": "G", "ent": "Lobos", "cls": "Western Europe"}, {"id": 14062, "dom": "G", "ent": "La Falda", "cls": "Australia and New Zealand"}, {"id": 14063, "dom": "G", "ent": "La Falda", "cls": "Caribbean"}, {"id": 14064, "dom": "G", "ent": "La Falda", "cls": "Central America"}, {"id": 14065, "dom": "G", "ent": "La Falda", "cls": "Central Asia"}, {"id": 14066, "dom": "G", "ent": "La Falda", "cls": "Eastern Africa"}, {"id": 14067, "dom": "G", "ent": "La Falda", "cls": "Eastern Asia"}, {"id": 14068, "dom": "G", "ent": "La Falda", "cls": "Eastern Europe"}, {"id": 14069, "dom": "G", "ent": "La Falda", "cls": "Melanesia"}, {"id": 14070, "dom": "G", "ent": "La Falda", "cls": "Micronesia"}, {"id": 14071, "dom": "G", "ent": "La Falda", "cls": "Middle Africa"}, {"id": 14072, "dom": "G", "ent": "La Falda", "cls": "Northern Africa"}, {"id": 14073, "dom": "G", "ent": "La Falda", "cls": "Northern America"}, {"id": 14074, "dom": "G", "ent": "La Falda", "cls": "Northern Europe"}, {"id": 14075, "dom": "G", "ent": "La Falda", "cls": "Polynesia"}, {"id": 14076, "dom": "G", "ent": "La Falda", "cls": "South America"}, {"id": 14077, "dom": "G", "ent": "La Falda", "cls": "South-eastern Asia"}, {"id": 14078, "dom": "G", "ent": "La Falda", "cls": "Southern Africa"}, {"id": 14079, "dom": "G", "ent": "La Falda", "cls": "Southern Asia"}, {"id": 14080, "dom": "G", "ent": "La Falda", "cls": "Southern Europe"}, {"id": 14081, "dom": "G", "ent": "La Falda", "cls": "Western Africa"}, {"id": 14082, "dom": "G", "ent": "La Falda", "cls": "Western Asia"}, {"id": 14083, "dom": "G", "ent": "La Falda", "cls": "Western Europe"}, {"id": 14084, "dom": "G", "ent": "Brigittenau", "cls": "Australia and New Zealand"}, {"id": 14085, "dom": "G", "ent": "Brigittenau", "cls": "Caribbean"}, {"id": 14086, "dom": "G", "ent": "Brigittenau", "cls": "Central America"}, {"id": 14087, "dom": "G", "ent": "Brigittenau", "cls": "Central Asia"}, {"id": 14088, "dom": "G", "ent": "Brigittenau", "cls": "Eastern Africa"}, {"id": 14089, "dom": "G", "ent": "Brigittenau", "cls": "Eastern Asia"}, {"id": 14090, "dom": "G", "ent": "Brigittenau", "cls": "Eastern Europe"}, {"id": 14091, "dom": "G", "ent": "Brigittenau", "cls": "Melanesia"}, {"id": 14092, "dom": "G", "ent": "Brigittenau", "cls": "Micronesia"}, {"id": 14093, "dom": "G", "ent": "Brigittenau", "cls": "Middle Africa"}, {"id": 14094, "dom": "G", "ent": "Brigittenau", "cls": "Northern Africa"}, {"id": 14095, "dom": "G", "ent": "Brigittenau", "cls": "Northern America"}, {"id": 14096, "dom": "G", "ent": "Brigittenau", "cls": "Northern Europe"}, {"id": 14097, "dom": "G", "ent": "Brigittenau", "cls": "Polynesia"}, {"id": 14098, "dom": "G", "ent": "Brigittenau", "cls": "South America"}, {"id": 14099, "dom": "G", "ent": "Brigittenau", "cls": "South-eastern Asia"}, {"id": 14100, "dom": "G", "ent": "Brigittenau", "cls": "Southern Africa"}, {"id": 14101, "dom": "G", "ent": "Brigittenau", "cls": "Southern Asia"}, {"id": 14102, "dom": "G", "ent": "Brigittenau", "cls": "Southern Europe"}, {"id": 14103, "dom": "G", "ent": "Brigittenau", "cls": "Western Africa"}, {"id": 14104, "dom": "G", "ent": "Brigittenau", "cls": "Western Asia"}, {"id": 14105, "dom": "G", "ent": "Brigittenau", "cls": "Western Europe"}, {"id": 14106, "dom": "G", "ent": "Alsergrund", "cls": "Australia and New Zealand"}, {"id": 14107, "dom": "G", "ent": "Alsergrund", "cls": "Caribbean"}, {"id": 14108, "dom": "G", "ent": "Alsergrund", "cls": "Central America"}, {"id": 14109, "dom": "G", "ent": "Alsergrund", "cls": "Central Asia"}, {"id": 14110, "dom": "G", "ent": "Alsergrund", "cls": "Eastern Africa"}, {"id": 14111, "dom": "G", "ent": "Alsergrund", "cls": "Eastern Asia"}, {"id": 14112, "dom": "G", "ent": "Alsergrund", "cls": "Eastern Europe"}, {"id": 14113, "dom": "G", "ent": "Alsergrund", "cls": "Melanesia"}, {"id": 14114, "dom": "G", "ent": "Alsergrund", "cls": "Micronesia"}, {"id": 14115, "dom": "G", "ent": "Alsergrund", "cls": "Middle Africa"}, {"id": 14116, "dom": "G", "ent": "Alsergrund", "cls": "Northern Africa"}, {"id": 14117, "dom": "G", "ent": "Alsergrund", "cls": "Northern America"}, {"id": 14118, "dom": "G", "ent": "Alsergrund", "cls": "Northern Europe"}, {"id": 14119, "dom": "G", "ent": "Alsergrund", "cls": "Polynesia"}, {"id": 14120, "dom": "G", "ent": "Alsergrund", "cls": "South America"}, {"id": 14121, "dom": "G", "ent": "Alsergrund", "cls": "South-eastern Asia"}, {"id": 14122, "dom": "G", "ent": "Alsergrund", "cls": "Southern Africa"}, {"id": 14123, "dom": "G", "ent": "Alsergrund", "cls": "Southern Asia"}, {"id": 14124, "dom": "G", "ent": "Alsergrund", "cls": "Southern Europe"}, {"id": 14125, "dom": "G", "ent": "Alsergrund", "cls": "Western Africa"}, {"id": 14126, "dom": "G", "ent": "Alsergrund", "cls": "Western Asia"}, {"id": 14127, "dom": "G", "ent": "Alsergrund", "cls": "Western Europe"}, {"id": 14128, "dom": "G", "ent": "Mariahilf", "cls": "Australia and New Zealand"}, {"id": 14129, "dom": "G", "ent": "Mariahilf", "cls": "Caribbean"}, {"id": 14130, "dom": "G", "ent": "Mariahilf", "cls": "Central America"}, {"id": 14131, "dom": "G", "ent": "Mariahilf", "cls": "Central Asia"}, {"id": 14132, "dom": "G", "ent": "Mariahilf", "cls": "Eastern Africa"}, {"id": 14133, "dom": "G", "ent": "Mariahilf", "cls": "Eastern Asia"}, {"id": 14134, "dom": "G", "ent": "Mariahilf", "cls": "Eastern Europe"}, {"id": 14135, "dom": "G", "ent": "Mariahilf", "cls": "Melanesia"}, {"id": 14136, "dom": "G", "ent": "Mariahilf", "cls": "Micronesia"}, {"id": 14137, "dom": "G", "ent": "Mariahilf", "cls": "Middle Africa"}, {"id": 14138, "dom": "G", "ent": "Mariahilf", "cls": "Northern Africa"}, {"id": 14139, "dom": "G", "ent": "Mariahilf", "cls": "Northern America"}, {"id": 14140, "dom": "G", "ent": "Mariahilf", "cls": "Northern Europe"}, {"id": 14141, "dom": "G", "ent": "Mariahilf", "cls": "Polynesia"}, {"id": 14142, "dom": "G", "ent": "Mariahilf", "cls": "South America"}, {"id": 14143, "dom": "G", "ent": "Mariahilf", "cls": "South-eastern Asia"}, {"id": 14144, "dom": "G", "ent": "Mariahilf", "cls": "Southern Africa"}, {"id": 14145, "dom": "G", "ent": "Mariahilf", "cls": "Southern Asia"}, {"id": 14146, "dom": "G", "ent": "Mariahilf", "cls": "Southern Europe"}, {"id": 14147, "dom": "G", "ent": "Mariahilf", "cls": "Western Africa"}, {"id": 14148, "dom": "G", "ent": "Mariahilf", "cls": "Western Asia"}, {"id": 14149, "dom": "G", "ent": "Mariahilf", "cls": "Western Europe"}, {"id": 14150, "dom": "G", "ent": "Sankt Leonhard", "cls": "Australia and New Zealand"}, {"id": 14151, "dom": "G", "ent": "Sankt Leonhard", "cls": "Caribbean"}, {"id": 14152, "dom": "G", "ent": "Sankt Leonhard", "cls": "Central America"}, {"id": 14153, "dom": "G", "ent": "Sankt Leonhard", "cls": "Central Asia"}, {"id": 14154, "dom": "G", "ent": "Sankt Leonhard", "cls": "Eastern Africa"}, {"id": 14155, "dom": "G", "ent": "Sankt Leonhard", "cls": "Eastern Asia"}, {"id": 14156, "dom": "G", "ent": "Sankt Leonhard", "cls": "Eastern Europe"}, {"id": 14157, "dom": "G", "ent": "Sankt Leonhard", "cls": "Melanesia"}, {"id": 14158, "dom": "G", "ent": "Sankt Leonhard", "cls": "Micronesia"}, {"id": 14159, "dom": "G", "ent": "Sankt Leonhard", "cls": "Middle Africa"}, {"id": 14160, "dom": "G", "ent": "Sankt Leonhard", "cls": "Northern Africa"}, {"id": 14161, "dom": "G", "ent": "Sankt Leonhard", "cls": "Northern America"}, {"id": 14162, "dom": "G", "ent": "Sankt Leonhard", "cls": "Northern Europe"}, {"id": 14163, "dom": "G", "ent": "Sankt Leonhard", "cls": "Polynesia"}, {"id": 14164, "dom": "G", "ent": "Sankt Leonhard", "cls": "South America"}, {"id": 14165, "dom": "G", "ent": "Sankt Leonhard", "cls": "South-eastern Asia"}, {"id": 14166, "dom": "G", "ent": "Sankt Leonhard", "cls": "Southern Africa"}, {"id": 14167, "dom": "G", "ent": "Sankt Leonhard", "cls": "Southern Asia"}, {"id": 14168, "dom": "G", "ent": "Sankt Leonhard", "cls": "Southern Europe"}, {"id": 14169, "dom": "G", "ent": "Sankt Leonhard", "cls": "Western Africa"}, {"id": 14170, "dom": "G", "ent": "Sankt Leonhard", "cls": "Western Asia"}, {"id": 14171, "dom": "G", "ent": "Sankt Leonhard", "cls": "Western Europe"}, {"id": 14172, "dom": "G", "ent": "Caloundra", "cls": "Australia and New Zealand"}, {"id": 14173, "dom": "G", "ent": "Caloundra", "cls": "Caribbean"}, {"id": 14174, "dom": "G", "ent": "Caloundra", "cls": "Central America"}, {"id": 14175, "dom": "G", "ent": "Caloundra", "cls": "Central Asia"}, {"id": 14176, "dom": "G", "ent": "Caloundra", "cls": "Eastern Africa"}, {"id": 14177, "dom": "G", "ent": "Caloundra", "cls": "Eastern Asia"}, {"id": 14178, "dom": "G", "ent": "Caloundra", "cls": "Eastern Europe"}, {"id": 14179, "dom": "G", "ent": "Caloundra", "cls": "Melanesia"}, {"id": 14180, "dom": "G", "ent": "Caloundra", "cls": "Micronesia"}, {"id": 14181, "dom": "G", "ent": "Caloundra", "cls": "Middle Africa"}, {"id": 14182, "dom": "G", "ent": "Caloundra", "cls": "Northern Africa"}, {"id": 14183, "dom": "G", "ent": "Caloundra", "cls": "Northern America"}, {"id": 14184, "dom": "G", "ent": "Caloundra", "cls": "Northern Europe"}, {"id": 14185, "dom": "G", "ent": "Caloundra", "cls": "Polynesia"}, {"id": 14186, "dom": "G", "ent": "Caloundra", "cls": "South America"}, {"id": 14187, "dom": "G", "ent": "Caloundra", "cls": "South-eastern Asia"}, {"id": 14188, "dom": "G", "ent": "Caloundra", "cls": "Southern Africa"}, {"id": 14189, "dom": "G", "ent": "Caloundra", "cls": "Southern Asia"}, {"id": 14190, "dom": "G", "ent": "Caloundra", "cls": "Southern Europe"}, {"id": 14191, "dom": "G", "ent": "Caloundra", "cls": "Western Africa"}, {"id": 14192, "dom": "G", "ent": "Caloundra", "cls": "Western Asia"}, {"id": 14193, "dom": "G", "ent": "Caloundra", "cls": "Western Europe"}, {"id": 14194, "dom": "G", "ent": "Randwick", "cls": "Australia and New Zealand"}, {"id": 14195, "dom": "G", "ent": "Randwick", "cls": "Caribbean"}, {"id": 14196, "dom": "G", "ent": "Randwick", "cls": "Central America"}, {"id": 14197, "dom": "G", "ent": "Randwick", "cls": "Central Asia"}, {"id": 14198, "dom": "G", "ent": "Randwick", "cls": "Eastern Africa"}, {"id": 14199, "dom": "G", "ent": "Randwick", "cls": "Eastern Asia"}, {"id": 14200, "dom": "G", "ent": "Randwick", "cls": "Eastern Europe"}, {"id": 14201, "dom": "G", "ent": "Randwick", "cls": "Melanesia"}, {"id": 14202, "dom": "G", "ent": "Randwick", "cls": "Micronesia"}, {"id": 14203, "dom": "G", "ent": "Randwick", "cls": "Middle Africa"}, {"id": 14204, "dom": "G", "ent": "Randwick", "cls": "Northern Africa"}, {"id": 14205, "dom": "G", "ent": "Randwick", "cls": "Northern America"}, {"id": 14206, "dom": "G", "ent": "Randwick", "cls": "Northern Europe"}, {"id": 14207, "dom": "G", "ent": "Randwick", "cls": "Polynesia"}, {"id": 14208, "dom": "G", "ent": "Randwick", "cls": "South America"}, {"id": 14209, "dom": "G", "ent": "Randwick", "cls": "South-eastern Asia"}, {"id": 14210, "dom": "G", "ent": "Randwick", "cls": "Southern Africa"}, {"id": 14211, "dom": "G", "ent": "Randwick", "cls": "Southern Asia"}, {"id": 14212, "dom": "G", "ent": "Randwick", "cls": "Southern Europe"}, {"id": 14213, "dom": "G", "ent": "Randwick", "cls": "Western Africa"}, {"id": 14214, "dom": "G", "ent": "Randwick", "cls": "Western Asia"}, {"id": 14215, "dom": "G", "ent": "Randwick", "cls": "Western Europe"}, {"id": 14216, "dom": "G", "ent": "Dandenong North", "cls": "Australia and New Zealand"}, {"id": 14217, "dom": "G", "ent": "Dandenong North", "cls": "Caribbean"}, {"id": 14218, "dom": "G", "ent": "Dandenong North", "cls": "Central America"}, {"id": 14219, "dom": "G", "ent": "Dandenong North", "cls": "Central Asia"}, {"id": 14220, "dom": "G", "ent": "Dandenong North", "cls": "Eastern Africa"}, {"id": 14221, "dom": "G", "ent": "Dandenong North", "cls": "Eastern Asia"}, {"id": 14222, "dom": "G", "ent": "Dandenong North", "cls": "Eastern Europe"}, {"id": 14223, "dom": "G", "ent": "Dandenong North", "cls": "Melanesia"}, {"id": 14224, "dom": "G", "ent": "Dandenong North", "cls": "Micronesia"}, {"id": 14225, "dom": "G", "ent": "Dandenong North", "cls": "Middle Africa"}, {"id": 14226, "dom": "G", "ent": "Dandenong North", "cls": "Northern Africa"}, {"id": 14227, "dom": "G", "ent": "Dandenong North", "cls": "Northern America"}, {"id": 14228, "dom": "G", "ent": "Dandenong North", "cls": "Northern Europe"}, {"id": 14229, "dom": "G", "ent": "Dandenong North", "cls": "Polynesia"}, {"id": 14230, "dom": "G", "ent": "Dandenong North", "cls": "South America"}, {"id": 14231, "dom": "G", "ent": "Dandenong North", "cls": "South-eastern Asia"}, {"id": 14232, "dom": "G", "ent": "Dandenong North", "cls": "Southern Africa"}, {"id": 14233, "dom": "G", "ent": "Dandenong North", "cls": "Southern Asia"}, {"id": 14234, "dom": "G", "ent": "Dandenong North", "cls": "Southern Europe"}, {"id": 14235, "dom": "G", "ent": "Dandenong North", "cls": "Western Africa"}, {"id": 14236, "dom": "G", "ent": "Dandenong North", "cls": "Western Asia"}, {"id": 14237, "dom": "G", "ent": "Dandenong North", "cls": "Western Europe"}, {"id": 14238, "dom": "G", "ent": "Hawthorn", "cls": "Australia and New Zealand"}, {"id": 14239, "dom": "G", "ent": "Hawthorn", "cls": "Caribbean"}, {"id": 14240, "dom": "G", "ent": "Hawthorn", "cls": "Central America"}, {"id": 14241, "dom": "G", "ent": "Hawthorn", "cls": "Central Asia"}, {"id": 14242, "dom": "G", "ent": "Hawthorn", "cls": "Eastern Africa"}, {"id": 14243, "dom": "G", "ent": "Hawthorn", "cls": "Eastern Asia"}, {"id": 14244, "dom": "G", "ent": "Hawthorn", "cls": "Eastern Europe"}, {"id": 14245, "dom": "G", "ent": "Hawthorn", "cls": "Melanesia"}, {"id": 14246, "dom": "G", "ent": "Hawthorn", "cls": "Micronesia"}, {"id": 14247, "dom": "G", "ent": "Hawthorn", "cls": "Middle Africa"}, {"id": 14248, "dom": "G", "ent": "Hawthorn", "cls": "Northern Africa"}, {"id": 14249, "dom": "G", "ent": "Hawthorn", "cls": "Northern America"}, {"id": 14250, "dom": "G", "ent": "Hawthorn", "cls": "Northern Europe"}, {"id": 14251, "dom": "G", "ent": "Hawthorn", "cls": "Polynesia"}, {"id": 14252, "dom": "G", "ent": "Hawthorn", "cls": "South America"}, {"id": 14253, "dom": "G", "ent": "Hawthorn", "cls": "South-eastern Asia"}, {"id": 14254, "dom": "G", "ent": "Hawthorn", "cls": "Southern Africa"}, {"id": 14255, "dom": "G", "ent": "Hawthorn", "cls": "Southern Asia"}, {"id": 14256, "dom": "G", "ent": "Hawthorn", "cls": "Southern Europe"}, {"id": 14257, "dom": "G", "ent": "Hawthorn", "cls": "Western Africa"}, {"id": 14258, "dom": "G", "ent": "Hawthorn", "cls": "Western Asia"}, {"id": 14259, "dom": "G", "ent": "Hawthorn", "cls": "Western Europe"}, {"id": 14260, "dom": "G", "ent": "Banja Luka", "cls": "Australia and New Zealand"}, {"id": 14261, "dom": "G", "ent": "Banja Luka", "cls": "Caribbean"}, {"id": 14262, "dom": "G", "ent": "Banja Luka", "cls": "Central America"}, {"id": 14263, "dom": "G", "ent": "Banja Luka", "cls": "Central Asia"}, {"id": 14264, "dom": "G", "ent": "Banja Luka", "cls": "Eastern Africa"}, {"id": 14265, "dom": "G", "ent": "Banja Luka", "cls": "Eastern Asia"}, {"id": 14266, "dom": "G", "ent": "Banja Luka", "cls": "Eastern Europe"}, {"id": 14267, "dom": "G", "ent": "Banja Luka", "cls": "Melanesia"}, {"id": 14268, "dom": "G", "ent": "Banja Luka", "cls": "Micronesia"}, {"id": 14269, "dom": "G", "ent": "Banja Luka", "cls": "Middle Africa"}, {"id": 14270, "dom": "G", "ent": "Banja Luka", "cls": "Northern Africa"}, {"id": 14271, "dom": "G", "ent": "Banja Luka", "cls": "Northern America"}, {"id": 14272, "dom": "G", "ent": "Banja Luka", "cls": "Northern Europe"}, {"id": 14273, "dom": "G", "ent": "Banja Luka", "cls": "Polynesia"}, {"id": 14274, "dom": "G", "ent": "Banja Luka", "cls": "South America"}, {"id": 14275, "dom": "G", "ent": "Banja Luka", "cls": "South-eastern Asia"}, {"id": 14276, "dom": "G", "ent": "Banja Luka", "cls": "Southern Africa"}, {"id": 14277, "dom": "G", "ent": "Banja Luka", "cls": "Southern Asia"}, {"id": 14278, "dom": "G", "ent": "Banja Luka", "cls": "Southern Europe"}, {"id": 14279, "dom": "G", "ent": "Banja Luka", "cls": "Western Africa"}, {"id": 14280, "dom": "G", "ent": "Banja Luka", "cls": "Western Asia"}, {"id": 14281, "dom": "G", "ent": "Banja Luka", "cls": "Western Europe"}, {"id": 14282, "dom": "G", "ent": "Bugojno", "cls": "Australia and New Zealand"}, {"id": 14283, "dom": "G", "ent": "Bugojno", "cls": "Caribbean"}, {"id": 14284, "dom": "G", "ent": "Bugojno", "cls": "Central America"}, {"id": 14285, "dom": "G", "ent": "Bugojno", "cls": "Central Asia"}, {"id": 14286, "dom": "G", "ent": "Bugojno", "cls": "Eastern Africa"}, {"id": 14287, "dom": "G", "ent": "Bugojno", "cls": "Eastern Asia"}, {"id": 14288, "dom": "G", "ent": "Bugojno", "cls": "Eastern Europe"}, {"id": 14289, "dom": "G", "ent": "Bugojno", "cls": "Melanesia"}, {"id": 14290, "dom": "G", "ent": "Bugojno", "cls": "Micronesia"}, {"id": 14291, "dom": "G", "ent": "Bugojno", "cls": "Middle Africa"}, {"id": 14292, "dom": "G", "ent": "Bugojno", "cls": "Northern Africa"}, {"id": 14293, "dom": "G", "ent": "Bugojno", "cls": "Northern America"}, {"id": 14294, "dom": "G", "ent": "Bugojno", "cls": "Northern Europe"}, {"id": 14295, "dom": "G", "ent": "Bugojno", "cls": "Polynesia"}, {"id": 14296, "dom": "G", "ent": "Bugojno", "cls": "South America"}, {"id": 14297, "dom": "G", "ent": "Bugojno", "cls": "South-eastern Asia"}, {"id": 14298, "dom": "G", "ent": "Bugojno", "cls": "Southern Africa"}, {"id": 14299, "dom": "G", "ent": "Bugojno", "cls": "Southern Asia"}, {"id": 14300, "dom": "G", "ent": "Bugojno", "cls": "Southern Europe"}, {"id": 14301, "dom": "G", "ent": "Bugojno", "cls": "Western Africa"}, {"id": 14302, "dom": "G", "ent": "Bugojno", "cls": "Western Asia"}, {"id": 14303, "dom": "G", "ent": "Bugojno", "cls": "Western Europe"}, {"id": 14304, "dom": "G", "ent": "Doboj", "cls": "Australia and New Zealand"}, {"id": 14305, "dom": "G", "ent": "Doboj", "cls": "Caribbean"}, {"id": 14306, "dom": "G", "ent": "Doboj", "cls": "Central America"}, {"id": 14307, "dom": "G", "ent": "Doboj", "cls": "Central Asia"}, {"id": 14308, "dom": "G", "ent": "Doboj", "cls": "Eastern Africa"}, {"id": 14309, "dom": "G", "ent": "Doboj", "cls": "Eastern Asia"}, {"id": 14310, "dom": "G", "ent": "Doboj", "cls": "Eastern Europe"}, {"id": 14311, "dom": "G", "ent": "Doboj", "cls": "Melanesia"}, {"id": 14312, "dom": "G", "ent": "Doboj", "cls": "Micronesia"}, {"id": 14313, "dom": "G", "ent": "Doboj", "cls": "Middle Africa"}, {"id": 14314, "dom": "G", "ent": "Doboj", "cls": "Northern Africa"}, {"id": 14315, "dom": "G", "ent": "Doboj", "cls": "Northern America"}, {"id": 14316, "dom": "G", "ent": "Doboj", "cls": "Northern Europe"}, {"id": 14317, "dom": "G", "ent": "Doboj", "cls": "Polynesia"}, {"id": 14318, "dom": "G", "ent": "Doboj", "cls": "South America"}, {"id": 14319, "dom": "G", "ent": "Doboj", "cls": "South-eastern Asia"}, {"id": 14320, "dom": "G", "ent": "Doboj", "cls": "Southern Africa"}, {"id": 14321, "dom": "G", "ent": "Doboj", "cls": "Southern Asia"}, {"id": 14322, "dom": "G", "ent": "Doboj", "cls": "Southern Europe"}, {"id": 14323, "dom": "G", "ent": "Doboj", "cls": "Western Africa"}, {"id": 14324, "dom": "G", "ent": "Doboj", "cls": "Western Asia"}, {"id": 14325, "dom": "G", "ent": "Doboj", "cls": "Western Europe"}, {"id": 14326, "dom": "G", "ent": "Cazin", "cls": "Australia and New Zealand"}, {"id": 14327, "dom": "G", "ent": "Cazin", "cls": "Caribbean"}, {"id": 14328, "dom": "G", "ent": "Cazin", "cls": "Central America"}, {"id": 14329, "dom": "G", "ent": "Cazin", "cls": "Central Asia"}, {"id": 14330, "dom": "G", "ent": "Cazin", "cls": "Eastern Africa"}, {"id": 14331, "dom": "G", "ent": "Cazin", "cls": "Eastern Asia"}, {"id": 14332, "dom": "G", "ent": "Cazin", "cls": "Eastern Europe"}, {"id": 14333, "dom": "G", "ent": "Cazin", "cls": "Melanesia"}, {"id": 14334, "dom": "G", "ent": "Cazin", "cls": "Micronesia"}, {"id": 14335, "dom": "G", "ent": "Cazin", "cls": "Middle Africa"}, {"id": 14336, "dom": "G", "ent": "Cazin", "cls": "Northern Africa"}, {"id": 14337, "dom": "G", "ent": "Cazin", "cls": "Northern America"}, {"id": 14338, "dom": "G", "ent": "Cazin", "cls": "Northern Europe"}, {"id": 14339, "dom": "G", "ent": "Cazin", "cls": "Polynesia"}, {"id": 14340, "dom": "G", "ent": "Cazin", "cls": "South America"}, {"id": 14341, "dom": "G", "ent": "Cazin", "cls": "South-eastern Asia"}, {"id": 14342, "dom": "G", "ent": "Cazin", "cls": "Southern Africa"}, {"id": 14343, "dom": "G", "ent": "Cazin", "cls": "Southern Asia"}, {"id": 14344, "dom": "G", "ent": "Cazin", "cls": "Southern Europe"}, {"id": 14345, "dom": "G", "ent": "Cazin", "cls": "Western Africa"}, {"id": 14346, "dom": "G", "ent": "Cazin", "cls": "Western Asia"}, {"id": 14347, "dom": "G", "ent": "Cazin", "cls": "Western Europe"}, {"id": 14348, "dom": "G", "ent": "Savar", "cls": "Australia and New Zealand"}, {"id": 14349, "dom": "G", "ent": "Savar", "cls": "Caribbean"}, {"id": 14350, "dom": "G", "ent": "Savar", "cls": "Central America"}, {"id": 14351, "dom": "G", "ent": "Savar", "cls": "Central Asia"}, {"id": 14352, "dom": "G", "ent": "Savar", "cls": "Eastern Africa"}, {"id": 14353, "dom": "G", "ent": "Savar", "cls": "Eastern Asia"}, {"id": 14354, "dom": "G", "ent": "Savar", "cls": "Eastern Europe"}, {"id": 14355, "dom": "G", "ent": "Savar", "cls": "Melanesia"}, {"id": 14356, "dom": "G", "ent": "Savar", "cls": "Micronesia"}, {"id": 14357, "dom": "G", "ent": "Savar", "cls": "Middle Africa"}, {"id": 14358, "dom": "G", "ent": "Savar", "cls": "Northern Africa"}, {"id": 14359, "dom": "G", "ent": "Savar", "cls": "Northern America"}, {"id": 14360, "dom": "G", "ent": "Savar", "cls": "Northern Europe"}, {"id": 14361, "dom": "G", "ent": "Savar", "cls": "Polynesia"}, {"id": 14362, "dom": "G", "ent": "Savar", "cls": "South America"}, {"id": 14363, "dom": "G", "ent": "Savar", "cls": "South-eastern Asia"}, {"id": 14364, "dom": "G", "ent": "Savar", "cls": "Southern Africa"}, {"id": 14365, "dom": "G", "ent": "Savar", "cls": "Southern Asia"}, {"id": 14366, "dom": "G", "ent": "Savar", "cls": "Southern Europe"}, {"id": 14367, "dom": "G", "ent": "Savar", "cls": "Western Africa"}, {"id": 14368, "dom": "G", "ent": "Savar", "cls": "Western Asia"}, {"id": 14369, "dom": "G", "ent": "Savar", "cls": "Western Europe"}, {"id": 14370, "dom": "G", "ent": "Barishal", "cls": "Australia and New Zealand"}, {"id": 14371, "dom": "G", "ent": "Barishal", "cls": "Caribbean"}, {"id": 14372, "dom": "G", "ent": "Barishal", "cls": "Central America"}, {"id": 14373, "dom": "G", "ent": "Barishal", "cls": "Central Asia"}, {"id": 14374, "dom": "G", "ent": "Barishal", "cls": "Eastern Africa"}, {"id": 14375, "dom": "G", "ent": "Barishal", "cls": "Eastern Asia"}, {"id": 14376, "dom": "G", "ent": "Barishal", "cls": "Eastern Europe"}, {"id": 14377, "dom": "G", "ent": "Barishal", "cls": "Melanesia"}, {"id": 14378, "dom": "G", "ent": "Barishal", "cls": "Micronesia"}, {"id": 14379, "dom": "G", "ent": "Barishal", "cls": "Middle Africa"}, {"id": 14380, "dom": "G", "ent": "Barishal", "cls": "Northern Africa"}, {"id": 14381, "dom": "G", "ent": "Barishal", "cls": "Northern America"}, {"id": 14382, "dom": "G", "ent": "Barishal", "cls": "Northern Europe"}, {"id": 14383, "dom": "G", "ent": "Barishal", "cls": "Polynesia"}, {"id": 14384, "dom": "G", "ent": "Barishal", "cls": "South America"}, {"id": 14385, "dom": "G", "ent": "Barishal", "cls": "South-eastern Asia"}, {"id": 14386, "dom": "G", "ent": "Barishal", "cls": "Southern Africa"}, {"id": 14387, "dom": "G", "ent": "Barishal", "cls": "Southern Asia"}, {"id": 14388, "dom": "G", "ent": "Barishal", "cls": "Southern Europe"}, {"id": 14389, "dom": "G", "ent": "Barishal", "cls": "Western Africa"}, {"id": 14390, "dom": "G", "ent": "Barishal", "cls": "Western Asia"}, {"id": 14391, "dom": "G", "ent": "Barishal", "cls": "Western Europe"}, {"id": 14392, "dom": "G", "ent": "Uttar Char Fasson", "cls": "Australia and New Zealand"}, {"id": 14393, "dom": "G", "ent": "Uttar Char Fasson", "cls": "Caribbean"}, {"id": 14394, "dom": "G", "ent": "Uttar Char Fasson", "cls": "Central America"}, {"id": 14395, "dom": "G", "ent": "Uttar Char Fasson", "cls": "Central Asia"}, {"id": 14396, "dom": "G", "ent": "Uttar Char Fasson", "cls": "Eastern Africa"}, {"id": 14397, "dom": "G", "ent": "Uttar Char Fasson", "cls": "Eastern Asia"}, {"id": 14398, "dom": "G", "ent": "Uttar Char Fasson", "cls": "Eastern Europe"}, {"id": 14399, "dom": "G", "ent": "Uttar Char Fasson", "cls": "Melanesia"}, {"id": 14400, "dom": "G", "ent": "Uttar Char Fasson", "cls": "Micronesia"}, {"id": 14401, "dom": "G", "ent": "Uttar Char Fasson", "cls": "Middle Africa"}, {"id": 14402, "dom": "G", "ent": "Uttar Char Fasson", "cls": "Northern Africa"}, {"id": 14403, "dom": "G", "ent": "Uttar Char Fasson", "cls": "Northern America"}, {"id": 14404, "dom": "G", "ent": "Uttar Char Fasson", "cls": "Northern Europe"}, {"id": 14405, "dom": "G", "ent": "Uttar Char Fasson", "cls": "Polynesia"}, {"id": 14406, "dom": "G", "ent": "Uttar Char Fasson", "cls": "South America"}, {"id": 14407, "dom": "G", "ent": "Uttar Char Fasson", "cls": "South-eastern Asia"}, {"id": 14408, "dom": "G", "ent": "Uttar Char Fasson", "cls": "Southern Africa"}, {"id": 14409, "dom": "G", "ent": "Uttar Char Fasson", "cls": "Southern Asia"}, {"id": 14410, "dom": "G", "ent": "Uttar Char Fasson", "cls": "Southern Europe"}, {"id": 14411, "dom": "G", "ent": "Uttar Char Fasson", "cls": "Western Africa"}, {"id": 14412, "dom": "G", "ent": "Uttar Char Fasson", "cls": "Western Asia"}, {"id": 14413, "dom": "G", "ent": "Uttar Char Fasson", "cls": "Western Europe"}, {"id": 14414, "dom": "G", "ent": "Srimangal", "cls": "Australia and New Zealand"}, {"id": 14415, "dom": "G", "ent": "Srimangal", "cls": "Caribbean"}, {"id": 14416, "dom": "G", "ent": "Srimangal", "cls": "Central America"}, {"id": 14417, "dom": "G", "ent": "Srimangal", "cls": "Central Asia"}, {"id": 14418, "dom": "G", "ent": "Srimangal", "cls": "Eastern Africa"}, {"id": 14419, "dom": "G", "ent": "Srimangal", "cls": "Eastern Asia"}, {"id": 14420, "dom": "G", "ent": "Srimangal", "cls": "Eastern Europe"}, {"id": 14421, "dom": "G", "ent": "Srimangal", "cls": "Melanesia"}, {"id": 14422, "dom": "G", "ent": "Srimangal", "cls": "Micronesia"}, {"id": 14423, "dom": "G", "ent": "Srimangal", "cls": "Middle Africa"}, {"id": 14424, "dom": "G", "ent": "Srimangal", "cls": "Northern Africa"}, {"id": 14425, "dom": "G", "ent": "Srimangal", "cls": "Northern America"}, {"id": 14426, "dom": "G", "ent": "Srimangal", "cls": "Northern Europe"}, {"id": 14427, "dom": "G", "ent": "Srimangal", "cls": "Polynesia"}, {"id": 14428, "dom": "G", "ent": "Srimangal", "cls": "South America"}, {"id": 14429, "dom": "G", "ent": "Srimangal", "cls": "South-eastern Asia"}, {"id": 14430, "dom": "G", "ent": "Srimangal", "cls": "Southern Africa"}, {"id": 14431, "dom": "G", "ent": "Srimangal", "cls": "Southern Asia"}, {"id": 14432, "dom": "G", "ent": "Srimangal", "cls": "Southern Europe"}, {"id": 14433, "dom": "G", "ent": "Srimangal", "cls": "Western Africa"}, {"id": 14434, "dom": "G", "ent": "Srimangal", "cls": "Western Asia"}, {"id": 14435, "dom": "G", "ent": "Srimangal", "cls": "Western Europe"}, {"id": 14436, "dom": "G", "ent": "Forest", "cls": "Australia and New Zealand"}, {"id": 14437, "dom": "G", "ent": "Forest", "cls": "Caribbean"}, {"id": 14438, "dom": "G", "ent": "Forest", "cls": "Central America"}, {"id": 14439, "dom": "G", "ent": "Forest", "cls": "Central Asia"}, {"id": 14440, "dom": "G", "ent": "Forest", "cls": "Eastern Africa"}, {"id": 14441, "dom": "G", "ent": "Forest", "cls": "Eastern Asia"}, {"id": 14442, "dom": "G", "ent": "Forest", "cls": "Eastern Europe"}, {"id": 14443, "dom": "G", "ent": "Forest", "cls": "Melanesia"}, {"id": 14444, "dom": "G", "ent": "Forest", "cls": "Micronesia"}, {"id": 14445, "dom": "G", "ent": "Forest", "cls": "Middle Africa"}, {"id": 14446, "dom": "G", "ent": "Forest", "cls": "Northern Africa"}, {"id": 14447, "dom": "G", "ent": "Forest", "cls": "Northern America"}, {"id": 14448, "dom": "G", "ent": "Forest", "cls": "Northern Europe"}, {"id": 14449, "dom": "G", "ent": "Forest", "cls": "Polynesia"}, {"id": 14450, "dom": "G", "ent": "Forest", "cls": "South America"}, {"id": 14451, "dom": "G", "ent": "Forest", "cls": "South-eastern Asia"}, {"id": 14452, "dom": "G", "ent": "Forest", "cls": "Southern Africa"}, {"id": 14453, "dom": "G", "ent": "Forest", "cls": "Southern Asia"}, {"id": 14454, "dom": "G", "ent": "Forest", "cls": "Southern Europe"}, {"id": 14455, "dom": "G", "ent": "Forest", "cls": "Western Africa"}, {"id": 14456, "dom": "G", "ent": "Forest", "cls": "Western Asia"}, {"id": 14457, "dom": "G", "ent": "Forest", "cls": "Western Europe"}, {"id": 14458, "dom": "G", "ent": "Izegem", "cls": "Australia and New Zealand"}, {"id": 14459, "dom": "G", "ent": "Izegem", "cls": "Caribbean"}, {"id": 14460, "dom": "G", "ent": "Izegem", "cls": "Central America"}, {"id": 14461, "dom": "G", "ent": "Izegem", "cls": "Central Asia"}, {"id": 14462, "dom": "G", "ent": "Izegem", "cls": "Eastern Africa"}, {"id": 14463, "dom": "G", "ent": "Izegem", "cls": "Eastern Asia"}, {"id": 14464, "dom": "G", "ent": "Izegem", "cls": "Eastern Europe"}, {"id": 14465, "dom": "G", "ent": "Izegem", "cls": "Melanesia"}, {"id": 14466, "dom": "G", "ent": "Izegem", "cls": "Micronesia"}, {"id": 14467, "dom": "G", "ent": "Izegem", "cls": "Middle Africa"}, {"id": 14468, "dom": "G", "ent": "Izegem", "cls": "Northern Africa"}, {"id": 14469, "dom": "G", "ent": "Izegem", "cls": "Northern America"}, {"id": 14470, "dom": "G", "ent": "Izegem", "cls": "Northern Europe"}, {"id": 14471, "dom": "G", "ent": "Izegem", "cls": "Polynesia"}, {"id": 14472, "dom": "G", "ent": "Izegem", "cls": "South America"}, {"id": 14473, "dom": "G", "ent": "Izegem", "cls": "South-eastern Asia"}, {"id": 14474, "dom": "G", "ent": "Izegem", "cls": "Southern Africa"}, {"id": 14475, "dom": "G", "ent": "Izegem", "cls": "Southern Asia"}, {"id": 14476, "dom": "G", "ent": "Izegem", "cls": "Southern Europe"}, {"id": 14477, "dom": "G", "ent": "Izegem", "cls": "Western Africa"}, {"id": 14478, "dom": "G", "ent": "Izegem", "cls": "Western Asia"}, {"id": 14479, "dom": "G", "ent": "Izegem", "cls": "Western Europe"}, {"id": 14480, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "Australia and New Zealand"}, {"id": 14481, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "Caribbean"}, {"id": 14482, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "Central America"}, {"id": 14483, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "Central Asia"}, {"id": 14484, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "Eastern Africa"}, {"id": 14485, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "Eastern Asia"}, {"id": 14486, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "Eastern Europe"}, {"id": 14487, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "Melanesia"}, {"id": 14488, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "Micronesia"}, {"id": 14489, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "Middle Africa"}, {"id": 14490, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "Northern Africa"}, {"id": 14491, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "Northern America"}, {"id": 14492, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "Northern Europe"}, {"id": 14493, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "Polynesia"}, {"id": 14494, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "South America"}, {"id": 14495, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "South-eastern Asia"}, {"id": 14496, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "Southern Africa"}, {"id": 14497, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "Southern Asia"}, {"id": 14498, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "Southern Europe"}, {"id": 14499, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "Western Africa"}, {"id": 14500, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "Western Asia"}, {"id": 14501, "dom": "G", "ent": "Watermael-Boitsfort", "cls": "Western Europe"}, {"id": 14502, "dom": "G", "ent": "Oostkamp", "cls": "Australia and New Zealand"}, {"id": 14503, "dom": "G", "ent": "Oostkamp", "cls": "Caribbean"}, {"id": 14504, "dom": "G", "ent": "Oostkamp", "cls": "Central America"}, {"id": 14505, "dom": "G", "ent": "Oostkamp", "cls": "Central Asia"}, {"id": 14506, "dom": "G", "ent": "Oostkamp", "cls": "Eastern Africa"}, {"id": 14507, "dom": "G", "ent": "Oostkamp", "cls": "Eastern Asia"}, {"id": 14508, "dom": "G", "ent": "Oostkamp", "cls": "Eastern Europe"}, {"id": 14509, "dom": "G", "ent": "Oostkamp", "cls": "Melanesia"}, {"id": 14510, "dom": "G", "ent": "Oostkamp", "cls": "Micronesia"}, {"id": 14511, "dom": "G", "ent": "Oostkamp", "cls": "Middle Africa"}, {"id": 14512, "dom": "G", "ent": "Oostkamp", "cls": "Northern Africa"}, {"id": 14513, "dom": "G", "ent": "Oostkamp", "cls": "Northern America"}, {"id": 14514, "dom": "G", "ent": "Oostkamp", "cls": "Northern Europe"}, {"id": 14515, "dom": "G", "ent": "Oostkamp", "cls": "Polynesia"}, {"id": 14516, "dom": "G", "ent": "Oostkamp", "cls": "South America"}, {"id": 14517, "dom": "G", "ent": "Oostkamp", "cls": "South-eastern Asia"}, {"id": 14518, "dom": "G", "ent": "Oostkamp", "cls": "Southern Africa"}, {"id": 14519, "dom": "G", "ent": "Oostkamp", "cls": "Southern Asia"}, {"id": 14520, "dom": "G", "ent": "Oostkamp", "cls": "Southern Europe"}, {"id": 14521, "dom": "G", "ent": "Oostkamp", "cls": "Western Africa"}, {"id": 14522, "dom": "G", "ent": "Oostkamp", "cls": "Western Asia"}, {"id": 14523, "dom": "G", "ent": "Oostkamp", "cls": "Western Europe"}, {"id": 14524, "dom": "G", "ent": "Koudougou", "cls": "Australia and New Zealand"}, {"id": 14525, "dom": "G", "ent": "Koudougou", "cls": "Caribbean"}, {"id": 14526, "dom": "G", "ent": "Koudougou", "cls": "Central America"}, {"id": 14527, "dom": "G", "ent": "Koudougou", "cls": "Central Asia"}, {"id": 14528, "dom": "G", "ent": "Koudougou", "cls": "Eastern Africa"}, {"id": 14529, "dom": "G", "ent": "Koudougou", "cls": "Eastern Asia"}, {"id": 14530, "dom": "G", "ent": "Koudougou", "cls": "Eastern Europe"}, {"id": 14531, "dom": "G", "ent": "Koudougou", "cls": "Melanesia"}, {"id": 14532, "dom": "G", "ent": "Koudougou", "cls": "Micronesia"}, {"id": 14533, "dom": "G", "ent": "Koudougou", "cls": "Middle Africa"}, {"id": 14534, "dom": "G", "ent": "Koudougou", "cls": "Northern Africa"}, {"id": 14535, "dom": "G", "ent": "Koudougou", "cls": "Northern America"}, {"id": 14536, "dom": "G", "ent": "Koudougou", "cls": "Northern Europe"}, {"id": 14537, "dom": "G", "ent": "Koudougou", "cls": "Polynesia"}, {"id": 14538, "dom": "G", "ent": "Koudougou", "cls": "South America"}, {"id": 14539, "dom": "G", "ent": "Koudougou", "cls": "South-eastern Asia"}, {"id": 14540, "dom": "G", "ent": "Koudougou", "cls": "Southern Africa"}, {"id": 14541, "dom": "G", "ent": "Koudougou", "cls": "Southern Asia"}, {"id": 14542, "dom": "G", "ent": "Koudougou", "cls": "Southern Europe"}, {"id": 14543, "dom": "G", "ent": "Koudougou", "cls": "Western Africa"}, {"id": 14544, "dom": "G", "ent": "Koudougou", "cls": "Western Asia"}, {"id": 14545, "dom": "G", "ent": "Koudougou", "cls": "Western Europe"}, {"id": 14546, "dom": "G", "ent": "Orodara", "cls": "Australia and New Zealand"}, {"id": 14547, "dom": "G", "ent": "Orodara", "cls": "Caribbean"}, {"id": 14548, "dom": "G", "ent": "Orodara", "cls": "Central America"}, {"id": 14549, "dom": "G", "ent": "Orodara", "cls": "Central Asia"}, {"id": 14550, "dom": "G", "ent": "Orodara", "cls": "Eastern Africa"}, {"id": 14551, "dom": "G", "ent": "Orodara", "cls": "Eastern Asia"}, {"id": 14552, "dom": "G", "ent": "Orodara", "cls": "Eastern Europe"}, {"id": 14553, "dom": "G", "ent": "Orodara", "cls": "Melanesia"}, {"id": 14554, "dom": "G", "ent": "Orodara", "cls": "Micronesia"}, {"id": 14555, "dom": "G", "ent": "Orodara", "cls": "Middle Africa"}, {"id": 14556, "dom": "G", "ent": "Orodara", "cls": "Northern Africa"}, {"id": 14557, "dom": "G", "ent": "Orodara", "cls": "Northern America"}, {"id": 14558, "dom": "G", "ent": "Orodara", "cls": "Northern Europe"}, {"id": 14559, "dom": "G", "ent": "Orodara", "cls": "Polynesia"}, {"id": 14560, "dom": "G", "ent": "Orodara", "cls": "South America"}, {"id": 14561, "dom": "G", "ent": "Orodara", "cls": "South-eastern Asia"}, {"id": 14562, "dom": "G", "ent": "Orodara", "cls": "Southern Africa"}, {"id": 14563, "dom": "G", "ent": "Orodara", "cls": "Southern Asia"}, {"id": 14564, "dom": "G", "ent": "Orodara", "cls": "Southern Europe"}, {"id": 14565, "dom": "G", "ent": "Orodara", "cls": "Western Africa"}, {"id": 14566, "dom": "G", "ent": "Orodara", "cls": "Western Asia"}, {"id": 14567, "dom": "G", "ent": "Orodara", "cls": "Western Europe"}, {"id": 14568, "dom": "G", "ent": "Korsimoro", "cls": "Australia and New Zealand"}, {"id": 14569, "dom": "G", "ent": "Korsimoro", "cls": "Caribbean"}, {"id": 14570, "dom": "G", "ent": "Korsimoro", "cls": "Central America"}, {"id": 14571, "dom": "G", "ent": "Korsimoro", "cls": "Central Asia"}, {"id": 14572, "dom": "G", "ent": "Korsimoro", "cls": "Eastern Africa"}, {"id": 14573, "dom": "G", "ent": "Korsimoro", "cls": "Eastern Asia"}, {"id": 14574, "dom": "G", "ent": "Korsimoro", "cls": "Eastern Europe"}, {"id": 14575, "dom": "G", "ent": "Korsimoro", "cls": "Melanesia"}, {"id": 14576, "dom": "G", "ent": "Korsimoro", "cls": "Micronesia"}, {"id": 14577, "dom": "G", "ent": "Korsimoro", "cls": "Middle Africa"}, {"id": 14578, "dom": "G", "ent": "Korsimoro", "cls": "Northern Africa"}, {"id": 14579, "dom": "G", "ent": "Korsimoro", "cls": "Northern America"}, {"id": 14580, "dom": "G", "ent": "Korsimoro", "cls": "Northern Europe"}, {"id": 14581, "dom": "G", "ent": "Korsimoro", "cls": "Polynesia"}, {"id": 14582, "dom": "G", "ent": "Korsimoro", "cls": "South America"}, {"id": 14583, "dom": "G", "ent": "Korsimoro", "cls": "South-eastern Asia"}, {"id": 14584, "dom": "G", "ent": "Korsimoro", "cls": "Southern Africa"}, {"id": 14585, "dom": "G", "ent": "Korsimoro", "cls": "Southern Asia"}, {"id": 14586, "dom": "G", "ent": "Korsimoro", "cls": "Southern Europe"}, {"id": 14587, "dom": "G", "ent": "Korsimoro", "cls": "Western Africa"}, {"id": 14588, "dom": "G", "ent": "Korsimoro", "cls": "Western Asia"}, {"id": 14589, "dom": "G", "ent": "Korsimoro", "cls": "Western Europe"}, {"id": 14590, "dom": "G", "ent": "Karankasso", "cls": "Australia and New Zealand"}, {"id": 14591, "dom": "G", "ent": "Karankasso", "cls": "Caribbean"}, {"id": 14592, "dom": "G", "ent": "Karankasso", "cls": "Central America"}, {"id": 14593, "dom": "G", "ent": "Karankasso", "cls": "Central Asia"}, {"id": 14594, "dom": "G", "ent": "Karankasso", "cls": "Eastern Africa"}, {"id": 14595, "dom": "G", "ent": "Karankasso", "cls": "Eastern Asia"}, {"id": 14596, "dom": "G", "ent": "Karankasso", "cls": "Eastern Europe"}, {"id": 14597, "dom": "G", "ent": "Karankasso", "cls": "Melanesia"}, {"id": 14598, "dom": "G", "ent": "Karankasso", "cls": "Micronesia"}, {"id": 14599, "dom": "G", "ent": "Karankasso", "cls": "Middle Africa"}, {"id": 14600, "dom": "G", "ent": "Karankasso", "cls": "Northern Africa"}, {"id": 14601, "dom": "G", "ent": "Karankasso", "cls": "Northern America"}, {"id": 14602, "dom": "G", "ent": "Karankasso", "cls": "Northern Europe"}, {"id": 14603, "dom": "G", "ent": "Karankasso", "cls": "Polynesia"}, {"id": 14604, "dom": "G", "ent": "Karankasso", "cls": "South America"}, {"id": 14605, "dom": "G", "ent": "Karankasso", "cls": "South-eastern Asia"}, {"id": 14606, "dom": "G", "ent": "Karankasso", "cls": "Southern Africa"}, {"id": 14607, "dom": "G", "ent": "Karankasso", "cls": "Southern Asia"}, {"id": 14608, "dom": "G", "ent": "Karankasso", "cls": "Southern Europe"}, {"id": 14609, "dom": "G", "ent": "Karankasso", "cls": "Western Africa"}, {"id": 14610, "dom": "G", "ent": "Karankasso", "cls": "Western Asia"}, {"id": 14611, "dom": "G", "ent": "Karankasso", "cls": "Western Europe"}, {"id": 14612, "dom": "G", "ent": "Stara Zagora", "cls": "Australia and New Zealand"}, {"id": 14613, "dom": "G", "ent": "Stara Zagora", "cls": "Caribbean"}, {"id": 14614, "dom": "G", "ent": "Stara Zagora", "cls": "Central America"}, {"id": 14615, "dom": "G", "ent": "Stara Zagora", "cls": "Central Asia"}, {"id": 14616, "dom": "G", "ent": "Stara Zagora", "cls": "Eastern Africa"}, {"id": 14617, "dom": "G", "ent": "Stara Zagora", "cls": "Eastern Asia"}, {"id": 14618, "dom": "G", "ent": "Stara Zagora", "cls": "Eastern Europe"}, {"id": 14619, "dom": "G", "ent": "Stara Zagora", "cls": "Melanesia"}, {"id": 14620, "dom": "G", "ent": "Stara Zagora", "cls": "Micronesia"}, {"id": 14621, "dom": "G", "ent": "Stara Zagora", "cls": "Middle Africa"}, {"id": 14622, "dom": "G", "ent": "Stara Zagora", "cls": "Northern Africa"}, {"id": 14623, "dom": "G", "ent": "Stara Zagora", "cls": "Northern America"}, {"id": 14624, "dom": "G", "ent": "Stara Zagora", "cls": "Northern Europe"}, {"id": 14625, "dom": "G", "ent": "Stara Zagora", "cls": "Polynesia"}, {"id": 14626, "dom": "G", "ent": "Stara Zagora", "cls": "South America"}, {"id": 14627, "dom": "G", "ent": "Stara Zagora", "cls": "South-eastern Asia"}, {"id": 14628, "dom": "G", "ent": "Stara Zagora", "cls": "Southern Africa"}, {"id": 14629, "dom": "G", "ent": "Stara Zagora", "cls": "Southern Asia"}, {"id": 14630, "dom": "G", "ent": "Stara Zagora", "cls": "Southern Europe"}, {"id": 14631, "dom": "G", "ent": "Stara Zagora", "cls": "Western Africa"}, {"id": 14632, "dom": "G", "ent": "Stara Zagora", "cls": "Western Asia"}, {"id": 14633, "dom": "G", "ent": "Stara Zagora", "cls": "Western Europe"}, {"id": 14634, "dom": "G", "ent": "Kazanlak", "cls": "Australia and New Zealand"}, {"id": 14635, "dom": "G", "ent": "Kazanlak", "cls": "Caribbean"}, {"id": 14636, "dom": "G", "ent": "Kazanlak", "cls": "Central America"}, {"id": 14637, "dom": "G", "ent": "Kazanlak", "cls": "Central Asia"}, {"id": 14638, "dom": "G", "ent": "Kazanlak", "cls": "Eastern Africa"}, {"id": 14639, "dom": "G", "ent": "Kazanlak", "cls": "Eastern Asia"}, {"id": 14640, "dom": "G", "ent": "Kazanlak", "cls": "Eastern Europe"}, {"id": 14641, "dom": "G", "ent": "Kazanlak", "cls": "Melanesia"}, {"id": 14642, "dom": "G", "ent": "Kazanlak", "cls": "Micronesia"}, {"id": 14643, "dom": "G", "ent": "Kazanlak", "cls": "Middle Africa"}, {"id": 14644, "dom": "G", "ent": "Kazanlak", "cls": "Northern Africa"}, {"id": 14645, "dom": "G", "ent": "Kazanlak", "cls": "Northern America"}, {"id": 14646, "dom": "G", "ent": "Kazanlak", "cls": "Northern Europe"}, {"id": 14647, "dom": "G", "ent": "Kazanlak", "cls": "Polynesia"}, {"id": 14648, "dom": "G", "ent": "Kazanlak", "cls": "South America"}, {"id": 14649, "dom": "G", "ent": "Kazanlak", "cls": "South-eastern Asia"}, {"id": 14650, "dom": "G", "ent": "Kazanlak", "cls": "Southern Africa"}, {"id": 14651, "dom": "G", "ent": "Kazanlak", "cls": "Southern Asia"}, {"id": 14652, "dom": "G", "ent": "Kazanlak", "cls": "Southern Europe"}, {"id": 14653, "dom": "G", "ent": "Kazanlak", "cls": "Western Africa"}, {"id": 14654, "dom": "G", "ent": "Kazanlak", "cls": "Western Asia"}, {"id": 14655, "dom": "G", "ent": "Kazanlak", "cls": "Western Europe"}, {"id": 14656, "dom": "G", "ent": "Targovishte", "cls": "Australia and New Zealand"}, {"id": 14657, "dom": "G", "ent": "Targovishte", "cls": "Caribbean"}, {"id": 14658, "dom": "G", "ent": "Targovishte", "cls": "Central America"}, {"id": 14659, "dom": "G", "ent": "Targovishte", "cls": "Central Asia"}, {"id": 14660, "dom": "G", "ent": "Targovishte", "cls": "Eastern Africa"}, {"id": 14661, "dom": "G", "ent": "Targovishte", "cls": "Eastern Asia"}, {"id": 14662, "dom": "G", "ent": "Targovishte", "cls": "Eastern Europe"}, {"id": 14663, "dom": "G", "ent": "Targovishte", "cls": "Melanesia"}, {"id": 14664, "dom": "G", "ent": "Targovishte", "cls": "Micronesia"}, {"id": 14665, "dom": "G", "ent": "Targovishte", "cls": "Middle Africa"}, {"id": 14666, "dom": "G", "ent": "Targovishte", "cls": "Northern Africa"}, {"id": 14667, "dom": "G", "ent": "Targovishte", "cls": "Northern America"}, {"id": 14668, "dom": "G", "ent": "Targovishte", "cls": "Northern Europe"}, {"id": 14669, "dom": "G", "ent": "Targovishte", "cls": "Polynesia"}, {"id": 14670, "dom": "G", "ent": "Targovishte", "cls": "South America"}, {"id": 14671, "dom": "G", "ent": "Targovishte", "cls": "South-eastern Asia"}, {"id": 14672, "dom": "G", "ent": "Targovishte", "cls": "Southern Africa"}, {"id": 14673, "dom": "G", "ent": "Targovishte", "cls": "Southern Asia"}, {"id": 14674, "dom": "G", "ent": "Targovishte", "cls": "Southern Europe"}, {"id": 14675, "dom": "G", "ent": "Targovishte", "cls": "Western Africa"}, {"id": 14676, "dom": "G", "ent": "Targovishte", "cls": "Western Asia"}, {"id": 14677, "dom": "G", "ent": "Targovishte", "cls": "Western Europe"}, {"id": 14678, "dom": "G", "ent": "Dupnitsa", "cls": "Australia and New Zealand"}, {"id": 14679, "dom": "G", "ent": "Dupnitsa", "cls": "Caribbean"}, {"id": 14680, "dom": "G", "ent": "Dupnitsa", "cls": "Central America"}, {"id": 14681, "dom": "G", "ent": "Dupnitsa", "cls": "Central Asia"}, {"id": 14682, "dom": "G", "ent": "Dupnitsa", "cls": "Eastern Africa"}, {"id": 14683, "dom": "G", "ent": "Dupnitsa", "cls": "Eastern Asia"}, {"id": 14684, "dom": "G", "ent": "Dupnitsa", "cls": "Eastern Europe"}, {"id": 14685, "dom": "G", "ent": "Dupnitsa", "cls": "Melanesia"}, {"id": 14686, "dom": "G", "ent": "Dupnitsa", "cls": "Micronesia"}, {"id": 14687, "dom": "G", "ent": "Dupnitsa", "cls": "Middle Africa"}, {"id": 14688, "dom": "G", "ent": "Dupnitsa", "cls": "Northern Africa"}, {"id": 14689, "dom": "G", "ent": "Dupnitsa", "cls": "Northern America"}, {"id": 14690, "dom": "G", "ent": "Dupnitsa", "cls": "Northern Europe"}, {"id": 14691, "dom": "G", "ent": "Dupnitsa", "cls": "Polynesia"}, {"id": 14692, "dom": "G", "ent": "Dupnitsa", "cls": "South America"}, {"id": 14693, "dom": "G", "ent": "Dupnitsa", "cls": "South-eastern Asia"}, {"id": 14694, "dom": "G", "ent": "Dupnitsa", "cls": "Southern Africa"}, {"id": 14695, "dom": "G", "ent": "Dupnitsa", "cls": "Southern Asia"}, {"id": 14696, "dom": "G", "ent": "Dupnitsa", "cls": "Southern Europe"}, {"id": 14697, "dom": "G", "ent": "Dupnitsa", "cls": "Western Africa"}, {"id": 14698, "dom": "G", "ent": "Dupnitsa", "cls": "Western Asia"}, {"id": 14699, "dom": "G", "ent": "Dupnitsa", "cls": "Western Europe"}, {"id": 14700, "dom": "G", "ent": "Ngozi", "cls": "Australia and New Zealand"}, {"id": 14701, "dom": "G", "ent": "Ngozi", "cls": "Caribbean"}, {"id": 14702, "dom": "G", "ent": "Ngozi", "cls": "Central America"}, {"id": 14703, "dom": "G", "ent": "Ngozi", "cls": "Central Asia"}, {"id": 14704, "dom": "G", "ent": "Ngozi", "cls": "Eastern Africa"}, {"id": 14705, "dom": "G", "ent": "Ngozi", "cls": "Eastern Asia"}, {"id": 14706, "dom": "G", "ent": "Ngozi", "cls": "Eastern Europe"}, {"id": 14707, "dom": "G", "ent": "Ngozi", "cls": "Melanesia"}, {"id": 14708, "dom": "G", "ent": "Ngozi", "cls": "Micronesia"}, {"id": 14709, "dom": "G", "ent": "Ngozi", "cls": "Middle Africa"}, {"id": 14710, "dom": "G", "ent": "Ngozi", "cls": "Northern Africa"}, {"id": 14711, "dom": "G", "ent": "Ngozi", "cls": "Northern America"}, {"id": 14712, "dom": "G", "ent": "Ngozi", "cls": "Northern Europe"}, {"id": 14713, "dom": "G", "ent": "Ngozi", "cls": "Polynesia"}, {"id": 14714, "dom": "G", "ent": "Ngozi", "cls": "South America"}, {"id": 14715, "dom": "G", "ent": "Ngozi", "cls": "South-eastern Asia"}, {"id": 14716, "dom": "G", "ent": "Ngozi", "cls": "Southern Africa"}, {"id": 14717, "dom": "G", "ent": "Ngozi", "cls": "Southern Asia"}, {"id": 14718, "dom": "G", "ent": "Ngozi", "cls": "Southern Europe"}, {"id": 14719, "dom": "G", "ent": "Ngozi", "cls": "Western Africa"}, {"id": 14720, "dom": "G", "ent": "Ngozi", "cls": "Western Asia"}, {"id": 14721, "dom": "G", "ent": "Ngozi", "cls": "Western Europe"}, {"id": 14722, "dom": "G", "ent": "Rumonge", "cls": "Australia and New Zealand"}, {"id": 14723, "dom": "G", "ent": "Rumonge", "cls": "Caribbean"}, {"id": 14724, "dom": "G", "ent": "Rumonge", "cls": "Central America"}, {"id": 14725, "dom": "G", "ent": "Rumonge", "cls": "Central Asia"}, {"id": 14726, "dom": "G", "ent": "Rumonge", "cls": "Eastern Africa"}, {"id": 14727, "dom": "G", "ent": "Rumonge", "cls": "Eastern Asia"}, {"id": 14728, "dom": "G", "ent": "Rumonge", "cls": "Eastern Europe"}, {"id": 14729, "dom": "G", "ent": "Rumonge", "cls": "Melanesia"}, {"id": 14730, "dom": "G", "ent": "Rumonge", "cls": "Micronesia"}, {"id": 14731, "dom": "G", "ent": "Rumonge", "cls": "Middle Africa"}, {"id": 14732, "dom": "G", "ent": "Rumonge", "cls": "Northern Africa"}, {"id": 14733, "dom": "G", "ent": "Rumonge", "cls": "Northern America"}, {"id": 14734, "dom": "G", "ent": "Rumonge", "cls": "Northern Europe"}, {"id": 14735, "dom": "G", "ent": "Rumonge", "cls": "Polynesia"}, {"id": 14736, "dom": "G", "ent": "Rumonge", "cls": "South America"}, {"id": 14737, "dom": "G", "ent": "Rumonge", "cls": "South-eastern Asia"}, {"id": 14738, "dom": "G", "ent": "Rumonge", "cls": "Southern Africa"}, {"id": 14739, "dom": "G", "ent": "Rumonge", "cls": "Southern Asia"}, {"id": 14740, "dom": "G", "ent": "Rumonge", "cls": "Southern Europe"}, {"id": 14741, "dom": "G", "ent": "Rumonge", "cls": "Western Africa"}, {"id": 14742, "dom": "G", "ent": "Rumonge", "cls": "Western Asia"}, {"id": 14743, "dom": "G", "ent": "Rumonge", "cls": "Western Europe"}, {"id": 14744, "dom": "G", "ent": "Kayanza", "cls": "Australia and New Zealand"}, {"id": 14745, "dom": "G", "ent": "Kayanza", "cls": "Caribbean"}, {"id": 14746, "dom": "G", "ent": "Kayanza", "cls": "Central America"}, {"id": 14747, "dom": "G", "ent": "Kayanza", "cls": "Central Asia"}, {"id": 14748, "dom": "G", "ent": "Kayanza", "cls": "Eastern Africa"}, {"id": 14749, "dom": "G", "ent": "Kayanza", "cls": "Eastern Asia"}, {"id": 14750, "dom": "G", "ent": "Kayanza", "cls": "Eastern Europe"}, {"id": 14751, "dom": "G", "ent": "Kayanza", "cls": "Melanesia"}, {"id": 14752, "dom": "G", "ent": "Kayanza", "cls": "Micronesia"}, {"id": 14753, "dom": "G", "ent": "Kayanza", "cls": "Middle Africa"}, {"id": 14754, "dom": "G", "ent": "Kayanza", "cls": "Northern Africa"}, {"id": 14755, "dom": "G", "ent": "Kayanza", "cls": "Northern America"}, {"id": 14756, "dom": "G", "ent": "Kayanza", "cls": "Northern Europe"}, {"id": 14757, "dom": "G", "ent": "Kayanza", "cls": "Polynesia"}, {"id": 14758, "dom": "G", "ent": "Kayanza", "cls": "South America"}, {"id": 14759, "dom": "G", "ent": "Kayanza", "cls": "South-eastern Asia"}, {"id": 14760, "dom": "G", "ent": "Kayanza", "cls": "Southern Africa"}, {"id": 14761, "dom": "G", "ent": "Kayanza", "cls": "Southern Asia"}, {"id": 14762, "dom": "G", "ent": "Kayanza", "cls": "Southern Europe"}, {"id": 14763, "dom": "G", "ent": "Kayanza", "cls": "Western Africa"}, {"id": 14764, "dom": "G", "ent": "Kayanza", "cls": "Western Asia"}, {"id": 14765, "dom": "G", "ent": "Kayanza", "cls": "Western Europe"}, {"id": 14766, "dom": "G", "ent": "Bubanza", "cls": "Australia and New Zealand"}, {"id": 14767, "dom": "G", "ent": "Bubanza", "cls": "Caribbean"}, {"id": 14768, "dom": "G", "ent": "Bubanza", "cls": "Central America"}, {"id": 14769, "dom": "G", "ent": "Bubanza", "cls": "Central Asia"}, {"id": 14770, "dom": "G", "ent": "Bubanza", "cls": "Eastern Africa"}, {"id": 14771, "dom": "G", "ent": "Bubanza", "cls": "Eastern Asia"}, {"id": 14772, "dom": "G", "ent": "Bubanza", "cls": "Eastern Europe"}, {"id": 14773, "dom": "G", "ent": "Bubanza", "cls": "Melanesia"}, {"id": 14774, "dom": "G", "ent": "Bubanza", "cls": "Micronesia"}, {"id": 14775, "dom": "G", "ent": "Bubanza", "cls": "Middle Africa"}, {"id": 14776, "dom": "G", "ent": "Bubanza", "cls": "Northern Africa"}, {"id": 14777, "dom": "G", "ent": "Bubanza", "cls": "Northern America"}, {"id": 14778, "dom": "G", "ent": "Bubanza", "cls": "Northern Europe"}, {"id": 14779, "dom": "G", "ent": "Bubanza", "cls": "Polynesia"}, {"id": 14780, "dom": "G", "ent": "Bubanza", "cls": "South America"}, {"id": 14781, "dom": "G", "ent": "Bubanza", "cls": "South-eastern Asia"}, {"id": 14782, "dom": "G", "ent": "Bubanza", "cls": "Southern Africa"}, {"id": 14783, "dom": "G", "ent": "Bubanza", "cls": "Southern Asia"}, {"id": 14784, "dom": "G", "ent": "Bubanza", "cls": "Southern Europe"}, {"id": 14785, "dom": "G", "ent": "Bubanza", "cls": "Western Africa"}, {"id": 14786, "dom": "G", "ent": "Bubanza", "cls": "Western Asia"}, {"id": 14787, "dom": "G", "ent": "Bubanza", "cls": "Western Europe"}, {"id": 14788, "dom": "G", "ent": "Bassila", "cls": "Australia and New Zealand"}, {"id": 14789, "dom": "G", "ent": "Bassila", "cls": "Caribbean"}, {"id": 14790, "dom": "G", "ent": "Bassila", "cls": "Central America"}, {"id": 14791, "dom": "G", "ent": "Bassila", "cls": "Central Asia"}, {"id": 14792, "dom": "G", "ent": "Bassila", "cls": "Eastern Africa"}, {"id": 14793, "dom": "G", "ent": "Bassila", "cls": "Eastern Asia"}, {"id": 14794, "dom": "G", "ent": "Bassila", "cls": "Eastern Europe"}, {"id": 14795, "dom": "G", "ent": "Bassila", "cls": "Melanesia"}, {"id": 14796, "dom": "G", "ent": "Bassila", "cls": "Micronesia"}, {"id": 14797, "dom": "G", "ent": "Bassila", "cls": "Middle Africa"}, {"id": 14798, "dom": "G", "ent": "Bassila", "cls": "Northern Africa"}, {"id": 14799, "dom": "G", "ent": "Bassila", "cls": "Northern America"}, {"id": 14800, "dom": "G", "ent": "Bassila", "cls": "Northern Europe"}, {"id": 14801, "dom": "G", "ent": "Bassila", "cls": "Polynesia"}, {"id": 14802, "dom": "G", "ent": "Bassila", "cls": "South America"}, {"id": 14803, "dom": "G", "ent": "Bassila", "cls": "South-eastern Asia"}, {"id": 14804, "dom": "G", "ent": "Bassila", "cls": "Southern Africa"}, {"id": 14805, "dom": "G", "ent": "Bassila", "cls": "Southern Asia"}, {"id": 14806, "dom": "G", "ent": "Bassila", "cls": "Southern Europe"}, {"id": 14807, "dom": "G", "ent": "Bassila", "cls": "Western Africa"}, {"id": 14808, "dom": "G", "ent": "Bassila", "cls": "Western Asia"}, {"id": 14809, "dom": "G", "ent": "Bassila", "cls": "Western Europe"}, {"id": 14810, "dom": "G", "ent": "Tchaourou", "cls": "Australia and New Zealand"}, {"id": 14811, "dom": "G", "ent": "Tchaourou", "cls": "Caribbean"}, {"id": 14812, "dom": "G", "ent": "Tchaourou", "cls": "Central America"}, {"id": 14813, "dom": "G", "ent": "Tchaourou", "cls": "Central Asia"}, {"id": 14814, "dom": "G", "ent": "Tchaourou", "cls": "Eastern Africa"}, {"id": 14815, "dom": "G", "ent": "Tchaourou", "cls": "Eastern Asia"}, {"id": 14816, "dom": "G", "ent": "Tchaourou", "cls": "Eastern Europe"}, {"id": 14817, "dom": "G", "ent": "Tchaourou", "cls": "Melanesia"}, {"id": 14818, "dom": "G", "ent": "Tchaourou", "cls": "Micronesia"}, {"id": 14819, "dom": "G", "ent": "Tchaourou", "cls": "Middle Africa"}, {"id": 14820, "dom": "G", "ent": "Tchaourou", "cls": "Northern Africa"}, {"id": 14821, "dom": "G", "ent": "Tchaourou", "cls": "Northern America"}, {"id": 14822, "dom": "G", "ent": "Tchaourou", "cls": "Northern Europe"}, {"id": 14823, "dom": "G", "ent": "Tchaourou", "cls": "Polynesia"}, {"id": 14824, "dom": "G", "ent": "Tchaourou", "cls": "South America"}, {"id": 14825, "dom": "G", "ent": "Tchaourou", "cls": "South-eastern Asia"}, {"id": 14826, "dom": "G", "ent": "Tchaourou", "cls": "Southern Africa"}, {"id": 14827, "dom": "G", "ent": "Tchaourou", "cls": "Southern Asia"}, {"id": 14828, "dom": "G", "ent": "Tchaourou", "cls": "Southern Europe"}, {"id": 14829, "dom": "G", "ent": "Tchaourou", "cls": "Western Africa"}, {"id": 14830, "dom": "G", "ent": "Tchaourou", "cls": "Western Asia"}, {"id": 14831, "dom": "G", "ent": "Tchaourou", "cls": "Western Europe"}, {"id": 14832, "dom": "G", "ent": "Kpozoun", "cls": "Australia and New Zealand"}, {"id": 14833, "dom": "G", "ent": "Kpozoun", "cls": "Caribbean"}, {"id": 14834, "dom": "G", "ent": "Kpozoun", "cls": "Central America"}, {"id": 14835, "dom": "G", "ent": "Kpozoun", "cls": "Central Asia"}, {"id": 14836, "dom": "G", "ent": "Kpozoun", "cls": "Eastern Africa"}, {"id": 14837, "dom": "G", "ent": "Kpozoun", "cls": "Eastern Asia"}, {"id": 14838, "dom": "G", "ent": "Kpozoun", "cls": "Eastern Europe"}, {"id": 14839, "dom": "G", "ent": "Kpozoun", "cls": "Melanesia"}, {"id": 14840, "dom": "G", "ent": "Kpozoun", "cls": "Micronesia"}, {"id": 14841, "dom": "G", "ent": "Kpozoun", "cls": "Middle Africa"}, {"id": 14842, "dom": "G", "ent": "Kpozoun", "cls": "Northern Africa"}, {"id": 14843, "dom": "G", "ent": "Kpozoun", "cls": "Northern America"}, {"id": 14844, "dom": "G", "ent": "Kpozoun", "cls": "Northern Europe"}, {"id": 14845, "dom": "G", "ent": "Kpozoun", "cls": "Polynesia"}, {"id": 14846, "dom": "G", "ent": "Kpozoun", "cls": "South America"}, {"id": 14847, "dom": "G", "ent": "Kpozoun", "cls": "South-eastern Asia"}, {"id": 14848, "dom": "G", "ent": "Kpozoun", "cls": "Southern Africa"}, {"id": 14849, "dom": "G", "ent": "Kpozoun", "cls": "Southern Asia"}, {"id": 14850, "dom": "G", "ent": "Kpozoun", "cls": "Southern Europe"}, {"id": 14851, "dom": "G", "ent": "Kpozoun", "cls": "Western Africa"}, {"id": 14852, "dom": "G", "ent": "Kpozoun", "cls": "Western Asia"}, {"id": 14853, "dom": "G", "ent": "Kpozoun", "cls": "Western Europe"}, {"id": 14854, "dom": "G", "ent": "Toukountouna", "cls": "Australia and New Zealand"}, {"id": 14855, "dom": "G", "ent": "Toukountouna", "cls": "Caribbean"}, {"id": 14856, "dom": "G", "ent": "Toukountouna", "cls": "Central America"}, {"id": 14857, "dom": "G", "ent": "Toukountouna", "cls": "Central Asia"}, {"id": 14858, "dom": "G", "ent": "Toukountouna", "cls": "Eastern Africa"}, {"id": 14859, "dom": "G", "ent": "Toukountouna", "cls": "Eastern Asia"}, {"id": 14860, "dom": "G", "ent": "Toukountouna", "cls": "Eastern Europe"}, {"id": 14861, "dom": "G", "ent": "Toukountouna", "cls": "Melanesia"}, {"id": 14862, "dom": "G", "ent": "Toukountouna", "cls": "Micronesia"}, {"id": 14863, "dom": "G", "ent": "Toukountouna", "cls": "Middle Africa"}, {"id": 14864, "dom": "G", "ent": "Toukountouna", "cls": "Northern Africa"}, {"id": 14865, "dom": "G", "ent": "Toukountouna", "cls": "Northern America"}, {"id": 14866, "dom": "G", "ent": "Toukountouna", "cls": "Northern Europe"}, {"id": 14867, "dom": "G", "ent": "Toukountouna", "cls": "Polynesia"}, {"id": 14868, "dom": "G", "ent": "Toukountouna", "cls": "South America"}, {"id": 14869, "dom": "G", "ent": "Toukountouna", "cls": "South-eastern Asia"}, {"id": 14870, "dom": "G", "ent": "Toukountouna", "cls": "Southern Africa"}, {"id": 14871, "dom": "G", "ent": "Toukountouna", "cls": "Southern Asia"}, {"id": 14872, "dom": "G", "ent": "Toukountouna", "cls": "Southern Europe"}, {"id": 14873, "dom": "G", "ent": "Toukountouna", "cls": "Western Africa"}, {"id": 14874, "dom": "G", "ent": "Toukountouna", "cls": "Western Asia"}, {"id": 14875, "dom": "G", "ent": "Toukountouna", "cls": "Western Europe"}, {"id": 14876, "dom": "G", "ent": "Sengkurong", "cls": "Australia and New Zealand"}, {"id": 14877, "dom": "G", "ent": "Sengkurong", "cls": "Caribbean"}, {"id": 14878, "dom": "G", "ent": "Sengkurong", "cls": "Central America"}, {"id": 14879, "dom": "G", "ent": "Sengkurong", "cls": "Central Asia"}, {"id": 14880, "dom": "G", "ent": "Sengkurong", "cls": "Eastern Africa"}, {"id": 14881, "dom": "G", "ent": "Sengkurong", "cls": "Eastern Asia"}, {"id": 14882, "dom": "G", "ent": "Sengkurong", "cls": "Eastern Europe"}, {"id": 14883, "dom": "G", "ent": "Sengkurong", "cls": "Melanesia"}, {"id": 14884, "dom": "G", "ent": "Sengkurong", "cls": "Micronesia"}, {"id": 14885, "dom": "G", "ent": "Sengkurong", "cls": "Middle Africa"}, {"id": 14886, "dom": "G", "ent": "Sengkurong", "cls": "Northern Africa"}, {"id": 14887, "dom": "G", "ent": "Sengkurong", "cls": "Northern America"}, {"id": 14888, "dom": "G", "ent": "Sengkurong", "cls": "Northern Europe"}, {"id": 14889, "dom": "G", "ent": "Sengkurong", "cls": "Polynesia"}, {"id": 14890, "dom": "G", "ent": "Sengkurong", "cls": "South America"}, {"id": 14891, "dom": "G", "ent": "Sengkurong", "cls": "South-eastern Asia"}, {"id": 14892, "dom": "G", "ent": "Sengkurong", "cls": "Southern Africa"}, {"id": 14893, "dom": "G", "ent": "Sengkurong", "cls": "Southern Asia"}, {"id": 14894, "dom": "G", "ent": "Sengkurong", "cls": "Southern Europe"}, {"id": 14895, "dom": "G", "ent": "Sengkurong", "cls": "Western Africa"}, {"id": 14896, "dom": "G", "ent": "Sengkurong", "cls": "Western Asia"}, {"id": 14897, "dom": "G", "ent": "Sengkurong", "cls": "Western Europe"}, {"id": 14898, "dom": "G", "ent": "Seria", "cls": "Australia and New Zealand"}, {"id": 14899, "dom": "G", "ent": "Seria", "cls": "Caribbean"}, {"id": 14900, "dom": "G", "ent": "Seria", "cls": "Central America"}, {"id": 14901, "dom": "G", "ent": "Seria", "cls": "Central Asia"}, {"id": 14902, "dom": "G", "ent": "Seria", "cls": "Eastern Africa"}, {"id": 14903, "dom": "G", "ent": "Seria", "cls": "Eastern Asia"}, {"id": 14904, "dom": "G", "ent": "Seria", "cls": "Eastern Europe"}, {"id": 14905, "dom": "G", "ent": "Seria", "cls": "Melanesia"}, {"id": 14906, "dom": "G", "ent": "Seria", "cls": "Micronesia"}, {"id": 14907, "dom": "G", "ent": "Seria", "cls": "Middle Africa"}, {"id": 14908, "dom": "G", "ent": "Seria", "cls": "Northern Africa"}, {"id": 14909, "dom": "G", "ent": "Seria", "cls": "Northern America"}, {"id": 14910, "dom": "G", "ent": "Seria", "cls": "Northern Europe"}, {"id": 14911, "dom": "G", "ent": "Seria", "cls": "Polynesia"}, {"id": 14912, "dom": "G", "ent": "Seria", "cls": "South America"}, {"id": 14913, "dom": "G", "ent": "Seria", "cls": "South-eastern Asia"}, {"id": 14914, "dom": "G", "ent": "Seria", "cls": "Southern Africa"}, {"id": 14915, "dom": "G", "ent": "Seria", "cls": "Southern Asia"}, {"id": 14916, "dom": "G", "ent": "Seria", "cls": "Southern Europe"}, {"id": 14917, "dom": "G", "ent": "Seria", "cls": "Western Africa"}, {"id": 14918, "dom": "G", "ent": "Seria", "cls": "Western Asia"}, {"id": 14919, "dom": "G", "ent": "Seria", "cls": "Western Europe"}, {"id": 14920, "dom": "G", "ent": "Tutong", "cls": "Australia and New Zealand"}, {"id": 14921, "dom": "G", "ent": "Tutong", "cls": "Caribbean"}, {"id": 14922, "dom": "G", "ent": "Tutong", "cls": "Central America"}, {"id": 14923, "dom": "G", "ent": "Tutong", "cls": "Central Asia"}, {"id": 14924, "dom": "G", "ent": "Tutong", "cls": "Eastern Africa"}, {"id": 14925, "dom": "G", "ent": "Tutong", "cls": "Eastern Asia"}, {"id": 14926, "dom": "G", "ent": "Tutong", "cls": "Eastern Europe"}, {"id": 14927, "dom": "G", "ent": "Tutong", "cls": "Melanesia"}, {"id": 14928, "dom": "G", "ent": "Tutong", "cls": "Micronesia"}, {"id": 14929, "dom": "G", "ent": "Tutong", "cls": "Middle Africa"}, {"id": 14930, "dom": "G", "ent": "Tutong", "cls": "Northern Africa"}, {"id": 14931, "dom": "G", "ent": "Tutong", "cls": "Northern America"}, {"id": 14932, "dom": "G", "ent": "Tutong", "cls": "Northern Europe"}, {"id": 14933, "dom": "G", "ent": "Tutong", "cls": "Polynesia"}, {"id": 14934, "dom": "G", "ent": "Tutong", "cls": "South America"}, {"id": 14935, "dom": "G", "ent": "Tutong", "cls": "South-eastern Asia"}, {"id": 14936, "dom": "G", "ent": "Tutong", "cls": "Southern Africa"}, {"id": 14937, "dom": "G", "ent": "Tutong", "cls": "Southern Asia"}, {"id": 14938, "dom": "G", "ent": "Tutong", "cls": "Southern Europe"}, {"id": 14939, "dom": "G", "ent": "Tutong", "cls": "Western Africa"}, {"id": 14940, "dom": "G", "ent": "Tutong", "cls": "Western Asia"}, {"id": 14941, "dom": "G", "ent": "Tutong", "cls": "Western Europe"}, {"id": 14942, "dom": "G", "ent": "Liang", "cls": "Australia and New Zealand"}, {"id": 14943, "dom": "G", "ent": "Liang", "cls": "Caribbean"}, {"id": 14944, "dom": "G", "ent": "Liang", "cls": "Central America"}, {"id": 14945, "dom": "G", "ent": "Liang", "cls": "Central Asia"}, {"id": 14946, "dom": "G", "ent": "Liang", "cls": "Eastern Africa"}, {"id": 14947, "dom": "G", "ent": "Liang", "cls": "Eastern Asia"}, {"id": 14948, "dom": "G", "ent": "Liang", "cls": "Eastern Europe"}, {"id": 14949, "dom": "G", "ent": "Liang", "cls": "Melanesia"}, {"id": 14950, "dom": "G", "ent": "Liang", "cls": "Micronesia"}, {"id": 14951, "dom": "G", "ent": "Liang", "cls": "Middle Africa"}, {"id": 14952, "dom": "G", "ent": "Liang", "cls": "Northern Africa"}, {"id": 14953, "dom": "G", "ent": "Liang", "cls": "Northern America"}, {"id": 14954, "dom": "G", "ent": "Liang", "cls": "Northern Europe"}, {"id": 14955, "dom": "G", "ent": "Liang", "cls": "Polynesia"}, {"id": 14956, "dom": "G", "ent": "Liang", "cls": "South America"}, {"id": 14957, "dom": "G", "ent": "Liang", "cls": "South-eastern Asia"}, {"id": 14958, "dom": "G", "ent": "Liang", "cls": "Southern Africa"}, {"id": 14959, "dom": "G", "ent": "Liang", "cls": "Southern Asia"}, {"id": 14960, "dom": "G", "ent": "Liang", "cls": "Southern Europe"}, {"id": 14961, "dom": "G", "ent": "Liang", "cls": "Western Africa"}, {"id": 14962, "dom": "G", "ent": "Liang", "cls": "Western Asia"}, {"id": 14963, "dom": "G", "ent": "Liang", "cls": "Western Europe"}, {"id": 14964, "dom": "G", "ent": "Riberalta", "cls": "Australia and New Zealand"}, {"id": 14965, "dom": "G", "ent": "Riberalta", "cls": "Caribbean"}, {"id": 14966, "dom": "G", "ent": "Riberalta", "cls": "Central America"}, {"id": 14967, "dom": "G", "ent": "Riberalta", "cls": "Central Asia"}, {"id": 14968, "dom": "G", "ent": "Riberalta", "cls": "Eastern Africa"}, {"id": 14969, "dom": "G", "ent": "Riberalta", "cls": "Eastern Asia"}, {"id": 14970, "dom": "G", "ent": "Riberalta", "cls": "Eastern Europe"}, {"id": 14971, "dom": "G", "ent": "Riberalta", "cls": "Melanesia"}, {"id": 14972, "dom": "G", "ent": "Riberalta", "cls": "Micronesia"}, {"id": 14973, "dom": "G", "ent": "Riberalta", "cls": "Middle Africa"}, {"id": 14974, "dom": "G", "ent": "Riberalta", "cls": "Northern Africa"}, {"id": 14975, "dom": "G", "ent": "Riberalta", "cls": "Northern America"}, {"id": 14976, "dom": "G", "ent": "Riberalta", "cls": "Northern Europe"}, {"id": 14977, "dom": "G", "ent": "Riberalta", "cls": "Polynesia"}, {"id": 14978, "dom": "G", "ent": "Riberalta", "cls": "South America"}, {"id": 14979, "dom": "G", "ent": "Riberalta", "cls": "South-eastern Asia"}, {"id": 14980, "dom": "G", "ent": "Riberalta", "cls": "Southern Africa"}, {"id": 14981, "dom": "G", "ent": "Riberalta", "cls": "Southern Asia"}, {"id": 14982, "dom": "G", "ent": "Riberalta", "cls": "Southern Europe"}, {"id": 14983, "dom": "G", "ent": "Riberalta", "cls": "Western Africa"}, {"id": 14984, "dom": "G", "ent": "Riberalta", "cls": "Western Asia"}, {"id": 14985, "dom": "G", "ent": "Riberalta", "cls": "Western Europe"}, {"id": 14986, "dom": "G", "ent": "Bermejo", "cls": "Australia and New Zealand"}, {"id": 14987, "dom": "G", "ent": "Bermejo", "cls": "Caribbean"}, {"id": 14988, "dom": "G", "ent": "Bermejo", "cls": "Central America"}, {"id": 14989, "dom": "G", "ent": "Bermejo", "cls": "Central Asia"}, {"id": 14990, "dom": "G", "ent": "Bermejo", "cls": "Eastern Africa"}, {"id": 14991, "dom": "G", "ent": "Bermejo", "cls": "Eastern Asia"}, {"id": 14992, "dom": "G", "ent": "Bermejo", "cls": "Eastern Europe"}, {"id": 14993, "dom": "G", "ent": "Bermejo", "cls": "Melanesia"}, {"id": 14994, "dom": "G", "ent": "Bermejo", "cls": "Micronesia"}, {"id": 14995, "dom": "G", "ent": "Bermejo", "cls": "Middle Africa"}, {"id": 14996, "dom": "G", "ent": "Bermejo", "cls": "Northern Africa"}, {"id": 14997, "dom": "G", "ent": "Bermejo", "cls": "Northern America"}, {"id": 14998, "dom": "G", "ent": "Bermejo", "cls": "Northern Europe"}, {"id": 14999, "dom": "G", "ent": "Bermejo", "cls": "Polynesia"}, {"id": 15000, "dom": "G", "ent": "Bermejo", "cls": "South America"}, {"id": 15001, "dom": "G", "ent": "Bermejo", "cls": "South-eastern Asia"}, {"id": 15002, "dom": "G", "ent": "Bermejo", "cls": "Southern Africa"}, {"id": 15003, "dom": "G", "ent": "Bermejo", "cls": "Southern Asia"}, {"id": 15004, "dom": "G", "ent": "Bermejo", "cls": "Southern Europe"}, {"id": 15005, "dom": "G", "ent": "Bermejo", "cls": "Western Africa"}, {"id": 15006, "dom": "G", "ent": "Bermejo", "cls": "Western Asia"}, {"id": 15007, "dom": "G", "ent": "Bermejo", "cls": "Western Europe"}, {"id": 15008, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "Australia and New Zealand"}, {"id": 15009, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "Caribbean"}, {"id": 15010, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "Central America"}, {"id": 15011, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "Central Asia"}, {"id": 15012, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "Eastern Africa"}, {"id": 15013, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "Eastern Asia"}, {"id": 15014, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "Eastern Europe"}, {"id": 15015, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "Melanesia"}, {"id": 15016, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "Micronesia"}, {"id": 15017, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "Middle Africa"}, {"id": 15018, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "Northern Africa"}, {"id": 15019, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "Northern America"}, {"id": 15020, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "Northern Europe"}, {"id": 15021, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "Polynesia"}, {"id": 15022, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "South America"}, {"id": 15023, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "South-eastern Asia"}, {"id": 15024, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "Southern Africa"}, {"id": 15025, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "Southern Asia"}, {"id": 15026, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "Southern Europe"}, {"id": 15027, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "Western Africa"}, {"id": 15028, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "Western Asia"}, {"id": 15029, "dom": "G", "ent": "San Ignacio de Velasco", "cls": "Western Europe"}, {"id": 15030, "dom": "G", "ent": "Tupiza", "cls": "Australia and New Zealand"}, {"id": 15031, "dom": "G", "ent": "Tupiza", "cls": "Caribbean"}, {"id": 15032, "dom": "G", "ent": "Tupiza", "cls": "Central America"}, {"id": 15033, "dom": "G", "ent": "Tupiza", "cls": "Central Asia"}, {"id": 15034, "dom": "G", "ent": "Tupiza", "cls": "Eastern Africa"}, {"id": 15035, "dom": "G", "ent": "Tupiza", "cls": "Eastern Asia"}, {"id": 15036, "dom": "G", "ent": "Tupiza", "cls": "Eastern Europe"}, {"id": 15037, "dom": "G", "ent": "Tupiza", "cls": "Melanesia"}, {"id": 15038, "dom": "G", "ent": "Tupiza", "cls": "Micronesia"}, {"id": 15039, "dom": "G", "ent": "Tupiza", "cls": "Middle Africa"}, {"id": 15040, "dom": "G", "ent": "Tupiza", "cls": "Northern Africa"}, {"id": 15041, "dom": "G", "ent": "Tupiza", "cls": "Northern America"}, {"id": 15042, "dom": "G", "ent": "Tupiza", "cls": "Northern Europe"}, {"id": 15043, "dom": "G", "ent": "Tupiza", "cls": "Polynesia"}, {"id": 15044, "dom": "G", "ent": "Tupiza", "cls": "South America"}, {"id": 15045, "dom": "G", "ent": "Tupiza", "cls": "South-eastern Asia"}, {"id": 15046, "dom": "G", "ent": "Tupiza", "cls": "Southern Africa"}, {"id": 15047, "dom": "G", "ent": "Tupiza", "cls": "Southern Asia"}, {"id": 15048, "dom": "G", "ent": "Tupiza", "cls": "Southern Europe"}, {"id": 15049, "dom": "G", "ent": "Tupiza", "cls": "Western Africa"}, {"id": 15050, "dom": "G", "ent": "Tupiza", "cls": "Western Asia"}, {"id": 15051, "dom": "G", "ent": "Tupiza", "cls": "Western Europe"}, {"id": 15052, "dom": "G", "ent": "Angra dos Reis", "cls": "Australia and New Zealand"}, {"id": 15053, "dom": "G", "ent": "Angra dos Reis", "cls": "Caribbean"}, {"id": 15054, "dom": "G", "ent": "Angra dos Reis", "cls": "Central America"}, {"id": 15055, "dom": "G", "ent": "Angra dos Reis", "cls": "Central Asia"}, {"id": 15056, "dom": "G", "ent": "Angra dos Reis", "cls": "Eastern Africa"}, {"id": 15057, "dom": "G", "ent": "Angra dos Reis", "cls": "Eastern Asia"}, {"id": 15058, "dom": "G", "ent": "Angra dos Reis", "cls": "Eastern Europe"}, {"id": 15059, "dom": "G", "ent": "Angra dos Reis", "cls": "Melanesia"}, {"id": 15060, "dom": "G", "ent": "Angra dos Reis", "cls": "Micronesia"}, {"id": 15061, "dom": "G", "ent": "Angra dos Reis", "cls": "Middle Africa"}, {"id": 15062, "dom": "G", "ent": "Angra dos Reis", "cls": "Northern Africa"}, {"id": 15063, "dom": "G", "ent": "Angra dos Reis", "cls": "Northern America"}, {"id": 15064, "dom": "G", "ent": "Angra dos Reis", "cls": "Northern Europe"}, {"id": 15065, "dom": "G", "ent": "Angra dos Reis", "cls": "Polynesia"}, {"id": 15066, "dom": "G", "ent": "Angra dos Reis", "cls": "South America"}, {"id": 15067, "dom": "G", "ent": "Angra dos Reis", "cls": "South-eastern Asia"}, {"id": 15068, "dom": "G", "ent": "Angra dos Reis", "cls": "Southern Africa"}, {"id": 15069, "dom": "G", "ent": "Angra dos Reis", "cls": "Southern Asia"}, {"id": 15070, "dom": "G", "ent": "Angra dos Reis", "cls": "Southern Europe"}, {"id": 15071, "dom": "G", "ent": "Angra dos Reis", "cls": "Western Africa"}, {"id": 15072, "dom": "G", "ent": "Angra dos Reis", "cls": "Western Asia"}, {"id": 15073, "dom": "G", "ent": "Angra dos Reis", "cls": "Western Europe"}, {"id": 15074, "dom": "G", "ent": "Serrinha", "cls": "Australia and New Zealand"}, {"id": 15075, "dom": "G", "ent": "Serrinha", "cls": "Caribbean"}, {"id": 15076, "dom": "G", "ent": "Serrinha", "cls": "Central America"}, {"id": 15077, "dom": "G", "ent": "Serrinha", "cls": "Central Asia"}, {"id": 15078, "dom": "G", "ent": "Serrinha", "cls": "Eastern Africa"}, {"id": 15079, "dom": "G", "ent": "Serrinha", "cls": "Eastern Asia"}, {"id": 15080, "dom": "G", "ent": "Serrinha", "cls": "Eastern Europe"}, {"id": 15081, "dom": "G", "ent": "Serrinha", "cls": "Melanesia"}, {"id": 15082, "dom": "G", "ent": "Serrinha", "cls": "Micronesia"}, {"id": 15083, "dom": "G", "ent": "Serrinha", "cls": "Middle Africa"}, {"id": 15084, "dom": "G", "ent": "Serrinha", "cls": "Northern Africa"}, {"id": 15085, "dom": "G", "ent": "Serrinha", "cls": "Northern America"}, {"id": 15086, "dom": "G", "ent": "Serrinha", "cls": "Northern Europe"}, {"id": 15087, "dom": "G", "ent": "Serrinha", "cls": "Polynesia"}, {"id": 15088, "dom": "G", "ent": "Serrinha", "cls": "South America"}, {"id": 15089, "dom": "G", "ent": "Serrinha", "cls": "South-eastern Asia"}, {"id": 15090, "dom": "G", "ent": "Serrinha", "cls": "Southern Africa"}, {"id": 15091, "dom": "G", "ent": "Serrinha", "cls": "Southern Asia"}, {"id": 15092, "dom": "G", "ent": "Serrinha", "cls": "Southern Europe"}, {"id": 15093, "dom": "G", "ent": "Serrinha", "cls": "Western Africa"}, {"id": 15094, "dom": "G", "ent": "Serrinha", "cls": "Western Asia"}, {"id": 15095, "dom": "G", "ent": "Serrinha", "cls": "Western Europe"}, {"id": 15096, "dom": "G", "ent": "Nova Soure", "cls": "Australia and New Zealand"}, {"id": 15097, "dom": "G", "ent": "Nova Soure", "cls": "Caribbean"}, {"id": 15098, "dom": "G", "ent": "Nova Soure", "cls": "Central America"}, {"id": 15099, "dom": "G", "ent": "Nova Soure", "cls": "Central Asia"}, {"id": 15100, "dom": "G", "ent": "Nova Soure", "cls": "Eastern Africa"}, {"id": 15101, "dom": "G", "ent": "Nova Soure", "cls": "Eastern Asia"}, {"id": 15102, "dom": "G", "ent": "Nova Soure", "cls": "Eastern Europe"}, {"id": 15103, "dom": "G", "ent": "Nova Soure", "cls": "Melanesia"}, {"id": 15104, "dom": "G", "ent": "Nova Soure", "cls": "Micronesia"}, {"id": 15105, "dom": "G", "ent": "Nova Soure", "cls": "Middle Africa"}, {"id": 15106, "dom": "G", "ent": "Nova Soure", "cls": "Northern Africa"}, {"id": 15107, "dom": "G", "ent": "Nova Soure", "cls": "Northern America"}, {"id": 15108, "dom": "G", "ent": "Nova Soure", "cls": "Northern Europe"}, {"id": 15109, "dom": "G", "ent": "Nova Soure", "cls": "Polynesia"}, {"id": 15110, "dom": "G", "ent": "Nova Soure", "cls": "South America"}, {"id": 15111, "dom": "G", "ent": "Nova Soure", "cls": "South-eastern Asia"}, {"id": 15112, "dom": "G", "ent": "Nova Soure", "cls": "Southern Africa"}, {"id": 15113, "dom": "G", "ent": "Nova Soure", "cls": "Southern Asia"}, {"id": 15114, "dom": "G", "ent": "Nova Soure", "cls": "Southern Europe"}, {"id": 15115, "dom": "G", "ent": "Nova Soure", "cls": "Western Africa"}, {"id": 15116, "dom": "G", "ent": "Nova Soure", "cls": "Western Asia"}, {"id": 15117, "dom": "G", "ent": "Nova Soure", "cls": "Western Europe"}, {"id": 15118, "dom": "G", "ent": "Campos Belos", "cls": "Australia and New Zealand"}, {"id": 15119, "dom": "G", "ent": "Campos Belos", "cls": "Caribbean"}, {"id": 15120, "dom": "G", "ent": "Campos Belos", "cls": "Central America"}, {"id": 15121, "dom": "G", "ent": "Campos Belos", "cls": "Central Asia"}, {"id": 15122, "dom": "G", "ent": "Campos Belos", "cls": "Eastern Africa"}, {"id": 15123, "dom": "G", "ent": "Campos Belos", "cls": "Eastern Asia"}, {"id": 15124, "dom": "G", "ent": "Campos Belos", "cls": "Eastern Europe"}, {"id": 15125, "dom": "G", "ent": "Campos Belos", "cls": "Melanesia"}, {"id": 15126, "dom": "G", "ent": "Campos Belos", "cls": "Micronesia"}, {"id": 15127, "dom": "G", "ent": "Campos Belos", "cls": "Middle Africa"}, {"id": 15128, "dom": "G", "ent": "Campos Belos", "cls": "Northern Africa"}, {"id": 15129, "dom": "G", "ent": "Campos Belos", "cls": "Northern America"}, {"id": 15130, "dom": "G", "ent": "Campos Belos", "cls": "Northern Europe"}, {"id": 15131, "dom": "G", "ent": "Campos Belos", "cls": "Polynesia"}, {"id": 15132, "dom": "G", "ent": "Campos Belos", "cls": "South America"}, {"id": 15133, "dom": "G", "ent": "Campos Belos", "cls": "South-eastern Asia"}, {"id": 15134, "dom": "G", "ent": "Campos Belos", "cls": "Southern Africa"}, {"id": 15135, "dom": "G", "ent": "Campos Belos", "cls": "Southern Asia"}, {"id": 15136, "dom": "G", "ent": "Campos Belos", "cls": "Southern Europe"}, {"id": 15137, "dom": "G", "ent": "Campos Belos", "cls": "Western Africa"}, {"id": 15138, "dom": "G", "ent": "Campos Belos", "cls": "Western Asia"}, {"id": 15139, "dom": "G", "ent": "Campos Belos", "cls": "Western Europe"}, {"id": 15140, "dom": "G", "ent": "Tlokweng", "cls": "Australia and New Zealand"}, {"id": 15141, "dom": "G", "ent": "Tlokweng", "cls": "Caribbean"}, {"id": 15142, "dom": "G", "ent": "Tlokweng", "cls": "Central America"}, {"id": 15143, "dom": "G", "ent": "Tlokweng", "cls": "Central Asia"}, {"id": 15144, "dom": "G", "ent": "Tlokweng", "cls": "Eastern Africa"}, {"id": 15145, "dom": "G", "ent": "Tlokweng", "cls": "Eastern Asia"}, {"id": 15146, "dom": "G", "ent": "Tlokweng", "cls": "Eastern Europe"}, {"id": 15147, "dom": "G", "ent": "Tlokweng", "cls": "Melanesia"}, {"id": 15148, "dom": "G", "ent": "Tlokweng", "cls": "Micronesia"}, {"id": 15149, "dom": "G", "ent": "Tlokweng", "cls": "Middle Africa"}, {"id": 15150, "dom": "G", "ent": "Tlokweng", "cls": "Northern Africa"}, {"id": 15151, "dom": "G", "ent": "Tlokweng", "cls": "Northern America"}, {"id": 15152, "dom": "G", "ent": "Tlokweng", "cls": "Northern Europe"}, {"id": 15153, "dom": "G", "ent": "Tlokweng", "cls": "Polynesia"}, {"id": 15154, "dom": "G", "ent": "Tlokweng", "cls": "South America"}, {"id": 15155, "dom": "G", "ent": "Tlokweng", "cls": "South-eastern Asia"}, {"id": 15156, "dom": "G", "ent": "Tlokweng", "cls": "Southern Africa"}, {"id": 15157, "dom": "G", "ent": "Tlokweng", "cls": "Southern Asia"}, {"id": 15158, "dom": "G", "ent": "Tlokweng", "cls": "Southern Europe"}, {"id": 15159, "dom": "G", "ent": "Tlokweng", "cls": "Western Africa"}, {"id": 15160, "dom": "G", "ent": "Tlokweng", "cls": "Western Asia"}, {"id": 15161, "dom": "G", "ent": "Tlokweng", "cls": "Western Europe"}, {"id": 15162, "dom": "G", "ent": "Kanye", "cls": "Australia and New Zealand"}, {"id": 15163, "dom": "G", "ent": "Kanye", "cls": "Caribbean"}, {"id": 15164, "dom": "G", "ent": "Kanye", "cls": "Central America"}, {"id": 15165, "dom": "G", "ent": "Kanye", "cls": "Central Asia"}, {"id": 15166, "dom": "G", "ent": "Kanye", "cls": "Eastern Africa"}, {"id": 15167, "dom": "G", "ent": "Kanye", "cls": "Eastern Asia"}, {"id": 15168, "dom": "G", "ent": "Kanye", "cls": "Eastern Europe"}, {"id": 15169, "dom": "G", "ent": "Kanye", "cls": "Melanesia"}, {"id": 15170, "dom": "G", "ent": "Kanye", "cls": "Micronesia"}, {"id": 15171, "dom": "G", "ent": "Kanye", "cls": "Middle Africa"}, {"id": 15172, "dom": "G", "ent": "Kanye", "cls": "Northern Africa"}, {"id": 15173, "dom": "G", "ent": "Kanye", "cls": "Northern America"}, {"id": 15174, "dom": "G", "ent": "Kanye", "cls": "Northern Europe"}, {"id": 15175, "dom": "G", "ent": "Kanye", "cls": "Polynesia"}, {"id": 15176, "dom": "G", "ent": "Kanye", "cls": "South America"}, {"id": 15177, "dom": "G", "ent": "Kanye", "cls": "South-eastern Asia"}, {"id": 15178, "dom": "G", "ent": "Kanye", "cls": "Southern Africa"}, {"id": 15179, "dom": "G", "ent": "Kanye", "cls": "Southern Asia"}, {"id": 15180, "dom": "G", "ent": "Kanye", "cls": "Southern Europe"}, {"id": 15181, "dom": "G", "ent": "Kanye", "cls": "Western Africa"}, {"id": 15182, "dom": "G", "ent": "Kanye", "cls": "Western Asia"}, {"id": 15183, "dom": "G", "ent": "Kanye", "cls": "Western Europe"}, {"id": 15184, "dom": "G", "ent": "Thamaga", "cls": "Australia and New Zealand"}, {"id": 15185, "dom": "G", "ent": "Thamaga", "cls": "Caribbean"}, {"id": 15186, "dom": "G", "ent": "Thamaga", "cls": "Central America"}, {"id": 15187, "dom": "G", "ent": "Thamaga", "cls": "Central Asia"}, {"id": 15188, "dom": "G", "ent": "Thamaga", "cls": "Eastern Africa"}, {"id": 15189, "dom": "G", "ent": "Thamaga", "cls": "Eastern Asia"}, {"id": 15190, "dom": "G", "ent": "Thamaga", "cls": "Eastern Europe"}, {"id": 15191, "dom": "G", "ent": "Thamaga", "cls": "Melanesia"}, {"id": 15192, "dom": "G", "ent": "Thamaga", "cls": "Micronesia"}, {"id": 15193, "dom": "G", "ent": "Thamaga", "cls": "Middle Africa"}, {"id": 15194, "dom": "G", "ent": "Thamaga", "cls": "Northern Africa"}, {"id": 15195, "dom": "G", "ent": "Thamaga", "cls": "Northern America"}, {"id": 15196, "dom": "G", "ent": "Thamaga", "cls": "Northern Europe"}, {"id": 15197, "dom": "G", "ent": "Thamaga", "cls": "Polynesia"}, {"id": 15198, "dom": "G", "ent": "Thamaga", "cls": "South America"}, {"id": 15199, "dom": "G", "ent": "Thamaga", "cls": "South-eastern Asia"}, {"id": 15200, "dom": "G", "ent": "Thamaga", "cls": "Southern Africa"}, {"id": 15201, "dom": "G", "ent": "Thamaga", "cls": "Southern Asia"}, {"id": 15202, "dom": "G", "ent": "Thamaga", "cls": "Southern Europe"}, {"id": 15203, "dom": "G", "ent": "Thamaga", "cls": "Western Africa"}, {"id": 15204, "dom": "G", "ent": "Thamaga", "cls": "Western Asia"}, {"id": 15205, "dom": "G", "ent": "Thamaga", "cls": "Western Europe"}, {"id": 15206, "dom": "G", "ent": "Janeng", "cls": "Australia and New Zealand"}, {"id": 15207, "dom": "G", "ent": "Janeng", "cls": "Caribbean"}, {"id": 15208, "dom": "G", "ent": "Janeng", "cls": "Central America"}, {"id": 15209, "dom": "G", "ent": "Janeng", "cls": "Central Asia"}, {"id": 15210, "dom": "G", "ent": "Janeng", "cls": "Eastern Africa"}, {"id": 15211, "dom": "G", "ent": "Janeng", "cls": "Eastern Asia"}, {"id": 15212, "dom": "G", "ent": "Janeng", "cls": "Eastern Europe"}, {"id": 15213, "dom": "G", "ent": "Janeng", "cls": "Melanesia"}, {"id": 15214, "dom": "G", "ent": "Janeng", "cls": "Micronesia"}, {"id": 15215, "dom": "G", "ent": "Janeng", "cls": "Middle Africa"}, {"id": 15216, "dom": "G", "ent": "Janeng", "cls": "Northern Africa"}, {"id": 15217, "dom": "G", "ent": "Janeng", "cls": "Northern America"}, {"id": 15218, "dom": "G", "ent": "Janeng", "cls": "Northern Europe"}, {"id": 15219, "dom": "G", "ent": "Janeng", "cls": "Polynesia"}, {"id": 15220, "dom": "G", "ent": "Janeng", "cls": "South America"}, {"id": 15221, "dom": "G", "ent": "Janeng", "cls": "South-eastern Asia"}, {"id": 15222, "dom": "G", "ent": "Janeng", "cls": "Southern Africa"}, {"id": 15223, "dom": "G", "ent": "Janeng", "cls": "Southern Asia"}, {"id": 15224, "dom": "G", "ent": "Janeng", "cls": "Southern Europe"}, {"id": 15225, "dom": "G", "ent": "Janeng", "cls": "Western Africa"}, {"id": 15226, "dom": "G", "ent": "Janeng", "cls": "Western Asia"}, {"id": 15227, "dom": "G", "ent": "Janeng", "cls": "Western Europe"}, {"id": 15228, "dom": "G", "ent": "Orsha", "cls": "Australia and New Zealand"}, {"id": 15229, "dom": "G", "ent": "Orsha", "cls": "Caribbean"}, {"id": 15230, "dom": "G", "ent": "Orsha", "cls": "Central America"}, {"id": 15231, "dom": "G", "ent": "Orsha", "cls": "Central Asia"}, {"id": 15232, "dom": "G", "ent": "Orsha", "cls": "Eastern Africa"}, {"id": 15233, "dom": "G", "ent": "Orsha", "cls": "Eastern Asia"}, {"id": 15234, "dom": "G", "ent": "Orsha", "cls": "Eastern Europe"}, {"id": 15235, "dom": "G", "ent": "Orsha", "cls": "Melanesia"}, {"id": 15236, "dom": "G", "ent": "Orsha", "cls": "Micronesia"}, {"id": 15237, "dom": "G", "ent": "Orsha", "cls": "Middle Africa"}, {"id": 15238, "dom": "G", "ent": "Orsha", "cls": "Northern Africa"}, {"id": 15239, "dom": "G", "ent": "Orsha", "cls": "Northern America"}, {"id": 15240, "dom": "G", "ent": "Orsha", "cls": "Northern Europe"}, {"id": 15241, "dom": "G", "ent": "Orsha", "cls": "Polynesia"}, {"id": 15242, "dom": "G", "ent": "Orsha", "cls": "South America"}, {"id": 15243, "dom": "G", "ent": "Orsha", "cls": "South-eastern Asia"}, {"id": 15244, "dom": "G", "ent": "Orsha", "cls": "Southern Africa"}, {"id": 15245, "dom": "G", "ent": "Orsha", "cls": "Southern Asia"}, {"id": 15246, "dom": "G", "ent": "Orsha", "cls": "Southern Europe"}, {"id": 15247, "dom": "G", "ent": "Orsha", "cls": "Western Africa"}, {"id": 15248, "dom": "G", "ent": "Orsha", "cls": "Western Asia"}, {"id": 15249, "dom": "G", "ent": "Orsha", "cls": "Western Europe"}, {"id": 15250, "dom": "G", "ent": "Slonim", "cls": "Australia and New Zealand"}, {"id": 15251, "dom": "G", "ent": "Slonim", "cls": "Caribbean"}, {"id": 15252, "dom": "G", "ent": "Slonim", "cls": "Central America"}, {"id": 15253, "dom": "G", "ent": "Slonim", "cls": "Central Asia"}, {"id": 15254, "dom": "G", "ent": "Slonim", "cls": "Eastern Africa"}, {"id": 15255, "dom": "G", "ent": "Slonim", "cls": "Eastern Asia"}, {"id": 15256, "dom": "G", "ent": "Slonim", "cls": "Eastern Europe"}, {"id": 15257, "dom": "G", "ent": "Slonim", "cls": "Melanesia"}, {"id": 15258, "dom": "G", "ent": "Slonim", "cls": "Micronesia"}, {"id": 15259, "dom": "G", "ent": "Slonim", "cls": "Middle Africa"}, {"id": 15260, "dom": "G", "ent": "Slonim", "cls": "Northern Africa"}, {"id": 15261, "dom": "G", "ent": "Slonim", "cls": "Northern America"}, {"id": 15262, "dom": "G", "ent": "Slonim", "cls": "Northern Europe"}, {"id": 15263, "dom": "G", "ent": "Slonim", "cls": "Polynesia"}, {"id": 15264, "dom": "G", "ent": "Slonim", "cls": "South America"}, {"id": 15265, "dom": "G", "ent": "Slonim", "cls": "South-eastern Asia"}, {"id": 15266, "dom": "G", "ent": "Slonim", "cls": "Southern Africa"}, {"id": 15267, "dom": "G", "ent": "Slonim", "cls": "Southern Asia"}, {"id": 15268, "dom": "G", "ent": "Slonim", "cls": "Southern Europe"}, {"id": 15269, "dom": "G", "ent": "Slonim", "cls": "Western Africa"}, {"id": 15270, "dom": "G", "ent": "Slonim", "cls": "Western Asia"}, {"id": 15271, "dom": "G", "ent": "Slonim", "cls": "Western Europe"}, {"id": 15272, "dom": "G", "ent": "Novogrudok", "cls": "Australia and New Zealand"}, {"id": 15273, "dom": "G", "ent": "Novogrudok", "cls": "Caribbean"}, {"id": 15274, "dom": "G", "ent": "Novogrudok", "cls": "Central America"}, {"id": 15275, "dom": "G", "ent": "Novogrudok", "cls": "Central Asia"}, {"id": 15276, "dom": "G", "ent": "Novogrudok", "cls": "Eastern Africa"}, {"id": 15277, "dom": "G", "ent": "Novogrudok", "cls": "Eastern Asia"}, {"id": 15278, "dom": "G", "ent": "Novogrudok", "cls": "Eastern Europe"}, {"id": 15279, "dom": "G", "ent": "Novogrudok", "cls": "Melanesia"}, {"id": 15280, "dom": "G", "ent": "Novogrudok", "cls": "Micronesia"}, {"id": 15281, "dom": "G", "ent": "Novogrudok", "cls": "Middle Africa"}, {"id": 15282, "dom": "G", "ent": "Novogrudok", "cls": "Northern Africa"}, {"id": 15283, "dom": "G", "ent": "Novogrudok", "cls": "Northern America"}, {"id": 15284, "dom": "G", "ent": "Novogrudok", "cls": "Northern Europe"}, {"id": 15285, "dom": "G", "ent": "Novogrudok", "cls": "Polynesia"}, {"id": 15286, "dom": "G", "ent": "Novogrudok", "cls": "South America"}, {"id": 15287, "dom": "G", "ent": "Novogrudok", "cls": "South-eastern Asia"}, {"id": 15288, "dom": "G", "ent": "Novogrudok", "cls": "Southern Africa"}, {"id": 15289, "dom": "G", "ent": "Novogrudok", "cls": "Southern Asia"}, {"id": 15290, "dom": "G", "ent": "Novogrudok", "cls": "Southern Europe"}, {"id": 15291, "dom": "G", "ent": "Novogrudok", "cls": "Western Africa"}, {"id": 15292, "dom": "G", "ent": "Novogrudok", "cls": "Western Asia"}, {"id": 15293, "dom": "G", "ent": "Novogrudok", "cls": "Western Europe"}, {"id": 15294, "dom": "G", "ent": "Ivatsevichi", "cls": "Australia and New Zealand"}, {"id": 15295, "dom": "G", "ent": "Ivatsevichi", "cls": "Caribbean"}, {"id": 15296, "dom": "G", "ent": "Ivatsevichi", "cls": "Central America"}, {"id": 15297, "dom": "G", "ent": "Ivatsevichi", "cls": "Central Asia"}, {"id": 15298, "dom": "G", "ent": "Ivatsevichi", "cls": "Eastern Africa"}, {"id": 15299, "dom": "G", "ent": "Ivatsevichi", "cls": "Eastern Asia"}, {"id": 15300, "dom": "G", "ent": "Ivatsevichi", "cls": "Eastern Europe"}, {"id": 15301, "dom": "G", "ent": "Ivatsevichi", "cls": "Melanesia"}, {"id": 15302, "dom": "G", "ent": "Ivatsevichi", "cls": "Micronesia"}, {"id": 15303, "dom": "G", "ent": "Ivatsevichi", "cls": "Middle Africa"}, {"id": 15304, "dom": "G", "ent": "Ivatsevichi", "cls": "Northern Africa"}, {"id": 15305, "dom": "G", "ent": "Ivatsevichi", "cls": "Northern America"}, {"id": 15306, "dom": "G", "ent": "Ivatsevichi", "cls": "Northern Europe"}, {"id": 15307, "dom": "G", "ent": "Ivatsevichi", "cls": "Polynesia"}, {"id": 15308, "dom": "G", "ent": "Ivatsevichi", "cls": "South America"}, {"id": 15309, "dom": "G", "ent": "Ivatsevichi", "cls": "South-eastern Asia"}, {"id": 15310, "dom": "G", "ent": "Ivatsevichi", "cls": "Southern Africa"}, {"id": 15311, "dom": "G", "ent": "Ivatsevichi", "cls": "Southern Asia"}, {"id": 15312, "dom": "G", "ent": "Ivatsevichi", "cls": "Southern Europe"}, {"id": 15313, "dom": "G", "ent": "Ivatsevichi", "cls": "Western Africa"}, {"id": 15314, "dom": "G", "ent": "Ivatsevichi", "cls": "Western Asia"}, {"id": 15315, "dom": "G", "ent": "Ivatsevichi", "cls": "Western Europe"}, {"id": 15316, "dom": "G", "ent": "Downtown Vancouver", "cls": "Australia and New Zealand"}, {"id": 15317, "dom": "G", "ent": "Downtown Vancouver", "cls": "Caribbean"}, {"id": 15318, "dom": "G", "ent": "Downtown Vancouver", "cls": "Central America"}, {"id": 15319, "dom": "G", "ent": "Downtown Vancouver", "cls": "Central Asia"}, {"id": 15320, "dom": "G", "ent": "Downtown Vancouver", "cls": "Eastern Africa"}, {"id": 15321, "dom": "G", "ent": "Downtown Vancouver", "cls": "Eastern Asia"}, {"id": 15322, "dom": "G", "ent": "Downtown Vancouver", "cls": "Eastern Europe"}, {"id": 15323, "dom": "G", "ent": "Downtown Vancouver", "cls": "Melanesia"}, {"id": 15324, "dom": "G", "ent": "Downtown Vancouver", "cls": "Micronesia"}, {"id": 15325, "dom": "G", "ent": "Downtown Vancouver", "cls": "Middle Africa"}, {"id": 15326, "dom": "G", "ent": "Downtown Vancouver", "cls": "Northern Africa"}, {"id": 15327, "dom": "G", "ent": "Downtown Vancouver", "cls": "Northern America"}, {"id": 15328, "dom": "G", "ent": "Downtown Vancouver", "cls": "Northern Europe"}, {"id": 15329, "dom": "G", "ent": "Downtown Vancouver", "cls": "Polynesia"}, {"id": 15330, "dom": "G", "ent": "Downtown Vancouver", "cls": "South America"}, {"id": 15331, "dom": "G", "ent": "Downtown Vancouver", "cls": "South-eastern Asia"}, {"id": 15332, "dom": "G", "ent": "Downtown Vancouver", "cls": "Southern Africa"}, {"id": 15333, "dom": "G", "ent": "Downtown Vancouver", "cls": "Southern Asia"}, {"id": 15334, "dom": "G", "ent": "Downtown Vancouver", "cls": "Southern Europe"}, {"id": 15335, "dom": "G", "ent": "Downtown Vancouver", "cls": "Western Africa"}, {"id": 15336, "dom": "G", "ent": "Downtown Vancouver", "cls": "Western Asia"}, {"id": 15337, "dom": "G", "ent": "Downtown Vancouver", "cls": "Western Europe"}, {"id": 15338, "dom": "G", "ent": "Courtenay", "cls": "Australia and New Zealand"}, {"id": 15339, "dom": "G", "ent": "Courtenay", "cls": "Caribbean"}, {"id": 15340, "dom": "G", "ent": "Courtenay", "cls": "Central America"}, {"id": 15341, "dom": "G", "ent": "Courtenay", "cls": "Central Asia"}, {"id": 15342, "dom": "G", "ent": "Courtenay", "cls": "Eastern Africa"}, {"id": 15343, "dom": "G", "ent": "Courtenay", "cls": "Eastern Asia"}, {"id": 15344, "dom": "G", "ent": "Courtenay", "cls": "Eastern Europe"}, {"id": 15345, "dom": "G", "ent": "Courtenay", "cls": "Melanesia"}, {"id": 15346, "dom": "G", "ent": "Courtenay", "cls": "Micronesia"}, {"id": 15347, "dom": "G", "ent": "Courtenay", "cls": "Middle Africa"}, {"id": 15348, "dom": "G", "ent": "Courtenay", "cls": "Northern Africa"}, {"id": 15349, "dom": "G", "ent": "Courtenay", "cls": "Northern America"}, {"id": 15350, "dom": "G", "ent": "Courtenay", "cls": "Northern Europe"}, {"id": 15351, "dom": "G", "ent": "Courtenay", "cls": "Polynesia"}, {"id": 15352, "dom": "G", "ent": "Courtenay", "cls": "South America"}, {"id": 15353, "dom": "G", "ent": "Courtenay", "cls": "South-eastern Asia"}, {"id": 15354, "dom": "G", "ent": "Courtenay", "cls": "Southern Africa"}, {"id": 15355, "dom": "G", "ent": "Courtenay", "cls": "Southern Asia"}, {"id": 15356, "dom": "G", "ent": "Courtenay", "cls": "Southern Europe"}, {"id": 15357, "dom": "G", "ent": "Courtenay", "cls": "Western Africa"}, {"id": 15358, "dom": "G", "ent": "Courtenay", "cls": "Western Asia"}, {"id": 15359, "dom": "G", "ent": "Courtenay", "cls": "Western Europe"}, {"id": 15360, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "Australia and New Zealand"}, {"id": 15361, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "Caribbean"}, {"id": 15362, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "Central America"}, {"id": 15363, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "Central Asia"}, {"id": 15364, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "Eastern Africa"}, {"id": 15365, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "Eastern Asia"}, {"id": 15366, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "Eastern Europe"}, {"id": 15367, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "Melanesia"}, {"id": 15368, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "Micronesia"}, {"id": 15369, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "Middle Africa"}, {"id": 15370, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "Northern Africa"}, {"id": 15371, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "Northern America"}, {"id": 15372, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "Northern Europe"}, {"id": 15373, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "Polynesia"}, {"id": 15374, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "South America"}, {"id": 15375, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "South-eastern Asia"}, {"id": 15376, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "Southern Africa"}, {"id": 15377, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "Southern Asia"}, {"id": 15378, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "Southern Europe"}, {"id": 15379, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "Western Africa"}, {"id": 15380, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "Western Asia"}, {"id": 15381, "dom": "G", "ent": "Willowridge-Martingrove-Richview", "cls": "Western Europe"}, {"id": 15382, "dom": "G", "ent": "Clarence-Rockland", "cls": "Australia and New Zealand"}, {"id": 15383, "dom": "G", "ent": "Clarence-Rockland", "cls": "Caribbean"}, {"id": 15384, "dom": "G", "ent": "Clarence-Rockland", "cls": "Central America"}, {"id": 15385, "dom": "G", "ent": "Clarence-Rockland", "cls": "Central Asia"}, {"id": 15386, "dom": "G", "ent": "Clarence-Rockland", "cls": "Eastern Africa"}, {"id": 15387, "dom": "G", "ent": "Clarence-Rockland", "cls": "Eastern Asia"}, {"id": 15388, "dom": "G", "ent": "Clarence-Rockland", "cls": "Eastern Europe"}, {"id": 15389, "dom": "G", "ent": "Clarence-Rockland", "cls": "Melanesia"}, {"id": 15390, "dom": "G", "ent": "Clarence-Rockland", "cls": "Micronesia"}, {"id": 15391, "dom": "G", "ent": "Clarence-Rockland", "cls": "Middle Africa"}, {"id": 15392, "dom": "G", "ent": "Clarence-Rockland", "cls": "Northern Africa"}, {"id": 15393, "dom": "G", "ent": "Clarence-Rockland", "cls": "Northern America"}, {"id": 15394, "dom": "G", "ent": "Clarence-Rockland", "cls": "Northern Europe"}, {"id": 15395, "dom": "G", "ent": "Clarence-Rockland", "cls": "Polynesia"}, {"id": 15396, "dom": "G", "ent": "Clarence-Rockland", "cls": "South America"}, {"id": 15397, "dom": "G", "ent": "Clarence-Rockland", "cls": "South-eastern Asia"}, {"id": 15398, "dom": "G", "ent": "Clarence-Rockland", "cls": "Southern Africa"}, {"id": 15399, "dom": "G", "ent": "Clarence-Rockland", "cls": "Southern Asia"}, {"id": 15400, "dom": "G", "ent": "Clarence-Rockland", "cls": "Southern Europe"}, {"id": 15401, "dom": "G", "ent": "Clarence-Rockland", "cls": "Western Africa"}, {"id": 15402, "dom": "G", "ent": "Clarence-Rockland", "cls": "Western Asia"}, {"id": 15403, "dom": "G", "ent": "Clarence-Rockland", "cls": "Western Europe"}, {"id": 15404, "dom": "G", "ent": "Bukavu", "cls": "Australia and New Zealand"}, {"id": 15405, "dom": "G", "ent": "Bukavu", "cls": "Caribbean"}, {"id": 15406, "dom": "G", "ent": "Bukavu", "cls": "Central America"}, {"id": 15407, "dom": "G", "ent": "Bukavu", "cls": "Central Asia"}, {"id": 15408, "dom": "G", "ent": "Bukavu", "cls": "Eastern Africa"}, {"id": 15409, "dom": "G", "ent": "Bukavu", "cls": "Eastern Asia"}, {"id": 15410, "dom": "G", "ent": "Bukavu", "cls": "Eastern Europe"}, {"id": 15411, "dom": "G", "ent": "Bukavu", "cls": "Melanesia"}, {"id": 15412, "dom": "G", "ent": "Bukavu", "cls": "Micronesia"}, {"id": 15413, "dom": "G", "ent": "Bukavu", "cls": "Middle Africa"}, {"id": 15414, "dom": "G", "ent": "Bukavu", "cls": "Northern Africa"}, {"id": 15415, "dom": "G", "ent": "Bukavu", "cls": "Northern America"}, {"id": 15416, "dom": "G", "ent": "Bukavu", "cls": "Northern Europe"}, {"id": 15417, "dom": "G", "ent": "Bukavu", "cls": "Polynesia"}, {"id": 15418, "dom": "G", "ent": "Bukavu", "cls": "South America"}, {"id": 15419, "dom": "G", "ent": "Bukavu", "cls": "South-eastern Asia"}, {"id": 15420, "dom": "G", "ent": "Bukavu", "cls": "Southern Africa"}, {"id": 15421, "dom": "G", "ent": "Bukavu", "cls": "Southern Asia"}, {"id": 15422, "dom": "G", "ent": "Bukavu", "cls": "Southern Europe"}, {"id": 15423, "dom": "G", "ent": "Bukavu", "cls": "Western Africa"}, {"id": 15424, "dom": "G", "ent": "Bukavu", "cls": "Western Asia"}, {"id": 15425, "dom": "G", "ent": "Bukavu", "cls": "Western Europe"}, {"id": 15426, "dom": "G", "ent": "Mwene", "cls": "Australia and New Zealand"}, {"id": 15427, "dom": "G", "ent": "Mwene", "cls": "Caribbean"}, {"id": 15428, "dom": "G", "ent": "Mwene", "cls": "Central America"}, {"id": 15429, "dom": "G", "ent": "Mwene", "cls": "Central Asia"}, {"id": 15430, "dom": "G", "ent": "Mwene", "cls": "Eastern Africa"}, {"id": 15431, "dom": "G", "ent": "Mwene", "cls": "Eastern Asia"}, {"id": 15432, "dom": "G", "ent": "Mwene", "cls": "Eastern Europe"}, {"id": 15433, "dom": "G", "ent": "Mwene", "cls": "Melanesia"}, {"id": 15434, "dom": "G", "ent": "Mwene", "cls": "Micronesia"}, {"id": 15435, "dom": "G", "ent": "Mwene", "cls": "Middle Africa"}, {"id": 15436, "dom": "G", "ent": "Mwene", "cls": "Northern Africa"}, {"id": 15437, "dom": "G", "ent": "Mwene", "cls": "Northern America"}, {"id": 15438, "dom": "G", "ent": "Mwene", "cls": "Northern Europe"}, {"id": 15439, "dom": "G", "ent": "Mwene", "cls": "Polynesia"}, {"id": 15440, "dom": "G", "ent": "Mwene", "cls": "South America"}, {"id": 15441, "dom": "G", "ent": "Mwene", "cls": "South-eastern Asia"}, {"id": 15442, "dom": "G", "ent": "Mwene", "cls": "Southern Africa"}, {"id": 15443, "dom": "G", "ent": "Mwene", "cls": "Southern Asia"}, {"id": 15444, "dom": "G", "ent": "Mwene", "cls": "Southern Europe"}, {"id": 15445, "dom": "G", "ent": "Mwene", "cls": "Western Africa"}, {"id": 15446, "dom": "G", "ent": "Mwene", "cls": "Western Asia"}, {"id": 15447, "dom": "G", "ent": "Mwene", "cls": "Western Europe"}, {"id": 15448, "dom": "G", "ent": "Djugu", "cls": "Australia and New Zealand"}, {"id": 15449, "dom": "G", "ent": "Djugu", "cls": "Caribbean"}, {"id": 15450, "dom": "G", "ent": "Djugu", "cls": "Central America"}, {"id": 15451, "dom": "G", "ent": "Djugu", "cls": "Central Asia"}, {"id": 15452, "dom": "G", "ent": "Djugu", "cls": "Eastern Africa"}, {"id": 15453, "dom": "G", "ent": "Djugu", "cls": "Eastern Asia"}, {"id": 15454, "dom": "G", "ent": "Djugu", "cls": "Eastern Europe"}, {"id": 15455, "dom": "G", "ent": "Djugu", "cls": "Melanesia"}, {"id": 15456, "dom": "G", "ent": "Djugu", "cls": "Micronesia"}, {"id": 15457, "dom": "G", "ent": "Djugu", "cls": "Middle Africa"}, {"id": 15458, "dom": "G", "ent": "Djugu", "cls": "Northern Africa"}, {"id": 15459, "dom": "G", "ent": "Djugu", "cls": "Northern America"}, {"id": 15460, "dom": "G", "ent": "Djugu", "cls": "Northern Europe"}, {"id": 15461, "dom": "G", "ent": "Djugu", "cls": "Polynesia"}, {"id": 15462, "dom": "G", "ent": "Djugu", "cls": "South America"}, {"id": 15463, "dom": "G", "ent": "Djugu", "cls": "South-eastern Asia"}, {"id": 15464, "dom": "G", "ent": "Djugu", "cls": "Southern Africa"}, {"id": 15465, "dom": "G", "ent": "Djugu", "cls": "Southern Asia"}, {"id": 15466, "dom": "G", "ent": "Djugu", "cls": "Southern Europe"}, {"id": 15467, "dom": "G", "ent": "Djugu", "cls": "Western Africa"}, {"id": 15468, "dom": "G", "ent": "Djugu", "cls": "Western Asia"}, {"id": 15469, "dom": "G", "ent": "Djugu", "cls": "Western Europe"}, {"id": 15470, "dom": "G", "ent": "Kabare", "cls": "Australia and New Zealand"}, {"id": 15471, "dom": "G", "ent": "Kabare", "cls": "Caribbean"}, {"id": 15472, "dom": "G", "ent": "Kabare", "cls": "Central America"}, {"id": 15473, "dom": "G", "ent": "Kabare", "cls": "Central Asia"}, {"id": 15474, "dom": "G", "ent": "Kabare", "cls": "Eastern Africa"}, {"id": 15475, "dom": "G", "ent": "Kabare", "cls": "Eastern Asia"}, {"id": 15476, "dom": "G", "ent": "Kabare", "cls": "Eastern Europe"}, {"id": 15477, "dom": "G", "ent": "Kabare", "cls": "Melanesia"}, {"id": 15478, "dom": "G", "ent": "Kabare", "cls": "Micronesia"}, {"id": 15479, "dom": "G", "ent": "Kabare", "cls": "Middle Africa"}, {"id": 15480, "dom": "G", "ent": "Kabare", "cls": "Northern Africa"}, {"id": 15481, "dom": "G", "ent": "Kabare", "cls": "Northern America"}, {"id": 15482, "dom": "G", "ent": "Kabare", "cls": "Northern Europe"}, {"id": 15483, "dom": "G", "ent": "Kabare", "cls": "Polynesia"}, {"id": 15484, "dom": "G", "ent": "Kabare", "cls": "South America"}, {"id": 15485, "dom": "G", "ent": "Kabare", "cls": "South-eastern Asia"}, {"id": 15486, "dom": "G", "ent": "Kabare", "cls": "Southern Africa"}, {"id": 15487, "dom": "G", "ent": "Kabare", "cls": "Southern Asia"}, {"id": 15488, "dom": "G", "ent": "Kabare", "cls": "Southern Europe"}, {"id": 15489, "dom": "G", "ent": "Kabare", "cls": "Western Africa"}, {"id": 15490, "dom": "G", "ent": "Kabare", "cls": "Western Asia"}, {"id": 15491, "dom": "G", "ent": "Kabare", "cls": "Western Europe"}, {"id": 15492, "dom": "G", "ent": "Bocaranga", "cls": "Australia and New Zealand"}, {"id": 15493, "dom": "G", "ent": "Bocaranga", "cls": "Caribbean"}, {"id": 15494, "dom": "G", "ent": "Bocaranga", "cls": "Central America"}, {"id": 15495, "dom": "G", "ent": "Bocaranga", "cls": "Central Asia"}, {"id": 15496, "dom": "G", "ent": "Bocaranga", "cls": "Eastern Africa"}, {"id": 15497, "dom": "G", "ent": "Bocaranga", "cls": "Eastern Asia"}, {"id": 15498, "dom": "G", "ent": "Bocaranga", "cls": "Eastern Europe"}, {"id": 15499, "dom": "G", "ent": "Bocaranga", "cls": "Melanesia"}, {"id": 15500, "dom": "G", "ent": "Bocaranga", "cls": "Micronesia"}, {"id": 15501, "dom": "G", "ent": "Bocaranga", "cls": "Middle Africa"}, {"id": 15502, "dom": "G", "ent": "Bocaranga", "cls": "Northern Africa"}, {"id": 15503, "dom": "G", "ent": "Bocaranga", "cls": "Northern America"}, {"id": 15504, "dom": "G", "ent": "Bocaranga", "cls": "Northern Europe"}, {"id": 15505, "dom": "G", "ent": "Bocaranga", "cls": "Polynesia"}, {"id": 15506, "dom": "G", "ent": "Bocaranga", "cls": "South America"}, {"id": 15507, "dom": "G", "ent": "Bocaranga", "cls": "South-eastern Asia"}, {"id": 15508, "dom": "G", "ent": "Bocaranga", "cls": "Southern Africa"}, {"id": 15509, "dom": "G", "ent": "Bocaranga", "cls": "Southern Asia"}, {"id": 15510, "dom": "G", "ent": "Bocaranga", "cls": "Southern Europe"}, {"id": 15511, "dom": "G", "ent": "Bocaranga", "cls": "Western Africa"}, {"id": 15512, "dom": "G", "ent": "Bocaranga", "cls": "Western Asia"}, {"id": 15513, "dom": "G", "ent": "Bocaranga", "cls": "Western Europe"}, {"id": 15514, "dom": "G", "ent": "Batangafo", "cls": "Australia and New Zealand"}, {"id": 15515, "dom": "G", "ent": "Batangafo", "cls": "Caribbean"}, {"id": 15516, "dom": "G", "ent": "Batangafo", "cls": "Central America"}, {"id": 15517, "dom": "G", "ent": "Batangafo", "cls": "Central Asia"}, {"id": 15518, "dom": "G", "ent": "Batangafo", "cls": "Eastern Africa"}, {"id": 15519, "dom": "G", "ent": "Batangafo", "cls": "Eastern Asia"}, {"id": 15520, "dom": "G", "ent": "Batangafo", "cls": "Eastern Europe"}, {"id": 15521, "dom": "G", "ent": "Batangafo", "cls": "Melanesia"}, {"id": 15522, "dom": "G", "ent": "Batangafo", "cls": "Micronesia"}, {"id": 15523, "dom": "G", "ent": "Batangafo", "cls": "Middle Africa"}, {"id": 15524, "dom": "G", "ent": "Batangafo", "cls": "Northern Africa"}, {"id": 15525, "dom": "G", "ent": "Batangafo", "cls": "Northern America"}, {"id": 15526, "dom": "G", "ent": "Batangafo", "cls": "Northern Europe"}, {"id": 15527, "dom": "G", "ent": "Batangafo", "cls": "Polynesia"}, {"id": 15528, "dom": "G", "ent": "Batangafo", "cls": "South America"}, {"id": 15529, "dom": "G", "ent": "Batangafo", "cls": "South-eastern Asia"}, {"id": 15530, "dom": "G", "ent": "Batangafo", "cls": "Southern Africa"}, {"id": 15531, "dom": "G", "ent": "Batangafo", "cls": "Southern Asia"}, {"id": 15532, "dom": "G", "ent": "Batangafo", "cls": "Southern Europe"}, {"id": 15533, "dom": "G", "ent": "Batangafo", "cls": "Western Africa"}, {"id": 15534, "dom": "G", "ent": "Batangafo", "cls": "Western Asia"}, {"id": 15535, "dom": "G", "ent": "Batangafo", "cls": "Western Europe"}, {"id": 15536, "dom": "G", "ent": "Dekoa", "cls": "Australia and New Zealand"}, {"id": 15537, "dom": "G", "ent": "Dekoa", "cls": "Caribbean"}, {"id": 15538, "dom": "G", "ent": "Dekoa", "cls": "Central America"}, {"id": 15539, "dom": "G", "ent": "Dekoa", "cls": "Central Asia"}, {"id": 15540, "dom": "G", "ent": "Dekoa", "cls": "Eastern Africa"}, {"id": 15541, "dom": "G", "ent": "Dekoa", "cls": "Eastern Asia"}, {"id": 15542, "dom": "G", "ent": "Dekoa", "cls": "Eastern Europe"}, {"id": 15543, "dom": "G", "ent": "Dekoa", "cls": "Melanesia"}, {"id": 15544, "dom": "G", "ent": "Dekoa", "cls": "Micronesia"}, {"id": 15545, "dom": "G", "ent": "Dekoa", "cls": "Middle Africa"}, {"id": 15546, "dom": "G", "ent": "Dekoa", "cls": "Northern Africa"}, {"id": 15547, "dom": "G", "ent": "Dekoa", "cls": "Northern America"}, {"id": 15548, "dom": "G", "ent": "Dekoa", "cls": "Northern Europe"}, {"id": 15549, "dom": "G", "ent": "Dekoa", "cls": "Polynesia"}, {"id": 15550, "dom": "G", "ent": "Dekoa", "cls": "South America"}, {"id": 15551, "dom": "G", "ent": "Dekoa", "cls": "South-eastern Asia"}, {"id": 15552, "dom": "G", "ent": "Dekoa", "cls": "Southern Africa"}, {"id": 15553, "dom": "G", "ent": "Dekoa", "cls": "Southern Asia"}, {"id": 15554, "dom": "G", "ent": "Dekoa", "cls": "Southern Europe"}, {"id": 15555, "dom": "G", "ent": "Dekoa", "cls": "Western Africa"}, {"id": 15556, "dom": "G", "ent": "Dekoa", "cls": "Western Asia"}, {"id": 15557, "dom": "G", "ent": "Dekoa", "cls": "Western Europe"}, {"id": 15558, "dom": "G", "ent": "Kabo", "cls": "Australia and New Zealand"}, {"id": 15559, "dom": "G", "ent": "Kabo", "cls": "Caribbean"}, {"id": 15560, "dom": "G", "ent": "Kabo", "cls": "Central America"}, {"id": 15561, "dom": "G", "ent": "Kabo", "cls": "Central Asia"}, {"id": 15562, "dom": "G", "ent": "Kabo", "cls": "Eastern Africa"}, {"id": 15563, "dom": "G", "ent": "Kabo", "cls": "Eastern Asia"}, {"id": 15564, "dom": "G", "ent": "Kabo", "cls": "Eastern Europe"}, {"id": 15565, "dom": "G", "ent": "Kabo", "cls": "Melanesia"}, {"id": 15566, "dom": "G", "ent": "Kabo", "cls": "Micronesia"}, {"id": 15567, "dom": "G", "ent": "Kabo", "cls": "Middle Africa"}, {"id": 15568, "dom": "G", "ent": "Kabo", "cls": "Northern Africa"}, {"id": 15569, "dom": "G", "ent": "Kabo", "cls": "Northern America"}, {"id": 15570, "dom": "G", "ent": "Kabo", "cls": "Northern Europe"}, {"id": 15571, "dom": "G", "ent": "Kabo", "cls": "Polynesia"}, {"id": 15572, "dom": "G", "ent": "Kabo", "cls": "South America"}, {"id": 15573, "dom": "G", "ent": "Kabo", "cls": "South-eastern Asia"}, {"id": 15574, "dom": "G", "ent": "Kabo", "cls": "Southern Africa"}, {"id": 15575, "dom": "G", "ent": "Kabo", "cls": "Southern Asia"}, {"id": 15576, "dom": "G", "ent": "Kabo", "cls": "Southern Europe"}, {"id": 15577, "dom": "G", "ent": "Kabo", "cls": "Western Africa"}, {"id": 15578, "dom": "G", "ent": "Kabo", "cls": "Western Asia"}, {"id": 15579, "dom": "G", "ent": "Kabo", "cls": "Western Europe"}, {"id": 15580, "dom": "G", "ent": "Owando", "cls": "Australia and New Zealand"}, {"id": 15581, "dom": "G", "ent": "Owando", "cls": "Caribbean"}, {"id": 15582, "dom": "G", "ent": "Owando", "cls": "Central America"}, {"id": 15583, "dom": "G", "ent": "Owando", "cls": "Central Asia"}, {"id": 15584, "dom": "G", "ent": "Owando", "cls": "Eastern Africa"}, {"id": 15585, "dom": "G", "ent": "Owando", "cls": "Eastern Asia"}, {"id": 15586, "dom": "G", "ent": "Owando", "cls": "Eastern Europe"}, {"id": 15587, "dom": "G", "ent": "Owando", "cls": "Melanesia"}, {"id": 15588, "dom": "G", "ent": "Owando", "cls": "Micronesia"}, {"id": 15589, "dom": "G", "ent": "Owando", "cls": "Middle Africa"}, {"id": 15590, "dom": "G", "ent": "Owando", "cls": "Northern Africa"}, {"id": 15591, "dom": "G", "ent": "Owando", "cls": "Northern America"}, {"id": 15592, "dom": "G", "ent": "Owando", "cls": "Northern Europe"}, {"id": 15593, "dom": "G", "ent": "Owando", "cls": "Polynesia"}, {"id": 15594, "dom": "G", "ent": "Owando", "cls": "South America"}, {"id": 15595, "dom": "G", "ent": "Owando", "cls": "South-eastern Asia"}, {"id": 15596, "dom": "G", "ent": "Owando", "cls": "Southern Africa"}, {"id": 15597, "dom": "G", "ent": "Owando", "cls": "Southern Asia"}, {"id": 15598, "dom": "G", "ent": "Owando", "cls": "Southern Europe"}, {"id": 15599, "dom": "G", "ent": "Owando", "cls": "Western Africa"}, {"id": 15600, "dom": "G", "ent": "Owando", "cls": "Western Asia"}, {"id": 15601, "dom": "G", "ent": "Owando", "cls": "Western Europe"}, {"id": 15602, "dom": "G", "ent": "Gamboma", "cls": "Australia and New Zealand"}, {"id": 15603, "dom": "G", "ent": "Gamboma", "cls": "Caribbean"}, {"id": 15604, "dom": "G", "ent": "Gamboma", "cls": "Central America"}, {"id": 15605, "dom": "G", "ent": "Gamboma", "cls": "Central Asia"}, {"id": 15606, "dom": "G", "ent": "Gamboma", "cls": "Eastern Africa"}, {"id": 15607, "dom": "G", "ent": "Gamboma", "cls": "Eastern Asia"}, {"id": 15608, "dom": "G", "ent": "Gamboma", "cls": "Eastern Europe"}, {"id": 15609, "dom": "G", "ent": "Gamboma", "cls": "Melanesia"}, {"id": 15610, "dom": "G", "ent": "Gamboma", "cls": "Micronesia"}, {"id": 15611, "dom": "G", "ent": "Gamboma", "cls": "Middle Africa"}, {"id": 15612, "dom": "G", "ent": "Gamboma", "cls": "Northern Africa"}, {"id": 15613, "dom": "G", "ent": "Gamboma", "cls": "Northern America"}, {"id": 15614, "dom": "G", "ent": "Gamboma", "cls": "Northern Europe"}, {"id": 15615, "dom": "G", "ent": "Gamboma", "cls": "Polynesia"}, {"id": 15616, "dom": "G", "ent": "Gamboma", "cls": "South America"}, {"id": 15617, "dom": "G", "ent": "Gamboma", "cls": "South-eastern Asia"}, {"id": 15618, "dom": "G", "ent": "Gamboma", "cls": "Southern Africa"}, {"id": 15619, "dom": "G", "ent": "Gamboma", "cls": "Southern Asia"}, {"id": 15620, "dom": "G", "ent": "Gamboma", "cls": "Southern Europe"}, {"id": 15621, "dom": "G", "ent": "Gamboma", "cls": "Western Africa"}, {"id": 15622, "dom": "G", "ent": "Gamboma", "cls": "Western Asia"}, {"id": 15623, "dom": "G", "ent": "Gamboma", "cls": "Western Europe"}, {"id": 15624, "dom": "G", "ent": "Mouyondzi", "cls": "Australia and New Zealand"}, {"id": 15625, "dom": "G", "ent": "Mouyondzi", "cls": "Caribbean"}, {"id": 15626, "dom": "G", "ent": "Mouyondzi", "cls": "Central America"}, {"id": 15627, "dom": "G", "ent": "Mouyondzi", "cls": "Central Asia"}, {"id": 15628, "dom": "G", "ent": "Mouyondzi", "cls": "Eastern Africa"}, {"id": 15629, "dom": "G", "ent": "Mouyondzi", "cls": "Eastern Asia"}, {"id": 15630, "dom": "G", "ent": "Mouyondzi", "cls": "Eastern Europe"}, {"id": 15631, "dom": "G", "ent": "Mouyondzi", "cls": "Melanesia"}, {"id": 15632, "dom": "G", "ent": "Mouyondzi", "cls": "Micronesia"}, {"id": 15633, "dom": "G", "ent": "Mouyondzi", "cls": "Middle Africa"}, {"id": 15634, "dom": "G", "ent": "Mouyondzi", "cls": "Northern Africa"}, {"id": 15635, "dom": "G", "ent": "Mouyondzi", "cls": "Northern America"}, {"id": 15636, "dom": "G", "ent": "Mouyondzi", "cls": "Northern Europe"}, {"id": 15637, "dom": "G", "ent": "Mouyondzi", "cls": "Polynesia"}, {"id": 15638, "dom": "G", "ent": "Mouyondzi", "cls": "South America"}, {"id": 15639, "dom": "G", "ent": "Mouyondzi", "cls": "South-eastern Asia"}, {"id": 15640, "dom": "G", "ent": "Mouyondzi", "cls": "Southern Africa"}, {"id": 15641, "dom": "G", "ent": "Mouyondzi", "cls": "Southern Asia"}, {"id": 15642, "dom": "G", "ent": "Mouyondzi", "cls": "Southern Europe"}, {"id": 15643, "dom": "G", "ent": "Mouyondzi", "cls": "Western Africa"}, {"id": 15644, "dom": "G", "ent": "Mouyondzi", "cls": "Western Asia"}, {"id": 15645, "dom": "G", "ent": "Mouyondzi", "cls": "Western Europe"}, {"id": 15646, "dom": "G", "ent": "Pokola", "cls": "Australia and New Zealand"}, {"id": 15647, "dom": "G", "ent": "Pokola", "cls": "Caribbean"}, {"id": 15648, "dom": "G", "ent": "Pokola", "cls": "Central America"}, {"id": 15649, "dom": "G", "ent": "Pokola", "cls": "Central Asia"}, {"id": 15650, "dom": "G", "ent": "Pokola", "cls": "Eastern Africa"}, {"id": 15651, "dom": "G", "ent": "Pokola", "cls": "Eastern Asia"}, {"id": 15652, "dom": "G", "ent": "Pokola", "cls": "Eastern Europe"}, {"id": 15653, "dom": "G", "ent": "Pokola", "cls": "Melanesia"}, {"id": 15654, "dom": "G", "ent": "Pokola", "cls": "Micronesia"}, {"id": 15655, "dom": "G", "ent": "Pokola", "cls": "Middle Africa"}, {"id": 15656, "dom": "G", "ent": "Pokola", "cls": "Northern Africa"}, {"id": 15657, "dom": "G", "ent": "Pokola", "cls": "Northern America"}, {"id": 15658, "dom": "G", "ent": "Pokola", "cls": "Northern Europe"}, {"id": 15659, "dom": "G", "ent": "Pokola", "cls": "Polynesia"}, {"id": 15660, "dom": "G", "ent": "Pokola", "cls": "South America"}, {"id": 15661, "dom": "G", "ent": "Pokola", "cls": "South-eastern Asia"}, {"id": 15662, "dom": "G", "ent": "Pokola", "cls": "Southern Africa"}, {"id": 15663, "dom": "G", "ent": "Pokola", "cls": "Southern Asia"}, {"id": 15664, "dom": "G", "ent": "Pokola", "cls": "Southern Europe"}, {"id": 15665, "dom": "G", "ent": "Pokola", "cls": "Western Africa"}, {"id": 15666, "dom": "G", "ent": "Pokola", "cls": "Western Asia"}, {"id": 15667, "dom": "G", "ent": "Pokola", "cls": "Western Europe"}, {"id": 15668, "dom": "G", "ent": "Sankt Gallen", "cls": "Australia and New Zealand"}, {"id": 15669, "dom": "G", "ent": "Sankt Gallen", "cls": "Caribbean"}, {"id": 15670, "dom": "G", "ent": "Sankt Gallen", "cls": "Central America"}, {"id": 15671, "dom": "G", "ent": "Sankt Gallen", "cls": "Central Asia"}, {"id": 15672, "dom": "G", "ent": "Sankt Gallen", "cls": "Eastern Africa"}, {"id": 15673, "dom": "G", "ent": "Sankt Gallen", "cls": "Eastern Asia"}, {"id": 15674, "dom": "G", "ent": "Sankt Gallen", "cls": "Eastern Europe"}, {"id": 15675, "dom": "G", "ent": "Sankt Gallen", "cls": "Melanesia"}, {"id": 15676, "dom": "G", "ent": "Sankt Gallen", "cls": "Micronesia"}, {"id": 15677, "dom": "G", "ent": "Sankt Gallen", "cls": "Middle Africa"}, {"id": 15678, "dom": "G", "ent": "Sankt Gallen", "cls": "Northern Africa"}, {"id": 15679, "dom": "G", "ent": "Sankt Gallen", "cls": "Northern America"}, {"id": 15680, "dom": "G", "ent": "Sankt Gallen", "cls": "Northern Europe"}, {"id": 15681, "dom": "G", "ent": "Sankt Gallen", "cls": "Polynesia"}, {"id": 15682, "dom": "G", "ent": "Sankt Gallen", "cls": "South America"}, {"id": 15683, "dom": "G", "ent": "Sankt Gallen", "cls": "South-eastern Asia"}, {"id": 15684, "dom": "G", "ent": "Sankt Gallen", "cls": "Southern Africa"}, {"id": 15685, "dom": "G", "ent": "Sankt Gallen", "cls": "Southern Asia"}, {"id": 15686, "dom": "G", "ent": "Sankt Gallen", "cls": "Southern Europe"}, {"id": 15687, "dom": "G", "ent": "Sankt Gallen", "cls": "Western Africa"}, {"id": 15688, "dom": "G", "ent": "Sankt Gallen", "cls": "Western Asia"}, {"id": 15689, "dom": "G", "ent": "Sankt Gallen", "cls": "Western Europe"}, {"id": 15690, "dom": "G", "ent": "Sitten", "cls": "Australia and New Zealand"}, {"id": 15691, "dom": "G", "ent": "Sitten", "cls": "Caribbean"}, {"id": 15692, "dom": "G", "ent": "Sitten", "cls": "Central America"}, {"id": 15693, "dom": "G", "ent": "Sitten", "cls": "Central Asia"}, {"id": 15694, "dom": "G", "ent": "Sitten", "cls": "Eastern Africa"}, {"id": 15695, "dom": "G", "ent": "Sitten", "cls": "Eastern Asia"}, {"id": 15696, "dom": "G", "ent": "Sitten", "cls": "Eastern Europe"}, {"id": 15697, "dom": "G", "ent": "Sitten", "cls": "Melanesia"}, {"id": 15698, "dom": "G", "ent": "Sitten", "cls": "Micronesia"}, {"id": 15699, "dom": "G", "ent": "Sitten", "cls": "Middle Africa"}, {"id": 15700, "dom": "G", "ent": "Sitten", "cls": "Northern Africa"}, {"id": 15701, "dom": "G", "ent": "Sitten", "cls": "Northern America"}, {"id": 15702, "dom": "G", "ent": "Sitten", "cls": "Northern Europe"}, {"id": 15703, "dom": "G", "ent": "Sitten", "cls": "Polynesia"}, {"id": 15704, "dom": "G", "ent": "Sitten", "cls": "South America"}, {"id": 15705, "dom": "G", "ent": "Sitten", "cls": "South-eastern Asia"}, {"id": 15706, "dom": "G", "ent": "Sitten", "cls": "Southern Africa"}, {"id": 15707, "dom": "G", "ent": "Sitten", "cls": "Southern Asia"}, {"id": 15708, "dom": "G", "ent": "Sitten", "cls": "Southern Europe"}, {"id": 15709, "dom": "G", "ent": "Sitten", "cls": "Western Africa"}, {"id": 15710, "dom": "G", "ent": "Sitten", "cls": "Western Asia"}, {"id": 15711, "dom": "G", "ent": "Sitten", "cls": "Western Europe"}, {"id": 15712, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "Australia and New Zealand"}, {"id": 15713, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "Caribbean"}, {"id": 15714, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "Central America"}, {"id": 15715, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "Central Asia"}, {"id": 15716, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "Eastern Africa"}, {"id": 15717, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "Eastern Asia"}, {"id": 15718, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "Eastern Europe"}, {"id": 15719, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "Melanesia"}, {"id": 15720, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "Micronesia"}, {"id": 15721, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "Middle Africa"}, {"id": 15722, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "Northern Africa"}, {"id": 15723, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "Northern America"}, {"id": 15724, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "Northern Europe"}, {"id": 15725, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "Polynesia"}, {"id": 15726, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "South America"}, {"id": 15727, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "South-eastern Asia"}, {"id": 15728, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "Southern Africa"}, {"id": 15729, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "Southern Asia"}, {"id": 15730, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "Southern Europe"}, {"id": 15731, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "Western Africa"}, {"id": 15732, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "Western Asia"}, {"id": 15733, "dom": "G", "ent": "Yverdon-les-Bains", "cls": "Western Europe"}, {"id": 15734, "dom": "G", "ent": "Wettingen", "cls": "Australia and New Zealand"}, {"id": 15735, "dom": "G", "ent": "Wettingen", "cls": "Caribbean"}, {"id": 15736, "dom": "G", "ent": "Wettingen", "cls": "Central America"}, {"id": 15737, "dom": "G", "ent": "Wettingen", "cls": "Central Asia"}, {"id": 15738, "dom": "G", "ent": "Wettingen", "cls": "Eastern Africa"}, {"id": 15739, "dom": "G", "ent": "Wettingen", "cls": "Eastern Asia"}, {"id": 15740, "dom": "G", "ent": "Wettingen", "cls": "Eastern Europe"}, {"id": 15741, "dom": "G", "ent": "Wettingen", "cls": "Melanesia"}, {"id": 15742, "dom": "G", "ent": "Wettingen", "cls": "Micronesia"}, {"id": 15743, "dom": "G", "ent": "Wettingen", "cls": "Middle Africa"}, {"id": 15744, "dom": "G", "ent": "Wettingen", "cls": "Northern Africa"}, {"id": 15745, "dom": "G", "ent": "Wettingen", "cls": "Northern America"}, {"id": 15746, "dom": "G", "ent": "Wettingen", "cls": "Northern Europe"}, {"id": 15747, "dom": "G", "ent": "Wettingen", "cls": "Polynesia"}, {"id": 15748, "dom": "G", "ent": "Wettingen", "cls": "South America"}, {"id": 15749, "dom": "G", "ent": "Wettingen", "cls": "South-eastern Asia"}, {"id": 15750, "dom": "G", "ent": "Wettingen", "cls": "Southern Africa"}, {"id": 15751, "dom": "G", "ent": "Wettingen", "cls": "Southern Asia"}, {"id": 15752, "dom": "G", "ent": "Wettingen", "cls": "Southern Europe"}, {"id": 15753, "dom": "G", "ent": "Wettingen", "cls": "Western Africa"}, {"id": 15754, "dom": "G", "ent": "Wettingen", "cls": "Western Asia"}, {"id": 15755, "dom": "G", "ent": "Wettingen", "cls": "Western Europe"}, {"id": 15756, "dom": "G", "ent": "Katiola", "cls": "Australia and New Zealand"}, {"id": 15757, "dom": "G", "ent": "Katiola", "cls": "Caribbean"}, {"id": 15758, "dom": "G", "ent": "Katiola", "cls": "Central America"}, {"id": 15759, "dom": "G", "ent": "Katiola", "cls": "Central Asia"}, {"id": 15760, "dom": "G", "ent": "Katiola", "cls": "Eastern Africa"}, {"id": 15761, "dom": "G", "ent": "Katiola", "cls": "Eastern Asia"}, {"id": 15762, "dom": "G", "ent": "Katiola", "cls": "Eastern Europe"}, {"id": 15763, "dom": "G", "ent": "Katiola", "cls": "Melanesia"}, {"id": 15764, "dom": "G", "ent": "Katiola", "cls": "Micronesia"}, {"id": 15765, "dom": "G", "ent": "Katiola", "cls": "Middle Africa"}, {"id": 15766, "dom": "G", "ent": "Katiola", "cls": "Northern Africa"}, {"id": 15767, "dom": "G", "ent": "Katiola", "cls": "Northern America"}, {"id": 15768, "dom": "G", "ent": "Katiola", "cls": "Northern Europe"}, {"id": 15769, "dom": "G", "ent": "Katiola", "cls": "Polynesia"}, {"id": 15770, "dom": "G", "ent": "Katiola", "cls": "South America"}, {"id": 15771, "dom": "G", "ent": "Katiola", "cls": "South-eastern Asia"}, {"id": 15772, "dom": "G", "ent": "Katiola", "cls": "Southern Africa"}, {"id": 15773, "dom": "G", "ent": "Katiola", "cls": "Southern Asia"}, {"id": 15774, "dom": "G", "ent": "Katiola", "cls": "Southern Europe"}, {"id": 15775, "dom": "G", "ent": "Katiola", "cls": "Western Africa"}, {"id": 15776, "dom": "G", "ent": "Katiola", "cls": "Western Asia"}, {"id": 15777, "dom": "G", "ent": "Katiola", "cls": "Western Europe"}, {"id": 15778, "dom": "G", "ent": "Tanda", "cls": "Australia and New Zealand"}, {"id": 15779, "dom": "G", "ent": "Tanda", "cls": "Caribbean"}, {"id": 15780, "dom": "G", "ent": "Tanda", "cls": "Central America"}, {"id": 15781, "dom": "G", "ent": "Tanda", "cls": "Central Asia"}, {"id": 15782, "dom": "G", "ent": "Tanda", "cls": "Eastern Africa"}, {"id": 15783, "dom": "G", "ent": "Tanda", "cls": "Eastern Asia"}, {"id": 15784, "dom": "G", "ent": "Tanda", "cls": "Eastern Europe"}, {"id": 15785, "dom": "G", "ent": "Tanda", "cls": "Melanesia"}, {"id": 15786, "dom": "G", "ent": "Tanda", "cls": "Micronesia"}, {"id": 15787, "dom": "G", "ent": "Tanda", "cls": "Middle Africa"}, {"id": 15788, "dom": "G", "ent": "Tanda", "cls": "Northern Africa"}, {"id": 15789, "dom": "G", "ent": "Tanda", "cls": "Northern America"}, {"id": 15790, "dom": "G", "ent": "Tanda", "cls": "Northern Europe"}, {"id": 15791, "dom": "G", "ent": "Tanda", "cls": "Polynesia"}, {"id": 15792, "dom": "G", "ent": "Tanda", "cls": "South America"}, {"id": 15793, "dom": "G", "ent": "Tanda", "cls": "South-eastern Asia"}, {"id": 15794, "dom": "G", "ent": "Tanda", "cls": "Southern Africa"}, {"id": 15795, "dom": "G", "ent": "Tanda", "cls": "Southern Asia"}, {"id": 15796, "dom": "G", "ent": "Tanda", "cls": "Southern Europe"}, {"id": 15797, "dom": "G", "ent": "Tanda", "cls": "Western Africa"}, {"id": 15798, "dom": "G", "ent": "Tanda", "cls": "Western Asia"}, {"id": 15799, "dom": "G", "ent": "Tanda", "cls": "Western Europe"}, {"id": 15800, "dom": "G", "ent": "Diourouzon", "cls": "Australia and New Zealand"}, {"id": 15801, "dom": "G", "ent": "Diourouzon", "cls": "Caribbean"}, {"id": 15802, "dom": "G", "ent": "Diourouzon", "cls": "Central America"}, {"id": 15803, "dom": "G", "ent": "Diourouzon", "cls": "Central Asia"}, {"id": 15804, "dom": "G", "ent": "Diourouzon", "cls": "Eastern Africa"}, {"id": 15805, "dom": "G", "ent": "Diourouzon", "cls": "Eastern Asia"}, {"id": 15806, "dom": "G", "ent": "Diourouzon", "cls": "Eastern Europe"}, {"id": 15807, "dom": "G", "ent": "Diourouzon", "cls": "Melanesia"}, {"id": 15808, "dom": "G", "ent": "Diourouzon", "cls": "Micronesia"}, {"id": 15809, "dom": "G", "ent": "Diourouzon", "cls": "Middle Africa"}, {"id": 15810, "dom": "G", "ent": "Diourouzon", "cls": "Northern Africa"}, {"id": 15811, "dom": "G", "ent": "Diourouzon", "cls": "Northern America"}, {"id": 15812, "dom": "G", "ent": "Diourouzon", "cls": "Northern Europe"}, {"id": 15813, "dom": "G", "ent": "Diourouzon", "cls": "Polynesia"}, {"id": 15814, "dom": "G", "ent": "Diourouzon", "cls": "South America"}, {"id": 15815, "dom": "G", "ent": "Diourouzon", "cls": "South-eastern Asia"}, {"id": 15816, "dom": "G", "ent": "Diourouzon", "cls": "Southern Africa"}, {"id": 15817, "dom": "G", "ent": "Diourouzon", "cls": "Southern Asia"}, {"id": 15818, "dom": "G", "ent": "Diourouzon", "cls": "Southern Europe"}, {"id": 15819, "dom": "G", "ent": "Diourouzon", "cls": "Western Africa"}, {"id": 15820, "dom": "G", "ent": "Diourouzon", "cls": "Western Asia"}, {"id": 15821, "dom": "G", "ent": "Diourouzon", "cls": "Western Europe"}, {"id": 15822, "dom": "G", "ent": "Nzianouan", "cls": "Australia and New Zealand"}, {"id": 15823, "dom": "G", "ent": "Nzianouan", "cls": "Caribbean"}, {"id": 15824, "dom": "G", "ent": "Nzianouan", "cls": "Central America"}, {"id": 15825, "dom": "G", "ent": "Nzianouan", "cls": "Central Asia"}, {"id": 15826, "dom": "G", "ent": "Nzianouan", "cls": "Eastern Africa"}, {"id": 15827, "dom": "G", "ent": "Nzianouan", "cls": "Eastern Asia"}, {"id": 15828, "dom": "G", "ent": "Nzianouan", "cls": "Eastern Europe"}, {"id": 15829, "dom": "G", "ent": "Nzianouan", "cls": "Melanesia"}, {"id": 15830, "dom": "G", "ent": "Nzianouan", "cls": "Micronesia"}, {"id": 15831, "dom": "G", "ent": "Nzianouan", "cls": "Middle Africa"}, {"id": 15832, "dom": "G", "ent": "Nzianouan", "cls": "Northern Africa"}, {"id": 15833, "dom": "G", "ent": "Nzianouan", "cls": "Northern America"}, {"id": 15834, "dom": "G", "ent": "Nzianouan", "cls": "Northern Europe"}, {"id": 15835, "dom": "G", "ent": "Nzianouan", "cls": "Polynesia"}, {"id": 15836, "dom": "G", "ent": "Nzianouan", "cls": "South America"}, {"id": 15837, "dom": "G", "ent": "Nzianouan", "cls": "South-eastern Asia"}, {"id": 15838, "dom": "G", "ent": "Nzianouan", "cls": "Southern Africa"}, {"id": 15839, "dom": "G", "ent": "Nzianouan", "cls": "Southern Asia"}, {"id": 15840, "dom": "G", "ent": "Nzianouan", "cls": "Southern Europe"}, {"id": 15841, "dom": "G", "ent": "Nzianouan", "cls": "Western Africa"}, {"id": 15842, "dom": "G", "ent": "Nzianouan", "cls": "Western Asia"}, {"id": 15843, "dom": "G", "ent": "Nzianouan", "cls": "Western Europe"}, {"id": 15844, "dom": "G", "ent": "La Serena", "cls": "Australia and New Zealand"}, {"id": 15845, "dom": "G", "ent": "La Serena", "cls": "Caribbean"}, {"id": 15846, "dom": "G", "ent": "La Serena", "cls": "Central America"}, {"id": 15847, "dom": "G", "ent": "La Serena", "cls": "Central Asia"}, {"id": 15848, "dom": "G", "ent": "La Serena", "cls": "Eastern Africa"}, {"id": 15849, "dom": "G", "ent": "La Serena", "cls": "Eastern Asia"}, {"id": 15850, "dom": "G", "ent": "La Serena", "cls": "Eastern Europe"}, {"id": 15851, "dom": "G", "ent": "La Serena", "cls": "Melanesia"}, {"id": 15852, "dom": "G", "ent": "La Serena", "cls": "Micronesia"}, {"id": 15853, "dom": "G", "ent": "La Serena", "cls": "Middle Africa"}, {"id": 15854, "dom": "G", "ent": "La Serena", "cls": "Northern Africa"}, {"id": 15855, "dom": "G", "ent": "La Serena", "cls": "Northern America"}, {"id": 15856, "dom": "G", "ent": "La Serena", "cls": "Northern Europe"}, {"id": 15857, "dom": "G", "ent": "La Serena", "cls": "Polynesia"}, {"id": 15858, "dom": "G", "ent": "La Serena", "cls": "South America"}, {"id": 15859, "dom": "G", "ent": "La Serena", "cls": "South-eastern Asia"}, {"id": 15860, "dom": "G", "ent": "La Serena", "cls": "Southern Africa"}, {"id": 15861, "dom": "G", "ent": "La Serena", "cls": "Southern Asia"}, {"id": 15862, "dom": "G", "ent": "La Serena", "cls": "Southern Europe"}, {"id": 15863, "dom": "G", "ent": "La Serena", "cls": "Western Africa"}, {"id": 15864, "dom": "G", "ent": "La Serena", "cls": "Western Asia"}, {"id": 15865, "dom": "G", "ent": "La Serena", "cls": "Western Europe"}, {"id": 15866, "dom": "G", "ent": "Renca", "cls": "Australia and New Zealand"}, {"id": 15867, "dom": "G", "ent": "Renca", "cls": "Caribbean"}, {"id": 15868, "dom": "G", "ent": "Renca", "cls": "Central America"}, {"id": 15869, "dom": "G", "ent": "Renca", "cls": "Central Asia"}, {"id": 15870, "dom": "G", "ent": "Renca", "cls": "Eastern Africa"}, {"id": 15871, "dom": "G", "ent": "Renca", "cls": "Eastern Asia"}, {"id": 15872, "dom": "G", "ent": "Renca", "cls": "Eastern Europe"}, {"id": 15873, "dom": "G", "ent": "Renca", "cls": "Melanesia"}, {"id": 15874, "dom": "G", "ent": "Renca", "cls": "Micronesia"}, {"id": 15875, "dom": "G", "ent": "Renca", "cls": "Middle Africa"}, {"id": 15876, "dom": "G", "ent": "Renca", "cls": "Northern Africa"}, {"id": 15877, "dom": "G", "ent": "Renca", "cls": "Northern America"}, {"id": 15878, "dom": "G", "ent": "Renca", "cls": "Northern Europe"}, {"id": 15879, "dom": "G", "ent": "Renca", "cls": "Polynesia"}, {"id": 15880, "dom": "G", "ent": "Renca", "cls": "South America"}, {"id": 15881, "dom": "G", "ent": "Renca", "cls": "South-eastern Asia"}, {"id": 15882, "dom": "G", "ent": "Renca", "cls": "Southern Africa"}, {"id": 15883, "dom": "G", "ent": "Renca", "cls": "Southern Asia"}, {"id": 15884, "dom": "G", "ent": "Renca", "cls": "Southern Europe"}, {"id": 15885, "dom": "G", "ent": "Renca", "cls": "Western Africa"}, {"id": 15886, "dom": "G", "ent": "Renca", "cls": "Western Asia"}, {"id": 15887, "dom": "G", "ent": "Renca", "cls": "Western Europe"}, {"id": 15888, "dom": "G", "ent": "Hacienda La Calera", "cls": "Australia and New Zealand"}, {"id": 15889, "dom": "G", "ent": "Hacienda La Calera", "cls": "Caribbean"}, {"id": 15890, "dom": "G", "ent": "Hacienda La Calera", "cls": "Central America"}, {"id": 15891, "dom": "G", "ent": "Hacienda La Calera", "cls": "Central Asia"}, {"id": 15892, "dom": "G", "ent": "Hacienda La Calera", "cls": "Eastern Africa"}, {"id": 15893, "dom": "G", "ent": "Hacienda La Calera", "cls": "Eastern Asia"}, {"id": 15894, "dom": "G", "ent": "Hacienda La Calera", "cls": "Eastern Europe"}, {"id": 15895, "dom": "G", "ent": "Hacienda La Calera", "cls": "Melanesia"}, {"id": 15896, "dom": "G", "ent": "Hacienda La Calera", "cls": "Micronesia"}, {"id": 15897, "dom": "G", "ent": "Hacienda La Calera", "cls": "Middle Africa"}, {"id": 15898, "dom": "G", "ent": "Hacienda La Calera", "cls": "Northern Africa"}, {"id": 15899, "dom": "G", "ent": "Hacienda La Calera", "cls": "Northern America"}, {"id": 15900, "dom": "G", "ent": "Hacienda La Calera", "cls": "Northern Europe"}, {"id": 15901, "dom": "G", "ent": "Hacienda La Calera", "cls": "Polynesia"}, {"id": 15902, "dom": "G", "ent": "Hacienda La Calera", "cls": "South America"}, {"id": 15903, "dom": "G", "ent": "Hacienda La Calera", "cls": "South-eastern Asia"}, {"id": 15904, "dom": "G", "ent": "Hacienda La Calera", "cls": "Southern Africa"}, {"id": 15905, "dom": "G", "ent": "Hacienda La Calera", "cls": "Southern Asia"}, {"id": 15906, "dom": "G", "ent": "Hacienda La Calera", "cls": "Southern Europe"}, {"id": 15907, "dom": "G", "ent": "Hacienda La Calera", "cls": "Western Africa"}, {"id": 15908, "dom": "G", "ent": "Hacienda La Calera", "cls": "Western Asia"}, {"id": 15909, "dom": "G", "ent": "Hacienda La Calera", "cls": "Western Europe"}, {"id": 15910, "dom": "G", "ent": "Angol", "cls": "Australia and New Zealand"}, {"id": 15911, "dom": "G", "ent": "Angol", "cls": "Caribbean"}, {"id": 15912, "dom": "G", "ent": "Angol", "cls": "Central America"}, {"id": 15913, "dom": "G", "ent": "Angol", "cls": "Central Asia"}, {"id": 15914, "dom": "G", "ent": "Angol", "cls": "Eastern Africa"}, {"id": 15915, "dom": "G", "ent": "Angol", "cls": "Eastern Asia"}, {"id": 15916, "dom": "G", "ent": "Angol", "cls": "Eastern Europe"}, {"id": 15917, "dom": "G", "ent": "Angol", "cls": "Melanesia"}, {"id": 15918, "dom": "G", "ent": "Angol", "cls": "Micronesia"}, {"id": 15919, "dom": "G", "ent": "Angol", "cls": "Middle Africa"}, {"id": 15920, "dom": "G", "ent": "Angol", "cls": "Northern Africa"}, {"id": 15921, "dom": "G", "ent": "Angol", "cls": "Northern America"}, {"id": 15922, "dom": "G", "ent": "Angol", "cls": "Northern Europe"}, {"id": 15923, "dom": "G", "ent": "Angol", "cls": "Polynesia"}, {"id": 15924, "dom": "G", "ent": "Angol", "cls": "South America"}, {"id": 15925, "dom": "G", "ent": "Angol", "cls": "South-eastern Asia"}, {"id": 15926, "dom": "G", "ent": "Angol", "cls": "Southern Africa"}, {"id": 15927, "dom": "G", "ent": "Angol", "cls": "Southern Asia"}, {"id": 15928, "dom": "G", "ent": "Angol", "cls": "Southern Europe"}, {"id": 15929, "dom": "G", "ent": "Angol", "cls": "Western Africa"}, {"id": 15930, "dom": "G", "ent": "Angol", "cls": "Western Asia"}, {"id": 15931, "dom": "G", "ent": "Angol", "cls": "Western Europe"}, {"id": 15932, "dom": "G", "ent": "Kribi", "cls": "Australia and New Zealand"}, {"id": 15933, "dom": "G", "ent": "Kribi", "cls": "Caribbean"}, {"id": 15934, "dom": "G", "ent": "Kribi", "cls": "Central America"}, {"id": 15935, "dom": "G", "ent": "Kribi", "cls": "Central Asia"}, {"id": 15936, "dom": "G", "ent": "Kribi", "cls": "Eastern Africa"}, {"id": 15937, "dom": "G", "ent": "Kribi", "cls": "Eastern Asia"}, {"id": 15938, "dom": "G", "ent": "Kribi", "cls": "Eastern Europe"}, {"id": 15939, "dom": "G", "ent": "Kribi", "cls": "Melanesia"}, {"id": 15940, "dom": "G", "ent": "Kribi", "cls": "Micronesia"}, {"id": 15941, "dom": "G", "ent": "Kribi", "cls": "Middle Africa"}, {"id": 15942, "dom": "G", "ent": "Kribi", "cls": "Northern Africa"}, {"id": 15943, "dom": "G", "ent": "Kribi", "cls": "Northern America"}, {"id": 15944, "dom": "G", "ent": "Kribi", "cls": "Northern Europe"}, {"id": 15945, "dom": "G", "ent": "Kribi", "cls": "Polynesia"}, {"id": 15946, "dom": "G", "ent": "Kribi", "cls": "South America"}, {"id": 15947, "dom": "G", "ent": "Kribi", "cls": "South-eastern Asia"}, {"id": 15948, "dom": "G", "ent": "Kribi", "cls": "Southern Africa"}, {"id": 15949, "dom": "G", "ent": "Kribi", "cls": "Southern Asia"}, {"id": 15950, "dom": "G", "ent": "Kribi", "cls": "Southern Europe"}, {"id": 15951, "dom": "G", "ent": "Kribi", "cls": "Western Africa"}, {"id": 15952, "dom": "G", "ent": "Kribi", "cls": "Western Asia"}, {"id": 15953, "dom": "G", "ent": "Kribi", "cls": "Western Europe"}, {"id": 15954, "dom": "G", "ent": "Tibati", "cls": "Australia and New Zealand"}, {"id": 15955, "dom": "G", "ent": "Tibati", "cls": "Caribbean"}, {"id": 15956, "dom": "G", "ent": "Tibati", "cls": "Central America"}, {"id": 15957, "dom": "G", "ent": "Tibati", "cls": "Central Asia"}, {"id": 15958, "dom": "G", "ent": "Tibati", "cls": "Eastern Africa"}, {"id": 15959, "dom": "G", "ent": "Tibati", "cls": "Eastern Asia"}, {"id": 15960, "dom": "G", "ent": "Tibati", "cls": "Eastern Europe"}, {"id": 15961, "dom": "G", "ent": "Tibati", "cls": "Melanesia"}, {"id": 15962, "dom": "G", "ent": "Tibati", "cls": "Micronesia"}, {"id": 15963, "dom": "G", "ent": "Tibati", "cls": "Middle Africa"}, {"id": 15964, "dom": "G", "ent": "Tibati", "cls": "Northern Africa"}, {"id": 15965, "dom": "G", "ent": "Tibati", "cls": "Northern America"}, {"id": 15966, "dom": "G", "ent": "Tibati", "cls": "Northern Europe"}, {"id": 15967, "dom": "G", "ent": "Tibati", "cls": "Polynesia"}, {"id": 15968, "dom": "G", "ent": "Tibati", "cls": "South America"}, {"id": 15969, "dom": "G", "ent": "Tibati", "cls": "South-eastern Asia"}, {"id": 15970, "dom": "G", "ent": "Tibati", "cls": "Southern Africa"}, {"id": 15971, "dom": "G", "ent": "Tibati", "cls": "Southern Asia"}, {"id": 15972, "dom": "G", "ent": "Tibati", "cls": "Southern Europe"}, {"id": 15973, "dom": "G", "ent": "Tibati", "cls": "Western Africa"}, {"id": 15974, "dom": "G", "ent": "Tibati", "cls": "Western Asia"}, {"id": 15975, "dom": "G", "ent": "Tibati", "cls": "Western Europe"}, {"id": 15976, "dom": "G", "ent": "Figuil", "cls": "Australia and New Zealand"}, {"id": 15977, "dom": "G", "ent": "Figuil", "cls": "Caribbean"}, {"id": 15978, "dom": "G", "ent": "Figuil", "cls": "Central America"}, {"id": 15979, "dom": "G", "ent": "Figuil", "cls": "Central Asia"}, {"id": 15980, "dom": "G", "ent": "Figuil", "cls": "Eastern Africa"}, {"id": 15981, "dom": "G", "ent": "Figuil", "cls": "Eastern Asia"}, {"id": 15982, "dom": "G", "ent": "Figuil", "cls": "Eastern Europe"}, {"id": 15983, "dom": "G", "ent": "Figuil", "cls": "Melanesia"}, {"id": 15984, "dom": "G", "ent": "Figuil", "cls": "Micronesia"}, {"id": 15985, "dom": "G", "ent": "Figuil", "cls": "Middle Africa"}, {"id": 15986, "dom": "G", "ent": "Figuil", "cls": "Northern Africa"}, {"id": 15987, "dom": "G", "ent": "Figuil", "cls": "Northern America"}, {"id": 15988, "dom": "G", "ent": "Figuil", "cls": "Northern Europe"}, {"id": 15989, "dom": "G", "ent": "Figuil", "cls": "Polynesia"}, {"id": 15990, "dom": "G", "ent": "Figuil", "cls": "South America"}, {"id": 15991, "dom": "G", "ent": "Figuil", "cls": "South-eastern Asia"}, {"id": 15992, "dom": "G", "ent": "Figuil", "cls": "Southern Africa"}, {"id": 15993, "dom": "G", "ent": "Figuil", "cls": "Southern Asia"}, {"id": 15994, "dom": "G", "ent": "Figuil", "cls": "Southern Europe"}, {"id": 15995, "dom": "G", "ent": "Figuil", "cls": "Western Africa"}, {"id": 15996, "dom": "G", "ent": "Figuil", "cls": "Western Asia"}, {"id": 15997, "dom": "G", "ent": "Figuil", "cls": "Western Europe"}, {"id": 15998, "dom": "G", "ent": "Makak", "cls": "Australia and New Zealand"}, {"id": 15999, "dom": "G", "ent": "Makak", "cls": "Caribbean"}, {"id": 16000, "dom": "G", "ent": "Makak", "cls": "Central America"}, {"id": 16001, "dom": "G", "ent": "Makak", "cls": "Central Asia"}, {"id": 16002, "dom": "G", "ent": "Makak", "cls": "Eastern Africa"}, {"id": 16003, "dom": "G", "ent": "Makak", "cls": "Eastern Asia"}, {"id": 16004, "dom": "G", "ent": "Makak", "cls": "Eastern Europe"}, {"id": 16005, "dom": "G", "ent": "Makak", "cls": "Melanesia"}, {"id": 16006, "dom": "G", "ent": "Makak", "cls": "Micronesia"}, {"id": 16007, "dom": "G", "ent": "Makak", "cls": "Middle Africa"}, {"id": 16008, "dom": "G", "ent": "Makak", "cls": "Northern Africa"}, {"id": 16009, "dom": "G", "ent": "Makak", "cls": "Northern America"}, {"id": 16010, "dom": "G", "ent": "Makak", "cls": "Northern Europe"}, {"id": 16011, "dom": "G", "ent": "Makak", "cls": "Polynesia"}, {"id": 16012, "dom": "G", "ent": "Makak", "cls": "South America"}, {"id": 16013, "dom": "G", "ent": "Makak", "cls": "South-eastern Asia"}, {"id": 16014, "dom": "G", "ent": "Makak", "cls": "Southern Africa"}, {"id": 16015, "dom": "G", "ent": "Makak", "cls": "Southern Asia"}, {"id": 16016, "dom": "G", "ent": "Makak", "cls": "Southern Europe"}, {"id": 16017, "dom": "G", "ent": "Makak", "cls": "Western Africa"}, {"id": 16018, "dom": "G", "ent": "Makak", "cls": "Western Asia"}, {"id": 16019, "dom": "G", "ent": "Makak", "cls": "Western Europe"}, {"id": 16020, "dom": "G", "ent": "Puqi", "cls": "Australia and New Zealand"}, {"id": 16021, "dom": "G", "ent": "Puqi", "cls": "Caribbean"}, {"id": 16022, "dom": "G", "ent": "Puqi", "cls": "Central America"}, {"id": 16023, "dom": "G", "ent": "Puqi", "cls": "Central Asia"}, {"id": 16024, "dom": "G", "ent": "Puqi", "cls": "Eastern Africa"}, {"id": 16025, "dom": "G", "ent": "Puqi", "cls": "Eastern Asia"}, {"id": 16026, "dom": "G", "ent": "Puqi", "cls": "Eastern Europe"}, {"id": 16027, "dom": "G", "ent": "Puqi", "cls": "Melanesia"}, {"id": 16028, "dom": "G", "ent": "Puqi", "cls": "Micronesia"}, {"id": 16029, "dom": "G", "ent": "Puqi", "cls": "Middle Africa"}, {"id": 16030, "dom": "G", "ent": "Puqi", "cls": "Northern Africa"}, {"id": 16031, "dom": "G", "ent": "Puqi", "cls": "Northern America"}, {"id": 16032, "dom": "G", "ent": "Puqi", "cls": "Northern Europe"}, {"id": 16033, "dom": "G", "ent": "Puqi", "cls": "Polynesia"}, {"id": 16034, "dom": "G", "ent": "Puqi", "cls": "South America"}, {"id": 16035, "dom": "G", "ent": "Puqi", "cls": "South-eastern Asia"}, {"id": 16036, "dom": "G", "ent": "Puqi", "cls": "Southern Africa"}, {"id": 16037, "dom": "G", "ent": "Puqi", "cls": "Southern Asia"}, {"id": 16038, "dom": "G", "ent": "Puqi", "cls": "Southern Europe"}, {"id": 16039, "dom": "G", "ent": "Puqi", "cls": "Western Africa"}, {"id": 16040, "dom": "G", "ent": "Puqi", "cls": "Western Asia"}, {"id": 16041, "dom": "G", "ent": "Puqi", "cls": "Western Europe"}, {"id": 16042, "dom": "G", "ent": "Qiuji", "cls": "Australia and New Zealand"}, {"id": 16043, "dom": "G", "ent": "Qiuji", "cls": "Caribbean"}, {"id": 16044, "dom": "G", "ent": "Qiuji", "cls": "Central America"}, {"id": 16045, "dom": "G", "ent": "Qiuji", "cls": "Central Asia"}, {"id": 16046, "dom": "G", "ent": "Qiuji", "cls": "Eastern Africa"}, {"id": 16047, "dom": "G", "ent": "Qiuji", "cls": "Eastern Asia"}, {"id": 16048, "dom": "G", "ent": "Qiuji", "cls": "Eastern Europe"}, {"id": 16049, "dom": "G", "ent": "Qiuji", "cls": "Melanesia"}, {"id": 16050, "dom": "G", "ent": "Qiuji", "cls": "Micronesia"}, {"id": 16051, "dom": "G", "ent": "Qiuji", "cls": "Middle Africa"}, {"id": 16052, "dom": "G", "ent": "Qiuji", "cls": "Northern Africa"}, {"id": 16053, "dom": "G", "ent": "Qiuji", "cls": "Northern America"}, {"id": 16054, "dom": "G", "ent": "Qiuji", "cls": "Northern Europe"}, {"id": 16055, "dom": "G", "ent": "Qiuji", "cls": "Polynesia"}, {"id": 16056, "dom": "G", "ent": "Qiuji", "cls": "South America"}, {"id": 16057, "dom": "G", "ent": "Qiuji", "cls": "South-eastern Asia"}, {"id": 16058, "dom": "G", "ent": "Qiuji", "cls": "Southern Africa"}, {"id": 16059, "dom": "G", "ent": "Qiuji", "cls": "Southern Asia"}, {"id": 16060, "dom": "G", "ent": "Qiuji", "cls": "Southern Europe"}, {"id": 16061, "dom": "G", "ent": "Qiuji", "cls": "Western Africa"}, {"id": 16062, "dom": "G", "ent": "Qiuji", "cls": "Western Asia"}, {"id": 16063, "dom": "G", "ent": "Qiuji", "cls": "Western Europe"}, {"id": 16064, "dom": "G", "ent": "Shiqiaozi", "cls": "Australia and New Zealand"}, {"id": 16065, "dom": "G", "ent": "Shiqiaozi", "cls": "Caribbean"}, {"id": 16066, "dom": "G", "ent": "Shiqiaozi", "cls": "Central America"}, {"id": 16067, "dom": "G", "ent": "Shiqiaozi", "cls": "Central Asia"}, {"id": 16068, "dom": "G", "ent": "Shiqiaozi", "cls": "Eastern Africa"}, {"id": 16069, "dom": "G", "ent": "Shiqiaozi", "cls": "Eastern Asia"}, {"id": 16070, "dom": "G", "ent": "Shiqiaozi", "cls": "Eastern Europe"}, {"id": 16071, "dom": "G", "ent": "Shiqiaozi", "cls": "Melanesia"}, {"id": 16072, "dom": "G", "ent": "Shiqiaozi", "cls": "Micronesia"}, {"id": 16073, "dom": "G", "ent": "Shiqiaozi", "cls": "Middle Africa"}, {"id": 16074, "dom": "G", "ent": "Shiqiaozi", "cls": "Northern Africa"}, {"id": 16075, "dom": "G", "ent": "Shiqiaozi", "cls": "Northern America"}, {"id": 16076, "dom": "G", "ent": "Shiqiaozi", "cls": "Northern Europe"}, {"id": 16077, "dom": "G", "ent": "Shiqiaozi", "cls": "Polynesia"}, {"id": 16078, "dom": "G", "ent": "Shiqiaozi", "cls": "South America"}, {"id": 16079, "dom": "G", "ent": "Shiqiaozi", "cls": "South-eastern Asia"}, {"id": 16080, "dom": "G", "ent": "Shiqiaozi", "cls": "Southern Africa"}, {"id": 16081, "dom": "G", "ent": "Shiqiaozi", "cls": "Southern Asia"}, {"id": 16082, "dom": "G", "ent": "Shiqiaozi", "cls": "Southern Europe"}, {"id": 16083, "dom": "G", "ent": "Shiqiaozi", "cls": "Western Africa"}, {"id": 16084, "dom": "G", "ent": "Shiqiaozi", "cls": "Western Asia"}, {"id": 16085, "dom": "G", "ent": "Shiqiaozi", "cls": "Western Europe"}, {"id": 16086, "dom": "G", "ent": "Chenlou", "cls": "Australia and New Zealand"}, {"id": 16087, "dom": "G", "ent": "Chenlou", "cls": "Caribbean"}, {"id": 16088, "dom": "G", "ent": "Chenlou", "cls": "Central America"}, {"id": 16089, "dom": "G", "ent": "Chenlou", "cls": "Central Asia"}, {"id": 16090, "dom": "G", "ent": "Chenlou", "cls": "Eastern Africa"}, {"id": 16091, "dom": "G", "ent": "Chenlou", "cls": "Eastern Asia"}, {"id": 16092, "dom": "G", "ent": "Chenlou", "cls": "Eastern Europe"}, {"id": 16093, "dom": "G", "ent": "Chenlou", "cls": "Melanesia"}, {"id": 16094, "dom": "G", "ent": "Chenlou", "cls": "Micronesia"}, {"id": 16095, "dom": "G", "ent": "Chenlou", "cls": "Middle Africa"}, {"id": 16096, "dom": "G", "ent": "Chenlou", "cls": "Northern Africa"}, {"id": 16097, "dom": "G", "ent": "Chenlou", "cls": "Northern America"}, {"id": 16098, "dom": "G", "ent": "Chenlou", "cls": "Northern Europe"}, {"id": 16099, "dom": "G", "ent": "Chenlou", "cls": "Polynesia"}, {"id": 16100, "dom": "G", "ent": "Chenlou", "cls": "South America"}, {"id": 16101, "dom": "G", "ent": "Chenlou", "cls": "South-eastern Asia"}, {"id": 16102, "dom": "G", "ent": "Chenlou", "cls": "Southern Africa"}, {"id": 16103, "dom": "G", "ent": "Chenlou", "cls": "Southern Asia"}, {"id": 16104, "dom": "G", "ent": "Chenlou", "cls": "Southern Europe"}, {"id": 16105, "dom": "G", "ent": "Chenlou", "cls": "Western Africa"}, {"id": 16106, "dom": "G", "ent": "Chenlou", "cls": "Western Asia"}, {"id": 16107, "dom": "G", "ent": "Chenlou", "cls": "Western Europe"}, {"id": 16108, "dom": "G", "ent": "Santa Marta", "cls": "Australia and New Zealand"}, {"id": 16109, "dom": "G", "ent": "Santa Marta", "cls": "Caribbean"}, {"id": 16110, "dom": "G", "ent": "Santa Marta", "cls": "Central America"}, {"id": 16111, "dom": "G", "ent": "Santa Marta", "cls": "Central Asia"}, {"id": 16112, "dom": "G", "ent": "Santa Marta", "cls": "Eastern Africa"}, {"id": 16113, "dom": "G", "ent": "Santa Marta", "cls": "Eastern Asia"}, {"id": 16114, "dom": "G", "ent": "Santa Marta", "cls": "Eastern Europe"}, {"id": 16115, "dom": "G", "ent": "Santa Marta", "cls": "Melanesia"}, {"id": 16116, "dom": "G", "ent": "Santa Marta", "cls": "Micronesia"}, {"id": 16117, "dom": "G", "ent": "Santa Marta", "cls": "Middle Africa"}, {"id": 16118, "dom": "G", "ent": "Santa Marta", "cls": "Northern Africa"}, {"id": 16119, "dom": "G", "ent": "Santa Marta", "cls": "Northern America"}, {"id": 16120, "dom": "G", "ent": "Santa Marta", "cls": "Northern Europe"}, {"id": 16121, "dom": "G", "ent": "Santa Marta", "cls": "Polynesia"}, {"id": 16122, "dom": "G", "ent": "Santa Marta", "cls": "South America"}, {"id": 16123, "dom": "G", "ent": "Santa Marta", "cls": "South-eastern Asia"}, {"id": 16124, "dom": "G", "ent": "Santa Marta", "cls": "Southern Africa"}, {"id": 16125, "dom": "G", "ent": "Santa Marta", "cls": "Southern Asia"}, {"id": 16126, "dom": "G", "ent": "Santa Marta", "cls": "Southern Europe"}, {"id": 16127, "dom": "G", "ent": "Santa Marta", "cls": "Western Africa"}, {"id": 16128, "dom": "G", "ent": "Santa Marta", "cls": "Western Asia"}, {"id": 16129, "dom": "G", "ent": "Santa Marta", "cls": "Western Europe"}, {"id": 16130, "dom": "G", "ent": "San Gil", "cls": "Australia and New Zealand"}, {"id": 16131, "dom": "G", "ent": "San Gil", "cls": "Caribbean"}, {"id": 16132, "dom": "G", "ent": "San Gil", "cls": "Central America"}, {"id": 16133, "dom": "G", "ent": "San Gil", "cls": "Central Asia"}, {"id": 16134, "dom": "G", "ent": "San Gil", "cls": "Eastern Africa"}, {"id": 16135, "dom": "G", "ent": "San Gil", "cls": "Eastern Asia"}, {"id": 16136, "dom": "G", "ent": "San Gil", "cls": "Eastern Europe"}, {"id": 16137, "dom": "G", "ent": "San Gil", "cls": "Melanesia"}, {"id": 16138, "dom": "G", "ent": "San Gil", "cls": "Micronesia"}, {"id": 16139, "dom": "G", "ent": "San Gil", "cls": "Middle Africa"}, {"id": 16140, "dom": "G", "ent": "San Gil", "cls": "Northern Africa"}, {"id": 16141, "dom": "G", "ent": "San Gil", "cls": "Northern America"}, {"id": 16142, "dom": "G", "ent": "San Gil", "cls": "Northern Europe"}, {"id": 16143, "dom": "G", "ent": "San Gil", "cls": "Polynesia"}, {"id": 16144, "dom": "G", "ent": "San Gil", "cls": "South America"}, {"id": 16145, "dom": "G", "ent": "San Gil", "cls": "South-eastern Asia"}, {"id": 16146, "dom": "G", "ent": "San Gil", "cls": "Southern Africa"}, {"id": 16147, "dom": "G", "ent": "San Gil", "cls": "Southern Asia"}, {"id": 16148, "dom": "G", "ent": "San Gil", "cls": "Southern Europe"}, {"id": 16149, "dom": "G", "ent": "San Gil", "cls": "Western Africa"}, {"id": 16150, "dom": "G", "ent": "San Gil", "cls": "Western Asia"}, {"id": 16151, "dom": "G", "ent": "San Gil", "cls": "Western Europe"}, {"id": 16152, "dom": "G", "ent": "Anaime", "cls": "Australia and New Zealand"}, {"id": 16153, "dom": "G", "ent": "Anaime", "cls": "Caribbean"}, {"id": 16154, "dom": "G", "ent": "Anaime", "cls": "Central America"}, {"id": 16155, "dom": "G", "ent": "Anaime", "cls": "Central Asia"}, {"id": 16156, "dom": "G", "ent": "Anaime", "cls": "Eastern Africa"}, {"id": 16157, "dom": "G", "ent": "Anaime", "cls": "Eastern Asia"}, {"id": 16158, "dom": "G", "ent": "Anaime", "cls": "Eastern Europe"}, {"id": 16159, "dom": "G", "ent": "Anaime", "cls": "Melanesia"}, {"id": 16160, "dom": "G", "ent": "Anaime", "cls": "Micronesia"}, {"id": 16161, "dom": "G", "ent": "Anaime", "cls": "Middle Africa"}, {"id": 16162, "dom": "G", "ent": "Anaime", "cls": "Northern Africa"}, {"id": 16163, "dom": "G", "ent": "Anaime", "cls": "Northern America"}, {"id": 16164, "dom": "G", "ent": "Anaime", "cls": "Northern Europe"}, {"id": 16165, "dom": "G", "ent": "Anaime", "cls": "Polynesia"}, {"id": 16166, "dom": "G", "ent": "Anaime", "cls": "South America"}, {"id": 16167, "dom": "G", "ent": "Anaime", "cls": "South-eastern Asia"}, {"id": 16168, "dom": "G", "ent": "Anaime", "cls": "Southern Africa"}, {"id": 16169, "dom": "G", "ent": "Anaime", "cls": "Southern Asia"}, {"id": 16170, "dom": "G", "ent": "Anaime", "cls": "Southern Europe"}, {"id": 16171, "dom": "G", "ent": "Anaime", "cls": "Western Africa"}, {"id": 16172, "dom": "G", "ent": "Anaime", "cls": "Western Asia"}, {"id": 16173, "dom": "G", "ent": "Anaime", "cls": "Western Europe"}, {"id": 16174, "dom": "G", "ent": "Cota", "cls": "Australia and New Zealand"}, {"id": 16175, "dom": "G", "ent": "Cota", "cls": "Caribbean"}, {"id": 16176, "dom": "G", "ent": "Cota", "cls": "Central America"}, {"id": 16177, "dom": "G", "ent": "Cota", "cls": "Central Asia"}, {"id": 16178, "dom": "G", "ent": "Cota", "cls": "Eastern Africa"}, {"id": 16179, "dom": "G", "ent": "Cota", "cls": "Eastern Asia"}, {"id": 16180, "dom": "G", "ent": "Cota", "cls": "Eastern Europe"}, {"id": 16181, "dom": "G", "ent": "Cota", "cls": "Melanesia"}, {"id": 16182, "dom": "G", "ent": "Cota", "cls": "Micronesia"}, {"id": 16183, "dom": "G", "ent": "Cota", "cls": "Middle Africa"}, {"id": 16184, "dom": "G", "ent": "Cota", "cls": "Northern Africa"}, {"id": 16185, "dom": "G", "ent": "Cota", "cls": "Northern America"}, {"id": 16186, "dom": "G", "ent": "Cota", "cls": "Northern Europe"}, {"id": 16187, "dom": "G", "ent": "Cota", "cls": "Polynesia"}, {"id": 16188, "dom": "G", "ent": "Cota", "cls": "South America"}, {"id": 16189, "dom": "G", "ent": "Cota", "cls": "South-eastern Asia"}, {"id": 16190, "dom": "G", "ent": "Cota", "cls": "Southern Africa"}, {"id": 16191, "dom": "G", "ent": "Cota", "cls": "Southern Asia"}, {"id": 16192, "dom": "G", "ent": "Cota", "cls": "Southern Europe"}, {"id": 16193, "dom": "G", "ent": "Cota", "cls": "Western Africa"}, {"id": 16194, "dom": "G", "ent": "Cota", "cls": "Western Asia"}, {"id": 16195, "dom": "G", "ent": "Cota", "cls": "Western Europe"}, {"id": 16196, "dom": "G", "ent": "San Vicente de Moravia", "cls": "Australia and New Zealand"}, {"id": 16197, "dom": "G", "ent": "San Vicente de Moravia", "cls": "Caribbean"}, {"id": 16198, "dom": "G", "ent": "San Vicente de Moravia", "cls": "Central America"}, {"id": 16199, "dom": "G", "ent": "San Vicente de Moravia", "cls": "Central Asia"}, {"id": 16200, "dom": "G", "ent": "San Vicente de Moravia", "cls": "Eastern Africa"}, {"id": 16201, "dom": "G", "ent": "San Vicente de Moravia", "cls": "Eastern Asia"}, {"id": 16202, "dom": "G", "ent": "San Vicente de Moravia", "cls": "Eastern Europe"}, {"id": 16203, "dom": "G", "ent": "San Vicente de Moravia", "cls": "Melanesia"}, {"id": 16204, "dom": "G", "ent": "San Vicente de Moravia", "cls": "Micronesia"}, {"id": 16205, "dom": "G", "ent": "San Vicente de Moravia", "cls": "Middle Africa"}, {"id": 16206, "dom": "G", "ent": "San Vicente de Moravia", "cls": "Northern Africa"}, {"id": 16207, "dom": "G", "ent": "San Vicente de Moravia", "cls": "Northern America"}, {"id": 16208, "dom": "G", "ent": "San Vicente de Moravia", "cls": "Northern Europe"}, {"id": 16209, "dom": "G", "ent": "San Vicente de Moravia", "cls": "Polynesia"}, {"id": 16210, "dom": "G", "ent": "San Vicente de Moravia", "cls": "South America"}, {"id": 16211, "dom": "G", "ent": "San Vicente de Moravia", "cls": "South-eastern Asia"}, {"id": 16212, "dom": "G", "ent": "San Vicente de Moravia", "cls": "Southern Africa"}, {"id": 16213, "dom": "G", "ent": "San Vicente de Moravia", "cls": "Southern Asia"}, {"id": 16214, "dom": "G", "ent": "San Vicente de Moravia", "cls": "Southern Europe"}, {"id": 16215, "dom": "G", "ent": "San Vicente de Moravia", "cls": "Western Africa"}, {"id": 16216, "dom": "G", "ent": "San Vicente de Moravia", "cls": "Western Asia"}, {"id": 16217, "dom": "G", "ent": "San Vicente de Moravia", "cls": "Western Europe"}, {"id": 16218, "dom": "G", "ent": "Quesada", "cls": "Australia and New Zealand"}, {"id": 16219, "dom": "G", "ent": "Quesada", "cls": "Caribbean"}, {"id": 16220, "dom": "G", "ent": "Quesada", "cls": "Central America"}, {"id": 16221, "dom": "G", "ent": "Quesada", "cls": "Central Asia"}, {"id": 16222, "dom": "G", "ent": "Quesada", "cls": "Eastern Africa"}, {"id": 16223, "dom": "G", "ent": "Quesada", "cls": "Eastern Asia"}, {"id": 16224, "dom": "G", "ent": "Quesada", "cls": "Eastern Europe"}, {"id": 16225, "dom": "G", "ent": "Quesada", "cls": "Melanesia"}, {"id": 16226, "dom": "G", "ent": "Quesada", "cls": "Micronesia"}, {"id": 16227, "dom": "G", "ent": "Quesada", "cls": "Middle Africa"}, {"id": 16228, "dom": "G", "ent": "Quesada", "cls": "Northern Africa"}, {"id": 16229, "dom": "G", "ent": "Quesada", "cls": "Northern America"}, {"id": 16230, "dom": "G", "ent": "Quesada", "cls": "Northern Europe"}, {"id": 16231, "dom": "G", "ent": "Quesada", "cls": "Polynesia"}, {"id": 16232, "dom": "G", "ent": "Quesada", "cls": "South America"}, {"id": 16233, "dom": "G", "ent": "Quesada", "cls": "South-eastern Asia"}, {"id": 16234, "dom": "G", "ent": "Quesada", "cls": "Southern Africa"}, {"id": 16235, "dom": "G", "ent": "Quesada", "cls": "Southern Asia"}, {"id": 16236, "dom": "G", "ent": "Quesada", "cls": "Southern Europe"}, {"id": 16237, "dom": "G", "ent": "Quesada", "cls": "Western Africa"}, {"id": 16238, "dom": "G", "ent": "Quesada", "cls": "Western Asia"}, {"id": 16239, "dom": "G", "ent": "Quesada", "cls": "Western Europe"}, {"id": 16240, "dom": "G", "ent": "Esparza", "cls": "Australia and New Zealand"}, {"id": 16241, "dom": "G", "ent": "Esparza", "cls": "Caribbean"}, {"id": 16242, "dom": "G", "ent": "Esparza", "cls": "Central America"}, {"id": 16243, "dom": "G", "ent": "Esparza", "cls": "Central Asia"}, {"id": 16244, "dom": "G", "ent": "Esparza", "cls": "Eastern Africa"}, {"id": 16245, "dom": "G", "ent": "Esparza", "cls": "Eastern Asia"}, {"id": 16246, "dom": "G", "ent": "Esparza", "cls": "Eastern Europe"}, {"id": 16247, "dom": "G", "ent": "Esparza", "cls": "Melanesia"}, {"id": 16248, "dom": "G", "ent": "Esparza", "cls": "Micronesia"}, {"id": 16249, "dom": "G", "ent": "Esparza", "cls": "Middle Africa"}, {"id": 16250, "dom": "G", "ent": "Esparza", "cls": "Northern Africa"}, {"id": 16251, "dom": "G", "ent": "Esparza", "cls": "Northern America"}, {"id": 16252, "dom": "G", "ent": "Esparza", "cls": "Northern Europe"}, {"id": 16253, "dom": "G", "ent": "Esparza", "cls": "Polynesia"}, {"id": 16254, "dom": "G", "ent": "Esparza", "cls": "South America"}, {"id": 16255, "dom": "G", "ent": "Esparza", "cls": "South-eastern Asia"}, {"id": 16256, "dom": "G", "ent": "Esparza", "cls": "Southern Africa"}, {"id": 16257, "dom": "G", "ent": "Esparza", "cls": "Southern Asia"}, {"id": 16258, "dom": "G", "ent": "Esparza", "cls": "Southern Europe"}, {"id": 16259, "dom": "G", "ent": "Esparza", "cls": "Western Africa"}, {"id": 16260, "dom": "G", "ent": "Esparza", "cls": "Western Asia"}, {"id": 16261, "dom": "G", "ent": "Esparza", "cls": "Western Europe"}, {"id": 16262, "dom": "G", "ent": "San Juan de Dios", "cls": "Australia and New Zealand"}, {"id": 16263, "dom": "G", "ent": "San Juan de Dios", "cls": "Caribbean"}, {"id": 16264, "dom": "G", "ent": "San Juan de Dios", "cls": "Central America"}, {"id": 16265, "dom": "G", "ent": "San Juan de Dios", "cls": "Central Asia"}, {"id": 16266, "dom": "G", "ent": "San Juan de Dios", "cls": "Eastern Africa"}, {"id": 16267, "dom": "G", "ent": "San Juan de Dios", "cls": "Eastern Asia"}, {"id": 16268, "dom": "G", "ent": "San Juan de Dios", "cls": "Eastern Europe"}, {"id": 16269, "dom": "G", "ent": "San Juan de Dios", "cls": "Melanesia"}, {"id": 16270, "dom": "G", "ent": "San Juan de Dios", "cls": "Micronesia"}, {"id": 16271, "dom": "G", "ent": "San Juan de Dios", "cls": "Middle Africa"}, {"id": 16272, "dom": "G", "ent": "San Juan de Dios", "cls": "Northern Africa"}, {"id": 16273, "dom": "G", "ent": "San Juan de Dios", "cls": "Northern America"}, {"id": 16274, "dom": "G", "ent": "San Juan de Dios", "cls": "Northern Europe"}, {"id": 16275, "dom": "G", "ent": "San Juan de Dios", "cls": "Polynesia"}, {"id": 16276, "dom": "G", "ent": "San Juan de Dios", "cls": "South America"}, {"id": 16277, "dom": "G", "ent": "San Juan de Dios", "cls": "South-eastern Asia"}, {"id": 16278, "dom": "G", "ent": "San Juan de Dios", "cls": "Southern Africa"}, {"id": 16279, "dom": "G", "ent": "San Juan de Dios", "cls": "Southern Asia"}, {"id": 16280, "dom": "G", "ent": "San Juan de Dios", "cls": "Southern Europe"}, {"id": 16281, "dom": "G", "ent": "San Juan de Dios", "cls": "Western Africa"}, {"id": 16282, "dom": "G", "ent": "San Juan de Dios", "cls": "Western Asia"}, {"id": 16283, "dom": "G", "ent": "San Juan de Dios", "cls": "Western Europe"}, {"id": 16284, "dom": "G", "ent": "Moa", "cls": "Australia and New Zealand"}, {"id": 16285, "dom": "G", "ent": "Moa", "cls": "Caribbean"}, {"id": 16286, "dom": "G", "ent": "Moa", "cls": "Central America"}, {"id": 16287, "dom": "G", "ent": "Moa", "cls": "Central Asia"}, {"id": 16288, "dom": "G", "ent": "Moa", "cls": "Eastern Africa"}, {"id": 16289, "dom": "G", "ent": "Moa", "cls": "Eastern Asia"}, {"id": 16290, "dom": "G", "ent": "Moa", "cls": "Eastern Europe"}, {"id": 16291, "dom": "G", "ent": "Moa", "cls": "Melanesia"}, {"id": 16292, "dom": "G", "ent": "Moa", "cls": "Micronesia"}, {"id": 16293, "dom": "G", "ent": "Moa", "cls": "Middle Africa"}, {"id": 16294, "dom": "G", "ent": "Moa", "cls": "Northern Africa"}, {"id": 16295, "dom": "G", "ent": "Moa", "cls": "Northern America"}, {"id": 16296, "dom": "G", "ent": "Moa", "cls": "Northern Europe"}, {"id": 16297, "dom": "G", "ent": "Moa", "cls": "Polynesia"}, {"id": 16298, "dom": "G", "ent": "Moa", "cls": "South America"}, {"id": 16299, "dom": "G", "ent": "Moa", "cls": "South-eastern Asia"}, {"id": 16300, "dom": "G", "ent": "Moa", "cls": "Southern Africa"}, {"id": 16301, "dom": "G", "ent": "Moa", "cls": "Southern Asia"}, {"id": 16302, "dom": "G", "ent": "Moa", "cls": "Southern Europe"}, {"id": 16303, "dom": "G", "ent": "Moa", "cls": "Western Africa"}, {"id": 16304, "dom": "G", "ent": "Moa", "cls": "Western Asia"}, {"id": 16305, "dom": "G", "ent": "Moa", "cls": "Western Europe"}, {"id": 16306, "dom": "G", "ent": "Placetas", "cls": "Australia and New Zealand"}, {"id": 16307, "dom": "G", "ent": "Placetas", "cls": "Caribbean"}, {"id": 16308, "dom": "G", "ent": "Placetas", "cls": "Central America"}, {"id": 16309, "dom": "G", "ent": "Placetas", "cls": "Central Asia"}, {"id": 16310, "dom": "G", "ent": "Placetas", "cls": "Eastern Africa"}, {"id": 16311, "dom": "G", "ent": "Placetas", "cls": "Eastern Asia"}, {"id": 16312, "dom": "G", "ent": "Placetas", "cls": "Eastern Europe"}, {"id": 16313, "dom": "G", "ent": "Placetas", "cls": "Melanesia"}, {"id": 16314, "dom": "G", "ent": "Placetas", "cls": "Micronesia"}, {"id": 16315, "dom": "G", "ent": "Placetas", "cls": "Middle Africa"}, {"id": 16316, "dom": "G", "ent": "Placetas", "cls": "Northern Africa"}, {"id": 16317, "dom": "G", "ent": "Placetas", "cls": "Northern America"}, {"id": 16318, "dom": "G", "ent": "Placetas", "cls": "Northern Europe"}, {"id": 16319, "dom": "G", "ent": "Placetas", "cls": "Polynesia"}, {"id": 16320, "dom": "G", "ent": "Placetas", "cls": "South America"}, {"id": 16321, "dom": "G", "ent": "Placetas", "cls": "South-eastern Asia"}, {"id": 16322, "dom": "G", "ent": "Placetas", "cls": "Southern Africa"}, {"id": 16323, "dom": "G", "ent": "Placetas", "cls": "Southern Asia"}, {"id": 16324, "dom": "G", "ent": "Placetas", "cls": "Southern Europe"}, {"id": 16325, "dom": "G", "ent": "Placetas", "cls": "Western Africa"}, {"id": 16326, "dom": "G", "ent": "Placetas", "cls": "Western Asia"}, {"id": 16327, "dom": "G", "ent": "Placetas", "cls": "Western Europe"}, {"id": 16328, "dom": "G", "ent": "Calimete", "cls": "Australia and New Zealand"}, {"id": 16329, "dom": "G", "ent": "Calimete", "cls": "Caribbean"}, {"id": 16330, "dom": "G", "ent": "Calimete", "cls": "Central America"}, {"id": 16331, "dom": "G", "ent": "Calimete", "cls": "Central Asia"}, {"id": 16332, "dom": "G", "ent": "Calimete", "cls": "Eastern Africa"}, {"id": 16333, "dom": "G", "ent": "Calimete", "cls": "Eastern Asia"}, {"id": 16334, "dom": "G", "ent": "Calimete", "cls": "Eastern Europe"}, {"id": 16335, "dom": "G", "ent": "Calimete", "cls": "Melanesia"}, {"id": 16336, "dom": "G", "ent": "Calimete", "cls": "Micronesia"}, {"id": 16337, "dom": "G", "ent": "Calimete", "cls": "Middle Africa"}, {"id": 16338, "dom": "G", "ent": "Calimete", "cls": "Northern Africa"}, {"id": 16339, "dom": "G", "ent": "Calimete", "cls": "Northern America"}, {"id": 16340, "dom": "G", "ent": "Calimete", "cls": "Northern Europe"}, {"id": 16341, "dom": "G", "ent": "Calimete", "cls": "Polynesia"}, {"id": 16342, "dom": "G", "ent": "Calimete", "cls": "South America"}, {"id": 16343, "dom": "G", "ent": "Calimete", "cls": "South-eastern Asia"}, {"id": 16344, "dom": "G", "ent": "Calimete", "cls": "Southern Africa"}, {"id": 16345, "dom": "G", "ent": "Calimete", "cls": "Southern Asia"}, {"id": 16346, "dom": "G", "ent": "Calimete", "cls": "Southern Europe"}, {"id": 16347, "dom": "G", "ent": "Calimete", "cls": "Western Africa"}, {"id": 16348, "dom": "G", "ent": "Calimete", "cls": "Western Asia"}, {"id": 16349, "dom": "G", "ent": "Calimete", "cls": "Western Europe"}, {"id": 16350, "dom": "G", "ent": "La Sierpe", "cls": "Australia and New Zealand"}, {"id": 16351, "dom": "G", "ent": "La Sierpe", "cls": "Caribbean"}, {"id": 16352, "dom": "G", "ent": "La Sierpe", "cls": "Central America"}, {"id": 16353, "dom": "G", "ent": "La Sierpe", "cls": "Central Asia"}, {"id": 16354, "dom": "G", "ent": "La Sierpe", "cls": "Eastern Africa"}, {"id": 16355, "dom": "G", "ent": "La Sierpe", "cls": "Eastern Asia"}, {"id": 16356, "dom": "G", "ent": "La Sierpe", "cls": "Eastern Europe"}, {"id": 16357, "dom": "G", "ent": "La Sierpe", "cls": "Melanesia"}, {"id": 16358, "dom": "G", "ent": "La Sierpe", "cls": "Micronesia"}, {"id": 16359, "dom": "G", "ent": "La Sierpe", "cls": "Middle Africa"}, {"id": 16360, "dom": "G", "ent": "La Sierpe", "cls": "Northern Africa"}, {"id": 16361, "dom": "G", "ent": "La Sierpe", "cls": "Northern America"}, {"id": 16362, "dom": "G", "ent": "La Sierpe", "cls": "Northern Europe"}, {"id": 16363, "dom": "G", "ent": "La Sierpe", "cls": "Polynesia"}, {"id": 16364, "dom": "G", "ent": "La Sierpe", "cls": "South America"}, {"id": 16365, "dom": "G", "ent": "La Sierpe", "cls": "South-eastern Asia"}, {"id": 16366, "dom": "G", "ent": "La Sierpe", "cls": "Southern Africa"}, {"id": 16367, "dom": "G", "ent": "La Sierpe", "cls": "Southern Asia"}, {"id": 16368, "dom": "G", "ent": "La Sierpe", "cls": "Southern Europe"}, {"id": 16369, "dom": "G", "ent": "La Sierpe", "cls": "Western Africa"}, {"id": 16370, "dom": "G", "ent": "La Sierpe", "cls": "Western Asia"}, {"id": 16371, "dom": "G", "ent": "La Sierpe", "cls": "Western Europe"}, {"id": 16372, "dom": "G", "ent": "Pilsen", "cls": "Australia and New Zealand"}, {"id": 16373, "dom": "G", "ent": "Pilsen", "cls": "Caribbean"}, {"id": 16374, "dom": "G", "ent": "Pilsen", "cls": "Central America"}, {"id": 16375, "dom": "G", "ent": "Pilsen", "cls": "Central Asia"}, {"id": 16376, "dom": "G", "ent": "Pilsen", "cls": "Eastern Africa"}, {"id": 16377, "dom": "G", "ent": "Pilsen", "cls": "Eastern Asia"}, {"id": 16378, "dom": "G", "ent": "Pilsen", "cls": "Eastern Europe"}, {"id": 16379, "dom": "G", "ent": "Pilsen", "cls": "Melanesia"}, {"id": 16380, "dom": "G", "ent": "Pilsen", "cls": "Micronesia"}, {"id": 16381, "dom": "G", "ent": "Pilsen", "cls": "Middle Africa"}, {"id": 16382, "dom": "G", "ent": "Pilsen", "cls": "Northern Africa"}, {"id": 16383, "dom": "G", "ent": "Pilsen", "cls": "Northern America"}, {"id": 16384, "dom": "G", "ent": "Pilsen", "cls": "Northern Europe"}, {"id": 16385, "dom": "G", "ent": "Pilsen", "cls": "Polynesia"}, {"id": 16386, "dom": "G", "ent": "Pilsen", "cls": "South America"}, {"id": 16387, "dom": "G", "ent": "Pilsen", "cls": "South-eastern Asia"}, {"id": 16388, "dom": "G", "ent": "Pilsen", "cls": "Southern Africa"}, {"id": 16389, "dom": "G", "ent": "Pilsen", "cls": "Southern Asia"}, {"id": 16390, "dom": "G", "ent": "Pilsen", "cls": "Southern Europe"}, {"id": 16391, "dom": "G", "ent": "Pilsen", "cls": "Western Africa"}, {"id": 16392, "dom": "G", "ent": "Pilsen", "cls": "Western Asia"}, {"id": 16393, "dom": "G", "ent": "Pilsen", "cls": "Western Europe"}, {"id": 16394, "dom": "G", "ent": "Opava", "cls": "Australia and New Zealand"}, {"id": 16395, "dom": "G", "ent": "Opava", "cls": "Caribbean"}, {"id": 16396, "dom": "G", "ent": "Opava", "cls": "Central America"}, {"id": 16397, "dom": "G", "ent": "Opava", "cls": "Central Asia"}, {"id": 16398, "dom": "G", "ent": "Opava", "cls": "Eastern Africa"}, {"id": 16399, "dom": "G", "ent": "Opava", "cls": "Eastern Asia"}, {"id": 16400, "dom": "G", "ent": "Opava", "cls": "Eastern Europe"}, {"id": 16401, "dom": "G", "ent": "Opava", "cls": "Melanesia"}, {"id": 16402, "dom": "G", "ent": "Opava", "cls": "Micronesia"}, {"id": 16403, "dom": "G", "ent": "Opava", "cls": "Middle Africa"}, {"id": 16404, "dom": "G", "ent": "Opava", "cls": "Northern Africa"}, {"id": 16405, "dom": "G", "ent": "Opava", "cls": "Northern America"}, {"id": 16406, "dom": "G", "ent": "Opava", "cls": "Northern Europe"}, {"id": 16407, "dom": "G", "ent": "Opava", "cls": "Polynesia"}, {"id": 16408, "dom": "G", "ent": "Opava", "cls": "South America"}, {"id": 16409, "dom": "G", "ent": "Opava", "cls": "South-eastern Asia"}, {"id": 16410, "dom": "G", "ent": "Opava", "cls": "Southern Africa"}, {"id": 16411, "dom": "G", "ent": "Opava", "cls": "Southern Asia"}, {"id": 16412, "dom": "G", "ent": "Opava", "cls": "Southern Europe"}, {"id": 16413, "dom": "G", "ent": "Opava", "cls": "Western Africa"}, {"id": 16414, "dom": "G", "ent": "Opava", "cls": "Western Asia"}, {"id": 16415, "dom": "G", "ent": "Opava", "cls": "Western Europe"}, {"id": 16416, "dom": "G", "ent": "Michle", "cls": "Australia and New Zealand"}, {"id": 16417, "dom": "G", "ent": "Michle", "cls": "Caribbean"}, {"id": 16418, "dom": "G", "ent": "Michle", "cls": "Central America"}, {"id": 16419, "dom": "G", "ent": "Michle", "cls": "Central Asia"}, {"id": 16420, "dom": "G", "ent": "Michle", "cls": "Eastern Africa"}, {"id": 16421, "dom": "G", "ent": "Michle", "cls": "Eastern Asia"}, {"id": 16422, "dom": "G", "ent": "Michle", "cls": "Eastern Europe"}, {"id": 16423, "dom": "G", "ent": "Michle", "cls": "Melanesia"}, {"id": 16424, "dom": "G", "ent": "Michle", "cls": "Micronesia"}, {"id": 16425, "dom": "G", "ent": "Michle", "cls": "Middle Africa"}, {"id": 16426, "dom": "G", "ent": "Michle", "cls": "Northern Africa"}, {"id": 16427, "dom": "G", "ent": "Michle", "cls": "Northern America"}, {"id": 16428, "dom": "G", "ent": "Michle", "cls": "Northern Europe"}, {"id": 16429, "dom": "G", "ent": "Michle", "cls": "Polynesia"}, {"id": 16430, "dom": "G", "ent": "Michle", "cls": "South America"}, {"id": 16431, "dom": "G", "ent": "Michle", "cls": "South-eastern Asia"}, {"id": 16432, "dom": "G", "ent": "Michle", "cls": "Southern Africa"}, {"id": 16433, "dom": "G", "ent": "Michle", "cls": "Southern Asia"}, {"id": 16434, "dom": "G", "ent": "Michle", "cls": "Southern Europe"}, {"id": 16435, "dom": "G", "ent": "Michle", "cls": "Western Africa"}, {"id": 16436, "dom": "G", "ent": "Michle", "cls": "Western Asia"}, {"id": 16437, "dom": "G", "ent": "Michle", "cls": "Western Europe"}, {"id": 16438, "dom": "G", "ent": "Bohnice", "cls": "Australia and New Zealand"}, {"id": 16439, "dom": "G", "ent": "Bohnice", "cls": "Caribbean"}, {"id": 16440, "dom": "G", "ent": "Bohnice", "cls": "Central America"}, {"id": 16441, "dom": "G", "ent": "Bohnice", "cls": "Central Asia"}, {"id": 16442, "dom": "G", "ent": "Bohnice", "cls": "Eastern Africa"}, {"id": 16443, "dom": "G", "ent": "Bohnice", "cls": "Eastern Asia"}, {"id": 16444, "dom": "G", "ent": "Bohnice", "cls": "Eastern Europe"}, {"id": 16445, "dom": "G", "ent": "Bohnice", "cls": "Melanesia"}, {"id": 16446, "dom": "G", "ent": "Bohnice", "cls": "Micronesia"}, {"id": 16447, "dom": "G", "ent": "Bohnice", "cls": "Middle Africa"}, {"id": 16448, "dom": "G", "ent": "Bohnice", "cls": "Northern Africa"}, {"id": 16449, "dom": "G", "ent": "Bohnice", "cls": "Northern America"}, {"id": 16450, "dom": "G", "ent": "Bohnice", "cls": "Northern Europe"}, {"id": 16451, "dom": "G", "ent": "Bohnice", "cls": "Polynesia"}, {"id": 16452, "dom": "G", "ent": "Bohnice", "cls": "South America"}, {"id": 16453, "dom": "G", "ent": "Bohnice", "cls": "South-eastern Asia"}, {"id": 16454, "dom": "G", "ent": "Bohnice", "cls": "Southern Africa"}, {"id": 16455, "dom": "G", "ent": "Bohnice", "cls": "Southern Asia"}, {"id": 16456, "dom": "G", "ent": "Bohnice", "cls": "Southern Europe"}, {"id": 16457, "dom": "G", "ent": "Bohnice", "cls": "Western Africa"}, {"id": 16458, "dom": "G", "ent": "Bohnice", "cls": "Western Asia"}, {"id": 16459, "dom": "G", "ent": "Bohnice", "cls": "Western Europe"}, {"id": 16460, "dom": "G", "ent": "Marzahn", "cls": "Australia and New Zealand"}, {"id": 16461, "dom": "G", "ent": "Marzahn", "cls": "Caribbean"}, {"id": 16462, "dom": "G", "ent": "Marzahn", "cls": "Central America"}, {"id": 16463, "dom": "G", "ent": "Marzahn", "cls": "Central Asia"}, {"id": 16464, "dom": "G", "ent": "Marzahn", "cls": "Eastern Africa"}, {"id": 16465, "dom": "G", "ent": "Marzahn", "cls": "Eastern Asia"}, {"id": 16466, "dom": "G", "ent": "Marzahn", "cls": "Eastern Europe"}, {"id": 16467, "dom": "G", "ent": "Marzahn", "cls": "Melanesia"}, {"id": 16468, "dom": "G", "ent": "Marzahn", "cls": "Micronesia"}, {"id": 16469, "dom": "G", "ent": "Marzahn", "cls": "Middle Africa"}, {"id": 16470, "dom": "G", "ent": "Marzahn", "cls": "Northern Africa"}, {"id": 16471, "dom": "G", "ent": "Marzahn", "cls": "Northern America"}, {"id": 16472, "dom": "G", "ent": "Marzahn", "cls": "Northern Europe"}, {"id": 16473, "dom": "G", "ent": "Marzahn", "cls": "Polynesia"}, {"id": 16474, "dom": "G", "ent": "Marzahn", "cls": "South America"}, {"id": 16475, "dom": "G", "ent": "Marzahn", "cls": "South-eastern Asia"}, {"id": 16476, "dom": "G", "ent": "Marzahn", "cls": "Southern Africa"}, {"id": 16477, "dom": "G", "ent": "Marzahn", "cls": "Southern Asia"}, {"id": 16478, "dom": "G", "ent": "Marzahn", "cls": "Southern Europe"}, {"id": 16479, "dom": "G", "ent": "Marzahn", "cls": "Western Africa"}, {"id": 16480, "dom": "G", "ent": "Marzahn", "cls": "Western Asia"}, {"id": 16481, "dom": "G", "ent": "Marzahn", "cls": "Western Europe"}, {"id": 16482, "dom": "G", "ent": "Bad Kreuznach", "cls": "Australia and New Zealand"}, {"id": 16483, "dom": "G", "ent": "Bad Kreuznach", "cls": "Caribbean"}, {"id": 16484, "dom": "G", "ent": "Bad Kreuznach", "cls": "Central America"}, {"id": 16485, "dom": "G", "ent": "Bad Kreuznach", "cls": "Central Asia"}, {"id": 16486, "dom": "G", "ent": "Bad Kreuznach", "cls": "Eastern Africa"}, {"id": 16487, "dom": "G", "ent": "Bad Kreuznach", "cls": "Eastern Asia"}, {"id": 16488, "dom": "G", "ent": "Bad Kreuznach", "cls": "Eastern Europe"}, {"id": 16489, "dom": "G", "ent": "Bad Kreuznach", "cls": "Melanesia"}, {"id": 16490, "dom": "G", "ent": "Bad Kreuznach", "cls": "Micronesia"}, {"id": 16491, "dom": "G", "ent": "Bad Kreuznach", "cls": "Middle Africa"}, {"id": 16492, "dom": "G", "ent": "Bad Kreuznach", "cls": "Northern Africa"}, {"id": 16493, "dom": "G", "ent": "Bad Kreuznach", "cls": "Northern America"}, {"id": 16494, "dom": "G", "ent": "Bad Kreuznach", "cls": "Northern Europe"}, {"id": 16495, "dom": "G", "ent": "Bad Kreuznach", "cls": "Polynesia"}, {"id": 16496, "dom": "G", "ent": "Bad Kreuznach", "cls": "South America"}, {"id": 16497, "dom": "G", "ent": "Bad Kreuznach", "cls": "South-eastern Asia"}, {"id": 16498, "dom": "G", "ent": "Bad Kreuznach", "cls": "Southern Africa"}, {"id": 16499, "dom": "G", "ent": "Bad Kreuznach", "cls": "Southern Asia"}, {"id": 16500, "dom": "G", "ent": "Bad Kreuznach", "cls": "Southern Europe"}, {"id": 16501, "dom": "G", "ent": "Bad Kreuznach", "cls": "Western Africa"}, {"id": 16502, "dom": "G", "ent": "Bad Kreuznach", "cls": "Western Asia"}, {"id": 16503, "dom": "G", "ent": "Bad Kreuznach", "cls": "Western Europe"}, {"id": 16504, "dom": "G", "ent": "Hille", "cls": "Australia and New Zealand"}, {"id": 16505, "dom": "G", "ent": "Hille", "cls": "Caribbean"}, {"id": 16506, "dom": "G", "ent": "Hille", "cls": "Central America"}, {"id": 16507, "dom": "G", "ent": "Hille", "cls": "Central Asia"}, {"id": 16508, "dom": "G", "ent": "Hille", "cls": "Eastern Africa"}, {"id": 16509, "dom": "G", "ent": "Hille", "cls": "Eastern Asia"}, {"id": 16510, "dom": "G", "ent": "Hille", "cls": "Eastern Europe"}, {"id": 16511, "dom": "G", "ent": "Hille", "cls": "Melanesia"}, {"id": 16512, "dom": "G", "ent": "Hille", "cls": "Micronesia"}, {"id": 16513, "dom": "G", "ent": "Hille", "cls": "Middle Africa"}, {"id": 16514, "dom": "G", "ent": "Hille", "cls": "Northern Africa"}, {"id": 16515, "dom": "G", "ent": "Hille", "cls": "Northern America"}, {"id": 16516, "dom": "G", "ent": "Hille", "cls": "Northern Europe"}, {"id": 16517, "dom": "G", "ent": "Hille", "cls": "Polynesia"}, {"id": 16518, "dom": "G", "ent": "Hille", "cls": "South America"}, {"id": 16519, "dom": "G", "ent": "Hille", "cls": "South-eastern Asia"}, {"id": 16520, "dom": "G", "ent": "Hille", "cls": "Southern Africa"}, {"id": 16521, "dom": "G", "ent": "Hille", "cls": "Southern Asia"}, {"id": 16522, "dom": "G", "ent": "Hille", "cls": "Southern Europe"}, {"id": 16523, "dom": "G", "ent": "Hille", "cls": "Western Africa"}, {"id": 16524, "dom": "G", "ent": "Hille", "cls": "Western Asia"}, {"id": 16525, "dom": "G", "ent": "Hille", "cls": "Western Europe"}, {"id": 16526, "dom": "G", "ent": "Vechelde", "cls": "Australia and New Zealand"}, {"id": 16527, "dom": "G", "ent": "Vechelde", "cls": "Caribbean"}, {"id": 16528, "dom": "G", "ent": "Vechelde", "cls": "Central America"}, {"id": 16529, "dom": "G", "ent": "Vechelde", "cls": "Central Asia"}, {"id": 16530, "dom": "G", "ent": "Vechelde", "cls": "Eastern Africa"}, {"id": 16531, "dom": "G", "ent": "Vechelde", "cls": "Eastern Asia"}, {"id": 16532, "dom": "G", "ent": "Vechelde", "cls": "Eastern Europe"}, {"id": 16533, "dom": "G", "ent": "Vechelde", "cls": "Melanesia"}, {"id": 16534, "dom": "G", "ent": "Vechelde", "cls": "Micronesia"}, {"id": 16535, "dom": "G", "ent": "Vechelde", "cls": "Middle Africa"}, {"id": 16536, "dom": "G", "ent": "Vechelde", "cls": "Northern Africa"}, {"id": 16537, "dom": "G", "ent": "Vechelde", "cls": "Northern America"}, {"id": 16538, "dom": "G", "ent": "Vechelde", "cls": "Northern Europe"}, {"id": 16539, "dom": "G", "ent": "Vechelde", "cls": "Polynesia"}, {"id": 16540, "dom": "G", "ent": "Vechelde", "cls": "South America"}, {"id": 16541, "dom": "G", "ent": "Vechelde", "cls": "South-eastern Asia"}, {"id": 16542, "dom": "G", "ent": "Vechelde", "cls": "Southern Africa"}, {"id": 16543, "dom": "G", "ent": "Vechelde", "cls": "Southern Asia"}, {"id": 16544, "dom": "G", "ent": "Vechelde", "cls": "Southern Europe"}, {"id": 16545, "dom": "G", "ent": "Vechelde", "cls": "Western Africa"}, {"id": 16546, "dom": "G", "ent": "Vechelde", "cls": "Western Asia"}, {"id": 16547, "dom": "G", "ent": "Vechelde", "cls": "Western Europe"}, {"id": 16548, "dom": "G", "ent": "Randers", "cls": "Australia and New Zealand"}, {"id": 16549, "dom": "G", "ent": "Randers", "cls": "Caribbean"}, {"id": 16550, "dom": "G", "ent": "Randers", "cls": "Central America"}, {"id": 16551, "dom": "G", "ent": "Randers", "cls": "Central Asia"}, {"id": 16552, "dom": "G", "ent": "Randers", "cls": "Eastern Africa"}, {"id": 16553, "dom": "G", "ent": "Randers", "cls": "Eastern Asia"}, {"id": 16554, "dom": "G", "ent": "Randers", "cls": "Eastern Europe"}, {"id": 16555, "dom": "G", "ent": "Randers", "cls": "Melanesia"}, {"id": 16556, "dom": "G", "ent": "Randers", "cls": "Micronesia"}, {"id": 16557, "dom": "G", "ent": "Randers", "cls": "Middle Africa"}, {"id": 16558, "dom": "G", "ent": "Randers", "cls": "Northern Africa"}, {"id": 16559, "dom": "G", "ent": "Randers", "cls": "Northern America"}, {"id": 16560, "dom": "G", "ent": "Randers", "cls": "Northern Europe"}, {"id": 16561, "dom": "G", "ent": "Randers", "cls": "Polynesia"}, {"id": 16562, "dom": "G", "ent": "Randers", "cls": "South America"}, {"id": 16563, "dom": "G", "ent": "Randers", "cls": "South-eastern Asia"}, {"id": 16564, "dom": "G", "ent": "Randers", "cls": "Southern Africa"}, {"id": 16565, "dom": "G", "ent": "Randers", "cls": "Southern Asia"}, {"id": 16566, "dom": "G", "ent": "Randers", "cls": "Southern Europe"}, {"id": 16567, "dom": "G", "ent": "Randers", "cls": "Western Africa"}, {"id": 16568, "dom": "G", "ent": "Randers", "cls": "Western Asia"}, {"id": 16569, "dom": "G", "ent": "Randers", "cls": "Western Europe"}, {"id": 16570, "dom": "G", "ent": "Hedensted", "cls": "Australia and New Zealand"}, {"id": 16571, "dom": "G", "ent": "Hedensted", "cls": "Caribbean"}, {"id": 16572, "dom": "G", "ent": "Hedensted", "cls": "Central America"}, {"id": 16573, "dom": "G", "ent": "Hedensted", "cls": "Central Asia"}, {"id": 16574, "dom": "G", "ent": "Hedensted", "cls": "Eastern Africa"}, {"id": 16575, "dom": "G", "ent": "Hedensted", "cls": "Eastern Asia"}, {"id": 16576, "dom": "G", "ent": "Hedensted", "cls": "Eastern Europe"}, {"id": 16577, "dom": "G", "ent": "Hedensted", "cls": "Melanesia"}, {"id": 16578, "dom": "G", "ent": "Hedensted", "cls": "Micronesia"}, {"id": 16579, "dom": "G", "ent": "Hedensted", "cls": "Middle Africa"}, {"id": 16580, "dom": "G", "ent": "Hedensted", "cls": "Northern Africa"}, {"id": 16581, "dom": "G", "ent": "Hedensted", "cls": "Northern America"}, {"id": 16582, "dom": "G", "ent": "Hedensted", "cls": "Northern Europe"}, {"id": 16583, "dom": "G", "ent": "Hedensted", "cls": "Polynesia"}, {"id": 16584, "dom": "G", "ent": "Hedensted", "cls": "South America"}, {"id": 16585, "dom": "G", "ent": "Hedensted", "cls": "South-eastern Asia"}, {"id": 16586, "dom": "G", "ent": "Hedensted", "cls": "Southern Africa"}, {"id": 16587, "dom": "G", "ent": "Hedensted", "cls": "Southern Asia"}, {"id": 16588, "dom": "G", "ent": "Hedensted", "cls": "Southern Europe"}, {"id": 16589, "dom": "G", "ent": "Hedensted", "cls": "Western Africa"}, {"id": 16590, "dom": "G", "ent": "Hedensted", "cls": "Western Asia"}, {"id": 16591, "dom": "G", "ent": "Hedensted", "cls": "Western Europe"}, {"id": 16592, "dom": "G", "ent": "Glostrup", "cls": "Australia and New Zealand"}, {"id": 16593, "dom": "G", "ent": "Glostrup", "cls": "Caribbean"}, {"id": 16594, "dom": "G", "ent": "Glostrup", "cls": "Central America"}, {"id": 16595, "dom": "G", "ent": "Glostrup", "cls": "Central Asia"}, {"id": 16596, "dom": "G", "ent": "Glostrup", "cls": "Eastern Africa"}, {"id": 16597, "dom": "G", "ent": "Glostrup", "cls": "Eastern Asia"}, {"id": 16598, "dom": "G", "ent": "Glostrup", "cls": "Eastern Europe"}, {"id": 16599, "dom": "G", "ent": "Glostrup", "cls": "Melanesia"}, {"id": 16600, "dom": "G", "ent": "Glostrup", "cls": "Micronesia"}, {"id": 16601, "dom": "G", "ent": "Glostrup", "cls": "Middle Africa"}, {"id": 16602, "dom": "G", "ent": "Glostrup", "cls": "Northern Africa"}, {"id": 16603, "dom": "G", "ent": "Glostrup", "cls": "Northern America"}, {"id": 16604, "dom": "G", "ent": "Glostrup", "cls": "Northern Europe"}, {"id": 16605, "dom": "G", "ent": "Glostrup", "cls": "Polynesia"}, {"id": 16606, "dom": "G", "ent": "Glostrup", "cls": "South America"}, {"id": 16607, "dom": "G", "ent": "Glostrup", "cls": "South-eastern Asia"}, {"id": 16608, "dom": "G", "ent": "Glostrup", "cls": "Southern Africa"}, {"id": 16609, "dom": "G", "ent": "Glostrup", "cls": "Southern Asia"}, {"id": 16610, "dom": "G", "ent": "Glostrup", "cls": "Southern Europe"}, {"id": 16611, "dom": "G", "ent": "Glostrup", "cls": "Western Africa"}, {"id": 16612, "dom": "G", "ent": "Glostrup", "cls": "Western Asia"}, {"id": 16613, "dom": "G", "ent": "Glostrup", "cls": "Western Europe"}, {"id": 16614, "dom": "G", "ent": "Middelfart", "cls": "Australia and New Zealand"}, {"id": 16615, "dom": "G", "ent": "Middelfart", "cls": "Caribbean"}, {"id": 16616, "dom": "G", "ent": "Middelfart", "cls": "Central America"}, {"id": 16617, "dom": "G", "ent": "Middelfart", "cls": "Central Asia"}, {"id": 16618, "dom": "G", "ent": "Middelfart", "cls": "Eastern Africa"}, {"id": 16619, "dom": "G", "ent": "Middelfart", "cls": "Eastern Asia"}, {"id": 16620, "dom": "G", "ent": "Middelfart", "cls": "Eastern Europe"}, {"id": 16621, "dom": "G", "ent": "Middelfart", "cls": "Melanesia"}, {"id": 16622, "dom": "G", "ent": "Middelfart", "cls": "Micronesia"}, {"id": 16623, "dom": "G", "ent": "Middelfart", "cls": "Middle Africa"}, {"id": 16624, "dom": "G", "ent": "Middelfart", "cls": "Northern Africa"}, {"id": 16625, "dom": "G", "ent": "Middelfart", "cls": "Northern America"}, {"id": 16626, "dom": "G", "ent": "Middelfart", "cls": "Northern Europe"}, {"id": 16627, "dom": "G", "ent": "Middelfart", "cls": "Polynesia"}, {"id": 16628, "dom": "G", "ent": "Middelfart", "cls": "South America"}, {"id": 16629, "dom": "G", "ent": "Middelfart", "cls": "South-eastern Asia"}, {"id": 16630, "dom": "G", "ent": "Middelfart", "cls": "Southern Africa"}, {"id": 16631, "dom": "G", "ent": "Middelfart", "cls": "Southern Asia"}, {"id": 16632, "dom": "G", "ent": "Middelfart", "cls": "Southern Europe"}, {"id": 16633, "dom": "G", "ent": "Middelfart", "cls": "Western Africa"}, {"id": 16634, "dom": "G", "ent": "Middelfart", "cls": "Western Asia"}, {"id": 16635, "dom": "G", "ent": "Middelfart", "cls": "Western Europe"}, {"id": 16636, "dom": "G", "ent": "Bonao", "cls": "Australia and New Zealand"}, {"id": 16637, "dom": "G", "ent": "Bonao", "cls": "Caribbean"}, {"id": 16638, "dom": "G", "ent": "Bonao", "cls": "Central America"}, {"id": 16639, "dom": "G", "ent": "Bonao", "cls": "Central Asia"}, {"id": 16640, "dom": "G", "ent": "Bonao", "cls": "Eastern Africa"}, {"id": 16641, "dom": "G", "ent": "Bonao", "cls": "Eastern Asia"}, {"id": 16642, "dom": "G", "ent": "Bonao", "cls": "Eastern Europe"}, {"id": 16643, "dom": "G", "ent": "Bonao", "cls": "Melanesia"}, {"id": 16644, "dom": "G", "ent": "Bonao", "cls": "Micronesia"}, {"id": 16645, "dom": "G", "ent": "Bonao", "cls": "Middle Africa"}, {"id": 16646, "dom": "G", "ent": "Bonao", "cls": "Northern Africa"}, {"id": 16647, "dom": "G", "ent": "Bonao", "cls": "Northern America"}, {"id": 16648, "dom": "G", "ent": "Bonao", "cls": "Northern Europe"}, {"id": 16649, "dom": "G", "ent": "Bonao", "cls": "Polynesia"}, {"id": 16650, "dom": "G", "ent": "Bonao", "cls": "South America"}, {"id": 16651, "dom": "G", "ent": "Bonao", "cls": "South-eastern Asia"}, {"id": 16652, "dom": "G", "ent": "Bonao", "cls": "Southern Africa"}, {"id": 16653, "dom": "G", "ent": "Bonao", "cls": "Southern Asia"}, {"id": 16654, "dom": "G", "ent": "Bonao", "cls": "Southern Europe"}, {"id": 16655, "dom": "G", "ent": "Bonao", "cls": "Western Africa"}, {"id": 16656, "dom": "G", "ent": "Bonao", "cls": "Western Asia"}, {"id": 16657, "dom": "G", "ent": "Bonao", "cls": "Western Europe"}, {"id": 16658, "dom": "G", "ent": "Villa Francisca", "cls": "Australia and New Zealand"}, {"id": 16659, "dom": "G", "ent": "Villa Francisca", "cls": "Caribbean"}, {"id": 16660, "dom": "G", "ent": "Villa Francisca", "cls": "Central America"}, {"id": 16661, "dom": "G", "ent": "Villa Francisca", "cls": "Central Asia"}, {"id": 16662, "dom": "G", "ent": "Villa Francisca", "cls": "Eastern Africa"}, {"id": 16663, "dom": "G", "ent": "Villa Francisca", "cls": "Eastern Asia"}, {"id": 16664, "dom": "G", "ent": "Villa Francisca", "cls": "Eastern Europe"}, {"id": 16665, "dom": "G", "ent": "Villa Francisca", "cls": "Melanesia"}, {"id": 16666, "dom": "G", "ent": "Villa Francisca", "cls": "Micronesia"}, {"id": 16667, "dom": "G", "ent": "Villa Francisca", "cls": "Middle Africa"}, {"id": 16668, "dom": "G", "ent": "Villa Francisca", "cls": "Northern Africa"}, {"id": 16669, "dom": "G", "ent": "Villa Francisca", "cls": "Northern America"}, {"id": 16670, "dom": "G", "ent": "Villa Francisca", "cls": "Northern Europe"}, {"id": 16671, "dom": "G", "ent": "Villa Francisca", "cls": "Polynesia"}, {"id": 16672, "dom": "G", "ent": "Villa Francisca", "cls": "South America"}, {"id": 16673, "dom": "G", "ent": "Villa Francisca", "cls": "South-eastern Asia"}, {"id": 16674, "dom": "G", "ent": "Villa Francisca", "cls": "Southern Africa"}, {"id": 16675, "dom": "G", "ent": "Villa Francisca", "cls": "Southern Asia"}, {"id": 16676, "dom": "G", "ent": "Villa Francisca", "cls": "Southern Europe"}, {"id": 16677, "dom": "G", "ent": "Villa Francisca", "cls": "Western Africa"}, {"id": 16678, "dom": "G", "ent": "Villa Francisca", "cls": "Western Asia"}, {"id": 16679, "dom": "G", "ent": "Villa Francisca", "cls": "Western Europe"}, {"id": 16680, "dom": "G", "ent": "Salcedo", "cls": "Australia and New Zealand"}, {"id": 16681, "dom": "G", "ent": "Salcedo", "cls": "Caribbean"}, {"id": 16682, "dom": "G", "ent": "Salcedo", "cls": "Central America"}, {"id": 16683, "dom": "G", "ent": "Salcedo", "cls": "Central Asia"}, {"id": 16684, "dom": "G", "ent": "Salcedo", "cls": "Eastern Africa"}, {"id": 16685, "dom": "G", "ent": "Salcedo", "cls": "Eastern Asia"}, {"id": 16686, "dom": "G", "ent": "Salcedo", "cls": "Eastern Europe"}, {"id": 16687, "dom": "G", "ent": "Salcedo", "cls": "Melanesia"}, {"id": 16688, "dom": "G", "ent": "Salcedo", "cls": "Micronesia"}, {"id": 16689, "dom": "G", "ent": "Salcedo", "cls": "Middle Africa"}, {"id": 16690, "dom": "G", "ent": "Salcedo", "cls": "Northern Africa"}, {"id": 16691, "dom": "G", "ent": "Salcedo", "cls": "Northern America"}, {"id": 16692, "dom": "G", "ent": "Salcedo", "cls": "Northern Europe"}, {"id": 16693, "dom": "G", "ent": "Salcedo", "cls": "Polynesia"}, {"id": 16694, "dom": "G", "ent": "Salcedo", "cls": "South America"}, {"id": 16695, "dom": "G", "ent": "Salcedo", "cls": "South-eastern Asia"}, {"id": 16696, "dom": "G", "ent": "Salcedo", "cls": "Southern Africa"}, {"id": 16697, "dom": "G", "ent": "Salcedo", "cls": "Southern Asia"}, {"id": 16698, "dom": "G", "ent": "Salcedo", "cls": "Southern Europe"}, {"id": 16699, "dom": "G", "ent": "Salcedo", "cls": "Western Africa"}, {"id": 16700, "dom": "G", "ent": "Salcedo", "cls": "Western Asia"}, {"id": 16701, "dom": "G", "ent": "Salcedo", "cls": "Western Europe"}, {"id": 16702, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "Australia and New Zealand"}, {"id": 16703, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "Caribbean"}, {"id": 16704, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "Central America"}, {"id": 16705, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "Central Asia"}, {"id": 16706, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "Eastern Africa"}, {"id": 16707, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "Eastern Asia"}, {"id": 16708, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "Eastern Europe"}, {"id": 16709, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "Melanesia"}, {"id": 16710, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "Micronesia"}, {"id": 16711, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "Middle Africa"}, {"id": 16712, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "Northern Africa"}, {"id": 16713, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "Northern America"}, {"id": 16714, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "Northern Europe"}, {"id": 16715, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "Polynesia"}, {"id": 16716, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "South America"}, {"id": 16717, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "South-eastern Asia"}, {"id": 16718, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "Southern Africa"}, {"id": 16719, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "Southern Asia"}, {"id": 16720, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "Southern Europe"}, {"id": 16721, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "Western Africa"}, {"id": 16722, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "Western Asia"}, {"id": 16723, "dom": "G", "ent": "Hato Mayor del Rey", "cls": "Western Europe"}, {"id": 16724, "dom": "G", "ent": "Sedrata", "cls": "Australia and New Zealand"}, {"id": 16725, "dom": "G", "ent": "Sedrata", "cls": "Caribbean"}, {"id": 16726, "dom": "G", "ent": "Sedrata", "cls": "Central America"}, {"id": 16727, "dom": "G", "ent": "Sedrata", "cls": "Central Asia"}, {"id": 16728, "dom": "G", "ent": "Sedrata", "cls": "Eastern Africa"}, {"id": 16729, "dom": "G", "ent": "Sedrata", "cls": "Eastern Asia"}, {"id": 16730, "dom": "G", "ent": "Sedrata", "cls": "Eastern Europe"}, {"id": 16731, "dom": "G", "ent": "Sedrata", "cls": "Melanesia"}, {"id": 16732, "dom": "G", "ent": "Sedrata", "cls": "Micronesia"}, {"id": 16733, "dom": "G", "ent": "Sedrata", "cls": "Middle Africa"}, {"id": 16734, "dom": "G", "ent": "Sedrata", "cls": "Northern Africa"}, {"id": 16735, "dom": "G", "ent": "Sedrata", "cls": "Northern America"}, {"id": 16736, "dom": "G", "ent": "Sedrata", "cls": "Northern Europe"}, {"id": 16737, "dom": "G", "ent": "Sedrata", "cls": "Polynesia"}, {"id": 16738, "dom": "G", "ent": "Sedrata", "cls": "South America"}, {"id": 16739, "dom": "G", "ent": "Sedrata", "cls": "South-eastern Asia"}, {"id": 16740, "dom": "G", "ent": "Sedrata", "cls": "Southern Africa"}, {"id": 16741, "dom": "G", "ent": "Sedrata", "cls": "Southern Asia"}, {"id": 16742, "dom": "G", "ent": "Sedrata", "cls": "Southern Europe"}, {"id": 16743, "dom": "G", "ent": "Sedrata", "cls": "Western Africa"}, {"id": 16744, "dom": "G", "ent": "Sedrata", "cls": "Western Asia"}, {"id": 16745, "dom": "G", "ent": "Sedrata", "cls": "Western Europe"}, {"id": 16746, "dom": "G", "ent": "Djidiouia", "cls": "Australia and New Zealand"}, {"id": 16747, "dom": "G", "ent": "Djidiouia", "cls": "Caribbean"}, {"id": 16748, "dom": "G", "ent": "Djidiouia", "cls": "Central America"}, {"id": 16749, "dom": "G", "ent": "Djidiouia", "cls": "Central Asia"}, {"id": 16750, "dom": "G", "ent": "Djidiouia", "cls": "Eastern Africa"}, {"id": 16751, "dom": "G", "ent": "Djidiouia", "cls": "Eastern Asia"}, {"id": 16752, "dom": "G", "ent": "Djidiouia", "cls": "Eastern Europe"}, {"id": 16753, "dom": "G", "ent": "Djidiouia", "cls": "Melanesia"}, {"id": 16754, "dom": "G", "ent": "Djidiouia", "cls": "Micronesia"}, {"id": 16755, "dom": "G", "ent": "Djidiouia", "cls": "Middle Africa"}, {"id": 16756, "dom": "G", "ent": "Djidiouia", "cls": "Northern Africa"}, {"id": 16757, "dom": "G", "ent": "Djidiouia", "cls": "Northern America"}, {"id": 16758, "dom": "G", "ent": "Djidiouia", "cls": "Northern Europe"}, {"id": 16759, "dom": "G", "ent": "Djidiouia", "cls": "Polynesia"}, {"id": 16760, "dom": "G", "ent": "Djidiouia", "cls": "South America"}, {"id": 16761, "dom": "G", "ent": "Djidiouia", "cls": "South-eastern Asia"}, {"id": 16762, "dom": "G", "ent": "Djidiouia", "cls": "Southern Africa"}, {"id": 16763, "dom": "G", "ent": "Djidiouia", "cls": "Southern Asia"}, {"id": 16764, "dom": "G", "ent": "Djidiouia", "cls": "Southern Europe"}, {"id": 16765, "dom": "G", "ent": "Djidiouia", "cls": "Western Africa"}, {"id": 16766, "dom": "G", "ent": "Djidiouia", "cls": "Western Asia"}, {"id": 16767, "dom": "G", "ent": "Djidiouia", "cls": "Western Europe"}, {"id": 16768, "dom": "G", "ent": "Grarem", "cls": "Australia and New Zealand"}, {"id": 16769, "dom": "G", "ent": "Grarem", "cls": "Caribbean"}, {"id": 16770, "dom": "G", "ent": "Grarem", "cls": "Central America"}, {"id": 16771, "dom": "G", "ent": "Grarem", "cls": "Central Asia"}, {"id": 16772, "dom": "G", "ent": "Grarem", "cls": "Eastern Africa"}, {"id": 16773, "dom": "G", "ent": "Grarem", "cls": "Eastern Asia"}, {"id": 16774, "dom": "G", "ent": "Grarem", "cls": "Eastern Europe"}, {"id": 16775, "dom": "G", "ent": "Grarem", "cls": "Melanesia"}, {"id": 16776, "dom": "G", "ent": "Grarem", "cls": "Micronesia"}, {"id": 16777, "dom": "G", "ent": "Grarem", "cls": "Middle Africa"}, {"id": 16778, "dom": "G", "ent": "Grarem", "cls": "Northern Africa"}, {"id": 16779, "dom": "G", "ent": "Grarem", "cls": "Northern America"}, {"id": 16780, "dom": "G", "ent": "Grarem", "cls": "Northern Europe"}, {"id": 16781, "dom": "G", "ent": "Grarem", "cls": "Polynesia"}, {"id": 16782, "dom": "G", "ent": "Grarem", "cls": "South America"}, {"id": 16783, "dom": "G", "ent": "Grarem", "cls": "South-eastern Asia"}, {"id": 16784, "dom": "G", "ent": "Grarem", "cls": "Southern Africa"}, {"id": 16785, "dom": "G", "ent": "Grarem", "cls": "Southern Asia"}, {"id": 16786, "dom": "G", "ent": "Grarem", "cls": "Southern Europe"}, {"id": 16787, "dom": "G", "ent": "Grarem", "cls": "Western Africa"}, {"id": 16788, "dom": "G", "ent": "Grarem", "cls": "Western Asia"}, {"id": 16789, "dom": "G", "ent": "Grarem", "cls": "Western Europe"}, {"id": 16790, "dom": "G", "ent": "Charef", "cls": "Australia and New Zealand"}, {"id": 16791, "dom": "G", "ent": "Charef", "cls": "Caribbean"}, {"id": 16792, "dom": "G", "ent": "Charef", "cls": "Central America"}, {"id": 16793, "dom": "G", "ent": "Charef", "cls": "Central Asia"}, {"id": 16794, "dom": "G", "ent": "Charef", "cls": "Eastern Africa"}, {"id": 16795, "dom": "G", "ent": "Charef", "cls": "Eastern Asia"}, {"id": 16796, "dom": "G", "ent": "Charef", "cls": "Eastern Europe"}, {"id": 16797, "dom": "G", "ent": "Charef", "cls": "Melanesia"}, {"id": 16798, "dom": "G", "ent": "Charef", "cls": "Micronesia"}, {"id": 16799, "dom": "G", "ent": "Charef", "cls": "Middle Africa"}, {"id": 16800, "dom": "G", "ent": "Charef", "cls": "Northern Africa"}, {"id": 16801, "dom": "G", "ent": "Charef", "cls": "Northern America"}, {"id": 16802, "dom": "G", "ent": "Charef", "cls": "Northern Europe"}, {"id": 16803, "dom": "G", "ent": "Charef", "cls": "Polynesia"}, {"id": 16804, "dom": "G", "ent": "Charef", "cls": "South America"}, {"id": 16805, "dom": "G", "ent": "Charef", "cls": "South-eastern Asia"}, {"id": 16806, "dom": "G", "ent": "Charef", "cls": "Southern Africa"}, {"id": 16807, "dom": "G", "ent": "Charef", "cls": "Southern Asia"}, {"id": 16808, "dom": "G", "ent": "Charef", "cls": "Southern Europe"}, {"id": 16809, "dom": "G", "ent": "Charef", "cls": "Western Africa"}, {"id": 16810, "dom": "G", "ent": "Charef", "cls": "Western Asia"}, {"id": 16811, "dom": "G", "ent": "Charef", "cls": "Western Europe"}, {"id": 16812, "dom": "G", "ent": "Ambato", "cls": "Australia and New Zealand"}, {"id": 16813, "dom": "G", "ent": "Ambato", "cls": "Caribbean"}, {"id": 16814, "dom": "G", "ent": "Ambato", "cls": "Central America"}, {"id": 16815, "dom": "G", "ent": "Ambato", "cls": "Central Asia"}, {"id": 16816, "dom": "G", "ent": "Ambato", "cls": "Eastern Africa"}, {"id": 16817, "dom": "G", "ent": "Ambato", "cls": "Eastern Asia"}, {"id": 16818, "dom": "G", "ent": "Ambato", "cls": "Eastern Europe"}, {"id": 16819, "dom": "G", "ent": "Ambato", "cls": "Melanesia"}, {"id": 16820, "dom": "G", "ent": "Ambato", "cls": "Micronesia"}, {"id": 16821, "dom": "G", "ent": "Ambato", "cls": "Middle Africa"}, {"id": 16822, "dom": "G", "ent": "Ambato", "cls": "Northern Africa"}, {"id": 16823, "dom": "G", "ent": "Ambato", "cls": "Northern America"}, {"id": 16824, "dom": "G", "ent": "Ambato", "cls": "Northern Europe"}, {"id": 16825, "dom": "G", "ent": "Ambato", "cls": "Polynesia"}, {"id": 16826, "dom": "G", "ent": "Ambato", "cls": "South America"}, {"id": 16827, "dom": "G", "ent": "Ambato", "cls": "South-eastern Asia"}, {"id": 16828, "dom": "G", "ent": "Ambato", "cls": "Southern Africa"}, {"id": 16829, "dom": "G", "ent": "Ambato", "cls": "Southern Asia"}, {"id": 16830, "dom": "G", "ent": "Ambato", "cls": "Southern Europe"}, {"id": 16831, "dom": "G", "ent": "Ambato", "cls": "Western Africa"}, {"id": 16832, "dom": "G", "ent": "Ambato", "cls": "Western Asia"}, {"id": 16833, "dom": "G", "ent": "Ambato", "cls": "Western Europe"}, {"id": 16834, "dom": "G", "ent": "Machala", "cls": "Australia and New Zealand"}, {"id": 16835, "dom": "G", "ent": "Machala", "cls": "Caribbean"}, {"id": 16836, "dom": "G", "ent": "Machala", "cls": "Central America"}, {"id": 16837, "dom": "G", "ent": "Machala", "cls": "Central Asia"}, {"id": 16838, "dom": "G", "ent": "Machala", "cls": "Eastern Africa"}, {"id": 16839, "dom": "G", "ent": "Machala", "cls": "Eastern Asia"}, {"id": 16840, "dom": "G", "ent": "Machala", "cls": "Eastern Europe"}, {"id": 16841, "dom": "G", "ent": "Machala", "cls": "Melanesia"}, {"id": 16842, "dom": "G", "ent": "Machala", "cls": "Micronesia"}, {"id": 16843, "dom": "G", "ent": "Machala", "cls": "Middle Africa"}, {"id": 16844, "dom": "G", "ent": "Machala", "cls": "Northern Africa"}, {"id": 16845, "dom": "G", "ent": "Machala", "cls": "Northern America"}, {"id": 16846, "dom": "G", "ent": "Machala", "cls": "Northern Europe"}, {"id": 16847, "dom": "G", "ent": "Machala", "cls": "Polynesia"}, {"id": 16848, "dom": "G", "ent": "Machala", "cls": "South America"}, {"id": 16849, "dom": "G", "ent": "Machala", "cls": "South-eastern Asia"}, {"id": 16850, "dom": "G", "ent": "Machala", "cls": "Southern Africa"}, {"id": 16851, "dom": "G", "ent": "Machala", "cls": "Southern Asia"}, {"id": 16852, "dom": "G", "ent": "Machala", "cls": "Southern Europe"}, {"id": 16853, "dom": "G", "ent": "Machala", "cls": "Western Africa"}, {"id": 16854, "dom": "G", "ent": "Machala", "cls": "Western Asia"}, {"id": 16855, "dom": "G", "ent": "Machala", "cls": "Western Europe"}, {"id": 16856, "dom": "G", "ent": "Ibarra", "cls": "Australia and New Zealand"}, {"id": 16857, "dom": "G", "ent": "Ibarra", "cls": "Caribbean"}, {"id": 16858, "dom": "G", "ent": "Ibarra", "cls": "Central America"}, {"id": 16859, "dom": "G", "ent": "Ibarra", "cls": "Central Asia"}, {"id": 16860, "dom": "G", "ent": "Ibarra", "cls": "Eastern Africa"}, {"id": 16861, "dom": "G", "ent": "Ibarra", "cls": "Eastern Asia"}, {"id": 16862, "dom": "G", "ent": "Ibarra", "cls": "Eastern Europe"}, {"id": 16863, "dom": "G", "ent": "Ibarra", "cls": "Melanesia"}, {"id": 16864, "dom": "G", "ent": "Ibarra", "cls": "Micronesia"}, {"id": 16865, "dom": "G", "ent": "Ibarra", "cls": "Middle Africa"}, {"id": 16866, "dom": "G", "ent": "Ibarra", "cls": "Northern Africa"}, {"id": 16867, "dom": "G", "ent": "Ibarra", "cls": "Northern America"}, {"id": 16868, "dom": "G", "ent": "Ibarra", "cls": "Northern Europe"}, {"id": 16869, "dom": "G", "ent": "Ibarra", "cls": "Polynesia"}, {"id": 16870, "dom": "G", "ent": "Ibarra", "cls": "South America"}, {"id": 16871, "dom": "G", "ent": "Ibarra", "cls": "South-eastern Asia"}, {"id": 16872, "dom": "G", "ent": "Ibarra", "cls": "Southern Africa"}, {"id": 16873, "dom": "G", "ent": "Ibarra", "cls": "Southern Asia"}, {"id": 16874, "dom": "G", "ent": "Ibarra", "cls": "Southern Europe"}, {"id": 16875, "dom": "G", "ent": "Ibarra", "cls": "Western Africa"}, {"id": 16876, "dom": "G", "ent": "Ibarra", "cls": "Western Asia"}, {"id": 16877, "dom": "G", "ent": "Ibarra", "cls": "Western Europe"}, {"id": 16878, "dom": "G", "ent": "Rosa Zarate", "cls": "Australia and New Zealand"}, {"id": 16879, "dom": "G", "ent": "Rosa Zarate", "cls": "Caribbean"}, {"id": 16880, "dom": "G", "ent": "Rosa Zarate", "cls": "Central America"}, {"id": 16881, "dom": "G", "ent": "Rosa Zarate", "cls": "Central Asia"}, {"id": 16882, "dom": "G", "ent": "Rosa Zarate", "cls": "Eastern Africa"}, {"id": 16883, "dom": "G", "ent": "Rosa Zarate", "cls": "Eastern Asia"}, {"id": 16884, "dom": "G", "ent": "Rosa Zarate", "cls": "Eastern Europe"}, {"id": 16885, "dom": "G", "ent": "Rosa Zarate", "cls": "Melanesia"}, {"id": 16886, "dom": "G", "ent": "Rosa Zarate", "cls": "Micronesia"}, {"id": 16887, "dom": "G", "ent": "Rosa Zarate", "cls": "Middle Africa"}, {"id": 16888, "dom": "G", "ent": "Rosa Zarate", "cls": "Northern Africa"}, {"id": 16889, "dom": "G", "ent": "Rosa Zarate", "cls": "Northern America"}, {"id": 16890, "dom": "G", "ent": "Rosa Zarate", "cls": "Northern Europe"}, {"id": 16891, "dom": "G", "ent": "Rosa Zarate", "cls": "Polynesia"}, {"id": 16892, "dom": "G", "ent": "Rosa Zarate", "cls": "South America"}, {"id": 16893, "dom": "G", "ent": "Rosa Zarate", "cls": "South-eastern Asia"}, {"id": 16894, "dom": "G", "ent": "Rosa Zarate", "cls": "Southern Africa"}, {"id": 16895, "dom": "G", "ent": "Rosa Zarate", "cls": "Southern Asia"}, {"id": 16896, "dom": "G", "ent": "Rosa Zarate", "cls": "Southern Europe"}, {"id": 16897, "dom": "G", "ent": "Rosa Zarate", "cls": "Western Africa"}, {"id": 16898, "dom": "G", "ent": "Rosa Zarate", "cls": "Western Asia"}, {"id": 16899, "dom": "G", "ent": "Rosa Zarate", "cls": "Western Europe"}, {"id": 16900, "dom": "G", "ent": "Tartu", "cls": "Australia and New Zealand"}, {"id": 16901, "dom": "G", "ent": "Tartu", "cls": "Caribbean"}, {"id": 16902, "dom": "G", "ent": "Tartu", "cls": "Central America"}, {"id": 16903, "dom": "G", "ent": "Tartu", "cls": "Central Asia"}, {"id": 16904, "dom": "G", "ent": "Tartu", "cls": "Eastern Africa"}, {"id": 16905, "dom": "G", "ent": "Tartu", "cls": "Eastern Asia"}, {"id": 16906, "dom": "G", "ent": "Tartu", "cls": "Eastern Europe"}, {"id": 16907, "dom": "G", "ent": "Tartu", "cls": "Melanesia"}, {"id": 16908, "dom": "G", "ent": "Tartu", "cls": "Micronesia"}, {"id": 16909, "dom": "G", "ent": "Tartu", "cls": "Middle Africa"}, {"id": 16910, "dom": "G", "ent": "Tartu", "cls": "Northern Africa"}, {"id": 16911, "dom": "G", "ent": "Tartu", "cls": "Northern America"}, {"id": 16912, "dom": "G", "ent": "Tartu", "cls": "Northern Europe"}, {"id": 16913, "dom": "G", "ent": "Tartu", "cls": "Polynesia"}, {"id": 16914, "dom": "G", "ent": "Tartu", "cls": "South America"}, {"id": 16915, "dom": "G", "ent": "Tartu", "cls": "South-eastern Asia"}, {"id": 16916, "dom": "G", "ent": "Tartu", "cls": "Southern Africa"}, {"id": 16917, "dom": "G", "ent": "Tartu", "cls": "Southern Asia"}, {"id": 16918, "dom": "G", "ent": "Tartu", "cls": "Southern Europe"}, {"id": 16919, "dom": "G", "ent": "Tartu", "cls": "Western Africa"}, {"id": 16920, "dom": "G", "ent": "Tartu", "cls": "Western Asia"}, {"id": 16921, "dom": "G", "ent": "Tartu", "cls": "Western Europe"}, {"id": 16922, "dom": "G", "ent": "Haabersti", "cls": "Australia and New Zealand"}, {"id": 16923, "dom": "G", "ent": "Haabersti", "cls": "Caribbean"}, {"id": 16924, "dom": "G", "ent": "Haabersti", "cls": "Central America"}, {"id": 16925, "dom": "G", "ent": "Haabersti", "cls": "Central Asia"}, {"id": 16926, "dom": "G", "ent": "Haabersti", "cls": "Eastern Africa"}, {"id": 16927, "dom": "G", "ent": "Haabersti", "cls": "Eastern Asia"}, {"id": 16928, "dom": "G", "ent": "Haabersti", "cls": "Eastern Europe"}, {"id": 16929, "dom": "G", "ent": "Haabersti", "cls": "Melanesia"}, {"id": 16930, "dom": "G", "ent": "Haabersti", "cls": "Micronesia"}, {"id": 16931, "dom": "G", "ent": "Haabersti", "cls": "Middle Africa"}, {"id": 16932, "dom": "G", "ent": "Haabersti", "cls": "Northern Africa"}, {"id": 16933, "dom": "G", "ent": "Haabersti", "cls": "Northern America"}, {"id": 16934, "dom": "G", "ent": "Haabersti", "cls": "Northern Europe"}, {"id": 16935, "dom": "G", "ent": "Haabersti", "cls": "Polynesia"}, {"id": 16936, "dom": "G", "ent": "Haabersti", "cls": "South America"}, {"id": 16937, "dom": "G", "ent": "Haabersti", "cls": "South-eastern Asia"}, {"id": 16938, "dom": "G", "ent": "Haabersti", "cls": "Southern Africa"}, {"id": 16939, "dom": "G", "ent": "Haabersti", "cls": "Southern Asia"}, {"id": 16940, "dom": "G", "ent": "Haabersti", "cls": "Southern Europe"}, {"id": 16941, "dom": "G", "ent": "Haabersti", "cls": "Western Africa"}, {"id": 16942, "dom": "G", "ent": "Haabersti", "cls": "Western Asia"}, {"id": 16943, "dom": "G", "ent": "Haabersti", "cls": "Western Europe"}, {"id": 16944, "dom": "G", "ent": "Kristiine", "cls": "Australia and New Zealand"}, {"id": 16945, "dom": "G", "ent": "Kristiine", "cls": "Caribbean"}, {"id": 16946, "dom": "G", "ent": "Kristiine", "cls": "Central America"}, {"id": 16947, "dom": "G", "ent": "Kristiine", "cls": "Central Asia"}, {"id": 16948, "dom": "G", "ent": "Kristiine", "cls": "Eastern Africa"}, {"id": 16949, "dom": "G", "ent": "Kristiine", "cls": "Eastern Asia"}, {"id": 16950, "dom": "G", "ent": "Kristiine", "cls": "Eastern Europe"}, {"id": 16951, "dom": "G", "ent": "Kristiine", "cls": "Melanesia"}, {"id": 16952, "dom": "G", "ent": "Kristiine", "cls": "Micronesia"}, {"id": 16953, "dom": "G", "ent": "Kristiine", "cls": "Middle Africa"}, {"id": 16954, "dom": "G", "ent": "Kristiine", "cls": "Northern Africa"}, {"id": 16955, "dom": "G", "ent": "Kristiine", "cls": "Northern America"}, {"id": 16956, "dom": "G", "ent": "Kristiine", "cls": "Northern Europe"}, {"id": 16957, "dom": "G", "ent": "Kristiine", "cls": "Polynesia"}, {"id": 16958, "dom": "G", "ent": "Kristiine", "cls": "South America"}, {"id": 16959, "dom": "G", "ent": "Kristiine", "cls": "South-eastern Asia"}, {"id": 16960, "dom": "G", "ent": "Kristiine", "cls": "Southern Africa"}, {"id": 16961, "dom": "G", "ent": "Kristiine", "cls": "Southern Asia"}, {"id": 16962, "dom": "G", "ent": "Kristiine", "cls": "Southern Europe"}, {"id": 16963, "dom": "G", "ent": "Kristiine", "cls": "Western Africa"}, {"id": 16964, "dom": "G", "ent": "Kristiine", "cls": "Western Asia"}, {"id": 16965, "dom": "G", "ent": "Kristiine", "cls": "Western Europe"}, {"id": 16966, "dom": "G", "ent": "Viljandi", "cls": "Australia and New Zealand"}, {"id": 16967, "dom": "G", "ent": "Viljandi", "cls": "Caribbean"}, {"id": 16968, "dom": "G", "ent": "Viljandi", "cls": "Central America"}, {"id": 16969, "dom": "G", "ent": "Viljandi", "cls": "Central Asia"}, {"id": 16970, "dom": "G", "ent": "Viljandi", "cls": "Eastern Africa"}, {"id": 16971, "dom": "G", "ent": "Viljandi", "cls": "Eastern Asia"}, {"id": 16972, "dom": "G", "ent": "Viljandi", "cls": "Eastern Europe"}, {"id": 16973, "dom": "G", "ent": "Viljandi", "cls": "Melanesia"}, {"id": 16974, "dom": "G", "ent": "Viljandi", "cls": "Micronesia"}, {"id": 16975, "dom": "G", "ent": "Viljandi", "cls": "Middle Africa"}, {"id": 16976, "dom": "G", "ent": "Viljandi", "cls": "Northern Africa"}, {"id": 16977, "dom": "G", "ent": "Viljandi", "cls": "Northern America"}, {"id": 16978, "dom": "G", "ent": "Viljandi", "cls": "Northern Europe"}, {"id": 16979, "dom": "G", "ent": "Viljandi", "cls": "Polynesia"}, {"id": 16980, "dom": "G", "ent": "Viljandi", "cls": "South America"}, {"id": 16981, "dom": "G", "ent": "Viljandi", "cls": "South-eastern Asia"}, {"id": 16982, "dom": "G", "ent": "Viljandi", "cls": "Southern Africa"}, {"id": 16983, "dom": "G", "ent": "Viljandi", "cls": "Southern Asia"}, {"id": 16984, "dom": "G", "ent": "Viljandi", "cls": "Southern Europe"}, {"id": 16985, "dom": "G", "ent": "Viljandi", "cls": "Western Africa"}, {"id": 16986, "dom": "G", "ent": "Viljandi", "cls": "Western Asia"}, {"id": 16987, "dom": "G", "ent": "Viljandi", "cls": "Western Europe"}, {"id": 16988, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "Australia and New Zealand"}, {"id": 16989, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "Caribbean"}, {"id": 16990, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "Central America"}, {"id": 16991, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "Central Asia"}, {"id": 16992, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "Eastern Africa"}, {"id": 16993, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "Eastern Asia"}, {"id": 16994, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "Eastern Europe"}, {"id": 16995, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "Melanesia"}, {"id": 16996, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "Micronesia"}, {"id": 16997, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "Middle Africa"}, {"id": 16998, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "Northern Africa"}, {"id": 16999, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "Northern America"}, {"id": 17000, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "Northern Europe"}, {"id": 17001, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "Polynesia"}, {"id": 17002, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "South America"}, {"id": 17003, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "South-eastern Asia"}, {"id": 17004, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "Southern Africa"}, {"id": 17005, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "Southern Asia"}, {"id": 17006, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "Southern Europe"}, {"id": 17007, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "Western Africa"}, {"id": 17008, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "Western Asia"}, {"id": 17009, "dom": "G", "ent": "Rivas-Vaciamadrid", "cls": "Western Europe"}, {"id": 17010, "dom": "G", "ent": "Chipiona", "cls": "Australia and New Zealand"}, {"id": 17011, "dom": "G", "ent": "Chipiona", "cls": "Caribbean"}, {"id": 17012, "dom": "G", "ent": "Chipiona", "cls": "Central America"}, {"id": 17013, "dom": "G", "ent": "Chipiona", "cls": "Central Asia"}, {"id": 17014, "dom": "G", "ent": "Chipiona", "cls": "Eastern Africa"}, {"id": 17015, "dom": "G", "ent": "Chipiona", "cls": "Eastern Asia"}, {"id": 17016, "dom": "G", "ent": "Chipiona", "cls": "Eastern Europe"}, {"id": 17017, "dom": "G", "ent": "Chipiona", "cls": "Melanesia"}, {"id": 17018, "dom": "G", "ent": "Chipiona", "cls": "Micronesia"}, {"id": 17019, "dom": "G", "ent": "Chipiona", "cls": "Middle Africa"}, {"id": 17020, "dom": "G", "ent": "Chipiona", "cls": "Northern Africa"}, {"id": 17021, "dom": "G", "ent": "Chipiona", "cls": "Northern America"}, {"id": 17022, "dom": "G", "ent": "Chipiona", "cls": "Northern Europe"}, {"id": 17023, "dom": "G", "ent": "Chipiona", "cls": "Polynesia"}, {"id": 17024, "dom": "G", "ent": "Chipiona", "cls": "South America"}, {"id": 17025, "dom": "G", "ent": "Chipiona", "cls": "South-eastern Asia"}, {"id": 17026, "dom": "G", "ent": "Chipiona", "cls": "Southern Africa"}, {"id": 17027, "dom": "G", "ent": "Chipiona", "cls": "Southern Asia"}, {"id": 17028, "dom": "G", "ent": "Chipiona", "cls": "Southern Europe"}, {"id": 17029, "dom": "G", "ent": "Chipiona", "cls": "Western Africa"}, {"id": 17030, "dom": "G", "ent": "Chipiona", "cls": "Western Asia"}, {"id": 17031, "dom": "G", "ent": "Chipiona", "cls": "Western Europe"}, {"id": 17032, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "Australia and New Zealand"}, {"id": 17033, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "Caribbean"}, {"id": 17034, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "Central America"}, {"id": 17035, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "Central Asia"}, {"id": 17036, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "Eastern Africa"}, {"id": 17037, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "Eastern Asia"}, {"id": 17038, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "Eastern Europe"}, {"id": 17039, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "Melanesia"}, {"id": 17040, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "Micronesia"}, {"id": 17041, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "Middle Africa"}, {"id": 17042, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "Northern Africa"}, {"id": 17043, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "Northern America"}, {"id": 17044, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "Northern Europe"}, {"id": 17045, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "Polynesia"}, {"id": 17046, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "South America"}, {"id": 17047, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "South-eastern Asia"}, {"id": 17048, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "Southern Africa"}, {"id": 17049, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "Southern Asia"}, {"id": 17050, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "Southern Europe"}, {"id": 17051, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "Western Africa"}, {"id": 17052, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "Western Asia"}, {"id": 17053, "dom": "G", "ent": "Castilleja de la Cuesta", "cls": "Western Europe"}, {"id": 17054, "dom": "G", "ent": "Carlet", "cls": "Australia and New Zealand"}, {"id": 17055, "dom": "G", "ent": "Carlet", "cls": "Caribbean"}, {"id": 17056, "dom": "G", "ent": "Carlet", "cls": "Central America"}, {"id": 17057, "dom": "G", "ent": "Carlet", "cls": "Central Asia"}, {"id": 17058, "dom": "G", "ent": "Carlet", "cls": "Eastern Africa"}, {"id": 17059, "dom": "G", "ent": "Carlet", "cls": "Eastern Asia"}, {"id": 17060, "dom": "G", "ent": "Carlet", "cls": "Eastern Europe"}, {"id": 17061, "dom": "G", "ent": "Carlet", "cls": "Melanesia"}, {"id": 17062, "dom": "G", "ent": "Carlet", "cls": "Micronesia"}, {"id": 17063, "dom": "G", "ent": "Carlet", "cls": "Middle Africa"}, {"id": 17064, "dom": "G", "ent": "Carlet", "cls": "Northern Africa"}, {"id": 17065, "dom": "G", "ent": "Carlet", "cls": "Northern America"}, {"id": 17066, "dom": "G", "ent": "Carlet", "cls": "Northern Europe"}, {"id": 17067, "dom": "G", "ent": "Carlet", "cls": "Polynesia"}, {"id": 17068, "dom": "G", "ent": "Carlet", "cls": "South America"}, {"id": 17069, "dom": "G", "ent": "Carlet", "cls": "South-eastern Asia"}, {"id": 17070, "dom": "G", "ent": "Carlet", "cls": "Southern Africa"}, {"id": 17071, "dom": "G", "ent": "Carlet", "cls": "Southern Asia"}, {"id": 17072, "dom": "G", "ent": "Carlet", "cls": "Southern Europe"}, {"id": 17073, "dom": "G", "ent": "Carlet", "cls": "Western Africa"}, {"id": 17074, "dom": "G", "ent": "Carlet", "cls": "Western Asia"}, {"id": 17075, "dom": "G", "ent": "Carlet", "cls": "Western Europe"}, {"id": 17076, "dom": "G", "ent": "Kibre Mengist", "cls": "Australia and New Zealand"}, {"id": 17077, "dom": "G", "ent": "Kibre Mengist", "cls": "Caribbean"}, {"id": 17078, "dom": "G", "ent": "Kibre Mengist", "cls": "Central America"}, {"id": 17079, "dom": "G", "ent": "Kibre Mengist", "cls": "Central Asia"}, {"id": 17080, "dom": "G", "ent": "Kibre Mengist", "cls": "Eastern Africa"}, {"id": 17081, "dom": "G", "ent": "Kibre Mengist", "cls": "Eastern Asia"}, {"id": 17082, "dom": "G", "ent": "Kibre Mengist", "cls": "Eastern Europe"}, {"id": 17083, "dom": "G", "ent": "Kibre Mengist", "cls": "Melanesia"}, {"id": 17084, "dom": "G", "ent": "Kibre Mengist", "cls": "Micronesia"}, {"id": 17085, "dom": "G", "ent": "Kibre Mengist", "cls": "Middle Africa"}, {"id": 17086, "dom": "G", "ent": "Kibre Mengist", "cls": "Northern Africa"}, {"id": 17087, "dom": "G", "ent": "Kibre Mengist", "cls": "Northern America"}, {"id": 17088, "dom": "G", "ent": "Kibre Mengist", "cls": "Northern Europe"}, {"id": 17089, "dom": "G", "ent": "Kibre Mengist", "cls": "Polynesia"}, {"id": 17090, "dom": "G", "ent": "Kibre Mengist", "cls": "South America"}, {"id": 17091, "dom": "G", "ent": "Kibre Mengist", "cls": "South-eastern Asia"}, {"id": 17092, "dom": "G", "ent": "Kibre Mengist", "cls": "Southern Africa"}, {"id": 17093, "dom": "G", "ent": "Kibre Mengist", "cls": "Southern Asia"}, {"id": 17094, "dom": "G", "ent": "Kibre Mengist", "cls": "Southern Europe"}, {"id": 17095, "dom": "G", "ent": "Kibre Mengist", "cls": "Western Africa"}, {"id": 17096, "dom": "G", "ent": "Kibre Mengist", "cls": "Western Asia"}, {"id": 17097, "dom": "G", "ent": "Kibre Mengist", "cls": "Western Europe"}, {"id": 17098, "dom": "G", "ent": "Tulu Bolo", "cls": "Australia and New Zealand"}, {"id": 17099, "dom": "G", "ent": "Tulu Bolo", "cls": "Caribbean"}, {"id": 17100, "dom": "G", "ent": "Tulu Bolo", "cls": "Central America"}, {"id": 17101, "dom": "G", "ent": "Tulu Bolo", "cls": "Central Asia"}, {"id": 17102, "dom": "G", "ent": "Tulu Bolo", "cls": "Eastern Africa"}, {"id": 17103, "dom": "G", "ent": "Tulu Bolo", "cls": "Eastern Asia"}, {"id": 17104, "dom": "G", "ent": "Tulu Bolo", "cls": "Eastern Europe"}, {"id": 17105, "dom": "G", "ent": "Tulu Bolo", "cls": "Melanesia"}, {"id": 17106, "dom": "G", "ent": "Tulu Bolo", "cls": "Micronesia"}, {"id": 17107, "dom": "G", "ent": "Tulu Bolo", "cls": "Middle Africa"}, {"id": 17108, "dom": "G", "ent": "Tulu Bolo", "cls": "Northern Africa"}, {"id": 17109, "dom": "G", "ent": "Tulu Bolo", "cls": "Northern America"}, {"id": 17110, "dom": "G", "ent": "Tulu Bolo", "cls": "Northern Europe"}, {"id": 17111, "dom": "G", "ent": "Tulu Bolo", "cls": "Polynesia"}, {"id": 17112, "dom": "G", "ent": "Tulu Bolo", "cls": "South America"}, {"id": 17113, "dom": "G", "ent": "Tulu Bolo", "cls": "South-eastern Asia"}, {"id": 17114, "dom": "G", "ent": "Tulu Bolo", "cls": "Southern Africa"}, {"id": 17115, "dom": "G", "ent": "Tulu Bolo", "cls": "Southern Asia"}, {"id": 17116, "dom": "G", "ent": "Tulu Bolo", "cls": "Southern Europe"}, {"id": 17117, "dom": "G", "ent": "Tulu Bolo", "cls": "Western Africa"}, {"id": 17118, "dom": "G", "ent": "Tulu Bolo", "cls": "Western Asia"}, {"id": 17119, "dom": "G", "ent": "Tulu Bolo", "cls": "Western Europe"}, {"id": 17120, "dom": "G", "ent": "Deder", "cls": "Australia and New Zealand"}, {"id": 17121, "dom": "G", "ent": "Deder", "cls": "Caribbean"}, {"id": 17122, "dom": "G", "ent": "Deder", "cls": "Central America"}, {"id": 17123, "dom": "G", "ent": "Deder", "cls": "Central Asia"}, {"id": 17124, "dom": "G", "ent": "Deder", "cls": "Eastern Africa"}, {"id": 17125, "dom": "G", "ent": "Deder", "cls": "Eastern Asia"}, {"id": 17126, "dom": "G", "ent": "Deder", "cls": "Eastern Europe"}, {"id": 17127, "dom": "G", "ent": "Deder", "cls": "Melanesia"}, {"id": 17128, "dom": "G", "ent": "Deder", "cls": "Micronesia"}, {"id": 17129, "dom": "G", "ent": "Deder", "cls": "Middle Africa"}, {"id": 17130, "dom": "G", "ent": "Deder", "cls": "Northern Africa"}, {"id": 17131, "dom": "G", "ent": "Deder", "cls": "Northern America"}, {"id": 17132, "dom": "G", "ent": "Deder", "cls": "Northern Europe"}, {"id": 17133, "dom": "G", "ent": "Deder", "cls": "Polynesia"}, {"id": 17134, "dom": "G", "ent": "Deder", "cls": "South America"}, {"id": 17135, "dom": "G", "ent": "Deder", "cls": "South-eastern Asia"}, {"id": 17136, "dom": "G", "ent": "Deder", "cls": "Southern Africa"}, {"id": 17137, "dom": "G", "ent": "Deder", "cls": "Southern Asia"}, {"id": 17138, "dom": "G", "ent": "Deder", "cls": "Southern Europe"}, {"id": 17139, "dom": "G", "ent": "Deder", "cls": "Western Africa"}, {"id": 17140, "dom": "G", "ent": "Deder", "cls": "Western Asia"}, {"id": 17141, "dom": "G", "ent": "Deder", "cls": "Western Europe"}, {"id": 17142, "dom": "G", "ent": "Adebaye", "cls": "Australia and New Zealand"}, {"id": 17143, "dom": "G", "ent": "Adebaye", "cls": "Caribbean"}, {"id": 17144, "dom": "G", "ent": "Adebaye", "cls": "Central America"}, {"id": 17145, "dom": "G", "ent": "Adebaye", "cls": "Central Asia"}, {"id": 17146, "dom": "G", "ent": "Adebaye", "cls": "Eastern Africa"}, {"id": 17147, "dom": "G", "ent": "Adebaye", "cls": "Eastern Asia"}, {"id": 17148, "dom": "G", "ent": "Adebaye", "cls": "Eastern Europe"}, {"id": 17149, "dom": "G", "ent": "Adebaye", "cls": "Melanesia"}, {"id": 17150, "dom": "G", "ent": "Adebaye", "cls": "Micronesia"}, {"id": 17151, "dom": "G", "ent": "Adebaye", "cls": "Middle Africa"}, {"id": 17152, "dom": "G", "ent": "Adebaye", "cls": "Northern Africa"}, {"id": 17153, "dom": "G", "ent": "Adebaye", "cls": "Northern America"}, {"id": 17154, "dom": "G", "ent": "Adebaye", "cls": "Northern Europe"}, {"id": 17155, "dom": "G", "ent": "Adebaye", "cls": "Polynesia"}, {"id": 17156, "dom": "G", "ent": "Adebaye", "cls": "South America"}, {"id": 17157, "dom": "G", "ent": "Adebaye", "cls": "South-eastern Asia"}, {"id": 17158, "dom": "G", "ent": "Adebaye", "cls": "Southern Africa"}, {"id": 17159, "dom": "G", "ent": "Adebaye", "cls": "Southern Asia"}, {"id": 17160, "dom": "G", "ent": "Adebaye", "cls": "Southern Europe"}, {"id": 17161, "dom": "G", "ent": "Adebaye", "cls": "Western Africa"}, {"id": 17162, "dom": "G", "ent": "Adebaye", "cls": "Western Asia"}, {"id": 17163, "dom": "G", "ent": "Adebaye", "cls": "Western Europe"}, {"id": 17164, "dom": "G", "ent": "East Helsinki", "cls": "Australia and New Zealand"}, {"id": 17165, "dom": "G", "ent": "East Helsinki", "cls": "Caribbean"}, {"id": 17166, "dom": "G", "ent": "East Helsinki", "cls": "Central America"}, {"id": 17167, "dom": "G", "ent": "East Helsinki", "cls": "Central Asia"}, {"id": 17168, "dom": "G", "ent": "East Helsinki", "cls": "Eastern Africa"}, {"id": 17169, "dom": "G", "ent": "East Helsinki", "cls": "Eastern Asia"}, {"id": 17170, "dom": "G", "ent": "East Helsinki", "cls": "Eastern Europe"}, {"id": 17171, "dom": "G", "ent": "East Helsinki", "cls": "Melanesia"}, {"id": 17172, "dom": "G", "ent": "East Helsinki", "cls": "Micronesia"}, {"id": 17173, "dom": "G", "ent": "East Helsinki", "cls": "Middle Africa"}, {"id": 17174, "dom": "G", "ent": "East Helsinki", "cls": "Northern Africa"}, {"id": 17175, "dom": "G", "ent": "East Helsinki", "cls": "Northern America"}, {"id": 17176, "dom": "G", "ent": "East Helsinki", "cls": "Northern Europe"}, {"id": 17177, "dom": "G", "ent": "East Helsinki", "cls": "Polynesia"}, {"id": 17178, "dom": "G", "ent": "East Helsinki", "cls": "South America"}, {"id": 17179, "dom": "G", "ent": "East Helsinki", "cls": "South-eastern Asia"}, {"id": 17180, "dom": "G", "ent": "East Helsinki", "cls": "Southern Africa"}, {"id": 17181, "dom": "G", "ent": "East Helsinki", "cls": "Southern Asia"}, {"id": 17182, "dom": "G", "ent": "East Helsinki", "cls": "Southern Europe"}, {"id": 17183, "dom": "G", "ent": "East Helsinki", "cls": "Western Africa"}, {"id": 17184, "dom": "G", "ent": "East Helsinki", "cls": "Western Asia"}, {"id": 17185, "dom": "G", "ent": "East Helsinki", "cls": "Western Europe"}, {"id": 17186, "dom": "G", "ent": "Kuopio", "cls": "Australia and New Zealand"}, {"id": 17187, "dom": "G", "ent": "Kuopio", "cls": "Caribbean"}, {"id": 17188, "dom": "G", "ent": "Kuopio", "cls": "Central America"}, {"id": 17189, "dom": "G", "ent": "Kuopio", "cls": "Central Asia"}, {"id": 17190, "dom": "G", "ent": "Kuopio", "cls": "Eastern Africa"}, {"id": 17191, "dom": "G", "ent": "Kuopio", "cls": "Eastern Asia"}, {"id": 17192, "dom": "G", "ent": "Kuopio", "cls": "Eastern Europe"}, {"id": 17193, "dom": "G", "ent": "Kuopio", "cls": "Melanesia"}, {"id": 17194, "dom": "G", "ent": "Kuopio", "cls": "Micronesia"}, {"id": 17195, "dom": "G", "ent": "Kuopio", "cls": "Middle Africa"}, {"id": 17196, "dom": "G", "ent": "Kuopio", "cls": "Northern Africa"}, {"id": 17197, "dom": "G", "ent": "Kuopio", "cls": "Northern America"}, {"id": 17198, "dom": "G", "ent": "Kuopio", "cls": "Northern Europe"}, {"id": 17199, "dom": "G", "ent": "Kuopio", "cls": "Polynesia"}, {"id": 17200, "dom": "G", "ent": "Kuopio", "cls": "South America"}, {"id": 17201, "dom": "G", "ent": "Kuopio", "cls": "South-eastern Asia"}, {"id": 17202, "dom": "G", "ent": "Kuopio", "cls": "Southern Africa"}, {"id": 17203, "dom": "G", "ent": "Kuopio", "cls": "Southern Asia"}, {"id": 17204, "dom": "G", "ent": "Kuopio", "cls": "Southern Europe"}, {"id": 17205, "dom": "G", "ent": "Kuopio", "cls": "Western Africa"}, {"id": 17206, "dom": "G", "ent": "Kuopio", "cls": "Western Asia"}, {"id": 17207, "dom": "G", "ent": "Kuopio", "cls": "Western Europe"}, {"id": 17208, "dom": "G", "ent": "Herttoniemi", "cls": "Australia and New Zealand"}, {"id": 17209, "dom": "G", "ent": "Herttoniemi", "cls": "Caribbean"}, {"id": 17210, "dom": "G", "ent": "Herttoniemi", "cls": "Central America"}, {"id": 17211, "dom": "G", "ent": "Herttoniemi", "cls": "Central Asia"}, {"id": 17212, "dom": "G", "ent": "Herttoniemi", "cls": "Eastern Africa"}, {"id": 17213, "dom": "G", "ent": "Herttoniemi", "cls": "Eastern Asia"}, {"id": 17214, "dom": "G", "ent": "Herttoniemi", "cls": "Eastern Europe"}, {"id": 17215, "dom": "G", "ent": "Herttoniemi", "cls": "Melanesia"}, {"id": 17216, "dom": "G", "ent": "Herttoniemi", "cls": "Micronesia"}, {"id": 17217, "dom": "G", "ent": "Herttoniemi", "cls": "Middle Africa"}, {"id": 17218, "dom": "G", "ent": "Herttoniemi", "cls": "Northern Africa"}, {"id": 17219, "dom": "G", "ent": "Herttoniemi", "cls": "Northern America"}, {"id": 17220, "dom": "G", "ent": "Herttoniemi", "cls": "Northern Europe"}, {"id": 17221, "dom": "G", "ent": "Herttoniemi", "cls": "Polynesia"}, {"id": 17222, "dom": "G", "ent": "Herttoniemi", "cls": "South America"}, {"id": 17223, "dom": "G", "ent": "Herttoniemi", "cls": "South-eastern Asia"}, {"id": 17224, "dom": "G", "ent": "Herttoniemi", "cls": "Southern Africa"}, {"id": 17225, "dom": "G", "ent": "Herttoniemi", "cls": "Southern Asia"}, {"id": 17226, "dom": "G", "ent": "Herttoniemi", "cls": "Southern Europe"}, {"id": 17227, "dom": "G", "ent": "Herttoniemi", "cls": "Western Africa"}, {"id": 17228, "dom": "G", "ent": "Herttoniemi", "cls": "Western Asia"}, {"id": 17229, "dom": "G", "ent": "Herttoniemi", "cls": "Western Europe"}, {"id": 17230, "dom": "G", "ent": "Raahe", "cls": "Australia and New Zealand"}, {"id": 17231, "dom": "G", "ent": "Raahe", "cls": "Caribbean"}, {"id": 17232, "dom": "G", "ent": "Raahe", "cls": "Central America"}, {"id": 17233, "dom": "G", "ent": "Raahe", "cls": "Central Asia"}, {"id": 17234, "dom": "G", "ent": "Raahe", "cls": "Eastern Africa"}, {"id": 17235, "dom": "G", "ent": "Raahe", "cls": "Eastern Asia"}, {"id": 17236, "dom": "G", "ent": "Raahe", "cls": "Eastern Europe"}, {"id": 17237, "dom": "G", "ent": "Raahe", "cls": "Melanesia"}, {"id": 17238, "dom": "G", "ent": "Raahe", "cls": "Micronesia"}, {"id": 17239, "dom": "G", "ent": "Raahe", "cls": "Middle Africa"}, {"id": 17240, "dom": "G", "ent": "Raahe", "cls": "Northern Africa"}, {"id": 17241, "dom": "G", "ent": "Raahe", "cls": "Northern America"}, {"id": 17242, "dom": "G", "ent": "Raahe", "cls": "Northern Europe"}, {"id": 17243, "dom": "G", "ent": "Raahe", "cls": "Polynesia"}, {"id": 17244, "dom": "G", "ent": "Raahe", "cls": "South America"}, {"id": 17245, "dom": "G", "ent": "Raahe", "cls": "South-eastern Asia"}, {"id": 17246, "dom": "G", "ent": "Raahe", "cls": "Southern Africa"}, {"id": 17247, "dom": "G", "ent": "Raahe", "cls": "Southern Asia"}, {"id": 17248, "dom": "G", "ent": "Raahe", "cls": "Southern Europe"}, {"id": 17249, "dom": "G", "ent": "Raahe", "cls": "Western Africa"}, {"id": 17250, "dom": "G", "ent": "Raahe", "cls": "Western Asia"}, {"id": 17251, "dom": "G", "ent": "Raahe", "cls": "Western Europe"}, {"id": 17252, "dom": "G", "ent": "Troyes", "cls": "Australia and New Zealand"}, {"id": 17253, "dom": "G", "ent": "Troyes", "cls": "Caribbean"}, {"id": 17254, "dom": "G", "ent": "Troyes", "cls": "Central America"}, {"id": 17255, "dom": "G", "ent": "Troyes", "cls": "Central Asia"}, {"id": 17256, "dom": "G", "ent": "Troyes", "cls": "Eastern Africa"}, {"id": 17257, "dom": "G", "ent": "Troyes", "cls": "Eastern Asia"}, {"id": 17258, "dom": "G", "ent": "Troyes", "cls": "Eastern Europe"}, {"id": 17259, "dom": "G", "ent": "Troyes", "cls": "Melanesia"}, {"id": 17260, "dom": "G", "ent": "Troyes", "cls": "Micronesia"}, {"id": 17261, "dom": "G", "ent": "Troyes", "cls": "Middle Africa"}, {"id": 17262, "dom": "G", "ent": "Troyes", "cls": "Northern Africa"}, {"id": 17263, "dom": "G", "ent": "Troyes", "cls": "Northern America"}, {"id": 17264, "dom": "G", "ent": "Troyes", "cls": "Northern Europe"}, {"id": 17265, "dom": "G", "ent": "Troyes", "cls": "Polynesia"}, {"id": 17266, "dom": "G", "ent": "Troyes", "cls": "South America"}, {"id": 17267, "dom": "G", "ent": "Troyes", "cls": "South-eastern Asia"}, {"id": 17268, "dom": "G", "ent": "Troyes", "cls": "Southern Africa"}, {"id": 17269, "dom": "G", "ent": "Troyes", "cls": "Southern Asia"}, {"id": 17270, "dom": "G", "ent": "Troyes", "cls": "Southern Europe"}, {"id": 17271, "dom": "G", "ent": "Troyes", "cls": "Western Africa"}, {"id": 17272, "dom": "G", "ent": "Troyes", "cls": "Western Asia"}, {"id": 17273, "dom": "G", "ent": "Troyes", "cls": "Western Europe"}, {"id": 17274, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "Australia and New Zealand"}, {"id": 17275, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "Caribbean"}, {"id": 17276, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "Central America"}, {"id": 17277, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "Central Asia"}, {"id": 17278, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "Eastern Africa"}, {"id": 17279, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "Eastern Asia"}, {"id": 17280, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "Eastern Europe"}, {"id": 17281, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "Melanesia"}, {"id": 17282, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "Micronesia"}, {"id": 17283, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "Middle Africa"}, {"id": 17284, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "Northern Africa"}, {"id": 17285, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "Northern America"}, {"id": 17286, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "Northern Europe"}, {"id": 17287, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "Polynesia"}, {"id": 17288, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "South America"}, {"id": 17289, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "South-eastern Asia"}, {"id": 17290, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "Southern Africa"}, {"id": 17291, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "Southern Asia"}, {"id": 17292, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "Southern Europe"}, {"id": 17293, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "Western Africa"}, {"id": 17294, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "Western Asia"}, {"id": 17295, "dom": "G", "ent": "Rosny-sous-Bois", "cls": "Western Europe"}, {"id": 17296, "dom": "G", "ent": "Savigny-le-Temple", "cls": "Australia and New Zealand"}, {"id": 17297, "dom": "G", "ent": "Savigny-le-Temple", "cls": "Caribbean"}, {"id": 17298, "dom": "G", "ent": "Savigny-le-Temple", "cls": "Central America"}, {"id": 17299, "dom": "G", "ent": "Savigny-le-Temple", "cls": "Central Asia"}, {"id": 17300, "dom": "G", "ent": "Savigny-le-Temple", "cls": "Eastern Africa"}, {"id": 17301, "dom": "G", "ent": "Savigny-le-Temple", "cls": "Eastern Asia"}, {"id": 17302, "dom": "G", "ent": "Savigny-le-Temple", "cls": "Eastern Europe"}, {"id": 17303, "dom": "G", "ent": "Savigny-le-Temple", "cls": "Melanesia"}, {"id": 17304, "dom": "G", "ent": "Savigny-le-Temple", "cls": "Micronesia"}, {"id": 17305, "dom": "G", "ent": "Savigny-le-Temple", "cls": "Middle Africa"}, {"id": 17306, "dom": "G", "ent": "Savigny-le-Temple", "cls": "Northern Africa"}, {"id": 17307, "dom": "G", "ent": "Savigny-le-Temple", "cls": "Northern America"}, {"id": 17308, "dom": "G", "ent": "Savigny-le-Temple", "cls": "Northern Europe"}, {"id": 17309, "dom": "G", "ent": "Savigny-le-Temple", "cls": "Polynesia"}, {"id": 17310, "dom": "G", "ent": "Savigny-le-Temple", "cls": "South America"}, {"id": 17311, "dom": "G", "ent": "Savigny-le-Temple", "cls": "South-eastern Asia"}, {"id": 17312, "dom": "G", "ent": "Savigny-le-Temple", "cls": "Southern Africa"}, {"id": 17313, "dom": "G", "ent": "Savigny-le-Temple", "cls": "Southern Asia"}, {"id": 17314, "dom": "G", "ent": "Savigny-le-Temple", "cls": "Southern Europe"}, {"id": 17315, "dom": "G", "ent": "Savigny-le-Temple", "cls": "Western Africa"}, {"id": 17316, "dom": "G", "ent": "Savigny-le-Temple", "cls": "Western Asia"}, {"id": 17317, "dom": "G", "ent": "Savigny-le-Temple", "cls": "Western Europe"}, {"id": 17318, "dom": "G", "ent": "Le Pontet", "cls": "Australia and New Zealand"}, {"id": 17319, "dom": "G", "ent": "Le Pontet", "cls": "Caribbean"}, {"id": 17320, "dom": "G", "ent": "Le Pontet", "cls": "Central America"}, {"id": 17321, "dom": "G", "ent": "Le Pontet", "cls": "Central Asia"}, {"id": 17322, "dom": "G", "ent": "Le Pontet", "cls": "Eastern Africa"}, {"id": 17323, "dom": "G", "ent": "Le Pontet", "cls": "Eastern Asia"}, {"id": 17324, "dom": "G", "ent": "Le Pontet", "cls": "Eastern Europe"}, {"id": 17325, "dom": "G", "ent": "Le Pontet", "cls": "Melanesia"}, {"id": 17326, "dom": "G", "ent": "Le Pontet", "cls": "Micronesia"}, {"id": 17327, "dom": "G", "ent": "Le Pontet", "cls": "Middle Africa"}, {"id": 17328, "dom": "G", "ent": "Le Pontet", "cls": "Northern Africa"}, {"id": 17329, "dom": "G", "ent": "Le Pontet", "cls": "Northern America"}, {"id": 17330, "dom": "G", "ent": "Le Pontet", "cls": "Northern Europe"}, {"id": 17331, "dom": "G", "ent": "Le Pontet", "cls": "Polynesia"}, {"id": 17332, "dom": "G", "ent": "Le Pontet", "cls": "South America"}, {"id": 17333, "dom": "G", "ent": "Le Pontet", "cls": "South-eastern Asia"}, {"id": 17334, "dom": "G", "ent": "Le Pontet", "cls": "Southern Africa"}, {"id": 17335, "dom": "G", "ent": "Le Pontet", "cls": "Southern Asia"}, {"id": 17336, "dom": "G", "ent": "Le Pontet", "cls": "Southern Europe"}, {"id": 17337, "dom": "G", "ent": "Le Pontet", "cls": "Western Africa"}, {"id": 17338, "dom": "G", "ent": "Le Pontet", "cls": "Western Asia"}, {"id": 17339, "dom": "G", "ent": "Le Pontet", "cls": "Western Europe"}, {"id": 17340, "dom": "G", "ent": "Oyem", "cls": "Australia and New Zealand"}, {"id": 17341, "dom": "G", "ent": "Oyem", "cls": "Caribbean"}, {"id": 17342, "dom": "G", "ent": "Oyem", "cls": "Central America"}, {"id": 17343, "dom": "G", "ent": "Oyem", "cls": "Central Asia"}, {"id": 17344, "dom": "G", "ent": "Oyem", "cls": "Eastern Africa"}, {"id": 17345, "dom": "G", "ent": "Oyem", "cls": "Eastern Asia"}, {"id": 17346, "dom": "G", "ent": "Oyem", "cls": "Eastern Europe"}, {"id": 17347, "dom": "G", "ent": "Oyem", "cls": "Melanesia"}, {"id": 17348, "dom": "G", "ent": "Oyem", "cls": "Micronesia"}, {"id": 17349, "dom": "G", "ent": "Oyem", "cls": "Middle Africa"}, {"id": 17350, "dom": "G", "ent": "Oyem", "cls": "Northern Africa"}, {"id": 17351, "dom": "G", "ent": "Oyem", "cls": "Northern America"}, {"id": 17352, "dom": "G", "ent": "Oyem", "cls": "Northern Europe"}, {"id": 17353, "dom": "G", "ent": "Oyem", "cls": "Polynesia"}, {"id": 17354, "dom": "G", "ent": "Oyem", "cls": "South America"}, {"id": 17355, "dom": "G", "ent": "Oyem", "cls": "South-eastern Asia"}, {"id": 17356, "dom": "G", "ent": "Oyem", "cls": "Southern Africa"}, {"id": 17357, "dom": "G", "ent": "Oyem", "cls": "Southern Asia"}, {"id": 17358, "dom": "G", "ent": "Oyem", "cls": "Southern Europe"}, {"id": 17359, "dom": "G", "ent": "Oyem", "cls": "Western Africa"}, {"id": 17360, "dom": "G", "ent": "Oyem", "cls": "Western Asia"}, {"id": 17361, "dom": "G", "ent": "Oyem", "cls": "Western Europe"}, {"id": 17362, "dom": "G", "ent": "Bitam", "cls": "Australia and New Zealand"}, {"id": 17363, "dom": "G", "ent": "Bitam", "cls": "Caribbean"}, {"id": 17364, "dom": "G", "ent": "Bitam", "cls": "Central America"}, {"id": 17365, "dom": "G", "ent": "Bitam", "cls": "Central Asia"}, {"id": 17366, "dom": "G", "ent": "Bitam", "cls": "Eastern Africa"}, {"id": 17367, "dom": "G", "ent": "Bitam", "cls": "Eastern Asia"}, {"id": 17368, "dom": "G", "ent": "Bitam", "cls": "Eastern Europe"}, {"id": 17369, "dom": "G", "ent": "Bitam", "cls": "Melanesia"}, {"id": 17370, "dom": "G", "ent": "Bitam", "cls": "Micronesia"}, {"id": 17371, "dom": "G", "ent": "Bitam", "cls": "Middle Africa"}, {"id": 17372, "dom": "G", "ent": "Bitam", "cls": "Northern Africa"}, {"id": 17373, "dom": "G", "ent": "Bitam", "cls": "Northern America"}, {"id": 17374, "dom": "G", "ent": "Bitam", "cls": "Northern Europe"}, {"id": 17375, "dom": "G", "ent": "Bitam", "cls": "Polynesia"}, {"id": 17376, "dom": "G", "ent": "Bitam", "cls": "South America"}, {"id": 17377, "dom": "G", "ent": "Bitam", "cls": "South-eastern Asia"}, {"id": 17378, "dom": "G", "ent": "Bitam", "cls": "Southern Africa"}, {"id": 17379, "dom": "G", "ent": "Bitam", "cls": "Southern Asia"}, {"id": 17380, "dom": "G", "ent": "Bitam", "cls": "Southern Europe"}, {"id": 17381, "dom": "G", "ent": "Bitam", "cls": "Western Africa"}, {"id": 17382, "dom": "G", "ent": "Bitam", "cls": "Western Asia"}, {"id": 17383, "dom": "G", "ent": "Bitam", "cls": "Western Europe"}, {"id": 17384, "dom": "G", "ent": "Oyam", "cls": "Australia and New Zealand"}, {"id": 17385, "dom": "G", "ent": "Oyam", "cls": "Caribbean"}, {"id": 17386, "dom": "G", "ent": "Oyam", "cls": "Central America"}, {"id": 17387, "dom": "G", "ent": "Oyam", "cls": "Central Asia"}, {"id": 17388, "dom": "G", "ent": "Oyam", "cls": "Eastern Africa"}, {"id": 17389, "dom": "G", "ent": "Oyam", "cls": "Eastern Asia"}, {"id": 17390, "dom": "G", "ent": "Oyam", "cls": "Eastern Europe"}, {"id": 17391, "dom": "G", "ent": "Oyam", "cls": "Melanesia"}, {"id": 17392, "dom": "G", "ent": "Oyam", "cls": "Micronesia"}, {"id": 17393, "dom": "G", "ent": "Oyam", "cls": "Middle Africa"}, {"id": 17394, "dom": "G", "ent": "Oyam", "cls": "Northern Africa"}, {"id": 17395, "dom": "G", "ent": "Oyam", "cls": "Northern America"}, {"id": 17396, "dom": "G", "ent": "Oyam", "cls": "Northern Europe"}, {"id": 17397, "dom": "G", "ent": "Oyam", "cls": "Polynesia"}, {"id": 17398, "dom": "G", "ent": "Oyam", "cls": "South America"}, {"id": 17399, "dom": "G", "ent": "Oyam", "cls": "South-eastern Asia"}, {"id": 17400, "dom": "G", "ent": "Oyam", "cls": "Southern Africa"}, {"id": 17401, "dom": "G", "ent": "Oyam", "cls": "Southern Asia"}, {"id": 17402, "dom": "G", "ent": "Oyam", "cls": "Southern Europe"}, {"id": 17403, "dom": "G", "ent": "Oyam", "cls": "Western Africa"}, {"id": 17404, "dom": "G", "ent": "Oyam", "cls": "Western Asia"}, {"id": 17405, "dom": "G", "ent": "Oyam", "cls": "Western Europe"}, {"id": 17406, "dom": "G", "ent": "Makokou", "cls": "Australia and New Zealand"}, {"id": 17407, "dom": "G", "ent": "Makokou", "cls": "Caribbean"}, {"id": 17408, "dom": "G", "ent": "Makokou", "cls": "Central America"}, {"id": 17409, "dom": "G", "ent": "Makokou", "cls": "Central Asia"}, {"id": 17410, "dom": "G", "ent": "Makokou", "cls": "Eastern Africa"}, {"id": 17411, "dom": "G", "ent": "Makokou", "cls": "Eastern Asia"}, {"id": 17412, "dom": "G", "ent": "Makokou", "cls": "Eastern Europe"}, {"id": 17413, "dom": "G", "ent": "Makokou", "cls": "Melanesia"}, {"id": 17414, "dom": "G", "ent": "Makokou", "cls": "Micronesia"}, {"id": 17415, "dom": "G", "ent": "Makokou", "cls": "Middle Africa"}, {"id": 17416, "dom": "G", "ent": "Makokou", "cls": "Northern Africa"}, {"id": 17417, "dom": "G", "ent": "Makokou", "cls": "Northern America"}, {"id": 17418, "dom": "G", "ent": "Makokou", "cls": "Northern Europe"}, {"id": 17419, "dom": "G", "ent": "Makokou", "cls": "Polynesia"}, {"id": 17420, "dom": "G", "ent": "Makokou", "cls": "South America"}, {"id": 17421, "dom": "G", "ent": "Makokou", "cls": "South-eastern Asia"}, {"id": 17422, "dom": "G", "ent": "Makokou", "cls": "Southern Africa"}, {"id": 17423, "dom": "G", "ent": "Makokou", "cls": "Southern Asia"}, {"id": 17424, "dom": "G", "ent": "Makokou", "cls": "Southern Europe"}, {"id": 17425, "dom": "G", "ent": "Makokou", "cls": "Western Africa"}, {"id": 17426, "dom": "G", "ent": "Makokou", "cls": "Western Asia"}, {"id": 17427, "dom": "G", "ent": "Makokou", "cls": "Western Europe"}, {"id": 17428, "dom": "G", "ent": "Bath", "cls": "Australia and New Zealand"}, {"id": 17429, "dom": "G", "ent": "Bath", "cls": "Caribbean"}, {"id": 17430, "dom": "G", "ent": "Bath", "cls": "Central America"}, {"id": 17431, "dom": "G", "ent": "Bath", "cls": "Central Asia"}, {"id": 17432, "dom": "G", "ent": "Bath", "cls": "Eastern Africa"}, {"id": 17433, "dom": "G", "ent": "Bath", "cls": "Eastern Asia"}, {"id": 17434, "dom": "G", "ent": "Bath", "cls": "Eastern Europe"}, {"id": 17435, "dom": "G", "ent": "Bath", "cls": "Melanesia"}, {"id": 17436, "dom": "G", "ent": "Bath", "cls": "Micronesia"}, {"id": 17437, "dom": "G", "ent": "Bath", "cls": "Middle Africa"}, {"id": 17438, "dom": "G", "ent": "Bath", "cls": "Northern Africa"}, {"id": 17439, "dom": "G", "ent": "Bath", "cls": "Northern America"}, {"id": 17440, "dom": "G", "ent": "Bath", "cls": "Northern Europe"}, {"id": 17441, "dom": "G", "ent": "Bath", "cls": "Polynesia"}, {"id": 17442, "dom": "G", "ent": "Bath", "cls": "South America"}, {"id": 17443, "dom": "G", "ent": "Bath", "cls": "South-eastern Asia"}, {"id": 17444, "dom": "G", "ent": "Bath", "cls": "Southern Africa"}, {"id": 17445, "dom": "G", "ent": "Bath", "cls": "Southern Asia"}, {"id": 17446, "dom": "G", "ent": "Bath", "cls": "Southern Europe"}, {"id": 17447, "dom": "G", "ent": "Bath", "cls": "Western Africa"}, {"id": 17448, "dom": "G", "ent": "Bath", "cls": "Western Asia"}, {"id": 17449, "dom": "G", "ent": "Bath", "cls": "Western Europe"}, {"id": 17450, "dom": "G", "ent": "Bicester", "cls": "Australia and New Zealand"}, {"id": 17451, "dom": "G", "ent": "Bicester", "cls": "Caribbean"}, {"id": 17452, "dom": "G", "ent": "Bicester", "cls": "Central America"}, {"id": 17453, "dom": "G", "ent": "Bicester", "cls": "Central Asia"}, {"id": 17454, "dom": "G", "ent": "Bicester", "cls": "Eastern Africa"}, {"id": 17455, "dom": "G", "ent": "Bicester", "cls": "Eastern Asia"}, {"id": 17456, "dom": "G", "ent": "Bicester", "cls": "Eastern Europe"}, {"id": 17457, "dom": "G", "ent": "Bicester", "cls": "Melanesia"}, {"id": 17458, "dom": "G", "ent": "Bicester", "cls": "Micronesia"}, {"id": 17459, "dom": "G", "ent": "Bicester", "cls": "Middle Africa"}, {"id": 17460, "dom": "G", "ent": "Bicester", "cls": "Northern Africa"}, {"id": 17461, "dom": "G", "ent": "Bicester", "cls": "Northern America"}, {"id": 17462, "dom": "G", "ent": "Bicester", "cls": "Northern Europe"}, {"id": 17463, "dom": "G", "ent": "Bicester", "cls": "Polynesia"}, {"id": 17464, "dom": "G", "ent": "Bicester", "cls": "South America"}, {"id": 17465, "dom": "G", "ent": "Bicester", "cls": "South-eastern Asia"}, {"id": 17466, "dom": "G", "ent": "Bicester", "cls": "Southern Africa"}, {"id": 17467, "dom": "G", "ent": "Bicester", "cls": "Southern Asia"}, {"id": 17468, "dom": "G", "ent": "Bicester", "cls": "Southern Europe"}, {"id": 17469, "dom": "G", "ent": "Bicester", "cls": "Western Africa"}, {"id": 17470, "dom": "G", "ent": "Bicester", "cls": "Western Asia"}, {"id": 17471, "dom": "G", "ent": "Bicester", "cls": "Western Europe"}, {"id": 17472, "dom": "G", "ent": "Bideford", "cls": "Australia and New Zealand"}, {"id": 17473, "dom": "G", "ent": "Bideford", "cls": "Caribbean"}, {"id": 17474, "dom": "G", "ent": "Bideford", "cls": "Central America"}, {"id": 17475, "dom": "G", "ent": "Bideford", "cls": "Central Asia"}, {"id": 17476, "dom": "G", "ent": "Bideford", "cls": "Eastern Africa"}, {"id": 17477, "dom": "G", "ent": "Bideford", "cls": "Eastern Asia"}, {"id": 17478, "dom": "G", "ent": "Bideford", "cls": "Eastern Europe"}, {"id": 17479, "dom": "G", "ent": "Bideford", "cls": "Melanesia"}, {"id": 17480, "dom": "G", "ent": "Bideford", "cls": "Micronesia"}, {"id": 17481, "dom": "G", "ent": "Bideford", "cls": "Middle Africa"}, {"id": 17482, "dom": "G", "ent": "Bideford", "cls": "Northern Africa"}, {"id": 17483, "dom": "G", "ent": "Bideford", "cls": "Northern America"}, {"id": 17484, "dom": "G", "ent": "Bideford", "cls": "Northern Europe"}, {"id": 17485, "dom": "G", "ent": "Bideford", "cls": "Polynesia"}, {"id": 17486, "dom": "G", "ent": "Bideford", "cls": "South America"}, {"id": 17487, "dom": "G", "ent": "Bideford", "cls": "South-eastern Asia"}, {"id": 17488, "dom": "G", "ent": "Bideford", "cls": "Southern Africa"}, {"id": 17489, "dom": "G", "ent": "Bideford", "cls": "Southern Asia"}, {"id": 17490, "dom": "G", "ent": "Bideford", "cls": "Southern Europe"}, {"id": 17491, "dom": "G", "ent": "Bideford", "cls": "Western Africa"}, {"id": 17492, "dom": "G", "ent": "Bideford", "cls": "Western Asia"}, {"id": 17493, "dom": "G", "ent": "Bideford", "cls": "Western Europe"}, {"id": 17494, "dom": "G", "ent": "Belle Vale", "cls": "Australia and New Zealand"}, {"id": 17495, "dom": "G", "ent": "Belle Vale", "cls": "Caribbean"}, {"id": 17496, "dom": "G", "ent": "Belle Vale", "cls": "Central America"}, {"id": 17497, "dom": "G", "ent": "Belle Vale", "cls": "Central Asia"}, {"id": 17498, "dom": "G", "ent": "Belle Vale", "cls": "Eastern Africa"}, {"id": 17499, "dom": "G", "ent": "Belle Vale", "cls": "Eastern Asia"}, {"id": 17500, "dom": "G", "ent": "Belle Vale", "cls": "Eastern Europe"}, {"id": 17501, "dom": "G", "ent": "Belle Vale", "cls": "Melanesia"}, {"id": 17502, "dom": "G", "ent": "Belle Vale", "cls": "Micronesia"}, {"id": 17503, "dom": "G", "ent": "Belle Vale", "cls": "Middle Africa"}, {"id": 17504, "dom": "G", "ent": "Belle Vale", "cls": "Northern Africa"}, {"id": 17505, "dom": "G", "ent": "Belle Vale", "cls": "Northern America"}, {"id": 17506, "dom": "G", "ent": "Belle Vale", "cls": "Northern Europe"}, {"id": 17507, "dom": "G", "ent": "Belle Vale", "cls": "Polynesia"}, {"id": 17508, "dom": "G", "ent": "Belle Vale", "cls": "South America"}, {"id": 17509, "dom": "G", "ent": "Belle Vale", "cls": "South-eastern Asia"}, {"id": 17510, "dom": "G", "ent": "Belle Vale", "cls": "Southern Africa"}, {"id": 17511, "dom": "G", "ent": "Belle Vale", "cls": "Southern Asia"}, {"id": 17512, "dom": "G", "ent": "Belle Vale", "cls": "Southern Europe"}, {"id": 17513, "dom": "G", "ent": "Belle Vale", "cls": "Western Africa"}, {"id": 17514, "dom": "G", "ent": "Belle Vale", "cls": "Western Asia"}, {"id": 17515, "dom": "G", "ent": "Belle Vale", "cls": "Western Europe"}, {"id": 17516, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "Australia and New Zealand"}, {"id": 17517, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "Caribbean"}, {"id": 17518, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "Central America"}, {"id": 17519, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "Central Asia"}, {"id": 17520, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "Eastern Africa"}, {"id": 17521, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "Eastern Asia"}, {"id": 17522, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "Eastern Europe"}, {"id": 17523, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "Melanesia"}, {"id": 17524, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "Micronesia"}, {"id": 17525, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "Middle Africa"}, {"id": 17526, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "Northern Africa"}, {"id": 17527, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "Northern America"}, {"id": 17528, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "Northern Europe"}, {"id": 17529, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "Polynesia"}, {"id": 17530, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "South America"}, {"id": 17531, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "South-eastern Asia"}, {"id": 17532, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "Southern Africa"}, {"id": 17533, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "Southern Asia"}, {"id": 17534, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "Southern Europe"}, {"id": 17535, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "Western Africa"}, {"id": 17536, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "Western Asia"}, {"id": 17537, "dom": "G", "ent": "Sekondi-Takoradi", "cls": "Western Europe"}, {"id": 17538, "dom": "G", "ent": "Kade", "cls": "Australia and New Zealand"}, {"id": 17539, "dom": "G", "ent": "Kade", "cls": "Caribbean"}, {"id": 17540, "dom": "G", "ent": "Kade", "cls": "Central America"}, {"id": 17541, "dom": "G", "ent": "Kade", "cls": "Central Asia"}, {"id": 17542, "dom": "G", "ent": "Kade", "cls": "Eastern Africa"}, {"id": 17543, "dom": "G", "ent": "Kade", "cls": "Eastern Asia"}, {"id": 17544, "dom": "G", "ent": "Kade", "cls": "Eastern Europe"}, {"id": 17545, "dom": "G", "ent": "Kade", "cls": "Melanesia"}, {"id": 17546, "dom": "G", "ent": "Kade", "cls": "Micronesia"}, {"id": 17547, "dom": "G", "ent": "Kade", "cls": "Middle Africa"}, {"id": 17548, "dom": "G", "ent": "Kade", "cls": "Northern Africa"}, {"id": 17549, "dom": "G", "ent": "Kade", "cls": "Northern America"}, {"id": 17550, "dom": "G", "ent": "Kade", "cls": "Northern Europe"}, {"id": 17551, "dom": "G", "ent": "Kade", "cls": "Polynesia"}, {"id": 17552, "dom": "G", "ent": "Kade", "cls": "South America"}, {"id": 17553, "dom": "G", "ent": "Kade", "cls": "South-eastern Asia"}, {"id": 17554, "dom": "G", "ent": "Kade", "cls": "Southern Africa"}, {"id": 17555, "dom": "G", "ent": "Kade", "cls": "Southern Asia"}, {"id": 17556, "dom": "G", "ent": "Kade", "cls": "Southern Europe"}, {"id": 17557, "dom": "G", "ent": "Kade", "cls": "Western Africa"}, {"id": 17558, "dom": "G", "ent": "Kade", "cls": "Western Asia"}, {"id": 17559, "dom": "G", "ent": "Kade", "cls": "Western Europe"}, {"id": 17560, "dom": "G", "ent": "Begoro", "cls": "Australia and New Zealand"}, {"id": 17561, "dom": "G", "ent": "Begoro", "cls": "Caribbean"}, {"id": 17562, "dom": "G", "ent": "Begoro", "cls": "Central America"}, {"id": 17563, "dom": "G", "ent": "Begoro", "cls": "Central Asia"}, {"id": 17564, "dom": "G", "ent": "Begoro", "cls": "Eastern Africa"}, {"id": 17565, "dom": "G", "ent": "Begoro", "cls": "Eastern Asia"}, {"id": 17566, "dom": "G", "ent": "Begoro", "cls": "Eastern Europe"}, {"id": 17567, "dom": "G", "ent": "Begoro", "cls": "Melanesia"}, {"id": 17568, "dom": "G", "ent": "Begoro", "cls": "Micronesia"}, {"id": 17569, "dom": "G", "ent": "Begoro", "cls": "Middle Africa"}, {"id": 17570, "dom": "G", "ent": "Begoro", "cls": "Northern Africa"}, {"id": 17571, "dom": "G", "ent": "Begoro", "cls": "Northern America"}, {"id": 17572, "dom": "G", "ent": "Begoro", "cls": "Northern Europe"}, {"id": 17573, "dom": "G", "ent": "Begoro", "cls": "Polynesia"}, {"id": 17574, "dom": "G", "ent": "Begoro", "cls": "South America"}, {"id": 17575, "dom": "G", "ent": "Begoro", "cls": "South-eastern Asia"}, {"id": 17576, "dom": "G", "ent": "Begoro", "cls": "Southern Africa"}, {"id": 17577, "dom": "G", "ent": "Begoro", "cls": "Southern Asia"}, {"id": 17578, "dom": "G", "ent": "Begoro", "cls": "Southern Europe"}, {"id": 17579, "dom": "G", "ent": "Begoro", "cls": "Western Africa"}, {"id": 17580, "dom": "G", "ent": "Begoro", "cls": "Western Asia"}, {"id": 17581, "dom": "G", "ent": "Begoro", "cls": "Western Europe"}, {"id": 17582, "dom": "G", "ent": "Nalerigu", "cls": "Australia and New Zealand"}, {"id": 17583, "dom": "G", "ent": "Nalerigu", "cls": "Caribbean"}, {"id": 17584, "dom": "G", "ent": "Nalerigu", "cls": "Central America"}, {"id": 17585, "dom": "G", "ent": "Nalerigu", "cls": "Central Asia"}, {"id": 17586, "dom": "G", "ent": "Nalerigu", "cls": "Eastern Africa"}, {"id": 17587, "dom": "G", "ent": "Nalerigu", "cls": "Eastern Asia"}, {"id": 17588, "dom": "G", "ent": "Nalerigu", "cls": "Eastern Europe"}, {"id": 17589, "dom": "G", "ent": "Nalerigu", "cls": "Melanesia"}, {"id": 17590, "dom": "G", "ent": "Nalerigu", "cls": "Micronesia"}, {"id": 17591, "dom": "G", "ent": "Nalerigu", "cls": "Middle Africa"}, {"id": 17592, "dom": "G", "ent": "Nalerigu", "cls": "Northern Africa"}, {"id": 17593, "dom": "G", "ent": "Nalerigu", "cls": "Northern America"}, {"id": 17594, "dom": "G", "ent": "Nalerigu", "cls": "Northern Europe"}, {"id": 17595, "dom": "G", "ent": "Nalerigu", "cls": "Polynesia"}, {"id": 17596, "dom": "G", "ent": "Nalerigu", "cls": "South America"}, {"id": 17597, "dom": "G", "ent": "Nalerigu", "cls": "South-eastern Asia"}, {"id": 17598, "dom": "G", "ent": "Nalerigu", "cls": "Southern Africa"}, {"id": 17599, "dom": "G", "ent": "Nalerigu", "cls": "Southern Asia"}, {"id": 17600, "dom": "G", "ent": "Nalerigu", "cls": "Southern Europe"}, {"id": 17601, "dom": "G", "ent": "Nalerigu", "cls": "Western Africa"}, {"id": 17602, "dom": "G", "ent": "Nalerigu", "cls": "Western Asia"}, {"id": 17603, "dom": "G", "ent": "Nalerigu", "cls": "Western Europe"}, {"id": 17604, "dom": "G", "ent": "Nema Kunku", "cls": "Australia and New Zealand"}, {"id": 17605, "dom": "G", "ent": "Nema Kunku", "cls": "Caribbean"}, {"id": 17606, "dom": "G", "ent": "Nema Kunku", "cls": "Central America"}, {"id": 17607, "dom": "G", "ent": "Nema Kunku", "cls": "Central Asia"}, {"id": 17608, "dom": "G", "ent": "Nema Kunku", "cls": "Eastern Africa"}, {"id": 17609, "dom": "G", "ent": "Nema Kunku", "cls": "Eastern Asia"}, {"id": 17610, "dom": "G", "ent": "Nema Kunku", "cls": "Eastern Europe"}, {"id": 17611, "dom": "G", "ent": "Nema Kunku", "cls": "Melanesia"}, {"id": 17612, "dom": "G", "ent": "Nema Kunku", "cls": "Micronesia"}, {"id": 17613, "dom": "G", "ent": "Nema Kunku", "cls": "Middle Africa"}, {"id": 17614, "dom": "G", "ent": "Nema Kunku", "cls": "Northern Africa"}, {"id": 17615, "dom": "G", "ent": "Nema Kunku", "cls": "Northern America"}, {"id": 17616, "dom": "G", "ent": "Nema Kunku", "cls": "Northern Europe"}, {"id": 17617, "dom": "G", "ent": "Nema Kunku", "cls": "Polynesia"}, {"id": 17618, "dom": "G", "ent": "Nema Kunku", "cls": "South America"}, {"id": 17619, "dom": "G", "ent": "Nema Kunku", "cls": "South-eastern Asia"}, {"id": 17620, "dom": "G", "ent": "Nema Kunku", "cls": "Southern Africa"}, {"id": 17621, "dom": "G", "ent": "Nema Kunku", "cls": "Southern Asia"}, {"id": 17622, "dom": "G", "ent": "Nema Kunku", "cls": "Southern Europe"}, {"id": 17623, "dom": "G", "ent": "Nema Kunku", "cls": "Western Africa"}, {"id": 17624, "dom": "G", "ent": "Nema Kunku", "cls": "Western Asia"}, {"id": 17625, "dom": "G", "ent": "Nema Kunku", "cls": "Western Europe"}, {"id": 17626, "dom": "G", "ent": "Lamin", "cls": "Australia and New Zealand"}, {"id": 17627, "dom": "G", "ent": "Lamin", "cls": "Caribbean"}, {"id": 17628, "dom": "G", "ent": "Lamin", "cls": "Central America"}, {"id": 17629, "dom": "G", "ent": "Lamin", "cls": "Central Asia"}, {"id": 17630, "dom": "G", "ent": "Lamin", "cls": "Eastern Africa"}, {"id": 17631, "dom": "G", "ent": "Lamin", "cls": "Eastern Asia"}, {"id": 17632, "dom": "G", "ent": "Lamin", "cls": "Eastern Europe"}, {"id": 17633, "dom": "G", "ent": "Lamin", "cls": "Melanesia"}, {"id": 17634, "dom": "G", "ent": "Lamin", "cls": "Micronesia"}, {"id": 17635, "dom": "G", "ent": "Lamin", "cls": "Middle Africa"}, {"id": 17636, "dom": "G", "ent": "Lamin", "cls": "Northern Africa"}, {"id": 17637, "dom": "G", "ent": "Lamin", "cls": "Northern America"}, {"id": 17638, "dom": "G", "ent": "Lamin", "cls": "Northern Europe"}, {"id": 17639, "dom": "G", "ent": "Lamin", "cls": "Polynesia"}, {"id": 17640, "dom": "G", "ent": "Lamin", "cls": "South America"}, {"id": 17641, "dom": "G", "ent": "Lamin", "cls": "South-eastern Asia"}, {"id": 17642, "dom": "G", "ent": "Lamin", "cls": "Southern Africa"}, {"id": 17643, "dom": "G", "ent": "Lamin", "cls": "Southern Asia"}, {"id": 17644, "dom": "G", "ent": "Lamin", "cls": "Southern Europe"}, {"id": 17645, "dom": "G", "ent": "Lamin", "cls": "Western Africa"}, {"id": 17646, "dom": "G", "ent": "Lamin", "cls": "Western Asia"}, {"id": 17647, "dom": "G", "ent": "Lamin", "cls": "Western Europe"}, {"id": 17648, "dom": "G", "ent": "Kotu", "cls": "Australia and New Zealand"}, {"id": 17649, "dom": "G", "ent": "Kotu", "cls": "Caribbean"}, {"id": 17650, "dom": "G", "ent": "Kotu", "cls": "Central America"}, {"id": 17651, "dom": "G", "ent": "Kotu", "cls": "Central Asia"}, {"id": 17652, "dom": "G", "ent": "Kotu", "cls": "Eastern Africa"}, {"id": 17653, "dom": "G", "ent": "Kotu", "cls": "Eastern Asia"}, {"id": 17654, "dom": "G", "ent": "Kotu", "cls": "Eastern Europe"}, {"id": 17655, "dom": "G", "ent": "Kotu", "cls": "Melanesia"}, {"id": 17656, "dom": "G", "ent": "Kotu", "cls": "Micronesia"}, {"id": 17657, "dom": "G", "ent": "Kotu", "cls": "Middle Africa"}, {"id": 17658, "dom": "G", "ent": "Kotu", "cls": "Northern Africa"}, {"id": 17659, "dom": "G", "ent": "Kotu", "cls": "Northern America"}, {"id": 17660, "dom": "G", "ent": "Kotu", "cls": "Northern Europe"}, {"id": 17661, "dom": "G", "ent": "Kotu", "cls": "Polynesia"}, {"id": 17662, "dom": "G", "ent": "Kotu", "cls": "South America"}, {"id": 17663, "dom": "G", "ent": "Kotu", "cls": "South-eastern Asia"}, {"id": 17664, "dom": "G", "ent": "Kotu", "cls": "Southern Africa"}, {"id": 17665, "dom": "G", "ent": "Kotu", "cls": "Southern Asia"}, {"id": 17666, "dom": "G", "ent": "Kotu", "cls": "Southern Europe"}, {"id": 17667, "dom": "G", "ent": "Kotu", "cls": "Western Africa"}, {"id": 17668, "dom": "G", "ent": "Kotu", "cls": "Western Asia"}, {"id": 17669, "dom": "G", "ent": "Kotu", "cls": "Western Europe"}, {"id": 17670, "dom": "G", "ent": "Tanji", "cls": "Australia and New Zealand"}, {"id": 17671, "dom": "G", "ent": "Tanji", "cls": "Caribbean"}, {"id": 17672, "dom": "G", "ent": "Tanji", "cls": "Central America"}, {"id": 17673, "dom": "G", "ent": "Tanji", "cls": "Central Asia"}, {"id": 17674, "dom": "G", "ent": "Tanji", "cls": "Eastern Africa"}, {"id": 17675, "dom": "G", "ent": "Tanji", "cls": "Eastern Asia"}, {"id": 17676, "dom": "G", "ent": "Tanji", "cls": "Eastern Europe"}, {"id": 17677, "dom": "G", "ent": "Tanji", "cls": "Melanesia"}, {"id": 17678, "dom": "G", "ent": "Tanji", "cls": "Micronesia"}, {"id": 17679, "dom": "G", "ent": "Tanji", "cls": "Middle Africa"}, {"id": 17680, "dom": "G", "ent": "Tanji", "cls": "Northern Africa"}, {"id": 17681, "dom": "G", "ent": "Tanji", "cls": "Northern America"}, {"id": 17682, "dom": "G", "ent": "Tanji", "cls": "Northern Europe"}, {"id": 17683, "dom": "G", "ent": "Tanji", "cls": "Polynesia"}, {"id": 17684, "dom": "G", "ent": "Tanji", "cls": "South America"}, {"id": 17685, "dom": "G", "ent": "Tanji", "cls": "South-eastern Asia"}, {"id": 17686, "dom": "G", "ent": "Tanji", "cls": "Southern Africa"}, {"id": 17687, "dom": "G", "ent": "Tanji", "cls": "Southern Asia"}, {"id": 17688, "dom": "G", "ent": "Tanji", "cls": "Southern Europe"}, {"id": 17689, "dom": "G", "ent": "Tanji", "cls": "Western Africa"}, {"id": 17690, "dom": "G", "ent": "Tanji", "cls": "Western Asia"}, {"id": 17691, "dom": "G", "ent": "Tanji", "cls": "Western Europe"}, {"id": 17692, "dom": "G", "ent": "Camayenne", "cls": "Australia and New Zealand"}, {"id": 17693, "dom": "G", "ent": "Camayenne", "cls": "Caribbean"}, {"id": 17694, "dom": "G", "ent": "Camayenne", "cls": "Central America"}, {"id": 17695, "dom": "G", "ent": "Camayenne", "cls": "Central Asia"}, {"id": 17696, "dom": "G", "ent": "Camayenne", "cls": "Eastern Africa"}, {"id": 17697, "dom": "G", "ent": "Camayenne", "cls": "Eastern Asia"}, {"id": 17698, "dom": "G", "ent": "Camayenne", "cls": "Eastern Europe"}, {"id": 17699, "dom": "G", "ent": "Camayenne", "cls": "Melanesia"}, {"id": 17700, "dom": "G", "ent": "Camayenne", "cls": "Micronesia"}, {"id": 17701, "dom": "G", "ent": "Camayenne", "cls": "Middle Africa"}, {"id": 17702, "dom": "G", "ent": "Camayenne", "cls": "Northern Africa"}, {"id": 17703, "dom": "G", "ent": "Camayenne", "cls": "Northern America"}, {"id": 17704, "dom": "G", "ent": "Camayenne", "cls": "Northern Europe"}, {"id": 17705, "dom": "G", "ent": "Camayenne", "cls": "Polynesia"}, {"id": 17706, "dom": "G", "ent": "Camayenne", "cls": "South America"}, {"id": 17707, "dom": "G", "ent": "Camayenne", "cls": "South-eastern Asia"}, {"id": 17708, "dom": "G", "ent": "Camayenne", "cls": "Southern Africa"}, {"id": 17709, "dom": "G", "ent": "Camayenne", "cls": "Southern Asia"}, {"id": 17710, "dom": "G", "ent": "Camayenne", "cls": "Southern Europe"}, {"id": 17711, "dom": "G", "ent": "Camayenne", "cls": "Western Africa"}, {"id": 17712, "dom": "G", "ent": "Camayenne", "cls": "Western Asia"}, {"id": 17713, "dom": "G", "ent": "Camayenne", "cls": "Western Europe"}, {"id": 17714, "dom": "G", "ent": "Fria", "cls": "Australia and New Zealand"}, {"id": 17715, "dom": "G", "ent": "Fria", "cls": "Caribbean"}, {"id": 17716, "dom": "G", "ent": "Fria", "cls": "Central America"}, {"id": 17717, "dom": "G", "ent": "Fria", "cls": "Central Asia"}, {"id": 17718, "dom": "G", "ent": "Fria", "cls": "Eastern Africa"}, {"id": 17719, "dom": "G", "ent": "Fria", "cls": "Eastern Asia"}, {"id": 17720, "dom": "G", "ent": "Fria", "cls": "Eastern Europe"}, {"id": 17721, "dom": "G", "ent": "Fria", "cls": "Melanesia"}, {"id": 17722, "dom": "G", "ent": "Fria", "cls": "Micronesia"}, {"id": 17723, "dom": "G", "ent": "Fria", "cls": "Middle Africa"}, {"id": 17724, "dom": "G", "ent": "Fria", "cls": "Northern Africa"}, {"id": 17725, "dom": "G", "ent": "Fria", "cls": "Northern America"}, {"id": 17726, "dom": "G", "ent": "Fria", "cls": "Northern Europe"}, {"id": 17727, "dom": "G", "ent": "Fria", "cls": "Polynesia"}, {"id": 17728, "dom": "G", "ent": "Fria", "cls": "South America"}, {"id": 17729, "dom": "G", "ent": "Fria", "cls": "South-eastern Asia"}, {"id": 17730, "dom": "G", "ent": "Fria", "cls": "Southern Africa"}, {"id": 17731, "dom": "G", "ent": "Fria", "cls": "Southern Asia"}, {"id": 17732, "dom": "G", "ent": "Fria", "cls": "Southern Europe"}, {"id": 17733, "dom": "G", "ent": "Fria", "cls": "Western Africa"}, {"id": 17734, "dom": "G", "ent": "Fria", "cls": "Western Asia"}, {"id": 17735, "dom": "G", "ent": "Fria", "cls": "Western Europe"}, {"id": 17736, "dom": "G", "ent": "Mandiana", "cls": "Australia and New Zealand"}, {"id": 17737, "dom": "G", "ent": "Mandiana", "cls": "Caribbean"}, {"id": 17738, "dom": "G", "ent": "Mandiana", "cls": "Central America"}, {"id": 17739, "dom": "G", "ent": "Mandiana", "cls": "Central Asia"}, {"id": 17740, "dom": "G", "ent": "Mandiana", "cls": "Eastern Africa"}, {"id": 17741, "dom": "G", "ent": "Mandiana", "cls": "Eastern Asia"}, {"id": 17742, "dom": "G", "ent": "Mandiana", "cls": "Eastern Europe"}, {"id": 17743, "dom": "G", "ent": "Mandiana", "cls": "Melanesia"}, {"id": 17744, "dom": "G", "ent": "Mandiana", "cls": "Micronesia"}, {"id": 17745, "dom": "G", "ent": "Mandiana", "cls": "Middle Africa"}, {"id": 17746, "dom": "G", "ent": "Mandiana", "cls": "Northern Africa"}, {"id": 17747, "dom": "G", "ent": "Mandiana", "cls": "Northern America"}, {"id": 17748, "dom": "G", "ent": "Mandiana", "cls": "Northern Europe"}, {"id": 17749, "dom": "G", "ent": "Mandiana", "cls": "Polynesia"}, {"id": 17750, "dom": "G", "ent": "Mandiana", "cls": "South America"}, {"id": 17751, "dom": "G", "ent": "Mandiana", "cls": "South-eastern Asia"}, {"id": 17752, "dom": "G", "ent": "Mandiana", "cls": "Southern Africa"}, {"id": 17753, "dom": "G", "ent": "Mandiana", "cls": "Southern Asia"}, {"id": 17754, "dom": "G", "ent": "Mandiana", "cls": "Southern Europe"}, {"id": 17755, "dom": "G", "ent": "Mandiana", "cls": "Western Africa"}, {"id": 17756, "dom": "G", "ent": "Mandiana", "cls": "Western Asia"}, {"id": 17757, "dom": "G", "ent": "Mandiana", "cls": "Western Europe"}, {"id": 17758, "dom": "G", "ent": "Moussadou", "cls": "Australia and New Zealand"}, {"id": 17759, "dom": "G", "ent": "Moussadou", "cls": "Caribbean"}, {"id": 17760, "dom": "G", "ent": "Moussadou", "cls": "Central America"}, {"id": 17761, "dom": "G", "ent": "Moussadou", "cls": "Central Asia"}, {"id": 17762, "dom": "G", "ent": "Moussadou", "cls": "Eastern Africa"}, {"id": 17763, "dom": "G", "ent": "Moussadou", "cls": "Eastern Asia"}, {"id": 17764, "dom": "G", "ent": "Moussadou", "cls": "Eastern Europe"}, {"id": 17765, "dom": "G", "ent": "Moussadou", "cls": "Melanesia"}, {"id": 17766, "dom": "G", "ent": "Moussadou", "cls": "Micronesia"}, {"id": 17767, "dom": "G", "ent": "Moussadou", "cls": "Middle Africa"}, {"id": 17768, "dom": "G", "ent": "Moussadou", "cls": "Northern Africa"}, {"id": 17769, "dom": "G", "ent": "Moussadou", "cls": "Northern America"}, {"id": 17770, "dom": "G", "ent": "Moussadou", "cls": "Northern Europe"}, {"id": 17771, "dom": "G", "ent": "Moussadou", "cls": "Polynesia"}, {"id": 17772, "dom": "G", "ent": "Moussadou", "cls": "South America"}, {"id": 17773, "dom": "G", "ent": "Moussadou", "cls": "South-eastern Asia"}, {"id": 17774, "dom": "G", "ent": "Moussadou", "cls": "Southern Africa"}, {"id": 17775, "dom": "G", "ent": "Moussadou", "cls": "Southern Asia"}, {"id": 17776, "dom": "G", "ent": "Moussadou", "cls": "Southern Europe"}, {"id": 17777, "dom": "G", "ent": "Moussadou", "cls": "Western Africa"}, {"id": 17778, "dom": "G", "ent": "Moussadou", "cls": "Western Asia"}, {"id": 17779, "dom": "G", "ent": "Moussadou", "cls": "Western Europe"}, {"id": 17780, "dom": "G", "ent": "Le Gosier", "cls": "Australia and New Zealand"}, {"id": 17781, "dom": "G", "ent": "Le Gosier", "cls": "Caribbean"}, {"id": 17782, "dom": "G", "ent": "Le Gosier", "cls": "Central America"}, {"id": 17783, "dom": "G", "ent": "Le Gosier", "cls": "Central Asia"}, {"id": 17784, "dom": "G", "ent": "Le Gosier", "cls": "Eastern Africa"}, {"id": 17785, "dom": "G", "ent": "Le Gosier", "cls": "Eastern Asia"}, {"id": 17786, "dom": "G", "ent": "Le Gosier", "cls": "Eastern Europe"}, {"id": 17787, "dom": "G", "ent": "Le Gosier", "cls": "Melanesia"}, {"id": 17788, "dom": "G", "ent": "Le Gosier", "cls": "Micronesia"}, {"id": 17789, "dom": "G", "ent": "Le Gosier", "cls": "Middle Africa"}, {"id": 17790, "dom": "G", "ent": "Le Gosier", "cls": "Northern Africa"}, {"id": 17791, "dom": "G", "ent": "Le Gosier", "cls": "Northern America"}, {"id": 17792, "dom": "G", "ent": "Le Gosier", "cls": "Northern Europe"}, {"id": 17793, "dom": "G", "ent": "Le Gosier", "cls": "Polynesia"}, {"id": 17794, "dom": "G", "ent": "Le Gosier", "cls": "South America"}, {"id": 17795, "dom": "G", "ent": "Le Gosier", "cls": "South-eastern Asia"}, {"id": 17796, "dom": "G", "ent": "Le Gosier", "cls": "Southern Africa"}, {"id": 17797, "dom": "G", "ent": "Le Gosier", "cls": "Southern Asia"}, {"id": 17798, "dom": "G", "ent": "Le Gosier", "cls": "Southern Europe"}, {"id": 17799, "dom": "G", "ent": "Le Gosier", "cls": "Western Africa"}, {"id": 17800, "dom": "G", "ent": "Le Gosier", "cls": "Western Asia"}, {"id": 17801, "dom": "G", "ent": "Le Gosier", "cls": "Western Europe"}, {"id": 17802, "dom": "G", "ent": "Petit-Bourg", "cls": "Australia and New Zealand"}, {"id": 17803, "dom": "G", "ent": "Petit-Bourg", "cls": "Caribbean"}, {"id": 17804, "dom": "G", "ent": "Petit-Bourg", "cls": "Central America"}, {"id": 17805, "dom": "G", "ent": "Petit-Bourg", "cls": "Central Asia"}, {"id": 17806, "dom": "G", "ent": "Petit-Bourg", "cls": "Eastern Africa"}, {"id": 17807, "dom": "G", "ent": "Petit-Bourg", "cls": "Eastern Asia"}, {"id": 17808, "dom": "G", "ent": "Petit-Bourg", "cls": "Eastern Europe"}, {"id": 17809, "dom": "G", "ent": "Petit-Bourg", "cls": "Melanesia"}, {"id": 17810, "dom": "G", "ent": "Petit-Bourg", "cls": "Micronesia"}, {"id": 17811, "dom": "G", "ent": "Petit-Bourg", "cls": "Middle Africa"}, {"id": 17812, "dom": "G", "ent": "Petit-Bourg", "cls": "Northern Africa"}, {"id": 17813, "dom": "G", "ent": "Petit-Bourg", "cls": "Northern America"}, {"id": 17814, "dom": "G", "ent": "Petit-Bourg", "cls": "Northern Europe"}, {"id": 17815, "dom": "G", "ent": "Petit-Bourg", "cls": "Polynesia"}, {"id": 17816, "dom": "G", "ent": "Petit-Bourg", "cls": "South America"}, {"id": 17817, "dom": "G", "ent": "Petit-Bourg", "cls": "South-eastern Asia"}, {"id": 17818, "dom": "G", "ent": "Petit-Bourg", "cls": "Southern Africa"}, {"id": 17819, "dom": "G", "ent": "Petit-Bourg", "cls": "Southern Asia"}, {"id": 17820, "dom": "G", "ent": "Petit-Bourg", "cls": "Southern Europe"}, {"id": 17821, "dom": "G", "ent": "Petit-Bourg", "cls": "Western Africa"}, {"id": 17822, "dom": "G", "ent": "Petit-Bourg", "cls": "Western Asia"}, {"id": 17823, "dom": "G", "ent": "Petit-Bourg", "cls": "Western Europe"}, {"id": 17824, "dom": "G", "ent": "Le Moule", "cls": "Australia and New Zealand"}, {"id": 17825, "dom": "G", "ent": "Le Moule", "cls": "Caribbean"}, {"id": 17826, "dom": "G", "ent": "Le Moule", "cls": "Central America"}, {"id": 17827, "dom": "G", "ent": "Le Moule", "cls": "Central Asia"}, {"id": 17828, "dom": "G", "ent": "Le Moule", "cls": "Eastern Africa"}, {"id": 17829, "dom": "G", "ent": "Le Moule", "cls": "Eastern Asia"}, {"id": 17830, "dom": "G", "ent": "Le Moule", "cls": "Eastern Europe"}, {"id": 17831, "dom": "G", "ent": "Le Moule", "cls": "Melanesia"}, {"id": 17832, "dom": "G", "ent": "Le Moule", "cls": "Micronesia"}, {"id": 17833, "dom": "G", "ent": "Le Moule", "cls": "Middle Africa"}, {"id": 17834, "dom": "G", "ent": "Le Moule", "cls": "Northern Africa"}, {"id": 17835, "dom": "G", "ent": "Le Moule", "cls": "Northern America"}, {"id": 17836, "dom": "G", "ent": "Le Moule", "cls": "Northern Europe"}, {"id": 17837, "dom": "G", "ent": "Le Moule", "cls": "Polynesia"}, {"id": 17838, "dom": "G", "ent": "Le Moule", "cls": "South America"}, {"id": 17839, "dom": "G", "ent": "Le Moule", "cls": "South-eastern Asia"}, {"id": 17840, "dom": "G", "ent": "Le Moule", "cls": "Southern Africa"}, {"id": 17841, "dom": "G", "ent": "Le Moule", "cls": "Southern Asia"}, {"id": 17842, "dom": "G", "ent": "Le Moule", "cls": "Southern Europe"}, {"id": 17843, "dom": "G", "ent": "Le Moule", "cls": "Western Africa"}, {"id": 17844, "dom": "G", "ent": "Le Moule", "cls": "Western Asia"}, {"id": 17845, "dom": "G", "ent": "Le Moule", "cls": "Western Europe"}, {"id": 17846, "dom": "G", "ent": "Basse-Terre", "cls": "Australia and New Zealand"}, {"id": 17847, "dom": "G", "ent": "Basse-Terre", "cls": "Caribbean"}, {"id": 17848, "dom": "G", "ent": "Basse-Terre", "cls": "Central America"}, {"id": 17849, "dom": "G", "ent": "Basse-Terre", "cls": "Central Asia"}, {"id": 17850, "dom": "G", "ent": "Basse-Terre", "cls": "Eastern Africa"}, {"id": 17851, "dom": "G", "ent": "Basse-Terre", "cls": "Eastern Asia"}, {"id": 17852, "dom": "G", "ent": "Basse-Terre", "cls": "Eastern Europe"}, {"id": 17853, "dom": "G", "ent": "Basse-Terre", "cls": "Melanesia"}, {"id": 17854, "dom": "G", "ent": "Basse-Terre", "cls": "Micronesia"}, {"id": 17855, "dom": "G", "ent": "Basse-Terre", "cls": "Middle Africa"}, {"id": 17856, "dom": "G", "ent": "Basse-Terre", "cls": "Northern Africa"}, {"id": 17857, "dom": "G", "ent": "Basse-Terre", "cls": "Northern America"}, {"id": 17858, "dom": "G", "ent": "Basse-Terre", "cls": "Northern Europe"}, {"id": 17859, "dom": "G", "ent": "Basse-Terre", "cls": "Polynesia"}, {"id": 17860, "dom": "G", "ent": "Basse-Terre", "cls": "South America"}, {"id": 17861, "dom": "G", "ent": "Basse-Terre", "cls": "South-eastern Asia"}, {"id": 17862, "dom": "G", "ent": "Basse-Terre", "cls": "Southern Africa"}, {"id": 17863, "dom": "G", "ent": "Basse-Terre", "cls": "Southern Asia"}, {"id": 17864, "dom": "G", "ent": "Basse-Terre", "cls": "Southern Europe"}, {"id": 17865, "dom": "G", "ent": "Basse-Terre", "cls": "Western Africa"}, {"id": 17866, "dom": "G", "ent": "Basse-Terre", "cls": "Western Asia"}, {"id": 17867, "dom": "G", "ent": "Basse-Terre", "cls": "Western Europe"}, {"id": 17868, "dom": "G", "ent": "Volos", "cls": "Australia and New Zealand"}, {"id": 17869, "dom": "G", "ent": "Volos", "cls": "Caribbean"}, {"id": 17870, "dom": "G", "ent": "Volos", "cls": "Central America"}, {"id": 17871, "dom": "G", "ent": "Volos", "cls": "Central Asia"}, {"id": 17872, "dom": "G", "ent": "Volos", "cls": "Eastern Africa"}, {"id": 17873, "dom": "G", "ent": "Volos", "cls": "Eastern Asia"}, {"id": 17874, "dom": "G", "ent": "Volos", "cls": "Eastern Europe"}, {"id": 17875, "dom": "G", "ent": "Volos", "cls": "Melanesia"}, {"id": 17876, "dom": "G", "ent": "Volos", "cls": "Micronesia"}, {"id": 17877, "dom": "G", "ent": "Volos", "cls": "Middle Africa"}, {"id": 17878, "dom": "G", "ent": "Volos", "cls": "Northern Africa"}, {"id": 17879, "dom": "G", "ent": "Volos", "cls": "Northern America"}, {"id": 17880, "dom": "G", "ent": "Volos", "cls": "Northern Europe"}, {"id": 17881, "dom": "G", "ent": "Volos", "cls": "Polynesia"}, {"id": 17882, "dom": "G", "ent": "Volos", "cls": "South America"}, {"id": 17883, "dom": "G", "ent": "Volos", "cls": "South-eastern Asia"}, {"id": 17884, "dom": "G", "ent": "Volos", "cls": "Southern Africa"}, {"id": 17885, "dom": "G", "ent": "Volos", "cls": "Southern Asia"}, {"id": 17886, "dom": "G", "ent": "Volos", "cls": "Southern Europe"}, {"id": 17887, "dom": "G", "ent": "Volos", "cls": "Western Africa"}, {"id": 17888, "dom": "G", "ent": "Volos", "cls": "Western Asia"}, {"id": 17889, "dom": "G", "ent": "Volos", "cls": "Western Europe"}, {"id": 17890, "dom": "G", "ent": "Agios Dimitrios", "cls": "Australia and New Zealand"}, {"id": 17891, "dom": "G", "ent": "Agios Dimitrios", "cls": "Caribbean"}, {"id": 17892, "dom": "G", "ent": "Agios Dimitrios", "cls": "Central America"}, {"id": 17893, "dom": "G", "ent": "Agios Dimitrios", "cls": "Central Asia"}, {"id": 17894, "dom": "G", "ent": "Agios Dimitrios", "cls": "Eastern Africa"}, {"id": 17895, "dom": "G", "ent": "Agios Dimitrios", "cls": "Eastern Asia"}, {"id": 17896, "dom": "G", "ent": "Agios Dimitrios", "cls": "Eastern Europe"}, {"id": 17897, "dom": "G", "ent": "Agios Dimitrios", "cls": "Melanesia"}, {"id": 17898, "dom": "G", "ent": "Agios Dimitrios", "cls": "Micronesia"}, {"id": 17899, "dom": "G", "ent": "Agios Dimitrios", "cls": "Middle Africa"}, {"id": 17900, "dom": "G", "ent": "Agios Dimitrios", "cls": "Northern Africa"}, {"id": 17901, "dom": "G", "ent": "Agios Dimitrios", "cls": "Northern America"}, {"id": 17902, "dom": "G", "ent": "Agios Dimitrios", "cls": "Northern Europe"}, {"id": 17903, "dom": "G", "ent": "Agios Dimitrios", "cls": "Polynesia"}, {"id": 17904, "dom": "G", "ent": "Agios Dimitrios", "cls": "South America"}, {"id": 17905, "dom": "G", "ent": "Agios Dimitrios", "cls": "South-eastern Asia"}, {"id": 17906, "dom": "G", "ent": "Agios Dimitrios", "cls": "Southern Africa"}, {"id": 17907, "dom": "G", "ent": "Agios Dimitrios", "cls": "Southern Asia"}, {"id": 17908, "dom": "G", "ent": "Agios Dimitrios", "cls": "Southern Europe"}, {"id": 17909, "dom": "G", "ent": "Agios Dimitrios", "cls": "Western Africa"}, {"id": 17910, "dom": "G", "ent": "Agios Dimitrios", "cls": "Western Asia"}, {"id": 17911, "dom": "G", "ent": "Agios Dimitrios", "cls": "Western Europe"}, {"id": 17912, "dom": "G", "ent": "Mytilene", "cls": "Australia and New Zealand"}, {"id": 17913, "dom": "G", "ent": "Mytilene", "cls": "Caribbean"}, {"id": 17914, "dom": "G", "ent": "Mytilene", "cls": "Central America"}, {"id": 17915, "dom": "G", "ent": "Mytilene", "cls": "Central Asia"}, {"id": 17916, "dom": "G", "ent": "Mytilene", "cls": "Eastern Africa"}, {"id": 17917, "dom": "G", "ent": "Mytilene", "cls": "Eastern Asia"}, {"id": 17918, "dom": "G", "ent": "Mytilene", "cls": "Eastern Europe"}, {"id": 17919, "dom": "G", "ent": "Mytilene", "cls": "Melanesia"}, {"id": 17920, "dom": "G", "ent": "Mytilene", "cls": "Micronesia"}, {"id": 17921, "dom": "G", "ent": "Mytilene", "cls": "Middle Africa"}, {"id": 17922, "dom": "G", "ent": "Mytilene", "cls": "Northern Africa"}, {"id": 17923, "dom": "G", "ent": "Mytilene", "cls": "Northern America"}, {"id": 17924, "dom": "G", "ent": "Mytilene", "cls": "Northern Europe"}, {"id": 17925, "dom": "G", "ent": "Mytilene", "cls": "Polynesia"}, {"id": 17926, "dom": "G", "ent": "Mytilene", "cls": "South America"}, {"id": 17927, "dom": "G", "ent": "Mytilene", "cls": "South-eastern Asia"}, {"id": 17928, "dom": "G", "ent": "Mytilene", "cls": "Southern Africa"}, {"id": 17929, "dom": "G", "ent": "Mytilene", "cls": "Southern Asia"}, {"id": 17930, "dom": "G", "ent": "Mytilene", "cls": "Southern Europe"}, {"id": 17931, "dom": "G", "ent": "Mytilene", "cls": "Western Africa"}, {"id": 17932, "dom": "G", "ent": "Mytilene", "cls": "Western Asia"}, {"id": 17933, "dom": "G", "ent": "Mytilene", "cls": "Western Europe"}, {"id": 17934, "dom": "G", "ent": "Neapoli", "cls": "Australia and New Zealand"}, {"id": 17935, "dom": "G", "ent": "Neapoli", "cls": "Caribbean"}, {"id": 17936, "dom": "G", "ent": "Neapoli", "cls": "Central America"}, {"id": 17937, "dom": "G", "ent": "Neapoli", "cls": "Central Asia"}, {"id": 17938, "dom": "G", "ent": "Neapoli", "cls": "Eastern Africa"}, {"id": 17939, "dom": "G", "ent": "Neapoli", "cls": "Eastern Asia"}, {"id": 17940, "dom": "G", "ent": "Neapoli", "cls": "Eastern Europe"}, {"id": 17941, "dom": "G", "ent": "Neapoli", "cls": "Melanesia"}, {"id": 17942, "dom": "G", "ent": "Neapoli", "cls": "Micronesia"}, {"id": 17943, "dom": "G", "ent": "Neapoli", "cls": "Middle Africa"}, {"id": 17944, "dom": "G", "ent": "Neapoli", "cls": "Northern Africa"}, {"id": 17945, "dom": "G", "ent": "Neapoli", "cls": "Northern America"}, {"id": 17946, "dom": "G", "ent": "Neapoli", "cls": "Northern Europe"}, {"id": 17947, "dom": "G", "ent": "Neapoli", "cls": "Polynesia"}, {"id": 17948, "dom": "G", "ent": "Neapoli", "cls": "South America"}, {"id": 17949, "dom": "G", "ent": "Neapoli", "cls": "South-eastern Asia"}, {"id": 17950, "dom": "G", "ent": "Neapoli", "cls": "Southern Africa"}, {"id": 17951, "dom": "G", "ent": "Neapoli", "cls": "Southern Asia"}, {"id": 17952, "dom": "G", "ent": "Neapoli", "cls": "Southern Europe"}, {"id": 17953, "dom": "G", "ent": "Neapoli", "cls": "Western Africa"}, {"id": 17954, "dom": "G", "ent": "Neapoli", "cls": "Western Asia"}, {"id": 17955, "dom": "G", "ent": "Neapoli", "cls": "Western Europe"}, {"id": 17956, "dom": "G", "ent": "Zacapa", "cls": "Australia and New Zealand"}, {"id": 17957, "dom": "G", "ent": "Zacapa", "cls": "Caribbean"}, {"id": 17958, "dom": "G", "ent": "Zacapa", "cls": "Central America"}, {"id": 17959, "dom": "G", "ent": "Zacapa", "cls": "Central Asia"}, {"id": 17960, "dom": "G", "ent": "Zacapa", "cls": "Eastern Africa"}, {"id": 17961, "dom": "G", "ent": "Zacapa", "cls": "Eastern Asia"}, {"id": 17962, "dom": "G", "ent": "Zacapa", "cls": "Eastern Europe"}, {"id": 17963, "dom": "G", "ent": "Zacapa", "cls": "Melanesia"}, {"id": 17964, "dom": "G", "ent": "Zacapa", "cls": "Micronesia"}, {"id": 17965, "dom": "G", "ent": "Zacapa", "cls": "Middle Africa"}, {"id": 17966, "dom": "G", "ent": "Zacapa", "cls": "Northern Africa"}, {"id": 17967, "dom": "G", "ent": "Zacapa", "cls": "Northern America"}, {"id": 17968, "dom": "G", "ent": "Zacapa", "cls": "Northern Europe"}, {"id": 17969, "dom": "G", "ent": "Zacapa", "cls": "Polynesia"}, {"id": 17970, "dom": "G", "ent": "Zacapa", "cls": "South America"}, {"id": 17971, "dom": "G", "ent": "Zacapa", "cls": "South-eastern Asia"}, {"id": 17972, "dom": "G", "ent": "Zacapa", "cls": "Southern Africa"}, {"id": 17973, "dom": "G", "ent": "Zacapa", "cls": "Southern Asia"}, {"id": 17974, "dom": "G", "ent": "Zacapa", "cls": "Southern Europe"}, {"id": 17975, "dom": "G", "ent": "Zacapa", "cls": "Western Africa"}, {"id": 17976, "dom": "G", "ent": "Zacapa", "cls": "Western Asia"}, {"id": 17977, "dom": "G", "ent": "Zacapa", "cls": "Western Europe"}, {"id": 17978, "dom": "G", "ent": "Yepocapa", "cls": "Australia and New Zealand"}, {"id": 17979, "dom": "G", "ent": "Yepocapa", "cls": "Caribbean"}, {"id": 17980, "dom": "G", "ent": "Yepocapa", "cls": "Central America"}, {"id": 17981, "dom": "G", "ent": "Yepocapa", "cls": "Central Asia"}, {"id": 17982, "dom": "G", "ent": "Yepocapa", "cls": "Eastern Africa"}, {"id": 17983, "dom": "G", "ent": "Yepocapa", "cls": "Eastern Asia"}, {"id": 17984, "dom": "G", "ent": "Yepocapa", "cls": "Eastern Europe"}, {"id": 17985, "dom": "G", "ent": "Yepocapa", "cls": "Melanesia"}, {"id": 17986, "dom": "G", "ent": "Yepocapa", "cls": "Micronesia"}, {"id": 17987, "dom": "G", "ent": "Yepocapa", "cls": "Middle Africa"}, {"id": 17988, "dom": "G", "ent": "Yepocapa", "cls": "Northern Africa"}, {"id": 17989, "dom": "G", "ent": "Yepocapa", "cls": "Northern America"}, {"id": 17990, "dom": "G", "ent": "Yepocapa", "cls": "Northern Europe"}, {"id": 17991, "dom": "G", "ent": "Yepocapa", "cls": "Polynesia"}, {"id": 17992, "dom": "G", "ent": "Yepocapa", "cls": "South America"}, {"id": 17993, "dom": "G", "ent": "Yepocapa", "cls": "South-eastern Asia"}, {"id": 17994, "dom": "G", "ent": "Yepocapa", "cls": "Southern Africa"}, {"id": 17995, "dom": "G", "ent": "Yepocapa", "cls": "Southern Asia"}, {"id": 17996, "dom": "G", "ent": "Yepocapa", "cls": "Southern Europe"}, {"id": 17997, "dom": "G", "ent": "Yepocapa", "cls": "Western Africa"}, {"id": 17998, "dom": "G", "ent": "Yepocapa", "cls": "Western Asia"}, {"id": 17999, "dom": "G", "ent": "Yepocapa", "cls": "Western Europe"}, {"id": 18000, "dom": "G", "ent": "Alotenango", "cls": "Australia and New Zealand"}, {"id": 18001, "dom": "G", "ent": "Alotenango", "cls": "Caribbean"}, {"id": 18002, "dom": "G", "ent": "Alotenango", "cls": "Central America"}, {"id": 18003, "dom": "G", "ent": "Alotenango", "cls": "Central Asia"}, {"id": 18004, "dom": "G", "ent": "Alotenango", "cls": "Eastern Africa"}, {"id": 18005, "dom": "G", "ent": "Alotenango", "cls": "Eastern Asia"}, {"id": 18006, "dom": "G", "ent": "Alotenango", "cls": "Eastern Europe"}, {"id": 18007, "dom": "G", "ent": "Alotenango", "cls": "Melanesia"}, {"id": 18008, "dom": "G", "ent": "Alotenango", "cls": "Micronesia"}, {"id": 18009, "dom": "G", "ent": "Alotenango", "cls": "Middle Africa"}, {"id": 18010, "dom": "G", "ent": "Alotenango", "cls": "Northern Africa"}, {"id": 18011, "dom": "G", "ent": "Alotenango", "cls": "Northern America"}, {"id": 18012, "dom": "G", "ent": "Alotenango", "cls": "Northern Europe"}, {"id": 18013, "dom": "G", "ent": "Alotenango", "cls": "Polynesia"}, {"id": 18014, "dom": "G", "ent": "Alotenango", "cls": "South America"}, {"id": 18015, "dom": "G", "ent": "Alotenango", "cls": "South-eastern Asia"}, {"id": 18016, "dom": "G", "ent": "Alotenango", "cls": "Southern Africa"}, {"id": 18017, "dom": "G", "ent": "Alotenango", "cls": "Southern Asia"}, {"id": 18018, "dom": "G", "ent": "Alotenango", "cls": "Southern Europe"}, {"id": 18019, "dom": "G", "ent": "Alotenango", "cls": "Western Africa"}, {"id": 18020, "dom": "G", "ent": "Alotenango", "cls": "Western Asia"}, {"id": 18021, "dom": "G", "ent": "Alotenango", "cls": "Western Europe"}, {"id": 18022, "dom": "G", "ent": "Tiquisate", "cls": "Australia and New Zealand"}, {"id": 18023, "dom": "G", "ent": "Tiquisate", "cls": "Caribbean"}, {"id": 18024, "dom": "G", "ent": "Tiquisate", "cls": "Central America"}, {"id": 18025, "dom": "G", "ent": "Tiquisate", "cls": "Central Asia"}, {"id": 18026, "dom": "G", "ent": "Tiquisate", "cls": "Eastern Africa"}, {"id": 18027, "dom": "G", "ent": "Tiquisate", "cls": "Eastern Asia"}, {"id": 18028, "dom": "G", "ent": "Tiquisate", "cls": "Eastern Europe"}, {"id": 18029, "dom": "G", "ent": "Tiquisate", "cls": "Melanesia"}, {"id": 18030, "dom": "G", "ent": "Tiquisate", "cls": "Micronesia"}, {"id": 18031, "dom": "G", "ent": "Tiquisate", "cls": "Middle Africa"}, {"id": 18032, "dom": "G", "ent": "Tiquisate", "cls": "Northern Africa"}, {"id": 18033, "dom": "G", "ent": "Tiquisate", "cls": "Northern America"}, {"id": 18034, "dom": "G", "ent": "Tiquisate", "cls": "Northern Europe"}, {"id": 18035, "dom": "G", "ent": "Tiquisate", "cls": "Polynesia"}, {"id": 18036, "dom": "G", "ent": "Tiquisate", "cls": "South America"}, {"id": 18037, "dom": "G", "ent": "Tiquisate", "cls": "South-eastern Asia"}, {"id": 18038, "dom": "G", "ent": "Tiquisate", "cls": "Southern Africa"}, {"id": 18039, "dom": "G", "ent": "Tiquisate", "cls": "Southern Asia"}, {"id": 18040, "dom": "G", "ent": "Tiquisate", "cls": "Southern Europe"}, {"id": 18041, "dom": "G", "ent": "Tiquisate", "cls": "Western Africa"}, {"id": 18042, "dom": "G", "ent": "Tiquisate", "cls": "Western Asia"}, {"id": 18043, "dom": "G", "ent": "Tiquisate", "cls": "Western Europe"}, {"id": 18044, "dom": "G", "ent": "Bairro Militar", "cls": "Australia and New Zealand"}, {"id": 18045, "dom": "G", "ent": "Bairro Militar", "cls": "Caribbean"}, {"id": 18046, "dom": "G", "ent": "Bairro Militar", "cls": "Central America"}, {"id": 18047, "dom": "G", "ent": "Bairro Militar", "cls": "Central Asia"}, {"id": 18048, "dom": "G", "ent": "Bairro Militar", "cls": "Eastern Africa"}, {"id": 18049, "dom": "G", "ent": "Bairro Militar", "cls": "Eastern Asia"}, {"id": 18050, "dom": "G", "ent": "Bairro Militar", "cls": "Eastern Europe"}, {"id": 18051, "dom": "G", "ent": "Bairro Militar", "cls": "Melanesia"}, {"id": 18052, "dom": "G", "ent": "Bairro Militar", "cls": "Micronesia"}, {"id": 18053, "dom": "G", "ent": "Bairro Militar", "cls": "Middle Africa"}, {"id": 18054, "dom": "G", "ent": "Bairro Militar", "cls": "Northern Africa"}, {"id": 18055, "dom": "G", "ent": "Bairro Militar", "cls": "Northern America"}, {"id": 18056, "dom": "G", "ent": "Bairro Militar", "cls": "Northern Europe"}, {"id": 18057, "dom": "G", "ent": "Bairro Militar", "cls": "Polynesia"}, {"id": 18058, "dom": "G", "ent": "Bairro Militar", "cls": "South America"}, {"id": 18059, "dom": "G", "ent": "Bairro Militar", "cls": "South-eastern Asia"}, {"id": 18060, "dom": "G", "ent": "Bairro Militar", "cls": "Southern Africa"}, {"id": 18061, "dom": "G", "ent": "Bairro Militar", "cls": "Southern Asia"}, {"id": 18062, "dom": "G", "ent": "Bairro Militar", "cls": "Southern Europe"}, {"id": 18063, "dom": "G", "ent": "Bairro Militar", "cls": "Western Africa"}, {"id": 18064, "dom": "G", "ent": "Bairro Militar", "cls": "Western Asia"}, {"id": 18065, "dom": "G", "ent": "Bairro Militar", "cls": "Western Europe"}, {"id": 18066, "dom": "G", "ent": "Reino de Bandim", "cls": "Australia and New Zealand"}, {"id": 18067, "dom": "G", "ent": "Reino de Bandim", "cls": "Caribbean"}, {"id": 18068, "dom": "G", "ent": "Reino de Bandim", "cls": "Central America"}, {"id": 18069, "dom": "G", "ent": "Reino de Bandim", "cls": "Central Asia"}, {"id": 18070, "dom": "G", "ent": "Reino de Bandim", "cls": "Eastern Africa"}, {"id": 18071, "dom": "G", "ent": "Reino de Bandim", "cls": "Eastern Asia"}, {"id": 18072, "dom": "G", "ent": "Reino de Bandim", "cls": "Eastern Europe"}, {"id": 18073, "dom": "G", "ent": "Reino de Bandim", "cls": "Melanesia"}, {"id": 18074, "dom": "G", "ent": "Reino de Bandim", "cls": "Micronesia"}, {"id": 18075, "dom": "G", "ent": "Reino de Bandim", "cls": "Middle Africa"}, {"id": 18076, "dom": "G", "ent": "Reino de Bandim", "cls": "Northern Africa"}, {"id": 18077, "dom": "G", "ent": "Reino de Bandim", "cls": "Northern America"}, {"id": 18078, "dom": "G", "ent": "Reino de Bandim", "cls": "Northern Europe"}, {"id": 18079, "dom": "G", "ent": "Reino de Bandim", "cls": "Polynesia"}, {"id": 18080, "dom": "G", "ent": "Reino de Bandim", "cls": "South America"}, {"id": 18081, "dom": "G", "ent": "Reino de Bandim", "cls": "South-eastern Asia"}, {"id": 18082, "dom": "G", "ent": "Reino de Bandim", "cls": "Southern Africa"}, {"id": 18083, "dom": "G", "ent": "Reino de Bandim", "cls": "Southern Asia"}, {"id": 18084, "dom": "G", "ent": "Reino de Bandim", "cls": "Southern Europe"}, {"id": 18085, "dom": "G", "ent": "Reino de Bandim", "cls": "Western Africa"}, {"id": 18086, "dom": "G", "ent": "Reino de Bandim", "cls": "Western Asia"}, {"id": 18087, "dom": "G", "ent": "Reino de Bandim", "cls": "Western Europe"}, {"id": 18088, "dom": "G", "ent": "Reino de Antula", "cls": "Australia and New Zealand"}, {"id": 18089, "dom": "G", "ent": "Reino de Antula", "cls": "Caribbean"}, {"id": 18090, "dom": "G", "ent": "Reino de Antula", "cls": "Central America"}, {"id": 18091, "dom": "G", "ent": "Reino de Antula", "cls": "Central Asia"}, {"id": 18092, "dom": "G", "ent": "Reino de Antula", "cls": "Eastern Africa"}, {"id": 18093, "dom": "G", "ent": "Reino de Antula", "cls": "Eastern Asia"}, {"id": 18094, "dom": "G", "ent": "Reino de Antula", "cls": "Eastern Europe"}, {"id": 18095, "dom": "G", "ent": "Reino de Antula", "cls": "Melanesia"}, {"id": 18096, "dom": "G", "ent": "Reino de Antula", "cls": "Micronesia"}, {"id": 18097, "dom": "G", "ent": "Reino de Antula", "cls": "Middle Africa"}, {"id": 18098, "dom": "G", "ent": "Reino de Antula", "cls": "Northern Africa"}, {"id": 18099, "dom": "G", "ent": "Reino de Antula", "cls": "Northern America"}, {"id": 18100, "dom": "G", "ent": "Reino de Antula", "cls": "Northern Europe"}, {"id": 18101, "dom": "G", "ent": "Reino de Antula", "cls": "Polynesia"}, {"id": 18102, "dom": "G", "ent": "Reino de Antula", "cls": "South America"}, {"id": 18103, "dom": "G", "ent": "Reino de Antula", "cls": "South-eastern Asia"}, {"id": 18104, "dom": "G", "ent": "Reino de Antula", "cls": "Southern Africa"}, {"id": 18105, "dom": "G", "ent": "Reino de Antula", "cls": "Southern Asia"}, {"id": 18106, "dom": "G", "ent": "Reino de Antula", "cls": "Southern Europe"}, {"id": 18107, "dom": "G", "ent": "Reino de Antula", "cls": "Western Africa"}, {"id": 18108, "dom": "G", "ent": "Reino de Antula", "cls": "Western Asia"}, {"id": 18109, "dom": "G", "ent": "Reino de Antula", "cls": "Western Europe"}, {"id": 18110, "dom": "G", "ent": "Plaque", "cls": "Australia and New Zealand"}, {"id": 18111, "dom": "G", "ent": "Plaque", "cls": "Caribbean"}, {"id": 18112, "dom": "G", "ent": "Plaque", "cls": "Central America"}, {"id": 18113, "dom": "G", "ent": "Plaque", "cls": "Central Asia"}, {"id": 18114, "dom": "G", "ent": "Plaque", "cls": "Eastern Africa"}, {"id": 18115, "dom": "G", "ent": "Plaque", "cls": "Eastern Asia"}, {"id": 18116, "dom": "G", "ent": "Plaque", "cls": "Eastern Europe"}, {"id": 18117, "dom": "G", "ent": "Plaque", "cls": "Melanesia"}, {"id": 18118, "dom": "G", "ent": "Plaque", "cls": "Micronesia"}, {"id": 18119, "dom": "G", "ent": "Plaque", "cls": "Middle Africa"}, {"id": 18120, "dom": "G", "ent": "Plaque", "cls": "Northern Africa"}, {"id": 18121, "dom": "G", "ent": "Plaque", "cls": "Northern America"}, {"id": 18122, "dom": "G", "ent": "Plaque", "cls": "Northern Europe"}, {"id": 18123, "dom": "G", "ent": "Plaque", "cls": "Polynesia"}, {"id": 18124, "dom": "G", "ent": "Plaque", "cls": "South America"}, {"id": 18125, "dom": "G", "ent": "Plaque", "cls": "South-eastern Asia"}, {"id": 18126, "dom": "G", "ent": "Plaque", "cls": "Southern Africa"}, {"id": 18127, "dom": "G", "ent": "Plaque", "cls": "Southern Asia"}, {"id": 18128, "dom": "G", "ent": "Plaque", "cls": "Southern Europe"}, {"id": 18129, "dom": "G", "ent": "Plaque", "cls": "Western Africa"}, {"id": 18130, "dom": "G", "ent": "Plaque", "cls": "Western Asia"}, {"id": 18131, "dom": "G", "ent": "Plaque", "cls": "Western Europe"}, {"id": 18132, "dom": "G", "ent": "Whampoa", "cls": "Australia and New Zealand"}, {"id": 18133, "dom": "G", "ent": "Whampoa", "cls": "Caribbean"}, {"id": 18134, "dom": "G", "ent": "Whampoa", "cls": "Central America"}, {"id": 18135, "dom": "G", "ent": "Whampoa", "cls": "Central Asia"}, {"id": 18136, "dom": "G", "ent": "Whampoa", "cls": "Eastern Africa"}, {"id": 18137, "dom": "G", "ent": "Whampoa", "cls": "Eastern Asia"}, {"id": 18138, "dom": "G", "ent": "Whampoa", "cls": "Eastern Europe"}, {"id": 18139, "dom": "G", "ent": "Whampoa", "cls": "Melanesia"}, {"id": 18140, "dom": "G", "ent": "Whampoa", "cls": "Micronesia"}, {"id": 18141, "dom": "G", "ent": "Whampoa", "cls": "Middle Africa"}, {"id": 18142, "dom": "G", "ent": "Whampoa", "cls": "Northern Africa"}, {"id": 18143, "dom": "G", "ent": "Whampoa", "cls": "Northern America"}, {"id": 18144, "dom": "G", "ent": "Whampoa", "cls": "Northern Europe"}, {"id": 18145, "dom": "G", "ent": "Whampoa", "cls": "Polynesia"}, {"id": 18146, "dom": "G", "ent": "Whampoa", "cls": "South America"}, {"id": 18147, "dom": "G", "ent": "Whampoa", "cls": "South-eastern Asia"}, {"id": 18148, "dom": "G", "ent": "Whampoa", "cls": "Southern Africa"}, {"id": 18149, "dom": "G", "ent": "Whampoa", "cls": "Southern Asia"}, {"id": 18150, "dom": "G", "ent": "Whampoa", "cls": "Southern Europe"}, {"id": 18151, "dom": "G", "ent": "Whampoa", "cls": "Western Africa"}, {"id": 18152, "dom": "G", "ent": "Whampoa", "cls": "Western Asia"}, {"id": 18153, "dom": "G", "ent": "Whampoa", "cls": "Western Europe"}, {"id": 18154, "dom": "G", "ent": "Shui Chuen O", "cls": "Australia and New Zealand"}, {"id": 18155, "dom": "G", "ent": "Shui Chuen O", "cls": "Caribbean"}, {"id": 18156, "dom": "G", "ent": "Shui Chuen O", "cls": "Central America"}, {"id": 18157, "dom": "G", "ent": "Shui Chuen O", "cls": "Central Asia"}, {"id": 18158, "dom": "G", "ent": "Shui Chuen O", "cls": "Eastern Africa"}, {"id": 18159, "dom": "G", "ent": "Shui Chuen O", "cls": "Eastern Asia"}, {"id": 18160, "dom": "G", "ent": "Shui Chuen O", "cls": "Eastern Europe"}, {"id": 18161, "dom": "G", "ent": "Shui Chuen O", "cls": "Melanesia"}, {"id": 18162, "dom": "G", "ent": "Shui Chuen O", "cls": "Micronesia"}, {"id": 18163, "dom": "G", "ent": "Shui Chuen O", "cls": "Middle Africa"}, {"id": 18164, "dom": "G", "ent": "Shui Chuen O", "cls": "Northern Africa"}, {"id": 18165, "dom": "G", "ent": "Shui Chuen O", "cls": "Northern America"}, {"id": 18166, "dom": "G", "ent": "Shui Chuen O", "cls": "Northern Europe"}, {"id": 18167, "dom": "G", "ent": "Shui Chuen O", "cls": "Polynesia"}, {"id": 18168, "dom": "G", "ent": "Shui Chuen O", "cls": "South America"}, {"id": 18169, "dom": "G", "ent": "Shui Chuen O", "cls": "South-eastern Asia"}, {"id": 18170, "dom": "G", "ent": "Shui Chuen O", "cls": "Southern Africa"}, {"id": 18171, "dom": "G", "ent": "Shui Chuen O", "cls": "Southern Asia"}, {"id": 18172, "dom": "G", "ent": "Shui Chuen O", "cls": "Southern Europe"}, {"id": 18173, "dom": "G", "ent": "Shui Chuen O", "cls": "Western Africa"}, {"id": 18174, "dom": "G", "ent": "Shui Chuen O", "cls": "Western Asia"}, {"id": 18175, "dom": "G", "ent": "Shui Chuen O", "cls": "Western Europe"}, {"id": 18176, "dom": "G", "ent": "Wo Che", "cls": "Australia and New Zealand"}, {"id": 18177, "dom": "G", "ent": "Wo Che", "cls": "Caribbean"}, {"id": 18178, "dom": "G", "ent": "Wo Che", "cls": "Central America"}, {"id": 18179, "dom": "G", "ent": "Wo Che", "cls": "Central Asia"}, {"id": 18180, "dom": "G", "ent": "Wo Che", "cls": "Eastern Africa"}, {"id": 18181, "dom": "G", "ent": "Wo Che", "cls": "Eastern Asia"}, {"id": 18182, "dom": "G", "ent": "Wo Che", "cls": "Eastern Europe"}, {"id": 18183, "dom": "G", "ent": "Wo Che", "cls": "Melanesia"}, {"id": 18184, "dom": "G", "ent": "Wo Che", "cls": "Micronesia"}, {"id": 18185, "dom": "G", "ent": "Wo Che", "cls": "Middle Africa"}, {"id": 18186, "dom": "G", "ent": "Wo Che", "cls": "Northern Africa"}, {"id": 18187, "dom": "G", "ent": "Wo Che", "cls": "Northern America"}, {"id": 18188, "dom": "G", "ent": "Wo Che", "cls": "Northern Europe"}, {"id": 18189, "dom": "G", "ent": "Wo Che", "cls": "Polynesia"}, {"id": 18190, "dom": "G", "ent": "Wo Che", "cls": "South America"}, {"id": 18191, "dom": "G", "ent": "Wo Che", "cls": "South-eastern Asia"}, {"id": 18192, "dom": "G", "ent": "Wo Che", "cls": "Southern Africa"}, {"id": 18193, "dom": "G", "ent": "Wo Che", "cls": "Southern Asia"}, {"id": 18194, "dom": "G", "ent": "Wo Che", "cls": "Southern Europe"}, {"id": 18195, "dom": "G", "ent": "Wo Che", "cls": "Western Africa"}, {"id": 18196, "dom": "G", "ent": "Wo Che", "cls": "Western Asia"}, {"id": 18197, "dom": "G", "ent": "Wo Che", "cls": "Western Europe"}, {"id": 18198, "dom": "G", "ent": "Mid Levels", "cls": "Australia and New Zealand"}, {"id": 18199, "dom": "G", "ent": "Mid Levels", "cls": "Caribbean"}, {"id": 18200, "dom": "G", "ent": "Mid Levels", "cls": "Central America"}, {"id": 18201, "dom": "G", "ent": "Mid Levels", "cls": "Central Asia"}, {"id": 18202, "dom": "G", "ent": "Mid Levels", "cls": "Eastern Africa"}, {"id": 18203, "dom": "G", "ent": "Mid Levels", "cls": "Eastern Asia"}, {"id": 18204, "dom": "G", "ent": "Mid Levels", "cls": "Eastern Europe"}, {"id": 18205, "dom": "G", "ent": "Mid Levels", "cls": "Melanesia"}, {"id": 18206, "dom": "G", "ent": "Mid Levels", "cls": "Micronesia"}, {"id": 18207, "dom": "G", "ent": "Mid Levels", "cls": "Middle Africa"}, {"id": 18208, "dom": "G", "ent": "Mid Levels", "cls": "Northern Africa"}, {"id": 18209, "dom": "G", "ent": "Mid Levels", "cls": "Northern America"}, {"id": 18210, "dom": "G", "ent": "Mid Levels", "cls": "Northern Europe"}, {"id": 18211, "dom": "G", "ent": "Mid Levels", "cls": "Polynesia"}, {"id": 18212, "dom": "G", "ent": "Mid Levels", "cls": "South America"}, {"id": 18213, "dom": "G", "ent": "Mid Levels", "cls": "South-eastern Asia"}, {"id": 18214, "dom": "G", "ent": "Mid Levels", "cls": "Southern Africa"}, {"id": 18215, "dom": "G", "ent": "Mid Levels", "cls": "Southern Asia"}, {"id": 18216, "dom": "G", "ent": "Mid Levels", "cls": "Southern Europe"}, {"id": 18217, "dom": "G", "ent": "Mid Levels", "cls": "Western Africa"}, {"id": 18218, "dom": "G", "ent": "Mid Levels", "cls": "Western Asia"}, {"id": 18219, "dom": "G", "ent": "Mid Levels", "cls": "Western Europe"}, {"id": 18220, "dom": "G", "ent": "El Progreso", "cls": "Australia and New Zealand"}, {"id": 18221, "dom": "G", "ent": "El Progreso", "cls": "Caribbean"}, {"id": 18222, "dom": "G", "ent": "El Progreso", "cls": "Central America"}, {"id": 18223, "dom": "G", "ent": "El Progreso", "cls": "Central Asia"}, {"id": 18224, "dom": "G", "ent": "El Progreso", "cls": "Eastern Africa"}, {"id": 18225, "dom": "G", "ent": "El Progreso", "cls": "Eastern Asia"}, {"id": 18226, "dom": "G", "ent": "El Progreso", "cls": "Eastern Europe"}, {"id": 18227, "dom": "G", "ent": "El Progreso", "cls": "Melanesia"}, {"id": 18228, "dom": "G", "ent": "El Progreso", "cls": "Micronesia"}, {"id": 18229, "dom": "G", "ent": "El Progreso", "cls": "Middle Africa"}, {"id": 18230, "dom": "G", "ent": "El Progreso", "cls": "Northern Africa"}, {"id": 18231, "dom": "G", "ent": "El Progreso", "cls": "Northern America"}, {"id": 18232, "dom": "G", "ent": "El Progreso", "cls": "Northern Europe"}, {"id": 18233, "dom": "G", "ent": "El Progreso", "cls": "Polynesia"}, {"id": 18234, "dom": "G", "ent": "El Progreso", "cls": "South America"}, {"id": 18235, "dom": "G", "ent": "El Progreso", "cls": "South-eastern Asia"}, {"id": 18236, "dom": "G", "ent": "El Progreso", "cls": "Southern Africa"}, {"id": 18237, "dom": "G", "ent": "El Progreso", "cls": "Southern Asia"}, {"id": 18238, "dom": "G", "ent": "El Progreso", "cls": "Southern Europe"}, {"id": 18239, "dom": "G", "ent": "El Progreso", "cls": "Western Africa"}, {"id": 18240, "dom": "G", "ent": "El Progreso", "cls": "Western Asia"}, {"id": 18241, "dom": "G", "ent": "El Progreso", "cls": "Western Europe"}, {"id": 18242, "dom": "G", "ent": "La Lima", "cls": "Australia and New Zealand"}, {"id": 18243, "dom": "G", "ent": "La Lima", "cls": "Caribbean"}, {"id": 18244, "dom": "G", "ent": "La Lima", "cls": "Central America"}, {"id": 18245, "dom": "G", "ent": "La Lima", "cls": "Central Asia"}, {"id": 18246, "dom": "G", "ent": "La Lima", "cls": "Eastern Africa"}, {"id": 18247, "dom": "G", "ent": "La Lima", "cls": "Eastern Asia"}, {"id": 18248, "dom": "G", "ent": "La Lima", "cls": "Eastern Europe"}, {"id": 18249, "dom": "G", "ent": "La Lima", "cls": "Melanesia"}, {"id": 18250, "dom": "G", "ent": "La Lima", "cls": "Micronesia"}, {"id": 18251, "dom": "G", "ent": "La Lima", "cls": "Middle Africa"}, {"id": 18252, "dom": "G", "ent": "La Lima", "cls": "Northern Africa"}, {"id": 18253, "dom": "G", "ent": "La Lima", "cls": "Northern America"}, {"id": 18254, "dom": "G", "ent": "La Lima", "cls": "Northern Europe"}, {"id": 18255, "dom": "G", "ent": "La Lima", "cls": "Polynesia"}, {"id": 18256, "dom": "G", "ent": "La Lima", "cls": "South America"}, {"id": 18257, "dom": "G", "ent": "La Lima", "cls": "South-eastern Asia"}, {"id": 18258, "dom": "G", "ent": "La Lima", "cls": "Southern Africa"}, {"id": 18259, "dom": "G", "ent": "La Lima", "cls": "Southern Asia"}, {"id": 18260, "dom": "G", "ent": "La Lima", "cls": "Southern Europe"}, {"id": 18261, "dom": "G", "ent": "La Lima", "cls": "Western Africa"}, {"id": 18262, "dom": "G", "ent": "La Lima", "cls": "Western Asia"}, {"id": 18263, "dom": "G", "ent": "La Lima", "cls": "Western Europe"}, {"id": 18264, "dom": "G", "ent": "Macuelizo", "cls": "Australia and New Zealand"}, {"id": 18265, "dom": "G", "ent": "Macuelizo", "cls": "Caribbean"}, {"id": 18266, "dom": "G", "ent": "Macuelizo", "cls": "Central America"}, {"id": 18267, "dom": "G", "ent": "Macuelizo", "cls": "Central Asia"}, {"id": 18268, "dom": "G", "ent": "Macuelizo", "cls": "Eastern Africa"}, {"id": 18269, "dom": "G", "ent": "Macuelizo", "cls": "Eastern Asia"}, {"id": 18270, "dom": "G", "ent": "Macuelizo", "cls": "Eastern Europe"}, {"id": 18271, "dom": "G", "ent": "Macuelizo", "cls": "Melanesia"}, {"id": 18272, "dom": "G", "ent": "Macuelizo", "cls": "Micronesia"}, {"id": 18273, "dom": "G", "ent": "Macuelizo", "cls": "Middle Africa"}, {"id": 18274, "dom": "G", "ent": "Macuelizo", "cls": "Northern Africa"}, {"id": 18275, "dom": "G", "ent": "Macuelizo", "cls": "Northern America"}, {"id": 18276, "dom": "G", "ent": "Macuelizo", "cls": "Northern Europe"}, {"id": 18277, "dom": "G", "ent": "Macuelizo", "cls": "Polynesia"}, {"id": 18278, "dom": "G", "ent": "Macuelizo", "cls": "South America"}, {"id": 18279, "dom": "G", "ent": "Macuelizo", "cls": "South-eastern Asia"}, {"id": 18280, "dom": "G", "ent": "Macuelizo", "cls": "Southern Africa"}, {"id": 18281, "dom": "G", "ent": "Macuelizo", "cls": "Southern Asia"}, {"id": 18282, "dom": "G", "ent": "Macuelizo", "cls": "Southern Europe"}, {"id": 18283, "dom": "G", "ent": "Macuelizo", "cls": "Western Africa"}, {"id": 18284, "dom": "G", "ent": "Macuelizo", "cls": "Western Asia"}, {"id": 18285, "dom": "G", "ent": "Macuelizo", "cls": "Western Europe"}, {"id": 18286, "dom": "G", "ent": "Marcovia", "cls": "Australia and New Zealand"}, {"id": 18287, "dom": "G", "ent": "Marcovia", "cls": "Caribbean"}, {"id": 18288, "dom": "G", "ent": "Marcovia", "cls": "Central America"}, {"id": 18289, "dom": "G", "ent": "Marcovia", "cls": "Central Asia"}, {"id": 18290, "dom": "G", "ent": "Marcovia", "cls": "Eastern Africa"}, {"id": 18291, "dom": "G", "ent": "Marcovia", "cls": "Eastern Asia"}, {"id": 18292, "dom": "G", "ent": "Marcovia", "cls": "Eastern Europe"}, {"id": 18293, "dom": "G", "ent": "Marcovia", "cls": "Melanesia"}, {"id": 18294, "dom": "G", "ent": "Marcovia", "cls": "Micronesia"}, {"id": 18295, "dom": "G", "ent": "Marcovia", "cls": "Middle Africa"}, {"id": 18296, "dom": "G", "ent": "Marcovia", "cls": "Northern Africa"}, {"id": 18297, "dom": "G", "ent": "Marcovia", "cls": "Northern America"}, {"id": 18298, "dom": "G", "ent": "Marcovia", "cls": "Northern Europe"}, {"id": 18299, "dom": "G", "ent": "Marcovia", "cls": "Polynesia"}, {"id": 18300, "dom": "G", "ent": "Marcovia", "cls": "South America"}, {"id": 18301, "dom": "G", "ent": "Marcovia", "cls": "South-eastern Asia"}, {"id": 18302, "dom": "G", "ent": "Marcovia", "cls": "Southern Africa"}, {"id": 18303, "dom": "G", "ent": "Marcovia", "cls": "Southern Asia"}, {"id": 18304, "dom": "G", "ent": "Marcovia", "cls": "Southern Europe"}, {"id": 18305, "dom": "G", "ent": "Marcovia", "cls": "Western Africa"}, {"id": 18306, "dom": "G", "ent": "Marcovia", "cls": "Western Asia"}, {"id": 18307, "dom": "G", "ent": "Marcovia", "cls": "Western Europe"}, {"id": 18308, "dom": "G", "ent": "Karlovac", "cls": "Australia and New Zealand"}, {"id": 18309, "dom": "G", "ent": "Karlovac", "cls": "Caribbean"}, {"id": 18310, "dom": "G", "ent": "Karlovac", "cls": "Central America"}, {"id": 18311, "dom": "G", "ent": "Karlovac", "cls": "Central Asia"}, {"id": 18312, "dom": "G", "ent": "Karlovac", "cls": "Eastern Africa"}, {"id": 18313, "dom": "G", "ent": "Karlovac", "cls": "Eastern Asia"}, {"id": 18314, "dom": "G", "ent": "Karlovac", "cls": "Eastern Europe"}, {"id": 18315, "dom": "G", "ent": "Karlovac", "cls": "Melanesia"}, {"id": 18316, "dom": "G", "ent": "Karlovac", "cls": "Micronesia"}, {"id": 18317, "dom": "G", "ent": "Karlovac", "cls": "Middle Africa"}, {"id": 18318, "dom": "G", "ent": "Karlovac", "cls": "Northern Africa"}, {"id": 18319, "dom": "G", "ent": "Karlovac", "cls": "Northern America"}, {"id": 18320, "dom": "G", "ent": "Karlovac", "cls": "Northern Europe"}, {"id": 18321, "dom": "G", "ent": "Karlovac", "cls": "Polynesia"}, {"id": 18322, "dom": "G", "ent": "Karlovac", "cls": "South America"}, {"id": 18323, "dom": "G", "ent": "Karlovac", "cls": "South-eastern Asia"}, {"id": 18324, "dom": "G", "ent": "Karlovac", "cls": "Southern Africa"}, {"id": 18325, "dom": "G", "ent": "Karlovac", "cls": "Southern Asia"}, {"id": 18326, "dom": "G", "ent": "Karlovac", "cls": "Southern Europe"}, {"id": 18327, "dom": "G", "ent": "Karlovac", "cls": "Western Africa"}, {"id": 18328, "dom": "G", "ent": "Karlovac", "cls": "Western Asia"}, {"id": 18329, "dom": "G", "ent": "Karlovac", "cls": "Western Europe"}, {"id": 18330, "dom": "G", "ent": "Velika Gorica", "cls": "Australia and New Zealand"}, {"id": 18331, "dom": "G", "ent": "Velika Gorica", "cls": "Caribbean"}, {"id": 18332, "dom": "G", "ent": "Velika Gorica", "cls": "Central America"}, {"id": 18333, "dom": "G", "ent": "Velika Gorica", "cls": "Central Asia"}, {"id": 18334, "dom": "G", "ent": "Velika Gorica", "cls": "Eastern Africa"}, {"id": 18335, "dom": "G", "ent": "Velika Gorica", "cls": "Eastern Asia"}, {"id": 18336, "dom": "G", "ent": "Velika Gorica", "cls": "Eastern Europe"}, {"id": 18337, "dom": "G", "ent": "Velika Gorica", "cls": "Melanesia"}, {"id": 18338, "dom": "G", "ent": "Velika Gorica", "cls": "Micronesia"}, {"id": 18339, "dom": "G", "ent": "Velika Gorica", "cls": "Middle Africa"}, {"id": 18340, "dom": "G", "ent": "Velika Gorica", "cls": "Northern Africa"}, {"id": 18341, "dom": "G", "ent": "Velika Gorica", "cls": "Northern America"}, {"id": 18342, "dom": "G", "ent": "Velika Gorica", "cls": "Northern Europe"}, {"id": 18343, "dom": "G", "ent": "Velika Gorica", "cls": "Polynesia"}, {"id": 18344, "dom": "G", "ent": "Velika Gorica", "cls": "South America"}, {"id": 18345, "dom": "G", "ent": "Velika Gorica", "cls": "South-eastern Asia"}, {"id": 18346, "dom": "G", "ent": "Velika Gorica", "cls": "Southern Africa"}, {"id": 18347, "dom": "G", "ent": "Velika Gorica", "cls": "Southern Asia"}, {"id": 18348, "dom": "G", "ent": "Velika Gorica", "cls": "Southern Europe"}, {"id": 18349, "dom": "G", "ent": "Velika Gorica", "cls": "Western Africa"}, {"id": 18350, "dom": "G", "ent": "Velika Gorica", "cls": "Western Asia"}, {"id": 18351, "dom": "G", "ent": "Velika Gorica", "cls": "Western Europe"}, {"id": 18352, "dom": "G", "ent": "Bjelovar", "cls": "Australia and New Zealand"}, {"id": 18353, "dom": "G", "ent": "Bjelovar", "cls": "Caribbean"}, {"id": 18354, "dom": "G", "ent": "Bjelovar", "cls": "Central America"}, {"id": 18355, "dom": "G", "ent": "Bjelovar", "cls": "Central Asia"}, {"id": 18356, "dom": "G", "ent": "Bjelovar", "cls": "Eastern Africa"}, {"id": 18357, "dom": "G", "ent": "Bjelovar", "cls": "Eastern Asia"}, {"id": 18358, "dom": "G", "ent": "Bjelovar", "cls": "Eastern Europe"}, {"id": 18359, "dom": "G", "ent": "Bjelovar", "cls": "Melanesia"}, {"id": 18360, "dom": "G", "ent": "Bjelovar", "cls": "Micronesia"}, {"id": 18361, "dom": "G", "ent": "Bjelovar", "cls": "Middle Africa"}, {"id": 18362, "dom": "G", "ent": "Bjelovar", "cls": "Northern Africa"}, {"id": 18363, "dom": "G", "ent": "Bjelovar", "cls": "Northern America"}, {"id": 18364, "dom": "G", "ent": "Bjelovar", "cls": "Northern Europe"}, {"id": 18365, "dom": "G", "ent": "Bjelovar", "cls": "Polynesia"}, {"id": 18366, "dom": "G", "ent": "Bjelovar", "cls": "South America"}, {"id": 18367, "dom": "G", "ent": "Bjelovar", "cls": "South-eastern Asia"}, {"id": 18368, "dom": "G", "ent": "Bjelovar", "cls": "Southern Africa"}, {"id": 18369, "dom": "G", "ent": "Bjelovar", "cls": "Southern Asia"}, {"id": 18370, "dom": "G", "ent": "Bjelovar", "cls": "Southern Europe"}, {"id": 18371, "dom": "G", "ent": "Bjelovar", "cls": "Western Africa"}, {"id": 18372, "dom": "G", "ent": "Bjelovar", "cls": "Western Asia"}, {"id": 18373, "dom": "G", "ent": "Bjelovar", "cls": "Western Europe"}, {"id": 18374, "dom": "G", "ent": "Liberpolis", "cls": "Australia and New Zealand"}, {"id": 18375, "dom": "G", "ent": "Liberpolis", "cls": "Caribbean"}, {"id": 18376, "dom": "G", "ent": "Liberpolis", "cls": "Central America"}, {"id": 18377, "dom": "G", "ent": "Liberpolis", "cls": "Central Asia"}, {"id": 18378, "dom": "G", "ent": "Liberpolis", "cls": "Eastern Africa"}, {"id": 18379, "dom": "G", "ent": "Liberpolis", "cls": "Eastern Asia"}, {"id": 18380, "dom": "G", "ent": "Liberpolis", "cls": "Eastern Europe"}, {"id": 18381, "dom": "G", "ent": "Liberpolis", "cls": "Melanesia"}, {"id": 18382, "dom": "G", "ent": "Liberpolis", "cls": "Micronesia"}, {"id": 18383, "dom": "G", "ent": "Liberpolis", "cls": "Middle Africa"}, {"id": 18384, "dom": "G", "ent": "Liberpolis", "cls": "Northern Africa"}, {"id": 18385, "dom": "G", "ent": "Liberpolis", "cls": "Northern America"}, {"id": 18386, "dom": "G", "ent": "Liberpolis", "cls": "Northern Europe"}, {"id": 18387, "dom": "G", "ent": "Liberpolis", "cls": "Polynesia"}, {"id": 18388, "dom": "G", "ent": "Liberpolis", "cls": "South America"}, {"id": 18389, "dom": "G", "ent": "Liberpolis", "cls": "South-eastern Asia"}, {"id": 18390, "dom": "G", "ent": "Liberpolis", "cls": "Southern Africa"}, {"id": 18391, "dom": "G", "ent": "Liberpolis", "cls": "Southern Asia"}, {"id": 18392, "dom": "G", "ent": "Liberpolis", "cls": "Southern Europe"}, {"id": 18393, "dom": "G", "ent": "Liberpolis", "cls": "Western Africa"}, {"id": 18394, "dom": "G", "ent": "Liberpolis", "cls": "Western Asia"}, {"id": 18395, "dom": "G", "ent": "Liberpolis", "cls": "Western Europe"}, {"id": 18396, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "Australia and New Zealand"}, {"id": 18397, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "Caribbean"}, {"id": 18398, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "Central America"}, {"id": 18399, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "Central Asia"}, {"id": 18400, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "Eastern Africa"}, {"id": 18401, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "Eastern Asia"}, {"id": 18402, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "Eastern Europe"}, {"id": 18403, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "Melanesia"}, {"id": 18404, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "Micronesia"}, {"id": 18405, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "Middle Africa"}, {"id": 18406, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "Northern Africa"}, {"id": 18407, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "Northern America"}, {"id": 18408, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "Northern Europe"}, {"id": 18409, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "Polynesia"}, {"id": 18410, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "South America"}, {"id": 18411, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "South-eastern Asia"}, {"id": 18412, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "Southern Africa"}, {"id": 18413, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "Southern Asia"}, {"id": 18414, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "Southern Europe"}, {"id": 18415, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "Western Africa"}, {"id": 18416, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "Western Asia"}, {"id": 18417, "dom": "G", "ent": "Croix-des-Bouquets", "cls": "Western Europe"}, {"id": 18418, "dom": "G", "ent": "Jacmel", "cls": "Australia and New Zealand"}, {"id": 18419, "dom": "G", "ent": "Jacmel", "cls": "Caribbean"}, {"id": 18420, "dom": "G", "ent": "Jacmel", "cls": "Central America"}, {"id": 18421, "dom": "G", "ent": "Jacmel", "cls": "Central Asia"}, {"id": 18422, "dom": "G", "ent": "Jacmel", "cls": "Eastern Africa"}, {"id": 18423, "dom": "G", "ent": "Jacmel", "cls": "Eastern Asia"}, {"id": 18424, "dom": "G", "ent": "Jacmel", "cls": "Eastern Europe"}, {"id": 18425, "dom": "G", "ent": "Jacmel", "cls": "Melanesia"}, {"id": 18426, "dom": "G", "ent": "Jacmel", "cls": "Micronesia"}, {"id": 18427, "dom": "G", "ent": "Jacmel", "cls": "Middle Africa"}, {"id": 18428, "dom": "G", "ent": "Jacmel", "cls": "Northern Africa"}, {"id": 18429, "dom": "G", "ent": "Jacmel", "cls": "Northern America"}, {"id": 18430, "dom": "G", "ent": "Jacmel", "cls": "Northern Europe"}, {"id": 18431, "dom": "G", "ent": "Jacmel", "cls": "Polynesia"}, {"id": 18432, "dom": "G", "ent": "Jacmel", "cls": "South America"}, {"id": 18433, "dom": "G", "ent": "Jacmel", "cls": "South-eastern Asia"}, {"id": 18434, "dom": "G", "ent": "Jacmel", "cls": "Southern Africa"}, {"id": 18435, "dom": "G", "ent": "Jacmel", "cls": "Southern Asia"}, {"id": 18436, "dom": "G", "ent": "Jacmel", "cls": "Southern Europe"}, {"id": 18437, "dom": "G", "ent": "Jacmel", "cls": "Western Africa"}, {"id": 18438, "dom": "G", "ent": "Jacmel", "cls": "Western Asia"}, {"id": 18439, "dom": "G", "ent": "Jacmel", "cls": "Western Europe"}, {"id": 18440, "dom": "G", "ent": "Verrettes", "cls": "Australia and New Zealand"}, {"id": 18441, "dom": "G", "ent": "Verrettes", "cls": "Caribbean"}, {"id": 18442, "dom": "G", "ent": "Verrettes", "cls": "Central America"}, {"id": 18443, "dom": "G", "ent": "Verrettes", "cls": "Central Asia"}, {"id": 18444, "dom": "G", "ent": "Verrettes", "cls": "Eastern Africa"}, {"id": 18445, "dom": "G", "ent": "Verrettes", "cls": "Eastern Asia"}, {"id": 18446, "dom": "G", "ent": "Verrettes", "cls": "Eastern Europe"}, {"id": 18447, "dom": "G", "ent": "Verrettes", "cls": "Melanesia"}, {"id": 18448, "dom": "G", "ent": "Verrettes", "cls": "Micronesia"}, {"id": 18449, "dom": "G", "ent": "Verrettes", "cls": "Middle Africa"}, {"id": 18450, "dom": "G", "ent": "Verrettes", "cls": "Northern Africa"}, {"id": 18451, "dom": "G", "ent": "Verrettes", "cls": "Northern America"}, {"id": 18452, "dom": "G", "ent": "Verrettes", "cls": "Northern Europe"}, {"id": 18453, "dom": "G", "ent": "Verrettes", "cls": "Polynesia"}, {"id": 18454, "dom": "G", "ent": "Verrettes", "cls": "South America"}, {"id": 18455, "dom": "G", "ent": "Verrettes", "cls": "South-eastern Asia"}, {"id": 18456, "dom": "G", "ent": "Verrettes", "cls": "Southern Africa"}, {"id": 18457, "dom": "G", "ent": "Verrettes", "cls": "Southern Asia"}, {"id": 18458, "dom": "G", "ent": "Verrettes", "cls": "Southern Europe"}, {"id": 18459, "dom": "G", "ent": "Verrettes", "cls": "Western Africa"}, {"id": 18460, "dom": "G", "ent": "Verrettes", "cls": "Western Asia"}, {"id": 18461, "dom": "G", "ent": "Verrettes", "cls": "Western Europe"}, {"id": 18462, "dom": "G", "ent": "Lenbe", "cls": "Australia and New Zealand"}, {"id": 18463, "dom": "G", "ent": "Lenbe", "cls": "Caribbean"}, {"id": 18464, "dom": "G", "ent": "Lenbe", "cls": "Central America"}, {"id": 18465, "dom": "G", "ent": "Lenbe", "cls": "Central Asia"}, {"id": 18466, "dom": "G", "ent": "Lenbe", "cls": "Eastern Africa"}, {"id": 18467, "dom": "G", "ent": "Lenbe", "cls": "Eastern Asia"}, {"id": 18468, "dom": "G", "ent": "Lenbe", "cls": "Eastern Europe"}, {"id": 18469, "dom": "G", "ent": "Lenbe", "cls": "Melanesia"}, {"id": 18470, "dom": "G", "ent": "Lenbe", "cls": "Micronesia"}, {"id": 18471, "dom": "G", "ent": "Lenbe", "cls": "Middle Africa"}, {"id": 18472, "dom": "G", "ent": "Lenbe", "cls": "Northern Africa"}, {"id": 18473, "dom": "G", "ent": "Lenbe", "cls": "Northern America"}, {"id": 18474, "dom": "G", "ent": "Lenbe", "cls": "Northern Europe"}, {"id": 18475, "dom": "G", "ent": "Lenbe", "cls": "Polynesia"}, {"id": 18476, "dom": "G", "ent": "Lenbe", "cls": "South America"}, {"id": 18477, "dom": "G", "ent": "Lenbe", "cls": "South-eastern Asia"}, {"id": 18478, "dom": "G", "ent": "Lenbe", "cls": "Southern Africa"}, {"id": 18479, "dom": "G", "ent": "Lenbe", "cls": "Southern Asia"}, {"id": 18480, "dom": "G", "ent": "Lenbe", "cls": "Southern Europe"}, {"id": 18481, "dom": "G", "ent": "Lenbe", "cls": "Western Africa"}, {"id": 18482, "dom": "G", "ent": "Lenbe", "cls": "Western Asia"}, {"id": 18483, "dom": "G", "ent": "Lenbe", "cls": "Western Europe"}, {"id": 18484, "dom": "G", "ent": "Nagykanizsa", "cls": "Australia and New Zealand"}, {"id": 18485, "dom": "G", "ent": "Nagykanizsa", "cls": "Caribbean"}, {"id": 18486, "dom": "G", "ent": "Nagykanizsa", "cls": "Central America"}, {"id": 18487, "dom": "G", "ent": "Nagykanizsa", "cls": "Central Asia"}, {"id": 18488, "dom": "G", "ent": "Nagykanizsa", "cls": "Eastern Africa"}, {"id": 18489, "dom": "G", "ent": "Nagykanizsa", "cls": "Eastern Asia"}, {"id": 18490, "dom": "G", "ent": "Nagykanizsa", "cls": "Eastern Europe"}, {"id": 18491, "dom": "G", "ent": "Nagykanizsa", "cls": "Melanesia"}, {"id": 18492, "dom": "G", "ent": "Nagykanizsa", "cls": "Micronesia"}, {"id": 18493, "dom": "G", "ent": "Nagykanizsa", "cls": "Middle Africa"}, {"id": 18494, "dom": "G", "ent": "Nagykanizsa", "cls": "Northern Africa"}, {"id": 18495, "dom": "G", "ent": "Nagykanizsa", "cls": "Northern America"}, {"id": 18496, "dom": "G", "ent": "Nagykanizsa", "cls": "Northern Europe"}, {"id": 18497, "dom": "G", "ent": "Nagykanizsa", "cls": "Polynesia"}, {"id": 18498, "dom": "G", "ent": "Nagykanizsa", "cls": "South America"}, {"id": 18499, "dom": "G", "ent": "Nagykanizsa", "cls": "South-eastern Asia"}, {"id": 18500, "dom": "G", "ent": "Nagykanizsa", "cls": "Southern Africa"}, {"id": 18501, "dom": "G", "ent": "Nagykanizsa", "cls": "Southern Asia"}, {"id": 18502, "dom": "G", "ent": "Nagykanizsa", "cls": "Southern Europe"}, {"id": 18503, "dom": "G", "ent": "Nagykanizsa", "cls": "Western Africa"}, {"id": 18504, "dom": "G", "ent": "Nagykanizsa", "cls": "Western Asia"}, {"id": 18505, "dom": "G", "ent": "Nagykanizsa", "cls": "Western Europe"}, {"id": 18506, "dom": "G", "ent": "Baja", "cls": "Australia and New Zealand"}, {"id": 18507, "dom": "G", "ent": "Baja", "cls": "Caribbean"}, {"id": 18508, "dom": "G", "ent": "Baja", "cls": "Central America"}, {"id": 18509, "dom": "G", "ent": "Baja", "cls": "Central Asia"}, {"id": 18510, "dom": "G", "ent": "Baja", "cls": "Eastern Africa"}, {"id": 18511, "dom": "G", "ent": "Baja", "cls": "Eastern Asia"}, {"id": 18512, "dom": "G", "ent": "Baja", "cls": "Eastern Europe"}, {"id": 18513, "dom": "G", "ent": "Baja", "cls": "Melanesia"}, {"id": 18514, "dom": "G", "ent": "Baja", "cls": "Micronesia"}, {"id": 18515, "dom": "G", "ent": "Baja", "cls": "Middle Africa"}, {"id": 18516, "dom": "G", "ent": "Baja", "cls": "Northern Africa"}, {"id": 18517, "dom": "G", "ent": "Baja", "cls": "Northern America"}, {"id": 18518, "dom": "G", "ent": "Baja", "cls": "Northern Europe"}, {"id": 18519, "dom": "G", "ent": "Baja", "cls": "Polynesia"}, {"id": 18520, "dom": "G", "ent": "Baja", "cls": "South America"}, {"id": 18521, "dom": "G", "ent": "Baja", "cls": "South-eastern Asia"}, {"id": 18522, "dom": "G", "ent": "Baja", "cls": "Southern Africa"}, {"id": 18523, "dom": "G", "ent": "Baja", "cls": "Southern Asia"}, {"id": 18524, "dom": "G", "ent": "Baja", "cls": "Southern Europe"}, {"id": 18525, "dom": "G", "ent": "Baja", "cls": "Western Africa"}, {"id": 18526, "dom": "G", "ent": "Baja", "cls": "Western Asia"}, {"id": 18527, "dom": "G", "ent": "Baja", "cls": "Western Europe"}, {"id": 18528, "dom": "G", "ent": "Keszthely", "cls": "Australia and New Zealand"}, {"id": 18529, "dom": "G", "ent": "Keszthely", "cls": "Caribbean"}, {"id": 18530, "dom": "G", "ent": "Keszthely", "cls": "Central America"}, {"id": 18531, "dom": "G", "ent": "Keszthely", "cls": "Central Asia"}, {"id": 18532, "dom": "G", "ent": "Keszthely", "cls": "Eastern Africa"}, {"id": 18533, "dom": "G", "ent": "Keszthely", "cls": "Eastern Asia"}, {"id": 18534, "dom": "G", "ent": "Keszthely", "cls": "Eastern Europe"}, {"id": 18535, "dom": "G", "ent": "Keszthely", "cls": "Melanesia"}, {"id": 18536, "dom": "G", "ent": "Keszthely", "cls": "Micronesia"}, {"id": 18537, "dom": "G", "ent": "Keszthely", "cls": "Middle Africa"}, {"id": 18538, "dom": "G", "ent": "Keszthely", "cls": "Northern Africa"}, {"id": 18539, "dom": "G", "ent": "Keszthely", "cls": "Northern America"}, {"id": 18540, "dom": "G", "ent": "Keszthely", "cls": "Northern Europe"}, {"id": 18541, "dom": "G", "ent": "Keszthely", "cls": "Polynesia"}, {"id": 18542, "dom": "G", "ent": "Keszthely", "cls": "South America"}, {"id": 18543, "dom": "G", "ent": "Keszthely", "cls": "South-eastern Asia"}, {"id": 18544, "dom": "G", "ent": "Keszthely", "cls": "Southern Africa"}, {"id": 18545, "dom": "G", "ent": "Keszthely", "cls": "Southern Asia"}, {"id": 18546, "dom": "G", "ent": "Keszthely", "cls": "Southern Europe"}, {"id": 18547, "dom": "G", "ent": "Keszthely", "cls": "Western Africa"}, {"id": 18548, "dom": "G", "ent": "Keszthely", "cls": "Western Asia"}, {"id": 18549, "dom": "G", "ent": "Keszthely", "cls": "Western Europe"}, {"id": 18550, "dom": "G", "ent": "Dunaharaszti", "cls": "Australia and New Zealand"}, {"id": 18551, "dom": "G", "ent": "Dunaharaszti", "cls": "Caribbean"}, {"id": 18552, "dom": "G", "ent": "Dunaharaszti", "cls": "Central America"}, {"id": 18553, "dom": "G", "ent": "Dunaharaszti", "cls": "Central Asia"}, {"id": 18554, "dom": "G", "ent": "Dunaharaszti", "cls": "Eastern Africa"}, {"id": 18555, "dom": "G", "ent": "Dunaharaszti", "cls": "Eastern Asia"}, {"id": 18556, "dom": "G", "ent": "Dunaharaszti", "cls": "Eastern Europe"}, {"id": 18557, "dom": "G", "ent": "Dunaharaszti", "cls": "Melanesia"}, {"id": 18558, "dom": "G", "ent": "Dunaharaszti", "cls": "Micronesia"}, {"id": 18559, "dom": "G", "ent": "Dunaharaszti", "cls": "Middle Africa"}, {"id": 18560, "dom": "G", "ent": "Dunaharaszti", "cls": "Northern Africa"}, {"id": 18561, "dom": "G", "ent": "Dunaharaszti", "cls": "Northern America"}, {"id": 18562, "dom": "G", "ent": "Dunaharaszti", "cls": "Northern Europe"}, {"id": 18563, "dom": "G", "ent": "Dunaharaszti", "cls": "Polynesia"}, {"id": 18564, "dom": "G", "ent": "Dunaharaszti", "cls": "South America"}, {"id": 18565, "dom": "G", "ent": "Dunaharaszti", "cls": "South-eastern Asia"}, {"id": 18566, "dom": "G", "ent": "Dunaharaszti", "cls": "Southern Africa"}, {"id": 18567, "dom": "G", "ent": "Dunaharaszti", "cls": "Southern Asia"}, {"id": 18568, "dom": "G", "ent": "Dunaharaszti", "cls": "Southern Europe"}, {"id": 18569, "dom": "G", "ent": "Dunaharaszti", "cls": "Western Africa"}, {"id": 18570, "dom": "G", "ent": "Dunaharaszti", "cls": "Western Asia"}, {"id": 18571, "dom": "G", "ent": "Dunaharaszti", "cls": "Western Europe"}, {"id": 18572, "dom": "G", "ent": "Letterkenny", "cls": "Australia and New Zealand"}, {"id": 18573, "dom": "G", "ent": "Letterkenny", "cls": "Caribbean"}, {"id": 18574, "dom": "G", "ent": "Letterkenny", "cls": "Central America"}, {"id": 18575, "dom": "G", "ent": "Letterkenny", "cls": "Central Asia"}, {"id": 18576, "dom": "G", "ent": "Letterkenny", "cls": "Eastern Africa"}, {"id": 18577, "dom": "G", "ent": "Letterkenny", "cls": "Eastern Asia"}, {"id": 18578, "dom": "G", "ent": "Letterkenny", "cls": "Eastern Europe"}, {"id": 18579, "dom": "G", "ent": "Letterkenny", "cls": "Melanesia"}, {"id": 18580, "dom": "G", "ent": "Letterkenny", "cls": "Micronesia"}, {"id": 18581, "dom": "G", "ent": "Letterkenny", "cls": "Middle Africa"}, {"id": 18582, "dom": "G", "ent": "Letterkenny", "cls": "Northern Africa"}, {"id": 18583, "dom": "G", "ent": "Letterkenny", "cls": "Northern America"}, {"id": 18584, "dom": "G", "ent": "Letterkenny", "cls": "Northern Europe"}, {"id": 18585, "dom": "G", "ent": "Letterkenny", "cls": "Polynesia"}, {"id": 18586, "dom": "G", "ent": "Letterkenny", "cls": "South America"}, {"id": 18587, "dom": "G", "ent": "Letterkenny", "cls": "South-eastern Asia"}, {"id": 18588, "dom": "G", "ent": "Letterkenny", "cls": "Southern Africa"}, {"id": 18589, "dom": "G", "ent": "Letterkenny", "cls": "Southern Asia"}, {"id": 18590, "dom": "G", "ent": "Letterkenny", "cls": "Southern Europe"}, {"id": 18591, "dom": "G", "ent": "Letterkenny", "cls": "Western Africa"}, {"id": 18592, "dom": "G", "ent": "Letterkenny", "cls": "Western Asia"}, {"id": 18593, "dom": "G", "ent": "Letterkenny", "cls": "Western Europe"}, {"id": 18594, "dom": "G", "ent": "Greystones", "cls": "Australia and New Zealand"}, {"id": 18595, "dom": "G", "ent": "Greystones", "cls": "Caribbean"}, {"id": 18596, "dom": "G", "ent": "Greystones", "cls": "Central America"}, {"id": 18597, "dom": "G", "ent": "Greystones", "cls": "Central Asia"}, {"id": 18598, "dom": "G", "ent": "Greystones", "cls": "Eastern Africa"}, {"id": 18599, "dom": "G", "ent": "Greystones", "cls": "Eastern Asia"}, {"id": 18600, "dom": "G", "ent": "Greystones", "cls": "Eastern Europe"}, {"id": 18601, "dom": "G", "ent": "Greystones", "cls": "Melanesia"}, {"id": 18602, "dom": "G", "ent": "Greystones", "cls": "Micronesia"}, {"id": 18603, "dom": "G", "ent": "Greystones", "cls": "Middle Africa"}, {"id": 18604, "dom": "G", "ent": "Greystones", "cls": "Northern Africa"}, {"id": 18605, "dom": "G", "ent": "Greystones", "cls": "Northern America"}, {"id": 18606, "dom": "G", "ent": "Greystones", "cls": "Northern Europe"}, {"id": 18607, "dom": "G", "ent": "Greystones", "cls": "Polynesia"}, {"id": 18608, "dom": "G", "ent": "Greystones", "cls": "South America"}, {"id": 18609, "dom": "G", "ent": "Greystones", "cls": "South-eastern Asia"}, {"id": 18610, "dom": "G", "ent": "Greystones", "cls": "Southern Africa"}, {"id": 18611, "dom": "G", "ent": "Greystones", "cls": "Southern Asia"}, {"id": 18612, "dom": "G", "ent": "Greystones", "cls": "Southern Europe"}, {"id": 18613, "dom": "G", "ent": "Greystones", "cls": "Western Africa"}, {"id": 18614, "dom": "G", "ent": "Greystones", "cls": "Western Asia"}, {"id": 18615, "dom": "G", "ent": "Greystones", "cls": "Western Europe"}, {"id": 18616, "dom": "G", "ent": "Balbriggan", "cls": "Australia and New Zealand"}, {"id": 18617, "dom": "G", "ent": "Balbriggan", "cls": "Caribbean"}, {"id": 18618, "dom": "G", "ent": "Balbriggan", "cls": "Central America"}, {"id": 18619, "dom": "G", "ent": "Balbriggan", "cls": "Central Asia"}, {"id": 18620, "dom": "G", "ent": "Balbriggan", "cls": "Eastern Africa"}, {"id": 18621, "dom": "G", "ent": "Balbriggan", "cls": "Eastern Asia"}, {"id": 18622, "dom": "G", "ent": "Balbriggan", "cls": "Eastern Europe"}, {"id": 18623, "dom": "G", "ent": "Balbriggan", "cls": "Melanesia"}, {"id": 18624, "dom": "G", "ent": "Balbriggan", "cls": "Micronesia"}, {"id": 18625, "dom": "G", "ent": "Balbriggan", "cls": "Middle Africa"}, {"id": 18626, "dom": "G", "ent": "Balbriggan", "cls": "Northern Africa"}, {"id": 18627, "dom": "G", "ent": "Balbriggan", "cls": "Northern America"}, {"id": 18628, "dom": "G", "ent": "Balbriggan", "cls": "Northern Europe"}, {"id": 18629, "dom": "G", "ent": "Balbriggan", "cls": "Polynesia"}, {"id": 18630, "dom": "G", "ent": "Balbriggan", "cls": "South America"}, {"id": 18631, "dom": "G", "ent": "Balbriggan", "cls": "South-eastern Asia"}, {"id": 18632, "dom": "G", "ent": "Balbriggan", "cls": "Southern Africa"}, {"id": 18633, "dom": "G", "ent": "Balbriggan", "cls": "Southern Asia"}, {"id": 18634, "dom": "G", "ent": "Balbriggan", "cls": "Southern Europe"}, {"id": 18635, "dom": "G", "ent": "Balbriggan", "cls": "Western Africa"}, {"id": 18636, "dom": "G", "ent": "Balbriggan", "cls": "Western Asia"}, {"id": 18637, "dom": "G", "ent": "Balbriggan", "cls": "Western Europe"}, {"id": 18638, "dom": "G", "ent": "Lucan", "cls": "Australia and New Zealand"}, {"id": 18639, "dom": "G", "ent": "Lucan", "cls": "Caribbean"}, {"id": 18640, "dom": "G", "ent": "Lucan", "cls": "Central America"}, {"id": 18641, "dom": "G", "ent": "Lucan", "cls": "Central Asia"}, {"id": 18642, "dom": "G", "ent": "Lucan", "cls": "Eastern Africa"}, {"id": 18643, "dom": "G", "ent": "Lucan", "cls": "Eastern Asia"}, {"id": 18644, "dom": "G", "ent": "Lucan", "cls": "Eastern Europe"}, {"id": 18645, "dom": "G", "ent": "Lucan", "cls": "Melanesia"}, {"id": 18646, "dom": "G", "ent": "Lucan", "cls": "Micronesia"}, {"id": 18647, "dom": "G", "ent": "Lucan", "cls": "Middle Africa"}, {"id": 18648, "dom": "G", "ent": "Lucan", "cls": "Northern Africa"}, {"id": 18649, "dom": "G", "ent": "Lucan", "cls": "Northern America"}, {"id": 18650, "dom": "G", "ent": "Lucan", "cls": "Northern Europe"}, {"id": 18651, "dom": "G", "ent": "Lucan", "cls": "Polynesia"}, {"id": 18652, "dom": "G", "ent": "Lucan", "cls": "South America"}, {"id": 18653, "dom": "G", "ent": "Lucan", "cls": "South-eastern Asia"}, {"id": 18654, "dom": "G", "ent": "Lucan", "cls": "Southern Africa"}, {"id": 18655, "dom": "G", "ent": "Lucan", "cls": "Southern Asia"}, {"id": 18656, "dom": "G", "ent": "Lucan", "cls": "Southern Europe"}, {"id": 18657, "dom": "G", "ent": "Lucan", "cls": "Western Africa"}, {"id": 18658, "dom": "G", "ent": "Lucan", "cls": "Western Asia"}, {"id": 18659, "dom": "G", "ent": "Lucan", "cls": "Western Europe"}, {"id": 18660, "dom": "G", "ent": "Ramat Gan", "cls": "Australia and New Zealand"}, {"id": 18661, "dom": "G", "ent": "Ramat Gan", "cls": "Caribbean"}, {"id": 18662, "dom": "G", "ent": "Ramat Gan", "cls": "Central America"}, {"id": 18663, "dom": "G", "ent": "Ramat Gan", "cls": "Central Asia"}, {"id": 18664, "dom": "G", "ent": "Ramat Gan", "cls": "Eastern Africa"}, {"id": 18665, "dom": "G", "ent": "Ramat Gan", "cls": "Eastern Asia"}, {"id": 18666, "dom": "G", "ent": "Ramat Gan", "cls": "Eastern Europe"}, {"id": 18667, "dom": "G", "ent": "Ramat Gan", "cls": "Melanesia"}, {"id": 18668, "dom": "G", "ent": "Ramat Gan", "cls": "Micronesia"}, {"id": 18669, "dom": "G", "ent": "Ramat Gan", "cls": "Middle Africa"}, {"id": 18670, "dom": "G", "ent": "Ramat Gan", "cls": "Northern Africa"}, {"id": 18671, "dom": "G", "ent": "Ramat Gan", "cls": "Northern America"}, {"id": 18672, "dom": "G", "ent": "Ramat Gan", "cls": "Northern Europe"}, {"id": 18673, "dom": "G", "ent": "Ramat Gan", "cls": "Polynesia"}, {"id": 18674, "dom": "G", "ent": "Ramat Gan", "cls": "South America"}, {"id": 18675, "dom": "G", "ent": "Ramat Gan", "cls": "South-eastern Asia"}, {"id": 18676, "dom": "G", "ent": "Ramat Gan", "cls": "Southern Africa"}, {"id": 18677, "dom": "G", "ent": "Ramat Gan", "cls": "Southern Asia"}, {"id": 18678, "dom": "G", "ent": "Ramat Gan", "cls": "Southern Europe"}, {"id": 18679, "dom": "G", "ent": "Ramat Gan", "cls": "Western Africa"}, {"id": 18680, "dom": "G", "ent": "Ramat Gan", "cls": "Western Asia"}, {"id": 18681, "dom": "G", "ent": "Ramat Gan", "cls": "Western Europe"}, {"id": 18682, "dom": "G", "ent": "Jaffa", "cls": "Australia and New Zealand"}, {"id": 18683, "dom": "G", "ent": "Jaffa", "cls": "Caribbean"}, {"id": 18684, "dom": "G", "ent": "Jaffa", "cls": "Central America"}, {"id": 18685, "dom": "G", "ent": "Jaffa", "cls": "Central Asia"}, {"id": 18686, "dom": "G", "ent": "Jaffa", "cls": "Eastern Africa"}, {"id": 18687, "dom": "G", "ent": "Jaffa", "cls": "Eastern Asia"}, {"id": 18688, "dom": "G", "ent": "Jaffa", "cls": "Eastern Europe"}, {"id": 18689, "dom": "G", "ent": "Jaffa", "cls": "Melanesia"}, {"id": 18690, "dom": "G", "ent": "Jaffa", "cls": "Micronesia"}, {"id": 18691, "dom": "G", "ent": "Jaffa", "cls": "Middle Africa"}, {"id": 18692, "dom": "G", "ent": "Jaffa", "cls": "Northern Africa"}, {"id": 18693, "dom": "G", "ent": "Jaffa", "cls": "Northern America"}, {"id": 18694, "dom": "G", "ent": "Jaffa", "cls": "Northern Europe"}, {"id": 18695, "dom": "G", "ent": "Jaffa", "cls": "Polynesia"}, {"id": 18696, "dom": "G", "ent": "Jaffa", "cls": "South America"}, {"id": 18697, "dom": "G", "ent": "Jaffa", "cls": "South-eastern Asia"}, {"id": 18698, "dom": "G", "ent": "Jaffa", "cls": "Southern Africa"}, {"id": 18699, "dom": "G", "ent": "Jaffa", "cls": "Southern Asia"}, {"id": 18700, "dom": "G", "ent": "Jaffa", "cls": "Southern Europe"}, {"id": 18701, "dom": "G", "ent": "Jaffa", "cls": "Western Africa"}, {"id": 18702, "dom": "G", "ent": "Jaffa", "cls": "Western Asia"}, {"id": 18703, "dom": "G", "ent": "Jaffa", "cls": "Western Europe"}, {"id": 18704, "dom": "G", "ent": "Givatayim", "cls": "Australia and New Zealand"}, {"id": 18705, "dom": "G", "ent": "Givatayim", "cls": "Caribbean"}, {"id": 18706, "dom": "G", "ent": "Givatayim", "cls": "Central America"}, {"id": 18707, "dom": "G", "ent": "Givatayim", "cls": "Central Asia"}, {"id": 18708, "dom": "G", "ent": "Givatayim", "cls": "Eastern Africa"}, {"id": 18709, "dom": "G", "ent": "Givatayim", "cls": "Eastern Asia"}, {"id": 18710, "dom": "G", "ent": "Givatayim", "cls": "Eastern Europe"}, {"id": 18711, "dom": "G", "ent": "Givatayim", "cls": "Melanesia"}, {"id": 18712, "dom": "G", "ent": "Givatayim", "cls": "Micronesia"}, {"id": 18713, "dom": "G", "ent": "Givatayim", "cls": "Middle Africa"}, {"id": 18714, "dom": "G", "ent": "Givatayim", "cls": "Northern Africa"}, {"id": 18715, "dom": "G", "ent": "Givatayim", "cls": "Northern America"}, {"id": 18716, "dom": "G", "ent": "Givatayim", "cls": "Northern Europe"}, {"id": 18717, "dom": "G", "ent": "Givatayim", "cls": "Polynesia"}, {"id": 18718, "dom": "G", "ent": "Givatayim", "cls": "South America"}, {"id": 18719, "dom": "G", "ent": "Givatayim", "cls": "South-eastern Asia"}, {"id": 18720, "dom": "G", "ent": "Givatayim", "cls": "Southern Africa"}, {"id": 18721, "dom": "G", "ent": "Givatayim", "cls": "Southern Asia"}, {"id": 18722, "dom": "G", "ent": "Givatayim", "cls": "Southern Europe"}, {"id": 18723, "dom": "G", "ent": "Givatayim", "cls": "Western Africa"}, {"id": 18724, "dom": "G", "ent": "Givatayim", "cls": "Western Asia"}, {"id": 18725, "dom": "G", "ent": "Givatayim", "cls": "Western Europe"}, {"id": 18726, "dom": "G", "ent": "Or Yehuda", "cls": "Australia and New Zealand"}, {"id": 18727, "dom": "G", "ent": "Or Yehuda", "cls": "Caribbean"}, {"id": 18728, "dom": "G", "ent": "Or Yehuda", "cls": "Central America"}, {"id": 18729, "dom": "G", "ent": "Or Yehuda", "cls": "Central Asia"}, {"id": 18730, "dom": "G", "ent": "Or Yehuda", "cls": "Eastern Africa"}, {"id": 18731, "dom": "G", "ent": "Or Yehuda", "cls": "Eastern Asia"}, {"id": 18732, "dom": "G", "ent": "Or Yehuda", "cls": "Eastern Europe"}, {"id": 18733, "dom": "G", "ent": "Or Yehuda", "cls": "Melanesia"}, {"id": 18734, "dom": "G", "ent": "Or Yehuda", "cls": "Micronesia"}, {"id": 18735, "dom": "G", "ent": "Or Yehuda", "cls": "Middle Africa"}, {"id": 18736, "dom": "G", "ent": "Or Yehuda", "cls": "Northern Africa"}, {"id": 18737, "dom": "G", "ent": "Or Yehuda", "cls": "Northern America"}, {"id": 18738, "dom": "G", "ent": "Or Yehuda", "cls": "Northern Europe"}, {"id": 18739, "dom": "G", "ent": "Or Yehuda", "cls": "Polynesia"}, {"id": 18740, "dom": "G", "ent": "Or Yehuda", "cls": "South America"}, {"id": 18741, "dom": "G", "ent": "Or Yehuda", "cls": "South-eastern Asia"}, {"id": 18742, "dom": "G", "ent": "Or Yehuda", "cls": "Southern Africa"}, {"id": 18743, "dom": "G", "ent": "Or Yehuda", "cls": "Southern Asia"}, {"id": 18744, "dom": "G", "ent": "Or Yehuda", "cls": "Southern Europe"}, {"id": 18745, "dom": "G", "ent": "Or Yehuda", "cls": "Western Africa"}, {"id": 18746, "dom": "G", "ent": "Or Yehuda", "cls": "Western Asia"}, {"id": 18747, "dom": "G", "ent": "Or Yehuda", "cls": "Western Europe"}, {"id": 18748, "dom": "G", "ent": "Pallichal", "cls": "Australia and New Zealand"}, {"id": 18749, "dom": "G", "ent": "Pallichal", "cls": "Caribbean"}, {"id": 18750, "dom": "G", "ent": "Pallichal", "cls": "Central America"}, {"id": 18751, "dom": "G", "ent": "Pallichal", "cls": "Central Asia"}, {"id": 18752, "dom": "G", "ent": "Pallichal", "cls": "Eastern Africa"}, {"id": 18753, "dom": "G", "ent": "Pallichal", "cls": "Eastern Asia"}, {"id": 18754, "dom": "G", "ent": "Pallichal", "cls": "Eastern Europe"}, {"id": 18755, "dom": "G", "ent": "Pallichal", "cls": "Melanesia"}, {"id": 18756, "dom": "G", "ent": "Pallichal", "cls": "Micronesia"}, {"id": 18757, "dom": "G", "ent": "Pallichal", "cls": "Middle Africa"}, {"id": 18758, "dom": "G", "ent": "Pallichal", "cls": "Northern Africa"}, {"id": 18759, "dom": "G", "ent": "Pallichal", "cls": "Northern America"}, {"id": 18760, "dom": "G", "ent": "Pallichal", "cls": "Northern Europe"}, {"id": 18761, "dom": "G", "ent": "Pallichal", "cls": "Polynesia"}, {"id": 18762, "dom": "G", "ent": "Pallichal", "cls": "South America"}, {"id": 18763, "dom": "G", "ent": "Pallichal", "cls": "South-eastern Asia"}, {"id": 18764, "dom": "G", "ent": "Pallichal", "cls": "Southern Africa"}, {"id": 18765, "dom": "G", "ent": "Pallichal", "cls": "Southern Asia"}, {"id": 18766, "dom": "G", "ent": "Pallichal", "cls": "Southern Europe"}, {"id": 18767, "dom": "G", "ent": "Pallichal", "cls": "Western Africa"}, {"id": 18768, "dom": "G", "ent": "Pallichal", "cls": "Western Asia"}, {"id": 18769, "dom": "G", "ent": "Pallichal", "cls": "Western Europe"}, {"id": 18770, "dom": "G", "ent": "Kotagiri", "cls": "Australia and New Zealand"}, {"id": 18771, "dom": "G", "ent": "Kotagiri", "cls": "Caribbean"}, {"id": 18772, "dom": "G", "ent": "Kotagiri", "cls": "Central America"}, {"id": 18773, "dom": "G", "ent": "Kotagiri", "cls": "Central Asia"}, {"id": 18774, "dom": "G", "ent": "Kotagiri", "cls": "Eastern Africa"}, {"id": 18775, "dom": "G", "ent": "Kotagiri", "cls": "Eastern Asia"}, {"id": 18776, "dom": "G", "ent": "Kotagiri", "cls": "Eastern Europe"}, {"id": 18777, "dom": "G", "ent": "Kotagiri", "cls": "Melanesia"}, {"id": 18778, "dom": "G", "ent": "Kotagiri", "cls": "Micronesia"}, {"id": 18779, "dom": "G", "ent": "Kotagiri", "cls": "Middle Africa"}, {"id": 18780, "dom": "G", "ent": "Kotagiri", "cls": "Northern Africa"}, {"id": 18781, "dom": "G", "ent": "Kotagiri", "cls": "Northern America"}, {"id": 18782, "dom": "G", "ent": "Kotagiri", "cls": "Northern Europe"}, {"id": 18783, "dom": "G", "ent": "Kotagiri", "cls": "Polynesia"}, {"id": 18784, "dom": "G", "ent": "Kotagiri", "cls": "South America"}, {"id": 18785, "dom": "G", "ent": "Kotagiri", "cls": "South-eastern Asia"}, {"id": 18786, "dom": "G", "ent": "Kotagiri", "cls": "Southern Africa"}, {"id": 18787, "dom": "G", "ent": "Kotagiri", "cls": "Southern Asia"}, {"id": 18788, "dom": "G", "ent": "Kotagiri", "cls": "Southern Europe"}, {"id": 18789, "dom": "G", "ent": "Kotagiri", "cls": "Western Africa"}, {"id": 18790, "dom": "G", "ent": "Kotagiri", "cls": "Western Asia"}, {"id": 18791, "dom": "G", "ent": "Kotagiri", "cls": "Western Europe"}, {"id": 18792, "dom": "G", "ent": "Palavakkam", "cls": "Australia and New Zealand"}, {"id": 18793, "dom": "G", "ent": "Palavakkam", "cls": "Caribbean"}, {"id": 18794, "dom": "G", "ent": "Palavakkam", "cls": "Central America"}, {"id": 18795, "dom": "G", "ent": "Palavakkam", "cls": "Central Asia"}, {"id": 18796, "dom": "G", "ent": "Palavakkam", "cls": "Eastern Africa"}, {"id": 18797, "dom": "G", "ent": "Palavakkam", "cls": "Eastern Asia"}, {"id": 18798, "dom": "G", "ent": "Palavakkam", "cls": "Eastern Europe"}, {"id": 18799, "dom": "G", "ent": "Palavakkam", "cls": "Melanesia"}, {"id": 18800, "dom": "G", "ent": "Palavakkam", "cls": "Micronesia"}, {"id": 18801, "dom": "G", "ent": "Palavakkam", "cls": "Middle Africa"}, {"id": 18802, "dom": "G", "ent": "Palavakkam", "cls": "Northern Africa"}, {"id": 18803, "dom": "G", "ent": "Palavakkam", "cls": "Northern America"}, {"id": 18804, "dom": "G", "ent": "Palavakkam", "cls": "Northern Europe"}, {"id": 18805, "dom": "G", "ent": "Palavakkam", "cls": "Polynesia"}, {"id": 18806, "dom": "G", "ent": "Palavakkam", "cls": "South America"}, {"id": 18807, "dom": "G", "ent": "Palavakkam", "cls": "South-eastern Asia"}, {"id": 18808, "dom": "G", "ent": "Palavakkam", "cls": "Southern Africa"}, {"id": 18809, "dom": "G", "ent": "Palavakkam", "cls": "Southern Asia"}, {"id": 18810, "dom": "G", "ent": "Palavakkam", "cls": "Southern Europe"}, {"id": 18811, "dom": "G", "ent": "Palavakkam", "cls": "Western Africa"}, {"id": 18812, "dom": "G", "ent": "Palavakkam", "cls": "Western Asia"}, {"id": 18813, "dom": "G", "ent": "Palavakkam", "cls": "Western Europe"}, {"id": 18814, "dom": "G", "ent": "Govardhan", "cls": "Australia and New Zealand"}, {"id": 18815, "dom": "G", "ent": "Govardhan", "cls": "Caribbean"}, {"id": 18816, "dom": "G", "ent": "Govardhan", "cls": "Central America"}, {"id": 18817, "dom": "G", "ent": "Govardhan", "cls": "Central Asia"}, {"id": 18818, "dom": "G", "ent": "Govardhan", "cls": "Eastern Africa"}, {"id": 18819, "dom": "G", "ent": "Govardhan", "cls": "Eastern Asia"}, {"id": 18820, "dom": "G", "ent": "Govardhan", "cls": "Eastern Europe"}, {"id": 18821, "dom": "G", "ent": "Govardhan", "cls": "Melanesia"}, {"id": 18822, "dom": "G", "ent": "Govardhan", "cls": "Micronesia"}, {"id": 18823, "dom": "G", "ent": "Govardhan", "cls": "Middle Africa"}, {"id": 18824, "dom": "G", "ent": "Govardhan", "cls": "Northern Africa"}, {"id": 18825, "dom": "G", "ent": "Govardhan", "cls": "Northern America"}, {"id": 18826, "dom": "G", "ent": "Govardhan", "cls": "Northern Europe"}, {"id": 18827, "dom": "G", "ent": "Govardhan", "cls": "Polynesia"}, {"id": 18828, "dom": "G", "ent": "Govardhan", "cls": "South America"}, {"id": 18829, "dom": "G", "ent": "Govardhan", "cls": "South-eastern Asia"}, {"id": 18830, "dom": "G", "ent": "Govardhan", "cls": "Southern Africa"}, {"id": 18831, "dom": "G", "ent": "Govardhan", "cls": "Southern Asia"}, {"id": 18832, "dom": "G", "ent": "Govardhan", "cls": "Southern Europe"}, {"id": 18833, "dom": "G", "ent": "Govardhan", "cls": "Western Africa"}, {"id": 18834, "dom": "G", "ent": "Govardhan", "cls": "Western Asia"}, {"id": 18835, "dom": "G", "ent": "Govardhan", "cls": "Western Europe"}, {"id": 18836, "dom": "G", "ent": "Basrah", "cls": "Australia and New Zealand"}, {"id": 18837, "dom": "G", "ent": "Basrah", "cls": "Caribbean"}, {"id": 18838, "dom": "G", "ent": "Basrah", "cls": "Central America"}, {"id": 18839, "dom": "G", "ent": "Basrah", "cls": "Central Asia"}, {"id": 18840, "dom": "G", "ent": "Basrah", "cls": "Eastern Africa"}, {"id": 18841, "dom": "G", "ent": "Basrah", "cls": "Eastern Asia"}, {"id": 18842, "dom": "G", "ent": "Basrah", "cls": "Eastern Europe"}, {"id": 18843, "dom": "G", "ent": "Basrah", "cls": "Melanesia"}, {"id": 18844, "dom": "G", "ent": "Basrah", "cls": "Micronesia"}, {"id": 18845, "dom": "G", "ent": "Basrah", "cls": "Middle Africa"}, {"id": 18846, "dom": "G", "ent": "Basrah", "cls": "Northern Africa"}, {"id": 18847, "dom": "G", "ent": "Basrah", "cls": "Northern America"}, {"id": 18848, "dom": "G", "ent": "Basrah", "cls": "Northern Europe"}, {"id": 18849, "dom": "G", "ent": "Basrah", "cls": "Polynesia"}, {"id": 18850, "dom": "G", "ent": "Basrah", "cls": "South America"}, {"id": 18851, "dom": "G", "ent": "Basrah", "cls": "South-eastern Asia"}, {"id": 18852, "dom": "G", "ent": "Basrah", "cls": "Southern Africa"}, {"id": 18853, "dom": "G", "ent": "Basrah", "cls": "Southern Asia"}, {"id": 18854, "dom": "G", "ent": "Basrah", "cls": "Southern Europe"}, {"id": 18855, "dom": "G", "ent": "Basrah", "cls": "Western Africa"}, {"id": 18856, "dom": "G", "ent": "Basrah", "cls": "Western Asia"}, {"id": 18857, "dom": "G", "ent": "Basrah", "cls": "Western Europe"}, {"id": 18858, "dom": "G", "ent": "Najaf", "cls": "Australia and New Zealand"}, {"id": 18859, "dom": "G", "ent": "Najaf", "cls": "Caribbean"}, {"id": 18860, "dom": "G", "ent": "Najaf", "cls": "Central America"}, {"id": 18861, "dom": "G", "ent": "Najaf", "cls": "Central Asia"}, {"id": 18862, "dom": "G", "ent": "Najaf", "cls": "Eastern Africa"}, {"id": 18863, "dom": "G", "ent": "Najaf", "cls": "Eastern Asia"}, {"id": 18864, "dom": "G", "ent": "Najaf", "cls": "Eastern Europe"}, {"id": 18865, "dom": "G", "ent": "Najaf", "cls": "Melanesia"}, {"id": 18866, "dom": "G", "ent": "Najaf", "cls": "Micronesia"}, {"id": 18867, "dom": "G", "ent": "Najaf", "cls": "Middle Africa"}, {"id": 18868, "dom": "G", "ent": "Najaf", "cls": "Northern Africa"}, {"id": 18869, "dom": "G", "ent": "Najaf", "cls": "Northern America"}, {"id": 18870, "dom": "G", "ent": "Najaf", "cls": "Northern Europe"}, {"id": 18871, "dom": "G", "ent": "Najaf", "cls": "Polynesia"}, {"id": 18872, "dom": "G", "ent": "Najaf", "cls": "South America"}, {"id": 18873, "dom": "G", "ent": "Najaf", "cls": "South-eastern Asia"}, {"id": 18874, "dom": "G", "ent": "Najaf", "cls": "Southern Africa"}, {"id": 18875, "dom": "G", "ent": "Najaf", "cls": "Southern Asia"}, {"id": 18876, "dom": "G", "ent": "Najaf", "cls": "Southern Europe"}, {"id": 18877, "dom": "G", "ent": "Najaf", "cls": "Western Africa"}, {"id": 18878, "dom": "G", "ent": "Najaf", "cls": "Western Asia"}, {"id": 18879, "dom": "G", "ent": "Najaf", "cls": "Western Europe"}, {"id": 18880, "dom": "G", "ent": "Al Diwaniyah", "cls": "Australia and New Zealand"}, {"id": 18881, "dom": "G", "ent": "Al Diwaniyah", "cls": "Caribbean"}, {"id": 18882, "dom": "G", "ent": "Al Diwaniyah", "cls": "Central America"}, {"id": 18883, "dom": "G", "ent": "Al Diwaniyah", "cls": "Central Asia"}, {"id": 18884, "dom": "G", "ent": "Al Diwaniyah", "cls": "Eastern Africa"}, {"id": 18885, "dom": "G", "ent": "Al Diwaniyah", "cls": "Eastern Asia"}, {"id": 18886, "dom": "G", "ent": "Al Diwaniyah", "cls": "Eastern Europe"}, {"id": 18887, "dom": "G", "ent": "Al Diwaniyah", "cls": "Melanesia"}, {"id": 18888, "dom": "G", "ent": "Al Diwaniyah", "cls": "Micronesia"}, {"id": 18889, "dom": "G", "ent": "Al Diwaniyah", "cls": "Middle Africa"}, {"id": 18890, "dom": "G", "ent": "Al Diwaniyah", "cls": "Northern Africa"}, {"id": 18891, "dom": "G", "ent": "Al Diwaniyah", "cls": "Northern America"}, {"id": 18892, "dom": "G", "ent": "Al Diwaniyah", "cls": "Northern Europe"}, {"id": 18893, "dom": "G", "ent": "Al Diwaniyah", "cls": "Polynesia"}, {"id": 18894, "dom": "G", "ent": "Al Diwaniyah", "cls": "South America"}, {"id": 18895, "dom": "G", "ent": "Al Diwaniyah", "cls": "South-eastern Asia"}, {"id": 18896, "dom": "G", "ent": "Al Diwaniyah", "cls": "Southern Africa"}, {"id": 18897, "dom": "G", "ent": "Al Diwaniyah", "cls": "Southern Asia"}, {"id": 18898, "dom": "G", "ent": "Al Diwaniyah", "cls": "Southern Europe"}, {"id": 18899, "dom": "G", "ent": "Al Diwaniyah", "cls": "Western Africa"}, {"id": 18900, "dom": "G", "ent": "Al Diwaniyah", "cls": "Western Asia"}, {"id": 18901, "dom": "G", "ent": "Al Diwaniyah", "cls": "Western Europe"}, {"id": 18902, "dom": "G", "ent": "Kufa", "cls": "Australia and New Zealand"}, {"id": 18903, "dom": "G", "ent": "Kufa", "cls": "Caribbean"}, {"id": 18904, "dom": "G", "ent": "Kufa", "cls": "Central America"}, {"id": 18905, "dom": "G", "ent": "Kufa", "cls": "Central Asia"}, {"id": 18906, "dom": "G", "ent": "Kufa", "cls": "Eastern Africa"}, {"id": 18907, "dom": "G", "ent": "Kufa", "cls": "Eastern Asia"}, {"id": 18908, "dom": "G", "ent": "Kufa", "cls": "Eastern Europe"}, {"id": 18909, "dom": "G", "ent": "Kufa", "cls": "Melanesia"}, {"id": 18910, "dom": "G", "ent": "Kufa", "cls": "Micronesia"}, {"id": 18911, "dom": "G", "ent": "Kufa", "cls": "Middle Africa"}, {"id": 18912, "dom": "G", "ent": "Kufa", "cls": "Northern Africa"}, {"id": 18913, "dom": "G", "ent": "Kufa", "cls": "Northern America"}, {"id": 18914, "dom": "G", "ent": "Kufa", "cls": "Northern Europe"}, {"id": 18915, "dom": "G", "ent": "Kufa", "cls": "Polynesia"}, {"id": 18916, "dom": "G", "ent": "Kufa", "cls": "South America"}, {"id": 18917, "dom": "G", "ent": "Kufa", "cls": "South-eastern Asia"}, {"id": 18918, "dom": "G", "ent": "Kufa", "cls": "Southern Africa"}, {"id": 18919, "dom": "G", "ent": "Kufa", "cls": "Southern Asia"}, {"id": 18920, "dom": "G", "ent": "Kufa", "cls": "Southern Europe"}, {"id": 18921, "dom": "G", "ent": "Kufa", "cls": "Western Africa"}, {"id": 18922, "dom": "G", "ent": "Kufa", "cls": "Western Asia"}, {"id": 18923, "dom": "G", "ent": "Kufa", "cls": "Western Europe"}, {"id": 18924, "dom": "G", "ent": "Kermanshah", "cls": "Australia and New Zealand"}, {"id": 18925, "dom": "G", "ent": "Kermanshah", "cls": "Caribbean"}, {"id": 18926, "dom": "G", "ent": "Kermanshah", "cls": "Central America"}, {"id": 18927, "dom": "G", "ent": "Kermanshah", "cls": "Central Asia"}, {"id": 18928, "dom": "G", "ent": "Kermanshah", "cls": "Eastern Africa"}, {"id": 18929, "dom": "G", "ent": "Kermanshah", "cls": "Eastern Asia"}, {"id": 18930, "dom": "G", "ent": "Kermanshah", "cls": "Eastern Europe"}, {"id": 18931, "dom": "G", "ent": "Kermanshah", "cls": "Melanesia"}, {"id": 18932, "dom": "G", "ent": "Kermanshah", "cls": "Micronesia"}, {"id": 18933, "dom": "G", "ent": "Kermanshah", "cls": "Middle Africa"}, {"id": 18934, "dom": "G", "ent": "Kermanshah", "cls": "Northern Africa"}, {"id": 18935, "dom": "G", "ent": "Kermanshah", "cls": "Northern America"}, {"id": 18936, "dom": "G", "ent": "Kermanshah", "cls": "Northern Europe"}, {"id": 18937, "dom": "G", "ent": "Kermanshah", "cls": "Polynesia"}, {"id": 18938, "dom": "G", "ent": "Kermanshah", "cls": "South America"}, {"id": 18939, "dom": "G", "ent": "Kermanshah", "cls": "South-eastern Asia"}, {"id": 18940, "dom": "G", "ent": "Kermanshah", "cls": "Southern Africa"}, {"id": 18941, "dom": "G", "ent": "Kermanshah", "cls": "Southern Asia"}, {"id": 18942, "dom": "G", "ent": "Kermanshah", "cls": "Southern Europe"}, {"id": 18943, "dom": "G", "ent": "Kermanshah", "cls": "Western Africa"}, {"id": 18944, "dom": "G", "ent": "Kermanshah", "cls": "Western Asia"}, {"id": 18945, "dom": "G", "ent": "Kermanshah", "cls": "Western Europe"}, {"id": 18946, "dom": "G", "ent": "Sirjan", "cls": "Australia and New Zealand"}, {"id": 18947, "dom": "G", "ent": "Sirjan", "cls": "Caribbean"}, {"id": 18948, "dom": "G", "ent": "Sirjan", "cls": "Central America"}, {"id": 18949, "dom": "G", "ent": "Sirjan", "cls": "Central Asia"}, {"id": 18950, "dom": "G", "ent": "Sirjan", "cls": "Eastern Africa"}, {"id": 18951, "dom": "G", "ent": "Sirjan", "cls": "Eastern Asia"}, {"id": 18952, "dom": "G", "ent": "Sirjan", "cls": "Eastern Europe"}, {"id": 18953, "dom": "G", "ent": "Sirjan", "cls": "Melanesia"}, {"id": 18954, "dom": "G", "ent": "Sirjan", "cls": "Micronesia"}, {"id": 18955, "dom": "G", "ent": "Sirjan", "cls": "Middle Africa"}, {"id": 18956, "dom": "G", "ent": "Sirjan", "cls": "Northern Africa"}, {"id": 18957, "dom": "G", "ent": "Sirjan", "cls": "Northern America"}, {"id": 18958, "dom": "G", "ent": "Sirjan", "cls": "Northern Europe"}, {"id": 18959, "dom": "G", "ent": "Sirjan", "cls": "Polynesia"}, {"id": 18960, "dom": "G", "ent": "Sirjan", "cls": "South America"}, {"id": 18961, "dom": "G", "ent": "Sirjan", "cls": "South-eastern Asia"}, {"id": 18962, "dom": "G", "ent": "Sirjan", "cls": "Southern Africa"}, {"id": 18963, "dom": "G", "ent": "Sirjan", "cls": "Southern Asia"}, {"id": 18964, "dom": "G", "ent": "Sirjan", "cls": "Southern Europe"}, {"id": 18965, "dom": "G", "ent": "Sirjan", "cls": "Western Africa"}, {"id": 18966, "dom": "G", "ent": "Sirjan", "cls": "Western Asia"}, {"id": 18967, "dom": "G", "ent": "Sirjan", "cls": "Western Europe"}, {"id": 18968, "dom": "G", "ent": "Hashtgerd", "cls": "Australia and New Zealand"}, {"id": 18969, "dom": "G", "ent": "Hashtgerd", "cls": "Caribbean"}, {"id": 18970, "dom": "G", "ent": "Hashtgerd", "cls": "Central America"}, {"id": 18971, "dom": "G", "ent": "Hashtgerd", "cls": "Central Asia"}, {"id": 18972, "dom": "G", "ent": "Hashtgerd", "cls": "Eastern Africa"}, {"id": 18973, "dom": "G", "ent": "Hashtgerd", "cls": "Eastern Asia"}, {"id": 18974, "dom": "G", "ent": "Hashtgerd", "cls": "Eastern Europe"}, {"id": 18975, "dom": "G", "ent": "Hashtgerd", "cls": "Melanesia"}, {"id": 18976, "dom": "G", "ent": "Hashtgerd", "cls": "Micronesia"}, {"id": 18977, "dom": "G", "ent": "Hashtgerd", "cls": "Middle Africa"}, {"id": 18978, "dom": "G", "ent": "Hashtgerd", "cls": "Northern Africa"}, {"id": 18979, "dom": "G", "ent": "Hashtgerd", "cls": "Northern America"}, {"id": 18980, "dom": "G", "ent": "Hashtgerd", "cls": "Northern Europe"}, {"id": 18981, "dom": "G", "ent": "Hashtgerd", "cls": "Polynesia"}, {"id": 18982, "dom": "G", "ent": "Hashtgerd", "cls": "South America"}, {"id": 18983, "dom": "G", "ent": "Hashtgerd", "cls": "South-eastern Asia"}, {"id": 18984, "dom": "G", "ent": "Hashtgerd", "cls": "Southern Africa"}, {"id": 18985, "dom": "G", "ent": "Hashtgerd", "cls": "Southern Asia"}, {"id": 18986, "dom": "G", "ent": "Hashtgerd", "cls": "Southern Europe"}, {"id": 18987, "dom": "G", "ent": "Hashtgerd", "cls": "Western Africa"}, {"id": 18988, "dom": "G", "ent": "Hashtgerd", "cls": "Western Asia"}, {"id": 18989, "dom": "G", "ent": "Hashtgerd", "cls": "Western Europe"}, {"id": 18990, "dom": "G", "ent": "Hoveyzeh", "cls": "Australia and New Zealand"}, {"id": 18991, "dom": "G", "ent": "Hoveyzeh", "cls": "Caribbean"}, {"id": 18992, "dom": "G", "ent": "Hoveyzeh", "cls": "Central America"}, {"id": 18993, "dom": "G", "ent": "Hoveyzeh", "cls": "Central Asia"}, {"id": 18994, "dom": "G", "ent": "Hoveyzeh", "cls": "Eastern Africa"}, {"id": 18995, "dom": "G", "ent": "Hoveyzeh", "cls": "Eastern Asia"}, {"id": 18996, "dom": "G", "ent": "Hoveyzeh", "cls": "Eastern Europe"}, {"id": 18997, "dom": "G", "ent": "Hoveyzeh", "cls": "Melanesia"}, {"id": 18998, "dom": "G", "ent": "Hoveyzeh", "cls": "Micronesia"}, {"id": 18999, "dom": "G", "ent": "Hoveyzeh", "cls": "Middle Africa"}, {"id": 19000, "dom": "G", "ent": "Hoveyzeh", "cls": "Northern Africa"}, {"id": 19001, "dom": "G", "ent": "Hoveyzeh", "cls": "Northern America"}, {"id": 19002, "dom": "G", "ent": "Hoveyzeh", "cls": "Northern Europe"}, {"id": 19003, "dom": "G", "ent": "Hoveyzeh", "cls": "Polynesia"}, {"id": 19004, "dom": "G", "ent": "Hoveyzeh", "cls": "South America"}, {"id": 19005, "dom": "G", "ent": "Hoveyzeh", "cls": "South-eastern Asia"}, {"id": 19006, "dom": "G", "ent": "Hoveyzeh", "cls": "Southern Africa"}, {"id": 19007, "dom": "G", "ent": "Hoveyzeh", "cls": "Southern Asia"}, {"id": 19008, "dom": "G", "ent": "Hoveyzeh", "cls": "Southern Europe"}, {"id": 19009, "dom": "G", "ent": "Hoveyzeh", "cls": "Western Africa"}, {"id": 19010, "dom": "G", "ent": "Hoveyzeh", "cls": "Western Asia"}, {"id": 19011, "dom": "G", "ent": "Hoveyzeh", "cls": "Western Europe"}, {"id": 19012, "dom": "G", "ent": "Circoiscrizione IV", "cls": "Australia and New Zealand"}, {"id": 19013, "dom": "G", "ent": "Circoiscrizione IV", "cls": "Caribbean"}, {"id": 19014, "dom": "G", "ent": "Circoiscrizione IV", "cls": "Central America"}, {"id": 19015, "dom": "G", "ent": "Circoiscrizione IV", "cls": "Central Asia"}, {"id": 19016, "dom": "G", "ent": "Circoiscrizione IV", "cls": "Eastern Africa"}, {"id": 19017, "dom": "G", "ent": "Circoiscrizione IV", "cls": "Eastern Asia"}, {"id": 19018, "dom": "G", "ent": "Circoiscrizione IV", "cls": "Eastern Europe"}, {"id": 19019, "dom": "G", "ent": "Circoiscrizione IV", "cls": "Melanesia"}, {"id": 19020, "dom": "G", "ent": "Circoiscrizione IV", "cls": "Micronesia"}, {"id": 19021, "dom": "G", "ent": "Circoiscrizione IV", "cls": "Middle Africa"}, {"id": 19022, "dom": "G", "ent": "Circoiscrizione IV", "cls": "Northern Africa"}, {"id": 19023, "dom": "G", "ent": "Circoiscrizione IV", "cls": "Northern America"}, {"id": 19024, "dom": "G", "ent": "Circoiscrizione IV", "cls": "Northern Europe"}, {"id": 19025, "dom": "G", "ent": "Circoiscrizione IV", "cls": "Polynesia"}, {"id": 19026, "dom": "G", "ent": "Circoiscrizione IV", "cls": "South America"}, {"id": 19027, "dom": "G", "ent": "Circoiscrizione IV", "cls": "South-eastern Asia"}, {"id": 19028, "dom": "G", "ent": "Circoiscrizione IV", "cls": "Southern Africa"}, {"id": 19029, "dom": "G", "ent": "Circoiscrizione IV", "cls": "Southern Asia"}, {"id": 19030, "dom": "G", "ent": "Circoiscrizione IV", "cls": "Southern Europe"}, {"id": 19031, "dom": "G", "ent": "Circoiscrizione IV", "cls": "Western Africa"}, {"id": 19032, "dom": "G", "ent": "Circoiscrizione IV", "cls": "Western Asia"}, {"id": 19033, "dom": "G", "ent": "Circoiscrizione IV", "cls": "Western Europe"}, {"id": 19034, "dom": "G", "ent": "Bacoli", "cls": "Australia and New Zealand"}, {"id": 19035, "dom": "G", "ent": "Bacoli", "cls": "Caribbean"}, {"id": 19036, "dom": "G", "ent": "Bacoli", "cls": "Central America"}, {"id": 19037, "dom": "G", "ent": "Bacoli", "cls": "Central Asia"}, {"id": 19038, "dom": "G", "ent": "Bacoli", "cls": "Eastern Africa"}, {"id": 19039, "dom": "G", "ent": "Bacoli", "cls": "Eastern Asia"}, {"id": 19040, "dom": "G", "ent": "Bacoli", "cls": "Eastern Europe"}, {"id": 19041, "dom": "G", "ent": "Bacoli", "cls": "Melanesia"}, {"id": 19042, "dom": "G", "ent": "Bacoli", "cls": "Micronesia"}, {"id": 19043, "dom": "G", "ent": "Bacoli", "cls": "Middle Africa"}, {"id": 19044, "dom": "G", "ent": "Bacoli", "cls": "Northern Africa"}, {"id": 19045, "dom": "G", "ent": "Bacoli", "cls": "Northern America"}, {"id": 19046, "dom": "G", "ent": "Bacoli", "cls": "Northern Europe"}, {"id": 19047, "dom": "G", "ent": "Bacoli", "cls": "Polynesia"}, {"id": 19048, "dom": "G", "ent": "Bacoli", "cls": "South America"}, {"id": 19049, "dom": "G", "ent": "Bacoli", "cls": "South-eastern Asia"}, {"id": 19050, "dom": "G", "ent": "Bacoli", "cls": "Southern Africa"}, {"id": 19051, "dom": "G", "ent": "Bacoli", "cls": "Southern Asia"}, {"id": 19052, "dom": "G", "ent": "Bacoli", "cls": "Southern Europe"}, {"id": 19053, "dom": "G", "ent": "Bacoli", "cls": "Western Africa"}, {"id": 19054, "dom": "G", "ent": "Bacoli", "cls": "Western Asia"}, {"id": 19055, "dom": "G", "ent": "Bacoli", "cls": "Western Europe"}, {"id": 19056, "dom": "G", "ent": "Villaricca", "cls": "Australia and New Zealand"}, {"id": 19057, "dom": "G", "ent": "Villaricca", "cls": "Caribbean"}, {"id": 19058, "dom": "G", "ent": "Villaricca", "cls": "Central America"}, {"id": 19059, "dom": "G", "ent": "Villaricca", "cls": "Central Asia"}, {"id": 19060, "dom": "G", "ent": "Villaricca", "cls": "Eastern Africa"}, {"id": 19061, "dom": "G", "ent": "Villaricca", "cls": "Eastern Asia"}, {"id": 19062, "dom": "G", "ent": "Villaricca", "cls": "Eastern Europe"}, {"id": 19063, "dom": "G", "ent": "Villaricca", "cls": "Melanesia"}, {"id": 19064, "dom": "G", "ent": "Villaricca", "cls": "Micronesia"}, {"id": 19065, "dom": "G", "ent": "Villaricca", "cls": "Middle Africa"}, {"id": 19066, "dom": "G", "ent": "Villaricca", "cls": "Northern Africa"}, {"id": 19067, "dom": "G", "ent": "Villaricca", "cls": "Northern America"}, {"id": 19068, "dom": "G", "ent": "Villaricca", "cls": "Northern Europe"}, {"id": 19069, "dom": "G", "ent": "Villaricca", "cls": "Polynesia"}, {"id": 19070, "dom": "G", "ent": "Villaricca", "cls": "South America"}, {"id": 19071, "dom": "G", "ent": "Villaricca", "cls": "South-eastern Asia"}, {"id": 19072, "dom": "G", "ent": "Villaricca", "cls": "Southern Africa"}, {"id": 19073, "dom": "G", "ent": "Villaricca", "cls": "Southern Asia"}, {"id": 19074, "dom": "G", "ent": "Villaricca", "cls": "Southern Europe"}, {"id": 19075, "dom": "G", "ent": "Villaricca", "cls": "Western Africa"}, {"id": 19076, "dom": "G", "ent": "Villaricca", "cls": "Western Asia"}, {"id": 19077, "dom": "G", "ent": "Villaricca", "cls": "Western Europe"}, {"id": 19078, "dom": "G", "ent": "Fossano", "cls": "Australia and New Zealand"}, {"id": 19079, "dom": "G", "ent": "Fossano", "cls": "Caribbean"}, {"id": 19080, "dom": "G", "ent": "Fossano", "cls": "Central America"}, {"id": 19081, "dom": "G", "ent": "Fossano", "cls": "Central Asia"}, {"id": 19082, "dom": "G", "ent": "Fossano", "cls": "Eastern Africa"}, {"id": 19083, "dom": "G", "ent": "Fossano", "cls": "Eastern Asia"}, {"id": 19084, "dom": "G", "ent": "Fossano", "cls": "Eastern Europe"}, {"id": 19085, "dom": "G", "ent": "Fossano", "cls": "Melanesia"}, {"id": 19086, "dom": "G", "ent": "Fossano", "cls": "Micronesia"}, {"id": 19087, "dom": "G", "ent": "Fossano", "cls": "Middle Africa"}, {"id": 19088, "dom": "G", "ent": "Fossano", "cls": "Northern Africa"}, {"id": 19089, "dom": "G", "ent": "Fossano", "cls": "Northern America"}, {"id": 19090, "dom": "G", "ent": "Fossano", "cls": "Northern Europe"}, {"id": 19091, "dom": "G", "ent": "Fossano", "cls": "Polynesia"}, {"id": 19092, "dom": "G", "ent": "Fossano", "cls": "South America"}, {"id": 19093, "dom": "G", "ent": "Fossano", "cls": "South-eastern Asia"}, {"id": 19094, "dom": "G", "ent": "Fossano", "cls": "Southern Africa"}, {"id": 19095, "dom": "G", "ent": "Fossano", "cls": "Southern Asia"}, {"id": 19096, "dom": "G", "ent": "Fossano", "cls": "Southern Europe"}, {"id": 19097, "dom": "G", "ent": "Fossano", "cls": "Western Africa"}, {"id": 19098, "dom": "G", "ent": "Fossano", "cls": "Western Asia"}, {"id": 19099, "dom": "G", "ent": "Fossano", "cls": "Western Europe"}, {"id": 19100, "dom": "G", "ent": "Mandeville", "cls": "Australia and New Zealand"}, {"id": 19101, "dom": "G", "ent": "Mandeville", "cls": "Caribbean"}, {"id": 19102, "dom": "G", "ent": "Mandeville", "cls": "Central America"}, {"id": 19103, "dom": "G", "ent": "Mandeville", "cls": "Central Asia"}, {"id": 19104, "dom": "G", "ent": "Mandeville", "cls": "Eastern Africa"}, {"id": 19105, "dom": "G", "ent": "Mandeville", "cls": "Eastern Asia"}, {"id": 19106, "dom": "G", "ent": "Mandeville", "cls": "Eastern Europe"}, {"id": 19107, "dom": "G", "ent": "Mandeville", "cls": "Melanesia"}, {"id": 19108, "dom": "G", "ent": "Mandeville", "cls": "Micronesia"}, {"id": 19109, "dom": "G", "ent": "Mandeville", "cls": "Middle Africa"}, {"id": 19110, "dom": "G", "ent": "Mandeville", "cls": "Northern Africa"}, {"id": 19111, "dom": "G", "ent": "Mandeville", "cls": "Northern America"}, {"id": 19112, "dom": "G", "ent": "Mandeville", "cls": "Northern Europe"}, {"id": 19113, "dom": "G", "ent": "Mandeville", "cls": "Polynesia"}, {"id": 19114, "dom": "G", "ent": "Mandeville", "cls": "South America"}, {"id": 19115, "dom": "G", "ent": "Mandeville", "cls": "South-eastern Asia"}, {"id": 19116, "dom": "G", "ent": "Mandeville", "cls": "Southern Africa"}, {"id": 19117, "dom": "G", "ent": "Mandeville", "cls": "Southern Asia"}, {"id": 19118, "dom": "G", "ent": "Mandeville", "cls": "Southern Europe"}, {"id": 19119, "dom": "G", "ent": "Mandeville", "cls": "Western Africa"}, {"id": 19120, "dom": "G", "ent": "Mandeville", "cls": "Western Asia"}, {"id": 19121, "dom": "G", "ent": "Mandeville", "cls": "Western Europe"}, {"id": 19122, "dom": "G", "ent": "May Pen", "cls": "Australia and New Zealand"}, {"id": 19123, "dom": "G", "ent": "May Pen", "cls": "Caribbean"}, {"id": 19124, "dom": "G", "ent": "May Pen", "cls": "Central America"}, {"id": 19125, "dom": "G", "ent": "May Pen", "cls": "Central Asia"}, {"id": 19126, "dom": "G", "ent": "May Pen", "cls": "Eastern Africa"}, {"id": 19127, "dom": "G", "ent": "May Pen", "cls": "Eastern Asia"}, {"id": 19128, "dom": "G", "ent": "May Pen", "cls": "Eastern Europe"}, {"id": 19129, "dom": "G", "ent": "May Pen", "cls": "Melanesia"}, {"id": 19130, "dom": "G", "ent": "May Pen", "cls": "Micronesia"}, {"id": 19131, "dom": "G", "ent": "May Pen", "cls": "Middle Africa"}, {"id": 19132, "dom": "G", "ent": "May Pen", "cls": "Northern Africa"}, {"id": 19133, "dom": "G", "ent": "May Pen", "cls": "Northern America"}, {"id": 19134, "dom": "G", "ent": "May Pen", "cls": "Northern Europe"}, {"id": 19135, "dom": "G", "ent": "May Pen", "cls": "Polynesia"}, {"id": 19136, "dom": "G", "ent": "May Pen", "cls": "South America"}, {"id": 19137, "dom": "G", "ent": "May Pen", "cls": "South-eastern Asia"}, {"id": 19138, "dom": "G", "ent": "May Pen", "cls": "Southern Africa"}, {"id": 19139, "dom": "G", "ent": "May Pen", "cls": "Southern Asia"}, {"id": 19140, "dom": "G", "ent": "May Pen", "cls": "Southern Europe"}, {"id": 19141, "dom": "G", "ent": "May Pen", "cls": "Western Africa"}, {"id": 19142, "dom": "G", "ent": "May Pen", "cls": "Western Asia"}, {"id": 19143, "dom": "G", "ent": "May Pen", "cls": "Western Europe"}, {"id": 19144, "dom": "G", "ent": "Ocho Rios", "cls": "Australia and New Zealand"}, {"id": 19145, "dom": "G", "ent": "Ocho Rios", "cls": "Caribbean"}, {"id": 19146, "dom": "G", "ent": "Ocho Rios", "cls": "Central America"}, {"id": 19147, "dom": "G", "ent": "Ocho Rios", "cls": "Central Asia"}, {"id": 19148, "dom": "G", "ent": "Ocho Rios", "cls": "Eastern Africa"}, {"id": 19149, "dom": "G", "ent": "Ocho Rios", "cls": "Eastern Asia"}, {"id": 19150, "dom": "G", "ent": "Ocho Rios", "cls": "Eastern Europe"}, {"id": 19151, "dom": "G", "ent": "Ocho Rios", "cls": "Melanesia"}, {"id": 19152, "dom": "G", "ent": "Ocho Rios", "cls": "Micronesia"}, {"id": 19153, "dom": "G", "ent": "Ocho Rios", "cls": "Middle Africa"}, {"id": 19154, "dom": "G", "ent": "Ocho Rios", "cls": "Northern Africa"}, {"id": 19155, "dom": "G", "ent": "Ocho Rios", "cls": "Northern America"}, {"id": 19156, "dom": "G", "ent": "Ocho Rios", "cls": "Northern Europe"}, {"id": 19157, "dom": "G", "ent": "Ocho Rios", "cls": "Polynesia"}, {"id": 19158, "dom": "G", "ent": "Ocho Rios", "cls": "South America"}, {"id": 19159, "dom": "G", "ent": "Ocho Rios", "cls": "South-eastern Asia"}, {"id": 19160, "dom": "G", "ent": "Ocho Rios", "cls": "Southern Africa"}, {"id": 19161, "dom": "G", "ent": "Ocho Rios", "cls": "Southern Asia"}, {"id": 19162, "dom": "G", "ent": "Ocho Rios", "cls": "Southern Europe"}, {"id": 19163, "dom": "G", "ent": "Ocho Rios", "cls": "Western Africa"}, {"id": 19164, "dom": "G", "ent": "Ocho Rios", "cls": "Western Asia"}, {"id": 19165, "dom": "G", "ent": "Ocho Rios", "cls": "Western Europe"}, {"id": 19166, "dom": "G", "ent": "Half Way Tree", "cls": "Australia and New Zealand"}, {"id": 19167, "dom": "G", "ent": "Half Way Tree", "cls": "Caribbean"}, {"id": 19168, "dom": "G", "ent": "Half Way Tree", "cls": "Central America"}, {"id": 19169, "dom": "G", "ent": "Half Way Tree", "cls": "Central Asia"}, {"id": 19170, "dom": "G", "ent": "Half Way Tree", "cls": "Eastern Africa"}, {"id": 19171, "dom": "G", "ent": "Half Way Tree", "cls": "Eastern Asia"}, {"id": 19172, "dom": "G", "ent": "Half Way Tree", "cls": "Eastern Europe"}, {"id": 19173, "dom": "G", "ent": "Half Way Tree", "cls": "Melanesia"}, {"id": 19174, "dom": "G", "ent": "Half Way Tree", "cls": "Micronesia"}, {"id": 19175, "dom": "G", "ent": "Half Way Tree", "cls": "Middle Africa"}, {"id": 19176, "dom": "G", "ent": "Half Way Tree", "cls": "Northern Africa"}, {"id": 19177, "dom": "G", "ent": "Half Way Tree", "cls": "Northern America"}, {"id": 19178, "dom": "G", "ent": "Half Way Tree", "cls": "Northern Europe"}, {"id": 19179, "dom": "G", "ent": "Half Way Tree", "cls": "Polynesia"}, {"id": 19180, "dom": "G", "ent": "Half Way Tree", "cls": "South America"}, {"id": 19181, "dom": "G", "ent": "Half Way Tree", "cls": "South-eastern Asia"}, {"id": 19182, "dom": "G", "ent": "Half Way Tree", "cls": "Southern Africa"}, {"id": 19183, "dom": "G", "ent": "Half Way Tree", "cls": "Southern Asia"}, {"id": 19184, "dom": "G", "ent": "Half Way Tree", "cls": "Southern Europe"}, {"id": 19185, "dom": "G", "ent": "Half Way Tree", "cls": "Western Africa"}, {"id": 19186, "dom": "G", "ent": "Half Way Tree", "cls": "Western Asia"}, {"id": 19187, "dom": "G", "ent": "Half Way Tree", "cls": "Western Europe"}, {"id": 19188, "dom": "G", "ent": "Zarqa", "cls": "Australia and New Zealand"}, {"id": 19189, "dom": "G", "ent": "Zarqa", "cls": "Caribbean"}, {"id": 19190, "dom": "G", "ent": "Zarqa", "cls": "Central America"}, {"id": 19191, "dom": "G", "ent": "Zarqa", "cls": "Central Asia"}, {"id": 19192, "dom": "G", "ent": "Zarqa", "cls": "Eastern Africa"}, {"id": 19193, "dom": "G", "ent": "Zarqa", "cls": "Eastern Asia"}, {"id": 19194, "dom": "G", "ent": "Zarqa", "cls": "Eastern Europe"}, {"id": 19195, "dom": "G", "ent": "Zarqa", "cls": "Melanesia"}, {"id": 19196, "dom": "G", "ent": "Zarqa", "cls": "Micronesia"}, {"id": 19197, "dom": "G", "ent": "Zarqa", "cls": "Middle Africa"}, {"id": 19198, "dom": "G", "ent": "Zarqa", "cls": "Northern Africa"}, {"id": 19199, "dom": "G", "ent": "Zarqa", "cls": "Northern America"}, {"id": 19200, "dom": "G", "ent": "Zarqa", "cls": "Northern Europe"}, {"id": 19201, "dom": "G", "ent": "Zarqa", "cls": "Polynesia"}, {"id": 19202, "dom": "G", "ent": "Zarqa", "cls": "South America"}, {"id": 19203, "dom": "G", "ent": "Zarqa", "cls": "South-eastern Asia"}, {"id": 19204, "dom": "G", "ent": "Zarqa", "cls": "Southern Africa"}, {"id": 19205, "dom": "G", "ent": "Zarqa", "cls": "Southern Asia"}, {"id": 19206, "dom": "G", "ent": "Zarqa", "cls": "Southern Europe"}, {"id": 19207, "dom": "G", "ent": "Zarqa", "cls": "Western Africa"}, {"id": 19208, "dom": "G", "ent": "Zarqa", "cls": "Western Asia"}, {"id": 19209, "dom": "G", "ent": "Zarqa", "cls": "Western Europe"}, {"id": 19210, "dom": "G", "ent": "Irbid", "cls": "Australia and New Zealand"}, {"id": 19211, "dom": "G", "ent": "Irbid", "cls": "Caribbean"}, {"id": 19212, "dom": "G", "ent": "Irbid", "cls": "Central America"}, {"id": 19213, "dom": "G", "ent": "Irbid", "cls": "Central Asia"}, {"id": 19214, "dom": "G", "ent": "Irbid", "cls": "Eastern Africa"}, {"id": 19215, "dom": "G", "ent": "Irbid", "cls": "Eastern Asia"}, {"id": 19216, "dom": "G", "ent": "Irbid", "cls": "Eastern Europe"}, {"id": 19217, "dom": "G", "ent": "Irbid", "cls": "Melanesia"}, {"id": 19218, "dom": "G", "ent": "Irbid", "cls": "Micronesia"}, {"id": 19219, "dom": "G", "ent": "Irbid", "cls": "Middle Africa"}, {"id": 19220, "dom": "G", "ent": "Irbid", "cls": "Northern Africa"}, {"id": 19221, "dom": "G", "ent": "Irbid", "cls": "Northern America"}, {"id": 19222, "dom": "G", "ent": "Irbid", "cls": "Northern Europe"}, {"id": 19223, "dom": "G", "ent": "Irbid", "cls": "Polynesia"}, {"id": 19224, "dom": "G", "ent": "Irbid", "cls": "South America"}, {"id": 19225, "dom": "G", "ent": "Irbid", "cls": "South-eastern Asia"}, {"id": 19226, "dom": "G", "ent": "Irbid", "cls": "Southern Africa"}, {"id": 19227, "dom": "G", "ent": "Irbid", "cls": "Southern Asia"}, {"id": 19228, "dom": "G", "ent": "Irbid", "cls": "Southern Europe"}, {"id": 19229, "dom": "G", "ent": "Irbid", "cls": "Western Africa"}, {"id": 19230, "dom": "G", "ent": "Irbid", "cls": "Western Asia"}, {"id": 19231, "dom": "G", "ent": "Irbid", "cls": "Western Europe"}, {"id": 19232, "dom": "G", "ent": "Marj Al Hamam", "cls": "Australia and New Zealand"}, {"id": 19233, "dom": "G", "ent": "Marj Al Hamam", "cls": "Caribbean"}, {"id": 19234, "dom": "G", "ent": "Marj Al Hamam", "cls": "Central America"}, {"id": 19235, "dom": "G", "ent": "Marj Al Hamam", "cls": "Central Asia"}, {"id": 19236, "dom": "G", "ent": "Marj Al Hamam", "cls": "Eastern Africa"}, {"id": 19237, "dom": "G", "ent": "Marj Al Hamam", "cls": "Eastern Asia"}, {"id": 19238, "dom": "G", "ent": "Marj Al Hamam", "cls": "Eastern Europe"}, {"id": 19239, "dom": "G", "ent": "Marj Al Hamam", "cls": "Melanesia"}, {"id": 19240, "dom": "G", "ent": "Marj Al Hamam", "cls": "Micronesia"}, {"id": 19241, "dom": "G", "ent": "Marj Al Hamam", "cls": "Middle Africa"}, {"id": 19242, "dom": "G", "ent": "Marj Al Hamam", "cls": "Northern Africa"}, {"id": 19243, "dom": "G", "ent": "Marj Al Hamam", "cls": "Northern America"}, {"id": 19244, "dom": "G", "ent": "Marj Al Hamam", "cls": "Northern Europe"}, {"id": 19245, "dom": "G", "ent": "Marj Al Hamam", "cls": "Polynesia"}, {"id": 19246, "dom": "G", "ent": "Marj Al Hamam", "cls": "South America"}, {"id": 19247, "dom": "G", "ent": "Marj Al Hamam", "cls": "South-eastern Asia"}, {"id": 19248, "dom": "G", "ent": "Marj Al Hamam", "cls": "Southern Africa"}, {"id": 19249, "dom": "G", "ent": "Marj Al Hamam", "cls": "Southern Asia"}, {"id": 19250, "dom": "G", "ent": "Marj Al Hamam", "cls": "Southern Europe"}, {"id": 19251, "dom": "G", "ent": "Marj Al Hamam", "cls": "Western Africa"}, {"id": 19252, "dom": "G", "ent": "Marj Al Hamam", "cls": "Western Asia"}, {"id": 19253, "dom": "G", "ent": "Marj Al Hamam", "cls": "Western Europe"}, {"id": 19254, "dom": "G", "ent": "Judita", "cls": "Australia and New Zealand"}, {"id": 19255, "dom": "G", "ent": "Judita", "cls": "Caribbean"}, {"id": 19256, "dom": "G", "ent": "Judita", "cls": "Central America"}, {"id": 19257, "dom": "G", "ent": "Judita", "cls": "Central Asia"}, {"id": 19258, "dom": "G", "ent": "Judita", "cls": "Eastern Africa"}, {"id": 19259, "dom": "G", "ent": "Judita", "cls": "Eastern Asia"}, {"id": 19260, "dom": "G", "ent": "Judita", "cls": "Eastern Europe"}, {"id": 19261, "dom": "G", "ent": "Judita", "cls": "Melanesia"}, {"id": 19262, "dom": "G", "ent": "Judita", "cls": "Micronesia"}, {"id": 19263, "dom": "G", "ent": "Judita", "cls": "Middle Africa"}, {"id": 19264, "dom": "G", "ent": "Judita", "cls": "Northern Africa"}, {"id": 19265, "dom": "G", "ent": "Judita", "cls": "Northern America"}, {"id": 19266, "dom": "G", "ent": "Judita", "cls": "Northern Europe"}, {"id": 19267, "dom": "G", "ent": "Judita", "cls": "Polynesia"}, {"id": 19268, "dom": "G", "ent": "Judita", "cls": "South America"}, {"id": 19269, "dom": "G", "ent": "Judita", "cls": "South-eastern Asia"}, {"id": 19270, "dom": "G", "ent": "Judita", "cls": "Southern Africa"}, {"id": 19271, "dom": "G", "ent": "Judita", "cls": "Southern Asia"}, {"id": 19272, "dom": "G", "ent": "Judita", "cls": "Southern Europe"}, {"id": 19273, "dom": "G", "ent": "Judita", "cls": "Western Africa"}, {"id": 19274, "dom": "G", "ent": "Judita", "cls": "Western Asia"}, {"id": 19275, "dom": "G", "ent": "Judita", "cls": "Western Europe"}, {"id": 19276, "dom": "G", "ent": "Tochigi", "cls": "Australia and New Zealand"}, {"id": 19277, "dom": "G", "ent": "Tochigi", "cls": "Caribbean"}, {"id": 19278, "dom": "G", "ent": "Tochigi", "cls": "Central America"}, {"id": 19279, "dom": "G", "ent": "Tochigi", "cls": "Central Asia"}, {"id": 19280, "dom": "G", "ent": "Tochigi", "cls": "Eastern Africa"}, {"id": 19281, "dom": "G", "ent": "Tochigi", "cls": "Eastern Asia"}, {"id": 19282, "dom": "G", "ent": "Tochigi", "cls": "Eastern Europe"}, {"id": 19283, "dom": "G", "ent": "Tochigi", "cls": "Melanesia"}, {"id": 19284, "dom": "G", "ent": "Tochigi", "cls": "Micronesia"}, {"id": 19285, "dom": "G", "ent": "Tochigi", "cls": "Middle Africa"}, {"id": 19286, "dom": "G", "ent": "Tochigi", "cls": "Northern Africa"}, {"id": 19287, "dom": "G", "ent": "Tochigi", "cls": "Northern America"}, {"id": 19288, "dom": "G", "ent": "Tochigi", "cls": "Northern Europe"}, {"id": 19289, "dom": "G", "ent": "Tochigi", "cls": "Polynesia"}, {"id": 19290, "dom": "G", "ent": "Tochigi", "cls": "South America"}, {"id": 19291, "dom": "G", "ent": "Tochigi", "cls": "South-eastern Asia"}, {"id": 19292, "dom": "G", "ent": "Tochigi", "cls": "Southern Africa"}, {"id": 19293, "dom": "G", "ent": "Tochigi", "cls": "Southern Asia"}, {"id": 19294, "dom": "G", "ent": "Tochigi", "cls": "Southern Europe"}, {"id": 19295, "dom": "G", "ent": "Tochigi", "cls": "Western Africa"}, {"id": 19296, "dom": "G", "ent": "Tochigi", "cls": "Western Asia"}, {"id": 19297, "dom": "G", "ent": "Tochigi", "cls": "Western Europe"}, {"id": 19298, "dom": "G", "ent": "Maruobara", "cls": "Australia and New Zealand"}, {"id": 19299, "dom": "G", "ent": "Maruobara", "cls": "Caribbean"}, {"id": 19300, "dom": "G", "ent": "Maruobara", "cls": "Central America"}, {"id": 19301, "dom": "G", "ent": "Maruobara", "cls": "Central Asia"}, {"id": 19302, "dom": "G", "ent": "Maruobara", "cls": "Eastern Africa"}, {"id": 19303, "dom": "G", "ent": "Maruobara", "cls": "Eastern Asia"}, {"id": 19304, "dom": "G", "ent": "Maruobara", "cls": "Eastern Europe"}, {"id": 19305, "dom": "G", "ent": "Maruobara", "cls": "Melanesia"}, {"id": 19306, "dom": "G", "ent": "Maruobara", "cls": "Micronesia"}, {"id": 19307, "dom": "G", "ent": "Maruobara", "cls": "Middle Africa"}, {"id": 19308, "dom": "G", "ent": "Maruobara", "cls": "Northern Africa"}, {"id": 19309, "dom": "G", "ent": "Maruobara", "cls": "Northern America"}, {"id": 19310, "dom": "G", "ent": "Maruobara", "cls": "Northern Europe"}, {"id": 19311, "dom": "G", "ent": "Maruobara", "cls": "Polynesia"}, {"id": 19312, "dom": "G", "ent": "Maruobara", "cls": "South America"}, {"id": 19313, "dom": "G", "ent": "Maruobara", "cls": "South-eastern Asia"}, {"id": 19314, "dom": "G", "ent": "Maruobara", "cls": "Southern Africa"}, {"id": 19315, "dom": "G", "ent": "Maruobara", "cls": "Southern Asia"}, {"id": 19316, "dom": "G", "ent": "Maruobara", "cls": "Southern Europe"}, {"id": 19317, "dom": "G", "ent": "Maruobara", "cls": "Western Africa"}, {"id": 19318, "dom": "G", "ent": "Maruobara", "cls": "Western Asia"}, {"id": 19319, "dom": "G", "ent": "Maruobara", "cls": "Western Europe"}, {"id": 19320, "dom": "G", "ent": "Yoshioka", "cls": "Australia and New Zealand"}, {"id": 19321, "dom": "G", "ent": "Yoshioka", "cls": "Caribbean"}, {"id": 19322, "dom": "G", "ent": "Yoshioka", "cls": "Central America"}, {"id": 19323, "dom": "G", "ent": "Yoshioka", "cls": "Central Asia"}, {"id": 19324, "dom": "G", "ent": "Yoshioka", "cls": "Eastern Africa"}, {"id": 19325, "dom": "G", "ent": "Yoshioka", "cls": "Eastern Asia"}, {"id": 19326, "dom": "G", "ent": "Yoshioka", "cls": "Eastern Europe"}, {"id": 19327, "dom": "G", "ent": "Yoshioka", "cls": "Melanesia"}, {"id": 19328, "dom": "G", "ent": "Yoshioka", "cls": "Micronesia"}, {"id": 19329, "dom": "G", "ent": "Yoshioka", "cls": "Middle Africa"}, {"id": 19330, "dom": "G", "ent": "Yoshioka", "cls": "Northern Africa"}, {"id": 19331, "dom": "G", "ent": "Yoshioka", "cls": "Northern America"}, {"id": 19332, "dom": "G", "ent": "Yoshioka", "cls": "Northern Europe"}, {"id": 19333, "dom": "G", "ent": "Yoshioka", "cls": "Polynesia"}, {"id": 19334, "dom": "G", "ent": "Yoshioka", "cls": "South America"}, {"id": 19335, "dom": "G", "ent": "Yoshioka", "cls": "South-eastern Asia"}, {"id": 19336, "dom": "G", "ent": "Yoshioka", "cls": "Southern Africa"}, {"id": 19337, "dom": "G", "ent": "Yoshioka", "cls": "Southern Asia"}, {"id": 19338, "dom": "G", "ent": "Yoshioka", "cls": "Southern Europe"}, {"id": 19339, "dom": "G", "ent": "Yoshioka", "cls": "Western Africa"}, {"id": 19340, "dom": "G", "ent": "Yoshioka", "cls": "Western Asia"}, {"id": 19341, "dom": "G", "ent": "Yoshioka", "cls": "Western Europe"}, {"id": 19342, "dom": "G", "ent": "Makubetsu", "cls": "Australia and New Zealand"}, {"id": 19343, "dom": "G", "ent": "Makubetsu", "cls": "Caribbean"}, {"id": 19344, "dom": "G", "ent": "Makubetsu", "cls": "Central America"}, {"id": 19345, "dom": "G", "ent": "Makubetsu", "cls": "Central Asia"}, {"id": 19346, "dom": "G", "ent": "Makubetsu", "cls": "Eastern Africa"}, {"id": 19347, "dom": "G", "ent": "Makubetsu", "cls": "Eastern Asia"}, {"id": 19348, "dom": "G", "ent": "Makubetsu", "cls": "Eastern Europe"}, {"id": 19349, "dom": "G", "ent": "Makubetsu", "cls": "Melanesia"}, {"id": 19350, "dom": "G", "ent": "Makubetsu", "cls": "Micronesia"}, {"id": 19351, "dom": "G", "ent": "Makubetsu", "cls": "Middle Africa"}, {"id": 19352, "dom": "G", "ent": "Makubetsu", "cls": "Northern Africa"}, {"id": 19353, "dom": "G", "ent": "Makubetsu", "cls": "Northern America"}, {"id": 19354, "dom": "G", "ent": "Makubetsu", "cls": "Northern Europe"}, {"id": 19355, "dom": "G", "ent": "Makubetsu", "cls": "Polynesia"}, {"id": 19356, "dom": "G", "ent": "Makubetsu", "cls": "South America"}, {"id": 19357, "dom": "G", "ent": "Makubetsu", "cls": "South-eastern Asia"}, {"id": 19358, "dom": "G", "ent": "Makubetsu", "cls": "Southern Africa"}, {"id": 19359, "dom": "G", "ent": "Makubetsu", "cls": "Southern Asia"}, {"id": 19360, "dom": "G", "ent": "Makubetsu", "cls": "Southern Europe"}, {"id": 19361, "dom": "G", "ent": "Makubetsu", "cls": "Western Africa"}, {"id": 19362, "dom": "G", "ent": "Makubetsu", "cls": "Western Asia"}, {"id": 19363, "dom": "G", "ent": "Makubetsu", "cls": "Western Europe"}, {"id": 19364, "dom": "G", "ent": "Mombasa", "cls": "Australia and New Zealand"}, {"id": 19365, "dom": "G", "ent": "Mombasa", "cls": "Caribbean"}, {"id": 19366, "dom": "G", "ent": "Mombasa", "cls": "Central America"}, {"id": 19367, "dom": "G", "ent": "Mombasa", "cls": "Central Asia"}, {"id": 19368, "dom": "G", "ent": "Mombasa", "cls": "Eastern Africa"}, {"id": 19369, "dom": "G", "ent": "Mombasa", "cls": "Eastern Asia"}, {"id": 19370, "dom": "G", "ent": "Mombasa", "cls": "Eastern Europe"}, {"id": 19371, "dom": "G", "ent": "Mombasa", "cls": "Melanesia"}, {"id": 19372, "dom": "G", "ent": "Mombasa", "cls": "Micronesia"}, {"id": 19373, "dom": "G", "ent": "Mombasa", "cls": "Middle Africa"}, {"id": 19374, "dom": "G", "ent": "Mombasa", "cls": "Northern Africa"}, {"id": 19375, "dom": "G", "ent": "Mombasa", "cls": "Northern America"}, {"id": 19376, "dom": "G", "ent": "Mombasa", "cls": "Northern Europe"}, {"id": 19377, "dom": "G", "ent": "Mombasa", "cls": "Polynesia"}, {"id": 19378, "dom": "G", "ent": "Mombasa", "cls": "South America"}, {"id": 19379, "dom": "G", "ent": "Mombasa", "cls": "South-eastern Asia"}, {"id": 19380, "dom": "G", "ent": "Mombasa", "cls": "Southern Africa"}, {"id": 19381, "dom": "G", "ent": "Mombasa", "cls": "Southern Asia"}, {"id": 19382, "dom": "G", "ent": "Mombasa", "cls": "Southern Europe"}, {"id": 19383, "dom": "G", "ent": "Mombasa", "cls": "Western Africa"}, {"id": 19384, "dom": "G", "ent": "Mombasa", "cls": "Western Asia"}, {"id": 19385, "dom": "G", "ent": "Mombasa", "cls": "Western Europe"}, {"id": 19386, "dom": "G", "ent": "Eldoret", "cls": "Australia and New Zealand"}, {"id": 19387, "dom": "G", "ent": "Eldoret", "cls": "Caribbean"}, {"id": 19388, "dom": "G", "ent": "Eldoret", "cls": "Central America"}, {"id": 19389, "dom": "G", "ent": "Eldoret", "cls": "Central Asia"}, {"id": 19390, "dom": "G", "ent": "Eldoret", "cls": "Eastern Africa"}, {"id": 19391, "dom": "G", "ent": "Eldoret", "cls": "Eastern Asia"}, {"id": 19392, "dom": "G", "ent": "Eldoret", "cls": "Eastern Europe"}, {"id": 19393, "dom": "G", "ent": "Eldoret", "cls": "Melanesia"}, {"id": 19394, "dom": "G", "ent": "Eldoret", "cls": "Micronesia"}, {"id": 19395, "dom": "G", "ent": "Eldoret", "cls": "Middle Africa"}, {"id": 19396, "dom": "G", "ent": "Eldoret", "cls": "Northern Africa"}, {"id": 19397, "dom": "G", "ent": "Eldoret", "cls": "Northern America"}, {"id": 19398, "dom": "G", "ent": "Eldoret", "cls": "Northern Europe"}, {"id": 19399, "dom": "G", "ent": "Eldoret", "cls": "Polynesia"}, {"id": 19400, "dom": "G", "ent": "Eldoret", "cls": "South America"}, {"id": 19401, "dom": "G", "ent": "Eldoret", "cls": "South-eastern Asia"}, {"id": 19402, "dom": "G", "ent": "Eldoret", "cls": "Southern Africa"}, {"id": 19403, "dom": "G", "ent": "Eldoret", "cls": "Southern Asia"}, {"id": 19404, "dom": "G", "ent": "Eldoret", "cls": "Southern Europe"}, {"id": 19405, "dom": "G", "ent": "Eldoret", "cls": "Western Africa"}, {"id": 19406, "dom": "G", "ent": "Eldoret", "cls": "Western Asia"}, {"id": 19407, "dom": "G", "ent": "Eldoret", "cls": "Western Europe"}, {"id": 19408, "dom": "G", "ent": "Garissa", "cls": "Australia and New Zealand"}, {"id": 19409, "dom": "G", "ent": "Garissa", "cls": "Caribbean"}, {"id": 19410, "dom": "G", "ent": "Garissa", "cls": "Central America"}, {"id": 19411, "dom": "G", "ent": "Garissa", "cls": "Central Asia"}, {"id": 19412, "dom": "G", "ent": "Garissa", "cls": "Eastern Africa"}, {"id": 19413, "dom": "G", "ent": "Garissa", "cls": "Eastern Asia"}, {"id": 19414, "dom": "G", "ent": "Garissa", "cls": "Eastern Europe"}, {"id": 19415, "dom": "G", "ent": "Garissa", "cls": "Melanesia"}, {"id": 19416, "dom": "G", "ent": "Garissa", "cls": "Micronesia"}, {"id": 19417, "dom": "G", "ent": "Garissa", "cls": "Middle Africa"}, {"id": 19418, "dom": "G", "ent": "Garissa", "cls": "Northern Africa"}, {"id": 19419, "dom": "G", "ent": "Garissa", "cls": "Northern America"}, {"id": 19420, "dom": "G", "ent": "Garissa", "cls": "Northern Europe"}, {"id": 19421, "dom": "G", "ent": "Garissa", "cls": "Polynesia"}, {"id": 19422, "dom": "G", "ent": "Garissa", "cls": "South America"}, {"id": 19423, "dom": "G", "ent": "Garissa", "cls": "South-eastern Asia"}, {"id": 19424, "dom": "G", "ent": "Garissa", "cls": "Southern Africa"}, {"id": 19425, "dom": "G", "ent": "Garissa", "cls": "Southern Asia"}, {"id": 19426, "dom": "G", "ent": "Garissa", "cls": "Southern Europe"}, {"id": 19427, "dom": "G", "ent": "Garissa", "cls": "Western Africa"}, {"id": 19428, "dom": "G", "ent": "Garissa", "cls": "Western Asia"}, {"id": 19429, "dom": "G", "ent": "Garissa", "cls": "Western Europe"}, {"id": 19430, "dom": "G", "ent": "Kimilili", "cls": "Australia and New Zealand"}, {"id": 19431, "dom": "G", "ent": "Kimilili", "cls": "Caribbean"}, {"id": 19432, "dom": "G", "ent": "Kimilili", "cls": "Central America"}, {"id": 19433, "dom": "G", "ent": "Kimilili", "cls": "Central Asia"}, {"id": 19434, "dom": "G", "ent": "Kimilili", "cls": "Eastern Africa"}, {"id": 19435, "dom": "G", "ent": "Kimilili", "cls": "Eastern Asia"}, {"id": 19436, "dom": "G", "ent": "Kimilili", "cls": "Eastern Europe"}, {"id": 19437, "dom": "G", "ent": "Kimilili", "cls": "Melanesia"}, {"id": 19438, "dom": "G", "ent": "Kimilili", "cls": "Micronesia"}, {"id": 19439, "dom": "G", "ent": "Kimilili", "cls": "Middle Africa"}, {"id": 19440, "dom": "G", "ent": "Kimilili", "cls": "Northern Africa"}, {"id": 19441, "dom": "G", "ent": "Kimilili", "cls": "Northern America"}, {"id": 19442, "dom": "G", "ent": "Kimilili", "cls": "Northern Europe"}, {"id": 19443, "dom": "G", "ent": "Kimilili", "cls": "Polynesia"}, {"id": 19444, "dom": "G", "ent": "Kimilili", "cls": "South America"}, {"id": 19445, "dom": "G", "ent": "Kimilili", "cls": "South-eastern Asia"}, {"id": 19446, "dom": "G", "ent": "Kimilili", "cls": "Southern Africa"}, {"id": 19447, "dom": "G", "ent": "Kimilili", "cls": "Southern Asia"}, {"id": 19448, "dom": "G", "ent": "Kimilili", "cls": "Southern Europe"}, {"id": 19449, "dom": "G", "ent": "Kimilili", "cls": "Western Africa"}, {"id": 19450, "dom": "G", "ent": "Kimilili", "cls": "Western Asia"}, {"id": 19451, "dom": "G", "ent": "Kimilili", "cls": "Western Europe"}, {"id": 19452, "dom": "G", "ent": "Balykchy", "cls": "Australia and New Zealand"}, {"id": 19453, "dom": "G", "ent": "Balykchy", "cls": "Caribbean"}, {"id": 19454, "dom": "G", "ent": "Balykchy", "cls": "Central America"}, {"id": 19455, "dom": "G", "ent": "Balykchy", "cls": "Central Asia"}, {"id": 19456, "dom": "G", "ent": "Balykchy", "cls": "Eastern Africa"}, {"id": 19457, "dom": "G", "ent": "Balykchy", "cls": "Eastern Asia"}, {"id": 19458, "dom": "G", "ent": "Balykchy", "cls": "Eastern Europe"}, {"id": 19459, "dom": "G", "ent": "Balykchy", "cls": "Melanesia"}, {"id": 19460, "dom": "G", "ent": "Balykchy", "cls": "Micronesia"}, {"id": 19461, "dom": "G", "ent": "Balykchy", "cls": "Middle Africa"}, {"id": 19462, "dom": "G", "ent": "Balykchy", "cls": "Northern Africa"}, {"id": 19463, "dom": "G", "ent": "Balykchy", "cls": "Northern America"}, {"id": 19464, "dom": "G", "ent": "Balykchy", "cls": "Northern Europe"}, {"id": 19465, "dom": "G", "ent": "Balykchy", "cls": "Polynesia"}, {"id": 19466, "dom": "G", "ent": "Balykchy", "cls": "South America"}, {"id": 19467, "dom": "G", "ent": "Balykchy", "cls": "South-eastern Asia"}, {"id": 19468, "dom": "G", "ent": "Balykchy", "cls": "Southern Africa"}, {"id": 19469, "dom": "G", "ent": "Balykchy", "cls": "Southern Asia"}, {"id": 19470, "dom": "G", "ent": "Balykchy", "cls": "Southern Europe"}, {"id": 19471, "dom": "G", "ent": "Balykchy", "cls": "Western Africa"}, {"id": 19472, "dom": "G", "ent": "Balykchy", "cls": "Western Asia"}, {"id": 19473, "dom": "G", "ent": "Balykchy", "cls": "Western Europe"}, {"id": 19474, "dom": "G", "ent": "Razzakov", "cls": "Australia and New Zealand"}, {"id": 19475, "dom": "G", "ent": "Razzakov", "cls": "Caribbean"}, {"id": 19476, "dom": "G", "ent": "Razzakov", "cls": "Central America"}, {"id": 19477, "dom": "G", "ent": "Razzakov", "cls": "Central Asia"}, {"id": 19478, "dom": "G", "ent": "Razzakov", "cls": "Eastern Africa"}, {"id": 19479, "dom": "G", "ent": "Razzakov", "cls": "Eastern Asia"}, {"id": 19480, "dom": "G", "ent": "Razzakov", "cls": "Eastern Europe"}, {"id": 19481, "dom": "G", "ent": "Razzakov", "cls": "Melanesia"}, {"id": 19482, "dom": "G", "ent": "Razzakov", "cls": "Micronesia"}, {"id": 19483, "dom": "G", "ent": "Razzakov", "cls": "Middle Africa"}, {"id": 19484, "dom": "G", "ent": "Razzakov", "cls": "Northern Africa"}, {"id": 19485, "dom": "G", "ent": "Razzakov", "cls": "Northern America"}, {"id": 19486, "dom": "G", "ent": "Razzakov", "cls": "Northern Europe"}, {"id": 19487, "dom": "G", "ent": "Razzakov", "cls": "Polynesia"}, {"id": 19488, "dom": "G", "ent": "Razzakov", "cls": "South America"}, {"id": 19489, "dom": "G", "ent": "Razzakov", "cls": "South-eastern Asia"}, {"id": 19490, "dom": "G", "ent": "Razzakov", "cls": "Southern Africa"}, {"id": 19491, "dom": "G", "ent": "Razzakov", "cls": "Southern Asia"}, {"id": 19492, "dom": "G", "ent": "Razzakov", "cls": "Southern Europe"}, {"id": 19493, "dom": "G", "ent": "Razzakov", "cls": "Western Africa"}, {"id": 19494, "dom": "G", "ent": "Razzakov", "cls": "Western Asia"}, {"id": 19495, "dom": "G", "ent": "Razzakov", "cls": "Western Europe"}, {"id": 19496, "dom": "G", "ent": "Iradan", "cls": "Australia and New Zealand"}, {"id": 19497, "dom": "G", "ent": "Iradan", "cls": "Caribbean"}, {"id": 19498, "dom": "G", "ent": "Iradan", "cls": "Central America"}, {"id": 19499, "dom": "G", "ent": "Iradan", "cls": "Central Asia"}, {"id": 19500, "dom": "G", "ent": "Iradan", "cls": "Eastern Africa"}, {"id": 19501, "dom": "G", "ent": "Iradan", "cls": "Eastern Asia"}, {"id": 19502, "dom": "G", "ent": "Iradan", "cls": "Eastern Europe"}, {"id": 19503, "dom": "G", "ent": "Iradan", "cls": "Melanesia"}, {"id": 19504, "dom": "G", "ent": "Iradan", "cls": "Micronesia"}, {"id": 19505, "dom": "G", "ent": "Iradan", "cls": "Middle Africa"}, {"id": 19506, "dom": "G", "ent": "Iradan", "cls": "Northern Africa"}, {"id": 19507, "dom": "G", "ent": "Iradan", "cls": "Northern America"}, {"id": 19508, "dom": "G", "ent": "Iradan", "cls": "Northern Europe"}, {"id": 19509, "dom": "G", "ent": "Iradan", "cls": "Polynesia"}, {"id": 19510, "dom": "G", "ent": "Iradan", "cls": "South America"}, {"id": 19511, "dom": "G", "ent": "Iradan", "cls": "South-eastern Asia"}, {"id": 19512, "dom": "G", "ent": "Iradan", "cls": "Southern Africa"}, {"id": 19513, "dom": "G", "ent": "Iradan", "cls": "Southern Asia"}, {"id": 19514, "dom": "G", "ent": "Iradan", "cls": "Southern Europe"}, {"id": 19515, "dom": "G", "ent": "Iradan", "cls": "Western Africa"}, {"id": 19516, "dom": "G", "ent": "Iradan", "cls": "Western Asia"}, {"id": 19517, "dom": "G", "ent": "Iradan", "cls": "Western Europe"}, {"id": 19518, "dom": "G", "ent": "Tash-Kumyr", "cls": "Australia and New Zealand"}, {"id": 19519, "dom": "G", "ent": "Tash-Kumyr", "cls": "Caribbean"}, {"id": 19520, "dom": "G", "ent": "Tash-Kumyr", "cls": "Central America"}, {"id": 19521, "dom": "G", "ent": "Tash-Kumyr", "cls": "Central Asia"}, {"id": 19522, "dom": "G", "ent": "Tash-Kumyr", "cls": "Eastern Africa"}, {"id": 19523, "dom": "G", "ent": "Tash-Kumyr", "cls": "Eastern Asia"}, {"id": 19524, "dom": "G", "ent": "Tash-Kumyr", "cls": "Eastern Europe"}, {"id": 19525, "dom": "G", "ent": "Tash-Kumyr", "cls": "Melanesia"}, {"id": 19526, "dom": "G", "ent": "Tash-Kumyr", "cls": "Micronesia"}, {"id": 19527, "dom": "G", "ent": "Tash-Kumyr", "cls": "Middle Africa"}, {"id": 19528, "dom": "G", "ent": "Tash-Kumyr", "cls": "Northern Africa"}, {"id": 19529, "dom": "G", "ent": "Tash-Kumyr", "cls": "Northern America"}, {"id": 19530, "dom": "G", "ent": "Tash-Kumyr", "cls": "Northern Europe"}, {"id": 19531, "dom": "G", "ent": "Tash-Kumyr", "cls": "Polynesia"}, {"id": 19532, "dom": "G", "ent": "Tash-Kumyr", "cls": "South America"}, {"id": 19533, "dom": "G", "ent": "Tash-Kumyr", "cls": "South-eastern Asia"}, {"id": 19534, "dom": "G", "ent": "Tash-Kumyr", "cls": "Southern Africa"}, {"id": 19535, "dom": "G", "ent": "Tash-Kumyr", "cls": "Southern Asia"}, {"id": 19536, "dom": "G", "ent": "Tash-Kumyr", "cls": "Southern Europe"}, {"id": 19537, "dom": "G", "ent": "Tash-Kumyr", "cls": "Western Africa"}, {"id": 19538, "dom": "G", "ent": "Tash-Kumyr", "cls": "Western Asia"}, {"id": 19539, "dom": "G", "ent": "Tash-Kumyr", "cls": "Western Europe"}, {"id": 19540, "dom": "G", "ent": "Kampong Cham", "cls": "Australia and New Zealand"}, {"id": 19541, "dom": "G", "ent": "Kampong Cham", "cls": "Caribbean"}, {"id": 19542, "dom": "G", "ent": "Kampong Cham", "cls": "Central America"}, {"id": 19543, "dom": "G", "ent": "Kampong Cham", "cls": "Central Asia"}, {"id": 19544, "dom": "G", "ent": "Kampong Cham", "cls": "Eastern Africa"}, {"id": 19545, "dom": "G", "ent": "Kampong Cham", "cls": "Eastern Asia"}, {"id": 19546, "dom": "G", "ent": "Kampong Cham", "cls": "Eastern Europe"}, {"id": 19547, "dom": "G", "ent": "Kampong Cham", "cls": "Melanesia"}, {"id": 19548, "dom": "G", "ent": "Kampong Cham", "cls": "Micronesia"}, {"id": 19549, "dom": "G", "ent": "Kampong Cham", "cls": "Middle Africa"}, {"id": 19550, "dom": "G", "ent": "Kampong Cham", "cls": "Northern Africa"}, {"id": 19551, "dom": "G", "ent": "Kampong Cham", "cls": "Northern America"}, {"id": 19552, "dom": "G", "ent": "Kampong Cham", "cls": "Northern Europe"}, {"id": 19553, "dom": "G", "ent": "Kampong Cham", "cls": "Polynesia"}, {"id": 19554, "dom": "G", "ent": "Kampong Cham", "cls": "South America"}, {"id": 19555, "dom": "G", "ent": "Kampong Cham", "cls": "South-eastern Asia"}, {"id": 19556, "dom": "G", "ent": "Kampong Cham", "cls": "Southern Africa"}, {"id": 19557, "dom": "G", "ent": "Kampong Cham", "cls": "Southern Asia"}, {"id": 19558, "dom": "G", "ent": "Kampong Cham", "cls": "Southern Europe"}, {"id": 19559, "dom": "G", "ent": "Kampong Cham", "cls": "Western Africa"}, {"id": 19560, "dom": "G", "ent": "Kampong Cham", "cls": "Western Asia"}, {"id": 19561, "dom": "G", "ent": "Kampong Cham", "cls": "Western Europe"}, {"id": 19562, "dom": "G", "ent": "Pursat", "cls": "Australia and New Zealand"}, {"id": 19563, "dom": "G", "ent": "Pursat", "cls": "Caribbean"}, {"id": 19564, "dom": "G", "ent": "Pursat", "cls": "Central America"}, {"id": 19565, "dom": "G", "ent": "Pursat", "cls": "Central Asia"}, {"id": 19566, "dom": "G", "ent": "Pursat", "cls": "Eastern Africa"}, {"id": 19567, "dom": "G", "ent": "Pursat", "cls": "Eastern Asia"}, {"id": 19568, "dom": "G", "ent": "Pursat", "cls": "Eastern Europe"}, {"id": 19569, "dom": "G", "ent": "Pursat", "cls": "Melanesia"}, {"id": 19570, "dom": "G", "ent": "Pursat", "cls": "Micronesia"}, {"id": 19571, "dom": "G", "ent": "Pursat", "cls": "Middle Africa"}, {"id": 19572, "dom": "G", "ent": "Pursat", "cls": "Northern Africa"}, {"id": 19573, "dom": "G", "ent": "Pursat", "cls": "Northern America"}, {"id": 19574, "dom": "G", "ent": "Pursat", "cls": "Northern Europe"}, {"id": 19575, "dom": "G", "ent": "Pursat", "cls": "Polynesia"}, {"id": 19576, "dom": "G", "ent": "Pursat", "cls": "South America"}, {"id": 19577, "dom": "G", "ent": "Pursat", "cls": "South-eastern Asia"}, {"id": 19578, "dom": "G", "ent": "Pursat", "cls": "Southern Africa"}, {"id": 19579, "dom": "G", "ent": "Pursat", "cls": "Southern Asia"}, {"id": 19580, "dom": "G", "ent": "Pursat", "cls": "Southern Europe"}, {"id": 19581, "dom": "G", "ent": "Pursat", "cls": "Western Africa"}, {"id": 19582, "dom": "G", "ent": "Pursat", "cls": "Western Asia"}, {"id": 19583, "dom": "G", "ent": "Pursat", "cls": "Western Europe"}, {"id": 19584, "dom": "G", "ent": "Lumphat", "cls": "Australia and New Zealand"}, {"id": 19585, "dom": "G", "ent": "Lumphat", "cls": "Caribbean"}, {"id": 19586, "dom": "G", "ent": "Lumphat", "cls": "Central America"}, {"id": 19587, "dom": "G", "ent": "Lumphat", "cls": "Central Asia"}, {"id": 19588, "dom": "G", "ent": "Lumphat", "cls": "Eastern Africa"}, {"id": 19589, "dom": "G", "ent": "Lumphat", "cls": "Eastern Asia"}, {"id": 19590, "dom": "G", "ent": "Lumphat", "cls": "Eastern Europe"}, {"id": 19591, "dom": "G", "ent": "Lumphat", "cls": "Melanesia"}, {"id": 19592, "dom": "G", "ent": "Lumphat", "cls": "Micronesia"}, {"id": 19593, "dom": "G", "ent": "Lumphat", "cls": "Middle Africa"}, {"id": 19594, "dom": "G", "ent": "Lumphat", "cls": "Northern Africa"}, {"id": 19595, "dom": "G", "ent": "Lumphat", "cls": "Northern America"}, {"id": 19596, "dom": "G", "ent": "Lumphat", "cls": "Northern Europe"}, {"id": 19597, "dom": "G", "ent": "Lumphat", "cls": "Polynesia"}, {"id": 19598, "dom": "G", "ent": "Lumphat", "cls": "South America"}, {"id": 19599, "dom": "G", "ent": "Lumphat", "cls": "South-eastern Asia"}, {"id": 19600, "dom": "G", "ent": "Lumphat", "cls": "Southern Africa"}, {"id": 19601, "dom": "G", "ent": "Lumphat", "cls": "Southern Asia"}, {"id": 19602, "dom": "G", "ent": "Lumphat", "cls": "Southern Europe"}, {"id": 19603, "dom": "G", "ent": "Lumphat", "cls": "Western Africa"}, {"id": 19604, "dom": "G", "ent": "Lumphat", "cls": "Western Asia"}, {"id": 19605, "dom": "G", "ent": "Lumphat", "cls": "Western Europe"}, {"id": 19606, "dom": "G", "ent": "Banlung", "cls": "Australia and New Zealand"}, {"id": 19607, "dom": "G", "ent": "Banlung", "cls": "Caribbean"}, {"id": 19608, "dom": "G", "ent": "Banlung", "cls": "Central America"}, {"id": 19609, "dom": "G", "ent": "Banlung", "cls": "Central Asia"}, {"id": 19610, "dom": "G", "ent": "Banlung", "cls": "Eastern Africa"}, {"id": 19611, "dom": "G", "ent": "Banlung", "cls": "Eastern Asia"}, {"id": 19612, "dom": "G", "ent": "Banlung", "cls": "Eastern Europe"}, {"id": 19613, "dom": "G", "ent": "Banlung", "cls": "Melanesia"}, {"id": 19614, "dom": "G", "ent": "Banlung", "cls": "Micronesia"}, {"id": 19615, "dom": "G", "ent": "Banlung", "cls": "Middle Africa"}, {"id": 19616, "dom": "G", "ent": "Banlung", "cls": "Northern Africa"}, {"id": 19617, "dom": "G", "ent": "Banlung", "cls": "Northern America"}, {"id": 19618, "dom": "G", "ent": "Banlung", "cls": "Northern Europe"}, {"id": 19619, "dom": "G", "ent": "Banlung", "cls": "Polynesia"}, {"id": 19620, "dom": "G", "ent": "Banlung", "cls": "South America"}, {"id": 19621, "dom": "G", "ent": "Banlung", "cls": "South-eastern Asia"}, {"id": 19622, "dom": "G", "ent": "Banlung", "cls": "Southern Africa"}, {"id": 19623, "dom": "G", "ent": "Banlung", "cls": "Southern Asia"}, {"id": 19624, "dom": "G", "ent": "Banlung", "cls": "Southern Europe"}, {"id": 19625, "dom": "G", "ent": "Banlung", "cls": "Western Africa"}, {"id": 19626, "dom": "G", "ent": "Banlung", "cls": "Western Asia"}, {"id": 19627, "dom": "G", "ent": "Banlung", "cls": "Western Europe"}, {"id": 19628, "dom": "G", "ent": "Rajin", "cls": "Australia and New Zealand"}, {"id": 19629, "dom": "G", "ent": "Rajin", "cls": "Caribbean"}, {"id": 19630, "dom": "G", "ent": "Rajin", "cls": "Central America"}, {"id": 19631, "dom": "G", "ent": "Rajin", "cls": "Central Asia"}, {"id": 19632, "dom": "G", "ent": "Rajin", "cls": "Eastern Africa"}, {"id": 19633, "dom": "G", "ent": "Rajin", "cls": "Eastern Asia"}, {"id": 19634, "dom": "G", "ent": "Rajin", "cls": "Eastern Europe"}, {"id": 19635, "dom": "G", "ent": "Rajin", "cls": "Melanesia"}, {"id": 19636, "dom": "G", "ent": "Rajin", "cls": "Micronesia"}, {"id": 19637, "dom": "G", "ent": "Rajin", "cls": "Middle Africa"}, {"id": 19638, "dom": "G", "ent": "Rajin", "cls": "Northern Africa"}, {"id": 19639, "dom": "G", "ent": "Rajin", "cls": "Northern America"}, {"id": 19640, "dom": "G", "ent": "Rajin", "cls": "Northern Europe"}, {"id": 19641, "dom": "G", "ent": "Rajin", "cls": "Polynesia"}, {"id": 19642, "dom": "G", "ent": "Rajin", "cls": "South America"}, {"id": 19643, "dom": "G", "ent": "Rajin", "cls": "South-eastern Asia"}, {"id": 19644, "dom": "G", "ent": "Rajin", "cls": "Southern Africa"}, {"id": 19645, "dom": "G", "ent": "Rajin", "cls": "Southern Asia"}, {"id": 19646, "dom": "G", "ent": "Rajin", "cls": "Southern Europe"}, {"id": 19647, "dom": "G", "ent": "Rajin", "cls": "Western Africa"}, {"id": 19648, "dom": "G", "ent": "Rajin", "cls": "Western Asia"}, {"id": 19649, "dom": "G", "ent": "Rajin", "cls": "Western Europe"}, {"id": 19650, "dom": "G", "ent": "Hoeyang", "cls": "Australia and New Zealand"}, {"id": 19651, "dom": "G", "ent": "Hoeyang", "cls": "Caribbean"}, {"id": 19652, "dom": "G", "ent": "Hoeyang", "cls": "Central America"}, {"id": 19653, "dom": "G", "ent": "Hoeyang", "cls": "Central Asia"}, {"id": 19654, "dom": "G", "ent": "Hoeyang", "cls": "Eastern Africa"}, {"id": 19655, "dom": "G", "ent": "Hoeyang", "cls": "Eastern Asia"}, {"id": 19656, "dom": "G", "ent": "Hoeyang", "cls": "Eastern Europe"}, {"id": 19657, "dom": "G", "ent": "Hoeyang", "cls": "Melanesia"}, {"id": 19658, "dom": "G", "ent": "Hoeyang", "cls": "Micronesia"}, {"id": 19659, "dom": "G", "ent": "Hoeyang", "cls": "Middle Africa"}, {"id": 19660, "dom": "G", "ent": "Hoeyang", "cls": "Northern Africa"}, {"id": 19661, "dom": "G", "ent": "Hoeyang", "cls": "Northern America"}, {"id": 19662, "dom": "G", "ent": "Hoeyang", "cls": "Northern Europe"}, {"id": 19663, "dom": "G", "ent": "Hoeyang", "cls": "Polynesia"}, {"id": 19664, "dom": "G", "ent": "Hoeyang", "cls": "South America"}, {"id": 19665, "dom": "G", "ent": "Hoeyang", "cls": "South-eastern Asia"}, {"id": 19666, "dom": "G", "ent": "Hoeyang", "cls": "Southern Africa"}, {"id": 19667, "dom": "G", "ent": "Hoeyang", "cls": "Southern Asia"}, {"id": 19668, "dom": "G", "ent": "Hoeyang", "cls": "Southern Europe"}, {"id": 19669, "dom": "G", "ent": "Hoeyang", "cls": "Western Africa"}, {"id": 19670, "dom": "G", "ent": "Hoeyang", "cls": "Western Asia"}, {"id": 19671, "dom": "G", "ent": "Hoeyang", "cls": "Western Europe"}, {"id": 19672, "dom": "G", "ent": "Chunghwa", "cls": "Australia and New Zealand"}, {"id": 19673, "dom": "G", "ent": "Chunghwa", "cls": "Caribbean"}, {"id": 19674, "dom": "G", "ent": "Chunghwa", "cls": "Central America"}, {"id": 19675, "dom": "G", "ent": "Chunghwa", "cls": "Central Asia"}, {"id": 19676, "dom": "G", "ent": "Chunghwa", "cls": "Eastern Africa"}, {"id": 19677, "dom": "G", "ent": "Chunghwa", "cls": "Eastern Asia"}, {"id": 19678, "dom": "G", "ent": "Chunghwa", "cls": "Eastern Europe"}, {"id": 19679, "dom": "G", "ent": "Chunghwa", "cls": "Melanesia"}, {"id": 19680, "dom": "G", "ent": "Chunghwa", "cls": "Micronesia"}, {"id": 19681, "dom": "G", "ent": "Chunghwa", "cls": "Middle Africa"}, {"id": 19682, "dom": "G", "ent": "Chunghwa", "cls": "Northern Africa"}, {"id": 19683, "dom": "G", "ent": "Chunghwa", "cls": "Northern America"}, {"id": 19684, "dom": "G", "ent": "Chunghwa", "cls": "Northern Europe"}, {"id": 19685, "dom": "G", "ent": "Chunghwa", "cls": "Polynesia"}, {"id": 19686, "dom": "G", "ent": "Chunghwa", "cls": "South America"}, {"id": 19687, "dom": "G", "ent": "Chunghwa", "cls": "South-eastern Asia"}, {"id": 19688, "dom": "G", "ent": "Chunghwa", "cls": "Southern Africa"}, {"id": 19689, "dom": "G", "ent": "Chunghwa", "cls": "Southern Asia"}, {"id": 19690, "dom": "G", "ent": "Chunghwa", "cls": "Southern Europe"}, {"id": 19691, "dom": "G", "ent": "Chunghwa", "cls": "Western Africa"}, {"id": 19692, "dom": "G", "ent": "Chunghwa", "cls": "Western Asia"}, {"id": 19693, "dom": "G", "ent": "Chunghwa", "cls": "Western Europe"}, {"id": 19694, "dom": "G", "ent": "Samho-rodongjagu", "cls": "Australia and New Zealand"}, {"id": 19695, "dom": "G", "ent": "Samho-rodongjagu", "cls": "Caribbean"}, {"id": 19696, "dom": "G", "ent": "Samho-rodongjagu", "cls": "Central America"}, {"id": 19697, "dom": "G", "ent": "Samho-rodongjagu", "cls": "Central Asia"}, {"id": 19698, "dom": "G", "ent": "Samho-rodongjagu", "cls": "Eastern Africa"}, {"id": 19699, "dom": "G", "ent": "Samho-rodongjagu", "cls": "Eastern Asia"}, {"id": 19700, "dom": "G", "ent": "Samho-rodongjagu", "cls": "Eastern Europe"}, {"id": 19701, "dom": "G", "ent": "Samho-rodongjagu", "cls": "Melanesia"}, {"id": 19702, "dom": "G", "ent": "Samho-rodongjagu", "cls": "Micronesia"}, {"id": 19703, "dom": "G", "ent": "Samho-rodongjagu", "cls": "Middle Africa"}, {"id": 19704, "dom": "G", "ent": "Samho-rodongjagu", "cls": "Northern Africa"}, {"id": 19705, "dom": "G", "ent": "Samho-rodongjagu", "cls": "Northern America"}, {"id": 19706, "dom": "G", "ent": "Samho-rodongjagu", "cls": "Northern Europe"}, {"id": 19707, "dom": "G", "ent": "Samho-rodongjagu", "cls": "Polynesia"}, {"id": 19708, "dom": "G", "ent": "Samho-rodongjagu", "cls": "South America"}, {"id": 19709, "dom": "G", "ent": "Samho-rodongjagu", "cls": "South-eastern Asia"}, {"id": 19710, "dom": "G", "ent": "Samho-rodongjagu", "cls": "Southern Africa"}, {"id": 19711, "dom": "G", "ent": "Samho-rodongjagu", "cls": "Southern Asia"}, {"id": 19712, "dom": "G", "ent": "Samho-rodongjagu", "cls": "Southern Europe"}, {"id": 19713, "dom": "G", "ent": "Samho-rodongjagu", "cls": "Western Africa"}, {"id": 19714, "dom": "G", "ent": "Samho-rodongjagu", "cls": "Western Asia"}, {"id": 19715, "dom": "G", "ent": "Samho-rodongjagu", "cls": "Western Europe"}, {"id": 19716, "dom": "G", "ent": "Hwaseong-si", "cls": "Australia and New Zealand"}, {"id": 19717, "dom": "G", "ent": "Hwaseong-si", "cls": "Caribbean"}, {"id": 19718, "dom": "G", "ent": "Hwaseong-si", "cls": "Central America"}, {"id": 19719, "dom": "G", "ent": "Hwaseong-si", "cls": "Central Asia"}, {"id": 19720, "dom": "G", "ent": "Hwaseong-si", "cls": "Eastern Africa"}, {"id": 19721, "dom": "G", "ent": "Hwaseong-si", "cls": "Eastern Asia"}, {"id": 19722, "dom": "G", "ent": "Hwaseong-si", "cls": "Eastern Europe"}, {"id": 19723, "dom": "G", "ent": "Hwaseong-si", "cls": "Melanesia"}, {"id": 19724, "dom": "G", "ent": "Hwaseong-si", "cls": "Micronesia"}, {"id": 19725, "dom": "G", "ent": "Hwaseong-si", "cls": "Middle Africa"}, {"id": 19726, "dom": "G", "ent": "Hwaseong-si", "cls": "Northern Africa"}, {"id": 19727, "dom": "G", "ent": "Hwaseong-si", "cls": "Northern America"}, {"id": 19728, "dom": "G", "ent": "Hwaseong-si", "cls": "Northern Europe"}, {"id": 19729, "dom": "G", "ent": "Hwaseong-si", "cls": "Polynesia"}, {"id": 19730, "dom": "G", "ent": "Hwaseong-si", "cls": "South America"}, {"id": 19731, "dom": "G", "ent": "Hwaseong-si", "cls": "South-eastern Asia"}, {"id": 19732, "dom": "G", "ent": "Hwaseong-si", "cls": "Southern Africa"}, {"id": 19733, "dom": "G", "ent": "Hwaseong-si", "cls": "Southern Asia"}, {"id": 19734, "dom": "G", "ent": "Hwaseong-si", "cls": "Southern Europe"}, {"id": 19735, "dom": "G", "ent": "Hwaseong-si", "cls": "Western Africa"}, {"id": 19736, "dom": "G", "ent": "Hwaseong-si", "cls": "Western Asia"}, {"id": 19737, "dom": "G", "ent": "Hwaseong-si", "cls": "Western Europe"}, {"id": 19738, "dom": "G", "ent": "Masan", "cls": "Australia and New Zealand"}, {"id": 19739, "dom": "G", "ent": "Masan", "cls": "Caribbean"}, {"id": 19740, "dom": "G", "ent": "Masan", "cls": "Central America"}, {"id": 19741, "dom": "G", "ent": "Masan", "cls": "Central Asia"}, {"id": 19742, "dom": "G", "ent": "Masan", "cls": "Eastern Africa"}, {"id": 19743, "dom": "G", "ent": "Masan", "cls": "Eastern Asia"}, {"id": 19744, "dom": "G", "ent": "Masan", "cls": "Eastern Europe"}, {"id": 19745, "dom": "G", "ent": "Masan", "cls": "Melanesia"}, {"id": 19746, "dom": "G", "ent": "Masan", "cls": "Micronesia"}, {"id": 19747, "dom": "G", "ent": "Masan", "cls": "Middle Africa"}, {"id": 19748, "dom": "G", "ent": "Masan", "cls": "Northern Africa"}, {"id": 19749, "dom": "G", "ent": "Masan", "cls": "Northern America"}, {"id": 19750, "dom": "G", "ent": "Masan", "cls": "Northern Europe"}, {"id": 19751, "dom": "G", "ent": "Masan", "cls": "Polynesia"}, {"id": 19752, "dom": "G", "ent": "Masan", "cls": "South America"}, {"id": 19753, "dom": "G", "ent": "Masan", "cls": "South-eastern Asia"}, {"id": 19754, "dom": "G", "ent": "Masan", "cls": "Southern Africa"}, {"id": 19755, "dom": "G", "ent": "Masan", "cls": "Southern Asia"}, {"id": 19756, "dom": "G", "ent": "Masan", "cls": "Southern Europe"}, {"id": 19757, "dom": "G", "ent": "Masan", "cls": "Western Africa"}, {"id": 19758, "dom": "G", "ent": "Masan", "cls": "Western Asia"}, {"id": 19759, "dom": "G", "ent": "Masan", "cls": "Western Europe"}, {"id": 19760, "dom": "G", "ent": "Gwangyang", "cls": "Australia and New Zealand"}, {"id": 19761, "dom": "G", "ent": "Gwangyang", "cls": "Caribbean"}, {"id": 19762, "dom": "G", "ent": "Gwangyang", "cls": "Central America"}, {"id": 19763, "dom": "G", "ent": "Gwangyang", "cls": "Central Asia"}, {"id": 19764, "dom": "G", "ent": "Gwangyang", "cls": "Eastern Africa"}, {"id": 19765, "dom": "G", "ent": "Gwangyang", "cls": "Eastern Asia"}, {"id": 19766, "dom": "G", "ent": "Gwangyang", "cls": "Eastern Europe"}, {"id": 19767, "dom": "G", "ent": "Gwangyang", "cls": "Melanesia"}, {"id": 19768, "dom": "G", "ent": "Gwangyang", "cls": "Micronesia"}, {"id": 19769, "dom": "G", "ent": "Gwangyang", "cls": "Middle Africa"}, {"id": 19770, "dom": "G", "ent": "Gwangyang", "cls": "Northern Africa"}, {"id": 19771, "dom": "G", "ent": "Gwangyang", "cls": "Northern America"}, {"id": 19772, "dom": "G", "ent": "Gwangyang", "cls": "Northern Europe"}, {"id": 19773, "dom": "G", "ent": "Gwangyang", "cls": "Polynesia"}, {"id": 19774, "dom": "G", "ent": "Gwangyang", "cls": "South America"}, {"id": 19775, "dom": "G", "ent": "Gwangyang", "cls": "South-eastern Asia"}, {"id": 19776, "dom": "G", "ent": "Gwangyang", "cls": "Southern Africa"}, {"id": 19777, "dom": "G", "ent": "Gwangyang", "cls": "Southern Asia"}, {"id": 19778, "dom": "G", "ent": "Gwangyang", "cls": "Southern Europe"}, {"id": 19779, "dom": "G", "ent": "Gwangyang", "cls": "Western Africa"}, {"id": 19780, "dom": "G", "ent": "Gwangyang", "cls": "Western Asia"}, {"id": 19781, "dom": "G", "ent": "Gwangyang", "cls": "Western Europe"}, {"id": 19782, "dom": "G", "ent": "Tallang-dong", "cls": "Australia and New Zealand"}, {"id": 19783, "dom": "G", "ent": "Tallang-dong", "cls": "Caribbean"}, {"id": 19784, "dom": "G", "ent": "Tallang-dong", "cls": "Central America"}, {"id": 19785, "dom": "G", "ent": "Tallang-dong", "cls": "Central Asia"}, {"id": 19786, "dom": "G", "ent": "Tallang-dong", "cls": "Eastern Africa"}, {"id": 19787, "dom": "G", "ent": "Tallang-dong", "cls": "Eastern Asia"}, {"id": 19788, "dom": "G", "ent": "Tallang-dong", "cls": "Eastern Europe"}, {"id": 19789, "dom": "G", "ent": "Tallang-dong", "cls": "Melanesia"}, {"id": 19790, "dom": "G", "ent": "Tallang-dong", "cls": "Micronesia"}, {"id": 19791, "dom": "G", "ent": "Tallang-dong", "cls": "Middle Africa"}, {"id": 19792, "dom": "G", "ent": "Tallang-dong", "cls": "Northern Africa"}, {"id": 19793, "dom": "G", "ent": "Tallang-dong", "cls": "Northern America"}, {"id": 19794, "dom": "G", "ent": "Tallang-dong", "cls": "Northern Europe"}, {"id": 19795, "dom": "G", "ent": "Tallang-dong", "cls": "Polynesia"}, {"id": 19796, "dom": "G", "ent": "Tallang-dong", "cls": "South America"}, {"id": 19797, "dom": "G", "ent": "Tallang-dong", "cls": "South-eastern Asia"}, {"id": 19798, "dom": "G", "ent": "Tallang-dong", "cls": "Southern Africa"}, {"id": 19799, "dom": "G", "ent": "Tallang-dong", "cls": "Southern Asia"}, {"id": 19800, "dom": "G", "ent": "Tallang-dong", "cls": "Southern Europe"}, {"id": 19801, "dom": "G", "ent": "Tallang-dong", "cls": "Western Africa"}, {"id": 19802, "dom": "G", "ent": "Tallang-dong", "cls": "Western Asia"}, {"id": 19803, "dom": "G", "ent": "Tallang-dong", "cls": "Western Europe"}, {"id": 19804, "dom": "G", "ent": "Ban Khoan", "cls": "Australia and New Zealand"}, {"id": 19805, "dom": "G", "ent": "Ban Khoan", "cls": "Caribbean"}, {"id": 19806, "dom": "G", "ent": "Ban Khoan", "cls": "Central America"}, {"id": 19807, "dom": "G", "ent": "Ban Khoan", "cls": "Central Asia"}, {"id": 19808, "dom": "G", "ent": "Ban Khoan", "cls": "Eastern Africa"}, {"id": 19809, "dom": "G", "ent": "Ban Khoan", "cls": "Eastern Asia"}, {"id": 19810, "dom": "G", "ent": "Ban Khoan", "cls": "Eastern Europe"}, {"id": 19811, "dom": "G", "ent": "Ban Khoan", "cls": "Melanesia"}, {"id": 19812, "dom": "G", "ent": "Ban Khoan", "cls": "Micronesia"}, {"id": 19813, "dom": "G", "ent": "Ban Khoan", "cls": "Middle Africa"}, {"id": 19814, "dom": "G", "ent": "Ban Khoan", "cls": "Northern Africa"}, {"id": 19815, "dom": "G", "ent": "Ban Khoan", "cls": "Northern America"}, {"id": 19816, "dom": "G", "ent": "Ban Khoan", "cls": "Northern Europe"}, {"id": 19817, "dom": "G", "ent": "Ban Khoan", "cls": "Polynesia"}, {"id": 19818, "dom": "G", "ent": "Ban Khoan", "cls": "South America"}, {"id": 19819, "dom": "G", "ent": "Ban Khoan", "cls": "South-eastern Asia"}, {"id": 19820, "dom": "G", "ent": "Ban Khoan", "cls": "Southern Africa"}, {"id": 19821, "dom": "G", "ent": "Ban Khoan", "cls": "Southern Asia"}, {"id": 19822, "dom": "G", "ent": "Ban Khoan", "cls": "Southern Europe"}, {"id": 19823, "dom": "G", "ent": "Ban Khoan", "cls": "Western Africa"}, {"id": 19824, "dom": "G", "ent": "Ban Khoan", "cls": "Western Asia"}, {"id": 19825, "dom": "G", "ent": "Ban Khoan", "cls": "Western Europe"}, {"id": 19826, "dom": "G", "ent": "Xam Nua", "cls": "Australia and New Zealand"}, {"id": 19827, "dom": "G", "ent": "Xam Nua", "cls": "Caribbean"}, {"id": 19828, "dom": "G", "ent": "Xam Nua", "cls": "Central America"}, {"id": 19829, "dom": "G", "ent": "Xam Nua", "cls": "Central Asia"}, {"id": 19830, "dom": "G", "ent": "Xam Nua", "cls": "Eastern Africa"}, {"id": 19831, "dom": "G", "ent": "Xam Nua", "cls": "Eastern Asia"}, {"id": 19832, "dom": "G", "ent": "Xam Nua", "cls": "Eastern Europe"}, {"id": 19833, "dom": "G", "ent": "Xam Nua", "cls": "Melanesia"}, {"id": 19834, "dom": "G", "ent": "Xam Nua", "cls": "Micronesia"}, {"id": 19835, "dom": "G", "ent": "Xam Nua", "cls": "Middle Africa"}, {"id": 19836, "dom": "G", "ent": "Xam Nua", "cls": "Northern Africa"}, {"id": 19837, "dom": "G", "ent": "Xam Nua", "cls": "Northern America"}, {"id": 19838, "dom": "G", "ent": "Xam Nua", "cls": "Northern Europe"}, {"id": 19839, "dom": "G", "ent": "Xam Nua", "cls": "Polynesia"}, {"id": 19840, "dom": "G", "ent": "Xam Nua", "cls": "South America"}, {"id": 19841, "dom": "G", "ent": "Xam Nua", "cls": "South-eastern Asia"}, {"id": 19842, "dom": "G", "ent": "Xam Nua", "cls": "Southern Africa"}, {"id": 19843, "dom": "G", "ent": "Xam Nua", "cls": "Southern Asia"}, {"id": 19844, "dom": "G", "ent": "Xam Nua", "cls": "Southern Europe"}, {"id": 19845, "dom": "G", "ent": "Xam Nua", "cls": "Western Africa"}, {"id": 19846, "dom": "G", "ent": "Xam Nua", "cls": "Western Asia"}, {"id": 19847, "dom": "G", "ent": "Xam Nua", "cls": "Western Europe"}, {"id": 19848, "dom": "G", "ent": "Pakxan", "cls": "Australia and New Zealand"}, {"id": 19849, "dom": "G", "ent": "Pakxan", "cls": "Caribbean"}, {"id": 19850, "dom": "G", "ent": "Pakxan", "cls": "Central America"}, {"id": 19851, "dom": "G", "ent": "Pakxan", "cls": "Central Asia"}, {"id": 19852, "dom": "G", "ent": "Pakxan", "cls": "Eastern Africa"}, {"id": 19853, "dom": "G", "ent": "Pakxan", "cls": "Eastern Asia"}, {"id": 19854, "dom": "G", "ent": "Pakxan", "cls": "Eastern Europe"}, {"id": 19855, "dom": "G", "ent": "Pakxan", "cls": "Melanesia"}, {"id": 19856, "dom": "G", "ent": "Pakxan", "cls": "Micronesia"}, {"id": 19857, "dom": "G", "ent": "Pakxan", "cls": "Middle Africa"}, {"id": 19858, "dom": "G", "ent": "Pakxan", "cls": "Northern Africa"}, {"id": 19859, "dom": "G", "ent": "Pakxan", "cls": "Northern America"}, {"id": 19860, "dom": "G", "ent": "Pakxan", "cls": "Northern Europe"}, {"id": 19861, "dom": "G", "ent": "Pakxan", "cls": "Polynesia"}, {"id": 19862, "dom": "G", "ent": "Pakxan", "cls": "South America"}, {"id": 19863, "dom": "G", "ent": "Pakxan", "cls": "South-eastern Asia"}, {"id": 19864, "dom": "G", "ent": "Pakxan", "cls": "Southern Africa"}, {"id": 19865, "dom": "G", "ent": "Pakxan", "cls": "Southern Asia"}, {"id": 19866, "dom": "G", "ent": "Pakxan", "cls": "Southern Europe"}, {"id": 19867, "dom": "G", "ent": "Pakxan", "cls": "Western Africa"}, {"id": 19868, "dom": "G", "ent": "Pakxan", "cls": "Western Asia"}, {"id": 19869, "dom": "G", "ent": "Pakxan", "cls": "Western Europe"}, {"id": 19870, "dom": "G", "ent": "Sekong", "cls": "Australia and New Zealand"}, {"id": 19871, "dom": "G", "ent": "Sekong", "cls": "Caribbean"}, {"id": 19872, "dom": "G", "ent": "Sekong", "cls": "Central America"}, {"id": 19873, "dom": "G", "ent": "Sekong", "cls": "Central Asia"}, {"id": 19874, "dom": "G", "ent": "Sekong", "cls": "Eastern Africa"}, {"id": 19875, "dom": "G", "ent": "Sekong", "cls": "Eastern Asia"}, {"id": 19876, "dom": "G", "ent": "Sekong", "cls": "Eastern Europe"}, {"id": 19877, "dom": "G", "ent": "Sekong", "cls": "Melanesia"}, {"id": 19878, "dom": "G", "ent": "Sekong", "cls": "Micronesia"}, {"id": 19879, "dom": "G", "ent": "Sekong", "cls": "Middle Africa"}, {"id": 19880, "dom": "G", "ent": "Sekong", "cls": "Northern Africa"}, {"id": 19881, "dom": "G", "ent": "Sekong", "cls": "Northern America"}, {"id": 19882, "dom": "G", "ent": "Sekong", "cls": "Northern Europe"}, {"id": 19883, "dom": "G", "ent": "Sekong", "cls": "Polynesia"}, {"id": 19884, "dom": "G", "ent": "Sekong", "cls": "South America"}, {"id": 19885, "dom": "G", "ent": "Sekong", "cls": "South-eastern Asia"}, {"id": 19886, "dom": "G", "ent": "Sekong", "cls": "Southern Africa"}, {"id": 19887, "dom": "G", "ent": "Sekong", "cls": "Southern Asia"}, {"id": 19888, "dom": "G", "ent": "Sekong", "cls": "Southern Europe"}, {"id": 19889, "dom": "G", "ent": "Sekong", "cls": "Western Africa"}, {"id": 19890, "dom": "G", "ent": "Sekong", "cls": "Western Asia"}, {"id": 19891, "dom": "G", "ent": "Sekong", "cls": "Western Europe"}, {"id": 19892, "dom": "G", "ent": "Aley", "cls": "Australia and New Zealand"}, {"id": 19893, "dom": "G", "ent": "Aley", "cls": "Caribbean"}, {"id": 19894, "dom": "G", "ent": "Aley", "cls": "Central America"}, {"id": 19895, "dom": "G", "ent": "Aley", "cls": "Central Asia"}, {"id": 19896, "dom": "G", "ent": "Aley", "cls": "Eastern Africa"}, {"id": 19897, "dom": "G", "ent": "Aley", "cls": "Eastern Asia"}, {"id": 19898, "dom": "G", "ent": "Aley", "cls": "Eastern Europe"}, {"id": 19899, "dom": "G", "ent": "Aley", "cls": "Melanesia"}, {"id": 19900, "dom": "G", "ent": "Aley", "cls": "Micronesia"}, {"id": 19901, "dom": "G", "ent": "Aley", "cls": "Middle Africa"}, {"id": 19902, "dom": "G", "ent": "Aley", "cls": "Northern Africa"}, {"id": 19903, "dom": "G", "ent": "Aley", "cls": "Northern America"}, {"id": 19904, "dom": "G", "ent": "Aley", "cls": "Northern Europe"}, {"id": 19905, "dom": "G", "ent": "Aley", "cls": "Polynesia"}, {"id": 19906, "dom": "G", "ent": "Aley", "cls": "South America"}, {"id": 19907, "dom": "G", "ent": "Aley", "cls": "South-eastern Asia"}, {"id": 19908, "dom": "G", "ent": "Aley", "cls": "Southern Africa"}, {"id": 19909, "dom": "G", "ent": "Aley", "cls": "Southern Asia"}, {"id": 19910, "dom": "G", "ent": "Aley", "cls": "Southern Europe"}, {"id": 19911, "dom": "G", "ent": "Aley", "cls": "Western Africa"}, {"id": 19912, "dom": "G", "ent": "Aley", "cls": "Western Asia"}, {"id": 19913, "dom": "G", "ent": "Aley", "cls": "Western Europe"}, {"id": 19914, "dom": "G", "ent": "Jounieh", "cls": "Australia and New Zealand"}, {"id": 19915, "dom": "G", "ent": "Jounieh", "cls": "Caribbean"}, {"id": 19916, "dom": "G", "ent": "Jounieh", "cls": "Central America"}, {"id": 19917, "dom": "G", "ent": "Jounieh", "cls": "Central Asia"}, {"id": 19918, "dom": "G", "ent": "Jounieh", "cls": "Eastern Africa"}, {"id": 19919, "dom": "G", "ent": "Jounieh", "cls": "Eastern Asia"}, {"id": 19920, "dom": "G", "ent": "Jounieh", "cls": "Eastern Europe"}, {"id": 19921, "dom": "G", "ent": "Jounieh", "cls": "Melanesia"}, {"id": 19922, "dom": "G", "ent": "Jounieh", "cls": "Micronesia"}, {"id": 19923, "dom": "G", "ent": "Jounieh", "cls": "Middle Africa"}, {"id": 19924, "dom": "G", "ent": "Jounieh", "cls": "Northern Africa"}, {"id": 19925, "dom": "G", "ent": "Jounieh", "cls": "Northern America"}, {"id": 19926, "dom": "G", "ent": "Jounieh", "cls": "Northern Europe"}, {"id": 19927, "dom": "G", "ent": "Jounieh", "cls": "Polynesia"}, {"id": 19928, "dom": "G", "ent": "Jounieh", "cls": "South America"}, {"id": 19929, "dom": "G", "ent": "Jounieh", "cls": "South-eastern Asia"}, {"id": 19930, "dom": "G", "ent": "Jounieh", "cls": "Southern Africa"}, {"id": 19931, "dom": "G", "ent": "Jounieh", "cls": "Southern Asia"}, {"id": 19932, "dom": "G", "ent": "Jounieh", "cls": "Southern Europe"}, {"id": 19933, "dom": "G", "ent": "Jounieh", "cls": "Western Africa"}, {"id": 19934, "dom": "G", "ent": "Jounieh", "cls": "Western Asia"}, {"id": 19935, "dom": "G", "ent": "Jounieh", "cls": "Western Europe"}, {"id": 19936, "dom": "G", "ent": "Ghazieh", "cls": "Australia and New Zealand"}, {"id": 19937, "dom": "G", "ent": "Ghazieh", "cls": "Caribbean"}, {"id": 19938, "dom": "G", "ent": "Ghazieh", "cls": "Central America"}, {"id": 19939, "dom": "G", "ent": "Ghazieh", "cls": "Central Asia"}, {"id": 19940, "dom": "G", "ent": "Ghazieh", "cls": "Eastern Africa"}, {"id": 19941, "dom": "G", "ent": "Ghazieh", "cls": "Eastern Asia"}, {"id": 19942, "dom": "G", "ent": "Ghazieh", "cls": "Eastern Europe"}, {"id": 19943, "dom": "G", "ent": "Ghazieh", "cls": "Melanesia"}, {"id": 19944, "dom": "G", "ent": "Ghazieh", "cls": "Micronesia"}, {"id": 19945, "dom": "G", "ent": "Ghazieh", "cls": "Middle Africa"}, {"id": 19946, "dom": "G", "ent": "Ghazieh", "cls": "Northern Africa"}, {"id": 19947, "dom": "G", "ent": "Ghazieh", "cls": "Northern America"}, {"id": 19948, "dom": "G", "ent": "Ghazieh", "cls": "Northern Europe"}, {"id": 19949, "dom": "G", "ent": "Ghazieh", "cls": "Polynesia"}, {"id": 19950, "dom": "G", "ent": "Ghazieh", "cls": "South America"}, {"id": 19951, "dom": "G", "ent": "Ghazieh", "cls": "South-eastern Asia"}, {"id": 19952, "dom": "G", "ent": "Ghazieh", "cls": "Southern Africa"}, {"id": 19953, "dom": "G", "ent": "Ghazieh", "cls": "Southern Asia"}, {"id": 19954, "dom": "G", "ent": "Ghazieh", "cls": "Southern Europe"}, {"id": 19955, "dom": "G", "ent": "Ghazieh", "cls": "Western Africa"}, {"id": 19956, "dom": "G", "ent": "Ghazieh", "cls": "Western Asia"}, {"id": 19957, "dom": "G", "ent": "Ghazieh", "cls": "Western Europe"}, {"id": 19958, "dom": "G", "ent": "El Hermel", "cls": "Australia and New Zealand"}, {"id": 19959, "dom": "G", "ent": "El Hermel", "cls": "Caribbean"}, {"id": 19960, "dom": "G", "ent": "El Hermel", "cls": "Central America"}, {"id": 19961, "dom": "G", "ent": "El Hermel", "cls": "Central Asia"}, {"id": 19962, "dom": "G", "ent": "El Hermel", "cls": "Eastern Africa"}, {"id": 19963, "dom": "G", "ent": "El Hermel", "cls": "Eastern Asia"}, {"id": 19964, "dom": "G", "ent": "El Hermel", "cls": "Eastern Europe"}, {"id": 19965, "dom": "G", "ent": "El Hermel", "cls": "Melanesia"}, {"id": 19966, "dom": "G", "ent": "El Hermel", "cls": "Micronesia"}, {"id": 19967, "dom": "G", "ent": "El Hermel", "cls": "Middle Africa"}, {"id": 19968, "dom": "G", "ent": "El Hermel", "cls": "Northern Africa"}, {"id": 19969, "dom": "G", "ent": "El Hermel", "cls": "Northern America"}, {"id": 19970, "dom": "G", "ent": "El Hermel", "cls": "Northern Europe"}, {"id": 19971, "dom": "G", "ent": "El Hermel", "cls": "Polynesia"}, {"id": 19972, "dom": "G", "ent": "El Hermel", "cls": "South America"}, {"id": 19973, "dom": "G", "ent": "El Hermel", "cls": "South-eastern Asia"}, {"id": 19974, "dom": "G", "ent": "El Hermel", "cls": "Southern Africa"}, {"id": 19975, "dom": "G", "ent": "El Hermel", "cls": "Southern Asia"}, {"id": 19976, "dom": "G", "ent": "El Hermel", "cls": "Southern Europe"}, {"id": 19977, "dom": "G", "ent": "El Hermel", "cls": "Western Africa"}, {"id": 19978, "dom": "G", "ent": "El Hermel", "cls": "Western Asia"}, {"id": 19979, "dom": "G", "ent": "El Hermel", "cls": "Western Europe"}, {"id": 19980, "dom": "G", "ent": "Jaffna", "cls": "Australia and New Zealand"}, {"id": 19981, "dom": "G", "ent": "Jaffna", "cls": "Caribbean"}, {"id": 19982, "dom": "G", "ent": "Jaffna", "cls": "Central America"}, {"id": 19983, "dom": "G", "ent": "Jaffna", "cls": "Central Asia"}, {"id": 19984, "dom": "G", "ent": "Jaffna", "cls": "Eastern Africa"}, {"id": 19985, "dom": "G", "ent": "Jaffna", "cls": "Eastern Asia"}, {"id": 19986, "dom": "G", "ent": "Jaffna", "cls": "Eastern Europe"}, {"id": 19987, "dom": "G", "ent": "Jaffna", "cls": "Melanesia"}, {"id": 19988, "dom": "G", "ent": "Jaffna", "cls": "Micronesia"}, {"id": 19989, "dom": "G", "ent": "Jaffna", "cls": "Middle Africa"}, {"id": 19990, "dom": "G", "ent": "Jaffna", "cls": "Northern Africa"}, {"id": 19991, "dom": "G", "ent": "Jaffna", "cls": "Northern America"}, {"id": 19992, "dom": "G", "ent": "Jaffna", "cls": "Northern Europe"}, {"id": 19993, "dom": "G", "ent": "Jaffna", "cls": "Polynesia"}, {"id": 19994, "dom": "G", "ent": "Jaffna", "cls": "South America"}, {"id": 19995, "dom": "G", "ent": "Jaffna", "cls": "South-eastern Asia"}, {"id": 19996, "dom": "G", "ent": "Jaffna", "cls": "Southern Africa"}, {"id": 19997, "dom": "G", "ent": "Jaffna", "cls": "Southern Asia"}, {"id": 19998, "dom": "G", "ent": "Jaffna", "cls": "Southern Europe"}, {"id": 19999, "dom": "G", "ent": "Jaffna", "cls": "Western Africa"}, {"id": 20000, "dom": "G", "ent": "Jaffna", "cls": "Western Asia"}, {"id": 20001, "dom": "G", "ent": "Jaffna", "cls": "Western Europe"}, {"id": 20002, "dom": "G", "ent": "Ambalangoda", "cls": "Australia and New Zealand"}, {"id": 20003, "dom": "G", "ent": "Ambalangoda", "cls": "Caribbean"}, {"id": 20004, "dom": "G", "ent": "Ambalangoda", "cls": "Central America"}, {"id": 20005, "dom": "G", "ent": "Ambalangoda", "cls": "Central Asia"}, {"id": 20006, "dom": "G", "ent": "Ambalangoda", "cls": "Eastern Africa"}, {"id": 20007, "dom": "G", "ent": "Ambalangoda", "cls": "Eastern Asia"}, {"id": 20008, "dom": "G", "ent": "Ambalangoda", "cls": "Eastern Europe"}, {"id": 20009, "dom": "G", "ent": "Ambalangoda", "cls": "Melanesia"}, {"id": 20010, "dom": "G", "ent": "Ambalangoda", "cls": "Micronesia"}, {"id": 20011, "dom": "G", "ent": "Ambalangoda", "cls": "Middle Africa"}, {"id": 20012, "dom": "G", "ent": "Ambalangoda", "cls": "Northern Africa"}, {"id": 20013, "dom": "G", "ent": "Ambalangoda", "cls": "Northern America"}, {"id": 20014, "dom": "G", "ent": "Ambalangoda", "cls": "Northern Europe"}, {"id": 20015, "dom": "G", "ent": "Ambalangoda", "cls": "Polynesia"}, {"id": 20016, "dom": "G", "ent": "Ambalangoda", "cls": "South America"}, {"id": 20017, "dom": "G", "ent": "Ambalangoda", "cls": "South-eastern Asia"}, {"id": 20018, "dom": "G", "ent": "Ambalangoda", "cls": "Southern Africa"}, {"id": 20019, "dom": "G", "ent": "Ambalangoda", "cls": "Southern Asia"}, {"id": 20020, "dom": "G", "ent": "Ambalangoda", "cls": "Southern Europe"}, {"id": 20021, "dom": "G", "ent": "Ambalangoda", "cls": "Western Africa"}, {"id": 20022, "dom": "G", "ent": "Ambalangoda", "cls": "Western Asia"}, {"id": 20023, "dom": "G", "ent": "Ambalangoda", "cls": "Western Europe"}, {"id": 20024, "dom": "G", "ent": "Kandana", "cls": "Australia and New Zealand"}, {"id": 20025, "dom": "G", "ent": "Kandana", "cls": "Caribbean"}, {"id": 20026, "dom": "G", "ent": "Kandana", "cls": "Central America"}, {"id": 20027, "dom": "G", "ent": "Kandana", "cls": "Central Asia"}, {"id": 20028, "dom": "G", "ent": "Kandana", "cls": "Eastern Africa"}, {"id": 20029, "dom": "G", "ent": "Kandana", "cls": "Eastern Asia"}, {"id": 20030, "dom": "G", "ent": "Kandana", "cls": "Eastern Europe"}, {"id": 20031, "dom": "G", "ent": "Kandana", "cls": "Melanesia"}, {"id": 20032, "dom": "G", "ent": "Kandana", "cls": "Micronesia"}, {"id": 20033, "dom": "G", "ent": "Kandana", "cls": "Middle Africa"}, {"id": 20034, "dom": "G", "ent": "Kandana", "cls": "Northern Africa"}, {"id": 20035, "dom": "G", "ent": "Kandana", "cls": "Northern America"}, {"id": 20036, "dom": "G", "ent": "Kandana", "cls": "Northern Europe"}, {"id": 20037, "dom": "G", "ent": "Kandana", "cls": "Polynesia"}, {"id": 20038, "dom": "G", "ent": "Kandana", "cls": "South America"}, {"id": 20039, "dom": "G", "ent": "Kandana", "cls": "South-eastern Asia"}, {"id": 20040, "dom": "G", "ent": "Kandana", "cls": "Southern Africa"}, {"id": 20041, "dom": "G", "ent": "Kandana", "cls": "Southern Asia"}, {"id": 20042, "dom": "G", "ent": "Kandana", "cls": "Southern Europe"}, {"id": 20043, "dom": "G", "ent": "Kandana", "cls": "Western Africa"}, {"id": 20044, "dom": "G", "ent": "Kandana", "cls": "Western Asia"}, {"id": 20045, "dom": "G", "ent": "Kandana", "cls": "Western Europe"}, {"id": 20046, "dom": "G", "ent": "Hambantota", "cls": "Australia and New Zealand"}, {"id": 20047, "dom": "G", "ent": "Hambantota", "cls": "Caribbean"}, {"id": 20048, "dom": "G", "ent": "Hambantota", "cls": "Central America"}, {"id": 20049, "dom": "G", "ent": "Hambantota", "cls": "Central Asia"}, {"id": 20050, "dom": "G", "ent": "Hambantota", "cls": "Eastern Africa"}, {"id": 20051, "dom": "G", "ent": "Hambantota", "cls": "Eastern Asia"}, {"id": 20052, "dom": "G", "ent": "Hambantota", "cls": "Eastern Europe"}, {"id": 20053, "dom": "G", "ent": "Hambantota", "cls": "Melanesia"}, {"id": 20054, "dom": "G", "ent": "Hambantota", "cls": "Micronesia"}, {"id": 20055, "dom": "G", "ent": "Hambantota", "cls": "Middle Africa"}, {"id": 20056, "dom": "G", "ent": "Hambantota", "cls": "Northern Africa"}, {"id": 20057, "dom": "G", "ent": "Hambantota", "cls": "Northern America"}, {"id": 20058, "dom": "G", "ent": "Hambantota", "cls": "Northern Europe"}, {"id": 20059, "dom": "G", "ent": "Hambantota", "cls": "Polynesia"}, {"id": 20060, "dom": "G", "ent": "Hambantota", "cls": "South America"}, {"id": 20061, "dom": "G", "ent": "Hambantota", "cls": "South-eastern Asia"}, {"id": 20062, "dom": "G", "ent": "Hambantota", "cls": "Southern Africa"}, {"id": 20063, "dom": "G", "ent": "Hambantota", "cls": "Southern Asia"}, {"id": 20064, "dom": "G", "ent": "Hambantota", "cls": "Southern Europe"}, {"id": 20065, "dom": "G", "ent": "Hambantota", "cls": "Western Africa"}, {"id": 20066, "dom": "G", "ent": "Hambantota", "cls": "Western Asia"}, {"id": 20067, "dom": "G", "ent": "Hambantota", "cls": "Western Europe"}, {"id": 20068, "dom": "G", "ent": "Buchanan", "cls": "Australia and New Zealand"}, {"id": 20069, "dom": "G", "ent": "Buchanan", "cls": "Caribbean"}, {"id": 20070, "dom": "G", "ent": "Buchanan", "cls": "Central America"}, {"id": 20071, "dom": "G", "ent": "Buchanan", "cls": "Central Asia"}, {"id": 20072, "dom": "G", "ent": "Buchanan", "cls": "Eastern Africa"}, {"id": 20073, "dom": "G", "ent": "Buchanan", "cls": "Eastern Asia"}, {"id": 20074, "dom": "G", "ent": "Buchanan", "cls": "Eastern Europe"}, {"id": 20075, "dom": "G", "ent": "Buchanan", "cls": "Melanesia"}, {"id": 20076, "dom": "G", "ent": "Buchanan", "cls": "Micronesia"}, {"id": 20077, "dom": "G", "ent": "Buchanan", "cls": "Middle Africa"}, {"id": 20078, "dom": "G", "ent": "Buchanan", "cls": "Northern Africa"}, {"id": 20079, "dom": "G", "ent": "Buchanan", "cls": "Northern America"}, {"id": 20080, "dom": "G", "ent": "Buchanan", "cls": "Northern Europe"}, {"id": 20081, "dom": "G", "ent": "Buchanan", "cls": "Polynesia"}, {"id": 20082, "dom": "G", "ent": "Buchanan", "cls": "South America"}, {"id": 20083, "dom": "G", "ent": "Buchanan", "cls": "South-eastern Asia"}, {"id": 20084, "dom": "G", "ent": "Buchanan", "cls": "Southern Africa"}, {"id": 20085, "dom": "G", "ent": "Buchanan", "cls": "Southern Asia"}, {"id": 20086, "dom": "G", "ent": "Buchanan", "cls": "Southern Europe"}, {"id": 20087, "dom": "G", "ent": "Buchanan", "cls": "Western Africa"}, {"id": 20088, "dom": "G", "ent": "Buchanan", "cls": "Western Asia"}, {"id": 20089, "dom": "G", "ent": "Buchanan", "cls": "Western Europe"}, {"id": 20090, "dom": "G", "ent": "Ganta", "cls": "Australia and New Zealand"}, {"id": 20091, "dom": "G", "ent": "Ganta", "cls": "Caribbean"}, {"id": 20092, "dom": "G", "ent": "Ganta", "cls": "Central America"}, {"id": 20093, "dom": "G", "ent": "Ganta", "cls": "Central Asia"}, {"id": 20094, "dom": "G", "ent": "Ganta", "cls": "Eastern Africa"}, {"id": 20095, "dom": "G", "ent": "Ganta", "cls": "Eastern Asia"}, {"id": 20096, "dom": "G", "ent": "Ganta", "cls": "Eastern Europe"}, {"id": 20097, "dom": "G", "ent": "Ganta", "cls": "Melanesia"}, {"id": 20098, "dom": "G", "ent": "Ganta", "cls": "Micronesia"}, {"id": 20099, "dom": "G", "ent": "Ganta", "cls": "Middle Africa"}, {"id": 20100, "dom": "G", "ent": "Ganta", "cls": "Northern Africa"}, {"id": 20101, "dom": "G", "ent": "Ganta", "cls": "Northern America"}, {"id": 20102, "dom": "G", "ent": "Ganta", "cls": "Northern Europe"}, {"id": 20103, "dom": "G", "ent": "Ganta", "cls": "Polynesia"}, {"id": 20104, "dom": "G", "ent": "Ganta", "cls": "South America"}, {"id": 20105, "dom": "G", "ent": "Ganta", "cls": "South-eastern Asia"}, {"id": 20106, "dom": "G", "ent": "Ganta", "cls": "Southern Africa"}, {"id": 20107, "dom": "G", "ent": "Ganta", "cls": "Southern Asia"}, {"id": 20108, "dom": "G", "ent": "Ganta", "cls": "Southern Europe"}, {"id": 20109, "dom": "G", "ent": "Ganta", "cls": "Western Africa"}, {"id": 20110, "dom": "G", "ent": "Ganta", "cls": "Western Asia"}, {"id": 20111, "dom": "G", "ent": "Ganta", "cls": "Western Europe"}, {"id": 20112, "dom": "G", "ent": "Harbel", "cls": "Australia and New Zealand"}, {"id": 20113, "dom": "G", "ent": "Harbel", "cls": "Caribbean"}, {"id": 20114, "dom": "G", "ent": "Harbel", "cls": "Central America"}, {"id": 20115, "dom": "G", "ent": "Harbel", "cls": "Central Asia"}, {"id": 20116, "dom": "G", "ent": "Harbel", "cls": "Eastern Africa"}, {"id": 20117, "dom": "G", "ent": "Harbel", "cls": "Eastern Asia"}, {"id": 20118, "dom": "G", "ent": "Harbel", "cls": "Eastern Europe"}, {"id": 20119, "dom": "G", "ent": "Harbel", "cls": "Melanesia"}, {"id": 20120, "dom": "G", "ent": "Harbel", "cls": "Micronesia"}, {"id": 20121, "dom": "G", "ent": "Harbel", "cls": "Middle Africa"}, {"id": 20122, "dom": "G", "ent": "Harbel", "cls": "Northern Africa"}, {"id": 20123, "dom": "G", "ent": "Harbel", "cls": "Northern America"}, {"id": 20124, "dom": "G", "ent": "Harbel", "cls": "Northern Europe"}, {"id": 20125, "dom": "G", "ent": "Harbel", "cls": "Polynesia"}, {"id": 20126, "dom": "G", "ent": "Harbel", "cls": "South America"}, {"id": 20127, "dom": "G", "ent": "Harbel", "cls": "South-eastern Asia"}, {"id": 20128, "dom": "G", "ent": "Harbel", "cls": "Southern Africa"}, {"id": 20129, "dom": "G", "ent": "Harbel", "cls": "Southern Asia"}, {"id": 20130, "dom": "G", "ent": "Harbel", "cls": "Southern Europe"}, {"id": 20131, "dom": "G", "ent": "Harbel", "cls": "Western Africa"}, {"id": 20132, "dom": "G", "ent": "Harbel", "cls": "Western Asia"}, {"id": 20133, "dom": "G", "ent": "Harbel", "cls": "Western Europe"}, {"id": 20134, "dom": "G", "ent": "Saclepea", "cls": "Australia and New Zealand"}, {"id": 20135, "dom": "G", "ent": "Saclepea", "cls": "Caribbean"}, {"id": 20136, "dom": "G", "ent": "Saclepea", "cls": "Central America"}, {"id": 20137, "dom": "G", "ent": "Saclepea", "cls": "Central Asia"}, {"id": 20138, "dom": "G", "ent": "Saclepea", "cls": "Eastern Africa"}, {"id": 20139, "dom": "G", "ent": "Saclepea", "cls": "Eastern Asia"}, {"id": 20140, "dom": "G", "ent": "Saclepea", "cls": "Eastern Europe"}, {"id": 20141, "dom": "G", "ent": "Saclepea", "cls": "Melanesia"}, {"id": 20142, "dom": "G", "ent": "Saclepea", "cls": "Micronesia"}, {"id": 20143, "dom": "G", "ent": "Saclepea", "cls": "Middle Africa"}, {"id": 20144, "dom": "G", "ent": "Saclepea", "cls": "Northern Africa"}, {"id": 20145, "dom": "G", "ent": "Saclepea", "cls": "Northern America"}, {"id": 20146, "dom": "G", "ent": "Saclepea", "cls": "Northern Europe"}, {"id": 20147, "dom": "G", "ent": "Saclepea", "cls": "Polynesia"}, {"id": 20148, "dom": "G", "ent": "Saclepea", "cls": "South America"}, {"id": 20149, "dom": "G", "ent": "Saclepea", "cls": "South-eastern Asia"}, {"id": 20150, "dom": "G", "ent": "Saclepea", "cls": "Southern Africa"}, {"id": 20151, "dom": "G", "ent": "Saclepea", "cls": "Southern Asia"}, {"id": 20152, "dom": "G", "ent": "Saclepea", "cls": "Southern Europe"}, {"id": 20153, "dom": "G", "ent": "Saclepea", "cls": "Western Africa"}, {"id": 20154, "dom": "G", "ent": "Saclepea", "cls": "Western Asia"}, {"id": 20155, "dom": "G", "ent": "Saclepea", "cls": "Western Europe"}, {"id": 20156, "dom": "G", "ent": "Maputsoe", "cls": "Australia and New Zealand"}, {"id": 20157, "dom": "G", "ent": "Maputsoe", "cls": "Caribbean"}, {"id": 20158, "dom": "G", "ent": "Maputsoe", "cls": "Central America"}, {"id": 20159, "dom": "G", "ent": "Maputsoe", "cls": "Central Asia"}, {"id": 20160, "dom": "G", "ent": "Maputsoe", "cls": "Eastern Africa"}, {"id": 20161, "dom": "G", "ent": "Maputsoe", "cls": "Eastern Asia"}, {"id": 20162, "dom": "G", "ent": "Maputsoe", "cls": "Eastern Europe"}, {"id": 20163, "dom": "G", "ent": "Maputsoe", "cls": "Melanesia"}, {"id": 20164, "dom": "G", "ent": "Maputsoe", "cls": "Micronesia"}, {"id": 20165, "dom": "G", "ent": "Maputsoe", "cls": "Middle Africa"}, {"id": 20166, "dom": "G", "ent": "Maputsoe", "cls": "Northern Africa"}, {"id": 20167, "dom": "G", "ent": "Maputsoe", "cls": "Northern America"}, {"id": 20168, "dom": "G", "ent": "Maputsoe", "cls": "Northern Europe"}, {"id": 20169, "dom": "G", "ent": "Maputsoe", "cls": "Polynesia"}, {"id": 20170, "dom": "G", "ent": "Maputsoe", "cls": "South America"}, {"id": 20171, "dom": "G", "ent": "Maputsoe", "cls": "South-eastern Asia"}, {"id": 20172, "dom": "G", "ent": "Maputsoe", "cls": "Southern Africa"}, {"id": 20173, "dom": "G", "ent": "Maputsoe", "cls": "Southern Asia"}, {"id": 20174, "dom": "G", "ent": "Maputsoe", "cls": "Southern Europe"}, {"id": 20175, "dom": "G", "ent": "Maputsoe", "cls": "Western Africa"}, {"id": 20176, "dom": "G", "ent": "Maputsoe", "cls": "Western Asia"}, {"id": 20177, "dom": "G", "ent": "Maputsoe", "cls": "Western Europe"}, {"id": 20178, "dom": "G", "ent": "Mohale's Hoek", "cls": "Australia and New Zealand"}, {"id": 20179, "dom": "G", "ent": "Mohale's Hoek", "cls": "Caribbean"}, {"id": 20180, "dom": "G", "ent": "Mohale's Hoek", "cls": "Central America"}, {"id": 20181, "dom": "G", "ent": "Mohale's Hoek", "cls": "Central Asia"}, {"id": 20182, "dom": "G", "ent": "Mohale's Hoek", "cls": "Eastern Africa"}, {"id": 20183, "dom": "G", "ent": "Mohale's Hoek", "cls": "Eastern Asia"}, {"id": 20184, "dom": "G", "ent": "Mohale's Hoek", "cls": "Eastern Europe"}, {"id": 20185, "dom": "G", "ent": "Mohale's Hoek", "cls": "Melanesia"}, {"id": 20186, "dom": "G", "ent": "Mohale's Hoek", "cls": "Micronesia"}, {"id": 20187, "dom": "G", "ent": "Mohale's Hoek", "cls": "Middle Africa"}, {"id": 20188, "dom": "G", "ent": "Mohale's Hoek", "cls": "Northern Africa"}, {"id": 20189, "dom": "G", "ent": "Mohale's Hoek", "cls": "Northern America"}, {"id": 20190, "dom": "G", "ent": "Mohale's Hoek", "cls": "Northern Europe"}, {"id": 20191, "dom": "G", "ent": "Mohale's Hoek", "cls": "Polynesia"}, {"id": 20192, "dom": "G", "ent": "Mohale's Hoek", "cls": "South America"}, {"id": 20193, "dom": "G", "ent": "Mohale's Hoek", "cls": "South-eastern Asia"}, {"id": 20194, "dom": "G", "ent": "Mohale's Hoek", "cls": "Southern Africa"}, {"id": 20195, "dom": "G", "ent": "Mohale's Hoek", "cls": "Southern Asia"}, {"id": 20196, "dom": "G", "ent": "Mohale's Hoek", "cls": "Southern Europe"}, {"id": 20197, "dom": "G", "ent": "Mohale's Hoek", "cls": "Western Africa"}, {"id": 20198, "dom": "G", "ent": "Mohale's Hoek", "cls": "Western Asia"}, {"id": 20199, "dom": "G", "ent": "Mohale's Hoek", "cls": "Western Europe"}, {"id": 20200, "dom": "G", "ent": "Mafeteng", "cls": "Australia and New Zealand"}, {"id": 20201, "dom": "G", "ent": "Mafeteng", "cls": "Caribbean"}, {"id": 20202, "dom": "G", "ent": "Mafeteng", "cls": "Central America"}, {"id": 20203, "dom": "G", "ent": "Mafeteng", "cls": "Central Asia"}, {"id": 20204, "dom": "G", "ent": "Mafeteng", "cls": "Eastern Africa"}, {"id": 20205, "dom": "G", "ent": "Mafeteng", "cls": "Eastern Asia"}, {"id": 20206, "dom": "G", "ent": "Mafeteng", "cls": "Eastern Europe"}, {"id": 20207, "dom": "G", "ent": "Mafeteng", "cls": "Melanesia"}, {"id": 20208, "dom": "G", "ent": "Mafeteng", "cls": "Micronesia"}, {"id": 20209, "dom": "G", "ent": "Mafeteng", "cls": "Middle Africa"}, {"id": 20210, "dom": "G", "ent": "Mafeteng", "cls": "Northern Africa"}, {"id": 20211, "dom": "G", "ent": "Mafeteng", "cls": "Northern America"}, {"id": 20212, "dom": "G", "ent": "Mafeteng", "cls": "Northern Europe"}, {"id": 20213, "dom": "G", "ent": "Mafeteng", "cls": "Polynesia"}, {"id": 20214, "dom": "G", "ent": "Mafeteng", "cls": "South America"}, {"id": 20215, "dom": "G", "ent": "Mafeteng", "cls": "South-eastern Asia"}, {"id": 20216, "dom": "G", "ent": "Mafeteng", "cls": "Southern Africa"}, {"id": 20217, "dom": "G", "ent": "Mafeteng", "cls": "Southern Asia"}, {"id": 20218, "dom": "G", "ent": "Mafeteng", "cls": "Southern Europe"}, {"id": 20219, "dom": "G", "ent": "Mafeteng", "cls": "Western Africa"}, {"id": 20220, "dom": "G", "ent": "Mafeteng", "cls": "Western Asia"}, {"id": 20221, "dom": "G", "ent": "Mafeteng", "cls": "Western Europe"}, {"id": 20222, "dom": "G", "ent": "Thaba-Tseka", "cls": "Australia and New Zealand"}, {"id": 20223, "dom": "G", "ent": "Thaba-Tseka", "cls": "Caribbean"}, {"id": 20224, "dom": "G", "ent": "Thaba-Tseka", "cls": "Central America"}, {"id": 20225, "dom": "G", "ent": "Thaba-Tseka", "cls": "Central Asia"}, {"id": 20226, "dom": "G", "ent": "Thaba-Tseka", "cls": "Eastern Africa"}, {"id": 20227, "dom": "G", "ent": "Thaba-Tseka", "cls": "Eastern Asia"}, {"id": 20228, "dom": "G", "ent": "Thaba-Tseka", "cls": "Eastern Europe"}, {"id": 20229, "dom": "G", "ent": "Thaba-Tseka", "cls": "Melanesia"}, {"id": 20230, "dom": "G", "ent": "Thaba-Tseka", "cls": "Micronesia"}, {"id": 20231, "dom": "G", "ent": "Thaba-Tseka", "cls": "Middle Africa"}, {"id": 20232, "dom": "G", "ent": "Thaba-Tseka", "cls": "Northern Africa"}, {"id": 20233, "dom": "G", "ent": "Thaba-Tseka", "cls": "Northern America"}, {"id": 20234, "dom": "G", "ent": "Thaba-Tseka", "cls": "Northern Europe"}, {"id": 20235, "dom": "G", "ent": "Thaba-Tseka", "cls": "Polynesia"}, {"id": 20236, "dom": "G", "ent": "Thaba-Tseka", "cls": "South America"}, {"id": 20237, "dom": "G", "ent": "Thaba-Tseka", "cls": "South-eastern Asia"}, {"id": 20238, "dom": "G", "ent": "Thaba-Tseka", "cls": "Southern Africa"}, {"id": 20239, "dom": "G", "ent": "Thaba-Tseka", "cls": "Southern Asia"}, {"id": 20240, "dom": "G", "ent": "Thaba-Tseka", "cls": "Southern Europe"}, {"id": 20241, "dom": "G", "ent": "Thaba-Tseka", "cls": "Western Africa"}, {"id": 20242, "dom": "G", "ent": "Thaba-Tseka", "cls": "Western Asia"}, {"id": 20243, "dom": "G", "ent": "Thaba-Tseka", "cls": "Western Europe"}, {"id": 20244, "dom": "G", "ent": "Kaunas", "cls": "Australia and New Zealand"}, {"id": 20245, "dom": "G", "ent": "Kaunas", "cls": "Caribbean"}, {"id": 20246, "dom": "G", "ent": "Kaunas", "cls": "Central America"}, {"id": 20247, "dom": "G", "ent": "Kaunas", "cls": "Central Asia"}, {"id": 20248, "dom": "G", "ent": "Kaunas", "cls": "Eastern Africa"}, {"id": 20249, "dom": "G", "ent": "Kaunas", "cls": "Eastern Asia"}, {"id": 20250, "dom": "G", "ent": "Kaunas", "cls": "Eastern Europe"}, {"id": 20251, "dom": "G", "ent": "Kaunas", "cls": "Melanesia"}, {"id": 20252, "dom": "G", "ent": "Kaunas", "cls": "Micronesia"}, {"id": 20253, "dom": "G", "ent": "Kaunas", "cls": "Middle Africa"}, {"id": 20254, "dom": "G", "ent": "Kaunas", "cls": "Northern Africa"}, {"id": 20255, "dom": "G", "ent": "Kaunas", "cls": "Northern America"}, {"id": 20256, "dom": "G", "ent": "Kaunas", "cls": "Northern Europe"}, {"id": 20257, "dom": "G", "ent": "Kaunas", "cls": "Polynesia"}, {"id": 20258, "dom": "G", "ent": "Kaunas", "cls": "South America"}, {"id": 20259, "dom": "G", "ent": "Kaunas", "cls": "South-eastern Asia"}, {"id": 20260, "dom": "G", "ent": "Kaunas", "cls": "Southern Africa"}, {"id": 20261, "dom": "G", "ent": "Kaunas", "cls": "Southern Asia"}, {"id": 20262, "dom": "G", "ent": "Kaunas", "cls": "Southern Europe"}, {"id": 20263, "dom": "G", "ent": "Kaunas", "cls": "Western Africa"}, {"id": 20264, "dom": "G", "ent": "Kaunas", "cls": "Western Asia"}, {"id": 20265, "dom": "G", "ent": "Kaunas", "cls": "Western Europe"}, {"id": 20266, "dom": "G", "ent": "Eiguliai", "cls": "Australia and New Zealand"}, {"id": 20267, "dom": "G", "ent": "Eiguliai", "cls": "Caribbean"}, {"id": 20268, "dom": "G", "ent": "Eiguliai", "cls": "Central America"}, {"id": 20269, "dom": "G", "ent": "Eiguliai", "cls": "Central Asia"}, {"id": 20270, "dom": "G", "ent": "Eiguliai", "cls": "Eastern Africa"}, {"id": 20271, "dom": "G", "ent": "Eiguliai", "cls": "Eastern Asia"}, {"id": 20272, "dom": "G", "ent": "Eiguliai", "cls": "Eastern Europe"}, {"id": 20273, "dom": "G", "ent": "Eiguliai", "cls": "Melanesia"}, {"id": 20274, "dom": "G", "ent": "Eiguliai", "cls": "Micronesia"}, {"id": 20275, "dom": "G", "ent": "Eiguliai", "cls": "Middle Africa"}, {"id": 20276, "dom": "G", "ent": "Eiguliai", "cls": "Northern Africa"}, {"id": 20277, "dom": "G", "ent": "Eiguliai", "cls": "Northern America"}, {"id": 20278, "dom": "G", "ent": "Eiguliai", "cls": "Northern Europe"}, {"id": 20279, "dom": "G", "ent": "Eiguliai", "cls": "Polynesia"}, {"id": 20280, "dom": "G", "ent": "Eiguliai", "cls": "South America"}, {"id": 20281, "dom": "G", "ent": "Eiguliai", "cls": "South-eastern Asia"}, {"id": 20282, "dom": "G", "ent": "Eiguliai", "cls": "Southern Africa"}, {"id": 20283, "dom": "G", "ent": "Eiguliai", "cls": "Southern Asia"}, {"id": 20284, "dom": "G", "ent": "Eiguliai", "cls": "Southern Europe"}, {"id": 20285, "dom": "G", "ent": "Eiguliai", "cls": "Western Africa"}, {"id": 20286, "dom": "G", "ent": "Eiguliai", "cls": "Western Asia"}, {"id": 20287, "dom": "G", "ent": "Eiguliai", "cls": "Western Europe"}, {"id": 20288, "dom": "G", "ent": "Naujininkai", "cls": "Australia and New Zealand"}, {"id": 20289, "dom": "G", "ent": "Naujininkai", "cls": "Caribbean"}, {"id": 20290, "dom": "G", "ent": "Naujininkai", "cls": "Central America"}, {"id": 20291, "dom": "G", "ent": "Naujininkai", "cls": "Central Asia"}, {"id": 20292, "dom": "G", "ent": "Naujininkai", "cls": "Eastern Africa"}, {"id": 20293, "dom": "G", "ent": "Naujininkai", "cls": "Eastern Asia"}, {"id": 20294, "dom": "G", "ent": "Naujininkai", "cls": "Eastern Europe"}, {"id": 20295, "dom": "G", "ent": "Naujininkai", "cls": "Melanesia"}, {"id": 20296, "dom": "G", "ent": "Naujininkai", "cls": "Micronesia"}, {"id": 20297, "dom": "G", "ent": "Naujininkai", "cls": "Middle Africa"}, {"id": 20298, "dom": "G", "ent": "Naujininkai", "cls": "Northern Africa"}, {"id": 20299, "dom": "G", "ent": "Naujininkai", "cls": "Northern America"}, {"id": 20300, "dom": "G", "ent": "Naujininkai", "cls": "Northern Europe"}, {"id": 20301, "dom": "G", "ent": "Naujininkai", "cls": "Polynesia"}, {"id": 20302, "dom": "G", "ent": "Naujininkai", "cls": "South America"}, {"id": 20303, "dom": "G", "ent": "Naujininkai", "cls": "South-eastern Asia"}, {"id": 20304, "dom": "G", "ent": "Naujininkai", "cls": "Southern Africa"}, {"id": 20305, "dom": "G", "ent": "Naujininkai", "cls": "Southern Asia"}, {"id": 20306, "dom": "G", "ent": "Naujininkai", "cls": "Southern Europe"}, {"id": 20307, "dom": "G", "ent": "Naujininkai", "cls": "Western Africa"}, {"id": 20308, "dom": "G", "ent": "Naujininkai", "cls": "Western Asia"}, {"id": 20309, "dom": "G", "ent": "Naujininkai", "cls": "Western Europe"}, {"id": 20310, "dom": "G", "ent": "Telsiai", "cls": "Australia and New Zealand"}, {"id": 20311, "dom": "G", "ent": "Telsiai", "cls": "Caribbean"}, {"id": 20312, "dom": "G", "ent": "Telsiai", "cls": "Central America"}, {"id": 20313, "dom": "G", "ent": "Telsiai", "cls": "Central Asia"}, {"id": 20314, "dom": "G", "ent": "Telsiai", "cls": "Eastern Africa"}, {"id": 20315, "dom": "G", "ent": "Telsiai", "cls": "Eastern Asia"}, {"id": 20316, "dom": "G", "ent": "Telsiai", "cls": "Eastern Europe"}, {"id": 20317, "dom": "G", "ent": "Telsiai", "cls": "Melanesia"}, {"id": 20318, "dom": "G", "ent": "Telsiai", "cls": "Micronesia"}, {"id": 20319, "dom": "G", "ent": "Telsiai", "cls": "Middle Africa"}, {"id": 20320, "dom": "G", "ent": "Telsiai", "cls": "Northern Africa"}, {"id": 20321, "dom": "G", "ent": "Telsiai", "cls": "Northern America"}, {"id": 20322, "dom": "G", "ent": "Telsiai", "cls": "Northern Europe"}, {"id": 20323, "dom": "G", "ent": "Telsiai", "cls": "Polynesia"}, {"id": 20324, "dom": "G", "ent": "Telsiai", "cls": "South America"}, {"id": 20325, "dom": "G", "ent": "Telsiai", "cls": "South-eastern Asia"}, {"id": 20326, "dom": "G", "ent": "Telsiai", "cls": "Southern Africa"}, {"id": 20327, "dom": "G", "ent": "Telsiai", "cls": "Southern Asia"}, {"id": 20328, "dom": "G", "ent": "Telsiai", "cls": "Southern Europe"}, {"id": 20329, "dom": "G", "ent": "Telsiai", "cls": "Western Africa"}, {"id": 20330, "dom": "G", "ent": "Telsiai", "cls": "Western Asia"}, {"id": 20331, "dom": "G", "ent": "Telsiai", "cls": "Western Europe"}, {"id": 20332, "dom": "G", "ent": "Daugavpils", "cls": "Australia and New Zealand"}, {"id": 20333, "dom": "G", "ent": "Daugavpils", "cls": "Caribbean"}, {"id": 20334, "dom": "G", "ent": "Daugavpils", "cls": "Central America"}, {"id": 20335, "dom": "G", "ent": "Daugavpils", "cls": "Central Asia"}, {"id": 20336, "dom": "G", "ent": "Daugavpils", "cls": "Eastern Africa"}, {"id": 20337, "dom": "G", "ent": "Daugavpils", "cls": "Eastern Asia"}, {"id": 20338, "dom": "G", "ent": "Daugavpils", "cls": "Eastern Europe"}, {"id": 20339, "dom": "G", "ent": "Daugavpils", "cls": "Melanesia"}, {"id": 20340, "dom": "G", "ent": "Daugavpils", "cls": "Micronesia"}, {"id": 20341, "dom": "G", "ent": "Daugavpils", "cls": "Middle Africa"}, {"id": 20342, "dom": "G", "ent": "Daugavpils", "cls": "Northern Africa"}, {"id": 20343, "dom": "G", "ent": "Daugavpils", "cls": "Northern America"}, {"id": 20344, "dom": "G", "ent": "Daugavpils", "cls": "Northern Europe"}, {"id": 20345, "dom": "G", "ent": "Daugavpils", "cls": "Polynesia"}, {"id": 20346, "dom": "G", "ent": "Daugavpils", "cls": "South America"}, {"id": 20347, "dom": "G", "ent": "Daugavpils", "cls": "South-eastern Asia"}, {"id": 20348, "dom": "G", "ent": "Daugavpils", "cls": "Southern Africa"}, {"id": 20349, "dom": "G", "ent": "Daugavpils", "cls": "Southern Asia"}, {"id": 20350, "dom": "G", "ent": "Daugavpils", "cls": "Southern Europe"}, {"id": 20351, "dom": "G", "ent": "Daugavpils", "cls": "Western Africa"}, {"id": 20352, "dom": "G", "ent": "Daugavpils", "cls": "Western Asia"}, {"id": 20353, "dom": "G", "ent": "Daugavpils", "cls": "Western Europe"}, {"id": 20354, "dom": "G", "ent": "Ventspils", "cls": "Australia and New Zealand"}, {"id": 20355, "dom": "G", "ent": "Ventspils", "cls": "Caribbean"}, {"id": 20356, "dom": "G", "ent": "Ventspils", "cls": "Central America"}, {"id": 20357, "dom": "G", "ent": "Ventspils", "cls": "Central Asia"}, {"id": 20358, "dom": "G", "ent": "Ventspils", "cls": "Eastern Africa"}, {"id": 20359, "dom": "G", "ent": "Ventspils", "cls": "Eastern Asia"}, {"id": 20360, "dom": "G", "ent": "Ventspils", "cls": "Eastern Europe"}, {"id": 20361, "dom": "G", "ent": "Ventspils", "cls": "Melanesia"}, {"id": 20362, "dom": "G", "ent": "Ventspils", "cls": "Micronesia"}, {"id": 20363, "dom": "G", "ent": "Ventspils", "cls": "Middle Africa"}, {"id": 20364, "dom": "G", "ent": "Ventspils", "cls": "Northern Africa"}, {"id": 20365, "dom": "G", "ent": "Ventspils", "cls": "Northern America"}, {"id": 20366, "dom": "G", "ent": "Ventspils", "cls": "Northern Europe"}, {"id": 20367, "dom": "G", "ent": "Ventspils", "cls": "Polynesia"}, {"id": 20368, "dom": "G", "ent": "Ventspils", "cls": "South America"}, {"id": 20369, "dom": "G", "ent": "Ventspils", "cls": "South-eastern Asia"}, {"id": 20370, "dom": "G", "ent": "Ventspils", "cls": "Southern Africa"}, {"id": 20371, "dom": "G", "ent": "Ventspils", "cls": "Southern Asia"}, {"id": 20372, "dom": "G", "ent": "Ventspils", "cls": "Southern Europe"}, {"id": 20373, "dom": "G", "ent": "Ventspils", "cls": "Western Africa"}, {"id": 20374, "dom": "G", "ent": "Ventspils", "cls": "Western Asia"}, {"id": 20375, "dom": "G", "ent": "Ventspils", "cls": "Western Europe"}, {"id": 20376, "dom": "G", "ent": "Valmiera", "cls": "Australia and New Zealand"}, {"id": 20377, "dom": "G", "ent": "Valmiera", "cls": "Caribbean"}, {"id": 20378, "dom": "G", "ent": "Valmiera", "cls": "Central America"}, {"id": 20379, "dom": "G", "ent": "Valmiera", "cls": "Central Asia"}, {"id": 20380, "dom": "G", "ent": "Valmiera", "cls": "Eastern Africa"}, {"id": 20381, "dom": "G", "ent": "Valmiera", "cls": "Eastern Asia"}, {"id": 20382, "dom": "G", "ent": "Valmiera", "cls": "Eastern Europe"}, {"id": 20383, "dom": "G", "ent": "Valmiera", "cls": "Melanesia"}, {"id": 20384, "dom": "G", "ent": "Valmiera", "cls": "Micronesia"}, {"id": 20385, "dom": "G", "ent": "Valmiera", "cls": "Middle Africa"}, {"id": 20386, "dom": "G", "ent": "Valmiera", "cls": "Northern Africa"}, {"id": 20387, "dom": "G", "ent": "Valmiera", "cls": "Northern America"}, {"id": 20388, "dom": "G", "ent": "Valmiera", "cls": "Northern Europe"}, {"id": 20389, "dom": "G", "ent": "Valmiera", "cls": "Polynesia"}, {"id": 20390, "dom": "G", "ent": "Valmiera", "cls": "South America"}, {"id": 20391, "dom": "G", "ent": "Valmiera", "cls": "South-eastern Asia"}, {"id": 20392, "dom": "G", "ent": "Valmiera", "cls": "Southern Africa"}, {"id": 20393, "dom": "G", "ent": "Valmiera", "cls": "Southern Asia"}, {"id": 20394, "dom": "G", "ent": "Valmiera", "cls": "Southern Europe"}, {"id": 20395, "dom": "G", "ent": "Valmiera", "cls": "Western Africa"}, {"id": 20396, "dom": "G", "ent": "Valmiera", "cls": "Western Asia"}, {"id": 20397, "dom": "G", "ent": "Valmiera", "cls": "Western Europe"}, {"id": 20398, "dom": "G", "ent": "Tukums", "cls": "Australia and New Zealand"}, {"id": 20399, "dom": "G", "ent": "Tukums", "cls": "Caribbean"}, {"id": 20400, "dom": "G", "ent": "Tukums", "cls": "Central America"}, {"id": 20401, "dom": "G", "ent": "Tukums", "cls": "Central Asia"}, {"id": 20402, "dom": "G", "ent": "Tukums", "cls": "Eastern Africa"}, {"id": 20403, "dom": "G", "ent": "Tukums", "cls": "Eastern Asia"}, {"id": 20404, "dom": "G", "ent": "Tukums", "cls": "Eastern Europe"}, {"id": 20405, "dom": "G", "ent": "Tukums", "cls": "Melanesia"}, {"id": 20406, "dom": "G", "ent": "Tukums", "cls": "Micronesia"}, {"id": 20407, "dom": "G", "ent": "Tukums", "cls": "Middle Africa"}, {"id": 20408, "dom": "G", "ent": "Tukums", "cls": "Northern Africa"}, {"id": 20409, "dom": "G", "ent": "Tukums", "cls": "Northern America"}, {"id": 20410, "dom": "G", "ent": "Tukums", "cls": "Northern Europe"}, {"id": 20411, "dom": "G", "ent": "Tukums", "cls": "Polynesia"}, {"id": 20412, "dom": "G", "ent": "Tukums", "cls": "South America"}, {"id": 20413, "dom": "G", "ent": "Tukums", "cls": "South-eastern Asia"}, {"id": 20414, "dom": "G", "ent": "Tukums", "cls": "Southern Africa"}, {"id": 20415, "dom": "G", "ent": "Tukums", "cls": "Southern Asia"}, {"id": 20416, "dom": "G", "ent": "Tukums", "cls": "Southern Europe"}, {"id": 20417, "dom": "G", "ent": "Tukums", "cls": "Western Africa"}, {"id": 20418, "dom": "G", "ent": "Tukums", "cls": "Western Asia"}, {"id": 20419, "dom": "G", "ent": "Tukums", "cls": "Western Europe"}, {"id": 20420, "dom": "G", "ent": "Sirte", "cls": "Australia and New Zealand"}, {"id": 20421, "dom": "G", "ent": "Sirte", "cls": "Caribbean"}, {"id": 20422, "dom": "G", "ent": "Sirte", "cls": "Central America"}, {"id": 20423, "dom": "G", "ent": "Sirte", "cls": "Central Asia"}, {"id": 20424, "dom": "G", "ent": "Sirte", "cls": "Eastern Africa"}, {"id": 20425, "dom": "G", "ent": "Sirte", "cls": "Eastern Asia"}, {"id": 20426, "dom": "G", "ent": "Sirte", "cls": "Eastern Europe"}, {"id": 20427, "dom": "G", "ent": "Sirte", "cls": "Melanesia"}, {"id": 20428, "dom": "G", "ent": "Sirte", "cls": "Micronesia"}, {"id": 20429, "dom": "G", "ent": "Sirte", "cls": "Middle Africa"}, {"id": 20430, "dom": "G", "ent": "Sirte", "cls": "Northern Africa"}, {"id": 20431, "dom": "G", "ent": "Sirte", "cls": "Northern America"}, {"id": 20432, "dom": "G", "ent": "Sirte", "cls": "Northern Europe"}, {"id": 20433, "dom": "G", "ent": "Sirte", "cls": "Polynesia"}, {"id": 20434, "dom": "G", "ent": "Sirte", "cls": "South America"}, {"id": 20435, "dom": "G", "ent": "Sirte", "cls": "South-eastern Asia"}, {"id": 20436, "dom": "G", "ent": "Sirte", "cls": "Southern Africa"}, {"id": 20437, "dom": "G", "ent": "Sirte", "cls": "Southern Asia"}, {"id": 20438, "dom": "G", "ent": "Sirte", "cls": "Southern Europe"}, {"id": 20439, "dom": "G", "ent": "Sirte", "cls": "Western Africa"}, {"id": 20440, "dom": "G", "ent": "Sirte", "cls": "Western Asia"}, {"id": 20441, "dom": "G", "ent": "Sirte", "cls": "Western Europe"}, {"id": 20442, "dom": "G", "ent": "Zuwarah", "cls": "Australia and New Zealand"}, {"id": 20443, "dom": "G", "ent": "Zuwarah", "cls": "Caribbean"}, {"id": 20444, "dom": "G", "ent": "Zuwarah", "cls": "Central America"}, {"id": 20445, "dom": "G", "ent": "Zuwarah", "cls": "Central Asia"}, {"id": 20446, "dom": "G", "ent": "Zuwarah", "cls": "Eastern Africa"}, {"id": 20447, "dom": "G", "ent": "Zuwarah", "cls": "Eastern Asia"}, {"id": 20448, "dom": "G", "ent": "Zuwarah", "cls": "Eastern Europe"}, {"id": 20449, "dom": "G", "ent": "Zuwarah", "cls": "Melanesia"}, {"id": 20450, "dom": "G", "ent": "Zuwarah", "cls": "Micronesia"}, {"id": 20451, "dom": "G", "ent": "Zuwarah", "cls": "Middle Africa"}, {"id": 20452, "dom": "G", "ent": "Zuwarah", "cls": "Northern Africa"}, {"id": 20453, "dom": "G", "ent": "Zuwarah", "cls": "Northern America"}, {"id": 20454, "dom": "G", "ent": "Zuwarah", "cls": "Northern Europe"}, {"id": 20455, "dom": "G", "ent": "Zuwarah", "cls": "Polynesia"}, {"id": 20456, "dom": "G", "ent": "Zuwarah", "cls": "South America"}, {"id": 20457, "dom": "G", "ent": "Zuwarah", "cls": "South-eastern Asia"}, {"id": 20458, "dom": "G", "ent": "Zuwarah", "cls": "Southern Africa"}, {"id": 20459, "dom": "G", "ent": "Zuwarah", "cls": "Southern Asia"}, {"id": 20460, "dom": "G", "ent": "Zuwarah", "cls": "Southern Europe"}, {"id": 20461, "dom": "G", "ent": "Zuwarah", "cls": "Western Africa"}, {"id": 20462, "dom": "G", "ent": "Zuwarah", "cls": "Western Asia"}, {"id": 20463, "dom": "G", "ent": "Zuwarah", "cls": "Western Europe"}, {"id": 20464, "dom": "G", "ent": "Ubari", "cls": "Australia and New Zealand"}, {"id": 20465, "dom": "G", "ent": "Ubari", "cls": "Caribbean"}, {"id": 20466, "dom": "G", "ent": "Ubari", "cls": "Central America"}, {"id": 20467, "dom": "G", "ent": "Ubari", "cls": "Central Asia"}, {"id": 20468, "dom": "G", "ent": "Ubari", "cls": "Eastern Africa"}, {"id": 20469, "dom": "G", "ent": "Ubari", "cls": "Eastern Asia"}, {"id": 20470, "dom": "G", "ent": "Ubari", "cls": "Eastern Europe"}, {"id": 20471, "dom": "G", "ent": "Ubari", "cls": "Melanesia"}, {"id": 20472, "dom": "G", "ent": "Ubari", "cls": "Micronesia"}, {"id": 20473, "dom": "G", "ent": "Ubari", "cls": "Middle Africa"}, {"id": 20474, "dom": "G", "ent": "Ubari", "cls": "Northern Africa"}, {"id": 20475, "dom": "G", "ent": "Ubari", "cls": "Northern America"}, {"id": 20476, "dom": "G", "ent": "Ubari", "cls": "Northern Europe"}, {"id": 20477, "dom": "G", "ent": "Ubari", "cls": "Polynesia"}, {"id": 20478, "dom": "G", "ent": "Ubari", "cls": "South America"}, {"id": 20479, "dom": "G", "ent": "Ubari", "cls": "South-eastern Asia"}, {"id": 20480, "dom": "G", "ent": "Ubari", "cls": "Southern Africa"}, {"id": 20481, "dom": "G", "ent": "Ubari", "cls": "Southern Asia"}, {"id": 20482, "dom": "G", "ent": "Ubari", "cls": "Southern Europe"}, {"id": 20483, "dom": "G", "ent": "Ubari", "cls": "Western Africa"}, {"id": 20484, "dom": "G", "ent": "Ubari", "cls": "Western Asia"}, {"id": 20485, "dom": "G", "ent": "Ubari", "cls": "Western Europe"}, {"id": 20486, "dom": "G", "ent": "Brak", "cls": "Australia and New Zealand"}, {"id": 20487, "dom": "G", "ent": "Brak", "cls": "Caribbean"}, {"id": 20488, "dom": "G", "ent": "Brak", "cls": "Central America"}, {"id": 20489, "dom": "G", "ent": "Brak", "cls": "Central Asia"}, {"id": 20490, "dom": "G", "ent": "Brak", "cls": "Eastern Africa"}, {"id": 20491, "dom": "G", "ent": "Brak", "cls": "Eastern Asia"}, {"id": 20492, "dom": "G", "ent": "Brak", "cls": "Eastern Europe"}, {"id": 20493, "dom": "G", "ent": "Brak", "cls": "Melanesia"}, {"id": 20494, "dom": "G", "ent": "Brak", "cls": "Micronesia"}, {"id": 20495, "dom": "G", "ent": "Brak", "cls": "Middle Africa"}, {"id": 20496, "dom": "G", "ent": "Brak", "cls": "Northern Africa"}, {"id": 20497, "dom": "G", "ent": "Brak", "cls": "Northern America"}, {"id": 20498, "dom": "G", "ent": "Brak", "cls": "Northern Europe"}, {"id": 20499, "dom": "G", "ent": "Brak", "cls": "Polynesia"}, {"id": 20500, "dom": "G", "ent": "Brak", "cls": "South America"}, {"id": 20501, "dom": "G", "ent": "Brak", "cls": "South-eastern Asia"}, {"id": 20502, "dom": "G", "ent": "Brak", "cls": "Southern Africa"}, {"id": 20503, "dom": "G", "ent": "Brak", "cls": "Southern Asia"}, {"id": 20504, "dom": "G", "ent": "Brak", "cls": "Southern Europe"}, {"id": 20505, "dom": "G", "ent": "Brak", "cls": "Western Africa"}, {"id": 20506, "dom": "G", "ent": "Brak", "cls": "Western Asia"}, {"id": 20507, "dom": "G", "ent": "Brak", "cls": "Western Europe"}, {"id": 20508, "dom": "G", "ent": "Skhirate", "cls": "Australia and New Zealand"}, {"id": 20509, "dom": "G", "ent": "Skhirate", "cls": "Caribbean"}, {"id": 20510, "dom": "G", "ent": "Skhirate", "cls": "Central America"}, {"id": 20511, "dom": "G", "ent": "Skhirate", "cls": "Central Asia"}, {"id": 20512, "dom": "G", "ent": "Skhirate", "cls": "Eastern Africa"}, {"id": 20513, "dom": "G", "ent": "Skhirate", "cls": "Eastern Asia"}, {"id": 20514, "dom": "G", "ent": "Skhirate", "cls": "Eastern Europe"}, {"id": 20515, "dom": "G", "ent": "Skhirate", "cls": "Melanesia"}, {"id": 20516, "dom": "G", "ent": "Skhirate", "cls": "Micronesia"}, {"id": 20517, "dom": "G", "ent": "Skhirate", "cls": "Middle Africa"}, {"id": 20518, "dom": "G", "ent": "Skhirate", "cls": "Northern Africa"}, {"id": 20519, "dom": "G", "ent": "Skhirate", "cls": "Northern America"}, {"id": 20520, "dom": "G", "ent": "Skhirate", "cls": "Northern Europe"}, {"id": 20521, "dom": "G", "ent": "Skhirate", "cls": "Polynesia"}, {"id": 20522, "dom": "G", "ent": "Skhirate", "cls": "South America"}, {"id": 20523, "dom": "G", "ent": "Skhirate", "cls": "South-eastern Asia"}, {"id": 20524, "dom": "G", "ent": "Skhirate", "cls": "Southern Africa"}, {"id": 20525, "dom": "G", "ent": "Skhirate", "cls": "Southern Asia"}, {"id": 20526, "dom": "G", "ent": "Skhirate", "cls": "Southern Europe"}, {"id": 20527, "dom": "G", "ent": "Skhirate", "cls": "Western Africa"}, {"id": 20528, "dom": "G", "ent": "Skhirate", "cls": "Western Asia"}, {"id": 20529, "dom": "G", "ent": "Skhirate", "cls": "Western Europe"}, {"id": 20530, "dom": "G", "ent": "Tinghir", "cls": "Australia and New Zealand"}, {"id": 20531, "dom": "G", "ent": "Tinghir", "cls": "Caribbean"}, {"id": 20532, "dom": "G", "ent": "Tinghir", "cls": "Central America"}, {"id": 20533, "dom": "G", "ent": "Tinghir", "cls": "Central Asia"}, {"id": 20534, "dom": "G", "ent": "Tinghir", "cls": "Eastern Africa"}, {"id": 20535, "dom": "G", "ent": "Tinghir", "cls": "Eastern Asia"}, {"id": 20536, "dom": "G", "ent": "Tinghir", "cls": "Eastern Europe"}, {"id": 20537, "dom": "G", "ent": "Tinghir", "cls": "Melanesia"}, {"id": 20538, "dom": "G", "ent": "Tinghir", "cls": "Micronesia"}, {"id": 20539, "dom": "G", "ent": "Tinghir", "cls": "Middle Africa"}, {"id": 20540, "dom": "G", "ent": "Tinghir", "cls": "Northern Africa"}, {"id": 20541, "dom": "G", "ent": "Tinghir", "cls": "Northern America"}, {"id": 20542, "dom": "G", "ent": "Tinghir", "cls": "Northern Europe"}, {"id": 20543, "dom": "G", "ent": "Tinghir", "cls": "Polynesia"}, {"id": 20544, "dom": "G", "ent": "Tinghir", "cls": "South America"}, {"id": 20545, "dom": "G", "ent": "Tinghir", "cls": "South-eastern Asia"}, {"id": 20546, "dom": "G", "ent": "Tinghir", "cls": "Southern Africa"}, {"id": 20547, "dom": "G", "ent": "Tinghir", "cls": "Southern Asia"}, {"id": 20548, "dom": "G", "ent": "Tinghir", "cls": "Southern Europe"}, {"id": 20549, "dom": "G", "ent": "Tinghir", "cls": "Western Africa"}, {"id": 20550, "dom": "G", "ent": "Tinghir", "cls": "Western Asia"}, {"id": 20551, "dom": "G", "ent": "Tinghir", "cls": "Western Europe"}, {"id": 20552, "dom": "G", "ent": "Ait Ourir", "cls": "Australia and New Zealand"}, {"id": 20553, "dom": "G", "ent": "Ait Ourir", "cls": "Caribbean"}, {"id": 20554, "dom": "G", "ent": "Ait Ourir", "cls": "Central America"}, {"id": 20555, "dom": "G", "ent": "Ait Ourir", "cls": "Central Asia"}, {"id": 20556, "dom": "G", "ent": "Ait Ourir", "cls": "Eastern Africa"}, {"id": 20557, "dom": "G", "ent": "Ait Ourir", "cls": "Eastern Asia"}, {"id": 20558, "dom": "G", "ent": "Ait Ourir", "cls": "Eastern Europe"}, {"id": 20559, "dom": "G", "ent": "Ait Ourir", "cls": "Melanesia"}, {"id": 20560, "dom": "G", "ent": "Ait Ourir", "cls": "Micronesia"}, {"id": 20561, "dom": "G", "ent": "Ait Ourir", "cls": "Middle Africa"}, {"id": 20562, "dom": "G", "ent": "Ait Ourir", "cls": "Northern Africa"}, {"id": 20563, "dom": "G", "ent": "Ait Ourir", "cls": "Northern America"}, {"id": 20564, "dom": "G", "ent": "Ait Ourir", "cls": "Northern Europe"}, {"id": 20565, "dom": "G", "ent": "Ait Ourir", "cls": "Polynesia"}, {"id": 20566, "dom": "G", "ent": "Ait Ourir", "cls": "South America"}, {"id": 20567, "dom": "G", "ent": "Ait Ourir", "cls": "South-eastern Asia"}, {"id": 20568, "dom": "G", "ent": "Ait Ourir", "cls": "Southern Africa"}, {"id": 20569, "dom": "G", "ent": "Ait Ourir", "cls": "Southern Asia"}, {"id": 20570, "dom": "G", "ent": "Ait Ourir", "cls": "Southern Europe"}, {"id": 20571, "dom": "G", "ent": "Ait Ourir", "cls": "Western Africa"}, {"id": 20572, "dom": "G", "ent": "Ait Ourir", "cls": "Western Asia"}, {"id": 20573, "dom": "G", "ent": "Ait Ourir", "cls": "Western Europe"}, {"id": 20574, "dom": "G", "ent": "Kelaat Mgouna", "cls": "Australia and New Zealand"}, {"id": 20575, "dom": "G", "ent": "Kelaat Mgouna", "cls": "Caribbean"}, {"id": 20576, "dom": "G", "ent": "Kelaat Mgouna", "cls": "Central America"}, {"id": 20577, "dom": "G", "ent": "Kelaat Mgouna", "cls": "Central Asia"}, {"id": 20578, "dom": "G", "ent": "Kelaat Mgouna", "cls": "Eastern Africa"}, {"id": 20579, "dom": "G", "ent": "Kelaat Mgouna", "cls": "Eastern Asia"}, {"id": 20580, "dom": "G", "ent": "Kelaat Mgouna", "cls": "Eastern Europe"}, {"id": 20581, "dom": "G", "ent": "Kelaat Mgouna", "cls": "Melanesia"}, {"id": 20582, "dom": "G", "ent": "Kelaat Mgouna", "cls": "Micronesia"}, {"id": 20583, "dom": "G", "ent": "Kelaat Mgouna", "cls": "Middle Africa"}, {"id": 20584, "dom": "G", "ent": "Kelaat Mgouna", "cls": "Northern Africa"}, {"id": 20585, "dom": "G", "ent": "Kelaat Mgouna", "cls": "Northern America"}, {"id": 20586, "dom": "G", "ent": "Kelaat Mgouna", "cls": "Northern Europe"}, {"id": 20587, "dom": "G", "ent": "Kelaat Mgouna", "cls": "Polynesia"}, {"id": 20588, "dom": "G", "ent": "Kelaat Mgouna", "cls": "South America"}, {"id": 20589, "dom": "G", "ent": "Kelaat Mgouna", "cls": "South-eastern Asia"}, {"id": 20590, "dom": "G", "ent": "Kelaat Mgouna", "cls": "Southern Africa"}, {"id": 20591, "dom": "G", "ent": "Kelaat Mgouna", "cls": "Southern Asia"}, {"id": 20592, "dom": "G", "ent": "Kelaat Mgouna", "cls": "Southern Europe"}, {"id": 20593, "dom": "G", "ent": "Kelaat Mgouna", "cls": "Western Africa"}, {"id": 20594, "dom": "G", "ent": "Kelaat Mgouna", "cls": "Western Asia"}, {"id": 20595, "dom": "G", "ent": "Kelaat Mgouna", "cls": "Western Europe"}, {"id": 20596, "dom": "G", "ent": "Tiraspol", "cls": "Australia and New Zealand"}, {"id": 20597, "dom": "G", "ent": "Tiraspol", "cls": "Caribbean"}, {"id": 20598, "dom": "G", "ent": "Tiraspol", "cls": "Central America"}, {"id": 20599, "dom": "G", "ent": "Tiraspol", "cls": "Central Asia"}, {"id": 20600, "dom": "G", "ent": "Tiraspol", "cls": "Eastern Africa"}, {"id": 20601, "dom": "G", "ent": "Tiraspol", "cls": "Eastern Asia"}, {"id": 20602, "dom": "G", "ent": "Tiraspol", "cls": "Eastern Europe"}, {"id": 20603, "dom": "G", "ent": "Tiraspol", "cls": "Melanesia"}, {"id": 20604, "dom": "G", "ent": "Tiraspol", "cls": "Micronesia"}, {"id": 20605, "dom": "G", "ent": "Tiraspol", "cls": "Middle Africa"}, {"id": 20606, "dom": "G", "ent": "Tiraspol", "cls": "Northern Africa"}, {"id": 20607, "dom": "G", "ent": "Tiraspol", "cls": "Northern America"}, {"id": 20608, "dom": "G", "ent": "Tiraspol", "cls": "Northern Europe"}, {"id": 20609, "dom": "G", "ent": "Tiraspol", "cls": "Polynesia"}, {"id": 20610, "dom": "G", "ent": "Tiraspol", "cls": "South America"}, {"id": 20611, "dom": "G", "ent": "Tiraspol", "cls": "South-eastern Asia"}, {"id": 20612, "dom": "G", "ent": "Tiraspol", "cls": "Southern Africa"}, {"id": 20613, "dom": "G", "ent": "Tiraspol", "cls": "Southern Asia"}, {"id": 20614, "dom": "G", "ent": "Tiraspol", "cls": "Southern Europe"}, {"id": 20615, "dom": "G", "ent": "Tiraspol", "cls": "Western Africa"}, {"id": 20616, "dom": "G", "ent": "Tiraspol", "cls": "Western Asia"}, {"id": 20617, "dom": "G", "ent": "Tiraspol", "cls": "Western Europe"}, {"id": 20618, "dom": "G", "ent": "Comrat", "cls": "Australia and New Zealand"}, {"id": 20619, "dom": "G", "ent": "Comrat", "cls": "Caribbean"}, {"id": 20620, "dom": "G", "ent": "Comrat", "cls": "Central America"}, {"id": 20621, "dom": "G", "ent": "Comrat", "cls": "Central Asia"}, {"id": 20622, "dom": "G", "ent": "Comrat", "cls": "Eastern Africa"}, {"id": 20623, "dom": "G", "ent": "Comrat", "cls": "Eastern Asia"}, {"id": 20624, "dom": "G", "ent": "Comrat", "cls": "Eastern Europe"}, {"id": 20625, "dom": "G", "ent": "Comrat", "cls": "Melanesia"}, {"id": 20626, "dom": "G", "ent": "Comrat", "cls": "Micronesia"}, {"id": 20627, "dom": "G", "ent": "Comrat", "cls": "Middle Africa"}, {"id": 20628, "dom": "G", "ent": "Comrat", "cls": "Northern Africa"}, {"id": 20629, "dom": "G", "ent": "Comrat", "cls": "Northern America"}, {"id": 20630, "dom": "G", "ent": "Comrat", "cls": "Northern Europe"}, {"id": 20631, "dom": "G", "ent": "Comrat", "cls": "Polynesia"}, {"id": 20632, "dom": "G", "ent": "Comrat", "cls": "South America"}, {"id": 20633, "dom": "G", "ent": "Comrat", "cls": "South-eastern Asia"}, {"id": 20634, "dom": "G", "ent": "Comrat", "cls": "Southern Africa"}, {"id": 20635, "dom": "G", "ent": "Comrat", "cls": "Southern Asia"}, {"id": 20636, "dom": "G", "ent": "Comrat", "cls": "Southern Europe"}, {"id": 20637, "dom": "G", "ent": "Comrat", "cls": "Western Africa"}, {"id": 20638, "dom": "G", "ent": "Comrat", "cls": "Western Asia"}, {"id": 20639, "dom": "G", "ent": "Comrat", "cls": "Western Europe"}, {"id": 20640, "dom": "G", "ent": "Drochia", "cls": "Australia and New Zealand"}, {"id": 20641, "dom": "G", "ent": "Drochia", "cls": "Caribbean"}, {"id": 20642, "dom": "G", "ent": "Drochia", "cls": "Central America"}, {"id": 20643, "dom": "G", "ent": "Drochia", "cls": "Central Asia"}, {"id": 20644, "dom": "G", "ent": "Drochia", "cls": "Eastern Africa"}, {"id": 20645, "dom": "G", "ent": "Drochia", "cls": "Eastern Asia"}, {"id": 20646, "dom": "G", "ent": "Drochia", "cls": "Eastern Europe"}, {"id": 20647, "dom": "G", "ent": "Drochia", "cls": "Melanesia"}, {"id": 20648, "dom": "G", "ent": "Drochia", "cls": "Micronesia"}, {"id": 20649, "dom": "G", "ent": "Drochia", "cls": "Middle Africa"}, {"id": 20650, "dom": "G", "ent": "Drochia", "cls": "Northern Africa"}, {"id": 20651, "dom": "G", "ent": "Drochia", "cls": "Northern America"}, {"id": 20652, "dom": "G", "ent": "Drochia", "cls": "Northern Europe"}, {"id": 20653, "dom": "G", "ent": "Drochia", "cls": "Polynesia"}, {"id": 20654, "dom": "G", "ent": "Drochia", "cls": "South America"}, {"id": 20655, "dom": "G", "ent": "Drochia", "cls": "South-eastern Asia"}, {"id": 20656, "dom": "G", "ent": "Drochia", "cls": "Southern Africa"}, {"id": 20657, "dom": "G", "ent": "Drochia", "cls": "Southern Asia"}, {"id": 20658, "dom": "G", "ent": "Drochia", "cls": "Southern Europe"}, {"id": 20659, "dom": "G", "ent": "Drochia", "cls": "Western Africa"}, {"id": 20660, "dom": "G", "ent": "Drochia", "cls": "Western Asia"}, {"id": 20661, "dom": "G", "ent": "Drochia", "cls": "Western Europe"}, {"id": 20662, "dom": "G", "ent": "Codru", "cls": "Australia and New Zealand"}, {"id": 20663, "dom": "G", "ent": "Codru", "cls": "Caribbean"}, {"id": 20664, "dom": "G", "ent": "Codru", "cls": "Central America"}, {"id": 20665, "dom": "G", "ent": "Codru", "cls": "Central Asia"}, {"id": 20666, "dom": "G", "ent": "Codru", "cls": "Eastern Africa"}, {"id": 20667, "dom": "G", "ent": "Codru", "cls": "Eastern Asia"}, {"id": 20668, "dom": "G", "ent": "Codru", "cls": "Eastern Europe"}, {"id": 20669, "dom": "G", "ent": "Codru", "cls": "Melanesia"}, {"id": 20670, "dom": "G", "ent": "Codru", "cls": "Micronesia"}, {"id": 20671, "dom": "G", "ent": "Codru", "cls": "Middle Africa"}, {"id": 20672, "dom": "G", "ent": "Codru", "cls": "Northern Africa"}, {"id": 20673, "dom": "G", "ent": "Codru", "cls": "Northern America"}, {"id": 20674, "dom": "G", "ent": "Codru", "cls": "Northern Europe"}, {"id": 20675, "dom": "G", "ent": "Codru", "cls": "Polynesia"}, {"id": 20676, "dom": "G", "ent": "Codru", "cls": "South America"}, {"id": 20677, "dom": "G", "ent": "Codru", "cls": "South-eastern Asia"}, {"id": 20678, "dom": "G", "ent": "Codru", "cls": "Southern Africa"}, {"id": 20679, "dom": "G", "ent": "Codru", "cls": "Southern Asia"}, {"id": 20680, "dom": "G", "ent": "Codru", "cls": "Southern Europe"}, {"id": 20681, "dom": "G", "ent": "Codru", "cls": "Western Africa"}, {"id": 20682, "dom": "G", "ent": "Codru", "cls": "Western Asia"}, {"id": 20683, "dom": "G", "ent": "Codru", "cls": "Western Europe"}, {"id": 20684, "dom": "G", "ent": "Ambanja", "cls": "Australia and New Zealand"}, {"id": 20685, "dom": "G", "ent": "Ambanja", "cls": "Caribbean"}, {"id": 20686, "dom": "G", "ent": "Ambanja", "cls": "Central America"}, {"id": 20687, "dom": "G", "ent": "Ambanja", "cls": "Central Asia"}, {"id": 20688, "dom": "G", "ent": "Ambanja", "cls": "Eastern Africa"}, {"id": 20689, "dom": "G", "ent": "Ambanja", "cls": "Eastern Asia"}, {"id": 20690, "dom": "G", "ent": "Ambanja", "cls": "Eastern Europe"}, {"id": 20691, "dom": "G", "ent": "Ambanja", "cls": "Melanesia"}, {"id": 20692, "dom": "G", "ent": "Ambanja", "cls": "Micronesia"}, {"id": 20693, "dom": "G", "ent": "Ambanja", "cls": "Middle Africa"}, {"id": 20694, "dom": "G", "ent": "Ambanja", "cls": "Northern Africa"}, {"id": 20695, "dom": "G", "ent": "Ambanja", "cls": "Northern America"}, {"id": 20696, "dom": "G", "ent": "Ambanja", "cls": "Northern Europe"}, {"id": 20697, "dom": "G", "ent": "Ambanja", "cls": "Polynesia"}, {"id": 20698, "dom": "G", "ent": "Ambanja", "cls": "South America"}, {"id": 20699, "dom": "G", "ent": "Ambanja", "cls": "South-eastern Asia"}, {"id": 20700, "dom": "G", "ent": "Ambanja", "cls": "Southern Africa"}, {"id": 20701, "dom": "G", "ent": "Ambanja", "cls": "Southern Asia"}, {"id": 20702, "dom": "G", "ent": "Ambanja", "cls": "Southern Europe"}, {"id": 20703, "dom": "G", "ent": "Ambanja", "cls": "Western Africa"}, {"id": 20704, "dom": "G", "ent": "Ambanja", "cls": "Western Asia"}, {"id": 20705, "dom": "G", "ent": "Ambanja", "cls": "Western Europe"}, {"id": 20706, "dom": "G", "ent": "Hell-Ville", "cls": "Australia and New Zealand"}, {"id": 20707, "dom": "G", "ent": "Hell-Ville", "cls": "Caribbean"}, {"id": 20708, "dom": "G", "ent": "Hell-Ville", "cls": "Central America"}, {"id": 20709, "dom": "G", "ent": "Hell-Ville", "cls": "Central Asia"}, {"id": 20710, "dom": "G", "ent": "Hell-Ville", "cls": "Eastern Africa"}, {"id": 20711, "dom": "G", "ent": "Hell-Ville", "cls": "Eastern Asia"}, {"id": 20712, "dom": "G", "ent": "Hell-Ville", "cls": "Eastern Europe"}, {"id": 20713, "dom": "G", "ent": "Hell-Ville", "cls": "Melanesia"}, {"id": 20714, "dom": "G", "ent": "Hell-Ville", "cls": "Micronesia"}, {"id": 20715, "dom": "G", "ent": "Hell-Ville", "cls": "Middle Africa"}, {"id": 20716, "dom": "G", "ent": "Hell-Ville", "cls": "Northern Africa"}, {"id": 20717, "dom": "G", "ent": "Hell-Ville", "cls": "Northern America"}, {"id": 20718, "dom": "G", "ent": "Hell-Ville", "cls": "Northern Europe"}, {"id": 20719, "dom": "G", "ent": "Hell-Ville", "cls": "Polynesia"}, {"id": 20720, "dom": "G", "ent": "Hell-Ville", "cls": "South America"}, {"id": 20721, "dom": "G", "ent": "Hell-Ville", "cls": "South-eastern Asia"}, {"id": 20722, "dom": "G", "ent": "Hell-Ville", "cls": "Southern Africa"}, {"id": 20723, "dom": "G", "ent": "Hell-Ville", "cls": "Southern Asia"}, {"id": 20724, "dom": "G", "ent": "Hell-Ville", "cls": "Southern Europe"}, {"id": 20725, "dom": "G", "ent": "Hell-Ville", "cls": "Western Africa"}, {"id": 20726, "dom": "G", "ent": "Hell-Ville", "cls": "Western Asia"}, {"id": 20727, "dom": "G", "ent": "Hell-Ville", "cls": "Western Europe"}, {"id": 20728, "dom": "G", "ent": "Didy", "cls": "Australia and New Zealand"}, {"id": 20729, "dom": "G", "ent": "Didy", "cls": "Caribbean"}, {"id": 20730, "dom": "G", "ent": "Didy", "cls": "Central America"}, {"id": 20731, "dom": "G", "ent": "Didy", "cls": "Central Asia"}, {"id": 20732, "dom": "G", "ent": "Didy", "cls": "Eastern Africa"}, {"id": 20733, "dom": "G", "ent": "Didy", "cls": "Eastern Asia"}, {"id": 20734, "dom": "G", "ent": "Didy", "cls": "Eastern Europe"}, {"id": 20735, "dom": "G", "ent": "Didy", "cls": "Melanesia"}, {"id": 20736, "dom": "G", "ent": "Didy", "cls": "Micronesia"}, {"id": 20737, "dom": "G", "ent": "Didy", "cls": "Middle Africa"}, {"id": 20738, "dom": "G", "ent": "Didy", "cls": "Northern Africa"}, {"id": 20739, "dom": "G", "ent": "Didy", "cls": "Northern America"}, {"id": 20740, "dom": "G", "ent": "Didy", "cls": "Northern Europe"}, {"id": 20741, "dom": "G", "ent": "Didy", "cls": "Polynesia"}, {"id": 20742, "dom": "G", "ent": "Didy", "cls": "South America"}, {"id": 20743, "dom": "G", "ent": "Didy", "cls": "South-eastern Asia"}, {"id": 20744, "dom": "G", "ent": "Didy", "cls": "Southern Africa"}, {"id": 20745, "dom": "G", "ent": "Didy", "cls": "Southern Asia"}, {"id": 20746, "dom": "G", "ent": "Didy", "cls": "Southern Europe"}, {"id": 20747, "dom": "G", "ent": "Didy", "cls": "Western Africa"}, {"id": 20748, "dom": "G", "ent": "Didy", "cls": "Western Asia"}, {"id": 20749, "dom": "G", "ent": "Didy", "cls": "Western Europe"}, {"id": 20750, "dom": "G", "ent": "Amboanjo", "cls": "Australia and New Zealand"}, {"id": 20751, "dom": "G", "ent": "Amboanjo", "cls": "Caribbean"}, {"id": 20752, "dom": "G", "ent": "Amboanjo", "cls": "Central America"}, {"id": 20753, "dom": "G", "ent": "Amboanjo", "cls": "Central Asia"}, {"id": 20754, "dom": "G", "ent": "Amboanjo", "cls": "Eastern Africa"}, {"id": 20755, "dom": "G", "ent": "Amboanjo", "cls": "Eastern Asia"}, {"id": 20756, "dom": "G", "ent": "Amboanjo", "cls": "Eastern Europe"}, {"id": 20757, "dom": "G", "ent": "Amboanjo", "cls": "Melanesia"}, {"id": 20758, "dom": "G", "ent": "Amboanjo", "cls": "Micronesia"}, {"id": 20759, "dom": "G", "ent": "Amboanjo", "cls": "Middle Africa"}, {"id": 20760, "dom": "G", "ent": "Amboanjo", "cls": "Northern Africa"}, {"id": 20761, "dom": "G", "ent": "Amboanjo", "cls": "Northern America"}, {"id": 20762, "dom": "G", "ent": "Amboanjo", "cls": "Northern Europe"}, {"id": 20763, "dom": "G", "ent": "Amboanjo", "cls": "Polynesia"}, {"id": 20764, "dom": "G", "ent": "Amboanjo", "cls": "South America"}, {"id": 20765, "dom": "G", "ent": "Amboanjo", "cls": "South-eastern Asia"}, {"id": 20766, "dom": "G", "ent": "Amboanjo", "cls": "Southern Africa"}, {"id": 20767, "dom": "G", "ent": "Amboanjo", "cls": "Southern Asia"}, {"id": 20768, "dom": "G", "ent": "Amboanjo", "cls": "Southern Europe"}, {"id": 20769, "dom": "G", "ent": "Amboanjo", "cls": "Western Africa"}, {"id": 20770, "dom": "G", "ent": "Amboanjo", "cls": "Western Asia"}, {"id": 20771, "dom": "G", "ent": "Amboanjo", "cls": "Western Europe"}, {"id": 20772, "dom": "G", "ent": "Tetovo", "cls": "Australia and New Zealand"}, {"id": 20773, "dom": "G", "ent": "Tetovo", "cls": "Caribbean"}, {"id": 20774, "dom": "G", "ent": "Tetovo", "cls": "Central America"}, {"id": 20775, "dom": "G", "ent": "Tetovo", "cls": "Central Asia"}, {"id": 20776, "dom": "G", "ent": "Tetovo", "cls": "Eastern Africa"}, {"id": 20777, "dom": "G", "ent": "Tetovo", "cls": "Eastern Asia"}, {"id": 20778, "dom": "G", "ent": "Tetovo", "cls": "Eastern Europe"}, {"id": 20779, "dom": "G", "ent": "Tetovo", "cls": "Melanesia"}, {"id": 20780, "dom": "G", "ent": "Tetovo", "cls": "Micronesia"}, {"id": 20781, "dom": "G", "ent": "Tetovo", "cls": "Middle Africa"}, {"id": 20782, "dom": "G", "ent": "Tetovo", "cls": "Northern Africa"}, {"id": 20783, "dom": "G", "ent": "Tetovo", "cls": "Northern America"}, {"id": 20784, "dom": "G", "ent": "Tetovo", "cls": "Northern Europe"}, {"id": 20785, "dom": "G", "ent": "Tetovo", "cls": "Polynesia"}, {"id": 20786, "dom": "G", "ent": "Tetovo", "cls": "South America"}, {"id": 20787, "dom": "G", "ent": "Tetovo", "cls": "South-eastern Asia"}, {"id": 20788, "dom": "G", "ent": "Tetovo", "cls": "Southern Africa"}, {"id": 20789, "dom": "G", "ent": "Tetovo", "cls": "Southern Asia"}, {"id": 20790, "dom": "G", "ent": "Tetovo", "cls": "Southern Europe"}, {"id": 20791, "dom": "G", "ent": "Tetovo", "cls": "Western Africa"}, {"id": 20792, "dom": "G", "ent": "Tetovo", "cls": "Western Asia"}, {"id": 20793, "dom": "G", "ent": "Tetovo", "cls": "Western Europe"}, {"id": 20794, "dom": "G", "ent": "Kochani", "cls": "Australia and New Zealand"}, {"id": 20795, "dom": "G", "ent": "Kochani", "cls": "Caribbean"}, {"id": 20796, "dom": "G", "ent": "Kochani", "cls": "Central America"}, {"id": 20797, "dom": "G", "ent": "Kochani", "cls": "Central Asia"}, {"id": 20798, "dom": "G", "ent": "Kochani", "cls": "Eastern Africa"}, {"id": 20799, "dom": "G", "ent": "Kochani", "cls": "Eastern Asia"}, {"id": 20800, "dom": "G", "ent": "Kochani", "cls": "Eastern Europe"}, {"id": 20801, "dom": "G", "ent": "Kochani", "cls": "Melanesia"}, {"id": 20802, "dom": "G", "ent": "Kochani", "cls": "Micronesia"}, {"id": 20803, "dom": "G", "ent": "Kochani", "cls": "Middle Africa"}, {"id": 20804, "dom": "G", "ent": "Kochani", "cls": "Northern Africa"}, {"id": 20805, "dom": "G", "ent": "Kochani", "cls": "Northern America"}, {"id": 20806, "dom": "G", "ent": "Kochani", "cls": "Northern Europe"}, {"id": 20807, "dom": "G", "ent": "Kochani", "cls": "Polynesia"}, {"id": 20808, "dom": "G", "ent": "Kochani", "cls": "South America"}, {"id": 20809, "dom": "G", "ent": "Kochani", "cls": "South-eastern Asia"}, {"id": 20810, "dom": "G", "ent": "Kochani", "cls": "Southern Africa"}, {"id": 20811, "dom": "G", "ent": "Kochani", "cls": "Southern Asia"}, {"id": 20812, "dom": "G", "ent": "Kochani", "cls": "Southern Europe"}, {"id": 20813, "dom": "G", "ent": "Kochani", "cls": "Western Africa"}, {"id": 20814, "dom": "G", "ent": "Kochani", "cls": "Western Asia"}, {"id": 20815, "dom": "G", "ent": "Kochani", "cls": "Western Europe"}, {"id": 20816, "dom": "G", "ent": "Lipkovo", "cls": "Australia and New Zealand"}, {"id": 20817, "dom": "G", "ent": "Lipkovo", "cls": "Caribbean"}, {"id": 20818, "dom": "G", "ent": "Lipkovo", "cls": "Central America"}, {"id": 20819, "dom": "G", "ent": "Lipkovo", "cls": "Central Asia"}, {"id": 20820, "dom": "G", "ent": "Lipkovo", "cls": "Eastern Africa"}, {"id": 20821, "dom": "G", "ent": "Lipkovo", "cls": "Eastern Asia"}, {"id": 20822, "dom": "G", "ent": "Lipkovo", "cls": "Eastern Europe"}, {"id": 20823, "dom": "G", "ent": "Lipkovo", "cls": "Melanesia"}, {"id": 20824, "dom": "G", "ent": "Lipkovo", "cls": "Micronesia"}, {"id": 20825, "dom": "G", "ent": "Lipkovo", "cls": "Middle Africa"}, {"id": 20826, "dom": "G", "ent": "Lipkovo", "cls": "Northern Africa"}, {"id": 20827, "dom": "G", "ent": "Lipkovo", "cls": "Northern America"}, {"id": 20828, "dom": "G", "ent": "Lipkovo", "cls": "Northern Europe"}, {"id": 20829, "dom": "G", "ent": "Lipkovo", "cls": "Polynesia"}, {"id": 20830, "dom": "G", "ent": "Lipkovo", "cls": "South America"}, {"id": 20831, "dom": "G", "ent": "Lipkovo", "cls": "South-eastern Asia"}, {"id": 20832, "dom": "G", "ent": "Lipkovo", "cls": "Southern Africa"}, {"id": 20833, "dom": "G", "ent": "Lipkovo", "cls": "Southern Asia"}, {"id": 20834, "dom": "G", "ent": "Lipkovo", "cls": "Southern Europe"}, {"id": 20835, "dom": "G", "ent": "Lipkovo", "cls": "Western Africa"}, {"id": 20836, "dom": "G", "ent": "Lipkovo", "cls": "Western Asia"}, {"id": 20837, "dom": "G", "ent": "Lipkovo", "cls": "Western Europe"}, {"id": 20838, "dom": "G", "ent": "Zelino", "cls": "Australia and New Zealand"}, {"id": 20839, "dom": "G", "ent": "Zelino", "cls": "Caribbean"}, {"id": 20840, "dom": "G", "ent": "Zelino", "cls": "Central America"}, {"id": 20841, "dom": "G", "ent": "Zelino", "cls": "Central Asia"}, {"id": 20842, "dom": "G", "ent": "Zelino", "cls": "Eastern Africa"}, {"id": 20843, "dom": "G", "ent": "Zelino", "cls": "Eastern Asia"}, {"id": 20844, "dom": "G", "ent": "Zelino", "cls": "Eastern Europe"}, {"id": 20845, "dom": "G", "ent": "Zelino", "cls": "Melanesia"}, {"id": 20846, "dom": "G", "ent": "Zelino", "cls": "Micronesia"}, {"id": 20847, "dom": "G", "ent": "Zelino", "cls": "Middle Africa"}, {"id": 20848, "dom": "G", "ent": "Zelino", "cls": "Northern Africa"}, {"id": 20849, "dom": "G", "ent": "Zelino", "cls": "Northern America"}, {"id": 20850, "dom": "G", "ent": "Zelino", "cls": "Northern Europe"}, {"id": 20851, "dom": "G", "ent": "Zelino", "cls": "Polynesia"}, {"id": 20852, "dom": "G", "ent": "Zelino", "cls": "South America"}, {"id": 20853, "dom": "G", "ent": "Zelino", "cls": "South-eastern Asia"}, {"id": 20854, "dom": "G", "ent": "Zelino", "cls": "Southern Africa"}, {"id": 20855, "dom": "G", "ent": "Zelino", "cls": "Southern Asia"}, {"id": 20856, "dom": "G", "ent": "Zelino", "cls": "Southern Europe"}, {"id": 20857, "dom": "G", "ent": "Zelino", "cls": "Western Africa"}, {"id": 20858, "dom": "G", "ent": "Zelino", "cls": "Western Asia"}, {"id": 20859, "dom": "G", "ent": "Zelino", "cls": "Western Europe"}, {"id": 20860, "dom": "G", "ent": "Douentza", "cls": "Australia and New Zealand"}, {"id": 20861, "dom": "G", "ent": "Douentza", "cls": "Caribbean"}, {"id": 20862, "dom": "G", "ent": "Douentza", "cls": "Central America"}, {"id": 20863, "dom": "G", "ent": "Douentza", "cls": "Central Asia"}, {"id": 20864, "dom": "G", "ent": "Douentza", "cls": "Eastern Africa"}, {"id": 20865, "dom": "G", "ent": "Douentza", "cls": "Eastern Asia"}, {"id": 20866, "dom": "G", "ent": "Douentza", "cls": "Eastern Europe"}, {"id": 20867, "dom": "G", "ent": "Douentza", "cls": "Melanesia"}, {"id": 20868, "dom": "G", "ent": "Douentza", "cls": "Micronesia"}, {"id": 20869, "dom": "G", "ent": "Douentza", "cls": "Middle Africa"}, {"id": 20870, "dom": "G", "ent": "Douentza", "cls": "Northern Africa"}, {"id": 20871, "dom": "G", "ent": "Douentza", "cls": "Northern America"}, {"id": 20872, "dom": "G", "ent": "Douentza", "cls": "Northern Europe"}, {"id": 20873, "dom": "G", "ent": "Douentza", "cls": "Polynesia"}, {"id": 20874, "dom": "G", "ent": "Douentza", "cls": "South America"}, {"id": 20875, "dom": "G", "ent": "Douentza", "cls": "South-eastern Asia"}, {"id": 20876, "dom": "G", "ent": "Douentza", "cls": "Southern Africa"}, {"id": 20877, "dom": "G", "ent": "Douentza", "cls": "Southern Asia"}, {"id": 20878, "dom": "G", "ent": "Douentza", "cls": "Southern Europe"}, {"id": 20879, "dom": "G", "ent": "Douentza", "cls": "Western Africa"}, {"id": 20880, "dom": "G", "ent": "Douentza", "cls": "Western Asia"}, {"id": 20881, "dom": "G", "ent": "Douentza", "cls": "Western Europe"}, {"id": 20882, "dom": "G", "ent": "Finnkolo", "cls": "Australia and New Zealand"}, {"id": 20883, "dom": "G", "ent": "Finnkolo", "cls": "Caribbean"}, {"id": 20884, "dom": "G", "ent": "Finnkolo", "cls": "Central America"}, {"id": 20885, "dom": "G", "ent": "Finnkolo", "cls": "Central Asia"}, {"id": 20886, "dom": "G", "ent": "Finnkolo", "cls": "Eastern Africa"}, {"id": 20887, "dom": "G", "ent": "Finnkolo", "cls": "Eastern Asia"}, {"id": 20888, "dom": "G", "ent": "Finnkolo", "cls": "Eastern Europe"}, {"id": 20889, "dom": "G", "ent": "Finnkolo", "cls": "Melanesia"}, {"id": 20890, "dom": "G", "ent": "Finnkolo", "cls": "Micronesia"}, {"id": 20891, "dom": "G", "ent": "Finnkolo", "cls": "Middle Africa"}, {"id": 20892, "dom": "G", "ent": "Finnkolo", "cls": "Northern Africa"}, {"id": 20893, "dom": "G", "ent": "Finnkolo", "cls": "Northern America"}, {"id": 20894, "dom": "G", "ent": "Finnkolo", "cls": "Northern Europe"}, {"id": 20895, "dom": "G", "ent": "Finnkolo", "cls": "Polynesia"}, {"id": 20896, "dom": "G", "ent": "Finnkolo", "cls": "South America"}, {"id": 20897, "dom": "G", "ent": "Finnkolo", "cls": "South-eastern Asia"}, {"id": 20898, "dom": "G", "ent": "Finnkolo", "cls": "Southern Africa"}, {"id": 20899, "dom": "G", "ent": "Finnkolo", "cls": "Southern Asia"}, {"id": 20900, "dom": "G", "ent": "Finnkolo", "cls": "Southern Europe"}, {"id": 20901, "dom": "G", "ent": "Finnkolo", "cls": "Western Africa"}, {"id": 20902, "dom": "G", "ent": "Finnkolo", "cls": "Western Asia"}, {"id": 20903, "dom": "G", "ent": "Finnkolo", "cls": "Western Europe"}, {"id": 20904, "dom": "G", "ent": "Kouri", "cls": "Australia and New Zealand"}, {"id": 20905, "dom": "G", "ent": "Kouri", "cls": "Caribbean"}, {"id": 20906, "dom": "G", "ent": "Kouri", "cls": "Central America"}, {"id": 20907, "dom": "G", "ent": "Kouri", "cls": "Central Asia"}, {"id": 20908, "dom": "G", "ent": "Kouri", "cls": "Eastern Africa"}, {"id": 20909, "dom": "G", "ent": "Kouri", "cls": "Eastern Asia"}, {"id": 20910, "dom": "G", "ent": "Kouri", "cls": "Eastern Europe"}, {"id": 20911, "dom": "G", "ent": "Kouri", "cls": "Melanesia"}, {"id": 20912, "dom": "G", "ent": "Kouri", "cls": "Micronesia"}, {"id": 20913, "dom": "G", "ent": "Kouri", "cls": "Middle Africa"}, {"id": 20914, "dom": "G", "ent": "Kouri", "cls": "Northern Africa"}, {"id": 20915, "dom": "G", "ent": "Kouri", "cls": "Northern America"}, {"id": 20916, "dom": "G", "ent": "Kouri", "cls": "Northern Europe"}, {"id": 20917, "dom": "G", "ent": "Kouri", "cls": "Polynesia"}, {"id": 20918, "dom": "G", "ent": "Kouri", "cls": "South America"}, {"id": 20919, "dom": "G", "ent": "Kouri", "cls": "South-eastern Asia"}, {"id": 20920, "dom": "G", "ent": "Kouri", "cls": "Southern Africa"}, {"id": 20921, "dom": "G", "ent": "Kouri", "cls": "Southern Asia"}, {"id": 20922, "dom": "G", "ent": "Kouri", "cls": "Southern Europe"}, {"id": 20923, "dom": "G", "ent": "Kouri", "cls": "Western Africa"}, {"id": 20924, "dom": "G", "ent": "Kouri", "cls": "Western Asia"}, {"id": 20925, "dom": "G", "ent": "Kouri", "cls": "Western Europe"}, {"id": 20926, "dom": "G", "ent": "Goundam", "cls": "Australia and New Zealand"}, {"id": 20927, "dom": "G", "ent": "Goundam", "cls": "Caribbean"}, {"id": 20928, "dom": "G", "ent": "Goundam", "cls": "Central America"}, {"id": 20929, "dom": "G", "ent": "Goundam", "cls": "Central Asia"}, {"id": 20930, "dom": "G", "ent": "Goundam", "cls": "Eastern Africa"}, {"id": 20931, "dom": "G", "ent": "Goundam", "cls": "Eastern Asia"}, {"id": 20932, "dom": "G", "ent": "Goundam", "cls": "Eastern Europe"}, {"id": 20933, "dom": "G", "ent": "Goundam", "cls": "Melanesia"}, {"id": 20934, "dom": "G", "ent": "Goundam", "cls": "Micronesia"}, {"id": 20935, "dom": "G", "ent": "Goundam", "cls": "Middle Africa"}, {"id": 20936, "dom": "G", "ent": "Goundam", "cls": "Northern Africa"}, {"id": 20937, "dom": "G", "ent": "Goundam", "cls": "Northern America"}, {"id": 20938, "dom": "G", "ent": "Goundam", "cls": "Northern Europe"}, {"id": 20939, "dom": "G", "ent": "Goundam", "cls": "Polynesia"}, {"id": 20940, "dom": "G", "ent": "Goundam", "cls": "South America"}, {"id": 20941, "dom": "G", "ent": "Goundam", "cls": "South-eastern Asia"}, {"id": 20942, "dom": "G", "ent": "Goundam", "cls": "Southern Africa"}, {"id": 20943, "dom": "G", "ent": "Goundam", "cls": "Southern Asia"}, {"id": 20944, "dom": "G", "ent": "Goundam", "cls": "Southern Europe"}, {"id": 20945, "dom": "G", "ent": "Goundam", "cls": "Western Africa"}, {"id": 20946, "dom": "G", "ent": "Goundam", "cls": "Western Asia"}, {"id": 20947, "dom": "G", "ent": "Goundam", "cls": "Western Europe"}, {"id": 20948, "dom": "G", "ent": "Kyimyindine", "cls": "Australia and New Zealand"}, {"id": 20949, "dom": "G", "ent": "Kyimyindine", "cls": "Caribbean"}, {"id": 20950, "dom": "G", "ent": "Kyimyindine", "cls": "Central America"}, {"id": 20951, "dom": "G", "ent": "Kyimyindine", "cls": "Central Asia"}, {"id": 20952, "dom": "G", "ent": "Kyimyindine", "cls": "Eastern Africa"}, {"id": 20953, "dom": "G", "ent": "Kyimyindine", "cls": "Eastern Asia"}, {"id": 20954, "dom": "G", "ent": "Kyimyindine", "cls": "Eastern Europe"}, {"id": 20955, "dom": "G", "ent": "Kyimyindine", "cls": "Melanesia"}, {"id": 20956, "dom": "G", "ent": "Kyimyindine", "cls": "Micronesia"}, {"id": 20957, "dom": "G", "ent": "Kyimyindine", "cls": "Middle Africa"}, {"id": 20958, "dom": "G", "ent": "Kyimyindine", "cls": "Northern Africa"}, {"id": 20959, "dom": "G", "ent": "Kyimyindine", "cls": "Northern America"}, {"id": 20960, "dom": "G", "ent": "Kyimyindine", "cls": "Northern Europe"}, {"id": 20961, "dom": "G", "ent": "Kyimyindine", "cls": "Polynesia"}, {"id": 20962, "dom": "G", "ent": "Kyimyindine", "cls": "South America"}, {"id": 20963, "dom": "G", "ent": "Kyimyindine", "cls": "South-eastern Asia"}, {"id": 20964, "dom": "G", "ent": "Kyimyindine", "cls": "Southern Africa"}, {"id": 20965, "dom": "G", "ent": "Kyimyindine", "cls": "Southern Asia"}, {"id": 20966, "dom": "G", "ent": "Kyimyindine", "cls": "Southern Europe"}, {"id": 20967, "dom": "G", "ent": "Kyimyindine", "cls": "Western Africa"}, {"id": 20968, "dom": "G", "ent": "Kyimyindine", "cls": "Western Asia"}, {"id": 20969, "dom": "G", "ent": "Kyimyindine", "cls": "Western Europe"}, {"id": 20970, "dom": "G", "ent": "Aungban", "cls": "Australia and New Zealand"}, {"id": 20971, "dom": "G", "ent": "Aungban", "cls": "Caribbean"}, {"id": 20972, "dom": "G", "ent": "Aungban", "cls": "Central America"}, {"id": 20973, "dom": "G", "ent": "Aungban", "cls": "Central Asia"}, {"id": 20974, "dom": "G", "ent": "Aungban", "cls": "Eastern Africa"}, {"id": 20975, "dom": "G", "ent": "Aungban", "cls": "Eastern Asia"}, {"id": 20976, "dom": "G", "ent": "Aungban", "cls": "Eastern Europe"}, {"id": 20977, "dom": "G", "ent": "Aungban", "cls": "Melanesia"}, {"id": 20978, "dom": "G", "ent": "Aungban", "cls": "Micronesia"}, {"id": 20979, "dom": "G", "ent": "Aungban", "cls": "Middle Africa"}, {"id": 20980, "dom": "G", "ent": "Aungban", "cls": "Northern Africa"}, {"id": 20981, "dom": "G", "ent": "Aungban", "cls": "Northern America"}, {"id": 20982, "dom": "G", "ent": "Aungban", "cls": "Northern Europe"}, {"id": 20983, "dom": "G", "ent": "Aungban", "cls": "Polynesia"}, {"id": 20984, "dom": "G", "ent": "Aungban", "cls": "South America"}, {"id": 20985, "dom": "G", "ent": "Aungban", "cls": "South-eastern Asia"}, {"id": 20986, "dom": "G", "ent": "Aungban", "cls": "Southern Africa"}, {"id": 20987, "dom": "G", "ent": "Aungban", "cls": "Southern Asia"}, {"id": 20988, "dom": "G", "ent": "Aungban", "cls": "Southern Europe"}, {"id": 20989, "dom": "G", "ent": "Aungban", "cls": "Western Africa"}, {"id": 20990, "dom": "G", "ent": "Aungban", "cls": "Western Asia"}, {"id": 20991, "dom": "G", "ent": "Aungban", "cls": "Western Europe"}, {"id": 20992, "dom": "G", "ent": "Zee Kone", "cls": "Australia and New Zealand"}, {"id": 20993, "dom": "G", "ent": "Zee Kone", "cls": "Caribbean"}, {"id": 20994, "dom": "G", "ent": "Zee Kone", "cls": "Central America"}, {"id": 20995, "dom": "G", "ent": "Zee Kone", "cls": "Central Asia"}, {"id": 20996, "dom": "G", "ent": "Zee Kone", "cls": "Eastern Africa"}, {"id": 20997, "dom": "G", "ent": "Zee Kone", "cls": "Eastern Asia"}, {"id": 20998, "dom": "G", "ent": "Zee Kone", "cls": "Eastern Europe"}, {"id": 20999, "dom": "G", "ent": "Zee Kone", "cls": "Melanesia"}, {"id": 21000, "dom": "G", "ent": "Zee Kone", "cls": "Micronesia"}, {"id": 21001, "dom": "G", "ent": "Zee Kone", "cls": "Middle Africa"}, {"id": 21002, "dom": "G", "ent": "Zee Kone", "cls": "Northern Africa"}, {"id": 21003, "dom": "G", "ent": "Zee Kone", "cls": "Northern America"}, {"id": 21004, "dom": "G", "ent": "Zee Kone", "cls": "Northern Europe"}, {"id": 21005, "dom": "G", "ent": "Zee Kone", "cls": "Polynesia"}, {"id": 21006, "dom": "G", "ent": "Zee Kone", "cls": "South America"}, {"id": 21007, "dom": "G", "ent": "Zee Kone", "cls": "South-eastern Asia"}, {"id": 21008, "dom": "G", "ent": "Zee Kone", "cls": "Southern Africa"}, {"id": 21009, "dom": "G", "ent": "Zee Kone", "cls": "Southern Asia"}, {"id": 21010, "dom": "G", "ent": "Zee Kone", "cls": "Southern Europe"}, {"id": 21011, "dom": "G", "ent": "Zee Kone", "cls": "Western Africa"}, {"id": 21012, "dom": "G", "ent": "Zee Kone", "cls": "Western Asia"}, {"id": 21013, "dom": "G", "ent": "Zee Kone", "cls": "Western Europe"}, {"id": 21014, "dom": "G", "ent": "Yesagyo", "cls": "Australia and New Zealand"}, {"id": 21015, "dom": "G", "ent": "Yesagyo", "cls": "Caribbean"}, {"id": 21016, "dom": "G", "ent": "Yesagyo", "cls": "Central America"}, {"id": 21017, "dom": "G", "ent": "Yesagyo", "cls": "Central Asia"}, {"id": 21018, "dom": "G", "ent": "Yesagyo", "cls": "Eastern Africa"}, {"id": 21019, "dom": "G", "ent": "Yesagyo", "cls": "Eastern Asia"}, {"id": 21020, "dom": "G", "ent": "Yesagyo", "cls": "Eastern Europe"}, {"id": 21021, "dom": "G", "ent": "Yesagyo", "cls": "Melanesia"}, {"id": 21022, "dom": "G", "ent": "Yesagyo", "cls": "Micronesia"}, {"id": 21023, "dom": "G", "ent": "Yesagyo", "cls": "Middle Africa"}, {"id": 21024, "dom": "G", "ent": "Yesagyo", "cls": "Northern Africa"}, {"id": 21025, "dom": "G", "ent": "Yesagyo", "cls": "Northern America"}, {"id": 21026, "dom": "G", "ent": "Yesagyo", "cls": "Northern Europe"}, {"id": 21027, "dom": "G", "ent": "Yesagyo", "cls": "Polynesia"}, {"id": 21028, "dom": "G", "ent": "Yesagyo", "cls": "South America"}, {"id": 21029, "dom": "G", "ent": "Yesagyo", "cls": "South-eastern Asia"}, {"id": 21030, "dom": "G", "ent": "Yesagyo", "cls": "Southern Africa"}, {"id": 21031, "dom": "G", "ent": "Yesagyo", "cls": "Southern Asia"}, {"id": 21032, "dom": "G", "ent": "Yesagyo", "cls": "Southern Europe"}, {"id": 21033, "dom": "G", "ent": "Yesagyo", "cls": "Western Africa"}, {"id": 21034, "dom": "G", "ent": "Yesagyo", "cls": "Western Asia"}, {"id": 21035, "dom": "G", "ent": "Yesagyo", "cls": "Western Europe"}, {"id": 21036, "dom": "G", "ent": "Nalayh", "cls": "Australia and New Zealand"}, {"id": 21037, "dom": "G", "ent": "Nalayh", "cls": "Caribbean"}, {"id": 21038, "dom": "G", "ent": "Nalayh", "cls": "Central America"}, {"id": 21039, "dom": "G", "ent": "Nalayh", "cls": "Central Asia"}, {"id": 21040, "dom": "G", "ent": "Nalayh", "cls": "Eastern Africa"}, {"id": 21041, "dom": "G", "ent": "Nalayh", "cls": "Eastern Asia"}, {"id": 21042, "dom": "G", "ent": "Nalayh", "cls": "Eastern Europe"}, {"id": 21043, "dom": "G", "ent": "Nalayh", "cls": "Melanesia"}, {"id": 21044, "dom": "G", "ent": "Nalayh", "cls": "Micronesia"}, {"id": 21045, "dom": "G", "ent": "Nalayh", "cls": "Middle Africa"}, {"id": 21046, "dom": "G", "ent": "Nalayh", "cls": "Northern Africa"}, {"id": 21047, "dom": "G", "ent": "Nalayh", "cls": "Northern America"}, {"id": 21048, "dom": "G", "ent": "Nalayh", "cls": "Northern Europe"}, {"id": 21049, "dom": "G", "ent": "Nalayh", "cls": "Polynesia"}, {"id": 21050, "dom": "G", "ent": "Nalayh", "cls": "South America"}, {"id": 21051, "dom": "G", "ent": "Nalayh", "cls": "South-eastern Asia"}, {"id": 21052, "dom": "G", "ent": "Nalayh", "cls": "Southern Africa"}, {"id": 21053, "dom": "G", "ent": "Nalayh", "cls": "Southern Asia"}, {"id": 21054, "dom": "G", "ent": "Nalayh", "cls": "Southern Europe"}, {"id": 21055, "dom": "G", "ent": "Nalayh", "cls": "Western Africa"}, {"id": 21056, "dom": "G", "ent": "Nalayh", "cls": "Western Asia"}, {"id": 21057, "dom": "G", "ent": "Nalayh", "cls": "Western Europe"}, {"id": 21058, "dom": "G", "ent": "Khovd", "cls": "Australia and New Zealand"}, {"id": 21059, "dom": "G", "ent": "Khovd", "cls": "Caribbean"}, {"id": 21060, "dom": "G", "ent": "Khovd", "cls": "Central America"}, {"id": 21061, "dom": "G", "ent": "Khovd", "cls": "Central Asia"}, {"id": 21062, "dom": "G", "ent": "Khovd", "cls": "Eastern Africa"}, {"id": 21063, "dom": "G", "ent": "Khovd", "cls": "Eastern Asia"}, {"id": 21064, "dom": "G", "ent": "Khovd", "cls": "Eastern Europe"}, {"id": 21065, "dom": "G", "ent": "Khovd", "cls": "Melanesia"}, {"id": 21066, "dom": "G", "ent": "Khovd", "cls": "Micronesia"}, {"id": 21067, "dom": "G", "ent": "Khovd", "cls": "Middle Africa"}, {"id": 21068, "dom": "G", "ent": "Khovd", "cls": "Northern Africa"}, {"id": 21069, "dom": "G", "ent": "Khovd", "cls": "Northern America"}, {"id": 21070, "dom": "G", "ent": "Khovd", "cls": "Northern Europe"}, {"id": 21071, "dom": "G", "ent": "Khovd", "cls": "Polynesia"}, {"id": 21072, "dom": "G", "ent": "Khovd", "cls": "South America"}, {"id": 21073, "dom": "G", "ent": "Khovd", "cls": "South-eastern Asia"}, {"id": 21074, "dom": "G", "ent": "Khovd", "cls": "Southern Africa"}, {"id": 21075, "dom": "G", "ent": "Khovd", "cls": "Southern Asia"}, {"id": 21076, "dom": "G", "ent": "Khovd", "cls": "Southern Europe"}, {"id": 21077, "dom": "G", "ent": "Khovd", "cls": "Western Africa"}, {"id": 21078, "dom": "G", "ent": "Khovd", "cls": "Western Asia"}, {"id": 21079, "dom": "G", "ent": "Khovd", "cls": "Western Europe"}, {"id": 21080, "dom": "G", "ent": "Tsetserleg", "cls": "Australia and New Zealand"}, {"id": 21081, "dom": "G", "ent": "Tsetserleg", "cls": "Caribbean"}, {"id": 21082, "dom": "G", "ent": "Tsetserleg", "cls": "Central America"}, {"id": 21083, "dom": "G", "ent": "Tsetserleg", "cls": "Central Asia"}, {"id": 21084, "dom": "G", "ent": "Tsetserleg", "cls": "Eastern Africa"}, {"id": 21085, "dom": "G", "ent": "Tsetserleg", "cls": "Eastern Asia"}, {"id": 21086, "dom": "G", "ent": "Tsetserleg", "cls": "Eastern Europe"}, {"id": 21087, "dom": "G", "ent": "Tsetserleg", "cls": "Melanesia"}, {"id": 21088, "dom": "G", "ent": "Tsetserleg", "cls": "Micronesia"}, {"id": 21089, "dom": "G", "ent": "Tsetserleg", "cls": "Middle Africa"}, {"id": 21090, "dom": "G", "ent": "Tsetserleg", "cls": "Northern Africa"}, {"id": 21091, "dom": "G", "ent": "Tsetserleg", "cls": "Northern America"}, {"id": 21092, "dom": "G", "ent": "Tsetserleg", "cls": "Northern Europe"}, {"id": 21093, "dom": "G", "ent": "Tsetserleg", "cls": "Polynesia"}, {"id": 21094, "dom": "G", "ent": "Tsetserleg", "cls": "South America"}, {"id": 21095, "dom": "G", "ent": "Tsetserleg", "cls": "South-eastern Asia"}, {"id": 21096, "dom": "G", "ent": "Tsetserleg", "cls": "Southern Africa"}, {"id": 21097, "dom": "G", "ent": "Tsetserleg", "cls": "Southern Asia"}, {"id": 21098, "dom": "G", "ent": "Tsetserleg", "cls": "Southern Europe"}, {"id": 21099, "dom": "G", "ent": "Tsetserleg", "cls": "Western Africa"}, {"id": 21100, "dom": "G", "ent": "Tsetserleg", "cls": "Western Asia"}, {"id": 21101, "dom": "G", "ent": "Tsetserleg", "cls": "Western Europe"}, {"id": 21102, "dom": "G", "ent": "Dzuunmod", "cls": "Australia and New Zealand"}, {"id": 21103, "dom": "G", "ent": "Dzuunmod", "cls": "Caribbean"}, {"id": 21104, "dom": "G", "ent": "Dzuunmod", "cls": "Central America"}, {"id": 21105, "dom": "G", "ent": "Dzuunmod", "cls": "Central Asia"}, {"id": 21106, "dom": "G", "ent": "Dzuunmod", "cls": "Eastern Africa"}, {"id": 21107, "dom": "G", "ent": "Dzuunmod", "cls": "Eastern Asia"}, {"id": 21108, "dom": "G", "ent": "Dzuunmod", "cls": "Eastern Europe"}, {"id": 21109, "dom": "G", "ent": "Dzuunmod", "cls": "Melanesia"}, {"id": 21110, "dom": "G", "ent": "Dzuunmod", "cls": "Micronesia"}, {"id": 21111, "dom": "G", "ent": "Dzuunmod", "cls": "Middle Africa"}, {"id": 21112, "dom": "G", "ent": "Dzuunmod", "cls": "Northern Africa"}, {"id": 21113, "dom": "G", "ent": "Dzuunmod", "cls": "Northern America"}, {"id": 21114, "dom": "G", "ent": "Dzuunmod", "cls": "Northern Europe"}, {"id": 21115, "dom": "G", "ent": "Dzuunmod", "cls": "Polynesia"}, {"id": 21116, "dom": "G", "ent": "Dzuunmod", "cls": "South America"}, {"id": 21117, "dom": "G", "ent": "Dzuunmod", "cls": "South-eastern Asia"}, {"id": 21118, "dom": "G", "ent": "Dzuunmod", "cls": "Southern Africa"}, {"id": 21119, "dom": "G", "ent": "Dzuunmod", "cls": "Southern Asia"}, {"id": 21120, "dom": "G", "ent": "Dzuunmod", "cls": "Southern Europe"}, {"id": 21121, "dom": "G", "ent": "Dzuunmod", "cls": "Western Africa"}, {"id": 21122, "dom": "G", "ent": "Dzuunmod", "cls": "Western Asia"}, {"id": 21123, "dom": "G", "ent": "Dzuunmod", "cls": "Western Europe"}, {"id": 21124, "dom": "G", "ent": "Mbera", "cls": "Australia and New Zealand"}, {"id": 21125, "dom": "G", "ent": "Mbera", "cls": "Caribbean"}, {"id": 21126, "dom": "G", "ent": "Mbera", "cls": "Central America"}, {"id": 21127, "dom": "G", "ent": "Mbera", "cls": "Central Asia"}, {"id": 21128, "dom": "G", "ent": "Mbera", "cls": "Eastern Africa"}, {"id": 21129, "dom": "G", "ent": "Mbera", "cls": "Eastern Asia"}, {"id": 21130, "dom": "G", "ent": "Mbera", "cls": "Eastern Europe"}, {"id": 21131, "dom": "G", "ent": "Mbera", "cls": "Melanesia"}, {"id": 21132, "dom": "G", "ent": "Mbera", "cls": "Micronesia"}, {"id": 21133, "dom": "G", "ent": "Mbera", "cls": "Middle Africa"}, {"id": 21134, "dom": "G", "ent": "Mbera", "cls": "Northern Africa"}, {"id": 21135, "dom": "G", "ent": "Mbera", "cls": "Northern America"}, {"id": 21136, "dom": "G", "ent": "Mbera", "cls": "Northern Europe"}, {"id": 21137, "dom": "G", "ent": "Mbera", "cls": "Polynesia"}, {"id": 21138, "dom": "G", "ent": "Mbera", "cls": "South America"}, {"id": 21139, "dom": "G", "ent": "Mbera", "cls": "South-eastern Asia"}, {"id": 21140, "dom": "G", "ent": "Mbera", "cls": "Southern Africa"}, {"id": 21141, "dom": "G", "ent": "Mbera", "cls": "Southern Asia"}, {"id": 21142, "dom": "G", "ent": "Mbera", "cls": "Southern Europe"}, {"id": 21143, "dom": "G", "ent": "Mbera", "cls": "Western Africa"}, {"id": 21144, "dom": "G", "ent": "Mbera", "cls": "Western Asia"}, {"id": 21145, "dom": "G", "ent": "Mbera", "cls": "Western Europe"}, {"id": 21146, "dom": "G", "ent": "Tevragh Zeina", "cls": "Australia and New Zealand"}, {"id": 21147, "dom": "G", "ent": "Tevragh Zeina", "cls": "Caribbean"}, {"id": 21148, "dom": "G", "ent": "Tevragh Zeina", "cls": "Central America"}, {"id": 21149, "dom": "G", "ent": "Tevragh Zeina", "cls": "Central Asia"}, {"id": 21150, "dom": "G", "ent": "Tevragh Zeina", "cls": "Eastern Africa"}, {"id": 21151, "dom": "G", "ent": "Tevragh Zeina", "cls": "Eastern Asia"}, {"id": 21152, "dom": "G", "ent": "Tevragh Zeina", "cls": "Eastern Europe"}, {"id": 21153, "dom": "G", "ent": "Tevragh Zeina", "cls": "Melanesia"}, {"id": 21154, "dom": "G", "ent": "Tevragh Zeina", "cls": "Micronesia"}, {"id": 21155, "dom": "G", "ent": "Tevragh Zeina", "cls": "Middle Africa"}, {"id": 21156, "dom": "G", "ent": "Tevragh Zeina", "cls": "Northern Africa"}, {"id": 21157, "dom": "G", "ent": "Tevragh Zeina", "cls": "Northern America"}, {"id": 21158, "dom": "G", "ent": "Tevragh Zeina", "cls": "Northern Europe"}, {"id": 21159, "dom": "G", "ent": "Tevragh Zeina", "cls": "Polynesia"}, {"id": 21160, "dom": "G", "ent": "Tevragh Zeina", "cls": "South America"}, {"id": 21161, "dom": "G", "ent": "Tevragh Zeina", "cls": "South-eastern Asia"}, {"id": 21162, "dom": "G", "ent": "Tevragh Zeina", "cls": "Southern Africa"}, {"id": 21163, "dom": "G", "ent": "Tevragh Zeina", "cls": "Southern Asia"}, {"id": 21164, "dom": "G", "ent": "Tevragh Zeina", "cls": "Southern Europe"}, {"id": 21165, "dom": "G", "ent": "Tevragh Zeina", "cls": "Western Africa"}, {"id": 21166, "dom": "G", "ent": "Tevragh Zeina", "cls": "Western Asia"}, {"id": 21167, "dom": "G", "ent": "Tevragh Zeina", "cls": "Western Europe"}, {"id": 21168, "dom": "G", "ent": "Aleg", "cls": "Australia and New Zealand"}, {"id": 21169, "dom": "G", "ent": "Aleg", "cls": "Caribbean"}, {"id": 21170, "dom": "G", "ent": "Aleg", "cls": "Central America"}, {"id": 21171, "dom": "G", "ent": "Aleg", "cls": "Central Asia"}, {"id": 21172, "dom": "G", "ent": "Aleg", "cls": "Eastern Africa"}, {"id": 21173, "dom": "G", "ent": "Aleg", "cls": "Eastern Asia"}, {"id": 21174, "dom": "G", "ent": "Aleg", "cls": "Eastern Europe"}, {"id": 21175, "dom": "G", "ent": "Aleg", "cls": "Melanesia"}, {"id": 21176, "dom": "G", "ent": "Aleg", "cls": "Micronesia"}, {"id": 21177, "dom": "G", "ent": "Aleg", "cls": "Middle Africa"}, {"id": 21178, "dom": "G", "ent": "Aleg", "cls": "Northern Africa"}, {"id": 21179, "dom": "G", "ent": "Aleg", "cls": "Northern America"}, {"id": 21180, "dom": "G", "ent": "Aleg", "cls": "Northern Europe"}, {"id": 21181, "dom": "G", "ent": "Aleg", "cls": "Polynesia"}, {"id": 21182, "dom": "G", "ent": "Aleg", "cls": "South America"}, {"id": 21183, "dom": "G", "ent": "Aleg", "cls": "South-eastern Asia"}, {"id": 21184, "dom": "G", "ent": "Aleg", "cls": "Southern Africa"}, {"id": 21185, "dom": "G", "ent": "Aleg", "cls": "Southern Asia"}, {"id": 21186, "dom": "G", "ent": "Aleg", "cls": "Southern Europe"}, {"id": 21187, "dom": "G", "ent": "Aleg", "cls": "Western Africa"}, {"id": 21188, "dom": "G", "ent": "Aleg", "cls": "Western Asia"}, {"id": 21189, "dom": "G", "ent": "Aleg", "cls": "Western Europe"}, {"id": 21190, "dom": "G", "ent": "Tembedgha", "cls": "Australia and New Zealand"}, {"id": 21191, "dom": "G", "ent": "Tembedgha", "cls": "Caribbean"}, {"id": 21192, "dom": "G", "ent": "Tembedgha", "cls": "Central America"}, {"id": 21193, "dom": "G", "ent": "Tembedgha", "cls": "Central Asia"}, {"id": 21194, "dom": "G", "ent": "Tembedgha", "cls": "Eastern Africa"}, {"id": 21195, "dom": "G", "ent": "Tembedgha", "cls": "Eastern Asia"}, {"id": 21196, "dom": "G", "ent": "Tembedgha", "cls": "Eastern Europe"}, {"id": 21197, "dom": "G", "ent": "Tembedgha", "cls": "Melanesia"}, {"id": 21198, "dom": "G", "ent": "Tembedgha", "cls": "Micronesia"}, {"id": 21199, "dom": "G", "ent": "Tembedgha", "cls": "Middle Africa"}, {"id": 21200, "dom": "G", "ent": "Tembedgha", "cls": "Northern Africa"}, {"id": 21201, "dom": "G", "ent": "Tembedgha", "cls": "Northern America"}, {"id": 21202, "dom": "G", "ent": "Tembedgha", "cls": "Northern Europe"}, {"id": 21203, "dom": "G", "ent": "Tembedgha", "cls": "Polynesia"}, {"id": 21204, "dom": "G", "ent": "Tembedgha", "cls": "South America"}, {"id": 21205, "dom": "G", "ent": "Tembedgha", "cls": "South-eastern Asia"}, {"id": 21206, "dom": "G", "ent": "Tembedgha", "cls": "Southern Africa"}, {"id": 21207, "dom": "G", "ent": "Tembedgha", "cls": "Southern Asia"}, {"id": 21208, "dom": "G", "ent": "Tembedgha", "cls": "Southern Europe"}, {"id": 21209, "dom": "G", "ent": "Tembedgha", "cls": "Western Africa"}, {"id": 21210, "dom": "G", "ent": "Tembedgha", "cls": "Western Asia"}, {"id": 21211, "dom": "G", "ent": "Tembedgha", "cls": "Western Europe"}, {"id": 21212, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "Australia and New Zealand"}, {"id": 21213, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "Caribbean"}, {"id": 21214, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "Central America"}, {"id": 21215, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "Central Asia"}, {"id": 21216, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "Eastern Africa"}, {"id": 21217, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "Eastern Asia"}, {"id": 21218, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "Eastern Europe"}, {"id": 21219, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "Melanesia"}, {"id": 21220, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "Micronesia"}, {"id": 21221, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "Middle Africa"}, {"id": 21222, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "Northern Africa"}, {"id": 21223, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "Northern America"}, {"id": 21224, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "Northern Europe"}, {"id": 21225, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "Polynesia"}, {"id": 21226, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "South America"}, {"id": 21227, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "South-eastern Asia"}, {"id": 21228, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "Southern Africa"}, {"id": 21229, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "Southern Asia"}, {"id": 21230, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "Southern Europe"}, {"id": 21231, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "Western Africa"}, {"id": 21232, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "Western Asia"}, {"id": 21233, "dom": "G", "ent": "Beau Bassin-Rose Hill", "cls": "Western Europe"}, {"id": 21234, "dom": "G", "ent": "Quatre Bornes", "cls": "Australia and New Zealand"}, {"id": 21235, "dom": "G", "ent": "Quatre Bornes", "cls": "Caribbean"}, {"id": 21236, "dom": "G", "ent": "Quatre Bornes", "cls": "Central America"}, {"id": 21237, "dom": "G", "ent": "Quatre Bornes", "cls": "Central Asia"}, {"id": 21238, "dom": "G", "ent": "Quatre Bornes", "cls": "Eastern Africa"}, {"id": 21239, "dom": "G", "ent": "Quatre Bornes", "cls": "Eastern Asia"}, {"id": 21240, "dom": "G", "ent": "Quatre Bornes", "cls": "Eastern Europe"}, {"id": 21241, "dom": "G", "ent": "Quatre Bornes", "cls": "Melanesia"}, {"id": 21242, "dom": "G", "ent": "Quatre Bornes", "cls": "Micronesia"}, {"id": 21243, "dom": "G", "ent": "Quatre Bornes", "cls": "Middle Africa"}, {"id": 21244, "dom": "G", "ent": "Quatre Bornes", "cls": "Northern Africa"}, {"id": 21245, "dom": "G", "ent": "Quatre Bornes", "cls": "Northern America"}, {"id": 21246, "dom": "G", "ent": "Quatre Bornes", "cls": "Northern Europe"}, {"id": 21247, "dom": "G", "ent": "Quatre Bornes", "cls": "Polynesia"}, {"id": 21248, "dom": "G", "ent": "Quatre Bornes", "cls": "South America"}, {"id": 21249, "dom": "G", "ent": "Quatre Bornes", "cls": "South-eastern Asia"}, {"id": 21250, "dom": "G", "ent": "Quatre Bornes", "cls": "Southern Africa"}, {"id": 21251, "dom": "G", "ent": "Quatre Bornes", "cls": "Southern Asia"}, {"id": 21252, "dom": "G", "ent": "Quatre Bornes", "cls": "Southern Europe"}, {"id": 21253, "dom": "G", "ent": "Quatre Bornes", "cls": "Western Africa"}, {"id": 21254, "dom": "G", "ent": "Quatre Bornes", "cls": "Western Asia"}, {"id": 21255, "dom": "G", "ent": "Quatre Bornes", "cls": "Western Europe"}, {"id": 21256, "dom": "G", "ent": "Goodlands", "cls": "Australia and New Zealand"}, {"id": 21257, "dom": "G", "ent": "Goodlands", "cls": "Caribbean"}, {"id": 21258, "dom": "G", "ent": "Goodlands", "cls": "Central America"}, {"id": 21259, "dom": "G", "ent": "Goodlands", "cls": "Central Asia"}, {"id": 21260, "dom": "G", "ent": "Goodlands", "cls": "Eastern Africa"}, {"id": 21261, "dom": "G", "ent": "Goodlands", "cls": "Eastern Asia"}, {"id": 21262, "dom": "G", "ent": "Goodlands", "cls": "Eastern Europe"}, {"id": 21263, "dom": "G", "ent": "Goodlands", "cls": "Melanesia"}, {"id": 21264, "dom": "G", "ent": "Goodlands", "cls": "Micronesia"}, {"id": 21265, "dom": "G", "ent": "Goodlands", "cls": "Middle Africa"}, {"id": 21266, "dom": "G", "ent": "Goodlands", "cls": "Northern Africa"}, {"id": 21267, "dom": "G", "ent": "Goodlands", "cls": "Northern America"}, {"id": 21268, "dom": "G", "ent": "Goodlands", "cls": "Northern Europe"}, {"id": 21269, "dom": "G", "ent": "Goodlands", "cls": "Polynesia"}, {"id": 21270, "dom": "G", "ent": "Goodlands", "cls": "South America"}, {"id": 21271, "dom": "G", "ent": "Goodlands", "cls": "South-eastern Asia"}, {"id": 21272, "dom": "G", "ent": "Goodlands", "cls": "Southern Africa"}, {"id": 21273, "dom": "G", "ent": "Goodlands", "cls": "Southern Asia"}, {"id": 21274, "dom": "G", "ent": "Goodlands", "cls": "Southern Europe"}, {"id": 21275, "dom": "G", "ent": "Goodlands", "cls": "Western Africa"}, {"id": 21276, "dom": "G", "ent": "Goodlands", "cls": "Western Asia"}, {"id": 21277, "dom": "G", "ent": "Goodlands", "cls": "Western Europe"}, {"id": 21278, "dom": "G", "ent": "Baie du Tombeau", "cls": "Australia and New Zealand"}, {"id": 21279, "dom": "G", "ent": "Baie du Tombeau", "cls": "Caribbean"}, {"id": 21280, "dom": "G", "ent": "Baie du Tombeau", "cls": "Central America"}, {"id": 21281, "dom": "G", "ent": "Baie du Tombeau", "cls": "Central Asia"}, {"id": 21282, "dom": "G", "ent": "Baie du Tombeau", "cls": "Eastern Africa"}, {"id": 21283, "dom": "G", "ent": "Baie du Tombeau", "cls": "Eastern Asia"}, {"id": 21284, "dom": "G", "ent": "Baie du Tombeau", "cls": "Eastern Europe"}, {"id": 21285, "dom": "G", "ent": "Baie du Tombeau", "cls": "Melanesia"}, {"id": 21286, "dom": "G", "ent": "Baie du Tombeau", "cls": "Micronesia"}, {"id": 21287, "dom": "G", "ent": "Baie du Tombeau", "cls": "Middle Africa"}, {"id": 21288, "dom": "G", "ent": "Baie du Tombeau", "cls": "Northern Africa"}, {"id": 21289, "dom": "G", "ent": "Baie du Tombeau", "cls": "Northern America"}, {"id": 21290, "dom": "G", "ent": "Baie du Tombeau", "cls": "Northern Europe"}, {"id": 21291, "dom": "G", "ent": "Baie du Tombeau", "cls": "Polynesia"}, {"id": 21292, "dom": "G", "ent": "Baie du Tombeau", "cls": "South America"}, {"id": 21293, "dom": "G", "ent": "Baie du Tombeau", "cls": "South-eastern Asia"}, {"id": 21294, "dom": "G", "ent": "Baie du Tombeau", "cls": "Southern Africa"}, {"id": 21295, "dom": "G", "ent": "Baie du Tombeau", "cls": "Southern Asia"}, {"id": 21296, "dom": "G", "ent": "Baie du Tombeau", "cls": "Southern Europe"}, {"id": 21297, "dom": "G", "ent": "Baie du Tombeau", "cls": "Western Africa"}, {"id": 21298, "dom": "G", "ent": "Baie du Tombeau", "cls": "Western Asia"}, {"id": 21299, "dom": "G", "ent": "Baie du Tombeau", "cls": "Western Europe"}, {"id": 21300, "dom": "G", "ent": "Salima", "cls": "Australia and New Zealand"}, {"id": 21301, "dom": "G", "ent": "Salima", "cls": "Caribbean"}, {"id": 21302, "dom": "G", "ent": "Salima", "cls": "Central America"}, {"id": 21303, "dom": "G", "ent": "Salima", "cls": "Central Asia"}, {"id": 21304, "dom": "G", "ent": "Salima", "cls": "Eastern Africa"}, {"id": 21305, "dom": "G", "ent": "Salima", "cls": "Eastern Asia"}, {"id": 21306, "dom": "G", "ent": "Salima", "cls": "Eastern Europe"}, {"id": 21307, "dom": "G", "ent": "Salima", "cls": "Melanesia"}, {"id": 21308, "dom": "G", "ent": "Salima", "cls": "Micronesia"}, {"id": 21309, "dom": "G", "ent": "Salima", "cls": "Middle Africa"}, {"id": 21310, "dom": "G", "ent": "Salima", "cls": "Northern Africa"}, {"id": 21311, "dom": "G", "ent": "Salima", "cls": "Northern America"}, {"id": 21312, "dom": "G", "ent": "Salima", "cls": "Northern Europe"}, {"id": 21313, "dom": "G", "ent": "Salima", "cls": "Polynesia"}, {"id": 21314, "dom": "G", "ent": "Salima", "cls": "South America"}, {"id": 21315, "dom": "G", "ent": "Salima", "cls": "South-eastern Asia"}, {"id": 21316, "dom": "G", "ent": "Salima", "cls": "Southern Africa"}, {"id": 21317, "dom": "G", "ent": "Salima", "cls": "Southern Asia"}, {"id": 21318, "dom": "G", "ent": "Salima", "cls": "Southern Europe"}, {"id": 21319, "dom": "G", "ent": "Salima", "cls": "Western Africa"}, {"id": 21320, "dom": "G", "ent": "Salima", "cls": "Western Asia"}, {"id": 21321, "dom": "G", "ent": "Salima", "cls": "Western Europe"}, {"id": 21322, "dom": "G", "ent": "Liwonde", "cls": "Australia and New Zealand"}, {"id": 21323, "dom": "G", "ent": "Liwonde", "cls": "Caribbean"}, {"id": 21324, "dom": "G", "ent": "Liwonde", "cls": "Central America"}, {"id": 21325, "dom": "G", "ent": "Liwonde", "cls": "Central Asia"}, {"id": 21326, "dom": "G", "ent": "Liwonde", "cls": "Eastern Africa"}, {"id": 21327, "dom": "G", "ent": "Liwonde", "cls": "Eastern Asia"}, {"id": 21328, "dom": "G", "ent": "Liwonde", "cls": "Eastern Europe"}, {"id": 21329, "dom": "G", "ent": "Liwonde", "cls": "Melanesia"}, {"id": 21330, "dom": "G", "ent": "Liwonde", "cls": "Micronesia"}, {"id": 21331, "dom": "G", "ent": "Liwonde", "cls": "Middle Africa"}, {"id": 21332, "dom": "G", "ent": "Liwonde", "cls": "Northern Africa"}, {"id": 21333, "dom": "G", "ent": "Liwonde", "cls": "Northern America"}, {"id": 21334, "dom": "G", "ent": "Liwonde", "cls": "Northern Europe"}, {"id": 21335, "dom": "G", "ent": "Liwonde", "cls": "Polynesia"}, {"id": 21336, "dom": "G", "ent": "Liwonde", "cls": "South America"}, {"id": 21337, "dom": "G", "ent": "Liwonde", "cls": "South-eastern Asia"}, {"id": 21338, "dom": "G", "ent": "Liwonde", "cls": "Southern Africa"}, {"id": 21339, "dom": "G", "ent": "Liwonde", "cls": "Southern Asia"}, {"id": 21340, "dom": "G", "ent": "Liwonde", "cls": "Southern Europe"}, {"id": 21341, "dom": "G", "ent": "Liwonde", "cls": "Western Africa"}, {"id": 21342, "dom": "G", "ent": "Liwonde", "cls": "Western Asia"}, {"id": 21343, "dom": "G", "ent": "Liwonde", "cls": "Western Europe"}, {"id": 21344, "dom": "G", "ent": "Ntcheu", "cls": "Australia and New Zealand"}, {"id": 21345, "dom": "G", "ent": "Ntcheu", "cls": "Caribbean"}, {"id": 21346, "dom": "G", "ent": "Ntcheu", "cls": "Central America"}, {"id": 21347, "dom": "G", "ent": "Ntcheu", "cls": "Central Asia"}, {"id": 21348, "dom": "G", "ent": "Ntcheu", "cls": "Eastern Africa"}, {"id": 21349, "dom": "G", "ent": "Ntcheu", "cls": "Eastern Asia"}, {"id": 21350, "dom": "G", "ent": "Ntcheu", "cls": "Eastern Europe"}, {"id": 21351, "dom": "G", "ent": "Ntcheu", "cls": "Melanesia"}, {"id": 21352, "dom": "G", "ent": "Ntcheu", "cls": "Micronesia"}, {"id": 21353, "dom": "G", "ent": "Ntcheu", "cls": "Middle Africa"}, {"id": 21354, "dom": "G", "ent": "Ntcheu", "cls": "Northern Africa"}, {"id": 21355, "dom": "G", "ent": "Ntcheu", "cls": "Northern America"}, {"id": 21356, "dom": "G", "ent": "Ntcheu", "cls": "Northern Europe"}, {"id": 21357, "dom": "G", "ent": "Ntcheu", "cls": "Polynesia"}, {"id": 21358, "dom": "G", "ent": "Ntcheu", "cls": "South America"}, {"id": 21359, "dom": "G", "ent": "Ntcheu", "cls": "South-eastern Asia"}, {"id": 21360, "dom": "G", "ent": "Ntcheu", "cls": "Southern Africa"}, {"id": 21361, "dom": "G", "ent": "Ntcheu", "cls": "Southern Asia"}, {"id": 21362, "dom": "G", "ent": "Ntcheu", "cls": "Southern Europe"}, {"id": 21363, "dom": "G", "ent": "Ntcheu", "cls": "Western Africa"}, {"id": 21364, "dom": "G", "ent": "Ntcheu", "cls": "Western Asia"}, {"id": 21365, "dom": "G", "ent": "Ntcheu", "cls": "Western Europe"}, {"id": 21366, "dom": "G", "ent": "Monkey Bay", "cls": "Australia and New Zealand"}, {"id": 21367, "dom": "G", "ent": "Monkey Bay", "cls": "Caribbean"}, {"id": 21368, "dom": "G", "ent": "Monkey Bay", "cls": "Central America"}, {"id": 21369, "dom": "G", "ent": "Monkey Bay", "cls": "Central Asia"}, {"id": 21370, "dom": "G", "ent": "Monkey Bay", "cls": "Eastern Africa"}, {"id": 21371, "dom": "G", "ent": "Monkey Bay", "cls": "Eastern Asia"}, {"id": 21372, "dom": "G", "ent": "Monkey Bay", "cls": "Eastern Europe"}, {"id": 21373, "dom": "G", "ent": "Monkey Bay", "cls": "Melanesia"}, {"id": 21374, "dom": "G", "ent": "Monkey Bay", "cls": "Micronesia"}, {"id": 21375, "dom": "G", "ent": "Monkey Bay", "cls": "Middle Africa"}, {"id": 21376, "dom": "G", "ent": "Monkey Bay", "cls": "Northern Africa"}, {"id": 21377, "dom": "G", "ent": "Monkey Bay", "cls": "Northern America"}, {"id": 21378, "dom": "G", "ent": "Monkey Bay", "cls": "Northern Europe"}, {"id": 21379, "dom": "G", "ent": "Monkey Bay", "cls": "Polynesia"}, {"id": 21380, "dom": "G", "ent": "Monkey Bay", "cls": "South America"}, {"id": 21381, "dom": "G", "ent": "Monkey Bay", "cls": "South-eastern Asia"}, {"id": 21382, "dom": "G", "ent": "Monkey Bay", "cls": "Southern Africa"}, {"id": 21383, "dom": "G", "ent": "Monkey Bay", "cls": "Southern Asia"}, {"id": 21384, "dom": "G", "ent": "Monkey Bay", "cls": "Southern Europe"}, {"id": 21385, "dom": "G", "ent": "Monkey Bay", "cls": "Western Africa"}, {"id": 21386, "dom": "G", "ent": "Monkey Bay", "cls": "Western Asia"}, {"id": 21387, "dom": "G", "ent": "Monkey Bay", "cls": "Western Europe"}, {"id": 21388, "dom": "G", "ent": "Agua Prieta", "cls": "Australia and New Zealand"}, {"id": 21389, "dom": "G", "ent": "Agua Prieta", "cls": "Caribbean"}, {"id": 21390, "dom": "G", "ent": "Agua Prieta", "cls": "Central America"}, {"id": 21391, "dom": "G", "ent": "Agua Prieta", "cls": "Central Asia"}, {"id": 21392, "dom": "G", "ent": "Agua Prieta", "cls": "Eastern Africa"}, {"id": 21393, "dom": "G", "ent": "Agua Prieta", "cls": "Eastern Asia"}, {"id": 21394, "dom": "G", "ent": "Agua Prieta", "cls": "Eastern Europe"}, {"id": 21395, "dom": "G", "ent": "Agua Prieta", "cls": "Melanesia"}, {"id": 21396, "dom": "G", "ent": "Agua Prieta", "cls": "Micronesia"}, {"id": 21397, "dom": "G", "ent": "Agua Prieta", "cls": "Middle Africa"}, {"id": 21398, "dom": "G", "ent": "Agua Prieta", "cls": "Northern Africa"}, {"id": 21399, "dom": "G", "ent": "Agua Prieta", "cls": "Northern America"}, {"id": 21400, "dom": "G", "ent": "Agua Prieta", "cls": "Northern Europe"}, {"id": 21401, "dom": "G", "ent": "Agua Prieta", "cls": "Polynesia"}, {"id": 21402, "dom": "G", "ent": "Agua Prieta", "cls": "South America"}, {"id": 21403, "dom": "G", "ent": "Agua Prieta", "cls": "South-eastern Asia"}, {"id": 21404, "dom": "G", "ent": "Agua Prieta", "cls": "Southern Africa"}, {"id": 21405, "dom": "G", "ent": "Agua Prieta", "cls": "Southern Asia"}, {"id": 21406, "dom": "G", "ent": "Agua Prieta", "cls": "Southern Europe"}, {"id": 21407, "dom": "G", "ent": "Agua Prieta", "cls": "Western Africa"}, {"id": 21408, "dom": "G", "ent": "Agua Prieta", "cls": "Western Asia"}, {"id": 21409, "dom": "G", "ent": "Agua Prieta", "cls": "Western Europe"}, {"id": 21410, "dom": "G", "ent": "La Barca", "cls": "Australia and New Zealand"}, {"id": 21411, "dom": "G", "ent": "La Barca", "cls": "Caribbean"}, {"id": 21412, "dom": "G", "ent": "La Barca", "cls": "Central America"}, {"id": 21413, "dom": "G", "ent": "La Barca", "cls": "Central Asia"}, {"id": 21414, "dom": "G", "ent": "La Barca", "cls": "Eastern Africa"}, {"id": 21415, "dom": "G", "ent": "La Barca", "cls": "Eastern Asia"}, {"id": 21416, "dom": "G", "ent": "La Barca", "cls": "Eastern Europe"}, {"id": 21417, "dom": "G", "ent": "La Barca", "cls": "Melanesia"}, {"id": 21418, "dom": "G", "ent": "La Barca", "cls": "Micronesia"}, {"id": 21419, "dom": "G", "ent": "La Barca", "cls": "Middle Africa"}, {"id": 21420, "dom": "G", "ent": "La Barca", "cls": "Northern Africa"}, {"id": 21421, "dom": "G", "ent": "La Barca", "cls": "Northern America"}, {"id": 21422, "dom": "G", "ent": "La Barca", "cls": "Northern Europe"}, {"id": 21423, "dom": "G", "ent": "La Barca", "cls": "Polynesia"}, {"id": 21424, "dom": "G", "ent": "La Barca", "cls": "South America"}, {"id": 21425, "dom": "G", "ent": "La Barca", "cls": "South-eastern Asia"}, {"id": 21426, "dom": "G", "ent": "La Barca", "cls": "Southern Africa"}, {"id": 21427, "dom": "G", "ent": "La Barca", "cls": "Southern Asia"}, {"id": 21428, "dom": "G", "ent": "La Barca", "cls": "Southern Europe"}, {"id": 21429, "dom": "G", "ent": "La Barca", "cls": "Western Africa"}, {"id": 21430, "dom": "G", "ent": "La Barca", "cls": "Western Asia"}, {"id": 21431, "dom": "G", "ent": "La Barca", "cls": "Western Europe"}, {"id": 21432, "dom": "G", "ent": "Zacoalco de Torres", "cls": "Australia and New Zealand"}, {"id": 21433, "dom": "G", "ent": "Zacoalco de Torres", "cls": "Caribbean"}, {"id": 21434, "dom": "G", "ent": "Zacoalco de Torres", "cls": "Central America"}, {"id": 21435, "dom": "G", "ent": "Zacoalco de Torres", "cls": "Central Asia"}, {"id": 21436, "dom": "G", "ent": "Zacoalco de Torres", "cls": "Eastern Africa"}, {"id": 21437, "dom": "G", "ent": "Zacoalco de Torres", "cls": "Eastern Asia"}, {"id": 21438, "dom": "G", "ent": "Zacoalco de Torres", "cls": "Eastern Europe"}, {"id": 21439, "dom": "G", "ent": "Zacoalco de Torres", "cls": "Melanesia"}, {"id": 21440, "dom": "G", "ent": "Zacoalco de Torres", "cls": "Micronesia"}, {"id": 21441, "dom": "G", "ent": "Zacoalco de Torres", "cls": "Middle Africa"}, {"id": 21442, "dom": "G", "ent": "Zacoalco de Torres", "cls": "Northern Africa"}, {"id": 21443, "dom": "G", "ent": "Zacoalco de Torres", "cls": "Northern America"}, {"id": 21444, "dom": "G", "ent": "Zacoalco de Torres", "cls": "Northern Europe"}, {"id": 21445, "dom": "G", "ent": "Zacoalco de Torres", "cls": "Polynesia"}, {"id": 21446, "dom": "G", "ent": "Zacoalco de Torres", "cls": "South America"}, {"id": 21447, "dom": "G", "ent": "Zacoalco de Torres", "cls": "South-eastern Asia"}, {"id": 21448, "dom": "G", "ent": "Zacoalco de Torres", "cls": "Southern Africa"}, {"id": 21449, "dom": "G", "ent": "Zacoalco de Torres", "cls": "Southern Asia"}, {"id": 21450, "dom": "G", "ent": "Zacoalco de Torres", "cls": "Southern Europe"}, {"id": 21451, "dom": "G", "ent": "Zacoalco de Torres", "cls": "Western Africa"}, {"id": 21452, "dom": "G", "ent": "Zacoalco de Torres", "cls": "Western Asia"}, {"id": 21453, "dom": "G", "ent": "Zacoalco de Torres", "cls": "Western Europe"}, {"id": 21454, "dom": "G", "ent": "Libres", "cls": "Australia and New Zealand"}, {"id": 21455, "dom": "G", "ent": "Libres", "cls": "Caribbean"}, {"id": 21456, "dom": "G", "ent": "Libres", "cls": "Central America"}, {"id": 21457, "dom": "G", "ent": "Libres", "cls": "Central Asia"}, {"id": 21458, "dom": "G", "ent": "Libres", "cls": "Eastern Africa"}, {"id": 21459, "dom": "G", "ent": "Libres", "cls": "Eastern Asia"}, {"id": 21460, "dom": "G", "ent": "Libres", "cls": "Eastern Europe"}, {"id": 21461, "dom": "G", "ent": "Libres", "cls": "Melanesia"}, {"id": 21462, "dom": "G", "ent": "Libres", "cls": "Micronesia"}, {"id": 21463, "dom": "G", "ent": "Libres", "cls": "Middle Africa"}, {"id": 21464, "dom": "G", "ent": "Libres", "cls": "Northern Africa"}, {"id": 21465, "dom": "G", "ent": "Libres", "cls": "Northern America"}, {"id": 21466, "dom": "G", "ent": "Libres", "cls": "Northern Europe"}, {"id": 21467, "dom": "G", "ent": "Libres", "cls": "Polynesia"}, {"id": 21468, "dom": "G", "ent": "Libres", "cls": "South America"}, {"id": 21469, "dom": "G", "ent": "Libres", "cls": "South-eastern Asia"}, {"id": 21470, "dom": "G", "ent": "Libres", "cls": "Southern Africa"}, {"id": 21471, "dom": "G", "ent": "Libres", "cls": "Southern Asia"}, {"id": 21472, "dom": "G", "ent": "Libres", "cls": "Southern Europe"}, {"id": 21473, "dom": "G", "ent": "Libres", "cls": "Western Africa"}, {"id": 21474, "dom": "G", "ent": "Libres", "cls": "Western Asia"}, {"id": 21475, "dom": "G", "ent": "Libres", "cls": "Western Europe"}, {"id": 21476, "dom": "G", "ent": "Hulu Langat", "cls": "Australia and New Zealand"}, {"id": 21477, "dom": "G", "ent": "Hulu Langat", "cls": "Caribbean"}, {"id": 21478, "dom": "G", "ent": "Hulu Langat", "cls": "Central America"}, {"id": 21479, "dom": "G", "ent": "Hulu Langat", "cls": "Central Asia"}, {"id": 21480, "dom": "G", "ent": "Hulu Langat", "cls": "Eastern Africa"}, {"id": 21481, "dom": "G", "ent": "Hulu Langat", "cls": "Eastern Asia"}, {"id": 21482, "dom": "G", "ent": "Hulu Langat", "cls": "Eastern Europe"}, {"id": 21483, "dom": "G", "ent": "Hulu Langat", "cls": "Melanesia"}, {"id": 21484, "dom": "G", "ent": "Hulu Langat", "cls": "Micronesia"}, {"id": 21485, "dom": "G", "ent": "Hulu Langat", "cls": "Middle Africa"}, {"id": 21486, "dom": "G", "ent": "Hulu Langat", "cls": "Northern Africa"}, {"id": 21487, "dom": "G", "ent": "Hulu Langat", "cls": "Northern America"}, {"id": 21488, "dom": "G", "ent": "Hulu Langat", "cls": "Northern Europe"}, {"id": 21489, "dom": "G", "ent": "Hulu Langat", "cls": "Polynesia"}, {"id": 21490, "dom": "G", "ent": "Hulu Langat", "cls": "South America"}, {"id": 21491, "dom": "G", "ent": "Hulu Langat", "cls": "South-eastern Asia"}, {"id": 21492, "dom": "G", "ent": "Hulu Langat", "cls": "Southern Africa"}, {"id": 21493, "dom": "G", "ent": "Hulu Langat", "cls": "Southern Asia"}, {"id": 21494, "dom": "G", "ent": "Hulu Langat", "cls": "Southern Europe"}, {"id": 21495, "dom": "G", "ent": "Hulu Langat", "cls": "Western Africa"}, {"id": 21496, "dom": "G", "ent": "Hulu Langat", "cls": "Western Asia"}, {"id": 21497, "dom": "G", "ent": "Hulu Langat", "cls": "Western Europe"}, {"id": 21498, "dom": "G", "ent": "Dungun", "cls": "Australia and New Zealand"}, {"id": 21499, "dom": "G", "ent": "Dungun", "cls": "Caribbean"}, {"id": 21500, "dom": "G", "ent": "Dungun", "cls": "Central America"}, {"id": 21501, "dom": "G", "ent": "Dungun", "cls": "Central Asia"}, {"id": 21502, "dom": "G", "ent": "Dungun", "cls": "Eastern Africa"}, {"id": 21503, "dom": "G", "ent": "Dungun", "cls": "Eastern Asia"}, {"id": 21504, "dom": "G", "ent": "Dungun", "cls": "Eastern Europe"}, {"id": 21505, "dom": "G", "ent": "Dungun", "cls": "Melanesia"}, {"id": 21506, "dom": "G", "ent": "Dungun", "cls": "Micronesia"}, {"id": 21507, "dom": "G", "ent": "Dungun", "cls": "Middle Africa"}, {"id": 21508, "dom": "G", "ent": "Dungun", "cls": "Northern Africa"}, {"id": 21509, "dom": "G", "ent": "Dungun", "cls": "Northern America"}, {"id": 21510, "dom": "G", "ent": "Dungun", "cls": "Northern Europe"}, {"id": 21511, "dom": "G", "ent": "Dungun", "cls": "Polynesia"}, {"id": 21512, "dom": "G", "ent": "Dungun", "cls": "South America"}, {"id": 21513, "dom": "G", "ent": "Dungun", "cls": "South-eastern Asia"}, {"id": 21514, "dom": "G", "ent": "Dungun", "cls": "Southern Africa"}, {"id": 21515, "dom": "G", "ent": "Dungun", "cls": "Southern Asia"}, {"id": 21516, "dom": "G", "ent": "Dungun", "cls": "Southern Europe"}, {"id": 21517, "dom": "G", "ent": "Dungun", "cls": "Western Africa"}, {"id": 21518, "dom": "G", "ent": "Dungun", "cls": "Western Asia"}, {"id": 21519, "dom": "G", "ent": "Dungun", "cls": "Western Europe"}, {"id": 21520, "dom": "G", "ent": "Sungkai", "cls": "Australia and New Zealand"}, {"id": 21521, "dom": "G", "ent": "Sungkai", "cls": "Caribbean"}, {"id": 21522, "dom": "G", "ent": "Sungkai", "cls": "Central America"}, {"id": 21523, "dom": "G", "ent": "Sungkai", "cls": "Central Asia"}, {"id": 21524, "dom": "G", "ent": "Sungkai", "cls": "Eastern Africa"}, {"id": 21525, "dom": "G", "ent": "Sungkai", "cls": "Eastern Asia"}, {"id": 21526, "dom": "G", "ent": "Sungkai", "cls": "Eastern Europe"}, {"id": 21527, "dom": "G", "ent": "Sungkai", "cls": "Melanesia"}, {"id": 21528, "dom": "G", "ent": "Sungkai", "cls": "Micronesia"}, {"id": 21529, "dom": "G", "ent": "Sungkai", "cls": "Middle Africa"}, {"id": 21530, "dom": "G", "ent": "Sungkai", "cls": "Northern Africa"}, {"id": 21531, "dom": "G", "ent": "Sungkai", "cls": "Northern America"}, {"id": 21532, "dom": "G", "ent": "Sungkai", "cls": "Northern Europe"}, {"id": 21533, "dom": "G", "ent": "Sungkai", "cls": "Polynesia"}, {"id": 21534, "dom": "G", "ent": "Sungkai", "cls": "South America"}, {"id": 21535, "dom": "G", "ent": "Sungkai", "cls": "South-eastern Asia"}, {"id": 21536, "dom": "G", "ent": "Sungkai", "cls": "Southern Africa"}, {"id": 21537, "dom": "G", "ent": "Sungkai", "cls": "Southern Asia"}, {"id": 21538, "dom": "G", "ent": "Sungkai", "cls": "Southern Europe"}, {"id": 21539, "dom": "G", "ent": "Sungkai", "cls": "Western Africa"}, {"id": 21540, "dom": "G", "ent": "Sungkai", "cls": "Western Asia"}, {"id": 21541, "dom": "G", "ent": "Sungkai", "cls": "Western Europe"}, {"id": 21542, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "Australia and New Zealand"}, {"id": 21543, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "Caribbean"}, {"id": 21544, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "Central America"}, {"id": 21545, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "Central Asia"}, {"id": 21546, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "Eastern Africa"}, {"id": 21547, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "Eastern Asia"}, {"id": 21548, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "Eastern Europe"}, {"id": 21549, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "Melanesia"}, {"id": 21550, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "Micronesia"}, {"id": 21551, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "Middle Africa"}, {"id": 21552, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "Northern Africa"}, {"id": 21553, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "Northern America"}, {"id": 21554, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "Northern Europe"}, {"id": 21555, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "Polynesia"}, {"id": 21556, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "South America"}, {"id": 21557, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "South-eastern Asia"}, {"id": 21558, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "Southern Africa"}, {"id": 21559, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "Southern Asia"}, {"id": 21560, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "Southern Europe"}, {"id": 21561, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "Western Africa"}, {"id": 21562, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "Western Asia"}, {"id": 21563, "dom": "G", "ent": "Kampung Tanjung Karang", "cls": "Western Europe"}, {"id": 21564, "dom": "G", "ent": "Monapo", "cls": "Australia and New Zealand"}, {"id": 21565, "dom": "G", "ent": "Monapo", "cls": "Caribbean"}, {"id": 21566, "dom": "G", "ent": "Monapo", "cls": "Central America"}, {"id": 21567, "dom": "G", "ent": "Monapo", "cls": "Central Asia"}, {"id": 21568, "dom": "G", "ent": "Monapo", "cls": "Eastern Africa"}, {"id": 21569, "dom": "G", "ent": "Monapo", "cls": "Eastern Asia"}, {"id": 21570, "dom": "G", "ent": "Monapo", "cls": "Eastern Europe"}, {"id": 21571, "dom": "G", "ent": "Monapo", "cls": "Melanesia"}, {"id": 21572, "dom": "G", "ent": "Monapo", "cls": "Micronesia"}, {"id": 21573, "dom": "G", "ent": "Monapo", "cls": "Middle Africa"}, {"id": 21574, "dom": "G", "ent": "Monapo", "cls": "Northern Africa"}, {"id": 21575, "dom": "G", "ent": "Monapo", "cls": "Northern America"}, {"id": 21576, "dom": "G", "ent": "Monapo", "cls": "Northern Europe"}, {"id": 21577, "dom": "G", "ent": "Monapo", "cls": "Polynesia"}, {"id": 21578, "dom": "G", "ent": "Monapo", "cls": "South America"}, {"id": 21579, "dom": "G", "ent": "Monapo", "cls": "South-eastern Asia"}, {"id": 21580, "dom": "G", "ent": "Monapo", "cls": "Southern Africa"}, {"id": 21581, "dom": "G", "ent": "Monapo", "cls": "Southern Asia"}, {"id": 21582, "dom": "G", "ent": "Monapo", "cls": "Southern Europe"}, {"id": 21583, "dom": "G", "ent": "Monapo", "cls": "Western Africa"}, {"id": 21584, "dom": "G", "ent": "Monapo", "cls": "Western Asia"}, {"id": 21585, "dom": "G", "ent": "Monapo", "cls": "Western Europe"}, {"id": 21586, "dom": "G", "ent": "Gondola", "cls": "Australia and New Zealand"}, {"id": 21587, "dom": "G", "ent": "Gondola", "cls": "Caribbean"}, {"id": 21588, "dom": "G", "ent": "Gondola", "cls": "Central America"}, {"id": 21589, "dom": "G", "ent": "Gondola", "cls": "Central Asia"}, {"id": 21590, "dom": "G", "ent": "Gondola", "cls": "Eastern Africa"}, {"id": 21591, "dom": "G", "ent": "Gondola", "cls": "Eastern Asia"}, {"id": 21592, "dom": "G", "ent": "Gondola", "cls": "Eastern Europe"}, {"id": 21593, "dom": "G", "ent": "Gondola", "cls": "Melanesia"}, {"id": 21594, "dom": "G", "ent": "Gondola", "cls": "Micronesia"}, {"id": 21595, "dom": "G", "ent": "Gondola", "cls": "Middle Africa"}, {"id": 21596, "dom": "G", "ent": "Gondola", "cls": "Northern Africa"}, {"id": 21597, "dom": "G", "ent": "Gondola", "cls": "Northern America"}, {"id": 21598, "dom": "G", "ent": "Gondola", "cls": "Northern Europe"}, {"id": 21599, "dom": "G", "ent": "Gondola", "cls": "Polynesia"}, {"id": 21600, "dom": "G", "ent": "Gondola", "cls": "South America"}, {"id": 21601, "dom": "G", "ent": "Gondola", "cls": "South-eastern Asia"}, {"id": 21602, "dom": "G", "ent": "Gondola", "cls": "Southern Africa"}, {"id": 21603, "dom": "G", "ent": "Gondola", "cls": "Southern Asia"}, {"id": 21604, "dom": "G", "ent": "Gondola", "cls": "Southern Europe"}, {"id": 21605, "dom": "G", "ent": "Gondola", "cls": "Western Africa"}, {"id": 21606, "dom": "G", "ent": "Gondola", "cls": "Western Asia"}, {"id": 21607, "dom": "G", "ent": "Gondola", "cls": "Western Europe"}, {"id": 21608, "dom": "G", "ent": "Namacurra", "cls": "Australia and New Zealand"}, {"id": 21609, "dom": "G", "ent": "Namacurra", "cls": "Caribbean"}, {"id": 21610, "dom": "G", "ent": "Namacurra", "cls": "Central America"}, {"id": 21611, "dom": "G", "ent": "Namacurra", "cls": "Central Asia"}, {"id": 21612, "dom": "G", "ent": "Namacurra", "cls": "Eastern Africa"}, {"id": 21613, "dom": "G", "ent": "Namacurra", "cls": "Eastern Asia"}, {"id": 21614, "dom": "G", "ent": "Namacurra", "cls": "Eastern Europe"}, {"id": 21615, "dom": "G", "ent": "Namacurra", "cls": "Melanesia"}, {"id": 21616, "dom": "G", "ent": "Namacurra", "cls": "Micronesia"}, {"id": 21617, "dom": "G", "ent": "Namacurra", "cls": "Middle Africa"}, {"id": 21618, "dom": "G", "ent": "Namacurra", "cls": "Northern Africa"}, {"id": 21619, "dom": "G", "ent": "Namacurra", "cls": "Northern America"}, {"id": 21620, "dom": "G", "ent": "Namacurra", "cls": "Northern Europe"}, {"id": 21621, "dom": "G", "ent": "Namacurra", "cls": "Polynesia"}, {"id": 21622, "dom": "G", "ent": "Namacurra", "cls": "South America"}, {"id": 21623, "dom": "G", "ent": "Namacurra", "cls": "South-eastern Asia"}, {"id": 21624, "dom": "G", "ent": "Namacurra", "cls": "Southern Africa"}, {"id": 21625, "dom": "G", "ent": "Namacurra", "cls": "Southern Asia"}, {"id": 21626, "dom": "G", "ent": "Namacurra", "cls": "Southern Europe"}, {"id": 21627, "dom": "G", "ent": "Namacurra", "cls": "Western Africa"}, {"id": 21628, "dom": "G", "ent": "Namacurra", "cls": "Western Asia"}, {"id": 21629, "dom": "G", "ent": "Namacurra", "cls": "Western Europe"}, {"id": 21630, "dom": "G", "ent": "Pebane", "cls": "Australia and New Zealand"}, {"id": 21631, "dom": "G", "ent": "Pebane", "cls": "Caribbean"}, {"id": 21632, "dom": "G", "ent": "Pebane", "cls": "Central America"}, {"id": 21633, "dom": "G", "ent": "Pebane", "cls": "Central Asia"}, {"id": 21634, "dom": "G", "ent": "Pebane", "cls": "Eastern Africa"}, {"id": 21635, "dom": "G", "ent": "Pebane", "cls": "Eastern Asia"}, {"id": 21636, "dom": "G", "ent": "Pebane", "cls": "Eastern Europe"}, {"id": 21637, "dom": "G", "ent": "Pebane", "cls": "Melanesia"}, {"id": 21638, "dom": "G", "ent": "Pebane", "cls": "Micronesia"}, {"id": 21639, "dom": "G", "ent": "Pebane", "cls": "Middle Africa"}, {"id": 21640, "dom": "G", "ent": "Pebane", "cls": "Northern Africa"}, {"id": 21641, "dom": "G", "ent": "Pebane", "cls": "Northern America"}, {"id": 21642, "dom": "G", "ent": "Pebane", "cls": "Northern Europe"}, {"id": 21643, "dom": "G", "ent": "Pebane", "cls": "Polynesia"}, {"id": 21644, "dom": "G", "ent": "Pebane", "cls": "South America"}, {"id": 21645, "dom": "G", "ent": "Pebane", "cls": "South-eastern Asia"}, {"id": 21646, "dom": "G", "ent": "Pebane", "cls": "Southern Africa"}, {"id": 21647, "dom": "G", "ent": "Pebane", "cls": "Southern Asia"}, {"id": 21648, "dom": "G", "ent": "Pebane", "cls": "Southern Europe"}, {"id": 21649, "dom": "G", "ent": "Pebane", "cls": "Western Africa"}, {"id": 21650, "dom": "G", "ent": "Pebane", "cls": "Western Asia"}, {"id": 21651, "dom": "G", "ent": "Pebane", "cls": "Western Europe"}, {"id": 21652, "dom": "G", "ent": "Rundu", "cls": "Australia and New Zealand"}, {"id": 21653, "dom": "G", "ent": "Rundu", "cls": "Caribbean"}, {"id": 21654, "dom": "G", "ent": "Rundu", "cls": "Central America"}, {"id": 21655, "dom": "G", "ent": "Rundu", "cls": "Central Asia"}, {"id": 21656, "dom": "G", "ent": "Rundu", "cls": "Eastern Africa"}, {"id": 21657, "dom": "G", "ent": "Rundu", "cls": "Eastern Asia"}, {"id": 21658, "dom": "G", "ent": "Rundu", "cls": "Eastern Europe"}, {"id": 21659, "dom": "G", "ent": "Rundu", "cls": "Melanesia"}, {"id": 21660, "dom": "G", "ent": "Rundu", "cls": "Micronesia"}, {"id": 21661, "dom": "G", "ent": "Rundu", "cls": "Middle Africa"}, {"id": 21662, "dom": "G", "ent": "Rundu", "cls": "Northern Africa"}, {"id": 21663, "dom": "G", "ent": "Rundu", "cls": "Northern America"}, {"id": 21664, "dom": "G", "ent": "Rundu", "cls": "Northern Europe"}, {"id": 21665, "dom": "G", "ent": "Rundu", "cls": "Polynesia"}, {"id": 21666, "dom": "G", "ent": "Rundu", "cls": "South America"}, {"id": 21667, "dom": "G", "ent": "Rundu", "cls": "South-eastern Asia"}, {"id": 21668, "dom": "G", "ent": "Rundu", "cls": "Southern Africa"}, {"id": 21669, "dom": "G", "ent": "Rundu", "cls": "Southern Asia"}, {"id": 21670, "dom": "G", "ent": "Rundu", "cls": "Southern Europe"}, {"id": 21671, "dom": "G", "ent": "Rundu", "cls": "Western Africa"}, {"id": 21672, "dom": "G", "ent": "Rundu", "cls": "Western Asia"}, {"id": 21673, "dom": "G", "ent": "Rundu", "cls": "Western Europe"}, {"id": 21674, "dom": "G", "ent": "Swakopmund", "cls": "Australia and New Zealand"}, {"id": 21675, "dom": "G", "ent": "Swakopmund", "cls": "Caribbean"}, {"id": 21676, "dom": "G", "ent": "Swakopmund", "cls": "Central America"}, {"id": 21677, "dom": "G", "ent": "Swakopmund", "cls": "Central Asia"}, {"id": 21678, "dom": "G", "ent": "Swakopmund", "cls": "Eastern Africa"}, {"id": 21679, "dom": "G", "ent": "Swakopmund", "cls": "Eastern Asia"}, {"id": 21680, "dom": "G", "ent": "Swakopmund", "cls": "Eastern Europe"}, {"id": 21681, "dom": "G", "ent": "Swakopmund", "cls": "Melanesia"}, {"id": 21682, "dom": "G", "ent": "Swakopmund", "cls": "Micronesia"}, {"id": 21683, "dom": "G", "ent": "Swakopmund", "cls": "Middle Africa"}, {"id": 21684, "dom": "G", "ent": "Swakopmund", "cls": "Northern Africa"}, {"id": 21685, "dom": "G", "ent": "Swakopmund", "cls": "Northern America"}, {"id": 21686, "dom": "G", "ent": "Swakopmund", "cls": "Northern Europe"}, {"id": 21687, "dom": "G", "ent": "Swakopmund", "cls": "Polynesia"}, {"id": 21688, "dom": "G", "ent": "Swakopmund", "cls": "South America"}, {"id": 21689, "dom": "G", "ent": "Swakopmund", "cls": "South-eastern Asia"}, {"id": 21690, "dom": "G", "ent": "Swakopmund", "cls": "Southern Africa"}, {"id": 21691, "dom": "G", "ent": "Swakopmund", "cls": "Southern Asia"}, {"id": 21692, "dom": "G", "ent": "Swakopmund", "cls": "Southern Europe"}, {"id": 21693, "dom": "G", "ent": "Swakopmund", "cls": "Western Africa"}, {"id": 21694, "dom": "G", "ent": "Swakopmund", "cls": "Western Asia"}, {"id": 21695, "dom": "G", "ent": "Swakopmund", "cls": "Western Europe"}, {"id": 21696, "dom": "G", "ent": "Oshakati", "cls": "Australia and New Zealand"}, {"id": 21697, "dom": "G", "ent": "Oshakati", "cls": "Caribbean"}, {"id": 21698, "dom": "G", "ent": "Oshakati", "cls": "Central America"}, {"id": 21699, "dom": "G", "ent": "Oshakati", "cls": "Central Asia"}, {"id": 21700, "dom": "G", "ent": "Oshakati", "cls": "Eastern Africa"}, {"id": 21701, "dom": "G", "ent": "Oshakati", "cls": "Eastern Asia"}, {"id": 21702, "dom": "G", "ent": "Oshakati", "cls": "Eastern Europe"}, {"id": 21703, "dom": "G", "ent": "Oshakati", "cls": "Melanesia"}, {"id": 21704, "dom": "G", "ent": "Oshakati", "cls": "Micronesia"}, {"id": 21705, "dom": "G", "ent": "Oshakati", "cls": "Middle Africa"}, {"id": 21706, "dom": "G", "ent": "Oshakati", "cls": "Northern Africa"}, {"id": 21707, "dom": "G", "ent": "Oshakati", "cls": "Northern America"}, {"id": 21708, "dom": "G", "ent": "Oshakati", "cls": "Northern Europe"}, {"id": 21709, "dom": "G", "ent": "Oshakati", "cls": "Polynesia"}, {"id": 21710, "dom": "G", "ent": "Oshakati", "cls": "South America"}, {"id": 21711, "dom": "G", "ent": "Oshakati", "cls": "South-eastern Asia"}, {"id": 21712, "dom": "G", "ent": "Oshakati", "cls": "Southern Africa"}, {"id": 21713, "dom": "G", "ent": "Oshakati", "cls": "Southern Asia"}, {"id": 21714, "dom": "G", "ent": "Oshakati", "cls": "Southern Europe"}, {"id": 21715, "dom": "G", "ent": "Oshakati", "cls": "Western Africa"}, {"id": 21716, "dom": "G", "ent": "Oshakati", "cls": "Western Asia"}, {"id": 21717, "dom": "G", "ent": "Oshakati", "cls": "Western Europe"}, {"id": 21718, "dom": "G", "ent": "Ondangwa", "cls": "Australia and New Zealand"}, {"id": 21719, "dom": "G", "ent": "Ondangwa", "cls": "Caribbean"}, {"id": 21720, "dom": "G", "ent": "Ondangwa", "cls": "Central America"}, {"id": 21721, "dom": "G", "ent": "Ondangwa", "cls": "Central Asia"}, {"id": 21722, "dom": "G", "ent": "Ondangwa", "cls": "Eastern Africa"}, {"id": 21723, "dom": "G", "ent": "Ondangwa", "cls": "Eastern Asia"}, {"id": 21724, "dom": "G", "ent": "Ondangwa", "cls": "Eastern Europe"}, {"id": 21725, "dom": "G", "ent": "Ondangwa", "cls": "Melanesia"}, {"id": 21726, "dom": "G", "ent": "Ondangwa", "cls": "Micronesia"}, {"id": 21727, "dom": "G", "ent": "Ondangwa", "cls": "Middle Africa"}, {"id": 21728, "dom": "G", "ent": "Ondangwa", "cls": "Northern Africa"}, {"id": 21729, "dom": "G", "ent": "Ondangwa", "cls": "Northern America"}, {"id": 21730, "dom": "G", "ent": "Ondangwa", "cls": "Northern Europe"}, {"id": 21731, "dom": "G", "ent": "Ondangwa", "cls": "Polynesia"}, {"id": 21732, "dom": "G", "ent": "Ondangwa", "cls": "South America"}, {"id": 21733, "dom": "G", "ent": "Ondangwa", "cls": "South-eastern Asia"}, {"id": 21734, "dom": "G", "ent": "Ondangwa", "cls": "Southern Africa"}, {"id": 21735, "dom": "G", "ent": "Ondangwa", "cls": "Southern Asia"}, {"id": 21736, "dom": "G", "ent": "Ondangwa", "cls": "Southern Europe"}, {"id": 21737, "dom": "G", "ent": "Ondangwa", "cls": "Western Africa"}, {"id": 21738, "dom": "G", "ent": "Ondangwa", "cls": "Western Asia"}, {"id": 21739, "dom": "G", "ent": "Ondangwa", "cls": "Western Europe"}, {"id": 21740, "dom": "G", "ent": "Agadez", "cls": "Australia and New Zealand"}, {"id": 21741, "dom": "G", "ent": "Agadez", "cls": "Caribbean"}, {"id": 21742, "dom": "G", "ent": "Agadez", "cls": "Central America"}, {"id": 21743, "dom": "G", "ent": "Agadez", "cls": "Central Asia"}, {"id": 21744, "dom": "G", "ent": "Agadez", "cls": "Eastern Africa"}, {"id": 21745, "dom": "G", "ent": "Agadez", "cls": "Eastern Asia"}, {"id": 21746, "dom": "G", "ent": "Agadez", "cls": "Eastern Europe"}, {"id": 21747, "dom": "G", "ent": "Agadez", "cls": "Melanesia"}, {"id": 21748, "dom": "G", "ent": "Agadez", "cls": "Micronesia"}, {"id": 21749, "dom": "G", "ent": "Agadez", "cls": "Middle Africa"}, {"id": 21750, "dom": "G", "ent": "Agadez", "cls": "Northern Africa"}, {"id": 21751, "dom": "G", "ent": "Agadez", "cls": "Northern America"}, {"id": 21752, "dom": "G", "ent": "Agadez", "cls": "Northern Europe"}, {"id": 21753, "dom": "G", "ent": "Agadez", "cls": "Polynesia"}, {"id": 21754, "dom": "G", "ent": "Agadez", "cls": "South America"}, {"id": 21755, "dom": "G", "ent": "Agadez", "cls": "South-eastern Asia"}, {"id": 21756, "dom": "G", "ent": "Agadez", "cls": "Southern Africa"}, {"id": 21757, "dom": "G", "ent": "Agadez", "cls": "Southern Asia"}, {"id": 21758, "dom": "G", "ent": "Agadez", "cls": "Southern Europe"}, {"id": 21759, "dom": "G", "ent": "Agadez", "cls": "Western Africa"}, {"id": 21760, "dom": "G", "ent": "Agadez", "cls": "Western Asia"}, {"id": 21761, "dom": "G", "ent": "Agadez", "cls": "Western Europe"}, {"id": 21762, "dom": "G", "ent": "Nguigmi", "cls": "Australia and New Zealand"}, {"id": 21763, "dom": "G", "ent": "Nguigmi", "cls": "Caribbean"}, {"id": 21764, "dom": "G", "ent": "Nguigmi", "cls": "Central America"}, {"id": 21765, "dom": "G", "ent": "Nguigmi", "cls": "Central Asia"}, {"id": 21766, "dom": "G", "ent": "Nguigmi", "cls": "Eastern Africa"}, {"id": 21767, "dom": "G", "ent": "Nguigmi", "cls": "Eastern Asia"}, {"id": 21768, "dom": "G", "ent": "Nguigmi", "cls": "Eastern Europe"}, {"id": 21769, "dom": "G", "ent": "Nguigmi", "cls": "Melanesia"}, {"id": 21770, "dom": "G", "ent": "Nguigmi", "cls": "Micronesia"}, {"id": 21771, "dom": "G", "ent": "Nguigmi", "cls": "Middle Africa"}, {"id": 21772, "dom": "G", "ent": "Nguigmi", "cls": "Northern Africa"}, {"id": 21773, "dom": "G", "ent": "Nguigmi", "cls": "Northern America"}, {"id": 21774, "dom": "G", "ent": "Nguigmi", "cls": "Northern Europe"}, {"id": 21775, "dom": "G", "ent": "Nguigmi", "cls": "Polynesia"}, {"id": 21776, "dom": "G", "ent": "Nguigmi", "cls": "South America"}, {"id": 21777, "dom": "G", "ent": "Nguigmi", "cls": "South-eastern Asia"}, {"id": 21778, "dom": "G", "ent": "Nguigmi", "cls": "Southern Africa"}, {"id": 21779, "dom": "G", "ent": "Nguigmi", "cls": "Southern Asia"}, {"id": 21780, "dom": "G", "ent": "Nguigmi", "cls": "Southern Europe"}, {"id": 21781, "dom": "G", "ent": "Nguigmi", "cls": "Western Africa"}, {"id": 21782, "dom": "G", "ent": "Nguigmi", "cls": "Western Asia"}, {"id": 21783, "dom": "G", "ent": "Nguigmi", "cls": "Western Europe"}, {"id": 21784, "dom": "G", "ent": "Mayahi", "cls": "Australia and New Zealand"}, {"id": 21785, "dom": "G", "ent": "Mayahi", "cls": "Caribbean"}, {"id": 21786, "dom": "G", "ent": "Mayahi", "cls": "Central America"}, {"id": 21787, "dom": "G", "ent": "Mayahi", "cls": "Central Asia"}, {"id": 21788, "dom": "G", "ent": "Mayahi", "cls": "Eastern Africa"}, {"id": 21789, "dom": "G", "ent": "Mayahi", "cls": "Eastern Asia"}, {"id": 21790, "dom": "G", "ent": "Mayahi", "cls": "Eastern Europe"}, {"id": 21791, "dom": "G", "ent": "Mayahi", "cls": "Melanesia"}, {"id": 21792, "dom": "G", "ent": "Mayahi", "cls": "Micronesia"}, {"id": 21793, "dom": "G", "ent": "Mayahi", "cls": "Middle Africa"}, {"id": 21794, "dom": "G", "ent": "Mayahi", "cls": "Northern Africa"}, {"id": 21795, "dom": "G", "ent": "Mayahi", "cls": "Northern America"}, {"id": 21796, "dom": "G", "ent": "Mayahi", "cls": "Northern Europe"}, {"id": 21797, "dom": "G", "ent": "Mayahi", "cls": "Polynesia"}, {"id": 21798, "dom": "G", "ent": "Mayahi", "cls": "South America"}, {"id": 21799, "dom": "G", "ent": "Mayahi", "cls": "South-eastern Asia"}, {"id": 21800, "dom": "G", "ent": "Mayahi", "cls": "Southern Africa"}, {"id": 21801, "dom": "G", "ent": "Mayahi", "cls": "Southern Asia"}, {"id": 21802, "dom": "G", "ent": "Mayahi", "cls": "Southern Europe"}, {"id": 21803, "dom": "G", "ent": "Mayahi", "cls": "Western Africa"}, {"id": 21804, "dom": "G", "ent": "Mayahi", "cls": "Western Asia"}, {"id": 21805, "dom": "G", "ent": "Mayahi", "cls": "Western Europe"}, {"id": 21806, "dom": "G", "ent": "Torodi", "cls": "Australia and New Zealand"}, {"id": 21807, "dom": "G", "ent": "Torodi", "cls": "Caribbean"}, {"id": 21808, "dom": "G", "ent": "Torodi", "cls": "Central America"}, {"id": 21809, "dom": "G", "ent": "Torodi", "cls": "Central Asia"}, {"id": 21810, "dom": "G", "ent": "Torodi", "cls": "Eastern Africa"}, {"id": 21811, "dom": "G", "ent": "Torodi", "cls": "Eastern Asia"}, {"id": 21812, "dom": "G", "ent": "Torodi", "cls": "Eastern Europe"}, {"id": 21813, "dom": "G", "ent": "Torodi", "cls": "Melanesia"}, {"id": 21814, "dom": "G", "ent": "Torodi", "cls": "Micronesia"}, {"id": 21815, "dom": "G", "ent": "Torodi", "cls": "Middle Africa"}, {"id": 21816, "dom": "G", "ent": "Torodi", "cls": "Northern Africa"}, {"id": 21817, "dom": "G", "ent": "Torodi", "cls": "Northern America"}, {"id": 21818, "dom": "G", "ent": "Torodi", "cls": "Northern Europe"}, {"id": 21819, "dom": "G", "ent": "Torodi", "cls": "Polynesia"}, {"id": 21820, "dom": "G", "ent": "Torodi", "cls": "South America"}, {"id": 21821, "dom": "G", "ent": "Torodi", "cls": "South-eastern Asia"}, {"id": 21822, "dom": "G", "ent": "Torodi", "cls": "Southern Africa"}, {"id": 21823, "dom": "G", "ent": "Torodi", "cls": "Southern Asia"}, {"id": 21824, "dom": "G", "ent": "Torodi", "cls": "Southern Europe"}, {"id": 21825, "dom": "G", "ent": "Torodi", "cls": "Western Africa"}, {"id": 21826, "dom": "G", "ent": "Torodi", "cls": "Western Asia"}, {"id": 21827, "dom": "G", "ent": "Torodi", "cls": "Western Europe"}, {"id": 21828, "dom": "G", "ent": "Abeokuta", "cls": "Australia and New Zealand"}, {"id": 21829, "dom": "G", "ent": "Abeokuta", "cls": "Caribbean"}, {"id": 21830, "dom": "G", "ent": "Abeokuta", "cls": "Central America"}, {"id": 21831, "dom": "G", "ent": "Abeokuta", "cls": "Central Asia"}, {"id": 21832, "dom": "G", "ent": "Abeokuta", "cls": "Eastern Africa"}, {"id": 21833, "dom": "G", "ent": "Abeokuta", "cls": "Eastern Asia"}, {"id": 21834, "dom": "G", "ent": "Abeokuta", "cls": "Eastern Europe"}, {"id": 21835, "dom": "G", "ent": "Abeokuta", "cls": "Melanesia"}, {"id": 21836, "dom": "G", "ent": "Abeokuta", "cls": "Micronesia"}, {"id": 21837, "dom": "G", "ent": "Abeokuta", "cls": "Middle Africa"}, {"id": 21838, "dom": "G", "ent": "Abeokuta", "cls": "Northern Africa"}, {"id": 21839, "dom": "G", "ent": "Abeokuta", "cls": "Northern America"}, {"id": 21840, "dom": "G", "ent": "Abeokuta", "cls": "Northern Europe"}, {"id": 21841, "dom": "G", "ent": "Abeokuta", "cls": "Polynesia"}, {"id": 21842, "dom": "G", "ent": "Abeokuta", "cls": "South America"}, {"id": 21843, "dom": "G", "ent": "Abeokuta", "cls": "South-eastern Asia"}, {"id": 21844, "dom": "G", "ent": "Abeokuta", "cls": "Southern Africa"}, {"id": 21845, "dom": "G", "ent": "Abeokuta", "cls": "Southern Asia"}, {"id": 21846, "dom": "G", "ent": "Abeokuta", "cls": "Southern Europe"}, {"id": 21847, "dom": "G", "ent": "Abeokuta", "cls": "Western Africa"}, {"id": 21848, "dom": "G", "ent": "Abeokuta", "cls": "Western Asia"}, {"id": 21849, "dom": "G", "ent": "Abeokuta", "cls": "Western Europe"}, {"id": 21850, "dom": "G", "ent": "Agege", "cls": "Australia and New Zealand"}, {"id": 21851, "dom": "G", "ent": "Agege", "cls": "Caribbean"}, {"id": 21852, "dom": "G", "ent": "Agege", "cls": "Central America"}, {"id": 21853, "dom": "G", "ent": "Agege", "cls": "Central Asia"}, {"id": 21854, "dom": "G", "ent": "Agege", "cls": "Eastern Africa"}, {"id": 21855, "dom": "G", "ent": "Agege", "cls": "Eastern Asia"}, {"id": 21856, "dom": "G", "ent": "Agege", "cls": "Eastern Europe"}, {"id": 21857, "dom": "G", "ent": "Agege", "cls": "Melanesia"}, {"id": 21858, "dom": "G", "ent": "Agege", "cls": "Micronesia"}, {"id": 21859, "dom": "G", "ent": "Agege", "cls": "Middle Africa"}, {"id": 21860, "dom": "G", "ent": "Agege", "cls": "Northern Africa"}, {"id": 21861, "dom": "G", "ent": "Agege", "cls": "Northern America"}, {"id": 21862, "dom": "G", "ent": "Agege", "cls": "Northern Europe"}, {"id": 21863, "dom": "G", "ent": "Agege", "cls": "Polynesia"}, {"id": 21864, "dom": "G", "ent": "Agege", "cls": "South America"}, {"id": 21865, "dom": "G", "ent": "Agege", "cls": "South-eastern Asia"}, {"id": 21866, "dom": "G", "ent": "Agege", "cls": "Southern Africa"}, {"id": 21867, "dom": "G", "ent": "Agege", "cls": "Southern Asia"}, {"id": 21868, "dom": "G", "ent": "Agege", "cls": "Southern Europe"}, {"id": 21869, "dom": "G", "ent": "Agege", "cls": "Western Africa"}, {"id": 21870, "dom": "G", "ent": "Agege", "cls": "Western Asia"}, {"id": 21871, "dom": "G", "ent": "Agege", "cls": "Western Europe"}, {"id": 21872, "dom": "G", "ent": "Nguru", "cls": "Australia and New Zealand"}, {"id": 21873, "dom": "G", "ent": "Nguru", "cls": "Caribbean"}, {"id": 21874, "dom": "G", "ent": "Nguru", "cls": "Central America"}, {"id": 21875, "dom": "G", "ent": "Nguru", "cls": "Central Asia"}, {"id": 21876, "dom": "G", "ent": "Nguru", "cls": "Eastern Africa"}, {"id": 21877, "dom": "G", "ent": "Nguru", "cls": "Eastern Asia"}, {"id": 21878, "dom": "G", "ent": "Nguru", "cls": "Eastern Europe"}, {"id": 21879, "dom": "G", "ent": "Nguru", "cls": "Melanesia"}, {"id": 21880, "dom": "G", "ent": "Nguru", "cls": "Micronesia"}, {"id": 21881, "dom": "G", "ent": "Nguru", "cls": "Middle Africa"}, {"id": 21882, "dom": "G", "ent": "Nguru", "cls": "Northern Africa"}, {"id": 21883, "dom": "G", "ent": "Nguru", "cls": "Northern America"}, {"id": 21884, "dom": "G", "ent": "Nguru", "cls": "Northern Europe"}, {"id": 21885, "dom": "G", "ent": "Nguru", "cls": "Polynesia"}, {"id": 21886, "dom": "G", "ent": "Nguru", "cls": "South America"}, {"id": 21887, "dom": "G", "ent": "Nguru", "cls": "South-eastern Asia"}, {"id": 21888, "dom": "G", "ent": "Nguru", "cls": "Southern Africa"}, {"id": 21889, "dom": "G", "ent": "Nguru", "cls": "Southern Asia"}, {"id": 21890, "dom": "G", "ent": "Nguru", "cls": "Southern Europe"}, {"id": 21891, "dom": "G", "ent": "Nguru", "cls": "Western Africa"}, {"id": 21892, "dom": "G", "ent": "Nguru", "cls": "Western Asia"}, {"id": 21893, "dom": "G", "ent": "Nguru", "cls": "Western Europe"}, {"id": 21894, "dom": "G", "ent": "Ipoti", "cls": "Australia and New Zealand"}, {"id": 21895, "dom": "G", "ent": "Ipoti", "cls": "Caribbean"}, {"id": 21896, "dom": "G", "ent": "Ipoti", "cls": "Central America"}, {"id": 21897, "dom": "G", "ent": "Ipoti", "cls": "Central Asia"}, {"id": 21898, "dom": "G", "ent": "Ipoti", "cls": "Eastern Africa"}, {"id": 21899, "dom": "G", "ent": "Ipoti", "cls": "Eastern Asia"}, {"id": 21900, "dom": "G", "ent": "Ipoti", "cls": "Eastern Europe"}, {"id": 21901, "dom": "G", "ent": "Ipoti", "cls": "Melanesia"}, {"id": 21902, "dom": "G", "ent": "Ipoti", "cls": "Micronesia"}, {"id": 21903, "dom": "G", "ent": "Ipoti", "cls": "Middle Africa"}, {"id": 21904, "dom": "G", "ent": "Ipoti", "cls": "Northern Africa"}, {"id": 21905, "dom": "G", "ent": "Ipoti", "cls": "Northern America"}, {"id": 21906, "dom": "G", "ent": "Ipoti", "cls": "Northern Europe"}, {"id": 21907, "dom": "G", "ent": "Ipoti", "cls": "Polynesia"}, {"id": 21908, "dom": "G", "ent": "Ipoti", "cls": "South America"}, {"id": 21909, "dom": "G", "ent": "Ipoti", "cls": "South-eastern Asia"}, {"id": 21910, "dom": "G", "ent": "Ipoti", "cls": "Southern Africa"}, {"id": 21911, "dom": "G", "ent": "Ipoti", "cls": "Southern Asia"}, {"id": 21912, "dom": "G", "ent": "Ipoti", "cls": "Southern Europe"}, {"id": 21913, "dom": "G", "ent": "Ipoti", "cls": "Western Africa"}, {"id": 21914, "dom": "G", "ent": "Ipoti", "cls": "Western Asia"}, {"id": 21915, "dom": "G", "ent": "Ipoti", "cls": "Western Europe"}, {"id": 21916, "dom": "G", "ent": "Chinandega", "cls": "Australia and New Zealand"}, {"id": 21917, "dom": "G", "ent": "Chinandega", "cls": "Caribbean"}, {"id": 21918, "dom": "G", "ent": "Chinandega", "cls": "Central America"}, {"id": 21919, "dom": "G", "ent": "Chinandega", "cls": "Central Asia"}, {"id": 21920, "dom": "G", "ent": "Chinandega", "cls": "Eastern Africa"}, {"id": 21921, "dom": "G", "ent": "Chinandega", "cls": "Eastern Asia"}, {"id": 21922, "dom": "G", "ent": "Chinandega", "cls": "Eastern Europe"}, {"id": 21923, "dom": "G", "ent": "Chinandega", "cls": "Melanesia"}, {"id": 21924, "dom": "G", "ent": "Chinandega", "cls": "Micronesia"}, {"id": 21925, "dom": "G", "ent": "Chinandega", "cls": "Middle Africa"}, {"id": 21926, "dom": "G", "ent": "Chinandega", "cls": "Northern Africa"}, {"id": 21927, "dom": "G", "ent": "Chinandega", "cls": "Northern America"}, {"id": 21928, "dom": "G", "ent": "Chinandega", "cls": "Northern Europe"}, {"id": 21929, "dom": "G", "ent": "Chinandega", "cls": "Polynesia"}, {"id": 21930, "dom": "G", "ent": "Chinandega", "cls": "South America"}, {"id": 21931, "dom": "G", "ent": "Chinandega", "cls": "South-eastern Asia"}, {"id": 21932, "dom": "G", "ent": "Chinandega", "cls": "Southern Africa"}, {"id": 21933, "dom": "G", "ent": "Chinandega", "cls": "Southern Asia"}, {"id": 21934, "dom": "G", "ent": "Chinandega", "cls": "Southern Europe"}, {"id": 21935, "dom": "G", "ent": "Chinandega", "cls": "Western Africa"}, {"id": 21936, "dom": "G", "ent": "Chinandega", "cls": "Western Asia"}, {"id": 21937, "dom": "G", "ent": "Chinandega", "cls": "Western Europe"}, {"id": 21938, "dom": "G", "ent": "Matagalpa", "cls": "Australia and New Zealand"}, {"id": 21939, "dom": "G", "ent": "Matagalpa", "cls": "Caribbean"}, {"id": 21940, "dom": "G", "ent": "Matagalpa", "cls": "Central America"}, {"id": 21941, "dom": "G", "ent": "Matagalpa", "cls": "Central Asia"}, {"id": 21942, "dom": "G", "ent": "Matagalpa", "cls": "Eastern Africa"}, {"id": 21943, "dom": "G", "ent": "Matagalpa", "cls": "Eastern Asia"}, {"id": 21944, "dom": "G", "ent": "Matagalpa", "cls": "Eastern Europe"}, {"id": 21945, "dom": "G", "ent": "Matagalpa", "cls": "Melanesia"}, {"id": 21946, "dom": "G", "ent": "Matagalpa", "cls": "Micronesia"}, {"id": 21947, "dom": "G", "ent": "Matagalpa", "cls": "Middle Africa"}, {"id": 21948, "dom": "G", "ent": "Matagalpa", "cls": "Northern Africa"}, {"id": 21949, "dom": "G", "ent": "Matagalpa", "cls": "Northern America"}, {"id": 21950, "dom": "G", "ent": "Matagalpa", "cls": "Northern Europe"}, {"id": 21951, "dom": "G", "ent": "Matagalpa", "cls": "Polynesia"}, {"id": 21952, "dom": "G", "ent": "Matagalpa", "cls": "South America"}, {"id": 21953, "dom": "G", "ent": "Matagalpa", "cls": "South-eastern Asia"}, {"id": 21954, "dom": "G", "ent": "Matagalpa", "cls": "Southern Africa"}, {"id": 21955, "dom": "G", "ent": "Matagalpa", "cls": "Southern Asia"}, {"id": 21956, "dom": "G", "ent": "Matagalpa", "cls": "Southern Europe"}, {"id": 21957, "dom": "G", "ent": "Matagalpa", "cls": "Western Africa"}, {"id": 21958, "dom": "G", "ent": "Matagalpa", "cls": "Western Asia"}, {"id": 21959, "dom": "G", "ent": "Matagalpa", "cls": "Western Europe"}, {"id": 21960, "dom": "G", "ent": "Mateare", "cls": "Australia and New Zealand"}, {"id": 21961, "dom": "G", "ent": "Mateare", "cls": "Caribbean"}, {"id": 21962, "dom": "G", "ent": "Mateare", "cls": "Central America"}, {"id": 21963, "dom": "G", "ent": "Mateare", "cls": "Central Asia"}, {"id": 21964, "dom": "G", "ent": "Mateare", "cls": "Eastern Africa"}, {"id": 21965, "dom": "G", "ent": "Mateare", "cls": "Eastern Asia"}, {"id": 21966, "dom": "G", "ent": "Mateare", "cls": "Eastern Europe"}, {"id": 21967, "dom": "G", "ent": "Mateare", "cls": "Melanesia"}, {"id": 21968, "dom": "G", "ent": "Mateare", "cls": "Micronesia"}, {"id": 21969, "dom": "G", "ent": "Mateare", "cls": "Middle Africa"}, {"id": 21970, "dom": "G", "ent": "Mateare", "cls": "Northern Africa"}, {"id": 21971, "dom": "G", "ent": "Mateare", "cls": "Northern America"}, {"id": 21972, "dom": "G", "ent": "Mateare", "cls": "Northern Europe"}, {"id": 21973, "dom": "G", "ent": "Mateare", "cls": "Polynesia"}, {"id": 21974, "dom": "G", "ent": "Mateare", "cls": "South America"}, {"id": 21975, "dom": "G", "ent": "Mateare", "cls": "South-eastern Asia"}, {"id": 21976, "dom": "G", "ent": "Mateare", "cls": "Southern Africa"}, {"id": 21977, "dom": "G", "ent": "Mateare", "cls": "Southern Asia"}, {"id": 21978, "dom": "G", "ent": "Mateare", "cls": "Southern Europe"}, {"id": 21979, "dom": "G", "ent": "Mateare", "cls": "Western Africa"}, {"id": 21980, "dom": "G", "ent": "Mateare", "cls": "Western Asia"}, {"id": 21981, "dom": "G", "ent": "Mateare", "cls": "Western Europe"}, {"id": 21982, "dom": "G", "ent": "Juigalpa", "cls": "Australia and New Zealand"}, {"id": 21983, "dom": "G", "ent": "Juigalpa", "cls": "Caribbean"}, {"id": 21984, "dom": "G", "ent": "Juigalpa", "cls": "Central America"}, {"id": 21985, "dom": "G", "ent": "Juigalpa", "cls": "Central Asia"}, {"id": 21986, "dom": "G", "ent": "Juigalpa", "cls": "Eastern Africa"}, {"id": 21987, "dom": "G", "ent": "Juigalpa", "cls": "Eastern Asia"}, {"id": 21988, "dom": "G", "ent": "Juigalpa", "cls": "Eastern Europe"}, {"id": 21989, "dom": "G", "ent": "Juigalpa", "cls": "Melanesia"}, {"id": 21990, "dom": "G", "ent": "Juigalpa", "cls": "Micronesia"}, {"id": 21991, "dom": "G", "ent": "Juigalpa", "cls": "Middle Africa"}, {"id": 21992, "dom": "G", "ent": "Juigalpa", "cls": "Northern Africa"}, {"id": 21993, "dom": "G", "ent": "Juigalpa", "cls": "Northern America"}, {"id": 21994, "dom": "G", "ent": "Juigalpa", "cls": "Northern Europe"}, {"id": 21995, "dom": "G", "ent": "Juigalpa", "cls": "Polynesia"}, {"id": 21996, "dom": "G", "ent": "Juigalpa", "cls": "South America"}, {"id": 21997, "dom": "G", "ent": "Juigalpa", "cls": "South-eastern Asia"}, {"id": 21998, "dom": "G", "ent": "Juigalpa", "cls": "Southern Africa"}, {"id": 21999, "dom": "G", "ent": "Juigalpa", "cls": "Southern Asia"}, {"id": 22000, "dom": "G", "ent": "Juigalpa", "cls": "Southern Europe"}, {"id": 22001, "dom": "G", "ent": "Juigalpa", "cls": "Western Africa"}, {"id": 22002, "dom": "G", "ent": "Juigalpa", "cls": "Western Asia"}, {"id": 22003, "dom": "G", "ent": "Juigalpa", "cls": "Western Europe"}, {"id": 22004, "dom": "G", "ent": "Nieuwegein", "cls": "Australia and New Zealand"}, {"id": 22005, "dom": "G", "ent": "Nieuwegein", "cls": "Caribbean"}, {"id": 22006, "dom": "G", "ent": "Nieuwegein", "cls": "Central America"}, {"id": 22007, "dom": "G", "ent": "Nieuwegein", "cls": "Central Asia"}, {"id": 22008, "dom": "G", "ent": "Nieuwegein", "cls": "Eastern Africa"}, {"id": 22009, "dom": "G", "ent": "Nieuwegein", "cls": "Eastern Asia"}, {"id": 22010, "dom": "G", "ent": "Nieuwegein", "cls": "Eastern Europe"}, {"id": 22011, "dom": "G", "ent": "Nieuwegein", "cls": "Melanesia"}, {"id": 22012, "dom": "G", "ent": "Nieuwegein", "cls": "Micronesia"}, {"id": 22013, "dom": "G", "ent": "Nieuwegein", "cls": "Middle Africa"}, {"id": 22014, "dom": "G", "ent": "Nieuwegein", "cls": "Northern Africa"}, {"id": 22015, "dom": "G", "ent": "Nieuwegein", "cls": "Northern America"}, {"id": 22016, "dom": "G", "ent": "Nieuwegein", "cls": "Northern Europe"}, {"id": 22017, "dom": "G", "ent": "Nieuwegein", "cls": "Polynesia"}, {"id": 22018, "dom": "G", "ent": "Nieuwegein", "cls": "South America"}, {"id": 22019, "dom": "G", "ent": "Nieuwegein", "cls": "South-eastern Asia"}, {"id": 22020, "dom": "G", "ent": "Nieuwegein", "cls": "Southern Africa"}, {"id": 22021, "dom": "G", "ent": "Nieuwegein", "cls": "Southern Asia"}, {"id": 22022, "dom": "G", "ent": "Nieuwegein", "cls": "Southern Europe"}, {"id": 22023, "dom": "G", "ent": "Nieuwegein", "cls": "Western Africa"}, {"id": 22024, "dom": "G", "ent": "Nieuwegein", "cls": "Western Asia"}, {"id": 22025, "dom": "G", "ent": "Nieuwegein", "cls": "Western Europe"}, {"id": 22026, "dom": "G", "ent": "Hoogeveen", "cls": "Australia and New Zealand"}, {"id": 22027, "dom": "G", "ent": "Hoogeveen", "cls": "Caribbean"}, {"id": 22028, "dom": "G", "ent": "Hoogeveen", "cls": "Central America"}, {"id": 22029, "dom": "G", "ent": "Hoogeveen", "cls": "Central Asia"}, {"id": 22030, "dom": "G", "ent": "Hoogeveen", "cls": "Eastern Africa"}, {"id": 22031, "dom": "G", "ent": "Hoogeveen", "cls": "Eastern Asia"}, {"id": 22032, "dom": "G", "ent": "Hoogeveen", "cls": "Eastern Europe"}, {"id": 22033, "dom": "G", "ent": "Hoogeveen", "cls": "Melanesia"}, {"id": 22034, "dom": "G", "ent": "Hoogeveen", "cls": "Micronesia"}, {"id": 22035, "dom": "G", "ent": "Hoogeveen", "cls": "Middle Africa"}, {"id": 22036, "dom": "G", "ent": "Hoogeveen", "cls": "Northern Africa"}, {"id": 22037, "dom": "G", "ent": "Hoogeveen", "cls": "Northern America"}, {"id": 22038, "dom": "G", "ent": "Hoogeveen", "cls": "Northern Europe"}, {"id": 22039, "dom": "G", "ent": "Hoogeveen", "cls": "Polynesia"}, {"id": 22040, "dom": "G", "ent": "Hoogeveen", "cls": "South America"}, {"id": 22041, "dom": "G", "ent": "Hoogeveen", "cls": "South-eastern Asia"}, {"id": 22042, "dom": "G", "ent": "Hoogeveen", "cls": "Southern Africa"}, {"id": 22043, "dom": "G", "ent": "Hoogeveen", "cls": "Southern Asia"}, {"id": 22044, "dom": "G", "ent": "Hoogeveen", "cls": "Southern Europe"}, {"id": 22045, "dom": "G", "ent": "Hoogeveen", "cls": "Western Africa"}, {"id": 22046, "dom": "G", "ent": "Hoogeveen", "cls": "Western Asia"}, {"id": 22047, "dom": "G", "ent": "Hoogeveen", "cls": "Western Europe"}, {"id": 22048, "dom": "G", "ent": "Ommoord", "cls": "Australia and New Zealand"}, {"id": 22049, "dom": "G", "ent": "Ommoord", "cls": "Caribbean"}, {"id": 22050, "dom": "G", "ent": "Ommoord", "cls": "Central America"}, {"id": 22051, "dom": "G", "ent": "Ommoord", "cls": "Central Asia"}, {"id": 22052, "dom": "G", "ent": "Ommoord", "cls": "Eastern Africa"}, {"id": 22053, "dom": "G", "ent": "Ommoord", "cls": "Eastern Asia"}, {"id": 22054, "dom": "G", "ent": "Ommoord", "cls": "Eastern Europe"}, {"id": 22055, "dom": "G", "ent": "Ommoord", "cls": "Melanesia"}, {"id": 22056, "dom": "G", "ent": "Ommoord", "cls": "Micronesia"}, {"id": 22057, "dom": "G", "ent": "Ommoord", "cls": "Middle Africa"}, {"id": 22058, "dom": "G", "ent": "Ommoord", "cls": "Northern Africa"}, {"id": 22059, "dom": "G", "ent": "Ommoord", "cls": "Northern America"}, {"id": 22060, "dom": "G", "ent": "Ommoord", "cls": "Northern Europe"}, {"id": 22061, "dom": "G", "ent": "Ommoord", "cls": "Polynesia"}, {"id": 22062, "dom": "G", "ent": "Ommoord", "cls": "South America"}, {"id": 22063, "dom": "G", "ent": "Ommoord", "cls": "South-eastern Asia"}, {"id": 22064, "dom": "G", "ent": "Ommoord", "cls": "Southern Africa"}, {"id": 22065, "dom": "G", "ent": "Ommoord", "cls": "Southern Asia"}, {"id": 22066, "dom": "G", "ent": "Ommoord", "cls": "Southern Europe"}, {"id": 22067, "dom": "G", "ent": "Ommoord", "cls": "Western Africa"}, {"id": 22068, "dom": "G", "ent": "Ommoord", "cls": "Western Asia"}, {"id": 22069, "dom": "G", "ent": "Ommoord", "cls": "Western Europe"}, {"id": 22070, "dom": "G", "ent": "Zeewolde", "cls": "Australia and New Zealand"}, {"id": 22071, "dom": "G", "ent": "Zeewolde", "cls": "Caribbean"}, {"id": 22072, "dom": "G", "ent": "Zeewolde", "cls": "Central America"}, {"id": 22073, "dom": "G", "ent": "Zeewolde", "cls": "Central Asia"}, {"id": 22074, "dom": "G", "ent": "Zeewolde", "cls": "Eastern Africa"}, {"id": 22075, "dom": "G", "ent": "Zeewolde", "cls": "Eastern Asia"}, {"id": 22076, "dom": "G", "ent": "Zeewolde", "cls": "Eastern Europe"}, {"id": 22077, "dom": "G", "ent": "Zeewolde", "cls": "Melanesia"}, {"id": 22078, "dom": "G", "ent": "Zeewolde", "cls": "Micronesia"}, {"id": 22079, "dom": "G", "ent": "Zeewolde", "cls": "Middle Africa"}, {"id": 22080, "dom": "G", "ent": "Zeewolde", "cls": "Northern Africa"}, {"id": 22081, "dom": "G", "ent": "Zeewolde", "cls": "Northern America"}, {"id": 22082, "dom": "G", "ent": "Zeewolde", "cls": "Northern Europe"}, {"id": 22083, "dom": "G", "ent": "Zeewolde", "cls": "Polynesia"}, {"id": 22084, "dom": "G", "ent": "Zeewolde", "cls": "South America"}, {"id": 22085, "dom": "G", "ent": "Zeewolde", "cls": "South-eastern Asia"}, {"id": 22086, "dom": "G", "ent": "Zeewolde", "cls": "Southern Africa"}, {"id": 22087, "dom": "G", "ent": "Zeewolde", "cls": "Southern Asia"}, {"id": 22088, "dom": "G", "ent": "Zeewolde", "cls": "Southern Europe"}, {"id": 22089, "dom": "G", "ent": "Zeewolde", "cls": "Western Africa"}, {"id": 22090, "dom": "G", "ent": "Zeewolde", "cls": "Western Asia"}, {"id": 22091, "dom": "G", "ent": "Zeewolde", "cls": "Western Europe"}, {"id": 22092, "dom": "G", "ent": "Nesttun", "cls": "Australia and New Zealand"}, {"id": 22093, "dom": "G", "ent": "Nesttun", "cls": "Caribbean"}, {"id": 22094, "dom": "G", "ent": "Nesttun", "cls": "Central America"}, {"id": 22095, "dom": "G", "ent": "Nesttun", "cls": "Central Asia"}, {"id": 22096, "dom": "G", "ent": "Nesttun", "cls": "Eastern Africa"}, {"id": 22097, "dom": "G", "ent": "Nesttun", "cls": "Eastern Asia"}, {"id": 22098, "dom": "G", "ent": "Nesttun", "cls": "Eastern Europe"}, {"id": 22099, "dom": "G", "ent": "Nesttun", "cls": "Melanesia"}, {"id": 22100, "dom": "G", "ent": "Nesttun", "cls": "Micronesia"}, {"id": 22101, "dom": "G", "ent": "Nesttun", "cls": "Middle Africa"}, {"id": 22102, "dom": "G", "ent": "Nesttun", "cls": "Northern Africa"}, {"id": 22103, "dom": "G", "ent": "Nesttun", "cls": "Northern America"}, {"id": 22104, "dom": "G", "ent": "Nesttun", "cls": "Northern Europe"}, {"id": 22105, "dom": "G", "ent": "Nesttun", "cls": "Polynesia"}, {"id": 22106, "dom": "G", "ent": "Nesttun", "cls": "South America"}, {"id": 22107, "dom": "G", "ent": "Nesttun", "cls": "South-eastern Asia"}, {"id": 22108, "dom": "G", "ent": "Nesttun", "cls": "Southern Africa"}, {"id": 22109, "dom": "G", "ent": "Nesttun", "cls": "Southern Asia"}, {"id": 22110, "dom": "G", "ent": "Nesttun", "cls": "Southern Europe"}, {"id": 22111, "dom": "G", "ent": "Nesttun", "cls": "Western Africa"}, {"id": 22112, "dom": "G", "ent": "Nesttun", "cls": "Western Asia"}, {"id": 22113, "dom": "G", "ent": "Nesttun", "cls": "Western Europe"}, {"id": 22114, "dom": "G", "ent": "Halden", "cls": "Australia and New Zealand"}, {"id": 22115, "dom": "G", "ent": "Halden", "cls": "Caribbean"}, {"id": 22116, "dom": "G", "ent": "Halden", "cls": "Central America"}, {"id": 22117, "dom": "G", "ent": "Halden", "cls": "Central Asia"}, {"id": 22118, "dom": "G", "ent": "Halden", "cls": "Eastern Africa"}, {"id": 22119, "dom": "G", "ent": "Halden", "cls": "Eastern Asia"}, {"id": 22120, "dom": "G", "ent": "Halden", "cls": "Eastern Europe"}, {"id": 22121, "dom": "G", "ent": "Halden", "cls": "Melanesia"}, {"id": 22122, "dom": "G", "ent": "Halden", "cls": "Micronesia"}, {"id": 22123, "dom": "G", "ent": "Halden", "cls": "Middle Africa"}, {"id": 22124, "dom": "G", "ent": "Halden", "cls": "Northern Africa"}, {"id": 22125, "dom": "G", "ent": "Halden", "cls": "Northern America"}, {"id": 22126, "dom": "G", "ent": "Halden", "cls": "Northern Europe"}, {"id": 22127, "dom": "G", "ent": "Halden", "cls": "Polynesia"}, {"id": 22128, "dom": "G", "ent": "Halden", "cls": "South America"}, {"id": 22129, "dom": "G", "ent": "Halden", "cls": "South-eastern Asia"}, {"id": 22130, "dom": "G", "ent": "Halden", "cls": "Southern Africa"}, {"id": 22131, "dom": "G", "ent": "Halden", "cls": "Southern Asia"}, {"id": 22132, "dom": "G", "ent": "Halden", "cls": "Southern Europe"}, {"id": 22133, "dom": "G", "ent": "Halden", "cls": "Western Africa"}, {"id": 22134, "dom": "G", "ent": "Halden", "cls": "Western Asia"}, {"id": 22135, "dom": "G", "ent": "Halden", "cls": "Western Europe"}, {"id": 22136, "dom": "G", "ent": "Harstad", "cls": "Australia and New Zealand"}, {"id": 22137, "dom": "G", "ent": "Harstad", "cls": "Caribbean"}, {"id": 22138, "dom": "G", "ent": "Harstad", "cls": "Central America"}, {"id": 22139, "dom": "G", "ent": "Harstad", "cls": "Central Asia"}, {"id": 22140, "dom": "G", "ent": "Harstad", "cls": "Eastern Africa"}, {"id": 22141, "dom": "G", "ent": "Harstad", "cls": "Eastern Asia"}, {"id": 22142, "dom": "G", "ent": "Harstad", "cls": "Eastern Europe"}, {"id": 22143, "dom": "G", "ent": "Harstad", "cls": "Melanesia"}, {"id": 22144, "dom": "G", "ent": "Harstad", "cls": "Micronesia"}, {"id": 22145, "dom": "G", "ent": "Harstad", "cls": "Middle Africa"}, {"id": 22146, "dom": "G", "ent": "Harstad", "cls": "Northern Africa"}, {"id": 22147, "dom": "G", "ent": "Harstad", "cls": "Northern America"}, {"id": 22148, "dom": "G", "ent": "Harstad", "cls": "Northern Europe"}, {"id": 22149, "dom": "G", "ent": "Harstad", "cls": "Polynesia"}, {"id": 22150, "dom": "G", "ent": "Harstad", "cls": "South America"}, {"id": 22151, "dom": "G", "ent": "Harstad", "cls": "South-eastern Asia"}, {"id": 22152, "dom": "G", "ent": "Harstad", "cls": "Southern Africa"}, {"id": 22153, "dom": "G", "ent": "Harstad", "cls": "Southern Asia"}, {"id": 22154, "dom": "G", "ent": "Harstad", "cls": "Southern Europe"}, {"id": 22155, "dom": "G", "ent": "Harstad", "cls": "Western Africa"}, {"id": 22156, "dom": "G", "ent": "Harstad", "cls": "Western Asia"}, {"id": 22157, "dom": "G", "ent": "Harstad", "cls": "Western Europe"}, {"id": 22158, "dom": "G", "ent": "Kongsberg", "cls": "Australia and New Zealand"}, {"id": 22159, "dom": "G", "ent": "Kongsberg", "cls": "Caribbean"}, {"id": 22160, "dom": "G", "ent": "Kongsberg", "cls": "Central America"}, {"id": 22161, "dom": "G", "ent": "Kongsberg", "cls": "Central Asia"}, {"id": 22162, "dom": "G", "ent": "Kongsberg", "cls": "Eastern Africa"}, {"id": 22163, "dom": "G", "ent": "Kongsberg", "cls": "Eastern Asia"}, {"id": 22164, "dom": "G", "ent": "Kongsberg", "cls": "Eastern Europe"}, {"id": 22165, "dom": "G", "ent": "Kongsberg", "cls": "Melanesia"}, {"id": 22166, "dom": "G", "ent": "Kongsberg", "cls": "Micronesia"}, {"id": 22167, "dom": "G", "ent": "Kongsberg", "cls": "Middle Africa"}, {"id": 22168, "dom": "G", "ent": "Kongsberg", "cls": "Northern Africa"}, {"id": 22169, "dom": "G", "ent": "Kongsberg", "cls": "Northern America"}, {"id": 22170, "dom": "G", "ent": "Kongsberg", "cls": "Northern Europe"}, {"id": 22171, "dom": "G", "ent": "Kongsberg", "cls": "Polynesia"}, {"id": 22172, "dom": "G", "ent": "Kongsberg", "cls": "South America"}, {"id": 22173, "dom": "G", "ent": "Kongsberg", "cls": "South-eastern Asia"}, {"id": 22174, "dom": "G", "ent": "Kongsberg", "cls": "Southern Africa"}, {"id": 22175, "dom": "G", "ent": "Kongsberg", "cls": "Southern Asia"}, {"id": 22176, "dom": "G", "ent": "Kongsberg", "cls": "Southern Europe"}, {"id": 22177, "dom": "G", "ent": "Kongsberg", "cls": "Western Africa"}, {"id": 22178, "dom": "G", "ent": "Kongsberg", "cls": "Western Asia"}, {"id": 22179, "dom": "G", "ent": "Kongsberg", "cls": "Western Europe"}, {"id": 22180, "dom": "G", "ent": "Madhyapur Thimi", "cls": "Australia and New Zealand"}, {"id": 22181, "dom": "G", "ent": "Madhyapur Thimi", "cls": "Caribbean"}, {"id": 22182, "dom": "G", "ent": "Madhyapur Thimi", "cls": "Central America"}, {"id": 22183, "dom": "G", "ent": "Madhyapur Thimi", "cls": "Central Asia"}, {"id": 22184, "dom": "G", "ent": "Madhyapur Thimi", "cls": "Eastern Africa"}, {"id": 22185, "dom": "G", "ent": "Madhyapur Thimi", "cls": "Eastern Asia"}, {"id": 22186, "dom": "G", "ent": "Madhyapur Thimi", "cls": "Eastern Europe"}, {"id": 22187, "dom": "G", "ent": "Madhyapur Thimi", "cls": "Melanesia"}, {"id": 22188, "dom": "G", "ent": "Madhyapur Thimi", "cls": "Micronesia"}, {"id": 22189, "dom": "G", "ent": "Madhyapur Thimi", "cls": "Middle Africa"}, {"id": 22190, "dom": "G", "ent": "Madhyapur Thimi", "cls": "Northern Africa"}, {"id": 22191, "dom": "G", "ent": "Madhyapur Thimi", "cls": "Northern America"}, {"id": 22192, "dom": "G", "ent": "Madhyapur Thimi", "cls": "Northern Europe"}, {"id": 22193, "dom": "G", "ent": "Madhyapur Thimi", "cls": "Polynesia"}, {"id": 22194, "dom": "G", "ent": "Madhyapur Thimi", "cls": "South America"}, {"id": 22195, "dom": "G", "ent": "Madhyapur Thimi", "cls": "South-eastern Asia"}, {"id": 22196, "dom": "G", "ent": "Madhyapur Thimi", "cls": "Southern Africa"}, {"id": 22197, "dom": "G", "ent": "Madhyapur Thimi", "cls": "Southern Asia"}, {"id": 22198, "dom": "G", "ent": "Madhyapur Thimi", "cls": "Southern Europe"}, {"id": 22199, "dom": "G", "ent": "Madhyapur Thimi", "cls": "Western Africa"}, {"id": 22200, "dom": "G", "ent": "Madhyapur Thimi", "cls": "Western Asia"}, {"id": 22201, "dom": "G", "ent": "Madhyapur Thimi", "cls": "Western Europe"}, {"id": 22202, "dom": "G", "ent": "Panauti", "cls": "Australia and New Zealand"}, {"id": 22203, "dom": "G", "ent": "Panauti", "cls": "Caribbean"}, {"id": 22204, "dom": "G", "ent": "Panauti", "cls": "Central America"}, {"id": 22205, "dom": "G", "ent": "Panauti", "cls": "Central Asia"}, {"id": 22206, "dom": "G", "ent": "Panauti", "cls": "Eastern Africa"}, {"id": 22207, "dom": "G", "ent": "Panauti", "cls": "Eastern Asia"}, {"id": 22208, "dom": "G", "ent": "Panauti", "cls": "Eastern Europe"}, {"id": 22209, "dom": "G", "ent": "Panauti", "cls": "Melanesia"}, {"id": 22210, "dom": "G", "ent": "Panauti", "cls": "Micronesia"}, {"id": 22211, "dom": "G", "ent": "Panauti", "cls": "Middle Africa"}, {"id": 22212, "dom": "G", "ent": "Panauti", "cls": "Northern Africa"}, {"id": 22213, "dom": "G", "ent": "Panauti", "cls": "Northern America"}, {"id": 22214, "dom": "G", "ent": "Panauti", "cls": "Northern Europe"}, {"id": 22215, "dom": "G", "ent": "Panauti", "cls": "Polynesia"}, {"id": 22216, "dom": "G", "ent": "Panauti", "cls": "South America"}, {"id": 22217, "dom": "G", "ent": "Panauti", "cls": "South-eastern Asia"}, {"id": 22218, "dom": "G", "ent": "Panauti", "cls": "Southern Africa"}, {"id": 22219, "dom": "G", "ent": "Panauti", "cls": "Southern Asia"}, {"id": 22220, "dom": "G", "ent": "Panauti", "cls": "Southern Europe"}, {"id": 22221, "dom": "G", "ent": "Panauti", "cls": "Western Africa"}, {"id": 22222, "dom": "G", "ent": "Panauti", "cls": "Western Asia"}, {"id": 22223, "dom": "G", "ent": "Panauti", "cls": "Western Europe"}, {"id": 22224, "dom": "G", "ent": "Siraha", "cls": "Australia and New Zealand"}, {"id": 22225, "dom": "G", "ent": "Siraha", "cls": "Caribbean"}, {"id": 22226, "dom": "G", "ent": "Siraha", "cls": "Central America"}, {"id": 22227, "dom": "G", "ent": "Siraha", "cls": "Central Asia"}, {"id": 22228, "dom": "G", "ent": "Siraha", "cls": "Eastern Africa"}, {"id": 22229, "dom": "G", "ent": "Siraha", "cls": "Eastern Asia"}, {"id": 22230, "dom": "G", "ent": "Siraha", "cls": "Eastern Europe"}, {"id": 22231, "dom": "G", "ent": "Siraha", "cls": "Melanesia"}, {"id": 22232, "dom": "G", "ent": "Siraha", "cls": "Micronesia"}, {"id": 22233, "dom": "G", "ent": "Siraha", "cls": "Middle Africa"}, {"id": 22234, "dom": "G", "ent": "Siraha", "cls": "Northern Africa"}, {"id": 22235, "dom": "G", "ent": "Siraha", "cls": "Northern America"}, {"id": 22236, "dom": "G", "ent": "Siraha", "cls": "Northern Europe"}, {"id": 22237, "dom": "G", "ent": "Siraha", "cls": "Polynesia"}, {"id": 22238, "dom": "G", "ent": "Siraha", "cls": "South America"}, {"id": 22239, "dom": "G", "ent": "Siraha", "cls": "South-eastern Asia"}, {"id": 22240, "dom": "G", "ent": "Siraha", "cls": "Southern Africa"}, {"id": 22241, "dom": "G", "ent": "Siraha", "cls": "Southern Asia"}, {"id": 22242, "dom": "G", "ent": "Siraha", "cls": "Southern Europe"}, {"id": 22243, "dom": "G", "ent": "Siraha", "cls": "Western Africa"}, {"id": 22244, "dom": "G", "ent": "Siraha", "cls": "Western Asia"}, {"id": 22245, "dom": "G", "ent": "Siraha", "cls": "Western Europe"}, {"id": 22246, "dom": "G", "ent": "Dailekh", "cls": "Australia and New Zealand"}, {"id": 22247, "dom": "G", "ent": "Dailekh", "cls": "Caribbean"}, {"id": 22248, "dom": "G", "ent": "Dailekh", "cls": "Central America"}, {"id": 22249, "dom": "G", "ent": "Dailekh", "cls": "Central Asia"}, {"id": 22250, "dom": "G", "ent": "Dailekh", "cls": "Eastern Africa"}, {"id": 22251, "dom": "G", "ent": "Dailekh", "cls": "Eastern Asia"}, {"id": 22252, "dom": "G", "ent": "Dailekh", "cls": "Eastern Europe"}, {"id": 22253, "dom": "G", "ent": "Dailekh", "cls": "Melanesia"}, {"id": 22254, "dom": "G", "ent": "Dailekh", "cls": "Micronesia"}, {"id": 22255, "dom": "G", "ent": "Dailekh", "cls": "Middle Africa"}, {"id": 22256, "dom": "G", "ent": "Dailekh", "cls": "Northern Africa"}, {"id": 22257, "dom": "G", "ent": "Dailekh", "cls": "Northern America"}, {"id": 22258, "dom": "G", "ent": "Dailekh", "cls": "Northern Europe"}, {"id": 22259, "dom": "G", "ent": "Dailekh", "cls": "Polynesia"}, {"id": 22260, "dom": "G", "ent": "Dailekh", "cls": "South America"}, {"id": 22261, "dom": "G", "ent": "Dailekh", "cls": "South-eastern Asia"}, {"id": 22262, "dom": "G", "ent": "Dailekh", "cls": "Southern Africa"}, {"id": 22263, "dom": "G", "ent": "Dailekh", "cls": "Southern Asia"}, {"id": 22264, "dom": "G", "ent": "Dailekh", "cls": "Southern Europe"}, {"id": 22265, "dom": "G", "ent": "Dailekh", "cls": "Western Africa"}, {"id": 22266, "dom": "G", "ent": "Dailekh", "cls": "Western Asia"}, {"id": 22267, "dom": "G", "ent": "Dailekh", "cls": "Western Europe"}, {"id": 22268, "dom": "G", "ent": "Porirua", "cls": "Australia and New Zealand"}, {"id": 22269, "dom": "G", "ent": "Porirua", "cls": "Caribbean"}, {"id": 22270, "dom": "G", "ent": "Porirua", "cls": "Central America"}, {"id": 22271, "dom": "G", "ent": "Porirua", "cls": "Central Asia"}, {"id": 22272, "dom": "G", "ent": "Porirua", "cls": "Eastern Africa"}, {"id": 22273, "dom": "G", "ent": "Porirua", "cls": "Eastern Asia"}, {"id": 22274, "dom": "G", "ent": "Porirua", "cls": "Eastern Europe"}, {"id": 22275, "dom": "G", "ent": "Porirua", "cls": "Melanesia"}, {"id": 22276, "dom": "G", "ent": "Porirua", "cls": "Micronesia"}, {"id": 22277, "dom": "G", "ent": "Porirua", "cls": "Middle Africa"}, {"id": 22278, "dom": "G", "ent": "Porirua", "cls": "Northern Africa"}, {"id": 22279, "dom": "G", "ent": "Porirua", "cls": "Northern America"}, {"id": 22280, "dom": "G", "ent": "Porirua", "cls": "Northern Europe"}, {"id": 22281, "dom": "G", "ent": "Porirua", "cls": "Polynesia"}, {"id": 22282, "dom": "G", "ent": "Porirua", "cls": "South America"}, {"id": 22283, "dom": "G", "ent": "Porirua", "cls": "South-eastern Asia"}, {"id": 22284, "dom": "G", "ent": "Porirua", "cls": "Southern Africa"}, {"id": 22285, "dom": "G", "ent": "Porirua", "cls": "Southern Asia"}, {"id": 22286, "dom": "G", "ent": "Porirua", "cls": "Southern Europe"}, {"id": 22287, "dom": "G", "ent": "Porirua", "cls": "Western Africa"}, {"id": 22288, "dom": "G", "ent": "Porirua", "cls": "Western Asia"}, {"id": 22289, "dom": "G", "ent": "Porirua", "cls": "Western Europe"}, {"id": 22290, "dom": "G", "ent": "Whangarei", "cls": "Australia and New Zealand"}, {"id": 22291, "dom": "G", "ent": "Whangarei", "cls": "Caribbean"}, {"id": 22292, "dom": "G", "ent": "Whangarei", "cls": "Central America"}, {"id": 22293, "dom": "G", "ent": "Whangarei", "cls": "Central Asia"}, {"id": 22294, "dom": "G", "ent": "Whangarei", "cls": "Eastern Africa"}, {"id": 22295, "dom": "G", "ent": "Whangarei", "cls": "Eastern Asia"}, {"id": 22296, "dom": "G", "ent": "Whangarei", "cls": "Eastern Europe"}, {"id": 22297, "dom": "G", "ent": "Whangarei", "cls": "Melanesia"}, {"id": 22298, "dom": "G", "ent": "Whangarei", "cls": "Micronesia"}, {"id": 22299, "dom": "G", "ent": "Whangarei", "cls": "Middle Africa"}, {"id": 22300, "dom": "G", "ent": "Whangarei", "cls": "Northern Africa"}, {"id": 22301, "dom": "G", "ent": "Whangarei", "cls": "Northern America"}, {"id": 22302, "dom": "G", "ent": "Whangarei", "cls": "Northern Europe"}, {"id": 22303, "dom": "G", "ent": "Whangarei", "cls": "Polynesia"}, {"id": 22304, "dom": "G", "ent": "Whangarei", "cls": "South America"}, {"id": 22305, "dom": "G", "ent": "Whangarei", "cls": "South-eastern Asia"}, {"id": 22306, "dom": "G", "ent": "Whangarei", "cls": "Southern Africa"}, {"id": 22307, "dom": "G", "ent": "Whangarei", "cls": "Southern Asia"}, {"id": 22308, "dom": "G", "ent": "Whangarei", "cls": "Southern Europe"}, {"id": 22309, "dom": "G", "ent": "Whangarei", "cls": "Western Africa"}, {"id": 22310, "dom": "G", "ent": "Whangarei", "cls": "Western Asia"}, {"id": 22311, "dom": "G", "ent": "Whangarei", "cls": "Western Europe"}, {"id": 22312, "dom": "G", "ent": "Paraparaumu", "cls": "Australia and New Zealand"}, {"id": 22313, "dom": "G", "ent": "Paraparaumu", "cls": "Caribbean"}, {"id": 22314, "dom": "G", "ent": "Paraparaumu", "cls": "Central America"}, {"id": 22315, "dom": "G", "ent": "Paraparaumu", "cls": "Central Asia"}, {"id": 22316, "dom": "G", "ent": "Paraparaumu", "cls": "Eastern Africa"}, {"id": 22317, "dom": "G", "ent": "Paraparaumu", "cls": "Eastern Asia"}, {"id": 22318, "dom": "G", "ent": "Paraparaumu", "cls": "Eastern Europe"}, {"id": 22319, "dom": "G", "ent": "Paraparaumu", "cls": "Melanesia"}, {"id": 22320, "dom": "G", "ent": "Paraparaumu", "cls": "Micronesia"}, {"id": 22321, "dom": "G", "ent": "Paraparaumu", "cls": "Middle Africa"}, {"id": 22322, "dom": "G", "ent": "Paraparaumu", "cls": "Northern Africa"}, {"id": 22323, "dom": "G", "ent": "Paraparaumu", "cls": "Northern America"}, {"id": 22324, "dom": "G", "ent": "Paraparaumu", "cls": "Northern Europe"}, {"id": 22325, "dom": "G", "ent": "Paraparaumu", "cls": "Polynesia"}, {"id": 22326, "dom": "G", "ent": "Paraparaumu", "cls": "South America"}, {"id": 22327, "dom": "G", "ent": "Paraparaumu", "cls": "South-eastern Asia"}, {"id": 22328, "dom": "G", "ent": "Paraparaumu", "cls": "Southern Africa"}, {"id": 22329, "dom": "G", "ent": "Paraparaumu", "cls": "Southern Asia"}, {"id": 22330, "dom": "G", "ent": "Paraparaumu", "cls": "Southern Europe"}, {"id": 22331, "dom": "G", "ent": "Paraparaumu", "cls": "Western Africa"}, {"id": 22332, "dom": "G", "ent": "Paraparaumu", "cls": "Western Asia"}, {"id": 22333, "dom": "G", "ent": "Paraparaumu", "cls": "Western Europe"}, {"id": 22334, "dom": "G", "ent": "Karori", "cls": "Australia and New Zealand"}, {"id": 22335, "dom": "G", "ent": "Karori", "cls": "Caribbean"}, {"id": 22336, "dom": "G", "ent": "Karori", "cls": "Central America"}, {"id": 22337, "dom": "G", "ent": "Karori", "cls": "Central Asia"}, {"id": 22338, "dom": "G", "ent": "Karori", "cls": "Eastern Africa"}, {"id": 22339, "dom": "G", "ent": "Karori", "cls": "Eastern Asia"}, {"id": 22340, "dom": "G", "ent": "Karori", "cls": "Eastern Europe"}, {"id": 22341, "dom": "G", "ent": "Karori", "cls": "Melanesia"}, {"id": 22342, "dom": "G", "ent": "Karori", "cls": "Micronesia"}, {"id": 22343, "dom": "G", "ent": "Karori", "cls": "Middle Africa"}, {"id": 22344, "dom": "G", "ent": "Karori", "cls": "Northern Africa"}, {"id": 22345, "dom": "G", "ent": "Karori", "cls": "Northern America"}, {"id": 22346, "dom": "G", "ent": "Karori", "cls": "Northern Europe"}, {"id": 22347, "dom": "G", "ent": "Karori", "cls": "Polynesia"}, {"id": 22348, "dom": "G", "ent": "Karori", "cls": "South America"}, {"id": 22349, "dom": "G", "ent": "Karori", "cls": "South-eastern Asia"}, {"id": 22350, "dom": "G", "ent": "Karori", "cls": "Southern Africa"}, {"id": 22351, "dom": "G", "ent": "Karori", "cls": "Southern Asia"}, {"id": 22352, "dom": "G", "ent": "Karori", "cls": "Southern Europe"}, {"id": 22353, "dom": "G", "ent": "Karori", "cls": "Western Africa"}, {"id": 22354, "dom": "G", "ent": "Karori", "cls": "Western Asia"}, {"id": 22355, "dom": "G", "ent": "Karori", "cls": "Western Europe"}, {"id": 22356, "dom": "G", "ent": "Rustaq", "cls": "Australia and New Zealand"}, {"id": 22357, "dom": "G", "ent": "Rustaq", "cls": "Caribbean"}, {"id": 22358, "dom": "G", "ent": "Rustaq", "cls": "Central America"}, {"id": 22359, "dom": "G", "ent": "Rustaq", "cls": "Central Asia"}, {"id": 22360, "dom": "G", "ent": "Rustaq", "cls": "Eastern Africa"}, {"id": 22361, "dom": "G", "ent": "Rustaq", "cls": "Eastern Asia"}, {"id": 22362, "dom": "G", "ent": "Rustaq", "cls": "Eastern Europe"}, {"id": 22363, "dom": "G", "ent": "Rustaq", "cls": "Melanesia"}, {"id": 22364, "dom": "G", "ent": "Rustaq", "cls": "Micronesia"}, {"id": 22365, "dom": "G", "ent": "Rustaq", "cls": "Middle Africa"}, {"id": 22366, "dom": "G", "ent": "Rustaq", "cls": "Northern Africa"}, {"id": 22367, "dom": "G", "ent": "Rustaq", "cls": "Northern America"}, {"id": 22368, "dom": "G", "ent": "Rustaq", "cls": "Northern Europe"}, {"id": 22369, "dom": "G", "ent": "Rustaq", "cls": "Polynesia"}, {"id": 22370, "dom": "G", "ent": "Rustaq", "cls": "South America"}, {"id": 22371, "dom": "G", "ent": "Rustaq", "cls": "South-eastern Asia"}, {"id": 22372, "dom": "G", "ent": "Rustaq", "cls": "Southern Africa"}, {"id": 22373, "dom": "G", "ent": "Rustaq", "cls": "Southern Asia"}, {"id": 22374, "dom": "G", "ent": "Rustaq", "cls": "Southern Europe"}, {"id": 22375, "dom": "G", "ent": "Rustaq", "cls": "Western Africa"}, {"id": 22376, "dom": "G", "ent": "Rustaq", "cls": "Western Asia"}, {"id": 22377, "dom": "G", "ent": "Rustaq", "cls": "Western Europe"}, {"id": 22378, "dom": "G", "ent": "Sohar", "cls": "Australia and New Zealand"}, {"id": 22379, "dom": "G", "ent": "Sohar", "cls": "Caribbean"}, {"id": 22380, "dom": "G", "ent": "Sohar", "cls": "Central America"}, {"id": 22381, "dom": "G", "ent": "Sohar", "cls": "Central Asia"}, {"id": 22382, "dom": "G", "ent": "Sohar", "cls": "Eastern Africa"}, {"id": 22383, "dom": "G", "ent": "Sohar", "cls": "Eastern Asia"}, {"id": 22384, "dom": "G", "ent": "Sohar", "cls": "Eastern Europe"}, {"id": 22385, "dom": "G", "ent": "Sohar", "cls": "Melanesia"}, {"id": 22386, "dom": "G", "ent": "Sohar", "cls": "Micronesia"}, {"id": 22387, "dom": "G", "ent": "Sohar", "cls": "Middle Africa"}, {"id": 22388, "dom": "G", "ent": "Sohar", "cls": "Northern Africa"}, {"id": 22389, "dom": "G", "ent": "Sohar", "cls": "Northern America"}, {"id": 22390, "dom": "G", "ent": "Sohar", "cls": "Northern Europe"}, {"id": 22391, "dom": "G", "ent": "Sohar", "cls": "Polynesia"}, {"id": 22392, "dom": "G", "ent": "Sohar", "cls": "South America"}, {"id": 22393, "dom": "G", "ent": "Sohar", "cls": "South-eastern Asia"}, {"id": 22394, "dom": "G", "ent": "Sohar", "cls": "Southern Africa"}, {"id": 22395, "dom": "G", "ent": "Sohar", "cls": "Southern Asia"}, {"id": 22396, "dom": "G", "ent": "Sohar", "cls": "Southern Europe"}, {"id": 22397, "dom": "G", "ent": "Sohar", "cls": "Western Africa"}, {"id": 22398, "dom": "G", "ent": "Sohar", "cls": "Western Asia"}, {"id": 22399, "dom": "G", "ent": "Sohar", "cls": "Western Europe"}, {"id": 22400, "dom": "G", "ent": "Sur", "cls": "Australia and New Zealand"}, {"id": 22401, "dom": "G", "ent": "Sur", "cls": "Caribbean"}, {"id": 22402, "dom": "G", "ent": "Sur", "cls": "Central America"}, {"id": 22403, "dom": "G", "ent": "Sur", "cls": "Central Asia"}, {"id": 22404, "dom": "G", "ent": "Sur", "cls": "Eastern Africa"}, {"id": 22405, "dom": "G", "ent": "Sur", "cls": "Eastern Asia"}, {"id": 22406, "dom": "G", "ent": "Sur", "cls": "Eastern Europe"}, {"id": 22407, "dom": "G", "ent": "Sur", "cls": "Melanesia"}, {"id": 22408, "dom": "G", "ent": "Sur", "cls": "Micronesia"}, {"id": 22409, "dom": "G", "ent": "Sur", "cls": "Middle Africa"}, {"id": 22410, "dom": "G", "ent": "Sur", "cls": "Northern Africa"}, {"id": 22411, "dom": "G", "ent": "Sur", "cls": "Northern America"}, {"id": 22412, "dom": "G", "ent": "Sur", "cls": "Northern Europe"}, {"id": 22413, "dom": "G", "ent": "Sur", "cls": "Polynesia"}, {"id": 22414, "dom": "G", "ent": "Sur", "cls": "South America"}, {"id": 22415, "dom": "G", "ent": "Sur", "cls": "South-eastern Asia"}, {"id": 22416, "dom": "G", "ent": "Sur", "cls": "Southern Africa"}, {"id": 22417, "dom": "G", "ent": "Sur", "cls": "Southern Asia"}, {"id": 22418, "dom": "G", "ent": "Sur", "cls": "Southern Europe"}, {"id": 22419, "dom": "G", "ent": "Sur", "cls": "Western Africa"}, {"id": 22420, "dom": "G", "ent": "Sur", "cls": "Western Asia"}, {"id": 22421, "dom": "G", "ent": "Sur", "cls": "Western Europe"}, {"id": 22422, "dom": "G", "ent": "Adam", "cls": "Australia and New Zealand"}, {"id": 22423, "dom": "G", "ent": "Adam", "cls": "Caribbean"}, {"id": 22424, "dom": "G", "ent": "Adam", "cls": "Central America"}, {"id": 22425, "dom": "G", "ent": "Adam", "cls": "Central Asia"}, {"id": 22426, "dom": "G", "ent": "Adam", "cls": "Eastern Africa"}, {"id": 22427, "dom": "G", "ent": "Adam", "cls": "Eastern Asia"}, {"id": 22428, "dom": "G", "ent": "Adam", "cls": "Eastern Europe"}, {"id": 22429, "dom": "G", "ent": "Adam", "cls": "Melanesia"}, {"id": 22430, "dom": "G", "ent": "Adam", "cls": "Micronesia"}, {"id": 22431, "dom": "G", "ent": "Adam", "cls": "Middle Africa"}, {"id": 22432, "dom": "G", "ent": "Adam", "cls": "Northern Africa"}, {"id": 22433, "dom": "G", "ent": "Adam", "cls": "Northern America"}, {"id": 22434, "dom": "G", "ent": "Adam", "cls": "Northern Europe"}, {"id": 22435, "dom": "G", "ent": "Adam", "cls": "Polynesia"}, {"id": 22436, "dom": "G", "ent": "Adam", "cls": "South America"}, {"id": 22437, "dom": "G", "ent": "Adam", "cls": "South-eastern Asia"}, {"id": 22438, "dom": "G", "ent": "Adam", "cls": "Southern Africa"}, {"id": 22439, "dom": "G", "ent": "Adam", "cls": "Southern Asia"}, {"id": 22440, "dom": "G", "ent": "Adam", "cls": "Southern Europe"}, {"id": 22441, "dom": "G", "ent": "Adam", "cls": "Western Africa"}, {"id": 22442, "dom": "G", "ent": "Adam", "cls": "Western Asia"}, {"id": 22443, "dom": "G", "ent": "Adam", "cls": "Western Europe"}, {"id": 22444, "dom": "G", "ent": "La Chorrera", "cls": "Australia and New Zealand"}, {"id": 22445, "dom": "G", "ent": "La Chorrera", "cls": "Caribbean"}, {"id": 22446, "dom": "G", "ent": "La Chorrera", "cls": "Central America"}, {"id": 22447, "dom": "G", "ent": "La Chorrera", "cls": "Central Asia"}, {"id": 22448, "dom": "G", "ent": "La Chorrera", "cls": "Eastern Africa"}, {"id": 22449, "dom": "G", "ent": "La Chorrera", "cls": "Eastern Asia"}, {"id": 22450, "dom": "G", "ent": "La Chorrera", "cls": "Eastern Europe"}, {"id": 22451, "dom": "G", "ent": "La Chorrera", "cls": "Melanesia"}, {"id": 22452, "dom": "G", "ent": "La Chorrera", "cls": "Micronesia"}, {"id": 22453, "dom": "G", "ent": "La Chorrera", "cls": "Middle Africa"}, {"id": 22454, "dom": "G", "ent": "La Chorrera", "cls": "Northern Africa"}, {"id": 22455, "dom": "G", "ent": "La Chorrera", "cls": "Northern America"}, {"id": 22456, "dom": "G", "ent": "La Chorrera", "cls": "Northern Europe"}, {"id": 22457, "dom": "G", "ent": "La Chorrera", "cls": "Polynesia"}, {"id": 22458, "dom": "G", "ent": "La Chorrera", "cls": "South America"}, {"id": 22459, "dom": "G", "ent": "La Chorrera", "cls": "South-eastern Asia"}, {"id": 22460, "dom": "G", "ent": "La Chorrera", "cls": "Southern Africa"}, {"id": 22461, "dom": "G", "ent": "La Chorrera", "cls": "Southern Asia"}, {"id": 22462, "dom": "G", "ent": "La Chorrera", "cls": "Southern Europe"}, {"id": 22463, "dom": "G", "ent": "La Chorrera", "cls": "Western Africa"}, {"id": 22464, "dom": "G", "ent": "La Chorrera", "cls": "Western Asia"}, {"id": 22465, "dom": "G", "ent": "La Chorrera", "cls": "Western Europe"}, {"id": 22466, "dom": "G", "ent": "Chilibre", "cls": "Australia and New Zealand"}, {"id": 22467, "dom": "G", "ent": "Chilibre", "cls": "Caribbean"}, {"id": 22468, "dom": "G", "ent": "Chilibre", "cls": "Central America"}, {"id": 22469, "dom": "G", "ent": "Chilibre", "cls": "Central Asia"}, {"id": 22470, "dom": "G", "ent": "Chilibre", "cls": "Eastern Africa"}, {"id": 22471, "dom": "G", "ent": "Chilibre", "cls": "Eastern Asia"}, {"id": 22472, "dom": "G", "ent": "Chilibre", "cls": "Eastern Europe"}, {"id": 22473, "dom": "G", "ent": "Chilibre", "cls": "Melanesia"}, {"id": 22474, "dom": "G", "ent": "Chilibre", "cls": "Micronesia"}, {"id": 22475, "dom": "G", "ent": "Chilibre", "cls": "Middle Africa"}, {"id": 22476, "dom": "G", "ent": "Chilibre", "cls": "Northern Africa"}, {"id": 22477, "dom": "G", "ent": "Chilibre", "cls": "Northern America"}, {"id": 22478, "dom": "G", "ent": "Chilibre", "cls": "Northern Europe"}, {"id": 22479, "dom": "G", "ent": "Chilibre", "cls": "Polynesia"}, {"id": 22480, "dom": "G", "ent": "Chilibre", "cls": "South America"}, {"id": 22481, "dom": "G", "ent": "Chilibre", "cls": "South-eastern Asia"}, {"id": 22482, "dom": "G", "ent": "Chilibre", "cls": "Southern Africa"}, {"id": 22483, "dom": "G", "ent": "Chilibre", "cls": "Southern Asia"}, {"id": 22484, "dom": "G", "ent": "Chilibre", "cls": "Southern Europe"}, {"id": 22485, "dom": "G", "ent": "Chilibre", "cls": "Western Africa"}, {"id": 22486, "dom": "G", "ent": "Chilibre", "cls": "Western Asia"}, {"id": 22487, "dom": "G", "ent": "Chilibre", "cls": "Western Europe"}, {"id": 22488, "dom": "G", "ent": "Changuinola", "cls": "Australia and New Zealand"}, {"id": 22489, "dom": "G", "ent": "Changuinola", "cls": "Caribbean"}, {"id": 22490, "dom": "G", "ent": "Changuinola", "cls": "Central America"}, {"id": 22491, "dom": "G", "ent": "Changuinola", "cls": "Central Asia"}, {"id": 22492, "dom": "G", "ent": "Changuinola", "cls": "Eastern Africa"}, {"id": 22493, "dom": "G", "ent": "Changuinola", "cls": "Eastern Asia"}, {"id": 22494, "dom": "G", "ent": "Changuinola", "cls": "Eastern Europe"}, {"id": 22495, "dom": "G", "ent": "Changuinola", "cls": "Melanesia"}, {"id": 22496, "dom": "G", "ent": "Changuinola", "cls": "Micronesia"}, {"id": 22497, "dom": "G", "ent": "Changuinola", "cls": "Middle Africa"}, {"id": 22498, "dom": "G", "ent": "Changuinola", "cls": "Northern Africa"}, {"id": 22499, "dom": "G", "ent": "Changuinola", "cls": "Northern America"}, {"id": 22500, "dom": "G", "ent": "Changuinola", "cls": "Northern Europe"}, {"id": 22501, "dom": "G", "ent": "Changuinola", "cls": "Polynesia"}, {"id": 22502, "dom": "G", "ent": "Changuinola", "cls": "South America"}, {"id": 22503, "dom": "G", "ent": "Changuinola", "cls": "South-eastern Asia"}, {"id": 22504, "dom": "G", "ent": "Changuinola", "cls": "Southern Africa"}, {"id": 22505, "dom": "G", "ent": "Changuinola", "cls": "Southern Asia"}, {"id": 22506, "dom": "G", "ent": "Changuinola", "cls": "Southern Europe"}, {"id": 22507, "dom": "G", "ent": "Changuinola", "cls": "Western Africa"}, {"id": 22508, "dom": "G", "ent": "Changuinola", "cls": "Western Asia"}, {"id": 22509, "dom": "G", "ent": "Changuinola", "cls": "Western Europe"}, {"id": 22510, "dom": "G", "ent": "Sabanitas", "cls": "Australia and New Zealand"}, {"id": 22511, "dom": "G", "ent": "Sabanitas", "cls": "Caribbean"}, {"id": 22512, "dom": "G", "ent": "Sabanitas", "cls": "Central America"}, {"id": 22513, "dom": "G", "ent": "Sabanitas", "cls": "Central Asia"}, {"id": 22514, "dom": "G", "ent": "Sabanitas", "cls": "Eastern Africa"}, {"id": 22515, "dom": "G", "ent": "Sabanitas", "cls": "Eastern Asia"}, {"id": 22516, "dom": "G", "ent": "Sabanitas", "cls": "Eastern Europe"}, {"id": 22517, "dom": "G", "ent": "Sabanitas", "cls": "Melanesia"}, {"id": 22518, "dom": "G", "ent": "Sabanitas", "cls": "Micronesia"}, {"id": 22519, "dom": "G", "ent": "Sabanitas", "cls": "Middle Africa"}, {"id": 22520, "dom": "G", "ent": "Sabanitas", "cls": "Northern Africa"}, {"id": 22521, "dom": "G", "ent": "Sabanitas", "cls": "Northern America"}, {"id": 22522, "dom": "G", "ent": "Sabanitas", "cls": "Northern Europe"}, {"id": 22523, "dom": "G", "ent": "Sabanitas", "cls": "Polynesia"}, {"id": 22524, "dom": "G", "ent": "Sabanitas", "cls": "South America"}, {"id": 22525, "dom": "G", "ent": "Sabanitas", "cls": "South-eastern Asia"}, {"id": 22526, "dom": "G", "ent": "Sabanitas", "cls": "Southern Africa"}, {"id": 22527, "dom": "G", "ent": "Sabanitas", "cls": "Southern Asia"}, {"id": 22528, "dom": "G", "ent": "Sabanitas", "cls": "Southern Europe"}, {"id": 22529, "dom": "G", "ent": "Sabanitas", "cls": "Western Africa"}, {"id": 22530, "dom": "G", "ent": "Sabanitas", "cls": "Western Asia"}, {"id": 22531, "dom": "G", "ent": "Sabanitas", "cls": "Western Europe"}, {"id": 22532, "dom": "G", "ent": "Talara", "cls": "Australia and New Zealand"}, {"id": 22533, "dom": "G", "ent": "Talara", "cls": "Caribbean"}, {"id": 22534, "dom": "G", "ent": "Talara", "cls": "Central America"}, {"id": 22535, "dom": "G", "ent": "Talara", "cls": "Central Asia"}, {"id": 22536, "dom": "G", "ent": "Talara", "cls": "Eastern Africa"}, {"id": 22537, "dom": "G", "ent": "Talara", "cls": "Eastern Asia"}, {"id": 22538, "dom": "G", "ent": "Talara", "cls": "Eastern Europe"}, {"id": 22539, "dom": "G", "ent": "Talara", "cls": "Melanesia"}, {"id": 22540, "dom": "G", "ent": "Talara", "cls": "Micronesia"}, {"id": 22541, "dom": "G", "ent": "Talara", "cls": "Middle Africa"}, {"id": 22542, "dom": "G", "ent": "Talara", "cls": "Northern Africa"}, {"id": 22543, "dom": "G", "ent": "Talara", "cls": "Northern America"}, {"id": 22544, "dom": "G", "ent": "Talara", "cls": "Northern Europe"}, {"id": 22545, "dom": "G", "ent": "Talara", "cls": "Polynesia"}, {"id": 22546, "dom": "G", "ent": "Talara", "cls": "South America"}, {"id": 22547, "dom": "G", "ent": "Talara", "cls": "South-eastern Asia"}, {"id": 22548, "dom": "G", "ent": "Talara", "cls": "Southern Africa"}, {"id": 22549, "dom": "G", "ent": "Talara", "cls": "Southern Asia"}, {"id": 22550, "dom": "G", "ent": "Talara", "cls": "Southern Europe"}, {"id": 22551, "dom": "G", "ent": "Talara", "cls": "Western Africa"}, {"id": 22552, "dom": "G", "ent": "Talara", "cls": "Western Asia"}, {"id": 22553, "dom": "G", "ent": "Talara", "cls": "Western Europe"}, {"id": 22554, "dom": "G", "ent": "Abancay", "cls": "Australia and New Zealand"}, {"id": 22555, "dom": "G", "ent": "Abancay", "cls": "Caribbean"}, {"id": 22556, "dom": "G", "ent": "Abancay", "cls": "Central America"}, {"id": 22557, "dom": "G", "ent": "Abancay", "cls": "Central Asia"}, {"id": 22558, "dom": "G", "ent": "Abancay", "cls": "Eastern Africa"}, {"id": 22559, "dom": "G", "ent": "Abancay", "cls": "Eastern Asia"}, {"id": 22560, "dom": "G", "ent": "Abancay", "cls": "Eastern Europe"}, {"id": 22561, "dom": "G", "ent": "Abancay", "cls": "Melanesia"}, {"id": 22562, "dom": "G", "ent": "Abancay", "cls": "Micronesia"}, {"id": 22563, "dom": "G", "ent": "Abancay", "cls": "Middle Africa"}, {"id": 22564, "dom": "G", "ent": "Abancay", "cls": "Northern Africa"}, {"id": 22565, "dom": "G", "ent": "Abancay", "cls": "Northern America"}, {"id": 22566, "dom": "G", "ent": "Abancay", "cls": "Northern Europe"}, {"id": 22567, "dom": "G", "ent": "Abancay", "cls": "Polynesia"}, {"id": 22568, "dom": "G", "ent": "Abancay", "cls": "South America"}, {"id": 22569, "dom": "G", "ent": "Abancay", "cls": "South-eastern Asia"}, {"id": 22570, "dom": "G", "ent": "Abancay", "cls": "Southern Africa"}, {"id": 22571, "dom": "G", "ent": "Abancay", "cls": "Southern Asia"}, {"id": 22572, "dom": "G", "ent": "Abancay", "cls": "Southern Europe"}, {"id": 22573, "dom": "G", "ent": "Abancay", "cls": "Western Africa"}, {"id": 22574, "dom": "G", "ent": "Abancay", "cls": "Western Asia"}, {"id": 22575, "dom": "G", "ent": "Abancay", "cls": "Western Europe"}, {"id": 22576, "dom": "G", "ent": "Picsi", "cls": "Australia and New Zealand"}, {"id": 22577, "dom": "G", "ent": "Picsi", "cls": "Caribbean"}, {"id": 22578, "dom": "G", "ent": "Picsi", "cls": "Central America"}, {"id": 22579, "dom": "G", "ent": "Picsi", "cls": "Central Asia"}, {"id": 22580, "dom": "G", "ent": "Picsi", "cls": "Eastern Africa"}, {"id": 22581, "dom": "G", "ent": "Picsi", "cls": "Eastern Asia"}, {"id": 22582, "dom": "G", "ent": "Picsi", "cls": "Eastern Europe"}, {"id": 22583, "dom": "G", "ent": "Picsi", "cls": "Melanesia"}, {"id": 22584, "dom": "G", "ent": "Picsi", "cls": "Micronesia"}, {"id": 22585, "dom": "G", "ent": "Picsi", "cls": "Middle Africa"}, {"id": 22586, "dom": "G", "ent": "Picsi", "cls": "Northern Africa"}, {"id": 22587, "dom": "G", "ent": "Picsi", "cls": "Northern America"}, {"id": 22588, "dom": "G", "ent": "Picsi", "cls": "Northern Europe"}, {"id": 22589, "dom": "G", "ent": "Picsi", "cls": "Polynesia"}, {"id": 22590, "dom": "G", "ent": "Picsi", "cls": "South America"}, {"id": 22591, "dom": "G", "ent": "Picsi", "cls": "South-eastern Asia"}, {"id": 22592, "dom": "G", "ent": "Picsi", "cls": "Southern Africa"}, {"id": 22593, "dom": "G", "ent": "Picsi", "cls": "Southern Asia"}, {"id": 22594, "dom": "G", "ent": "Picsi", "cls": "Southern Europe"}, {"id": 22595, "dom": "G", "ent": "Picsi", "cls": "Western Africa"}, {"id": 22596, "dom": "G", "ent": "Picsi", "cls": "Western Asia"}, {"id": 22597, "dom": "G", "ent": "Picsi", "cls": "Western Europe"}, {"id": 22598, "dom": "G", "ent": "Quillabamba", "cls": "Australia and New Zealand"}, {"id": 22599, "dom": "G", "ent": "Quillabamba", "cls": "Caribbean"}, {"id": 22600, "dom": "G", "ent": "Quillabamba", "cls": "Central America"}, {"id": 22601, "dom": "G", "ent": "Quillabamba", "cls": "Central Asia"}, {"id": 22602, "dom": "G", "ent": "Quillabamba", "cls": "Eastern Africa"}, {"id": 22603, "dom": "G", "ent": "Quillabamba", "cls": "Eastern Asia"}, {"id": 22604, "dom": "G", "ent": "Quillabamba", "cls": "Eastern Europe"}, {"id": 22605, "dom": "G", "ent": "Quillabamba", "cls": "Melanesia"}, {"id": 22606, "dom": "G", "ent": "Quillabamba", "cls": "Micronesia"}, {"id": 22607, "dom": "G", "ent": "Quillabamba", "cls": "Middle Africa"}, {"id": 22608, "dom": "G", "ent": "Quillabamba", "cls": "Northern Africa"}, {"id": 22609, "dom": "G", "ent": "Quillabamba", "cls": "Northern America"}, {"id": 22610, "dom": "G", "ent": "Quillabamba", "cls": "Northern Europe"}, {"id": 22611, "dom": "G", "ent": "Quillabamba", "cls": "Polynesia"}, {"id": 22612, "dom": "G", "ent": "Quillabamba", "cls": "South America"}, {"id": 22613, "dom": "G", "ent": "Quillabamba", "cls": "South-eastern Asia"}, {"id": 22614, "dom": "G", "ent": "Quillabamba", "cls": "Southern Africa"}, {"id": 22615, "dom": "G", "ent": "Quillabamba", "cls": "Southern Asia"}, {"id": 22616, "dom": "G", "ent": "Quillabamba", "cls": "Southern Europe"}, {"id": 22617, "dom": "G", "ent": "Quillabamba", "cls": "Western Africa"}, {"id": 22618, "dom": "G", "ent": "Quillabamba", "cls": "Western Asia"}, {"id": 22619, "dom": "G", "ent": "Quillabamba", "cls": "Western Europe"}, {"id": 22620, "dom": "G", "ent": "Brookes Point", "cls": "Australia and New Zealand"}, {"id": 22621, "dom": "G", "ent": "Brookes Point", "cls": "Caribbean"}, {"id": 22622, "dom": "G", "ent": "Brookes Point", "cls": "Central America"}, {"id": 22623, "dom": "G", "ent": "Brookes Point", "cls": "Central Asia"}, {"id": 22624, "dom": "G", "ent": "Brookes Point", "cls": "Eastern Africa"}, {"id": 22625, "dom": "G", "ent": "Brookes Point", "cls": "Eastern Asia"}, {"id": 22626, "dom": "G", "ent": "Brookes Point", "cls": "Eastern Europe"}, {"id": 22627, "dom": "G", "ent": "Brookes Point", "cls": "Melanesia"}, {"id": 22628, "dom": "G", "ent": "Brookes Point", "cls": "Micronesia"}, {"id": 22629, "dom": "G", "ent": "Brookes Point", "cls": "Middle Africa"}, {"id": 22630, "dom": "G", "ent": "Brookes Point", "cls": "Northern Africa"}, {"id": 22631, "dom": "G", "ent": "Brookes Point", "cls": "Northern America"}, {"id": 22632, "dom": "G", "ent": "Brookes Point", "cls": "Northern Europe"}, {"id": 22633, "dom": "G", "ent": "Brookes Point", "cls": "Polynesia"}, {"id": 22634, "dom": "G", "ent": "Brookes Point", "cls": "South America"}, {"id": 22635, "dom": "G", "ent": "Brookes Point", "cls": "South-eastern Asia"}, {"id": 22636, "dom": "G", "ent": "Brookes Point", "cls": "Southern Africa"}, {"id": 22637, "dom": "G", "ent": "Brookes Point", "cls": "Southern Asia"}, {"id": 22638, "dom": "G", "ent": "Brookes Point", "cls": "Southern Europe"}, {"id": 22639, "dom": "G", "ent": "Brookes Point", "cls": "Western Africa"}, {"id": 22640, "dom": "G", "ent": "Brookes Point", "cls": "Western Asia"}, {"id": 22641, "dom": "G", "ent": "Brookes Point", "cls": "Western Europe"}, {"id": 22642, "dom": "G", "ent": "Itogon", "cls": "Australia and New Zealand"}, {"id": 22643, "dom": "G", "ent": "Itogon", "cls": "Caribbean"}, {"id": 22644, "dom": "G", "ent": "Itogon", "cls": "Central America"}, {"id": 22645, "dom": "G", "ent": "Itogon", "cls": "Central Asia"}, {"id": 22646, "dom": "G", "ent": "Itogon", "cls": "Eastern Africa"}, {"id": 22647, "dom": "G", "ent": "Itogon", "cls": "Eastern Asia"}, {"id": 22648, "dom": "G", "ent": "Itogon", "cls": "Eastern Europe"}, {"id": 22649, "dom": "G", "ent": "Itogon", "cls": "Melanesia"}, {"id": 22650, "dom": "G", "ent": "Itogon", "cls": "Micronesia"}, {"id": 22651, "dom": "G", "ent": "Itogon", "cls": "Middle Africa"}, {"id": 22652, "dom": "G", "ent": "Itogon", "cls": "Northern Africa"}, {"id": 22653, "dom": "G", "ent": "Itogon", "cls": "Northern America"}, {"id": 22654, "dom": "G", "ent": "Itogon", "cls": "Northern Europe"}, {"id": 22655, "dom": "G", "ent": "Itogon", "cls": "Polynesia"}, {"id": 22656, "dom": "G", "ent": "Itogon", "cls": "South America"}, {"id": 22657, "dom": "G", "ent": "Itogon", "cls": "South-eastern Asia"}, {"id": 22658, "dom": "G", "ent": "Itogon", "cls": "Southern Africa"}, {"id": 22659, "dom": "G", "ent": "Itogon", "cls": "Southern Asia"}, {"id": 22660, "dom": "G", "ent": "Itogon", "cls": "Southern Europe"}, {"id": 22661, "dom": "G", "ent": "Itogon", "cls": "Western Africa"}, {"id": 22662, "dom": "G", "ent": "Itogon", "cls": "Western Asia"}, {"id": 22663, "dom": "G", "ent": "Itogon", "cls": "Western Europe"}, {"id": 22664, "dom": "G", "ent": "Jaen", "cls": "Australia and New Zealand"}, {"id": 22665, "dom": "G", "ent": "Jaen", "cls": "Caribbean"}, {"id": 22666, "dom": "G", "ent": "Jaen", "cls": "Central America"}, {"id": 22667, "dom": "G", "ent": "Jaen", "cls": "Central Asia"}, {"id": 22668, "dom": "G", "ent": "Jaen", "cls": "Eastern Africa"}, {"id": 22669, "dom": "G", "ent": "Jaen", "cls": "Eastern Asia"}, {"id": 22670, "dom": "G", "ent": "Jaen", "cls": "Eastern Europe"}, {"id": 22671, "dom": "G", "ent": "Jaen", "cls": "Melanesia"}, {"id": 22672, "dom": "G", "ent": "Jaen", "cls": "Micronesia"}, {"id": 22673, "dom": "G", "ent": "Jaen", "cls": "Middle Africa"}, {"id": 22674, "dom": "G", "ent": "Jaen", "cls": "Northern Africa"}, {"id": 22675, "dom": "G", "ent": "Jaen", "cls": "Northern America"}, {"id": 22676, "dom": "G", "ent": "Jaen", "cls": "Northern Europe"}, {"id": 22677, "dom": "G", "ent": "Jaen", "cls": "Polynesia"}, {"id": 22678, "dom": "G", "ent": "Jaen", "cls": "South America"}, {"id": 22679, "dom": "G", "ent": "Jaen", "cls": "South-eastern Asia"}, {"id": 22680, "dom": "G", "ent": "Jaen", "cls": "Southern Africa"}, {"id": 22681, "dom": "G", "ent": "Jaen", "cls": "Southern Asia"}, {"id": 22682, "dom": "G", "ent": "Jaen", "cls": "Southern Europe"}, {"id": 22683, "dom": "G", "ent": "Jaen", "cls": "Western Africa"}, {"id": 22684, "dom": "G", "ent": "Jaen", "cls": "Western Asia"}, {"id": 22685, "dom": "G", "ent": "Jaen", "cls": "Western Europe"}, {"id": 22686, "dom": "G", "ent": "Pililla", "cls": "Australia and New Zealand"}, {"id": 22687, "dom": "G", "ent": "Pililla", "cls": "Caribbean"}, {"id": 22688, "dom": "G", "ent": "Pililla", "cls": "Central America"}, {"id": 22689, "dom": "G", "ent": "Pililla", "cls": "Central Asia"}, {"id": 22690, "dom": "G", "ent": "Pililla", "cls": "Eastern Africa"}, {"id": 22691, "dom": "G", "ent": "Pililla", "cls": "Eastern Asia"}, {"id": 22692, "dom": "G", "ent": "Pililla", "cls": "Eastern Europe"}, {"id": 22693, "dom": "G", "ent": "Pililla", "cls": "Melanesia"}, {"id": 22694, "dom": "G", "ent": "Pililla", "cls": "Micronesia"}, {"id": 22695, "dom": "G", "ent": "Pililla", "cls": "Middle Africa"}, {"id": 22696, "dom": "G", "ent": "Pililla", "cls": "Northern Africa"}, {"id": 22697, "dom": "G", "ent": "Pililla", "cls": "Northern America"}, {"id": 22698, "dom": "G", "ent": "Pililla", "cls": "Northern Europe"}, {"id": 22699, "dom": "G", "ent": "Pililla", "cls": "Polynesia"}, {"id": 22700, "dom": "G", "ent": "Pililla", "cls": "South America"}, {"id": 22701, "dom": "G", "ent": "Pililla", "cls": "South-eastern Asia"}, {"id": 22702, "dom": "G", "ent": "Pililla", "cls": "Southern Africa"}, {"id": 22703, "dom": "G", "ent": "Pililla", "cls": "Southern Asia"}, {"id": 22704, "dom": "G", "ent": "Pililla", "cls": "Southern Europe"}, {"id": 22705, "dom": "G", "ent": "Pililla", "cls": "Western Africa"}, {"id": 22706, "dom": "G", "ent": "Pililla", "cls": "Western Asia"}, {"id": 22707, "dom": "G", "ent": "Pililla", "cls": "Western Europe"}, {"id": 22708, "dom": "G", "ent": "Dera Ismail Khan", "cls": "Australia and New Zealand"}, {"id": 22709, "dom": "G", "ent": "Dera Ismail Khan", "cls": "Caribbean"}, {"id": 22710, "dom": "G", "ent": "Dera Ismail Khan", "cls": "Central America"}, {"id": 22711, "dom": "G", "ent": "Dera Ismail Khan", "cls": "Central Asia"}, {"id": 22712, "dom": "G", "ent": "Dera Ismail Khan", "cls": "Eastern Africa"}, {"id": 22713, "dom": "G", "ent": "Dera Ismail Khan", "cls": "Eastern Asia"}, {"id": 22714, "dom": "G", "ent": "Dera Ismail Khan", "cls": "Eastern Europe"}, {"id": 22715, "dom": "G", "ent": "Dera Ismail Khan", "cls": "Melanesia"}, {"id": 22716, "dom": "G", "ent": "Dera Ismail Khan", "cls": "Micronesia"}, {"id": 22717, "dom": "G", "ent": "Dera Ismail Khan", "cls": "Middle Africa"}, {"id": 22718, "dom": "G", "ent": "Dera Ismail Khan", "cls": "Northern Africa"}, {"id": 22719, "dom": "G", "ent": "Dera Ismail Khan", "cls": "Northern America"}, {"id": 22720, "dom": "G", "ent": "Dera Ismail Khan", "cls": "Northern Europe"}, {"id": 22721, "dom": "G", "ent": "Dera Ismail Khan", "cls": "Polynesia"}, {"id": 22722, "dom": "G", "ent": "Dera Ismail Khan", "cls": "South America"}, {"id": 22723, "dom": "G", "ent": "Dera Ismail Khan", "cls": "South-eastern Asia"}, {"id": 22724, "dom": "G", "ent": "Dera Ismail Khan", "cls": "Southern Africa"}, {"id": 22725, "dom": "G", "ent": "Dera Ismail Khan", "cls": "Southern Asia"}, {"id": 22726, "dom": "G", "ent": "Dera Ismail Khan", "cls": "Southern Europe"}, {"id": 22727, "dom": "G", "ent": "Dera Ismail Khan", "cls": "Western Africa"}, {"id": 22728, "dom": "G", "ent": "Dera Ismail Khan", "cls": "Western Asia"}, {"id": 22729, "dom": "G", "ent": "Dera Ismail Khan", "cls": "Western Europe"}, {"id": 22730, "dom": "G", "ent": "Phalia", "cls": "Australia and New Zealand"}, {"id": 22731, "dom": "G", "ent": "Phalia", "cls": "Caribbean"}, {"id": 22732, "dom": "G", "ent": "Phalia", "cls": "Central America"}, {"id": 22733, "dom": "G", "ent": "Phalia", "cls": "Central Asia"}, {"id": 22734, "dom": "G", "ent": "Phalia", "cls": "Eastern Africa"}, {"id": 22735, "dom": "G", "ent": "Phalia", "cls": "Eastern Asia"}, {"id": 22736, "dom": "G", "ent": "Phalia", "cls": "Eastern Europe"}, {"id": 22737, "dom": "G", "ent": "Phalia", "cls": "Melanesia"}, {"id": 22738, "dom": "G", "ent": "Phalia", "cls": "Micronesia"}, {"id": 22739, "dom": "G", "ent": "Phalia", "cls": "Middle Africa"}, {"id": 22740, "dom": "G", "ent": "Phalia", "cls": "Northern Africa"}, {"id": 22741, "dom": "G", "ent": "Phalia", "cls": "Northern America"}, {"id": 22742, "dom": "G", "ent": "Phalia", "cls": "Northern Europe"}, {"id": 22743, "dom": "G", "ent": "Phalia", "cls": "Polynesia"}, {"id": 22744, "dom": "G", "ent": "Phalia", "cls": "South America"}, {"id": 22745, "dom": "G", "ent": "Phalia", "cls": "South-eastern Asia"}, {"id": 22746, "dom": "G", "ent": "Phalia", "cls": "Southern Africa"}, {"id": 22747, "dom": "G", "ent": "Phalia", "cls": "Southern Asia"}, {"id": 22748, "dom": "G", "ent": "Phalia", "cls": "Southern Europe"}, {"id": 22749, "dom": "G", "ent": "Phalia", "cls": "Western Africa"}, {"id": 22750, "dom": "G", "ent": "Phalia", "cls": "Western Asia"}, {"id": 22751, "dom": "G", "ent": "Phalia", "cls": "Western Europe"}, {"id": 22752, "dom": "G", "ent": "Sillanwali", "cls": "Australia and New Zealand"}, {"id": 22753, "dom": "G", "ent": "Sillanwali", "cls": "Caribbean"}, {"id": 22754, "dom": "G", "ent": "Sillanwali", "cls": "Central America"}, {"id": 22755, "dom": "G", "ent": "Sillanwali", "cls": "Central Asia"}, {"id": 22756, "dom": "G", "ent": "Sillanwali", "cls": "Eastern Africa"}, {"id": 22757, "dom": "G", "ent": "Sillanwali", "cls": "Eastern Asia"}, {"id": 22758, "dom": "G", "ent": "Sillanwali", "cls": "Eastern Europe"}, {"id": 22759, "dom": "G", "ent": "Sillanwali", "cls": "Melanesia"}, {"id": 22760, "dom": "G", "ent": "Sillanwali", "cls": "Micronesia"}, {"id": 22761, "dom": "G", "ent": "Sillanwali", "cls": "Middle Africa"}, {"id": 22762, "dom": "G", "ent": "Sillanwali", "cls": "Northern Africa"}, {"id": 22763, "dom": "G", "ent": "Sillanwali", "cls": "Northern America"}, {"id": 22764, "dom": "G", "ent": "Sillanwali", "cls": "Northern Europe"}, {"id": 22765, "dom": "G", "ent": "Sillanwali", "cls": "Polynesia"}, {"id": 22766, "dom": "G", "ent": "Sillanwali", "cls": "South America"}, {"id": 22767, "dom": "G", "ent": "Sillanwali", "cls": "South-eastern Asia"}, {"id": 22768, "dom": "G", "ent": "Sillanwali", "cls": "Southern Africa"}, {"id": 22769, "dom": "G", "ent": "Sillanwali", "cls": "Southern Asia"}, {"id": 22770, "dom": "G", "ent": "Sillanwali", "cls": "Southern Europe"}, {"id": 22771, "dom": "G", "ent": "Sillanwali", "cls": "Western Africa"}, {"id": 22772, "dom": "G", "ent": "Sillanwali", "cls": "Western Asia"}, {"id": 22773, "dom": "G", "ent": "Sillanwali", "cls": "Western Europe"}, {"id": 22774, "dom": "G", "ent": "Harappa", "cls": "Australia and New Zealand"}, {"id": 22775, "dom": "G", "ent": "Harappa", "cls": "Caribbean"}, {"id": 22776, "dom": "G", "ent": "Harappa", "cls": "Central America"}, {"id": 22777, "dom": "G", "ent": "Harappa", "cls": "Central Asia"}, {"id": 22778, "dom": "G", "ent": "Harappa", "cls": "Eastern Africa"}, {"id": 22779, "dom": "G", "ent": "Harappa", "cls": "Eastern Asia"}, {"id": 22780, "dom": "G", "ent": "Harappa", "cls": "Eastern Europe"}, {"id": 22781, "dom": "G", "ent": "Harappa", "cls": "Melanesia"}, {"id": 22782, "dom": "G", "ent": "Harappa", "cls": "Micronesia"}, {"id": 22783, "dom": "G", "ent": "Harappa", "cls": "Middle Africa"}, {"id": 22784, "dom": "G", "ent": "Harappa", "cls": "Northern Africa"}, {"id": 22785, "dom": "G", "ent": "Harappa", "cls": "Northern America"}, {"id": 22786, "dom": "G", "ent": "Harappa", "cls": "Northern Europe"}, {"id": 22787, "dom": "G", "ent": "Harappa", "cls": "Polynesia"}, {"id": 22788, "dom": "G", "ent": "Harappa", "cls": "South America"}, {"id": 22789, "dom": "G", "ent": "Harappa", "cls": "South-eastern Asia"}, {"id": 22790, "dom": "G", "ent": "Harappa", "cls": "Southern Africa"}, {"id": 22791, "dom": "G", "ent": "Harappa", "cls": "Southern Asia"}, {"id": 22792, "dom": "G", "ent": "Harappa", "cls": "Southern Europe"}, {"id": 22793, "dom": "G", "ent": "Harappa", "cls": "Western Africa"}, {"id": 22794, "dom": "G", "ent": "Harappa", "cls": "Western Asia"}, {"id": 22795, "dom": "G", "ent": "Harappa", "cls": "Western Europe"}, {"id": 22796, "dom": "G", "ent": "Konin", "cls": "Australia and New Zealand"}, {"id": 22797, "dom": "G", "ent": "Konin", "cls": "Caribbean"}, {"id": 22798, "dom": "G", "ent": "Konin", "cls": "Central America"}, {"id": 22799, "dom": "G", "ent": "Konin", "cls": "Central Asia"}, {"id": 22800, "dom": "G", "ent": "Konin", "cls": "Eastern Africa"}, {"id": 22801, "dom": "G", "ent": "Konin", "cls": "Eastern Asia"}, {"id": 22802, "dom": "G", "ent": "Konin", "cls": "Eastern Europe"}, {"id": 22803, "dom": "G", "ent": "Konin", "cls": "Melanesia"}, {"id": 22804, "dom": "G", "ent": "Konin", "cls": "Micronesia"}, {"id": 22805, "dom": "G", "ent": "Konin", "cls": "Middle Africa"}, {"id": 22806, "dom": "G", "ent": "Konin", "cls": "Northern Africa"}, {"id": 22807, "dom": "G", "ent": "Konin", "cls": "Northern America"}, {"id": 22808, "dom": "G", "ent": "Konin", "cls": "Northern Europe"}, {"id": 22809, "dom": "G", "ent": "Konin", "cls": "Polynesia"}, {"id": 22810, "dom": "G", "ent": "Konin", "cls": "South America"}, {"id": 22811, "dom": "G", "ent": "Konin", "cls": "South-eastern Asia"}, {"id": 22812, "dom": "G", "ent": "Konin", "cls": "Southern Africa"}, {"id": 22813, "dom": "G", "ent": "Konin", "cls": "Southern Asia"}, {"id": 22814, "dom": "G", "ent": "Konin", "cls": "Southern Europe"}, {"id": 22815, "dom": "G", "ent": "Konin", "cls": "Western Africa"}, {"id": 22816, "dom": "G", "ent": "Konin", "cls": "Western Asia"}, {"id": 22817, "dom": "G", "ent": "Konin", "cls": "Western Europe"}, {"id": 22818, "dom": "G", "ent": "Nysa", "cls": "Australia and New Zealand"}, {"id": 22819, "dom": "G", "ent": "Nysa", "cls": "Caribbean"}, {"id": 22820, "dom": "G", "ent": "Nysa", "cls": "Central America"}, {"id": 22821, "dom": "G", "ent": "Nysa", "cls": "Central Asia"}, {"id": 22822, "dom": "G", "ent": "Nysa", "cls": "Eastern Africa"}, {"id": 22823, "dom": "G", "ent": "Nysa", "cls": "Eastern Asia"}, {"id": 22824, "dom": "G", "ent": "Nysa", "cls": "Eastern Europe"}, {"id": 22825, "dom": "G", "ent": "Nysa", "cls": "Melanesia"}, {"id": 22826, "dom": "G", "ent": "Nysa", "cls": "Micronesia"}, {"id": 22827, "dom": "G", "ent": "Nysa", "cls": "Middle Africa"}, {"id": 22828, "dom": "G", "ent": "Nysa", "cls": "Northern Africa"}, {"id": 22829, "dom": "G", "ent": "Nysa", "cls": "Northern America"}, {"id": 22830, "dom": "G", "ent": "Nysa", "cls": "Northern Europe"}, {"id": 22831, "dom": "G", "ent": "Nysa", "cls": "Polynesia"}, {"id": 22832, "dom": "G", "ent": "Nysa", "cls": "South America"}, {"id": 22833, "dom": "G", "ent": "Nysa", "cls": "South-eastern Asia"}, {"id": 22834, "dom": "G", "ent": "Nysa", "cls": "Southern Africa"}, {"id": 22835, "dom": "G", "ent": "Nysa", "cls": "Southern Asia"}, {"id": 22836, "dom": "G", "ent": "Nysa", "cls": "Southern Europe"}, {"id": 22837, "dom": "G", "ent": "Nysa", "cls": "Western Africa"}, {"id": 22838, "dom": "G", "ent": "Nysa", "cls": "Western Asia"}, {"id": 22839, "dom": "G", "ent": "Nysa", "cls": "Western Europe"}, {"id": 22840, "dom": "G", "ent": "Krapkowice", "cls": "Australia and New Zealand"}, {"id": 22841, "dom": "G", "ent": "Krapkowice", "cls": "Caribbean"}, {"id": 22842, "dom": "G", "ent": "Krapkowice", "cls": "Central America"}, {"id": 22843, "dom": "G", "ent": "Krapkowice", "cls": "Central Asia"}, {"id": 22844, "dom": "G", "ent": "Krapkowice", "cls": "Eastern Africa"}, {"id": 22845, "dom": "G", "ent": "Krapkowice", "cls": "Eastern Asia"}, {"id": 22846, "dom": "G", "ent": "Krapkowice", "cls": "Eastern Europe"}, {"id": 22847, "dom": "G", "ent": "Krapkowice", "cls": "Melanesia"}, {"id": 22848, "dom": "G", "ent": "Krapkowice", "cls": "Micronesia"}, {"id": 22849, "dom": "G", "ent": "Krapkowice", "cls": "Middle Africa"}, {"id": 22850, "dom": "G", "ent": "Krapkowice", "cls": "Northern Africa"}, {"id": 22851, "dom": "G", "ent": "Krapkowice", "cls": "Northern America"}, {"id": 22852, "dom": "G", "ent": "Krapkowice", "cls": "Northern Europe"}, {"id": 22853, "dom": "G", "ent": "Krapkowice", "cls": "Polynesia"}, {"id": 22854, "dom": "G", "ent": "Krapkowice", "cls": "South America"}, {"id": 22855, "dom": "G", "ent": "Krapkowice", "cls": "South-eastern Asia"}, {"id": 22856, "dom": "G", "ent": "Krapkowice", "cls": "Southern Africa"}, {"id": 22857, "dom": "G", "ent": "Krapkowice", "cls": "Southern Asia"}, {"id": 22858, "dom": "G", "ent": "Krapkowice", "cls": "Southern Europe"}, {"id": 22859, "dom": "G", "ent": "Krapkowice", "cls": "Western Africa"}, {"id": 22860, "dom": "G", "ent": "Krapkowice", "cls": "Western Asia"}, {"id": 22861, "dom": "G", "ent": "Krapkowice", "cls": "Western Europe"}, {"id": 22862, "dom": "G", "ent": "Osowa", "cls": "Australia and New Zealand"}, {"id": 22863, "dom": "G", "ent": "Osowa", "cls": "Caribbean"}, {"id": 22864, "dom": "G", "ent": "Osowa", "cls": "Central America"}, {"id": 22865, "dom": "G", "ent": "Osowa", "cls": "Central Asia"}, {"id": 22866, "dom": "G", "ent": "Osowa", "cls": "Eastern Africa"}, {"id": 22867, "dom": "G", "ent": "Osowa", "cls": "Eastern Asia"}, {"id": 22868, "dom": "G", "ent": "Osowa", "cls": "Eastern Europe"}, {"id": 22869, "dom": "G", "ent": "Osowa", "cls": "Melanesia"}, {"id": 22870, "dom": "G", "ent": "Osowa", "cls": "Micronesia"}, {"id": 22871, "dom": "G", "ent": "Osowa", "cls": "Middle Africa"}, {"id": 22872, "dom": "G", "ent": "Osowa", "cls": "Northern Africa"}, {"id": 22873, "dom": "G", "ent": "Osowa", "cls": "Northern America"}, {"id": 22874, "dom": "G", "ent": "Osowa", "cls": "Northern Europe"}, {"id": 22875, "dom": "G", "ent": "Osowa", "cls": "Polynesia"}, {"id": 22876, "dom": "G", "ent": "Osowa", "cls": "South America"}, {"id": 22877, "dom": "G", "ent": "Osowa", "cls": "South-eastern Asia"}, {"id": 22878, "dom": "G", "ent": "Osowa", "cls": "Southern Africa"}, {"id": 22879, "dom": "G", "ent": "Osowa", "cls": "Southern Asia"}, {"id": 22880, "dom": "G", "ent": "Osowa", "cls": "Southern Europe"}, {"id": 22881, "dom": "G", "ent": "Osowa", "cls": "Western Africa"}, {"id": 22882, "dom": "G", "ent": "Osowa", "cls": "Western Asia"}, {"id": 22883, "dom": "G", "ent": "Osowa", "cls": "Western Europe"}, {"id": 22884, "dom": "G", "ent": "Caguas", "cls": "Australia and New Zealand"}, {"id": 22885, "dom": "G", "ent": "Caguas", "cls": "Caribbean"}, {"id": 22886, "dom": "G", "ent": "Caguas", "cls": "Central America"}, {"id": 22887, "dom": "G", "ent": "Caguas", "cls": "Central Asia"}, {"id": 22888, "dom": "G", "ent": "Caguas", "cls": "Eastern Africa"}, {"id": 22889, "dom": "G", "ent": "Caguas", "cls": "Eastern Asia"}, {"id": 22890, "dom": "G", "ent": "Caguas", "cls": "Eastern Europe"}, {"id": 22891, "dom": "G", "ent": "Caguas", "cls": "Melanesia"}, {"id": 22892, "dom": "G", "ent": "Caguas", "cls": "Micronesia"}, {"id": 22893, "dom": "G", "ent": "Caguas", "cls": "Middle Africa"}, {"id": 22894, "dom": "G", "ent": "Caguas", "cls": "Northern Africa"}, {"id": 22895, "dom": "G", "ent": "Caguas", "cls": "Northern America"}, {"id": 22896, "dom": "G", "ent": "Caguas", "cls": "Northern Europe"}, {"id": 22897, "dom": "G", "ent": "Caguas", "cls": "Polynesia"}, {"id": 22898, "dom": "G", "ent": "Caguas", "cls": "South America"}, {"id": 22899, "dom": "G", "ent": "Caguas", "cls": "South-eastern Asia"}, {"id": 22900, "dom": "G", "ent": "Caguas", "cls": "Southern Africa"}, {"id": 22901, "dom": "G", "ent": "Caguas", "cls": "Southern Asia"}, {"id": 22902, "dom": "G", "ent": "Caguas", "cls": "Southern Europe"}, {"id": 22903, "dom": "G", "ent": "Caguas", "cls": "Western Africa"}, {"id": 22904, "dom": "G", "ent": "Caguas", "cls": "Western Asia"}, {"id": 22905, "dom": "G", "ent": "Caguas", "cls": "Western Europe"}, {"id": 22906, "dom": "G", "ent": "Trujillo Alto", "cls": "Australia and New Zealand"}, {"id": 22907, "dom": "G", "ent": "Trujillo Alto", "cls": "Caribbean"}, {"id": 22908, "dom": "G", "ent": "Trujillo Alto", "cls": "Central America"}, {"id": 22909, "dom": "G", "ent": "Trujillo Alto", "cls": "Central Asia"}, {"id": 22910, "dom": "G", "ent": "Trujillo Alto", "cls": "Eastern Africa"}, {"id": 22911, "dom": "G", "ent": "Trujillo Alto", "cls": "Eastern Asia"}, {"id": 22912, "dom": "G", "ent": "Trujillo Alto", "cls": "Eastern Europe"}, {"id": 22913, "dom": "G", "ent": "Trujillo Alto", "cls": "Melanesia"}, {"id": 22914, "dom": "G", "ent": "Trujillo Alto", "cls": "Micronesia"}, {"id": 22915, "dom": "G", "ent": "Trujillo Alto", "cls": "Middle Africa"}, {"id": 22916, "dom": "G", "ent": "Trujillo Alto", "cls": "Northern Africa"}, {"id": 22917, "dom": "G", "ent": "Trujillo Alto", "cls": "Northern America"}, {"id": 22918, "dom": "G", "ent": "Trujillo Alto", "cls": "Northern Europe"}, {"id": 22919, "dom": "G", "ent": "Trujillo Alto", "cls": "Polynesia"}, {"id": 22920, "dom": "G", "ent": "Trujillo Alto", "cls": "South America"}, {"id": 22921, "dom": "G", "ent": "Trujillo Alto", "cls": "South-eastern Asia"}, {"id": 22922, "dom": "G", "ent": "Trujillo Alto", "cls": "Southern Africa"}, {"id": 22923, "dom": "G", "ent": "Trujillo Alto", "cls": "Southern Asia"}, {"id": 22924, "dom": "G", "ent": "Trujillo Alto", "cls": "Southern Europe"}, {"id": 22925, "dom": "G", "ent": "Trujillo Alto", "cls": "Western Africa"}, {"id": 22926, "dom": "G", "ent": "Trujillo Alto", "cls": "Western Asia"}, {"id": 22927, "dom": "G", "ent": "Trujillo Alto", "cls": "Western Europe"}, {"id": 22928, "dom": "G", "ent": "Barceloneta", "cls": "Australia and New Zealand"}, {"id": 22929, "dom": "G", "ent": "Barceloneta", "cls": "Caribbean"}, {"id": 22930, "dom": "G", "ent": "Barceloneta", "cls": "Central America"}, {"id": 22931, "dom": "G", "ent": "Barceloneta", "cls": "Central Asia"}, {"id": 22932, "dom": "G", "ent": "Barceloneta", "cls": "Eastern Africa"}, {"id": 22933, "dom": "G", "ent": "Barceloneta", "cls": "Eastern Asia"}, {"id": 22934, "dom": "G", "ent": "Barceloneta", "cls": "Eastern Europe"}, {"id": 22935, "dom": "G", "ent": "Barceloneta", "cls": "Melanesia"}, {"id": 22936, "dom": "G", "ent": "Barceloneta", "cls": "Micronesia"}, {"id": 22937, "dom": "G", "ent": "Barceloneta", "cls": "Middle Africa"}, {"id": 22938, "dom": "G", "ent": "Barceloneta", "cls": "Northern Africa"}, {"id": 22939, "dom": "G", "ent": "Barceloneta", "cls": "Northern America"}, {"id": 22940, "dom": "G", "ent": "Barceloneta", "cls": "Northern Europe"}, {"id": 22941, "dom": "G", "ent": "Barceloneta", "cls": "Polynesia"}, {"id": 22942, "dom": "G", "ent": "Barceloneta", "cls": "South America"}, {"id": 22943, "dom": "G", "ent": "Barceloneta", "cls": "South-eastern Asia"}, {"id": 22944, "dom": "G", "ent": "Barceloneta", "cls": "Southern Africa"}, {"id": 22945, "dom": "G", "ent": "Barceloneta", "cls": "Southern Asia"}, {"id": 22946, "dom": "G", "ent": "Barceloneta", "cls": "Southern Europe"}, {"id": 22947, "dom": "G", "ent": "Barceloneta", "cls": "Western Africa"}, {"id": 22948, "dom": "G", "ent": "Barceloneta", "cls": "Western Asia"}, {"id": 22949, "dom": "G", "ent": "Barceloneta", "cls": "Western Europe"}, {"id": 22950, "dom": "G", "ent": "Aguadilla", "cls": "Australia and New Zealand"}, {"id": 22951, "dom": "G", "ent": "Aguadilla", "cls": "Caribbean"}, {"id": 22952, "dom": "G", "ent": "Aguadilla", "cls": "Central America"}, {"id": 22953, "dom": "G", "ent": "Aguadilla", "cls": "Central Asia"}, {"id": 22954, "dom": "G", "ent": "Aguadilla", "cls": "Eastern Africa"}, {"id": 22955, "dom": "G", "ent": "Aguadilla", "cls": "Eastern Asia"}, {"id": 22956, "dom": "G", "ent": "Aguadilla", "cls": "Eastern Europe"}, {"id": 22957, "dom": "G", "ent": "Aguadilla", "cls": "Melanesia"}, {"id": 22958, "dom": "G", "ent": "Aguadilla", "cls": "Micronesia"}, {"id": 22959, "dom": "G", "ent": "Aguadilla", "cls": "Middle Africa"}, {"id": 22960, "dom": "G", "ent": "Aguadilla", "cls": "Northern Africa"}, {"id": 22961, "dom": "G", "ent": "Aguadilla", "cls": "Northern America"}, {"id": 22962, "dom": "G", "ent": "Aguadilla", "cls": "Northern Europe"}, {"id": 22963, "dom": "G", "ent": "Aguadilla", "cls": "Polynesia"}, {"id": 22964, "dom": "G", "ent": "Aguadilla", "cls": "South America"}, {"id": 22965, "dom": "G", "ent": "Aguadilla", "cls": "South-eastern Asia"}, {"id": 22966, "dom": "G", "ent": "Aguadilla", "cls": "Southern Africa"}, {"id": 22967, "dom": "G", "ent": "Aguadilla", "cls": "Southern Asia"}, {"id": 22968, "dom": "G", "ent": "Aguadilla", "cls": "Southern Europe"}, {"id": 22969, "dom": "G", "ent": "Aguadilla", "cls": "Western Africa"}, {"id": 22970, "dom": "G", "ent": "Aguadilla", "cls": "Western Asia"}, {"id": 22971, "dom": "G", "ent": "Aguadilla", "cls": "Western Europe"}, {"id": 22972, "dom": "G", "ent": "Hebron", "cls": "Australia and New Zealand"}, {"id": 22973, "dom": "G", "ent": "Hebron", "cls": "Caribbean"}, {"id": 22974, "dom": "G", "ent": "Hebron", "cls": "Central America"}, {"id": 22975, "dom": "G", "ent": "Hebron", "cls": "Central Asia"}, {"id": 22976, "dom": "G", "ent": "Hebron", "cls": "Eastern Africa"}, {"id": 22977, "dom": "G", "ent": "Hebron", "cls": "Eastern Asia"}, {"id": 22978, "dom": "G", "ent": "Hebron", "cls": "Eastern Europe"}, {"id": 22979, "dom": "G", "ent": "Hebron", "cls": "Melanesia"}, {"id": 22980, "dom": "G", "ent": "Hebron", "cls": "Micronesia"}, {"id": 22981, "dom": "G", "ent": "Hebron", "cls": "Middle Africa"}, {"id": 22982, "dom": "G", "ent": "Hebron", "cls": "Northern Africa"}, {"id": 22983, "dom": "G", "ent": "Hebron", "cls": "Northern America"}, {"id": 22984, "dom": "G", "ent": "Hebron", "cls": "Northern Europe"}, {"id": 22985, "dom": "G", "ent": "Hebron", "cls": "Polynesia"}, {"id": 22986, "dom": "G", "ent": "Hebron", "cls": "South America"}, {"id": 22987, "dom": "G", "ent": "Hebron", "cls": "South-eastern Asia"}, {"id": 22988, "dom": "G", "ent": "Hebron", "cls": "Southern Africa"}, {"id": 22989, "dom": "G", "ent": "Hebron", "cls": "Southern Asia"}, {"id": 22990, "dom": "G", "ent": "Hebron", "cls": "Southern Europe"}, {"id": 22991, "dom": "G", "ent": "Hebron", "cls": "Western Africa"}, {"id": 22992, "dom": "G", "ent": "Hebron", "cls": "Western Asia"}, {"id": 22993, "dom": "G", "ent": "Hebron", "cls": "Western Europe"}, {"id": 22994, "dom": "G", "ent": "Old City", "cls": "Australia and New Zealand"}, {"id": 22995, "dom": "G", "ent": "Old City", "cls": "Caribbean"}, {"id": 22996, "dom": "G", "ent": "Old City", "cls": "Central America"}, {"id": 22997, "dom": "G", "ent": "Old City", "cls": "Central Asia"}, {"id": 22998, "dom": "G", "ent": "Old City", "cls": "Eastern Africa"}, {"id": 22999, "dom": "G", "ent": "Old City", "cls": "Eastern Asia"}, {"id": 23000, "dom": "G", "ent": "Old City", "cls": "Eastern Europe"}, {"id": 23001, "dom": "G", "ent": "Old City", "cls": "Melanesia"}, {"id": 23002, "dom": "G", "ent": "Old City", "cls": "Micronesia"}, {"id": 23003, "dom": "G", "ent": "Old City", "cls": "Middle Africa"}, {"id": 23004, "dom": "G", "ent": "Old City", "cls": "Northern Africa"}, {"id": 23005, "dom": "G", "ent": "Old City", "cls": "Northern America"}, {"id": 23006, "dom": "G", "ent": "Old City", "cls": "Northern Europe"}, {"id": 23007, "dom": "G", "ent": "Old City", "cls": "Polynesia"}, {"id": 23008, "dom": "G", "ent": "Old City", "cls": "South America"}, {"id": 23009, "dom": "G", "ent": "Old City", "cls": "South-eastern Asia"}, {"id": 23010, "dom": "G", "ent": "Old City", "cls": "Southern Africa"}, {"id": 23011, "dom": "G", "ent": "Old City", "cls": "Southern Asia"}, {"id": 23012, "dom": "G", "ent": "Old City", "cls": "Southern Europe"}, {"id": 23013, "dom": "G", "ent": "Old City", "cls": "Western Africa"}, {"id": 23014, "dom": "G", "ent": "Old City", "cls": "Western Asia"}, {"id": 23015, "dom": "G", "ent": "Old City", "cls": "Western Europe"}, {"id": 23016, "dom": "G", "ent": "Surif City", "cls": "Australia and New Zealand"}, {"id": 23017, "dom": "G", "ent": "Surif City", "cls": "Caribbean"}, {"id": 23018, "dom": "G", "ent": "Surif City", "cls": "Central America"}, {"id": 23019, "dom": "G", "ent": "Surif City", "cls": "Central Asia"}, {"id": 23020, "dom": "G", "ent": "Surif City", "cls": "Eastern Africa"}, {"id": 23021, "dom": "G", "ent": "Surif City", "cls": "Eastern Asia"}, {"id": 23022, "dom": "G", "ent": "Surif City", "cls": "Eastern Europe"}, {"id": 23023, "dom": "G", "ent": "Surif City", "cls": "Melanesia"}, {"id": 23024, "dom": "G", "ent": "Surif City", "cls": "Micronesia"}, {"id": 23025, "dom": "G", "ent": "Surif City", "cls": "Middle Africa"}, {"id": 23026, "dom": "G", "ent": "Surif City", "cls": "Northern Africa"}, {"id": 23027, "dom": "G", "ent": "Surif City", "cls": "Northern America"}, {"id": 23028, "dom": "G", "ent": "Surif City", "cls": "Northern Europe"}, {"id": 23029, "dom": "G", "ent": "Surif City", "cls": "Polynesia"}, {"id": 23030, "dom": "G", "ent": "Surif City", "cls": "South America"}, {"id": 23031, "dom": "G", "ent": "Surif City", "cls": "South-eastern Asia"}, {"id": 23032, "dom": "G", "ent": "Surif City", "cls": "Southern Africa"}, {"id": 23033, "dom": "G", "ent": "Surif City", "cls": "Southern Asia"}, {"id": 23034, "dom": "G", "ent": "Surif City", "cls": "Southern Europe"}, {"id": 23035, "dom": "G", "ent": "Surif City", "cls": "Western Africa"}, {"id": 23036, "dom": "G", "ent": "Surif City", "cls": "Western Asia"}, {"id": 23037, "dom": "G", "ent": "Surif City", "cls": "Western Europe"}, {"id": 23038, "dom": "G", "ent": "Jericho", "cls": "Australia and New Zealand"}, {"id": 23039, "dom": "G", "ent": "Jericho", "cls": "Caribbean"}, {"id": 23040, "dom": "G", "ent": "Jericho", "cls": "Central America"}, {"id": 23041, "dom": "G", "ent": "Jericho", "cls": "Central Asia"}, {"id": 23042, "dom": "G", "ent": "Jericho", "cls": "Eastern Africa"}, {"id": 23043, "dom": "G", "ent": "Jericho", "cls": "Eastern Asia"}, {"id": 23044, "dom": "G", "ent": "Jericho", "cls": "Eastern Europe"}, {"id": 23045, "dom": "G", "ent": "Jericho", "cls": "Melanesia"}, {"id": 23046, "dom": "G", "ent": "Jericho", "cls": "Micronesia"}, {"id": 23047, "dom": "G", "ent": "Jericho", "cls": "Middle Africa"}, {"id": 23048, "dom": "G", "ent": "Jericho", "cls": "Northern Africa"}, {"id": 23049, "dom": "G", "ent": "Jericho", "cls": "Northern America"}, {"id": 23050, "dom": "G", "ent": "Jericho", "cls": "Northern Europe"}, {"id": 23051, "dom": "G", "ent": "Jericho", "cls": "Polynesia"}, {"id": 23052, "dom": "G", "ent": "Jericho", "cls": "South America"}, {"id": 23053, "dom": "G", "ent": "Jericho", "cls": "South-eastern Asia"}, {"id": 23054, "dom": "G", "ent": "Jericho", "cls": "Southern Africa"}, {"id": 23055, "dom": "G", "ent": "Jericho", "cls": "Southern Asia"}, {"id": 23056, "dom": "G", "ent": "Jericho", "cls": "Southern Europe"}, {"id": 23057, "dom": "G", "ent": "Jericho", "cls": "Western Africa"}, {"id": 23058, "dom": "G", "ent": "Jericho", "cls": "Western Asia"}, {"id": 23059, "dom": "G", "ent": "Jericho", "cls": "Western Europe"}, {"id": 23060, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "Australia and New Zealand"}, {"id": 23061, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "Caribbean"}, {"id": 23062, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "Central America"}, {"id": 23063, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "Central Asia"}, {"id": 23064, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "Eastern Africa"}, {"id": 23065, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "Eastern Asia"}, {"id": 23066, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "Eastern Europe"}, {"id": 23067, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "Melanesia"}, {"id": 23068, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "Micronesia"}, {"id": 23069, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "Middle Africa"}, {"id": 23070, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "Northern Africa"}, {"id": 23071, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "Northern America"}, {"id": 23072, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "Northern Europe"}, {"id": 23073, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "Polynesia"}, {"id": 23074, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "South America"}, {"id": 23075, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "South-eastern Asia"}, {"id": 23076, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "Southern Africa"}, {"id": 23077, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "Southern Asia"}, {"id": 23078, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "Southern Europe"}, {"id": 23079, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "Western Africa"}, {"id": 23080, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "Western Asia"}, {"id": 23081, "dom": "G", "ent": "Vila Nova de Gaia", "cls": "Western Europe"}, {"id": 23082, "dom": "G", "ent": "Arrentela", "cls": "Australia and New Zealand"}, {"id": 23083, "dom": "G", "ent": "Arrentela", "cls": "Caribbean"}, {"id": 23084, "dom": "G", "ent": "Arrentela", "cls": "Central America"}, {"id": 23085, "dom": "G", "ent": "Arrentela", "cls": "Central Asia"}, {"id": 23086, "dom": "G", "ent": "Arrentela", "cls": "Eastern Africa"}, {"id": 23087, "dom": "G", "ent": "Arrentela", "cls": "Eastern Asia"}, {"id": 23088, "dom": "G", "ent": "Arrentela", "cls": "Eastern Europe"}, {"id": 23089, "dom": "G", "ent": "Arrentela", "cls": "Melanesia"}, {"id": 23090, "dom": "G", "ent": "Arrentela", "cls": "Micronesia"}, {"id": 23091, "dom": "G", "ent": "Arrentela", "cls": "Middle Africa"}, {"id": 23092, "dom": "G", "ent": "Arrentela", "cls": "Northern Africa"}, {"id": 23093, "dom": "G", "ent": "Arrentela", "cls": "Northern America"}, {"id": 23094, "dom": "G", "ent": "Arrentela", "cls": "Northern Europe"}, {"id": 23095, "dom": "G", "ent": "Arrentela", "cls": "Polynesia"}, {"id": 23096, "dom": "G", "ent": "Arrentela", "cls": "South America"}, {"id": 23097, "dom": "G", "ent": "Arrentela", "cls": "South-eastern Asia"}, {"id": 23098, "dom": "G", "ent": "Arrentela", "cls": "Southern Africa"}, {"id": 23099, "dom": "G", "ent": "Arrentela", "cls": "Southern Asia"}, {"id": 23100, "dom": "G", "ent": "Arrentela", "cls": "Southern Europe"}, {"id": 23101, "dom": "G", "ent": "Arrentela", "cls": "Western Africa"}, {"id": 23102, "dom": "G", "ent": "Arrentela", "cls": "Western Asia"}, {"id": 23103, "dom": "G", "ent": "Arrentela", "cls": "Western Europe"}, {"id": 23104, "dom": "G", "ent": "Pontinha", "cls": "Australia and New Zealand"}, {"id": 23105, "dom": "G", "ent": "Pontinha", "cls": "Caribbean"}, {"id": 23106, "dom": "G", "ent": "Pontinha", "cls": "Central America"}, {"id": 23107, "dom": "G", "ent": "Pontinha", "cls": "Central Asia"}, {"id": 23108, "dom": "G", "ent": "Pontinha", "cls": "Eastern Africa"}, {"id": 23109, "dom": "G", "ent": "Pontinha", "cls": "Eastern Asia"}, {"id": 23110, "dom": "G", "ent": "Pontinha", "cls": "Eastern Europe"}, {"id": 23111, "dom": "G", "ent": "Pontinha", "cls": "Melanesia"}, {"id": 23112, "dom": "G", "ent": "Pontinha", "cls": "Micronesia"}, {"id": 23113, "dom": "G", "ent": "Pontinha", "cls": "Middle Africa"}, {"id": 23114, "dom": "G", "ent": "Pontinha", "cls": "Northern Africa"}, {"id": 23115, "dom": "G", "ent": "Pontinha", "cls": "Northern America"}, {"id": 23116, "dom": "G", "ent": "Pontinha", "cls": "Northern Europe"}, {"id": 23117, "dom": "G", "ent": "Pontinha", "cls": "Polynesia"}, {"id": 23118, "dom": "G", "ent": "Pontinha", "cls": "South America"}, {"id": 23119, "dom": "G", "ent": "Pontinha", "cls": "South-eastern Asia"}, {"id": 23120, "dom": "G", "ent": "Pontinha", "cls": "Southern Africa"}, {"id": 23121, "dom": "G", "ent": "Pontinha", "cls": "Southern Asia"}, {"id": 23122, "dom": "G", "ent": "Pontinha", "cls": "Southern Europe"}, {"id": 23123, "dom": "G", "ent": "Pontinha", "cls": "Western Africa"}, {"id": 23124, "dom": "G", "ent": "Pontinha", "cls": "Western Asia"}, {"id": 23125, "dom": "G", "ent": "Pontinha", "cls": "Western Europe"}, {"id": 23126, "dom": "G", "ent": "Baguim do Monte", "cls": "Australia and New Zealand"}, {"id": 23127, "dom": "G", "ent": "Baguim do Monte", "cls": "Caribbean"}, {"id": 23128, "dom": "G", "ent": "Baguim do Monte", "cls": "Central America"}, {"id": 23129, "dom": "G", "ent": "Baguim do Monte", "cls": "Central Asia"}, {"id": 23130, "dom": "G", "ent": "Baguim do Monte", "cls": "Eastern Africa"}, {"id": 23131, "dom": "G", "ent": "Baguim do Monte", "cls": "Eastern Asia"}, {"id": 23132, "dom": "G", "ent": "Baguim do Monte", "cls": "Eastern Europe"}, {"id": 23133, "dom": "G", "ent": "Baguim do Monte", "cls": "Melanesia"}, {"id": 23134, "dom": "G", "ent": "Baguim do Monte", "cls": "Micronesia"}, {"id": 23135, "dom": "G", "ent": "Baguim do Monte", "cls": "Middle Africa"}, {"id": 23136, "dom": "G", "ent": "Baguim do Monte", "cls": "Northern Africa"}, {"id": 23137, "dom": "G", "ent": "Baguim do Monte", "cls": "Northern America"}, {"id": 23138, "dom": "G", "ent": "Baguim do Monte", "cls": "Northern Europe"}, {"id": 23139, "dom": "G", "ent": "Baguim do Monte", "cls": "Polynesia"}, {"id": 23140, "dom": "G", "ent": "Baguim do Monte", "cls": "South America"}, {"id": 23141, "dom": "G", "ent": "Baguim do Monte", "cls": "South-eastern Asia"}, {"id": 23142, "dom": "G", "ent": "Baguim do Monte", "cls": "Southern Africa"}, {"id": 23143, "dom": "G", "ent": "Baguim do Monte", "cls": "Southern Asia"}, {"id": 23144, "dom": "G", "ent": "Baguim do Monte", "cls": "Southern Europe"}, {"id": 23145, "dom": "G", "ent": "Baguim do Monte", "cls": "Western Africa"}, {"id": 23146, "dom": "G", "ent": "Baguim do Monte", "cls": "Western Asia"}, {"id": 23147, "dom": "G", "ent": "Baguim do Monte", "cls": "Western Europe"}, {"id": 23148, "dom": "G", "ent": "Fernando de la Mora", "cls": "Australia and New Zealand"}, {"id": 23149, "dom": "G", "ent": "Fernando de la Mora", "cls": "Caribbean"}, {"id": 23150, "dom": "G", "ent": "Fernando de la Mora", "cls": "Central America"}, {"id": 23151, "dom": "G", "ent": "Fernando de la Mora", "cls": "Central Asia"}, {"id": 23152, "dom": "G", "ent": "Fernando de la Mora", "cls": "Eastern Africa"}, {"id": 23153, "dom": "G", "ent": "Fernando de la Mora", "cls": "Eastern Asia"}, {"id": 23154, "dom": "G", "ent": "Fernando de la Mora", "cls": "Eastern Europe"}, {"id": 23155, "dom": "G", "ent": "Fernando de la Mora", "cls": "Melanesia"}, {"id": 23156, "dom": "G", "ent": "Fernando de la Mora", "cls": "Micronesia"}, {"id": 23157, "dom": "G", "ent": "Fernando de la Mora", "cls": "Middle Africa"}, {"id": 23158, "dom": "G", "ent": "Fernando de la Mora", "cls": "Northern Africa"}, {"id": 23159, "dom": "G", "ent": "Fernando de la Mora", "cls": "Northern America"}, {"id": 23160, "dom": "G", "ent": "Fernando de la Mora", "cls": "Northern Europe"}, {"id": 23161, "dom": "G", "ent": "Fernando de la Mora", "cls": "Polynesia"}, {"id": 23162, "dom": "G", "ent": "Fernando de la Mora", "cls": "South America"}, {"id": 23163, "dom": "G", "ent": "Fernando de la Mora", "cls": "South-eastern Asia"}, {"id": 23164, "dom": "G", "ent": "Fernando de la Mora", "cls": "Southern Africa"}, {"id": 23165, "dom": "G", "ent": "Fernando de la Mora", "cls": "Southern Asia"}, {"id": 23166, "dom": "G", "ent": "Fernando de la Mora", "cls": "Southern Europe"}, {"id": 23167, "dom": "G", "ent": "Fernando de la Mora", "cls": "Western Africa"}, {"id": 23168, "dom": "G", "ent": "Fernando de la Mora", "cls": "Western Asia"}, {"id": 23169, "dom": "G", "ent": "Fernando de la Mora", "cls": "Western Europe"}, {"id": 23170, "dom": "G", "ent": "Villa Hayes", "cls": "Australia and New Zealand"}, {"id": 23171, "dom": "G", "ent": "Villa Hayes", "cls": "Caribbean"}, {"id": 23172, "dom": "G", "ent": "Villa Hayes", "cls": "Central America"}, {"id": 23173, "dom": "G", "ent": "Villa Hayes", "cls": "Central Asia"}, {"id": 23174, "dom": "G", "ent": "Villa Hayes", "cls": "Eastern Africa"}, {"id": 23175, "dom": "G", "ent": "Villa Hayes", "cls": "Eastern Asia"}, {"id": 23176, "dom": "G", "ent": "Villa Hayes", "cls": "Eastern Europe"}, {"id": 23177, "dom": "G", "ent": "Villa Hayes", "cls": "Melanesia"}, {"id": 23178, "dom": "G", "ent": "Villa Hayes", "cls": "Micronesia"}, {"id": 23179, "dom": "G", "ent": "Villa Hayes", "cls": "Middle Africa"}, {"id": 23180, "dom": "G", "ent": "Villa Hayes", "cls": "Northern Africa"}, {"id": 23181, "dom": "G", "ent": "Villa Hayes", "cls": "Northern America"}, {"id": 23182, "dom": "G", "ent": "Villa Hayes", "cls": "Northern Europe"}, {"id": 23183, "dom": "G", "ent": "Villa Hayes", "cls": "Polynesia"}, {"id": 23184, "dom": "G", "ent": "Villa Hayes", "cls": "South America"}, {"id": 23185, "dom": "G", "ent": "Villa Hayes", "cls": "South-eastern Asia"}, {"id": 23186, "dom": "G", "ent": "Villa Hayes", "cls": "Southern Africa"}, {"id": 23187, "dom": "G", "ent": "Villa Hayes", "cls": "Southern Asia"}, {"id": 23188, "dom": "G", "ent": "Villa Hayes", "cls": "Southern Europe"}, {"id": 23189, "dom": "G", "ent": "Villa Hayes", "cls": "Western Africa"}, {"id": 23190, "dom": "G", "ent": "Villa Hayes", "cls": "Western Asia"}, {"id": 23191, "dom": "G", "ent": "Villa Hayes", "cls": "Western Europe"}, {"id": 23192, "dom": "G", "ent": "Coronel Oviedo", "cls": "Australia and New Zealand"}, {"id": 23193, "dom": "G", "ent": "Coronel Oviedo", "cls": "Caribbean"}, {"id": 23194, "dom": "G", "ent": "Coronel Oviedo", "cls": "Central America"}, {"id": 23195, "dom": "G", "ent": "Coronel Oviedo", "cls": "Central Asia"}, {"id": 23196, "dom": "G", "ent": "Coronel Oviedo", "cls": "Eastern Africa"}, {"id": 23197, "dom": "G", "ent": "Coronel Oviedo", "cls": "Eastern Asia"}, {"id": 23198, "dom": "G", "ent": "Coronel Oviedo", "cls": "Eastern Europe"}, {"id": 23199, "dom": "G", "ent": "Coronel Oviedo", "cls": "Melanesia"}, {"id": 23200, "dom": "G", "ent": "Coronel Oviedo", "cls": "Micronesia"}, {"id": 23201, "dom": "G", "ent": "Coronel Oviedo", "cls": "Middle Africa"}, {"id": 23202, "dom": "G", "ent": "Coronel Oviedo", "cls": "Northern Africa"}, {"id": 23203, "dom": "G", "ent": "Coronel Oviedo", "cls": "Northern America"}, {"id": 23204, "dom": "G", "ent": "Coronel Oviedo", "cls": "Northern Europe"}, {"id": 23205, "dom": "G", "ent": "Coronel Oviedo", "cls": "Polynesia"}, {"id": 23206, "dom": "G", "ent": "Coronel Oviedo", "cls": "South America"}, {"id": 23207, "dom": "G", "ent": "Coronel Oviedo", "cls": "South-eastern Asia"}, {"id": 23208, "dom": "G", "ent": "Coronel Oviedo", "cls": "Southern Africa"}, {"id": 23209, "dom": "G", "ent": "Coronel Oviedo", "cls": "Southern Asia"}, {"id": 23210, "dom": "G", "ent": "Coronel Oviedo", "cls": "Southern Europe"}, {"id": 23211, "dom": "G", "ent": "Coronel Oviedo", "cls": "Western Africa"}, {"id": 23212, "dom": "G", "ent": "Coronel Oviedo", "cls": "Western Asia"}, {"id": 23213, "dom": "G", "ent": "Coronel Oviedo", "cls": "Western Europe"}, {"id": 23214, "dom": "G", "ent": "Loma Plata", "cls": "Australia and New Zealand"}, {"id": 23215, "dom": "G", "ent": "Loma Plata", "cls": "Caribbean"}, {"id": 23216, "dom": "G", "ent": "Loma Plata", "cls": "Central America"}, {"id": 23217, "dom": "G", "ent": "Loma Plata", "cls": "Central Asia"}, {"id": 23218, "dom": "G", "ent": "Loma Plata", "cls": "Eastern Africa"}, {"id": 23219, "dom": "G", "ent": "Loma Plata", "cls": "Eastern Asia"}, {"id": 23220, "dom": "G", "ent": "Loma Plata", "cls": "Eastern Europe"}, {"id": 23221, "dom": "G", "ent": "Loma Plata", "cls": "Melanesia"}, {"id": 23222, "dom": "G", "ent": "Loma Plata", "cls": "Micronesia"}, {"id": 23223, "dom": "G", "ent": "Loma Plata", "cls": "Middle Africa"}, {"id": 23224, "dom": "G", "ent": "Loma Plata", "cls": "Northern Africa"}, {"id": 23225, "dom": "G", "ent": "Loma Plata", "cls": "Northern America"}, {"id": 23226, "dom": "G", "ent": "Loma Plata", "cls": "Northern Europe"}, {"id": 23227, "dom": "G", "ent": "Loma Plata", "cls": "Polynesia"}, {"id": 23228, "dom": "G", "ent": "Loma Plata", "cls": "South America"}, {"id": 23229, "dom": "G", "ent": "Loma Plata", "cls": "South-eastern Asia"}, {"id": 23230, "dom": "G", "ent": "Loma Plata", "cls": "Southern Africa"}, {"id": 23231, "dom": "G", "ent": "Loma Plata", "cls": "Southern Asia"}, {"id": 23232, "dom": "G", "ent": "Loma Plata", "cls": "Southern Europe"}, {"id": 23233, "dom": "G", "ent": "Loma Plata", "cls": "Western Africa"}, {"id": 23234, "dom": "G", "ent": "Loma Plata", "cls": "Western Asia"}, {"id": 23235, "dom": "G", "ent": "Loma Plata", "cls": "Western Europe"}, {"id": 23236, "dom": "G", "ent": "Turda", "cls": "Australia and New Zealand"}, {"id": 23237, "dom": "G", "ent": "Turda", "cls": "Caribbean"}, {"id": 23238, "dom": "G", "ent": "Turda", "cls": "Central America"}, {"id": 23239, "dom": "G", "ent": "Turda", "cls": "Central Asia"}, {"id": 23240, "dom": "G", "ent": "Turda", "cls": "Eastern Africa"}, {"id": 23241, "dom": "G", "ent": "Turda", "cls": "Eastern Asia"}, {"id": 23242, "dom": "G", "ent": "Turda", "cls": "Eastern Europe"}, {"id": 23243, "dom": "G", "ent": "Turda", "cls": "Melanesia"}, {"id": 23244, "dom": "G", "ent": "Turda", "cls": "Micronesia"}, {"id": 23245, "dom": "G", "ent": "Turda", "cls": "Middle Africa"}, {"id": 23246, "dom": "G", "ent": "Turda", "cls": "Northern Africa"}, {"id": 23247, "dom": "G", "ent": "Turda", "cls": "Northern America"}, {"id": 23248, "dom": "G", "ent": "Turda", "cls": "Northern Europe"}, {"id": 23249, "dom": "G", "ent": "Turda", "cls": "Polynesia"}, {"id": 23250, "dom": "G", "ent": "Turda", "cls": "South America"}, {"id": 23251, "dom": "G", "ent": "Turda", "cls": "South-eastern Asia"}, {"id": 23252, "dom": "G", "ent": "Turda", "cls": "Southern Africa"}, {"id": 23253, "dom": "G", "ent": "Turda", "cls": "Southern Asia"}, {"id": 23254, "dom": "G", "ent": "Turda", "cls": "Southern Europe"}, {"id": 23255, "dom": "G", "ent": "Turda", "cls": "Western Africa"}, {"id": 23256, "dom": "G", "ent": "Turda", "cls": "Western Asia"}, {"id": 23257, "dom": "G", "ent": "Turda", "cls": "Western Europe"}, {"id": 23258, "dom": "G", "ent": "Lugoj", "cls": "Australia and New Zealand"}, {"id": 23259, "dom": "G", "ent": "Lugoj", "cls": "Caribbean"}, {"id": 23260, "dom": "G", "ent": "Lugoj", "cls": "Central America"}, {"id": 23261, "dom": "G", "ent": "Lugoj", "cls": "Central Asia"}, {"id": 23262, "dom": "G", "ent": "Lugoj", "cls": "Eastern Africa"}, {"id": 23263, "dom": "G", "ent": "Lugoj", "cls": "Eastern Asia"}, {"id": 23264, "dom": "G", "ent": "Lugoj", "cls": "Eastern Europe"}, {"id": 23265, "dom": "G", "ent": "Lugoj", "cls": "Melanesia"}, {"id": 23266, "dom": "G", "ent": "Lugoj", "cls": "Micronesia"}, {"id": 23267, "dom": "G", "ent": "Lugoj", "cls": "Middle Africa"}, {"id": 23268, "dom": "G", "ent": "Lugoj", "cls": "Northern Africa"}, {"id": 23269, "dom": "G", "ent": "Lugoj", "cls": "Northern America"}, {"id": 23270, "dom": "G", "ent": "Lugoj", "cls": "Northern Europe"}, {"id": 23271, "dom": "G", "ent": "Lugoj", "cls": "Polynesia"}, {"id": 23272, "dom": "G", "ent": "Lugoj", "cls": "South America"}, {"id": 23273, "dom": "G", "ent": "Lugoj", "cls": "South-eastern Asia"}, {"id": 23274, "dom": "G", "ent": "Lugoj", "cls": "Southern Africa"}, {"id": 23275, "dom": "G", "ent": "Lugoj", "cls": "Southern Asia"}, {"id": 23276, "dom": "G", "ent": "Lugoj", "cls": "Southern Europe"}, {"id": 23277, "dom": "G", "ent": "Lugoj", "cls": "Western Africa"}, {"id": 23278, "dom": "G", "ent": "Lugoj", "cls": "Western Asia"}, {"id": 23279, "dom": "G", "ent": "Lugoj", "cls": "Western Europe"}, {"id": 23280, "dom": "G", "ent": "Gherla", "cls": "Australia and New Zealand"}, {"id": 23281, "dom": "G", "ent": "Gherla", "cls": "Caribbean"}, {"id": 23282, "dom": "G", "ent": "Gherla", "cls": "Central America"}, {"id": 23283, "dom": "G", "ent": "Gherla", "cls": "Central Asia"}, {"id": 23284, "dom": "G", "ent": "Gherla", "cls": "Eastern Africa"}, {"id": 23285, "dom": "G", "ent": "Gherla", "cls": "Eastern Asia"}, {"id": 23286, "dom": "G", "ent": "Gherla", "cls": "Eastern Europe"}, {"id": 23287, "dom": "G", "ent": "Gherla", "cls": "Melanesia"}, {"id": 23288, "dom": "G", "ent": "Gherla", "cls": "Micronesia"}, {"id": 23289, "dom": "G", "ent": "Gherla", "cls": "Middle Africa"}, {"id": 23290, "dom": "G", "ent": "Gherla", "cls": "Northern Africa"}, {"id": 23291, "dom": "G", "ent": "Gherla", "cls": "Northern America"}, {"id": 23292, "dom": "G", "ent": "Gherla", "cls": "Northern Europe"}, {"id": 23293, "dom": "G", "ent": "Gherla", "cls": "Polynesia"}, {"id": 23294, "dom": "G", "ent": "Gherla", "cls": "South America"}, {"id": 23295, "dom": "G", "ent": "Gherla", "cls": "South-eastern Asia"}, {"id": 23296, "dom": "G", "ent": "Gherla", "cls": "Southern Africa"}, {"id": 23297, "dom": "G", "ent": "Gherla", "cls": "Southern Asia"}, {"id": 23298, "dom": "G", "ent": "Gherla", "cls": "Southern Europe"}, {"id": 23299, "dom": "G", "ent": "Gherla", "cls": "Western Africa"}, {"id": 23300, "dom": "G", "ent": "Gherla", "cls": "Western Asia"}, {"id": 23301, "dom": "G", "ent": "Gherla", "cls": "Western Europe"}, {"id": 23302, "dom": "G", "ent": "Gura Humorului", "cls": "Australia and New Zealand"}, {"id": 23303, "dom": "G", "ent": "Gura Humorului", "cls": "Caribbean"}, {"id": 23304, "dom": "G", "ent": "Gura Humorului", "cls": "Central America"}, {"id": 23305, "dom": "G", "ent": "Gura Humorului", "cls": "Central Asia"}, {"id": 23306, "dom": "G", "ent": "Gura Humorului", "cls": "Eastern Africa"}, {"id": 23307, "dom": "G", "ent": "Gura Humorului", "cls": "Eastern Asia"}, {"id": 23308, "dom": "G", "ent": "Gura Humorului", "cls": "Eastern Europe"}, {"id": 23309, "dom": "G", "ent": "Gura Humorului", "cls": "Melanesia"}, {"id": 23310, "dom": "G", "ent": "Gura Humorului", "cls": "Micronesia"}, {"id": 23311, "dom": "G", "ent": "Gura Humorului", "cls": "Middle Africa"}, {"id": 23312, "dom": "G", "ent": "Gura Humorului", "cls": "Northern Africa"}, {"id": 23313, "dom": "G", "ent": "Gura Humorului", "cls": "Northern America"}, {"id": 23314, "dom": "G", "ent": "Gura Humorului", "cls": "Northern Europe"}, {"id": 23315, "dom": "G", "ent": "Gura Humorului", "cls": "Polynesia"}, {"id": 23316, "dom": "G", "ent": "Gura Humorului", "cls": "South America"}, {"id": 23317, "dom": "G", "ent": "Gura Humorului", "cls": "South-eastern Asia"}, {"id": 23318, "dom": "G", "ent": "Gura Humorului", "cls": "Southern Africa"}, {"id": 23319, "dom": "G", "ent": "Gura Humorului", "cls": "Southern Asia"}, {"id": 23320, "dom": "G", "ent": "Gura Humorului", "cls": "Southern Europe"}, {"id": 23321, "dom": "G", "ent": "Gura Humorului", "cls": "Western Africa"}, {"id": 23322, "dom": "G", "ent": "Gura Humorului", "cls": "Western Asia"}, {"id": 23323, "dom": "G", "ent": "Gura Humorului", "cls": "Western Europe"}, {"id": 23324, "dom": "G", "ent": "Novi Pazar", "cls": "Australia and New Zealand"}, {"id": 23325, "dom": "G", "ent": "Novi Pazar", "cls": "Caribbean"}, {"id": 23326, "dom": "G", "ent": "Novi Pazar", "cls": "Central America"}, {"id": 23327, "dom": "G", "ent": "Novi Pazar", "cls": "Central Asia"}, {"id": 23328, "dom": "G", "ent": "Novi Pazar", "cls": "Eastern Africa"}, {"id": 23329, "dom": "G", "ent": "Novi Pazar", "cls": "Eastern Asia"}, {"id": 23330, "dom": "G", "ent": "Novi Pazar", "cls": "Eastern Europe"}, {"id": 23331, "dom": "G", "ent": "Novi Pazar", "cls": "Melanesia"}, {"id": 23332, "dom": "G", "ent": "Novi Pazar", "cls": "Micronesia"}, {"id": 23333, "dom": "G", "ent": "Novi Pazar", "cls": "Middle Africa"}, {"id": 23334, "dom": "G", "ent": "Novi Pazar", "cls": "Northern Africa"}, {"id": 23335, "dom": "G", "ent": "Novi Pazar", "cls": "Northern America"}, {"id": 23336, "dom": "G", "ent": "Novi Pazar", "cls": "Northern Europe"}, {"id": 23337, "dom": "G", "ent": "Novi Pazar", "cls": "Polynesia"}, {"id": 23338, "dom": "G", "ent": "Novi Pazar", "cls": "South America"}, {"id": 23339, "dom": "G", "ent": "Novi Pazar", "cls": "South-eastern Asia"}, {"id": 23340, "dom": "G", "ent": "Novi Pazar", "cls": "Southern Africa"}, {"id": 23341, "dom": "G", "ent": "Novi Pazar", "cls": "Southern Asia"}, {"id": 23342, "dom": "G", "ent": "Novi Pazar", "cls": "Southern Europe"}, {"id": 23343, "dom": "G", "ent": "Novi Pazar", "cls": "Western Africa"}, {"id": 23344, "dom": "G", "ent": "Novi Pazar", "cls": "Western Asia"}, {"id": 23345, "dom": "G", "ent": "Novi Pazar", "cls": "Western Europe"}, {"id": 23346, "dom": "G", "ent": "Kikinda", "cls": "Australia and New Zealand"}, {"id": 23347, "dom": "G", "ent": "Kikinda", "cls": "Caribbean"}, {"id": 23348, "dom": "G", "ent": "Kikinda", "cls": "Central America"}, {"id": 23349, "dom": "G", "ent": "Kikinda", "cls": "Central Asia"}, {"id": 23350, "dom": "G", "ent": "Kikinda", "cls": "Eastern Africa"}, {"id": 23351, "dom": "G", "ent": "Kikinda", "cls": "Eastern Asia"}, {"id": 23352, "dom": "G", "ent": "Kikinda", "cls": "Eastern Europe"}, {"id": 23353, "dom": "G", "ent": "Kikinda", "cls": "Melanesia"}, {"id": 23354, "dom": "G", "ent": "Kikinda", "cls": "Micronesia"}, {"id": 23355, "dom": "G", "ent": "Kikinda", "cls": "Middle Africa"}, {"id": 23356, "dom": "G", "ent": "Kikinda", "cls": "Northern Africa"}, {"id": 23357, "dom": "G", "ent": "Kikinda", "cls": "Northern America"}, {"id": 23358, "dom": "G", "ent": "Kikinda", "cls": "Northern Europe"}, {"id": 23359, "dom": "G", "ent": "Kikinda", "cls": "Polynesia"}, {"id": 23360, "dom": "G", "ent": "Kikinda", "cls": "South America"}, {"id": 23361, "dom": "G", "ent": "Kikinda", "cls": "South-eastern Asia"}, {"id": 23362, "dom": "G", "ent": "Kikinda", "cls": "Southern Africa"}, {"id": 23363, "dom": "G", "ent": "Kikinda", "cls": "Southern Asia"}, {"id": 23364, "dom": "G", "ent": "Kikinda", "cls": "Southern Europe"}, {"id": 23365, "dom": "G", "ent": "Kikinda", "cls": "Western Africa"}, {"id": 23366, "dom": "G", "ent": "Kikinda", "cls": "Western Asia"}, {"id": 23367, "dom": "G", "ent": "Kikinda", "cls": "Western Europe"}, {"id": 23368, "dom": "G", "ent": "Sremska Mitrovica", "cls": "Australia and New Zealand"}, {"id": 23369, "dom": "G", "ent": "Sremska Mitrovica", "cls": "Caribbean"}, {"id": 23370, "dom": "G", "ent": "Sremska Mitrovica", "cls": "Central America"}, {"id": 23371, "dom": "G", "ent": "Sremska Mitrovica", "cls": "Central Asia"}, {"id": 23372, "dom": "G", "ent": "Sremska Mitrovica", "cls": "Eastern Africa"}, {"id": 23373, "dom": "G", "ent": "Sremska Mitrovica", "cls": "Eastern Asia"}, {"id": 23374, "dom": "G", "ent": "Sremska Mitrovica", "cls": "Eastern Europe"}, {"id": 23375, "dom": "G", "ent": "Sremska Mitrovica", "cls": "Melanesia"}, {"id": 23376, "dom": "G", "ent": "Sremska Mitrovica", "cls": "Micronesia"}, {"id": 23377, "dom": "G", "ent": "Sremska Mitrovica", "cls": "Middle Africa"}, {"id": 23378, "dom": "G", "ent": "Sremska Mitrovica", "cls": "Northern Africa"}, {"id": 23379, "dom": "G", "ent": "Sremska Mitrovica", "cls": "Northern America"}, {"id": 23380, "dom": "G", "ent": "Sremska Mitrovica", "cls": "Northern Europe"}, {"id": 23381, "dom": "G", "ent": "Sremska Mitrovica", "cls": "Polynesia"}, {"id": 23382, "dom": "G", "ent": "Sremska Mitrovica", "cls": "South America"}, {"id": 23383, "dom": "G", "ent": "Sremska Mitrovica", "cls": "South-eastern Asia"}, {"id": 23384, "dom": "G", "ent": "Sremska Mitrovica", "cls": "Southern Africa"}, {"id": 23385, "dom": "G", "ent": "Sremska Mitrovica", "cls": "Southern Asia"}, {"id": 23386, "dom": "G", "ent": "Sremska Mitrovica", "cls": "Southern Europe"}, {"id": 23387, "dom": "G", "ent": "Sremska Mitrovica", "cls": "Western Africa"}, {"id": 23388, "dom": "G", "ent": "Sremska Mitrovica", "cls": "Western Asia"}, {"id": 23389, "dom": "G", "ent": "Sremska Mitrovica", "cls": "Western Europe"}, {"id": 23390, "dom": "G", "ent": "Vrbas", "cls": "Australia and New Zealand"}, {"id": 23391, "dom": "G", "ent": "Vrbas", "cls": "Caribbean"}, {"id": 23392, "dom": "G", "ent": "Vrbas", "cls": "Central America"}, {"id": 23393, "dom": "G", "ent": "Vrbas", "cls": "Central Asia"}, {"id": 23394, "dom": "G", "ent": "Vrbas", "cls": "Eastern Africa"}, {"id": 23395, "dom": "G", "ent": "Vrbas", "cls": "Eastern Asia"}, {"id": 23396, "dom": "G", "ent": "Vrbas", "cls": "Eastern Europe"}, {"id": 23397, "dom": "G", "ent": "Vrbas", "cls": "Melanesia"}, {"id": 23398, "dom": "G", "ent": "Vrbas", "cls": "Micronesia"}, {"id": 23399, "dom": "G", "ent": "Vrbas", "cls": "Middle Africa"}, {"id": 23400, "dom": "G", "ent": "Vrbas", "cls": "Northern Africa"}, {"id": 23401, "dom": "G", "ent": "Vrbas", "cls": "Northern America"}, {"id": 23402, "dom": "G", "ent": "Vrbas", "cls": "Northern Europe"}, {"id": 23403, "dom": "G", "ent": "Vrbas", "cls": "Polynesia"}, {"id": 23404, "dom": "G", "ent": "Vrbas", "cls": "South America"}, {"id": 23405, "dom": "G", "ent": "Vrbas", "cls": "South-eastern Asia"}, {"id": 23406, "dom": "G", "ent": "Vrbas", "cls": "Southern Africa"}, {"id": 23407, "dom": "G", "ent": "Vrbas", "cls": "Southern Asia"}, {"id": 23408, "dom": "G", "ent": "Vrbas", "cls": "Southern Europe"}, {"id": 23409, "dom": "G", "ent": "Vrbas", "cls": "Western Africa"}, {"id": 23410, "dom": "G", "ent": "Vrbas", "cls": "Western Asia"}, {"id": 23411, "dom": "G", "ent": "Vrbas", "cls": "Western Europe"}, {"id": 23412, "dom": "G", "ent": "Cyangugu", "cls": "Australia and New Zealand"}, {"id": 23413, "dom": "G", "ent": "Cyangugu", "cls": "Caribbean"}, {"id": 23414, "dom": "G", "ent": "Cyangugu", "cls": "Central America"}, {"id": 23415, "dom": "G", "ent": "Cyangugu", "cls": "Central Asia"}, {"id": 23416, "dom": "G", "ent": "Cyangugu", "cls": "Eastern Africa"}, {"id": 23417, "dom": "G", "ent": "Cyangugu", "cls": "Eastern Asia"}, {"id": 23418, "dom": "G", "ent": "Cyangugu", "cls": "Eastern Europe"}, {"id": 23419, "dom": "G", "ent": "Cyangugu", "cls": "Melanesia"}, {"id": 23420, "dom": "G", "ent": "Cyangugu", "cls": "Micronesia"}, {"id": 23421, "dom": "G", "ent": "Cyangugu", "cls": "Middle Africa"}, {"id": 23422, "dom": "G", "ent": "Cyangugu", "cls": "Northern Africa"}, {"id": 23423, "dom": "G", "ent": "Cyangugu", "cls": "Northern America"}, {"id": 23424, "dom": "G", "ent": "Cyangugu", "cls": "Northern Europe"}, {"id": 23425, "dom": "G", "ent": "Cyangugu", "cls": "Polynesia"}, {"id": 23426, "dom": "G", "ent": "Cyangugu", "cls": "South America"}, {"id": 23427, "dom": "G", "ent": "Cyangugu", "cls": "South-eastern Asia"}, {"id": 23428, "dom": "G", "ent": "Cyangugu", "cls": "Southern Africa"}, {"id": 23429, "dom": "G", "ent": "Cyangugu", "cls": "Southern Asia"}, {"id": 23430, "dom": "G", "ent": "Cyangugu", "cls": "Southern Europe"}, {"id": 23431, "dom": "G", "ent": "Cyangugu", "cls": "Western Africa"}, {"id": 23432, "dom": "G", "ent": "Cyangugu", "cls": "Western Asia"}, {"id": 23433, "dom": "G", "ent": "Cyangugu", "cls": "Western Europe"}, {"id": 23434, "dom": "G", "ent": "Nyabitekeli", "cls": "Australia and New Zealand"}, {"id": 23435, "dom": "G", "ent": "Nyabitekeli", "cls": "Caribbean"}, {"id": 23436, "dom": "G", "ent": "Nyabitekeli", "cls": "Central America"}, {"id": 23437, "dom": "G", "ent": "Nyabitekeli", "cls": "Central Asia"}, {"id": 23438, "dom": "G", "ent": "Nyabitekeli", "cls": "Eastern Africa"}, {"id": 23439, "dom": "G", "ent": "Nyabitekeli", "cls": "Eastern Asia"}, {"id": 23440, "dom": "G", "ent": "Nyabitekeli", "cls": "Eastern Europe"}, {"id": 23441, "dom": "G", "ent": "Nyabitekeli", "cls": "Melanesia"}, {"id": 23442, "dom": "G", "ent": "Nyabitekeli", "cls": "Micronesia"}, {"id": 23443, "dom": "G", "ent": "Nyabitekeli", "cls": "Middle Africa"}, {"id": 23444, "dom": "G", "ent": "Nyabitekeli", "cls": "Northern Africa"}, {"id": 23445, "dom": "G", "ent": "Nyabitekeli", "cls": "Northern America"}, {"id": 23446, "dom": "G", "ent": "Nyabitekeli", "cls": "Northern Europe"}, {"id": 23447, "dom": "G", "ent": "Nyabitekeli", "cls": "Polynesia"}, {"id": 23448, "dom": "G", "ent": "Nyabitekeli", "cls": "South America"}, {"id": 23449, "dom": "G", "ent": "Nyabitekeli", "cls": "South-eastern Asia"}, {"id": 23450, "dom": "G", "ent": "Nyabitekeli", "cls": "Southern Africa"}, {"id": 23451, "dom": "G", "ent": "Nyabitekeli", "cls": "Southern Asia"}, {"id": 23452, "dom": "G", "ent": "Nyabitekeli", "cls": "Southern Europe"}, {"id": 23453, "dom": "G", "ent": "Nyabitekeli", "cls": "Western Africa"}, {"id": 23454, "dom": "G", "ent": "Nyabitekeli", "cls": "Western Asia"}, {"id": 23455, "dom": "G", "ent": "Nyabitekeli", "cls": "Western Europe"}, {"id": 23456, "dom": "G", "ent": "Kayonza", "cls": "Australia and New Zealand"}, {"id": 23457, "dom": "G", "ent": "Kayonza", "cls": "Caribbean"}, {"id": 23458, "dom": "G", "ent": "Kayonza", "cls": "Central America"}, {"id": 23459, "dom": "G", "ent": "Kayonza", "cls": "Central Asia"}, {"id": 23460, "dom": "G", "ent": "Kayonza", "cls": "Eastern Africa"}, {"id": 23461, "dom": "G", "ent": "Kayonza", "cls": "Eastern Asia"}, {"id": 23462, "dom": "G", "ent": "Kayonza", "cls": "Eastern Europe"}, {"id": 23463, "dom": "G", "ent": "Kayonza", "cls": "Melanesia"}, {"id": 23464, "dom": "G", "ent": "Kayonza", "cls": "Micronesia"}, {"id": 23465, "dom": "G", "ent": "Kayonza", "cls": "Middle Africa"}, {"id": 23466, "dom": "G", "ent": "Kayonza", "cls": "Northern Africa"}, {"id": 23467, "dom": "G", "ent": "Kayonza", "cls": "Northern America"}, {"id": 23468, "dom": "G", "ent": "Kayonza", "cls": "Northern Europe"}, {"id": 23469, "dom": "G", "ent": "Kayonza", "cls": "Polynesia"}, {"id": 23470, "dom": "G", "ent": "Kayonza", "cls": "South America"}, {"id": 23471, "dom": "G", "ent": "Kayonza", "cls": "South-eastern Asia"}, {"id": 23472, "dom": "G", "ent": "Kayonza", "cls": "Southern Africa"}, {"id": 23473, "dom": "G", "ent": "Kayonza", "cls": "Southern Asia"}, {"id": 23474, "dom": "G", "ent": "Kayonza", "cls": "Southern Europe"}, {"id": 23475, "dom": "G", "ent": "Kayonza", "cls": "Western Africa"}, {"id": 23476, "dom": "G", "ent": "Kayonza", "cls": "Western Asia"}, {"id": 23477, "dom": "G", "ent": "Kayonza", "cls": "Western Europe"}, {"id": 23478, "dom": "G", "ent": "Umutara", "cls": "Australia and New Zealand"}, {"id": 23479, "dom": "G", "ent": "Umutara", "cls": "Caribbean"}, {"id": 23480, "dom": "G", "ent": "Umutara", "cls": "Central America"}, {"id": 23481, "dom": "G", "ent": "Umutara", "cls": "Central Asia"}, {"id": 23482, "dom": "G", "ent": "Umutara", "cls": "Eastern Africa"}, {"id": 23483, "dom": "G", "ent": "Umutara", "cls": "Eastern Asia"}, {"id": 23484, "dom": "G", "ent": "Umutara", "cls": "Eastern Europe"}, {"id": 23485, "dom": "G", "ent": "Umutara", "cls": "Melanesia"}, {"id": 23486, "dom": "G", "ent": "Umutara", "cls": "Micronesia"}, {"id": 23487, "dom": "G", "ent": "Umutara", "cls": "Middle Africa"}, {"id": 23488, "dom": "G", "ent": "Umutara", "cls": "Northern Africa"}, {"id": 23489, "dom": "G", "ent": "Umutara", "cls": "Northern America"}, {"id": 23490, "dom": "G", "ent": "Umutara", "cls": "Northern Europe"}, {"id": 23491, "dom": "G", "ent": "Umutara", "cls": "Polynesia"}, {"id": 23492, "dom": "G", "ent": "Umutara", "cls": "South America"}, {"id": 23493, "dom": "G", "ent": "Umutara", "cls": "South-eastern Asia"}, {"id": 23494, "dom": "G", "ent": "Umutara", "cls": "Southern Africa"}, {"id": 23495, "dom": "G", "ent": "Umutara", "cls": "Southern Asia"}, {"id": 23496, "dom": "G", "ent": "Umutara", "cls": "Southern Europe"}, {"id": 23497, "dom": "G", "ent": "Umutara", "cls": "Western Africa"}, {"id": 23498, "dom": "G", "ent": "Umutara", "cls": "Western Asia"}, {"id": 23499, "dom": "G", "ent": "Umutara", "cls": "Western Europe"}, {"id": 23500, "dom": "G", "ent": "Madinah", "cls": "Australia and New Zealand"}, {"id": 23501, "dom": "G", "ent": "Madinah", "cls": "Caribbean"}, {"id": 23502, "dom": "G", "ent": "Madinah", "cls": "Central America"}, {"id": 23503, "dom": "G", "ent": "Madinah", "cls": "Central Asia"}, {"id": 23504, "dom": "G", "ent": "Madinah", "cls": "Eastern Africa"}, {"id": 23505, "dom": "G", "ent": "Madinah", "cls": "Eastern Asia"}, {"id": 23506, "dom": "G", "ent": "Madinah", "cls": "Eastern Europe"}, {"id": 23507, "dom": "G", "ent": "Madinah", "cls": "Melanesia"}, {"id": 23508, "dom": "G", "ent": "Madinah", "cls": "Micronesia"}, {"id": 23509, "dom": "G", "ent": "Madinah", "cls": "Middle Africa"}, {"id": 23510, "dom": "G", "ent": "Madinah", "cls": "Northern Africa"}, {"id": 23511, "dom": "G", "ent": "Madinah", "cls": "Northern America"}, {"id": 23512, "dom": "G", "ent": "Madinah", "cls": "Northern Europe"}, {"id": 23513, "dom": "G", "ent": "Madinah", "cls": "Polynesia"}, {"id": 23514, "dom": "G", "ent": "Madinah", "cls": "South America"}, {"id": 23515, "dom": "G", "ent": "Madinah", "cls": "South-eastern Asia"}, {"id": 23516, "dom": "G", "ent": "Madinah", "cls": "Southern Africa"}, {"id": 23517, "dom": "G", "ent": "Madinah", "cls": "Southern Asia"}, {"id": 23518, "dom": "G", "ent": "Madinah", "cls": "Southern Europe"}, {"id": 23519, "dom": "G", "ent": "Madinah", "cls": "Western Africa"}, {"id": 23520, "dom": "G", "ent": "Madinah", "cls": "Western Asia"}, {"id": 23521, "dom": "G", "ent": "Madinah", "cls": "Western Europe"}, {"id": 23522, "dom": "G", "ent": "Sakakah", "cls": "Australia and New Zealand"}, {"id": 23523, "dom": "G", "ent": "Sakakah", "cls": "Caribbean"}, {"id": 23524, "dom": "G", "ent": "Sakakah", "cls": "Central America"}, {"id": 23525, "dom": "G", "ent": "Sakakah", "cls": "Central Asia"}, {"id": 23526, "dom": "G", "ent": "Sakakah", "cls": "Eastern Africa"}, {"id": 23527, "dom": "G", "ent": "Sakakah", "cls": "Eastern Asia"}, {"id": 23528, "dom": "G", "ent": "Sakakah", "cls": "Eastern Europe"}, {"id": 23529, "dom": "G", "ent": "Sakakah", "cls": "Melanesia"}, {"id": 23530, "dom": "G", "ent": "Sakakah", "cls": "Micronesia"}, {"id": 23531, "dom": "G", "ent": "Sakakah", "cls": "Middle Africa"}, {"id": 23532, "dom": "G", "ent": "Sakakah", "cls": "Northern Africa"}, {"id": 23533, "dom": "G", "ent": "Sakakah", "cls": "Northern America"}, {"id": 23534, "dom": "G", "ent": "Sakakah", "cls": "Northern Europe"}, {"id": 23535, "dom": "G", "ent": "Sakakah", "cls": "Polynesia"}, {"id": 23536, "dom": "G", "ent": "Sakakah", "cls": "South America"}, {"id": 23537, "dom": "G", "ent": "Sakakah", "cls": "South-eastern Asia"}, {"id": 23538, "dom": "G", "ent": "Sakakah", "cls": "Southern Africa"}, {"id": 23539, "dom": "G", "ent": "Sakakah", "cls": "Southern Asia"}, {"id": 23540, "dom": "G", "ent": "Sakakah", "cls": "Southern Europe"}, {"id": 23541, "dom": "G", "ent": "Sakakah", "cls": "Western Africa"}, {"id": 23542, "dom": "G", "ent": "Sakakah", "cls": "Western Asia"}, {"id": 23543, "dom": "G", "ent": "Sakakah", "cls": "Western Europe"}, {"id": 23544, "dom": "G", "ent": "King Faisal Military City", "cls": "Australia and New Zealand"}, {"id": 23545, "dom": "G", "ent": "King Faisal Military City", "cls": "Caribbean"}, {"id": 23546, "dom": "G", "ent": "King Faisal Military City", "cls": "Central America"}, {"id": 23547, "dom": "G", "ent": "King Faisal Military City", "cls": "Central Asia"}, {"id": 23548, "dom": "G", "ent": "King Faisal Military City", "cls": "Eastern Africa"}, {"id": 23549, "dom": "G", "ent": "King Faisal Military City", "cls": "Eastern Asia"}, {"id": 23550, "dom": "G", "ent": "King Faisal Military City", "cls": "Eastern Europe"}, {"id": 23551, "dom": "G", "ent": "King Faisal Military City", "cls": "Melanesia"}, {"id": 23552, "dom": "G", "ent": "King Faisal Military City", "cls": "Micronesia"}, {"id": 23553, "dom": "G", "ent": "King Faisal Military City", "cls": "Middle Africa"}, {"id": 23554, "dom": "G", "ent": "King Faisal Military City", "cls": "Northern Africa"}, {"id": 23555, "dom": "G", "ent": "King Faisal Military City", "cls": "Northern America"}, {"id": 23556, "dom": "G", "ent": "King Faisal Military City", "cls": "Northern Europe"}, {"id": 23557, "dom": "G", "ent": "King Faisal Military City", "cls": "Polynesia"}, {"id": 23558, "dom": "G", "ent": "King Faisal Military City", "cls": "South America"}, {"id": 23559, "dom": "G", "ent": "King Faisal Military City", "cls": "South-eastern Asia"}, {"id": 23560, "dom": "G", "ent": "King Faisal Military City", "cls": "Southern Africa"}, {"id": 23561, "dom": "G", "ent": "King Faisal Military City", "cls": "Southern Asia"}, {"id": 23562, "dom": "G", "ent": "King Faisal Military City", "cls": "Southern Europe"}, {"id": 23563, "dom": "G", "ent": "King Faisal Military City", "cls": "Western Africa"}, {"id": 23564, "dom": "G", "ent": "King Faisal Military City", "cls": "Western Asia"}, {"id": 23565, "dom": "G", "ent": "King Faisal Military City", "cls": "Western Europe"}, {"id": 23566, "dom": "G", "ent": "Afif", "cls": "Australia and New Zealand"}, {"id": 23567, "dom": "G", "ent": "Afif", "cls": "Caribbean"}, {"id": 23568, "dom": "G", "ent": "Afif", "cls": "Central America"}, {"id": 23569, "dom": "G", "ent": "Afif", "cls": "Central Asia"}, {"id": 23570, "dom": "G", "ent": "Afif", "cls": "Eastern Africa"}, {"id": 23571, "dom": "G", "ent": "Afif", "cls": "Eastern Asia"}, {"id": 23572, "dom": "G", "ent": "Afif", "cls": "Eastern Europe"}, {"id": 23573, "dom": "G", "ent": "Afif", "cls": "Melanesia"}, {"id": 23574, "dom": "G", "ent": "Afif", "cls": "Micronesia"}, {"id": 23575, "dom": "G", "ent": "Afif", "cls": "Middle Africa"}, {"id": 23576, "dom": "G", "ent": "Afif", "cls": "Northern Africa"}, {"id": 23577, "dom": "G", "ent": "Afif", "cls": "Northern America"}, {"id": 23578, "dom": "G", "ent": "Afif", "cls": "Northern Europe"}, {"id": 23579, "dom": "G", "ent": "Afif", "cls": "Polynesia"}, {"id": 23580, "dom": "G", "ent": "Afif", "cls": "South America"}, {"id": 23581, "dom": "G", "ent": "Afif", "cls": "South-eastern Asia"}, {"id": 23582, "dom": "G", "ent": "Afif", "cls": "Southern Africa"}, {"id": 23583, "dom": "G", "ent": "Afif", "cls": "Southern Asia"}, {"id": 23584, "dom": "G", "ent": "Afif", "cls": "Southern Europe"}, {"id": 23585, "dom": "G", "ent": "Afif", "cls": "Western Africa"}, {"id": 23586, "dom": "G", "ent": "Afif", "cls": "Western Asia"}, {"id": 23587, "dom": "G", "ent": "Afif", "cls": "Western Europe"}, {"id": 23588, "dom": "G", "ent": "Kassala", "cls": "Australia and New Zealand"}, {"id": 23589, "dom": "G", "ent": "Kassala", "cls": "Caribbean"}, {"id": 23590, "dom": "G", "ent": "Kassala", "cls": "Central America"}, {"id": 23591, "dom": "G", "ent": "Kassala", "cls": "Central Asia"}, {"id": 23592, "dom": "G", "ent": "Kassala", "cls": "Eastern Africa"}, {"id": 23593, "dom": "G", "ent": "Kassala", "cls": "Eastern Asia"}, {"id": 23594, "dom": "G", "ent": "Kassala", "cls": "Eastern Europe"}, {"id": 23595, "dom": "G", "ent": "Kassala", "cls": "Melanesia"}, {"id": 23596, "dom": "G", "ent": "Kassala", "cls": "Micronesia"}, {"id": 23597, "dom": "G", "ent": "Kassala", "cls": "Middle Africa"}, {"id": 23598, "dom": "G", "ent": "Kassala", "cls": "Northern Africa"}, {"id": 23599, "dom": "G", "ent": "Kassala", "cls": "Northern America"}, {"id": 23600, "dom": "G", "ent": "Kassala", "cls": "Northern Europe"}, {"id": 23601, "dom": "G", "ent": "Kassala", "cls": "Polynesia"}, {"id": 23602, "dom": "G", "ent": "Kassala", "cls": "South America"}, {"id": 23603, "dom": "G", "ent": "Kassala", "cls": "South-eastern Asia"}, {"id": 23604, "dom": "G", "ent": "Kassala", "cls": "Southern Africa"}, {"id": 23605, "dom": "G", "ent": "Kassala", "cls": "Southern Asia"}, {"id": 23606, "dom": "G", "ent": "Kassala", "cls": "Southern Europe"}, {"id": 23607, "dom": "G", "ent": "Kassala", "cls": "Western Africa"}, {"id": 23608, "dom": "G", "ent": "Kassala", "cls": "Western Asia"}, {"id": 23609, "dom": "G", "ent": "Kassala", "cls": "Western Europe"}, {"id": 23610, "dom": "G", "ent": "El Geneina Fort", "cls": "Australia and New Zealand"}, {"id": 23611, "dom": "G", "ent": "El Geneina Fort", "cls": "Caribbean"}, {"id": 23612, "dom": "G", "ent": "El Geneina Fort", "cls": "Central America"}, {"id": 23613, "dom": "G", "ent": "El Geneina Fort", "cls": "Central Asia"}, {"id": 23614, "dom": "G", "ent": "El Geneina Fort", "cls": "Eastern Africa"}, {"id": 23615, "dom": "G", "ent": "El Geneina Fort", "cls": "Eastern Asia"}, {"id": 23616, "dom": "G", "ent": "El Geneina Fort", "cls": "Eastern Europe"}, {"id": 23617, "dom": "G", "ent": "El Geneina Fort", "cls": "Melanesia"}, {"id": 23618, "dom": "G", "ent": "El Geneina Fort", "cls": "Micronesia"}, {"id": 23619, "dom": "G", "ent": "El Geneina Fort", "cls": "Middle Africa"}, {"id": 23620, "dom": "G", "ent": "El Geneina Fort", "cls": "Northern Africa"}, {"id": 23621, "dom": "G", "ent": "El Geneina Fort", "cls": "Northern America"}, {"id": 23622, "dom": "G", "ent": "El Geneina Fort", "cls": "Northern Europe"}, {"id": 23623, "dom": "G", "ent": "El Geneina Fort", "cls": "Polynesia"}, {"id": 23624, "dom": "G", "ent": "El Geneina Fort", "cls": "South America"}, {"id": 23625, "dom": "G", "ent": "El Geneina Fort", "cls": "South-eastern Asia"}, {"id": 23626, "dom": "G", "ent": "El Geneina Fort", "cls": "Southern Africa"}, {"id": 23627, "dom": "G", "ent": "El Geneina Fort", "cls": "Southern Asia"}, {"id": 23628, "dom": "G", "ent": "El Geneina Fort", "cls": "Southern Europe"}, {"id": 23629, "dom": "G", "ent": "El Geneina Fort", "cls": "Western Africa"}, {"id": 23630, "dom": "G", "ent": "El Geneina Fort", "cls": "Western Asia"}, {"id": 23631, "dom": "G", "ent": "El Geneina Fort", "cls": "Western Europe"}, {"id": 23632, "dom": "G", "ent": "Kadugli", "cls": "Australia and New Zealand"}, {"id": 23633, "dom": "G", "ent": "Kadugli", "cls": "Caribbean"}, {"id": 23634, "dom": "G", "ent": "Kadugli", "cls": "Central America"}, {"id": 23635, "dom": "G", "ent": "Kadugli", "cls": "Central Asia"}, {"id": 23636, "dom": "G", "ent": "Kadugli", "cls": "Eastern Africa"}, {"id": 23637, "dom": "G", "ent": "Kadugli", "cls": "Eastern Asia"}, {"id": 23638, "dom": "G", "ent": "Kadugli", "cls": "Eastern Europe"}, {"id": 23639, "dom": "G", "ent": "Kadugli", "cls": "Melanesia"}, {"id": 23640, "dom": "G", "ent": "Kadugli", "cls": "Micronesia"}, {"id": 23641, "dom": "G", "ent": "Kadugli", "cls": "Middle Africa"}, {"id": 23642, "dom": "G", "ent": "Kadugli", "cls": "Northern Africa"}, {"id": 23643, "dom": "G", "ent": "Kadugli", "cls": "Northern America"}, {"id": 23644, "dom": "G", "ent": "Kadugli", "cls": "Northern Europe"}, {"id": 23645, "dom": "G", "ent": "Kadugli", "cls": "Polynesia"}, {"id": 23646, "dom": "G", "ent": "Kadugli", "cls": "South America"}, {"id": 23647, "dom": "G", "ent": "Kadugli", "cls": "South-eastern Asia"}, {"id": 23648, "dom": "G", "ent": "Kadugli", "cls": "Southern Africa"}, {"id": 23649, "dom": "G", "ent": "Kadugli", "cls": "Southern Asia"}, {"id": 23650, "dom": "G", "ent": "Kadugli", "cls": "Southern Europe"}, {"id": 23651, "dom": "G", "ent": "Kadugli", "cls": "Western Africa"}, {"id": 23652, "dom": "G", "ent": "Kadugli", "cls": "Western Asia"}, {"id": 23653, "dom": "G", "ent": "Kadugli", "cls": "Western Europe"}, {"id": 23654, "dom": "G", "ent": "Al Hasaheisa", "cls": "Australia and New Zealand"}, {"id": 23655, "dom": "G", "ent": "Al Hasaheisa", "cls": "Caribbean"}, {"id": 23656, "dom": "G", "ent": "Al Hasaheisa", "cls": "Central America"}, {"id": 23657, "dom": "G", "ent": "Al Hasaheisa", "cls": "Central Asia"}, {"id": 23658, "dom": "G", "ent": "Al Hasaheisa", "cls": "Eastern Africa"}, {"id": 23659, "dom": "G", "ent": "Al Hasaheisa", "cls": "Eastern Asia"}, {"id": 23660, "dom": "G", "ent": "Al Hasaheisa", "cls": "Eastern Europe"}, {"id": 23661, "dom": "G", "ent": "Al Hasaheisa", "cls": "Melanesia"}, {"id": 23662, "dom": "G", "ent": "Al Hasaheisa", "cls": "Micronesia"}, {"id": 23663, "dom": "G", "ent": "Al Hasaheisa", "cls": "Middle Africa"}, {"id": 23664, "dom": "G", "ent": "Al Hasaheisa", "cls": "Northern Africa"}, {"id": 23665, "dom": "G", "ent": "Al Hasaheisa", "cls": "Northern America"}, {"id": 23666, "dom": "G", "ent": "Al Hasaheisa", "cls": "Northern Europe"}, {"id": 23667, "dom": "G", "ent": "Al Hasaheisa", "cls": "Polynesia"}, {"id": 23668, "dom": "G", "ent": "Al Hasaheisa", "cls": "South America"}, {"id": 23669, "dom": "G", "ent": "Al Hasaheisa", "cls": "South-eastern Asia"}, {"id": 23670, "dom": "G", "ent": "Al Hasaheisa", "cls": "Southern Africa"}, {"id": 23671, "dom": "G", "ent": "Al Hasaheisa", "cls": "Southern Asia"}, {"id": 23672, "dom": "G", "ent": "Al Hasaheisa", "cls": "Southern Europe"}, {"id": 23673, "dom": "G", "ent": "Al Hasaheisa", "cls": "Western Africa"}, {"id": 23674, "dom": "G", "ent": "Al Hasaheisa", "cls": "Western Asia"}, {"id": 23675, "dom": "G", "ent": "Al Hasaheisa", "cls": "Western Europe"}, {"id": 23676, "dom": "G", "ent": "Vasastaden", "cls": "Australia and New Zealand"}, {"id": 23677, "dom": "G", "ent": "Vasastaden", "cls": "Caribbean"}, {"id": 23678, "dom": "G", "ent": "Vasastaden", "cls": "Central America"}, {"id": 23679, "dom": "G", "ent": "Vasastaden", "cls": "Central Asia"}, {"id": 23680, "dom": "G", "ent": "Vasastaden", "cls": "Eastern Africa"}, {"id": 23681, "dom": "G", "ent": "Vasastaden", "cls": "Eastern Asia"}, {"id": 23682, "dom": "G", "ent": "Vasastaden", "cls": "Eastern Europe"}, {"id": 23683, "dom": "G", "ent": "Vasastaden", "cls": "Melanesia"}, {"id": 23684, "dom": "G", "ent": "Vasastaden", "cls": "Micronesia"}, {"id": 23685, "dom": "G", "ent": "Vasastaden", "cls": "Middle Africa"}, {"id": 23686, "dom": "G", "ent": "Vasastaden", "cls": "Northern Africa"}, {"id": 23687, "dom": "G", "ent": "Vasastaden", "cls": "Northern America"}, {"id": 23688, "dom": "G", "ent": "Vasastaden", "cls": "Northern Europe"}, {"id": 23689, "dom": "G", "ent": "Vasastaden", "cls": "Polynesia"}, {"id": 23690, "dom": "G", "ent": "Vasastaden", "cls": "South America"}, {"id": 23691, "dom": "G", "ent": "Vasastaden", "cls": "South-eastern Asia"}, {"id": 23692, "dom": "G", "ent": "Vasastaden", "cls": "Southern Africa"}, {"id": 23693, "dom": "G", "ent": "Vasastaden", "cls": "Southern Asia"}, {"id": 23694, "dom": "G", "ent": "Vasastaden", "cls": "Southern Europe"}, {"id": 23695, "dom": "G", "ent": "Vasastaden", "cls": "Western Africa"}, {"id": 23696, "dom": "G", "ent": "Vasastaden", "cls": "Western Asia"}, {"id": 23697, "dom": "G", "ent": "Vasastaden", "cls": "Western Europe"}, {"id": 23698, "dom": "G", "ent": "Tumba", "cls": "Australia and New Zealand"}, {"id": 23699, "dom": "G", "ent": "Tumba", "cls": "Caribbean"}, {"id": 23700, "dom": "G", "ent": "Tumba", "cls": "Central America"}, {"id": 23701, "dom": "G", "ent": "Tumba", "cls": "Central Asia"}, {"id": 23702, "dom": "G", "ent": "Tumba", "cls": "Eastern Africa"}, {"id": 23703, "dom": "G", "ent": "Tumba", "cls": "Eastern Asia"}, {"id": 23704, "dom": "G", "ent": "Tumba", "cls": "Eastern Europe"}, {"id": 23705, "dom": "G", "ent": "Tumba", "cls": "Melanesia"}, {"id": 23706, "dom": "G", "ent": "Tumba", "cls": "Micronesia"}, {"id": 23707, "dom": "G", "ent": "Tumba", "cls": "Middle Africa"}, {"id": 23708, "dom": "G", "ent": "Tumba", "cls": "Northern Africa"}, {"id": 23709, "dom": "G", "ent": "Tumba", "cls": "Northern America"}, {"id": 23710, "dom": "G", "ent": "Tumba", "cls": "Northern Europe"}, {"id": 23711, "dom": "G", "ent": "Tumba", "cls": "Polynesia"}, {"id": 23712, "dom": "G", "ent": "Tumba", "cls": "South America"}, {"id": 23713, "dom": "G", "ent": "Tumba", "cls": "South-eastern Asia"}, {"id": 23714, "dom": "G", "ent": "Tumba", "cls": "Southern Africa"}, {"id": 23715, "dom": "G", "ent": "Tumba", "cls": "Southern Asia"}, {"id": 23716, "dom": "G", "ent": "Tumba", "cls": "Southern Europe"}, {"id": 23717, "dom": "G", "ent": "Tumba", "cls": "Western Africa"}, {"id": 23718, "dom": "G", "ent": "Tumba", "cls": "Western Asia"}, {"id": 23719, "dom": "G", "ent": "Tumba", "cls": "Western Europe"}, {"id": 23720, "dom": "G", "ent": "Sundbyberg", "cls": "Australia and New Zealand"}, {"id": 23721, "dom": "G", "ent": "Sundbyberg", "cls": "Caribbean"}, {"id": 23722, "dom": "G", "ent": "Sundbyberg", "cls": "Central America"}, {"id": 23723, "dom": "G", "ent": "Sundbyberg", "cls": "Central Asia"}, {"id": 23724, "dom": "G", "ent": "Sundbyberg", "cls": "Eastern Africa"}, {"id": 23725, "dom": "G", "ent": "Sundbyberg", "cls": "Eastern Asia"}, {"id": 23726, "dom": "G", "ent": "Sundbyberg", "cls": "Eastern Europe"}, {"id": 23727, "dom": "G", "ent": "Sundbyberg", "cls": "Melanesia"}, {"id": 23728, "dom": "G", "ent": "Sundbyberg", "cls": "Micronesia"}, {"id": 23729, "dom": "G", "ent": "Sundbyberg", "cls": "Middle Africa"}, {"id": 23730, "dom": "G", "ent": "Sundbyberg", "cls": "Northern Africa"}, {"id": 23731, "dom": "G", "ent": "Sundbyberg", "cls": "Northern America"}, {"id": 23732, "dom": "G", "ent": "Sundbyberg", "cls": "Northern Europe"}, {"id": 23733, "dom": "G", "ent": "Sundbyberg", "cls": "Polynesia"}, {"id": 23734, "dom": "G", "ent": "Sundbyberg", "cls": "South America"}, {"id": 23735, "dom": "G", "ent": "Sundbyberg", "cls": "South-eastern Asia"}, {"id": 23736, "dom": "G", "ent": "Sundbyberg", "cls": "Southern Africa"}, {"id": 23737, "dom": "G", "ent": "Sundbyberg", "cls": "Southern Asia"}, {"id": 23738, "dom": "G", "ent": "Sundbyberg", "cls": "Southern Europe"}, {"id": 23739, "dom": "G", "ent": "Sundbyberg", "cls": "Western Africa"}, {"id": 23740, "dom": "G", "ent": "Sundbyberg", "cls": "Western Asia"}, {"id": 23741, "dom": "G", "ent": "Sundbyberg", "cls": "Western Europe"}, {"id": 23742, "dom": "G", "ent": "Gustavsberg", "cls": "Australia and New Zealand"}, {"id": 23743, "dom": "G", "ent": "Gustavsberg", "cls": "Caribbean"}, {"id": 23744, "dom": "G", "ent": "Gustavsberg", "cls": "Central America"}, {"id": 23745, "dom": "G", "ent": "Gustavsberg", "cls": "Central Asia"}, {"id": 23746, "dom": "G", "ent": "Gustavsberg", "cls": "Eastern Africa"}, {"id": 23747, "dom": "G", "ent": "Gustavsberg", "cls": "Eastern Asia"}, {"id": 23748, "dom": "G", "ent": "Gustavsberg", "cls": "Eastern Europe"}, {"id": 23749, "dom": "G", "ent": "Gustavsberg", "cls": "Melanesia"}, {"id": 23750, "dom": "G", "ent": "Gustavsberg", "cls": "Micronesia"}, {"id": 23751, "dom": "G", "ent": "Gustavsberg", "cls": "Middle Africa"}, {"id": 23752, "dom": "G", "ent": "Gustavsberg", "cls": "Northern Africa"}, {"id": 23753, "dom": "G", "ent": "Gustavsberg", "cls": "Northern America"}, {"id": 23754, "dom": "G", "ent": "Gustavsberg", "cls": "Northern Europe"}, {"id": 23755, "dom": "G", "ent": "Gustavsberg", "cls": "Polynesia"}, {"id": 23756, "dom": "G", "ent": "Gustavsberg", "cls": "South America"}, {"id": 23757, "dom": "G", "ent": "Gustavsberg", "cls": "South-eastern Asia"}, {"id": 23758, "dom": "G", "ent": "Gustavsberg", "cls": "Southern Africa"}, {"id": 23759, "dom": "G", "ent": "Gustavsberg", "cls": "Southern Asia"}, {"id": 23760, "dom": "G", "ent": "Gustavsberg", "cls": "Southern Europe"}, {"id": 23761, "dom": "G", "ent": "Gustavsberg", "cls": "Western Africa"}, {"id": 23762, "dom": "G", "ent": "Gustavsberg", "cls": "Western Asia"}, {"id": 23763, "dom": "G", "ent": "Gustavsberg", "cls": "Western Europe"}, {"id": 23764, "dom": "G", "ent": "Kangkar", "cls": "Australia and New Zealand"}, {"id": 23765, "dom": "G", "ent": "Kangkar", "cls": "Caribbean"}, {"id": 23766, "dom": "G", "ent": "Kangkar", "cls": "Central America"}, {"id": 23767, "dom": "G", "ent": "Kangkar", "cls": "Central Asia"}, {"id": 23768, "dom": "G", "ent": "Kangkar", "cls": "Eastern Africa"}, {"id": 23769, "dom": "G", "ent": "Kangkar", "cls": "Eastern Asia"}, {"id": 23770, "dom": "G", "ent": "Kangkar", "cls": "Eastern Europe"}, {"id": 23771, "dom": "G", "ent": "Kangkar", "cls": "Melanesia"}, {"id": 23772, "dom": "G", "ent": "Kangkar", "cls": "Micronesia"}, {"id": 23773, "dom": "G", "ent": "Kangkar", "cls": "Middle Africa"}, {"id": 23774, "dom": "G", "ent": "Kangkar", "cls": "Northern Africa"}, {"id": 23775, "dom": "G", "ent": "Kangkar", "cls": "Northern America"}, {"id": 23776, "dom": "G", "ent": "Kangkar", "cls": "Northern Europe"}, {"id": 23777, "dom": "G", "ent": "Kangkar", "cls": "Polynesia"}, {"id": 23778, "dom": "G", "ent": "Kangkar", "cls": "South America"}, {"id": 23779, "dom": "G", "ent": "Kangkar", "cls": "South-eastern Asia"}, {"id": 23780, "dom": "G", "ent": "Kangkar", "cls": "Southern Africa"}, {"id": 23781, "dom": "G", "ent": "Kangkar", "cls": "Southern Asia"}, {"id": 23782, "dom": "G", "ent": "Kangkar", "cls": "Southern Europe"}, {"id": 23783, "dom": "G", "ent": "Kangkar", "cls": "Western Africa"}, {"id": 23784, "dom": "G", "ent": "Kangkar", "cls": "Western Asia"}, {"id": 23785, "dom": "G", "ent": "Kangkar", "cls": "Western Europe"}, {"id": 23786, "dom": "G", "ent": "Senja", "cls": "Australia and New Zealand"}, {"id": 23787, "dom": "G", "ent": "Senja", "cls": "Caribbean"}, {"id": 23788, "dom": "G", "ent": "Senja", "cls": "Central America"}, {"id": 23789, "dom": "G", "ent": "Senja", "cls": "Central Asia"}, {"id": 23790, "dom": "G", "ent": "Senja", "cls": "Eastern Africa"}, {"id": 23791, "dom": "G", "ent": "Senja", "cls": "Eastern Asia"}, {"id": 23792, "dom": "G", "ent": "Senja", "cls": "Eastern Europe"}, {"id": 23793, "dom": "G", "ent": "Senja", "cls": "Melanesia"}, {"id": 23794, "dom": "G", "ent": "Senja", "cls": "Micronesia"}, {"id": 23795, "dom": "G", "ent": "Senja", "cls": "Middle Africa"}, {"id": 23796, "dom": "G", "ent": "Senja", "cls": "Northern Africa"}, {"id": 23797, "dom": "G", "ent": "Senja", "cls": "Northern America"}, {"id": 23798, "dom": "G", "ent": "Senja", "cls": "Northern Europe"}, {"id": 23799, "dom": "G", "ent": "Senja", "cls": "Polynesia"}, {"id": 23800, "dom": "G", "ent": "Senja", "cls": "South America"}, {"id": 23801, "dom": "G", "ent": "Senja", "cls": "South-eastern Asia"}, {"id": 23802, "dom": "G", "ent": "Senja", "cls": "Southern Africa"}, {"id": 23803, "dom": "G", "ent": "Senja", "cls": "Southern Asia"}, {"id": 23804, "dom": "G", "ent": "Senja", "cls": "Southern Europe"}, {"id": 23805, "dom": "G", "ent": "Senja", "cls": "Western Africa"}, {"id": 23806, "dom": "G", "ent": "Senja", "cls": "Western Asia"}, {"id": 23807, "dom": "G", "ent": "Senja", "cls": "Western Europe"}, {"id": 23808, "dom": "G", "ent": "Telok Blangah", "cls": "Australia and New Zealand"}, {"id": 23809, "dom": "G", "ent": "Telok Blangah", "cls": "Caribbean"}, {"id": 23810, "dom": "G", "ent": "Telok Blangah", "cls": "Central America"}, {"id": 23811, "dom": "G", "ent": "Telok Blangah", "cls": "Central Asia"}, {"id": 23812, "dom": "G", "ent": "Telok Blangah", "cls": "Eastern Africa"}, {"id": 23813, "dom": "G", "ent": "Telok Blangah", "cls": "Eastern Asia"}, {"id": 23814, "dom": "G", "ent": "Telok Blangah", "cls": "Eastern Europe"}, {"id": 23815, "dom": "G", "ent": "Telok Blangah", "cls": "Melanesia"}, {"id": 23816, "dom": "G", "ent": "Telok Blangah", "cls": "Micronesia"}, {"id": 23817, "dom": "G", "ent": "Telok Blangah", "cls": "Middle Africa"}, {"id": 23818, "dom": "G", "ent": "Telok Blangah", "cls": "Northern Africa"}, {"id": 23819, "dom": "G", "ent": "Telok Blangah", "cls": "Northern America"}, {"id": 23820, "dom": "G", "ent": "Telok Blangah", "cls": "Northern Europe"}, {"id": 23821, "dom": "G", "ent": "Telok Blangah", "cls": "Polynesia"}, {"id": 23822, "dom": "G", "ent": "Telok Blangah", "cls": "South America"}, {"id": 23823, "dom": "G", "ent": "Telok Blangah", "cls": "South-eastern Asia"}, {"id": 23824, "dom": "G", "ent": "Telok Blangah", "cls": "Southern Africa"}, {"id": 23825, "dom": "G", "ent": "Telok Blangah", "cls": "Southern Asia"}, {"id": 23826, "dom": "G", "ent": "Telok Blangah", "cls": "Southern Europe"}, {"id": 23827, "dom": "G", "ent": "Telok Blangah", "cls": "Western Africa"}, {"id": 23828, "dom": "G", "ent": "Telok Blangah", "cls": "Western Asia"}, {"id": 23829, "dom": "G", "ent": "Telok Blangah", "cls": "Western Europe"}, {"id": 23830, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "Australia and New Zealand"}, {"id": 23831, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "Caribbean"}, {"id": 23832, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "Central America"}, {"id": 23833, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "Central Asia"}, {"id": 23834, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "Eastern Africa"}, {"id": 23835, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "Eastern Asia"}, {"id": 23836, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "Eastern Europe"}, {"id": 23837, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "Melanesia"}, {"id": 23838, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "Micronesia"}, {"id": 23839, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "Middle Africa"}, {"id": 23840, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "Northern Africa"}, {"id": 23841, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "Northern America"}, {"id": 23842, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "Northern Europe"}, {"id": 23843, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "Polynesia"}, {"id": 23844, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "South America"}, {"id": 23845, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "South-eastern Asia"}, {"id": 23846, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "Southern Africa"}, {"id": 23847, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "Southern Asia"}, {"id": 23848, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "Southern Europe"}, {"id": 23849, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "Western Africa"}, {"id": 23850, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "Western Asia"}, {"id": 23851, "dom": "G", "ent": "Tiong Bahru Estate", "cls": "Western Europe"}, {"id": 23852, "dom": "G", "ent": "Maribor", "cls": "Australia and New Zealand"}, {"id": 23853, "dom": "G", "ent": "Maribor", "cls": "Caribbean"}, {"id": 23854, "dom": "G", "ent": "Maribor", "cls": "Central America"}, {"id": 23855, "dom": "G", "ent": "Maribor", "cls": "Central Asia"}, {"id": 23856, "dom": "G", "ent": "Maribor", "cls": "Eastern Africa"}, {"id": 23857, "dom": "G", "ent": "Maribor", "cls": "Eastern Asia"}, {"id": 23858, "dom": "G", "ent": "Maribor", "cls": "Eastern Europe"}, {"id": 23859, "dom": "G", "ent": "Maribor", "cls": "Melanesia"}, {"id": 23860, "dom": "G", "ent": "Maribor", "cls": "Micronesia"}, {"id": 23861, "dom": "G", "ent": "Maribor", "cls": "Middle Africa"}, {"id": 23862, "dom": "G", "ent": "Maribor", "cls": "Northern Africa"}, {"id": 23863, "dom": "G", "ent": "Maribor", "cls": "Northern America"}, {"id": 23864, "dom": "G", "ent": "Maribor", "cls": "Northern Europe"}, {"id": 23865, "dom": "G", "ent": "Maribor", "cls": "Polynesia"}, {"id": 23866, "dom": "G", "ent": "Maribor", "cls": "South America"}, {"id": 23867, "dom": "G", "ent": "Maribor", "cls": "South-eastern Asia"}, {"id": 23868, "dom": "G", "ent": "Maribor", "cls": "Southern Africa"}, {"id": 23869, "dom": "G", "ent": "Maribor", "cls": "Southern Asia"}, {"id": 23870, "dom": "G", "ent": "Maribor", "cls": "Southern Europe"}, {"id": 23871, "dom": "G", "ent": "Maribor", "cls": "Western Africa"}, {"id": 23872, "dom": "G", "ent": "Maribor", "cls": "Western Asia"}, {"id": 23873, "dom": "G", "ent": "Maribor", "cls": "Western Europe"}, {"id": 23874, "dom": "G", "ent": "Kranj", "cls": "Australia and New Zealand"}, {"id": 23875, "dom": "G", "ent": "Kranj", "cls": "Caribbean"}, {"id": 23876, "dom": "G", "ent": "Kranj", "cls": "Central America"}, {"id": 23877, "dom": "G", "ent": "Kranj", "cls": "Central Asia"}, {"id": 23878, "dom": "G", "ent": "Kranj", "cls": "Eastern Africa"}, {"id": 23879, "dom": "G", "ent": "Kranj", "cls": "Eastern Asia"}, {"id": 23880, "dom": "G", "ent": "Kranj", "cls": "Eastern Europe"}, {"id": 23881, "dom": "G", "ent": "Kranj", "cls": "Melanesia"}, {"id": 23882, "dom": "G", "ent": "Kranj", "cls": "Micronesia"}, {"id": 23883, "dom": "G", "ent": "Kranj", "cls": "Middle Africa"}, {"id": 23884, "dom": "G", "ent": "Kranj", "cls": "Northern Africa"}, {"id": 23885, "dom": "G", "ent": "Kranj", "cls": "Northern America"}, {"id": 23886, "dom": "G", "ent": "Kranj", "cls": "Northern Europe"}, {"id": 23887, "dom": "G", "ent": "Kranj", "cls": "Polynesia"}, {"id": 23888, "dom": "G", "ent": "Kranj", "cls": "South America"}, {"id": 23889, "dom": "G", "ent": "Kranj", "cls": "South-eastern Asia"}, {"id": 23890, "dom": "G", "ent": "Kranj", "cls": "Southern Africa"}, {"id": 23891, "dom": "G", "ent": "Kranj", "cls": "Southern Asia"}, {"id": 23892, "dom": "G", "ent": "Kranj", "cls": "Southern Europe"}, {"id": 23893, "dom": "G", "ent": "Kranj", "cls": "Western Africa"}, {"id": 23894, "dom": "G", "ent": "Kranj", "cls": "Western Asia"}, {"id": 23895, "dom": "G", "ent": "Kranj", "cls": "Western Europe"}, {"id": 23896, "dom": "G", "ent": "Novo Mesto", "cls": "Australia and New Zealand"}, {"id": 23897, "dom": "G", "ent": "Novo Mesto", "cls": "Caribbean"}, {"id": 23898, "dom": "G", "ent": "Novo Mesto", "cls": "Central America"}, {"id": 23899, "dom": "G", "ent": "Novo Mesto", "cls": "Central Asia"}, {"id": 23900, "dom": "G", "ent": "Novo Mesto", "cls": "Eastern Africa"}, {"id": 23901, "dom": "G", "ent": "Novo Mesto", "cls": "Eastern Asia"}, {"id": 23902, "dom": "G", "ent": "Novo Mesto", "cls": "Eastern Europe"}, {"id": 23903, "dom": "G", "ent": "Novo Mesto", "cls": "Melanesia"}, {"id": 23904, "dom": "G", "ent": "Novo Mesto", "cls": "Micronesia"}, {"id": 23905, "dom": "G", "ent": "Novo Mesto", "cls": "Middle Africa"}, {"id": 23906, "dom": "G", "ent": "Novo Mesto", "cls": "Northern Africa"}, {"id": 23907, "dom": "G", "ent": "Novo Mesto", "cls": "Northern America"}, {"id": 23908, "dom": "G", "ent": "Novo Mesto", "cls": "Northern Europe"}, {"id": 23909, "dom": "G", "ent": "Novo Mesto", "cls": "Polynesia"}, {"id": 23910, "dom": "G", "ent": "Novo Mesto", "cls": "South America"}, {"id": 23911, "dom": "G", "ent": "Novo Mesto", "cls": "South-eastern Asia"}, {"id": 23912, "dom": "G", "ent": "Novo Mesto", "cls": "Southern Africa"}, {"id": 23913, "dom": "G", "ent": "Novo Mesto", "cls": "Southern Asia"}, {"id": 23914, "dom": "G", "ent": "Novo Mesto", "cls": "Southern Europe"}, {"id": 23915, "dom": "G", "ent": "Novo Mesto", "cls": "Western Africa"}, {"id": 23916, "dom": "G", "ent": "Novo Mesto", "cls": "Western Asia"}, {"id": 23917, "dom": "G", "ent": "Novo Mesto", "cls": "Western Europe"}, {"id": 23918, "dom": "G", "ent": "Trbovlje", "cls": "Australia and New Zealand"}, {"id": 23919, "dom": "G", "ent": "Trbovlje", "cls": "Caribbean"}, {"id": 23920, "dom": "G", "ent": "Trbovlje", "cls": "Central America"}, {"id": 23921, "dom": "G", "ent": "Trbovlje", "cls": "Central Asia"}, {"id": 23922, "dom": "G", "ent": "Trbovlje", "cls": "Eastern Africa"}, {"id": 23923, "dom": "G", "ent": "Trbovlje", "cls": "Eastern Asia"}, {"id": 23924, "dom": "G", "ent": "Trbovlje", "cls": "Eastern Europe"}, {"id": 23925, "dom": "G", "ent": "Trbovlje", "cls": "Melanesia"}, {"id": 23926, "dom": "G", "ent": "Trbovlje", "cls": "Micronesia"}, {"id": 23927, "dom": "G", "ent": "Trbovlje", "cls": "Middle Africa"}, {"id": 23928, "dom": "G", "ent": "Trbovlje", "cls": "Northern Africa"}, {"id": 23929, "dom": "G", "ent": "Trbovlje", "cls": "Northern America"}, {"id": 23930, "dom": "G", "ent": "Trbovlje", "cls": "Northern Europe"}, {"id": 23931, "dom": "G", "ent": "Trbovlje", "cls": "Polynesia"}, {"id": 23932, "dom": "G", "ent": "Trbovlje", "cls": "South America"}, {"id": 23933, "dom": "G", "ent": "Trbovlje", "cls": "South-eastern Asia"}, {"id": 23934, "dom": "G", "ent": "Trbovlje", "cls": "Southern Africa"}, {"id": 23935, "dom": "G", "ent": "Trbovlje", "cls": "Southern Asia"}, {"id": 23936, "dom": "G", "ent": "Trbovlje", "cls": "Southern Europe"}, {"id": 23937, "dom": "G", "ent": "Trbovlje", "cls": "Western Africa"}, {"id": 23938, "dom": "G", "ent": "Trbovlje", "cls": "Western Asia"}, {"id": 23939, "dom": "G", "ent": "Trbovlje", "cls": "Western Europe"}, {"id": 23940, "dom": "G", "ent": "Trnava", "cls": "Australia and New Zealand"}, {"id": 23941, "dom": "G", "ent": "Trnava", "cls": "Caribbean"}, {"id": 23942, "dom": "G", "ent": "Trnava", "cls": "Central America"}, {"id": 23943, "dom": "G", "ent": "Trnava", "cls": "Central Asia"}, {"id": 23944, "dom": "G", "ent": "Trnava", "cls": "Eastern Africa"}, {"id": 23945, "dom": "G", "ent": "Trnava", "cls": "Eastern Asia"}, {"id": 23946, "dom": "G", "ent": "Trnava", "cls": "Eastern Europe"}, {"id": 23947, "dom": "G", "ent": "Trnava", "cls": "Melanesia"}, {"id": 23948, "dom": "G", "ent": "Trnava", "cls": "Micronesia"}, {"id": 23949, "dom": "G", "ent": "Trnava", "cls": "Middle Africa"}, {"id": 23950, "dom": "G", "ent": "Trnava", "cls": "Northern Africa"}, {"id": 23951, "dom": "G", "ent": "Trnava", "cls": "Northern America"}, {"id": 23952, "dom": "G", "ent": "Trnava", "cls": "Northern Europe"}, {"id": 23953, "dom": "G", "ent": "Trnava", "cls": "Polynesia"}, {"id": 23954, "dom": "G", "ent": "Trnava", "cls": "South America"}, {"id": 23955, "dom": "G", "ent": "Trnava", "cls": "South-eastern Asia"}, {"id": 23956, "dom": "G", "ent": "Trnava", "cls": "Southern Africa"}, {"id": 23957, "dom": "G", "ent": "Trnava", "cls": "Southern Asia"}, {"id": 23958, "dom": "G", "ent": "Trnava", "cls": "Southern Europe"}, {"id": 23959, "dom": "G", "ent": "Trnava", "cls": "Western Africa"}, {"id": 23960, "dom": "G", "ent": "Trnava", "cls": "Western Asia"}, {"id": 23961, "dom": "G", "ent": "Trnava", "cls": "Western Europe"}, {"id": 23962, "dom": "G", "ent": "Michalovce", "cls": "Australia and New Zealand"}, {"id": 23963, "dom": "G", "ent": "Michalovce", "cls": "Caribbean"}, {"id": 23964, "dom": "G", "ent": "Michalovce", "cls": "Central America"}, {"id": 23965, "dom": "G", "ent": "Michalovce", "cls": "Central Asia"}, {"id": 23966, "dom": "G", "ent": "Michalovce", "cls": "Eastern Africa"}, {"id": 23967, "dom": "G", "ent": "Michalovce", "cls": "Eastern Asia"}, {"id": 23968, "dom": "G", "ent": "Michalovce", "cls": "Eastern Europe"}, {"id": 23969, "dom": "G", "ent": "Michalovce", "cls": "Melanesia"}, {"id": 23970, "dom": "G", "ent": "Michalovce", "cls": "Micronesia"}, {"id": 23971, "dom": "G", "ent": "Michalovce", "cls": "Middle Africa"}, {"id": 23972, "dom": "G", "ent": "Michalovce", "cls": "Northern Africa"}, {"id": 23973, "dom": "G", "ent": "Michalovce", "cls": "Northern America"}, {"id": 23974, "dom": "G", "ent": "Michalovce", "cls": "Northern Europe"}, {"id": 23975, "dom": "G", "ent": "Michalovce", "cls": "Polynesia"}, {"id": 23976, "dom": "G", "ent": "Michalovce", "cls": "South America"}, {"id": 23977, "dom": "G", "ent": "Michalovce", "cls": "South-eastern Asia"}, {"id": 23978, "dom": "G", "ent": "Michalovce", "cls": "Southern Africa"}, {"id": 23979, "dom": "G", "ent": "Michalovce", "cls": "Southern Asia"}, {"id": 23980, "dom": "G", "ent": "Michalovce", "cls": "Southern Europe"}, {"id": 23981, "dom": "G", "ent": "Michalovce", "cls": "Western Africa"}, {"id": 23982, "dom": "G", "ent": "Michalovce", "cls": "Western Asia"}, {"id": 23983, "dom": "G", "ent": "Michalovce", "cls": "Western Europe"}, {"id": 23984, "dom": "G", "ent": "Bardejov", "cls": "Australia and New Zealand"}, {"id": 23985, "dom": "G", "ent": "Bardejov", "cls": "Caribbean"}, {"id": 23986, "dom": "G", "ent": "Bardejov", "cls": "Central America"}, {"id": 23987, "dom": "G", "ent": "Bardejov", "cls": "Central Asia"}, {"id": 23988, "dom": "G", "ent": "Bardejov", "cls": "Eastern Africa"}, {"id": 23989, "dom": "G", "ent": "Bardejov", "cls": "Eastern Asia"}, {"id": 23990, "dom": "G", "ent": "Bardejov", "cls": "Eastern Europe"}, {"id": 23991, "dom": "G", "ent": "Bardejov", "cls": "Melanesia"}, {"id": 23992, "dom": "G", "ent": "Bardejov", "cls": "Micronesia"}, {"id": 23993, "dom": "G", "ent": "Bardejov", "cls": "Middle Africa"}, {"id": 23994, "dom": "G", "ent": "Bardejov", "cls": "Northern Africa"}, {"id": 23995, "dom": "G", "ent": "Bardejov", "cls": "Northern America"}, {"id": 23996, "dom": "G", "ent": "Bardejov", "cls": "Northern Europe"}, {"id": 23997, "dom": "G", "ent": "Bardejov", "cls": "Polynesia"}, {"id": 23998, "dom": "G", "ent": "Bardejov", "cls": "South America"}, {"id": 23999, "dom": "G", "ent": "Bardejov", "cls": "South-eastern Asia"}, {"id": 24000, "dom": "G", "ent": "Bardejov", "cls": "Southern Africa"}, {"id": 24001, "dom": "G", "ent": "Bardejov", "cls": "Southern Asia"}, {"id": 24002, "dom": "G", "ent": "Bardejov", "cls": "Southern Europe"}, {"id": 24003, "dom": "G", "ent": "Bardejov", "cls": "Western Africa"}, {"id": 24004, "dom": "G", "ent": "Bardejov", "cls": "Western Asia"}, {"id": 24005, "dom": "G", "ent": "Bardejov", "cls": "Western Europe"}, {"id": 24006, "dom": "G", "ent": "Malacky", "cls": "Australia and New Zealand"}, {"id": 24007, "dom": "G", "ent": "Malacky", "cls": "Caribbean"}, {"id": 24008, "dom": "G", "ent": "Malacky", "cls": "Central America"}, {"id": 24009, "dom": "G", "ent": "Malacky", "cls": "Central Asia"}, {"id": 24010, "dom": "G", "ent": "Malacky", "cls": "Eastern Africa"}, {"id": 24011, "dom": "G", "ent": "Malacky", "cls": "Eastern Asia"}, {"id": 24012, "dom": "G", "ent": "Malacky", "cls": "Eastern Europe"}, {"id": 24013, "dom": "G", "ent": "Malacky", "cls": "Melanesia"}, {"id": 24014, "dom": "G", "ent": "Malacky", "cls": "Micronesia"}, {"id": 24015, "dom": "G", "ent": "Malacky", "cls": "Middle Africa"}, {"id": 24016, "dom": "G", "ent": "Malacky", "cls": "Northern Africa"}, {"id": 24017, "dom": "G", "ent": "Malacky", "cls": "Northern America"}, {"id": 24018, "dom": "G", "ent": "Malacky", "cls": "Northern Europe"}, {"id": 24019, "dom": "G", "ent": "Malacky", "cls": "Polynesia"}, {"id": 24020, "dom": "G", "ent": "Malacky", "cls": "South America"}, {"id": 24021, "dom": "G", "ent": "Malacky", "cls": "South-eastern Asia"}, {"id": 24022, "dom": "G", "ent": "Malacky", "cls": "Southern Africa"}, {"id": 24023, "dom": "G", "ent": "Malacky", "cls": "Southern Asia"}, {"id": 24024, "dom": "G", "ent": "Malacky", "cls": "Southern Europe"}, {"id": 24025, "dom": "G", "ent": "Malacky", "cls": "Western Africa"}, {"id": 24026, "dom": "G", "ent": "Malacky", "cls": "Western Asia"}, {"id": 24027, "dom": "G", "ent": "Malacky", "cls": "Western Europe"}, {"id": 24028, "dom": "G", "ent": "Kenema", "cls": "Australia and New Zealand"}, {"id": 24029, "dom": "G", "ent": "Kenema", "cls": "Caribbean"}, {"id": 24030, "dom": "G", "ent": "Kenema", "cls": "Central America"}, {"id": 24031, "dom": "G", "ent": "Kenema", "cls": "Central Asia"}, {"id": 24032, "dom": "G", "ent": "Kenema", "cls": "Eastern Africa"}, {"id": 24033, "dom": "G", "ent": "Kenema", "cls": "Eastern Asia"}, {"id": 24034, "dom": "G", "ent": "Kenema", "cls": "Eastern Europe"}, {"id": 24035, "dom": "G", "ent": "Kenema", "cls": "Melanesia"}, {"id": 24036, "dom": "G", "ent": "Kenema", "cls": "Micronesia"}, {"id": 24037, "dom": "G", "ent": "Kenema", "cls": "Middle Africa"}, {"id": 24038, "dom": "G", "ent": "Kenema", "cls": "Northern Africa"}, {"id": 24039, "dom": "G", "ent": "Kenema", "cls": "Northern America"}, {"id": 24040, "dom": "G", "ent": "Kenema", "cls": "Northern Europe"}, {"id": 24041, "dom": "G", "ent": "Kenema", "cls": "Polynesia"}, {"id": 24042, "dom": "G", "ent": "Kenema", "cls": "South America"}, {"id": 24043, "dom": "G", "ent": "Kenema", "cls": "South-eastern Asia"}, {"id": 24044, "dom": "G", "ent": "Kenema", "cls": "Southern Africa"}, {"id": 24045, "dom": "G", "ent": "Kenema", "cls": "Southern Asia"}, {"id": 24046, "dom": "G", "ent": "Kenema", "cls": "Southern Europe"}, {"id": 24047, "dom": "G", "ent": "Kenema", "cls": "Western Africa"}, {"id": 24048, "dom": "G", "ent": "Kenema", "cls": "Western Asia"}, {"id": 24049, "dom": "G", "ent": "Kenema", "cls": "Western Europe"}, {"id": 24050, "dom": "G", "ent": "Port Loko", "cls": "Australia and New Zealand"}, {"id": 24051, "dom": "G", "ent": "Port Loko", "cls": "Caribbean"}, {"id": 24052, "dom": "G", "ent": "Port Loko", "cls": "Central America"}, {"id": 24053, "dom": "G", "ent": "Port Loko", "cls": "Central Asia"}, {"id": 24054, "dom": "G", "ent": "Port Loko", "cls": "Eastern Africa"}, {"id": 24055, "dom": "G", "ent": "Port Loko", "cls": "Eastern Asia"}, {"id": 24056, "dom": "G", "ent": "Port Loko", "cls": "Eastern Europe"}, {"id": 24057, "dom": "G", "ent": "Port Loko", "cls": "Melanesia"}, {"id": 24058, "dom": "G", "ent": "Port Loko", "cls": "Micronesia"}, {"id": 24059, "dom": "G", "ent": "Port Loko", "cls": "Middle Africa"}, {"id": 24060, "dom": "G", "ent": "Port Loko", "cls": "Northern Africa"}, {"id": 24061, "dom": "G", "ent": "Port Loko", "cls": "Northern America"}, {"id": 24062, "dom": "G", "ent": "Port Loko", "cls": "Northern Europe"}, {"id": 24063, "dom": "G", "ent": "Port Loko", "cls": "Polynesia"}, {"id": 24064, "dom": "G", "ent": "Port Loko", "cls": "South America"}, {"id": 24065, "dom": "G", "ent": "Port Loko", "cls": "South-eastern Asia"}, {"id": 24066, "dom": "G", "ent": "Port Loko", "cls": "Southern Africa"}, {"id": 24067, "dom": "G", "ent": "Port Loko", "cls": "Southern Asia"}, {"id": 24068, "dom": "G", "ent": "Port Loko", "cls": "Southern Europe"}, {"id": 24069, "dom": "G", "ent": "Port Loko", "cls": "Western Africa"}, {"id": 24070, "dom": "G", "ent": "Port Loko", "cls": "Western Asia"}, {"id": 24071, "dom": "G", "ent": "Port Loko", "cls": "Western Europe"}, {"id": 24072, "dom": "G", "ent": "Lunsar", "cls": "Australia and New Zealand"}, {"id": 24073, "dom": "G", "ent": "Lunsar", "cls": "Caribbean"}, {"id": 24074, "dom": "G", "ent": "Lunsar", "cls": "Central America"}, {"id": 24075, "dom": "G", "ent": "Lunsar", "cls": "Central Asia"}, {"id": 24076, "dom": "G", "ent": "Lunsar", "cls": "Eastern Africa"}, {"id": 24077, "dom": "G", "ent": "Lunsar", "cls": "Eastern Asia"}, {"id": 24078, "dom": "G", "ent": "Lunsar", "cls": "Eastern Europe"}, {"id": 24079, "dom": "G", "ent": "Lunsar", "cls": "Melanesia"}, {"id": 24080, "dom": "G", "ent": "Lunsar", "cls": "Micronesia"}, {"id": 24081, "dom": "G", "ent": "Lunsar", "cls": "Middle Africa"}, {"id": 24082, "dom": "G", "ent": "Lunsar", "cls": "Northern Africa"}, {"id": 24083, "dom": "G", "ent": "Lunsar", "cls": "Northern America"}, {"id": 24084, "dom": "G", "ent": "Lunsar", "cls": "Northern Europe"}, {"id": 24085, "dom": "G", "ent": "Lunsar", "cls": "Polynesia"}, {"id": 24086, "dom": "G", "ent": "Lunsar", "cls": "South America"}, {"id": 24087, "dom": "G", "ent": "Lunsar", "cls": "South-eastern Asia"}, {"id": 24088, "dom": "G", "ent": "Lunsar", "cls": "Southern Africa"}, {"id": 24089, "dom": "G", "ent": "Lunsar", "cls": "Southern Asia"}, {"id": 24090, "dom": "G", "ent": "Lunsar", "cls": "Southern Europe"}, {"id": 24091, "dom": "G", "ent": "Lunsar", "cls": "Western Africa"}, {"id": 24092, "dom": "G", "ent": "Lunsar", "cls": "Western Asia"}, {"id": 24093, "dom": "G", "ent": "Lunsar", "cls": "Western Europe"}, {"id": 24094, "dom": "G", "ent": "Goderich", "cls": "Australia and New Zealand"}, {"id": 24095, "dom": "G", "ent": "Goderich", "cls": "Caribbean"}, {"id": 24096, "dom": "G", "ent": "Goderich", "cls": "Central America"}, {"id": 24097, "dom": "G", "ent": "Goderich", "cls": "Central Asia"}, {"id": 24098, "dom": "G", "ent": "Goderich", "cls": "Eastern Africa"}, {"id": 24099, "dom": "G", "ent": "Goderich", "cls": "Eastern Asia"}, {"id": 24100, "dom": "G", "ent": "Goderich", "cls": "Eastern Europe"}, {"id": 24101, "dom": "G", "ent": "Goderich", "cls": "Melanesia"}, {"id": 24102, "dom": "G", "ent": "Goderich", "cls": "Micronesia"}, {"id": 24103, "dom": "G", "ent": "Goderich", "cls": "Middle Africa"}, {"id": 24104, "dom": "G", "ent": "Goderich", "cls": "Northern Africa"}, {"id": 24105, "dom": "G", "ent": "Goderich", "cls": "Northern America"}, {"id": 24106, "dom": "G", "ent": "Goderich", "cls": "Northern Europe"}, {"id": 24107, "dom": "G", "ent": "Goderich", "cls": "Polynesia"}, {"id": 24108, "dom": "G", "ent": "Goderich", "cls": "South America"}, {"id": 24109, "dom": "G", "ent": "Goderich", "cls": "South-eastern Asia"}, {"id": 24110, "dom": "G", "ent": "Goderich", "cls": "Southern Africa"}, {"id": 24111, "dom": "G", "ent": "Goderich", "cls": "Southern Asia"}, {"id": 24112, "dom": "G", "ent": "Goderich", "cls": "Southern Europe"}, {"id": 24113, "dom": "G", "ent": "Goderich", "cls": "Western Africa"}, {"id": 24114, "dom": "G", "ent": "Goderich", "cls": "Western Asia"}, {"id": 24115, "dom": "G", "ent": "Goderich", "cls": "Western Europe"}, {"id": 24116, "dom": "G", "ent": "Kolda", "cls": "Australia and New Zealand"}, {"id": 24117, "dom": "G", "ent": "Kolda", "cls": "Caribbean"}, {"id": 24118, "dom": "G", "ent": "Kolda", "cls": "Central America"}, {"id": 24119, "dom": "G", "ent": "Kolda", "cls": "Central Asia"}, {"id": 24120, "dom": "G", "ent": "Kolda", "cls": "Eastern Africa"}, {"id": 24121, "dom": "G", "ent": "Kolda", "cls": "Eastern Asia"}, {"id": 24122, "dom": "G", "ent": "Kolda", "cls": "Eastern Europe"}, {"id": 24123, "dom": "G", "ent": "Kolda", "cls": "Melanesia"}, {"id": 24124, "dom": "G", "ent": "Kolda", "cls": "Micronesia"}, {"id": 24125, "dom": "G", "ent": "Kolda", "cls": "Middle Africa"}, {"id": 24126, "dom": "G", "ent": "Kolda", "cls": "Northern Africa"}, {"id": 24127, "dom": "G", "ent": "Kolda", "cls": "Northern America"}, {"id": 24128, "dom": "G", "ent": "Kolda", "cls": "Northern Europe"}, {"id": 24129, "dom": "G", "ent": "Kolda", "cls": "Polynesia"}, {"id": 24130, "dom": "G", "ent": "Kolda", "cls": "South America"}, {"id": 24131, "dom": "G", "ent": "Kolda", "cls": "South-eastern Asia"}, {"id": 24132, "dom": "G", "ent": "Kolda", "cls": "Southern Africa"}, {"id": 24133, "dom": "G", "ent": "Kolda", "cls": "Southern Asia"}, {"id": 24134, "dom": "G", "ent": "Kolda", "cls": "Southern Europe"}, {"id": 24135, "dom": "G", "ent": "Kolda", "cls": "Western Africa"}, {"id": 24136, "dom": "G", "ent": "Kolda", "cls": "Western Asia"}, {"id": 24137, "dom": "G", "ent": "Kolda", "cls": "Western Europe"}, {"id": 24138, "dom": "G", "ent": "Mboro", "cls": "Australia and New Zealand"}, {"id": 24139, "dom": "G", "ent": "Mboro", "cls": "Caribbean"}, {"id": 24140, "dom": "G", "ent": "Mboro", "cls": "Central America"}, {"id": 24141, "dom": "G", "ent": "Mboro", "cls": "Central Asia"}, {"id": 24142, "dom": "G", "ent": "Mboro", "cls": "Eastern Africa"}, {"id": 24143, "dom": "G", "ent": "Mboro", "cls": "Eastern Asia"}, {"id": 24144, "dom": "G", "ent": "Mboro", "cls": "Eastern Europe"}, {"id": 24145, "dom": "G", "ent": "Mboro", "cls": "Melanesia"}, {"id": 24146, "dom": "G", "ent": "Mboro", "cls": "Micronesia"}, {"id": 24147, "dom": "G", "ent": "Mboro", "cls": "Middle Africa"}, {"id": 24148, "dom": "G", "ent": "Mboro", "cls": "Northern Africa"}, {"id": 24149, "dom": "G", "ent": "Mboro", "cls": "Northern America"}, {"id": 24150, "dom": "G", "ent": "Mboro", "cls": "Northern Europe"}, {"id": 24151, "dom": "G", "ent": "Mboro", "cls": "Polynesia"}, {"id": 24152, "dom": "G", "ent": "Mboro", "cls": "South America"}, {"id": 24153, "dom": "G", "ent": "Mboro", "cls": "South-eastern Asia"}, {"id": 24154, "dom": "G", "ent": "Mboro", "cls": "Southern Africa"}, {"id": 24155, "dom": "G", "ent": "Mboro", "cls": "Southern Asia"}, {"id": 24156, "dom": "G", "ent": "Mboro", "cls": "Southern Europe"}, {"id": 24157, "dom": "G", "ent": "Mboro", "cls": "Western Africa"}, {"id": 24158, "dom": "G", "ent": "Mboro", "cls": "Western Asia"}, {"id": 24159, "dom": "G", "ent": "Mboro", "cls": "Western Europe"}, {"id": 24160, "dom": "G", "ent": "Kayar", "cls": "Australia and New Zealand"}, {"id": 24161, "dom": "G", "ent": "Kayar", "cls": "Caribbean"}, {"id": 24162, "dom": "G", "ent": "Kayar", "cls": "Central America"}, {"id": 24163, "dom": "G", "ent": "Kayar", "cls": "Central Asia"}, {"id": 24164, "dom": "G", "ent": "Kayar", "cls": "Eastern Africa"}, {"id": 24165, "dom": "G", "ent": "Kayar", "cls": "Eastern Asia"}, {"id": 24166, "dom": "G", "ent": "Kayar", "cls": "Eastern Europe"}, {"id": 24167, "dom": "G", "ent": "Kayar", "cls": "Melanesia"}, {"id": 24168, "dom": "G", "ent": "Kayar", "cls": "Micronesia"}, {"id": 24169, "dom": "G", "ent": "Kayar", "cls": "Middle Africa"}, {"id": 24170, "dom": "G", "ent": "Kayar", "cls": "Northern Africa"}, {"id": 24171, "dom": "G", "ent": "Kayar", "cls": "Northern America"}, {"id": 24172, "dom": "G", "ent": "Kayar", "cls": "Northern Europe"}, {"id": 24173, "dom": "G", "ent": "Kayar", "cls": "Polynesia"}, {"id": 24174, "dom": "G", "ent": "Kayar", "cls": "South America"}, {"id": 24175, "dom": "G", "ent": "Kayar", "cls": "South-eastern Asia"}, {"id": 24176, "dom": "G", "ent": "Kayar", "cls": "Southern Africa"}, {"id": 24177, "dom": "G", "ent": "Kayar", "cls": "Southern Asia"}, {"id": 24178, "dom": "G", "ent": "Kayar", "cls": "Southern Europe"}, {"id": 24179, "dom": "G", "ent": "Kayar", "cls": "Western Africa"}, {"id": 24180, "dom": "G", "ent": "Kayar", "cls": "Western Asia"}, {"id": 24181, "dom": "G", "ent": "Kayar", "cls": "Western Europe"}, {"id": 24182, "dom": "G", "ent": "Kbombole", "cls": "Australia and New Zealand"}, {"id": 24183, "dom": "G", "ent": "Kbombole", "cls": "Caribbean"}, {"id": 24184, "dom": "G", "ent": "Kbombole", "cls": "Central America"}, {"id": 24185, "dom": "G", "ent": "Kbombole", "cls": "Central Asia"}, {"id": 24186, "dom": "G", "ent": "Kbombole", "cls": "Eastern Africa"}, {"id": 24187, "dom": "G", "ent": "Kbombole", "cls": "Eastern Asia"}, {"id": 24188, "dom": "G", "ent": "Kbombole", "cls": "Eastern Europe"}, {"id": 24189, "dom": "G", "ent": "Kbombole", "cls": "Melanesia"}, {"id": 24190, "dom": "G", "ent": "Kbombole", "cls": "Micronesia"}, {"id": 24191, "dom": "G", "ent": "Kbombole", "cls": "Middle Africa"}, {"id": 24192, "dom": "G", "ent": "Kbombole", "cls": "Northern Africa"}, {"id": 24193, "dom": "G", "ent": "Kbombole", "cls": "Northern America"}, {"id": 24194, "dom": "G", "ent": "Kbombole", "cls": "Northern Europe"}, {"id": 24195, "dom": "G", "ent": "Kbombole", "cls": "Polynesia"}, {"id": 24196, "dom": "G", "ent": "Kbombole", "cls": "South America"}, {"id": 24197, "dom": "G", "ent": "Kbombole", "cls": "South-eastern Asia"}, {"id": 24198, "dom": "G", "ent": "Kbombole", "cls": "Southern Africa"}, {"id": 24199, "dom": "G", "ent": "Kbombole", "cls": "Southern Asia"}, {"id": 24200, "dom": "G", "ent": "Kbombole", "cls": "Southern Europe"}, {"id": 24201, "dom": "G", "ent": "Kbombole", "cls": "Western Africa"}, {"id": 24202, "dom": "G", "ent": "Kbombole", "cls": "Western Asia"}, {"id": 24203, "dom": "G", "ent": "Kbombole", "cls": "Western Europe"}, {"id": 24204, "dom": "G", "ent": "Berbera", "cls": "Australia and New Zealand"}, {"id": 24205, "dom": "G", "ent": "Berbera", "cls": "Caribbean"}, {"id": 24206, "dom": "G", "ent": "Berbera", "cls": "Central America"}, {"id": 24207, "dom": "G", "ent": "Berbera", "cls": "Central Asia"}, {"id": 24208, "dom": "G", "ent": "Berbera", "cls": "Eastern Africa"}, {"id": 24209, "dom": "G", "ent": "Berbera", "cls": "Eastern Asia"}, {"id": 24210, "dom": "G", "ent": "Berbera", "cls": "Eastern Europe"}, {"id": 24211, "dom": "G", "ent": "Berbera", "cls": "Melanesia"}, {"id": 24212, "dom": "G", "ent": "Berbera", "cls": "Micronesia"}, {"id": 24213, "dom": "G", "ent": "Berbera", "cls": "Middle Africa"}, {"id": 24214, "dom": "G", "ent": "Berbera", "cls": "Northern Africa"}, {"id": 24215, "dom": "G", "ent": "Berbera", "cls": "Northern America"}, {"id": 24216, "dom": "G", "ent": "Berbera", "cls": "Northern Europe"}, {"id": 24217, "dom": "G", "ent": "Berbera", "cls": "Polynesia"}, {"id": 24218, "dom": "G", "ent": "Berbera", "cls": "South America"}, {"id": 24219, "dom": "G", "ent": "Berbera", "cls": "South-eastern Asia"}, {"id": 24220, "dom": "G", "ent": "Berbera", "cls": "Southern Africa"}, {"id": 24221, "dom": "G", "ent": "Berbera", "cls": "Southern Asia"}, {"id": 24222, "dom": "G", "ent": "Berbera", "cls": "Southern Europe"}, {"id": 24223, "dom": "G", "ent": "Berbera", "cls": "Western Africa"}, {"id": 24224, "dom": "G", "ent": "Berbera", "cls": "Western Asia"}, {"id": 24225, "dom": "G", "ent": "Berbera", "cls": "Western Europe"}, {"id": 24226, "dom": "G", "ent": "Balanbale", "cls": "Australia and New Zealand"}, {"id": 24227, "dom": "G", "ent": "Balanbale", "cls": "Caribbean"}, {"id": 24228, "dom": "G", "ent": "Balanbale", "cls": "Central America"}, {"id": 24229, "dom": "G", "ent": "Balanbale", "cls": "Central Asia"}, {"id": 24230, "dom": "G", "ent": "Balanbale", "cls": "Eastern Africa"}, {"id": 24231, "dom": "G", "ent": "Balanbale", "cls": "Eastern Asia"}, {"id": 24232, "dom": "G", "ent": "Balanbale", "cls": "Eastern Europe"}, {"id": 24233, "dom": "G", "ent": "Balanbale", "cls": "Melanesia"}, {"id": 24234, "dom": "G", "ent": "Balanbale", "cls": "Micronesia"}, {"id": 24235, "dom": "G", "ent": "Balanbale", "cls": "Middle Africa"}, {"id": 24236, "dom": "G", "ent": "Balanbale", "cls": "Northern Africa"}, {"id": 24237, "dom": "G", "ent": "Balanbale", "cls": "Northern America"}, {"id": 24238, "dom": "G", "ent": "Balanbale", "cls": "Northern Europe"}, {"id": 24239, "dom": "G", "ent": "Balanbale", "cls": "Polynesia"}, {"id": 24240, "dom": "G", "ent": "Balanbale", "cls": "South America"}, {"id": 24241, "dom": "G", "ent": "Balanbale", "cls": "South-eastern Asia"}, {"id": 24242, "dom": "G", "ent": "Balanbale", "cls": "Southern Africa"}, {"id": 24243, "dom": "G", "ent": "Balanbale", "cls": "Southern Asia"}, {"id": 24244, "dom": "G", "ent": "Balanbale", "cls": "Southern Europe"}, {"id": 24245, "dom": "G", "ent": "Balanbale", "cls": "Western Africa"}, {"id": 24246, "dom": "G", "ent": "Balanbale", "cls": "Western Asia"}, {"id": 24247, "dom": "G", "ent": "Balanbale", "cls": "Western Europe"}, {"id": 24248, "dom": "G", "ent": "Dhabad", "cls": "Australia and New Zealand"}, {"id": 24249, "dom": "G", "ent": "Dhabad", "cls": "Caribbean"}, {"id": 24250, "dom": "G", "ent": "Dhabad", "cls": "Central America"}, {"id": 24251, "dom": "G", "ent": "Dhabad", "cls": "Central Asia"}, {"id": 24252, "dom": "G", "ent": "Dhabad", "cls": "Eastern Africa"}, {"id": 24253, "dom": "G", "ent": "Dhabad", "cls": "Eastern Asia"}, {"id": 24254, "dom": "G", "ent": "Dhabad", "cls": "Eastern Europe"}, {"id": 24255, "dom": "G", "ent": "Dhabad", "cls": "Melanesia"}, {"id": 24256, "dom": "G", "ent": "Dhabad", "cls": "Micronesia"}, {"id": 24257, "dom": "G", "ent": "Dhabad", "cls": "Middle Africa"}, {"id": 24258, "dom": "G", "ent": "Dhabad", "cls": "Northern Africa"}, {"id": 24259, "dom": "G", "ent": "Dhabad", "cls": "Northern America"}, {"id": 24260, "dom": "G", "ent": "Dhabad", "cls": "Northern Europe"}, {"id": 24261, "dom": "G", "ent": "Dhabad", "cls": "Polynesia"}, {"id": 24262, "dom": "G", "ent": "Dhabad", "cls": "South America"}, {"id": 24263, "dom": "G", "ent": "Dhabad", "cls": "South-eastern Asia"}, {"id": 24264, "dom": "G", "ent": "Dhabad", "cls": "Southern Africa"}, {"id": 24265, "dom": "G", "ent": "Dhabad", "cls": "Southern Asia"}, {"id": 24266, "dom": "G", "ent": "Dhabad", "cls": "Southern Europe"}, {"id": 24267, "dom": "G", "ent": "Dhabad", "cls": "Western Africa"}, {"id": 24268, "dom": "G", "ent": "Dhabad", "cls": "Western Asia"}, {"id": 24269, "dom": "G", "ent": "Dhabad", "cls": "Western Europe"}, {"id": 24270, "dom": "G", "ent": "Ceerigaabo", "cls": "Australia and New Zealand"}, {"id": 24271, "dom": "G", "ent": "Ceerigaabo", "cls": "Caribbean"}, {"id": 24272, "dom": "G", "ent": "Ceerigaabo", "cls": "Central America"}, {"id": 24273, "dom": "G", "ent": "Ceerigaabo", "cls": "Central Asia"}, {"id": 24274, "dom": "G", "ent": "Ceerigaabo", "cls": "Eastern Africa"}, {"id": 24275, "dom": "G", "ent": "Ceerigaabo", "cls": "Eastern Asia"}, {"id": 24276, "dom": "G", "ent": "Ceerigaabo", "cls": "Eastern Europe"}, {"id": 24277, "dom": "G", "ent": "Ceerigaabo", "cls": "Melanesia"}, {"id": 24278, "dom": "G", "ent": "Ceerigaabo", "cls": "Micronesia"}, {"id": 24279, "dom": "G", "ent": "Ceerigaabo", "cls": "Middle Africa"}, {"id": 24280, "dom": "G", "ent": "Ceerigaabo", "cls": "Northern Africa"}, {"id": 24281, "dom": "G", "ent": "Ceerigaabo", "cls": "Northern America"}, {"id": 24282, "dom": "G", "ent": "Ceerigaabo", "cls": "Northern Europe"}, {"id": 24283, "dom": "G", "ent": "Ceerigaabo", "cls": "Polynesia"}, {"id": 24284, "dom": "G", "ent": "Ceerigaabo", "cls": "South America"}, {"id": 24285, "dom": "G", "ent": "Ceerigaabo", "cls": "South-eastern Asia"}, {"id": 24286, "dom": "G", "ent": "Ceerigaabo", "cls": "Southern Africa"}, {"id": 24287, "dom": "G", "ent": "Ceerigaabo", "cls": "Southern Asia"}, {"id": 24288, "dom": "G", "ent": "Ceerigaabo", "cls": "Southern Europe"}, {"id": 24289, "dom": "G", "ent": "Ceerigaabo", "cls": "Western Africa"}, {"id": 24290, "dom": "G", "ent": "Ceerigaabo", "cls": "Western Asia"}, {"id": 24291, "dom": "G", "ent": "Ceerigaabo", "cls": "Western Europe"}, {"id": 24292, "dom": "G", "ent": "Pontbuiten", "cls": "Australia and New Zealand"}, {"id": 24293, "dom": "G", "ent": "Pontbuiten", "cls": "Caribbean"}, {"id": 24294, "dom": "G", "ent": "Pontbuiten", "cls": "Central America"}, {"id": 24295, "dom": "G", "ent": "Pontbuiten", "cls": "Central Asia"}, {"id": 24296, "dom": "G", "ent": "Pontbuiten", "cls": "Eastern Africa"}, {"id": 24297, "dom": "G", "ent": "Pontbuiten", "cls": "Eastern Asia"}, {"id": 24298, "dom": "G", "ent": "Pontbuiten", "cls": "Eastern Europe"}, {"id": 24299, "dom": "G", "ent": "Pontbuiten", "cls": "Melanesia"}, {"id": 24300, "dom": "G", "ent": "Pontbuiten", "cls": "Micronesia"}, {"id": 24301, "dom": "G", "ent": "Pontbuiten", "cls": "Middle Africa"}, {"id": 24302, "dom": "G", "ent": "Pontbuiten", "cls": "Northern Africa"}, {"id": 24303, "dom": "G", "ent": "Pontbuiten", "cls": "Northern America"}, {"id": 24304, "dom": "G", "ent": "Pontbuiten", "cls": "Northern Europe"}, {"id": 24305, "dom": "G", "ent": "Pontbuiten", "cls": "Polynesia"}, {"id": 24306, "dom": "G", "ent": "Pontbuiten", "cls": "South America"}, {"id": 24307, "dom": "G", "ent": "Pontbuiten", "cls": "South-eastern Asia"}, {"id": 24308, "dom": "G", "ent": "Pontbuiten", "cls": "Southern Africa"}, {"id": 24309, "dom": "G", "ent": "Pontbuiten", "cls": "Southern Asia"}, {"id": 24310, "dom": "G", "ent": "Pontbuiten", "cls": "Southern Europe"}, {"id": 24311, "dom": "G", "ent": "Pontbuiten", "cls": "Western Africa"}, {"id": 24312, "dom": "G", "ent": "Pontbuiten", "cls": "Western Asia"}, {"id": 24313, "dom": "G", "ent": "Pontbuiten", "cls": "Western Europe"}, {"id": 24314, "dom": "G", "ent": "Rainville", "cls": "Australia and New Zealand"}, {"id": 24315, "dom": "G", "ent": "Rainville", "cls": "Caribbean"}, {"id": 24316, "dom": "G", "ent": "Rainville", "cls": "Central America"}, {"id": 24317, "dom": "G", "ent": "Rainville", "cls": "Central Asia"}, {"id": 24318, "dom": "G", "ent": "Rainville", "cls": "Eastern Africa"}, {"id": 24319, "dom": "G", "ent": "Rainville", "cls": "Eastern Asia"}, {"id": 24320, "dom": "G", "ent": "Rainville", "cls": "Eastern Europe"}, {"id": 24321, "dom": "G", "ent": "Rainville", "cls": "Melanesia"}, {"id": 24322, "dom": "G", "ent": "Rainville", "cls": "Micronesia"}, {"id": 24323, "dom": "G", "ent": "Rainville", "cls": "Middle Africa"}, {"id": 24324, "dom": "G", "ent": "Rainville", "cls": "Northern Africa"}, {"id": 24325, "dom": "G", "ent": "Rainville", "cls": "Northern America"}, {"id": 24326, "dom": "G", "ent": "Rainville", "cls": "Northern Europe"}, {"id": 24327, "dom": "G", "ent": "Rainville", "cls": "Polynesia"}, {"id": 24328, "dom": "G", "ent": "Rainville", "cls": "South America"}, {"id": 24329, "dom": "G", "ent": "Rainville", "cls": "South-eastern Asia"}, {"id": 24330, "dom": "G", "ent": "Rainville", "cls": "Southern Africa"}, {"id": 24331, "dom": "G", "ent": "Rainville", "cls": "Southern Asia"}, {"id": 24332, "dom": "G", "ent": "Rainville", "cls": "Southern Europe"}, {"id": 24333, "dom": "G", "ent": "Rainville", "cls": "Western Africa"}, {"id": 24334, "dom": "G", "ent": "Rainville", "cls": "Western Asia"}, {"id": 24335, "dom": "G", "ent": "Rainville", "cls": "Western Europe"}, {"id": 24336, "dom": "G", "ent": "Lelydorp", "cls": "Australia and New Zealand"}, {"id": 24337, "dom": "G", "ent": "Lelydorp", "cls": "Caribbean"}, {"id": 24338, "dom": "G", "ent": "Lelydorp", "cls": "Central America"}, {"id": 24339, "dom": "G", "ent": "Lelydorp", "cls": "Central Asia"}, {"id": 24340, "dom": "G", "ent": "Lelydorp", "cls": "Eastern Africa"}, {"id": 24341, "dom": "G", "ent": "Lelydorp", "cls": "Eastern Asia"}, {"id": 24342, "dom": "G", "ent": "Lelydorp", "cls": "Eastern Europe"}, {"id": 24343, "dom": "G", "ent": "Lelydorp", "cls": "Melanesia"}, {"id": 24344, "dom": "G", "ent": "Lelydorp", "cls": "Micronesia"}, {"id": 24345, "dom": "G", "ent": "Lelydorp", "cls": "Middle Africa"}, {"id": 24346, "dom": "G", "ent": "Lelydorp", "cls": "Northern Africa"}, {"id": 24347, "dom": "G", "ent": "Lelydorp", "cls": "Northern America"}, {"id": 24348, "dom": "G", "ent": "Lelydorp", "cls": "Northern Europe"}, {"id": 24349, "dom": "G", "ent": "Lelydorp", "cls": "Polynesia"}, {"id": 24350, "dom": "G", "ent": "Lelydorp", "cls": "South America"}, {"id": 24351, "dom": "G", "ent": "Lelydorp", "cls": "South-eastern Asia"}, {"id": 24352, "dom": "G", "ent": "Lelydorp", "cls": "Southern Africa"}, {"id": 24353, "dom": "G", "ent": "Lelydorp", "cls": "Southern Asia"}, {"id": 24354, "dom": "G", "ent": "Lelydorp", "cls": "Southern Europe"}, {"id": 24355, "dom": "G", "ent": "Lelydorp", "cls": "Western Africa"}, {"id": 24356, "dom": "G", "ent": "Lelydorp", "cls": "Western Asia"}, {"id": 24357, "dom": "G", "ent": "Lelydorp", "cls": "Western Europe"}, {"id": 24358, "dom": "G", "ent": "Tammenga", "cls": "Australia and New Zealand"}, {"id": 24359, "dom": "G", "ent": "Tammenga", "cls": "Caribbean"}, {"id": 24360, "dom": "G", "ent": "Tammenga", "cls": "Central America"}, {"id": 24361, "dom": "G", "ent": "Tammenga", "cls": "Central Asia"}, {"id": 24362, "dom": "G", "ent": "Tammenga", "cls": "Eastern Africa"}, {"id": 24363, "dom": "G", "ent": "Tammenga", "cls": "Eastern Asia"}, {"id": 24364, "dom": "G", "ent": "Tammenga", "cls": "Eastern Europe"}, {"id": 24365, "dom": "G", "ent": "Tammenga", "cls": "Melanesia"}, {"id": 24366, "dom": "G", "ent": "Tammenga", "cls": "Micronesia"}, {"id": 24367, "dom": "G", "ent": "Tammenga", "cls": "Middle Africa"}, {"id": 24368, "dom": "G", "ent": "Tammenga", "cls": "Northern Africa"}, {"id": 24369, "dom": "G", "ent": "Tammenga", "cls": "Northern America"}, {"id": 24370, "dom": "G", "ent": "Tammenga", "cls": "Northern Europe"}, {"id": 24371, "dom": "G", "ent": "Tammenga", "cls": "Polynesia"}, {"id": 24372, "dom": "G", "ent": "Tammenga", "cls": "South America"}, {"id": 24373, "dom": "G", "ent": "Tammenga", "cls": "South-eastern Asia"}, {"id": 24374, "dom": "G", "ent": "Tammenga", "cls": "Southern Africa"}, {"id": 24375, "dom": "G", "ent": "Tammenga", "cls": "Southern Asia"}, {"id": 24376, "dom": "G", "ent": "Tammenga", "cls": "Southern Europe"}, {"id": 24377, "dom": "G", "ent": "Tammenga", "cls": "Western Africa"}, {"id": 24378, "dom": "G", "ent": "Tammenga", "cls": "Western Asia"}, {"id": 24379, "dom": "G", "ent": "Tammenga", "cls": "Western Europe"}, {"id": 24380, "dom": "G", "ent": "Malakal", "cls": "Australia and New Zealand"}, {"id": 24381, "dom": "G", "ent": "Malakal", "cls": "Caribbean"}, {"id": 24382, "dom": "G", "ent": "Malakal", "cls": "Central America"}, {"id": 24383, "dom": "G", "ent": "Malakal", "cls": "Central Asia"}, {"id": 24384, "dom": "G", "ent": "Malakal", "cls": "Eastern Africa"}, {"id": 24385, "dom": "G", "ent": "Malakal", "cls": "Eastern Asia"}, {"id": 24386, "dom": "G", "ent": "Malakal", "cls": "Eastern Europe"}, {"id": 24387, "dom": "G", "ent": "Malakal", "cls": "Melanesia"}, {"id": 24388, "dom": "G", "ent": "Malakal", "cls": "Micronesia"}, {"id": 24389, "dom": "G", "ent": "Malakal", "cls": "Middle Africa"}, {"id": 24390, "dom": "G", "ent": "Malakal", "cls": "Northern Africa"}, {"id": 24391, "dom": "G", "ent": "Malakal", "cls": "Northern America"}, {"id": 24392, "dom": "G", "ent": "Malakal", "cls": "Northern Europe"}, {"id": 24393, "dom": "G", "ent": "Malakal", "cls": "Polynesia"}, {"id": 24394, "dom": "G", "ent": "Malakal", "cls": "South America"}, {"id": 24395, "dom": "G", "ent": "Malakal", "cls": "South-eastern Asia"}, {"id": 24396, "dom": "G", "ent": "Malakal", "cls": "Southern Africa"}, {"id": 24397, "dom": "G", "ent": "Malakal", "cls": "Southern Asia"}, {"id": 24398, "dom": "G", "ent": "Malakal", "cls": "Southern Europe"}, {"id": 24399, "dom": "G", "ent": "Malakal", "cls": "Western Africa"}, {"id": 24400, "dom": "G", "ent": "Malakal", "cls": "Western Asia"}, {"id": 24401, "dom": "G", "ent": "Malakal", "cls": "Western Europe"}, {"id": 24402, "dom": "G", "ent": "Wau", "cls": "Australia and New Zealand"}, {"id": 24403, "dom": "G", "ent": "Wau", "cls": "Caribbean"}, {"id": 24404, "dom": "G", "ent": "Wau", "cls": "Central America"}, {"id": 24405, "dom": "G", "ent": "Wau", "cls": "Central Asia"}, {"id": 24406, "dom": "G", "ent": "Wau", "cls": "Eastern Africa"}, {"id": 24407, "dom": "G", "ent": "Wau", "cls": "Eastern Asia"}, {"id": 24408, "dom": "G", "ent": "Wau", "cls": "Eastern Europe"}, {"id": 24409, "dom": "G", "ent": "Wau", "cls": "Melanesia"}, {"id": 24410, "dom": "G", "ent": "Wau", "cls": "Micronesia"}, {"id": 24411, "dom": "G", "ent": "Wau", "cls": "Middle Africa"}, {"id": 24412, "dom": "G", "ent": "Wau", "cls": "Northern Africa"}, {"id": 24413, "dom": "G", "ent": "Wau", "cls": "Northern America"}, {"id": 24414, "dom": "G", "ent": "Wau", "cls": "Northern Europe"}, {"id": 24415, "dom": "G", "ent": "Wau", "cls": "Polynesia"}, {"id": 24416, "dom": "G", "ent": "Wau", "cls": "South America"}, {"id": 24417, "dom": "G", "ent": "Wau", "cls": "South-eastern Asia"}, {"id": 24418, "dom": "G", "ent": "Wau", "cls": "Southern Africa"}, {"id": 24419, "dom": "G", "ent": "Wau", "cls": "Southern Asia"}, {"id": 24420, "dom": "G", "ent": "Wau", "cls": "Southern Europe"}, {"id": 24421, "dom": "G", "ent": "Wau", "cls": "Western Africa"}, {"id": 24422, "dom": "G", "ent": "Wau", "cls": "Western Asia"}, {"id": 24423, "dom": "G", "ent": "Wau", "cls": "Western Europe"}, {"id": 24424, "dom": "G", "ent": "Aweil", "cls": "Australia and New Zealand"}, {"id": 24425, "dom": "G", "ent": "Aweil", "cls": "Caribbean"}, {"id": 24426, "dom": "G", "ent": "Aweil", "cls": "Central America"}, {"id": 24427, "dom": "G", "ent": "Aweil", "cls": "Central Asia"}, {"id": 24428, "dom": "G", "ent": "Aweil", "cls": "Eastern Africa"}, {"id": 24429, "dom": "G", "ent": "Aweil", "cls": "Eastern Asia"}, {"id": 24430, "dom": "G", "ent": "Aweil", "cls": "Eastern Europe"}, {"id": 24431, "dom": "G", "ent": "Aweil", "cls": "Melanesia"}, {"id": 24432, "dom": "G", "ent": "Aweil", "cls": "Micronesia"}, {"id": 24433, "dom": "G", "ent": "Aweil", "cls": "Middle Africa"}, {"id": 24434, "dom": "G", "ent": "Aweil", "cls": "Northern Africa"}, {"id": 24435, "dom": "G", "ent": "Aweil", "cls": "Northern America"}, {"id": 24436, "dom": "G", "ent": "Aweil", "cls": "Northern Europe"}, {"id": 24437, "dom": "G", "ent": "Aweil", "cls": "Polynesia"}, {"id": 24438, "dom": "G", "ent": "Aweil", "cls": "South America"}, {"id": 24439, "dom": "G", "ent": "Aweil", "cls": "South-eastern Asia"}, {"id": 24440, "dom": "G", "ent": "Aweil", "cls": "Southern Africa"}, {"id": 24441, "dom": "G", "ent": "Aweil", "cls": "Southern Asia"}, {"id": 24442, "dom": "G", "ent": "Aweil", "cls": "Southern Europe"}, {"id": 24443, "dom": "G", "ent": "Aweil", "cls": "Western Africa"}, {"id": 24444, "dom": "G", "ent": "Aweil", "cls": "Western Asia"}, {"id": 24445, "dom": "G", "ent": "Aweil", "cls": "Western Europe"}, {"id": 24446, "dom": "G", "ent": "Kafia Kingi", "cls": "Australia and New Zealand"}, {"id": 24447, "dom": "G", "ent": "Kafia Kingi", "cls": "Caribbean"}, {"id": 24448, "dom": "G", "ent": "Kafia Kingi", "cls": "Central America"}, {"id": 24449, "dom": "G", "ent": "Kafia Kingi", "cls": "Central Asia"}, {"id": 24450, "dom": "G", "ent": "Kafia Kingi", "cls": "Eastern Africa"}, {"id": 24451, "dom": "G", "ent": "Kafia Kingi", "cls": "Eastern Asia"}, {"id": 24452, "dom": "G", "ent": "Kafia Kingi", "cls": "Eastern Europe"}, {"id": 24453, "dom": "G", "ent": "Kafia Kingi", "cls": "Melanesia"}, {"id": 24454, "dom": "G", "ent": "Kafia Kingi", "cls": "Micronesia"}, {"id": 24455, "dom": "G", "ent": "Kafia Kingi", "cls": "Middle Africa"}, {"id": 24456, "dom": "G", "ent": "Kafia Kingi", "cls": "Northern Africa"}, {"id": 24457, "dom": "G", "ent": "Kafia Kingi", "cls": "Northern America"}, {"id": 24458, "dom": "G", "ent": "Kafia Kingi", "cls": "Northern Europe"}, {"id": 24459, "dom": "G", "ent": "Kafia Kingi", "cls": "Polynesia"}, {"id": 24460, "dom": "G", "ent": "Kafia Kingi", "cls": "South America"}, {"id": 24461, "dom": "G", "ent": "Kafia Kingi", "cls": "South-eastern Asia"}, {"id": 24462, "dom": "G", "ent": "Kafia Kingi", "cls": "Southern Africa"}, {"id": 24463, "dom": "G", "ent": "Kafia Kingi", "cls": "Southern Asia"}, {"id": 24464, "dom": "G", "ent": "Kafia Kingi", "cls": "Southern Europe"}, {"id": 24465, "dom": "G", "ent": "Kafia Kingi", "cls": "Western Africa"}, {"id": 24466, "dom": "G", "ent": "Kafia Kingi", "cls": "Western Asia"}, {"id": 24467, "dom": "G", "ent": "Kafia Kingi", "cls": "Western Europe"}, {"id": 24468, "dom": "G", "ent": "Santa Tecla", "cls": "Australia and New Zealand"}, {"id": 24469, "dom": "G", "ent": "Santa Tecla", "cls": "Caribbean"}, {"id": 24470, "dom": "G", "ent": "Santa Tecla", "cls": "Central America"}, {"id": 24471, "dom": "G", "ent": "Santa Tecla", "cls": "Central Asia"}, {"id": 24472, "dom": "G", "ent": "Santa Tecla", "cls": "Eastern Africa"}, {"id": 24473, "dom": "G", "ent": "Santa Tecla", "cls": "Eastern Asia"}, {"id": 24474, "dom": "G", "ent": "Santa Tecla", "cls": "Eastern Europe"}, {"id": 24475, "dom": "G", "ent": "Santa Tecla", "cls": "Melanesia"}, {"id": 24476, "dom": "G", "ent": "Santa Tecla", "cls": "Micronesia"}, {"id": 24477, "dom": "G", "ent": "Santa Tecla", "cls": "Middle Africa"}, {"id": 24478, "dom": "G", "ent": "Santa Tecla", "cls": "Northern Africa"}, {"id": 24479, "dom": "G", "ent": "Santa Tecla", "cls": "Northern America"}, {"id": 24480, "dom": "G", "ent": "Santa Tecla", "cls": "Northern Europe"}, {"id": 24481, "dom": "G", "ent": "Santa Tecla", "cls": "Polynesia"}, {"id": 24482, "dom": "G", "ent": "Santa Tecla", "cls": "South America"}, {"id": 24483, "dom": "G", "ent": "Santa Tecla", "cls": "South-eastern Asia"}, {"id": 24484, "dom": "G", "ent": "Santa Tecla", "cls": "Southern Africa"}, {"id": 24485, "dom": "G", "ent": "Santa Tecla", "cls": "Southern Asia"}, {"id": 24486, "dom": "G", "ent": "Santa Tecla", "cls": "Southern Europe"}, {"id": 24487, "dom": "G", "ent": "Santa Tecla", "cls": "Western Africa"}, {"id": 24488, "dom": "G", "ent": "Santa Tecla", "cls": "Western Asia"}, {"id": 24489, "dom": "G", "ent": "Santa Tecla", "cls": "Western Europe"}, {"id": 24490, "dom": "G", "ent": "Delgado", "cls": "Australia and New Zealand"}, {"id": 24491, "dom": "G", "ent": "Delgado", "cls": "Caribbean"}, {"id": 24492, "dom": "G", "ent": "Delgado", "cls": "Central America"}, {"id": 24493, "dom": "G", "ent": "Delgado", "cls": "Central Asia"}, {"id": 24494, "dom": "G", "ent": "Delgado", "cls": "Eastern Africa"}, {"id": 24495, "dom": "G", "ent": "Delgado", "cls": "Eastern Asia"}, {"id": 24496, "dom": "G", "ent": "Delgado", "cls": "Eastern Europe"}, {"id": 24497, "dom": "G", "ent": "Delgado", "cls": "Melanesia"}, {"id": 24498, "dom": "G", "ent": "Delgado", "cls": "Micronesia"}, {"id": 24499, "dom": "G", "ent": "Delgado", "cls": "Middle Africa"}, {"id": 24500, "dom": "G", "ent": "Delgado", "cls": "Northern Africa"}, {"id": 24501, "dom": "G", "ent": "Delgado", "cls": "Northern America"}, {"id": 24502, "dom": "G", "ent": "Delgado", "cls": "Northern Europe"}, {"id": 24503, "dom": "G", "ent": "Delgado", "cls": "Polynesia"}, {"id": 24504, "dom": "G", "ent": "Delgado", "cls": "South America"}, {"id": 24505, "dom": "G", "ent": "Delgado", "cls": "South-eastern Asia"}, {"id": 24506, "dom": "G", "ent": "Delgado", "cls": "Southern Africa"}, {"id": 24507, "dom": "G", "ent": "Delgado", "cls": "Southern Asia"}, {"id": 24508, "dom": "G", "ent": "Delgado", "cls": "Southern Europe"}, {"id": 24509, "dom": "G", "ent": "Delgado", "cls": "Western Africa"}, {"id": 24510, "dom": "G", "ent": "Delgado", "cls": "Western Asia"}, {"id": 24511, "dom": "G", "ent": "Delgado", "cls": "Western Europe"}, {"id": 24512, "dom": "G", "ent": "San Rafael Oriente", "cls": "Australia and New Zealand"}, {"id": 24513, "dom": "G", "ent": "San Rafael Oriente", "cls": "Caribbean"}, {"id": 24514, "dom": "G", "ent": "San Rafael Oriente", "cls": "Central America"}, {"id": 24515, "dom": "G", "ent": "San Rafael Oriente", "cls": "Central Asia"}, {"id": 24516, "dom": "G", "ent": "San Rafael Oriente", "cls": "Eastern Africa"}, {"id": 24517, "dom": "G", "ent": "San Rafael Oriente", "cls": "Eastern Asia"}, {"id": 24518, "dom": "G", "ent": "San Rafael Oriente", "cls": "Eastern Europe"}, {"id": 24519, "dom": "G", "ent": "San Rafael Oriente", "cls": "Melanesia"}, {"id": 24520, "dom": "G", "ent": "San Rafael Oriente", "cls": "Micronesia"}, {"id": 24521, "dom": "G", "ent": "San Rafael Oriente", "cls": "Middle Africa"}, {"id": 24522, "dom": "G", "ent": "San Rafael Oriente", "cls": "Northern Africa"}, {"id": 24523, "dom": "G", "ent": "San Rafael Oriente", "cls": "Northern America"}, {"id": 24524, "dom": "G", "ent": "San Rafael Oriente", "cls": "Northern Europe"}, {"id": 24525, "dom": "G", "ent": "San Rafael Oriente", "cls": "Polynesia"}, {"id": 24526, "dom": "G", "ent": "San Rafael Oriente", "cls": "South America"}, {"id": 24527, "dom": "G", "ent": "San Rafael Oriente", "cls": "South-eastern Asia"}, {"id": 24528, "dom": "G", "ent": "San Rafael Oriente", "cls": "Southern Africa"}, {"id": 24529, "dom": "G", "ent": "San Rafael Oriente", "cls": "Southern Asia"}, {"id": 24530, "dom": "G", "ent": "San Rafael Oriente", "cls": "Southern Europe"}, {"id": 24531, "dom": "G", "ent": "San Rafael Oriente", "cls": "Western Africa"}, {"id": 24532, "dom": "G", "ent": "San Rafael Oriente", "cls": "Western Asia"}, {"id": 24533, "dom": "G", "ent": "San Rafael Oriente", "cls": "Western Europe"}, {"id": 24534, "dom": "G", "ent": "Sonzacate", "cls": "Australia and New Zealand"}, {"id": 24535, "dom": "G", "ent": "Sonzacate", "cls": "Caribbean"}, {"id": 24536, "dom": "G", "ent": "Sonzacate", "cls": "Central America"}, {"id": 24537, "dom": "G", "ent": "Sonzacate", "cls": "Central Asia"}, {"id": 24538, "dom": "G", "ent": "Sonzacate", "cls": "Eastern Africa"}, {"id": 24539, "dom": "G", "ent": "Sonzacate", "cls": "Eastern Asia"}, {"id": 24540, "dom": "G", "ent": "Sonzacate", "cls": "Eastern Europe"}, {"id": 24541, "dom": "G", "ent": "Sonzacate", "cls": "Melanesia"}, {"id": 24542, "dom": "G", "ent": "Sonzacate", "cls": "Micronesia"}, {"id": 24543, "dom": "G", "ent": "Sonzacate", "cls": "Middle Africa"}, {"id": 24544, "dom": "G", "ent": "Sonzacate", "cls": "Northern Africa"}, {"id": 24545, "dom": "G", "ent": "Sonzacate", "cls": "Northern America"}, {"id": 24546, "dom": "G", "ent": "Sonzacate", "cls": "Northern Europe"}, {"id": 24547, "dom": "G", "ent": "Sonzacate", "cls": "Polynesia"}, {"id": 24548, "dom": "G", "ent": "Sonzacate", "cls": "South America"}, {"id": 24549, "dom": "G", "ent": "Sonzacate", "cls": "South-eastern Asia"}, {"id": 24550, "dom": "G", "ent": "Sonzacate", "cls": "Southern Africa"}, {"id": 24551, "dom": "G", "ent": "Sonzacate", "cls": "Southern Asia"}, {"id": 24552, "dom": "G", "ent": "Sonzacate", "cls": "Southern Europe"}, {"id": 24553, "dom": "G", "ent": "Sonzacate", "cls": "Western Africa"}, {"id": 24554, "dom": "G", "ent": "Sonzacate", "cls": "Western Asia"}, {"id": 24555, "dom": "G", "ent": "Sonzacate", "cls": "Western Europe"}, {"id": 24556, "dom": "G", "ent": "Homs", "cls": "Australia and New Zealand"}, {"id": 24557, "dom": "G", "ent": "Homs", "cls": "Caribbean"}, {"id": 24558, "dom": "G", "ent": "Homs", "cls": "Central America"}, {"id": 24559, "dom": "G", "ent": "Homs", "cls": "Central Asia"}, {"id": 24560, "dom": "G", "ent": "Homs", "cls": "Eastern Africa"}, {"id": 24561, "dom": "G", "ent": "Homs", "cls": "Eastern Asia"}, {"id": 24562, "dom": "G", "ent": "Homs", "cls": "Eastern Europe"}, {"id": 24563, "dom": "G", "ent": "Homs", "cls": "Melanesia"}, {"id": 24564, "dom": "G", "ent": "Homs", "cls": "Micronesia"}, {"id": 24565, "dom": "G", "ent": "Homs", "cls": "Middle Africa"}, {"id": 24566, "dom": "G", "ent": "Homs", "cls": "Northern Africa"}, {"id": 24567, "dom": "G", "ent": "Homs", "cls": "Northern America"}, {"id": 24568, "dom": "G", "ent": "Homs", "cls": "Northern Europe"}, {"id": 24569, "dom": "G", "ent": "Homs", "cls": "Polynesia"}, {"id": 24570, "dom": "G", "ent": "Homs", "cls": "South America"}, {"id": 24571, "dom": "G", "ent": "Homs", "cls": "South-eastern Asia"}, {"id": 24572, "dom": "G", "ent": "Homs", "cls": "Southern Africa"}, {"id": 24573, "dom": "G", "ent": "Homs", "cls": "Southern Asia"}, {"id": 24574, "dom": "G", "ent": "Homs", "cls": "Southern Europe"}, {"id": 24575, "dom": "G", "ent": "Homs", "cls": "Western Africa"}, {"id": 24576, "dom": "G", "ent": "Homs", "cls": "Western Asia"}, {"id": 24577, "dom": "G", "ent": "Homs", "cls": "Western Europe"}, {"id": 24578, "dom": "G", "ent": "Al-Tabqa", "cls": "Australia and New Zealand"}, {"id": 24579, "dom": "G", "ent": "Al-Tabqa", "cls": "Caribbean"}, {"id": 24580, "dom": "G", "ent": "Al-Tabqa", "cls": "Central America"}, {"id": 24581, "dom": "G", "ent": "Al-Tabqa", "cls": "Central Asia"}, {"id": 24582, "dom": "G", "ent": "Al-Tabqa", "cls": "Eastern Africa"}, {"id": 24583, "dom": "G", "ent": "Al-Tabqa", "cls": "Eastern Asia"}, {"id": 24584, "dom": "G", "ent": "Al-Tabqa", "cls": "Eastern Europe"}, {"id": 24585, "dom": "G", "ent": "Al-Tabqa", "cls": "Melanesia"}, {"id": 24586, "dom": "G", "ent": "Al-Tabqa", "cls": "Micronesia"}, {"id": 24587, "dom": "G", "ent": "Al-Tabqa", "cls": "Middle Africa"}, {"id": 24588, "dom": "G", "ent": "Al-Tabqa", "cls": "Northern Africa"}, {"id": 24589, "dom": "G", "ent": "Al-Tabqa", "cls": "Northern America"}, {"id": 24590, "dom": "G", "ent": "Al-Tabqa", "cls": "Northern Europe"}, {"id": 24591, "dom": "G", "ent": "Al-Tabqa", "cls": "Polynesia"}, {"id": 24592, "dom": "G", "ent": "Al-Tabqa", "cls": "South America"}, {"id": 24593, "dom": "G", "ent": "Al-Tabqa", "cls": "South-eastern Asia"}, {"id": 24594, "dom": "G", "ent": "Al-Tabqa", "cls": "Southern Africa"}, {"id": 24595, "dom": "G", "ent": "Al-Tabqa", "cls": "Southern Asia"}, {"id": 24596, "dom": "G", "ent": "Al-Tabqa", "cls": "Southern Europe"}, {"id": 24597, "dom": "G", "ent": "Al-Tabqa", "cls": "Western Africa"}, {"id": 24598, "dom": "G", "ent": "Al-Tabqa", "cls": "Western Asia"}, {"id": 24599, "dom": "G", "ent": "Al-Tabqa", "cls": "Western Europe"}, {"id": 24600, "dom": "G", "ent": "Al Qaryatayn", "cls": "Australia and New Zealand"}, {"id": 24601, "dom": "G", "ent": "Al Qaryatayn", "cls": "Caribbean"}, {"id": 24602, "dom": "G", "ent": "Al Qaryatayn", "cls": "Central America"}, {"id": 24603, "dom": "G", "ent": "Al Qaryatayn", "cls": "Central Asia"}, {"id": 24604, "dom": "G", "ent": "Al Qaryatayn", "cls": "Eastern Africa"}, {"id": 24605, "dom": "G", "ent": "Al Qaryatayn", "cls": "Eastern Asia"}, {"id": 24606, "dom": "G", "ent": "Al Qaryatayn", "cls": "Eastern Europe"}, {"id": 24607, "dom": "G", "ent": "Al Qaryatayn", "cls": "Melanesia"}, {"id": 24608, "dom": "G", "ent": "Al Qaryatayn", "cls": "Micronesia"}, {"id": 24609, "dom": "G", "ent": "Al Qaryatayn", "cls": "Middle Africa"}, {"id": 24610, "dom": "G", "ent": "Al Qaryatayn", "cls": "Northern Africa"}, {"id": 24611, "dom": "G", "ent": "Al Qaryatayn", "cls": "Northern America"}, {"id": 24612, "dom": "G", "ent": "Al Qaryatayn", "cls": "Northern Europe"}, {"id": 24613, "dom": "G", "ent": "Al Qaryatayn", "cls": "Polynesia"}, {"id": 24614, "dom": "G", "ent": "Al Qaryatayn", "cls": "South America"}, {"id": 24615, "dom": "G", "ent": "Al Qaryatayn", "cls": "South-eastern Asia"}, {"id": 24616, "dom": "G", "ent": "Al Qaryatayn", "cls": "Southern Africa"}, {"id": 24617, "dom": "G", "ent": "Al Qaryatayn", "cls": "Southern Asia"}, {"id": 24618, "dom": "G", "ent": "Al Qaryatayn", "cls": "Southern Europe"}, {"id": 24619, "dom": "G", "ent": "Al Qaryatayn", "cls": "Western Africa"}, {"id": 24620, "dom": "G", "ent": "Al Qaryatayn", "cls": "Western Asia"}, {"id": 24621, "dom": "G", "ent": "Al Qaryatayn", "cls": "Western Europe"}, {"id": 24622, "dom": "G", "ent": "Tallkalakh", "cls": "Australia and New Zealand"}, {"id": 24623, "dom": "G", "ent": "Tallkalakh", "cls": "Caribbean"}, {"id": 24624, "dom": "G", "ent": "Tallkalakh", "cls": "Central America"}, {"id": 24625, "dom": "G", "ent": "Tallkalakh", "cls": "Central Asia"}, {"id": 24626, "dom": "G", "ent": "Tallkalakh", "cls": "Eastern Africa"}, {"id": 24627, "dom": "G", "ent": "Tallkalakh", "cls": "Eastern Asia"}, {"id": 24628, "dom": "G", "ent": "Tallkalakh", "cls": "Eastern Europe"}, {"id": 24629, "dom": "G", "ent": "Tallkalakh", "cls": "Melanesia"}, {"id": 24630, "dom": "G", "ent": "Tallkalakh", "cls": "Micronesia"}, {"id": 24631, "dom": "G", "ent": "Tallkalakh", "cls": "Middle Africa"}, {"id": 24632, "dom": "G", "ent": "Tallkalakh", "cls": "Northern Africa"}, {"id": 24633, "dom": "G", "ent": "Tallkalakh", "cls": "Northern America"}, {"id": 24634, "dom": "G", "ent": "Tallkalakh", "cls": "Northern Europe"}, {"id": 24635, "dom": "G", "ent": "Tallkalakh", "cls": "Polynesia"}, {"id": 24636, "dom": "G", "ent": "Tallkalakh", "cls": "South America"}, {"id": 24637, "dom": "G", "ent": "Tallkalakh", "cls": "South-eastern Asia"}, {"id": 24638, "dom": "G", "ent": "Tallkalakh", "cls": "Southern Africa"}, {"id": 24639, "dom": "G", "ent": "Tallkalakh", "cls": "Southern Asia"}, {"id": 24640, "dom": "G", "ent": "Tallkalakh", "cls": "Southern Europe"}, {"id": 24641, "dom": "G", "ent": "Tallkalakh", "cls": "Western Africa"}, {"id": 24642, "dom": "G", "ent": "Tallkalakh", "cls": "Western Asia"}, {"id": 24643, "dom": "G", "ent": "Tallkalakh", "cls": "Western Europe"}, {"id": 24644, "dom": "G", "ent": "Kelo", "cls": "Australia and New Zealand"}, {"id": 24645, "dom": "G", "ent": "Kelo", "cls": "Caribbean"}, {"id": 24646, "dom": "G", "ent": "Kelo", "cls": "Central America"}, {"id": 24647, "dom": "G", "ent": "Kelo", "cls": "Central Asia"}, {"id": 24648, "dom": "G", "ent": "Kelo", "cls": "Eastern Africa"}, {"id": 24649, "dom": "G", "ent": "Kelo", "cls": "Eastern Asia"}, {"id": 24650, "dom": "G", "ent": "Kelo", "cls": "Eastern Europe"}, {"id": 24651, "dom": "G", "ent": "Kelo", "cls": "Melanesia"}, {"id": 24652, "dom": "G", "ent": "Kelo", "cls": "Micronesia"}, {"id": 24653, "dom": "G", "ent": "Kelo", "cls": "Middle Africa"}, {"id": 24654, "dom": "G", "ent": "Kelo", "cls": "Northern Africa"}, {"id": 24655, "dom": "G", "ent": "Kelo", "cls": "Northern America"}, {"id": 24656, "dom": "G", "ent": "Kelo", "cls": "Northern Europe"}, {"id": 24657, "dom": "G", "ent": "Kelo", "cls": "Polynesia"}, {"id": 24658, "dom": "G", "ent": "Kelo", "cls": "South America"}, {"id": 24659, "dom": "G", "ent": "Kelo", "cls": "South-eastern Asia"}, {"id": 24660, "dom": "G", "ent": "Kelo", "cls": "Southern Africa"}, {"id": 24661, "dom": "G", "ent": "Kelo", "cls": "Southern Asia"}, {"id": 24662, "dom": "G", "ent": "Kelo", "cls": "Southern Europe"}, {"id": 24663, "dom": "G", "ent": "Kelo", "cls": "Western Africa"}, {"id": 24664, "dom": "G", "ent": "Kelo", "cls": "Western Asia"}, {"id": 24665, "dom": "G", "ent": "Kelo", "cls": "Western Europe"}, {"id": 24666, "dom": "G", "ent": "Doba", "cls": "Australia and New Zealand"}, {"id": 24667, "dom": "G", "ent": "Doba", "cls": "Caribbean"}, {"id": 24668, "dom": "G", "ent": "Doba", "cls": "Central America"}, {"id": 24669, "dom": "G", "ent": "Doba", "cls": "Central Asia"}, {"id": 24670, "dom": "G", "ent": "Doba", "cls": "Eastern Africa"}, {"id": 24671, "dom": "G", "ent": "Doba", "cls": "Eastern Asia"}, {"id": 24672, "dom": "G", "ent": "Doba", "cls": "Eastern Europe"}, {"id": 24673, "dom": "G", "ent": "Doba", "cls": "Melanesia"}, {"id": 24674, "dom": "G", "ent": "Doba", "cls": "Micronesia"}, {"id": 24675, "dom": "G", "ent": "Doba", "cls": "Middle Africa"}, {"id": 24676, "dom": "G", "ent": "Doba", "cls": "Northern Africa"}, {"id": 24677, "dom": "G", "ent": "Doba", "cls": "Northern America"}, {"id": 24678, "dom": "G", "ent": "Doba", "cls": "Northern Europe"}, {"id": 24679, "dom": "G", "ent": "Doba", "cls": "Polynesia"}, {"id": 24680, "dom": "G", "ent": "Doba", "cls": "South America"}, {"id": 24681, "dom": "G", "ent": "Doba", "cls": "South-eastern Asia"}, {"id": 24682, "dom": "G", "ent": "Doba", "cls": "Southern Africa"}, {"id": 24683, "dom": "G", "ent": "Doba", "cls": "Southern Asia"}, {"id": 24684, "dom": "G", "ent": "Doba", "cls": "Southern Europe"}, {"id": 24685, "dom": "G", "ent": "Doba", "cls": "Western Africa"}, {"id": 24686, "dom": "G", "ent": "Doba", "cls": "Western Asia"}, {"id": 24687, "dom": "G", "ent": "Doba", "cls": "Western Europe"}, {"id": 24688, "dom": "G", "ent": "Pala", "cls": "Australia and New Zealand"}, {"id": 24689, "dom": "G", "ent": "Pala", "cls": "Caribbean"}, {"id": 24690, "dom": "G", "ent": "Pala", "cls": "Central America"}, {"id": 24691, "dom": "G", "ent": "Pala", "cls": "Central Asia"}, {"id": 24692, "dom": "G", "ent": "Pala", "cls": "Eastern Africa"}, {"id": 24693, "dom": "G", "ent": "Pala", "cls": "Eastern Asia"}, {"id": 24694, "dom": "G", "ent": "Pala", "cls": "Eastern Europe"}, {"id": 24695, "dom": "G", "ent": "Pala", "cls": "Melanesia"}, {"id": 24696, "dom": "G", "ent": "Pala", "cls": "Micronesia"}, {"id": 24697, "dom": "G", "ent": "Pala", "cls": "Middle Africa"}, {"id": 24698, "dom": "G", "ent": "Pala", "cls": "Northern Africa"}, {"id": 24699, "dom": "G", "ent": "Pala", "cls": "Northern America"}, {"id": 24700, "dom": "G", "ent": "Pala", "cls": "Northern Europe"}, {"id": 24701, "dom": "G", "ent": "Pala", "cls": "Polynesia"}, {"id": 24702, "dom": "G", "ent": "Pala", "cls": "South America"}, {"id": 24703, "dom": "G", "ent": "Pala", "cls": "South-eastern Asia"}, {"id": 24704, "dom": "G", "ent": "Pala", "cls": "Southern Africa"}, {"id": 24705, "dom": "G", "ent": "Pala", "cls": "Southern Asia"}, {"id": 24706, "dom": "G", "ent": "Pala", "cls": "Southern Europe"}, {"id": 24707, "dom": "G", "ent": "Pala", "cls": "Western Africa"}, {"id": 24708, "dom": "G", "ent": "Pala", "cls": "Western Asia"}, {"id": 24709, "dom": "G", "ent": "Pala", "cls": "Western Europe"}, {"id": 24710, "dom": "G", "ent": "Massaguet", "cls": "Australia and New Zealand"}, {"id": 24711, "dom": "G", "ent": "Massaguet", "cls": "Caribbean"}, {"id": 24712, "dom": "G", "ent": "Massaguet", "cls": "Central America"}, {"id": 24713, "dom": "G", "ent": "Massaguet", "cls": "Central Asia"}, {"id": 24714, "dom": "G", "ent": "Massaguet", "cls": "Eastern Africa"}, {"id": 24715, "dom": "G", "ent": "Massaguet", "cls": "Eastern Asia"}, {"id": 24716, "dom": "G", "ent": "Massaguet", "cls": "Eastern Europe"}, {"id": 24717, "dom": "G", "ent": "Massaguet", "cls": "Melanesia"}, {"id": 24718, "dom": "G", "ent": "Massaguet", "cls": "Micronesia"}, {"id": 24719, "dom": "G", "ent": "Massaguet", "cls": "Middle Africa"}, {"id": 24720, "dom": "G", "ent": "Massaguet", "cls": "Northern Africa"}, {"id": 24721, "dom": "G", "ent": "Massaguet", "cls": "Northern America"}, {"id": 24722, "dom": "G", "ent": "Massaguet", "cls": "Northern Europe"}, {"id": 24723, "dom": "G", "ent": "Massaguet", "cls": "Polynesia"}, {"id": 24724, "dom": "G", "ent": "Massaguet", "cls": "South America"}, {"id": 24725, "dom": "G", "ent": "Massaguet", "cls": "South-eastern Asia"}, {"id": 24726, "dom": "G", "ent": "Massaguet", "cls": "Southern Africa"}, {"id": 24727, "dom": "G", "ent": "Massaguet", "cls": "Southern Asia"}, {"id": 24728, "dom": "G", "ent": "Massaguet", "cls": "Southern Europe"}, {"id": 24729, "dom": "G", "ent": "Massaguet", "cls": "Western Africa"}, {"id": 24730, "dom": "G", "ent": "Massaguet", "cls": "Western Asia"}, {"id": 24731, "dom": "G", "ent": "Massaguet", "cls": "Western Europe"}, {"id": 24732, "dom": "G", "ent": "Bassar", "cls": "Australia and New Zealand"}, {"id": 24733, "dom": "G", "ent": "Bassar", "cls": "Caribbean"}, {"id": 24734, "dom": "G", "ent": "Bassar", "cls": "Central America"}, {"id": 24735, "dom": "G", "ent": "Bassar", "cls": "Central Asia"}, {"id": 24736, "dom": "G", "ent": "Bassar", "cls": "Eastern Africa"}, {"id": 24737, "dom": "G", "ent": "Bassar", "cls": "Eastern Asia"}, {"id": 24738, "dom": "G", "ent": "Bassar", "cls": "Eastern Europe"}, {"id": 24739, "dom": "G", "ent": "Bassar", "cls": "Melanesia"}, {"id": 24740, "dom": "G", "ent": "Bassar", "cls": "Micronesia"}, {"id": 24741, "dom": "G", "ent": "Bassar", "cls": "Middle Africa"}, {"id": 24742, "dom": "G", "ent": "Bassar", "cls": "Northern Africa"}, {"id": 24743, "dom": "G", "ent": "Bassar", "cls": "Northern America"}, {"id": 24744, "dom": "G", "ent": "Bassar", "cls": "Northern Europe"}, {"id": 24745, "dom": "G", "ent": "Bassar", "cls": "Polynesia"}, {"id": 24746, "dom": "G", "ent": "Bassar", "cls": "South America"}, {"id": 24747, "dom": "G", "ent": "Bassar", "cls": "South-eastern Asia"}, {"id": 24748, "dom": "G", "ent": "Bassar", "cls": "Southern Africa"}, {"id": 24749, "dom": "G", "ent": "Bassar", "cls": "Southern Asia"}, {"id": 24750, "dom": "G", "ent": "Bassar", "cls": "Southern Europe"}, {"id": 24751, "dom": "G", "ent": "Bassar", "cls": "Western Africa"}, {"id": 24752, "dom": "G", "ent": "Bassar", "cls": "Western Asia"}, {"id": 24753, "dom": "G", "ent": "Bassar", "cls": "Western Europe"}, {"id": 24754, "dom": "G", "ent": "Sotouboua", "cls": "Australia and New Zealand"}, {"id": 24755, "dom": "G", "ent": "Sotouboua", "cls": "Caribbean"}, {"id": 24756, "dom": "G", "ent": "Sotouboua", "cls": "Central America"}, {"id": 24757, "dom": "G", "ent": "Sotouboua", "cls": "Central Asia"}, {"id": 24758, "dom": "G", "ent": "Sotouboua", "cls": "Eastern Africa"}, {"id": 24759, "dom": "G", "ent": "Sotouboua", "cls": "Eastern Asia"}, {"id": 24760, "dom": "G", "ent": "Sotouboua", "cls": "Eastern Europe"}, {"id": 24761, "dom": "G", "ent": "Sotouboua", "cls": "Melanesia"}, {"id": 24762, "dom": "G", "ent": "Sotouboua", "cls": "Micronesia"}, {"id": 24763, "dom": "G", "ent": "Sotouboua", "cls": "Middle Africa"}, {"id": 24764, "dom": "G", "ent": "Sotouboua", "cls": "Northern Africa"}, {"id": 24765, "dom": "G", "ent": "Sotouboua", "cls": "Northern America"}, {"id": 24766, "dom": "G", "ent": "Sotouboua", "cls": "Northern Europe"}, {"id": 24767, "dom": "G", "ent": "Sotouboua", "cls": "Polynesia"}, {"id": 24768, "dom": "G", "ent": "Sotouboua", "cls": "South America"}, {"id": 24769, "dom": "G", "ent": "Sotouboua", "cls": "South-eastern Asia"}, {"id": 24770, "dom": "G", "ent": "Sotouboua", "cls": "Southern Africa"}, {"id": 24771, "dom": "G", "ent": "Sotouboua", "cls": "Southern Asia"}, {"id": 24772, "dom": "G", "ent": "Sotouboua", "cls": "Southern Europe"}, {"id": 24773, "dom": "G", "ent": "Sotouboua", "cls": "Western Africa"}, {"id": 24774, "dom": "G", "ent": "Sotouboua", "cls": "Western Asia"}, {"id": 24775, "dom": "G", "ent": "Sotouboua", "cls": "Western Europe"}, {"id": 24776, "dom": "G", "ent": "Bafilo", "cls": "Australia and New Zealand"}, {"id": 24777, "dom": "G", "ent": "Bafilo", "cls": "Caribbean"}, {"id": 24778, "dom": "G", "ent": "Bafilo", "cls": "Central America"}, {"id": 24779, "dom": "G", "ent": "Bafilo", "cls": "Central Asia"}, {"id": 24780, "dom": "G", "ent": "Bafilo", "cls": "Eastern Africa"}, {"id": 24781, "dom": "G", "ent": "Bafilo", "cls": "Eastern Asia"}, {"id": 24782, "dom": "G", "ent": "Bafilo", "cls": "Eastern Europe"}, {"id": 24783, "dom": "G", "ent": "Bafilo", "cls": "Melanesia"}, {"id": 24784, "dom": "G", "ent": "Bafilo", "cls": "Micronesia"}, {"id": 24785, "dom": "G", "ent": "Bafilo", "cls": "Middle Africa"}, {"id": 24786, "dom": "G", "ent": "Bafilo", "cls": "Northern Africa"}, {"id": 24787, "dom": "G", "ent": "Bafilo", "cls": "Northern America"}, {"id": 24788, "dom": "G", "ent": "Bafilo", "cls": "Northern Europe"}, {"id": 24789, "dom": "G", "ent": "Bafilo", "cls": "Polynesia"}, {"id": 24790, "dom": "G", "ent": "Bafilo", "cls": "South America"}, {"id": 24791, "dom": "G", "ent": "Bafilo", "cls": "South-eastern Asia"}, {"id": 24792, "dom": "G", "ent": "Bafilo", "cls": "Southern Africa"}, {"id": 24793, "dom": "G", "ent": "Bafilo", "cls": "Southern Asia"}, {"id": 24794, "dom": "G", "ent": "Bafilo", "cls": "Southern Europe"}, {"id": 24795, "dom": "G", "ent": "Bafilo", "cls": "Western Africa"}, {"id": 24796, "dom": "G", "ent": "Bafilo", "cls": "Western Asia"}, {"id": 24797, "dom": "G", "ent": "Bafilo", "cls": "Western Europe"}, {"id": 24798, "dom": "G", "ent": "Tabligbo", "cls": "Australia and New Zealand"}, {"id": 24799, "dom": "G", "ent": "Tabligbo", "cls": "Caribbean"}, {"id": 24800, "dom": "G", "ent": "Tabligbo", "cls": "Central America"}, {"id": 24801, "dom": "G", "ent": "Tabligbo", "cls": "Central Asia"}, {"id": 24802, "dom": "G", "ent": "Tabligbo", "cls": "Eastern Africa"}, {"id": 24803, "dom": "G", "ent": "Tabligbo", "cls": "Eastern Asia"}, {"id": 24804, "dom": "G", "ent": "Tabligbo", "cls": "Eastern Europe"}, {"id": 24805, "dom": "G", "ent": "Tabligbo", "cls": "Melanesia"}, {"id": 24806, "dom": "G", "ent": "Tabligbo", "cls": "Micronesia"}, {"id": 24807, "dom": "G", "ent": "Tabligbo", "cls": "Middle Africa"}, {"id": 24808, "dom": "G", "ent": "Tabligbo", "cls": "Northern Africa"}, {"id": 24809, "dom": "G", "ent": "Tabligbo", "cls": "Northern America"}, {"id": 24810, "dom": "G", "ent": "Tabligbo", "cls": "Northern Europe"}, {"id": 24811, "dom": "G", "ent": "Tabligbo", "cls": "Polynesia"}, {"id": 24812, "dom": "G", "ent": "Tabligbo", "cls": "South America"}, {"id": 24813, "dom": "G", "ent": "Tabligbo", "cls": "South-eastern Asia"}, {"id": 24814, "dom": "G", "ent": "Tabligbo", "cls": "Southern Africa"}, {"id": 24815, "dom": "G", "ent": "Tabligbo", "cls": "Southern Asia"}, {"id": 24816, "dom": "G", "ent": "Tabligbo", "cls": "Southern Europe"}, {"id": 24817, "dom": "G", "ent": "Tabligbo", "cls": "Western Africa"}, {"id": 24818, "dom": "G", "ent": "Tabligbo", "cls": "Western Asia"}, {"id": 24819, "dom": "G", "ent": "Tabligbo", "cls": "Western Europe"}, {"id": 24820, "dom": "G", "ent": "Suan Luang", "cls": "Australia and New Zealand"}, {"id": 24821, "dom": "G", "ent": "Suan Luang", "cls": "Caribbean"}, {"id": 24822, "dom": "G", "ent": "Suan Luang", "cls": "Central America"}, {"id": 24823, "dom": "G", "ent": "Suan Luang", "cls": "Central Asia"}, {"id": 24824, "dom": "G", "ent": "Suan Luang", "cls": "Eastern Africa"}, {"id": 24825, "dom": "G", "ent": "Suan Luang", "cls": "Eastern Asia"}, {"id": 24826, "dom": "G", "ent": "Suan Luang", "cls": "Eastern Europe"}, {"id": 24827, "dom": "G", "ent": "Suan Luang", "cls": "Melanesia"}, {"id": 24828, "dom": "G", "ent": "Suan Luang", "cls": "Micronesia"}, {"id": 24829, "dom": "G", "ent": "Suan Luang", "cls": "Middle Africa"}, {"id": 24830, "dom": "G", "ent": "Suan Luang", "cls": "Northern Africa"}, {"id": 24831, "dom": "G", "ent": "Suan Luang", "cls": "Northern America"}, {"id": 24832, "dom": "G", "ent": "Suan Luang", "cls": "Northern Europe"}, {"id": 24833, "dom": "G", "ent": "Suan Luang", "cls": "Polynesia"}, {"id": 24834, "dom": "G", "ent": "Suan Luang", "cls": "South America"}, {"id": 24835, "dom": "G", "ent": "Suan Luang", "cls": "South-eastern Asia"}, {"id": 24836, "dom": "G", "ent": "Suan Luang", "cls": "Southern Africa"}, {"id": 24837, "dom": "G", "ent": "Suan Luang", "cls": "Southern Asia"}, {"id": 24838, "dom": "G", "ent": "Suan Luang", "cls": "Southern Europe"}, {"id": 24839, "dom": "G", "ent": "Suan Luang", "cls": "Western Africa"}, {"id": 24840, "dom": "G", "ent": "Suan Luang", "cls": "Western Asia"}, {"id": 24841, "dom": "G", "ent": "Suan Luang", "cls": "Western Europe"}, {"id": 24842, "dom": "G", "ent": "Bangkok Yai", "cls": "Australia and New Zealand"}, {"id": 24843, "dom": "G", "ent": "Bangkok Yai", "cls": "Caribbean"}, {"id": 24844, "dom": "G", "ent": "Bangkok Yai", "cls": "Central America"}, {"id": 24845, "dom": "G", "ent": "Bangkok Yai", "cls": "Central Asia"}, {"id": 24846, "dom": "G", "ent": "Bangkok Yai", "cls": "Eastern Africa"}, {"id": 24847, "dom": "G", "ent": "Bangkok Yai", "cls": "Eastern Asia"}, {"id": 24848, "dom": "G", "ent": "Bangkok Yai", "cls": "Eastern Europe"}, {"id": 24849, "dom": "G", "ent": "Bangkok Yai", "cls": "Melanesia"}, {"id": 24850, "dom": "G", "ent": "Bangkok Yai", "cls": "Micronesia"}, {"id": 24851, "dom": "G", "ent": "Bangkok Yai", "cls": "Middle Africa"}, {"id": 24852, "dom": "G", "ent": "Bangkok Yai", "cls": "Northern Africa"}, {"id": 24853, "dom": "G", "ent": "Bangkok Yai", "cls": "Northern America"}, {"id": 24854, "dom": "G", "ent": "Bangkok Yai", "cls": "Northern Europe"}, {"id": 24855, "dom": "G", "ent": "Bangkok Yai", "cls": "Polynesia"}, {"id": 24856, "dom": "G", "ent": "Bangkok Yai", "cls": "South America"}, {"id": 24857, "dom": "G", "ent": "Bangkok Yai", "cls": "South-eastern Asia"}, {"id": 24858, "dom": "G", "ent": "Bangkok Yai", "cls": "Southern Africa"}, {"id": 24859, "dom": "G", "ent": "Bangkok Yai", "cls": "Southern Asia"}, {"id": 24860, "dom": "G", "ent": "Bangkok Yai", "cls": "Southern Europe"}, {"id": 24861, "dom": "G", "ent": "Bangkok Yai", "cls": "Western Africa"}, {"id": 24862, "dom": "G", "ent": "Bangkok Yai", "cls": "Western Asia"}, {"id": 24863, "dom": "G", "ent": "Bangkok Yai", "cls": "Western Europe"}, {"id": 24864, "dom": "G", "ent": "Pran Buri", "cls": "Australia and New Zealand"}, {"id": 24865, "dom": "G", "ent": "Pran Buri", "cls": "Caribbean"}, {"id": 24866, "dom": "G", "ent": "Pran Buri", "cls": "Central America"}, {"id": 24867, "dom": "G", "ent": "Pran Buri", "cls": "Central Asia"}, {"id": 24868, "dom": "G", "ent": "Pran Buri", "cls": "Eastern Africa"}, {"id": 24869, "dom": "G", "ent": "Pran Buri", "cls": "Eastern Asia"}, {"id": 24870, "dom": "G", "ent": "Pran Buri", "cls": "Eastern Europe"}, {"id": 24871, "dom": "G", "ent": "Pran Buri", "cls": "Melanesia"}, {"id": 24872, "dom": "G", "ent": "Pran Buri", "cls": "Micronesia"}, {"id": 24873, "dom": "G", "ent": "Pran Buri", "cls": "Middle Africa"}, {"id": 24874, "dom": "G", "ent": "Pran Buri", "cls": "Northern Africa"}, {"id": 24875, "dom": "G", "ent": "Pran Buri", "cls": "Northern America"}, {"id": 24876, "dom": "G", "ent": "Pran Buri", "cls": "Northern Europe"}, {"id": 24877, "dom": "G", "ent": "Pran Buri", "cls": "Polynesia"}, {"id": 24878, "dom": "G", "ent": "Pran Buri", "cls": "South America"}, {"id": 24879, "dom": "G", "ent": "Pran Buri", "cls": "South-eastern Asia"}, {"id": 24880, "dom": "G", "ent": "Pran Buri", "cls": "Southern Africa"}, {"id": 24881, "dom": "G", "ent": "Pran Buri", "cls": "Southern Asia"}, {"id": 24882, "dom": "G", "ent": "Pran Buri", "cls": "Southern Europe"}, {"id": 24883, "dom": "G", "ent": "Pran Buri", "cls": "Western Africa"}, {"id": 24884, "dom": "G", "ent": "Pran Buri", "cls": "Western Asia"}, {"id": 24885, "dom": "G", "ent": "Pran Buri", "cls": "Western Europe"}, {"id": 24886, "dom": "G", "ent": "Bang Bon Nuea", "cls": "Australia and New Zealand"}, {"id": 24887, "dom": "G", "ent": "Bang Bon Nuea", "cls": "Caribbean"}, {"id": 24888, "dom": "G", "ent": "Bang Bon Nuea", "cls": "Central America"}, {"id": 24889, "dom": "G", "ent": "Bang Bon Nuea", "cls": "Central Asia"}, {"id": 24890, "dom": "G", "ent": "Bang Bon Nuea", "cls": "Eastern Africa"}, {"id": 24891, "dom": "G", "ent": "Bang Bon Nuea", "cls": "Eastern Asia"}, {"id": 24892, "dom": "G", "ent": "Bang Bon Nuea", "cls": "Eastern Europe"}, {"id": 24893, "dom": "G", "ent": "Bang Bon Nuea", "cls": "Melanesia"}, {"id": 24894, "dom": "G", "ent": "Bang Bon Nuea", "cls": "Micronesia"}, {"id": 24895, "dom": "G", "ent": "Bang Bon Nuea", "cls": "Middle Africa"}, {"id": 24896, "dom": "G", "ent": "Bang Bon Nuea", "cls": "Northern Africa"}, {"id": 24897, "dom": "G", "ent": "Bang Bon Nuea", "cls": "Northern America"}, {"id": 24898, "dom": "G", "ent": "Bang Bon Nuea", "cls": "Northern Europe"}, {"id": 24899, "dom": "G", "ent": "Bang Bon Nuea", "cls": "Polynesia"}, {"id": 24900, "dom": "G", "ent": "Bang Bon Nuea", "cls": "South America"}, {"id": 24901, "dom": "G", "ent": "Bang Bon Nuea", "cls": "South-eastern Asia"}, {"id": 24902, "dom": "G", "ent": "Bang Bon Nuea", "cls": "Southern Africa"}, {"id": 24903, "dom": "G", "ent": "Bang Bon Nuea", "cls": "Southern Asia"}, {"id": 24904, "dom": "G", "ent": "Bang Bon Nuea", "cls": "Southern Europe"}, {"id": 24905, "dom": "G", "ent": "Bang Bon Nuea", "cls": "Western Africa"}, {"id": 24906, "dom": "G", "ent": "Bang Bon Nuea", "cls": "Western Asia"}, {"id": 24907, "dom": "G", "ent": "Bang Bon Nuea", "cls": "Western Europe"}, {"id": 24908, "dom": "G", "ent": "Boshkengash", "cls": "Australia and New Zealand"}, {"id": 24909, "dom": "G", "ent": "Boshkengash", "cls": "Caribbean"}, {"id": 24910, "dom": "G", "ent": "Boshkengash", "cls": "Central America"}, {"id": 24911, "dom": "G", "ent": "Boshkengash", "cls": "Central Asia"}, {"id": 24912, "dom": "G", "ent": "Boshkengash", "cls": "Eastern Africa"}, {"id": 24913, "dom": "G", "ent": "Boshkengash", "cls": "Eastern Asia"}, {"id": 24914, "dom": "G", "ent": "Boshkengash", "cls": "Eastern Europe"}, {"id": 24915, "dom": "G", "ent": "Boshkengash", "cls": "Melanesia"}, {"id": 24916, "dom": "G", "ent": "Boshkengash", "cls": "Micronesia"}, {"id": 24917, "dom": "G", "ent": "Boshkengash", "cls": "Middle Africa"}, {"id": 24918, "dom": "G", "ent": "Boshkengash", "cls": "Northern Africa"}, {"id": 24919, "dom": "G", "ent": "Boshkengash", "cls": "Northern America"}, {"id": 24920, "dom": "G", "ent": "Boshkengash", "cls": "Northern Europe"}, {"id": 24921, "dom": "G", "ent": "Boshkengash", "cls": "Polynesia"}, {"id": 24922, "dom": "G", "ent": "Boshkengash", "cls": "South America"}, {"id": 24923, "dom": "G", "ent": "Boshkengash", "cls": "South-eastern Asia"}, {"id": 24924, "dom": "G", "ent": "Boshkengash", "cls": "Southern Africa"}, {"id": 24925, "dom": "G", "ent": "Boshkengash", "cls": "Southern Asia"}, {"id": 24926, "dom": "G", "ent": "Boshkengash", "cls": "Southern Europe"}, {"id": 24927, "dom": "G", "ent": "Boshkengash", "cls": "Western Africa"}, {"id": 24928, "dom": "G", "ent": "Boshkengash", "cls": "Western Asia"}, {"id": 24929, "dom": "G", "ent": "Boshkengash", "cls": "Western Europe"}, {"id": 24930, "dom": "G", "ent": "Norak", "cls": "Australia and New Zealand"}, {"id": 24931, "dom": "G", "ent": "Norak", "cls": "Caribbean"}, {"id": 24932, "dom": "G", "ent": "Norak", "cls": "Central America"}, {"id": 24933, "dom": "G", "ent": "Norak", "cls": "Central Asia"}, {"id": 24934, "dom": "G", "ent": "Norak", "cls": "Eastern Africa"}, {"id": 24935, "dom": "G", "ent": "Norak", "cls": "Eastern Asia"}, {"id": 24936, "dom": "G", "ent": "Norak", "cls": "Eastern Europe"}, {"id": 24937, "dom": "G", "ent": "Norak", "cls": "Melanesia"}, {"id": 24938, "dom": "G", "ent": "Norak", "cls": "Micronesia"}, {"id": 24939, "dom": "G", "ent": "Norak", "cls": "Middle Africa"}, {"id": 24940, "dom": "G", "ent": "Norak", "cls": "Northern Africa"}, {"id": 24941, "dom": "G", "ent": "Norak", "cls": "Northern America"}, {"id": 24942, "dom": "G", "ent": "Norak", "cls": "Northern Europe"}, {"id": 24943, "dom": "G", "ent": "Norak", "cls": "Polynesia"}, {"id": 24944, "dom": "G", "ent": "Norak", "cls": "South America"}, {"id": 24945, "dom": "G", "ent": "Norak", "cls": "South-eastern Asia"}, {"id": 24946, "dom": "G", "ent": "Norak", "cls": "Southern Africa"}, {"id": 24947, "dom": "G", "ent": "Norak", "cls": "Southern Asia"}, {"id": 24948, "dom": "G", "ent": "Norak", "cls": "Southern Europe"}, {"id": 24949, "dom": "G", "ent": "Norak", "cls": "Western Africa"}, {"id": 24950, "dom": "G", "ent": "Norak", "cls": "Western Asia"}, {"id": 24951, "dom": "G", "ent": "Norak", "cls": "Western Europe"}, {"id": 24952, "dom": "G", "ent": "Istiqlol", "cls": "Australia and New Zealand"}, {"id": 24953, "dom": "G", "ent": "Istiqlol", "cls": "Caribbean"}, {"id": 24954, "dom": "G", "ent": "Istiqlol", "cls": "Central America"}, {"id": 24955, "dom": "G", "ent": "Istiqlol", "cls": "Central Asia"}, {"id": 24956, "dom": "G", "ent": "Istiqlol", "cls": "Eastern Africa"}, {"id": 24957, "dom": "G", "ent": "Istiqlol", "cls": "Eastern Asia"}, {"id": 24958, "dom": "G", "ent": "Istiqlol", "cls": "Eastern Europe"}, {"id": 24959, "dom": "G", "ent": "Istiqlol", "cls": "Melanesia"}, {"id": 24960, "dom": "G", "ent": "Istiqlol", "cls": "Micronesia"}, {"id": 24961, "dom": "G", "ent": "Istiqlol", "cls": "Middle Africa"}, {"id": 24962, "dom": "G", "ent": "Istiqlol", "cls": "Northern Africa"}, {"id": 24963, "dom": "G", "ent": "Istiqlol", "cls": "Northern America"}, {"id": 24964, "dom": "G", "ent": "Istiqlol", "cls": "Northern Europe"}, {"id": 24965, "dom": "G", "ent": "Istiqlol", "cls": "Polynesia"}, {"id": 24966, "dom": "G", "ent": "Istiqlol", "cls": "South America"}, {"id": 24967, "dom": "G", "ent": "Istiqlol", "cls": "South-eastern Asia"}, {"id": 24968, "dom": "G", "ent": "Istiqlol", "cls": "Southern Africa"}, {"id": 24969, "dom": "G", "ent": "Istiqlol", "cls": "Southern Asia"}, {"id": 24970, "dom": "G", "ent": "Istiqlol", "cls": "Southern Europe"}, {"id": 24971, "dom": "G", "ent": "Istiqlol", "cls": "Western Africa"}, {"id": 24972, "dom": "G", "ent": "Istiqlol", "cls": "Western Asia"}, {"id": 24973, "dom": "G", "ent": "Istiqlol", "cls": "Western Europe"}, {"id": 24974, "dom": "G", "ent": "Vakhsh", "cls": "Australia and New Zealand"}, {"id": 24975, "dom": "G", "ent": "Vakhsh", "cls": "Caribbean"}, {"id": 24976, "dom": "G", "ent": "Vakhsh", "cls": "Central America"}, {"id": 24977, "dom": "G", "ent": "Vakhsh", "cls": "Central Asia"}, {"id": 24978, "dom": "G", "ent": "Vakhsh", "cls": "Eastern Africa"}, {"id": 24979, "dom": "G", "ent": "Vakhsh", "cls": "Eastern Asia"}, {"id": 24980, "dom": "G", "ent": "Vakhsh", "cls": "Eastern Europe"}, {"id": 24981, "dom": "G", "ent": "Vakhsh", "cls": "Melanesia"}, {"id": 24982, "dom": "G", "ent": "Vakhsh", "cls": "Micronesia"}, {"id": 24983, "dom": "G", "ent": "Vakhsh", "cls": "Middle Africa"}, {"id": 24984, "dom": "G", "ent": "Vakhsh", "cls": "Northern Africa"}, {"id": 24985, "dom": "G", "ent": "Vakhsh", "cls": "Northern America"}, {"id": 24986, "dom": "G", "ent": "Vakhsh", "cls": "Northern Europe"}, {"id": 24987, "dom": "G", "ent": "Vakhsh", "cls": "Polynesia"}, {"id": 24988, "dom": "G", "ent": "Vakhsh", "cls": "South America"}, {"id": 24989, "dom": "G", "ent": "Vakhsh", "cls": "South-eastern Asia"}, {"id": 24990, "dom": "G", "ent": "Vakhsh", "cls": "Southern Africa"}, {"id": 24991, "dom": "G", "ent": "Vakhsh", "cls": "Southern Asia"}, {"id": 24992, "dom": "G", "ent": "Vakhsh", "cls": "Southern Europe"}, {"id": 24993, "dom": "G", "ent": "Vakhsh", "cls": "Western Africa"}, {"id": 24994, "dom": "G", "ent": "Vakhsh", "cls": "Western Asia"}, {"id": 24995, "dom": "G", "ent": "Vakhsh", "cls": "Western Europe"}, {"id": 24996, "dom": "G", "ent": "Balkanabat", "cls": "Australia and New Zealand"}, {"id": 24997, "dom": "G", "ent": "Balkanabat", "cls": "Caribbean"}, {"id": 24998, "dom": "G", "ent": "Balkanabat", "cls": "Central America"}, {"id": 24999, "dom": "G", "ent": "Balkanabat", "cls": "Central Asia"}, {"id": 25000, "dom": "G", "ent": "Balkanabat", "cls": "Eastern Africa"}, {"id": 25001, "dom": "G", "ent": "Balkanabat", "cls": "Eastern Asia"}, {"id": 25002, "dom": "G", "ent": "Balkanabat", "cls": "Eastern Europe"}, {"id": 25003, "dom": "G", "ent": "Balkanabat", "cls": "Melanesia"}, {"id": 25004, "dom": "G", "ent": "Balkanabat", "cls": "Micronesia"}, {"id": 25005, "dom": "G", "ent": "Balkanabat", "cls": "Middle Africa"}, {"id": 25006, "dom": "G", "ent": "Balkanabat", "cls": "Northern Africa"}, {"id": 25007, "dom": "G", "ent": "Balkanabat", "cls": "Northern America"}, {"id": 25008, "dom": "G", "ent": "Balkanabat", "cls": "Northern Europe"}, {"id": 25009, "dom": "G", "ent": "Balkanabat", "cls": "Polynesia"}, {"id": 25010, "dom": "G", "ent": "Balkanabat", "cls": "South America"}, {"id": 25011, "dom": "G", "ent": "Balkanabat", "cls": "South-eastern Asia"}, {"id": 25012, "dom": "G", "ent": "Balkanabat", "cls": "Southern Africa"}, {"id": 25013, "dom": "G", "ent": "Balkanabat", "cls": "Southern Asia"}, {"id": 25014, "dom": "G", "ent": "Balkanabat", "cls": "Southern Europe"}, {"id": 25015, "dom": "G", "ent": "Balkanabat", "cls": "Western Africa"}, {"id": 25016, "dom": "G", "ent": "Balkanabat", "cls": "Western Asia"}, {"id": 25017, "dom": "G", "ent": "Balkanabat", "cls": "Western Europe"}, {"id": 25018, "dom": "G", "ent": "Bayramaly", "cls": "Australia and New Zealand"}, {"id": 25019, "dom": "G", "ent": "Bayramaly", "cls": "Caribbean"}, {"id": 25020, "dom": "G", "ent": "Bayramaly", "cls": "Central America"}, {"id": 25021, "dom": "G", "ent": "Bayramaly", "cls": "Central Asia"}, {"id": 25022, "dom": "G", "ent": "Bayramaly", "cls": "Eastern Africa"}, {"id": 25023, "dom": "G", "ent": "Bayramaly", "cls": "Eastern Asia"}, {"id": 25024, "dom": "G", "ent": "Bayramaly", "cls": "Eastern Europe"}, {"id": 25025, "dom": "G", "ent": "Bayramaly", "cls": "Melanesia"}, {"id": 25026, "dom": "G", "ent": "Bayramaly", "cls": "Micronesia"}, {"id": 25027, "dom": "G", "ent": "Bayramaly", "cls": "Middle Africa"}, {"id": 25028, "dom": "G", "ent": "Bayramaly", "cls": "Northern Africa"}, {"id": 25029, "dom": "G", "ent": "Bayramaly", "cls": "Northern America"}, {"id": 25030, "dom": "G", "ent": "Bayramaly", "cls": "Northern Europe"}, {"id": 25031, "dom": "G", "ent": "Bayramaly", "cls": "Polynesia"}, {"id": 25032, "dom": "G", "ent": "Bayramaly", "cls": "South America"}, {"id": 25033, "dom": "G", "ent": "Bayramaly", "cls": "South-eastern Asia"}, {"id": 25034, "dom": "G", "ent": "Bayramaly", "cls": "Southern Africa"}, {"id": 25035, "dom": "G", "ent": "Bayramaly", "cls": "Southern Asia"}, {"id": 25036, "dom": "G", "ent": "Bayramaly", "cls": "Southern Europe"}, {"id": 25037, "dom": "G", "ent": "Bayramaly", "cls": "Western Africa"}, {"id": 25038, "dom": "G", "ent": "Bayramaly", "cls": "Western Asia"}, {"id": 25039, "dom": "G", "ent": "Bayramaly", "cls": "Western Europe"}, {"id": 25040, "dom": "G", "ent": "Yylanly", "cls": "Australia and New Zealand"}, {"id": 25041, "dom": "G", "ent": "Yylanly", "cls": "Caribbean"}, {"id": 25042, "dom": "G", "ent": "Yylanly", "cls": "Central America"}, {"id": 25043, "dom": "G", "ent": "Yylanly", "cls": "Central Asia"}, {"id": 25044, "dom": "G", "ent": "Yylanly", "cls": "Eastern Africa"}, {"id": 25045, "dom": "G", "ent": "Yylanly", "cls": "Eastern Asia"}, {"id": 25046, "dom": "G", "ent": "Yylanly", "cls": "Eastern Europe"}, {"id": 25047, "dom": "G", "ent": "Yylanly", "cls": "Melanesia"}, {"id": 25048, "dom": "G", "ent": "Yylanly", "cls": "Micronesia"}, {"id": 25049, "dom": "G", "ent": "Yylanly", "cls": "Middle Africa"}, {"id": 25050, "dom": "G", "ent": "Yylanly", "cls": "Northern Africa"}, {"id": 25051, "dom": "G", "ent": "Yylanly", "cls": "Northern America"}, {"id": 25052, "dom": "G", "ent": "Yylanly", "cls": "Northern Europe"}, {"id": 25053, "dom": "G", "ent": "Yylanly", "cls": "Polynesia"}, {"id": 25054, "dom": "G", "ent": "Yylanly", "cls": "South America"}, {"id": 25055, "dom": "G", "ent": "Yylanly", "cls": "South-eastern Asia"}, {"id": 25056, "dom": "G", "ent": "Yylanly", "cls": "Southern Africa"}, {"id": 25057, "dom": "G", "ent": "Yylanly", "cls": "Southern Asia"}, {"id": 25058, "dom": "G", "ent": "Yylanly", "cls": "Southern Europe"}, {"id": 25059, "dom": "G", "ent": "Yylanly", "cls": "Western Africa"}, {"id": 25060, "dom": "G", "ent": "Yylanly", "cls": "Western Asia"}, {"id": 25061, "dom": "G", "ent": "Yylanly", "cls": "Western Europe"}, {"id": 25062, "dom": "G", "ent": "Gubadag", "cls": "Australia and New Zealand"}, {"id": 25063, "dom": "G", "ent": "Gubadag", "cls": "Caribbean"}, {"id": 25064, "dom": "G", "ent": "Gubadag", "cls": "Central America"}, {"id": 25065, "dom": "G", "ent": "Gubadag", "cls": "Central Asia"}, {"id": 25066, "dom": "G", "ent": "Gubadag", "cls": "Eastern Africa"}, {"id": 25067, "dom": "G", "ent": "Gubadag", "cls": "Eastern Asia"}, {"id": 25068, "dom": "G", "ent": "Gubadag", "cls": "Eastern Europe"}, {"id": 25069, "dom": "G", "ent": "Gubadag", "cls": "Melanesia"}, {"id": 25070, "dom": "G", "ent": "Gubadag", "cls": "Micronesia"}, {"id": 25071, "dom": "G", "ent": "Gubadag", "cls": "Middle Africa"}, {"id": 25072, "dom": "G", "ent": "Gubadag", "cls": "Northern Africa"}, {"id": 25073, "dom": "G", "ent": "Gubadag", "cls": "Northern America"}, {"id": 25074, "dom": "G", "ent": "Gubadag", "cls": "Northern Europe"}, {"id": 25075, "dom": "G", "ent": "Gubadag", "cls": "Polynesia"}, {"id": 25076, "dom": "G", "ent": "Gubadag", "cls": "South America"}, {"id": 25077, "dom": "G", "ent": "Gubadag", "cls": "South-eastern Asia"}, {"id": 25078, "dom": "G", "ent": "Gubadag", "cls": "Southern Africa"}, {"id": 25079, "dom": "G", "ent": "Gubadag", "cls": "Southern Asia"}, {"id": 25080, "dom": "G", "ent": "Gubadag", "cls": "Southern Europe"}, {"id": 25081, "dom": "G", "ent": "Gubadag", "cls": "Western Africa"}, {"id": 25082, "dom": "G", "ent": "Gubadag", "cls": "Western Asia"}, {"id": 25083, "dom": "G", "ent": "Gubadag", "cls": "Western Europe"}, {"id": 25084, "dom": "G", "ent": "Menzel Bourguiba", "cls": "Australia and New Zealand"}, {"id": 25085, "dom": "G", "ent": "Menzel Bourguiba", "cls": "Caribbean"}, {"id": 25086, "dom": "G", "ent": "Menzel Bourguiba", "cls": "Central America"}, {"id": 25087, "dom": "G", "ent": "Menzel Bourguiba", "cls": "Central Asia"}, {"id": 25088, "dom": "G", "ent": "Menzel Bourguiba", "cls": "Eastern Africa"}, {"id": 25089, "dom": "G", "ent": "Menzel Bourguiba", "cls": "Eastern Asia"}, {"id": 25090, "dom": "G", "ent": "Menzel Bourguiba", "cls": "Eastern Europe"}, {"id": 25091, "dom": "G", "ent": "Menzel Bourguiba", "cls": "Melanesia"}, {"id": 25092, "dom": "G", "ent": "Menzel Bourguiba", "cls": "Micronesia"}, {"id": 25093, "dom": "G", "ent": "Menzel Bourguiba", "cls": "Middle Africa"}, {"id": 25094, "dom": "G", "ent": "Menzel Bourguiba", "cls": "Northern Africa"}, {"id": 25095, "dom": "G", "ent": "Menzel Bourguiba", "cls": "Northern America"}, {"id": 25096, "dom": "G", "ent": "Menzel Bourguiba", "cls": "Northern Europe"}, {"id": 25097, "dom": "G", "ent": "Menzel Bourguiba", "cls": "Polynesia"}, {"id": 25098, "dom": "G", "ent": "Menzel Bourguiba", "cls": "South America"}, {"id": 25099, "dom": "G", "ent": "Menzel Bourguiba", "cls": "South-eastern Asia"}, {"id": 25100, "dom": "G", "ent": "Menzel Bourguiba", "cls": "Southern Africa"}, {"id": 25101, "dom": "G", "ent": "Menzel Bourguiba", "cls": "Southern Asia"}, {"id": 25102, "dom": "G", "ent": "Menzel Bourguiba", "cls": "Southern Europe"}, {"id": 25103, "dom": "G", "ent": "Menzel Bourguiba", "cls": "Western Africa"}, {"id": 25104, "dom": "G", "ent": "Menzel Bourguiba", "cls": "Western Asia"}, {"id": 25105, "dom": "G", "ent": "Menzel Bourguiba", "cls": "Western Europe"}, {"id": 25106, "dom": "G", "ent": "El Kef", "cls": "Australia and New Zealand"}, {"id": 25107, "dom": "G", "ent": "El Kef", "cls": "Caribbean"}, {"id": 25108, "dom": "G", "ent": "El Kef", "cls": "Central America"}, {"id": 25109, "dom": "G", "ent": "El Kef", "cls": "Central Asia"}, {"id": 25110, "dom": "G", "ent": "El Kef", "cls": "Eastern Africa"}, {"id": 25111, "dom": "G", "ent": "El Kef", "cls": "Eastern Asia"}, {"id": 25112, "dom": "G", "ent": "El Kef", "cls": "Eastern Europe"}, {"id": 25113, "dom": "G", "ent": "El Kef", "cls": "Melanesia"}, {"id": 25114, "dom": "G", "ent": "El Kef", "cls": "Micronesia"}, {"id": 25115, "dom": "G", "ent": "El Kef", "cls": "Middle Africa"}, {"id": 25116, "dom": "G", "ent": "El Kef", "cls": "Northern Africa"}, {"id": 25117, "dom": "G", "ent": "El Kef", "cls": "Northern America"}, {"id": 25118, "dom": "G", "ent": "El Kef", "cls": "Northern Europe"}, {"id": 25119, "dom": "G", "ent": "El Kef", "cls": "Polynesia"}, {"id": 25120, "dom": "G", "ent": "El Kef", "cls": "South America"}, {"id": 25121, "dom": "G", "ent": "El Kef", "cls": "South-eastern Asia"}, {"id": 25122, "dom": "G", "ent": "El Kef", "cls": "Southern Africa"}, {"id": 25123, "dom": "G", "ent": "El Kef", "cls": "Southern Asia"}, {"id": 25124, "dom": "G", "ent": "El Kef", "cls": "Southern Europe"}, {"id": 25125, "dom": "G", "ent": "El Kef", "cls": "Western Africa"}, {"id": 25126, "dom": "G", "ent": "El Kef", "cls": "Western Asia"}, {"id": 25127, "dom": "G", "ent": "El Kef", "cls": "Western Europe"}, {"id": 25128, "dom": "G", "ent": "Jendouba", "cls": "Australia and New Zealand"}, {"id": 25129, "dom": "G", "ent": "Jendouba", "cls": "Caribbean"}, {"id": 25130, "dom": "G", "ent": "Jendouba", "cls": "Central America"}, {"id": 25131, "dom": "G", "ent": "Jendouba", "cls": "Central Asia"}, {"id": 25132, "dom": "G", "ent": "Jendouba", "cls": "Eastern Africa"}, {"id": 25133, "dom": "G", "ent": "Jendouba", "cls": "Eastern Asia"}, {"id": 25134, "dom": "G", "ent": "Jendouba", "cls": "Eastern Europe"}, {"id": 25135, "dom": "G", "ent": "Jendouba", "cls": "Melanesia"}, {"id": 25136, "dom": "G", "ent": "Jendouba", "cls": "Micronesia"}, {"id": 25137, "dom": "G", "ent": "Jendouba", "cls": "Middle Africa"}, {"id": 25138, "dom": "G", "ent": "Jendouba", "cls": "Northern Africa"}, {"id": 25139, "dom": "G", "ent": "Jendouba", "cls": "Northern America"}, {"id": 25140, "dom": "G", "ent": "Jendouba", "cls": "Northern Europe"}, {"id": 25141, "dom": "G", "ent": "Jendouba", "cls": "Polynesia"}, {"id": 25142, "dom": "G", "ent": "Jendouba", "cls": "South America"}, {"id": 25143, "dom": "G", "ent": "Jendouba", "cls": "South-eastern Asia"}, {"id": 25144, "dom": "G", "ent": "Jendouba", "cls": "Southern Africa"}, {"id": 25145, "dom": "G", "ent": "Jendouba", "cls": "Southern Asia"}, {"id": 25146, "dom": "G", "ent": "Jendouba", "cls": "Southern Europe"}, {"id": 25147, "dom": "G", "ent": "Jendouba", "cls": "Western Africa"}, {"id": 25148, "dom": "G", "ent": "Jendouba", "cls": "Western Asia"}, {"id": 25149, "dom": "G", "ent": "Jendouba", "cls": "Western Europe"}, {"id": 25150, "dom": "G", "ent": "Borj Cedria", "cls": "Australia and New Zealand"}, {"id": 25151, "dom": "G", "ent": "Borj Cedria", "cls": "Caribbean"}, {"id": 25152, "dom": "G", "ent": "Borj Cedria", "cls": "Central America"}, {"id": 25153, "dom": "G", "ent": "Borj Cedria", "cls": "Central Asia"}, {"id": 25154, "dom": "G", "ent": "Borj Cedria", "cls": "Eastern Africa"}, {"id": 25155, "dom": "G", "ent": "Borj Cedria", "cls": "Eastern Asia"}, {"id": 25156, "dom": "G", "ent": "Borj Cedria", "cls": "Eastern Europe"}, {"id": 25157, "dom": "G", "ent": "Borj Cedria", "cls": "Melanesia"}, {"id": 25158, "dom": "G", "ent": "Borj Cedria", "cls": "Micronesia"}, {"id": 25159, "dom": "G", "ent": "Borj Cedria", "cls": "Middle Africa"}, {"id": 25160, "dom": "G", "ent": "Borj Cedria", "cls": "Northern Africa"}, {"id": 25161, "dom": "G", "ent": "Borj Cedria", "cls": "Northern America"}, {"id": 25162, "dom": "G", "ent": "Borj Cedria", "cls": "Northern Europe"}, {"id": 25163, "dom": "G", "ent": "Borj Cedria", "cls": "Polynesia"}, {"id": 25164, "dom": "G", "ent": "Borj Cedria", "cls": "South America"}, {"id": 25165, "dom": "G", "ent": "Borj Cedria", "cls": "South-eastern Asia"}, {"id": 25166, "dom": "G", "ent": "Borj Cedria", "cls": "Southern Africa"}, {"id": 25167, "dom": "G", "ent": "Borj Cedria", "cls": "Southern Asia"}, {"id": 25168, "dom": "G", "ent": "Borj Cedria", "cls": "Southern Europe"}, {"id": 25169, "dom": "G", "ent": "Borj Cedria", "cls": "Western Africa"}, {"id": 25170, "dom": "G", "ent": "Borj Cedria", "cls": "Western Asia"}, {"id": 25171, "dom": "G", "ent": "Borj Cedria", "cls": "Western Europe"}, {"id": 25172, "dom": "G", "ent": "Arima", "cls": "Australia and New Zealand"}, {"id": 25173, "dom": "G", "ent": "Arima", "cls": "Caribbean"}, {"id": 25174, "dom": "G", "ent": "Arima", "cls": "Central America"}, {"id": 25175, "dom": "G", "ent": "Arima", "cls": "Central Asia"}, {"id": 25176, "dom": "G", "ent": "Arima", "cls": "Eastern Africa"}, {"id": 25177, "dom": "G", "ent": "Arima", "cls": "Eastern Asia"}, {"id": 25178, "dom": "G", "ent": "Arima", "cls": "Eastern Europe"}, {"id": 25179, "dom": "G", "ent": "Arima", "cls": "Melanesia"}, {"id": 25180, "dom": "G", "ent": "Arima", "cls": "Micronesia"}, {"id": 25181, "dom": "G", "ent": "Arima", "cls": "Middle Africa"}, {"id": 25182, "dom": "G", "ent": "Arima", "cls": "Northern Africa"}, {"id": 25183, "dom": "G", "ent": "Arima", "cls": "Northern America"}, {"id": 25184, "dom": "G", "ent": "Arima", "cls": "Northern Europe"}, {"id": 25185, "dom": "G", "ent": "Arima", "cls": "Polynesia"}, {"id": 25186, "dom": "G", "ent": "Arima", "cls": "South America"}, {"id": 25187, "dom": "G", "ent": "Arima", "cls": "South-eastern Asia"}, {"id": 25188, "dom": "G", "ent": "Arima", "cls": "Southern Africa"}, {"id": 25189, "dom": "G", "ent": "Arima", "cls": "Southern Asia"}, {"id": 25190, "dom": "G", "ent": "Arima", "cls": "Southern Europe"}, {"id": 25191, "dom": "G", "ent": "Arima", "cls": "Western Africa"}, {"id": 25192, "dom": "G", "ent": "Arima", "cls": "Western Asia"}, {"id": 25193, "dom": "G", "ent": "Arima", "cls": "Western Europe"}, {"id": 25194, "dom": "G", "ent": "Diego Martin", "cls": "Australia and New Zealand"}, {"id": 25195, "dom": "G", "ent": "Diego Martin", "cls": "Caribbean"}, {"id": 25196, "dom": "G", "ent": "Diego Martin", "cls": "Central America"}, {"id": 25197, "dom": "G", "ent": "Diego Martin", "cls": "Central Asia"}, {"id": 25198, "dom": "G", "ent": "Diego Martin", "cls": "Eastern Africa"}, {"id": 25199, "dom": "G", "ent": "Diego Martin", "cls": "Eastern Asia"}, {"id": 25200, "dom": "G", "ent": "Diego Martin", "cls": "Eastern Europe"}, {"id": 25201, "dom": "G", "ent": "Diego Martin", "cls": "Melanesia"}, {"id": 25202, "dom": "G", "ent": "Diego Martin", "cls": "Micronesia"}, {"id": 25203, "dom": "G", "ent": "Diego Martin", "cls": "Middle Africa"}, {"id": 25204, "dom": "G", "ent": "Diego Martin", "cls": "Northern Africa"}, {"id": 25205, "dom": "G", "ent": "Diego Martin", "cls": "Northern America"}, {"id": 25206, "dom": "G", "ent": "Diego Martin", "cls": "Northern Europe"}, {"id": 25207, "dom": "G", "ent": "Diego Martin", "cls": "Polynesia"}, {"id": 25208, "dom": "G", "ent": "Diego Martin", "cls": "South America"}, {"id": 25209, "dom": "G", "ent": "Diego Martin", "cls": "South-eastern Asia"}, {"id": 25210, "dom": "G", "ent": "Diego Martin", "cls": "Southern Africa"}, {"id": 25211, "dom": "G", "ent": "Diego Martin", "cls": "Southern Asia"}, {"id": 25212, "dom": "G", "ent": "Diego Martin", "cls": "Southern Europe"}, {"id": 25213, "dom": "G", "ent": "Diego Martin", "cls": "Western Africa"}, {"id": 25214, "dom": "G", "ent": "Diego Martin", "cls": "Western Asia"}, {"id": 25215, "dom": "G", "ent": "Diego Martin", "cls": "Western Europe"}, {"id": 25216, "dom": "G", "ent": "Point Fortin", "cls": "Australia and New Zealand"}, {"id": 25217, "dom": "G", "ent": "Point Fortin", "cls": "Caribbean"}, {"id": 25218, "dom": "G", "ent": "Point Fortin", "cls": "Central America"}, {"id": 25219, "dom": "G", "ent": "Point Fortin", "cls": "Central Asia"}, {"id": 25220, "dom": "G", "ent": "Point Fortin", "cls": "Eastern Africa"}, {"id": 25221, "dom": "G", "ent": "Point Fortin", "cls": "Eastern Asia"}, {"id": 25222, "dom": "G", "ent": "Point Fortin", "cls": "Eastern Europe"}, {"id": 25223, "dom": "G", "ent": "Point Fortin", "cls": "Melanesia"}, {"id": 25224, "dom": "G", "ent": "Point Fortin", "cls": "Micronesia"}, {"id": 25225, "dom": "G", "ent": "Point Fortin", "cls": "Middle Africa"}, {"id": 25226, "dom": "G", "ent": "Point Fortin", "cls": "Northern Africa"}, {"id": 25227, "dom": "G", "ent": "Point Fortin", "cls": "Northern America"}, {"id": 25228, "dom": "G", "ent": "Point Fortin", "cls": "Northern Europe"}, {"id": 25229, "dom": "G", "ent": "Point Fortin", "cls": "Polynesia"}, {"id": 25230, "dom": "G", "ent": "Point Fortin", "cls": "South America"}, {"id": 25231, "dom": "G", "ent": "Point Fortin", "cls": "South-eastern Asia"}, {"id": 25232, "dom": "G", "ent": "Point Fortin", "cls": "Southern Africa"}, {"id": 25233, "dom": "G", "ent": "Point Fortin", "cls": "Southern Asia"}, {"id": 25234, "dom": "G", "ent": "Point Fortin", "cls": "Southern Europe"}, {"id": 25235, "dom": "G", "ent": "Point Fortin", "cls": "Western Africa"}, {"id": 25236, "dom": "G", "ent": "Point Fortin", "cls": "Western Asia"}, {"id": 25237, "dom": "G", "ent": "Point Fortin", "cls": "Western Europe"}, {"id": 25238, "dom": "G", "ent": "Tunapuna", "cls": "Australia and New Zealand"}, {"id": 25239, "dom": "G", "ent": "Tunapuna", "cls": "Caribbean"}, {"id": 25240, "dom": "G", "ent": "Tunapuna", "cls": "Central America"}, {"id": 25241, "dom": "G", "ent": "Tunapuna", "cls": "Central Asia"}, {"id": 25242, "dom": "G", "ent": "Tunapuna", "cls": "Eastern Africa"}, {"id": 25243, "dom": "G", "ent": "Tunapuna", "cls": "Eastern Asia"}, {"id": 25244, "dom": "G", "ent": "Tunapuna", "cls": "Eastern Europe"}, {"id": 25245, "dom": "G", "ent": "Tunapuna", "cls": "Melanesia"}, {"id": 25246, "dom": "G", "ent": "Tunapuna", "cls": "Micronesia"}, {"id": 25247, "dom": "G", "ent": "Tunapuna", "cls": "Middle Africa"}, {"id": 25248, "dom": "G", "ent": "Tunapuna", "cls": "Northern Africa"}, {"id": 25249, "dom": "G", "ent": "Tunapuna", "cls": "Northern America"}, {"id": 25250, "dom": "G", "ent": "Tunapuna", "cls": "Northern Europe"}, {"id": 25251, "dom": "G", "ent": "Tunapuna", "cls": "Polynesia"}, {"id": 25252, "dom": "G", "ent": "Tunapuna", "cls": "South America"}, {"id": 25253, "dom": "G", "ent": "Tunapuna", "cls": "South-eastern Asia"}, {"id": 25254, "dom": "G", "ent": "Tunapuna", "cls": "Southern Africa"}, {"id": 25255, "dom": "G", "ent": "Tunapuna", "cls": "Southern Asia"}, {"id": 25256, "dom": "G", "ent": "Tunapuna", "cls": "Southern Europe"}, {"id": 25257, "dom": "G", "ent": "Tunapuna", "cls": "Western Africa"}, {"id": 25258, "dom": "G", "ent": "Tunapuna", "cls": "Western Asia"}, {"id": 25259, "dom": "G", "ent": "Tunapuna", "cls": "Western Europe"}, {"id": 25260, "dom": "G", "ent": "Mbuguni", "cls": "Australia and New Zealand"}, {"id": 25261, "dom": "G", "ent": "Mbuguni", "cls": "Caribbean"}, {"id": 25262, "dom": "G", "ent": "Mbuguni", "cls": "Central America"}, {"id": 25263, "dom": "G", "ent": "Mbuguni", "cls": "Central Asia"}, {"id": 25264, "dom": "G", "ent": "Mbuguni", "cls": "Eastern Africa"}, {"id": 25265, "dom": "G", "ent": "Mbuguni", "cls": "Eastern Asia"}, {"id": 25266, "dom": "G", "ent": "Mbuguni", "cls": "Eastern Europe"}, {"id": 25267, "dom": "G", "ent": "Mbuguni", "cls": "Melanesia"}, {"id": 25268, "dom": "G", "ent": "Mbuguni", "cls": "Micronesia"}, {"id": 25269, "dom": "G", "ent": "Mbuguni", "cls": "Middle Africa"}, {"id": 25270, "dom": "G", "ent": "Mbuguni", "cls": "Northern Africa"}, {"id": 25271, "dom": "G", "ent": "Mbuguni", "cls": "Northern America"}, {"id": 25272, "dom": "G", "ent": "Mbuguni", "cls": "Northern Europe"}, {"id": 25273, "dom": "G", "ent": "Mbuguni", "cls": "Polynesia"}, {"id": 25274, "dom": "G", "ent": "Mbuguni", "cls": "South America"}, {"id": 25275, "dom": "G", "ent": "Mbuguni", "cls": "South-eastern Asia"}, {"id": 25276, "dom": "G", "ent": "Mbuguni", "cls": "Southern Africa"}, {"id": 25277, "dom": "G", "ent": "Mbuguni", "cls": "Southern Asia"}, {"id": 25278, "dom": "G", "ent": "Mbuguni", "cls": "Southern Europe"}, {"id": 25279, "dom": "G", "ent": "Mbuguni", "cls": "Western Africa"}, {"id": 25280, "dom": "G", "ent": "Mbuguni", "cls": "Western Asia"}, {"id": 25281, "dom": "G", "ent": "Mbuguni", "cls": "Western Europe"}, {"id": 25282, "dom": "G", "ent": "Namanyere", "cls": "Australia and New Zealand"}, {"id": 25283, "dom": "G", "ent": "Namanyere", "cls": "Caribbean"}, {"id": 25284, "dom": "G", "ent": "Namanyere", "cls": "Central America"}, {"id": 25285, "dom": "G", "ent": "Namanyere", "cls": "Central Asia"}, {"id": 25286, "dom": "G", "ent": "Namanyere", "cls": "Eastern Africa"}, {"id": 25287, "dom": "G", "ent": "Namanyere", "cls": "Eastern Asia"}, {"id": 25288, "dom": "G", "ent": "Namanyere", "cls": "Eastern Europe"}, {"id": 25289, "dom": "G", "ent": "Namanyere", "cls": "Melanesia"}, {"id": 25290, "dom": "G", "ent": "Namanyere", "cls": "Micronesia"}, {"id": 25291, "dom": "G", "ent": "Namanyere", "cls": "Middle Africa"}, {"id": 25292, "dom": "G", "ent": "Namanyere", "cls": "Northern Africa"}, {"id": 25293, "dom": "G", "ent": "Namanyere", "cls": "Northern America"}, {"id": 25294, "dom": "G", "ent": "Namanyere", "cls": "Northern Europe"}, {"id": 25295, "dom": "G", "ent": "Namanyere", "cls": "Polynesia"}, {"id": 25296, "dom": "G", "ent": "Namanyere", "cls": "South America"}, {"id": 25297, "dom": "G", "ent": "Namanyere", "cls": "South-eastern Asia"}, {"id": 25298, "dom": "G", "ent": "Namanyere", "cls": "Southern Africa"}, {"id": 25299, "dom": "G", "ent": "Namanyere", "cls": "Southern Asia"}, {"id": 25300, "dom": "G", "ent": "Namanyere", "cls": "Southern Europe"}, {"id": 25301, "dom": "G", "ent": "Namanyere", "cls": "Western Africa"}, {"id": 25302, "dom": "G", "ent": "Namanyere", "cls": "Western Asia"}, {"id": 25303, "dom": "G", "ent": "Namanyere", "cls": "Western Europe"}, {"id": 25304, "dom": "G", "ent": "Ngudu", "cls": "Australia and New Zealand"}, {"id": 25305, "dom": "G", "ent": "Ngudu", "cls": "Caribbean"}, {"id": 25306, "dom": "G", "ent": "Ngudu", "cls": "Central America"}, {"id": 25307, "dom": "G", "ent": "Ngudu", "cls": "Central Asia"}, {"id": 25308, "dom": "G", "ent": "Ngudu", "cls": "Eastern Africa"}, {"id": 25309, "dom": "G", "ent": "Ngudu", "cls": "Eastern Asia"}, {"id": 25310, "dom": "G", "ent": "Ngudu", "cls": "Eastern Europe"}, {"id": 25311, "dom": "G", "ent": "Ngudu", "cls": "Melanesia"}, {"id": 25312, "dom": "G", "ent": "Ngudu", "cls": "Micronesia"}, {"id": 25313, "dom": "G", "ent": "Ngudu", "cls": "Middle Africa"}, {"id": 25314, "dom": "G", "ent": "Ngudu", "cls": "Northern Africa"}, {"id": 25315, "dom": "G", "ent": "Ngudu", "cls": "Northern America"}, {"id": 25316, "dom": "G", "ent": "Ngudu", "cls": "Northern Europe"}, {"id": 25317, "dom": "G", "ent": "Ngudu", "cls": "Polynesia"}, {"id": 25318, "dom": "G", "ent": "Ngudu", "cls": "South America"}, {"id": 25319, "dom": "G", "ent": "Ngudu", "cls": "South-eastern Asia"}, {"id": 25320, "dom": "G", "ent": "Ngudu", "cls": "Southern Africa"}, {"id": 25321, "dom": "G", "ent": "Ngudu", "cls": "Southern Asia"}, {"id": 25322, "dom": "G", "ent": "Ngudu", "cls": "Southern Europe"}, {"id": 25323, "dom": "G", "ent": "Ngudu", "cls": "Western Africa"}, {"id": 25324, "dom": "G", "ent": "Ngudu", "cls": "Western Asia"}, {"id": 25325, "dom": "G", "ent": "Ngudu", "cls": "Western Europe"}, {"id": 25326, "dom": "G", "ent": "Kibara", "cls": "Australia and New Zealand"}, {"id": 25327, "dom": "G", "ent": "Kibara", "cls": "Caribbean"}, {"id": 25328, "dom": "G", "ent": "Kibara", "cls": "Central America"}, {"id": 25329, "dom": "G", "ent": "Kibara", "cls": "Central Asia"}, {"id": 25330, "dom": "G", "ent": "Kibara", "cls": "Eastern Africa"}, {"id": 25331, "dom": "G", "ent": "Kibara", "cls": "Eastern Asia"}, {"id": 25332, "dom": "G", "ent": "Kibara", "cls": "Eastern Europe"}, {"id": 25333, "dom": "G", "ent": "Kibara", "cls": "Melanesia"}, {"id": 25334, "dom": "G", "ent": "Kibara", "cls": "Micronesia"}, {"id": 25335, "dom": "G", "ent": "Kibara", "cls": "Middle Africa"}, {"id": 25336, "dom": "G", "ent": "Kibara", "cls": "Northern Africa"}, {"id": 25337, "dom": "G", "ent": "Kibara", "cls": "Northern America"}, {"id": 25338, "dom": "G", "ent": "Kibara", "cls": "Northern Europe"}, {"id": 25339, "dom": "G", "ent": "Kibara", "cls": "Polynesia"}, {"id": 25340, "dom": "G", "ent": "Kibara", "cls": "South America"}, {"id": 25341, "dom": "G", "ent": "Kibara", "cls": "South-eastern Asia"}, {"id": 25342, "dom": "G", "ent": "Kibara", "cls": "Southern Africa"}, {"id": 25343, "dom": "G", "ent": "Kibara", "cls": "Southern Asia"}, {"id": 25344, "dom": "G", "ent": "Kibara", "cls": "Southern Europe"}, {"id": 25345, "dom": "G", "ent": "Kibara", "cls": "Western Africa"}, {"id": 25346, "dom": "G", "ent": "Kibara", "cls": "Western Asia"}, {"id": 25347, "dom": "G", "ent": "Kibara", "cls": "Western Europe"}, {"id": 25348, "dom": "G", "ent": "Kryukiv", "cls": "Australia and New Zealand"}, {"id": 25349, "dom": "G", "ent": "Kryukiv", "cls": "Caribbean"}, {"id": 25350, "dom": "G", "ent": "Kryukiv", "cls": "Central America"}, {"id": 25351, "dom": "G", "ent": "Kryukiv", "cls": "Central Asia"}, {"id": 25352, "dom": "G", "ent": "Kryukiv", "cls": "Eastern Africa"}, {"id": 25353, "dom": "G", "ent": "Kryukiv", "cls": "Eastern Asia"}, {"id": 25354, "dom": "G", "ent": "Kryukiv", "cls": "Eastern Europe"}, {"id": 25355, "dom": "G", "ent": "Kryukiv", "cls": "Melanesia"}, {"id": 25356, "dom": "G", "ent": "Kryukiv", "cls": "Micronesia"}, {"id": 25357, "dom": "G", "ent": "Kryukiv", "cls": "Middle Africa"}, {"id": 25358, "dom": "G", "ent": "Kryukiv", "cls": "Northern Africa"}, {"id": 25359, "dom": "G", "ent": "Kryukiv", "cls": "Northern America"}, {"id": 25360, "dom": "G", "ent": "Kryukiv", "cls": "Northern Europe"}, {"id": 25361, "dom": "G", "ent": "Kryukiv", "cls": "Polynesia"}, {"id": 25362, "dom": "G", "ent": "Kryukiv", "cls": "South America"}, {"id": 25363, "dom": "G", "ent": "Kryukiv", "cls": "South-eastern Asia"}, {"id": 25364, "dom": "G", "ent": "Kryukiv", "cls": "Southern Africa"}, {"id": 25365, "dom": "G", "ent": "Kryukiv", "cls": "Southern Asia"}, {"id": 25366, "dom": "G", "ent": "Kryukiv", "cls": "Southern Europe"}, {"id": 25367, "dom": "G", "ent": "Kryukiv", "cls": "Western Africa"}, {"id": 25368, "dom": "G", "ent": "Kryukiv", "cls": "Western Asia"}, {"id": 25369, "dom": "G", "ent": "Kryukiv", "cls": "Western Europe"}, {"id": 25370, "dom": "G", "ent": "Debaltseve", "cls": "Australia and New Zealand"}, {"id": 25371, "dom": "G", "ent": "Debaltseve", "cls": "Caribbean"}, {"id": 25372, "dom": "G", "ent": "Debaltseve", "cls": "Central America"}, {"id": 25373, "dom": "G", "ent": "Debaltseve", "cls": "Central Asia"}, {"id": 25374, "dom": "G", "ent": "Debaltseve", "cls": "Eastern Africa"}, {"id": 25375, "dom": "G", "ent": "Debaltseve", "cls": "Eastern Asia"}, {"id": 25376, "dom": "G", "ent": "Debaltseve", "cls": "Eastern Europe"}, {"id": 25377, "dom": "G", "ent": "Debaltseve", "cls": "Melanesia"}, {"id": 25378, "dom": "G", "ent": "Debaltseve", "cls": "Micronesia"}, {"id": 25379, "dom": "G", "ent": "Debaltseve", "cls": "Middle Africa"}, {"id": 25380, "dom": "G", "ent": "Debaltseve", "cls": "Northern Africa"}, {"id": 25381, "dom": "G", "ent": "Debaltseve", "cls": "Northern America"}, {"id": 25382, "dom": "G", "ent": "Debaltseve", "cls": "Northern Europe"}, {"id": 25383, "dom": "G", "ent": "Debaltseve", "cls": "Polynesia"}, {"id": 25384, "dom": "G", "ent": "Debaltseve", "cls": "South America"}, {"id": 25385, "dom": "G", "ent": "Debaltseve", "cls": "South-eastern Asia"}, {"id": 25386, "dom": "G", "ent": "Debaltseve", "cls": "Southern Africa"}, {"id": 25387, "dom": "G", "ent": "Debaltseve", "cls": "Southern Asia"}, {"id": 25388, "dom": "G", "ent": "Debaltseve", "cls": "Southern Europe"}, {"id": 25389, "dom": "G", "ent": "Debaltseve", "cls": "Western Africa"}, {"id": 25390, "dom": "G", "ent": "Debaltseve", "cls": "Western Asia"}, {"id": 25391, "dom": "G", "ent": "Debaltseve", "cls": "Western Europe"}, {"id": 25392, "dom": "G", "ent": "Korostyshiv", "cls": "Australia and New Zealand"}, {"id": 25393, "dom": "G", "ent": "Korostyshiv", "cls": "Caribbean"}, {"id": 25394, "dom": "G", "ent": "Korostyshiv", "cls": "Central America"}, {"id": 25395, "dom": "G", "ent": "Korostyshiv", "cls": "Central Asia"}, {"id": 25396, "dom": "G", "ent": "Korostyshiv", "cls": "Eastern Africa"}, {"id": 25397, "dom": "G", "ent": "Korostyshiv", "cls": "Eastern Asia"}, {"id": 25398, "dom": "G", "ent": "Korostyshiv", "cls": "Eastern Europe"}, {"id": 25399, "dom": "G", "ent": "Korostyshiv", "cls": "Melanesia"}, {"id": 25400, "dom": "G", "ent": "Korostyshiv", "cls": "Micronesia"}, {"id": 25401, "dom": "G", "ent": "Korostyshiv", "cls": "Middle Africa"}, {"id": 25402, "dom": "G", "ent": "Korostyshiv", "cls": "Northern Africa"}, {"id": 25403, "dom": "G", "ent": "Korostyshiv", "cls": "Northern America"}, {"id": 25404, "dom": "G", "ent": "Korostyshiv", "cls": "Northern Europe"}, {"id": 25405, "dom": "G", "ent": "Korostyshiv", "cls": "Polynesia"}, {"id": 25406, "dom": "G", "ent": "Korostyshiv", "cls": "South America"}, {"id": 25407, "dom": "G", "ent": "Korostyshiv", "cls": "South-eastern Asia"}, {"id": 25408, "dom": "G", "ent": "Korostyshiv", "cls": "Southern Africa"}, {"id": 25409, "dom": "G", "ent": "Korostyshiv", "cls": "Southern Asia"}, {"id": 25410, "dom": "G", "ent": "Korostyshiv", "cls": "Southern Europe"}, {"id": 25411, "dom": "G", "ent": "Korostyshiv", "cls": "Western Africa"}, {"id": 25412, "dom": "G", "ent": "Korostyshiv", "cls": "Western Asia"}, {"id": 25413, "dom": "G", "ent": "Korostyshiv", "cls": "Western Europe"}, {"id": 25414, "dom": "G", "ent": "Bakhmach", "cls": "Australia and New Zealand"}, {"id": 25415, "dom": "G", "ent": "Bakhmach", "cls": "Caribbean"}, {"id": 25416, "dom": "G", "ent": "Bakhmach", "cls": "Central America"}, {"id": 25417, "dom": "G", "ent": "Bakhmach", "cls": "Central Asia"}, {"id": 25418, "dom": "G", "ent": "Bakhmach", "cls": "Eastern Africa"}, {"id": 25419, "dom": "G", "ent": "Bakhmach", "cls": "Eastern Asia"}, {"id": 25420, "dom": "G", "ent": "Bakhmach", "cls": "Eastern Europe"}, {"id": 25421, "dom": "G", "ent": "Bakhmach", "cls": "Melanesia"}, {"id": 25422, "dom": "G", "ent": "Bakhmach", "cls": "Micronesia"}, {"id": 25423, "dom": "G", "ent": "Bakhmach", "cls": "Middle Africa"}, {"id": 25424, "dom": "G", "ent": "Bakhmach", "cls": "Northern Africa"}, {"id": 25425, "dom": "G", "ent": "Bakhmach", "cls": "Northern America"}, {"id": 25426, "dom": "G", "ent": "Bakhmach", "cls": "Northern Europe"}, {"id": 25427, "dom": "G", "ent": "Bakhmach", "cls": "Polynesia"}, {"id": 25428, "dom": "G", "ent": "Bakhmach", "cls": "South America"}, {"id": 25429, "dom": "G", "ent": "Bakhmach", "cls": "South-eastern Asia"}, {"id": 25430, "dom": "G", "ent": "Bakhmach", "cls": "Southern Africa"}, {"id": 25431, "dom": "G", "ent": "Bakhmach", "cls": "Southern Asia"}, {"id": 25432, "dom": "G", "ent": "Bakhmach", "cls": "Southern Europe"}, {"id": 25433, "dom": "G", "ent": "Bakhmach", "cls": "Western Africa"}, {"id": 25434, "dom": "G", "ent": "Bakhmach", "cls": "Western Asia"}, {"id": 25435, "dom": "G", "ent": "Bakhmach", "cls": "Western Europe"}, {"id": 25436, "dom": "G", "ent": "Butaleja", "cls": "Australia and New Zealand"}, {"id": 25437, "dom": "G", "ent": "Butaleja", "cls": "Caribbean"}, {"id": 25438, "dom": "G", "ent": "Butaleja", "cls": "Central America"}, {"id": 25439, "dom": "G", "ent": "Butaleja", "cls": "Central Asia"}, {"id": 25440, "dom": "G", "ent": "Butaleja", "cls": "Eastern Africa"}, {"id": 25441, "dom": "G", "ent": "Butaleja", "cls": "Eastern Asia"}, {"id": 25442, "dom": "G", "ent": "Butaleja", "cls": "Eastern Europe"}, {"id": 25443, "dom": "G", "ent": "Butaleja", "cls": "Melanesia"}, {"id": 25444, "dom": "G", "ent": "Butaleja", "cls": "Micronesia"}, {"id": 25445, "dom": "G", "ent": "Butaleja", "cls": "Middle Africa"}, {"id": 25446, "dom": "G", "ent": "Butaleja", "cls": "Northern Africa"}, {"id": 25447, "dom": "G", "ent": "Butaleja", "cls": "Northern America"}, {"id": 25448, "dom": "G", "ent": "Butaleja", "cls": "Northern Europe"}, {"id": 25449, "dom": "G", "ent": "Butaleja", "cls": "Polynesia"}, {"id": 25450, "dom": "G", "ent": "Butaleja", "cls": "South America"}, {"id": 25451, "dom": "G", "ent": "Butaleja", "cls": "South-eastern Asia"}, {"id": 25452, "dom": "G", "ent": "Butaleja", "cls": "Southern Africa"}, {"id": 25453, "dom": "G", "ent": "Butaleja", "cls": "Southern Asia"}, {"id": 25454, "dom": "G", "ent": "Butaleja", "cls": "Southern Europe"}, {"id": 25455, "dom": "G", "ent": "Butaleja", "cls": "Western Africa"}, {"id": 25456, "dom": "G", "ent": "Butaleja", "cls": "Western Asia"}, {"id": 25457, "dom": "G", "ent": "Butaleja", "cls": "Western Europe"}, {"id": 25458, "dom": "G", "ent": "Anaka", "cls": "Australia and New Zealand"}, {"id": 25459, "dom": "G", "ent": "Anaka", "cls": "Caribbean"}, {"id": 25460, "dom": "G", "ent": "Anaka", "cls": "Central America"}, {"id": 25461, "dom": "G", "ent": "Anaka", "cls": "Central Asia"}, {"id": 25462, "dom": "G", "ent": "Anaka", "cls": "Eastern Africa"}, {"id": 25463, "dom": "G", "ent": "Anaka", "cls": "Eastern Asia"}, {"id": 25464, "dom": "G", "ent": "Anaka", "cls": "Eastern Europe"}, {"id": 25465, "dom": "G", "ent": "Anaka", "cls": "Melanesia"}, {"id": 25466, "dom": "G", "ent": "Anaka", "cls": "Micronesia"}, {"id": 25467, "dom": "G", "ent": "Anaka", "cls": "Middle Africa"}, {"id": 25468, "dom": "G", "ent": "Anaka", "cls": "Northern Africa"}, {"id": 25469, "dom": "G", "ent": "Anaka", "cls": "Northern America"}, {"id": 25470, "dom": "G", "ent": "Anaka", "cls": "Northern Europe"}, {"id": 25471, "dom": "G", "ent": "Anaka", "cls": "Polynesia"}, {"id": 25472, "dom": "G", "ent": "Anaka", "cls": "South America"}, {"id": 25473, "dom": "G", "ent": "Anaka", "cls": "South-eastern Asia"}, {"id": 25474, "dom": "G", "ent": "Anaka", "cls": "Southern Africa"}, {"id": 25475, "dom": "G", "ent": "Anaka", "cls": "Southern Asia"}, {"id": 25476, "dom": "G", "ent": "Anaka", "cls": "Southern Europe"}, {"id": 25477, "dom": "G", "ent": "Anaka", "cls": "Western Africa"}, {"id": 25478, "dom": "G", "ent": "Anaka", "cls": "Western Asia"}, {"id": 25479, "dom": "G", "ent": "Anaka", "cls": "Western Europe"}, {"id": 25480, "dom": "G", "ent": "Lwakyomba", "cls": "Australia and New Zealand"}, {"id": 25481, "dom": "G", "ent": "Lwakyomba", "cls": "Caribbean"}, {"id": 25482, "dom": "G", "ent": "Lwakyomba", "cls": "Central America"}, {"id": 25483, "dom": "G", "ent": "Lwakyomba", "cls": "Central Asia"}, {"id": 25484, "dom": "G", "ent": "Lwakyomba", "cls": "Eastern Africa"}, {"id": 25485, "dom": "G", "ent": "Lwakyomba", "cls": "Eastern Asia"}, {"id": 25486, "dom": "G", "ent": "Lwakyomba", "cls": "Eastern Europe"}, {"id": 25487, "dom": "G", "ent": "Lwakyomba", "cls": "Melanesia"}, {"id": 25488, "dom": "G", "ent": "Lwakyomba", "cls": "Micronesia"}, {"id": 25489, "dom": "G", "ent": "Lwakyomba", "cls": "Middle Africa"}, {"id": 25490, "dom": "G", "ent": "Lwakyomba", "cls": "Northern Africa"}, {"id": 25491, "dom": "G", "ent": "Lwakyomba", "cls": "Northern America"}, {"id": 25492, "dom": "G", "ent": "Lwakyomba", "cls": "Northern Europe"}, {"id": 25493, "dom": "G", "ent": "Lwakyomba", "cls": "Polynesia"}, {"id": 25494, "dom": "G", "ent": "Lwakyomba", "cls": "South America"}, {"id": 25495, "dom": "G", "ent": "Lwakyomba", "cls": "South-eastern Asia"}, {"id": 25496, "dom": "G", "ent": "Lwakyomba", "cls": "Southern Africa"}, {"id": 25497, "dom": "G", "ent": "Lwakyomba", "cls": "Southern Asia"}, {"id": 25498, "dom": "G", "ent": "Lwakyomba", "cls": "Southern Europe"}, {"id": 25499, "dom": "G", "ent": "Lwakyomba", "cls": "Western Africa"}, {"id": 25500, "dom": "G", "ent": "Lwakyomba", "cls": "Western Asia"}, {"id": 25501, "dom": "G", "ent": "Lwakyomba", "cls": "Western Europe"}, {"id": 25502, "dom": "G", "ent": "Iryango", "cls": "Australia and New Zealand"}, {"id": 25503, "dom": "G", "ent": "Iryango", "cls": "Caribbean"}, {"id": 25504, "dom": "G", "ent": "Iryango", "cls": "Central America"}, {"id": 25505, "dom": "G", "ent": "Iryango", "cls": "Central Asia"}, {"id": 25506, "dom": "G", "ent": "Iryango", "cls": "Eastern Africa"}, {"id": 25507, "dom": "G", "ent": "Iryango", "cls": "Eastern Asia"}, {"id": 25508, "dom": "G", "ent": "Iryango", "cls": "Eastern Europe"}, {"id": 25509, "dom": "G", "ent": "Iryango", "cls": "Melanesia"}, {"id": 25510, "dom": "G", "ent": "Iryango", "cls": "Micronesia"}, {"id": 25511, "dom": "G", "ent": "Iryango", "cls": "Middle Africa"}, {"id": 25512, "dom": "G", "ent": "Iryango", "cls": "Northern Africa"}, {"id": 25513, "dom": "G", "ent": "Iryango", "cls": "Northern America"}, {"id": 25514, "dom": "G", "ent": "Iryango", "cls": "Northern Europe"}, {"id": 25515, "dom": "G", "ent": "Iryango", "cls": "Polynesia"}, {"id": 25516, "dom": "G", "ent": "Iryango", "cls": "South America"}, {"id": 25517, "dom": "G", "ent": "Iryango", "cls": "South-eastern Asia"}, {"id": 25518, "dom": "G", "ent": "Iryango", "cls": "Southern Africa"}, {"id": 25519, "dom": "G", "ent": "Iryango", "cls": "Southern Asia"}, {"id": 25520, "dom": "G", "ent": "Iryango", "cls": "Southern Europe"}, {"id": 25521, "dom": "G", "ent": "Iryango", "cls": "Western Africa"}, {"id": 25522, "dom": "G", "ent": "Iryango", "cls": "Western Asia"}, {"id": 25523, "dom": "G", "ent": "Iryango", "cls": "Western Europe"}, {"id": 25524, "dom": "G", "ent": "The Villages", "cls": "Australia and New Zealand"}, {"id": 25525, "dom": "G", "ent": "The Villages", "cls": "Caribbean"}, {"id": 25526, "dom": "G", "ent": "The Villages", "cls": "Central America"}, {"id": 25527, "dom": "G", "ent": "The Villages", "cls": "Central Asia"}, {"id": 25528, "dom": "G", "ent": "The Villages", "cls": "Eastern Africa"}, {"id": 25529, "dom": "G", "ent": "The Villages", "cls": "Eastern Asia"}, {"id": 25530, "dom": "G", "ent": "The Villages", "cls": "Eastern Europe"}, {"id": 25531, "dom": "G", "ent": "The Villages", "cls": "Melanesia"}, {"id": 25532, "dom": "G", "ent": "The Villages", "cls": "Micronesia"}, {"id": 25533, "dom": "G", "ent": "The Villages", "cls": "Middle Africa"}, {"id": 25534, "dom": "G", "ent": "The Villages", "cls": "Northern Africa"}, {"id": 25535, "dom": "G", "ent": "The Villages", "cls": "Northern America"}, {"id": 25536, "dom": "G", "ent": "The Villages", "cls": "Northern Europe"}, {"id": 25537, "dom": "G", "ent": "The Villages", "cls": "Polynesia"}, {"id": 25538, "dom": "G", "ent": "The Villages", "cls": "South America"}, {"id": 25539, "dom": "G", "ent": "The Villages", "cls": "South-eastern Asia"}, {"id": 25540, "dom": "G", "ent": "The Villages", "cls": "Southern Africa"}, {"id": 25541, "dom": "G", "ent": "The Villages", "cls": "Southern Asia"}, {"id": 25542, "dom": "G", "ent": "The Villages", "cls": "Southern Europe"}, {"id": 25543, "dom": "G", "ent": "The Villages", "cls": "Western Africa"}, {"id": 25544, "dom": "G", "ent": "The Villages", "cls": "Western Asia"}, {"id": 25545, "dom": "G", "ent": "The Villages", "cls": "Western Europe"}, {"id": 25546, "dom": "G", "ent": "Southglenn", "cls": "Australia and New Zealand"}, {"id": 25547, "dom": "G", "ent": "Southglenn", "cls": "Caribbean"}, {"id": 25548, "dom": "G", "ent": "Southglenn", "cls": "Central America"}, {"id": 25549, "dom": "G", "ent": "Southglenn", "cls": "Central Asia"}, {"id": 25550, "dom": "G", "ent": "Southglenn", "cls": "Eastern Africa"}, {"id": 25551, "dom": "G", "ent": "Southglenn", "cls": "Eastern Asia"}, {"id": 25552, "dom": "G", "ent": "Southglenn", "cls": "Eastern Europe"}, {"id": 25553, "dom": "G", "ent": "Southglenn", "cls": "Melanesia"}, {"id": 25554, "dom": "G", "ent": "Southglenn", "cls": "Micronesia"}, {"id": 25555, "dom": "G", "ent": "Southglenn", "cls": "Middle Africa"}, {"id": 25556, "dom": "G", "ent": "Southglenn", "cls": "Northern Africa"}, {"id": 25557, "dom": "G", "ent": "Southglenn", "cls": "Northern America"}, {"id": 25558, "dom": "G", "ent": "Southglenn", "cls": "Northern Europe"}, {"id": 25559, "dom": "G", "ent": "Southglenn", "cls": "Polynesia"}, {"id": 25560, "dom": "G", "ent": "Southglenn", "cls": "South America"}, {"id": 25561, "dom": "G", "ent": "Southglenn", "cls": "South-eastern Asia"}, {"id": 25562, "dom": "G", "ent": "Southglenn", "cls": "Southern Africa"}, {"id": 25563, "dom": "G", "ent": "Southglenn", "cls": "Southern Asia"}, {"id": 25564, "dom": "G", "ent": "Southglenn", "cls": "Southern Europe"}, {"id": 25565, "dom": "G", "ent": "Southglenn", "cls": "Western Africa"}, {"id": 25566, "dom": "G", "ent": "Southglenn", "cls": "Western Asia"}, {"id": 25567, "dom": "G", "ent": "Southglenn", "cls": "Western Europe"}, {"id": 25568, "dom": "G", "ent": "Wheat Ridge", "cls": "Australia and New Zealand"}, {"id": 25569, "dom": "G", "ent": "Wheat Ridge", "cls": "Caribbean"}, {"id": 25570, "dom": "G", "ent": "Wheat Ridge", "cls": "Central America"}, {"id": 25571, "dom": "G", "ent": "Wheat Ridge", "cls": "Central Asia"}, {"id": 25572, "dom": "G", "ent": "Wheat Ridge", "cls": "Eastern Africa"}, {"id": 25573, "dom": "G", "ent": "Wheat Ridge", "cls": "Eastern Asia"}, {"id": 25574, "dom": "G", "ent": "Wheat Ridge", "cls": "Eastern Europe"}, {"id": 25575, "dom": "G", "ent": "Wheat Ridge", "cls": "Melanesia"}, {"id": 25576, "dom": "G", "ent": "Wheat Ridge", "cls": "Micronesia"}, {"id": 25577, "dom": "G", "ent": "Wheat Ridge", "cls": "Middle Africa"}, {"id": 25578, "dom": "G", "ent": "Wheat Ridge", "cls": "Northern Africa"}, {"id": 25579, "dom": "G", "ent": "Wheat Ridge", "cls": "Northern America"}, {"id": 25580, "dom": "G", "ent": "Wheat Ridge", "cls": "Northern Europe"}, {"id": 25581, "dom": "G", "ent": "Wheat Ridge", "cls": "Polynesia"}, {"id": 25582, "dom": "G", "ent": "Wheat Ridge", "cls": "South America"}, {"id": 25583, "dom": "G", "ent": "Wheat Ridge", "cls": "South-eastern Asia"}, {"id": 25584, "dom": "G", "ent": "Wheat Ridge", "cls": "Southern Africa"}, {"id": 25585, "dom": "G", "ent": "Wheat Ridge", "cls": "Southern Asia"}, {"id": 25586, "dom": "G", "ent": "Wheat Ridge", "cls": "Southern Europe"}, {"id": 25587, "dom": "G", "ent": "Wheat Ridge", "cls": "Western Africa"}, {"id": 25588, "dom": "G", "ent": "Wheat Ridge", "cls": "Western Asia"}, {"id": 25589, "dom": "G", "ent": "Wheat Ridge", "cls": "Western Europe"}, {"id": 25590, "dom": "G", "ent": "Carteret", "cls": "Australia and New Zealand"}, {"id": 25591, "dom": "G", "ent": "Carteret", "cls": "Caribbean"}, {"id": 25592, "dom": "G", "ent": "Carteret", "cls": "Central America"}, {"id": 25593, "dom": "G", "ent": "Carteret", "cls": "Central Asia"}, {"id": 25594, "dom": "G", "ent": "Carteret", "cls": "Eastern Africa"}, {"id": 25595, "dom": "G", "ent": "Carteret", "cls": "Eastern Asia"}, {"id": 25596, "dom": "G", "ent": "Carteret", "cls": "Eastern Europe"}, {"id": 25597, "dom": "G", "ent": "Carteret", "cls": "Melanesia"}, {"id": 25598, "dom": "G", "ent": "Carteret", "cls": "Micronesia"}, {"id": 25599, "dom": "G", "ent": "Carteret", "cls": "Middle Africa"}, {"id": 25600, "dom": "G", "ent": "Carteret", "cls": "Northern Africa"}, {"id": 25601, "dom": "G", "ent": "Carteret", "cls": "Northern America"}, {"id": 25602, "dom": "G", "ent": "Carteret", "cls": "Northern Europe"}, {"id": 25603, "dom": "G", "ent": "Carteret", "cls": "Polynesia"}, {"id": 25604, "dom": "G", "ent": "Carteret", "cls": "South America"}, {"id": 25605, "dom": "G", "ent": "Carteret", "cls": "South-eastern Asia"}, {"id": 25606, "dom": "G", "ent": "Carteret", "cls": "Southern Africa"}, {"id": 25607, "dom": "G", "ent": "Carteret", "cls": "Southern Asia"}, {"id": 25608, "dom": "G", "ent": "Carteret", "cls": "Southern Europe"}, {"id": 25609, "dom": "G", "ent": "Carteret", "cls": "Western Africa"}, {"id": 25610, "dom": "G", "ent": "Carteret", "cls": "Western Asia"}, {"id": 25611, "dom": "G", "ent": "Carteret", "cls": "Western Europe"}, {"id": 25612, "dom": "G", "ent": "Fray Bentos", "cls": "Australia and New Zealand"}, {"id": 25613, "dom": "G", "ent": "Fray Bentos", "cls": "Caribbean"}, {"id": 25614, "dom": "G", "ent": "Fray Bentos", "cls": "Central America"}, {"id": 25615, "dom": "G", "ent": "Fray Bentos", "cls": "Central Asia"}, {"id": 25616, "dom": "G", "ent": "Fray Bentos", "cls": "Eastern Africa"}, {"id": 25617, "dom": "G", "ent": "Fray Bentos", "cls": "Eastern Asia"}, {"id": 25618, "dom": "G", "ent": "Fray Bentos", "cls": "Eastern Europe"}, {"id": 25619, "dom": "G", "ent": "Fray Bentos", "cls": "Melanesia"}, {"id": 25620, "dom": "G", "ent": "Fray Bentos", "cls": "Micronesia"}, {"id": 25621, "dom": "G", "ent": "Fray Bentos", "cls": "Middle Africa"}, {"id": 25622, "dom": "G", "ent": "Fray Bentos", "cls": "Northern Africa"}, {"id": 25623, "dom": "G", "ent": "Fray Bentos", "cls": "Northern America"}, {"id": 25624, "dom": "G", "ent": "Fray Bentos", "cls": "Northern Europe"}, {"id": 25625, "dom": "G", "ent": "Fray Bentos", "cls": "Polynesia"}, {"id": 25626, "dom": "G", "ent": "Fray Bentos", "cls": "South America"}, {"id": 25627, "dom": "G", "ent": "Fray Bentos", "cls": "South-eastern Asia"}, {"id": 25628, "dom": "G", "ent": "Fray Bentos", "cls": "Southern Africa"}, {"id": 25629, "dom": "G", "ent": "Fray Bentos", "cls": "Southern Asia"}, {"id": 25630, "dom": "G", "ent": "Fray Bentos", "cls": "Southern Europe"}, {"id": 25631, "dom": "G", "ent": "Fray Bentos", "cls": "Western Africa"}, {"id": 25632, "dom": "G", "ent": "Fray Bentos", "cls": "Western Asia"}, {"id": 25633, "dom": "G", "ent": "Fray Bentos", "cls": "Western Europe"}, {"id": 25634, "dom": "G", "ent": "Treinta y Tres", "cls": "Australia and New Zealand"}, {"id": 25635, "dom": "G", "ent": "Treinta y Tres", "cls": "Caribbean"}, {"id": 25636, "dom": "G", "ent": "Treinta y Tres", "cls": "Central America"}, {"id": 25637, "dom": "G", "ent": "Treinta y Tres", "cls": "Central Asia"}, {"id": 25638, "dom": "G", "ent": "Treinta y Tres", "cls": "Eastern Africa"}, {"id": 25639, "dom": "G", "ent": "Treinta y Tres", "cls": "Eastern Asia"}, {"id": 25640, "dom": "G", "ent": "Treinta y Tres", "cls": "Eastern Europe"}, {"id": 25641, "dom": "G", "ent": "Treinta y Tres", "cls": "Melanesia"}, {"id": 25642, "dom": "G", "ent": "Treinta y Tres", "cls": "Micronesia"}, {"id": 25643, "dom": "G", "ent": "Treinta y Tres", "cls": "Middle Africa"}, {"id": 25644, "dom": "G", "ent": "Treinta y Tres", "cls": "Northern Africa"}, {"id": 25645, "dom": "G", "ent": "Treinta y Tres", "cls": "Northern America"}, {"id": 25646, "dom": "G", "ent": "Treinta y Tres", "cls": "Northern Europe"}, {"id": 25647, "dom": "G", "ent": "Treinta y Tres", "cls": "Polynesia"}, {"id": 25648, "dom": "G", "ent": "Treinta y Tres", "cls": "South America"}, {"id": 25649, "dom": "G", "ent": "Treinta y Tres", "cls": "South-eastern Asia"}, {"id": 25650, "dom": "G", "ent": "Treinta y Tres", "cls": "Southern Africa"}, {"id": 25651, "dom": "G", "ent": "Treinta y Tres", "cls": "Southern Asia"}, {"id": 25652, "dom": "G", "ent": "Treinta y Tres", "cls": "Southern Europe"}, {"id": 25653, "dom": "G", "ent": "Treinta y Tres", "cls": "Western Africa"}, {"id": 25654, "dom": "G", "ent": "Treinta y Tres", "cls": "Western Asia"}, {"id": 25655, "dom": "G", "ent": "Treinta y Tres", "cls": "Western Europe"}, {"id": 25656, "dom": "G", "ent": "Carmelo", "cls": "Australia and New Zealand"}, {"id": 25657, "dom": "G", "ent": "Carmelo", "cls": "Caribbean"}, {"id": 25658, "dom": "G", "ent": "Carmelo", "cls": "Central America"}, {"id": 25659, "dom": "G", "ent": "Carmelo", "cls": "Central Asia"}, {"id": 25660, "dom": "G", "ent": "Carmelo", "cls": "Eastern Africa"}, {"id": 25661, "dom": "G", "ent": "Carmelo", "cls": "Eastern Asia"}, {"id": 25662, "dom": "G", "ent": "Carmelo", "cls": "Eastern Europe"}, {"id": 25663, "dom": "G", "ent": "Carmelo", "cls": "Melanesia"}, {"id": 25664, "dom": "G", "ent": "Carmelo", "cls": "Micronesia"}, {"id": 25665, "dom": "G", "ent": "Carmelo", "cls": "Middle Africa"}, {"id": 25666, "dom": "G", "ent": "Carmelo", "cls": "Northern Africa"}, {"id": 25667, "dom": "G", "ent": "Carmelo", "cls": "Northern America"}, {"id": 25668, "dom": "G", "ent": "Carmelo", "cls": "Northern Europe"}, {"id": 25669, "dom": "G", "ent": "Carmelo", "cls": "Polynesia"}, {"id": 25670, "dom": "G", "ent": "Carmelo", "cls": "South America"}, {"id": 25671, "dom": "G", "ent": "Carmelo", "cls": "South-eastern Asia"}, {"id": 25672, "dom": "G", "ent": "Carmelo", "cls": "Southern Africa"}, {"id": 25673, "dom": "G", "ent": "Carmelo", "cls": "Southern Asia"}, {"id": 25674, "dom": "G", "ent": "Carmelo", "cls": "Southern Europe"}, {"id": 25675, "dom": "G", "ent": "Carmelo", "cls": "Western Africa"}, {"id": 25676, "dom": "G", "ent": "Carmelo", "cls": "Western Asia"}, {"id": 25677, "dom": "G", "ent": "Carmelo", "cls": "Western Europe"}, {"id": 25678, "dom": "G", "ent": "Young", "cls": "Australia and New Zealand"}, {"id": 25679, "dom": "G", "ent": "Young", "cls": "Caribbean"}, {"id": 25680, "dom": "G", "ent": "Young", "cls": "Central America"}, {"id": 25681, "dom": "G", "ent": "Young", "cls": "Central Asia"}, {"id": 25682, "dom": "G", "ent": "Young", "cls": "Eastern Africa"}, {"id": 25683, "dom": "G", "ent": "Young", "cls": "Eastern Asia"}, {"id": 25684, "dom": "G", "ent": "Young", "cls": "Eastern Europe"}, {"id": 25685, "dom": "G", "ent": "Young", "cls": "Melanesia"}, {"id": 25686, "dom": "G", "ent": "Young", "cls": "Micronesia"}, {"id": 25687, "dom": "G", "ent": "Young", "cls": "Middle Africa"}, {"id": 25688, "dom": "G", "ent": "Young", "cls": "Northern Africa"}, {"id": 25689, "dom": "G", "ent": "Young", "cls": "Northern America"}, {"id": 25690, "dom": "G", "ent": "Young", "cls": "Northern Europe"}, {"id": 25691, "dom": "G", "ent": "Young", "cls": "Polynesia"}, {"id": 25692, "dom": "G", "ent": "Young", "cls": "South America"}, {"id": 25693, "dom": "G", "ent": "Young", "cls": "South-eastern Asia"}, {"id": 25694, "dom": "G", "ent": "Young", "cls": "Southern Africa"}, {"id": 25695, "dom": "G", "ent": "Young", "cls": "Southern Asia"}, {"id": 25696, "dom": "G", "ent": "Young", "cls": "Southern Europe"}, {"id": 25697, "dom": "G", "ent": "Young", "cls": "Western Africa"}, {"id": 25698, "dom": "G", "ent": "Young", "cls": "Western Asia"}, {"id": 25699, "dom": "G", "ent": "Young", "cls": "Western Europe"}, {"id": 25700, "dom": "G", "ent": "Kogon Shahri", "cls": "Australia and New Zealand"}, {"id": 25701, "dom": "G", "ent": "Kogon Shahri", "cls": "Caribbean"}, {"id": 25702, "dom": "G", "ent": "Kogon Shahri", "cls": "Central America"}, {"id": 25703, "dom": "G", "ent": "Kogon Shahri", "cls": "Central Asia"}, {"id": 25704, "dom": "G", "ent": "Kogon Shahri", "cls": "Eastern Africa"}, {"id": 25705, "dom": "G", "ent": "Kogon Shahri", "cls": "Eastern Asia"}, {"id": 25706, "dom": "G", "ent": "Kogon Shahri", "cls": "Eastern Europe"}, {"id": 25707, "dom": "G", "ent": "Kogon Shahri", "cls": "Melanesia"}, {"id": 25708, "dom": "G", "ent": "Kogon Shahri", "cls": "Micronesia"}, {"id": 25709, "dom": "G", "ent": "Kogon Shahri", "cls": "Middle Africa"}, {"id": 25710, "dom": "G", "ent": "Kogon Shahri", "cls": "Northern Africa"}, {"id": 25711, "dom": "G", "ent": "Kogon Shahri", "cls": "Northern America"}, {"id": 25712, "dom": "G", "ent": "Kogon Shahri", "cls": "Northern Europe"}, {"id": 25713, "dom": "G", "ent": "Kogon Shahri", "cls": "Polynesia"}, {"id": 25714, "dom": "G", "ent": "Kogon Shahri", "cls": "South America"}, {"id": 25715, "dom": "G", "ent": "Kogon Shahri", "cls": "South-eastern Asia"}, {"id": 25716, "dom": "G", "ent": "Kogon Shahri", "cls": "Southern Africa"}, {"id": 25717, "dom": "G", "ent": "Kogon Shahri", "cls": "Southern Asia"}, {"id": 25718, "dom": "G", "ent": "Kogon Shahri", "cls": "Southern Europe"}, {"id": 25719, "dom": "G", "ent": "Kogon Shahri", "cls": "Western Africa"}, {"id": 25720, "dom": "G", "ent": "Kogon Shahri", "cls": "Western Asia"}, {"id": 25721, "dom": "G", "ent": "Kogon Shahri", "cls": "Western Europe"}, {"id": 25722, "dom": "G", "ent": "Kitob", "cls": "Australia and New Zealand"}, {"id": 25723, "dom": "G", "ent": "Kitob", "cls": "Caribbean"}, {"id": 25724, "dom": "G", "ent": "Kitob", "cls": "Central America"}, {"id": 25725, "dom": "G", "ent": "Kitob", "cls": "Central Asia"}, {"id": 25726, "dom": "G", "ent": "Kitob", "cls": "Eastern Africa"}, {"id": 25727, "dom": "G", "ent": "Kitob", "cls": "Eastern Asia"}, {"id": 25728, "dom": "G", "ent": "Kitob", "cls": "Eastern Europe"}, {"id": 25729, "dom": "G", "ent": "Kitob", "cls": "Melanesia"}, {"id": 25730, "dom": "G", "ent": "Kitob", "cls": "Micronesia"}, {"id": 25731, "dom": "G", "ent": "Kitob", "cls": "Middle Africa"}, {"id": 25732, "dom": "G", "ent": "Kitob", "cls": "Northern Africa"}, {"id": 25733, "dom": "G", "ent": "Kitob", "cls": "Northern America"}, {"id": 25734, "dom": "G", "ent": "Kitob", "cls": "Northern Europe"}, {"id": 25735, "dom": "G", "ent": "Kitob", "cls": "Polynesia"}, {"id": 25736, "dom": "G", "ent": "Kitob", "cls": "South America"}, {"id": 25737, "dom": "G", "ent": "Kitob", "cls": "South-eastern Asia"}, {"id": 25738, "dom": "G", "ent": "Kitob", "cls": "Southern Africa"}, {"id": 25739, "dom": "G", "ent": "Kitob", "cls": "Southern Asia"}, {"id": 25740, "dom": "G", "ent": "Kitob", "cls": "Southern Europe"}, {"id": 25741, "dom": "G", "ent": "Kitob", "cls": "Western Africa"}, {"id": 25742, "dom": "G", "ent": "Kitob", "cls": "Western Asia"}, {"id": 25743, "dom": "G", "ent": "Kitob", "cls": "Western Europe"}, {"id": 25744, "dom": "G", "ent": "Shofirkon Shahri", "cls": "Australia and New Zealand"}, {"id": 25745, "dom": "G", "ent": "Shofirkon Shahri", "cls": "Caribbean"}, {"id": 25746, "dom": "G", "ent": "Shofirkon Shahri", "cls": "Central America"}, {"id": 25747, "dom": "G", "ent": "Shofirkon Shahri", "cls": "Central Asia"}, {"id": 25748, "dom": "G", "ent": "Shofirkon Shahri", "cls": "Eastern Africa"}, {"id": 25749, "dom": "G", "ent": "Shofirkon Shahri", "cls": "Eastern Asia"}, {"id": 25750, "dom": "G", "ent": "Shofirkon Shahri", "cls": "Eastern Europe"}, {"id": 25751, "dom": "G", "ent": "Shofirkon Shahri", "cls": "Melanesia"}, {"id": 25752, "dom": "G", "ent": "Shofirkon Shahri", "cls": "Micronesia"}, {"id": 25753, "dom": "G", "ent": "Shofirkon Shahri", "cls": "Middle Africa"}, {"id": 25754, "dom": "G", "ent": "Shofirkon Shahri", "cls": "Northern Africa"}, {"id": 25755, "dom": "G", "ent": "Shofirkon Shahri", "cls": "Northern America"}, {"id": 25756, "dom": "G", "ent": "Shofirkon Shahri", "cls": "Northern Europe"}, {"id": 25757, "dom": "G", "ent": "Shofirkon Shahri", "cls": "Polynesia"}, {"id": 25758, "dom": "G", "ent": "Shofirkon Shahri", "cls": "South America"}, {"id": 25759, "dom": "G", "ent": "Shofirkon Shahri", "cls": "South-eastern Asia"}, {"id": 25760, "dom": "G", "ent": "Shofirkon Shahri", "cls": "Southern Africa"}, {"id": 25761, "dom": "G", "ent": "Shofirkon Shahri", "cls": "Southern Asia"}, {"id": 25762, "dom": "G", "ent": "Shofirkon Shahri", "cls": "Southern Europe"}, {"id": 25763, "dom": "G", "ent": "Shofirkon Shahri", "cls": "Western Africa"}, {"id": 25764, "dom": "G", "ent": "Shofirkon Shahri", "cls": "Western Asia"}, {"id": 25765, "dom": "G", "ent": "Shofirkon Shahri", "cls": "Western Europe"}, {"id": 25766, "dom": "G", "ent": "Chelak", "cls": "Australia and New Zealand"}, {"id": 25767, "dom": "G", "ent": "Chelak", "cls": "Caribbean"}, {"id": 25768, "dom": "G", "ent": "Chelak", "cls": "Central America"}, {"id": 25769, "dom": "G", "ent": "Chelak", "cls": "Central Asia"}, {"id": 25770, "dom": "G", "ent": "Chelak", "cls": "Eastern Africa"}, {"id": 25771, "dom": "G", "ent": "Chelak", "cls": "Eastern Asia"}, {"id": 25772, "dom": "G", "ent": "Chelak", "cls": "Eastern Europe"}, {"id": 25773, "dom": "G", "ent": "Chelak", "cls": "Melanesia"}, {"id": 25774, "dom": "G", "ent": "Chelak", "cls": "Micronesia"}, {"id": 25775, "dom": "G", "ent": "Chelak", "cls": "Middle Africa"}, {"id": 25776, "dom": "G", "ent": "Chelak", "cls": "Northern Africa"}, {"id": 25777, "dom": "G", "ent": "Chelak", "cls": "Northern America"}, {"id": 25778, "dom": "G", "ent": "Chelak", "cls": "Northern Europe"}, {"id": 25779, "dom": "G", "ent": "Chelak", "cls": "Polynesia"}, {"id": 25780, "dom": "G", "ent": "Chelak", "cls": "South America"}, {"id": 25781, "dom": "G", "ent": "Chelak", "cls": "South-eastern Asia"}, {"id": 25782, "dom": "G", "ent": "Chelak", "cls": "Southern Africa"}, {"id": 25783, "dom": "G", "ent": "Chelak", "cls": "Southern Asia"}, {"id": 25784, "dom": "G", "ent": "Chelak", "cls": "Southern Europe"}, {"id": 25785, "dom": "G", "ent": "Chelak", "cls": "Western Africa"}, {"id": 25786, "dom": "G", "ent": "Chelak", "cls": "Western Asia"}, {"id": 25787, "dom": "G", "ent": "Chelak", "cls": "Western Europe"}, {"id": 25788, "dom": "G", "ent": "Ciudad Guayana", "cls": "Australia and New Zealand"}, {"id": 25789, "dom": "G", "ent": "Ciudad Guayana", "cls": "Caribbean"}, {"id": 25790, "dom": "G", "ent": "Ciudad Guayana", "cls": "Central America"}, {"id": 25791, "dom": "G", "ent": "Ciudad Guayana", "cls": "Central Asia"}, {"id": 25792, "dom": "G", "ent": "Ciudad Guayana", "cls": "Eastern Africa"}, {"id": 25793, "dom": "G", "ent": "Ciudad Guayana", "cls": "Eastern Asia"}, {"id": 25794, "dom": "G", "ent": "Ciudad Guayana", "cls": "Eastern Europe"}, {"id": 25795, "dom": "G", "ent": "Ciudad Guayana", "cls": "Melanesia"}, {"id": 25796, "dom": "G", "ent": "Ciudad Guayana", "cls": "Micronesia"}, {"id": 25797, "dom": "G", "ent": "Ciudad Guayana", "cls": "Middle Africa"}, {"id": 25798, "dom": "G", "ent": "Ciudad Guayana", "cls": "Northern Africa"}, {"id": 25799, "dom": "G", "ent": "Ciudad Guayana", "cls": "Northern America"}, {"id": 25800, "dom": "G", "ent": "Ciudad Guayana", "cls": "Northern Europe"}, {"id": 25801, "dom": "G", "ent": "Ciudad Guayana", "cls": "Polynesia"}, {"id": 25802, "dom": "G", "ent": "Ciudad Guayana", "cls": "South America"}, {"id": 25803, "dom": "G", "ent": "Ciudad Guayana", "cls": "South-eastern Asia"}, {"id": 25804, "dom": "G", "ent": "Ciudad Guayana", "cls": "Southern Africa"}, {"id": 25805, "dom": "G", "ent": "Ciudad Guayana", "cls": "Southern Asia"}, {"id": 25806, "dom": "G", "ent": "Ciudad Guayana", "cls": "Southern Europe"}, {"id": 25807, "dom": "G", "ent": "Ciudad Guayana", "cls": "Western Africa"}, {"id": 25808, "dom": "G", "ent": "Ciudad Guayana", "cls": "Western Asia"}, {"id": 25809, "dom": "G", "ent": "Ciudad Guayana", "cls": "Western Europe"}, {"id": 25810, "dom": "G", "ent": "La Victoria", "cls": "Australia and New Zealand"}, {"id": 25811, "dom": "G", "ent": "La Victoria", "cls": "Caribbean"}, {"id": 25812, "dom": "G", "ent": "La Victoria", "cls": "Central America"}, {"id": 25813, "dom": "G", "ent": "La Victoria", "cls": "Central Asia"}, {"id": 25814, "dom": "G", "ent": "La Victoria", "cls": "Eastern Africa"}, {"id": 25815, "dom": "G", "ent": "La Victoria", "cls": "Eastern Asia"}, {"id": 25816, "dom": "G", "ent": "La Victoria", "cls": "Eastern Europe"}, {"id": 25817, "dom": "G", "ent": "La Victoria", "cls": "Melanesia"}, {"id": 25818, "dom": "G", "ent": "La Victoria", "cls": "Micronesia"}, {"id": 25819, "dom": "G", "ent": "La Victoria", "cls": "Middle Africa"}, {"id": 25820, "dom": "G", "ent": "La Victoria", "cls": "Northern Africa"}, {"id": 25821, "dom": "G", "ent": "La Victoria", "cls": "Northern America"}, {"id": 25822, "dom": "G", "ent": "La Victoria", "cls": "Northern Europe"}, {"id": 25823, "dom": "G", "ent": "La Victoria", "cls": "Polynesia"}, {"id": 25824, "dom": "G", "ent": "La Victoria", "cls": "South America"}, {"id": 25825, "dom": "G", "ent": "La Victoria", "cls": "South-eastern Asia"}, {"id": 25826, "dom": "G", "ent": "La Victoria", "cls": "Southern Africa"}, {"id": 25827, "dom": "G", "ent": "La Victoria", "cls": "Southern Asia"}, {"id": 25828, "dom": "G", "ent": "La Victoria", "cls": "Southern Europe"}, {"id": 25829, "dom": "G", "ent": "La Victoria", "cls": "Western Africa"}, {"id": 25830, "dom": "G", "ent": "La Victoria", "cls": "Western Asia"}, {"id": 25831, "dom": "G", "ent": "La Victoria", "cls": "Western Europe"}, {"id": 25832, "dom": "G", "ent": "Carora", "cls": "Australia and New Zealand"}, {"id": 25833, "dom": "G", "ent": "Carora", "cls": "Caribbean"}, {"id": 25834, "dom": "G", "ent": "Carora", "cls": "Central America"}, {"id": 25835, "dom": "G", "ent": "Carora", "cls": "Central Asia"}, {"id": 25836, "dom": "G", "ent": "Carora", "cls": "Eastern Africa"}, {"id": 25837, "dom": "G", "ent": "Carora", "cls": "Eastern Asia"}, {"id": 25838, "dom": "G", "ent": "Carora", "cls": "Eastern Europe"}, {"id": 25839, "dom": "G", "ent": "Carora", "cls": "Melanesia"}, {"id": 25840, "dom": "G", "ent": "Carora", "cls": "Micronesia"}, {"id": 25841, "dom": "G", "ent": "Carora", "cls": "Middle Africa"}, {"id": 25842, "dom": "G", "ent": "Carora", "cls": "Northern Africa"}, {"id": 25843, "dom": "G", "ent": "Carora", "cls": "Northern America"}, {"id": 25844, "dom": "G", "ent": "Carora", "cls": "Northern Europe"}, {"id": 25845, "dom": "G", "ent": "Carora", "cls": "Polynesia"}, {"id": 25846, "dom": "G", "ent": "Carora", "cls": "South America"}, {"id": 25847, "dom": "G", "ent": "Carora", "cls": "South-eastern Asia"}, {"id": 25848, "dom": "G", "ent": "Carora", "cls": "Southern Africa"}, {"id": 25849, "dom": "G", "ent": "Carora", "cls": "Southern Asia"}, {"id": 25850, "dom": "G", "ent": "Carora", "cls": "Southern Europe"}, {"id": 25851, "dom": "G", "ent": "Carora", "cls": "Western Africa"}, {"id": 25852, "dom": "G", "ent": "Carora", "cls": "Western Asia"}, {"id": 25853, "dom": "G", "ent": "Carora", "cls": "Western Europe"}, {"id": 25854, "dom": "G", "ent": "La Azulita", "cls": "Australia and New Zealand"}, {"id": 25855, "dom": "G", "ent": "La Azulita", "cls": "Caribbean"}, {"id": 25856, "dom": "G", "ent": "La Azulita", "cls": "Central America"}, {"id": 25857, "dom": "G", "ent": "La Azulita", "cls": "Central Asia"}, {"id": 25858, "dom": "G", "ent": "La Azulita", "cls": "Eastern Africa"}, {"id": 25859, "dom": "G", "ent": "La Azulita", "cls": "Eastern Asia"}, {"id": 25860, "dom": "G", "ent": "La Azulita", "cls": "Eastern Europe"}, {"id": 25861, "dom": "G", "ent": "La Azulita", "cls": "Melanesia"}, {"id": 25862, "dom": "G", "ent": "La Azulita", "cls": "Micronesia"}, {"id": 25863, "dom": "G", "ent": "La Azulita", "cls": "Middle Africa"}, {"id": 25864, "dom": "G", "ent": "La Azulita", "cls": "Northern Africa"}, {"id": 25865, "dom": "G", "ent": "La Azulita", "cls": "Northern America"}, {"id": 25866, "dom": "G", "ent": "La Azulita", "cls": "Northern Europe"}, {"id": 25867, "dom": "G", "ent": "La Azulita", "cls": "Polynesia"}, {"id": 25868, "dom": "G", "ent": "La Azulita", "cls": "South America"}, {"id": 25869, "dom": "G", "ent": "La Azulita", "cls": "South-eastern Asia"}, {"id": 25870, "dom": "G", "ent": "La Azulita", "cls": "Southern Africa"}, {"id": 25871, "dom": "G", "ent": "La Azulita", "cls": "Southern Asia"}, {"id": 25872, "dom": "G", "ent": "La Azulita", "cls": "Southern Europe"}, {"id": 25873, "dom": "G", "ent": "La Azulita", "cls": "Western Africa"}, {"id": 25874, "dom": "G", "ent": "La Azulita", "cls": "Western Asia"}, {"id": 25875, "dom": "G", "ent": "La Azulita", "cls": "Western Europe"}, {"id": 25876, "dom": "G", "ent": "Haiphong", "cls": "Australia and New Zealand"}, {"id": 25877, "dom": "G", "ent": "Haiphong", "cls": "Caribbean"}, {"id": 25878, "dom": "G", "ent": "Haiphong", "cls": "Central America"}, {"id": 25879, "dom": "G", "ent": "Haiphong", "cls": "Central Asia"}, {"id": 25880, "dom": "G", "ent": "Haiphong", "cls": "Eastern Africa"}, {"id": 25881, "dom": "G", "ent": "Haiphong", "cls": "Eastern Asia"}, {"id": 25882, "dom": "G", "ent": "Haiphong", "cls": "Eastern Europe"}, {"id": 25883, "dom": "G", "ent": "Haiphong", "cls": "Melanesia"}, {"id": 25884, "dom": "G", "ent": "Haiphong", "cls": "Micronesia"}, {"id": 25885, "dom": "G", "ent": "Haiphong", "cls": "Middle Africa"}, {"id": 25886, "dom": "G", "ent": "Haiphong", "cls": "Northern Africa"}, {"id": 25887, "dom": "G", "ent": "Haiphong", "cls": "Northern America"}, {"id": 25888, "dom": "G", "ent": "Haiphong", "cls": "Northern Europe"}, {"id": 25889, "dom": "G", "ent": "Haiphong", "cls": "Polynesia"}, {"id": 25890, "dom": "G", "ent": "Haiphong", "cls": "South America"}, {"id": 25891, "dom": "G", "ent": "Haiphong", "cls": "South-eastern Asia"}, {"id": 25892, "dom": "G", "ent": "Haiphong", "cls": "Southern Africa"}, {"id": 25893, "dom": "G", "ent": "Haiphong", "cls": "Southern Asia"}, {"id": 25894, "dom": "G", "ent": "Haiphong", "cls": "Southern Europe"}, {"id": 25895, "dom": "G", "ent": "Haiphong", "cls": "Western Africa"}, {"id": 25896, "dom": "G", "ent": "Haiphong", "cls": "Western Asia"}, {"id": 25897, "dom": "G", "ent": "Haiphong", "cls": "Western Europe"}, {"id": 25898, "dom": "G", "ent": "Phu Quoc", "cls": "Australia and New Zealand"}, {"id": 25899, "dom": "G", "ent": "Phu Quoc", "cls": "Caribbean"}, {"id": 25900, "dom": "G", "ent": "Phu Quoc", "cls": "Central America"}, {"id": 25901, "dom": "G", "ent": "Phu Quoc", "cls": "Central Asia"}, {"id": 25902, "dom": "G", "ent": "Phu Quoc", "cls": "Eastern Africa"}, {"id": 25903, "dom": "G", "ent": "Phu Quoc", "cls": "Eastern Asia"}, {"id": 25904, "dom": "G", "ent": "Phu Quoc", "cls": "Eastern Europe"}, {"id": 25905, "dom": "G", "ent": "Phu Quoc", "cls": "Melanesia"}, {"id": 25906, "dom": "G", "ent": "Phu Quoc", "cls": "Micronesia"}, {"id": 25907, "dom": "G", "ent": "Phu Quoc", "cls": "Middle Africa"}, {"id": 25908, "dom": "G", "ent": "Phu Quoc", "cls": "Northern Africa"}, {"id": 25909, "dom": "G", "ent": "Phu Quoc", "cls": "Northern America"}, {"id": 25910, "dom": "G", "ent": "Phu Quoc", "cls": "Northern Europe"}, {"id": 25911, "dom": "G", "ent": "Phu Quoc", "cls": "Polynesia"}, {"id": 25912, "dom": "G", "ent": "Phu Quoc", "cls": "South America"}, {"id": 25913, "dom": "G", "ent": "Phu Quoc", "cls": "South-eastern Asia"}, {"id": 25914, "dom": "G", "ent": "Phu Quoc", "cls": "Southern Africa"}, {"id": 25915, "dom": "G", "ent": "Phu Quoc", "cls": "Southern Asia"}, {"id": 25916, "dom": "G", "ent": "Phu Quoc", "cls": "Southern Europe"}, {"id": 25917, "dom": "G", "ent": "Phu Quoc", "cls": "Western Africa"}, {"id": 25918, "dom": "G", "ent": "Phu Quoc", "cls": "Western Asia"}, {"id": 25919, "dom": "G", "ent": "Phu Quoc", "cls": "Western Europe"}, {"id": 25920, "dom": "G", "ent": "Lai Vung", "cls": "Australia and New Zealand"}, {"id": 25921, "dom": "G", "ent": "Lai Vung", "cls": "Caribbean"}, {"id": 25922, "dom": "G", "ent": "Lai Vung", "cls": "Central America"}, {"id": 25923, "dom": "G", "ent": "Lai Vung", "cls": "Central Asia"}, {"id": 25924, "dom": "G", "ent": "Lai Vung", "cls": "Eastern Africa"}, {"id": 25925, "dom": "G", "ent": "Lai Vung", "cls": "Eastern Asia"}, {"id": 25926, "dom": "G", "ent": "Lai Vung", "cls": "Eastern Europe"}, {"id": 25927, "dom": "G", "ent": "Lai Vung", "cls": "Melanesia"}, {"id": 25928, "dom": "G", "ent": "Lai Vung", "cls": "Micronesia"}, {"id": 25929, "dom": "G", "ent": "Lai Vung", "cls": "Middle Africa"}, {"id": 25930, "dom": "G", "ent": "Lai Vung", "cls": "Northern Africa"}, {"id": 25931, "dom": "G", "ent": "Lai Vung", "cls": "Northern America"}, {"id": 25932, "dom": "G", "ent": "Lai Vung", "cls": "Northern Europe"}, {"id": 25933, "dom": "G", "ent": "Lai Vung", "cls": "Polynesia"}, {"id": 25934, "dom": "G", "ent": "Lai Vung", "cls": "South America"}, {"id": 25935, "dom": "G", "ent": "Lai Vung", "cls": "South-eastern Asia"}, {"id": 25936, "dom": "G", "ent": "Lai Vung", "cls": "Southern Africa"}, {"id": 25937, "dom": "G", "ent": "Lai Vung", "cls": "Southern Asia"}, {"id": 25938, "dom": "G", "ent": "Lai Vung", "cls": "Southern Europe"}, {"id": 25939, "dom": "G", "ent": "Lai Vung", "cls": "Western Africa"}, {"id": 25940, "dom": "G", "ent": "Lai Vung", "cls": "Western Asia"}, {"id": 25941, "dom": "G", "ent": "Lai Vung", "cls": "Western Europe"}, {"id": 25942, "dom": "G", "ent": "Mui Ne", "cls": "Australia and New Zealand"}, {"id": 25943, "dom": "G", "ent": "Mui Ne", "cls": "Caribbean"}, {"id": 25944, "dom": "G", "ent": "Mui Ne", "cls": "Central America"}, {"id": 25945, "dom": "G", "ent": "Mui Ne", "cls": "Central Asia"}, {"id": 25946, "dom": "G", "ent": "Mui Ne", "cls": "Eastern Africa"}, {"id": 25947, "dom": "G", "ent": "Mui Ne", "cls": "Eastern Asia"}, {"id": 25948, "dom": "G", "ent": "Mui Ne", "cls": "Eastern Europe"}, {"id": 25949, "dom": "G", "ent": "Mui Ne", "cls": "Melanesia"}, {"id": 25950, "dom": "G", "ent": "Mui Ne", "cls": "Micronesia"}, {"id": 25951, "dom": "G", "ent": "Mui Ne", "cls": "Middle Africa"}, {"id": 25952, "dom": "G", "ent": "Mui Ne", "cls": "Northern Africa"}, {"id": 25953, "dom": "G", "ent": "Mui Ne", "cls": "Northern America"}, {"id": 25954, "dom": "G", "ent": "Mui Ne", "cls": "Northern Europe"}, {"id": 25955, "dom": "G", "ent": "Mui Ne", "cls": "Polynesia"}, {"id": 25956, "dom": "G", "ent": "Mui Ne", "cls": "South America"}, {"id": 25957, "dom": "G", "ent": "Mui Ne", "cls": "South-eastern Asia"}, {"id": 25958, "dom": "G", "ent": "Mui Ne", "cls": "Southern Africa"}, {"id": 25959, "dom": "G", "ent": "Mui Ne", "cls": "Southern Asia"}, {"id": 25960, "dom": "G", "ent": "Mui Ne", "cls": "Southern Europe"}, {"id": 25961, "dom": "G", "ent": "Mui Ne", "cls": "Western Africa"}, {"id": 25962, "dom": "G", "ent": "Mui Ne", "cls": "Western Asia"}, {"id": 25963, "dom": "G", "ent": "Mui Ne", "cls": "Western Europe"}, {"id": 25964, "dom": "G", "ent": "Al Bayda", "cls": "Australia and New Zealand"}, {"id": 25965, "dom": "G", "ent": "Al Bayda", "cls": "Caribbean"}, {"id": 25966, "dom": "G", "ent": "Al Bayda", "cls": "Central America"}, {"id": 25967, "dom": "G", "ent": "Al Bayda", "cls": "Central Asia"}, {"id": 25968, "dom": "G", "ent": "Al Bayda", "cls": "Eastern Africa"}, {"id": 25969, "dom": "G", "ent": "Al Bayda", "cls": "Eastern Asia"}, {"id": 25970, "dom": "G", "ent": "Al Bayda", "cls": "Eastern Europe"}, {"id": 25971, "dom": "G", "ent": "Al Bayda", "cls": "Melanesia"}, {"id": 25972, "dom": "G", "ent": "Al Bayda", "cls": "Micronesia"}, {"id": 25973, "dom": "G", "ent": "Al Bayda", "cls": "Middle Africa"}, {"id": 25974, "dom": "G", "ent": "Al Bayda", "cls": "Northern Africa"}, {"id": 25975, "dom": "G", "ent": "Al Bayda", "cls": "Northern America"}, {"id": 25976, "dom": "G", "ent": "Al Bayda", "cls": "Northern Europe"}, {"id": 25977, "dom": "G", "ent": "Al Bayda", "cls": "Polynesia"}, {"id": 25978, "dom": "G", "ent": "Al Bayda", "cls": "South America"}, {"id": 25979, "dom": "G", "ent": "Al Bayda", "cls": "South-eastern Asia"}, {"id": 25980, "dom": "G", "ent": "Al Bayda", "cls": "Southern Africa"}, {"id": 25981, "dom": "G", "ent": "Al Bayda", "cls": "Southern Asia"}, {"id": 25982, "dom": "G", "ent": "Al Bayda", "cls": "Southern Europe"}, {"id": 25983, "dom": "G", "ent": "Al Bayda", "cls": "Western Africa"}, {"id": 25984, "dom": "G", "ent": "Al Bayda", "cls": "Western Asia"}, {"id": 25985, "dom": "G", "ent": "Al Bayda", "cls": "Western Europe"}, {"id": 25986, "dom": "G", "ent": "Ataq", "cls": "Australia and New Zealand"}, {"id": 25987, "dom": "G", "ent": "Ataq", "cls": "Caribbean"}, {"id": 25988, "dom": "G", "ent": "Ataq", "cls": "Central America"}, {"id": 25989, "dom": "G", "ent": "Ataq", "cls": "Central Asia"}, {"id": 25990, "dom": "G", "ent": "Ataq", "cls": "Eastern Africa"}, {"id": 25991, "dom": "G", "ent": "Ataq", "cls": "Eastern Asia"}, {"id": 25992, "dom": "G", "ent": "Ataq", "cls": "Eastern Europe"}, {"id": 25993, "dom": "G", "ent": "Ataq", "cls": "Melanesia"}, {"id": 25994, "dom": "G", "ent": "Ataq", "cls": "Micronesia"}, {"id": 25995, "dom": "G", "ent": "Ataq", "cls": "Middle Africa"}, {"id": 25996, "dom": "G", "ent": "Ataq", "cls": "Northern Africa"}, {"id": 25997, "dom": "G", "ent": "Ataq", "cls": "Northern America"}, {"id": 25998, "dom": "G", "ent": "Ataq", "cls": "Northern Europe"}, {"id": 25999, "dom": "G", "ent": "Ataq", "cls": "Polynesia"}, {"id": 26000, "dom": "G", "ent": "Ataq", "cls": "South America"}, {"id": 26001, "dom": "G", "ent": "Ataq", "cls": "South-eastern Asia"}, {"id": 26002, "dom": "G", "ent": "Ataq", "cls": "Southern Africa"}, {"id": 26003, "dom": "G", "ent": "Ataq", "cls": "Southern Asia"}, {"id": 26004, "dom": "G", "ent": "Ataq", "cls": "Southern Europe"}, {"id": 26005, "dom": "G", "ent": "Ataq", "cls": "Western Africa"}, {"id": 26006, "dom": "G", "ent": "Ataq", "cls": "Western Asia"}, {"id": 26007, "dom": "G", "ent": "Ataq", "cls": "Western Europe"}, {"id": 26008, "dom": "G", "ent": "Ma'rib", "cls": "Australia and New Zealand"}, {"id": 26009, "dom": "G", "ent": "Ma'rib", "cls": "Caribbean"}, {"id": 26010, "dom": "G", "ent": "Ma'rib", "cls": "Central America"}, {"id": 26011, "dom": "G", "ent": "Ma'rib", "cls": "Central Asia"}, {"id": 26012, "dom": "G", "ent": "Ma'rib", "cls": "Eastern Africa"}, {"id": 26013, "dom": "G", "ent": "Ma'rib", "cls": "Eastern Asia"}, {"id": 26014, "dom": "G", "ent": "Ma'rib", "cls": "Eastern Europe"}, {"id": 26015, "dom": "G", "ent": "Ma'rib", "cls": "Melanesia"}, {"id": 26016, "dom": "G", "ent": "Ma'rib", "cls": "Micronesia"}, {"id": 26017, "dom": "G", "ent": "Ma'rib", "cls": "Middle Africa"}, {"id": 26018, "dom": "G", "ent": "Ma'rib", "cls": "Northern Africa"}, {"id": 26019, "dom": "G", "ent": "Ma'rib", "cls": "Northern America"}, {"id": 26020, "dom": "G", "ent": "Ma'rib", "cls": "Northern Europe"}, {"id": 26021, "dom": "G", "ent": "Ma'rib", "cls": "Polynesia"}, {"id": 26022, "dom": "G", "ent": "Ma'rib", "cls": "South America"}, {"id": 26023, "dom": "G", "ent": "Ma'rib", "cls": "South-eastern Asia"}, {"id": 26024, "dom": "G", "ent": "Ma'rib", "cls": "Southern Africa"}, {"id": 26025, "dom": "G", "ent": "Ma'rib", "cls": "Southern Asia"}, {"id": 26026, "dom": "G", "ent": "Ma'rib", "cls": "Southern Europe"}, {"id": 26027, "dom": "G", "ent": "Ma'rib", "cls": "Western Africa"}, {"id": 26028, "dom": "G", "ent": "Ma'rib", "cls": "Western Asia"}, {"id": 26029, "dom": "G", "ent": "Ma'rib", "cls": "Western Europe"}, {"id": 26030, "dom": "G", "ent": "Mocha", "cls": "Australia and New Zealand"}, {"id": 26031, "dom": "G", "ent": "Mocha", "cls": "Caribbean"}, {"id": 26032, "dom": "G", "ent": "Mocha", "cls": "Central America"}, {"id": 26033, "dom": "G", "ent": "Mocha", "cls": "Central Asia"}, {"id": 26034, "dom": "G", "ent": "Mocha", "cls": "Eastern Africa"}, {"id": 26035, "dom": "G", "ent": "Mocha", "cls": "Eastern Asia"}, {"id": 26036, "dom": "G", "ent": "Mocha", "cls": "Eastern Europe"}, {"id": 26037, "dom": "G", "ent": "Mocha", "cls": "Melanesia"}, {"id": 26038, "dom": "G", "ent": "Mocha", "cls": "Micronesia"}, {"id": 26039, "dom": "G", "ent": "Mocha", "cls": "Middle Africa"}, {"id": 26040, "dom": "G", "ent": "Mocha", "cls": "Northern Africa"}, {"id": 26041, "dom": "G", "ent": "Mocha", "cls": "Northern America"}, {"id": 26042, "dom": "G", "ent": "Mocha", "cls": "Northern Europe"}, {"id": 26043, "dom": "G", "ent": "Mocha", "cls": "Polynesia"}, {"id": 26044, "dom": "G", "ent": "Mocha", "cls": "South America"}, {"id": 26045, "dom": "G", "ent": "Mocha", "cls": "South-eastern Asia"}, {"id": 26046, "dom": "G", "ent": "Mocha", "cls": "Southern Africa"}, {"id": 26047, "dom": "G", "ent": "Mocha", "cls": "Southern Asia"}, {"id": 26048, "dom": "G", "ent": "Mocha", "cls": "Southern Europe"}, {"id": 26049, "dom": "G", "ent": "Mocha", "cls": "Western Africa"}, {"id": 26050, "dom": "G", "ent": "Mocha", "cls": "Western Asia"}, {"id": 26051, "dom": "G", "ent": "Mocha", "cls": "Western Europe"}, {"id": 26052, "dom": "G", "ent": "Centurion", "cls": "Australia and New Zealand"}, {"id": 26053, "dom": "G", "ent": "Centurion", "cls": "Caribbean"}, {"id": 26054, "dom": "G", "ent": "Centurion", "cls": "Central America"}, {"id": 26055, "dom": "G", "ent": "Centurion", "cls": "Central Asia"}, {"id": 26056, "dom": "G", "ent": "Centurion", "cls": "Eastern Africa"}, {"id": 26057, "dom": "G", "ent": "Centurion", "cls": "Eastern Asia"}, {"id": 26058, "dom": "G", "ent": "Centurion", "cls": "Eastern Europe"}, {"id": 26059, "dom": "G", "ent": "Centurion", "cls": "Melanesia"}, {"id": 26060, "dom": "G", "ent": "Centurion", "cls": "Micronesia"}, {"id": 26061, "dom": "G", "ent": "Centurion", "cls": "Middle Africa"}, {"id": 26062, "dom": "G", "ent": "Centurion", "cls": "Northern Africa"}, {"id": 26063, "dom": "G", "ent": "Centurion", "cls": "Northern America"}, {"id": 26064, "dom": "G", "ent": "Centurion", "cls": "Northern Europe"}, {"id": 26065, "dom": "G", "ent": "Centurion", "cls": "Polynesia"}, {"id": 26066, "dom": "G", "ent": "Centurion", "cls": "South America"}, {"id": 26067, "dom": "G", "ent": "Centurion", "cls": "South-eastern Asia"}, {"id": 26068, "dom": "G", "ent": "Centurion", "cls": "Southern Africa"}, {"id": 26069, "dom": "G", "ent": "Centurion", "cls": "Southern Asia"}, {"id": 26070, "dom": "G", "ent": "Centurion", "cls": "Southern Europe"}, {"id": 26071, "dom": "G", "ent": "Centurion", "cls": "Western Africa"}, {"id": 26072, "dom": "G", "ent": "Centurion", "cls": "Western Asia"}, {"id": 26073, "dom": "G", "ent": "Centurion", "cls": "Western Europe"}, {"id": 26074, "dom": "G", "ent": "Kroonstad", "cls": "Australia and New Zealand"}, {"id": 26075, "dom": "G", "ent": "Kroonstad", "cls": "Caribbean"}, {"id": 26076, "dom": "G", "ent": "Kroonstad", "cls": "Central America"}, {"id": 26077, "dom": "G", "ent": "Kroonstad", "cls": "Central Asia"}, {"id": 26078, "dom": "G", "ent": "Kroonstad", "cls": "Eastern Africa"}, {"id": 26079, "dom": "G", "ent": "Kroonstad", "cls": "Eastern Asia"}, {"id": 26080, "dom": "G", "ent": "Kroonstad", "cls": "Eastern Europe"}, {"id": 26081, "dom": "G", "ent": "Kroonstad", "cls": "Melanesia"}, {"id": 26082, "dom": "G", "ent": "Kroonstad", "cls": "Micronesia"}, {"id": 26083, "dom": "G", "ent": "Kroonstad", "cls": "Middle Africa"}, {"id": 26084, "dom": "G", "ent": "Kroonstad", "cls": "Northern Africa"}, {"id": 26085, "dom": "G", "ent": "Kroonstad", "cls": "Northern America"}, {"id": 26086, "dom": "G", "ent": "Kroonstad", "cls": "Northern Europe"}, {"id": 26087, "dom": "G", "ent": "Kroonstad", "cls": "Polynesia"}, {"id": 26088, "dom": "G", "ent": "Kroonstad", "cls": "South America"}, {"id": 26089, "dom": "G", "ent": "Kroonstad", "cls": "South-eastern Asia"}, {"id": 26090, "dom": "G", "ent": "Kroonstad", "cls": "Southern Africa"}, {"id": 26091, "dom": "G", "ent": "Kroonstad", "cls": "Southern Asia"}, {"id": 26092, "dom": "G", "ent": "Kroonstad", "cls": "Southern Europe"}, {"id": 26093, "dom": "G", "ent": "Kroonstad", "cls": "Western Africa"}, {"id": 26094, "dom": "G", "ent": "Kroonstad", "cls": "Western Asia"}, {"id": 26095, "dom": "G", "ent": "Kroonstad", "cls": "Western Europe"}, {"id": 26096, "dom": "G", "ent": "Harrismith", "cls": "Australia and New Zealand"}, {"id": 26097, "dom": "G", "ent": "Harrismith", "cls": "Caribbean"}, {"id": 26098, "dom": "G", "ent": "Harrismith", "cls": "Central America"}, {"id": 26099, "dom": "G", "ent": "Harrismith", "cls": "Central Asia"}, {"id": 26100, "dom": "G", "ent": "Harrismith", "cls": "Eastern Africa"}, {"id": 26101, "dom": "G", "ent": "Harrismith", "cls": "Eastern Asia"}, {"id": 26102, "dom": "G", "ent": "Harrismith", "cls": "Eastern Europe"}, {"id": 26103, "dom": "G", "ent": "Harrismith", "cls": "Melanesia"}, {"id": 26104, "dom": "G", "ent": "Harrismith", "cls": "Micronesia"}, {"id": 26105, "dom": "G", "ent": "Harrismith", "cls": "Middle Africa"}, {"id": 26106, "dom": "G", "ent": "Harrismith", "cls": "Northern Africa"}, {"id": 26107, "dom": "G", "ent": "Harrismith", "cls": "Northern America"}, {"id": 26108, "dom": "G", "ent": "Harrismith", "cls": "Northern Europe"}, {"id": 26109, "dom": "G", "ent": "Harrismith", "cls": "Polynesia"}, {"id": 26110, "dom": "G", "ent": "Harrismith", "cls": "South America"}, {"id": 26111, "dom": "G", "ent": "Harrismith", "cls": "South-eastern Asia"}, {"id": 26112, "dom": "G", "ent": "Harrismith", "cls": "Southern Africa"}, {"id": 26113, "dom": "G", "ent": "Harrismith", "cls": "Southern Asia"}, {"id": 26114, "dom": "G", "ent": "Harrismith", "cls": "Southern Europe"}, {"id": 26115, "dom": "G", "ent": "Harrismith", "cls": "Western Africa"}, {"id": 26116, "dom": "G", "ent": "Harrismith", "cls": "Western Asia"}, {"id": 26117, "dom": "G", "ent": "Harrismith", "cls": "Western Europe"}, {"id": 26118, "dom": "G", "ent": "Burgersdorp", "cls": "Australia and New Zealand"}, {"id": 26119, "dom": "G", "ent": "Burgersdorp", "cls": "Caribbean"}, {"id": 26120, "dom": "G", "ent": "Burgersdorp", "cls": "Central America"}, {"id": 26121, "dom": "G", "ent": "Burgersdorp", "cls": "Central Asia"}, {"id": 26122, "dom": "G", "ent": "Burgersdorp", "cls": "Eastern Africa"}, {"id": 26123, "dom": "G", "ent": "Burgersdorp", "cls": "Eastern Asia"}, {"id": 26124, "dom": "G", "ent": "Burgersdorp", "cls": "Eastern Europe"}, {"id": 26125, "dom": "G", "ent": "Burgersdorp", "cls": "Melanesia"}, {"id": 26126, "dom": "G", "ent": "Burgersdorp", "cls": "Micronesia"}, {"id": 26127, "dom": "G", "ent": "Burgersdorp", "cls": "Middle Africa"}, {"id": 26128, "dom": "G", "ent": "Burgersdorp", "cls": "Northern Africa"}, {"id": 26129, "dom": "G", "ent": "Burgersdorp", "cls": "Northern America"}, {"id": 26130, "dom": "G", "ent": "Burgersdorp", "cls": "Northern Europe"}, {"id": 26131, "dom": "G", "ent": "Burgersdorp", "cls": "Polynesia"}, {"id": 26132, "dom": "G", "ent": "Burgersdorp", "cls": "South America"}, {"id": 26133, "dom": "G", "ent": "Burgersdorp", "cls": "South-eastern Asia"}, {"id": 26134, "dom": "G", "ent": "Burgersdorp", "cls": "Southern Africa"}, {"id": 26135, "dom": "G", "ent": "Burgersdorp", "cls": "Southern Asia"}, {"id": 26136, "dom": "G", "ent": "Burgersdorp", "cls": "Southern Europe"}, {"id": 26137, "dom": "G", "ent": "Burgersdorp", "cls": "Western Africa"}, {"id": 26138, "dom": "G", "ent": "Burgersdorp", "cls": "Western Asia"}, {"id": 26139, "dom": "G", "ent": "Burgersdorp", "cls": "Western Europe"}, {"id": 26140, "dom": "G", "ent": "Mufulira", "cls": "Australia and New Zealand"}, {"id": 26141, "dom": "G", "ent": "Mufulira", "cls": "Caribbean"}, {"id": 26142, "dom": "G", "ent": "Mufulira", "cls": "Central America"}, {"id": 26143, "dom": "G", "ent": "Mufulira", "cls": "Central Asia"}, {"id": 26144, "dom": "G", "ent": "Mufulira", "cls": "Eastern Africa"}, {"id": 26145, "dom": "G", "ent": "Mufulira", "cls": "Eastern Asia"}, {"id": 26146, "dom": "G", "ent": "Mufulira", "cls": "Eastern Europe"}, {"id": 26147, "dom": "G", "ent": "Mufulira", "cls": "Melanesia"}, {"id": 26148, "dom": "G", "ent": "Mufulira", "cls": "Micronesia"}, {"id": 26149, "dom": "G", "ent": "Mufulira", "cls": "Middle Africa"}, {"id": 26150, "dom": "G", "ent": "Mufulira", "cls": "Northern Africa"}, {"id": 26151, "dom": "G", "ent": "Mufulira", "cls": "Northern America"}, {"id": 26152, "dom": "G", "ent": "Mufulira", "cls": "Northern Europe"}, {"id": 26153, "dom": "G", "ent": "Mufulira", "cls": "Polynesia"}, {"id": 26154, "dom": "G", "ent": "Mufulira", "cls": "South America"}, {"id": 26155, "dom": "G", "ent": "Mufulira", "cls": "South-eastern Asia"}, {"id": 26156, "dom": "G", "ent": "Mufulira", "cls": "Southern Africa"}, {"id": 26157, "dom": "G", "ent": "Mufulira", "cls": "Southern Asia"}, {"id": 26158, "dom": "G", "ent": "Mufulira", "cls": "Southern Europe"}, {"id": 26159, "dom": "G", "ent": "Mufulira", "cls": "Western Africa"}, {"id": 26160, "dom": "G", "ent": "Mufulira", "cls": "Western Asia"}, {"id": 26161, "dom": "G", "ent": "Mufulira", "cls": "Western Europe"}, {"id": 26162, "dom": "G", "ent": "Luwingu", "cls": "Australia and New Zealand"}, {"id": 26163, "dom": "G", "ent": "Luwingu", "cls": "Caribbean"}, {"id": 26164, "dom": "G", "ent": "Luwingu", "cls": "Central America"}, {"id": 26165, "dom": "G", "ent": "Luwingu", "cls": "Central Asia"}, {"id": 26166, "dom": "G", "ent": "Luwingu", "cls": "Eastern Africa"}, {"id": 26167, "dom": "G", "ent": "Luwingu", "cls": "Eastern Asia"}, {"id": 26168, "dom": "G", "ent": "Luwingu", "cls": "Eastern Europe"}, {"id": 26169, "dom": "G", "ent": "Luwingu", "cls": "Melanesia"}, {"id": 26170, "dom": "G", "ent": "Luwingu", "cls": "Micronesia"}, {"id": 26171, "dom": "G", "ent": "Luwingu", "cls": "Middle Africa"}, {"id": 26172, "dom": "G", "ent": "Luwingu", "cls": "Northern Africa"}, {"id": 26173, "dom": "G", "ent": "Luwingu", "cls": "Northern America"}, {"id": 26174, "dom": "G", "ent": "Luwingu", "cls": "Northern Europe"}, {"id": 26175, "dom": "G", "ent": "Luwingu", "cls": "Polynesia"}, {"id": 26176, "dom": "G", "ent": "Luwingu", "cls": "South America"}, {"id": 26177, "dom": "G", "ent": "Luwingu", "cls": "South-eastern Asia"}, {"id": 26178, "dom": "G", "ent": "Luwingu", "cls": "Southern Africa"}, {"id": 26179, "dom": "G", "ent": "Luwingu", "cls": "Southern Asia"}, {"id": 26180, "dom": "G", "ent": "Luwingu", "cls": "Southern Europe"}, {"id": 26181, "dom": "G", "ent": "Luwingu", "cls": "Western Africa"}, {"id": 26182, "dom": "G", "ent": "Luwingu", "cls": "Western Asia"}, {"id": 26183, "dom": "G", "ent": "Luwingu", "cls": "Western Europe"}, {"id": 26184, "dom": "G", "ent": "Manyinga", "cls": "Australia and New Zealand"}, {"id": 26185, "dom": "G", "ent": "Manyinga", "cls": "Caribbean"}, {"id": 26186, "dom": "G", "ent": "Manyinga", "cls": "Central America"}, {"id": 26187, "dom": "G", "ent": "Manyinga", "cls": "Central Asia"}, {"id": 26188, "dom": "G", "ent": "Manyinga", "cls": "Eastern Africa"}, {"id": 26189, "dom": "G", "ent": "Manyinga", "cls": "Eastern Asia"}, {"id": 26190, "dom": "G", "ent": "Manyinga", "cls": "Eastern Europe"}, {"id": 26191, "dom": "G", "ent": "Manyinga", "cls": "Melanesia"}, {"id": 26192, "dom": "G", "ent": "Manyinga", "cls": "Micronesia"}, {"id": 26193, "dom": "G", "ent": "Manyinga", "cls": "Middle Africa"}, {"id": 26194, "dom": "G", "ent": "Manyinga", "cls": "Northern Africa"}, {"id": 26195, "dom": "G", "ent": "Manyinga", "cls": "Northern America"}, {"id": 26196, "dom": "G", "ent": "Manyinga", "cls": "Northern Europe"}, {"id": 26197, "dom": "G", "ent": "Manyinga", "cls": "Polynesia"}, {"id": 26198, "dom": "G", "ent": "Manyinga", "cls": "South America"}, {"id": 26199, "dom": "G", "ent": "Manyinga", "cls": "South-eastern Asia"}, {"id": 26200, "dom": "G", "ent": "Manyinga", "cls": "Southern Africa"}, {"id": 26201, "dom": "G", "ent": "Manyinga", "cls": "Southern Asia"}, {"id": 26202, "dom": "G", "ent": "Manyinga", "cls": "Southern Europe"}, {"id": 26203, "dom": "G", "ent": "Manyinga", "cls": "Western Africa"}, {"id": 26204, "dom": "G", "ent": "Manyinga", "cls": "Western Asia"}, {"id": 26205, "dom": "G", "ent": "Manyinga", "cls": "Western Europe"}, {"id": 26206, "dom": "G", "ent": "Chiengi", "cls": "Australia and New Zealand"}, {"id": 26207, "dom": "G", "ent": "Chiengi", "cls": "Caribbean"}, {"id": 26208, "dom": "G", "ent": "Chiengi", "cls": "Central America"}, {"id": 26209, "dom": "G", "ent": "Chiengi", "cls": "Central Asia"}, {"id": 26210, "dom": "G", "ent": "Chiengi", "cls": "Eastern Africa"}, {"id": 26211, "dom": "G", "ent": "Chiengi", "cls": "Eastern Asia"}, {"id": 26212, "dom": "G", "ent": "Chiengi", "cls": "Eastern Europe"}, {"id": 26213, "dom": "G", "ent": "Chiengi", "cls": "Melanesia"}, {"id": 26214, "dom": "G", "ent": "Chiengi", "cls": "Micronesia"}, {"id": 26215, "dom": "G", "ent": "Chiengi", "cls": "Middle Africa"}, {"id": 26216, "dom": "G", "ent": "Chiengi", "cls": "Northern Africa"}, {"id": 26217, "dom": "G", "ent": "Chiengi", "cls": "Northern America"}, {"id": 26218, "dom": "G", "ent": "Chiengi", "cls": "Northern Europe"}, {"id": 26219, "dom": "G", "ent": "Chiengi", "cls": "Polynesia"}, {"id": 26220, "dom": "G", "ent": "Chiengi", "cls": "South America"}, {"id": 26221, "dom": "G", "ent": "Chiengi", "cls": "South-eastern Asia"}, {"id": 26222, "dom": "G", "ent": "Chiengi", "cls": "Southern Africa"}, {"id": 26223, "dom": "G", "ent": "Chiengi", "cls": "Southern Asia"}, {"id": 26224, "dom": "G", "ent": "Chiengi", "cls": "Southern Europe"}, {"id": 26225, "dom": "G", "ent": "Chiengi", "cls": "Western Africa"}, {"id": 26226, "dom": "G", "ent": "Chiengi", "cls": "Western Asia"}, {"id": 26227, "dom": "G", "ent": "Chiengi", "cls": "Western Europe"}, {"id": 26228, "dom": "G", "ent": "Chegutu", "cls": "Australia and New Zealand"}, {"id": 26229, "dom": "G", "ent": "Chegutu", "cls": "Caribbean"}, {"id": 26230, "dom": "G", "ent": "Chegutu", "cls": "Central America"}, {"id": 26231, "dom": "G", "ent": "Chegutu", "cls": "Central Asia"}, {"id": 26232, "dom": "G", "ent": "Chegutu", "cls": "Eastern Africa"}, {"id": 26233, "dom": "G", "ent": "Chegutu", "cls": "Eastern Asia"}, {"id": 26234, "dom": "G", "ent": "Chegutu", "cls": "Eastern Europe"}, {"id": 26235, "dom": "G", "ent": "Chegutu", "cls": "Melanesia"}, {"id": 26236, "dom": "G", "ent": "Chegutu", "cls": "Micronesia"}, {"id": 26237, "dom": "G", "ent": "Chegutu", "cls": "Middle Africa"}, {"id": 26238, "dom": "G", "ent": "Chegutu", "cls": "Northern Africa"}, {"id": 26239, "dom": "G", "ent": "Chegutu", "cls": "Northern America"}, {"id": 26240, "dom": "G", "ent": "Chegutu", "cls": "Northern Europe"}, {"id": 26241, "dom": "G", "ent": "Chegutu", "cls": "Polynesia"}, {"id": 26242, "dom": "G", "ent": "Chegutu", "cls": "South America"}, {"id": 26243, "dom": "G", "ent": "Chegutu", "cls": "South-eastern Asia"}, {"id": 26244, "dom": "G", "ent": "Chegutu", "cls": "Southern Africa"}, {"id": 26245, "dom": "G", "ent": "Chegutu", "cls": "Southern Asia"}, {"id": 26246, "dom": "G", "ent": "Chegutu", "cls": "Southern Europe"}, {"id": 26247, "dom": "G", "ent": "Chegutu", "cls": "Western Africa"}, {"id": 26248, "dom": "G", "ent": "Chegutu", "cls": "Western Asia"}, {"id": 26249, "dom": "G", "ent": "Chegutu", "cls": "Western Europe"}, {"id": 26250, "dom": "G", "ent": "Chiredzi", "cls": "Australia and New Zealand"}, {"id": 26251, "dom": "G", "ent": "Chiredzi", "cls": "Caribbean"}, {"id": 26252, "dom": "G", "ent": "Chiredzi", "cls": "Central America"}, {"id": 26253, "dom": "G", "ent": "Chiredzi", "cls": "Central Asia"}, {"id": 26254, "dom": "G", "ent": "Chiredzi", "cls": "Eastern Africa"}, {"id": 26255, "dom": "G", "ent": "Chiredzi", "cls": "Eastern Asia"}, {"id": 26256, "dom": "G", "ent": "Chiredzi", "cls": "Eastern Europe"}, {"id": 26257, "dom": "G", "ent": "Chiredzi", "cls": "Melanesia"}, {"id": 26258, "dom": "G", "ent": "Chiredzi", "cls": "Micronesia"}, {"id": 26259, "dom": "G", "ent": "Chiredzi", "cls": "Middle Africa"}, {"id": 26260, "dom": "G", "ent": "Chiredzi", "cls": "Northern Africa"}, {"id": 26261, "dom": "G", "ent": "Chiredzi", "cls": "Northern America"}, {"id": 26262, "dom": "G", "ent": "Chiredzi", "cls": "Northern Europe"}, {"id": 26263, "dom": "G", "ent": "Chiredzi", "cls": "Polynesia"}, {"id": 26264, "dom": "G", "ent": "Chiredzi", "cls": "South America"}, {"id": 26265, "dom": "G", "ent": "Chiredzi", "cls": "South-eastern Asia"}, {"id": 26266, "dom": "G", "ent": "Chiredzi", "cls": "Southern Africa"}, {"id": 26267, "dom": "G", "ent": "Chiredzi", "cls": "Southern Asia"}, {"id": 26268, "dom": "G", "ent": "Chiredzi", "cls": "Southern Europe"}, {"id": 26269, "dom": "G", "ent": "Chiredzi", "cls": "Western Africa"}, {"id": 26270, "dom": "G", "ent": "Chiredzi", "cls": "Western Asia"}, {"id": 26271, "dom": "G", "ent": "Chiredzi", "cls": "Western Europe"}, {"id": 26272, "dom": "G", "ent": "Gokwe", "cls": "Australia and New Zealand"}, {"id": 26273, "dom": "G", "ent": "Gokwe", "cls": "Caribbean"}, {"id": 26274, "dom": "G", "ent": "Gokwe", "cls": "Central America"}, {"id": 26275, "dom": "G", "ent": "Gokwe", "cls": "Central Asia"}, {"id": 26276, "dom": "G", "ent": "Gokwe", "cls": "Eastern Africa"}, {"id": 26277, "dom": "G", "ent": "Gokwe", "cls": "Eastern Asia"}, {"id": 26278, "dom": "G", "ent": "Gokwe", "cls": "Eastern Europe"}, {"id": 26279, "dom": "G", "ent": "Gokwe", "cls": "Melanesia"}, {"id": 26280, "dom": "G", "ent": "Gokwe", "cls": "Micronesia"}, {"id": 26281, "dom": "G", "ent": "Gokwe", "cls": "Middle Africa"}, {"id": 26282, "dom": "G", "ent": "Gokwe", "cls": "Northern Africa"}, {"id": 26283, "dom": "G", "ent": "Gokwe", "cls": "Northern America"}, {"id": 26284, "dom": "G", "ent": "Gokwe", "cls": "Northern Europe"}, {"id": 26285, "dom": "G", "ent": "Gokwe", "cls": "Polynesia"}, {"id": 26286, "dom": "G", "ent": "Gokwe", "cls": "South America"}, {"id": 26287, "dom": "G", "ent": "Gokwe", "cls": "South-eastern Asia"}, {"id": 26288, "dom": "G", "ent": "Gokwe", "cls": "Southern Africa"}, {"id": 26289, "dom": "G", "ent": "Gokwe", "cls": "Southern Asia"}, {"id": 26290, "dom": "G", "ent": "Gokwe", "cls": "Southern Europe"}, {"id": 26291, "dom": "G", "ent": "Gokwe", "cls": "Western Africa"}, {"id": 26292, "dom": "G", "ent": "Gokwe", "cls": "Western Asia"}, {"id": 26293, "dom": "G", "ent": "Gokwe", "cls": "Western Europe"}, {"id": 26294, "dom": "G", "ent": "Gwanda", "cls": "Australia and New Zealand"}, {"id": 26295, "dom": "G", "ent": "Gwanda", "cls": "Caribbean"}, {"id": 26296, "dom": "G", "ent": "Gwanda", "cls": "Central America"}, {"id": 26297, "dom": "G", "ent": "Gwanda", "cls": "Central Asia"}, {"id": 26298, "dom": "G", "ent": "Gwanda", "cls": "Eastern Africa"}, {"id": 26299, "dom": "G", "ent": "Gwanda", "cls": "Eastern Asia"}, {"id": 26300, "dom": "G", "ent": "Gwanda", "cls": "Eastern Europe"}, {"id": 26301, "dom": "G", "ent": "Gwanda", "cls": "Melanesia"}, {"id": 26302, "dom": "G", "ent": "Gwanda", "cls": "Micronesia"}, {"id": 26303, "dom": "G", "ent": "Gwanda", "cls": "Middle Africa"}, {"id": 26304, "dom": "G", "ent": "Gwanda", "cls": "Northern Africa"}, {"id": 26305, "dom": "G", "ent": "Gwanda", "cls": "Northern America"}, {"id": 26306, "dom": "G", "ent": "Gwanda", "cls": "Northern Europe"}, {"id": 26307, "dom": "G", "ent": "Gwanda", "cls": "Polynesia"}, {"id": 26308, "dom": "G", "ent": "Gwanda", "cls": "South America"}, {"id": 26309, "dom": "G", "ent": "Gwanda", "cls": "South-eastern Asia"}, {"id": 26310, "dom": "G", "ent": "Gwanda", "cls": "Southern Africa"}, {"id": 26311, "dom": "G", "ent": "Gwanda", "cls": "Southern Asia"}, {"id": 26312, "dom": "G", "ent": "Gwanda", "cls": "Southern Europe"}, {"id": 26313, "dom": "G", "ent": "Gwanda", "cls": "Western Africa"}, {"id": 26314, "dom": "G", "ent": "Gwanda", "cls": "Western Asia"}, {"id": 26315, "dom": "G", "ent": "Gwanda", "cls": "Western Europe"}]}''')
json.dump(ITEMS, open('/content/e101_items_nogold.json', 'w'))
print(len(ITEMS['items']), '主張')

In [ ]:
# 場面ごとの学習（analysis/e11/e113_box.py と同じ）
"""E11-3 の箱（E10-1 の部分と全体・病気・地理）。E11-2 と同じ作り方（e112_box.py の数のまま、物の語で5つに分け、種5つの平均）。
python3 e113_box.py E101_RESULTS_DIR e101_items.json DOM out.json
組：一覧なしの鎖（N）の段（意味→次の語の意味、重み＝段の NLI。一種の段と「〜にある」の段の両方）と、着き（点→全体・分類、重み＝着きの NLI、点×全部の分類）。
答え漏れを防ぐ：その組の物の語の、物の語そのものの点からの着きは学習から外す。WordNet も正解も使わない。"""
import json, sys, os, gzip, time, numpy as np




def torch_threads():
    import torch; torch.set_num_threads(1)

lgz = lambda d, n: json.load(gzip.open(os.path.join(d, n + ".json.gz"), "rt", encoding="utf-8"))


def main(rdir, items_path, dom, out, only_fold=None):
    torch_threads()
    items = [it for it in json.load(open(items_path))["items"] if it["dom"] == dom]
    res = lgz(rdir, f"E101_res_{dom}")
    ST = {}
    for d, tag, w, k, g, t, e in lgz(rdir, "E101_steps"):
        if d == dom and tag == "N": ST[(w, k, g, t)] = e
    A = np.array(res["arrive"], dtype=np.float32).max(axis=2)
    nid = {tuple(n): i for i, n in enumerate(res["nodes"])}; classes = res["classes"]; cidx = {c: j for j, c in enumerate(classes)}
    ch = {key.split("|", 1)[1]: v for key, v in res["chains"].items() if key.startswith("N|")}
    used = sorted({tuple(n) for nd, lk in ch.values() for n in nd})
    idx = {n: i for i, n in enumerate(used)}; N = len(used) + len(classes)
    links = sorted({(tuple(a), tuple(b), t) for nd, lk in ch.values() for a, b, t in lk if tuple(a) != tuple(b)})
    sp = np.array([[idx[a], idx[b]] for a, b, t in links]); sw = np.array([ST.get((a[0], a[1], b[0], t), 0.0) for a, b, t in links])
    ents = sorted({it["ent"] for it in items}); rng = np.random.default_rng(0)
    fold = {e: f % 5 for f, e in enumerate(rng.permutation(ents))}
    print(dom, "点", N, "段", len(links), "着き", len(used) * len(classes), flush=True)
    roots_of = {}
    for n in used: roots_of.setdefault(n[0], []).append(idx[n])
    out_s = np.zeros((5, len(items)))
    t0 = time.time()
    for f in ([int(only_fold)] if only_fold is not None else range(5)):
        drop = {e for e in ents if fold[e] == f}
        ap, aw = [], []
        for n in used:
            if n[0] in drop: continue
            for c, j in cidx.items():
                ap.append([idx[n], len(used) + j]); aw.append(float(A[nid[n], j]))
        P = np.vstack([sp, np.array(ap)]); W = np.concatenate([sw, np.array(aw)])
        te = [i for i, it in enumerate(items) if fold[it["ent"]] == f]
        for s in range(5):
            m = BXM.train(N, P, W, seed=100 * f + s)
            for i in te:
                x = items[i]["ent"]; roots = roots_of.get(x, [])
                out_s[s, i] = BXM.score(m, np.array(roots), np.full(len(roots), len(used) + cidx[items[i]["cls"]])).max() if roots else 0.0
        print(f"  {dom} 組{f + 1}/5 {time.time() - t0:.0f}秒", flush=True)
    if only_fold is not None:   # 組ごとに別の処理で流した時：その組の問題の値だけ保存（あとで merge でまとめる）
        te = [i for i, it in enumerate(items) if fold[it["ent"]] == int(only_fold)]
        json.dump({"dom": dom, "fold": int(only_fold), "idx": te, "vals": out_s[:, te].round(5).tolist()}, open(out, "w")); return
    json.dump({"dom": dom, "ids": [it["id"] for it in items], "box": out_s.mean(0).round(5).tolist(),
               "seed_spread": out_s.std(0).mean().item()}, open(out, "w"))


def merge(items_path, dom, out, *parts):
    items = [it for it in json.load(open(items_path))["items"] if it["dom"] == dom]
    V = np.full((5, len(items)), np.nan)
    for p in parts:
        d = json.load(open(p)); V[:, d["idx"]] = np.array(d["vals"])
    assert not np.isnan(V).any()
    json.dump({"dom": dom, "ids": [it["id"] for it in items], "box": V.mean(0).round(5).tolist(),
               "seed_spread": V.std(0).mean().item()}, open(out, "w"))

In [ ]:
for dom in "GDP":
    path = f"{OUT}/box_{dom}.json"
    if os.path.exists(path):
        print(dom, "済み"); continue
    t = time.time(); main(R101, "/content/e101_items_nogold.json", dom, path)
    print(dom, f"{time.time() - t:.0f}秒", flush=True)